# Architecture




In [ ]:
GrandNexus/
│
├── core/
│   ├── nexus_core.py
│   ├── errorRecoveryManager.py
│   └── registry.py
│
├── memory/
│   ├── memory_manager.py
│   ├── episodic.py
│   ├── semantic.py
│   └── working.py
│
├── cognition/
├── cognition_manager.py      # Orchestrateur central de la cognition
│
├── perception/
│   ├── perception_core.py    # Module de base pour l'interprétation des entrées
│   ├── symbolic_parser.py    # Parsing des expressions symboliques/mathématiques
│   ├── code_analyzer.py      # Analyse de code et structures formelles
│   ├── pattern_recognizer.py # Détection de motifs récurrents
│   └── filters/              # Filtres attentionnels et contextuels
│
├── reasoning/
│   ├── reasoning_core.py     # Core du système de raisonnement
│   ├── neural_reasoner.py    # Raisonnement par réseaux neuronaux (embeddings)
│   ├── hybrid_inference.py   # Module neuro-symbolique (LNN, DeepProbLog, etc.)
│   ├── planning.py           # Planification et décomposition d'objectifs
│   └── uncertainty.py        # Gestion de l'incertitude et raisonnement probabiliste
│
├── learning/
│   ├── learning_core.py      # Core du système d'apprentissage
│   ├── continuous_learner.py # Apprentissage continu (Avalanche)
│   ├── meta_learner.py       # Méta-apprentissage (learn2learn)
│   ├── reinforcement.py      # Apprentissage par renforcement
│   ├── intrinsic_motivation.py # Motivation et curiosité computationnelle
│   └── curriculum.py         # Gestion du curriculum d'apprentissage
│
└── metacognition/
    ├── monitor.py            # Surveillance des processus cognitifs
    ├── confidence.py         # Évaluation de la confiance/incertitude
    ├── adaptation.py         # Ajustement dynamique des paramètres
    └── reflection.py         # Auto-analyse des processus de raisonnement
│
├── graph/
│   ├── graph_engine.py
│   ├── pattern_mining.py
│   ├── graph_pattern_interface.py
│   └── graph_module_manager.py
│
├── computation/
│   └── resource_manager.py
│
└── llm_interface/
    ├── prompt_templates.py
    ├── response_parser.py
    └── knowledge_extractor.py


SyntaxError: invalid character '│' (U+2502) (<ipython-input-1-e454451c0d31>, line 2)

# Core System

1.   NexusCore
2.   ErrorRecovery



# NexusCore

In [ ]:
import logging
import threading
import time
import uuid
import sqlite3
import json  # For serializing/deserializing module configurations
from typing import Dict, Any, Optional, List, Callable, Union, Tuple, Set  # Importer Set
import heapq  # Correctly manage priorities


class NexusCore:
    """
    Central class orchestrating the GrandNexus system.
    Provides a unified access point for module management,
    lifecycle control, and inter-module communication.

    Features:
        - Registration of modules with optional dependencies
        - Topological sorting of dependencies for startup and shutdown
        - Hooks for lifecycle events (pre_start, post_start, pre_stop, post_stop)
        - Priority-based message queue with optional background processing
        - Persistence of module state, dependencies, and configuration using SQLite
        - Extensive logging and error checks
        - Convenience methods to add or list modules, get module status and configuration
    """

    def __init__(self, config: Optional[Dict[str, Any]] = None, enable_async_messaging: bool = True, db_file: str = "nexuscore.db"):
        """
        Initializes the NexusCore with an optional configuration.

        Args:
            config (dict, optional):  Global configuration dictionary. Defaults to None.
            enable_async_messaging (bool): If True, use a background thread to process the message queue. Defaults to True.
            db_file (str): Path to the SQLite database file. Defaults to "nexuscore.db".
        """
        self.instance_id: str = str(uuid.uuid4())[:8]
        self.logger: logging.Logger = self._setup_logger()
        self.config: Dict[str, Any] = config or {}
        self.modules: Dict[str, Dict[str, Any]] = {}  # {module_name: {instance, dependencies, status, config}}
        self.running: bool = False
        self.start_time: float = 0.0
        self.stop_event: threading.Event = threading.Event()

        # Hooks for lifecycle customization
        self.pre_start_hooks: List[Callable[['NexusCore'], None]] = []
        self.post_start_hooks: List[Callable[['NexusCore'], None]] = []
        self.pre_stop_hooks: List[Callable[['NexusCore'], None]] = []
        self.post_stop_hooks: List[Callable[['NexusCore'], None]] = []

        # Priority-based Message queue
        self.message_queue: List[Tuple[int, Dict[str, Any]]] = []  # (priority, message)
        self.lock: threading.RLock = threading.RLock()

        self.enable_async_messaging: bool = enable_async_messaging
        self.message_thread: Optional[threading.Thread] = None

        self.db_file: str = db_file
        self._init_db()
        self._load_modules_from_db()

        self.logger.info(f"NexusCore initialized with ID={self.instance_id}, Async Messaging: {enable_async_messaging}")

        # Initialize error recovery manager
        self.error_recovery = ErrorRecoveryManager(self)

        self.logger.info(f"NexusCore initialized with ID={self.instance_id}, Async Messaging: {enable_async_messaging}")


    def _setup_logger(self) -> logging.Logger:
        """Configures and returns a logger for the core."""
        logger = logging.getLogger("NexusCore")
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - [%(name)s] - %(levelname)s - %(message)s')
            handler.setFormatter(formatter)
            logger.addHandler(handler)
            logger.setLevel(logging.INFO)
        return logger

    def _init_db(self):
        """Initializes the SQLite database."""
        try:
            with sqlite3.connect(self.db_file) as conn:
                cursor = conn.cursor()
                cursor.execute("""
                    CREATE TABLE IF NOT EXISTS modules (
                        name TEXT PRIMARY KEY,
                        status TEXT,
                        configuration TEXT,  -- Store configuration as JSON
                        last_updated REAL
                    )
                """)
                cursor.execute("""
                    CREATE TABLE IF NOT EXISTS dependencies (
                        module_name TEXT,
                        dependency_name TEXT,
                        FOREIGN KEY (module_name) REFERENCES modules(name) ON DELETE CASCADE,
                        FOREIGN KEY (dependency_name) REFERENCES modules(name) ON DELETE CASCADE,
                        PRIMARY KEY (module_name, dependency_name)
                    )
                """)
                # You could add a 'metadata' table here, similar to nsytz's version.
                conn.commit()
            self.logger.info("Database initialized successfully.")

        except sqlite3.Error as e:
            self.logger.error(f"Database initialization failed: {e}")
            raise  # Re-raise to prevent startup if DB init fails

    def _load_modules_from_db(self):
        """Loads module information from the database."""
        try:
            with sqlite3.connect(self.db_file) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT name, status, configuration FROM modules")
                for name, status, configuration in cursor.fetchall():
                    self.modules[name] = {
                        "instance": None,  # Placeholder, will be filled on register
                        "dependencies": [],  # Will be filled below
                        "status": status,
                        "config": json.loads(configuration) if configuration else {}  # Load config
                    }

                cursor.execute("SELECT module_name, dependency_name FROM dependencies")
                for module_name, dependency_name in cursor.fetchall():
                    if module_name in self.modules:
                        self.modules[module_name]["dependencies"].append(dependency_name)
        except sqlite3.Error as e:
            self.logger.error(f"Error loading modules from database: {e}")
            #  Don't re-raise here.  Allow registration, but log the error

    def _persist_module(self, name: str):
        """Persists module information to the database."""
        with self.lock:
            if name not in self.modules:
                return # Nothing to persist.

            module_info = self.modules[name]
            status = module_info["status"]
            config = module_info.get("config", {})  # Get config, default to empty dict
            config_json = json.dumps(config)       # Serialize configuration
            dependencies = module_info["dependencies"]

            try:
                with sqlite3.connect(self.db_file) as conn:
                    cursor = conn.cursor()
                    cursor.execute("""
                        INSERT OR REPLACE INTO modules (name, status, configuration, last_updated)
                        VALUES (?, ?, ?, ?)
                    """, (name, status, config_json, time.time()))

                    # Clear existing dependencies
                    cursor.execute("DELETE FROM dependencies WHERE module_name = ?", (name,))
                    # Insert new dependencies
                    for dep in dependencies:
                        cursor.execute("INSERT INTO dependencies (module_name, dependency_name) VALUES (?, ?)", (name, dep))
                    conn.commit()
            except sqlite3.Error as e:
                self.logger.error(f"Error persisting module '{name}' to database: {e}")
                # Consider if you want to re-raise, or handle the error here

    def register_module(self, name: str, module: Any, dependencies: Optional[List[str]] = None, config: Optional[Dict[str, Any]] = None) -> None:
        """
        Registers a module in the system.

        Args:
            name (str): Unique identifier for the module.
            module (Any): Instance of the module to register.
            dependencies (List[str], optional): List of module names this module depends on. Defaults to None.
            config (Dict[str, Any], optional): Initial configuration for the module.
        """
        with self.lock:
            if name in self.modules:
                self.logger.warning(f"Module '{name}' is already registered. Overwriting the existing module entry.")

            # Check dependencies
            if dependencies:
                for dep in dependencies:
                    if dep not in self.modules and dep != name: # Allow self-dependency.
                        self.logger.error(f"Dependency '{dep}' not found for module '{name}'.  Registration may fail.")
                        #  Don't raise an exception here; allow the system to try to continue.

            self.modules[name] = {
                "instance": module,
                "dependencies": dependencies or [],
                "status": "registered",
                "config": config or {}  # Store initial configuration
            }
            self.logger.info(f"Module '{name}' registered successfully with status='registered'.")
            self._persist_module(name)  # Persist after *successful* registration

    def get_module(self, name: str) -> Optional[Any]:
        """Retrieves a module by its name."""
        with self.lock:
            module_info = self.modules.get(name)
            if module_info:
                return module_info["instance"]
            return None

    def get_module_config(self, name: str) -> Optional[Dict[str, Any]]:
        """Retrieves the configuration of a module."""
        with self.lock:
            module_info = self.modules.get(name)
            if module_info:
                return module_info.get("config")  # Return config, or None
            return None

    def set_module_config(self, name: str, config: Dict[str, Any]) -> None:
        """
        Sets or updates the configuration for a module.

        Args:
             name (str): The name of the module
             config (Dict[str, Any]):  The configuration dictionary
        """
        with self.lock:
            if name not in self.modules:
                self.logger.error(f"Cannot set configuration: Module '{name}' not found.")
                return

            module_info = self.modules[name]
            module_info["config"] = config

            # Call set_configuration if available.
            instance = module_info['instance']
            if instance and hasattr(instance, 'set_configuration') and callable(getattr(instance, 'set_configuration')):
                try:
                    instance.set_configuration(config)
                    self.logger.info(f"Module '{name}' configuration updated and applied.")
                except Exception as e:
                     self.logger.error(f"Error applying configuration to module '{name}': {e}")
                     #  Don't change the module status here, as the module might still be functional

            self._persist_module(name) # Persist new config
            self.logger.info(f"Module '{name}' configuration updated in registry.")


    def list_modules(self) -> List[str]:
        """Lists all registered modules by name."""
        with self.lock:
            return list(self.modules.keys())

    def start(self) -> bool:
      """
      Starts the NexusCore and all registered modules in the order of their dependencies.

      Returns:
          bool: True if started successfully, False otherwise
      """
      if self.running:
          self.logger.warning("NexusCore is already running.")
          return True

      self.logger.info(f"Starting NexusCore [{self.instance_id}]")

      # Execute pre-start hooks
      pre_hooks_success = self._execute_hooks('pre_start')
      if not pre_hooks_success:
          self.logger.warning("Some pre-start hooks failed, but continuing with startup")

      self.stop_event.clear()
      self.start_time = time.time()
      self.running = True

      # Determine startup order based on dependencies
      try:
          start_order = self._resolve_dependencies()
          self.logger.info(f"Module startup order resolved to: {start_order}")
      except ValueError as e:
          self.logger.error(f"Failed to start NexusCore: {e}")
          self.running = False  # Reset running flag
          return False  # Exit if dependency resolution fails

      # Start each module in the resolved order
      start_success = True
      for module_name in start_order:
          if not self._start_module(module_name):
              start_success = False
              # Continue starting other modules unless configured otherwise
              if self.config.get("fail_fast", False):
                  break

      # Start the background thread for message processing if enabled
      if self.enable_async_messaging:
          self._start_async_message_processing()

      # Execute post-start hooks
      post_hooks_success = self._execute_hooks('post_start')
      if not post_hooks_success:
          self.logger.warning("Some post-start hooks failed")

      if start_success:
          self.logger.info(f"NexusCore [{self.instance_id}] started successfully.")
      else:
          self.logger.warning(f"NexusCore [{self.instance_id}] started with some module failures.")

      return start_success

    def stop(self) -> bool:
      """
      Gracefully stops NexusCore and all modules.

      Returns:
          bool: True if stopped successfully, False otherwise
      """
      if not self.running:
          self.logger.warning("NexusCore is not running.")
          return True

      self.logger.info(f"Stopping NexusCore [{self.instance_id}]")

      # Execute pre-stop hooks
      pre_hooks_success = self._execute_hooks('pre_stop')
      if not pre_hooks_success:
          self.logger.warning("Some pre-stop hooks failed, but continuing with shutdown")

      self.running = False
      self.stop_event.set()

      # Stop the asynchronous message processing thread if running
      if self.message_thread and self.message_thread.is_alive():
          self.logger.info("Waiting for the message processing thread to finish...")
          self.message_thread.join(timeout=5.0)  # Give it some time to exit
          if self.message_thread.is_alive():
              self.logger.warning("Message processing thread did not exit cleanly within 5 seconds.")

      # Determine the order to stop modules (reverse of the startup order)
      stop_success = True
      try:
          stop_order = list(reversed(self._resolve_dependencies()))
          self.logger.info(f"Module shutdown order resolved to: {stop_order}")
      except ValueError as e:
          self.logger.error(f"Error during shutdown dependency resolution: {e}. Forcing shutdown of known modules.")
          stop_order = list(self.modules.keys())  # Attempt to stop all registered, even with bad dependency graph.

      # Stop modules in reverse order
      for module_name in stop_order:
          if not self._stop_module(module_name):
              stop_success = False
              # Continue stopping other modules

      # Execute post-stop hooks
      post_hooks_success = self._execute_hooks('post_stop')
      if not post_hooks_success:
          self.logger.warning("Some post-stop hooks failed")

      uptime = time.time() - self.start_time
      self.logger.info(f"NexusCore [{self.instance_id}] stopped. Total uptime: {uptime:.2f}s")

      return stop_success

      # Arrêter le gestionnaire d'erreurs
      self.error_recovery.shutdown()
      return success

    # Amélioration pour NexusCore._resolve_dependencies()
    def _resolve_dependencies(self) -> List[str]:
        """
        Determines the startup (and shutdown) order of modules based on dependencies.
        Uses topological sorting (Kahn's Algorithm) with improved cycle detection and error handling.

        Returns:
            List[str]: Names of modules in the correct startup order.
        Raises:
            ValueError: If a circular dependency is detected, with detailed information about the cycle.
        """
        with self.lock:
            # Build the dependency graph and track reverse dependencies for cycle detection
            graph = {name: set(info["dependencies"]) for name, info in self.modules.items()}
            reverse_graph = {name: set() for name in self.modules}

            # Populate reverse graph for cycle detection
            for node, deps in graph.items():
                for dep in deps:
                    if dep in reverse_graph:  # Check if dependency is registered
                        reverse_graph[dep].add(node)
                    else:
                        self.logger.warning(f"Module '{node}' depends on unknown module '{dep}'")

            # Kahn's Algorithm with cycle detection
            in_degree = {node: 0 for node in graph}
            for node in graph:
                for dep in graph.get(node, []):
                    if dep in in_degree:  # Check if dependency is registered
                        in_degree[dep] += 1

            # Create priority queue for processing nodes
            queue = [(0, node) for node, degree in in_degree.items() if degree == 0]
            heapq.heapify(queue)
            order = []

            # Process nodes in priority order
            while queue:
                _, node = heapq.heappop(queue)
                order.append(node)

                for neighbor in list(graph.get(node, [])):
                    if neighbor in in_degree:  # Check if neighbor is registered
                        in_degree[neighbor] -= 1
                        if in_degree[neighbor] == 0:
                            # Use module priority in heap if available, default to 0
                            priority = self.modules.get(neighbor, {}).get("priority", 0)
                            heapq.heappush(queue, (priority, neighbor))

            # Check for circular dependencies
            if len(order) != len(self.modules):
                # Find and report the cycle
                remaining = set(self.modules.keys()) - set(order)
                cycles = self._find_dependency_cycles(remaining, graph, reverse_graph)
                cycle_str = '; '.join([' -> '.join(cycle) for cycle in cycles])
                raise ValueError(f"Circular dependency detected: {cycle_str}")

            return order

    # New helper method for cycle detection
    def _find_dependency_cycles(self, nodes: Set[str], graph: Dict[str, Set[str]],
                              reverse_graph: Dict[str, Set[str]]) -> List[List[str]]:
        """
        Helper method to find and report cycles in the dependency graph.

        Args:
            nodes: Set of nodes that are part of a cycle
            graph: Forward dependency graph
            reverse_graph: Reverse dependency graph for cycle detection

        Returns:
            List of cycles, where each cycle is a list of module names
        """
        cycles = []
        visited = set()

        def dfs(node, path, visited_in_path):
            if node in visited_in_path:
                # Found a cycle
                cycle_start = path.index(node)
                cycles.append(path[cycle_start:] + [node])
                return

            visited.add(node)
            visited_in_path.add(node)
            path.append(node)

            for neighbor in graph.get(node, []):
                if neighbor in nodes and neighbor not in visited:
                    dfs(neighbor, path, visited_in_path.copy())

            path.pop()
            visited_in_path.remove(node)

        # Try to find cycles starting from each remaining node
        for node in nodes:
            if node not in visited:
                dfs(node, [], set())

        return cycles


    def _start_module(self, name: str) -> bool:
      """
      Starts a specific module by name, calling its start() method if available.

      Args:
          name (str): Name of the module to start.

      Returns:
          bool: True if module started successfully, False otherwise
      """
      with self.lock:
          if name not in self.modules:
              self.logger.error(f"Cannot start '{name}': Module not found.")
              return False

          module_info = self.modules[name]
          if module_info["status"] == "running":
              self.logger.debug(f"Module '{name}' is already running.")
              return True

          instance = module_info["instance"]
          try:
              start_method = getattr(instance, "start", None)
              if callable(start_method):
                  start_method()
                  self.logger.info(f"Module '{name}' started successfully.")
              else:
                  self.logger.debug(f"Module '{name}' has no start() method. Marking as 'running' anyway.")
              module_info["status"] = "running"
              self._persist_module(name)  # Persist new status
              return True
          except Exception as e:
              # Appeler error_recovery
              recovery_result = self.error_recovery.handle_error(name, e, "start")
              if not recovery_result["success"]:
                  self.logger.error(f"Error while starting module '{name}': {e}", exc_info=True)
                  module_info["status"] = "error"
                  module_info.setdefault("error_info", {})
                  module_info["error_info"]["last_error"] = str(e)
                  module_info["error_info"]["last_error_time"] = time.time()
                  self._persist_module(name)  # Persist error status
              return recovery_result["success"]


    def _stop_module(self, name: str) -> bool:
      """
      Stops a specific module by name, calling its stop() method if available.

      Args:
          name (str): Name of the module to stop.

      Returns:
          bool: True if module stopped successfully, False otherwise
      """
      with self.lock:
          if name not in self.modules:
              self.logger.error(f"Cannot stop '{name}': Module not found.")
              return False

          module_info = self.modules[name]
          if module_info["status"] != "running":
              self.logger.debug(f"Module '{name}' is not currently running. Status is '{module_info['status']}'.")
              return True  # Consider it a success if it's already not running

          instance = module_info["instance"]
          try:
              stop_method = getattr(instance, "stop", None)
              if callable(stop_method):
                  stop_method()
                  self.logger.info(f"Module '{name}' stopped successfully.")
              else:
                  self.logger.debug(f"Module '{name}' has no stop() method. Marking as 'stopped'.")
              module_info["status"] = "stopped"
              self._persist_module(name)  # Persist
              return True
          except Exception as e:
              self.logger.error(f"Error while stopping module '{name}': {e}", exc_info=True)
              module_info["status"] = "error"
              module_info.setdefault("error_info", {})
              module_info["error_info"]["last_error"] = str(e)
              module_info["error_info"]["last_error_time"] = time.time()
              self._persist_module(name)  # Persist
              return False

    # Améliorations pour les hooks du cycle de vie

    def add_hook(self, hook_type: str, hook_func: Callable[['NexusCore'], None]) -> bool:
        """
        Adds a hook to the core lifecycle.

        Args:
            hook_type (str): Type of hook ('pre_start', 'post_start', 'pre_stop', 'post_stop').
            hook_func (Callable[['NexusCore'], None]): Function to execute during the hook.

        Returns:
            bool: True if the hook was added successfully, False otherwise.
        """
        valid_hooks = {'pre_start', 'post_start', 'pre_stop', 'post_stop'}

        if hook_type not in valid_hooks:
            self.logger.warning(f"Unknown hook type: {hook_type}. Valid types are: {valid_hooks}")
            return False

        if not callable(hook_func):
            self.logger.warning(f"Hook function must be callable")
            return False

        hook_list = getattr(self, f"{hook_type}_hooks")

        # Avoid duplicate hooks
        if hook_func in hook_list:
            self.logger.debug(f"Hook function already registered for {hook_type}")
            return True

        hook_list.append(hook_func)
        self.logger.debug(f"Added {hook_type} hook: {hook_func.__name__}")
        return True

    def remove_hook(self, hook_type: str, hook_func: Callable[['NexusCore'], None]) -> bool:
        """
        Removes a previously added hook from the core lifecycle.

        Args:
            hook_type (str): Type of hook ('pre_start', 'post_start', 'pre_stop', 'post_stop').
            hook_func (Callable[['NexusCore'], None]): Function to remove.

        Returns:
            bool: True if the hook was removed successfully, False if not found.
        """
        valid_hooks = {'pre_start', 'post_start', 'pre_stop', 'post_stop'}

        if hook_type not in valid_hooks:
            self.logger.warning(f"Unknown hook type: {hook_type}. Valid types are: {valid_hooks}")
            return False

        hook_list = getattr(self, f"{hook_type}_hooks")

        if hook_func not in hook_list:
            self.logger.debug(f"Hook function not found in {hook_type} hooks")
            return False

        hook_list.remove(hook_func)
        self.logger.debug(f"Removed {hook_type} hook: {hook_func.__name__}")
        return True

    def _execute_hooks(self, hook_type: str) -> bool:
        """
        Internal method to execute all hooks of a given type.

        Args:
            hook_type (str): Type of hooks to execute

        Returns:
            bool: True if all hooks executed successfully, False if any hooks failed
        """
        hook_list = getattr(self, f"{hook_type}_hooks")
        all_succeeded = True

        for i, hook in enumerate(hook_list):
            try:
                self.logger.debug(f"Executing {hook_type} hook {i+1}/{len(hook_list)}: {hook.__name__}")
                hook(self)
            except Exception as e:
                all_succeeded = False
                self.logger.error(f"Error in {hook_type} hook {hook.__name__}: {e}", exc_info=True)
                # Continue with other hooks even if one fails

        return all_succeeded

    def send_message(self, source: str, target: str, message_type: str, content: Any, priority: int = 1) -> None:
        """
        Sends a message from one module to another.  Adds the message to the priority queue.

        Args:
            source (str): Source module name.
            target (str): Target module name.
            message_type (str): Type/category of the message.
            content (Any): The message payload.
            priority (int):  Message priority (lower is higher).
        """

        message = {
            "source": source,
            "target": target,
            "type": message_type,
            "content": content,
            "timestamp": time.time()
        }

        with self.lock:
            heapq.heappush(self.message_queue, (priority, message))  # Use heapq
            self.logger.debug(f"Message from '{source}' to '{target}' (type: {message_type}, priority: {priority}) queued.")

        # If asynchronous messaging is disabled, process messages immediately
        if not self.enable_async_messaging:
            self._process_messages()


    def _process_messages(self):
        """
        Process messages from the queue.  This is either called immediately after
        send_message (if async is disabled) or runs in a separate thread (if async is enabled).
        """
        with self.lock:
            while self.message_queue:
                priority, message = heapq.heappop(self.message_queue) # Use heappop
                self._deliver_message(message)



    def _deliver_message(self, message: Dict[str, Any]) -> None:
        """
        Deliver the message directly to the target's handle_message method, if it exists.
        This is used both in synchronous mode or by the background thread in asynchronous mode.

        Args:
            message (dict): The message to be delivered.
        """
        target = message["target"]
        with self.lock:
            if target in self.modules:
                target_module = self.modules[target]["instance"]
                handle_method = getattr(target_module, "handle_message", None)
                if callable(handle_method):
                    try:
                        handle_method(message)
                    except Exception as e:
                        self.logger.error(f"Error processing message by '{target}': {e}")

    def _start_async_message_processing(self) -> None:
        """
        Spawns a background thread that continuously processes messages from the queue
        until the core is stopped.  The thread calls _deliver_message for each message.
        """

        def message_loop():
            self.logger.info("Asynchronous message processing thread started.")
            while self.running and not self.stop_event.is_set():
                self._process_messages()  # Process all available messages
                time.sleep(0.01)  # Avoid busy-waiting if queue is empty, check frequently
            self.logger.info("Asynchronous message processing thread exiting.")

        self.message_thread = threading.Thread(target=message_loop, daemon=True)
        self.message_thread.start()

    def get_status(self) -> Dict[str, Any]:
        """
        Retrieves the current status of NexusCore and its modules,
        including health information from ErrorRecoveryManager.

        Returns:
            Dict[str, Any]: Dictionary containing status information.
        """
        with self.lock:
            if self.running:
                uptime = time.time() - self.start_time
            else:
                uptime = 0

            modules_status = {}
            for name, info in self.modules.items():
                modules_status[name] = {
                    "status": info["status"],
                    "dependencies": info["dependencies"],
                    "config": info.get("config", {})  # Include configuration
                }

                # Add health information from error recovery manager
                if hasattr(self, "error_recovery"):
                    health_info = self.error_recovery.get_module_health_status(name)
                    modules_status[name]["health"] = health_info

                # If the module has a get_status() method, include that info
                instance = info["instance"]
                get_status_method = getattr(instance, "get_status", None)
                if callable(get_status_method):
                    try:
                        module_status = get_status_method()
                        if isinstance(module_status, dict):
                            modules_status[name].update(module_status)
                    except Exception as e:
                        self.logger.error(f"Error retrieving status for '{name}': {e}")

            status = {
                "instance_id": self.instance_id,
                "running": self.running,
                "uptime": uptime,
                "modules": modules_status,
                "queue_size": len(self.message_queue),
                "async_messaging_enabled": self.enable_async_messaging
            }

            # Add recovery enabled flag if error recovery is available
            if hasattr(self, "error_recovery"):
                status["recovery_enabled"] = True

            return status



    def update_module_status(self, module_name: str, new_status: str):
        """Updates the status of a module and persists it."""
        with self.lock:
            if module_name in self.modules:
                self.modules[module_name]['status'] = new_status
                self._persist_module(module_name)
                self.logger.info(f"Module '{module_name}' status updated to '{new_status}'")
            else:
                self.logger.error(f"Cannot update status: Module '{module_name}' not found")

# Error Recovery

In [ ]:
"""
error_recovery.py - Module de gestion de récupération d'erreurs pour GrandNexus

Ce module implémente un système de récupération d'erreurs qui collabore avec NexusCore
sans modifier son paradigme d'orchestration fondamental. Il fournit des stratégies
de récupération adaptatives, une surveillance de l'état des modules, et des mécanismes
d'isolation pour les modules problématiques.
"""

import threading
import time
import logging
import uuid
from typing import Dict, List, Any, Optional, Callable, Tuple, Set

class ErrorRecoveryManager:
    """
    Gestionnaire de récupération d'erreurs pour GrandNexus.

    Cette classe implémente diverses stratégies pour récupérer des erreurs,
    y compris les tentatives automatiques avec backoff exponentiel, la surveillance
    de l'état des modules, et l'isolation des modules problématiques.

    Elle est conçue pour collaborer avec NexusCore sans modifier son paradigme
    d'orchestration fondamental.
    """

    # Niveaux de gravité des erreurs
    SEVERITY_LOW = 1     # Problèmes mineurs, peuvent être ignorés
    SEVERITY_MEDIUM = 2  # Problèmes significatifs, doivent être traités
    SEVERITY_HIGH = 3    # Problèmes critiques, nécessitent une attention immédiate

    # Stratégies de récupération
    STRATEGY_IGNORE = "ignore"           # Journaliser et continuer
    STRATEGY_RETRY = "retry"             # Réessayer l'opération
    STRATEGY_RESTART = "restart"         # Redémarrer le module
    STRATEGY_ISOLATE = "isolate"         # Isoler le module
    STRATEGY_NOTIFY = "notify"           # Notifier les administrateurs

    def __init__(self, core_reference):
        """
        Initialise le gestionnaire de récupération d'erreurs.

        Args:
            core_reference: Référence à l'instance NexusCore (utilisée uniquement pour les callbacks)
        """
        self.core = core_reference
        self.logger = logging.getLogger("ErrorRecoveryManager")

        # Suivi de l'état des modules et des tentatives de récupération
        self.module_health: Dict[str, Dict[str, Any]] = {}
        self.recovery_attempts: Dict[str, Dict[str, Any]] = {}

        # Configuration des stratégies de récupération
        self.max_retry_attempts = 3
        self.max_restart_attempts = 2
        self.backoff_base = 2  # Base pour le backoff exponentiel
        self.health_check_interval = 60  # secondes

        # Démarrer le thread de surveillance de l'état
        self.stop_monitoring = threading.Event()
        self.monitoring_thread = threading.Thread(target=self._health_monitoring_loop, daemon=True)
        self.monitoring_thread.start()

        self.logger.info("ErrorRecoveryManager initialisé")

    def handle_error(self, module_name: str, error: Exception, operation: str,
                    severity: int = SEVERITY_MEDIUM) -> Dict[str, Any]:
        """
        Gère une erreur survenue dans un module.

        Args:
            module_name: Nom du module où l'erreur s'est produite
            error: L'exception qui a été levée
            operation: L'opération qui était en cours (ex: "start", "stop", "message")
            severity: Niveau de gravité de l'erreur

        Returns:
            Dict contenant l'action de récupération prise et le résultat
        """
        self.logger.error(f"Erreur dans le module '{module_name}' pendant {operation}: {error}")

        # Initialiser le suivi de l'état du module si inexistant
        if module_name not in self.module_health:
            self.module_health[module_name] = {
                "error_count": 0,
                "last_error_time": 0,
                "consecutive_errors": 0,
                "status": "healthy"
            }

        # Mettre à jour l'état du module
        health = self.module_health[module_name]
        health["error_count"] += 1
        health["last_error_time"] = time.time()
        health["consecutive_errors"] += 1

        # Déterminer la stratégie de récupération en fonction de la gravité et du contexte
        strategy = self._determine_strategy(module_name, error, operation, severity)

        # Exécuter la stratégie de récupération
        result = self._execute_strategy(module_name, strategy, error, operation)

        # Mettre à jour l'état du module en fonction du résultat de la récupération
        if result.get("success", False):
            health["consecutive_errors"] = 0
            if health["status"] == "degraded":
                health["status"] = "recovering"
        else:
            if health["consecutive_errors"] > 3:
                health["status"] = "degraded"
            if health["consecutive_errors"] > 5:
                health["status"] = "critical"

        return result

    def _determine_strategy(self, module_name: str, error: Exception,
                           operation: str, severity: int) -> str:
        """
        Détermine la stratégie de récupération appropriée en fonction du contexte.

        Args:
            module_name: Nom du module
            error: L'exception qui s'est produite
            operation: L'opération en cours
            severity: Niveau de gravité de l'erreur

        Returns:
            La stratégie de récupération sélectionnée
        """
        # Obtenir les informations sur l'état du module
        health = self.module_health.get(module_name, {})
        consecutive_errors = health.get("consecutive_errors", 0)

        # Obtenir les informations sur les tentatives de récupération
        if module_name not in self.recovery_attempts:
            self.recovery_attempts[module_name] = {
                "retry_count": 0,
                "restart_count": 0,
                "last_attempt_time": 0
            }
        attempts = self.recovery_attempts[module_name]

        # Logique pour déterminer la stratégie
        if severity == self.SEVERITY_LOW:
            return self.STRATEGY_IGNORE

        elif severity == self.SEVERITY_MEDIUM:
            # Pour les opérations start/stop, essayer de redémarrer
            if operation in ["start", "stop"]:
                if attempts["restart_count"] < self.max_restart_attempts:
                    return self.STRATEGY_RESTART
                else:
                    return self.STRATEGY_ISOLATE

            # Pour le traitement des messages, essayer de réessayer
            elif operation == "message":
                if attempts["retry_count"] < self.max_retry_attempts:
                    return self.STRATEGY_RETRY
                else:
                    return self.STRATEGY_IGNORE

            # Par défaut pour une gravité moyenne
            return self.STRATEGY_RETRY

        elif severity == self.SEVERITY_HIGH:
            # Pour les erreurs critiques, essayer de redémarrer une fois puis isoler
            if attempts["restart_count"] < 1:
                return self.STRATEGY_RESTART
            else:
                return self.STRATEGY_ISOLATE

        # Stratégie par défaut
        return self.STRATEGY_RETRY

    def _execute_strategy(self, module_name: str, strategy: str,
                         error: Exception, operation: str) -> Dict[str, Any]:
        """
        Exécute la stratégie de récupération sélectionnée.

        Args:
            module_name: Nom du module
            strategy: La stratégie de récupération à exécuter
            error: L'exception qui s'est produite
            operation: L'opération en cours

        Returns:
            Dict contenant le résultat de l'action de récupération
        """
        result = {
            "module": module_name,
            "strategy": strategy,
            "operation": operation,
            "success": False,
            "message": ""
        }

        attempts = self.recovery_attempts.get(module_name, {
            "retry_count": 0,
            "restart_count": 0,
            "last_attempt_time": 0
        })

        # Calculer le temps de backoff si nécessaire
        current_time = time.time()
        time_since_last_attempt = current_time - attempts.get("last_attempt_time", 0)

        if strategy == self.STRATEGY_IGNORE:
            result["success"] = True
            result["message"] = "Erreur journalisée et ignorée"

        elif strategy == self.STRATEGY_RETRY:
            # Mettre à jour le compteur de tentatives
            attempts["retry_count"] += 1
            attempts["last_attempt_time"] = current_time

            # Calculer le temps de backoff
            backoff_time = self._calculate_backoff(attempts["retry_count"])

            # Si nous devons attendre, le faire
            if time_since_last_attempt < backoff_time:
                time.sleep(backoff_time - time_since_last_attempt)

            # Réessayer l'opération via un callback à NexusCore
            try:
                # Utiliser les callbacks pour maintenir la séparation des responsabilités
                if operation == "start":
                    success = self._callback_start_module(module_name)
                    if success:
                        result["success"] = True
                        result["message"] = f"Module '{module_name}' démarré avec succès après nouvelle tentative"
                elif operation == "stop":
                    success = self._callback_stop_module(module_name)
                    if success:
                        result["success"] = True
                        result["message"] = f"Module '{module_name}' arrêté avec succès après nouvelle tentative"
                elif operation == "message":
                    # Nous ne pouvons pas réessayer la livraison de message directement ici
                    result["success"] = False
                    result["message"] = "Nouvelle tentative de message non implémentée dans ce contexte"
            except Exception as e:
                result["success"] = False
                result["message"] = f"Nouvelle tentative échouée: {str(e)}"

        elif strategy == self.STRATEGY_RESTART:
            # Mettre à jour le compteur de redémarrages
            attempts["restart_count"] += 1
            attempts["last_attempt_time"] = current_time

            # Calculer le temps de backoff
            backoff_time = self._calculate_backoff(attempts["restart_count"])

            # Si nous devons attendre, le faire
            if time_since_last_attempt < backoff_time:
                time.sleep(backoff_time - time_since_last_attempt)

            # Essayer d'arrêter puis de démarrer le module via des callbacks
            try:
                # D'abord essayer d'arrêter s'il est en cours d'exécution
                module_status = self._callback_get_module_status(module_name)
                if module_status == "running":
                    self._callback_stop_module(module_name)

                # Puis essayer de le démarrer
                success = self._callback_start_module(module_name)
                if success:
                    result["success"] = True
                    result["message"] = f"Module '{module_name}' redémarré avec succès"
                else:
                    result["success"] = False
                    result["message"] = f"Échec du redémarrage du module '{module_name}'"
            except Exception as e:
                result["success"] = False
                result["message"] = f"Redémarrage échoué: {str(e)}"

        elif strategy == self.STRATEGY_ISOLATE:
            try:
                # Marquer le module comme isolé via un callback
                success = self._callback_update_module_status(module_name, "isolated")
                if success:
                    # Journaliser l'isolation
                    self.logger.warning(f"Module '{module_name}' a été isolé en raison d'erreurs persistantes")

                    result["success"] = True
                    result["message"] = f"Module '{module_name}' a été isolé"
                else:
                    result["success"] = False
                    result["message"] = f"Impossible d'isoler: Module '{module_name}' non trouvé"
            except Exception as e:
                result["success"] = False
                result["message"] = f"Isolation échouée: {str(e)}"

        elif strategy == self.STRATEGY_NOTIFY:
            # Ceci s'intégrerait généralement à un système de notification externe
            # Pour l'instant, nous le journalisons simplement de manière visible
            self.logger.critical(f"ERREUR CRITIQUE dans le module '{module_name}': {error}")
            result["success"] = True
            result["message"] = "Notification d'erreur journalisée"

        # Mettre à jour les tentatives de récupération
        self.recovery_attempts[module_name] = attempts

        return result

    def _calculate_backoff(self, attempt_count: int) -> float:
        """
        Calcule le temps de backoff exponentiel basé sur le nombre de tentatives.

        Args:
            attempt_count: Nombre de tentatives jusqu'à présent

        Returns:
            Temps de backoff en secondes
        """
        # Backoff exponentiel avec jitter
        backoff = (self.backoff_base ** attempt_count) * (0.5 + 0.5 * (uuid.uuid4().int % 100) / 100)
        return min(backoff, 60)  # Plafonner à 60 secondes

    def _health_monitoring_loop(self):
        """
        Thread d'arrière-plan qui vérifie périodiquement l'état des modules.
        """
        self.logger.info("Thread de surveillance de l'état démarré")

        while not self.stop_monitoring.is_set():
            try:
                self._check_module_health()
                time.sleep(self.health_check_interval)
            except Exception as e:
                self.logger.error(f"Erreur dans la surveillance de l'état: {e}")
                time.sleep(5)  # Court sommeil en cas d'erreur

        self.logger.info("Thread de surveillance de l'état arrêté")

    def _check_module_health(self):
        """
        Vérifie l'état de tous les modules et tente une récupération si nécessaire.
        """
        # Obtenir la liste des modules et leur état via un callback
        modules_info = self._callback_get_all_modules_info()

        for module_name, module_info in modules_info.items():
            # Ignorer les modules qui ne sont pas en cours d'exécution
            if module_info.get("status") != "running":
                continue

            # Vérifier si le module a une méthode health_check
            has_health_check = self._callback_has_health_check(module_name)

            if has_health_check:
                try:
                    # Appeler la méthode health_check du module via un callback
                    health_status = self._callback_check_module_health(module_name)

                    # Si la vérification de l'état échoue, tenter une récupération
                    if not health_status.get("healthy", True):
                        self.logger.warning(f"Vérification de l'état échouée pour le module '{module_name}': {health_status.get('message', 'Aucun détail')}")

                        # Créer une erreur synthétique pour la récupération
                        error = Exception(f"Vérification de l'état échouée: {health_status.get('message', 'Aucun détail')}")
                        self.handle_error(module_name, error, "health_check", self.SEVERITY_MEDIUM)
                except Exception as e:
                    self.logger.error(f"Erreur pendant la vérification de l'état du module '{module_name}': {e}")

    def reset_recovery_attempts(self, module_name: str):
        """
        Réinitialise les tentatives de récupération pour un module.

        Args:
            module_name: Nom du module
        """
        if module_name in self.recovery_attempts:
            self.recovery_attempts[module_name] = {
                "retry_count": 0,
                "restart_count": 0,
                "last_attempt_time": 0
            }
            self.logger.info(f"Tentatives de récupération réinitialisées pour le module '{module_name}'")

    def get_module_health_status(self, module_name: str = None) -> Dict[str, Any]:
        """
        Obtient l'état de santé d'un module spécifique ou de tous les modules.

        Args:
            module_name: Nom du module, ou None pour tous les modules

        Returns:
            Dict contenant les informations sur l'état de santé
        """
        if module_name:
            return self.module_health.get(module_name, {
                "error_count": 0,
                "last_error_time": 0,
                "consecutive_errors": 0,
                "status": "unknown"
            })
        else:
            return self.module_health

    def shutdown(self):
        """
        Arrête le gestionnaire de récupération d'erreurs.
        """
        self.stop_monitoring.set()
        if self.monitoring_thread.is_alive():
            self.monitoring_thread.join(timeout=5.0)
        self.logger.info("Arrêt d'ErrorRecoveryManager terminé")

    # Méthodes de callback pour maintenir la séparation des responsabilités
    # Ces méthodes appellent NexusCore sans modifier son paradigme d'orchestration

    def _callback_start_module(self, module_name: str) -> bool:
        """
        Callback pour démarrer un module via NexusCore.

        Args:
            module_name: Nom du module à démarrer

        Returns:
            bool: True si le démarrage a réussi, False sinon
        """
        try:
            # Appeler la méthode _start_module de NexusCore
            # Cette approche maintient la séparation des responsabilités
            self.core._start_module(module_name)
            return True
        except Exception as e:
            self.logger.error(f"Callback de démarrage échoué pour '{module_name}': {e}")
            return False

    def _callback_stop_module(self, module_name: str) -> bool:
        """
        Callback pour arrêter un module via NexusCore.

        Args:
            module_name: Nom du module à arrêter

        Returns:
            bool: True si l'arrêt a réussi, False sinon
        """
        try:
            # Appeler la méthode _stop_module de NexusCore
            self.core._stop_module(module_name)
            return True
        except Exception as e:
            self.logger.error(f"Callback d'arrêt échoué pour '{module_name}': {e}")
            return False

    def _callback_get_module_status(self, module_name: str) -> str:
        """
        Callback pour obtenir l'état d'un module via NexusCore.

        Args:
            module_name: Nom du module

        Returns:
            str: État du module
        """
        try:
            # Accéder à l'état du module via NexusCore
            with self.core.lock:
                if module_name in self.core.modules:
                    return self.core.modules[module_name]["status"]
                return "unknown"
        except Exception as e:
            self.logger.error(f"Callback d'obtention d'état échoué pour '{module_name}': {e}")
            return "unknown"

    def _callback_update_module_status(self, module_name: str, status: str) -> bool:
        """
        Callback pour mettre à jour l'état d'un module via NexusCore.

        Args:
            module_name: Nom du module
            status: Nouvel état

        Returns:
            bool: True si la mise à jour a réussi, False sinon
        """
        try:
            # Utiliser la méthode update_module_status de NexusCore si elle existe
            if hasattr(self.core, "update_module_status") and callable(getattr(self.core, "update_module_status")):
                self.core.update_module_status(module_name, status)
                return True

            # Sinon, mettre à jour directement (moins idéal)
            with self.core.lock:
                if module_name in self.core.modules:
                    self.core.modules[module_name]["status"] = status
                    self.core._persist_module(module_name)
                    return True
                return False
        except Exception as e:
            self.logger.error(f"Callback de mise à jour d'état échoué pour '{module_name}': {e}")
            return False

    def _callback_get_all_modules_info(self) -> Dict[str, Dict[str, Any]]:
        """
        Callback pour obtenir les informations sur tous les modules via NexusCore.

        Returns:
            Dict: Informations sur tous les modules
        """
        try:
            # Accéder aux informations sur les modules via NexusCore
            with self.core.lock:
                return {name: info.copy() for name, info in self.core.modules.items()}
        except Exception as e:
            self.logger.error(f"Callback d'obtention des informations sur les modules échoué: {e}")
            return {}

    def _callback_has_health_check(self, module_name: str) -> bool:
        """
        Callback pour vérifier si un module a une méthode health_check.

        Args:
            module_name: Nom du module

        Returns:
            bool: True si le module a une méthode health_check, False sinon
        """
        try:
            # Vérifier si le module a une méthode health_check
            with self.core.lock:
                if module_name in self.core.modules:
                    instance = self.core.modules[module_name]["instance"]
                    return hasattr(instance, "health_check") and callable(getattr(instance, "health_check"))
                return False
        except Exception as e:
            self.logger.error(f"Callback de vérification de health_check échoué pour '{module_name}': {e}")
            return False

    def _callback_check_module_health(self, module_name: str) -> Dict[str, Any]:
        """
        Callback pour vérifier l'état de santé d'un module.

        Args:
            module_name: Nom du module

        Returns:
            Dict: Résultat de la vérification de l'état de santé
        """
        try:
            # Appeler la méthode health_check du module
            with self.core.lock:
                if module_name in self.core.modules:
                    instance = self.core.modules[module_name]["instance"]
                    health_check = getattr(instance, "health_check", None)
                    if callable(health_check):
                        return health_check()
                return {"healthy": True}
        except Exception as e:
            self.logger.error(f"Callback de vérification de l'état de santé échoué pour '{module_name}': {e}")
            return {"healthy": False, "message": f"Erreur lors de la vérification: {str(e)}"}


# Classe d'extension pour NexusCore qui préserve son paradigme d'orchestration
class ErrorRecoveryExtension:
    """
    Extension pour NexusCore qui ajoute des capacités de récupération d'erreurs
    sans modifier le paradigme d'orchestration fondamental.

    Cette classe est conçue pour être utilisée comme un mixin ou une extension
    de NexusCore, en ajoutant des points d'intégration minimaux pour la gestion
    d'erreurs.
    """

    def initialize_error_recovery(self):
        """
        Initialise le gestionnaire de récupération d'erreurs.
        Cette méthode doit être appelée après l'initialisation de NexusCore.
        """
        self.error_recovery = ErrorRecoveryManager(self)
        self.logger.info("Gestionnaire de récupération d'erreurs initialisé")

    def _start_module_with_recovery(self, name: str) -> None:
        """
        Version améliorée de _start_module avec récupération d'erreurs.

        Args:
            name (str): Nom du module à démarrer.
        """
        with self.lock:
            if name not in self.modules:
                self.logger.error(f"Impossible de démarrer '{name}': Module non trouvé.")
                return

            module_info = self.modules[name]
            if module_info["status"] == "running":
                self.logger.debug(f"Module '{name}' est déjà en cours d'exécution.")
                return

            # Ignorer les modules isolés
            if module_info["status"] == "isolated":
                self.logger.warning(f"Module '{name}' est isolé en raison d'erreurs précédentes. Démarrage ignoré.")
                return

            instance = module_info["instance"]
            try:
                start_method = getattr(instance, "start", None)
                if callable(start_method):
                    start_method()
                    self.logger.info(f"Module '{name}' démarré avec succès.")
                else:
                    self.logger.debug(f"Module '{name}' n'a pas de méthode start(). Marqué comme 'running' quand même.")
                module_info["status"] = "running"
                self._persist_module(name)  # Persister le nouvel état

                # Réinitialiser les tentatives de récupération en cas de démarrage réussi
                self.error_recovery.reset_recovery_attempts(name)

            except Exception as e:
                # Utiliser le gestionnaire de récupération d'erreurs
                recovery_result = self.error_recovery.handle_error(name, e, "start")

                if not recovery_result["success"]:
                    self.logger.error(f"Erreur lors du démarrage du module '{name}': {e}")
                    module_info["status"] = "error"
                    self._persist_module(name)  # Persister l'état d'erreur

    def _stop_module_with_recovery(self, name: str) -> None:
        """
        Version améliorée de _stop_module avec récupération d'erreurs.

        Args:
            name (str): Nom du module à arrêter.
        """
        with self.lock:
            if name not in self.modules:
                self.logger.error(f"Impossible d'arrêter '{name}': Module non trouvé.")
                return

            module_info = self.modules[name]
            if module_info["status"] != "running":
                self.logger.debug(f"Module '{name}' n'est pas en cours d'exécution. État actuel: '{module_info['status']}'.")
                return

            instance = module_info["instance"]
            try:
                stop_method = getattr(instance, "stop", None)
                if callable(stop_method):
                    stop_method()
                    self.logger.info(f"Module '{name}' arrêté avec succès.")
                else:
                    self.logger.debug(f"Module '{name}' n'a pas de méthode stop(). Marqué comme 'stopped' quand même.")
                module_info["status"] = "stopped"
                self._persist_module(name)  # Persister

                # Réinitialiser les tentatives de récupération en cas d'arrêt réussi
                self.error_recovery.reset_recovery_attempts(name)

            except Exception as e:
                # Utiliser le gestionnaire de récupération d'erreurs
                recovery_result = self.error_recovery.handle_error(name, e, "stop")

                if not recovery_result["success"]:
                    self.logger.error(f"Erreur lors de l'arrêt du module '{name}': {e}")
                    module_info["status"] = "error"
                    self._persist_module(name)  # Persister

    def _deliver_message_with_recovery(self, message: Dict[str, Any]) -> None:
        """
        Version améliorée de _deliver_message avec récupération d'erreurs.

        Args:
            message (dict): Le message à livrer.
        """
        target = message["target"]
        with self.lock:
            if target in self.modules:
                module_info = self.modules[target]

                # Ignorer la livraison de messages aux modules isolés
                if module_info["status"] == "isolated":
                    self.logger.warning(f"Livraison de message ignorée: Module '{target}' est isolé.")
                    return

                target_module = module_info["instance"]
                handle_method = getattr(target_module, "handle_message", None)
                if callable(handle_method):
                    try:
                        handle_method(message)
                    except Exception as e:
                        # Utiliser le gestionnaire de récupération d'erreurs
                        recovery_result = self.error_recovery.handle_error(target, e, "message")

                        if not recovery_result["success"]:
                            self.logger.error(f"Erreur lors du traitement du message par '{target}': {e}")

    def reset_module(self, name: str) -> bool:
        """
        Réinitialise un module qui est en état d'erreur ou isolé, permettant de le démarrer à nouveau.

        Args:
            name (str): Nom du module à réinitialiser

        Returns:
            bool: True si la réinitialisation a réussi, False sinon
        """
        with self.lock:
            if name not in self.modules:
                self.logger.error(f"Impossible de réinitialiser '{name}': Module non trouvé.")
                return False

            module_info = self.modules[name]
            current_status = module_info["status"]

            if current_status in ["error", "isolated"]:
                # Réinitialiser l'état du module
                module_info["status"] = "registered"
                self._persist_module(name)

                # Réinitialiser les tentatives de récupération
                self.error_recovery.reset_recovery_attempts(name)

                self.logger.info(f"Module '{name}' a été réinitialisé de '{current_status}' à 'registered'")
                return True
            else:
                self.logger.warning(f"Module '{name}' est en état '{current_status}', pas 'error' ou 'isolated'. Aucune réinitialisation nécessaire.")
                return False

    def get_enhanced_status(self) -> Dict[str, Any]:
        """
        Version améliorée de get_status qui inclut les informations de santé.

        Returns:
            Dict[str, Any]: Dictionnaire contenant les informations d'état.
        """
        # Appeler d'abord la méthode get_status originale
        status = self.get_status()

        # Ajouter les informations de santé pour chaque module
        for name in status["modules"]:
            health_info = self.error_recovery.get_module_health_status(name)
            status["modules"][name]["health"] = health_info

        # Ajouter des informations sur la récupération d'erreurs
        status["recovery_enabled"] = True

        return status

    def shutdown_with_recovery(self) -> None:
        """
        Version améliorée de stop qui arrête également le gestionnaire de récupération d'erreurs.
        """
        # Arrêter d'abord NexusCore normalement
        self.stop()

        # Puis arrêter le gestionnaire de récupération d'erreurs
        if hasattr(self, "error_recovery"):
            self.error_recovery.shutdown()




# Memory Manager

In [ ]:
"""
memory_manager.py - Hierarchical Memory System for GrandNexus

This module implements a multi-level memory architecture:
- Short-term, medium-term, and long-term memory stores
- Adaptive consolidation based on relevance, usage frequency, and recency
- Background maintenance thread for periodic decay, pruning, and auto-saving
- Flexible serialization (JSON or pickle) for storing memory to disk
- Rich metadata indexing and searching
- Thread-safe operations for concurrency

It combines ideas from the Claude/Gemini prototypes and extends them with:
- Weighted scoring for consolidation and eviction
- Configurable decay rates per tier
- Hooks for integrated logging and potential life-cycle linking
- Optional synergy with other modules, e.g., hooking into a 'registry' or 'core' if needed
"""

import logging
import threading
import time
import json
import os
import math
import pickle
import uuid
import heapq
import numpy as np
from collections import defaultdict
from typing import Dict, List, Any, Optional, Tuple, Union, Set


class MemoryItem:
        """
        Represents a single piece of information stored in the memory system.

        Attributes:
            memory_id (str): Unique identifier for this memory (UUID4 by default).
            content (Any): The actual data stored (can be text, JSON, custom class, etc.).
            metadata (Dict[str, Any]): Arbitrary metadata about this memory (source, type, tags, etc.).
            relevance (float): Importance score in [0,1].
            created_at (float): Timestamp when the memory was created.
            last_accessed (float): Timestamp of the most recent access.
            access_count (int): Number of times this memory has been accessed.
            embedding (Optional[np.ndarray]): Vector representation for similarity comparisons.
            expiration (Optional[float]): Time when this memory should be considered expired, if applicable.
        """

        def __init__(self,
                    content: Any,
                    metadata: Optional[Dict[str, Any]] = None,
                    relevance: float = 0.5,
                    memory_id: Optional[str] = None,
                    embedding: Optional[np.ndarray] = None,
                    expiration: Optional[float] = None):
            self.content = content
            self.metadata = metadata or {}
            self.relevance = max(0.0, min(1.0, relevance))  # clamp
            self.created_at = time.time()
            self.last_accessed = self.created_at
            self.access_count = 0
            self.memory_id = memory_id or str(uuid.uuid4())
            self.embedding = embedding
            self.expiration = expiration
            self.modified_count = 0  # Track modifications

        def access(self) -> None:
            """Mark this memory as accessed, updating last_accessed and incrementing the counter."""
            self.last_accessed = time.time()
            self.access_count += 1

        def update_relevance(self,
                            new_relevance: Optional[float] = None,
                            increment: Optional[float] = None,
                            context: Optional[Dict[str, Any]] = None) -> None:
            """
            Update the relevance score either by setting it absolutely (new_relevance)
            or by incrementing it (increment). Optional context can be provided for logging.

            Args:
                new_relevance: Optional absolute value to set (between 0-1)
                increment: Optional amount to add to current relevance
                context: Optional contextual information about why relevance is changing
            """
            old_relevance = self.relevance

            if new_relevance is not None:
                self.relevance = max(0.0, min(1.0, new_relevance))
            elif increment is not None:
                self.relevance = max(0.0, min(1.0, self.relevance + increment))

            # Record modification and context
            if old_relevance != self.relevance:
                self.modified_count += 1
                if context and isinstance(context, dict):
                    # Optionally store relevance change history in metadata
                    history = self.metadata.setdefault("relevance_history", [])
                    history.append({
                        "timestamp": time.time(),
                        "old": old_relevance,
                        "new": self.relevance,
                        "context": context
                    })

        def update_content(self, new_content: Any, preserve_metadata: bool = True) -> None:
            """
            Update the content of this memory item.

            Args:
                new_content: The new content to store
                preserve_metadata: If True, keep existing metadata, otherwise reset it
            """
            self.content = new_content
            if not preserve_metadata:
                self.metadata = {}
            self.modified_count += 1
            self.last_accessed = time.time()  # Consider this an access

        def update_embedding(self, new_embedding: Optional[np.ndarray] = None) -> None:
            """Update the embedding vector for this memory."""
            self.embedding = new_embedding
            self.modified_count += 1

        def set_expiration(self, expiration_time: Optional[float]) -> None:
            """Set or clear the expiration time for this memory."""
            self.expiration = expiration_time

        def is_expired(self) -> bool:
            """Check if this memory has expired."""
            if self.expiration is None:
                return False
            return time.time() >= self.expiration

        def age(self) -> float:
            """Returns how many seconds have passed since creation."""
            return time.time() - self.created_at

        def idle_time(self) -> float:
            """Returns how many seconds have passed since last access."""
            return time.time() - self.last_accessed

        def to_dict(self) -> Dict[str, Any]:
            """
            Convert MemoryItem to a dictionary for serialization.
            Note: embeddings must be handled separately if they need serialization.
            """
            result = {
                "memory_id": self.memory_id,
                "content": self.content,
                "metadata": self.metadata,
                "relevance": self.relevance,
                "created_at": self.created_at,
                "last_accessed": self.last_accessed,
                "access_count": self.access_count,
                "modified_count": self.modified_count
            }

            if self.expiration is not None:
                result["expiration"] = self.expiration

            # Embedding handling needs special care
            if self.embedding is not None:
                if isinstance(self.embedding, np.ndarray):
                    # Convert to list for JSON serialization
                    result["embedding_shape"] = self.embedding.shape
                    result["embedding_data"] = self.embedding.tolist()
                else:
                    result["embedding_data"] = self.embedding

            return result

        @classmethod
        def from_dict(cls, data: Dict[str, Any]) -> 'MemoryItem':
            """
            Create a MemoryItem from a dictionary, presumably read from disk.
            """
            # Handle embedding if present
            embedding = None
            if "embedding_data" in data:
                if "embedding_shape" in data:
                    # Reconstruct numpy array
                    embedding = np.array(data["embedding_data"])
                else:
                    embedding = data["embedding_data"]

            # Create the item with basic attributes
            item = cls(
                content=data["content"],
                metadata=data["metadata"],
                relevance=data["relevance"],
                memory_id=data["memory_id"],
                embedding=embedding,
                expiration=data.get("expiration")
            )

            # Restore additional attributes
            item.created_at = data["created_at"]
            item.last_accessed = data["last_accessed"]
            item.access_count = data["access_count"]
            if "modified_count" in data:
                item.modified_count = data["modified_count"]

            return item

        def __repr__(self) -> str:
            expiry_info = f", expires={self.expiration:.1f}" if self.expiration else ""
            return (f"<MemoryItem id={self.memory_id[-8:]}, rel={self.relevance:.2f}, "
                    f"age={self.age():.1f}s, acc={self.access_count}{expiry_info}>")


class NexusMemoryManager:
    """
    Hierarchical memory manager for GrandNexus, featuring:
      - short-term, medium-term, and long-term stores
      - dynamic consolidation and relevance decay
      - flexible persistence (pickle or JSON)
      - powerful metadata indexing
      - capacity-limited tiers with eviction

    Typical usage:
      1) Instantiate with desired capacities, intervals, thresholds, etc.
      2) Call start() to enable the background maintenance loop
      3) Store / retrieve items via store(), retrieve(), or query()
      4) stop() to gracefully shut down (auto-saves if configured)

    The memory manager is thread-safe, using an RLock for synchronization.
    """

    def __init__(self,
                 short_term_capacity: int = 100,
                 medium_term_capacity: int = 300,
                 long_term_capacity: int = 5000,
                 consolidation_interval: float = 60.0,
                 storage_file: Optional[str] = "nexus_memory.dat",
                 serialization_format: str = "pickle",  # can be "json" or "pickle"
                 auto_save: bool = True,
                 save_interval: float = 300.0,
                 relevance_threshold: float = 0.3,
                 short_decay_rate: float = 0.05,
                 medium_decay_rate: float = 0.02,
                 long_decay_rate: float = 0.005,
                 threshold_short_to_medium: float = 0.4,
                 threshold_medium_to_long: float = 0.7,
                 consolidation_access_for_short: int = 3,
                 consolidation_access_for_medium: int = 10
                 ):
        """
        Args:
            short_term_capacity (int): Maximum items in short-term store
            medium_term_capacity (int): Max items in medium-term
            long_term_capacity (int): Max items in long-term
            consolidation_interval (float): How often (sec) we run the consolidation/decay/prune loop
            storage_file (str, optional): Path to persist memories. If None, no persistence.
            serialization_format (str): "json" or "pickle" for on-disk format
            auto_save (bool): If True, we auto-save periodically
            save_interval (float): How often (sec) to auto-save
            relevance_threshold (float): Memories with relevance below this get pruned
            short_decay_rate (float): Decay factor for short-term store
            medium_decay_rate (float): Decay factor for medium-term
            long_decay_rate (float): Decay factor for long-term
            threshold_short_to_medium (float): Relevance threshold for short->medium promotion
            threshold_medium_to_long (float): Relevance threshold for medium->long promotion
            consolidation_access_for_short (int): Access count threshold for short->medium
            consolidation_access_for_medium (int): Access count threshold for medium->long
        """
        self.logger = logging.getLogger("NexusMemoryManager")

        # capacities
        self.short_term_capacity = short_term_capacity
        self.medium_term_capacity = medium_term_capacity
        self.long_term_capacity = long_term_capacity

        # data stores: memory_id -> MemoryItem
        self.short_term: Dict[str, MemoryItem] = {}
        self.medium_term: Dict[str, MemoryItem] = {}
        self.long_term: Dict[str, MemoryItem] = {}

        # intervals & config
        self.consolidation_interval = consolidation_interval
        self.storage_file = storage_file
        self.serialization_format = serialization_format
        self.auto_save = auto_save
        self.save_interval = save_interval
        self.relevance_threshold = relevance_threshold

        self.short_decay_rate = short_decay_rate
        self.medium_decay_rate = medium_decay_rate
        self.long_decay_rate = long_decay_rate

        # thresholds for internal promotions
        self.threshold_short_to_medium = threshold_short_to_medium
        self.threshold_medium_to_long = threshold_medium_to_long

        # optional additional constraints
        self.consolidation_access_for_short = consolidation_access_for_short
        self.consolidation_access_for_medium = consolidation_access_for_medium

        # concurrency
        self.lock = threading.RLock()
        self.running = False
        self.maintenance_thread: Optional[threading.Thread] = None

        # indexing (metadata_key -> {value -> set(memory_ids)})
        self.indexes: Dict[str, Dict[Any, Set[str]]] = defaultdict(lambda: defaultdict(set))

        # Attempt to load existing memories
        if self.storage_file and os.path.exists(self.storage_file):
            self._load_memories()

        self.logger.info(
            f"NexusMemoryManager initialized: short={short_term_capacity}, "
            f"medium={medium_term_capacity}, long={long_term_capacity}, file={self.storage_file}"
        )

    def start(self) -> None:
        """Starts the background maintenance thread if not already running."""
        with self.lock:
            if self.running:
                self.logger.warning("NexusMemoryManager is already running.")
                return
            self.running = True

        self.maintenance_thread = threading.Thread(
            target=self._maintenance_loop,
            name="NexusMemoryMaintenance",
            daemon=True
        )
        self.maintenance_thread.start()
        self.logger.info("NexusMemoryManager started with background maintenance loop.")

    def stop(self) -> None:
        """Stops the maintenance thread and performs a final save if configured."""
        with self.lock:
            if not self.running:
                self.logger.warning("NexusMemoryManager is not running.")
                return
            self.running = False

        if self.maintenance_thread:
            self.maintenance_thread.join(timeout=5.0)

        if self.storage_file and self.auto_save:
            self._save_memories()

        self.logger.info("NexusMemoryManager stopped.")

    def store(self,
              content: Any,
              metadata: Optional[Dict[str, Any]] = None,
              relevance: float = 0.5,
              memory_id: Optional[str] = None,
              embedding: Optional[np.ndarray] = None,
              expiration: Optional[float] = None) -> str:
        """
        Store a new memory item and decide which tier (short/medium/long) it goes into
        based on the initial relevance. Updates the index and ensures capacities.

        Args:
            content: The data to store
            metadata: Additional information about this memory
            relevance: Importance score from 0.0 to 1.0
            memory_id: Optional custom ID (UUID generated if None)
            embedding: Optional vector representation for similarity searches
            expiration: Optional timestamp when this memory should expire

        Returns:
            memory_id: The identifier of the stored memory
        """
        metadata = metadata or {}
        item = MemoryItem(content, metadata, relevance, memory_id, embedding, expiration)

        with self.lock:
            # Determine which store to use based on relevance thresholds
            if item.relevance >= self.threshold_medium_to_long:
                self.long_term[item.memory_id] = item
                tier = "long-term"
            elif item.relevance >= self.threshold_short_to_medium:
                self.medium_term[item.memory_id] = item
                tier = "medium-term"
            else:
                self.short_term[item.memory_id] = item
                tier = "short-term"

            self._index_memory(item)
            self._check_capacities()

            # Enhance logging with more context
            expiry_info = f", expires at {time.strftime('%H:%M:%S', time.localtime(expiration))}" if expiration else ""
            self.logger.info(f"Stored item {item.memory_id} in {tier} memory (relevance={item.relevance:.2f}{expiry_info})")

            # For semantic memories, optionally extract concepts and add additional indexing
            if metadata.get("memory_type") == "semantic" and self.config.synergy_with_memory:
                self._process_semantic_memory(item)

        return item.memory_id

    def retrieve(self, memory_id: str) -> Optional[MemoryItem]:
        """
        Retrieve a memory by ID from any store. Mark it accessed.
        """
        with self.lock:
            item = (self.short_term.get(memory_id)
                    or self.medium_term.get(memory_id)
                    or self.long_term.get(memory_id))
            if item:
                item.access()
                self.logger.debug(f"Retrieved memory {memory_id} (count={item.access_count})")
                return item
            else:
                self.logger.warning(f"Memory {memory_id} not found.")
                return None

    def retrieve_content(self, memory_id: str) -> Any:
        """Convenience method to retrieve only the content of a memory."""
        item = self.retrieve(memory_id)
        return item.content if item else None

    def delete(self, memory_id: str) -> bool:
        """Delete a memory from whichever store it is in."""
        with self.lock:
            item = None
            if memory_id in self.short_term:
                item = self.short_term.pop(memory_id)
            elif memory_id in self.medium_term:
                item = self.medium_term.pop(memory_id)
            elif memory_id in self.long_term:
                item = self.long_term.pop(memory_id)

            if item:
                self._remove_from_indexes(item)
                self.logger.info(f"Deleted memory {memory_id} from memory.")
                return True
            else:
                self.logger.warning(f"Delete failed: Memory {memory_id} not found.")
                return False

    def query(self,
              metadata_filter: Optional[Dict[str, Any]] = None,
              min_relevance: float = 0.0,
              max_results: int = 10,
              include_content: bool = True) -> List[Union[str, Tuple[str, Any]]]:
        """
        Query memories by (optional) metadata filters and minimal relevance.
        Returns up to max_results memory IDs or (memory_id, content) tuples.
        """
        with self.lock:
            candidate_ids = self._get_candidate_ids(metadata_filter)

            results: List[Union[str, Tuple[str, Any]]] = []
            for mid in candidate_ids:
                item = (self.short_term.get(mid)
                        or self.medium_term.get(mid)
                        or self.long_term.get(mid))
                if not item:
                    continue
                if item.relevance < min_relevance:
                    continue
                # Check metadata
                if metadata_filter and not all(item.metadata.get(k) == v for k, v in metadata_filter.items()):
                    continue
                # Mark it accessed
                item.access()
                # Decide what to return
                if include_content:
                    results.append((mid, item.content))
                else:
                    results.append(mid)
                if len(results) >= max_results:
                    break
            return results

    def update_relevance(self,
                         memory_id: str,
                         new_relevance: Optional[float] = None,
                         increment: Optional[float] = None) -> bool:
        """
        Update the relevance of a memory item. If this triggers a threshold crossing,
        the item may be moved between short/medium/long.
        """
        with self.lock:
            item = (self.short_term.get(memory_id)
                    or self.medium_term.get(memory_id)
                    or self.long_term.get(memory_id))
            if not item:
                self.logger.warning(f"Cannot update relevance: {memory_id} not found.")
                return False

            old_rel = item.relevance
            item.update_relevance(new_relevance, increment)
            self.logger.info(f"Updated relevance for {memory_id}: {old_rel:.2f} -> {item.relevance:.2f}")
            self._check_store_transition(item)
            return True

    def get_stats(self) -> Dict[str, Any]:
        """Obtain statistics about how many items each store holds, capacities, etc."""
        with self.lock:
            return {
                "short_count": len(self.short_term),
                "medium_count": len(self.medium_term),
                "long_count": len(self.long_term),
                "short_capacity": self.short_term_capacity,
                "medium_capacity": self.medium_term_capacity,
                "long_capacity": self.long_term_capacity,
                "total_memories": (len(self.short_term)
                                   + len(self.medium_term)
                                   + len(self.long_term))
            }

    def clear_memories(self, store_name: Optional[str] = None) -> int:
        """
        Clear the specified store or all stores if store_name is None.
        Returns the number of items cleared.
        """
        with self.lock:
            count = 0
            if store_name in (None, "short_term"):
                for m in list(self.short_term.values()):
                    self._remove_from_indexes(m)
                count += len(self.short_term)
                self.short_term.clear()
            if store_name in (None, "medium_term"):
                for m in list(self.medium_term.values()):
                    self._remove_from_indexes(m)
                count += len(self.medium_term)
                self.medium_term.clear()
            if store_name in (None, "long_term"):
                for m in list(self.long_term.values()):
                    self._remove_from_indexes(m)
                count += len(self.long_term)
                self.long_term.clear()

            if store_name is None:
                self._clear_all_indexes()

            self.logger.info(f"Cleared {count} memories from {store_name or 'all stores'}.")
            return count

    # -----------------------------------------------------
    #  Internal / Protected Methods
    # -----------------------------------------------------

    def _maintenance_loop(self) -> None:
        """Background thread that handles consolidation, decay, pruning, and auto-save."""
        self.logger.info("Memory maintenance loop started.")
        last_consolidation = time.time()
        last_save = time.time()

        while True:
            with self.lock:
                if not self.running:
                    break
            time.sleep(1.0)  # check every second

            now = time.time()
            if now - last_consolidation >= self.consolidation_interval:
                try:
                    with self.lock:
                        self._consolidate_memories()
                        self._apply_relevance_decay()
                        self._prune_low_relevance()
                    last_consolidation = now
                except Exception as e:
                    self.logger.exception(f"Error during memory consolidation: {e}")

            if self.auto_save and self.storage_file and (now - last_save >= self.save_interval):
                try:
                    with self.lock:
                        self._save_memories()
                    last_save = now
                except Exception as e:
                    self.logger.exception(f"Error during auto-save: {e}")

        self.logger.info("Memory maintenance loop ended.")

    def _consolidate_memories(self) -> None:
        """
        Periodically promote items from short-term -> medium-term or
        medium-term -> long-term based on relevance thresholds, access counts,
        and additional heuristics for cognitive-like memory management.
        """
        # For tracking consolidation statistics
        stats = {
            "short_to_medium": 0,
            "medium_to_long": 0,
            "short_expired": 0,
            "medium_expired": 0,
            "long_expired": 0
        }

        # Check for and handle expired memories first
        now = time.time()
        expired_short = self._prune_expired_memories(self.short_term)
        expired_medium = self._prune_expired_memories(self.medium_term)
        expired_long = self._prune_expired_memories(self.long_term)

        stats["short_expired"] = expired_short
        stats["medium_expired"] = expired_medium
        stats["long_expired"] = expired_long

        # short->medium promotion
        to_medium = []
        for mid, item in self.short_term.items():
            if self._should_promote_to_medium(item):
                to_medium.append(mid)

        for mid in to_medium:
            if mid in self.short_term:  # double-check
                mem = self.short_term.pop(mid)
                # Adjust relevance to ensure it meets medium-term threshold
                if mem.relevance < self.threshold_short_to_medium:
                    mem.relevance = self.threshold_short_to_medium
                self.medium_term[mid] = mem
                stats["short_to_medium"] += 1

        # medium->long promotion
        to_long = []
        for mid, item in self.medium_term.items():
            if self._should_promote_to_long(item):
                to_long.append(mid)

        for mid in to_long:
            if mid in self.medium_term:
                mem = self.medium_term.pop(mid)
                # Adjust relevance to ensure it meets long-term threshold
                if mem.relevance < self.threshold_medium_to_long:
                    mem.relevance = self.threshold_medium_to_long
                self.long_term[mid] = mem
                stats["medium_to_long"] += 1

        # After consolidation, re-check capacities
        self._check_capacities()

        # Log consolidation statistics if any changes were made
        total_changes = sum(stats.values())
        if total_changes > 0:
            self.logger.info(f"Memory consolidation: {stats['short_to_medium']} items promoted to medium-term, "
                            f"{stats['medium_to_long']} to long-term. "
                            f"Expired: {stats['short_expired']} short, {stats['medium_expired']} medium, "
                            f"{stats['long_expired']} long.")

    def _should_promote_to_medium(self, item: MemoryItem) -> bool:
        """
        Determine if a short-term memory should be promoted to medium-term.
        Uses multiple heuristics for a more cognitive-like memory system.
        """
        # Core criteria from configuration
        if item.relevance >= self.threshold_short_to_medium:
            return True

        if item.access_count >= self.consolidation_access_for_short:
            return True

        # Additional heuristics
        # 1. Recent frequent access relative to age
        if item.age() > 0 and item.access_count / item.age() > 0.1:  # Accessed every 10 seconds on average
            return True

        # 2. High relevance combined with recent access
        if item.relevance > 0.3 and (time.time() - item.last_accessed) < 60:  # Relevance > 0.3 and accessed in last minute
            return True

        # 3. Item has been modified multiple times, suggesting importance
        if item.modified_count > 2:
            return True

        # 4. Tagged as important in metadata
        if item.metadata.get("importance", 0) > 0.7:
            return True

        # 5. Associated with other long-term memories (requires implementation)
        if item.metadata.get("associated_with_long_term", False):
            return True

        return False

    def _should_promote_to_long(self, item: MemoryItem) -> bool:
        """
        Determine if a medium-term memory should be promoted to long-term.
        Uses multiple heuristics for a more cognitive-like memory system.
        """
        # Core criteria from configuration
        if item.relevance >= self.threshold_medium_to_long:
            return True

        if item.access_count >= self.consolidation_access_for_medium:
            return True

        # Additional heuristics
        # 1. Consistently accessed over long period
        if item.age() > 3600 and item.access_count > 5:  # Older than 1 hour and accessed multiple times
            return True

        # 2. High-quality embeddings (if applicable)
        if "embedding_quality" in item.metadata and item.metadata["embedding_quality"] > 0.8:
            return True

        # 3. Explicitly marked for long-term storage
        if item.metadata.get("store_long_term", False):
            return True

        # 4. Contains critical information types
        critical_types = {"core_concept", "fundamental_rule", "identity_info", "critical_fact"}
        if any(tag in critical_types for tag in item.metadata.get("tags", [])):
            return True

        return False

    def _prune_expired_memories(self, memory_store: Dict[str, MemoryItem]) -> int:
        """
        Remove expired memories from the given store.
        Returns the number of items removed.
        """
        expired_ids = []
        now = time.time()

        for mid, memory in memory_store.items():
            if memory.expiration and now >= memory.expiration:
                expired_ids.append(mid)

        for mid in expired_ids:
            memory = memory_store.pop(mid)
            self._remove_from_indexes(memory)

        return len(expired_ids)

    def _apply_relevance_decay(self) -> None:
        """
        Apply time-based decay to relevance. Items in short-term decay faster, etc.
        This is a simple approach where we reduce relevance proportionally to how long
        they've been idle since last_accessed.
        """
        now = time.time()

        def apply_decay(store: Dict[str, MemoryItem], decay_rate: float):
            for mem in store.values():
                idle_time = now - mem.last_accessed
                # scale the decay by a factor of (idle_time / consolidation_interval) for instance
                factor = (idle_time / self.consolidation_interval)
                decay_amount = decay_rate * factor
                if decay_amount > 0:
                    mem.update_relevance(increment=-decay_amount)

        with self.lock:
            apply_decay(self.short_term, self.short_decay_rate)
            apply_decay(self.medium_term, self.medium_decay_rate)
            apply_decay(self.long_term, self.long_decay_rate)

    def _prune_low_relevance(self) -> None:
        """
        Remove items whose relevance is below the global threshold from each store.
        """
        pruned_count = 0

        def prune_store(store: Dict[str, MemoryItem]) -> None:
            nonlocal pruned_count
            to_remove = [mid for mid, m in store.items() if m.relevance < self.relevance_threshold]
            for mid in to_remove:
                memory = store.pop(mid)
                self._remove_from_indexes(memory)
            pruned_count += len(to_remove)

        with self.lock:
            prune_store(self.short_term)
            prune_store(self.medium_term)
            prune_store(self.long_term)

        if pruned_count:
            self.logger.info(f"Pruned {pruned_count} low-relevance items.")

    def _check_capacities(self) -> None:
        """
        Ensure that none of the stores exceed capacity. If they do, we evict the
        lowest "score" items.
        """
        if len(self.short_term) > self.short_term_capacity:
            over = len(self.short_term) - self.short_term_capacity
            self._evict_from_store("short_term", over)

        if len(self.medium_term) > self.medium_term_capacity:
            over = len(self.medium_term) - self.medium_term_capacity
            self._evict_from_store("medium_term", over)

        if len(self.long_term) > self.long_term_capacity:
            over = len(self.long_term) - self.long_term_capacity
            self._evict_from_store("long_term", over)

    def _evict_from_store(self, store_name: str, count: int) -> None:
        """
        Evict `count` items from the specified store, choosing the ones
        with the 'lowest score'. For example, we can define a simple score
        that combines relevance, access_count, and recency.
        """
        if count <= 0:
            return

        store = getattr(self, store_name, None)
        if not store:
            return

        heap: List[Tuple[float, str]] = []
        now = time.time()

        def score(mem: MemoryItem) -> float:
            # a simple combined metric: relevance * log(access+1), then penalize old last_access
            # lower => more likely to be evicted
            access_factor = math.log(mem.access_count + 1)
            recency_factor = 1.0 / (1.0 + (now - mem.last_accessed))
            # we want a higher value => better => so we invert it for the min-heap
            return -(mem.relevance * access_factor * recency_factor)

        # build a min-heap by pushing negative scores
        for mid, mem in store.items():
            heap.append((score(mem), mid))
        heapq.heapify(heap)

        evicted = 0
        while evicted < count and heap:
            _, mem_id = heapq.heappop(heap)
            if mem_id in store:
                memory = store.pop(mem_id)
                self._remove_from_indexes(memory)
                evicted += 1

        self.logger.info(f"Evicted {evicted} items from {store_name} to respect capacity.")

    def _check_store_transition(self, memory: MemoryItem) -> None:
        """
        If the memory's new relevance crosses a threshold, move it to
        the appropriate store. Similar logic to store() method.
        """
        mid = memory.memory_id
        # figure out which store it belongs in
        if memory.relevance >= self.threshold_medium_to_long:
            target_store = "long_term"
        elif memory.relevance >= self.threshold_short_to_medium:
            target_store = "medium_term"
        else:
            target_store = "short_term"

        # see which store it's currently in
        in_store = None
        if mid in self.short_term:
            in_store = "short_term"
        elif mid in self.medium_term:
            in_store = "medium_term"
        elif mid in self.long_term:
            in_store = "long_term"

        if in_store != target_store:
            # remove from old store
            if in_store == "short_term":
                del self.short_term[mid]
            elif in_store == "medium_term":
                del self.medium_term[mid]
            elif in_store == "long_term":
                del self.long_term[mid]

            # put it in the correct store
            getattr(self, target_store)[mid] = memory
            self.logger.info(f"Moved memory {mid} from {in_store} to {target_store} after relevance change.")
            self._check_capacities()

    def _index_memory(self, memory: MemoryItem) -> None:
        """
        Add references to the 'indexes' dictionary for quick searching by metadata keys.
        We only index simple, hashable values (str, int, etc.).
        """
        mid = memory.memory_id
        for k, v in memory.metadata.items():
            if isinstance(v, (str, int, float, bool)):
                self.indexes[k][v].add(mid)

    def _remove_from_indexes(self, memory: MemoryItem) -> None:
        """Remove memory's references from the indexes."""
        mid = memory.memory_id
        for k, v in memory.metadata.items():
            if isinstance(v, (str, int, float, bool)) and v in self.indexes[k]:
                self.indexes[k][v].discard(mid)
                if not self.indexes[k][v]:
                    del self.indexes[k][v]
            if not self.indexes[k]:
                del self.indexes[k]

    def _clear_all_indexes(self) -> None:
        """Clear the entire indexing structure."""
        self.indexes = defaultdict(lambda: defaultdict(set))

    def _get_candidate_ids(self, metadata_filter: Optional[Dict[str, Any]]) -> List[str]:
        """Return a list of memory IDs that match the given metadata filter via indexes."""
        if not metadata_filter:
            # return all known IDs
            return (list(self.short_term.keys())
                    + list(self.medium_term.keys())
                    + list(self.long_term.keys()))

        candidate_sets: List[Set[str]] = []
        for k, v in metadata_filter.items():
            if k in self.indexes and v in self.indexes[k]:
                candidate_sets.append(self.indexes[k][v])
            else:
                # no match
                return []

        # intersect all sets
        if not candidate_sets:
            return []
        result = candidate_sets[0]
        for s in candidate_sets[1:]:
            result = result.intersection(s)

        return list(result)


    def _process_semantic_memory(self, item: MemoryItem) -> None:
        """
        Special processing for semantic memories, such as concept extraction
        and integration with other memory systems.
        """
        # Extract potential concepts from content
        concepts = []
        if isinstance(item.content, str):
            # Simple keyword extraction (placeholder for more sophisticated NLP)
            # In a real implementation, you might use a keyword extractor or NER model
            words = item.content.lower().split()
            # Filter out common stop words and short words
            stop_words = {'the', 'and', 'a', 'an', 'in', 'on', 'at', 'to', 'for', 'of', 'with'}
            potential_concepts = [w for w in words if w not in stop_words and len(w) > 3]
            # Remove duplicates
            concepts = list(set(potential_concepts))

        # Store extracted concepts in metadata
        if concepts:
            item.metadata["extracted_concepts"] = concepts
            self.logger.debug(f"Extracted concepts from memory {item.memory_id}: {concepts}")

        # Check for relationships with existing items
        if hasattr(self, "semantic_memory") and self.semantic_memory:
            # This would require access to the SemanticMemory module
            # For now, just mark that semantic processing was attempted
            item.metadata["semantic_processed"] = True
            self.logger.debug(f"Semantic memory processing attempted for {item.memory_id}")
        else:
            # Basic standalone processing
            item.metadata["semantic_processed"] = True
            self.logger.debug(f"Processed semantic memory {item.memory_id}")

    def _save_memories(self) -> None:
        """Persist all memories in short/medium/long to disk using the chosen format."""
        if not self.storage_file:
            return

        data_to_save = {
            "short_term": {mid: mem.to_dict() for mid, mem in self.short_term.items()},
            "medium_term": {mid: mem.to_dict() for mid, mem in self.medium_term.items()},
            "long_term": {mid: mem.to_dict() for mid, mem in self.long_term.items()}
        }

        try:
            if self.serialization_format == "pickle":
                with open(self.storage_file, 'wb') as f:
                    pickle.dump(data_to_save, f)
            elif self.serialization_format == "json":
                with open(self.storage_file, 'w') as f:
                    json.dump(data_to_save, f)
            else:
                self.logger.error(f"Unsupported serialization format: {self.serialization_format}")
                return

            self.logger.info(f"Saved memories to {self.storage_file} [{self.serialization_format}].")

        except Exception as e:
            self.logger.exception(f"Error saving memories: {e}")

    def _load_memories(self) -> None:
        """Load memories from disk into short/medium/long, re-building indexes."""
        if not self.storage_file or not os.path.exists(self.storage_file):
            return

        try:
            if self.serialization_format == "pickle":
                with open(self.storage_file, 'rb') as f:
                    loaded = pickle.load(f)
            elif self.serialization_format == "json":
                with open(self.storage_file, 'r') as f:
                    loaded = json.load(f)
            else:
                self.logger.error(f"Unsupported format: {self.serialization_format}")
                return

            self.short_term.clear()
            self.medium_term.clear()
            self.long_term.clear()
            self._clear_all_indexes()

            # Rebuild from loaded data
            for store_name in ("short_term", "medium_term", "long_term"):
                store_dict = loaded.get(store_name, {})
                for mid, mem_data in store_dict.items():
                    item = MemoryItem.from_dict(mem_data)
                    getattr(self, store_name)[mid] = item
                    self._index_memory(item)

            self.logger.info(f"Loaded memories from {self.storage_file} [{self.serialization_format}].")

        except Exception as e:
            self.logger.exception(f"Error loading memories: {e}")


# Registry

In [ ]:
"""
registry.py - Advanced Module Registry for GrandNexus

This module implements:
- Enhanced module registration and dependency management
- Bidirectional tracking of dependencies and dependents
- Module lifecycle state tracking and logging of state changes
- Interface verification to ensure basic compatibility
- Dynamic discovery of modules from a Python package or path
- Automatic sync with NexusCore upon registration (if desired)
- Extended metadata handling (version, author, config, etc.)
"""

import logging
import importlib
import pkgutil
import inspect
import threading
import time
from typing import Dict, List, Any, Optional, Callable, Set, Union

class ModuleRegistry:
    """
    Advanced registry for GrandNexus modules.

    The ModuleRegistry is responsible for:
        - Keeping a centralized record of modules (by name).
        - Storing module dependencies (which other modules a given module depends on).
        - Storing dependents (which modules depend on a given module).
        - Tracking module metadata, including lifecycle states, timestamps, version, etc.
        - Managing the registration and unregistration process, ensuring no orphan dependencies.
        - Optionally synchronizing module info with NexusCore (ex: register_module calls).
        - Providing a method to discover and automatically register modules from a Python package.

    This registry can be seen as a companion to NexusCore, focusing on structural and
    informational aspects, while NexusCore handles the operational lifecycle (start/stop).
    """

    VALID_STATES: Set[str] = {
        "REGISTERED",  # Initially registered but not yet started
        "STARTING",    # In the process of starting
        "RUNNING",     # Successfully started and operational
        "STOPPING",    # In the process of stopping
        "STOPPED",     # Successfully stopped
        "ERROR"        # Error state (start, stop, or runtime error)
    }

    def __init__(self, core: Any):
        """
        Initialize the ModuleRegistry.

        Args:
            core: A reference to the NexusCore instance (or similar) that orchestrates
                  the system's lifecycle. This registry can delegate calls to that
                  instance, such as 'register_module', 'update_module_status', etc.
        """
        self.logger: logging.Logger = logging.getLogger("ModuleRegistry")
        self.core = core

        # The following dicts store key info about modules:
        #   self.modules       : { module_name: module_instance }
        #   self.dependencies  : { module_name: [dependencies...] }
        #   self.dependents    : { module_name: set(modules_that_depend_on_it) }
        #   self.metadata      : { module_name: { arbitrary_metadata... } }

        self.modules: Dict[str, Any] = {}
        self.dependencies: Dict[str, List[str]] = {}
        self.dependents: Dict[str, Set[str]] = {}
        self.metadata: Dict[str, Dict[str, Any]] = {}

        # Lock to ensure thread safety
        self.lock: threading.RLock = threading.RLock()

        self.logger.info("ModuleRegistry initialized successfully.")

    def register(
        self,
        name: str,
        module_instance: Any,
        dependencies: Optional[List[str]] = None,
        metadata: Optional[Dict[str, Any]] = None,
        sync_with_core: bool = True,
        auto_verify: bool = True
    ) -> bool:
        """
        Registers a module along with its dependencies and any custom metadata.

        By default, this method also registers the module with the NexusCore (sync_with_core=True).

        Args:
            name (str):
                The unique identifier for the module. Typically a short string or camelCase name.
            module_instance (Any):
                The Python object (class instance) representing the module. Should have
                at least a 'start()' or 'handle_message()' method to be recognized.
            dependencies (List[str], optional):
                Names of other modules upon which this module depends. If any of those
                modules are not yet registered, a warning is logged. Defaults to None.
            metadata (Dict[str, Any], optional):
                Additional information about the module (version, config, author, etc.).
                This dictionary is stored internally; it may also be partially sent to
                the NexusCore if relevant. Defaults to None.
            sync_with_core (bool):
                If True, automatically invokes 'core.register_module' to keep the
                NexusCore's records aligned. If False, no call is made. Defaults to True.
            auto_verify (bool):
                If True, automatically calls _verify_module_interface to check for
                required methods. Set to False to skip verification. Defaults to True.

        Returns:
            bool: True if registration was successful, False otherwise.

        Raises:
            ValueError:
                If the module fails interface verification (no start() or handle_message()).
        """
        deps = dependencies or []
        meta = metadata or {}

        try:
            with self.lock:
                # If this module is already registered, remove its old dependency links
                if name in self.modules:
                    self.logger.warning(f"Module '{name}' was already registered. Overwriting old entry.")
                    self._remove_dependency_links(name)

                # Validate minimal interface
                if auto_verify:
                    self._verify_module_interface(name, module_instance)

                # Check if dependencies exist in the registry
                for dep in deps:
                    if dep not in self.modules and dep != name:
                        # We do not forcibly raise an error, but we do log it
                        self.logger.warning(f"Dependency '{dep}' for module '{name}' is not yet registered. "
                                            "This may cause startup order issues.")

                # Store the module and its dependencies
                self.modules[name] = module_instance
                self.dependencies[name] = deps.copy()  # store a shallow copy
                self.metadata.setdefault(name, {})
                self.metadata[name].update(meta)       # merge new metadata
                # Set up bidirectional links
                for dep in deps:
                    self.dependents.setdefault(dep, set()).add(name)
                if name not in self.dependents:
                    self.dependents[name] = set()

                # Mark the module's initial state
                self._set_module_state(name, "REGISTERED")

                # Optionally inform the NexusCore
                if sync_with_core and hasattr(self.core, "register_module") and callable(self.core.register_module):
                    # If there's a 'config' key in the metadata, pass it as config param
                    if 'config' in meta:
                        self.core.register_module(name, module_instance, deps, meta['config'])
                    else:
                        self.core.register_module(name, module_instance, deps)

                self.logger.info(f"Module '{name}' registered with dependencies={deps}.")
                return True

        except Exception as e:
            self.logger.error(f"Failed to register module '{name}': {e}", exc_info=True)
            return False

    def unregister(self, name: str) -> bool:
        """
        Unregister a module from the registry, removing any references to it.

        This method checks if any other modules still depend on the one being removed.
        If so, it raises ValueError to prevent accidentally removing a module that is
        still in use.

        Args:
            name (str): The name of the module to unregister.

        Returns:
            bool: True if the unregistration succeeded, False if it could not be done
                  (e.g., module wasn't found).

        Raises:
            ValueError:
                If other modules still depend on the one being removed.
        """
        with self.lock:
            if name not in self.modules:
                self.logger.warning(f"Cannot unregister '{name}': module not found in registry.")
                return False

            # If something depends on this module, we can't remove it
            if name in self.dependents and self.dependents[name]:
                blocking_deps = ", ".join(sorted(self.dependents[name]))
                raise ValueError(
                    f"Cannot unregister module '{name}': the following modules depend on it: {blocking_deps}"
                )

            # Remove references
            self._remove_dependency_links(name)
            del self.modules[name]
            del self.dependencies[name]
            if name in self.dependents:
                del self.dependents[name]
            if name in self.metadata:
                del self.metadata[name]

            # Optionally, one could also call `self.core.unregister_module` if that were desired,
            # but the code from Gemini doesn't do that by default.

            self.logger.info(f"Module '{name}' successfully unregistered.")
            return True

    def get(self, name: str) -> Optional[Any]:
        """
        Retrieve the module instance by name.

        Args:
            name (str): The name of the module to retrieve.

        Returns:
            Optional[Any]: The module instance, or None if it doesn't exist.
        """
        with self.lock:
            return self.modules.get(name)

    def get_all_modules(self) -> Dict[str, Any]:
        """
        Return a dictionary of all registered modules.

        Returns:
            Dict[str, Any]: Copy of the internal registry map (name -> module_instance).
        """
        with self.lock:
            return dict(self.modules)

    def list_module_names(self) -> List[str]:
        """
        Get a list of all module names currently in the registry.

        Returns:
            List[str]: Sorted list of module names.
        """
        with self.lock:
            return sorted(self.modules.keys())

    def get_dependencies(self, name: str) -> List[str]:
        """
        Return the list of modules that the specified module depends on.

        Args:
            name (str): The module whose dependencies we want to retrieve.

        Returns:
            List[str]: Names of the modules that 'name' depends on.

        Raises:
            KeyError: If 'name' is not in the registry.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            return list(self.dependencies[name])

    def get_dependents(self, name: str) -> List[str]:
        """
        Return the list of modules that depend on the specified module.

        Args:
            name (str): The module name for which we want to retrieve dependents.

        Returns:
            List[str]: Names of the modules that depend on 'name'.

        Raises:
            KeyError: If 'name' is not in the registry.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            return sorted(self.dependents.get(name, set()))

    def get_module_metadata(self, name: str) -> Dict[str, Any]:
        """
        Retrieve the metadata dictionary for a given module.

        Args:
            name (str): Name of the module.

        Returns:
            Dict[str, Any]: A (copied) dictionary of the metadata for that module,
                            or an empty dict if none is stored.

        Raises:
            KeyError: If the module is not in the registry.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            meta = self.metadata.get(name, {})
            return dict(meta)  # return a shallow copy

    def set_module_metadata(self, name: str, new_meta: Dict[str, Any]) -> None:
        """
        Set or update the metadata for a given module.

        This merges the existing metadata with 'new_meta', overwriting any
        keys that appear in both.

        Args:
            name (str): The module name to update.
            new_meta (Dict[str, Any]): The new metadata to store.

        Raises:
            KeyError: If the module is not in the registry.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            self.metadata.setdefault(name, {})
            self.metadata[name].update(new_meta)
            self.logger.debug(f"Module '{name}' metadata updated with {new_meta}.")

    def get_module_state(self, name: str) -> str:
        """
        Get the current lifecycle state of a module.

        Args:
            name (str): Name of the module.

        Returns:
            str: The module's current state (REGISTERED, RUNNING, etc.), or "UNKNOWN"
                 if no state is recorded.

        Raises:
            KeyError: If 'name' is not registered.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            return self.metadata.get(name, {}).get("state", "UNKNOWN")

    def set_module_state(self, name: str, state: str, error_reason: Optional[str] = None) -> None:
        """
        Public method to forcibly set a module's state. Typically, internal calls use
        the protected _set_module_state, but you may invoke this method if you need
        to set or override the state externally.

        Args:
            name (str): The module name whose state is changing.
            state (str): The new state (must be in VALID_STATES).
            error_reason (str, optional): Explanation if we are transitioning to ERROR.

        Raises:
            KeyError: If the module is not in the registry.
            ValueError: If the provided state is invalid.
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry.")
            self._set_module_state(name, state, error_reason=error_reason)

    def resolve_dependency_order(self) -> List[str]:
        """
        Determine a valid startup (or shutdown) order based on module dependencies,
        typically by delegating to the NexusCore's method, if implemented.

        Returns:
            List[str]: A list of module names in an order that respects dependencies.

        Raises:
            ValueError: If circular dependencies are detected or if the resolution fails.
        """
        if hasattr(self.core, "_resolve_dependencies") and callable(self.core._resolve_dependencies):
            self.logger.debug("Resolving dependency order by delegating to NexusCore.")
            return self.core._resolve_dependencies()
        else:
            # If for any reason we cannot delegate to NexusCore, we can do a local topological sort
            self.logger.debug("NexusCore has no _resolve_dependencies method. Performing local topological sort.")
            return self._local_topo_sort()

    def get_module_dependencies(self, name: str, recursive: bool = False, max_depth: int = 10) -> List[str]:
        """
        Get the dependencies of a module, with option for recursive dependency resolution.

        Args:
            name (str): Module name to get dependencies for
            recursive (bool): If True, include all indirect dependencies
            max_depth (int): Maximum recursion depth for recursive dependencies

        Returns:
            List[str]: List of module names that this module depends on

        Raises:
            KeyError: If module name not found
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry")

            # Direct dependencies
            direct_deps = list(self.dependencies[name])

            # If not recursive, return direct dependencies
            if not recursive:
                return direct_deps

            # Recursive dependency resolution
            all_deps = set(direct_deps)
            to_process = deque([(dep, 1) for dep in direct_deps])  # (dep, depth)

            while to_process:
                dep, depth = to_process.popleft()

                # Stop if we reach max depth
                if depth >= max_depth:
                    continue

                if dep in self.dependencies:
                    for indirect_dep in self.dependencies[dep]:
                        if indirect_dep not in all_deps:
                            all_deps.add(indirect_dep)
                            to_process.append((indirect_dep, depth + 1))

            return list(all_deps)

    def get_module_dependents(self, name: str, recursive: bool = False, max_depth: int = 10) -> List[str]:
        """
        Get modules that depend on the specified module, with option for recursive resolution.

        Args:
            name (str): Module name to get dependents for
            recursive (bool): If True, include all indirect dependents
            max_depth (int): Maximum recursion depth for recursive dependents

        Returns:
            List[str]: List of module names that depend on this module

        Raises:
            KeyError: If module name not found
        """
        with self.lock:
            if name not in self.modules:
                raise KeyError(f"Module '{name}' not found in the registry")

            # Direct dependents
            direct_deps = list(self.dependents.get(name, set()))

            # If not recursive, return direct dependents
            if not recursive:
                return direct_deps

            # Recursive dependent resolution
            all_deps = set(direct_deps)
            to_process = deque([(dep, 1) for dep in direct_deps])  # (dep, depth)

            while to_process:
                dep, depth = to_process.popleft()

                # Stop if we reach max depth
                if depth >= max_depth:
                    continue

                if dep in self.dependents:
                    for indirect_dep in self.dependents[dep]:
                        if indirect_dep not in all_deps:
                            all_deps.add(indirect_dep)
                            to_process.append((indirect_dep, depth + 1))

            return list(all_deps)

    def check_dependency_cycles(self) -> List[List[str]]:
        """
        Check for dependency cycles in the registry.

        Returns:
            List[List[str]]: List of cycles found, where each cycle is a list of module names
            forming a circular dependency chain. Empty list if no cycles found.
        """
        with self.lock:
            # Convert dependencies to a directed graph
            graph = {name: set(deps) for name, deps in self.dependencies.items()}

            # Build reverse graph
            reverse_graph = {name: set() for name in self.modules}
            for node, deps in graph.items():
                for dep in deps:
                    if dep in reverse_graph:
                        reverse_graph[dep].add(node)

            # Try to do a topological sort
            in_degree = {node: 0 for node in graph}
            for node in graph:
                for dep in graph.get(node, []):
                    if dep in in_degree:
                        in_degree[dep] += 1

            queue = [node for node, degree in in_degree.items() if degree == 0]
            visited = set()

            while queue:
                node = queue.pop(0)
                visited.add(node)

                for neighbor in list(graph.get(node, [])):
                    if neighbor in in_degree:
                        in_degree[neighbor] -= 1
                        if in_degree[neighbor] == 0:
                            queue.append(neighbor)

            # If topological sort visited all nodes, there are no cycles
            if len(visited) == len(self.modules):
                return []

            # Find cycles using DFS
            remaining = set(self.modules.keys()) - visited

            # Use _find_dependency_cycles from NexusCore if available
            if hasattr(self.core, "_find_dependency_cycles") and callable(self.core._find_dependency_cycles):
                return self.core._find_dependency_cycles(remaining, graph, reverse_graph)

            # Fallback implementation
            cycles = []
            cycle_visited = set()

            def find_cycle(node, path, visited_in_path):
                if node in visited_in_path:
                    # Found a cycle
                    cycle_start = path.index(node)
                    cycles.append(path[cycle_start:] + [node])
                    return

                cycle_visited.add(node)
                visited_in_path.add(node)
                path.append(node)

                for neighbor in graph.get(node, []):
                    if neighbor in remaining and neighbor not in cycle_visited:
                        find_cycle(neighbor, path, visited_in_path.copy())

                path.pop()
                visited_in_path.remove(node)

            # Try to find cycles starting from each remaining node
            for node in remaining:
                if node not in cycle_visited:
                    find_cycle(node, [], set())

            return cycles

    def get_dependency_graph_summary(self) -> Dict[str, Any]:
        """
        Creates a summary of the dependency graph structure for visualization or debugging.

        Returns:
            Dict with:
            - 'modules': list of all module names
            - 'dependencies': dict mapping module names to their dependencies
            - 'dependents': dict mapping module names to their dependents
            - 'orphaned': list of modules with no dependencies or dependents
            - 'cycles': list of dependency cycles if any
            - 'stats': general statistics about the graph
        """
        with self.lock:
            modules = list(self.modules.keys())
            dependencies = {name: list(deps) for name, deps in self.dependencies.items()}
            dependents = {name: list(deps) for name, deps in self.dependents.items()}

            # Find orphaned modules (no dependencies or dependents)
            orphaned = [name for name in modules
                        if not dependencies.get(name) and not dependents.get(name)]

            # Check for cycles
            cycles = self.check_dependency_cycles()

            # Calculate some stats
            stats = {
                "total_modules": len(modules),
                "total_dependency_links": sum(len(deps) for deps in dependencies.values()),
                "max_dependencies": max((len(deps) for deps in dependencies.values()), default=0),
                "max_dependents": max((len(deps) for deps in dependents.values()), default=0),
                "orphaned_count": len(orphaned),
                "cycles_count": len(cycles)
            }

            return {
                "modules": modules,
                "dependencies": dependencies,
                "dependents": dependents,
                "orphaned": orphaned,
                "cycles": cycles,
                "stats": stats
            }

    def discover_modules(self, package_path: str) -> List[str]:
        """
        Dynamically discover and auto-register modules from the specified Python package.

        This scans for submodules, imports them, and looks for a class whose name matches
        the submodule's last part (case-insensitive). If found, an instance is created
        and automatically registered under that name.

        Args:
            package_path (str): The dotted path to the Python package to scan.

        Returns:
            List[str]: A list of module names that were discovered and registered.
        """
        discovered = []
        try:
            package = importlib.import_module(package_path)
            if not hasattr(package, "__path__"):
                self.logger.warning(f"Package '{package_path}' has no __path__. Cannot discover submodules.")
                return discovered

            for _, submodule_name, is_pkg in pkgutil.iter_modules(package.__path__, package.__name__ + "."):
                if is_pkg:
                    # We skip sub-packages in this example
                    continue

                try:
                    submod = importlib.import_module(submodule_name)
                    # The last element of submodule_name might be something like "some_module"
                    short_name = submodule_name.split(".")[-1].lower()

                    # Search for a class matching short_name
                    for attr_name, attr_value in vars(submod).items():
                        if inspect.isclass(attr_value):
                            if attr_name.lower() == short_name:
                                instance = attr_value()
                                # The registry name is the class name, but lowercased
                                module_name = attr_name.lower()
                                self.register(module_name, instance)
                                discovered.append(module_name)
                                break

                except Exception as e:
                    self.logger.error(f"Error loading submodule '{submodule_name}': {e}", exc_info=True)

        except Exception as e:
            self.logger.error(f"Error discovering modules in '{package_path}': {e}", exc_info=True)

        if discovered:
            self.logger.info(f"Discovered and registered modules: {discovered}")
        else:
            self.logger.info(f"No modules discovered in '{package_path}'.")

        return discovered

    ############################################################
    #            INTERNAL (PROTECTED) METHODS                  #
    ############################################################

    def _set_module_state(self, name: str, state: str, error_reason: Optional[str] = None) -> None:
        """
        Protected method to change a module's state internally, recording the time
        of the change and also notifying the NexusCore if it has a suitable method.

        Args:
            name (str): The module name.
            state (str): A string in VALID_STATES.
            error_reason (str, optional): If transitioning to ERROR, explain the cause.

        Raises:
            ValueError: If the new state is not recognized.
        """
        if state not in self.VALID_STATES:
            raise ValueError(f"Invalid state '{state}'. Must be one of: {sorted(self.VALID_STATES)}")

        with self.lock:
            old_state = self.metadata.get(name, {}).get("state", "UNKNOWN")
            self.metadata.setdefault(name, {})
            self.metadata[name]["state"] = state
            self.metadata[name]["state_changed"] = time.time()
            if state == "ERROR" and error_reason:
                self.metadata[name]["error_reason"] = error_reason

            self.logger.info(f"Module '{name}' changed state: {old_state} -> {state} "
                             f"{'(reason: ' + error_reason + ')' if error_reason else ''}")

            # Attempt to update the core's view as well
            if hasattr(self.core, "update_module_status") and callable(self.core.update_module_status):
                try:
                    self.core.update_module_status(name, state)
                except Exception as e:
                    self.logger.error(f"Error updating NexusCore module status for '{name}': {e}", exc_info=True)

    def _verify_module_interface(self, name: str, module: Any) -> None:
        """
        Ensure that a module has at least a minimal set of expected methods.

        By default, we require that a module has either a 'start' method or a 'handle_message'
        method. You may expand this logic if your project demands more strict interface checks.

        Args:
            name (str): The module name for logging.
            module (Any): The module instance.

        Raises:
            ValueError: If the module doesn't have either start() or handle_message() method.
        """
        has_start = hasattr(module, "start") and callable(getattr(module, "start"))
        has_handle_message = hasattr(module, "handle_message") and callable(getattr(module, "handle_message"))
        has_stop = hasattr(module, "stop") and callable(getattr(module, "stop"))

        if not (has_start or has_handle_message):
            msg = (f"Module '{name}' provides neither start() nor handle_message(). "
                  f"It might not function properly within GrandNexus.")
            self.logger.warning(msg)
            raise ValueError(msg)

        if has_start and not has_stop:
            self.logger.warning(f"Module '{name}' has start() but no stop() method. "
                              "This may cause issues during shutdown.")

        # Check for additional recommended interfaces
        recommended_methods = {
            "get_status": "report module status",
            "set_configuration": "receive configuration updates"
        }

        for method, purpose in recommended_methods.items():
            if not (hasattr(module, method) and callable(getattr(module, method))):
                self.logger.debug(f"Module '{name}' does not implement optional '{method}()' to {purpose}")

    def _remove_dependency_links(self, name: str) -> None:
        """
        Remove any references to 'name' in the dependencies or dependents structures.

        This is typically used when a module is unregistered or re-registered.
        """
        # For each of 'name's dependencies, remove 'name' from the dep's dependents set
        deps = self.dependencies.get(name, [])
        for dep in deps:
            if dep in self.dependents and name in self.dependents[dep]:
                self.dependents[dep].remove(name)

    def _local_topo_sort(self) -> List[str]:
        """
        Perform a local topological sort of the modules using DFS, to handle dependencies
        if we can't rely on NexusCore.

        Returns:
            List[str]: A valid topological ordering (modules in the correct sequence).

        Raises:
            ValueError: If a cycle is detected.
        """
        visited = set()
        temp_visited = set()
        result = []

        def visit(node: str):
            if node in temp_visited:
                raise ValueError(f"Circular dependency detected involving '{node}'.")
            if node not in visited:
                temp_visited.add(node)
                for neighbor in self.dependencies.get(node, []):
                    if neighbor in self.modules:  # only consider known modules
                        visit(neighbor)
                temp_visited.remove(node)
                visited.add(node)
                result.append(node)

        with self.lock:
            for node in self.modules.keys():
                if node not in visited:
                    visit(node)

        # The result is reversed so that dependencies appear first
        return list(reversed(result))


# Episodic

In [ ]:
"""
episodic.py - Enhanced Episodic Memory Module for GrandNexus

This enhanced implementation combines concepts from:
- BioNeMo: For efficient retrieval mechanisms and embedding structures
- LIDA (Learning Intelligent Distribution Agent): For consciousness-inspired episodic structure
- ACT-R: For activation/decay mechanisms and attention-based memory processes
- HTM (Hierarchical Temporal Memory): For temporal encoding and sequence detection

Features:
- Event-Episode-Epoch hierarchy to organize memory temporally
- Sparse distributed representations for efficient storage and matching
- Activation-based retrieval with context sensitivity
- Salience-guided memory consolidation for long-term storage
- Temporal and causal associative links between episodes
"""

import logging
import time
import uuid
import heapq
import math
import threading
import numpy as np
from collections import defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import (
    Any, Dict, List, Optional, Set, Tuple, Callable, Union, Deque
)

# -----------------------------------------------------------------------------
# Enumerations and Types
# -----------------------------------------------------------------------------

class MemoryStrength(Enum):
    """Levels of memory strength, inspired by LIDA's activation levels"""
    UNCONSCIOUS = 1      # Very weak activation, not readily accessible
    PERIPHERAL = 2       # Weak activation, accessible with cues
    PRECONSCIOUS = 3     # Moderate activation, easily accessible with cues
    CONSCIOUS = 4        # Strong activation, part of current context
    FOCAL = 5            # Strongest activation, central to current focus


class EpisodeType(Enum):
    """Types of episodes for better organization and retrieval"""
    PERCEPTION = auto()   # Direct sensory experiences
    INTERACTION = auto()  # Agent interactions with environment
    COMMUNICATION = auto() # Communication with users or other agents
    DECISION = auto()     # Decision-making processes
    REFLECTION = auto()   # Self-reflection and metacognition
    SYSTEM = auto()       # System operations or administrative tasks
    PROCEDURAL = auto()   # Step-by-step procedures or workflows
    GENERAL = auto()      # General/unspecified episode type


class RetrievalMode(Enum):
    """Different modes for memory retrieval, affecting how matches are scored"""
    EXACT = auto()         # Exact match on specified criteria
    SIMILARITY = auto()    # Similarity-based matching using embeddings
    TEMPORAL = auto()      # Retrieval based on temporal relationships
    CAUSAL = auto()        # Retrieval that emphasizes cause-and-effect
    ASSOCIATIVE = auto()   # Retrieval based on associative links
    CONTEXT = auto()       # Context-sensitive retrieval including current state
    MIXED = auto()         # Balanced approach using multiple factors

# -----------------------------------------------------------------------------
# Data Structures
# -----------------------------------------------------------------------------

@dataclass
class EpisodeEvent:
    """
    Represents an atomic event within an episode.
    Inspired by ACT-R's declarative memory chunks and HTM's sparse representations.

    Attributes:
        event_id: Unique identifier for the event
        timestamp: When the event occurred
        content: The actual data (text, sensory data, etc.)
        metadata: Additional context or attributes
        importance: How salient/important this event is (0-1)
        embedding: Vector representation for similarity
        activation: Current activation level (ACT-R concept)
        sparse_rep: Sparse distributed representation (HTM concept)
        causal_links: IDs of events this may have caused/influenced
        predecessor_events: IDs of events that directly preceded this
    """
    event_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    timestamp: float = field(default_factory=time.time)
    content: Any = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    importance: float = 0.5
    embedding: Optional[np.ndarray] = None
    activation: float = 1.0  # Initial full activation
    sparse_rep: Optional[np.ndarray] = None  # Sparse distributed representation
    causal_links: List[str] = field(default_factory=list)
    predecessor_events: List[str] = field(default_factory=list)

    def __post_init__(self):
        # Set memory strength to CONSCIOUS initially
        self.memory_strength = MemoryStrength.CONSCIOUS
        self.access_count: int = 0
        self.last_access_time: float = self.timestamp

    def access(self) -> None:
        """Mark this event as accessed, updating activation and metadata."""
        self.access_count += 1
        self.last_access_time = time.time()
        self.activation = 1.0  # Full activation upon access

        # Update memory strength based on access
        if self.access_count > 5:
            self.memory_strength = MemoryStrength.FOCAL
        elif self.access_count > 2:
            self.memory_strength = MemoryStrength.CONSCIOUS

    def decay_activation(self, decay_rate: float, current_time: float) -> None:
        """
        Apply ACT-R-inspired decay to activation based on time since last access.

        Args:
            decay_rate: Rate at which activation decays
            current_time: Current timestamp for decay calculation
        """
        time_since_access = max(0.001, current_time - self.last_access_time)

        # ACT-R-style power law decay: A = B * (T^-d)
        # Where B is base activation, T is time since access, d is decay parameter
        base_level = 1.0
        decay_factor = math.pow(time_since_access, -decay_rate)

        # Incorporate access count (more accesses = slower decay)
        access_boost = math.log(1 + self.access_count) * 0.1

        self.activation = base_level * decay_factor + access_boost

        # Cap activation within range [0, 1]
        self.activation = max(0.0, min(1.0, self.activation))

        # Update memory strength based on activation
        if self.activation > 0.8:
            self.memory_strength = MemoryStrength.CONSCIOUS
        elif self.activation > 0.5:
            self.memory_strength = MemoryStrength.PRECONSCIOUS
        elif self.activation > 0.2:
            self.memory_strength = MemoryStrength.PERIPHERAL
        else:
            self.memory_strength = MemoryStrength.UNCONSCIOUS

    def __repr__(self) -> str:
        return (f"<EpisodeEvent id={self.event_id[-8:]}, time={self.timestamp:.1f}, "
                f"imp={self.importance:.2f}, act={self.activation:.2f}, "
                f"strength={self.memory_strength.name}>")


@dataclass
class Episode:
    """
    Represents a coherent collection of events, inspired by LIDA's episodic memory
    and HTM's sequence representations.

    Attributes:
        episode_id: Unique identifier for this episode
        episode_type: Type of episode (perception, interaction, etc.)
        start_time: When the episode began
        end_time: When the episode ended (None if ongoing)
        events: Sequence of events in this episode
        context: Contextual information about the environment, agent state, etc.
        tags: Semantic labels for easier retrieval
        embedding: Vector representation of the entire episode
        importance: Overall salience/importance of the episode
        activation: Current activation level (decays over time)
        location: Optional spatial reference
        associations: Links to related episodes
        summary: Brief description of episode content
    """
    episode_id: str
    episode_type: EpisodeType = EpisodeType.GENERAL
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    events: List[EpisodeEvent] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)
    tags: Set[str] = field(default_factory=set)
    embedding: Optional[np.ndarray] = None
    importance: float = 0.5
    activation: float = 1.0
    location: Optional[str] = None
    associations: Dict[str, float] = field(default_factory=dict)  # episode_id -> strength
    summary: Optional[str] = None

    def __post_init__(self):
        # Set memory strength to CONSCIOUS initially
        self.memory_strength = MemoryStrength.CONSCIOUS
        self.access_count: int = 0
        self.last_access_time: float = self.start_time
        self.temporal_context: List[str] = []  # Episode IDs that occurred before/after

        # HTM-inspired fields for sequence learning
        self.sequence_predictors: Dict[str, Dict[str, float]] = defaultdict(dict)  # event -> (next_event -> probability)

    def add_event(self, event: EpisodeEvent) -> None:
        """
        Adds an event to the episode, maintaining temporal order and
        updating HTM-style sequence predictors.
        """
        if not self.events:
            self.events.append(event)
            return

        # If the new event is out of order, insert it at the right position
        if event.timestamp < self.events[-1].timestamp:
            # Find the correct position
            for i, existing_event in enumerate(self.events):
                if event.timestamp < existing_event.timestamp:
                    # Update predecessor relationships
                    if i > 0:
                        event.predecessor_events.append(self.events[i-1].event_id)
                    if i < len(self.events):
                        self.events[i].predecessor_events.append(event.event_id)

                    # Insert at correct position
                    self.events.insert(i, event)
                    break
        else:
            # Add at the end (normal case)
            if self.events:
                # Update predecessor relationship
                event.predecessor_events.append(self.events[-1].event_id)

                # Update HTM-style sequence predictors
                last_event_id = self.events[-1].event_id
                event_type = event.metadata.get("type", "generic")

                # Increment count for this sequence
                if last_event_id in self.sequence_predictors:
                    self.sequence_predictors[last_event_id][event.event_id] = self.sequence_predictors[last_event_id].get(event.event_id, 0) + 1
                else:
                    self.sequence_predictors[last_event_id] = {event.event_id: 1}

            self.events.append(event)

        # Update episode importance based on event importance
        self.importance = max(self.importance, event.importance)

        # Update episode's last access time
        self.last_access_time = time.time()

    def close(self) -> None:
        """Marks this episode as finished by setting end_time."""
        if self.end_time is None:
            self.end_time = time.time()

        # Normalize sequence predictor probabilities
        for source_event, targets in self.sequence_predictors.items():
            total = sum(targets.values())
            if total > 0:
                for target_event in targets:
                    targets[target_event] /= total

    def is_active(self) -> bool:
        """Returns True if the episode has not been closed yet."""
        return self.end_time is None

    def duration(self) -> float:
        """Return the length of time from start_time to end_time (or now if ongoing)."""
        if self.end_time is None:
            return time.time() - self.start_time
        else:
            return self.end_time - self.start_time

    def access(self) -> None:
        """Mark this episode as accessed, updating activation and metadata."""
        self.access_count += 1
        self.last_access_time = time.time()
        self.activation = 1.0  # Full activation upon access

        # Update memory strength based on access
        if self.access_count > 5:
            self.memory_strength = MemoryStrength.FOCAL
        elif self.access_count > 2:
            self.memory_strength = MemoryStrength.CONSCIOUS

    def decay_activation(self, decay_rate: float, current_time: float) -> None:
        """
        Apply ACT-R-inspired decay to activation based on time since last access.
        """
        time_since_access = max(0.001, current_time - self.last_access_time)

        # Decay formula similar to event decay but slower for episodes
        base_level = 1.0
        decay_factor = math.pow(time_since_access, -decay_rate * 0.5)  # Slower decay for episodes

        # Incorporate access count, importance, and recency
        access_boost = math.log(1 + self.access_count) * 0.1
        importance_boost = self.importance * 0.2

        self.activation = base_level * decay_factor + access_boost + importance_boost

        # Cap activation within range [0, 1]
        self.activation = max(0.0, min(1.0, self.activation))

        # Update memory strength based on activation
        if self.activation > 0.8:
            self.memory_strength = MemoryStrength.CONSCIOUS
        elif self.activation > 0.5:
            self.memory_strength = MemoryStrength.PRECONSCIOUS
        elif self.activation > 0.2:
            self.memory_strength = MemoryStrength.PERIPHERAL
        else:
            self.memory_strength = MemoryStrength.UNCONSCIOUS

    def predict_next_event_type(self, current_event_id: str) -> Optional[str]:
        """
        HTM-inspired prediction of what might come next in a sequence.

        Args:
            current_event_id: ID of the current event

        Returns:
            ID of the most likely next event, or None if no prediction
        """
        if current_event_id not in self.sequence_predictors:
            return None

        # Get probabilities for next events
        next_event_probs = self.sequence_predictors[current_event_id]
        if not next_event_probs:
            return None

        # Return the most probable next event
        return max(next_event_probs.items(), key=lambda x: x[1])[0]

    def generate_summary(self) -> str:
        """
        Generate a brief summary of this episode if not already set.
        In a real implementation, this might use LLM integration.
        """
        if self.summary:
            return self.summary

        # Simple summary generation based on events and context
        event_count = len(self.events)
        duration = self.duration()

        if event_count == 0:
            summary = f"Empty {self.episode_type.name.lower()} episode"
        else:
            # Get the first and last event contents
            first_event = self.events[0].content
            last_event = self.events[-1].content

            # Try to extract content strings
            if isinstance(first_event, str):
                first_content = first_event[:30]
            else:
                first_content = str(type(first_event).__name__)

            if isinstance(last_event, str):
                last_content = last_event[:30]
            else:
                last_content = str(type(last_event).__name__)

            summary = (f"{self.episode_type.name.capitalize()} episode with {event_count} events "
                      f"over {duration:.1f}s, from '{first_content}...' to '{last_content}...'")

        self.summary = summary
        return summary

    def __repr__(self) -> str:
        status = "active" if self.is_active() else "closed"
        return (f"<Episode id={self.episode_id}, type={self.episode_type.name}, "
                f"events={len(self.events)}, {status}, imp={self.importance:.2f}, "
                f"act={self.activation:.2f}, strength={self.memory_strength.name}>")


@dataclass
class Epoch:
    """
    Represents a higher-level time period containing multiple episodes.
    This creates a 3-level hierarchy: Event -> Episode -> Epoch.
    Inspired by LIDA's hierarchical structure and HTM's temporal pooling.

    Attributes:
        epoch_id: Unique identifier for this epoch
        name: Human-readable name for this epoch
        start_time: When the epoch began
        end_time: When the epoch ended (None if ongoing)
        episodes: Episodes contained in this epoch
        themes: Detected themes or patterns across episodes
        embedding: Vector representation of the entire epoch
        importance: Overall salience/importance of the epoch
    """
    epoch_id: str
    name: str
    start_time: float
    end_time: Optional[float] = None
    episodes: List[str] = field(default_factory=list)  # List of episode_ids
    themes: Dict[str, float] = field(default_factory=dict)  # theme -> strength
    embedding: Optional[np.ndarray] = None
    importance: float = 0.5

    def is_active(self) -> bool:
        """Returns True if the epoch has not been closed yet."""
        return self.end_time is None

    def add_episode(self, episode_id: str) -> None:
        """Adds an episode to this epoch."""
        if episode_id not in self.episodes:
            self.episodes.append(episode_id)

    def close(self) -> None:
        """Marks this epoch as finished by setting end_time."""
        if self.end_time is None:
            self.end_time = time.time()

    def duration(self) -> float:
        """Return the length of time from start_time to end_time (or now if ongoing)."""
        if self.end_time is None:
            return time.time() - self.start_time
        else:
            return self.end_time - self.start_time

    def __repr__(self) -> str:
        status = "active" if self.is_active() else "closed"
        return (f"<Epoch id={self.epoch_id}, name='{self.name}', "
                f"episodes={len(self.episodes)}, {status}, imp={self.importance:.2f}>")


@dataclass
class EpisodicQuery:
    """
    Enhanced query structure for retrieving memories from episodic memory.
    Supports multiple retrieval modes and filters.

    Attributes:
        text: Optional text to match against episode/event content
        embedding: Optional embedding vector for similarity search
        tags: Optional set of tags to filter by
        time_range: Optional (start_time, end_time) to filter by
        episode_types: Optional set of episode types to filter by
        importance_threshold: Minimum importance score for retrieved episodes
        activation_threshold: Minimum activation for retrieved episodes
        memory_strength: Minimum memory strength for retrieved episodes
        retrieval_mode: Strategy for ranking and scoring matches
        max_results: Maximum number of results to return
        include_events: Whether to include individual events in results
        recursive: Whether to include associated episodes
        context_factors: Additional context to influence retrieval
    """
    text: Optional[str] = None
    embedding: Optional[np.ndarray] = None
    tags: Optional[Set[str]] = None
    time_range: Optional[Tuple[float, float]] = None
    episode_types: Optional[Set[EpisodeType]] = None
    importance_threshold: float = 0.0
    activation_threshold: float = 0.0
    memory_strength: Optional[MemoryStrength] = None
    retrieval_mode: RetrievalMode = RetrievalMode.MIXED
    max_results: int = 10
    include_events: bool = False
    recursive: bool = False
    context_factors: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        # Convert string episode types to enum values if needed
        if self.episode_types and any(isinstance(t, str) for t in self.episode_types):
            types = set()
            for t in self.episode_types:
                if isinstance(t, str):
                    try:
                        types.add(EpisodeType[t.upper()])
                    except KeyError:
                        pass  # Ignore invalid types
                else:
                    types.add(t)
            self.episode_types = types

        # Convert string memory strength to enum value if needed
        if isinstance(self.memory_strength, str):
            try:
                self.memory_strength = MemoryStrength[self.memory_strength.upper()]
            except KeyError:
                self.memory_strength = None

        # Convert string retrieval mode to enum value if needed
        if isinstance(self.retrieval_mode, str):
            try:
                self.retrieval_mode = RetrievalMode[self.retrieval_mode.upper()]
            except KeyError:
                self.retrieval_mode = RetrievalMode.MIXED


@dataclass
class EpisodicMemoryConfig:
    """
    Configuration for the EpisodicMemory system.

    Attributes:
        short_term_capacity: Max episodes in short-term memory
        long_term_capacity: Max episodes in long-term memory
        event_decay_rate: How quickly event activations decay (ACT-R concept)
        episode_decay_rate: How quickly episode activations decay (ACT-R concept)
        consolidation_interval: How often to run consolidation (seconds)
        embedding_dimension: Size of embedding vectors
        enable_memory_optimization: Whether to use memory optimization techniques
        similarity_threshold: Threshold for considering items similar
        embedding_function: Function to generate embeddings from content
        enable_predictive_coding: Whether to use HTM-style predictive coding
        enable_temporal_pooling: Whether to use HTM-style temporal pooling
        auto_generate_summaries: Whether to auto-generate episode summaries
        auto_tagging: Whether to automatically generate tags for episodes
        max_associative_distance: Max distance for associative retrieval
        attention_span: Number of episodes to keep in attentional focus
        consciousness_threshold: Activation threshold for "conscious" access
        temporal_context_size: Size of temporal context window
        max_epochs: Maximum number of epochs to maintain
        consolidation_scheduler: How to schedule consolidation (periodic/adaptive)
    """
    short_term_capacity: int = 100
    long_term_capacity: int = 1000
    event_decay_rate: float = 0.5
    episode_decay_rate: float = 0.3
    consolidation_interval: float = 300.0  # 5 minutes
    embedding_dimension: int = 256
    enable_memory_optimization: bool = True
    similarity_threshold: float = 0.7
    embedding_function: Optional[Callable[[Any], np.ndarray]] = None
    enable_predictive_coding: bool = True
    enable_temporal_pooling: bool = True
    auto_generate_summaries: bool = True
    auto_tagging: bool = True
    max_associative_distance: int = 3
    attention_span: int = 5
    consciousness_threshold: float = 0.7
    temporal_context_size: int = 10
    max_epochs: int = 100
    consolidation_scheduler: str = "adaptive"  # "periodic" or "adaptive"


@dataclass
class EpisodicMemoryState:
    """
    Internal state for episodic memory, used to track current context
    and attentional focus.
    """
    current_episode_id: Optional[str] = None
    attentional_focus: List[str] = field(default_factory=list)  # episode_ids in focus
    temporal_context: List[str] = field(default_factory=list)  # recent episode_ids
    active_concepts: Dict[str, float] = field(default_factory=dict)  # concept_id -> activation
    last_consolidation_time: float = field(default_factory=time.time)
    operation_count: int = 0
    current_epoch_id: Optional[str] = None


class EpisodicMemory:
    """
    Enhanced episodic memory system incorporating concepts from:
    - LIDA: Consciousness-inspired memory structure
    - ACT-R: Activation-based retrieval and decay
    - HTM: Hierarchical temporal memory patterns

    Features:
    - Three-level memory hierarchy: Events -> Episodes -> Epochs
    - Activation-based memory with ACT-R-inspired decay
    - Consciousness-like attentional mechanisms
    - HTM-inspired sequence learning and prediction
    - Associative linking between related episodes
    - Content-addressable memory with embedding similarity
    """

    def __init__(self, config: EpisodicMemoryConfig):
        """Initialize EpisodicMemory with the provided configuration."""
        self.logger = logging.getLogger("EpisodicMemory")
        self.config = config
        self.lock = threading.RLock()

        # Primary memory stores
        self.episodes: Dict[str, Episode] = {}
        self.events: Dict[str, EpisodeEvent] = {}
        self.epochs: Dict[str, Epoch] = {}

        # Organizational structures
        self.short_term_episodes: List[str] = []  # episode_ids in short-term memory
        self.long_term_episodes: List[str] = []   # episode_ids in long-term memory

        # Indexing structures for efficient retrieval
        self.episodes_by_type: Dict[EpisodeType, List[str]] = {
            etype: [] for etype in EpisodeType
        }
        self.episodes_by_tag: Dict[str, List[str]] = defaultdict(list)
        self.episode_timeline: List[Tuple[float, str]] = []  # (timestamp, episode_id) sorted by time

        # ACT-R inspired activation tracking
        self.episode_activations: Dict[str, float] = {}
        self.event_activations: Dict[str, float] = {}

        # HTM inspired structures
        self.temporal_sequences: Dict[str, Dict[str, float]] = defaultdict(dict)  # ep_id -> (next_ep_id -> probability)
        self.active_predictions: Dict[str, float] = {}  # predicted_episode_id -> confidence

        # LIDA inspired attention and consciousness structures
        self.state = EpisodicMemoryState()

        # Time tracking for adaptive consolidation
        self.last_decay_time = time.time()
        self.last_consolidation_time = time.time()

        self.logger.info(f"EpisodicMemory initialized (short={config.short_term_capacity}, "
                        f"long={config.long_term_capacity})")

    # -------------------------------------------------------------------------
    # Episode Creation and Management
    # -------------------------------------------------------------------------

    def create_episode(self,
                      episode_type: Union[EpisodeType, str] = EpisodeType.GENERAL,
                      context: Dict[str, Any] = None,
                      tags: Set[str] = None,
                      episode_id: Optional[str] = None,
                      importance: float = 0.5,
                      location: Optional[str] = None) -> Episode:
        """
        Creates a new episode and makes it the current active episode.

        Args:
            episode_type: Type of episode being created
            context: Contextual information for this episode
            tags: Semantic tags for categorization
            episode_id: Optional custom ID (generated if None)
            importance: Initial importance score (0-1)
            location: Optional location identifier

        Returns:
            The newly created Episode object
        """
        with self.lock:
            # Convert string episode type to enum if needed
            if isinstance(episode_type, str):
                try:
                    episode_type = EpisodeType[episode_type.upper()]
                except KeyError:
                    episode_type = EpisodeType.GENERAL

            # Generate ID if not provided
            if episode_id is None:
                episode_id = str(uuid.uuid4())

            # Check for duplicates
            if episode_id in self.episodes:
                self.logger.warning(f"Episode {episode_id} already exists. Returning existing episode.")
                return self.episodes[episode_id]

            # Create the episode
            episode = Episode(
                episode_id=episode_id,
                episode_type=episode_type,
                context=context or {},
                tags=tags or set(),
                importance=importance,
                location=location
            )

            # Store the episode
            self.episodes[episode_id] = episode
            self.short_term_episodes.append(episode_id)
            self.episode_activations[episode_id] = 1.0  # Full initial activation

            # Update indices
            self.episodes_by_type[episode_type].append(episode_id)
            for tag in episode.tags:
                self.episodes_by_tag[tag].append(episode_id)

            # Add to timeline
            self.episode_timeline.append((episode.start_time, episode_id))
            self._sort_timeline()

            # Update attention state
            self._update_attention(episode_id)

            # Make this the current episode
            self.state.current_episode_id = episode_id

            # Assign to current epoch if one is active
            if self.state.current_epoch_id and self.state.current_epoch_id in self.epochs:
                current_epoch = self.epochs[self.state.current_epoch_id]
                if current_epoch.is_active():
                    current_epoch.add_episode(episode_id)

            # Auto-tag if enabled
            if self.config.auto_tagging:
                self._auto_tag_episode(episode)

            # Increment operation count
            self.state.operation_count += 1

            # Check if we should run adaptive consolidation
            if (self.config.consolidation_scheduler == "adaptive" and
                self.state.operation_count % 10 == 0):  # Every 10 operations
                self._consolidate_memory()

            self.logger.info(f"Created new episode {episode_id} of type {episode_type.name}")
            return episode

    def add_event(self,
                 episode_id: str,
                 content: Any,
                 metadata: Dict[str, Any] = None,
                 importance: float = 0.5,
                 event_id: Optional[str] = None) -> Optional[str]:
        """
        Add a new event to the specified episode.

        Args:
            episode_id: ID of the episode to add this event to
            content: Event content (text, sensor data, etc.)
            metadata: Additional context or attributes
            importance: Event importance (0-1)
            event_id: Optional custom ID (generated if None)

        Returns:
            The event_id if successful, None otherwise
        """
        with self.lock:
            # Check if episode exists and is active
            if episode_id not in self.episodes:
                self.logger.warning(f"Cannot add event: Episode {episode_id} not found.")
                return None

            episode = self.episodes[episode_id]
            if not episode.is_active():
                self.logger.warning(f"Cannot add event: Episode {episode_id} is closed.")
                return None

            # Generate ID if not provided
            if event_id is None:
                event_id = str(uuid.uuid4())

            # Check for duplicates
            if event_id in self.events:
                self.logger.warning(f"Event {event_id} already exists. Using existing event.")
                return event_id

            # Create event
            event = EpisodeEvent(
                event_id=event_id,
                timestamp=time.time(),
                content=content,
                metadata=metadata or {},
                importance=importance
            )

            # Generate embedding if function provided
            if self.config.embedding_function:
                try:
                    event.embedding = self.config.embedding_function(content)
                except Exception as e:
                    self.logger.error(f"Error generating embedding for event {event_id}: {e}")

            # Generate sparse representation (HTM-inspired)
            if self.config.enable_predictive_coding:
                event.sparse_rep = self._generate_sparse_representation(content)

            # Store event
            self.events[event_id] = event
            self.event_activations[event_id] = 1.0  # Full initial activation

            # Add to episode
            episode.add_event(event)

            # Update episode importance if this event is more important
            if importance > episode.importance:
                episode.importance = importance

            # Update episode's activation
            episode.activation = 1.0
            self.episode_activations[episode_id] = 1.0

            # Check for HTM-style predictions and update accuracy
            self._update_predictions(episode, event)

            # Increment operation count
            self.state.operation_count += 1

            # Check if we should run adaptive consolidation
            if (self.config.consolidation_scheduler == "adaptive" and
                self.state.operation_count % 20 == 0):  # Every 20 operations
                self._consolidate_memory()

            self.logger.debug(f"Added event {event_id} to episode {episode_id}")
            return event_id

    def close_episode(self, episode_id: str) -> bool:
        """
        Mark an episode as complete by setting its end_time.
        If embedding_function is provided, computes an embedding for the episode.

        Args:
            episode_id: ID of the episode to close

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            if episode_id not in self.episodes:
                self.logger.warning(f"Cannot close episode {episode_id}: Not found.")
                return False

            episode = self.episodes[episode_id]
            if not episode.is_active():
                self.logger.debug(f"Episode {episode_id} is already closed.")
                return True

            # Close the episode
            episode.close()

            # Generate embedding if function provided
            if self.config.embedding_function and episode.events:
                try:
                    # Create a combined representation of all events
                    all_content = []
                    for event in episode.events:
                        if isinstance(event.content, str):
                            all_content.append(event.content)
                        else:
                            # Try to get a string representation
                            all_content.append(str(event.content))

                    combined_content = " ".join(all_content)
                    episode.embedding = self.config.embedding_function(combined_content)
                except Exception as e:
                    self.logger.error(f"Error generating embedding for episode {episode_id}: {e}")

            # Generate summary if auto-generation is enabled
            if self.config.auto_generate_summaries:
                episode.generate_summary()

            # Update temporal links in context
            self._update_temporal_links(episode_id)

            # Make this episode no longer the current one
            if self.state.current_episode_id == episode_id:
                self.state.current_episode_id = None

            # If this was in the attentional focus, update attention
            if episode_id in self.state.attentional_focus:
                self.state.attentional_focus.remove(episode_id)

            # HTM-inspired sequence learning
            if self.config.enable_predictive_coding:
                self._learn_episode_sequence(episode_id)

            # Check if we should consolidate memory after closing an episode
            if self.config.consolidation_scheduler == "periodic":
                current_time = time.time()
                if current_time - self.last_consolidation_time >= self.config.consolidation_interval:
                    self._consolidate_memory()

            self.logger.info(f"Closed episode {episode_id} with {len(episode.events)} events.")
            return True

    def create_epoch(self,
                    name: str,
                    episodes: List[str] = None,
                    epoch_id: Optional[str] = None,
                    importance: float = 0.5) -> Epoch:
        """
        Creates a new epoch to group episodes together.

        Args:
            name: Human-readable name for this epoch
            episodes: Optional list of episode IDs to include initially
            epoch_id: Optional custom ID (generated if None)
            importance: Initial importance score (0-1)

        Returns:
            The newly created Epoch object
        """
        with self.lock:
            # Generate ID if not provided
            if epoch_id is None:
                epoch_id = str(uuid.uuid4())

            # Check for duplicates
            if epoch_id in self.epochs:
                self.logger.warning(f"Epoch {epoch_id} already exists. Returning existing epoch.")
                return self.epochs[epoch_id]

            # Create the epoch
            epoch = Epoch(
                epoch_id=epoch_id,
                name=name,
                start_time=time.time(),
                episodes=[],
                importance=importance
            )

            # Add episodes if provided
            if episodes:
                valid_episodes = [ep_id for ep_id in episodes if ep_id in self.episodes]
                epoch.episodes = valid_episodes

            # Store the epoch
            self.epochs[epoch_id] = epoch

            # Make this the current epoch
            self.state.current_epoch_id = epoch_id

            self.logger.info(f"Created new epoch {epoch_id} named '{name}' with {len(epoch.episodes)} episodes")
            return epoch

    def close_epoch(self, epoch_id: str) -> bool:
        """
        Mark an epoch as complete by setting its end_time.

        Args:
            epoch_id: ID of the epoch to close

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            if epoch_id not in self.epochs:
                self.logger.warning(f"Cannot close epoch {epoch_id}: Not found.")
                return False

            epoch = self.epochs[epoch_id]
            if not epoch.is_active():
                self.logger.debug(f"Epoch {epoch_id} is already closed.")
                return True

            # Close the epoch
            epoch.close()

            # No longer the current epoch
            if self.state.current_epoch_id == epoch_id:
                self.state.current_epoch_id = None

            # If we have episodic memory integration, we could analyze themes here
            if self.config.auto_generate_summaries:
                self._analyze_epoch_themes(epoch)

            self.logger.info(f"Closed epoch {epoch_id} named '{epoch.name}'")
            return True

    # -------------------------------------------------------------------------
    # Memory Retrieval
    # -------------------------------------------------------------------------

    def get_episode(self, episode_id: str, update_activation: bool = True) -> Optional[Episode]:
        """
        Retrieve an episode by ID, optionally updating its activation.

        Args:
            episode_id: ID of the episode to retrieve
            update_activation: Whether to update activation on access

        Returns:
            The Episode object or None if not found
        """
        with self.lock:
            episode = self.episodes.get(episode_id)
            if not episode:
                return None

            if update_activation:
                # Update episode activation
                episode.access()
                self.episode_activations[episode_id] = episode.activation

                # Update attention
                self._update_attention(episode_id)

            return episode

    def get_event(self, event_id: str, update_activation: bool = True) -> Optional[EpisodeEvent]:
        """
        Retrieve an event by ID, optionally updating its activation.

        Args:
            event_id: ID of the event to retrieve
            update_activation: Whether to update activation on access

        Returns:
            The EpisodeEvent object or None if not found
        """
        with self.lock:
            event = self.events.get(event_id)
            if not event:
                return None

            if update_activation:
                # Update event activation
                event.access()
                self.event_activations[event_id] = event.activation

            return event

    def get_epoch(self, epoch_id: str) -> Optional[Epoch]:
        """
        Retrieve an epoch by ID.

        Args:
            epoch_id: ID of the epoch to retrieve

        Returns:
            The Epoch object or None if not found
        """
        with self.lock:
            return self.epochs.get(epoch_id)

    def replay_episode(self,
                      episode_id: str,
                      callback: Callable[[EpisodeEvent], None],
                      update_activation: bool = True) -> bool:
        """
        Replay each event in the specified episode by calling the callback function.

        Args:
            episode_id: ID of the episode to replay
            callback: Function to call for each event
            update_activation: Whether to update activations during replay

        Returns:
            True if replay successful, False otherwise
        """
        with self.lock:
            episode = self.episodes.get(episode_id)
            if not episode:
                self.logger.warning(f"Cannot replay episode {episode_id}: Not found.")
                return False

            if update_activation:
                # Update episode activation
                episode.access()
                self.episode_activations[episode_id] = episode.activation

                # Update attention
                self._update_attention(episode_id)

            # Replay each event in order
            for event in episode.events:
                if update_activation:
                    # Update event activation
                    event.access()
                    self.event_activations[event.event_id] = event.activation

                # Call callback with the event
                try:
                    callback(event)
                except Exception as e:
                    self.logger.error(f"Error in replay callback for event {event.event_id}: {e}")
                    return False

            return True

    def retrieve_episodes(self, query: EpisodicQuery) -> List[Episode]:
        """
        Enhanced retrieval of episodes based on various criteria.

        Args:
            query: EpisodicQuery object with search parameters

        Returns:
            List of matching episodes, sorted by relevance
        """
        with self.lock:
            # If using consciousness-based retrieval, prime the activation landscape
            if query.retrieval_mode == RetrievalMode.CONTEXT:
                self._prime_activations_for_context(query)

            # Get all candidate episodes
            candidates = self._get_candidate_episodes(query)

            # Score candidates based on retrieval mode
            scored_episodes = []
            for episode_id in candidates:
                episode = self.episodes[episode_id]
                score = self._score_episode(episode, query)
                scored_episodes.append((score, episode))

            # Sort by score (descending)
            scored_episodes.sort(reverse=True, key=lambda x: x[0])

            # Get top results
            top_episodes = [episode for _, episode in scored_episodes[:query.max_results]]

            # If recursive retrieval is enabled, include associated episodes
            if query.recursive and top_episodes:
                associated_episodes = self._get_associated_episodes(top_episodes)

                # Add associated episodes not already in results
                added_ids = {ep.episode_id for ep in top_episodes}
                for score, episode in associated_episodes:
                    if episode.episode_id not in added_ids and len(top_episodes) < query.max_results:
                        top_episodes.append(episode)
                        added_ids.add(episode.episode_id)

            # Update activations and attention for retrieved episodes
            for episode in top_episodes:
                episode.access()
                self.episode_activations[episode.episode_id] = episode.activation

                # Only update attention for the top few episodes
                if len(self.state.attentional_focus) < self.config.attention_span:
                    self._update_attention(episode.episode_id)

            # If include_events is True, also retrieve and activate the events
            if query.include_events:
                for episode in top_episodes:
                    for event in episode.events:
                        event.access()
                        self.event_activations[event.event_id] = event.activation

            self.logger.debug(f"Retrieved {len(top_episodes)} episodes matching query")
            return top_episodes

    def retrieve_events(self,
                       text: Optional[str] = None,
                       embedding: Optional[np.ndarray] = None,
                       time_range: Optional[Tuple[float, float]] = None,
                       importance_threshold: float = 0.0,
                       max_results: int = 20) -> List[EpisodeEvent]:
        """
        Retrieve individual events matching the criteria.

        Args:
            text: Optional text to match against event content
            embedding: Optional embedding for similarity search
            time_range: Optional time range to filter by
            importance_threshold: Minimum importance for events
            max_results: Maximum number of events to return

        Returns:
            List of matching events, sorted by relevance
        """
        with self.lock:
            candidates = []

            # Check all events
            for event_id, event in self.events.items():
                # Apply filters
                if importance_threshold > 0 and event.importance < importance_threshold:
                    continue

                if time_range:
                    start_time, end_time = time_range
                    if event.timestamp < start_time or event.timestamp > end_time:
                        continue

                candidates.append(event)

            # Score candidates
            scored_events = []
            for event in candidates:
                score = 0.0

                # Text matching
                if text and isinstance(event.content, str):
                    if text.lower() in event.content.lower():
                        score += 0.5

                # Embedding similarity
                if embedding is not None and event.embedding is not None:
                    similarity = self._vector_similarity(embedding, event.embedding)
                    score += similarity * 0.7

                # Importance and activation boost
                score += event.importance * 0.2
                score += event.activation * 0.1

                scored_events.append((score, event))

            # Sort by score
            scored_events.sort(reverse=True, key=lambda x: x[0])

            # Get top results
            top_events = [event for _, event in scored_events[:max_results]]

            # Update activations for retrieved events
            for event in top_events:
                event.access()
                self.event_activations[event.event_id] = event.activation

            return top_events

    def get_temporal_context(self) -> List[Episode]:
        """
        Get the current temporal context - episodes that are temporally adjacent
        to the current focus of attention.

        Returns:
            List of episodes in the temporal context
        """
        with self.lock:
            # Get episodes in the temporal context
            context_episodes = []
            for episode_id in self.state.temporal_context:
                if episode_id in self.episodes:
                    context_episodes.append(self.episodes[episode_id])

            return context_episodes

    def get_attentional_focus(self) -> List[Episode]:
        """
        Get episodes currently in the attentional focus.

        Returns:
            List of episodes in the attentional focus
        """
        with self.lock:
            focus_episodes = []
            for episode_id in self.state.attentional_focus:
                if episode_id in self.episodes:
                    focus_episodes.append(self.episodes[episode_id])

            return focus_episodes

    def query_by_time_range(self, start_time: float, end_time: float) -> List[Episode]:
        """
        Find episodes that overlap with the given time range.

        Args:
            start_time: Start of the time range
            end_time: End of the time range

        Returns:
            List of episodes that overlap with the time range
        """
        with self.lock:
            results = []

            # Linear search through episodes
            # (A more efficient implementation would use a time-indexed data structure)
            for episode_id, episode in self.episodes.items():
                episode_end = episode.end_time if episode.end_time is not None else time.time()

                # Check for overlap
                if not (episode_end < start_time or episode.start_time > end_time):
                    results.append(episode)

            # Sort by start time
            results.sort(key=lambda ep: ep.start_time)
            return results

    def query_by_tag(self, tag: str) -> List[Episode]:
        """
        Find episodes that have the specified tag.

        Args:
            tag: Tag to search for

        Returns:
            List of episodes with the tag
        """
        with self.lock:
            results = []

            # Use the tag index
            for episode_id in self.episodes_by_tag.get(tag, []):
                if episode_id in self.episodes:
                    results.append(self.episodes[episode_id])

            return results

    def predict_next_episodes(self,
                             current_episode_id: Optional[str] = None,
                             top_n: int = 3) -> List[Tuple[Episode, float]]:
        """
        Predict the most likely next episodes based on learned temporal patterns.
        Uses HTM-inspired sequence memory.

        Args:
            current_episode_id: ID of the current episode, or None to use the last one
            top_n: Number of predictions to return

        Returns:
            List of (episode, probability) tuples
        """
        with self.lock:
            # If no current episode specified, use the current or last episode
            if current_episode_id is None:
                if self.state.current_episode_id:
                    current_episode_id = self.state.current_episode_id
                elif self.state.temporal_context:
                    current_episode_id = self.state.temporal_context[-1]
                else:
                    return []

            # Check if episode exists
            if current_episode_id not in self.episodes:
                return []

            # Get predictions
            if current_episode_id in self.temporal_sequences:
                predictions = self.temporal_sequences[current_episode_id]

                # Sort by probability
                sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)

                # Get top N
                results = []
                for pred_id, prob in sorted_predictions[:top_n]:
                    if pred_id in self.episodes:
                        results.append((self.episodes[pred_id], prob))

                return results

            return []

    # -------------------------------------------------------------------------
    # Memory Maintenance and Optimization
    # -------------------------------------------------------------------------

    def clear_all(self) -> None:
        """
        Clear all episodes, events, and epochs from memory.
        """
        with self.lock:
            self.episodes.clear()
            self.events.clear()
            self.epochs.clear()

            self.short_term_episodes.clear()
            self.long_term_episodes.clear()

            for etype in EpisodeType:
                self.episodes_by_type[etype].clear()

            self.episodes_by_tag.clear()
            self.episode_timeline.clear()

            self.episode_activations.clear()
            self.event_activations.clear()

            self.temporal_sequences.clear()
            self.active_predictions.clear()

            self.state = EpisodicMemoryState()

            self.last_decay_time = time.time()
            self.last_consolidation_time = time.time()

            self.logger.info("Episodic memory cleared")

    def _consolidate_memory(self) -> None:
        """
        Perform memory maintenance operations:
        1. Decay event and episode activations
        2. Move episodes between short-term and long-term memory
        3. Prune episodes if above capacity
        4. Update associations between episodes

        This implements concepts from ACT-R (activation decay) and
        LIDA (memory consolidation).
        """
        with self.lock:
            current_time = time.time()

            # Update decay time
            self.last_consolidation_time = current_time

            # Apply activation decay
            self._apply_activation_decay()

            # Check short-term capacity
            if len(self.short_term_episodes) > self.config.short_term_capacity:
                # Select episodes to move to long-term memory
                to_consolidate = len(self.short_term_episodes) - self.config.short_term_capacity
                self._move_episodes_to_long_term(to_consolidate)

            # Check long-term capacity
            if len(self.long_term_episodes) > self.config.long_term_capacity:
                # Remove excess episodes
                to_remove = len(self.long_term_episodes) - self.config.long_term_capacity
                self._prune_episodes(to_remove)

            # Update associations between episodes
            self._update_episode_associations()

            # Update temporal sequence models
            if self.config.enable_predictive_coding:
                self._update_sequence_models()

            # Update epochs if needed
            self._update_epochs()

            self.logger.debug(f"Memory consolidation complete. Short-term: {len(self.short_term_episodes)}, "
                             f"Long-term: {len(self.long_term_episodes)}")

    def _apply_activation_decay(self) -> None:
        """
        Apply ACT-R-inspired activation decay to events and episodes.
        """
        current_time = time.time()
        time_since_decay = current_time - self.last_decay_time
        self.last_decay_time = current_time

        # Only decay if enough time has passed (efficiency optimization)
        if time_since_decay < 1.0:
            return

        # Apply decay to events
        for event_id, event in self.events.items():
            event.decay_activation(self.config.event_decay_rate, current_time)
            self.event_activations[event_id] = event.activation

        # Apply decay to episodes
        for episode_id, episode in self.episodes.items():
            episode.decay_activation(self.config.episode_decay_rate, current_time)
            self.episode_activations[episode_id] = episode.activation

    def _move_episodes_to_long_term(self, count: int) -> None:
        """
        Move episodes from short-term to long-term memory based on
        consolidation criteria (importance, activation, etc.).

        Args:
            count: Number of episodes to move
        """
        if count <= 0:
            return

        # Score episodes for consolidation
        scored_episodes = []
        for episode_id in self.short_term_episodes:
            if episode_id in self.episodes:
                episode = self.episodes[episode_id]

                # Compute consolidation score
                # Episodes with high importance but low activation are prime candidates
                consolidation_score = (
                    episode.importance * 0.4 +    # Higher importance -> more likely to consolidate
                    (1.0 - episode.activation) * 0.3 +  # Lower activation -> more likely to consolidate
                    (episode.is_active() == False) * 0.3  # Closed episodes are prioritized
                )

                scored_episodes.append((consolidation_score, episode_id))

        # Sort by score (descending)
        scored_episodes.sort(reverse=True)

        # Select episodes to move
        to_move = [ep_id for _, ep_id in scored_episodes[:count]]

        # Move to long-term memory
        for episode_id in to_move:
            if episode_id in self.short_term_episodes:
                self.short_term_episodes.remove(episode_id)
                self.long_term_episodes.append(episode_id)

                # Generate summary if not already done
                episode = self.episodes[episode_id]
                if self.config.auto_generate_summaries and not episode.summary:
                    episode.generate_summary()

        self.logger.debug(f"Moved {len(to_move)} episodes to long-term memory")

    def _prune_episodes(self, count: int) -> None:
        """
        Remove the least important/relevant episodes from long-term memory.

        Args:
            count: Number of episodes to remove
        """
        if count <= 0:
            return

        # Score episodes for removal
        scored_episodes = []
        for episode_id in self.long_term_episodes:
            if episode_id in self.episodes:
                episode = self.episodes[episode_id]

                # Compute removal score
                # Lower score = higher priority for removal
                removal_score = (
                    episode.importance * 0.4 +          # Higher importance -> less likely to remove
                    episode.activation * 0.3 +          # Higher activation -> less likely to remove
                    (len(episode.events) > 0) * 0.2 +   # Episodes with events are less likely to be removed
                    (len(episode.associations) > 0) * 0.1  # Episodes with associations are less likely to be removed
                )

                scored_episodes.append((removal_score, episode_id))

        # Sort by score (ascending - lowest scores removed first)
        scored_episodes.sort()

        # Select episodes to remove
        to_remove = [ep_id for _, ep_id in scored_episodes[:count]]

        # Remove episodes
        for episode_id in to_remove:
            self._remove_episode(episode_id)

        self.logger.debug(f"Pruned {len(to_remove)} episodes from long-term memory")

    def _remove_episode(self, episode_id: str) -> None:
        """
        Internal method to remove an episode and its events.

        Args:
            episode_id: ID of the episode to remove
        """
        if episode_id not in self.episodes:
            return

        episode = self.episodes[episode_id]

        # Remove from both memory stores
        if episode_id in self.short_term_episodes:
            self.short_term_episodes.remove(episode_id)
        if episode_id in self.long_term_episodes:
            self.long_term_episodes.remove(episode_id)

        # Remove from type index
        if episode.episode_type in self.episodes_by_type and episode_id in self.episodes_by_type[episode.episode_type]:
            self.episodes_by_type[episode.episode_type].remove(episode_id)

        # Remove from tag index
        for tag in episode.tags:
            if tag in self.episodes_by_tag and episode_id in self.episodes_by_tag[tag]:
                self.episodes_by_tag[tag].remove(episode_id)

        # Remove from timeline
        self.episode_timeline = [(t, eid) for t, eid in self.episode_timeline if eid != episode_id]

        # Remove from attention and temporal context
        if episode_id in self.state.attentional_focus:
            self.state.attentional_focus.remove(episode_id)
        if episode_id in self.state.temporal_context:
            self.state.temporal_context.remove(episode_id)

        # Remove from activations
        if episode_id in self.episode_activations:
            del self.episode_activations[episode_id]

        # Remove from temporal sequences
        if episode_id in self.temporal_sequences:
            del self.temporal_sequences[episode_id]
        for seq in self.temporal_sequences.values():
            if episode_id in seq:
                del seq[episode_id]

        # Remove from epochs
        for epoch_id, epoch in self.epochs.items():
            if episode_id in epoch.episodes:
                epoch.episodes.remove(episode_id)

        # Remove associated events
        for event in episode.events:
            if event.event_id in self.events:
                del self.events[event.event_id]
            if event.event_id in self.event_activations:
                del self.event_activations[event.event_id]

        # Remove episode itself
        del self.episodes[episode_id]

    def _update_episode_associations(self) -> None:
        """
        Update associations between episodes based on:
        - Shared tags
        - Temporal proximity
        - Content similarity
        - Causal relationships

        This implements concepts from LIDA (associative networks).
        """
        # This can be computationally expensive, so we'll limit updates
        # to recently accessed episodes
        current_time = time.time()
        recent_threshold = current_time - 3600  # Last hour

        recent_episodes = [
            episode_id for episode_id, episode in self.episodes.items()
            if episode.last_access_time >= recent_threshold or episode.activation > 0.5
        ]

        # Limit to a reasonable number
        if len(recent_episodes) > 50:
            # Select the most recently accessed ones
            recent_episodes = sorted(
                recent_episodes,
                key=lambda ep_id: self.episodes[ep_id].last_access_time,
                reverse=True
            )[:50]

        # Update associations between these episodes
        for i, ep1_id in enumerate(recent_episodes):
            if ep1_id not in self.episodes:
                continue

            ep1 = self.episodes[ep1_id]

            for j in range(i+1, len(recent_episodes)):
                ep2_id = recent_episodes[j]
                if ep2_id not in self.episodes:
                    continue

                ep2 = self.episodes[ep2_id]

                # Calculate association strength
                assoc_strength = 0.0

                # Tag similarity
                tag_intersection = ep1.tags.intersection(ep2.tags)
                tag_union = ep1.tags.union(ep2.tags)
                if tag_union:
                    tag_similarity = len(tag_intersection) / len(tag_union)
                    assoc_strength += tag_similarity * 0.3

                # Temporal proximity
                if ep1.end_time and ep2.start_time:
                    # Episodes that occur close together in time
                    time_diff = abs(ep1.end_time - ep2.start_time)
                    time_factor = math.exp(-time_diff / 3600)  # Decay factor, 1 hour half-life
                    assoc_strength += time_factor * 0.2

                # Embedding similarity
                if ep1.embedding is not None and ep2.embedding is not None:
                    embedding_sim = self._vector_similarity(ep1.embedding, ep2.embedding)
                    assoc_strength += embedding_sim * 0.4

                # Context similarity
                context_match = sum(1 for k, v in ep1.context.items()
                                  if k in ep2.context and ep2.context[k] == v)
                context_total = len(set(ep1.context.keys()).union(ep2.context.keys()))
                if context_total > 0:
                    context_sim = context_match / context_total
                    assoc_strength += context_sim * 0.1

                # Only store significant associations
                if assoc_strength > 0.3:
                    ep1.associations[ep2_id] = assoc_strength
                    ep2.associations[ep1_id] = assoc_strength

    def _update_sequence_models(self) -> None:
        """
        Update HTM-inspired sequence models for episode prediction.
        """
        # Get timeline of recently closed episodes
        closed_episodes = [(t, eid) for t, eid in self.episode_timeline
                        if eid in self.episodes and not self.episodes[eid].is_active()]

        if len(closed_episodes) < 2:
            return

        # Update sequence model based on observed transitions
        for i in range(len(closed_episodes) - 1):
            current_ep_id = closed_episodes[i][1]
            next_ep_id = closed_episodes[i+1][1]

            # Add or update transition probability
            if current_ep_id not in self.temporal_sequences:
                self.temporal_sequences[current_ep_id] = {next_ep_id: 1.0}
            else:
                self.temporal_sequences[current_ep_id][next_ep_id] = self.temporal_sequences[current_ep_id].get(next_ep_id, 0) + 1.0

        # Normalize probabilities
        for source_id, transitions in self.temporal_sequences.items():
            total = sum(transitions.values())
            if total > 0:
                for target_id in transitions:
                    transitions[target_id] /= total

    def _update_epochs(self) -> None:
        """
        Update epoch information, such as theme detection.
        """
        # Find active epochs that have been updated
        for epoch_id, epoch in self.epochs.items():
            if not epoch.is_active():
                continue

            # Check if we have new episodes in this epoch
            episodes_updated = False
            for episode_id in epoch.episodes:
                if episode_id in self.episodes:
                    episode = self.episodes[episode_id]
                    if episode.last_access_time > self.last_consolidation_time:
                        episodes_updated = True
                        break

            if not episodes_updated:
                continue

            # Update epoch themes
            self._analyze_epoch_themes(epoch)

    def _analyze_epoch_themes(self, epoch: Epoch) -> None:
        """
        Analyze themes across episodes in an epoch.
        """
        # Skip epochs with too few episodes
        if len(epoch.episodes) < 2:
            return

        # Collect tags from all episodes
        all_tags = defaultdict(int)
        for episode_id in epoch.episodes:
            if episode_id in self.episodes:
                episode = self.episodes[episode_id]
                for tag in episode.tags:
                    all_tags[tag] += 1

        # Identify common themes
        total_episodes = len(epoch.episodes)
        themes = {}
        for tag, count in all_tags.items():
            if count >= 2:  # At least 2 episodes have this tag
                theme_strength = count / total_episodes
                themes[tag] = theme_strength

        # Update epoch themes
        epoch.themes = themes

    def _update_temporal_links(self, episode_id: str) -> None:
        """
        Update temporal links when an episode is closed.
        """
        # Add to temporal context
        if episode_id not in self.state.temporal_context:
            self.state.temporal_context.append(episode_id)
            # Keep only the most recent episodes in the context
            if len(self.state.temporal_context) > self.config.temporal_context_size:
                self.state.temporal_context.pop(0)

    def _update_attention(self, episode_id: str) -> None:
        """
        Update attentional focus with a newly accessed episode.
        Based on LIDA's attentional mechanisms.
        """
        # If already in focus, move to the end (most recent)
        if episode_id in self.state.attentional_focus:
            self.state.attentional_focus.remove(episode_id)
            self.state.attentional_focus.append(episode_id)
        else:
            # Add to focus
            self.state.attentional_focus.append(episode_id)

            # Limit focus size
            if len(self.state.attentional_focus) > self.config.attention_span:
                self.state.attentional_focus.pop(0)

    def _update_predictions(self, episode: Episode, event: EpisodeEvent) -> None:
        """
        Update HTM-style predictions based on a new event.
        """
        if not self.config.enable_predictive_coding or len(episode.events) < 2:
            return

        # Get previous event(s)
        prev_events = [e for e in episode.events if e.event_id != event.event_id and e.timestamp < event.timestamp]
        if not prev_events:
            return

        prev_event = max(prev_events, key=lambda e: e.timestamp)

        # Update sequence predictions
        if prev_event.event_id in episode.sequence_predictors:
            # Check if our prediction was correct
            predicted_events = episode.sequence_predictors[prev_event.event_id]
            for predicted_id, prob in list(predicted_events.items()):
                # Adjust probability based on match/miss
                if predicted_id == event.event_id:
                    # Correct prediction - increase probability
                    predicted_events[predicted_id] = min(1.0, prob * 1.1)
                else:
                    # Incorrect - decrease probability
                    predicted_events[predicted_id] = max(0.01, prob * 0.9)

        # Make new predictions based on this event
        episode.sequence_predictors.setdefault(event.event_id, {})

    def _learn_episode_sequence(self, episode_id: str) -> None:
        """
        Learn sequence patterns when an episode is closed.
        """
        if not self.config.enable_predictive_coding:
            return

        # Find episodes that preceded and followed this one
        timeline_idx = None
        for i, (_, eid) in enumerate(self.episode_timeline):
            if eid == episode_id:
                timeline_idx = i
                break

        if timeline_idx is None:
            return

        # Get preceding episode
        if timeline_idx > 0:
            prev_id = self.episode_timeline[timeline_idx-1][1]
            # Update sequence model
            if prev_id not in self.temporal_sequences:
                self.temporal_sequences[prev_id] = {episode_id: 1.0}
            else:
                self.temporal_sequences[prev_id][episode_id] = self.temporal_sequences[prev_id].get(episode_id, 0) + 1.0
                # Normalize
                total = sum(self.temporal_sequences[prev_id].values())
                for target_id in self.temporal_sequences[prev_id]:
                    self.temporal_sequences[prev_id][target_id] /= total

    def _prime_activations_for_context(self, query: EpisodicQuery) -> None:
        """
        Prime the activation landscape to enhance context-sensitive retrieval.
        """
        # Boost activation of episodes in the current temporal context
        for episode_id in self.state.temporal_context:
            if episode_id in self.episodes:
                episode = self.episodes[episode_id]
                episode.activation = min(1.0, episode.activation * 1.2)
                self.episode_activations[episode_id] = episode.activation

        # Boost activation of episodes in the attentional focus
        for episode_id in self.state.attentional_focus:
            if episode_id in self.episodes:
                episode = self.episodes[episode_id]
                episode.activation = min(1.0, episode.activation * 1.5)
                self.episode_activations[episode_id] = episode.activation

        # If context factors are provided, boost matching episodes
        if query.context_factors:
            for episode_id, episode in self.episodes.items():
                match_score = 0
                for k, v in query.context_factors.items():
                    if k in episode.context and episode.context[k] == v:
                        match_score += 1

                if match_score > 0:
                    boost_factor = 1.0 + (match_score / len(query.context_factors)) * 0.5
                    episode.activation = min(1.0, episode.activation * boost_factor)
                    self.episode_activations[episode_id] = episode.activation

    def _get_candidate_episodes(self, query: EpisodicQuery) -> List[str]:
        """
        Get the initial set of candidate episodes based on query filters.
        """
        candidates = set()

        # Apply hard filters first

        # Filter by episode type
        if query.episode_types:
            for etype in query.episode_types:
                candidates.update(self.episodes_by_type[etype])
        else:
            # Use all episodes
            candidates.update(self.episodes.keys())

        # Filter by tags (if specified)
        if query.tags:
            tag_matches = set()
            for tag in query.tags:
                tag_matches.update(self.episodes_by_tag.get(tag, []))
            if tag_matches:
                candidates = candidates.intersection(tag_matches)

        # Filter by time range (if specified)
        if query.time_range:
            start_time, end_time = query.time_range
            time_matches = set()
            for episode_id in candidates:
                episode = self.episodes[episode_id]
                episode_end = episode.end_time if episode.end_time is not None else time.time()
                if not (episode_end < start_time or episode.start_time > end_time):
                    time_matches.add(episode_id)
            candidates = time_matches

        # Filter by memory strength (if specified)
        if query.memory_strength:
            strength_matches = set()
            for episode_id in candidates:
                episode = self.episodes[episode_id]
                if episode.memory_strength.value >= query.memory_strength.value:
                    strength_matches.add(episode_id)
            candidates = strength_matches

        # Filter by activation threshold
        activation_matches = set()
        for episode_id in candidates:
            if episode_id in self.episode_activations:
                activation = self.episode_activations[episode_id]
                if activation >= query.activation_threshold:
                    activation_matches.add(episode_id)
        candidates = activation_matches

        # Filter by importance threshold
        importance_matches = set()
        for episode_id in candidates:
            episode = self.episodes[episode_id]
            if episode.importance >= query.importance_threshold:
                importance_matches.add(episode_id)
        candidates = importance_matches

        return list(candidates)

    def _score_episode(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score an episode based on how well it matches the query.
        The scoring method depends on the retrieval mode.
        """
        score = 0.0

        # Base score components
        if query.retrieval_mode == RetrievalMode.EXACT:
            # Exact mode prioritizes precise matches
            score += self._exact_match_score(episode, query)

        elif query.retrieval_mode == RetrievalMode.SIMILARITY:
            # Similarity mode prioritizes embedding similarity
            score += self._similarity_score(episode, query)

        elif query.retrieval_mode == RetrievalMode.TEMPORAL:
            # Temporal mode prioritizes recency and sequence patterns
            score += self._temporal_score(episode, query)

        elif query.retrieval_mode == RetrievalMode.CAUSAL:
            # Causal mode prioritizes cause-effect relationships
            score += self._causal_score(episode, query)

        elif query.retrieval_mode == RetrievalMode.ASSOCIATIVE:
            # Associative mode prioritizes linked episodes
            score += self._associative_score(episode, query)

        elif query.retrieval_mode == RetrievalMode.CONTEXT:
            # Context mode prioritizes contextual relevance
            score += self._context_score(episode, query)

        else:  # RetrievalMode.MIXED or fallback
            # Mixed mode uses a balanced approach
            score += self._exact_match_score(episode, query) * 0.2
            score += self._similarity_score(episode, query) * 0.3
            score += self._temporal_score(episode, query) * 0.15
            score += self._causal_score(episode, query) * 0.1
            score += self._associative_score(episode, query) * 0.1
            score += self._context_score(episode, query) * 0.15

        # Additional factors that apply to all modes

        # Boost for episodes in attentional focus
        if episode.episode_id in self.state.attentional_focus:
            score += 0.3

        # Boost for episodes in temporal context
        if episode.episode_id in self.state.temporal_context:
            score += 0.2

        # Boost for high activation (already in consciousness)
        score += episode.activation * 0.2

        # Boost for high importance
        score += episode.importance * 0.1

        return score

    def _exact_match_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on exact matches to query criteria.
        """
        score = 0.0

        # Text matching
        if query.text:
            # Check episode summary
            if episode.summary and query.text.lower() in episode.summary.lower():
                score += 0.5

            # Check event content
            content_matches = 0
            for event in episode.events:
                if isinstance(event.content, str) and query.text.lower() in event.content.lower():
                    content_matches += 1

            if content_matches > 0:
                # Logarithmic scoring to prevent domination by episodes with many events
                score += min(0.5, 0.1 * math.log(1 + content_matches))

        # Tag matching
        if query.tags:
            matching_tags = len(query.tags.intersection(episode.tags))
            if matching_tags > 0:
                score += 0.5 * (matching_tags / len(query.tags))

        return score

    def _similarity_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on embedding similarity.
        """
        score = 0.0

        # Embedding similarity
        if query.embedding is not None and episode.embedding is not None:
            similarity = self._vector_similarity(query.embedding, episode.embedding)
            score += similarity

        return score

    def _temporal_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on temporal factors (recency, sequence patterns).
        """
        score = 0.0

        # Recency factor
        current_time = time.time()
        episode_time = episode.end_time if episode.end_time else episode.start_time
        time_diff = current_time - episode_time
        recency = math.exp(-time_diff / 86400)  # 1-day half-life
        score += recency * 0.5

        # Sequence factor
        if self.state.current_episode_id in self.temporal_sequences:
            predictions = self.temporal_sequences[self.state.current_episode_id]
            if episode.episode_id in predictions:
                sequence_prob = predictions[episode.episode_id]
                score += sequence_prob * 0.5

        return score

    def _causal_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on causal relationships.
        """
        score = 0.0

        # Causal links through events
        causal_count = 0
        for event in episode.events:
            causal_count += len(event.causal_links)

        if causal_count > 0:
            # Logarithmic scaling to prevent domination
            score += min(0.5, 0.1 * math.log(1 + causal_count))

        # If current episode is active, check for causal links
        if self.state.current_episode_id and self.state.current_episode_id in self.episodes:
            current_ep = self.episodes[self.state.current_episode_id]
            for event in current_ep.events:
                if episode.episode_id in event.causal_links:
                    score += 0.5
                    break

        return score

    def _associative_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on associative links to other episodes.
        """
        score = 0.0

        # Associations with episodes in attentional focus
        for focus_id in self.state.attentional_focus:
            if focus_id in episode.associations:
                assoc_strength = episode.associations[focus_id]
                score += assoc_strength * 0.5

        # Associations with episodes matching the query
        if query.text and self.config.embedding_function:
            try:
                query_embedding = self.config.embedding_function(query.text)
                for assoc_id, strength in episode.associations.items():
                    if assoc_id in self.episodes:
                        assoc_ep = self.episodes[assoc_id]
                        if assoc_ep.embedding is not None:
                            sim = self._vector_similarity(query_embedding, assoc_ep.embedding)
                            if sim > 0.7:  # High similarity threshold
                                score += strength * sim * 0.5
            except Exception as e:
                self.logger.error(f"Error computing associative score: {e}")

        return score

    def _context_score(self, episode: Episode, query: EpisodicQuery) -> float:
        """
        Score based on contextual relevance.
        """
        score = 0.0

        # Context factors
        if query.context_factors:
            matches = 0
            for k, v in query.context_factors.items():
                if k in episode.context and episode.context[k] == v:
                    matches += 1

            if matches > 0:
                score += 0.5 * (matches / len(query.context_factors))

        # Location match
        if "location" in query.context_factors and episode.location:
            if query.context_factors["location"] == episode.location:
                score += 0.3

        return score

    def _get_associated_episodes(self, episodes: List[Episode]) -> List[Tuple[float, Episode]]:
        """
        Find episodes associated with the given list of episodes.

        Args:
            episodes: List of episodes to find associations for

        Returns:
            List of (score, episode) tuples
        """
        associated = {}

        # Collect all associations
        for episode in episodes:
            for assoc_id, strength in episode.associations.items():
                if assoc_id in self.episodes and assoc_id != episode.episode_id:
                    if assoc_id in associated:
                        associated[assoc_id] = max(associated[assoc_id], strength)
                    else:
                        associated[assoc_id] = strength

        # Convert to list of (score, episode) tuples
        result = []
        for assoc_id, strength in associated.items():
            result.append((strength, self.episodes[assoc_id]))

        # Sort by strength
        result.sort(reverse=True, key=lambda x: x[0])

        return result

    def _sort_timeline(self) -> None:
        """Sort the episode timeline by timestamp."""
        self.episode_timeline.sort(key=lambda x: x[0])

    def _generate_sparse_representation(self, content: Any) -> Optional[np.ndarray]:
        """
        Generate a sparse distributed representation (SDR) of content.
        Inspired by HTM's sparse encodings.

        This is a simplified implementation that:
        1. Creates a sparse binary vector
        2. Sets a small percentage of bits to 1
        3. Positions of 1s represent semantic features

        Args:
            content: Content to generate SDR for

        Returns:
            Sparse binary vector or None if cannot be generated
        """
        if not self.config.enable_predictive_coding:
            return None

        # SDR parameters
        n = 1024  # Total bits
        w = 40    # Active bits (sparsity ~4%)

        # Initialize empty SDR
        sdr = np.zeros(n, dtype=np.int8)

        try:
            # Simple hash-based approach for deterministic SDRs
            if isinstance(content, str):
                # For strings, hash character sequences
                chunks = [content[i:i+3] for i in range(0, len(content), 3)]
                for chunk in chunks:
                    # Get a hash value for this chunk
                    hash_val = hash(chunk) % n
                    # Set that bit to 1
                    sdr[hash_val] = 1
            elif isinstance(content, (int, float)):
                # For numbers, use value-based encoding
                val = float(content)
                # Scale to 0-1 range (assuming reasonable bounds)
                normalized = (val + 1000) / 2000
                # Set w bits based on this value
                center = int(normalized * n)
                half_width = w // 2
                start = max(0, center - half_width)
                end = min(n, center + half_width)
                sdr[start:end] = 1
            else:
                # For other types, use hash of string representation
                s = str(content)
                hash_val = hash(s)
                for i in range(w):
                    bit_pos = (hash_val + i * 16777619) % n  # FNV-like hash spreading
                    sdr[bit_pos] = 1
        except Exception as e:
            self.logger.error(f"Error generating SDR: {e}")
            return None

        # Ensure we have exactly w bits set to 1
        active_bits = np.sum(sdr)
        if active_bits < w:
            # Need to add more active bits
            zero_positions = np.where(sdr == 0)[0]
            to_activate = np.random.choice(zero_positions, size=w-active_bits, replace=False)
            sdr[to_activate] = 1
        elif active_bits > w:
            # Need to remove some active bits
            one_positions = np.where(sdr == 1)[0]
            to_deactivate = np.random.choice(one_positions, size=active_bits-w, replace=False)
            sdr[to_deactivate] = 0

        return sdr

    def _vector_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Calculate similarity between two vectors.

        Args:
            vec1: First vector
            vec2: Second vector

        Returns:
            Similarity score between 0 and 1
        """
        # Check if SDRs
        if vec1.dtype == np.int8 and vec2.dtype == np.int8 and np.max(vec1) <= 1 and np.max(vec2) <= 1:
            # For SDRs, use overlap measure
            intersection = np.sum(np.logical_and(vec1, vec2))
            union = np.sum(np.logical_or(vec1, vec2))
            if union == 0:
                return 0.0
            return intersection / union

        # Otherwise use cosine similarity
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)

        if norm1 == 0 or norm2 == 0:
            return 0.0

        dot_product = np.dot(vec1, vec2)
        similarity = dot_product / (norm1 * norm2)

        # Ensure result is in [0, 1] range
        return max(0.0, min(1.0, similarity))

    def _auto_tag_episode(self, episode: Episode) -> None:
        """
        Automatically generate tags for an episode based on its content.
        """
        if not self.config.auto_tagging:
            return

        # This is a simplified implementation
        # A real implementation might use more sophisticated NLP

        # Collect words from content and context
        words = set()

        # Add words from summary if available
        if episode.summary:
            words.update(episode.summary.lower().split())

        # Add words from context
        for k, v in episode.context.items():
            if isinstance(v, str):
                words.update(v.lower().split())

        # Add words from event content
        for event in episode.events:
            if isinstance(event.content, str):
                words.update(event.content.lower().split())

        # Filter out common stop words
        stop_words = {"the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "with", "by", "of", "is", "are", "was", "were"}
        words = words - stop_words

        # Keep only words of reasonable length
        words = {w for w in words if len(w) >= 4 and len(w) <= 20}

        # Add as tags (limit to top 5)
        for word in list(words)[:5]:
            episode.tags.add(word)
            # Update tag index
            self.episodes_by_tag[word].append(episode.episode_id)

# Semantic

In [ ]:
"""
semantic.py - Enhanced Semantic Memory for GrandNexus

This module combines the best features of the original implementation with
approaches inspired by BioNeMo for biological ontologies, providing:

1. Multi-level abstraction for concepts
2. Dynamic weights for different relation types
3. Improved ontological reasoning with transitive inference
4. Activation spreading for related concepts
5. Context-aware retrieval with relevance scoring
"""

import logging
import threading
import uuid
import math
import numpy as np
from dataclasses import dataclass, field
from typing import (
    Any, Dict, List, Optional, Set, Tuple, Callable, Union
)
from collections import defaultdict, deque
from enum import Enum, auto
# Ajout à verifier si il est cohérent
from grandnexus.graph.graph_engine import AbstractGraphEngine, create_graph_engine, GraphQuery, GraphNode


class AbstractionLevel(Enum):
    """Represents different levels of abstraction for concepts."""
    CONCRETE = 1      # Very specific instances or entities
    SPECIFIC = 2      # Specific types/categories
    GENERAL = 3       # General categories
    ABSTRACT = 4      # Abstract concepts
    UNIVERSAL = 5     # Universal principles or concepts


class RelationType(str, Enum):
    """Standard relation types with predefined semantics."""
    IS_A = "is_a"                 # Inheritance/subsumption (e.g., "dog is_a animal")
    PART_OF = "part_of"           # Composition (e.g., "wheel part_of car")
    HAS_PROPERTY = "has_property" # Attributes (e.g., "apple has_property red")
    RELATED_TO = "related_to"     # Generic association
    CAUSES = "causes"             # Causal relationship
    PRECEDED_BY = "preceded_by"   # Temporal precedence
    FOLLOWED_BY = "followed_by"   # Temporal succession
    SIMILAR_TO = "similar_to"     # Similarity or analogy
    OPPOSITE_OF = "opposite_of"   # Opposition or antonymy
    INSTANCE_OF = "instance_of"   # Instance relationship (e.g., "Fido instance_of dog")
    DEFINED_AS = "defined_as"     # Definition relationship


@dataclass
class Concept:
    """
    Represents a concept in the semantic memory with enhanced BioNeMo-inspired features.

    Attributes:
        concept_id (str): Unique identifier.
        name (str): Human-readable name (e.g., "Dog").
        description (str): Optional textual description.
        properties (Dict[str, Any]): Arbitrary attributes or key-value pairs.
        embedding (Optional[np.ndarray]): Vector representation for similarity calculations.
        synonyms (Set[str]): Alternate names or labels for the concept.
        importance (float): A measure of how relevant or "important" this concept is (0..1).
        abstraction_level (AbstractionLevel): The level of abstraction of this concept.
        activation (float): Current activation level (for spreading activation).
        last_accessed (float): Timestamp of last access (for recency effects).
        access_count (int): Number of times this concept has been accessed.
        confidence (float): Confidence in the concept's validity (0..1).
        source (str): Source of the concept (e.g., "user", "llm", "reasoning").
    """
    concept_id: str
    name: str
    description: str = ""
    properties: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None
    synonyms: Set[str] = field(default_factory=set)
    importance: float = 0.5
    abstraction_level: AbstractionLevel = AbstractionLevel.SPECIFIC
    activation: float = 0.0
    last_accessed: float = field(default_factory=lambda: 0.0)
    access_count: int = 0
    confidence: float = 1.0
    source: str = "user"

    def __repr__(self):
        return (f"<Concept id={self.concept_id}, "
                f"name={self.name}, level={self.abstraction_level.name}, "
                f"importance={self.importance:.2f}, activation={self.activation:.2f}>")


@dataclass
class Relation:
    """
    Represents a relationship between two concepts with enhanced features.

    Attributes:
        relation_id (str): Unique identifier for this relation.
        source_concept_id (str): ID of the source concept.
        relation_type (str): Type/category of the relationship.
        target_concept_id (str): ID of the target concept.
        weight (float): Strength/confidence of the relationship (0..1).
        properties (Dict[str, Any]): Additional info about the relation.
        bidirectional (bool): Whether the relation works in both directions.
        transitive (bool): Whether the relation is transitive.
        temporal (bool): Whether the relation has temporal significance.
        created_at (float): Timestamp when relation was created.
        confidence (float): Confidence in this relation's validity (0..1).
        source (str): Source of this relation (e.g., "user", "inference").
    """
    relation_id: str
    source_concept_id: str
    relation_type: Union[RelationType, str]
    target_concept_id: str
    weight: float = 1.0
    properties: Dict[str, Any] = field(default_factory=dict)
    bidirectional: bool = False
    transitive: bool = False
    temporal: bool = False
    created_at: float = field(default_factory=lambda: 0.0)
    confidence: float = 1.0
    source: str = "user"

    def __post_init__(self):
        """Automatically set typical properties for standard relation types."""
        if isinstance(self.relation_type, str) and not isinstance(self.relation_type, RelationType):
            # Try to convert string to RelationType if possible
            try:
                self.relation_type = RelationType(self.relation_type)
            except ValueError:
                # Keep as string if not a standard type
                pass

        # Set defaults for standard relation types
        if isinstance(self.relation_type, RelationType):
            if self.relation_type == RelationType.IS_A:
                self.transitive = True
            elif self.relation_type == RelationType.PART_OF:
                self.transitive = True
            elif self.relation_type == RelationType.SIMILAR_TO:
                self.bidirectional = True
            elif self.relation_type == RelationType.OPPOSITE_OF:
                self.bidirectional = True
            elif self.relation_type == RelationType.PRECEDED_BY or self.relation_type == RelationType.FOLLOWED_BY:
                self.temporal = True

    def __repr__(self):
        return (f"<Relation id={self.relation_id}, "
                f"{self.source_concept_id} -{self.relation_type}-> "
                f"{self.target_concept_id}, weight={self.weight:.2f}, confidence={self.confidence:.2f}>")


@dataclass
class SemanticQuery:
    """
    Enhanced query container for specifying parameters when searching for concepts.

    Attributes:
        text (Optional[str]): A text string to match against a concept's name or synonyms.
        embedding (Optional[np.ndarray]): If set, we do an embedding-based similarity search.
        similarity_metric (str): 'cosine' or 'euclidean'.
        importance_threshold (float): Minimum concept importance required.
        max_results (int): Limit on returned concepts.
        relationship_filters (Optional[List[Tuple[str, str]]]):
            A list of (relation_type, concept_id). The concept must have a relationship of
            that type to the specified concept_id.
        synonyms_match (bool): If True, we also match `text` against synonyms.
        abstraction_levels (Optional[List[AbstractionLevel]]): Filter by abstraction levels.
        min_confidence (float): Minimum confidence level for retrieved concepts.
        context_boost (Dict[str, float]): Boost certain concept types or properties.
        recency_bias (float): Weight to give to recently accessed concepts (0-1).
        activation_threshold (float): Only return concepts with activation above this.
        include_inferred (bool): Whether to include inferred relationships.
        sort_by (str): How to sort results ('relevance', 'importance', 'confidence', etc.)
    """
    text: Optional[str] = None
    embedding: Optional[np.ndarray] = None
    similarity_metric: str = "cosine"
    importance_threshold: float = 0.0
    max_results: int = 20
    relationship_filters: Optional[List[Tuple[Union[RelationType, str], str]]] = None
    synonyms_match: bool = True
    abstraction_levels: Optional[List[AbstractionLevel]] = None
    min_confidence: float = 0.0
    context_boost: Dict[str, float] = field(default_factory=dict)
    recency_bias: float = 0.2
    activation_threshold: float = 0.0
    include_inferred: bool = True
    sort_by: str = "relevance"


class SemanticMemoryConfig:
    """
    Enhanced configuration for SemanticMemory.

    Attributes:
        embedding_function (Optional[Callable[[str], np.ndarray]]):
            A function that takes a string (name + synonyms) and returns an np.ndarray embedding.
        auto_link_graph (bool): If True, we call _sync_to_graph_subsystem whenever a concept changes.
        recency_decay (float): For advanced scoring if we track recency.
        max_concepts (int): If > 0, we limit the number of concepts and prune if exceeded.
        prune_policy (str): E.g. 'lowest_importance', 'oldest', or 'lowest_score'.
        activation_decay (float): Rate at which activation decays over time.
        activation_spread_factor (float): How much activation spreads to neighbors.
        relation_confidence_threshold (float): Minimum confidence for relations to be used in inference.
        inference_depth_limit (int): Maximum depth for transitive inference.
        enable_spreading_activation (bool): Whether to use spreading activation for search.
        max_activation_spreading_steps (int): Maximum steps for spreading activation.
        ontology_validation (bool): Validate ontological consistency when adding relations.
        default_concept_confidence (float): Default confidence for new concepts if not specified.
        context_window_size (int): Number of recent concepts to maintain in context window.
        similarity_threshold (float): Threshold for considering two embeddings similar.
        use_memory_consolidation (bool): Whether to periodically consolidate memory.
        consolidation_interval (int): How many operations before consolidation.
    """
    def __init__(self,
                 embedding_function: Optional[Callable[[str], np.ndarray]] = None,
                 auto_link_graph: bool = False,
                 recency_decay: float = 0.01,
                 max_concepts: int = 0,
                 prune_policy: str = "lowest_importance",
                 activation_decay: float = 0.05,
                 activation_spread_factor: float = 0.5,
                 relation_confidence_threshold: float = 0.5,
                 inference_depth_limit: int = 5,
                 enable_spreading_activation: bool = True,
                 max_activation_spreading_steps: int = 3,
                 ontology_validation: bool = True,
                 default_concept_confidence: float = 0.8,
                 context_window_size: int = 50,
                 similarity_threshold: float = 0.75,
                 use_memory_consolidation: bool = True,
                 consolidation_interval: int = 100):
        self.embedding_function = embedding_function
        self.auto_link_graph = auto_link_graph
        self.recency_decay = recency_decay
        self.max_concepts = max_concepts
        self.prune_policy = prune_policy
        self.activation_decay = activation_decay
        self.activation_spread_factor = activation_spread_factor
        self.relation_confidence_threshold = relation_confidence_threshold
        self.inference_depth_limit = inference_depth_limit
        self.enable_spreading_activation = enable_spreading_activation
        self.max_activation_spreading_steps = max_activation_spreading_steps
        self.ontology_validation = ontology_validation
        self.default_concept_confidence = default_concept_confidence
        self.context_window_size = context_window_size
        self.similarity_threshold = similarity_threshold
        self.use_memory_consolidation = use_memory_consolidation
        self.consolidation_interval = consolidation_interval


class SemanticMemory:
    """
    Enhanced semantic memory for GrandNexus.
    Stores concepts, each with synonyms, properties, abstraction levels, and importance.
    Also stores relations with weights, confidence, and specialized attributes.

    Main capabilities:
     - Add / get / update / remove concepts with abstraction levels
     - Add / get / update / remove relations with dynamic weights
     - Query concepts using text, embedding, or relationship filters
     - Advanced inference with transitive relationships
     - Spreading activation for context-aware retrieval
     - Ontological validation and consistency checking
    """

    def __init__(self, config: SemanticMemoryConfig, graph_engine: Optional[AbstractGraphEngine] = None):
        """Initialize the semantic memory with the given configuration."""
        self.logger = logging.getLogger("SemanticMemory")
        self.config = config
        self.lock = threading.RLock()

        # Graph engine integration - NEW
        self.graph_engine = graph_engine
        if self.graph_engine is None and config.auto_link_graph:
            self.logger.info("Creating default in-memory graph engine for semantic memory")
            self.graph_engine = create_graph_engine("inmemory")

        # Primary storage
        self.concepts: Dict[str, Concept] = {}
        self.relations: Dict[str, Relation] = {}

        # Enhanced indexing structures beyond the original implementation
        self.relations_by_source: Dict[str, List[Relation]] = defaultdict(list)
        self.relations_by_target: Dict[str, List[Relation]] = defaultdict(list)
        self.relations_by_type: Dict[str, List[Relation]] = defaultdict(list)

        # BioNeMo-inspired additions
        self.concepts_by_abstraction: Dict[AbstractionLevel, List[str]] = {
            level: [] for level in AbstractionLevel
        }
        self.context_window: deque[str] = deque(maxlen=config.context_window_size)
        self.concept_activations: Dict[str, float] = defaultdict(float)
        self.operation_counter: int = 0

        self.logger.info(f"SemanticMemory initialized (auto_link_graph={config.auto_link_graph}, "
                        f"ontology_validation={config.ontology_validation}, "
                        f"graph_engine={'enabled' if self.graph_engine else 'disabled'})")

    def _sync_concept_to_graph(self, concept: Concept) -> None:
        """Synchronize a concept to the graph engine if enabled."""
        if not self.graph_engine or not self.config.auto_link_graph:
            return

        try:
            # Add or update node in graph
            properties = {
                "type": "concept",
                "name": concept.name,
                "description": concept.description,
                "abstraction_level": concept.abstraction_level.value,
                "importance": concept.importance,
                "confidence": concept.confidence,
                "source": concept.source,
                **concept.properties
            }

            # Add synonyms as a property
            if concept.synonyms:
                properties["synonyms"] = list(concept.synonyms)

            # Check if node exists
            existing_nodes = self.graph_engine.query_nodes({"type": "concept", "name": concept.name})

            if existing_nodes:
                # Update existing node
                self.graph_engine.update_node(concept.concept_id, properties)
            else:
                # Add new node
                self.graph_engine.add_node(
                    node_id=concept.concept_id,
                    label="Concept",
                    properties=properties
                )

            # Store embedding if available
            if concept.embedding is not None and hasattr(self.graph_engine.get_node(concept.concept_id), 'embedding'):
                node = self.graph_engine.get_node(concept.concept_id)
                node.embedding = concept.embedding

        except Exception as e:
            self.logger.warning(f"Failed to sync concept {concept.concept_id} to graph: {e}")

    def _sync_relation_to_graph(self, relation: Relation) -> None:
        """Synchronize a relation to the graph engine if enabled."""
        if not self.graph_engine or not self.config.auto_link_graph:
            return

        try:
            # Add edge in graph
            properties = {
                "confidence": relation.confidence,
                "source": relation.source,
                "temporal": relation.temporal,
                **relation.properties
            }

            self.graph_engine.add_edge(
                source_id=relation.source_concept_id,
                target_id=relation.target_concept_id,
                edge_type=str(relation.relation_type),
                edge_id=relation.relation_id,
                properties=properties,
                weight=relation.weight,
                bidirectional=relation.bidirectional
            )

        except Exception as e:
            self.logger.warning(f"Failed to sync relation {relation.relation_id} to graph: {e}")

    def _navigate_knowledge_graph(self, start_concept_id: str, query: str, max_depth: int = 3) -> List[str]:
        """
        Navigate the knowledge graph to find related concepts based on a query.

        Args:
            start_concept_id: Starting concept ID
            query: Query string to guide navigation
            max_depth: Maximum depth to traverse

        Returns:
            List of relevant concept IDs found
        """
        if not self.graph_engine:
            return []

        try:
            # Use graph engine to execute query
            graph_query = GraphQuery(
                start_nodes=[start_concept_id],
                node_criteria={"type": "concept"},
                max_depth=max_depth,
                min_activation=0.1
            )

            results = self.graph_engine.execute_query(graph_query)

            # Extract concept IDs from results
            concept_ids = list(results["nodes"].keys())

            # If we have an embedding function, we can rank by similarity to query
            if self.config.embedding_function and query:
                query_embedding = self.config.embedding_function(query)

                # Rank concepts by embedding similarity
                ranked_concepts = []
                for concept_id in concept_ids:
                    if concept_id in self.concepts and self.concepts[concept_id].embedding is not None:
                        similarity = self._similarity(
                            query_embedding,
                            self.concepts[concept_id].embedding,
                            self.config.similarity_metric
                        )
                        ranked_concepts.append((concept_id, similarity))

                # Sort by similarity and return IDs
                ranked_concepts.sort(key=lambda x: x[1], reverse=True)
                return [cid for cid, _ in ranked_concepts]

            return concept_ids

        except Exception as e:
            self.logger.warning(f"Failed to navigate knowledge graph: {e}")
            return []

    # --------------------------------------------------------------------------
    # CONCEPT METHODS
    # --------------------------------------------------------------------------

    def add_concept(self,
                   concept: Union[Concept, Dict[str, Any]]) -> str:
        """
        Insert a new concept. If config.embedding_function is set, compute an embedding from
        concept.name + synonyms. Supports both Concept objects and dictionaries.

        Args:
            concept: Either a Concept object or a dictionary with concept attributes

        Returns:
            str: The concept_id of the created/updated concept
        """
        with self.lock:
            # Convert dictionary to Concept if needed
            if isinstance(concept, dict):
                concept_id = concept.get("concept_id") or str(uuid.uuid4())

                # Handle abstraction_level if provided as string
                abstraction_level = concept.get("abstraction_level", AbstractionLevel.SPECIFIC)
                if isinstance(abstraction_level, str):
                    try:
                        abstraction_level = AbstractionLevel[abstraction_level.upper()]
                    except KeyError:
                        abstraction_level = AbstractionLevel.SPECIFIC

                concept_obj = Concept(
                    concept_id=concept_id,
                    name=concept.get("name", ""),
                    description=concept.get("description", ""),
                    properties=concept.get("properties", {}),
                    embedding=concept.get("embedding"),
                    synonyms=set(concept.get("synonyms", [])),
                    importance=concept.get("importance", 0.5),
                    abstraction_level=abstraction_level,
                    confidence=concept.get("confidence", self.config.default_concept_confidence),
                    source=concept.get("source", "user")
                )
                concept = concept_obj
            else:
                # Ensure concept_id exists
                if not concept.concept_id:
                    concept.concept_id = str(uuid.uuid4())

            # Check if concept already exists
            if concept.concept_id in self.concepts:
                self.logger.warning(f"Concept {concept.concept_id} already exists. Use update_concept to modify.")
                return concept.concept_id

            # Generate embedding if needed
            if self.config.embedding_function and not concept.embedding:
                text_for_embedding = concept.name
                if concept.synonyms:
                    text_for_embedding += " " + " ".join(concept.synonyms)
                if concept.description:
                    text_for_embedding += " " + concept.description

                try:
                    concept.embedding = self.config.embedding_function(text_for_embedding)
                except Exception as e:
                    self.logger.error(f"Failed embedding for concept {concept.concept_id}: {e}", exc_info=True)

            # Set timestamps and initialize activation
            import time
            current_time = time.time()
            concept.last_accessed = current_time
            concept.activation = 1.0  # New concepts start highly activated

            # Store the concept
            self.concepts[concept.concept_id] = concept

            # Update indices
            self.concepts_by_abstraction[concept.abstraction_level].append(concept.concept_id)
            self.context_window.append(concept.concept_id)
            self.concept_activations[concept.concept_id] = 1.0

            # Track operation count for potential consolidation
            self.operation_counter += 1
            if self.config.use_memory_consolidation and self.operation_counter >= self.config.consolidation_interval:
                self._consolidate_memory()
                self.operation_counter = 0

            # Sync to graph if enabled
            if self.config.auto_link_graph:
                self._sync_to_graph_subsystem(concept)

            self.logger.debug(f"Added concept: {concept}")
            # Sync to graph engine - NEW
            self._sync_concept_to_graph(concept)


        return concept.concept_id

    def get_concept(self, concept_id: str, update_activation: bool = True) -> Optional[Concept]:
        """
        Retrieve a concept by ID, optionally updating its activation and access metadata.

        Args:
            concept_id: ID of the concept to retrieve
            update_activation: Whether to update activation and access timestamp/count

        Returns:
            The Concept object or None if not found
        """
        with self.lock:
            concept = self.concepts.get(concept_id)
            if not concept:
                return None

            if update_activation:
                import time
                current_time = time.time()
                concept.last_accessed = current_time
                concept.access_count += 1
                concept.activation = 1.0  # Full activation when directly accessed
                self.concept_activations[concept_id] = 1.0

                # Update context window
                if concept_id in self.context_window:
                    # Move to most recent position if already in window
                    self.context_window.remove(concept_id)
                self.context_window.append(concept_id)

            return concept

    def update_concept(self,
                      concept: Union[Concept, Dict[str, Any]],
                      partial_update: bool = False) -> bool:
        """
        Update an existing concept. If synonyms changed, we might recalc embedding.

        Args:
            concept: The concept object or dictionary with updated values
            partial_update: If True, only update the provided fields

        Returns:
            bool: True if the update was successful
        """
        with self.lock:
            # Convert dictionary to concept if needed
            concept_id = None
            if isinstance(concept, dict):
                concept_id = concept.get("concept_id")
                if not concept_id:
                    self.logger.error("Cannot update concept: concept_id not provided in dictionary")
                    return False
            else:
                concept_id = concept.concept_id

            # Check if concept exists
            if concept_id not in self.concepts:
                self.logger.warning(f"Cannot update concept {concept_id}, it doesn't exist. Use add_concept.")
                return False

            existing_concept = self.concepts[concept_id]

            # Update the concept
            if isinstance(concept, dict):
                # Partial update with dictionary
                if "name" in concept:
                    existing_concept.name = concept["name"]
                if "description" in concept:
                    existing_concept.description = concept["description"]
                if "properties" in concept:
                    if partial_update:
                        existing_concept.properties.update(concept["properties"])
                    else:
                        existing_concept.properties = concept["properties"]
                if "synonyms" in concept:
                    existing_concept.synonyms = set(concept["synonyms"])
                if "importance" in concept:
                    existing_concept.importance = concept["importance"]
                if "abstraction_level" in concept:
                    abstraction_level = concept["abstraction_level"]
                    if isinstance(abstraction_level, str):
                        try:
                            abstraction_level = AbstractionLevel[abstraction_level.upper()]
                        except KeyError:
                            abstraction_level = existing_concept.abstraction_level

                    # Update abstraction index if changed
                    if abstraction_level != existing_concept.abstraction_level:
                        self.concepts_by_abstraction[existing_concept.abstraction_level].remove(concept_id)
                        existing_concept.abstraction_level = abstraction_level
                        self.concepts_by_abstraction[abstraction_level].append(concept_id)
                if "confidence" in concept:
                    existing_concept.confidence = concept["confidence"]
                if "source" in concept:
                    existing_concept.source = concept["source"]
                if "embedding" in concept:
                    existing_concept.embedding = concept["embedding"]
            else:
                # Replace with new concept but keep activation data
                old_activation = existing_concept.activation
                old_access_count = existing_concept.access_count
                old_last_accessed = existing_concept.last_accessed

                # Update abstraction index if changed
                if concept.abstraction_level != existing_concept.abstraction_level:
                    self.concepts_by_abstraction[existing_concept.abstraction_level].remove(concept_id)
                    self.concepts_by_abstraction[concept.abstraction_level].append(concept_id)

                # Replace the concept
                self.concepts[concept_id] = concept

                # Preserve activation data
                concept.activation = old_activation
                concept.access_count = old_access_count
                concept.last_accessed = old_last_accessed

            # Recalculate embedding if necessary
            need_embedding_update = False
            if isinstance(concept, dict):
                need_embedding_update = "name" in concept or "synonyms" in concept or "description" in concept
            else:
                need_embedding_update = True

            if need_embedding_update and self.config.embedding_function:
                try:
                    updated_concept = self.concepts[concept_id]
                    text_for_embedding = updated_concept.name
                    if updated_concept.synonyms:
                        text_for_embedding += " " + " ".join(updated_concept.synonyms)
                    if updated_concept.description:
                        text_for_embedding += " " + updated_concept.description

                    updated_concept.embedding = self.config.embedding_function(text_for_embedding)
                except Exception as e:
                    self.logger.error(f"Failed to update embedding for concept {concept_id}: {e}", exc_info=True)

            # Update context window
            if concept_id in self.context_window:
                self.context_window.remove(concept_id)
            self.context_window.append(concept_id)

            # Track operation
            self.operation_counter += 1

            # Sync to graph if enabled
            if self.config.auto_link_graph:
                self._sync_to_graph_subsystem(self.concepts[concept_id])

            self.logger.debug(f"Updated concept: {self.concepts[concept_id]}")
            return True

    def remove_concept(self, concept_id: str) -> bool:
        """
        Remove a concept and all its relations from memory.

        Args:
            concept_id: ID of the concept to remove

        Returns:
            bool: True if removed, False if not found
        """
        with self.lock:
            concept = self.concepts.pop(concept_id, None)
            if not concept:
                self.logger.warning(f"Cannot remove concept {concept_id}: doesn't exist.")
                return False

            # Remove from abstraction index
            if concept_id in self.concepts_by_abstraction[concept.abstraction_level]:
                self.concepts_by_abstraction[concept.abstraction_level].remove(concept_id)

            # Remove from context window and activations
            if concept_id in self.context_window:
                self.context_window.remove(concept_id)
            if concept_id in self.concept_activations:
                del self.concept_activations[concept_id]

            # Remove all relations involving this concept
            relations_to_remove = []
            for rel_id, relation in self.relations.items():
                if relation.source_concept_id == concept_id or relation.target_concept_id == concept_id:
                    relations_to_remove.append(rel_id)

            for rel_id in relations_to_remove:
                self.remove_relation(rel_id)

            self.logger.debug(f"Removed concept {concept_id} and {len(relations_to_remove)} related relations")
            return True

    # --------------------------------------------------------------------------
    # RELATION METHODS
    # --------------------------------------------------------------------------

    def add_relation(self,
                    relation: Union[Relation, Dict[str, Any]]) -> str:
        """
        Create a new relationship between two existing concepts.

        Args:
            relation: A Relation object or dictionary with relation data

        Returns:
            str: The relation_id of the created relation
        """
        with self.lock:
            # Convert dictionary to Relation if needed
            if isinstance(relation, dict):
                relation_id = relation.get("relation_id") or str(uuid.uuid4())
                source_id = relation.get("source_concept_id")
                target_id = relation.get("target_concept_id")
                relation_type = relation.get("relation_type")

                if not source_id or not target_id or not relation_type:
                    self.logger.error("Cannot add relation: missing required fields (source_id, target_id, relation_type)")
                    return ""

                relation_obj = Relation(
                    relation_id=relation_id,
                    source_concept_id=source_id,
                    relation_type=relation_type,
                    target_concept_id=target_id,
                    weight=relation.get("weight", 1.0),
                    properties=relation.get("properties", {}),
                    bidirectional=relation.get("bidirectional", False),
                    transitive=relation.get("transitive", False),
                    temporal=relation.get("temporal", False),
                    confidence=relation.get("confidence", self.config.default_concept_confidence),
                    source=relation.get("source", "user")
                )
                relation = relation_obj
            else:
                # Ensure relation_id exists
                if not relation.relation_id:
                    relation.relation_id = str(uuid.uuid4())

            # Check if relation already exists
            if relation.relation_id in self.relations:
                self.logger.warning(f"Relation {relation.relation_id} already exists. Use update_relation to modify.")
                return relation.relation_id

            # Verify concepts exist
            if relation.source_concept_id not in self.concepts:
                self.logger.warning(f"Source concept {relation.source_concept_id} doesn't exist.")
                return ""
            if relation.target_concept_id not in self.concepts:
                self.logger.warning(f"Target concept {relation.target_concept_id} doesn't exist.")
                return ""

            # Set created timestamp
            import time
            relation.created_at = time.time()

            # Validate ontology if enabled
            if self.config.ontology_validation:
                validation_result = self._validate_relation_ontology(relation)
                if not validation_result["valid"]:
                    self.logger.warning(f"Ontology validation failed for relation: {validation_result['reason']}")
                    # We might still add it but with reduced confidence
                    relation.confidence *= 0.5

            # Store the relation
            self.relations[relation.relation_id] = relation

            # Update indices
            self.relations_by_source[relation.source_concept_id].append(relation)
            self.relations_by_target[relation.target_concept_id].append(relation)
            self.relations_by_type[str(relation.relation_type)].append(relation)

            # Handle bidirectional relations by creating the inverse
            if relation.bidirectional and relation.source_concept_id != relation.target_concept_id:
                inverse_id = f"inverse_{relation.relation_id}"
                inverse_relation = Relation(
                    relation_id=inverse_id,
                    source_concept_id=relation.target_concept_id,
                    relation_type=relation.relation_type,
                    target_concept_id=relation.source_concept_id,
                    weight=relation.weight,
                    properties=dict(relation.properties),
                    bidirectional=True,
                    transitive=relation.transitive,
                    temporal=relation.temporal,
                    created_at=relation.created_at,
                    confidence=relation.confidence,
                    source=f"inverse_of_{relation.source}"
                )

                self.relations[inverse_id] = inverse_relation
                self.relations_by_source[inverse_relation.source_concept_id].append(inverse_relation)
                self.relations_by_target[inverse_relation.target_concept_id].append(inverse_relation)
                self.relations_by_type[str(inverse_relation.relation_type)].append(inverse_relation)

            # Update context for both concepts
            self.context_window.append(relation.source_concept_id)
            self.context_window.append(relation.target_concept_id)

            # Spread activation between related concepts
            source_concept = self.concepts[relation.source_concept_id]
            target_concept = self.concepts[relation.target_concept_id]

            # Source activates target
            activation_spread = source_concept.activation * relation.weight * self.config.activation_spread_factor
            target_concept.activation = min(1.0, target_concept.activation + activation_spread)
            self.concept_activations[relation.target_concept_id] = target_concept.activation

            # Target activates source (weaker)
            reverse_spread = target_concept.activation * relation.weight * self.config.activation_spread_factor * 0.5
            source_concept.activation = min(1.0, source_concept.activation + reverse_spread)
            self.concept_activations[relation.source_concept_id] = source_concept.activation

            # Track operation
            self.operation_counter += 1

            self.logger.debug(f"Added relation: {relation}")

            self._sync_relation_to_graph(relation)
            return relation.relation_id

    def get_relation(self, relation_id: str) -> Optional[Relation]:
        """
        Retrieve a relation by its ID.

        Args:
            relation_id: ID of the relation to retrieve

        Returns:
            The Relation object or None if not found
        """
        with self.lock:
            return self.relations.get(relation_id)

    def update_relation(self,
                       relation: Union[Relation, Dict[str, Any]],
                       partial_update: bool = False) -> bool:
        """
        Update an existing relation.

        Args:
            relation: The relation object or dictionary with updated values
            partial_update: If True, only update the provided fields

        Returns:
            bool: True if the update was successful
        """
        with self.lock:
            # Convert dictionary to relation if needed
            relation_id = None
            if isinstance(relation, dict):
                relation_id = relation.get("relation_id")
                if not relation_id:
                    self.logger.error("Cannot update relation: relation_id not provided in dictionary")
                    return False
            else:
                relation_id = relation.relation_id

            # Check if relation exists
            if relation_id not in self.relations:
                self.logger.warning(f"Cannot update relation {relation_id}, it doesn't exist. Use add_relation.")
                return False

            existing_relation = self.relations[relation_id]

            # Remember old indices for removal
            old_source_id = existing_relation.source_concept_id
            old_target_id = existing_relation.target_concept_id
            old_type = str(existing_relation.relation_type)

            # Update the relation
            if isinstance(relation, dict):
                # Partial update with dictionary
                changed_source = False
                changed_target = False
                changed_type = False

                if "source_concept_id" in relation and relation["source_concept_id"] != existing_relation.source_concept_id:
                    if relation["source_concept_id"] in self.concepts:
                        existing_relation.source_concept_id = relation["source_concept_id"]
                        changed_source = True
                    else:
                        self.logger.warning(f"Source concept {relation['source_concept_id']} doesn't exist.")
                        return False

                if "target_concept_id" in relation and relation["target_concept_id"] != existing_relation.target_concept_id:
                    if relation["target_concept_id"] in self.concepts:
                        existing_relation.target_concept_id = relation["target_concept_id"]
                        changed_target = True
                    else:
                        self.logger.warning(f"Target concept {relation['target_concept_id']} doesn't exist.")
                        return False

                if "relation_type" in relation and relation["relation_type"] != existing_relation.relation_type:
                    existing_relation.relation_type = relation["relation_type"]
                    changed_type = True

                if "weight" in relation:
                    existing_relation.weight = relation["weight"]
                if "properties" in relation:
                    if partial_update:
                        existing_relation.properties.update(relation["properties"])
                    else:
                        existing_relation.properties = relation["properties"]
                if "bidirectional" in relation:
                    existing_relation.bidirectional = relation["bidirectional"]
                if "transitive" in relation:
                    existing_relation.transitive = relation["transitive"]
                if "temporal" in relation:
                    existing_relation.temporal = relation["temporal"]
                if "confidence" in relation:
                    existing_relation.confidence = relation["confidence"]
                if "source" in relation:
                    existing_relation.source = relation["source"]

                # Update indices if necessary
                if changed_source or changed_target or changed_type:
                    # Remove from old indices
                    if old_source_id in self.relations_by_source and existing_relation in self.relations_by_source[old_source_id]:
                        self.relations_by_source[old_source_id].remove(existing_relation)
                    if old_target_id in self.relations_by_target and existing_relation in self.relations_by_target[old_target_id]:
                        self.relations_by_target[old_target_id].remove(existing_relation)
                    if old_type in self.relations_by_type and existing_relation in self.relations_by_type[old_type]:
                        self.relations_by_type[old_type].remove(existing_relation)

                    # Add to new indices
                    self.relations_by_source[existing_relation.source_concept_id].append(existing_relation)
                    self.relations_by_target[existing_relation.target_concept_id].append(existing_relation)
                    self.relations_by_type[str(existing_relation.relation_type)].append(existing_relation)
            else:
                # Full replacement
                old_relation = self.relations[relation_id]

                # Remove from old indices
                if old_source_id in self.relations_by_source and old_relation in self.relations_by_source[old_source_id]:
                    self.relations_by_source[old_source_id].remove(old_relation)
                if old_target_id in self.relations_by_target and old_relation in self.relations_by_target[old_target_id]:
                    self.relations_by_target[old_target_id].remove(old_relation)
                if old_type in self.relations_by_type and old_relation in self.relations_by_type[old_type]:
                    self.relations_by_type[old_type].remove(old_relation)

                # Verify concepts exist
                if relation.source_concept_id not in self.concepts:
                    self.logger.warning(f"Source concept {relation.source_concept_id} doesn't exist.")
                    return False
                if relation.target_concept_id not in self.concepts:
                    self.logger.warning(f"Target concept {relation.target_concept_id} doesn't exist.")
                    return False

                # Replace relation and update indices
                self.relations[relation_id] = relation
                self.relations_by_source[relation.source_concept_id].append(relation)
                self.relations_by_target[relation.target_concept_id].append(relation)
                self.relations_by_type[str(relation.relation_type)].append(relation)

            # Validate ontology if enabled
            if self.config.ontology_validation:
                validation_result = self._validate_relation_ontology(self.relations[relation_id])
                if not validation_result["valid"]:
                    self.logger.warning(f"Ontology validation failed for updated relation: {validation_result['reason']}")
                    # Reduce confidence
                    self.relations[relation_id].confidence *= 0.5

            # Update context window
            self.context_window.append(self.relations[relation_id].source_concept_id)
            self.context_window.append(self.relations[relation_id].target_concept_id)

            # Track operation
            self.operation_counter += 1

            self.logger.debug(f"Updated relation: {self.relations[relation_id]}")
            return True

    def remove_relation(self, relation_id: str) -> bool:
        """
        Remove a relation from the semantic memory.

        Args:
            relation_id: ID of the relation to remove

        Returns:
            bool: True if removed, False if not found
        """
        with self.lock:
            relation = self.relations.pop(relation_id, None)
            if not relation:
                self.logger.warning(f"Cannot remove relation {relation_id}: doesn't exist.")
                return False

            # Remove from indices
            if relation in self.relations_by_source[relation.source_concept_id]:
                self.relations_by_source[relation.source_concept_id].remove(relation)
            if relation in self.relations_by_target[relation.target_concept_id]:
                self.relations_by_target[relation.target_concept_id].remove(relation)

            rel_type = str(relation.relation_type)
            if relation in self.relations_by_type[rel_type]:
                self.relations_by_type[rel_type].remove(relation)

            # If it was bidirectional, try to remove the inverse relation too
            if relation.bidirectional:
                inverse_id = f"inverse_{relation_id}"
                if inverse_id in self.relations:
                    self.remove_relation(inverse_id)

            # For completeness, also check if this relation is an inverse of another
            if relation_id.startswith("inverse_"):
                original_id = relation_id[8:]  # Remove "inverse_" prefix
                if original_id in self.relations:
                    original = self.relations[original_id]
                    if original.bidirectional:
                        original.bidirectional = False  # No longer bidirectional

            self.logger.debug(f"Removed relation {relation_id}")
            return True

    def get_relations_by_source(self, concept_id: str) -> List[Relation]:
        """
        Get all relations where the specified concept is the source.

        Args:
            concept_id: ID of the source concept

        Returns:
            List of relations
        """
        with self.lock:
            return list(self.relations_by_source.get(concept_id, []))

    def get_relations_by_target(self, concept_id: str) -> List[Relation]:
        """
        Get all relations where the specified concept is the target.

        Args:
            concept_id: ID of the target concept

        Returns:
            List of relations
        """
        with self.lock:
            return list(self.relations_by_target.get(concept_id, []))

    def get_relations_by_type(self, relation_type: Union[RelationType, str]) -> List[Relation]:
        """
        Get all relations of the specified type.

        Args:
            relation_type: Type of relations to retrieve

        Returns:
            List of relations
        """
        with self.lock:
            return list(self.relations_by_type.get(str(relation_type), []))

    def query_relations(self,
                       source_concept_id: Optional[str] = None,
                       relation_type: Optional[Union[RelationType, str]] = None,
                       target_concept_id: Optional[str] = None,
                       min_weight: float = 0.0,
                       min_confidence: float = 0.0) -> List[Relation]:
        """
        Flexible query for relations based on source, type, target, and other criteria.

        Args:
            source_concept_id: Optional source concept ID filter
            relation_type: Optional relation type filter
            target_concept_id: Optional target concept ID filter
            min_weight: Minimum relation weight
            min_confidence: Minimum confidence level

        Returns:
            List of matching relations
        """
        with self.lock:
            results = []

            # Determine the most efficient way to search
            if source_concept_id and source_concept_id in self.relations_by_source:
                # Start with source-based index
                candidates = self.relations_by_source[source_concept_id]
            elif target_concept_id and target_concept_id in self.relations_by_target:
                # Start with target-based index
                candidates = self.relations_by_target[target_concept_id]
            elif relation_type and str(relation_type) in self.relations_by_type:
                # Start with type-based index
                candidates = self.relations_by_type[str(relation_type)]
            else:
                # No specific index, search all relations
                candidates = list(self.relations.values())

            # Apply all filters
            for relation in candidates:
                if (source_concept_id is None or relation.source_concept_id == source_concept_id) and \
                   (target_concept_id is None or relation.target_concept_id == target_concept_id) and \
                   (relation_type is None or str(relation.relation_type) == str(relation_type)) and \
                   relation.weight >= min_weight and \
                   relation.confidence >= min_confidence:
                    results.append(relation)

            return results

    # --------------------------------------------------------------------------
    # QUERY CONCEPTS
    # --------------------------------------------------------------------------

    def query_concepts(self, query: SemanticQuery) -> List[Concept]:
        """
        Enhanced concept retrieval using advanced criteria including activation,
        abstraction levels, and spreading activation.

        Args:
            query: SemanticQuery object with search parameters

        Returns:
            List of matching concepts
        """
        with self.lock:
            # Optional spreading activation
            if self.config.enable_spreading_activation and query.text:
                self._apply_spreading_activation(query.text)

            # Build candidate set
            candidates = set()

            # Apply text-based filtering if specified
            if query.text is not None:
                # Find concepts matching the text query
                for concept_id, concept in self.concepts.items():
                    if query.text.lower() in concept.name.lower():
                        candidates.add(concept_id)
                    elif query.synonyms_match and concept.synonyms:
                        if any(query.text.lower() in syn.lower() for syn in concept.synonyms):
                            candidates.add(concept_id)

                # If no direct matches found and embedding_function is available, try embedding similarity
                if not candidates and self.config.embedding_function and query.embedding is None:
                    try:
                        query.embedding = self.config.embedding_function(query.text)
                    except Exception as e:
                        self.logger.error(f"Error generating embedding for query text: {e}", exc_info=True)

            # Apply relationship filters if specified
            if query.relationship_filters:
                rel_candidates = set()
                first_filter = True

                for rel_type, related_concept_id in query.relationship_filters:
                    filter_matches = set()

                    # Find concepts with the specified relationship
                    if related_concept_id in self.concepts:
                        # Find concepts that have this relationship
                        relations = self.query_relations(
                            relation_type=rel_type,
                            target_concept_id=related_concept_id,
                            min_confidence=query.min_confidence
                        )

                        # Add sources to candidates
                        for rel in relations:
                            filter_matches.add(rel.source_concept_id)

                        # Include inferred relationships if requested
                        if query.include_inferred and isinstance(rel_type, RelationType) and rel_type == RelationType.IS_A:
                            # Get transitive is_a relationships
                            transitive_matches = self._find_transitive_sources(related_concept_id, rel_type)
                            filter_matches.update(transitive_matches)

                    # Intersect with previous filters' results
                    if first_filter:
                        rel_candidates = filter_matches
                        first_filter = False
                    else:
                        rel_candidates &= filter_matches

                # If we have text matches, intersect with relationship matches
                if query.text is not None:
                    if candidates:
                        candidates &= rel_candidates
                    else:
                        candidates = rel_candidates
                else:
                    candidates = rel_candidates

            # If neither text nor relationship filters specified, use all concepts
            if query.text is None and not query.relationship_filters:
                candidates = set(self.concepts.keys())

            # Apply abstraction level filtering if specified
            if query.abstraction_levels:
                valid_levels = set(query.abstraction_levels)
                candidates = {cid for cid in candidates
                             if self.concepts[cid].abstraction_level in valid_levels}

            # Apply activation threshold if specified
            if query.activation_threshold > 0:
                candidates = {cid for cid in candidates
                             if self.concepts[cid].activation >= query.activation_threshold}

            # Apply importance threshold filtering
            if query.importance_threshold > 0:
                candidates = {cid for cid in candidates
                             if self.concepts[cid].importance >= query.importance_threshold}

            # Apply confidence threshold filtering
            if query.min_confidence > 0:
                candidates = {cid for cid in candidates
                             if self.concepts[cid].confidence >= query.min_confidence}

            # Compute scores and rank results
            scored_results = []
            for concept_id in candidates:
                concept = self.concepts[concept_id]
                # Calculate combined score
                score = self._compute_concept_score(concept, query)
                scored_results.append((score, concept))

            # Sort by score (descending)
            scored_results.sort(reverse=True, key=lambda x: x[0])

            # Get top results
            top_concepts = [concept for _, concept in scored_results[:query.max_results]]

            # Update context window and activations for retrieved concepts
            for concept in top_concepts:
                if concept.concept_id in self.context_window:
                    self.context_window.remove(concept.concept_id)
                self.context_window.append(concept.concept_id)

                # Increase activation for retrieved concepts
                concept.activation = min(1.0, concept.activation + 0.2)
                self.concept_activations[concept.concept_id] = concept.activation

                # Update access metadata
                import time
                concept.last_accessed = time.time()
                concept.access_count += 1

            return top_concepts

    def _compute_concept_score(self, concept: Concept, query: SemanticQuery) -> float:
        """
        Calculate a combined score for a concept based on the query parameters.

        Args:
            concept: Concept to score
            query: Query parameters

        Returns:
            Float score (higher is better)
        """
        score = 0.0

        # Importance component
        score += concept.importance * 0.2

        # Activation component
        score += concept.activation * 0.1

        # Recency component (if recency_bias > 0)
        if query.recency_bias > 0 and concept.last_accessed > 0:
            import time
            time_factor = math.exp(-self.config.recency_decay * (time.time() - concept.last_accessed))
            score += time_factor * query.recency_bias

        # Text match component
        if query.text is not None:
            if query.text.lower() in concept.name.lower():
                score += 0.5
            elif query.synonyms_match and any(query.text.lower() in syn.lower() for syn in concept.synonyms):
                score += 0.3

        # Embedding similarity component
        if query.embedding is not None and concept.embedding is not None:
            similarity = self._similarity(query.embedding, concept.embedding, query.similarity_metric)
            score += similarity * 0.3

        # Context boost component
        for boost_key, boost_value in query.context_boost.items():
            if boost_key in concept.properties:
                score += boost_value
            elif boost_key == concept.abstraction_level.name:
                score += boost_value

        # Apply confidence factor
        score *= concept.confidence

        return score

    def _similarity(self, vec1: np.ndarray, vec2: np.ndarray, metric: str) -> float:
        """
        Compute vector similarity using the specified metric.

        Args:
            vec1: First vector
            vec2: Second vector
            metric: Similarity metric name

        Returns:
            Similarity score (higher means more similar)
        """
        if metric == "cosine":
            dot_product = np.dot(vec1, vec2)
            norm1 = np.linalg.norm(vec1)
            norm2 = np.linalg.norm(vec2)
            if norm1 > 0 and norm2 > 0:
                return dot_product / (norm1 * norm2)
            return 0.0
        elif metric == "euclidean":
            # Convert distance to similarity (lower distance = higher similarity)
            dist = np.linalg.norm(vec1 - vec2)
            return 1.0 / (1.0 + dist)
        else:
            self.logger.warning(f"Unknown similarity metric: {metric}, defaulting to cosine")
            return self._similarity(vec1, vec2, "cosine")

    def _apply_spreading_activation(self, seed_text: str) -> None:
        """
        Apply spreading activation from concepts matching the seed text.

        Args:
            seed_text: Text to search for initial concepts to activate
        """
        # Find initial concepts to activate
        activated_concepts = set()

        # Find concepts matching the seed text
        for concept_id, concept in self.concepts.items():
            if seed_text.lower() in concept.name.lower():
                activated_concepts.add(concept_id)
                concept.activation = 1.0  # Full activation
                self.concept_activations[concept_id] = 1.0
            elif any(seed_text.lower() in syn.lower() for syn in concept.synonyms):
                activated_concepts.add(concept_id)
                concept.activation = 0.8  # High but not full activation
                self.concept_activations[concept_id] = 0.8

        # Process spreading in steps
        for step in range(self.config.max_activation_spreading_steps):
            newly_activated = set()

            # Process each activated concept
            for concept_id in activated_concepts:
                # Spread to related concepts (outgoing relations)
                for relation in self.relations_by_source.get(concept_id, []):
                    if relation.target_concept_id not in activated_concepts:
                        target_concept = self.concepts[relation.target_concept_id]
                        # Calculate spread amount
                        spread = self.concepts[concept_id].activation * relation.weight * self.config.activation_spread_factor
                        # Apply spread with decay based on step
                        decay_factor = 1.0 / (step + 1)
                        spread *= decay_factor
                        # Update activation
                        target_concept.activation = min(1.0, target_concept.activation + spread)
                        self.concept_activations[relation.target_concept_id] = target_concept.activation
                        # Mark as newly activated if significant
                        if spread > 0.05:
                            newly_activated.add(relation.target_concept_id)

                # Also consider incoming relations (weaker)
                for relation in self.relations_by_target.get(concept_id, []):
                    if relation.source_concept_id not in activated_concepts:
                        source_concept = self.concepts[relation.source_concept_id]
                        # Calculate reverse spread (weaker)
                        spread = self.concepts[concept_id].activation * relation.weight * self.config.activation_spread_factor * 0.5
                        # Apply spread with decay based on step
                        decay_factor = 1.0 / (step + 1)
                        spread *= decay_factor
                        # Update activation
                        source_concept.activation = min(1.0, source_concept.activation + spread)
                        self.concept_activations[relation.source_concept_id] = source_concept.activation
                        # Mark as newly activated if significant
                        if spread > 0.05:
                            newly_activated.add(relation.source_concept_id)

            # Add newly activated concepts
            activated_concepts.update(newly_activated)

            # If no new concepts were activated, stop spreading
            if not newly_activated:
                break

    # --------------------------------------------------------------------------
    # INFERENCE METHODS
    # --------------------------------------------------------------------------

    def infer_relation(self,
                      source_concept_id: str,
                      relation_type: Union[RelationType, str],
                      target_concept_id: str,
                      min_confidence: float = 0.5) -> bool:
        """
        Check if a relation between source and target exists directly or can be inferred.

        Args:
            source_concept_id: Source concept ID
            relation_type: Relation type to check
            target_concept_id: Target concept ID
            min_confidence: Minimum confidence threshold

        Returns:
            bool: True if the relation exists or can be inferred
        """
        with self.lock:
            # Check if concepts exist
            if source_concept_id not in self.concepts or target_concept_id not in self.concepts:
                return False

            # First, check for direct relations
            direct_relations = self.query_relations(
                source_concept_id=source_concept_id,
                relation_type=relation_type,
                target_concept_id=target_concept_id,
                min_confidence=min_confidence
            )

            if direct_relations:
                return True

            # For transitive relations, try inference
            if isinstance(relation_type, RelationType) and \
               (relation_type == RelationType.IS_A or relation_type == RelationType.PART_OF) or \
               (isinstance(relation_type, str) and relation_type in ("is_a", "part_of")):
                return self._check_transitive_relation(
                    source_concept_id,
                    relation_type,
                    target_concept_id,
                    min_confidence,
                    depth=0,
                    visited=set()
                )

            return False

    def get_related_concepts(self,
                            concept_id: str,
                            relation_type: Optional[Union[RelationType, str]] = None,
                            max_depth: int = 1,
                            min_confidence: float = 0.0) -> Dict[str, float]:
        """
        Get concepts related to the given concept with relationship strength scores.

        Args:
            concept_id: ID of the concept to find relations for
            relation_type: Optional specific relation type to filter by
            max_depth: Maximum traversal depth for relations (1 = direct relations only)
            min_confidence: Minimum confidence threshold for relations

        Returns:
            Dictionary mapping related concept IDs to relevance scores
        """
        with self.lock:
            if concept_id not in self.concepts:
                return {}

            result = {}  # concept_id -> relevance score
            visited = set([concept_id])  # Avoid cycles

            def traverse_relations(current_id: str, depth: int, path_weight: float):
                if depth > max_depth:
                    return

                # Get outgoing relations
                relations = []
                if relation_type is not None:
                    relations = self.query_relations(
                        source_concept_id=current_id,
                        relation_type=relation_type,
                        min_confidence=min_confidence
                    )
                else:
                    relations = self.get_relations_by_source(current_id)
                    relations = [r for r in relations if r.confidence >= min_confidence]

                # Process each relation
                for rel in relations:
                    if rel.target_concept_id not in visited:
                        # Calculate new weight (diminishes with depth)
                        new_weight = path_weight * rel.weight * (1.0 / (depth + 1))

                        # Add or update the target in results
                        result[rel.target_concept_id] = max(
                            result.get(rel.target_concept_id, 0.0),
                            new_weight
                        )

                        # Mark as visited
                        visited.add(rel.target_concept_id)

                        # Recurse deeper
                        traverse_relations(rel.target_concept_id, depth + 1, new_weight)

            # Start traversal
            traverse_relations(concept_id, 1, 1.0)

            return result

    def _check_transitive_relation(self,
                                 source_id: str,
                                 relation_type: Union[RelationType, str],
                                 target_id: str,
                                 min_confidence: float,
                                 depth: int,
                                 visited: Set[str]) -> bool:
        """
        Recursive helper to check for transitive relations.

        Args:
            source_id: Current source concept
            relation_type: Relation type to check
            target_id: Ultimate target concept
            min_confidence: Minimum confidence threshold
            depth: Current recursion depth
            visited: Set of already visited concepts to prevent cycles

        Returns:
            bool: True if a valid path exists
        """
        # Prevent excessive recursion or cycles
        if depth >= self.config.inference_depth_limit or source_id in visited:
            return False

        # Mark as visited
        visited.add(source_id)

        # Get direct relations of the specified type
        relations = self.query_relations(
            source_concept_id=source_id,
            relation_type=relation_type,
            min_confidence=min_confidence
        )

        # Check each relation
        for relation in relations:
            # Check if this relation leads to target
            if relation.target_concept_id == target_id:
                return True

            # Recursively check from this target
            if self._check_transitive_relation(
                relation.target_concept_id,
                relation_type,
                target_id,
                min_confidence,
                depth + 1,
                visited
            ):
                return True

        return False

    def _find_transitive_sources(self,
                               concept_id: str,
                               relation_type: Union[RelationType, str]) -> Set[str]:
        """
        Find all concepts that have a transitive relationship to the given concept.
        This is useful for IS_A relationships (find all subtypes).

        Args:
            concept_id: Target concept ID
            relation_type: Relation type to traverse

        Returns:
            Set of concept IDs with transitive relationships
        """
        results = set()
        rel_type_str = str(relation_type)

        # Get all direct relations of this type targeting the concept
        direct_relations = self.query_relations(
            relation_type=relation_type,
            target_concept_id=concept_id
        )

        # Add direct sources
        for rel in direct_relations:
            results.add(rel.source_concept_id)

        # For each direct source, recursively find its sources
        for rel in direct_relations:
            # Only recurse for transitive relation types
            if rel.transitive:
                indirect_sources = self._find_transitive_sources(rel.source_concept_id, relation_type)
                results.update(indirect_sources)

        return results

    def _validate_relation_ontology(self, relation: Relation) -> Dict[str, Any]:
        """
        Validate if a relation is consistent with ontological constraints.

        Args:
            relation: The relation to validate

        Returns:
            Dict with 'valid' (bool) and optional 'reason' for failure
        """
        # Default valid result
        result = {"valid": True}

        # Skip validation if not enabled
        if not self.config.ontology_validation:
            return result

        source_concept = self.concepts.get(relation.source_concept_id)
        target_concept = self.concepts.get(relation.target_concept_id)

        if not source_concept or not target_concept:
            result["valid"] = False
            result["reason"] = "Source or target concept not found"
            return result

        # Check abstraction level consistency for IS_A relationships
        if isinstance(relation.relation_type, RelationType) and relation.relation_type == RelationType.IS_A:
            # For IS_A, source should generally be at the same or lower abstraction level than target
            if source_concept.abstraction_level.value > target_concept.abstraction_level.value:
                result["valid"] = False
                result["reason"] = f"Invalid IS_A abstraction levels: {source_concept.abstraction_level.name} -> {target_concept.abstraction_level.name}"
                return result

        # Check for circularity in transitive relationships
        if relation.transitive and relation.source_concept_id != relation.target_concept_id:
            # Check if the target already has a path to the source (which would create a cycle)
            if self._check_transitive_relation(
                relation.target_concept_id,
                relation.relation_type,
                relation.source_concept_id,
                0.0,  # No confidence threshold for this check
                0,
                set()
            ):
                result["valid"] = False
                result["reason"] = "Relation would create a circular dependency"
                return result

        return result

    # --------------------------------------------------------------------------
    # MEMORY MANAGEMENT METHODS
    # --------------------------------------------------------------------------

    def _decay_activations(self) -> None:
        """
        Apply decay to all concept activations based on config.activation_decay.
        """
        with self.lock:
            for concept_id, concept in self.concepts.items():
                if concept.activation > 0:
                    # Apply exponential decay
                    concept.activation *= (1.0 - self.config.activation_decay)
                    if concept.activation < 0.01:  # Minimum threshold
                        concept.activation = 0.0
                    self.concept_activations[concept_id] = concept.activation

    def _consolidate_memory(self) -> None:
        """
        Perform memory consolidation operations:
        1. Decay activations
        2. Prune concepts if over capacity
        3. Update concept importances based on access patterns
        """
        with self.lock:
            # Decay activations
            self._decay_activations()

            # Adjust concept importances based on access patterns
            self._update_concept_importances()

            # Prune if necessary
            if self.config.max_concepts > 0 and len(self.concepts) > self.config.max_concepts:
                self._prune_concepts(len(self.concepts) - self.config.max_concepts)

    def _update_concept_importances(self) -> None:
        """
        Update concept importances based on access patterns and relationship structure.
        """
        import time
        current_time = time.time()

        for concept_id, concept in self.concepts.items():
            # Initialize factors
            recency_factor = 0.0
            centrality_factor = 0.0
            access_factor = 0.0

            # Recency factor - how recently the concept was accessed
            if concept.last_accessed > 0:
                time_since_access = current_time - concept.last_accessed
                recency_factor = math.exp(-self.config.recency_decay * time_since_access)

            # Centrality factor - how many relations involve this concept
            outgoing_relations = len(self.relations_by_source.get(concept_id, []))
            incoming_relations = len(self.relations_by_target.get(concept_id, []))
            relation_count = outgoing_relations + incoming_relations
            if relation_count > 0:
                centrality_factor = min(1.0, math.log(1 + relation_count) / 5.0)

            # Access factor - how frequently the concept is accessed
            if concept.access_count > 0:
                access_factor = min(1.0, math.log(1 + concept.access_count) / 5.0)

            # Combine factors into new importance
            # Higher weights for recency and centrality, lower for access count
            new_importance = (
                recency_factor * 0.4 +
                centrality_factor * 0.4 +
                access_factor * 0.2
            )

            # Smooth change by blending with current importance
            concept.importance = 0.7 * concept.importance + 0.3 * new_importance

    def _prune_concepts(self, count: int) -> None:
        """
        Remove the least important concepts based on prune_policy.

        Args:
            count: Number of concepts to remove
        """
        if count <= 0:
            return

        candidates = []

        if self.config.prune_policy == "lowest_importance":
            # Sort by importance (ascending)
            candidates = sorted(self.concepts.items(), key=lambda x: x[1].importance)
        elif self.config.prune_policy == "lowest_activation":
            # Sort by activation (ascending)
            candidates = sorted(self.concepts.items(), key=lambda x: x[1].activation)
        elif self.config.prune_policy == "lowest_confidence":
            # Sort by confidence (ascending)
            candidates = sorted(self.concepts.items(), key=lambda x: x[1].confidence)
        else:
            # Default to lowest importance
            candidates = sorted(self.concepts.items(), key=lambda x: x[1].importance)

        # Remove the least important concepts
        for i in range(min(count, len(candidates))):
            concept_id, _ = candidates[i]
            self.remove_concept(concept_id)

        self.logger.info(f"Pruned {count} concepts using policy '{self.config.prune_policy}'")

    def clear_all(self) -> None:
        """
        Clear all concepts and relations from memory.
        """
        with self.lock:
            self.concepts.clear()
            self.relations.clear()
            self.relations_by_source.clear()
            self.relations_by_target.clear()
            self.relations_by_type.clear()
            self.concepts_by_abstraction = {level: [] for level in AbstractionLevel}
            self.context_window.clear()
            self.concept_activations.clear()
            self.operation_counter = 0

        self.logger.info("Semantic memory cleared.")

    # --------------------------------------------------------------------------
    # GRAPH INTEGRATION
    # --------------------------------------------------------------------------

    def _sync_to_graph_subsystem(self, concept: Concept) -> None:
        """
        Sync concept to an external graph engine (if configured).
        This is a stub for integration with graph_engine.py.

        Args:
            concept: Concept to sync to graph
        """
        if not self.config.auto_link_graph:
            return

        self.logger.debug(f"Would sync concept {concept.concept_id} to graph subsystem (stub)")
        # In a real implementation, call graph_engine methods

    # --------------------------------------------------------------------------
    # IMPORT/EXPORT METHODS
    # --------------------------------------------------------------------------

    def export_to_dict(self) -> Dict[str, Any]:
        """
        Export the semantic memory state to a dictionary for serialization.

        Returns:
            Dictionary representation of the semantic memory
        """
        with self.lock:
            # Convert concepts to serializable dicts
            concepts_dict = {}
            for cid, concept in self.concepts.items():
                concept_dict = {
                    "concept_id": concept.concept_id,
                    "name": concept.name,
                    "description": concept.description,
                    "properties": concept.properties,
                    "synonyms": list(concept.synonyms),
                    "importance": concept.importance,
                    "abstraction_level": concept.abstraction_level.name,
                    "activation": concept.activation,
                    "last_accessed": concept.last_accessed,
                    "access_count": concept.access_count,
                    "confidence": concept.confidence,
                    "source": concept.source
                }

                # Handle embeddings specially
                if concept.embedding is not None:
                    concept_dict["embedding"] = concept.embedding.tolist()

                concepts_dict[cid] = concept_dict

            # Convert relations to serializable dicts
            relations_dict = {}
            for rid, relation in self.relations.items():
                relation_dict = {
                    "relation_id": relation.relation_id,
                    "source_concept_id": relation.source_concept_id,
                    "relation_type": str(relation.relation_type),
                    "target_concept_id": relation.target_concept_id,
                    "weight": relation.weight,
                    "properties": relation.properties,
                    "bidirectional": relation.bidirectional,
                    "transitive": relation.transitive,
                    "temporal": relation.temporal,
                    "created_at": relation.created_at,
                    "confidence": relation.confidence,
                    "source": relation.source
                }

                relations_dict[rid] = relation_dict

            # Build the export dict
            export_data = {
                "concepts": concepts_dict,
                "relations": relations_dict,
                "context_window": list(self.context_window),
                "config": {
                    "max_concepts": self.config.max_concepts,
                    "prune_policy": self.config.prune_policy,
                    "activation_decay": self.config.activation_decay,
                    "activation_spread_factor": self.config.activation_spread_factor,
                    "relation_confidence_threshold": self.config.relation_confidence_threshold,
                    "inference_depth_limit": self.config.inference_depth_limit,
                    "enable_spreading_activation": self.config.enable_spreading_activation,
                    "max_activation_spreading_steps": self.config.max_activation_spreading_steps,
                    "ontology_validation": self.config.ontology_validation,
                    "default_concept_confidence": self.config.default_concept_confidence,
                    "context_window_size": self.config.context_window_size,
                    "similarity_threshold": self.config.similarity_threshold,
                    "use_memory_consolidation": self.config.use_memory_consolidation,
                    "consolidation_interval": self.config.consolidation_interval
                }
            }

            return export_data

    def import_from_dict(self, data: Dict[str, Any]) -> bool:
        """
        Import semantic memory state from a dictionary.

        Args:
            data: Dictionary representation of semantic memory state

        Returns:
            bool: True if import was successful
        """
        with self.lock:
            try:
                # Clear existing data
                self.clear_all()

                # Update config values if present
                if "config" in data:
                    for key, value in data["config"].items():
                        if hasattr(self.config, key):
                            setattr(self.config, key, value)

                # Import concepts
                for cid, concept_dict in data.get("concepts", {}).items():
                    # Handle embeddings
                    embedding = None
                    if "embedding" in concept_dict:
                        embedding = np.array(concept_dict["embedding"])

                    # Parse abstraction level
                    abstraction_level = AbstractionLevel.SPECIFIC
                    if "abstraction_level" in concept_dict:
                        level_name = concept_dict["abstraction_level"]
                        try:
                            abstraction_level = AbstractionLevel[level_name]
                        except KeyError:
                            pass

                    concept = Concept(
                        concept_id=concept_dict["concept_id"],
                        name=concept_dict["name"],
                        description=concept_dict.get("description", ""),
                        properties=concept_dict.get("properties", {}),
                        embedding=embedding,
                        synonyms=set(concept_dict.get("synonyms", [])),
                        importance=concept_dict.get("importance", 0.5),
                        abstraction_level=abstraction_level,
                        activation=concept_dict.get("activation", 0.0),
                        last_accessed=concept_dict.get("last_accessed", 0.0),
                        access_count=concept_dict.get("access_count", 0),
                        confidence=concept_dict.get("confidence", 1.0),
                        source=concept_dict.get("source", "user")
                    )

                    # Store without using add_concept to preserve exact values
                    self.concepts[concept.concept_id] = concept
                    self.concept_activations[concept.concept_id] = concept.activation
                    self.concepts_by_abstraction[concept.abstraction_level].append(concept.concept_id)

                # Import relations
                for rid, relation_dict in data.get("relations", {}).items():
                    relation = Relation(
                        relation_id=relation_dict["relation_id"],
                        source_concept_id=relation_dict["source_concept_id"],
                        relation_type=relation_dict["relation_type"],
                        target_concept_id=relation_dict["target_concept_id"],
                        weight=relation_dict.get("weight", 1.0),
                        properties=relation_dict.get("properties", {}),
                        bidirectional=relation_dict.get("bidirectional", False),
                        transitive=relation_dict.get("transitive", False),
                        temporal=relation_dict.get("temporal", False),
                        created_at=relation_dict.get("created_at", 0.0),
                        confidence=relation_dict.get("confidence", 1.0),
                        source=relation_dict.get("source", "user")
                    )

                    # Store without using add_relation to preserve exact values
                    self.relations[relation.relation_id] = relation
                    self.relations_by_source[relation.source_concept_id].append(relation)
                    self.relations_by_target[relation.target_concept_id].append(relation)
                    self.relations_by_type[str(relation.relation_type)].append(relation)

                # Import context window
                self.context_window = deque(
                    data.get("context_window", []),
                    maxlen=self.config.context_window_size
                )

                self.logger.info(f"Imported {len(self.concepts)} concepts and {len(self.relations)} relations")
                return True

            except Exception as e:
                self.logger.error(f"Error importing semantic memory: {e}", exc_info=True)
                # Try to cleanup after failed import
                self.clear_all()
                return False

    # --------------------------------------------------------------------------
    # STATS & DIAGNOSTICS
    # --------------------------------------------------------------------------

    def get_stats(self) -> Dict[str, Any]:
        """
        Get statistics about the semantic memory state.

        Returns:
            Dictionary with statistics
        """
        with self.lock:
            stats = {
                "concept_count": len(self.concepts),
                "relation_count": len(self.relations),
                "abstraction_level_counts": {
                    level.name: len(concepts)
                    for level, concepts in self.concepts_by_abstraction.items()
                },
                "relation_type_counts": {
                    rtype: len(relations)
                    for rtype, relations in self.relations_by_type.items()
                },
                "avg_importance": sum(c.importance for c in self.concepts.values()) / max(1, len(self.concepts)),
                "avg_confidence": sum(c.confidence for c in self.concepts.values()) / max(1, len(self.concepts)),
                "avg_relations_per_concept": len(self.relations) / max(1, len(self.concepts)),
                "activated_concepts": len([c for c in self.concepts.values() if c.activation > 0]),
                "context_window_size": len(self.context_window),
                "operation_counter": self.operation_counter
            }

            return stats

    def analyze_concept_network(self) -> Dict[str, Any]:
        """
        Analyze the concept network structure to identify key concepts,
        clusters, and other network properties.

        Returns:
            Dictionary with network analysis results
        """
        with self.lock:
            if not self.concepts:
                return {"error": "No concepts in memory"}

            try:
                # This would require networkx, which might not be available
                # We'll use a more basic approach that doesn't require external dependencies

                # Find central concepts (most connections)
                connection_counts = {}
                for cid in self.concepts:
                    outgoing = len(self.relations_by_source.get(cid, []))
                    incoming = len(self.relations_by_target.get(cid, []))
                    connection_counts[cid] = outgoing + incoming

                # Sort by connection count
                central_concepts = sorted(
                    connection_counts.items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:10]  # Top 10

                # Find isolated concepts (no connections)
                isolated_concepts = [
                    cid for cid in self.concepts
                    if len(self.relations_by_source.get(cid, [])) == 0 and
                       len(self.relations_by_target.get(cid, [])) == 0
                ]

                # Find concept clusters (very basic approach)
                # Group concepts that share relations
                basic_clusters = []
                remaining = set(self.concepts.keys())

                while remaining:
                    # Start a new cluster
                    if not remaining:
                        break

                    seed = next(iter(remaining))
                    cluster = {seed}
                    to_process = {seed}

                    # Grow the cluster
                    while to_process:
                        current = to_process.pop()
                        # Find connected concepts
                        for rel in self.relations_by_source.get(current, []):
                            if rel.target_concept_id in remaining:
                                cluster.add(rel.target_concept_id)
                                to_process.add(rel.target_concept_id)

                        for rel in self.relations_by_target.get(current, []):
                            if rel.source_concept_id in remaining:
                                cluster.add(rel.source_concept_id)
                                to_process.add(rel.source_concept_id)

                    # Only add clusters with multiple concepts
                    if len(cluster) > 1:
                        basic_clusters.append(list(cluster))

                    # Remove cluster concepts from remaining
                    remainin
                    g -= cluster

                return {
                    "central_concepts": [
                        {"concept_id": cid, "name": self.concepts[cid].name, "connections": count}
                        for cid, count in central_concepts
                    ],
                    "isolated_concepts": [
                        {"concept_id": cid, "name": self.concepts[cid].name}
                        for cid in isolated_concepts
                    ],
                    "basic_clusters": [
                        {
                            "size": len(cluster),
                            "concepts": [
                                {"concept_id": cid, "name": self.concepts[cid].name}
                                for cid in cluster
                            ]
                        }
                        for cluster in basic_clusters
                    ],
                    "total_concepts": len(self.concepts),
                    "total_relations": len(self.relations),
                    "average_connectivity": sum(connection_counts.values()) / max(1, len(self.concepts))
                }

            except Exception as e:
                self.logger.error(f"Error analyzing concept network: {e}", exc_info=True)
                return {"error": f"Analysis failed: {str(e)}"}

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 373)

In [ ]:
    def _sync_concept_to_graph(self, concept: Concept) -> None:
        """Synchronize a concept to the graph engine if enabled."""
        if not self.graph_engine or not self.config.auto_link_graph:
            return

        try:
            # Add or update node in graph
            properties = {
                "type": "concept",
                "name": concept.name,
                "description": concept.description,
                "abstraction_level": concept.abstraction_level.value,
                "importance": concept.importance,
                "confidence": concept.confidence,
                "source": concept.source,
                **concept.properties
            }

            # Add synonyms as a property
            if concept.synonyms:
                properties["synonyms"] = list(concept.synonyms)

            # Check if node exists
            existing_nodes = self.graph_engine.query_nodes({"type": "concept", "name": concept.name})

            if existing_nodes:
                # Update existing node
                self.graph_engine.update_node(concept.concept_id, properties)
            else:
                # Add new node
                self.graph_engine.add_node(
                    node_id=concept.concept_id,
                    label="Concept",
                    properties=properties
                )

            # Store embedding if available
            if concept.embedding is not None and hasattr(self.graph_engine.get_node(concept.concept_id), 'embedding'):
                node = self.graph_engine.get_node(concept.concept_id)
                node.embedding = concept.embedding

        except Exception as e:
            self.logger.warning(f"Failed to sync concept {concept.concept_id} to graph: {e}")

    def _sync_relation_to_graph(self, relation: Relation) -> None:
        """Synchronize a relation to the graph engine if enabled."""
        if not self.graph_engine or not self.config.auto_link_graph:
            return

        try:
            # Add edge in graph
            properties = {
                "confidence": relation.confidence,
                "source": relation.source,
                "temporal": relation.temporal,
                **relation.properties
            }

            self.graph_engine.add_edge(
                source_id=relation.source_concept_id,
                target_id=relation.target_concept_id,
                edge_type=str(relation.relation_type),
                edge_id=relation.relation_id,
                properties=properties,
                weight=relation.weight,
                bidirectional=relation.bidirectional
            )

        except Exception as e:
            self.logger.warning(f"Failed to sync relation {relation.relation_id} to graph: {e}")

    def _navigate_knowledge_graph(self, start_concept_id: str, query: str, max_depth: int = 3) -> List[str]:
        """
        Navigate the knowledge graph to find related concepts based on a query.

        Args:
            start_concept_id: Starting concept ID
            query: Query string to guide navigation
            max_depth: Maximum depth to traverse

        Returns:
            List of relevant concept IDs found
        """
        if not self.graph_engine:
            return []

        try:
            # Use graph engine to execute query
            graph_query = GraphQuery(
                start_nodes=[start_concept_id],
                node_criteria={"type": "concept"},
                max_depth=max_depth,
                min_activation=0.1
            )

            results = self.graph_engine.execute_query(graph_query)

            # Extract concept IDs from results
            concept_ids = list(results["nodes"].keys())

            # If we have an embedding function, we can rank by similarity to query
            if self.config.embedding_function and query:
                query_embedding = self.config.embedding_function(query)

                # Rank concepts by embedding similarity
                ranked_concepts = []
                for concept_id in concept_ids:
                    if concept_id in self.concepts and self.concepts[concept_id].embedding is not None:
                        similarity = self._similarity(
                            query_embedding,
                            self.concepts[concept_id].embedding,
                            self.config.similarity_metric
                        )
                        ranked_concepts.append((concept_id, similarity))

                # Sort by similarity and return IDs
                ranked_concepts.sort(key=lambda x: x[1], reverse=True)
                return [cid for cid, _ in ranked_concepts]

            return concept_ids

        except Exception as e:
            self.logger.warning(f"Failed to navigate knowledge graph: {e}")
            return []

# Working Memory

In [ ]:
"""
working.py - Hybrid Working Memory for GrandNexus

This module unifies:
  1) Solid LRU / LFU / priority eviction logic.
  2) Text + embedding-based retrieval approach (like a "Gemini + ChatGPT" synergy).
  3) Optional expiration times for short-lived items.
  4) Recency bias if desired.
  5) Potential auto-transfer to another store on eviction.

It is fully thread-safe via RLock, and can be integrated with other memory or
cognition modules in GrandNexus.
"""

import logging
import threading
import time
import uuid
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Callable, Tuple, Union
import numpy as np


@dataclass
class WorkingMemoryItem:
    """
    Represents a single item stored in working memory.

    Attributes:
        item_id (str): Unique identifier.
        content (Any): Actual data (text, partial results, etc.).
        source (str): Which module or process inserted it.
        timestamp (float): Creation or last update time.
        importance (float): Salience or relevance (0..1).
        metadata (Dict[str, Any]): Additional info.
        expiration_time (Optional[float]): If set, we consider the item expired after that time.
        embedding (Optional[np.ndarray]): If config provides an embedding_function, we store the computed vector here.
    """
    item_id: str
    content: Any
    source: str
    timestamp: float = field(default_factory=time.time)
    importance: float = 0.5
    metadata: Dict[str, Any] = field(default_factory=dict)
    expiration_time: Optional[float] = None
    embedding: Optional[np.ndarray] = None

    def __repr__(self) -> str:
        return (f"<WorkingMemoryItem id={self.item_id[-8:]}, "
                f"source={self.source}, importance={self.importance:.2f}>")

class WorkingMemoryConfig:
    """
    Configuration for the WorkingMemory.

    Attributes:
        capacity (int): Max number of items allowed.
        eviction_policy (str): 'LRU', 'LFU', or 'priority'.
        default_importance (float): Default if importance not provided
        expiration_enabled (bool): If True, we consider item.expiration_time for removal
        default_expiration_time (Optional[float]): If set, used when adding items
        embedding_function (Optional[Callable[[Any], np.ndarray]]): For item embeddings
        recency_bias (float): Weighted factor for recency in retrieval scoring
        auto_transfer_callback (Optional[Callable[[WorkingMemoryItem], None]]):
            If set, we call it before eviction, allowing items to be "saved" elsewhere
    """
    def __init__(self,
                 capacity: int = 100,
                 eviction_policy: str = "LRU",
                 default_importance: float = 0.5,
                 expiration_enabled: bool = False,
                 default_expiration_time: Optional[float] = None,
                 embedding_function: Optional[Callable[[Any], np.ndarray]] = None,
                 recency_bias: float = 0.0,
                 auto_transfer_callback: Optional[Callable[[WorkingMemoryItem], None]] = None
                 ):
        self.capacity = capacity
        self.eviction_policy = eviction_policy
        self.default_importance = default_importance
        self.expiration_enabled = expiration_enabled
        self.default_expiration_time = default_expiration_time
        self.embedding_function = embedding_function
        self.recency_bias = recency_bias
        self.auto_transfer_callback = auto_transfer_callback


class WorkingMemory:
    """
    A robust working memory module for short-lived data, with concurrency,
    multiple eviction policies, item embeddings, text-based + embedding retrieval, etc.

    - store_item(): Add new item
    - get_item(): Retrieve by ID
    - retrieve(): High-level retrieval by text / embedding / importance
    - update_item(), remove_item()
    - Evictions done automatically when capacity is exceeded
    - Optionally call auto_transfer_callback on items about to be evicted
    - Clear or get_all_items for debugging
    """

    def __init__(self, config: WorkingMemoryConfig):
        self.logger = logging.getLogger("WorkingMemory")
        self.config = config
        self.lock = threading.RLock()

        # item_id -> WorkingMemoryItem
        self.memory: Dict[str, WorkingMemoryItem] = {}

        # LRU uses a deque for the order (most recently used at the right)
        self._access_order: deque[str] = deque()
        # LFU uses a map from item_id -> access_count
        self._access_counts: Dict[str, int] = defaultdict(int)

        # Recent retrievals/operations cache for optimizing frequently accessed items
        self._recent_operations_cache: Dict[str, Any] = {}
        self._cache_hits = 0
        self._cache_misses = 0

        self.logger.info(f"WorkingMemory initialized: capacity={config.capacity}, eviction={config.eviction_policy}")

        # Register with memory manager if available
        if hasattr(config, 'memory_manager') and config.memory_manager:
            self._register_with_memory_manager(config.memory_manager)

    def store_item(self,
                   content: Any,
                   source: str,
                   item_id: Optional[str] = None,
                   importance: Optional[float] = None,
                   metadata: Optional[Dict[str, Any]] = None,
                   expiration_time: Optional[float] = None,
                   embedding: Optional[np.ndarray] = None) -> str:
        """
        Add a new item to working memory. May trigger eviction if over capacity.

        Args:
            content: The item content
            source: Which module or subsystem created it
            item_id: If you want to specify a custom ID (else we create a new UUID)
            importance: 0..1, if omitted use config.default_importance
            metadata: Additional info
            expiration_time: If omitted but expiration_enabled, use default_expiration_time
            embedding: Pre-computed embedding vector for similarity searches

        Returns:
            The item_id of the newly stored item
        """
        with self.lock:
            item_id = item_id or str(uuid.uuid4())
            if item_id in self.memory:
                self.logger.warning(f"Item {item_id} already exists; overwriting.")
                self._remove_tracking(item_id)

            if importance is None:
                importance = self.config.default_importance
            if self.config.expiration_enabled and expiration_time is None and self.config.default_expiration_time is not None:
                expiration_time = time.time() + self.config.default_expiration_time

            metadata = metadata or {}
            metadata["source"] = source

            # Create the item
            now = time.time()
            item = WorkingMemoryItem(
                item_id=item_id,
                content=content,
                source=source,
                timestamp=now,
                importance=importance,
                metadata=metadata,
                expiration_time=expiration_time,
                embedding=embedding
            )

            # Generate embedding if needed and function provided
            if embedding is None and self.config.embedding_function:
                try:
                    item.embedding = self.config.embedding_function(content)
                except Exception as e:
                    self.logger.error(f"Failed to embed content for item {item_id}: {e}", exc_info=True)

            # Store the item
            self.memory[item_id] = item

            # Track access patterns based on policy
            if self.config.eviction_policy == "LRU":
                self._access_order.append(item_id)
            elif self.config.eviction_policy == "LFU":
                self._access_counts[item_id] = 1

            # Clear operation cache entry if it exists
            if item_id in self._recent_operations_cache:
                del self._recent_operations_cache[item_id]

            # Check capacity and evict if needed
            self._evict_if_needed()

            # Log with detailed context
            expiry_str = f", expires in {int(expiration_time - now)}s" if expiration_time else ""
            self.logger.debug(f"Stored item {item_id} from {source}, importance={importance:.2f}{expiry_str}")

            # Notify listeners if configured
            if hasattr(self.config, 'on_store_callbacks') and self.config.on_store_callbacks:
                for callback in self.config.on_store_callbacks:
                    try:
                        callback(item)
                    except Exception as e:
                        self.logger.error(f"Error in on_store callback: {e}", exc_info=True)

            return item_id

    def get_item(self, item_id: str) -> Optional[WorkingMemoryItem]:
        """
        Retrieve an item by ID, checking expiration, updating LRU or LFU data.
        Uses caching for frequently accessed items.
        """
        # Check cache first for frequent operations
        if item_id in self._recent_operations_cache:
            self._cache_hits += 1
            return self._recent_operations_cache[item_id]

        self._cache_misses += 1

        with self.lock:
            it = self.memory.get(item_id)
            if not it:
                return None

            # Check expiration
            if self.config.expiration_enabled and it.expiration_time is not None:
                if time.time() > it.expiration_time:
                    self.logger.debug(f"Item {item_id} expired; removing.")
                    self.remove_item(item_id)
                    return None

            # Access update
            self._record_access(item_id)

            # Cache for future operations if frequently accessed
            if it.access_count > 3:  # Only cache items accessed multiple times
                self._recent_operations_cache[item_id] = it
                # Keep cache size reasonable
                if len(self._recent_operations_cache) > min(100, self.config.capacity // 5):
                    # Remove oldest entries when cache gets too large
                    oldest = next(iter(self._recent_operations_cache))
                    del self._recent_operations_cache[oldest]

            return it

    def retrieve(self,
                 query_text: Optional[str] = None,
                 query_embedding: Optional[np.ndarray] = None,
                 similarity_metric: str = "cosine",
                 min_importance: float = 0.0,
                 max_results: int = 10,
                 metadata_filter: Optional[Dict[str, Any]] = None,
                 source_filter: Optional[Union[str, List[str]]] = None) -> List[WorkingMemoryItem]:
        """
        A combined retrieval approach that merges:
          - text-based matching (if content is str and query_text is given)
          - embedding-based similarity (if query_embedding is provided and item has an embedding)
          - item importance
          - optional recency bias
          - metadata and source filtering

        We compute a total score for each item, then sort descending and return the top N.
        """
        t0 = time.time()

        # Prepare source filter as a set for faster lookup
        if source_filter:
            if isinstance(source_filter, str):
                source_filter_set = {source_filter}
            else:
                source_filter_set = set(source_filter)
        else:
            source_filter_set = None

        with self.lock:
            # Pre-filter expired items
            now = time.time()
            valid_items = []
            for item in self.memory.values():
                # Skip expired items
                if self.config.expiration_enabled and item.expiration_time is not None and now > item.expiration_time:
                    continue

                # Skip items below minimum importance
                if item.importance < min_importance:
                    continue

                # Apply source filter if specified
                if source_filter_set and item.source not in source_filter_set:
                    continue

                # Apply metadata filter if specified
                if metadata_filter:
                    if not all(item.metadata.get(k) == v for k, v in metadata_filter.items()):
                        continue

                valid_items.append(item)

            # Exit early if no valid items
            if not valid_items:
                self.logger.debug("retrieve: no valid items matched filters")
                return []

            # Calculate scores for valid items
            scored_items = []
            for item in valid_items:
                score = self._calculate_retrieval_score(
                    item,
                    query_text,
                    query_embedding,
                    similarity_metric,
                    now
                )
                scored_items.append((score, item))

            # Sort by score descending and take top N
            scored_items.sort(reverse=True)
            results = [item for _, item in scored_items[:max_results]]

            # Update access records for returned items
            for item in results:
                self._record_access(item.item_id)

            dt = time.time() - t0
            self.logger.debug(f"retrieve: returned {len(results)} items in {dt:.4f}s from {len(valid_items)} candidates")
            return results

    def _calculate_retrieval_score(self,
                                 item: WorkingMemoryItem,
                                 query_text: Optional[str],
                                 query_embedding: Optional[np.ndarray],
                                 similarity_metric: str,
                                 now: float) -> float:
        """
        Calculate a combined score for an item during retrieval.
        Considers text match, embedding similarity, importance, and recency.
        """
        score = item.importance  # Start with base importance

        # Text match component
        if query_text and isinstance(item.content, str):
            text_score = self._text_similarity(item.content, query_text)
            score += text_score

        # Embedding similarity component
        if query_embedding is not None and item.embedding is not None:
            try:
                sim = self._vector_similarity(query_embedding, item.embedding, similarity_metric)
                score += sim
            except Exception as e:
                self.logger.error(f"Error computing embedding similarity: {e}", exc_info=True)

        # Recency bias component
        if self.config.recency_bias > 0.0:
            age = now - item.timestamp
            # Exponential decay based on recency
            recency_factor = np.exp(-age * self.config.recency_decay_rate)
            recency_bonus = self.config.recency_bias * recency_factor
            score += recency_bonus

        # Access frequency component (optional)
        if hasattr(self.config, 'access_frequency_weight') and self.config.access_frequency_weight > 0:
            access_bonus = math.log(1 + item.access_count) * self.config.access_frequency_weight
            score += access_bonus

        return score

    def _text_similarity(self, text1: str, text2: str) -> float:
        """
        Calculate text similarity between two strings.
        Uses a simple approach based on word overlap and substring matching.
        """
        # Method 1: Simple word overlap
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())

        if not words1 or not words2:
            return 0.0

        overlap = len(words1.intersection(words2))
        word_sim = overlap / (len(words1) + len(words2) - overlap)  # Jaccard similarity

        # Method 2: Substring match
        substring_sim = 0.0
        if text2.lower() in text1.lower():
            substring_sim = len(text2) / len(text1)

        # Return max of the two methods
        return max(word_sim, substring_sim)

    def _vector_similarity(self, vec1: np.ndarray, vec2: np.ndarray, metric: str) -> float:
        """
        Compute vector similarity using the specified metric.
        """
        # Ensure inputs are numpy arrays
        if not isinstance(vec1, np.ndarray):
            vec1 = np.array(vec1)
        if not isinstance(vec2, np.ndarray):
            vec2 = np.array(vec2)

        # Ensure dimensions match
        if vec1.shape != vec2.shape:
            if len(vec1.shape) == 1 and len(vec2.shape) == 1:
                # Reshape to match dimensions if both are 1D but different lengths
                min_len = min(vec1.shape[0], vec2.shape[0])
                vec1 = vec1[:min_len]
                vec2 = vec2[:min_len]
            else:
                self.logger.warning(f"Incompatible embedding dimensions: {vec1.shape} vs {vec2.shape}")
                return 0.0

        if metric == "cosine":
            # Calculate cosine similarity: dot(a, b) / (||a|| * ||b||)
            norm1 = np.linalg.norm(vec1)
            norm2 = np.linalg.norm(vec2)

            if norm1 > 0 and norm2 > 0:
                return np.dot(vec1, vec2) / (norm1 * norm2)
            return 0.0

        elif metric == "euclidean":
            # Calculate negative Euclidean distance (negative since higher values = more similar)
            return -np.linalg.norm(vec1 - vec2)

        elif metric == "dot":
            # Simple dot product
            return float(np.dot(vec1, vec2))

        else:
            self.logger.warning(f"Unknown similarity metric: {metric}, defaulting to cosine")
            return self._vector_similarity(vec1, vec2, "cosine")

    def update_item(self,
                    item_id: str,
                    new_content: Optional[Any] = None,
                    new_importance: Optional[float] = None,
                    new_metadata: Optional[Dict[str, Any]] = None,
                    new_embedding: Optional[np.ndarray] = None,
                    reset_expiration: bool = False) -> bool:
        """
        Update an existing item (content, importance, metadata).
        Recompute embedding if content changes.

        Args:
            item_id: The ID of the item to update
            new_content: New content to set (None to keep current)
            new_importance: New importance to set (None to keep current)
            new_metadata: Metadata to update (None to keep current)
            new_embedding: New embedding vector (None to recompute if needed)
            reset_expiration: If True, reset the expiration timer

        Returns:
            bool: True if successful, False if item not found
        """
        with self.lock:
            it = self.memory.get(item_id)
            if not it:
                self.logger.warning(f"update_item: item {item_id} not found.")
                return False

            modified = False

            # Update content if provided
            if new_content is not None:
                it.content = new_content
                modified = True

                # Auto-recompute embedding if content changed and no new embedding provided
                if new_embedding is None and self.config.embedding_function:
                    try:
                        it.embedding = self.config.embedding_function(new_content)
                    except Exception as e:
                        self.logger.error(f"Embedding update failed for {item_id}: {e}", exc_info=True)

            # Update embedding if provided
            if new_embedding is not None:
                it.embedding = new_embedding
                modified = True

            # Update importance if provided
            if new_importance is not None:
                it.importance = max(0.0, min(1.0, new_importance))
                modified = True

            # Update metadata if provided
            if new_metadata is not None:
                it.metadata.update(new_metadata)
                modified = True

            # Reset expiration if requested
            if reset_expiration and it.expiration_time is not None:
                if self.config.default_expiration_time:
                    it.expiration_time = time.time() + self.config.default_expiration_time
                    modified = True

            # Update timestamp and access tracking
            if modified:
                it.timestamp = time.time()
                self._record_access(item_id)

                # Clear cache entry if it exists
                if item_id in self._recent_operations_cache:
                    del self._recent_operations_cache[item_id]

                self.logger.debug(f"update_item => item {item_id} updated.")

                # Notify listeners if configured
                if hasattr(self.config, 'on_update_callbacks') and self.config.on_update_callbacks:
                    for callback in self.config.on_update_callbacks:
                        try:
                            callback(it)
                        except Exception as e:
                            self.logger.error(f"Error in on_update callback: {e}", exc_info=True)

            return True

    def remove_item(self, item_id: str) -> bool:
        """
        Remove an item from working memory.

        Args:
            item_id: The item ID to remove

        Returns:
            bool: True if item was removed, False if not found
        """
        with self.lock:
            if item_id not in self.memory:
                self.logger.warning(f"Cannot remove item {item_id}: not found.")
                return False

            # Get the item before removing it for callbacks
            item = self.memory[item_id]

            # Remove from tracking structures
            self._remove_tracking(item_id)

            # Remove from memory
            del self.memory[item_id]

            # Remove from operation cache
            if item_id in self._recent_operations_cache:
                del self._recent_operations_cache[item_id]

            self.logger.debug(f"Removed item {item_id} from working memory.")

            # Notify listeners if configured
            if hasattr(self.config, 'on_remove_callbacks') and self.config.on_remove_callbacks:
                for callback in self.config.on_remove_callbacks:
                    try:
                        callback(item)
                    except Exception as e:
                        self.logger.error(f"Error in on_remove callback: {e}", exc_info=True)

            return True

    def _evict_if_needed(self) -> None:
        """
        Check if we're over capacity and evict items if needed.
        """
        if len(self.memory) <= self.config.capacity:
            return

        # Calculate how many items to evict
        to_evict = len(self.memory) - self.config.capacity
        self.logger.debug(f"Need to evict {to_evict} items to respect capacity")

        # Use the appropriate eviction policy
        if self.config.eviction_policy == "LRU":
            self._evict_lru(to_evict)
        elif self.config.eviction_policy == "LFU":
            self._evict_lfu(to_evict)
        elif self.config.eviction_policy == "priority":
            self._evict_priority(to_evict)
        else:
            self.logger.warning(f"Unknown eviction policy: {self.config.eviction_policy}, defaulting to LRU")
            self._evict_lru(to_evict)

    def _evict_lru(self, count: int) -> None:
        """
        Evict the least recently used items.
        """
        evicted = 0
        while evicted < count and self._access_order:
            # Remove from the left (least recently used)
            item_id = self._access_order.popleft()
            if item_id in self.memory:
                item = self.memory[item_id]
                self._auto_transfer_if_needed(item)
                del self.memory[item_id]

                # Remove from other tracking structures
                if item_id in self._access_counts:
                    del self._access_counts[item_id]
                if item_id in self._recent_operations_cache:
                    del self._recent_operations_cache[item_id]

                evicted += 1

        self.logger.debug(f"LRU eviction: removed {evicted} items")

    def _evict_lfu(self, count: int) -> None:
        """
        Evict the least frequently used items.
        """
        if not self._access_counts:
            return

        # Sort items by access count (ascending)
        sorted_items = sorted(self._access_counts.items(), key=lambda x: x[1])

        evicted = 0
        for item_id, _ in sorted_items:
            if item_id in self.memory and evicted < count:
                item = self.memory[item_id]
                self._auto_transfer_if_needed(item)
                del self.memory[item_id]

                # Remove from other tracking structures
                try:
                    self._access_order.remove(item_id)
                except ValueError:
                    pass
                del self._access_counts[item_id]
                if item_id in self._recent_operations_cache:
                    del self._recent_operations_cache[item_id]

                evicted += 1

            if evicted >= count:
                break

        self.logger.debug(f"LFU eviction: removed {evicted} items")

    def _evict_priority(self, count: int) -> None:
        """
        Evict items based on priority (importance, recency, etc.)
        """
        # Build scoring function
        now = time.time()

        def score_item(item: WorkingMemoryItem) -> float:
            """
            Calculate a score for each item - lower score = higher priority to keep
            (i.e., higher score = more likely to be evicted)
            """
            importance_factor = 1.0 - item.importance  # Invert importance (lower importance = higher score)
            recency_factor = (now - item.timestamp) / 3600.0  # Hours since last update
            access_factor = 1.0 / (1.0 + math.log(1.0 + item.access_count))  # Inverse of access count

            # Combined score (higher = more likely to evict)
            return (importance_factor * 0.6) + (recency_factor * 0.3) + (access_factor * 0.1)

        # Calculate scores for all items
        scored_items = [(score_item(item), item_id) for item_id, item in self.memory.items()]

        # Sort by score (highest first = most likely to evict)
        scored_items.sort(reverse=True)

        # Evict the highest-scored items
        evicted = 0
        for _, item_id in scored_items:
            if evicted >= count:
                break

            if item_id in self.memory:
                item = self.memory[item_id]
                self._auto_transfer_if_needed(item)
                del self.memory[item_id]

                # Remove from tracking structures
                self._remove_tracking(item_id)
                if item_id in self._recent_operations_cache:
                    del self._recent_operations_cache[item_id]

                evicted += 1

        self.logger.debug(f"Priority eviction: removed {evicted} items")

    def _auto_transfer_if_needed(self, item: WorkingMemoryItem) -> None:
        """
        If auto_transfer_callback is configured, call it with the item
        about to be evicted.
        """
        if not hasattr(self.config, 'auto_transfer_callback') or not self.config.auto_transfer_callback:
            return

        try:
            self.config.auto_transfer_callback(item)
            self.logger.debug(f"Auto-transferred item {item.item_id} to another memory store")
        except Exception as e:
            self.logger.error(f"Error in auto_transfer_callback for {item.item_id}: {e}", exc_info=True)

    def _record_access(self, item_id: str) -> None:
        """
        Update access tracking structures when an item is accessed.
        """
        if item_id not in self.memory:
            return

        # Update the item's access count
        self.memory[item_id].access_count += 1

        # Update LRU tracking if enabled
        if self.config.eviction_policy == "LRU":
            # Move to the end of the access order (most recently used)
            try:
                self._access_order.remove(item_id)
            except ValueError:
                pass  # Not found, which is fine
            self._access_order.append(item_id)

        # Update LFU tracking if enabled
        elif self.config.eviction_policy == "LFU":
            self._access_counts[item_id] += 1

    def _remove_tracking(self, item_id: str) -> None:
        """
        Remove an item from all tracking structures.
        """
        # LRU tracking
        try:
            self._access_order.remove(item_id)
        except ValueError:
            pass  # Not found, which is fine

        # LFU tracking
        if item_id in self._access_counts:
            del self._access_counts[item_id]

        # Operation cache
        if item_id in self._recent_operations_cache:
            del self._recent_operations_cache[item_id]

    def clear(self) -> None:
        """
        Remove all items from working memory.
        """
        with self.lock:
            # Get copy of items for callbacks
            items = list(self.memory.values())

            # Clear all data structures
            self.memory.clear()
            self._access_order.clear()
            self._access_counts.clear()
            self._recent_operations_cache.clear()
            self._cache_hits = 0
            self._cache_misses = 0

            self.logger.info(f"Working memory cleared ({len(items)} items removed)")

            # Notify listeners if configured
            if hasattr(self.config, 'on_clear_callbacks') and self.config.on_clear_callbacks:
                for callback in self.config.on_clear_callbacks:
                    try:
                        callback(items)
                    except Exception as e:
                        self.logger.error(f"Error in on_clear callback: {e}", exc_info=True)

    def get_all_items(self) -> List[WorkingMemoryItem]:
        """
        Returns a snapshot (copy) of all items currently in working memory.
        """
        with self.lock:
            return list(self.memory.values())

    def get_stats(self) -> Dict[str, Any]:
        """
        Return statistics about the working memory.
        """
        with self.lock:
            return {
                "capacity": self.config.capacity,
                "current_size": len(self.memory),
                "eviction_policy": self.config.eviction_policy,
                "access_order_size": len(self._access_order),
                "lfu_tracking_size": len(self._access_counts),
                "cache_hits": self._cache_hits,
                "cache_misses": self._cache_misses,
                "cache_hit_ratio": self._cache_hits / (self._cache_hits + self._cache_misses) if (self._cache_hits + self._cache_misses) > 0 else 0,
                "items_by_source": self._count_items_by_source(),
                "average_importance": self._calculate_average_importance()
            }

    def _count_items_by_source(self) -> Dict[str, int]:
        """
        Count items by source.
        """
        counts = defaultdict(int)
        for item in self.memory.values():
            counts[item.source] += 1
        return dict(counts)

    def _calculate_average_importance(self) -> float:
        """
        Calculate the average importance of all items.
        """
        if not self.memory:
            return 0.0

        total = sum(item.importance for item in self.memory.values())
        return total / len(self.memory)

    def _register_with_memory_manager(self, memory_manager: Any) -> None:
        """
        Register this working memory instance with a memory manager.
        This allows for integration with the broader memory system.
        """
        try:
            if hasattr(memory_manager, 'register_memory_module'):
                memory_manager.register_memory_module('working_memory', self)
                self.logger.info(f"Registered with memory manager: {memory_manager}")
            else:
                self.logger.warning(f"Memory manager does not support registration: {memory_manager}")
        except Exception as e:
            self.logger.error(f"Error registering with memory manager: {e}", exc_info=True)


# Cognition Manager

In [ ]:
"""
cognition_manager.py - Central Cognitive Orchestrator for GrandNexus

This module serves as the central coordinator for all cognitive processes in the GrandNexus
architecture. It orchestrates the interaction between perception, reasoning, learning, and
metacognition modules, creating an emergent cognitive system that is greater than the sum
of its parts.

Key responsibilities:
- Managing cognitive workflows and information flow between modules
- Coordinating perceptual input processing through various filters and analyzers
- Orchestrating reasoning processes using different reasoning strategies
- Facilitating learning from experiences and feedback
- Enabling metacognitive monitoring and adaptation of cognitive processes
- Creating a cognitive workspace where modules can contribute and interact

Design principles:
- Organic interaction rather than rigid control - modules influence each other
- Emergence from simple interaction rules rather than top-down prescription
- Flexible cognitive pathways that can evolve based on experience
- Metacognitive awareness and self-modification capabilities
- Balance between specialization (modularity) and integration (holism)
"""

import logging
import threading
import time
import uuid
from enum import Enum, auto
from typing import Any, Dict, List, Optional, Tuple, Union, Callable, Set
from dataclasses import dataclass, field
from collections import defaultdict, deque
import random
import json
import numpy as np

# Imports from GrandNexus modules
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory

# Cognitive modules imports
# Perception
from grandnexus.cognition.perception.perception_core import PerceptionCore
from grandnexus.cognition.perception.symbolic_parser import SymbolicParser
from grandnexus.cognition.perception.code_analyzer import CodeAnalyzer
from grandnexus.cognition.perception.pattern_recognizer import PatternRecognizer

# Reasoning
from grandnexus.cognition.reasoning.reasoning_core import ReasoningCore
from grandnexus.cognition.reasoning.symbolic_engine import SymbolicEngine
from grandnexus.cognition.reasoning.neural_reasoner import NeuralReasoner
from grandnexus.cognition.reasoning.hybrid_inference import HybridInferenceEngine
from grandnexus.cognition.reasoning.planning import PlanningEngine
from grandnexus.cognition.reasoning.uncertainty import UncertaintyManager

# Learning
from grandnexus.cognition.learning.learning_core import LearningCore
from grandnexus.cognition.learning.continuous_learner import ContinuousLearner
from grandnexus.cognition.learning.meta_learner import MetaLearner
from grandnexus.cognition.learning.reinforcement import ReinforcementLearner
from grandnexus.cognition.learning.intrinsic_motivation import IntrinsicMotivation
from grandnexus.cognition.learning.curriculum import CurriculumManager

# Metacognition
from grandnexus.cognition.metacognition.monitor import CognitiveMonitor
from grandnexus.cognition.metacognition.confidence import ConfidenceEvaluator
from grandnexus.cognition.metacognition.adaptation import AdaptationManager
from grandnexus.cognition.metacognition.reflection import Reflector

# Define cognitive event types
class CognitiveEventType(Enum):
    """Types of cognitive events that can occur in the system"""
    PERCEPTION = auto()      # New perception input
    REASONING = auto()       # Reasoning process event
    LEARNING = auto()        # Learning event
    METACOGNITION = auto()   # Metacognitive reflection or adaptation
    GOAL = auto()            # Goal-related event
    ATTENTION = auto()       # Attentional shift
    ACTION = auto()          # Action taken by the system
    ERROR = auto()           # Error condition
    INTEGRATION = auto()     # Cross-module integration event

@dataclass
class CognitiveEvent:
    """Represents a cognitive event in the system"""
    event_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    event_type: CognitiveEventType = CognitiveEventType.PERCEPTION
    timestamp: float = field(default_factory=time.time)
    source_module: str = ""
    content: Any = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    priority: float = 0.5    # Higher values indicate higher priority
    processed: bool = False  # Whether this event has been processed

@dataclass
class CognitiveTask:
    """Represents a cognitive task to be executed"""
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    task_type: str = ""      # Type of task (perception, reasoning, etc.)
    description: str = ""
    source: str = ""         # Module that created this task
    parameters: Dict[str, Any] = field(default_factory=dict)
    dependencies: List[str] = field(default_factory=list)  # IDs of tasks this depends on
    priority: float = 0.5
    deadline: Optional[float] = None
    status: str = "pending"  # pending, running, completed, failed
    result: Any = None
    created_at: float = field(default_factory=time.time)

@dataclass
class CognitiveState:
    """Represents the current cognitive state of the system"""
    attention_focus: Dict[str, float] = field(default_factory=dict)  # Concept -> activation
    active_goals: List[Dict[str, Any]] = field(default_factory=list)
    active_contexts: Dict[str, float] = field(default_factory=dict)  # Context -> strength
    recent_events: List[CognitiveEvent] = field(default_factory=list)
    working_load: float = 0.0  # Cognitive load measure (0.0-1.0)
    uncertainty: float = 0.0   # Global uncertainty measure (0.0-1.0)
    arousal: float = 0.5       # Cognitive arousal/activation level (0.0-1.0)
    mode: str = "normal"       # Cognitive mode: normal, focused, exploratory, etc.
    metadata: Dict[str, Any] = field(default_factory=dict)

class CognitionManager:
    """
    Central orchestrator for GrandNexus's cognitive functions, coordinating
    between perception, reasoning, learning, and metacognitive processes.

    This manager facilitates the emergence of a flexible, adaptive cognitive
    system by establishing pathways for information flow and feedback loops
    between specialized modules.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None):
        """
        Initialize the cognition manager.

        Args:
            nexus_core: Reference to the NexusCore for system-wide coordination
            working_memory: Reference to working memory for temporary storage
            semantic_memory: Reference to semantic memory for structured knowledge
            episodic_memory: Reference to episodic memory for experience-based storage
        """
        self.logger = logging.getLogger("GrandNexus.Cognition.Manager")
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Cognitive state
        self.cognitive_state = CognitiveState()

        # Event management
        self.event_queue = deque()
        self.event_history = deque(maxlen=1000)

        # Task management
        self.task_queue = []  # Heap queue for priority-based tasks
        self.active_tasks = {}  # task_id -> task
        self.completed_tasks = deque(maxlen=100)

        # Module references
        self.perception_modules = {}
        self.reasoning_modules = {}
        self.learning_modules = {}
        self.metacognition_modules = {}

        # Cognitive pathways (dynamic connections between modules)
        self.pathways = defaultdict(float)  # (source, target, event_type) -> strength

        # Metrics tracking
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"CognitionManager initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize all cognitive modules and establish base pathways"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Initialize all cognitive modules
                self._initialize_perception_modules()
                self._initialize_reasoning_modules()
                self._initialize_learning_modules()
                self._initialize_metacognition_modules()

                # Establish initial cognitive pathways
                self._establish_initial_pathways()

                # Connect to NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("CognitionManager initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"CognitionManager initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the cognition manager and begin cognitive processing"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start all cognitive modules
                self._start_perception_modules()
                self._start_reasoning_modules()
                self._start_learning_modules()
                self._start_metacognition_modules()

                # Start cognitive threads
                self._start_cognitive_threads()

                self.running = True
                self.logger.info("CognitionManager started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start CognitionManager: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the cognition manager and all cognitive processes"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Stop all cognitive threads
                self._stop_cognitive_threads()

                # Stop all cognitive modules
                self._stop_metacognition_modules()
                self._stop_learning_modules()
                self._stop_reasoning_modules()
                self._stop_perception_modules()

                self.running = False
                self.logger.info("CognitionManager stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping CognitionManager: {str(e)}")
                return False

    def process_input(self, input_data: Any, input_type: str = "text",
                     metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Process a new input through the cognitive pipeline.

        Args:
            input_data: The input data to process
            input_type: Type of input (text, image, code, etc.)
            metadata: Additional information about the input

        Returns:
            task_id: ID for tracking the processing task
        """
        if not self.initialized:
            self.initialize()

        if metadata is None:
            metadata = {}

        # Create a cognitive task for this input
        task = CognitiveTask(
            task_type="perception",
            description=f"Process {input_type} input",
            source="external",
            parameters={
                "input_data": input_data,
                "input_type": input_type
            },
            priority=metadata.get("priority", 0.5)
        )

        # Create a corresponding cognitive event
        event = CognitiveEvent(
            event_type=CognitiveEventType.PERCEPTION,
            source_module="external",
            content={
                "input_data": input_data,
                "input_type": input_type,
                "task_id": task.task_id
            },
            metadata=metadata,
            priority=metadata.get("priority", 0.5)
        )

        with self.lock:
            # Add the task to the queue
            self._add_task(task)

            # Add the event to the queue
            self.event_queue.append(event)

        self.logger.info(f"Input processing task created: {task.task_id}")
        return task.task_id

    def add_goal(self, description: str, parameters: Optional[Dict[str, Any]] = None,
                priority: float = 0.5, deadline: Optional[float] = None) -> str:
        """
        Add a new goal for the cognitive system to pursue.

        Args:
            description: Description of the goal
            parameters: Additional parameters for the goal
            priority: Goal priority (0.0-1.0)
            deadline: Optional deadline timestamp

        Returns:
            goal_id: ID for tracking the goal
        """
        if not self.initialized:
            self.initialize()

        if parameters is None:
            parameters = {}

        # Create a goal object
        goal_id = str(uuid.uuid4())
        goal = {
            "goal_id": goal_id,
            "description": description,
            "parameters": parameters,
            "priority": priority,
            "deadline": deadline,
            "created_at": time.time(),
            "status": "active"
        }

        # Create a corresponding cognitive event
        event = CognitiveEvent(
            event_type=CognitiveEventType.GOAL,
            source_module="external",
            content={
                "goal": goal,
                "action": "add"
            },
            priority=priority
        )

        with self.lock:
            # Add the goal to active goals
            self.cognitive_state.active_goals.append(goal)

            # Add the event to the queue
            self.event_queue.append(event)

        self.logger.info(f"Goal added: {goal_id} - {description}")
        return goal_id

    def update_goal(self, goal_id: str, status: Optional[str] = None,
                  priority: Optional[float] = None,
                  parameters: Optional[Dict[str, Any]] = None) -> bool:
        """
        Update an existing goal.

        Args:
            goal_id: ID of the goal to update
            status: New status for the goal (active, completed, abandoned)
            priority: New priority for the goal
            parameters: New or updated parameters

        Returns:
            success: Whether the update was successful
        """
        with self.lock:
            # Find the goal
            for i, goal in enumerate(self.cognitive_state.active_goals):
                if goal["goal_id"] == goal_id:
                    # Update fields as specified
                    updates = {}

                    if status is not None:
                        goal["status"] = status
                        updates["status"] = status

                        # If goal is no longer active, we might remove it
                        if status in ["completed", "abandoned"]:
                            self.cognitive_state.active_goals.pop(i)

                    if priority is not None:
                        goal["priority"] = priority
                        updates["priority"] = priority

                    if parameters is not None:
                        goal["parameters"].update(parameters)
                        updates["parameters"] = parameters

                    # Create event for goal update
                    event = CognitiveEvent(
                        event_type=CognitiveEventType.GOAL,
                        source_module="external",
                        content={
                            "goal_id": goal_id,
                            "action": "update",
                            "updates": updates
                        },
                        priority=goal.get("priority", 0.5)
                    )

                    # Add the event to the queue
                    self.event_queue.append(event)

                    self.logger.info(f"Goal updated: {goal_id}")
                    return True

            self.logger.warning(f"Goal not found: {goal_id}")
            return False

    def get_goal_status(self, goal_id: str) -> Optional[Dict[str, Any]]:
        """
        Get the current status of a goal.

        Args:
            goal_id: ID of the goal

        Returns:
            goal_info: Dictionary with goal information
        """
        with self.lock:
            for goal in self.cognitive_state.active_goals:
                if goal["goal_id"] == goal_id:
                    return goal.copy()

            # Check completed tasks for this goal
            for task in self.completed_tasks:
                if task.task_type == "goal" and task.parameters.get("goal_id") == goal_id:
                    return {
                        "goal_id": goal_id,
                        "status": "completed" if task.status == "completed" else "failed",
                        "result": task.result
                    }

            return None

    def get_cognitive_state(self) -> Dict[str, Any]:
        """
        Get a snapshot of the current cognitive state.

        Returns:
            state_dict: Dictionary representation of the cognitive state
        """
        with self.lock:
            # Convert to dictionary and make a deep copy
            state_dict = {
                "attention_focus": dict(self.cognitive_state.attention_focus),
                "active_goals": [goal.copy() for goal in self.cognitive_state.active_goals],
                "active_contexts": dict(self.cognitive_state.active_contexts),
                "recent_events": [
                    {
                        "event_id": e.event_id,
                        "event_type": e.event_type.name,
                        "timestamp": e.timestamp,
                        "source_module": e.source_module,
                        "priority": e.priority,
                        "processed": e.processed
                    }
                    for e in self.cognitive_state.recent_events
                ],
                "working_load": self.cognitive_state.working_load,
                "uncertainty": self.cognitive_state.uncertainty,
                "arousal": self.cognitive_state.arousal,
                "mode": self.cognitive_state.mode,
                "metadata": dict(self.cognitive_state.metadata)
            }

            return state_dict

    def shift_attention(self, focus_items: Dict[str, float]) -> None:
        """
        Shift the system's attention to specific concepts or items.

        Args:
            focus_items: Dictionary mapping concepts to activation levels
        """
        with self.lock:
            # Update attention focus
            for concept, activation in focus_items.items():
                self.cognitive_state.attention_focus[concept] = activation

            # Create a corresponding cognitive event
            event = CognitiveEvent(
                event_type=CognitiveEventType.ATTENTION,
                source_module="external",
                content={
                    "focus_items": focus_items
                },
                priority=0.7  # Attention shifts are important
            )

            # Add the event to the queue
            self.event_queue.append(event)

        self.logger.info(f"Attention shifted to {list(focus_items.keys())}")

    def get_task_status(self, task_id: str) -> Optional[Dict[str, Any]]:
        """
        Get the current status of a task.

        Args:
            task_id: ID of the task

        Returns:
            task_info: Dictionary with task information
        """
        with self.lock:
            # Check active tasks
            if task_id in self.active_tasks:
                task = self.active_tasks[task_id]
                return {
                    "task_id": task.task_id,
                    "task_type": task.task_type,
                    "status": task.status,
                    "progress": self._estimate_task_progress(task),
                    "created_at": task.created_at
                }

            # Check task queue
            for task in self.task_queue:
                if task.task_id == task_id:
                    return {
                        "task_id": task.task_id,
                        "task_type": task.task_type,
                        "status": "pending",
                        "progress": 0.0,
                        "created_at": task.created_at
                    }

            # Check completed tasks
            for task in self.completed_tasks:
                if task.task_id == task_id:
                    return {
                        "task_id": task.task_id,
                        "task_type": task.task_type,
                        "status": task.status,
                        "result": task.result,
                        "created_at": task.created_at
                    }

            return None

    def register_module(self, module_type: str, module_name: str,
                       module_instance: Any) -> bool:
        """
        Register a new cognitive module with the manager.

        Args:
            module_type: Type of module (perception, reasoning, learning, metacognition)
            module_name: Name for the module
            module_instance: The module instance to register

        Returns:
            success: Whether registration was successful
        """
        with self.lock:
            if module_type == "perception":
                self.perception_modules[module_name] = module_instance
            elif module_type == "reasoning":
                self.reasoning_modules[module_name] = module_instance
            elif module_type == "learning":
                self.learning_modules[module_name] = module_instance
            elif module_type == "metacognition":
                self.metacognition_modules[module_name] = module_instance
            else:
                self.logger.error(f"Unknown module type: {module_type}")
                return False

            self.logger.info(f"Registered {module_type} module: {module_name}")
            return True

    def add_cognitive_pathway(self, source: str, target: str,
                            event_type: CognitiveEventType,
                            strength: float = 0.5) -> None:
        """
        Add or strengthen a cognitive pathway between modules.

        Args:
            source: Source module name
            target: Target module name
            event_type: Type of event that flows through this pathway
            strength: Connection strength (0.0-1.0)
        """
        with self.lock:
            key = (source, target, event_type)
            self.pathways[key] = strength

            self.logger.info(f"Added pathway: {source} -> {target} ({event_type.name}, {strength:.2f})")

    def submit_cognitive_event(self, event: CognitiveEvent) -> None:
        """
        Submit a cognitive event for processing.

        Args:
            event: The cognitive event to process
        """
        with self.lock:
            self.event_queue.append(event)

            self.logger.debug(f"Event submitted: {event.event_type.name} from {event.source_module}")

    def get_metrics(self) -> Dict[str, Dict[str, float]]:
        """
        Get performance metrics for cognitive processes.

        Returns:
            metrics_dict: Dictionary of metrics
        """
        with self.lock:
            metrics_dict = {}

            for key, values in self.metrics.items():
                if values:
                    metrics_dict[key] = {
                        "current": values[-1],
                        "average": sum(values) / len(values),
                        "min": min(values),
                        "max": max(values)
                    }

            return metrics_dict

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _initialize_perception_modules(self) -> None:
        """Initialize perception modules"""
        try:
            # Initialize PerceptionCore
            perception_core = PerceptionCore(
                nexus_core=self.nexus_core,
                working_memory=self.working_memory,
                semantic_memory=self.semantic_memory
            )
            perception_core.initialize()
            self.perception_modules["perception_core"] = perception_core

            # Initialize SymbolicParser
            symbolic_parser = SymbolicParser(
                nexus_core=self.nexus_core,
                semantic_memory=self.semantic_memory,
                working_memory=self.working_memory
            )
            symbolic_parser.initialize()
            self.perception_modules["symbolic_parser"] = symbolic_parser

            # Initialize CodeAnalyzer
            code_analyzer = CodeAnalyzer(
                nexus_core=self.nexus_core,
                semantic_memory=self.semantic_memory
            )
            code_analyzer.initialize()
            self.perception_modules["code_analyzer"] = code_analyzer

            # Initialize PatternRecognizer
            pattern_recognizer = PatternRecognizer(
                nexus_core=self.nexus_core,
                semantic_memory=self.semantic_memory,
                episodic_memory=self.episodic_memory
            )
            pattern_recognizer.initialize()
            self.perception_modules["pattern_recognizer"] = pattern_recognizer

            self.logger.info("Perception modules initialized")
        except Exception as e:
            self.logger.error(f"Error initializing perception modules: {str(e)}")
            raise

    def _initialize_reasoning_modules(self) -> None:
        """Initialize reasoning modules"""
        try:
            # Initialize ReasoningCore
            reasoning_core = ReasoningCore(
                nexus_core=self.nexus_core,
                working_memory=self.working_memory,
                semantic_memory=self.semantic_memory,
                episodic_memory=self.episodic_memory
            )
            reasoning_core.initialize()
            self.reasoning_modules["reasoning_core"] = reasoning_core

            # Initialize SymbolicEngine
            symbolic_engine = SymbolicEngine(
                nexus_core=self.nexus_core,
                semantic_memory=self.semantic_memory
            )
            symbolic_engine.initialize()
            self.reasoning_modules["symbolic_engine"] = symbolic_engine

            # Initialize NeuralReasoner
            neural_reasoner = NeuralReasoner(
                nexus_core=self.nexus_core,
                semantic_memory=self.semantic_memory
            )
            neural_reasoner.initialize()
            self.reasoning_modules["neural_reasoner"] = neural_reasoner

            # Initialize HybridInferenceEngine
            hybrid_inference = HybridInferenceEngine(
                nexus_core=self.nexus_core,
                symbolic_engine=symbolic_engine,
                neural_reasoner=neural_reasoner
            )
            hybrid_inference.initialize()
            self.reasoning_modules["hybrid_inference"] = hybrid_inference

            # Initialize PlanningEngine
            planning_engine = PlanningEngine(
                nexus_core=self.nexus_core,
                working_memory=self.working_memory,
                semantic_memory=self.semantic_memory
            )
            planning_engine.initialize()
            self.reasoning_modules["planning_engine"] = planning_engine

            # Initialize UncertaintyManager
            uncertainty_manager = UncertaintyManager(
                nexus_core=self.nexus_core
            )
            uncertainty_manager.initialize()
            self.reasoning_modules["uncertainty_manager"] = uncertainty_manager

            self.logger.info("Reasoning modules initialized")
        except Exception as e:
            self.logger.error(f"Error initializing reasoning modules: {str(e)}")
            raise

    def _initialize_learning_modules(self) -> None:
        """Initialize learning modules"""
        try:
            # Initialize LearningCore
            learning_core = LearningCore(
                nexus_core=self.nexus_core,
                working_memory=self.working_memory,
                semantic_memory=self.semantic_memory,
                episodic_memory=self.episodic_memory
            )
            learning_core.initialize()
            self.learning_modules["learning_core"] = learning_core

            # Initialize ContinuousLearner
            continuous_learner = ContinuousLearner(
                nexus_core=self.nexus_core,
                learning_core=learning_core
            )
            continuous_learner.initialize()
            self.learning_modules["continuous_learner"] = continuous_learner

            # Initialize MetaLearner
            meta_learner = MetaLearner(
                nexus_core=self.nexus_core,
                learning_core=learning_core
            )
            meta_learner.initialize()
            self.learning_modules["meta_learner"] = meta_learner

            # Initialize ReinforcementLearner
            reinforcement_learner = ReinforcementLearner(
                nexus_core=self.nexus_core,
                learning_core=learning_core
            )
            reinforcement_learner.initialize()
            self.learning_modules["reinforcement_learner"] = reinforcement_learner

            # Initialize IntrinsicMotivation
            intrinsic_motivation = IntrinsicMotivation(
                nexus_core=self.nexus_core,
                learning_core=learning_core
            )
            intrinsic_motivation.initialize()
            self.learning_modules["intrinsic_motivation"] = intrinsic_motivation

            # Initialize CurriculumManager
            curriculum_manager = CurriculumManager(
                nexus_core=self.nexus_core,
                learning_core=learning_core
            )
            curriculum_manager.initialize()
            self.learning_modules["curriculum_manager"] = curriculum_manager

            self.logger.info("Learning modules initialized")
        except Exception as e:
            self.logger.error(f"Error initializing learning modules: {str(e)}")
            raise

    def _initialize_metacognition_modules(self) -> None:
        """Initialize metacognition modules"""
        try:
            # Initialize CognitiveMonitor
            cognitive_monitor = CognitiveMonitor(
                nexus_core=self.nexus_core
            )
            cognitive_monitor.initialize()
            self.metacognition_modules["cognitive_monitor"] = cognitive_monitor

            # Initialize ConfidenceEvaluator
            confidence_evaluator = ConfidenceEvaluator(
                nexus_core=self.nexus_core
            )
            confidence_evaluator.initialize()
            self.metacognition_modules["confidence_evaluator"] = confidence_evaluator

            # Initialize AdaptationManager
            adaptation_manager = AdaptationManager(
                nexus_core=self.nexus_core
            )
            adaptation_manager.initialize()
            self.metacognition_modules["adaptation_manager"] = adaptation_manager

            # Initialize Reflector
            reflector = Reflector(
                nexus_core=self.nexus_core,
                episodic_memory=self.episodic_memory
            )
            reflector.initialize()
            self.metacognition_modules["reflector"] = reflector

            self.logger.info("Metacognition modules initialized")
        except Exception as e:
            self.logger.error(f"Error initializing metacognition modules: {str(e)}")
            raise

    def _establish_initial_pathways(self) -> None:
        """Establish initial cognitive pathways between modules"""
        # Perception to Reasoning pathways
        self.add_cognitive_pathway("perception_core", "reasoning_core",
                                  CognitiveEventType.PERCEPTION, 0.8)
        self.add_cognitive_pathway("symbolic_parser", "symbolic_engine",
                                  CognitiveEventType.PERCEPTION, 0.9)
        self.add_cognitive_pathway("code_analyzer", "reasoning_core",
                                  CognitiveEventType.PERCEPTION, 0.7)
        self.add_cognitive_pathway("pattern_recognizer", "neural_reasoner",
                                  CognitiveEventType.PERCEPTION, 0.8)

        # Reasoning to Learning pathways
        self.add_cognitive_pathway("reasoning_core", "learning_core",
                                  CognitiveEventType.REASONING, 0.7)
        self.add_cognitive_pathway("symbolic_engine", "learning_core",
                                  CognitiveEventType.REASONING, 0.6)
        self.add_cognitive_pathway("neural_reasoner", "continuous_learner",
                                  CognitiveEventType.REASONING, 0.7)
        self.add_cognitive_pathway("hybrid_inference", "meta_learner",
                                  CognitiveEventType.REASONING, 0.8)

        # Learning to Metacognition pathways
        self.add_cognitive_pathway("learning_core", "cognitive_monitor",
                                  CognitiveEventType.LEARNING, 0.7)
        self.add_cognitive_pathway("continuous_learner", "adaptation_manager",
                                  CognitiveEventType.LEARNING, 0.6)
        self.add_cognitive_pathway("meta_learner", "reflector",
                                  CognitiveEventType.LEARNING, 0.8)

        # Metacognition feedback loops
        self.add_cognitive_pathway("cognitive_monitor", "reasoning_core",
                                  CognitiveEventType.METACOGNITION, 0.7)
        self.add_cognitive_pathway("confidence_evaluator", "uncertainty_manager",
                                  CognitiveEventType.METACOGNITION, 0.9)
        self.add_cognitive_pathway("adaptation_manager", "learning_core",
                                  CognitiveEventType.METACOGNITION, 0.8)
        self.add_cognitive_pathway("reflector", "intrinsic_motivation",
                                  CognitiveEventType.METACOGNITION, 0.7)

        # Goal-related pathways
        self.add_cognitive_pathway("reasoning_core", "planning_engine",
                                  CognitiveEventType.GOAL, 0.9)
        self.add_cognitive_pathway("planning_engine", "reinforcement_learner",
                                  CognitiveEventType.GOAL, 0.7)

        # Attention-related pathways
        self.add_cognitive_pathway("cognitive_monitor", "perception_core",
                                  CognitiveEventType.ATTENTION, 0.8)
        self.add_cognitive_pathway("intrinsic_motivation", "perception_core",
                                  CognitiveEventType.ATTENTION, 0.7)

        self.logger.info("Initial cognitive pathways established")

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="cognition_manager",
                module=self,
                dependencies=dependencies
            )

            # Set up message handlers - specific implementation would depend on NexusCore

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _start_perception_modules(self) -> None:
        """Start all perception modules"""
        for name, module in self.perception_modules.items():
            try:
                if hasattr(module, "start") and callable(module.start):
                    module.start()
                    self.logger.info(f"Started perception module: {name}")
            except Exception as e:
                self.logger.error(f"Error starting perception module {name}: {str(e)}")

    def _start_reasoning_modules(self) -> None:
        """Start all reasoning modules"""
        for name, module in self.reasoning_modules.items():
            try:
                if hasattr(module, "start") and callable(module.start):
                    module.start()
                    self.logger.info(f"Started reasoning module: {name}")
            except Exception as e:
                self.logger.error(f"Error starting reasoning module {name}: {str(e)}")

    def _start_learning_modules(self) -> None:
        """Start all learning modules"""
        for name, module in self.learning_modules.items():
            try:
                if hasattr(module, "start") and callable(module.start):
                    module.start()
                    self.logger.info(f"Started learning module: {name}")
            except Exception as e:
                self.logger.error(f"Error starting learning module {name}: {str(e)}")

    def _start_metacognition_modules(self) -> None:
        """Start all metacognition modules"""
        for name, module in self.metacognition_modules.items():
            try:
                if hasattr(module, "start") and callable(module.start):
                    module.start()
                    self.logger.info(f"Started metacognition module: {name}")
            except Exception as e:
                self.logger.error(f"Error starting metacognition module {name}: {str(e)}")

    def _stop_perception_modules(self) -> None:
        """Stop all perception modules"""
        for name, module in self.perception_modules.items():
            try:
                if hasattr(module, "stop") and callable(module.stop):
                    module.stop()
                    self.logger.info(f"Stopped perception module: {name}")
            except Exception as e:
                self.logger.error(f"Error stopping perception module {name}: {str(e)}")

    def _stop_reasoning_modules(self) -> None:
        """Stop all reasoning modules"""
        for name, module in self.reasoning_modules.items():
            try:
                if hasattr(module, "stop") and callable(module.stop):
                    module.stop()
                    self.logger.info(f"Stopped reasoning module: {name}")
            except Exception as e:
                self.logger.error(f"Error stopping reasoning module {name}: {str(e)}")

    def _stop_learning_modules(self) -> None:
        """Stop all learning modules"""
        for name, module in self.learning_modules.items():
            try:
                if hasattr(module, "stop") and callable(module.stop):
                    module.stop()
                    self.logger.info(f"Stopped learning module: {name}")
            except Exception as e:
                self.logger.error(f"Error stopping learning module {name}: {str(e)}")

    def _stop_metacognition_modules(self) -> None:
        """Stop all metacognition modules"""
        for name, module in self.metacognition_modules.items():
            try:
                if hasattr(module, "stop") and callable(module.stop):
                    module.stop()
                    self.logger.info(f"Stopped metacognition module: {name}")
            except Exception as e:
                self.logger.error(f"Error stopping metacognition module {name}: {str(e)}")

    def _start_cognitive_threads(self) -> None:
        """Start cognitive processing threads"""
        # Create and start threads for different cognitive processes
        self.stop_threads = False

        # Event processing thread
        self.event_thread = threading.Thread(
            target=self._event_processing_loop,
            name="event_processor",
            daemon=True
        )
        self.event_thread.start()

        # Task execution thread
        self.task_thread = threading.Thread(
            target=self._task_execution_loop,
            name="task_executor",
            daemon=True
        )
        self.task_thread.start()

        # State update thread
        self.state_thread = threading.Thread(
            target=self._state_update_loop,
            name="state_updater",
            daemon=True
        )
        self.state_thread.start()

        # Metacognitive monitoring thread
        self.monitor_thread = threading.Thread(
            target=self._metacognitive_monitoring_loop,
            name="meta_monitor",
            daemon=True
        )
        self.monitor_thread.start()

        self.logger.info("Cognitive threads started")

    def _stop_cognitive_threads(self) -> None:
        """Stop cognitive processing threads"""
        self.stop_threads = True

        # Wait for threads to exit gracefully
        if hasattr(self, 'event_thread') and self.event_thread.is_alive():
            self.event_thread.join(timeout=2.0)

        if hasattr(self, 'task_thread') and self.task_thread.is_alive():
            self.task_thread.join(timeout=2.0)

        if hasattr(self, 'state_thread') and self.state_thread.is_alive():
            self.state_thread.join(timeout=2.0)

        if hasattr(self, 'monitor_thread') and self.monitor_thread.is_alive():
            self.monitor_thread.join(timeout=2.0)

        self.logger.info("Cognitive threads stopped")

    def _event_processing_loop(self) -> None:
        """Main event processing loop"""
        self.logger.info("Event processing loop started")

        while not self.stop_threads:
            try:
                # Process events in queue
                events_processed = self._process_events_batch()

                # If no events processed, sleep briefly
                if not events_processed:
                    time.sleep(0.01)

            except Exception as e:
                self.logger.error(f"Error in event processing loop: {str(e)}")
                time.sleep(0.1)  # Sleep longer on error

        self.logger.info("Event processing loop stopped")

    def _process_events_batch(self, max_events: int = 10) -> int:
        """
        Process a batch of events from the queue.

        Args:
            max_events: Maximum number of events to process in this batch

        Returns:
            processed_count: Number of events processed
        """
        processed_count = 0

        with self.lock:
            # Process up to max_events
            for _ in range(min(max_events, len(self.event_queue))):
                if not self.event_queue:
                    break

                # Get the next event
                event = self.event_queue.popleft()

                # Add to event history
                self.event_history.append(event)

                # Add to recent events in cognitive state
                self.cognitive_state.recent_events.append(event)
                if len(self.cognitive_state.recent_events) > 20:
                    self.cognitive_state.recent_events.pop(0)

                # Release lock during actual processing to avoid blocking
                self.lock.release()
                try:
                    # Process the event
                    self._process_single_event(event)
                    processed_count += 1
                finally:
                    # Re-acquire lock
                    self.lock.acquire()

        return processed_count

    def _process_single_event(self, event: CognitiveEvent) -> None:
        """
        Process a single cognitive event.

        Args:
            event: The event to process
        """
        try:
            # Update metrics
            self.metrics["events_processed"].append(1.0)

            # Mark as processed
            event.processed = True

            # Find relevant pathways for this event
            pathways = []
            for (source, target, event_type), strength in self.pathways.items():
                if source == event.source_module and event_type == event.event_type:
                    pathways.append((target, strength))

            # Route event to target modules based on pathways
            for target, strength in pathways:
                # Check if strength exceeds threshold based on event priority
                threshold = 0.3 - (event.priority * 0.2)  # Lower threshold for higher priority

                if strength >= threshold:
                    self._route_event_to_module(event, target)

            # Special handling based on event type
            if event.event_type == CognitiveEventType.PERCEPTION:
                self._handle_perception_event(event)
            elif event.event_type == CognitiveEventType.REASONING:
                self._handle_reasoning_event(event)
            elif event.event_type == CognitiveEventType.LEARNING:
                self._handle_learning_event(event)
            elif event.event_type == CognitiveEventType.METACOGNITION:
                self._handle_metacognition_event(event)
            elif event.event_type == CognitiveEventType.GOAL:
                self._handle_goal_event(event)
            elif event.event_type == CognitiveEventType.ATTENTION:
                self._handle_attention_event(event)

        except Exception as e:
            self.logger.error(f"Error processing event {event.event_id}: {str(e)}")

            # Generate an error event
            error_event = CognitiveEvent(
                event_type=CognitiveEventType.ERROR,
                source_module="cognition_manager",
                content={
                    "error": str(e),
                    "original_event": {
                        "event_id": event.event_id,
                        "event_type": event.event_type.name,
                        "source_module": event.source_module
                    }
                },
                priority=0.8  # Errors are high priority
            )

            # Add error event to queue
            with self.lock:
                self.event_queue.append(error_event)

    def _route_event_to_module(self, event: CognitiveEvent, target_module: str) -> None:
        """
        Route an event to a specific module.

        Args:
            event: The event to route
            target_module: The target module name
        """
        # Find the target module in the appropriate category
        module = None

        if target_module in self.perception_modules:
            module = self.perception_modules[target_module]
        elif target_module in self.reasoning_modules:
            module = self.reasoning_modules[target_module]
        elif target_module in self.learning_modules:
            module = self.learning_modules[target_module]
        elif target_module in self.metacognition_modules:
            module = self.metacognition_modules[target_module]

        if module is None:
            self.logger.warning(f"Target module not found: {target_module}")
            return

        # Check if module has event handling capability
        if hasattr(module, "handle_event") and callable(module.handle_event):
            try:
                module.handle_event(event)
                self.logger.debug(f"Routed event {event.event_id} to {target_module}")
            except Exception as e:
                self.logger.error(f"Error routing event to {target_module}: {str(e)}")
        else:
            self.logger.warning(f"Module {target_module} cannot handle events")

    def _handle_perception_event(self, event: CognitiveEvent) -> None:
        """
        Handle a perception event.

        Args:
            event: The perception event to handle
        """
        # Check if this is a new input to process
        if event.source_module == "external" and "input_data" in event.content:
            # Extract information from event
            input_data = event.content["input_data"]
            input_type = event.content.get("input_type", "text")
            task_id = event.content.get("task_id")

            # Create a task to process this input if not already created
            if task_id is None:
                task = CognitiveTask(
                    task_type="perception",
                    description=f"Process {input_type} input",
                    source="external",
                    parameters={
                        "input_data": input_data,
                        "input_type": input_type
                    },
                    priority=event.priority
                )

                with self.lock:
                    self._add_task(task)
                    task_id = task.task_id

            # Route to appropriate perception module based on input type
            if input_type == "text":
                if "perception_core" in self.perception_modules:
                    self.perception_modules["perception_core"].process_text(
                        input_data,
                        metadata={"task_id": task_id, "priority": event.priority}
                    )
            elif input_type == "symbolic":
                if "symbolic_parser" in self.perception_modules:
                    self.perception_modules["symbolic_parser"].parse(
                        input_data,
                        context={"task_id": task_id}
                    )
            elif input_type == "code":
                if "code_analyzer" in self.perception_modules:
                    self.perception_modules["code_analyzer"].analyze(
                        input_data,
                        {"task_id": task_id}
                    )
            elif input_type == "pattern":
                if "pattern_recognizer" in self.perception_modules:
                    self.perception_modules["pattern_recognizer"].recognize(
                        input_data,
                        {"task_id": task_id}
                    )
            else:
                self.logger.warning(f"Unsupported input type: {input_type}")

    def _handle_reasoning_event(self, event: CognitiveEvent) -> None:
        """
        Handle a reasoning event.

        Args:
            event: The reasoning event to handle
        """
        # Implementation would depend on specific reasoning events
        pass

    def _handle_learning_event(self, event: CognitiveEvent) -> None:
        """
        Handle a learning event.

        Args:
            event: The learning event to handle
        """
        # Implementation would depend on specific learning events
        pass

    def _handle_metacognition_event(self, event: CognitiveEvent) -> None:
        """
        Handle a metacognition event.

        Args:
            event: The metacognition event to handle
        """
        # Implementation would depend on specific metacognition events
        pass

    def _handle_goal_event(self, event: CognitiveEvent) -> None:
        """
        Handle a goal-related event.

        Args:
            event: The goal event to handle
        """
        if "action" in event.content:
            action = event.content["action"]

            if action == "add":
                goal = event.content.get("goal")
                if goal and "goal_id" in goal:
                    # Update cognitive state with new goal
                    with self.lock:
                        # Check if goal already exists
                        for existing in self.cognitive_state.active_goals:
                            if existing["goal_id"] == goal["goal_id"]:
                                return

                        # Add new goal
                        self.cognitive_state.active_goals.append(goal)

                    # Route to planning engine
                    if "planning_engine" in self.reasoning_modules:
                        self.reasoning_modules["planning_engine"].add_goal(goal)

            elif action == "update":
                goal_id = event.content.get("goal_id")
                updates = event.content.get("updates", {})

                if goal_id:
                    # Update goal in cognitive state
                    with self.lock:
                        for goal in self.cognitive_state.active_goals:
                            if goal["goal_id"] == goal_id:
                                # Apply updates
                                for key, value in updates.items():
                                    if key == "status" and value in ["completed", "abandoned"]:
                                        # If goal is completed or abandoned, might remove it
                                        self.cognitive_state.active_goals.remove(goal)
                                        break
                                    elif isinstance(value, dict) and isinstance(goal.get(key, {}), dict):
                                        # Merge dictionaries for nested updates
                                        goal.setdefault(key, {}).update(value)
                                    else:
                                        goal[key] = value
                                break

                    # Route to planning engine
                    if "planning_engine" in self.reasoning_modules:
                        self.reasoning_modules["planning_engine"].update_goal(goal_id, updates)

    def _handle_attention_event(self, event: CognitiveEvent) -> None:
        """
        Handle an attention-related event.

        Args:
            event: The attention event to handle
        """
        if "focus_items" in event.content:
            focus_items = event.content["focus_items"]

            # Update attention focus in cognitive state
            with self.lock:
                # Apply decay to existing focus items
                for concept in list(self.cognitive_state.attention_focus.keys()):
                    self.cognitive_state.attention_focus[concept] *= 0.8

                    # Remove concepts with negligible activation
                    if self.cognitive_state.attention_focus[concept] < 0.1:
                        del self.cognitive_state.attention_focus[concept]

                # Add or update new focus items
                for concept, activation in focus_items.items():
                    if concept in self.cognitive_state.attention_focus:
                        # Combine activations (max)
                        self.cognitive_state.attention_focus[concept] = max(
                            self.cognitive_state.attention_focus[concept],
                            activation
                        )
                    else:
                        # Add new focus item
                        self.cognitive_state.attention_focus[concept] = activation

            # Notify perception modules of attention shift
            if "perception_core" in self.perception_modules:
                self.perception_modules["perception_core"].shift_attention(focus_items)

    def _task_execution_loop(self) -> None:
        """Main task execution loop"""
        self.logger.info("Task execution loop started")

        while not self.stop_threads:
            try:
                # Get the highest priority task
                task = self._get_next_task()

                if task:
                    # Execute the task
                    self._execute_task(task)
                else:
                    # No tasks available, sleep briefly
                    time.sleep(0.01)

            except Exception as e:
                self.logger.error(f"Error in task execution loop: {str(e)}")
                time.sleep(0.1)  # Sleep longer on error

        self.logger.info("Task execution loop stopped")

    def _get_next_task(self) -> Optional[CognitiveTask]:
        """
        Get the next task to execute based on priority and dependencies.

        Returns:
            task: The next task to execute, or None if no tasks are ready
        """
        with self.lock:
            # Check if there are tasks in the queue
            if not self.task_queue:
                return None

            # Find the highest priority task that has all dependencies satisfied
            eligible_tasks = []
            for task in self.task_queue:
                # Check if all dependencies are satisfied
                dependencies_satisfied = True

                for dep_id in task.dependencies:
                    # Check if dependency is complete
                    dep_complete = False

                    # Check in completed tasks
                    for completed_task in self.completed_tasks:
                        if completed_task.task_id == dep_id and completed_task.status == "completed":
                            dep_complete = True
                            break

                    if not dep_complete:
                        dependencies_satisfied = False
                        break

                if dependencies_satisfied:
                    eligible_tasks.append(task)

            if not eligible_tasks:
                return None

            # Sort by priority (higher value = higher priority)
            eligible_tasks.sort(key=lambda t: t.priority, reverse=True)

            # Get the highest priority task
            selected_task = eligible_tasks[0]

            # Remove from queue
            self.task_queue.remove(selected_task)

            # Mark as active
            selected_task.status = "running"
            self.active_tasks[selected_task.task_id] = selected_task

            return selected_task

    def _add_task(self, task: CognitiveTask) -> None:
        """
        Add a task to the queue.

        Args:
            task: The task to add
        """
        self.task_queue.append(task)
        self.logger.debug(f"Task added to queue: {task.task_id}")

    def _execute_task(self, task: CognitiveTask) -> None:
        """
        Execute a cognitive task.

        Args:
            task: The task to execute
        """
        self.logger.info(f"Executing task: {task.task_id} - {task.description}")

        try:
            # Update metrics
            self.metrics["tasks_executed"].append(1.0)

            # Execute based on task type
            if task.task_type == "perception":
                self._execute_perception_task(task)
            elif task.task_type == "reasoning":
                self._execute_reasoning_task(task)
            elif task.task_type == "learning":
                self._execute_learning_task(task)
            elif task.task_type == "metacognition":
                self._execute_metacognition_task(task)
            elif task.task_type == "goal":
                self._execute_goal_task(task)
            else:
                self.logger.warning(f"Unknown task type: {task.task_type}")

            # Mark task as completed
            with self.lock:
                task.status = "completed"
                if task.task_id in self.active_tasks:
                    del self.active_tasks[task.task_id]
                self.completed_tasks.append(task)

            self.logger.info(f"Task completed: {task.task_id}")

            # Generate task completion event
            completion_event = CognitiveEvent(
                event_type=CognitiveEventType.INTEGRATION,
                source_module="cognition_manager",
                content={
                    "completed_task": {
                        "task_id": task.task_id,
                        "task_type": task.task_type,
                        "result": task.result
                    }
                }
            )

            # Add event to queue
            with self.lock:
                self.event_queue.append(completion_event)

        except Exception as e:
            self.logger.error(f"Error executing task {task.task_id}: {str(e)}")

            # Mark task as failed
            with self.lock:
                task.status = "failed"
                task.result = {"error": str(e)}
                if task.task_id in self.active_tasks:
                    del self.active_tasks[task.task_id]
                self.completed_tasks.append(task)

            # Generate error event
            error_event = CognitiveEvent(
                event_type=CognitiveEventType.ERROR,
                source_module="cognition_manager",
                content={
                    "error": str(e),
                    "failed_task": {
                        "task_id": task.task_id,
                        "task_type": task.task_type
                    }
                },
                priority=0.7
            )

            # Add error event to queue
            with self.lock:
                self.event_queue.append(error_event)

    def _execute_perception_task(self, task: CognitiveTask) -> None:
        """
        Execute a perception task.

        Args:
            task: The perception task to execute
        """
        input_data = task.parameters.get("input_data")
        input_type = task.parameters.get("input_type", "text")

        if input_data is None:
            raise ValueError("No input data provided")

        # Default result is empty
        task.result = {}

        # Process based on input type
        if input_type == "text":
            if "perception_core" in self.perception_modules:
                result = self.perception_modules["perception_core"].process_text(
                    input_data,
                    metadata={"task_id": task.task_id}
                )
                task.result["perception"] = result
        elif input_type == "symbolic":
            if "symbolic_parser" in self.perception_modules:
                result = self.perception_modules["symbolic_parser"].parse(
                    input_data,
                    context={"task_id": task.task_id}
                )
                task.result["parsed"] = result
        elif input_type == "code":
            if "code_analyzer" in self.perception_modules:
                result = self.perception_modules["code_analyzer"].analyze(
                    input_data,
                    {"task_id": task.task_id}
                )
                task.result["code_analysis"] = result
        elif input_type == "pattern":
            if "pattern_recognizer" in self.perception_modules:
                result = self.perception_modules["pattern_recognizer"].recognize(
                    input_data,
                    {"task_id": task.task_id}
                )
                task.result["patterns"] = result
        else:
            raise ValueError(f"Unsupported input type: {input_type}")

    def _execute_reasoning_task(self, task: CognitiveTask) -> None:
        """
        Execute a reasoning task.

        Args:
            task: The reasoning task to execute
        """
        # Implementation would depend on reasoning task types
        pass

    def _execute_learning_task(self, task: CognitiveTask) -> None:
        """
        Execute a learning task.

        Args:
            task: The learning task to execute
        """
        # Implementation would depend on learning task types
        pass

    def _execute_metacognition_task(self, task: CognitiveTask) -> None:
        """
        Execute a metacognition task.

        Args:
            task: The metacognition task to execute
        """
        task_subtype = task.parameters.get("subtype")

        if task_subtype == "monitor":
            # Execute monitoring task
            if "cognitive_monitor" in self.metacognition_modules:
                result = self.metacognition_modules["cognitive_monitor"].perform_monitoring(
                    task.parameters.get("target_modules", []),
                    task.parameters.get("metrics", [])
                )
                task.result = {"monitoring_results": result}

        elif task_subtype == "evaluate_confidence":
            # Execute confidence evaluation task
            if "confidence_evaluator" in self.metacognition_modules:
                result = self.metacognition_modules["confidence_evaluator"].evaluate(
                    task.parameters.get("content"),
                    task.parameters.get("context", {})
                )
                task.result = {"confidence": result}

        elif task_subtype == "adapt":
            # Execute adaptation task
            if "adaptation_manager" in self.metacognition_modules:
                result = self.metacognition_modules["adaptation_manager"].adapt(
                    task.parameters.get("module"),
                    task.parameters.get("parameters", {}),
                    task.parameters.get("context", {})
                )
                task.result = {"adaptation_result": result}

        elif task_subtype == "reflect":
            # Execute reflection task
            if "reflector" in self.metacognition_modules:
                result = self.metacognition_modules["reflector"].reflect_on(
                    task.parameters.get("topic"),
                    task.parameters.get("depth", 1),
                    task.parameters.get("context", {})
                )
                task.result = {"reflection": result}

        else:
            raise ValueError(f"Unknown metacognition task subtype: {task_subtype}")

    def _execute_goal_task(self, task: CognitiveTask) -> None:
        """
        Execute a goal-related task.

        Args:
            task: The goal task to execute
        """
        action = task.parameters.get("action")
        goal_id = task.parameters.get("goal_id")

        if action == "plan":
            # Create plan for a goal
            if "planning_engine" in self.reasoning_modules:
                plan = self.reasoning_modules["planning_engine"].create_plan(
                    goal_id,
                    task.parameters.get("constraints", {})
                )
                task.result = {"plan": plan}

        elif action == "execute_step":
            # Execute a step in a plan
            if "planning_engine" in self.reasoning_modules:
                step_result = self.reasoning_modules["planning_engine"].execute_step(
                    goal_id,
                    task.parameters.get("step_id")
                )
                task.result = {"step_result": step_result}

        elif action == "evaluate":
            # Evaluate progress on a goal
            goal_progress = self._evaluate_goal_progress(goal_id)
            task.result = {"progress": goal_progress}

        elif action == "revise":
            # Revise a goal or plan
            if "planning_engine" in self.reasoning_modules:
                revised_plan = self.reasoning_modules["planning_engine"].revise_plan(
                    goal_id,
                    task.parameters.get("reason"),
                    task.parameters.get("constraints", {})
                )
                task.result = {"revised_plan": revised_plan}

        else:
            raise ValueError(f"Unknown goal task action: {action}")

    def _evaluate_goal_progress(self, goal_id: str) -> Dict[str, Any]:
        """
        Evaluate progress on a goal.

        Args:
            goal_id: ID of the goal to evaluate

        Returns:
            progress_info: Dictionary with progress information
        """
        # Find the goal
        goal = None
        with self.lock:
            for g in self.cognitive_state.active_goals:
                if g["goal_id"] == goal_id:
                    goal = g
                    break

        if goal is None:
            return {"error": f"Goal not found: {goal_id}"}

        # Check if goal has a plan
        if "planning_engine" in self.reasoning_modules:
            plan = self.reasoning_modules["planning_engine"].get_plan(goal_id)

            if plan:
                # Calculate progress based on completed steps
                completed_steps = sum(1 for step in plan.get("steps", [])
                                    if step.get("status") == "completed")
                total_steps = len(plan.get("steps", []))

                if total_steps > 0:
                    progress_pct = (completed_steps / total_steps) * 100
                else:
                    progress_pct = 0.0

                return {
                    "goal_id": goal_id,
                    "has_plan": True,
                    "completed_steps": completed_steps,
                    "total_steps": total_steps,
                    "progress_pct": progress_pct,
                    "status": goal.get("status", "active")
                }

        # No plan available
        return {
            "goal_id": goal_id,
            "has_plan": False,
            "status": goal.get("status", "active")
        }

    def _state_update_loop(self) -> None:
        """Loop to periodically update the cognitive state"""
        self.logger.info("State update loop started")

        while not self.stop_threads:
            try:
                # Update the cognitive state
                self._update_cognitive_state()

                # Sleep for a short time
                time.sleep(0.1)

            except Exception as e:
                self.logger.error(f"Error in state update loop: {str(e)}")
                time.sleep(0.5)  # Sleep longer on error

        self.logger.info("State update loop stopped")

    def _update_cognitive_state(self) -> None:
        """Update the cognitive state based on current system state"""
        with self.lock:
            # Update working load based on queue lengths
            queue_size = len(self.event_queue) + len(self.task_queue) + len(self.active_tasks)
            max_queue_size = 200  # Arbitrary maximum for normalization

            self.cognitive_state.working_load = min(1.0, queue_size / max_queue_size)

            # Update arousal based on recent activity and importance
            recent_activity = len(self.cognitive_state.recent_events)
            avg_importance = (sum(goal.get("priority", 0.5) for goal in self.cognitive_state.active_goals) /
                            max(1, len(self.cognitive_state.active_goals)))

            self.cognitive_state.arousal = 0.3 * (recent_activity / 20) + 0.7 * avg_importance

            # Decay old context activations
            for context in list(self.cognitive_state.active_contexts.keys()):
                self.cognitive_state.active_contexts[context] *= 0.95

                # Remove contexts with negligible activation
                if self.cognitive_state.active_contexts[context] < 0.05:
                    del self.cognitive_state.active_contexts[context]

            # Update uncertainty if confidence evaluator is available
            if "confidence_evaluator" in self.metacognition_modules:
                try:
                    # Get current uncertainty estimate
                    uncertainty = self.metacognition_modules["confidence_evaluator"].get_global_uncertainty()
                    self.cognitive_state.uncertainty = uncertainty
                except Exception as e:
                    self.logger.warning(f"Error getting uncertainty: {str(e)}")

    def _metacognitive_monitoring_loop(self) -> None:
        """Loop to perform periodic metacognitive monitoring"""
        self.logger.info("Metacognitive monitoring loop started")

        monitor_interval = 1.0  # seconds
        last_monitor_time = time.time()

        while not self.stop_threads:
            try:
                current_time = time.time()

                # Check if it's time to monitor
                if current_time - last_monitor_time >= monitor_interval:
                    self._perform_metacognitive_monitoring()
                    last_monitor_time = current_time

                # Sleep for a short time
                time.sleep(0.1)

            except Exception as e:
                self.logger.error(f"Error in metacognitive monitoring loop: {str(e)}")
                time.sleep(1.0)  # Sleep longer on error

        self.logger.info("Metacognitive monitoring loop stopped")

    def _perform_metacognitive_monitoring(self) -> None:
        """Perform metacognitive monitoring of the system"""
        try:
            # Skip if cognitive monitor is not available
            if "cognitive_monitor" not in self.metacognition_modules:
                return

            # Monitor active modules
            monitor_results = self.metacognition_modules["cognitive_monitor"].quick_scan()

            # Check for issues
            issues = monitor_results.get("issues", [])

            for issue in issues:
                severity = issue.get("severity", "info")

                if severity in ["warning", "error"]:
                    # Create metacognition event for this issue
                    event = CognitiveEvent(
                        event_type=CognitiveEventType.METACOGNITION,
                        source_module="cognitive_monitor",
                        content={"issue": issue},
                        priority=0.7 if severity == "error" else 0.5
                    )

                    with self.lock:
                        self.event_queue.append(event)

                    # Log the issue
                    self.logger.warning(f"Metacognitive issue detected: {issue}")

            # Update cognitive state with monitoring results
            with self.lock:
                self.cognitive_state.metadata["last_monitoring"] = {
                    "timestamp": time.time(),
                    "summary": monitor_results.get("summary", {})
                }

        except Exception as e:
            self.logger.error(f"Error in metacognitive monitoring: {str(e)}")

    def _estimate_task_progress(self, task: CognitiveTask) -> float:
        """
        Estimate the progress of a task.

        Args:
            task: The task to estimate progress for

        Returns:
            progress: Estimated progress as a fraction (0.0-1.0)
        """
        # For now, just use a simple heuristic based on task type and time elapsed
        if task.status == "completed":
            return 1.0
        elif task.status == "failed":
            return 0.0
        elif task.status == "pending":
            return 0.0

        # For running tasks, estimate based on elapsed time
        time_elapsed = time.time() - task.created_at

        # Estimate expected duration based on task type
        if task.task_type == "perception":
            expected_duration = 2.0  # seconds
        elif task.task_type == "reasoning":
            expected_duration = 5.0
        elif task.task_type == "learning":
            expected_duration = 10.0
        elif task.task_type == "metacognition":
            expected_duration = 3.0
        elif task.task_type == "goal":
            expected_duration = 7.0
        else:
            expected_duration = 5.0

        # Calculate progress fraction
        progress = min(0.95, time_elapsed / expected_duration)

        return progress

# Perception Core

In [ ]:
"""
perception_core.py - Core Perception Module for GrandNexus

This module provides the foundation for perception capabilities in GrandNexus.
It processes incoming data, interpreting and structuring different types of inputs
including text, mathematical expressions, code, structured data, and conceptual relationships.

The perception module serves as the initial layer of cognitive processing,
transforming raw inputs into structured representations that can be utilized
by reasoning, learning, and memory systems.

Key features:
- Multi-modal input processing
- Context-aware perception filters
- Pattern recognition in various data types
- Symbolic parsing capabilities
- Integration with memory for context enrichment
"""

import logging
import time
import uuid
import re
import json
from typing import Dict, List, Any, Optional, Union, Tuple, Set, Type, Callable
from dataclasses import dataclass, field
from enum import Enum, auto
import threading

# Will be imported from other modules
from grandnexus.cognition.cognition_manager import (
    CognitionManager, CognitiveModuleBase, CognitiveProcessType,
    CognitiveMode, CognitiveContext, CognitiveRequest
)
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.llm_interface.llm_engine import LLMEngine


class InputType(Enum):
    """Types of inputs that the perception module can process"""
    TEXT = auto()             # Natural language text
    MATHEMATICAL = auto()      # Mathematical expressions, formulas, equations
    CODE = auto()             # Programming code in various languages
    STRUCTURED_DATA = auto()  # JSON, XML, CSV, etc.
    GRAPH = auto()            # Graph structures, relationships, networks
    CONCEPT = auto()          # Conceptual or abstract notions
    MIXED = auto()            # Mixed or ambiguous input types
    UNKNOWN = auto()          # Unrecognized input


class PerceptionFilter(Enum):
    """Filters that can be applied during perception"""
    NONE = auto()             # No filtering
    RELEVANCE = auto()        # Filter by relevance to current context
    NOVELTY = auto()          # Filter by novelty (emphasis on new information)
    SALIENCE = auto()         # Filter by saliency (emphasis on important features)
    ABSTRACTION = auto()       # Filter by level of abstraction
    TEMPORAL = auto()          # Filter by recency or temporal relevance
    SEMANTIC = auto()          # Filter by semantic meaning or category


@dataclass
class PerceptionResult:
    """Result of a perception process"""
    result_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    input_type: InputType = InputType.UNKNOWN
    parsed_data: Any = None
    structured_representation: Any = None
    detected_patterns: List[Dict[str, Any]] = field(default_factory=list)
    confidence: float = 1.0
    metadata: Dict[str, Any] = field(default_factory=dict)
    processing_time: float = 0.0
    errors: List[str] = field(default_factory=list)


class PerceptionCore(CognitiveModuleBase):
    """
    Core class for the perception module in GrandNexus.

    Provides capabilities for processing inputs of different types,
    extracting structured information, and preparing it for higher
    cognitive functions like reasoning and learning.
    """

    def __init__(self,
                 cognition_manager: Optional[CognitionManager] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 llm_engine: Optional[LLMEngine] = None):
        """
        Initialize the Perception Core.

        Args:
            cognition_manager: Reference to the CognitionManager
            working_memory: Reference to the WorkingMemory module
            semantic_memory: Reference to the SemanticMemory module
            llm_engine: Reference to the LLM interface
        """
        self.cognition_manager = cognition_manager
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.llm_engine = llm_engine

        # Logger setup
        self.logger = logging.getLogger("PerceptionCore")

        # Internal state
        self.lock = threading.RLock()
        self.input_type_detectors: Dict[InputType, Callable] = {}
        self.input_type_parsers: Dict[InputType, Dict[str, Callable]] = {
            input_type: {} for input_type in InputType
        }
        self.perception_filters: Dict[PerceptionFilter, Callable] = {}

        # Initialize default detectors and parsers
        self._initialize_default_components()

        self.logger.info("PerceptionCore initialized")

    def _initialize_default_components(self) -> None:
        """Initialize default input type detectors and parsers"""
        # Register input type detectors
        self.register_input_type_detector(InputType.TEXT, self._detect_text_input)
        self.register_input_type_detector(InputType.MATHEMATICAL, self._detect_mathematical_input)
        self.register_input_type_detector(InputType.CODE, self._detect_code_input)
        self.register_input_type_detector(InputType.STRUCTURED_DATA, self._detect_structured_data)

        # Register input parsers
        self.register_input_parser(InputType.TEXT, "default", self._parse_text_input)
        self.register_input_parser(InputType.MATHEMATICAL, "default", self._parse_mathematical_input)
        self.register_input_parser(InputType.CODE, "default", self._parse_code_input)
        self.register_input_parser(InputType.STRUCTURED_DATA, "default", self._parse_structured_data)

        # Register perception filters
        self.register_perception_filter(PerceptionFilter.RELEVANCE, self._filter_by_relevance)
        self.register_perception_filter(PerceptionFilter.NOVELTY, self._filter_by_novelty)

    def register_input_type_detector(self, input_type: InputType, detector_func: Callable) -> None:
        """
        Register a function to detect a specific input type.

        Args:
            input_type: The input type to detect
            detector_func: Function that returns confidence (0-1) of input being the specified type
        """
        with self.lock:
            self.input_type_detectors[input_type] = detector_func
            self.logger.debug(f"Registered detector for {input_type.name}")

    def register_input_parser(self, input_type: InputType, parser_name: str, parser_func: Callable) -> None:
        """
        Register a parser for a specific input type.

        Args:
            input_type: The input type to parse
            parser_name: Name of the parser (allows multiple parsers per input type)
            parser_func: Function that parses input of the specified type
        """
        with self.lock:
            self.input_type_parsers[input_type][parser_name] = parser_func
            self.logger.debug(f"Registered parser '{parser_name}' for {input_type.name}")

    def register_perception_filter(self, filter_type: PerceptionFilter, filter_func: Callable) -> None:
        """
        Register a perception filter function.

        Args:
            filter_type: The type of filter
            filter_func: Function that applies the filter
        """
        with self.lock:
            self.perception_filters[filter_type] = filter_func
            self.logger.debug(f"Registered filter for {filter_type.name}")

    def unregister_input_type_detector(self, input_type: InputType) -> bool:
        """
        Unregister an input type detector.

        Args:
            input_type: The input type detector to remove

        Returns:
            bool: True if successful, False otherwise
        """
        with self.lock:
            if input_type in self.input_type_detectors:
                del self.input_type_detectors[input_type]
                self.logger.debug(f"Unregistered detector for {input_type.name}")
                return True
            return False

    def unregister_input_parser(self, input_type: InputType, parser_name: str) -> bool:
        """
        Unregister an input parser.

        Args:
            input_type: The input type
            parser_name: Name of the parser to remove

        Returns:
            bool: True if successful, False otherwise
        """
        with self.lock:
            if input_type in self.input_type_parsers and parser_name in self.input_type_parsers[input_type]:
                del self.input_type_parsers[input_type][parser_name]
                self.logger.debug(f"Unregistered parser '{parser_name}' for {input_type.name}")
                return True
            return False

    def unregister_perception_filter(self, filter_type: PerceptionFilter) -> bool:
        """
        Unregister a perception filter.

        Args:
            filter_type: The filter type to remove

        Returns:
            bool: True if successful, False otherwise
        """
        with self.lock:
            if filter_type in self.perception_filters:
                del self.perception_filters[filter_type]
                self.logger.debug(f"Unregistered filter for {filter_type.name}")
                return True
            return False

    @CognitionManager.cognitive_capability(
        capability_type=CognitiveProcessType.PERCEPTION,
        description="Detect input type and structure",
        priority=50,
        compatible_modes={CognitiveMode.REACTIVE, CognitiveMode.DELIBERATIVE}
    )
    def perceive_input(self,
                     request: CognitiveRequest,
                     context: CognitiveContext,
                     query: Any,
                     **kwargs) -> Dict[str, Any]:
        """
        Main perception capability to process input data.

        This function performs the following steps:
        1. Determine the type of input
        2. Parse the input according to its type
        3. Apply any filters based on context
        4. Return structured results

        Args:
            request: The cognitive request
            context: Cognitive context
            query: The input data to process
            **kwargs: Additional arguments

        Returns:
            Dict[str, Any]: Perception results
        """
        start_time = time.time()

        try:
            self.logger.info(f"Processing perception request for query type: {type(query)}")

            # Create a perception result object
            result = PerceptionResult()

            # If input is not a string, convert to string or handle specially
            if not isinstance(query, str):
                if isinstance(query, dict):
                    # Handle dictionary input
                    result.input_type = InputType.STRUCTURED_DATA
                    result.parsed_data = query
                    result.structured_representation = query
                elif isinstance(query, (list, tuple)):
                    # Handle list/tuple input
                    result.input_type = InputType.STRUCTURED_DATA
                    result.parsed_data = query
                    result.structured_representation = {"items": query}
                else:
                    # Convert to string
                    query = str(query)
                    # Then determine type
                    result.input_type = self._determine_input_type(query)
            else:
                # Determine the type of input
                result.input_type = self._determine_input_type(query)

            # Parse the input based on its type
            result = self._parse_input(query, result)

            # Apply filters if specified in context
            if "perception_filters" in context.parameters:
                filters = context.parameters["perception_filters"]
                result = self._apply_filters(result, filters, context)

            # Detect patterns in the input
            result.detected_patterns = self._detect_patterns(query, result)

            # Update processing time
            end_time = time.time()
            result.processing_time = end_time - start_time

            # Convert result to dictionary for return
            return self._perception_result_to_dict(result)

        except Exception as e:
            self.logger.error(f"Error in perceive_input: {e}", exc_info=True)
            return {
                "error": str(e),
                "input_type": InputType.UNKNOWN.name,
                "processing_time": time.time() - start_time
            }

    def _determine_input_type(self, input_data: Any) -> InputType:
        """
        Determine the type of input data.

        Args:
            input_data: The input data to analyze

        Returns:
            InputType: The detected input type
        """
        type_confidences = {}

        for input_type, detector in self.input_type_detectors.items():
            try:
                confidence = detector(input_data)
                type_confidences[input_type] = confidence
            except Exception as e:
                self.logger.debug(f"Error in detector for {input_type.name}: {e}")
                type_confidences[input_type] = 0.0

        # If no detector returned a positive confidence, default to TEXT
        if not type_confidences or max(type_confidences.values()) <= 0.2:
            return InputType.TEXT

        # Return the type with highest confidence
        return max(type_confidences.items(), key=lambda x: x[1])[0]

    def _parse_input(self, input_data: Any, result: PerceptionResult) -> PerceptionResult:
        """
        Parse the input data based on its detected type.

        Args:
            input_data: The input data to parse
            result: The current perception result

        Returns:
            PerceptionResult: Updated perception result
        """
        input_type = result.input_type

        # If we don't have any parsers for this type, leave as is
        if input_type not in self.input_type_parsers or not self.input_type_parsers[input_type]:
            result.parsed_data = input_data
            return result

        # Get default parser for this input type
        parser = self.input_type_parsers[input_type].get("default")

        if not parser:
            # If no default parser, use the first available
            parser = next(iter(self.input_type_parsers[input_type].values()))

        try:
            # Parse the input
            parsed_data = parser(input_data)
            result.parsed_data = parsed_data

            # Create structured representation
            if input_type == InputType.TEXT:
                result.structured_representation = {"text": parsed_data}
            elif input_type == InputType.MATHEMATICAL:
                result.structured_representation = {"expression": parsed_data}
            elif input_type == InputType.CODE:
                result.structured_representation = parsed_data
            elif input_type == InputType.STRUCTURED_DATA:
                result.structured_representation = parsed_data
            else:
                result.structured_representation = {"data": parsed_data}

        except Exception as e:
            self.logger.error(f"Error parsing input of type {input_type.name}: {e}", exc_info=True)
            result.errors.append(f"Parsing error: {str(e)}")
            result.confidence = 0.5  # Reduce confidence due to parsing error

        return result

    def _apply_filters(self,
                     result: PerceptionResult,
                     filters: List[Union[PerceptionFilter, str]],
                     context: CognitiveContext) -> PerceptionResult:
        """
        Apply perception filters to the result.

        Args:
            result: The perception result to filter
            filters: List of filters to apply
            context: The cognitive context

        Returns:
            PerceptionResult: Filtered perception result
        """
        for filter_item in filters:
            # Convert string to enum if needed
            if isinstance(filter_item, str):
                try:
                    filter_item = PerceptionFilter[filter_item.upper()]
                except KeyError:
                    self.logger.warning(f"Unknown filter type: {filter_item}")
                    continue

            # Get filter function
            filter_func = self.perception_filters.get(filter_item)

            if filter_func:
                try:
                    result = filter_func(result, context)
                except Exception as e:
                    self.logger.error(f"Error applying filter {filter_item.name}: {e}", exc_info=True)

        return result

    def _detect_patterns(self, input_data: Any, result: PerceptionResult) -> List[Dict[str, Any]]:
        """
        Detect patterns in the input data.

        Args:
            input_data: The original input data
            result: The current perception result

        Returns:
            List[Dict[str, Any]]: Detected patterns
        """
        patterns = []

        # Simple pattern detection based on input type
        if result.input_type == InputType.TEXT:
            # Detect URLs
            url_pattern = re.compile(r'https?://\S+')
            urls = url_pattern.findall(input_data) if isinstance(input_data, str) else []
            if urls:
                patterns.append({
                    "pattern_type": "url",
                    "instances": urls,
                    "confidence": 1.0
                })

            # Detect email addresses
            email_pattern = re.compile(r'\S+@\S+\.\S+')
            emails = email_pattern.findall(input_data) if isinstance(input_data, str) else []
            if emails:
                patterns.append({
                    "pattern_type": "email",
                    "instances": emails,
                    "confidence": 1.0
                })

        elif result.input_type == InputType.CODE:
            # Detect function definitions (simplified)
            if isinstance(input_data, str):
                function_pattern = re.compile(r'def\s+(\w+)\s*\(')
                functions = function_pattern.findall(input_data)
                if functions:
                    patterns.append({
                        "pattern_type": "function_definition",
                        "instances": functions,
                        "confidence": 0.9
                    })

        # Add more specialized pattern detection in future implementations

        return patterns

    def _perception_result_to_dict(self, result: PerceptionResult) -> Dict[str, Any]:
        """
        Convert a PerceptionResult to a dictionary.

        Args:
            result: The perception result to convert

        Returns:
            Dict[str, Any]: Dictionary representation of the result
        """
        return {
            "result_id": result.result_id,
            "input_type": result.input_type.name,
            "parsed_data": result.parsed_data,
            "structured_representation": result.structured_representation,
            "detected_patterns": result.detected_patterns,
            "confidence": result.confidence,
            "metadata": result.metadata,
            "processing_time": result.processing_time,
            "errors": result.errors
        }

    #------------------------------------------------------------------
    # Input type detectors
    #------------------------------------------------------------------

    def _detect_text_input(self, input_data: Any) -> float:
        """
        Detect if input is natural language text.

        Args:
            input_data: The input data to analyze

        Returns:
            float: Confidence that input is text (0-1)
        """
        if not isinstance(input_data, str):
            return 0.0

        # Text is our default, but we can still apply some heuristics

        # Check for structured data markers
        if input_data.strip().startswith(("{", "[", "<")) and input_data.strip().endswith(("}", "]", ">")):
            # Might be JSON, XML, etc.
            return 0.3

        # Check for code markers
        code_indicators = ["def ", "class ", "function", "import ", "from ", "#include", "public class"]
        if any(indicator in input_data for indicator in code_indicators):
            return 0.4

        # Check for mathematical expressions
        math_symbols = ["=", "+", "-", "*", "/", "^", "\\frac", "\\sum", "\\int"]
        math_symbol_count = sum(1 for symbol in math_symbols if symbol in input_data)
        if math_symbol_count > 3:
            return 0.4

        # Default confidence for text
        return 0.8

    def _detect_mathematical_input(self, input_data: Any) -> float:
        """
        Detect if input contains mathematical expressions.

        Args:
            input_data: The input data to analyze

        Returns:
            float: Confidence that input is mathematical (0-1)
        """
        if not isinstance(input_data, str):
            return 0.0

        # Check for LaTeX math delimiters
        if "\\begin{equation}" in input_data or "\\begin{align}" in input_data:
            return 0.9

        if "$" in input_data and input_data.count("$") >= 2:
            return 0.8

        # Check for common math operators and patterns
        math_operators = ["+", "-", "*", "/", "=", "<", ">", "±", "≤", "≥", "≠", "≈", "∫", "∑"]
        math_functions = ["sin", "cos", "tan", "log", "ln", "exp", "sqrt", "max", "min"]

        # Count math operators and functions
        operator_count = sum(input_data.count(op) for op in math_operators)
        function_count = sum(1 for func in math_functions if func in input_data)

        # Check for equation patterns
        equation_patterns = ["=", "equation", "formula", "solve", "compute"]
        has_equation_pattern = any(pattern in input_data.lower() for pattern in equation_patterns)

        # Calculate confidence score
        score = 0.0
        if has_equation_pattern:
            score += 0.3

        # Add score based on operator and function density
        if len(input_data) > 0:
            op_density = operator_count / len(input_data)
            score += min(0.5, op_density * 10)  # Cap at 0.5 for operator density
            score += min(0.3, function_count * 0.1)  # 0.1 per function, up to 0.3

        return min(0.95, score)  # Cap at 0.95

    def _detect_code_input(self, input_data: Any) -> float:
        """
        Detect if input is programming code.

        Args:
            input_data: The input data to analyze

        Returns:
            float: Confidence that input is code (0-1)
        """
        if not isinstance(input_data, str):
            return 0.0

        # Common code indicators
        code_keywords = [
            "function", "def ", "class ", "if ", "else", "for ", "while ", "return", "import ",
            "from ", "var ", "let ", "const ", "public", "private", "protected", "#include",
            "package", "namespace", "using "
        ]

        code_symbols = ["{}", "[]", "()", ";", "=>", "->", "<<", ">>", "===", "!==", "&&", "||"]

        # Count occurrences
        keyword_count = sum(input_data.count(keyword) for keyword in code_keywords)
        symbol_count = sum(input_data.count(symbol[0]) for symbol in code_symbols if len(symbol) == 1)

        # Check for indentation patterns
        indentation_pattern = re.compile(r'^( {2,}|\t+)', re.MULTILINE)
        indentation_matches = indentation_pattern.findall(input_data)

        # Check for common code structures
        function_def = re.search(r'(function|def|public|private)\s+\w+\s*\(', input_data)
        class_def = re.search(r'(class|interface|struct|enum)\s+\w+', input_data)
        import_statement = re.search(r'(import|from|#include|using)\s+[\w\.]+', input_data)

        # Calculate confidence score
        score = 0.0

        if function_def or class_def:
            score += 0.4

        if import_statement:
            score += 0.3

        if len(indentation_matches) > 2:
            score += 0.2

        # Add score based on keyword and symbol density
        if len(input_data) > 0:
            keyword_factor = min(0.3, keyword_count * 0.05)  # 0.05 per keyword, up to 0.3
            symbol_factor = min(0.2, symbol_count * 0.02)   # 0.02 per symbol, up to 0.2
            score += keyword_factor + symbol_factor

        return min(0.95, score)  # Cap at 0.95

    def _detect_structured_data(self, input_data: Any) -> float:
        """
        Detect if input is structured data (JSON, XML, etc.).

        Args:
            input_data: The input data to analyze

        Returns:
            float: Confidence that input is structured data (0-1)
        """
        # If it's already a dict or list, it's definitely structured data
        if isinstance(input_data, (dict, list)):
            return 1.0

        if not isinstance(input_data, str):
            return 0.0

        input_data = input_data.strip()

        # Check for JSON format
        if (input_data.startswith("{") and input_data.endswith("}")) or (input_data.startswith("[") and input_data.endswith("]")):
            try:
                json.loads(input_data)
                return 0.95  # High confidence for valid JSON
            except json.JSONDecodeError:
                # Looks like JSON but isn't valid
                return 0.5

        # Check for XML format
        if input_data.startswith("<") and input_data.endswith(">"):
            if input_data.startswith("<?xml") or ("</" in input_data and ">" in input_data):
                return 0.9

        # Check for CSV-like format
        if "," in input_data:
            lines = input_data.split("\n")
            if len(lines) > 1:
                comma_counts = [line.count(",") for line in lines if line.strip()]
                if len(comma_counts) > 1 and len(set(comma_counts[:min(5, len(comma_counts))])) <= 1:
                    # Consistent number of commas in first few lines suggests CSV
                    return 0.8

        # Check for key-value pairs
        kv_pattern = re.compile(r'^\s*[\w-]+\s*[:=]\s*.+$', re.MULTILINE)
        kv_matches = kv_pattern.findall(input_data)
        if len(kv_matches) > 2 and len(kv_matches) / input_data.count("\n") > 0.5:
            return 0.7  # Looks like key-value configuration format

        return 0.1  # Low confidence

    #------------------------------------------------------------------
    # Input parsers
    #------------------------------------------------------------------

    def _parse_text_input(self, input_data: Any) -> Any:
        """
        Parse natural language text input.

        Args:
            input_data: The text input to parse

        Returns:
            Any: Parsed text data
        """
        if not isinstance(input_data, str):
            return str(input_data)

        # For text, we might extract entities, keywords, etc.
        # For now, we'll just return the text itself with basic structure

        # Split into paragraphs
        paragraphs = [p.strip() for p in input_data.split("\n\n") if p.strip()]

        # Extract potential keywords (simple implementation)
        words = input_data.lower().split()
        # Filter common stopwords (very simplified)
        stopwords = {"the", "a", "an", "in", "on", "at", "to", "for", "of", "with", "by"}
        keywords = [word for word in words if word not in stopwords and len(word) > 3]
        # Count word frequencies
        word_counts = {}
        for word in keywords:
            word_counts[word] = word_counts.get(word, 0) + 1
        # Sort by frequency
        potential_keywords = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

        return {
            "text": input_data,
            "paragraphs": paragraphs,
            "potential_keywords": [k[0] for k in potential_keywords],
            "word_count": len(words),
            "paragraph_count": len(paragraphs)
        }

    def _parse_mathematical_input(self, input_data: Any) -> Any:
        """
        Parse mathematical expressions.

        Args:
            input_data: The mathematical input to parse

        Returns:
            Any: Parsed mathematical data
        """
        if not isinstance(input_data, str):
            return {"expression": str(input_data), "parsed": False}

        # Basic identification of expression type
        expression_type = "unknown"
        properties = {}

        # Check for equation
        if "=" in input_data:
            expression_type = "equation"
            sides = input_data.split("=", 1)
            properties["left_side"] = sides[0].strip()
            properties["right_side"] = sides[1].strip()

        # Check for function definition
        elif ":" in input_data or "->" in input_data:
            expression_type = "function_definition"
            # Try to identify function name and mapping
            if "->" in input_data:
                parts = input_data.split("->", 1)
                properties["domain"] = parts[0].strip()
                properties["codomain"] = parts[1].strip()
            elif ":" in input_data:
                parts = input_data.split(":", 1)
                properties["name"] = parts[0].strip()
                properties["definition"] = parts[1].strip()

        # Check for inequality
        elif any(op in input_data for op in ["<", ">", "≤", "≥", "≠"]):
            expression_type = "inequality"
            # Identify the operator
            for op in ["≤", "≥", "≠", "<", ">"]:
                if op in input_data:
                    sides = input_data.split(op, 1)
                    properties["left_side"] = sides[0].strip()
                    properties["right_side"] = sides[1].strip()
                    properties["operator"] = op
                    break

        # Check for expression (no equality/inequality)
        else:
            expression_type = "expression"
            properties["expression"] = input_data.strip()

        return {
            "expression": input_data,
            "type": expression_type,
            "properties": properties,
            "parsed": True
        }

    def _parse_code_input(self, input_data: Any) -> Any:
        """
        Parse programming code input.

        Args:
            input_data: The code input to parse

        Returns:
            Any: Parsed code data with structure information
        """
        if not isinstance(input_data, str):
            return {"code": str(input_data), "parsed": False}

        # Try to determine language
        language = self._detect_code_language(input_data)

        # Extract functions and classes (basic regex-based approach)
        functions = []
        classes = []
        imports = []

        # Extract function definitions
        if language in ["python", "unknown"]:
            # Python-style function definitions
            function_pattern = re.compile(r'def\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\(([^)]*)\)(?:\s*->?[^:]*)?:', re.MULTILINE)
            func_matches = function_pattern.findall(input_data)
            for name, params in func_matches:
                functions.append({
                    "name": name,
                    "parameters": [p.strip() for p in params.split(",") if p.strip()],
                    "language": language
                })

            # Python-style class definitions
            class_pattern = re.compile(r'class\s+([a-zA-Z_][a-zA-Z0-9_]*)(?:\s*\(([^)]*)\))?:', re.MULTILINE)
            class_matches = class_pattern.findall(input_data)
            for name, parents in class_matches:
                classes.append({
                    "name": name,
                    "parents": [p.strip() for p in parents.split(",") if p.strip()],
                    "language": language
                })

            # Python-style imports
            import_pattern = re.compile(r'(from\s+[a-zA-Z_][a-zA-Z0-9_.]+\s+import|import)\s+([^#\n]+)', re.MULTILINE)
            import_matches = import_pattern.findall(input_data)
            for import_type, import_names in import_matches:
                imports.append({
                    "statement": (import_type + " " + import_names).strip(),
                    "language": language
                })

        elif language == "javascript":
            # JavaScript-style function definitions (including ES6 arrow functions)
            function_pattern = re.compile(r'(?:function\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\(([^)]*)\)|(?:const|let|var)\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*=\s*(?:function\s*\(([^)]*)\)|(?:\(([^)]*)\)\s*=>)))', re.MULTILINE)
            func_matches = function_pattern.findall(input_data)
            for match in func_matches:
                if match[0]:  # Standard function
                    functions.append({
                        "name": match[0],
                        "parameters": [p.strip() for p in match[1].split(",") if p.strip()],
                        "language": language
                    })
                elif match[2]:  # Arrow function or function expression
                    params = match[3] if match[3] else match[4]
                    functions.append({
                        "name": match[2],
                        "parameters": [p.strip() for p in params.split(",") if p.strip()],
                        "language": language
                    })

            # JavaScript-style class definitions
            class_pattern = re.compile(r'class\s+([a-zA-Z_][a-zA-Z0-9_]*)(?:\s+extends\s+([a-zA-Z_][a-zA-Z0-9_.]*))?', re.MULTILINE)
            class_matches = class_pattern.findall(input_data)
            for name, parent in class_matches:
                parents = [parent] if parent else []
                classes.append({
                    "name": name,
                    "parents": parents,
                    "language": language
                })

            # JavaScript-style imports
            import_pattern = re.compile(r'import\s+(?:{[^}]+}|[a-zA-Z_][a-zA-Z0-9_]*)\s+from\s+[\'"][^\'")]+[\'"]', re.MULTILINE)
            import_matches = import_pattern.findall(input_data)
            for import_stmt in import_matches:
                imports.append({
                    "statement": import_stmt.strip(),
                    "language": language
                })

        # Count lines of code
        lines = input_data.split("\n")
        non_empty_lines = sum(1 for line in lines if line.strip())

        # Look for comments
        comment_patterns = {
            "python": r'#.*$',
            "javascript": r'(?://.*$|/\*[\s\S]*?\*/)',
            "unknown": r'(?://.*$|/\*[\s\S]*?\*/|#.*$)'
        }

        comment_pattern = re.compile(comment_patterns.get(language, comment_patterns["unknown"]), re.MULTILINE)
        comments = comment_pattern.findall(input_data)

        return {
            "code": input_data,
            "language": language,
            "functions": functions,
            "classes": classes,
            "imports": imports,
            "line_count": len(lines),
            "non_empty_line_count": non_empty_lines,
            "comment_count": len(comments),
            "parsed": True
        }

    def _detect_code_language(self, code: str) -> str:
        """
        Attempt to detect the programming language of a code snippet.

        Args:
            code: The code snippet

        Returns:
            str: Detected language or "unknown"
        """
        # Simple heuristics for language detection
        if re.search(r'def\s+\w+\s*\(|import\s+\w+|from\s+\w+\s+import', code):
            return "python"
        elif re.search(r'(?:function\s+\w+|var\s+|let\s+|const\s+|=>|{|}|;$)', code):
            return "javascript"
        elif re.search(r'#include\s+<\w+\.h>|int\s+main\s*\(\s*\)', code):
            return "c"
        elif re.search(r'public\s+class|public\s+static\s+void\s+main|import\s+java\.\w+', code):
            return "java"
        else:
            return "unknown"

    def _parse_structured_data(self, input_data: Any) -> Any:
        """
        Parse structured data input (JSON, XML, etc.).

        Args:
            input_data: The structured data input to parse

        Returns:
            Any: Parsed structured data
        """
        # If already a dict or list, return as is
        if isinstance(input_data, (dict, list)):
            return input_data

        if not isinstance(input_data, str):
            return {"data": str(input_data), "parsed": False}

        input_data = input_data.strip()
        data_format = "unknown"
        parsed_data = None

        # Try to parse as JSON
        if (input_data.startswith("{") and input_data.endswith("}")) or (input_data.startswith("[") and input_data.endswith("]")):
            try:
                parsed_data = json.loads(input_data)
                data_format = "json"
            except json.JSONDecodeError:
                # Not valid JSON
                pass

        # Try to parse as CSV (simplified)
        if data_format == "unknown" and "," in input_data:
            lines = input_data.split("\n")
            if len(lines) > 1:
                # Check if the first line could be a header
                header = [h.strip() for h in lines[0].split(",")]
                if all(h for h in header):  # Non-empty header values
                    try:
                        parsed_data = []
                        for i, line in enumerate(lines[1:], 1):
                            if not line.strip():
                                continue
                            values = [v.strip() for v in line.split(",")]
                            if len(values) == len(header):
                                row = {header[j]: values[j] for j in range(len(header))}
                                parsed_data.append(row)
                            else:
                                # Inconsistent column count, not a valid CSV
                                parsed_data = None
                                break
                        if parsed_data:
                            data_format = "csv"
                    except Exception:
                        # Error during CSV parsing
                        parsed_data = None

        # Try to parse as key-value pairs
        if data_format == "unknown":
            lines = input_data.split("\n")
            kv_data = {}
            valid_kv = True
            for line in lines:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if ":" in line:
                    key, value = line.split(":", 1)
                    kv_data[key.strip()] = value.strip()
                elif "=" in line:
                    key, value = line.split("=", 1)
                    kv_data[key.strip()] = value.strip()
                else:
                    valid_kv = False
                    break
            if valid_kv and kv_data:
                parsed_data = kv_data
                data_format = "key-value"

        # If we couldn't parse it, just return the raw string
        if parsed_data is None:
            return {
                "data": input_data,
                "format": data_format,
                "parsed": False
            }

        return {
            "data": parsed_data,
            "format": data_format,
            "raw": input_data,
            "parsed": True
        }

    #------------------------------------------------------------------
    # Perception filters
    #------------------------------------------------------------------

    def _filter_by_relevance(self, result: PerceptionResult, context: CognitiveContext) -> PerceptionResult:
        """
        Filter perception result based on relevance to current context.

        Args:
            result: The perception result to filter
            context: The cognitive context

        Returns:
            PerceptionResult: Filtered perception result
        """
        # Simple relevance filtering, could be expanded significantly

        # If no context parameters or no structured representation, return as is
        if not context.parameters or not result.structured_representation:
            return result

        # If there's a current topic or focus in the context
        if "topic" in context.parameters:
            current_topic = context.parameters["topic"]

            # If dealing with text
            if result.input_type == InputType.TEXT and isinstance(result.parsed_data, dict):
                # Check if we have keywords and filter based on topic relevance
                if "potential_keywords" in result.parsed_data:
                    # Keep keywords potentially relevant to the topic
                    original_keywords = result.parsed_data["potential_keywords"]
                    topic_keywords = [k for k in original_keywords if self._is_related(k, current_topic)]

                    # Update the parsed data
                    if topic_keywords:
                        result.parsed_data["potential_keywords"] = topic_keywords
                        result.parsed_data["topic_filtered"] = True

                        # Set a relevance metadata score
                        result.metadata["relevance_score"] = len(topic_keywords) / len(original_keywords)

        return result

    def _is_related(self, term: str, topic: str) -> bool:
        """
        Simple helper to check if a term is related to a topic.
        Could be expanded with semantic similarity, knowledge graph, etc.

        Args:
            term: The term to check
            topic: The topic to check against

        Returns:
            bool: True if related, False otherwise
        """
        # Basic implementation - check if the term is contained in the topic or vice versa
        return term.lower() in topic.lower() or topic.lower() in term.lower()

    def _filter_by_novelty(self, result: PerceptionResult, context: CognitiveContext) -> PerceptionResult:
        """
        Filter perception result based on novelty (emphasize new information).

        Args:
            result: The perception result to filter
            context: The cognitive context

        Returns:
            PerceptionResult: Filtered perception result with novelty information
        """
        # If working memory is not available, we can't determine novelty
        if not self.working_memory:
            return result

        # Initialize novelty score as neutral
        novelty_score = 0.5

        # If we have text input with keywords
        if result.input_type == InputType.TEXT and isinstance(result.parsed_data, dict) and "potential_keywords" in result.parsed_data:
            keywords = result.parsed_data["potential_keywords"]

            # Check which keywords are new relative to working memory
            new_keywords = []
            for keyword in keywords:
                # Simplified check - a more sophisticated approach would use embeddings
                query_result = self.working_memory.retrieve(
                    query_text=keyword,
                    max_results=1
                )

                # If no results, it's potentially new
                if not query_result:
                    new_keywords.append(keyword)

            # Calculate novelty score based on proportion of new keywords
            if keywords:
                novelty_score = len(new_keywords) / len(keywords)

            # Add novelty information to result
            result.metadata["novelty_score"] = novelty_score
            result.metadata["new_keywords"] = new_keywords

        return result

    #------------------------------------------------------------------
    # Helper methods
    #------------------------------------------------------------------

    @CognitionManager.cognitive_capability(
        capability_type=CognitiveProcessType.PERCEPTION,
        description="Extract structured information from text",
        priority=60,
        compatible_modes={CognitiveMode.REACTIVE, CognitiveMode.DELIBERATIVE}
    )
    def extract_structured_info(self,
                              request: CognitiveRequest,
                              context: CognitiveContext,
                              query: Any,
                              **kwargs) -> Dict[str, Any]:
        """
        Extract structured information from text.
        This is a higher-level capability that builds on basic perception.

        Args:
            request: The cognitive request
            context: Cognitive context
            query: The input data to process
            **kwargs: Additional arguments

        Returns:
            Dict[str, Any]: Extracted structured information
        """
        # First perform basic perception
        perception_result = self.perceive_input(request, context, query, **kwargs)

        # If there was an error in basic perception, return it
        if "error" in perception_result:
            return perception_result

        # If input is not text, return the basic perception result
        if perception_result["input_type"] != "TEXT":
            return {
                "message": "Extraction of structured information is only supported for text input",
                "basic_perception": perception_result
            }

        # Extract structured information using more advanced techniques
        # In a full implementation, this could use NLP techniques or call a LLM
        structured_info = self._extract_structured_info_from_text(query)

        return {
            "structured_info": structured_info,
            "basic_perception": perception_result
        }

    def _extract_structured_info_from_text(self, text: str) -> Dict[str, Any]:
        """
        Extract structured information from text.
        This is a simplified implementation.

        Args:
            text: The text to analyze

        Returns:
            Dict[str, Any]: Extracted structured information
        """
        # Simple extraction of entities (people, locations, dates)
        entities = {
            "people": [],
            "locations": [],
            "dates": [],
            "organizations": []
        }

        # Very simple name detection (looking for capitalized words)
        name_pattern = re.compile(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b')
        potential_names = name_pattern.findall(text)

        # Simple date detection
        date_patterns = [
            r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',  # MM/DD/YYYY or DD/MM/YYYY
            r'\b\d{1,2}-\d{1,2}-\d{2,4}\b',  # MM-DD-YYYY or DD-MM-YYYY
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?,?\s+\d{2,4}\b'  # Month DD, YYYY
        ]

        dates = []
        for pattern in date_patterns:
            dates.extend(re.findall(pattern, text))

        # Attempt to categorize entities
        for name in potential_names:
            # This is a very simplistic approach; a real implementation would use NER
            if len(name.split()) >= 2:
                # Assume people usually have first and last names
                entities["people"].append(name)
            elif name in ["America", "USA", "Canada", "Mexico", "China", "Russia", "London", "Paris", "Tokyo"]:
                # Just a few example locations
                entities["locations"].append(name)
            elif "Inc" in name or "Corp" in name or "Company" in name:
                entities["organizations"].append(name)

        entities["dates"] = dates

        # Extract key phrases (very simplified)
        sentences = re.split(r'[.!?]+', text)
        key_phrases = []

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            # Look for sentences that might contain key information
            if any(indicator in sentence.lower() for indicator in [
                "important", "significant", "key", "critical", "essential",
                "main", "primary", "major", "crucial", "vital"
            ]):
                key_phrases.append(sentence)

        # Extract potential action items (very simplified)
        action_items = []
        action_indicators = ["need to", "should", "must", "have to", "required to"]

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            if any(indicator in sentence.lower() for indicator in action_indicators):
                action_items.append(sentence)

        return {
            "entities": entities,
            "key_phrases": key_phrases,
            "action_items": action_items
        }

    @CognitionManager.cognitive_capability(
        capability_type=CognitiveProcessType.PERCEPTION,
        description="Classify input content",
        priority=70,
        compatible_modes={CognitiveMode.DELIBERATIVE}
    )
    def classify_content(self,
                       request: CognitiveRequest,
                       context: CognitiveContext,
                       query: Any,
                       **kwargs) -> Dict[str, Any]:
        """
        Classify the content of input data.

        Args:
            request: The cognitive request
            context: Cognitive context
            query: The input data to classify
            **kwargs: Additional arguments

        Returns:
            Dict[str, Any]: Classification results
        """
        # First perform basic perception
        perception_result = self.perceive_input(request, context, query, **kwargs)

        # If there was an error in basic perception, return it
        if "error" in perception_result:
            return perception_result

        # Build classification based on input type
        classification = {
            "categories": [],
            "confidence": 0.0,
            "topic_areas": []
        }

        # Use LLM for classification if available
        if self.llm_engine:
            try:
                # Create a classification prompt
                if isinstance(query, str):
                    prompt = f"Classify the following content into categories and topics:\n\n{query[:1000]}\n\nProvide the response as JSON with 'categories' (list of category names) and 'topic_areas' (list of relevant topics)."

                    # Call LLM for classification
                    llm_response = self.llm_engine.complete({
                        "prompt": prompt,
                        "max_tokens": 200,
                        "temperature": 0.3
                    })

                    # Try to parse JSON response
                    try:
                        if hasattr(llm_response, 'text'):
                            llm_text = llm_response.text
                        else:
                            llm_text = str(llm_response)

                        # Extract JSON part if exists
                        json_match = re.search(r'```json\s*(.*?)\s*```', llm_text, re.DOTALL)
                        if json_match:
                            llm_result = json.loads(json_match.group(1))
                        else:
                            # Try to find JSON-like structure
                            json_match = re.search(r'({[^{]*"categories"[^}]*})', llm_text, re.DOTALL)
                            if json_match:
                                llm_result = json.loads(json_match.group(1))
                            else:
                                # Fallback to rule-based classification
                                raise ValueError("Could not parse LLM response as JSON")

                        # Update classification with LLM results
                        if "categories" in llm_result:
                            classification["categories"] = llm_result["categories"]
                        if "topic_areas" in llm_result:
                            classification["topic_areas"] = llm_result["topic_areas"]

                        classification["confidence"] = 0.8  # Higher confidence with LLM
                        classification["method"] = "llm"

                    except (json.JSONDecodeError, ValueError, AttributeError) as e:
                        self.logger.warning(f"Error parsing LLM classification response: {e}")
                        # Fall back to rule-based classification
                        classification.update(self._rule_based_classification(query, perception_result))
                else:
                    # Non-string input, use rule-based
                    classification.update(self._rule_based_classification(query, perception_result))
            except Exception as e:
                self.logger.error(f"Error in LLM classification: {e}")
                # Fall back to rule-based classification
                classification.update(self._rule_based_classification(query, perception_result))
        else:
            # No LLM available, use rule-based classification
            classification.update(self._rule_based_classification(query, perception_result))

        return {
            "classification": classification,
            "basic_perception": perception_result
        }

    def _rule_based_classification(self, query: Any, perception_result: Dict[str, Any]) -> Dict[str, Any]:
        """
        Perform rule-based classification of content.

        Args:
            query: The original query
            perception_result: Result of basic perception

        Returns:
            Dict[str, Any]: Classification results
        """
        classification = {
            "categories": [],
            "confidence": 0.5,  # Lower confidence for rule-based
            "topic_areas": [],
            "method": "rule-based"
        }

        input_type = perception_result.get("input_type", "UNKNOWN")

        # Classify based on input type
        if input_type == "TEXT":
            text = query if isinstance(query, str) else str(query)

            # Simple topic detection based on keywords
            topic_indicators = {
                "technology": ["computer", "software", "hardware", "tech", "digital", "internet", "web", "online", "code", "program"],
                "science": ["science", "scientific", "research", "experiment", "theory", "hypothesis", "lab", "study"],
                "business": ["business", "company", "market", "finance", "economic", "investment", "stock", "profit", "revenue"],
                "health": ["health", "medical", "doctor", "patient", "disease", "treatment", "medicine", "hospital", "clinic"],
                "education": ["education", "school", "student", "learn", "teach", "course", "class", "professor", "academic"],
                "politics": ["politic", "government", "election", "vote", "party", "policy", "law", "regulation", "president"]
            }

            # Check for topics
            text_lower = text.lower()
            for topic, keywords in topic_indicators.items():
                if any(keyword in text_lower for keyword in keywords):
                    classification["topic_areas"].append(topic)

            # Classify content type
            if len(text.split()) > 100:
                classification["categories"].append("long-form")
            else:
                classification["categories"].append("short-form")

            if "?" in text:
                classification["categories"].append("question")

            if len(text.split("\n")) > 5:
                classification["categories"].append("multi-paragraph")

        elif input_type == "CODE":
            classification["categories"].append("code")
            language = perception_result.get("parsed_data", {}).get("language", "unknown")
            if language != "unknown":
                classification["categories"].append(f"{language}-code")

        elif input_type == "MATHEMATICAL":
            classification["categories"].append("mathematical")
            math_type = perception_result.get("parsed_data", {}).get("type", "expression")
            classification["categories"].append(f"math-{math_type}")

        elif input_type == "STRUCTURED_DATA":
            classification["categories"].append("structured-data")
            data_format = perception_result.get("parsed_data", {}).get("format", "unknown")
            if data_format != "unknown":
                classification["categories"].append(f"{data_format}-format")

        return classification


# Standalone execution for testing
if __name__ == "__main__":
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Create a PerceptionCore instance
    perception = PerceptionCore()

    # Create a simple context for testing
    context = CognitiveContext(
        parameters={"perception_filters": [PerceptionFilter.RELEVANCE]}
    )

    # Test with different input types
    test_inputs = [
        "This is a simple text input for testing perception capabilities.",
        "x^2 + 3x + 2 = 0",
        "def hello_world():\n    print('Hello, world!')",
        '{"name": "John", "age": 30, "city": "New York"}'
    ]

    for input_data in test_inputs:
        request = CognitiveRequest(query=input_data, context=context)
        result = perception.perceive_input(request, context, input_data)
        print(f"\nPerception result for: {input_data[:30]}...")
        print(f"Input type: {result['input_type']}")
        print(f"Confidence: {result['confidence']}")
        if result['detected_patterns']:
            print(f"Detected patterns: {result['detected_patterns']}")
        print("-" * 40)

# Symbolic Parser

In [ ]:
"""
symbolic_parser.py - Symbolic Expression Parsing Module for GrandNexus

This module provides functionality for parsing and interpreting symbolic expressions
such as mathematical formulas, logical statements, and structured patterns. It serves
as a bridge between textual representations and formal computational structures.

The module supports:
- Mathematical expression parsing and evaluation
- Logical formula parsing and interpretation
- Custom DSL (Domain Specific Language) processing
- Structural pattern recognition in symbolic forms

Key design principles:
- Extensibility: Easy to add new symbolic languages and parsing strategies
- Progressive capacity: Start with simple expressions, evolve to handle complex ones
- Integration: Connect with semantic memory for concept recognition and lookup
- Emergence: Support discovery of patterns and structures in symbolic content
"""

import logging
import re
import uuid
import math
from typing import Any, Dict, List, Optional, Set, Tuple, Union, Callable
from enum import Enum, auto
from dataclasses import dataclass, field
import ast
import operator
import threading
from collections import defaultdict, deque

# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.working import WorkingMemory

# Import sympy with try/except to make it optional
try:
    import sympy
    from sympy.parsing.sympy_parser import parse_expr, standard_transformations
    from sympy import Symbol, symbols, solve, Eq, simplify
    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False

# Define symbolic expression types
class SymbolicType(Enum):
    """Types of symbolic expressions the parser can handle"""
    MATHEMATICAL = auto()    # Mathematical formulas, equations
    LOGICAL = auto()         # Logical propositions, boolean algebra
    CODE = auto()            # Code snippets or DSL expressions
    PATTERN = auto()         # Pattern expressions or templates
    CONSTRAINT = auto()      # Constraint expressions
    UNKNOWN = auto()         # Unable to determine type

@dataclass
class SymbolicAST:
    """Abstract Syntax Tree for parsed symbolic expressions"""
    node_type: str
    value: Any = None
    children: List["SymbolicAST"] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def traverse(self, callback: Callable[["SymbolicAST"], None]) -> None:
        """
        Traverse the AST, calling the callback for each node.

        Args:
            callback: Function to call on each node
        """
        callback(self)
        for child in self.children:
            child.traverse(callback)

    def to_dict(self) -> Dict[str, Any]:
        """Convert AST to dictionary representation"""
        result = {
            "node_type": self.node_type,
            "value": self.value,
            "metadata": self.metadata,
            "children": [child.to_dict() for child in self.children]
        }
        return result

    def __str__(self) -> str:
        """String representation of the AST node"""
        if not self.children:
            return f"{self.node_type}({self.value})"
        return f"{self.node_type}({self.value}, {len(self.children)} children)"

@dataclass
class ParseResult:
    """Result of parsing a symbolic expression"""
    success: bool
    expr_type: SymbolicType
    ast: Optional[SymbolicAST] = None
    sympy_expr: Optional[Any] = None  # SymPy expression if applicable
    python_ast: Optional[ast.AST] = None  # Python AST if applicable
    error_message: Optional[str] = None
    confidence: float = 1.0
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        """Initialize id and timestamp after creation"""
        self.id = str(uuid.uuid4())
        self.timestamp = field(default_factory=lambda: __import__('time').time())

    def is_valid(self) -> bool:
        """Check if parse result is valid and usable"""
        return self.success and (self.ast is not None or
                                 self.sympy_expr is not None or
                                 self.python_ast is not None)

    def to_dict(self) -> Dict[str, Any]:
        """Convert parse result to dictionary representation"""
        result = {
            "success": self.success,
            "expr_type": self.expr_type.name,
            "confidence": self.confidence,
            "metadata": self.metadata
        }

        if self.error_message:
            result["error_message"] = self.error_message

        if self.ast:
            result["ast"] = self.ast.to_dict()

        # Can't directly serialize sympy expressions or Python AST
        # Just indicate their presence
        if self.sympy_expr is not None:
            result["has_sympy_expr"] = True

        if self.python_ast is not None:
            result["has_python_ast"] = True

        return result

class SymbolicParser:
    """
    Core symbolic parser for GrandNexus, capable of understanding
    and interpreting various forms of symbolic expressions.

    This parser serves as an interface between textual/string representations
    and structured symbolic forms that can be manipulated, reasoned about,
    and integrated into GrandNexus's cognitive processes.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None):
        """
        Initialize the symbolic parser.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            semantic_memory: Reference to semantic memory for concept lookup
            working_memory: Reference to working memory for context
        """
        self.logger = logging.getLogger("GrandNexus.Perception.SymbolicParser")
        self.nexus_core = nexus_core
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Parsers registry for different symbolic types
        self.parsers = {}

        # Recognized symbol registry
        self.known_symbols = {}
        self.symbol_patterns = []

        # Cache recently parsed expressions
        self.expression_cache = {}
        self.expression_history = deque(maxlen=100)

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"SymbolicParser initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the parser and load parsers for different expression types"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register built-in parsers
                self._register_default_parsers()

                # Initialize symbol registry
                self._initialize_symbol_registry()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("SymbolicParser initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"SymbolicParser initialization failed: {str(e)}")
                return False

    def parse(self, expression: str, expected_type: Optional[SymbolicType] = None,
             context: Optional[Dict[str, Any]] = None) -> ParseResult:
        """
        Parse a symbolic expression into a structured form.

        Args:
            expression: The expression string to parse
            expected_type: Optional hint about the expected expression type
            context: Optional contextual information to aid parsing

        Returns:
            ParseResult containing the parsed expression
        """
        if not self.initialized:
            self.initialize()

        expression = expression.strip()

        # Check cache for identical expression
        cache_key = (expression, str(expected_type), str(sorted(context.items())) if context else None)
        if cache_key in self.expression_cache:
            return self.expression_cache[cache_key]

        # Prepare context if not provided
        if context is None:
            context = {}

        # Determine the expression type if not specified
        expr_type = expected_type if expected_type else self._detect_expression_type(expression, context)

        # Select the appropriate parser
        if expr_type in self.parsers:
            parser_func = self.parsers[expr_type]
            try:
                # Parse the expression
                result = parser_func(expression, context)
                # Update metrics
                self.metrics["parse_success_rate"].append(1.0 if result.success else 0.0)

                # Cache successful results
                if result.success:
                    self.expression_cache[cache_key] = result
                    self.expression_history.append(expression)

                return result
            except Exception as e:
                self.logger.error(f"Error parsing expression '{expression}': {str(e)}")
                self.metrics["parse_success_rate"].append(0.0)
                return ParseResult(
                    success=False,
                    expr_type=expr_type,
                    error_message=f"Parser error: {str(e)}",
                    confidence=0.0
                )
        else:
            self.logger.warning(f"No parser available for expression type: {expr_type}")
            self.metrics["parse_success_rate"].append(0.0)
            return ParseResult(
                success=False,
                expr_type=expr_type,
                error_message=f"No parser available for type: {expr_type}",
                confidence=0.0
            )

    def evaluate(self, parse_result: ParseResult,
                 variable_values: Optional[Dict[str, Any]] = None,
                 context: Optional[Dict[str, Any]] = None) -> Any:
        """
        Evaluate a parsed expression with given variable values.

        Args:
            parse_result: The parsed expression to evaluate
            variable_values: Dictionary mapping variable names to values
            context: Additional contextual information for evaluation

        Returns:
            The result of evaluation, or None if evaluation fails
        """
        if not parse_result.success:
            self.logger.warning("Cannot evaluate unsuccessful parse result")
            return None

        # Prepare variable values if not provided
        if variable_values is None:
            variable_values = {}

        # Prepare context if not provided
        if context is None:
            context = {}

        try:
            # Handle different expression types
            if parse_result.expr_type == SymbolicType.MATHEMATICAL:
                return self._evaluate_mathematical(parse_result, variable_values, context)
            elif parse_result.expr_type == SymbolicType.LOGICAL:
                return self._evaluate_logical(parse_result, variable_values, context)
            elif parse_result.expr_type == SymbolicType.CODE:
                return self._evaluate_code(parse_result, variable_values, context)
            elif parse_result.expr_type == SymbolicType.PATTERN:
                return self._evaluate_pattern(parse_result, variable_values, context)
            else:
                self.logger.warning(f"Evaluation not supported for type: {parse_result.expr_type}")
                return None
        except Exception as e:
            self.logger.error(f"Error evaluating expression: {str(e)}")
            return None

    def transform(self, parse_result: ParseResult,
                 transformation: str,
                 context: Optional[Dict[str, Any]] = None) -> ParseResult:
        """
        Apply a transformation to a parsed expression.

        Args:
            parse_result: The parsed expression to transform
            transformation: The type of transformation to apply (e.g., "simplify", "factor")
            context: Additional contextual information for transformation

        Returns:
            A new ParseResult with the transformed expression
        """
        if not parse_result.success:
            self.logger.warning("Cannot transform unsuccessful parse result")
            return parse_result

        # Prepare context if not provided
        if context is None:
            context = {}

        try:
            # Handle different expression types
            if parse_result.expr_type == SymbolicType.MATHEMATICAL:
                return self._transform_mathematical(parse_result, transformation, context)
            elif parse_result.expr_type == SymbolicType.LOGICAL:
                return self._transform_logical(parse_result, transformation, context)
            else:
                self.logger.warning(f"Transformation not supported for type: {parse_result.expr_type}")
                return parse_result
        except Exception as e:
            self.logger.error(f"Error transforming expression: {str(e)}")
            return parse_result

    def find_symbols(self, expression: str) -> List[Dict[str, Any]]:
        """
        Extract symbols (variables, constants, functions) from an expression.

        Args:
            expression: The expression string to analyze

        Returns:
            List of dictionaries containing symbol information
        """
        # Parse the expression first
        parse_result = self.parse(expression)
        if not parse_result.success:
            return []

        symbols = []
        try:
            # Extract symbols based on expression type
            if parse_result.expr_type == SymbolicType.MATHEMATICAL and parse_result.sympy_expr is not None:
                # Extract symbols from sympy expression
                if HAS_SYMPY:
                    sympy_symbols = list(parse_result.sympy_expr.free_symbols)
                    symbols = [
                        {
                            "name": str(sym),
                            "type": "variable",
                            "metadata": {
                                "is_known": str(sym) in self.known_symbols
                            }
                        }
                        for sym in sympy_symbols
                    ]
            elif parse_result.ast is not None:
                # Extract symbols from AST by traversing
                found_symbols = set()

                def collect_symbols(node):
                    if node.node_type == "variable":
                        found_symbols.add(node.value)

                parse_result.ast.traverse(collect_symbols)

                symbols = [
                    {
                        "name": sym,
                        "type": "variable",
                        "metadata": {
                            "is_known": sym in self.known_symbols
                        }
                    }
                    for sym in found_symbols
                ]
        except Exception as e:
            self.logger.error(f"Error extracting symbols: {str(e)}")

        return symbols

    def register_symbol(self, name: str, symbol_type: str,
                       value: Optional[Any] = None,
                       metadata: Optional[Dict[str, Any]] = None) -> bool:
        """
        Register a known symbol for future reference.

        Args:
            name: The symbol name
            symbol_type: Type of symbol (variable, constant, function, etc.)
            value: Optional value or definition
            metadata: Additional information about the symbol

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            if metadata is None:
                metadata = {}

            self.known_symbols[name] = {
                "type": symbol_type,
                "value": value,
                "metadata": metadata
            }

            self.logger.info(f"Registered symbol: {name} ({symbol_type})")
            return True

    def register_parser(self, expr_type: SymbolicType,
                       parser_func: Callable[[str, Dict[str, Any]], ParseResult]) -> None:
        """
        Register a custom parser for a specific expression type.

        Args:
            expr_type: The type of expression this parser handles
            parser_func: Function that takes (expression, context) and returns ParseResult
        """
        with self.lock:
            self.parsers[expr_type] = parser_func
            self.logger.info(f"Registered parser for expression type: {expr_type}")

    def get_parsing_metrics(self) -> Dict[str, Any]:
        """Get current parsing performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_default_parsers(self) -> None:
        """Register built-in parsers for different expression types"""
        self.register_parser(SymbolicType.MATHEMATICAL, self._parse_mathematical)
        self.register_parser(SymbolicType.LOGICAL, self._parse_logical)
        self.register_parser(SymbolicType.CODE, self._parse_code)
        self.register_parser(SymbolicType.PATTERN, self._parse_pattern)
        self.register_parser(SymbolicType.CONSTRAINT, self._parse_constraint)

    def _initialize_symbol_registry(self) -> None:
        """Initialize registry with common mathematical and logical symbols"""
        # Mathematical constants
        self.register_symbol("pi", "constant", math.pi, {"description": "Ratio of circumference to diameter"})
        self.register_symbol("e", "constant", math.e, {"description": "Base of natural logarithm"})

        # Mathematical functions
        self.register_symbol("sin", "function", math.sin, {"description": "Sine function"})
        self.register_symbol("cos", "function", math.cos, {"description": "Cosine function"})
        self.register_symbol("tan", "function", math.tan, {"description": "Tangent function"})
        self.register_symbol("log", "function", math.log, {"description": "Natural logarithm"})
        self.register_symbol("exp", "function", math.exp, {"description": "Exponential function"})

        # Logical operators
        self.register_symbol("and", "operator", lambda x, y: x and y, {"description": "Logical AND"})
        self.register_symbol("or", "operator", lambda x, y: x or y, {"description": "Logical OR"})
        self.register_symbol("not", "operator", lambda x: not x, {"description": "Logical NOT"})

        # Common symbol patterns
        self.symbol_patterns = [
            (r'\b[a-zA-Z]\b', "single_letter_variable"),
            (r'\b[a-zA-Z][0-9]\b', "indexed_variable"),
            (r'\b[a-zA-Z][a-zA-Z0-9_]*\b', "identifier")
        ]

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.working_memory:
                dependencies.append("working_memory")

            self.nexus_core.register_module(
                name="symbolic_parser",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _detect_expression_type(self, expression: str, context: Dict[str, Any]) -> SymbolicType:
        """
        Determine the type of symbolic expression.

        Args:
            expression: The expression string to analyze
            context: Contextual information that might help with detection

        Returns:
            SymbolicType indicating the detected type
        """
        # Check context for explicit type hint
        if context and "expr_type" in context:
            hint = context["expr_type"]
            if isinstance(hint, SymbolicType):
                return hint
            elif isinstance(hint, str) and hasattr(SymbolicType, hint.upper()):
                return getattr(SymbolicType, hint.upper())

        # Apply heuristics to detect type

        # Check for mathematical expressions
        if re.search(r'[-+*/^=<>()]|\b(sin|cos|tan|log|exp)\b', expression):
            return SymbolicType.MATHEMATICAL

        # Check for logical expressions
        if re.search(r'\b(and|or|not|if|then|implies|forall|exists)\b|[∧∨¬⟹∀∃]', expression, re.IGNORECASE):
            return SymbolicType.LOGICAL

        # Check for code-like expressions
        if re.search(r'[;{}]|\b(function|def|return|for|while)\b', expression):
            return SymbolicType.CODE

        # Check for pattern expressions
        if re.search(r'[*?+]|\{\d+,\d*\}|\[\w+\]', expression):
            return SymbolicType.PATTERN

        # Default to mathematical if it contains common operators
        if re.search(r'[-+*/]', expression):
            return SymbolicType.MATHEMATICAL

        return SymbolicType.UNKNOWN

    def _parse_mathematical(self, expression: str, context: Dict[str, Any]) -> ParseResult:
        """
        Parse a mathematical expression.

        Args:
            expression: The expression string to parse
            context: Additional context for parsing

        Returns:
            ParseResult containing the parsed mathematical expression
        """
        # Use SymPy for advanced mathematical parsing if available
        if HAS_SYMPY:
            try:
                # Parse with sympy
                sympy_expr = parse_expr(expression, transformations=standard_transformations)

                # Create a basic AST representation
                ast_root = self._sympy_to_ast(sympy_expr)

                return ParseResult(
                    success=True,
                    expr_type=SymbolicType.MATHEMATICAL,
                    ast=ast_root,
                    sympy_expr=sympy_expr,
                    confidence=1.0
                )
            except Exception as e:
                self.logger.warning(f"SymPy parsing failed: {str(e)}, falling back to basic parsing")

        # Fall back to basic parsing if SymPy is not available or fails
        try:
            ast_root = self._basic_math_parse(expression)

            return ParseResult(
                success=True,
                expr_type=SymbolicType.MATHEMATICAL,
                ast=ast_root,
                confidence=0.8,  # Lower confidence for basic parsing
                metadata={"parser": "basic"}
            )
        except Exception as e:
            self.logger.error(f"Basic math parsing failed: {str(e)}")
            return ParseResult(
                success=False,
                expr_type=SymbolicType.MATHEMATICAL,
                error_message=f"Parsing error: {str(e)}",
                confidence=0.0
            )

    def _parse_logical(self, expression: str, context: Dict[str, Any]) -> ParseResult:
        """
        Parse a logical expression.

        Args:
            expression: The expression string to parse
            context: Additional context for parsing

        Returns:
            ParseResult containing the parsed logical expression
        """
        # Normalize logical operators
        normalized = expression.lower()
        normalized = normalized.replace("&&", " and ")
        normalized = normalized.replace("||", " or ")
        normalized = normalized.replace("!", "not ")

        try:
            # Try to parse as Python expression if it's simple enough
            try:
                # Convert to a Python boolean expression
                # This is a simplistic approach - a real implementation would use a proper logical parser
                python_expr = normalized
                python_expr = re.sub(r'\b(implies|⟹)\b', "<=", python_expr)
                python_expr = re.sub(r'\b(iff|⟺)\b', "==", python_expr)

                # Parse with Python's ast
                python_ast = ast.parse(python_expr, mode='eval')

                # Convert to our AST format
                ast_root = self._python_ast_to_symbolic_ast(python_ast)

                return ParseResult(
                    success=True,
                    expr_type=SymbolicType.LOGICAL,
                    ast=ast_root,
                    python_ast=python_ast,
                    confidence=0.9,
                    metadata={"parser": "python_ast"}
                )
            except SyntaxError:
                # Fall back to basic parsing
                raise ValueError("Not parsable as Python expression")

            # A real implementation would have a proper logical expression parser here
            # For now, we'll just create a basic AST for demonstration

        except Exception as e:
            self.logger.error(f"Logical parsing failed: {str(e)}")

            # Create a very basic AST even if parsing fails
            # In a real implementation, this would be more sophisticated
            ast_root = SymbolicAST(node_type="expression", value=expression)

            return ParseResult(
                success=True,  # Let's say it succeeds but with low confidence
                expr_type=SymbolicType.LOGICAL,
                ast=ast_root,
                confidence=0.5,
                metadata={"parser": "basic", "warning": str(e)}
            )

    def _parse_code(self, expression: str, context: Dict[str, Any]) -> ParseResult:
        """
        Parse a code expression or snippet.

        Args:
            expression: The code string to parse
            context: Additional context for parsing

        Returns:
            ParseResult containing the parsed code
        """
        try:
            # Try to parse as Python code
            python_ast = ast.parse(expression)

            # Convert to our AST format
            ast_root = self._python_ast_to_symbolic_ast(python_ast)

            return ParseResult(
                success=True,
                expr_type=SymbolicType.CODE,
                ast=ast_root,
                python_ast=python_ast,
                confidence=0.9,
                metadata={"language": "python"}
            )
        except SyntaxError as e:
            # Not valid Python code
            self.logger.info(f"Code is not valid Python: {str(e)}")

            # For other languages, we would have additional parsers
            # For now, just create a generic code AST
            ast_root = SymbolicAST(node_type="code", value=expression)

            return ParseResult(
                success=True,
                expr_type=SymbolicType.CODE,
                ast=ast_root,
                confidence=0.5,
                metadata={"language": "unknown"}
            )
        except Exception as e:
            self.logger.error(f"Code parsing failed: {str(e)}")
            return ParseResult(
                success=False,
                expr_type=SymbolicType.CODE,
                error_message=f"Parsing error: {str(e)}",
                confidence=0.0
            )

    def _parse_pattern(self, expression: str, context: Dict[str, Any]) -> ParseResult:
        """
        Parse a pattern expression.

        Args:
            expression: The pattern string to parse
            context: Additional context for parsing

        Returns:
            ParseResult containing the parsed pattern
        """
        try:
            # Check if it looks like a regex pattern
            if re.search(r'[*+?{}()\[\]\|\.]', expression):
                # Try to compile as regex to validate
                try:
                    re.compile(expression)
                    pattern_type = "regex"
                    confidence = 0.9
                except re.error:
                    pattern_type = "invalid_regex"
                    confidence = 0.3
            else:
                # Treat as a simple glob-like pattern
                pattern_type = "glob"
                confidence = 0.7

            # Create a basic AST for the pattern
            ast_root = SymbolicAST(
                node_type="pattern",
                value=expression,
                metadata={"pattern_type": pattern_type}
            )

            return ParseResult(
                success=True,
                expr_type=SymbolicType.PATTERN,
                ast=ast_root,
                confidence=confidence,
                metadata={"pattern_type": pattern_type}
            )
        except Exception as e:
            self.logger.error(f"Pattern parsing failed: {str(e)}")
            return ParseResult(
                success=False,
                expr_type=SymbolicType.PATTERN,
                error_message=f"Parsing error: {str(e)}",
                confidence=0.0
            )

    def _parse_constraint(self, expression: str, context: Dict[str, Any]) -> ParseResult:
        """
        Parse a constraint expression.

        Args:
            expression: The constraint string to parse
            context: Additional context for parsing

        Returns:
            ParseResult containing the parsed constraint
        """
        # Constraints are often mathematical equalities or inequalities
        if re.search(r'[<>=]', expression):
            # Try to parse as mathematical expression first
            math_result = self._parse_mathematical(expression, context)
            if math_result.success:
                # It's a mathematical constraint
                math_result.expr_type = SymbolicType.CONSTRAINT
                math_result.metadata["constraint_type"] = "mathematical"
                return math_result

        # For other types of constraints, a more specialized parser would be used
        # For now, create a basic constraint AST
        ast_root = SymbolicAST(node_type="constraint", value=expression)

        return ParseResult(
            success=True,
            expr_type=SymbolicType.CONSTRAINT,
            ast=ast_root,
            confidence=0.6,
            metadata={"constraint_type": "general"}
        )

    def _sympy_to_ast(self, sympy_expr) -> SymbolicAST:
        """
        Convert a SymPy expression to our internal AST format.

        Args:
            sympy_expr: The SymPy expression to convert

        Returns:
            SymbolicAST representation of the expression
        """
        if not HAS_SYMPY:
            raise ImportError("SymPy is not available")

        # Handle different types of expressions
        if sympy_expr.is_number:
            return SymbolicAST(node_type="number", value=float(sympy_expr))

        elif sympy_expr.is_symbol:
            return SymbolicAST(node_type="variable", value=str(sympy_expr))

        elif sympy_expr.is_Add:
            # Addition operation
            ast_node = SymbolicAST(node_type="operation", value="add")
            for arg in sympy_expr.args:
                ast_node.children.append(self._sympy_to_ast(arg))
            return ast_node

        elif sympy_expr.is_Mul:
            # Multiplication operation
            ast_node = SymbolicAST(node_type="operation", value="multiply")
            for arg in sympy_expr.args:
                ast_node.children.append(self._sympy_to_ast(arg))
            return ast_node

        elif sympy_expr.is_Pow:
            # Power operation
            ast_node = SymbolicAST(node_type="operation", value="power")
            # Add base and exponent as children
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        elif isinstance(sympy_expr, sympy.Function):
            # Function call
            func_name = type(sympy_expr).__name__.lower()
            ast_node = SymbolicAST(node_type="function", value=func_name)
            # Add arguments as children
            for arg in sympy_expr.args:
                ast_node.children.append(self._sympy_to_ast(arg))
            return ast_node

        # Handle equations
        elif isinstance(sympy_expr, sympy.Eq):
            ast_node = SymbolicAST(node_type="equation", value="equals")
            # Add left and right sides as children
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        # Handle inequalities
        elif isinstance(sympy_expr, sympy.Gt):
            ast_node = SymbolicAST(node_type="inequality", value="greater_than")
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        elif isinstance(sympy_expr, sympy.Lt):
            ast_node = SymbolicAST(node_type="inequality", value="less_than")
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        elif isinstance(sympy_expr, sympy.Ge):
            ast_node = SymbolicAST(node_type="inequality", value="greater_equal")
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        elif isinstance(sympy_expr, sympy.Le):
            ast_node = SymbolicAST(node_type="inequality", value="less_equal")
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[0]))
            ast_node.children.append(self._sympy_to_ast(sympy_expr.args[1]))
            return ast_node

        else:
            # Generic expression, just convert to string
            ast_node = SymbolicAST(node_type="expression", value=str(sympy_expr))
            return ast_node

    def _basic_math_parse(self, expression: str) -> SymbolicAST:
        """
        Simple recursive descent parser for basic mathematical expressions.
        Only handles +, -, *, /, ^, and parentheses.

        Args:
            expression: The expression string to parse

        Returns:
            SymbolicAST representation of the expression
        """
        # Remove whitespace
        expression = expression.strip()

        # Handle empty expression
        if not expression:
            return SymbolicAST(node_type="empty", value="")

        # Handle parenthesized expressions
        if expression.startswith('(') and expression.endswith(')'):
            # Check if the parentheses are balanced and outer-most
            count = 0
            for i, char in enumerate(expression):
                if char == '(':
                    count += 1
                elif char == ')':
                    count -= 1
                if count == 0 and i < len(expression) - 1:
                    # Found a closing parenthesis before the end, not outer-most
                    break
            else:
                # Parentheses are outer-most, parse the inner expression
                return self._basic_math_parse(expression[1:-1])

        # Look for operators: + and - (lowest precedence)
        count = 0
        for i in range(len(expression) - 1, -1, -1):  # Scan right to left
            char = expression[i]
            if char == '(':
                count += 1
            elif char == ')':
                count -= 1
            elif count == 0 and char in '+-' and i > 0 and expression[i-1] not in '+-*/^(':
                # Found an operator outside of parentheses
                # Make sure it's not a unary operator or part of another operator
                left = expression[:i]
                right = expression[i+1:]

                op_node = SymbolicAST(
                    node_type="operation",
                    value="add" if char == '+' else "subtract"
                )
                op_node.children.append(self._basic_math_parse(left))
                op_node.children.append(self._basic_math_parse(right))
                return op_node

        # No + or - found, look for * and / (medium precedence)
        count = 0
        for i in range(len(expression) - 1, -1, -1):  # Scan right to left
            char = expression[i]
            if char == '(':
                count += 1
            elif char == ')':
                count -= 1
            elif count == 0 and char in '*/':
                # Found an operator outside of parentheses
                left = expression[:i]
                right = expression[i+1:]

                op_node = SymbolicAST(
                    node_type="operation",
                    value="multiply" if char == '*' else "divide"
                )
                op_node.children.append(self._basic_math_parse(left))
                op_node.children.append(self._basic_math_parse(right))
                return op_node

        # No +, -, *, / found, look for ^ (highest precedence)
        count = 0
        for i in range(len(expression) - 1, -1, -1):  # Scan right to left
            char = expression[i]
            if char == '(':
                count += 1
            elif char == ')':
                count -= 1
            elif count == 0 and char == '^':
                # Found an operator outside of parentheses
                left = expression[:i]
                right = expression[i+1:]

                op_node = SymbolicAST(
                    node_type="operation",
                    value="power"
                )
                op_node.children.append(self._basic_math_parse(left))
                op_node.children.append(self._basic_math_parse(right))
                return op_node

        # No operators found, check for functions
        match = re.match(r'(\w+)\((.+)\)$', expression)
        if match:
            func_name = match.group(1)
            args_str = match.group(2)

            # Parse arguments (simple comma-separated list)
            # This is naive and doesn't handle nested functions properly
            args = []
            current_arg = ""
            paren_count = 0

            for char in args_str:
                if char == '(':
                    paren_count += 1
                    current_arg += char
                elif char == ')':
                    paren_count -= 1
                    current_arg += char
                elif char == ',' and paren_count == 0:
                    args.append(current_arg.strip())
                    current_arg = ""
                else:
                    current_arg += char

            if current_arg:
                args.append(current_arg.strip())

            # Create function node
            func_node = SymbolicAST(node_type="function", value=func_name)

            # Parse each argument
            for arg in args:
                func_node.children.append(self._basic_math_parse(arg))

            return func_node

        # No operators or functions, check for number
        try:
            value = float(expression)
            return SymbolicAST(node_type="number", value=value)
        except ValueError:
            pass

        # Must be a variable
        return SymbolicAST(node_type="variable", value=expression)

    def _python_ast_to_symbolic_ast(self, python_ast) -> SymbolicAST:
        """
        Convert a Python AST to our internal AST format.

        Args:
            python_ast: The Python AST to convert

        Returns:
            SymbolicAST representation of the expression
        """
        # Handle different types of expressions

        if isinstance(python_ast, ast.Module):
            # Multiple statements
            module_node = SymbolicAST(node_type="module", value="module")
            for statement in python_ast.body:
                module_node.children.append(self._python_ast_to_symbolic_ast(statement))
            return module_node

        elif isinstance(python_ast, ast.Expr):
            # Expression statement
            return self._python_ast_to_symbolic_ast(python_ast.value)

        elif isinstance(python_ast, ast.Expression):
            # Expression wrapper
            return self._python_ast_to_symbolic_ast(python_ast.body)

        elif isinstance(python_ast, ast.Constant):
            # Literal value (Python 3.8+)
            if isinstance(python_ast.value, bool):
                return SymbolicAST(node_type="boolean", value=python_ast.value)
            elif isinstance(python_ast.value, (int, float)):
                return SymbolicAST(node_type="number", value=python_ast.value)
            elif isinstance(python_ast.value, str):
                return SymbolicAST(node_type="string", value=python_ast.value)
            else:
                return SymbolicAST(node_type="constant", value=str(python_ast.value))

        elif isinstance(python_ast, ast.Num):
            # Number literal (older Python versions)
            return SymbolicAST(node_type="number", value=python_ast.n)

        elif isinstance(python_ast, ast.Str):
            # String literal (older Python versions)
            return SymbolicAST(node_type="string", value=python_ast.s)

        elif isinstance(python_ast, ast.NameConstant):
            # Name constant (older Python versions)
            return SymbolicAST(node_type="constant", value=str(python_ast.value))

        elif isinstance(python_ast, ast.Name):
            # Variable name
            return SymbolicAST(node_type="variable", value=python_ast.id)

        elif isinstance(python_ast, ast.BinOp):
            # Binary operation
            op_map = {
                ast.Add: "add",
                ast.Sub: "subtract",
                ast.Mult: "multiply",
                ast.Div: "divide",
                ast.Pow: "power",
                ast.Mod: "modulo",
                ast.FloorDiv: "floor_divide",
            }

            op_type = type(python_ast.op)
            op_name = op_map.get(op_type, op_type.__name__.lower())

            op_node = SymbolicAST(node_type="operation", value=op_name)
            op_node.children.append(self._python_ast_to_symbolic_ast(python_ast.left))
            op_node.children.append(self._python_ast_to_symbolic_ast(python_ast.right))

            return op_node

        elif isinstance(python_ast, ast.UnaryOp):
            # Unary operation
            op_map = {
                ast.USub: "negate",
                ast.UAdd: "positive",
                ast.Not: "not",
                ast.Invert: "invert",
            }

            op_type = type(python_ast.op)
            op_name = op_map.get(op_type, op_type.__name__.lower())

            op_node = SymbolicAST(node_type="operation", value=op_name)
            op_node.children.append(self._python_ast_to_symbolic_ast(python_ast.operand))

            return op_node

        elif isinstance(python_ast, ast.Compare):
            # Comparison operation
            op_map = {
                ast.Eq: "equals",
                ast.NotEq: "not_equals",
                ast.Lt: "less_than",
                ast.LtE: "less_equal",
                ast.Gt: "greater_than",
                ast.GtE: "greater_equal",
                ast.Is: "is",
                ast.IsNot: "is_not",
                ast.In: "in",
                ast.NotIn: "not_in",
            }

            # Handle multiple comparisons (e.g., a < b < c)
            if len(python_ast.ops) == 1:
                # Simple comparison
                op_type = type(python_ast.ops[0])
                op_name = op_map.get(op_type, op_type.__name__.lower())

                if op_name in ["equals", "not_equals"]:
                    node_type = "equation"
                else:
                    node_type = "inequality"

                op_node = SymbolicAST(node_type=node_type, value=op_name)
                op_node.children.append(self._python_ast_to_symbolic_ast(python_ast.left))
                op_node.children.append(self._python_ast_to_symbolic_ast(python_ast.comparators[0]))

                return op_node
            else:
                # Multiple comparisons, create a logical AND of comparisons
                and_node = SymbolicAST(node_type="operation", value="and")

                left = python_ast.left
                for i, op in enumerate(python_ast.ops):
                    op_type = type(op)
                    op_name = op_map.get(op_type, op_type.__name__.lower())

                    if op_name in ["equals", "not_equals"]:
                        node_type = "equation"
                    else:
                        node_type = "inequality"

                    comp_node = SymbolicAST(node_type=node_type, value=op_name)
                    comp_node.children.append(self._python_ast_to_symbolic_ast(left))
                    comp_node.children.append(self._python_ast_to_symbolic_ast(python_ast.comparators[i]))

                    and_node.children.append(comp_node)

                    # For chained comparisons, the right side becomes the left for the next comparison
                    left = python_ast.comparators[i]

                return and_node

        elif isinstance(python_ast, ast.BoolOp):
            # Boolean operation (and, or)
            op_map = {
                ast.And: "and",
                ast.Or: "or",
            }

            op_type = type(python_ast.op)
            op_name = op_map.get(op_type, op_type.__name__.lower())

            op_node = SymbolicAST(node_type="operation", value=op_name)

            # Add all values as children
            for value in python_ast.values:
                op_node.children.append(self._python_ast_to_symbolic_ast(value))

            return op_node

        elif isinstance(python_ast, ast.Call):
            # Function call
            func_node = SymbolicAST(node_type="function", value=self._get_func_name(python_ast.func))

            # Add arguments as children
            for arg in python_ast.args:
                func_node.children.append(self._python_ast_to_symbolic_ast(arg))

            # Add keyword arguments as metadata
            if python_ast.keywords:
                func_node.metadata["keywords"] = {}
                for keyword in python_ast.keywords:
                    key = keyword.arg
                    value_ast = self._python_ast_to_symbolic_ast(keyword.value)
                    func_node.metadata["keywords"][key] = value_ast

            return func_node

        # Add more conversions as needed for other AST node types

        else:
            # Generic node, just convert to string
            return SymbolicAST(node_type="unknown", value=str(python_ast.__class__.__name__))

    def _get_func_name(self, func_node) -> str:
        """
        Extract function name from a func AST node.

        Args:
            func_node: The function node from Python AST

        Returns:
            String representation of the function name
        """
        if isinstance(func_node, ast.Name):
            return func_node.id
        elif isinstance(func_node, ast.Attribute):
            base = self._get_func_name(func_node.value)
            return f"{base}.{func_node.attr}"
        else:
            return str(func_node.__class__.__name__)

    def _evaluate_mathematical(self, parse_result: ParseResult,
                              variable_values: Dict[str, Any],
                              context: Dict[str, Any]) -> Any:
        """
        Evaluate a mathematical expression with given variable values.

        Args:
            parse_result: The parsed mathematical expression
            variable_values: Dictionary mapping variable names to values
            context: Additional contextual information

        Returns:
            The result of evaluation
        """
        if HAS_SYMPY and parse_result.sympy_expr is not None:
            # Evaluate using SymPy
            expr = parse_result.sympy_expr

            # Substitute variables with values
            for var_name, value in variable_values.items():
                if var_name in [str(sym) for sym in expr.free_symbols]:
                    expr = expr.subs(sympy.Symbol(var_name), value)

            # Try to evaluate to a numerical value
            try:
                result = float(expr.evalf())
                return result
            except (TypeError, ValueError):
                # If it can't be converted to float, return the SymPy expression
                return expr

        elif parse_result.ast is not None:
            # Evaluate using our own AST walker
            return self._evaluate_ast(parse_result.ast, variable_values, context)

        else:
            raise ValueError("No evaluable expression found in parse result")

    def _evaluate_logical(self, parse_result: ParseResult,
                         variable_values: Dict[str, Any],
                         context: Dict[str, Any]) -> Any:
        """
        Evaluate a logical expression with given variable values.

        Args:
            parse_result: The parsed logical expression
            variable_values: Dictionary mapping variable names to values
            context: Additional contextual information

        Returns:
            The result of evaluation (typically a boolean)
        """
        if parse_result.ast is not None:
            # Evaluate using our own AST walker
            return self._evaluate_ast(parse_result.ast, variable_values, context)

        elif parse_result.python_ast is not None:
            # Evaluate using Python's eval (use with caution!)
            # This is simplified and not secure for production

            # Create a safe environment with only the variables provided
            safe_env = {k: v for k, v in variable_values.items()}

            # Add basic logical operations
            safe_env.update({
                'and': operator.and_,
                'or': operator.or_,
                'not': operator.not_,
                'True': True,
                'False': False
            })

            # Compile and evaluate the AST
            code = compile(parse_result.python_ast, '<string>', 'eval')
            return eval(code, {"__builtins__": {}}, safe_env)

        else:
            raise ValueError("No evaluable expression found in parse result")

    def _evaluate_code(self, parse_result: ParseResult,
                      variable_values: Dict[str, Any],
                      context: Dict[str, Any]) -> Any:
        """
        Evaluate a code expression with given variable values.

        Args:
            parse_result: The parsed code expression
            variable_values: Dictionary mapping variable names to values
            context: Additional contextual information

        Returns:
            The result of evaluation
        """
        if parse_result.python_ast is not None:
            # This is a simplified and not secure evaluation
            # In a real system, use a sandbox or restricted execution environment

            # Create an environment with only the variables provided
            safe_env = {k: v for k, v in variable_values.items()}

            # Compile and evaluate the AST
            mode = 'exec' if isinstance(parse_result.python_ast, ast.Module) else 'eval'
            code = compile(parse_result.python_ast, '<string>', mode)

            # For 'exec' mode, we need to capture potential return values
            local_env = {}
            exec(code, {"__builtins__": {}}, local_env)

            # Return the updated local environment as the result
            return local_env

        elif parse_result.ast is not None:
            # For non-Python code, we would need a custom interpreter
            # This is just a placeholder
            self.logger.warning("Custom code interpretation not implemented")
            return None

        else:
            raise ValueError("No evaluable code found in parse result")

    def _evaluate_pattern(self, parse_result: ParseResult,
                         variable_values: Dict[str, Any],
                         context: Dict[str, Any]) -> Any:
        """
        Evaluate a pattern expression, typically by matching it against a target.

        Args:
            parse_result: The parsed pattern expression
            variable_values: Dictionary with target string to match against
            context: Additional contextual information

        Returns:
            Match results, typically a boolean or match object
        """
        if not parse_result.ast:
            raise ValueError("No pattern AST found in parse result")

        pattern_type = parse_result.metadata.get("pattern_type", "unknown")
        pattern_str = parse_result.ast.value

        # Get the target string to match against
        target = variable_values.get("target", "")
        if not target:
            raise ValueError("No target string provided for pattern matching")

        if pattern_type == "regex":
            # Regular expression matching
            try:
                match = re.search(pattern_str, target)
                if match:
                    # Return match object details
                    return {
                        "matched": True,
                        "span": match.span(),
                        "groups": match.groups(),
                        "group_dict": match.groupdict()
                    }
                else:
                    return {"matched": False}
            except re.error as e:
                raise ValueError(f"Invalid regex pattern: {str(e)}")

        elif pattern_type == "glob":
            # Simple glob pattern matching
            # Convert glob to regex
            regex = pattern_str.replace("*", ".*").replace("?", ".")
            regex = f"^{regex}$"  # Match whole string

            match = re.match(regex, target)
            return {"matched": bool(match)}

        else:
            raise ValueError(f"Unsupported pattern type: {pattern_type}")

    def _evaluate_ast(self, ast_node: SymbolicAST,
                     variable_values: Dict[str, Any],
                     context: Dict[str, Any]) -> Any:
        """
        Evaluate an AST node with given variable values.

        Args:
            ast_node: The AST node to evaluate
            variable_values: Dictionary mapping variable names to values
            context: Additional contextual information

        Returns:
            The result of evaluation
        """
        # Handle different node types
        if ast_node.node_type == "number":
            return ast_node.value

        elif ast_node.node_type == "boolean":
            return ast_node.value

        elif ast_node.node_type == "string":
            return ast_node.value

        elif ast_node.node_type == "variable":
            var_name = ast_node.value
            if var_name in variable_values:
                return variable_values[var_name]
            elif var_name in self.known_symbols:
                symbol_info = self.known_symbols[var_name]
                if "value" in symbol_info:
                    return symbol_info["value"]

            raise ValueError(f"Unknown variable: {var_name}")

        elif ast_node.node_type == "operation":
            op = ast_node.value

            if op == "add":
                if len(ast_node.children) != 2:
                    raise ValueError("Add operation requires exactly 2 operands")
                left = self._evaluate_ast(ast_node.children[0], variable_values, context)
                right = self._evaluate_ast(ast_node.children[1], variable_values, context)
                return left + right

            elif op == "subtract":
                if len(ast_node.children) != 2:
                    raise ValueError("Subtract operation requires exactly 2 operands")
                left = self._evaluate_ast(ast_node.children[0], variable_values, context)
                right = self._evaluate_ast(ast_node.children[1], variable_values, context)
                return left - right

            elif op == "multiply":
                if len(ast_node.children) != 2:
                    raise ValueError("Multiply operation requires exactly 2 operands")
                left = self._evaluate_ast(ast_node.children[0], variable_values, context)
                right = self._evaluate_ast(ast_node.children[1], variable_values, context)
                return left * right

            elif op == "divide":
                if len(ast_node.children) != 2:
                    raise ValueError("Divide operation requires exactly 2 operands")
                left = self._evaluate_ast(ast_node.children[0], variable_values, context)
                right = self._evaluate_ast(ast_node.children[1], variable_values, context)
                return left / right

            elif op == "power":
                if len(ast_node.children) != 2:
                    raise ValueError("Power operation requires exactly 2 operands")
                base = self._evaluate_ast(ast_node.children[0], variable_values, context)
                exponent = self._evaluate_ast(ast_node.children[1], variable_values, context)
                return base ** exponent

            elif op == "negate":
                if len(ast_node.children) != 1:
                    raise ValueError("Negate operation requires exactly 1 operand")
                operand = self._evaluate_ast(ast_node.children[0], variable_values, context)
                return -operand

            elif op == "and":
                results = [self._evaluate_ast(child, variable_values, context) for child in ast_node.children]
                return all(results)

            elif op == "or":
                results = [self._evaluate_ast(child, variable_values, context) for child in ast_node.children]
                return any(results)

            elif op == "not":
                if len(ast_node.children) != 1:
                    raise ValueError("Not operation requires exactly 1 operand")
                operand = self._evaluate_ast(ast_node.children[0], variable_values, context)
                return not operand

            else:
                raise ValueError(f"Unsupported operation: {op}")

        elif ast_node.node_type == "function":
            func_name = ast_node.value
            args = [self._evaluate_ast(child, variable_values, context) for child in ast_node.children]

            # Check if function is in known symbols
            if func_name in self.known_symbols:
                symbol_info = self.known_symbols[func_name]
                if symbol_info["type"] == "function" and "value" in symbol_info:
                    func = symbol_info["value"]
                    return func(*args)

            # Check built-in functions
            if func_name == "sin":
                return math.sin(args[0])
            elif func_name == "cos":
                return math.cos(args[0])
            elif func_name == "tan":
                return math.tan(args[0])
            elif func_name == "log":
                return math.log(args[0])
            elif func_name == "exp":
                return math.exp(args[0])
            else:
                raise ValueError(f"Unknown function: {func_name}")

        elif ast_node.node_type in ["equation", "inequality"]:
            op = ast_node.value

            if len(ast_node.children) != 2:
                raise ValueError(f"{op} requires exactly 2 operands")

            left = self._evaluate_ast(ast_node.children[0], variable_values, context)
            right = self._evaluate_ast(ast_node.children[1], variable_values, context)

            if op == "equals":
                return left == right
            elif op == "not_equals":
                return left != right
            elif op == "less_than":
                return left < right
            elif op == "less_equal":
                return left <= right
            elif op == "greater_than":
                return left > right
            elif op == "greater_equal":
                return left >= right
            else:
                raise ValueError(f"Unsupported comparison: {op}")

        else:
            raise ValueError(f"Unsupported node type: {ast_node.node_type}")

    def _transform_mathematical(self, parse_result: ParseResult,
                               transformation: str,
                               context: Dict[str, Any]) -> ParseResult:
        """
        Apply a transformation to a mathematical expression.

        Args:
            parse_result: The parsed mathematical expression
            transformation: The type of transformation to apply
            context: Additional contextual information

        Returns:
            A new ParseResult with the transformed expression
        """
        if not HAS_SYMPY or parse_result.sympy_expr is None:
            self.logger.warning("SymPy is required for mathematical transformations")
            return parse_result

        try:
            expr = parse_result.sympy_expr

            # Apply the requested transformation
            if transformation == "simplify":
                result = sympy.simplify(expr)
            elif transformation == "expand":
                result = sympy.expand(expr)
            elif transformation == "factor":
                result = sympy.factor(expr)
            elif transformation == "collect":
                if "var" in context:
                    var_name = context["var"]
                    var = sympy.Symbol(var_name)
                    result = sympy.collect(expr, var)
                else:
                    raise ValueError("Collect transformation requires 'var' in context")
            elif transformation == "solve":
                if isinstance(expr, sympy.Eq):
                    # Solve equation for x by default, or specified variable
                    var_name = context.get("var", "x")
                    var = sympy.Symbol(var_name)
                    result = sympy.solve(expr, var)

                    # Create a new result with the solution
                    # This is a bit special since we're changing the expression type
                    new_ast = SymbolicAST(node_type="solution", value=str(result))
                    return ParseResult(
                        success=True,
                        expr_type=SymbolicType.MATHEMATICAL,
                        ast=new_ast,
                        sympy_expr=result,  # This will be a list for solve, not an expr
                        confidence=parse_result.confidence,
                        metadata={**parse_result.metadata, "transformation": transformation}
                    )
                else:
                    raise ValueError("Solve transformation requires an equation")
            else:
                raise ValueError(f"Unsupported transformation: {transformation}")

            # Create a new AST from the transformed expression
            new_ast = self._sympy_to_ast(result)

            # Create a new ParseResult with the transformed expression
            return ParseResult(
                success=True,
                expr_type=parse_result.expr_type,
                ast=new_ast,
                sympy_expr=result,
                confidence=parse_result.confidence,
                metadata={**parse_result.metadata, "transformation": transformation}
            )

        except Exception as e:
            self.logger.error(f"Error applying transformation '{transformation}': {str(e)}")
            return parse_result

    def _transform_logical(self, parse_result: ParseResult,
                          transformation: str,
                          context: Dict[str, Any]) -> ParseResult:
        """
        Apply a transformation to a logical expression.

        Args:
            parse_result: The parsed logical expression
            transformation: The type of transformation to apply
            context: Additional contextual information

        Returns:
            A new ParseResult with the transformed expression
        """
        # This is a simplified implementation
        # A real implementation would use a logical transformation library

        if not parse_result.ast:
            self.logger.warning("AST required for logical transformations")
            return parse_result

        try:
            # Very basic transformations that don't really transform anything
            # In a real system, this would use proper logical equivalence rules

            if transformation == "negate":
                # Create a NOT wrapper around the expression
                new_ast = SymbolicAST(node_type="operation", value="not")
                new_ast.children.append(parse_result.ast)

                return ParseResult(
                    success=True,
                    expr_type=parse_result.expr_type,
                    ast=new_ast,
                    confidence=parse_result.confidence,
                    metadata={**parse_result.metadata, "transformation": transformation}
                )

            elif transformation == "distribute":
                # This would actually distribute AND over OR or vice versa
                # But we'll just return the original for now
                self.logger.warning("Logical distribution not fully implemented")
                return parse_result

            else:
                raise ValueError(f"Unsupported logical transformation: {transformation}")

        except Exception as e:
            self.logger.error(f"Error applying logical transformation '{transformation}': {str(e)}")
            return parse_result

ModuleNotFoundError: No module named 'grandnexus'

# Symbolic Engine

In [ ]:
"""
symbolic_engine.py - Formal Symbolic Reasoning Engine for GrandNexus

This module provides a robust symbolic reasoning system capable of formal logical operations,
theorem proving, and knowledge representation. It serves as the foundation for GrandNexus's
symbolic reasoning capabilities, complementing the neural reasoning components.

Key features:
- First-order logic representation and manipulation
- Inference rules and deduction mechanisms
- Knowledge base management with belief revision
- Integration with the semantic graph for knowledge retrieval
- Support for different reasoning strategies (forward/backward chaining, resolution)
- Uncertainty handling via probabilistic logic extensions
- Flexible rule systems that can be extended or adapted

The design emphasizes:
- Modularity: Different reasoning strategies can be swapped or combined
- Explainability: All inference steps can be traced and explained
- Integration: Seamless connection with other cognitive modules
- Extensibility: New logical systems or reasoning approaches can be added
"""

import logging
import time
import threading
import uuid
import re
from typing import Dict, List, Any, Optional, Set, Tuple, Union, Callable
from enum import Enum, auto
from dataclasses import dataclass, field
from collections import defaultdict, deque
import copy

# Assuming these exist or will exist in the GrandNexus architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.perception.symbolic_parser import SymbolicParser, ParseResult, SymbolicType
from grandnexus.graph.graph_engine import create_graph_engine, AbstractGraphEngine

# Try to import common symbolic reasoning libraries
try:
    import sympy
    from sympy.logic.boolalg import to_cnf, simplify_logic
    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False

try:
    from pyke import knowledge_engine
    HAS_PYKE = True
except ImportError:
    HAS_PYKE = False

# -----------------------------------------------------------------------------
# Core Data Structures
# -----------------------------------------------------------------------------

class SymbolicEntityType(Enum):
    """Types of symbolic entities in the reasoning system"""
    CONSTANT = auto()    # Individual constant (e.g., "a", "John")
    VARIABLE = auto()    # Logical variable (e.g., "x", "Y")
    FUNCTION = auto()    # Function symbol (e.g., "father_of")
    PREDICATE = auto()   # Predicate symbol (e.g., "isHuman")
    OPERATOR = auto()    # Logical operator (e.g., "AND", "OR")
    QUANTIFIER = auto()  # Logical quantifier (e.g., "∀", "∃")
    TERM = auto()        # Compound term (function application)
    ATOM = auto()        # Atomic formula (predicate application)
    FORMULA = auto()     # Compound formula (logical expressions)
    RULE = auto()        # Inference rule or axiom

class InferenceStrategy(Enum):
    """Reasoning strategies supported by the engine"""
    FORWARD_CHAINING = auto()  # Data-driven reasoning
    BACKWARD_CHAINING = auto() # Goal-driven reasoning
    RESOLUTION = auto()        # Resolution refutation
    NATURAL_DEDUCTION = auto() # Natural deduction system
    TABLEAUX = auto()          # Semantic tableaux method
    HYBRID = auto()            # Combined strategy

@dataclass
class SymbolicEntity:
    """Base class for all symbolic entities in the reasoning system"""
    entity_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    entity_type: SymbolicEntityType = SymbolicEntityType.CONSTANT
    symbol: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __hash__(self):
        return hash(self.entity_id)

    def __eq__(self, other):
        if not isinstance(other, SymbolicEntity):
            return False
        return self.entity_id == other.entity_id

    def __str__(self):
        return self.symbol

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        return {
            "entity_id": self.entity_id,
            "entity_type": self.entity_type.name,
            "symbol": self.symbol,
            "metadata": self.metadata
        }

@dataclass
class Term(SymbolicEntity):
    """Represents a term in first-order logic (constant, variable, or function application)"""
    arguments: List[Union["Term", "SymbolicEntity"]] = field(default_factory=list)

    def __post_init__(self):
        if not self.entity_type:
            if not self.arguments:
                # Simple term (constant or variable)
                if self.symbol.isupper() or self.symbol.startswith('?'):
                    self.entity_type = SymbolicEntityType.VARIABLE
                else:
                    self.entity_type = SymbolicEntityType.CONSTANT
            else:
                # Function application
                self.entity_type = SymbolicEntityType.TERM

    def __str__(self):
        if not self.arguments:
            return self.symbol

        args_str = ", ".join(str(arg) for arg in self.arguments)
        return f"{self.symbol}({args_str})"

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        result = super().to_dict()
        result["arguments"] = [
            arg.to_dict() if hasattr(arg, "to_dict") else str(arg)
            for arg in self.arguments
        ]
        return result

@dataclass
class Formula(SymbolicEntity):
    """Represents a logical formula (atomic or compound)"""
    operator: Optional[str] = None
    arguments: List[Union["Formula", Term]] = field(default_factory=list)
    quantifier: Optional[str] = None
    variables: List[Term] = field(default_factory=list)

    def __post_init__(self):
        if not self.entity_type:
            if not self.operator and not self.quantifier:
                self.entity_type = SymbolicEntityType.ATOM
            else:
                self.entity_type = SymbolicEntityType.FORMULA

    def __str__(self):
        if self.quantifier:
            vars_str = ", ".join(str(var) for var in self.variables)
            arg_str = str(self.arguments[0]) if self.arguments else ""
            return f"{self.quantifier}{vars_str}. {arg_str}"

        if not self.operator:
            # Atomic formula
            args_str = ", ".join(str(arg) for arg in self.arguments)
            return f"{self.symbol}({args_str})"

        # Compound formula
        if len(self.arguments) == 1:
            # Unary operator (e.g., NOT)
            return f"{self.operator}({str(self.arguments[0])})"

        # Binary or n-ary operator
        args_str = f" {self.operator} ".join(str(arg) for arg in self.arguments)
        return f"({args_str})"

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        result = super().to_dict()
        result["operator"] = self.operator
        result["arguments"] = [
            arg.to_dict() if hasattr(arg, "to_dict") else str(arg)
            for arg in self.arguments
        ]
        result["quantifier"] = self.quantifier
        result["variables"] = [var.to_dict() for var in self.variables]
        return result

@dataclass
class Rule(SymbolicEntity):
    """Represents an inference rule or axiom"""
    premises: List[Formula] = field(default_factory=list)
    conclusion: Optional[Formula] = None
    bidirectional: bool = False
    weight: float = 1.0

    def __post_init__(self):
        self.entity_type = SymbolicEntityType.RULE

    def __str__(self):
        premises_str = ", ".join(str(premise) for premise in self.premises)
        conclusion_str = str(self.conclusion) if self.conclusion else ""
        connector = "⟺" if self.bidirectional else "⟹"
        return f"{premises_str} {connector} {conclusion_str}"

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        result = super().to_dict()
        result["premises"] = [premise.to_dict() for premise in self.premises]
        result["conclusion"] = self.conclusion.to_dict() if self.conclusion else None
        result["bidirectional"] = self.bidirectional
        result["weight"] = self.weight
        return result

@dataclass
class InferenceStep:
    """Represents a single step in an inference process"""
    step_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    rule_id: Optional[str] = None
    premises: List[str] = field(default_factory=list)  # IDs of formulas used as premises
    conclusion: Optional[str] = None  # ID of concluded formula
    substitutions: Dict[str, str] = field(default_factory=dict)  # Variable substitutions
    confidence: float = 1.0
    timestamp: float = field(default_factory=time.time)

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        return {
            "step_id": self.step_id,
            "rule_id": self.rule_id,
            "premises": self.premises,
            "conclusion": self.conclusion,
            "substitutions": self.substitutions,
            "confidence": self.confidence,
            "timestamp": self.timestamp
        }

@dataclass
class InferenceResult:
    """Result of a symbolic reasoning process"""
    success: bool
    proved_formulas: List[Formula] = field(default_factory=list)
    inference_steps: List[InferenceStep] = field(default_factory=list)
    confidence: float = 1.0
    execution_time: float = 0.0
    message: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        return {
            "success": self.success,
            "proved_formulas": [f.to_dict() for f in self.proved_formulas],
            "inference_steps": [step.to_dict() for step in self.inference_steps],
            "confidence": self.confidence,
            "execution_time": self.execution_time,
            "message": self.message
        }

class KnowledgeBase:
    """
    Knowledge base for symbolic reasoning, storing facts, rules, and supporting
    belief revision and consistency management.
    """

    def __init__(self):
        """Initialize the knowledge base"""
        self.facts = {}  # Formula ID -> Formula
        self.rules = {}  # Rule ID -> Rule
        self.predicates = {}  # Predicate symbol -> metadata
        self.functions = {}  # Function symbol -> metadata

        # Indexes for efficient querying
        self.fact_index = defaultdict(set)  # Predicate symbol -> set of fact IDs
        self.rule_index = defaultdict(set)  # Predicate in conclusion -> set of rule IDs

        # Belief tracking
        self.confidence = {}  # Formula/Rule ID -> confidence value
        self.provenance = {}  # Formula/Rule ID -> source information
        self.timestamps = {}  # Formula/Rule ID -> timestamp

        # For conflict resolution
        self.dependencies = defaultdict(set)  # Formula ID -> set of IDs it depends on
        self.contradictions = set()  # Pairs of contradictory formula IDs

    def add_fact(self, formula: Formula, confidence: float = 1.0,
                source: Optional[str] = None) -> bool:
        """
        Add a fact to the knowledge base.

        Args:
            formula: The fact to add
            confidence: Confidence value (0.0 to 1.0)
            source: Optional source information

        Returns:
            True if added successfully, False if rejected
        """
        # Check for existing identical fact
        for existing_id, existing_formula in self.facts.items():
            if self._formulas_equivalent(formula, existing_formula):
                # Update confidence if new fact has higher confidence
                if confidence > self.confidence.get(existing_id, 0.0):
                    self.confidence[existing_id] = confidence
                    if source:
                        self.provenance[existing_id] = source
                    self.timestamps[existing_id] = time.time()
                return True

        # Add new fact
        self.facts[formula.entity_id] = formula
        self.confidence[formula.entity_id] = confidence
        self.timestamps[formula.entity_id] = time.time()

        if source:
            self.provenance[formula.entity_id] = source

        # Update indexes
        if formula.entity_type == SymbolicEntityType.ATOM:
            self.fact_index[formula.symbol].add(formula.entity_id)

        # Check for contradictions
        self._check_contradictions(formula)

        return True

    def add_rule(self, rule: Rule, confidence: float = 1.0,
                source: Optional[str] = None) -> bool:
        """
        Add a rule to the knowledge base.

        Args:
            rule: The rule to add
            confidence: Confidence value (0.0 to 1.0)
            source: Optional source information

        Returns:
            True if added successfully, False if rejected
        """
        # Check for existing identical rule
        for existing_id, existing_rule in self.rules.items():
            if self._rules_equivalent(rule, existing_rule):
                # Update confidence if new rule has higher confidence
                if confidence > self.confidence.get(existing_id, 0.0):
                    self.confidence[existing_id] = confidence
                    if source:
                        self.provenance[existing_id] = source
                    self.timestamps[existing_id] = time.time()
                return True

        # Add new rule
        self.rules[rule.entity_id] = rule
        self.confidence[rule.entity_id] = confidence
        self.timestamps[rule.entity_id] = time.time()

        if source:
            self.provenance[rule.entity_id] = source

        # Update indexes
        if rule.conclusion and rule.conclusion.entity_type == SymbolicEntityType.ATOM:
            self.rule_index[rule.conclusion.symbol].add(rule.entity_id)

        return True

    def get_fact(self, fact_id: str) -> Optional[Formula]:
        """Retrieve a fact by ID"""
        return self.facts.get(fact_id)

    def get_rule(self, rule_id: str) -> Optional[Rule]:
        """Retrieve a rule by ID"""
        return self.rules.get(rule_id)

    def query_facts(self, predicate: Optional[str] = None,
                   confidence_threshold: float = 0.0) -> List[Formula]:
        """
        Query facts based on predicate and confidence.

        Args:
            predicate: Optional predicate symbol to filter by
            confidence_threshold: Minimum confidence value

        Returns:
            List of matching facts
        """
        if predicate:
            fact_ids = self.fact_index.get(predicate, set())
        else:
            fact_ids = set(self.facts.keys())

        # Filter by confidence
        facts = []
        for fact_id in fact_ids:
            if self.confidence.get(fact_id, 0.0) >= confidence_threshold:
                facts.append(self.facts[fact_id])

        return facts

    def query_rules(self, conclusion_predicate: Optional[str] = None,
                   confidence_threshold: float = 0.0) -> List[Rule]:
        """
        Query rules based on conclusion predicate and confidence.

        Args:
            conclusion_predicate: Optional predicate symbol in the conclusion
            confidence_threshold: Minimum confidence value

        Returns:
            List of matching rules
        """
        if conclusion_predicate:
            rule_ids = self.rule_index.get(conclusion_predicate, set())
        else:
            rule_ids = set(self.rules.keys())

        # Filter by confidence
        rules = []
        for rule_id in rule_ids:
            if self.confidence.get(rule_id, 0.0) >= confidence_threshold:
                rules.append(self.rules[rule_id])

        return rules

    def remove_fact(self, fact_id: str) -> bool:
        """Remove a fact from the knowledge base"""
        if fact_id not in self.facts:
            return False

        fact = self.facts[fact_id]

        # Remove from indexes
        if fact.entity_type == SymbolicEntityType.ATOM:
            self.fact_index[fact.symbol].discard(fact_id)

        # Remove from contradictions
        to_remove = set()
        for pair in self.contradictions:
            if fact_id in pair:
                to_remove.add(pair)
        self.contradictions -= to_remove

        # Remove from dependencies
        self.dependencies.pop(fact_id, None)
        for deps in self.dependencies.values():
            deps.discard(fact_id)

        # Remove fact
        del self.facts[fact_id]
        self.confidence.pop(fact_id, None)
        self.provenance.pop(fact_id, None)
        self.timestamps.pop(fact_id, None)

        return True

    def remove_rule(self, rule_id: str) -> bool:
        """Remove a rule from the knowledge base"""
        if rule_id not in self.rules:
            return False

        rule = self.rules[rule_id]

        # Remove from indexes
        if rule.conclusion and rule.conclusion.entity_type == SymbolicEntityType.ATOM:
            self.rule_index[rule.conclusion.symbol].discard(rule_id)

        # Remove rule
        del self.rules[rule_id]
        self.confidence.pop(rule_id, None)
        self.provenance.pop(rule_id, None)
        self.timestamps.pop(rule_id, None)

        return True

    def get_contradictions(self) -> List[Tuple[Formula, Formula]]:
        """Get all contradictory facts in the knowledge base"""
        result = []
        for fact1_id, fact2_id in self.contradictions:
            if fact1_id in self.facts and fact2_id in self.facts:
                result.append((self.facts[fact1_id], self.facts[fact2_id]))
        return result

    def resolve_contradiction(self, fact1_id: str, fact2_id: str,
                             keep_id: Optional[str] = None) -> None:
        """
        Resolve a contradiction by removing one of the facts.

        Args:
            fact1_id: ID of the first fact
            fact2_id: ID of the second fact
            keep_id: ID of the fact to keep (if None, keep the highest confidence one)
        """
        if (fact1_id, fact2_id) not in self.contradictions and (fact2_id, fact1_id) not in self.contradictions:
            return

        # Determine which fact to keep
        if keep_id is None:
            conf1 = self.confidence.get(fact1_id, 0.0)
            conf2 = self.confidence.get(fact2_id, 0.0)
            keep_id = fact1_id if conf1 >= conf2 else fact2_id

        # Remove the other fact
        remove_id = fact2_id if keep_id == fact1_id else fact1_id
        self.remove_fact(remove_id)

        # Remove from contradictions
        self.contradictions.discard((fact1_id, fact2_id))
        self.contradictions.discard((fact2_id, fact1_id))

    def clear(self) -> None:
        """Clear the knowledge base"""
        self.facts.clear()
        self.rules.clear()
        self.fact_index.clear()
        self.rule_index.clear()
        self.confidence.clear()
        self.provenance.clear()
        self.timestamps.clear()
        self.dependencies.clear()
        self.contradictions.clear()

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _formulas_equivalent(self, formula1: Formula, formula2: Formula) -> bool:
        """
        Check if two formulas are logically equivalent.
        This is a simplified implementation - a real system would use more
        sophisticated formula normalization and equivalence checking.

        Args:
            formula1: First formula
            formula2: Second formula

        Returns:
            True if formulas are equivalent, False otherwise
        """
        # Simple syntactic equality check
        # In a real system, this would use a proper logical equivalence check
        return str(formula1) == str(formula2)

    def _rules_equivalent(self, rule1: Rule, rule2: Rule) -> bool:
        """
        Check if two rules are logically equivalent.

        Args:
            rule1: First rule
            rule2: Second rule

        Returns:
            True if rules are equivalent, False otherwise
        """
        # Check bidirectionality
        if rule1.bidirectional != rule2.bidirectional:
            return False

        # Check conclusion equality
        if not self._formulas_equivalent(rule1.conclusion, rule2.conclusion):
            return False

        # Check premises (order-independent)
        if len(rule1.premises) != len(rule2.premises):
            return False

        # For simplicity, just check that all premises in rule1 have an equivalent in rule2
        # This is a simplification - a real system would handle more complex equivalence
        for premise1 in rule1.premises:
            found_match = False
            for premise2 in rule2.premises:
                if self._formulas_equivalent(premise1, premise2):
                    found_match = True
                    break
            if not found_match:
                return False

        return True

    def _check_contradictions(self, formula: Formula) -> None:
        """
        Check if a formula contradicts existing facts in the knowledge base.

        Args:
            formula: The formula to check
        """
        # This is a simplified implementation
        # A real system would use more sophisticated contradiction detection

        if formula.entity_type != SymbolicEntityType.ATOM:
            return

        # Look for simple negation contradictions
        for existing_id, existing_formula in self.facts.items():
            if existing_formula.entity_type != SymbolicEntityType.ATOM:
                continue

            # Check for negation contradiction
            # This is naive - a real system would check more complex contradictions
            if (existing_formula.symbol == formula.symbol and
                len(existing_formula.arguments) == len(formula.arguments)):

                # Check if one is a negation of the other
                # This is extremely simplified - real systems use unification and logical inference
                if existing_formula.metadata.get("negated", False) != formula.metadata.get("negated", False):

                    # Check if arguments match
                    args_match = True
                    for arg1, arg2 in zip(existing_formula.arguments, formula.arguments):
                        if str(arg1) != str(arg2):
                            args_match = False
                            break

                    if args_match:
                        self.contradictions.add((existing_id, formula.entity_id))

class SymbolicEngine:
    """
    Core symbolic reasoning engine for GrandNexus, supporting formal logical
    operations, theorem proving, and knowledge representation.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 symbolic_parser: Optional[SymbolicParser] = None,
                 graph_engine: Optional[AbstractGraphEngine] = None):
        """
        Initialize the symbolic reasoning engine.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            semantic_memory: Reference to semantic memory for knowledge retrieval
            working_memory: Reference to working memory for context
            symbolic_parser: Reference to symbolic parser for parsing logical expressions
            graph_engine: Reference to graph engine for graph-based reasoning
        """
        self.logger = logging.getLogger("GrandNexus.Reasoning.SymbolicEngine")
        self.nexus_core = nexus_core
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory
        self.symbolic_parser = symbolic_parser
        self.graph_engine = graph_engine

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Knowledge base
        self.knowledge_base = KnowledgeBase()

        # Inference rules registry
        self.inference_rules = {}
        self.inference_strategies = {}

        # Reasoning statistics
        self.stats = defaultdict(lambda: defaultdict(float))

        self.logger.info(f"SymbolicEngine initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the engine and load standard inference rules"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register standard inference rules
                self._register_standard_inference_rules()

                # Register inference strategies
                self._register_inference_strategies()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("SymbolicEngine initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"SymbolicEngine initialization failed: {str(e)}")
                return False

    def parse_formula(self, formula_str: str, context: Optional[Dict[str, Any]] = None) -> Optional[Formula]:
        """
        Parse a logical formula from string representation.

        Args:
            formula_str: String representation of the formula
            context: Optional context information

        Returns:
            Parsed Formula object, or None if parsing fails
        """
        if not self.symbolic_parser:
            self.logger.error("No symbolic parser available")
            return None

        if not self.initialized:
            self.initialize()

        # Set context for logical expression parsing
        if context is None:
            context = {}
        context["expr_type"] = SymbolicType.LOGICAL

        # Parse the formula using symbolic parser
        parse_result = self.symbolic_parser.parse(formula_str, expected_type=SymbolicType.LOGICAL, context=context)

        if not parse_result.success:
            self.logger.warning(f"Failed to parse formula: {formula_str}")
            return None

        # Convert the parsed result to a Formula object
        try:
            formula = self._parse_result_to_formula(parse_result)
            return formula
        except Exception as e:
            self.logger.error(f"Error converting parse result to formula: {str(e)}")
            return None

    def parse_rule(self, rule_str: str, context: Optional[Dict[str, Any]] = None) -> Optional[Rule]:
        """
        Parse a logical rule from string representation.

        Args:
            rule_str: String representation of the rule
            context: Optional context information

        Returns:
            Parsed Rule object, or None if parsing fails
        """
        if not self.symbolic_parser:
            self.logger.error("No symbolic parser available")
            return None

        if not self.initialized:
            self.initialize()

        # Split into premises and conclusion
        if "=>" in rule_str:
            parts = rule_str.split("=>")
            bidirectional = False
        elif "<=>" in rule_str:
            parts = rule_str.split("<=>")
            bidirectional = True
        else:
            self.logger.warning(f"Rule must contain => or <=> operator: {rule_str}")
            return None

        if len(parts) != 2:
            self.logger.warning(f"Invalid rule format: {rule_str}")
            return None

        premises_str = parts[0].strip()
        conclusion_str = parts[1].strip()

        # Parse premises
        premises = []
        for premise_str in premises_str.split(","):
            premise = self.parse_formula(premise_str.strip(), context)
            if premise:
                premises.append(premise)
            else:
                self.logger.warning(f"Failed to parse premise: {premise_str}")
                return None

        # Parse conclusion
        conclusion = self.parse_formula(conclusion_str, context)
        if not conclusion:
            self.logger.warning(f"Failed to parse conclusion: {conclusion_str}")
            return None

        # Create rule
        rule = Rule(
            symbol=rule_str,
            premises=premises,
            conclusion=conclusion,
            bidirectional=bidirectional
        )

        return rule

    def add_fact(self, fact: Union[Formula, str], confidence: float = 1.0,
                source: Optional[str] = None) -> bool:
        """
        Add a fact to the knowledge base.

        Args:
            fact: The fact to add (either Formula object or string representation)
            confidence: Confidence value (0.0 to 1.0)
            source: Optional source information

        Returns:
            True if added successfully, False otherwise
        """
        if not self.initialized:
            self.initialize()

        # Parse if string
        if isinstance(fact, str):
            formula = self.parse_formula(fact)
            if not formula:
                return False
        else:
            formula = fact

        # Add to knowledge base
        return self.knowledge_base.add_fact(formula, confidence, source)

    def add_rule(self, rule: Union[Rule, str], confidence: float = 1.0,
                source: Optional[str] = None) -> bool:
        """
        Add a rule to the knowledge base.

        Args:
            rule: The rule to add (either Rule object or string representation)
            confidence: Confidence value (0.0 to 1.0)
            source: Optional source information

        Returns:
            True if added successfully, False otherwise
        """
        if not self.initialized:
            self.initialize()

        # Parse if string
        if isinstance(rule, str):
            rule_obj = self.parse_rule(rule)
            if not rule_obj:
                return False
        else:
            rule_obj = rule

        # Add to knowledge base
        return self.knowledge_base.add_rule(rule_obj, confidence, source)

    def query_facts(self, predicate: Optional[str] = None,
                   confidence_threshold: float = 0.0) -> List[Formula]:
        """
        Query facts from the knowledge base.

        Args:
            predicate: Optional predicate symbol to filter by
            confidence_threshold: Minimum confidence value

        Returns:
            List of matching facts
        """
        if not self.initialized:
            self.initialize()

        return self.knowledge_base.query_facts(predicate, confidence_threshold)

    def query_rules(self, conclusion_predicate: Optional[str] = None,
                   confidence_threshold: float = 0.0) -> List[Rule]:
        """
        Query rules from the knowledge base.

        Args:
            conclusion_predicate: Optional predicate symbol in the conclusion
            confidence_threshold: Minimum confidence value

        Returns:
            List of matching rules
        """
        if not self.initialized:
            self.initialize()

        return self.knowledge_base.query_rules(conclusion_predicate, confidence_threshold)

    def prove(self, goal: Union[Formula, str],
             strategy: InferenceStrategy = InferenceStrategy.BACKWARD_CHAINING,
             context: Optional[Dict[str, Any]] = None) -> InferenceResult:
        """
        Attempt to prove a goal using the specified inference strategy.

        Args:
            goal: The goal formula to prove
            strategy: The inference strategy to use
            context: Optional contextual information

        Returns:
            InferenceResult object with the proof result
        """
        if not self.initialized:
            self.initialize()

        # Parse goal if string
        if isinstance(goal, str):
            goal_formula = self.parse_formula(goal)
            if not goal_formula:
                return InferenceResult(
                    success=False,
                    message="Failed to parse goal formula"
                )
        else:
            goal_formula = goal

        # Prepare context
        if context is None:
            context = {}

        # Select strategy
        strategy_func = self.inference_strategies.get(strategy)
        if not strategy_func:
            return InferenceResult(
                success=False,
                message=f"Unsupported inference strategy: {strategy}"
            )

        # Record execution time
        start_time = time.time()

        # Execute strategy
        try:
            result = strategy_func(goal_formula, context)
            result.execution_time = time.time() - start_time

            # Update stats
            self.stats["proof_attempts"][strategy.name] += 1
            if result.success:
                self.stats["proof_successes"][strategy.name] += 1

            return result
        except Exception as e:
            self.logger.error(f"Error during proof: {str(e)}")
            return InferenceResult(
                success=False,
                message=f"Error during proof: {str(e)}",
                execution_time=time.time() - start_time
            )

    def check_consistency(self) -> Dict[str, Any]:
        """
        Check the consistency of the knowledge base.

        Returns:
            Dictionary with consistency information
        """
        if not self.initialized:
            self.initialize()

        contradictions = self.knowledge_base.get_contradictions()

        return {
            "consistent": len(contradictions) == 0,
            "contradictions": [{
                "fact1": str(fact1),
                "fact2": str(fact2),
                "fact1_id": fact1.entity_id,
                "fact2_id": fact2.entity_id
            } for fact1, fact2 in contradictions]
        }

    def resolve_contradictions(self, strategy: str = "confidence") -> int:
        """
        Resolve contradictions in the knowledge base.

        Args:
            strategy: Strategy for resolving contradictions
                     ("confidence", "recency", "custom")

        Returns:
            Number of contradictions resolved
        """
        if not self.initialized:
            self.initialize()

        contradictions = self.knowledge_base.get_contradictions()
        resolved = 0

        for fact1, fact2 in contradictions:
            fact1_id = fact1.entity_id
            fact2_id = fact2.entity_id

            if strategy == "confidence":
                # Keep the fact with higher confidence
                conf1 = self.knowledge_base.confidence.get(fact1_id, 0.0)
                conf2 = self.knowledge_base.confidence.get(fact2_id, 0.0)
                keep_id = fact1_id if conf1 >= conf2 else fact2_id

            elif strategy == "recency":
                # Keep the most recent fact
                time1 = self.knowledge_base.timestamps.get(fact1_id, 0.0)
                time2 = self.knowledge_base.timestamps.get(fact2_id, 0.0)
                keep_id = fact1_id if time1 >= time2 else fact2_id

            else:
                # Default to confidence
                conf1 = self.knowledge_base.confidence.get(fact1_id, 0.0)
                conf2 = self.knowledge_base.confidence.get(fact2_id, 0.0)
                keep_id = fact1_id if conf1 >= conf2 else fact2_id

            self.knowledge_base.resolve_contradiction(fact1_id, fact2_id, keep_id)
            resolved += 1

        return resolved

    def clear_knowledge_base(self) -> None:
        """Clear the knowledge base"""
        if not self.initialized:
            self.initialize()

        self.knowledge_base.clear()

    def get_stats(self) -> Dict[str, Any]:
        """Get reasoning statistics"""
        return {k: dict(v) for k, v in self.stats.items()}

    def register_inference_rule(self, rule_name: str, rule_func: Callable) -> bool:
        """
        Register a custom inference rule.

        Args:
            rule_name: Name of the inference rule
            rule_func: Function implementing the rule

        Returns:
            True if registered successfully, False otherwise
        """
        with self.lock:
            self.inference_rules[rule_name] = rule_func
            self.logger.info(f"Registered inference rule: {rule_name}")
            return True

    def register_inference_strategy(self, strategy: InferenceStrategy,
                                  strategy_func: Callable) -> bool:
        """
        Register a custom inference strategy.

        Args:
            strategy: The inference strategy type
            strategy_func: Function implementing the strategy

        Returns:
            True if registered successfully, False otherwise
        """
        with self.lock:
            self.inference_strategies[strategy] = strategy_func
            self.logger.info(f"Registered inference strategy: {strategy.name}")
            return True

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.symbolic_parser:
                dependencies.append("symbolic_parser")
            if self.graph_engine:
                dependencies.append("graph_engine")

            self.nexus_core.register_module(
                name="symbolic_engine",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _register_standard_inference_rules(self) -> None:
        """Register standard logical inference rules"""
        # Modus Ponens: A, A => B ⊢ B
        self.register_inference_rule("modus_ponens", self._apply_modus_ponens)

        # Modus Tollens: ¬B, A => B ⊢ ¬A
        self.register_inference_rule("modus_tollens", self._apply_modus_tollens)

        # Conjunction Introduction: A, B ⊢ A ∧ B
        self.register_inference_rule("conjunction_introduction", self._apply_conjunction_introduction)

        # Conjunction Elimination: A ∧ B ⊢ A, B
        self.register_inference_rule("conjunction_elimination", self._apply_conjunction_elimination)

        # Disjunction Introduction: A ⊢ A ∨ B
        self.register_inference_rule("disjunction_introduction", self._apply_disjunction_introduction)

        # Disjunction Elimination: A ∨ B, ¬A ⊢ B
        self.register_inference_rule("disjunction_elimination", self._apply_disjunction_elimination)

        # Double Negation Elimination: ¬¬A ⊢ A
        self.register_inference_rule("double_negation_elimination", self._apply_double_negation_elimination)

    def _register_inference_strategies(self) -> None:
        """Register inference strategies"""
        self.register_inference_strategy(InferenceStrategy.FORWARD_CHAINING, self._forward_chaining)
        self.register_inference_strategy(InferenceStrategy.BACKWARD_CHAINING, self._backward_chaining)
        self.register_inference_strategy(InferenceStrategy.RESOLUTION, self._resolution)
        self.register_inference_strategy(InferenceStrategy.HYBRID, self._hybrid_inference)

    def _parse_result_to_formula(self, parse_result: ParseResult) -> Formula:
        """
        Convert a parse result to a Formula object.

        Args:
            parse_result: The parse result from symbolic parser

        Returns:
            Formula object representing the parsed expression
        """
        # This is a simplified implementation
        # A real implementation would carefully convert the AST

        if not parse_result.ast:
            raise ValueError("Parse result has no AST")

        ast_node = parse_result.ast

        if ast_node.node_type == "equation":
            # Handle equation (A = B)
            if len(ast_node.children) != 2:
                raise ValueError("Equation must have exactly 2 children")

            left = self._ast_node_to_term(ast_node.children[0])
            right = self._ast_node_to_term(ast_node.children[1])

            return Formula(
                entity_type=SymbolicEntityType.ATOM,
                symbol="equals",
                arguments=[left, right]
            )

        elif ast_node.node_type == "operation":
            # Handle logical operations (AND, OR, NOT, etc.)
            op = ast_node.value.upper()

            children = []
            for child in ast_node.children:
                children.append(self._parse_result_to_formula(ParseResult(
                    success=True,
                    expr_type=SymbolicType.LOGICAL,
                    ast=child
                )))

            return Formula(
                entity_type=SymbolicEntityType.FORMULA,
                symbol=op,
                operator=op,
                arguments=children
            )

        elif ast_node.node_type == "function" or ast_node.node_type == "atom":
            # Handle predicate application (P(x, y))
            func_name = ast_node.value

            args = []
            for child in ast_node.children:
                args.append(self._ast_node_to_term(child))

            return Formula(
                entity_type=SymbolicEntityType.ATOM,
                symbol=func_name,
                arguments=args
            )

        # Default case
        return Formula(
            entity_type=SymbolicEntityType.FORMULA,
            symbol=str(ast_node)
        )

    def _ast_node_to_term(self, ast_node) -> Term:
        """
        Convert an AST node to a Term object.

        Args:
            ast_node: The AST node to convert

        Returns:
            Term object representing the AST node
        """
        if ast_node.node_type == "variable":
            return Term(
                entity_type=SymbolicEntityType.VARIABLE,
                symbol=ast_node.value
            )

        elif ast_node.node_type == "constant" or ast_node.node_type == "number":
            return Term(
                entity_type=SymbolicEntityType.CONSTANT,
                symbol=str(ast_node.value)
            )

        elif ast_node.node_type == "function":
            func_name = ast_node.value

            args = []
            for child in ast_node.children:
                args.append(self._ast_node_to_term(child))

            return Term(
                entity_type=SymbolicEntityType.TERM,
                symbol=func_name,
                arguments=args
            )

        # Default case
        return Term(
            entity_type=SymbolicEntityType.CONSTANT,
            symbol=str(ast_node)
        )

    # -----------------------------------------------------------------------
    # Inference rules implementation
    # -----------------------------------------------------------------------

    def _apply_modus_ponens(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply modus ponens: A, A => B ⊢ B

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        # This is a simplified implementation
        # A real implementation would use unification and proper logical inference

        if len(premises) < 2:
            return None

        # For all pairs of premises
        for i, premise1 in enumerate(premises):
            for j, premise2 in enumerate(premises):
                if i == j:
                    continue

                # Check if premise2 is an implication
                if (premise2.entity_type == SymbolicEntityType.FORMULA and
                    premise2.operator == "IMPLIES" and
                    len(premise2.arguments) == 2):

                    # Check if premise1 matches the antecedent
                    antecedent = premise2.arguments[0]
                    if self._formulas_match(premise1, antecedent):
                        # Return the consequent
                        return premise2.arguments[1]

        return None

    def _apply_modus_tollens(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply modus tollens: ¬B, A => B ⊢ ¬A

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        # This is a simplified implementation
        # A real implementation would use unification and proper logical inference

        if len(premises) < 2:
            return None

        # For all pairs of premises
        for i, premise1 in enumerate(premises):
            for j, premise2 in enumerate(premises):
                if i == j:
                    continue

                # Check if premise1 is a negation
                if (premise1.entity_type == SymbolicEntityType.FORMULA and
                    premise1.operator == "NOT" and
                    len(premise1.arguments) == 1):

                    negated_formula = premise1.arguments[0]

                    # Check if premise2 is an implication
                    if (premise2.entity_type == SymbolicEntityType.FORMULA and
                        premise2.operator == "IMPLIES" and
                        len(premise2.arguments) == 2):

                        # Check if negated_formula matches the consequent
                        consequent = premise2.arguments[1]
                        if self._formulas_match(negated_formula, consequent):
                            # Return the negation of the antecedent
                            antecedent = premise2.arguments[0]
                            return Formula(
                                entity_type=SymbolicEntityType.FORMULA,
                                operator="NOT",
                                arguments=[antecedent]
                            )

        return None

    def _apply_conjunction_introduction(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply conjunction introduction: A, B ⊢ A ∧ B

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        if len(premises) < 2:
            return None

        # Take the first two premises and create a conjunction
        return Formula(
            entity_type=SymbolicEntityType.FORMULA,
            operator="AND",
            arguments=premises[:2]
        )

    def _apply_conjunction_elimination(self, premises: List[Formula]) -> Optional[List[Formula]]:
        """
        Apply conjunction elimination: A ∧ B ⊢ A, B

        Args:
            premises: List of premises to apply the rule to

        Returns:
            List of derived formulas, or None if the rule is not applicable
        """
        if not premises:
            return None

        # Look for a conjunction
        for premise in premises:
            if (premise.entity_type == SymbolicEntityType.FORMULA and
                premise.operator == "AND" and
                len(premise.arguments) >= 2):

                # Return the conjuncts
                return premise.arguments

        return None

    def _apply_disjunction_introduction(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply disjunction introduction: A ⊢ A ∨ B

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        if not premises:
            return None

        # Take the first premise and create a disjunction with a dummy formula
        dummy = Formula(
            entity_type=SymbolicEntityType.ATOM,
            symbol="dummy_formula"
        )

        return Formula(
            entity_type=SymbolicEntityType.FORMULA,
            operator="OR",
            arguments=[premises[0], dummy]
        )

    def _apply_disjunction_elimination(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply disjunction elimination: A ∨ B, ¬A ⊢ B

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        if len(premises) < 2:
            return None

        # For all pairs of premises
        for i, premise1 in enumerate(premises):
            for j, premise2 in enumerate(premises):
                if i == j:
                    continue

                # Check if premise1 is a disjunction
                if (premise1.entity_type == SymbolicEntityType.FORMULA and
                    premise1.operator == "OR" and
                    len(premise1.arguments) == 2):

                    disjunct1 = premise1.arguments[0]
                    disjunct2 = premise1.arguments[1]

                    # Check if premise2 is a negation of disjunct1
                    if (premise2.entity_type == SymbolicEntityType.FORMULA and
                        premise2.operator == "NOT" and
                        len(premise2.arguments) == 1 and
                        self._formulas_match(premise2.arguments[0], disjunct1)):

                        # Return disjunct2
                        return disjunct2

                    # Check if premise2 is a negation of disjunct2
                    if (premise2.entity_type == SymbolicEntityType.FORMULA and
                        premise2.operator == "NOT" and
                        len(premise2.arguments) == 1 and
                        self._formulas_match(premise2.arguments[0], disjunct2)):

                        # Return disjunct1
                        return disjunct1

        return None

    def _apply_double_negation_elimination(self, premises: List[Formula]) -> Optional[Formula]:
        """
        Apply double negation elimination: ¬¬A ⊢ A

        Args:
            premises: List of premises to apply the rule to

        Returns:
            The derived formula, or None if the rule is not applicable
        """
        if not premises:
            return None

        # Look for a double negation
        for premise in premises:
            if (premise.entity_type == SymbolicEntityType.FORMULA and
                premise.operator == "NOT" and
                len(premise.arguments) == 1 and
                premise.arguments[0].entity_type == SymbolicEntityType.FORMULA and
                premise.arguments[0].operator == "NOT" and
                len(premise.arguments[0].arguments) == 1):

                # Return the doubly negated formula
                return premise.arguments[0].arguments[0]

        return None

    def _formulas_match(self, formula1: Formula, formula2: Formula) -> bool:
        """
        Check if two formulas match (for rule application).
        This is a simplified implementation that just checks string equality.
        A real implementation would use unification.

        Args:
            formula1: First formula
            formula2: Second formula

        Returns:
            True if formulas match, False otherwise
        """
        return str(formula1) == str(formula2)

    # -----------------------------------------------------------------------
    # Inference strategies implementation
    # -----------------------------------------------------------------------

    def _forward_chaining(self, goal: Formula, context: Dict[str, Any]) -> InferenceResult:
        """
        Apply forward chaining inference to prove a goal.

        Args:
            goal: The goal formula to prove
            context: Contextual information for inference

        Returns:
            InferenceResult object with the proof result
        """
        # This is a simplified implementation
        # A real implementation would be more sophisticated

        # Initialize result
        result = InferenceResult(success=False)

        # Get all facts and rules
        facts = list(self.knowledge_base.facts.values())
        rules = list(self.knowledge_base.rules.values())

        # Keep track of derived formulas
        derived = set(str(fact) for fact in facts)
        fact_id_map = {str(fact): fact.entity_id for fact in facts}

        # Add facts to proved formulas
        result.proved_formulas.extend(facts)

        # Set a limit on inference steps to avoid infinite loops
        max_steps = context.get("max_steps", 100)
        steps = 0

        # Forward chaining loop
        while steps < max_steps:
            steps += 1
            new_fact_derived = False

            # Try to apply each rule
            for rule in rules:
                if not rule.premises or not rule.conclusion:
                    continue

                # Check if all premises are satisfied
                premises_satisfied = True
                premise_ids = []

                for premise in rule.premises:
                    if str(premise) not in derived:
                        premises_satisfied = False
                        break
                    premise_ids.append(fact_id_map.get(str(premise), "unknown"))

                if not premises_satisfied:
                    continue

                # Apply the rule to derive a new fact
                conclusion = rule.conclusion

                # Check if the conclusion is already derived
                if str(conclusion) in derived:
                    continue

                # Derive the conclusion
                derived.add(str(conclusion))
                fact_id_map[str(conclusion)] = conclusion.entity_id
                result.proved_formulas.append(conclusion)

                # Record inference step
                result.inference_steps.append(InferenceStep(
                    rule_id=rule.entity_id,
                    premises=premise_ids,
                    conclusion=conclusion.entity_id
                ))

                new_fact_derived = True

                # Check if the goal is reached
                if str(conclusion) == str(goal):
                    result.success = True
                    return result

            # If no new facts were derived, we've reached a fixed point
            if not new_fact_derived:
                break

        # Check if the goal is already in the derived facts
        if str(goal) in derived:
            result.success = True

        return result

    def _backward_chaining(self, goal: Formula, context: Dict[str, Any]) -> InferenceResult:
        """
        Apply backward chaining inference to prove a goal.

        Args:
            goal: The goal formula to prove
            context: Contextual information for inference

        Returns:
            InferenceResult object with the proof result
        """
        # This is a simplified implementation
        # A real implementation would be more sophisticated

        # Initialize result
        result = InferenceResult(success=False)

        # Get all facts
        facts = list(self.knowledge_base.facts.values())

        # Initialize agenda with the goal
        agenda = [goal]

        # Keep track of proved formulas and already processed goals
        proved = {str(fact): fact for fact in facts}
        processed_goals = set()

        # Keep track of inference steps
        inference_steps = []

        # Set a limit on inference steps to avoid infinite loops
        max_steps = context.get("max_steps", 100)
        steps = 0

        # Backward chaining loop
        while agenda and steps < max_steps:
            steps += 1
            current_goal = agenda.pop()

            # Skip if already processed
            if str(current_goal) in processed_goals:
                continue

            processed_goals.add(str(current_goal))

            # Check if the goal is already proved
            if str(current_goal) in proved:
                continue

            # Get relevant rules (rules whose conclusion matches the goal)
            relevant_rules = []
            for rule in self.knowledge_base.rules.values():
                if rule.conclusion and self._formulas_match(rule.conclusion, current_goal):
                    relevant_rules.append(rule)

            # Try each relevant rule
            goal_proved = False

            for rule in relevant_rules:
                # Check if all premises can be proved
                all_premises_proved = True
                rule_premises = []

                for premise in rule.premises:
                    if str(premise) in proved:
                        # Premise already proved
                        rule_premises.append(proved[str(premise)])
                    else:
                        # Add premise to agenda
                        agenda.append(premise)
                        all_premises_proved = False

                if all_premises_proved:
                    # All premises are proved, so the current goal is proved
                    proved[str(current_goal)] = current_goal

                    # Record inference step
                    inference_steps.append(InferenceStep(
                        rule_id=rule.entity_id,
                        premises=[premise.entity_id for premise in rule_premises],
                        conclusion=current_goal.entity_id
                    ))

                    goal_proved = True
                    break

            # If no rule could prove the goal, it remains unproved
            if not goal_proved:
                # Try to find the goal in the facts
                for fact in facts:
                    if self._formulas_match(fact, current_goal):
                        proved[str(current_goal)] = current_goal
                        goal_proved = True
                        break

            # If still not proved, try to apply inference rules directly
            if not goal_proved:
                for rule_name, rule_func in self.inference_rules.items():
                    derived = rule_func([proved[str(p)] for p in proved])
                    if derived and self._formulas_match(derived, current_goal):
                        proved[str(current_goal)] = current_goal
                        goal_proved = True
                        break

        # Check if the original goal was proved
        result.success = str(goal) in proved

        # Add all proved formulas to the result
        result.proved_formulas = list(proved.values())

        # Add inference steps to the result
        result.inference_steps = inference_steps

        return result

    def _resolution(self, goal: Formula, context: Dict[str, Any]) -> InferenceResult:
        """
        Apply resolution inference to prove a goal.

        Args:
            goal: The goal formula to prove
            context: Contextual information for inference

        Returns:
            InferenceResult object with the proof result
        """
        # Resolution is more complex and typically used for refutation proofs
        # This is a very simplified placeholder implementation

        # For now, fall back to backward chaining
        self.logger.warning("Resolution strategy not fully implemented, falling back to backward chaining")
        return self._backward_chaining(goal, context)

    def _hybrid_inference(self, goal: Formula, context: Dict[str, Any]) -> InferenceResult:
        """
        Apply a hybrid inference strategy to prove a goal.

        Args:
            goal: The goal formula to prove
            context: Contextual information for inference

        Returns:
            InferenceResult object with the proof result
        """
        # Try backward chaining first
        result = self._backward_chaining(goal, context)

        # If successful, return the result
        if result.success:
            return result

        # If not, try forward chaining
        forward_result = self._forward_chaining(goal, context)

        # If forward chaining is successful, return that result
        if forward_result.success:
            return forward_result

        # If neither succeeded, return the backward chaining result
        # (could also choose to return a combined result)
        return result

ModuleNotFoundError: No module named 'grandnexus'

# Code Analyzer

In [ ]:
"""
code_analyzer.py - Code Analysis Module for GrandNexus

This module provides functionality for analyzing, understanding, and processing
code structures across various programming languages. It serves as a specialized
component of the perception system, focusing on code as a form of structured input.

The module supports:
- Code structure analysis (AST generation and traversal)
- Semantic understanding of code components
- Pattern recognition in coding structures
- Integration with symbolic and neural reasoning systems
- Cross-language code analysis (Python, JavaScript, etc.)

Key design principles:
- Language-agnostic where possible, specialized where necessary
- Progressive understanding (from syntax to semantics to intent)
- Support for both static and dynamic code analysis
- Extensible plugin system for different programming languages
"""

import logging
import ast
import re
import uuid
import threading
import time
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Set, Tuple, Union, Callable
from collections import defaultdict, deque

# Assuming these exist or will exist in the GrandNexus architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.working import WorkingMemory

# Try to import optional dependencies
try:
    import astroid
    HAS_ASTROID = True
except ImportError:
    HAS_ASTROID = False

try:
    import esprima
    HAS_ESPRIMA = True
except ImportError:
    HAS_ESPRIMA = False

class CodeLanguage(Enum):
    """Supported programming languages for code analysis"""
    PYTHON = auto()
    JAVASCRIPT = auto()
    JAVA = auto()
    CPP = auto()
    RUST = auto()
    GO = auto()
    RUBY = auto()
    PHP = auto()
    CSHARP = auto()
    TYPESCRIPT = auto()
    HTML = auto()
    CSS = auto()
    SQL = auto()
    SHELL = auto()
    UNKNOWN = auto()

class CodeNodeType(Enum):
    """Types of code nodes in the unified AST representation"""
    MODULE = auto()          # Complete module/file
    FUNCTION = auto()        # Function/method definition
    CLASS = auto()           # Class/struct definition
    VARIABLE = auto()        # Variable declaration/reference
    EXPRESSION = auto()      # Expression
    STATEMENT = auto()       # Statement (e.g., assignment)
    CONTROL_FLOW = auto()    # Control flow (if, for, while)
    IMPORT = auto()          # Import/include statement
    LITERAL = auto()         # Literal value (string, number)
    OPERATOR = auto()        # Operator (+, -, etc.)
    COMMENT = auto()         # Comment
    DECORATOR = auto()       # Decorator/annotation
    ARGUMENT = auto()        # Function argument
    ATTRIBUTE = auto()       # Object attribute/property
    CALL = auto()            # Function/method call
    BLOCK = auto()           # Code block
    UNKNOWN = auto()         # Unknown node type

@dataclass
class CodeMetrics:
    """Metrics calculated from code analysis"""
    loc: int = 0                      # Lines of code
    lloc: int = 0                     # Logical lines of code
    comment_lines: int = 0            # Comment lines
    cyclomatic_complexity: int = 0    # Cyclomatic complexity
    function_count: int = 0           # Number of functions
    class_count: int = 0              # Number of classes
    parameter_count: Dict[str, int] = field(default_factory=dict)  # Parameters per function
    nesting_depth: int = 0            # Maximum nesting depth
    dependency_count: int = 0         # Number of dependencies/imports
    maintainability_index: float = 0.0  # Maintainability index
    halstead_metrics: Dict[str, float] = field(default_factory=dict)  # Halstead complexity metrics

@dataclass
class CodeNode:
    """Node in the unified code AST representation"""
    node_id: str
    node_type: CodeNodeType
    language: CodeLanguage
    name: Optional[str] = None
    value: Optional[Any] = None
    code: Optional[str] = None
    line_start: int = 0
    line_end: int = 0
    column_start: int = 0
    column_end: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)
    parent_id: Optional[str] = None
    children: List[str] = field(default_factory=list)  # List of child node IDs

    def __post_init__(self):
        """Initialize additional fields after creation"""
        self.creation_time = time.time()

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        return {
            "node_id": self.node_id,
            "node_type": self.node_type.name,
            "language": self.language.name,
            "name": self.name,
            "value": str(self.value) if self.value is not None else None,
            "code": self.code,
            "line_start": self.line_start,
            "line_end": self.line_end,
            "column_start": self.column_start,
            "column_end": self.column_end,
            "metadata": self.metadata,
            "parent_id": self.parent_id,
            "children": self.children
        }

@dataclass
class CodeAnalysisResult:
    """Result of a code analysis operation"""
    success: bool
    language: CodeLanguage
    root_node_id: Optional[str] = None
    metrics: Optional[CodeMetrics] = None
    error_message: Optional[str] = None
    warnings: List[Dict[str, Any]] = field(default_factory=list)
    timing: Dict[str, float] = field(default_factory=dict)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        """Initialize additional fields after creation"""
        self.id = str(uuid.uuid4())
        self.timestamp = time.time()

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary representation"""
        result = {
            "id": self.id,
            "timestamp": self.timestamp,
            "success": self.success,
            "language": self.language.name,
            "root_node_id": self.root_node_id,
            "timing": self.timing,
            "metadata": self.metadata
        }

        if self.error_message:
            result["error_message"] = self.error_message

        if self.warnings:
            result["warnings"] = self.warnings

        if self.metrics:
            result["metrics"] = {
                "loc": self.metrics.loc,
                "lloc": self.metrics.lloc,
                "comment_lines": self.metrics.comment_lines,
                "cyclomatic_complexity": self.metrics.cyclomatic_complexity,
                "function_count": self.metrics.function_count,
                "class_count": self.metrics.class_count,
                "parameter_count": self.metrics.parameter_count,
                "nesting_depth": self.metrics.nesting_depth,
                "dependency_count": self.metrics.dependency_count,
                "maintainability_index": self.metrics.maintainability_index,
                "halstead_metrics": self.metrics.halstead_metrics
            }

        return result

class CodeAnalyzer:
    """
    Core code analysis module for GrandNexus.

    This module provides functionality for parsing, analyzing, and understanding
    code across multiple programming languages. It serves as a specialized
    component of the perception subsystem focused on code structures.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None):
        """
        Initialize the code analyzer.

        Args:
            nexus_core: Reference to the NexusCore for system-wide coordination
            semantic_memory: Reference to semantic memory for concept storage
            working_memory: Reference to working memory for context
        """
        self.logger = logging.getLogger("GrandNexus.Perception.CodeAnalyzer")
        self.nexus_core = nexus_core
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Node storage
        self.nodes = {}  # Map of node_id -> CodeNode

        # Language parsers registry
        self.language_parsers = {}

        # Analysis plugins registry
        self.analysis_plugins = {}

        # Recent results cache
        self.result_cache = {}
        self.recent_results = deque(maxlen=20)

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"CodeAnalyzer initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the code analyzer and register language parsers"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register built-in language parsers
                self._register_default_parsers()

                # Register built-in analysis plugins
                self._register_default_plugins()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("CodeAnalyzer initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"CodeAnalyzer initialization failed: {str(e)}")
                return False

    def analyze(self, code: str, language: Optional[CodeLanguage] = None,
               filename: Optional[str] = None,
               analysis_options: Optional[Dict[str, Any]] = None) -> CodeAnalysisResult:
        """
        Analyze a code snippet or file.

        Args:
            code: The source code to analyze
            language: Optional language hint, will auto-detect if not provided
            filename: Optional filename for context
            analysis_options: Additional options for the analysis

        Returns:
            CodeAnalysisResult containing the analysis results
        """
        if not self.initialized:
            self.initialize()

        # Start timing
        start_time = time.time()
        timing = {}

        # Check cache for identical code
        cache_key = hash(code)
        if cache_key in self.result_cache:
            return self.result_cache[cache_key]

        # Prepare options
        if analysis_options is None:
            analysis_options = {}

        # Detect language if not provided
        detected_language = language
        if detected_language is None:
            detect_start = time.time()
            detected_language = self._detect_language(code, filename)
            timing["language_detection"] = time.time() - detect_start

        # Select the appropriate parser
        if detected_language in self.language_parsers:
            parser_func = self.language_parsers[detected_language]
            try:
                # Parse the code
                parse_start = time.time()
                parse_result = parser_func(code, filename, analysis_options)
                timing["parsing"] = time.time() - parse_start

                # Gather code metrics
                metrics_start = time.time()
                metrics = self._calculate_metrics(code, detected_language, parse_result["root_node_id"])
                timing["metrics"] = time.time() - metrics_start

                # Run analysis plugins
                plugins_start = time.time()
                warnings = self._run_analysis_plugins(detected_language, parse_result["root_node_id"], analysis_options)
                timing["plugins"] = time.time() - plugins_start

                # Create result
                result = CodeAnalysisResult(
                    success=True,
                    language=detected_language,
                    root_node_id=parse_result["root_node_id"],
                    metrics=metrics,
                    warnings=warnings,
                    timing=timing,
                    metadata={"filename": filename} if filename else {}
                )

                # Update metrics
                self.metrics["analysis_success_rate"].append(1.0)
                self.metrics["analysis_time"].append(time.time() - start_time)

                # Cache successful results
                self.result_cache[cache_key] = result
                self.recent_results.append(result.id)

                return result

            except Exception as e:
                self.logger.error(f"Error analyzing code: {str(e)}")
                self.metrics["analysis_success_rate"].append(0.0)
                return CodeAnalysisResult(
                    success=False,
                    language=detected_language,
                    error_message=f"Analysis error: {str(e)}",
                    timing=timing
                )
        else:
            self.logger.warning(f"No parser available for language: {detected_language}")
            self.metrics["analysis_success_rate"].append(0.0)
            return CodeAnalysisResult(
                success=False,
                language=detected_language,
                error_message=f"No parser available for language: {detected_language}",
                timing=timing
            )

    def get_node(self, node_id: str) -> Optional[CodeNode]:
        """
        Retrieve a code node by ID.

        Args:
            node_id: The ID of the node to retrieve

        Returns:
            The CodeNode if found, None otherwise
        """
        with self.lock:
            return self.nodes.get(node_id)

    def get_subtree(self, node_id: str) -> Dict[str, CodeNode]:
        """
        Retrieve a node and all its descendants.

        Args:
            node_id: The ID of the root node

        Returns:
            Dictionary mapping node IDs to CodeNodes
        """
        with self.lock:
            subtree = {}
            self._collect_subtree(node_id, subtree)
            return subtree

    def find_nodes(self, criteria: Dict[str, Any]) -> List[CodeNode]:
        """
        Find nodes matching specified criteria.

        Args:
            criteria: Dictionary of criteria to match against node properties

        Returns:
            List of matching CodeNodes
        """
        with self.lock:
            results = []

            for node in self.nodes.values():
                match = True
                for key, value in criteria.items():
                    if key == "node_type":
                        if isinstance(value, CodeNodeType):
                            if node.node_type != value:
                                match = False
                                break
                        elif isinstance(value, str):
                            if node.node_type.name != value:
                                match = False
                                break
                    elif key == "language":
                        if isinstance(value, CodeLanguage):
                            if node.language != value:
                                match = False
                                break
                        elif isinstance(value, str):
                            if node.language.name != value:
                                match = False
                                break
                    elif hasattr(node, key):
                        if getattr(node, key) != value:
                            match = False
                            break
                    elif key in node.metadata:
                        if node.metadata[key] != value:
                            match = False
                            break
                    else:
                        match = False
                        break

                if match:
                    results.append(node)

            return results

    def register_language_parser(self, language: CodeLanguage,
                                parser_func: Callable[[str, Optional[str], Dict[str, Any]], Dict[str, Any]]) -> None:
        """
        Register a parser for a specific language.

        Args:
            language: The language this parser handles
            parser_func: Function that takes (code, filename, options) and returns parse result
        """
        with self.lock:
            self.language_parsers[language] = parser_func
            self.logger.info(f"Registered parser for language: {language}")

    def register_analysis_plugin(self, plugin_id: str,
                                plugin_func: Callable[[CodeLanguage, str, Dict[str, Any]], List[Dict[str, Any]]]) -> None:
        """
        Register an analysis plugin.

        Args:
            plugin_id: Unique identifier for the plugin
            plugin_func: Function that takes (language, root_node_id, options) and returns warnings
        """
        with self.lock:
            self.analysis_plugins[plugin_id] = plugin_func
            self.logger.info(f"Registered analysis plugin: {plugin_id}")

    def get_analysis_metrics(self) -> Dict[str, Any]:
        """Get current analysis performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def _register_default_parsers(self) -> None:
        """Register built-in language parsers"""
        self.register_language_parser(CodeLanguage.PYTHON, self._parse_python)

        if HAS_ESPRIMA:
            self.register_language_parser(CodeLanguage.JAVASCRIPT, self._parse_javascript)

        # Other language parsers would be registered here

    def _register_default_plugins(self) -> None:
        """Register built-in analysis plugins"""
        self.register_analysis_plugin("complexity_checker", self._plugin_complexity_checker)
        self.register_analysis_plugin("variable_naming", self._plugin_variable_naming)
        self.register_analysis_plugin("security_checker", self._plugin_security_checker)

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.working_memory:
                dependencies.append("working_memory")

            self.nexus_core.register_module(
                name="code_analyzer",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _detect_language(self, code: str, filename: Optional[str] = None) -> CodeLanguage:
        """
        Detect the programming language of the code.

        Args:
            code: The source code to analyze
            filename: Optional filename for context

        Returns:
            Detected CodeLanguage
        """
        # First check file extension if available
        if filename:
            ext = filename.split('.')[-1].lower()
            if ext in ['py', 'pyw']:
                return CodeLanguage.PYTHON
            elif ext in ['js']:
                return CodeLanguage.JAVASCRIPT
            elif ext in ['java']:
                return CodeLanguage.JAVA
            elif ext in ['cpp', 'cc', 'cxx', 'c++']:
                return CodeLanguage.CPP
            elif ext in ['rs']:
                return CodeLanguage.RUST
            elif ext in ['go']:
                return CodeLanguage.GO
            elif ext in ['rb']:
                return CodeLanguage.RUBY
            elif ext in ['php']:
                return CodeLanguage.PHP
            elif ext in ['cs']:
                return CodeLanguage.CSHARP
            elif ext in ['ts']:
                return CodeLanguage.TYPESCRIPT
            elif ext in ['html', 'htm']:
                return CodeLanguage.HTML
            elif ext in ['css']:
                return CodeLanguage.CSS
            elif ext in ['sql']:
                return CodeLanguage.SQL
            elif ext in ['sh', 'bash']:
                return CodeLanguage.SHELL

        # Language detection heuristics based on code content

        # Python indicators
        if re.search(r'\bdef\s+\w+\s*\(', code) or re.search(r'\bimport\s+\w+', code) or re.search(r'\bclass\s+\w+\s*:', code):
            return CodeLanguage.PYTHON

        # JavaScript indicators
        if re.search(r'\bfunction\s+\w+\s*\(', code) or re.search(r'\bconst\s+\w+\s*=', code) or re.search(r'\blet\s+\w+\s*=', code):
            return CodeLanguage.JAVASCRIPT

        # Java indicators
        if re.search(r'\bpublic\s+class\s+\w+', code) or re.search(r'\bprivate\s+\w+\s+\w+\s*\(', code):
            return CodeLanguage.JAVA

        # C++ indicators
        if re.search(r'\#include\s*<\w+>', code) or re.search(r'\bstd::', code):
            return CodeLanguage.CPP

        # Default to UNKNOWN if no patterns match
        return CodeLanguage.UNKNOWN

    def _parse_python(self, code: str, filename: Optional[str],
                     options: Dict[str, Any]) -> Dict[str, Any]:
        """
        Parse Python code using Python's built-in ast module.

        Args:
            code: Python source code
            filename: Optional filename for context
            options: Additional parsing options

        Returns:
            Dictionary with parsing results
        """
        try:
            # Parse the code with Python's ast module
            tree = ast.parse(code, filename or "<string>")

            # Transform to our unified representation
            root_id = self._create_node_id()

            # Create the root module node
            root_node = CodeNode(
                node_id=root_id,
                node_type=CodeNodeType.MODULE,
                language=CodeLanguage.PYTHON,
                name=filename or "<string>",
                code=code,
                line_start=0,
                line_end=len(code.splitlines())
            )

            # Store the node
            self.nodes[root_id] = root_node

            # Process the AST recursively
            self._process_python_ast(tree, root_id, code)

            return {
                "root_node_id": root_id,
                "language": CodeLanguage.PYTHON
            }

        except SyntaxError as e:
            self.logger.error(f"Python syntax error: {str(e)}")
            raise ValueError(f"Python syntax error: {str(e)}")

        except Exception as e:
            self.logger.error(f"Error parsing Python code: {str(e)}")
            raise

    def _parse_javascript(self, code: str, filename: Optional[str],
                         options: Dict[str, Any]) -> Dict[str, Any]:
        """
        Parse JavaScript code using esprima (if available).

        Args:
            code: JavaScript source code
            filename: Optional filename for context
            options: Additional parsing options

        Returns:
            Dictionary with parsing results
        """
        if not HAS_ESPRIMA:
            raise ImportError("esprima is required for JavaScript parsing")

        try:
            # Parse the code with esprima
            tree = esprima.parseScript(code, {'loc': True, 'comment': True})

            # Transform to our unified representation
            root_id = self._create_node_id()

            # Create the root module node
            root_node = CodeNode(
                node_id=root_id,
                node_type=CodeNodeType.MODULE,
                language=CodeLanguage.JAVASCRIPT,
                name=filename or "<string>",
                code=code,
                line_start=0,
                line_end=len(code.splitlines())
            )

            # Store the node
            self.nodes[root_id] = root_node

            # Process the AST recursively
            self._process_javascript_ast(tree, root_id, code)

            return {
                "root_node_id": root_id,
                "language": CodeLanguage.JAVASCRIPT
            }

        except Exception as e:
            self.logger.error(f"Error parsing JavaScript code: {str(e)}")
            raise

    def _process_python_ast(self, node: ast.AST, parent_id: str, code: str) -> None:
        """
        Process a Python AST node and add it to our representation.

        Args:
            node: Python AST node
            parent_id: ID of the parent node
            code: Original source code
        """
        # Skip if node is None
        if node is None:
            return

        # Get the parent node
        parent_node = self.nodes[parent_id]

        # Handle different node types
        if isinstance(node, ast.Module):
            # Module already created, just process children
            for child in node.body:
                self._process_python_ast(child, parent_id, code)

        elif isinstance(node, ast.FunctionDef):
            # Create function node
            node_id = self._create_node_id()

            func_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.FUNCTION,
                language=CodeLanguage.PYTHON,
                name=node.name,
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "is_async": isinstance(node, ast.AsyncFunctionDef),
                    "decorators": len(node.decorator_list)
                }
            )

            # Extract function code if we have line numbers
            if hasattr(node, 'end_lineno'):
                lines = code.splitlines()
                if 0 <= node.lineno - 1 < len(lines) and 0 <= node.end_lineno - 1 < len(lines):
                    func_code = '\n'.join(lines[node.lineno - 1:node.end_lineno])
                    func_node.code = func_code

            # Store the node
            self.nodes[node_id] = func_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process arguments
            if hasattr(node, 'args'):
                args_id = self._create_node_id()
                args_node = CodeNode(
                    node_id=args_id,
                    node_type=CodeNodeType.ARGUMENT,
                    language=CodeLanguage.PYTHON,
                    name="arguments",
                    parent_id=node_id,
                    metadata={
                        "arg_count": len(node.args.args),
                        "has_vararg": node.args.vararg is not None,
                        "has_kwarg": node.args.kwarg is not None
                    }
                )
                self.nodes[args_id] = args_node
                func_node.children.append(args_id)

                # Process individual arguments
                for arg in node.args.args:
                    arg_id = self._create_node_id()
                    arg_node = CodeNode(
                        node_id=arg_id,
                        node_type=CodeNodeType.ARGUMENT,
                        language=CodeLanguage.PYTHON,
                        name=arg.arg,
                        parent_id=args_id,
                        line_start=getattr(arg, 'lineno', 0),
                        column_start=getattr(arg, 'col_offset', 0)
                    )
                    self.nodes[arg_id] = arg_node
                    args_node.children.append(arg_id)

            # Process function body
            for child in node.body:
                self._process_python_ast(child, node_id, code)

        elif isinstance(node, ast.ClassDef):
            # Create class node
            node_id = self._create_node_id()

            class_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.CLASS,
                language=CodeLanguage.PYTHON,
                name=node.name,
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "bases": [self._get_name_from_python_expr(base) for base in node.bases],
                    "decorators": len(node.decorator_list)
                }
            )

            # Extract class code if we have line numbers
            if hasattr(node, 'end_lineno'):
                lines = code.splitlines()
                if 0 <= node.lineno - 1 < len(lines) and 0 <= node.end_lineno - 1 < len(lines):
                    class_code = '\n'.join(lines[node.lineno - 1:node.end_lineno])
                    class_node.code = class_code

            # Store the node
            self.nodes[node_id] = class_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process class body
            for child in node.body:
                self._process_python_ast(child, node_id, code)

        elif isinstance(node, ast.Assign):
            # Create assignment node
            node_id = self._create_node_id()

            # Get target names (simplistic, just for display)
            target_names = []
            for target in node.targets:
                if isinstance(target, ast.Name):
                    target_names.append(target.id)
                elif isinstance(target, ast.Attribute):
                    target_names.append(f"{self._get_name_from_python_expr(target.value)}.{target.attr}")
                else:
                    target_names.append("complex_target")

            assign_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.STATEMENT,
                language=CodeLanguage.PYTHON,
                name=",".join(target_names),
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "statement_type": "assignment",
                    "targets": target_names
                }
            )

            # Store the node
            self.nodes[node_id] = assign_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process value
            value_id = self._create_node_id()
            value_node = CodeNode(
                node_id=value_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                parent_id=node_id,
                line_start=getattr(node.value, 'lineno', node.lineno),
                column_start=getattr(node.value, 'col_offset', 0)
            )
            self.nodes[value_id] = value_node
            assign_node.children.append(value_id)

            # Process the value expression
            self._process_python_ast(node.value, value_id, code)

        elif isinstance(node, ast.If):
            # Create if statement node
            node_id = self._create_node_id()

            if_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.CONTROL_FLOW,
                language=CodeLanguage.PYTHON,
                name="if",
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "control_type": "if",
                    "has_else": bool(node.orelse)
                }
            )

            # Store the node
            self.nodes[node_id] = if_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process test condition
            test_id = self._create_node_id()
            test_node = CodeNode(
                node_id=test_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                name="condition",
                parent_id=node_id,
                line_start=getattr(node.test, 'lineno', node.lineno),
                column_start=getattr(node.test, 'col_offset', 0)
            )
            self.nodes[test_id] = test_node
            if_node.children.append(test_id)

            # Process test expression
            self._process_python_ast(node.test, test_id, code)

            # Process if body
            body_id = self._create_node_id()
            body_node = CodeNode(
                node_id=body_id,
                node_type=CodeNodeType.BLOCK,
                language=CodeLanguage.PYTHON,
                name="if_body",
                parent_id=node_id
            )
            self.nodes[body_id] = body_node
            if_node.children.append(body_id)

            for child in node.body:
                self._process_python_ast(child, body_id, code)

            # Process else body if present
            if node.orelse:
                orelse_id = self._create_node_id()
                orelse_node = CodeNode(
                    node_id=orelse_id,
                    node_type=CodeNodeType.BLOCK,
                    language=CodeLanguage.PYTHON,
                    name="else_body",
                    parent_id=node_id
                )
                self.nodes[orelse_id] = orelse_node
                if_node.children.append(orelse_id)

                for child in node.orelse:
                    self._process_python_ast(child, orelse_id, code)

        elif isinstance(node, ast.For) or isinstance(node, ast.AsyncFor):
            # Create for loop node
            node_id = self._create_node_id()

            for_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.CONTROL_FLOW,
                language=CodeLanguage.PYTHON,
                name="for",
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "control_type": "for",
                    "is_async": isinstance(node, ast.AsyncFor),
                    "has_else": bool(node.orelse)
                }
            )

            # Store the node
            self.nodes[node_id] = for_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process target
            target_id = self._create_node_id()
            target_node = CodeNode(
                node_id=target_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                name="target",
                parent_id=node_id,
                line_start=getattr(node.target, 'lineno', node.lineno),
                column_start=getattr(node.target, 'col_offset', 0)
            )
            self.nodes[target_id] = target_node
            for_node.children.append(target_id)

            # Process target expression
            self._process_python_ast(node.target, target_id, code)

            # Process iter
            iter_id = self._create_node_id()
            iter_node = CodeNode(
                node_id=iter_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                name="iter",
                parent_id=node_id,
                line_start=getattr(node.iter, 'lineno', node.lineno),
                column_start=getattr(node.iter, 'col_offset', 0)
            )
            self.nodes[iter_id] = iter_node
            for_node.children.append(iter_id)

            # Process iter expression
            self._process_python_ast(node.iter, iter_id, code)

            # Process for body
            body_id = self._create_node_id()
            body_node = CodeNode(
                node_id=body_id,
                node_type=CodeNodeType.BLOCK,
                language=CodeLanguage.PYTHON,
                name="for_body",
                parent_id=node_id
            )
            self.nodes[body_id] = body_node
            for_node.children.append(body_id)

            for child in node.body:
                self._process_python_ast(child, body_id, code)

            # Process else body if present
            if node.orelse:
                orelse_id = self._create_node_id()
                orelse_node = CodeNode(
                    node_id=orelse_id,
                    node_type=CodeNodeType.BLOCK,
                    language=CodeLanguage.PYTHON,
                    name="else_body",
                    parent_id=node_id
                )
                self.nodes[orelse_id] = orelse_node
                for_node.children.append(orelse_id)

                for child in node.orelse:
                    self._process_python_ast(child, orelse_id, code)

        elif isinstance(node, ast.Call):
            # Create function call node
            node_id = self._create_node_id()

            func_name = self._get_name_from_python_expr(node.func)

            call_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.CALL,
                language=CodeLanguage.PYTHON,
                name=func_name,
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "arg_count": len(node.args),
                    "kwarg_count": len(node.keywords)
                }
            )

            # Store the node
            self.nodes[node_id] = call_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process function expression
            func_id = self._create_node_id()
            func_node = CodeNode(
                node_id=func_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                name="function",
                parent_id=node_id,
                line_start=getattr(node.func, 'lineno', node.lineno),
                column_start=getattr(node.func, 'col_offset', 0)
            )
            self.nodes[func_id] = func_node
            call_node.children.append(func_id)

            self._process_python_ast(node.func, func_id, code)

            # Process arguments
            if node.args:
                args_id = self._create_node_id()
                args_node = CodeNode(
                    node_id=args_id,
                    node_type=CodeNodeType.ARGUMENT,
                    language=CodeLanguage.PYTHON,
                    name="arguments",
                    parent_id=node_id
                )
                self.nodes[args_id] = args_node
                call_node.children.append(args_id)

                for arg in node.args:
                    self._process_python_ast(arg, args_id, code)

            # Process keyword arguments
            if node.keywords:
                kwargs_id = self._create_node_id()
                kwargs_node = CodeNode(
                    node_id=kwargs_id,
                    node_type=CodeNodeType.ARGUMENT,
                    language=CodeLanguage.PYTHON,
                    name="keyword_arguments",
                    parent_id=node_id
                )
                self.nodes[kwargs_id] = kwargs_node
                call_node.children.append(kwargs_id)

                for kw in node.keywords:
                    kw_id = self._create_node_id()
                    kw_name = kw.arg if kw.arg else "**"
                    kw_node = CodeNode(
                        node_id=kw_id,
                        node_type=CodeNodeType.ARGUMENT,
                        language=CodeLanguage.PYTHON,
                        name=kw_name,
                        parent_id=kwargs_id,
                        line_start=getattr(kw, 'lineno', 0),
                        column_start=getattr(kw, 'col_offset', 0)
                    )
                    self.nodes[kw_id] = kw_node
                    kwargs_node.children.append(kw_id)

                    self._process_python_ast(kw.value, kw_id, code)

        elif isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
            # Create import node
            node_id = self._create_node_id()

            if isinstance(node, ast.Import):
                import_names = [name.name for name in node.names]
                module_name = None
            else:  # ImportFrom
                import_names = [name.name for name in node.names]
                module_name = node.module

            import_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.IMPORT,
                language=CodeLanguage.PYTHON,
                name=', '.join(import_names),
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id,
                metadata={
                    "is_from": isinstance(node, ast.ImportFrom),
                    "module": module_name,
                    "level": getattr(node, 'level', 0) if isinstance(node, ast.ImportFrom) else 0,
                    "names": import_names
                }
            )

            # Store the node
            self.nodes[node_id] = import_node

            # Add to parent's children
            parent_node.children.append(node_id)

        elif isinstance(node, ast.Expr):
            # Create expression statement node
            node_id = self._create_node_id()

            expr_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.EXPRESSION,
                language=CodeLanguage.PYTHON,
                line_start=node.lineno,
                line_end=getattr(node, 'end_lineno', node.lineno) if hasattr(node, 'end_lineno') else node.lineno,
                column_start=node.col_offset,
                column_end=getattr(node, 'end_col_offset', node.col_offset) if hasattr(node, 'end_col_offset') else node.col_offset,
                parent_id=parent_id
            )

            # Store the node
            self.nodes[node_id] = expr_node

            # Add to parent's children
            parent_node.children.append(node_id)

            # Process the expression value
            self._process_python_ast(node.value, node_id, code)

        elif isinstance(node, ast.Constant):
            # Create literal node for constant (Python 3.8+)
            node_id = self._create_node_id()

            # Determine the type of constant
            value_type = type(node.value).__name__

            literal_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.LITERAL,
                language=CodeLanguage.PYTHON,
                value=node.value,
                line_start=getattr(node, 'lineno', 0),
                line_end=getattr(node, 'end_lineno', getattr(node, 'lineno', 0)) if hasattr(node, 'end_lineno') else getattr(node, 'lineno', 0),
                column_start=getattr(node, 'col_offset', 0),
                column_end=getattr(node, 'end_col_offset', getattr(node, 'col_offset', 0)) if hasattr(node, 'end_col_offset') else getattr(node, 'col_offset', 0),
                parent_id=parent_id,
                metadata={
                    "value_type": value_type
                }
            )

            # Store the node
            self.nodes[node_id] = literal_node

            # Add to parent's children
            parent_node.children.append(node_id)

        elif isinstance(node, ast.Name):
            # Create variable reference node
            node_id = self._create_node_id()

            name_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.VARIABLE,
                language=CodeLanguage.PYTHON,
                name=node.id,
                line_start=getattr(node, 'lineno', 0),
                line_end=getattr(node, 'end_lineno', getattr(node, 'lineno', 0)) if hasattr(node, 'end_lineno') else getattr(node, 'lineno', 0),
                column_start=getattr(node, 'col_offset', 0),
                column_end=getattr(node, 'end_col_offset', getattr(node, 'col_offset', 0)) if hasattr(node, 'end_col_offset') else getattr(node, 'col_offset', 0),
                parent_id=parent_id,
                metadata={
                    "ctx": node.ctx.__class__.__name__
                }
            )

            # Store the node
            self.nodes[node_id] = name_node

            # Add to parent's children
            parent_node.children.append(node_id)

        # Add handlers for other node types as needed

        # For nodes we don't explicitly handle, create a generic unknown node
        else:
            node_id = self._create_node_id()

            unknown_node = CodeNode(
                node_id=node_id,
                node_type=CodeNodeType.UNKNOWN,
                language=CodeLanguage.PYTHON,
                name=node.__class__.__name__,
                line_start=getattr(node, 'lineno', 0),
                line_end=getattr(node, 'end_lineno', getattr(node, 'lineno', 0)) if hasattr(node, 'end_lineno') else getattr(node, 'lineno', 0),
                column_start=getattr(node, 'col_offset', 0),
                column_end=getattr(node, 'end_col_offset', getattr(node, 'col_offset', 0)) if hasattr(node, 'end_col_offset') else getattr(node, 'col_offset', 0),
                parent_id=parent_id
            )

            # Store the node
            self.nodes[node_id] = unknown_node

            # Add to parent's children
            parent_node.children.append(node_id)

    def _process_javascript_ast(self, node: Any, parent_id: str, code: str) -> None:
        """
        Process a JavaScript AST node (from esprima) and add it to our representation.

        Args:
            node: Esprima AST node
            parent_id: ID of the parent node
            code: Original source code
        """
        # This would be implemented similarly to _process_python_ast
        # but with esprima's node types
        pass

    def _get_name_from_python_expr(self, expr: ast.AST) -> str:
        """
        Extract a displayable name from a Python expression.

        Args:
            expr: Python AST expression node

        Returns:
            String representation of the expression
        """
        if isinstance(expr, ast.Name):
            return expr.id
        elif isinstance(expr, ast.Attribute):
            return f"{self._get_name_from_python_expr(expr.value)}.{expr.attr}"
        elif isinstance(expr, ast.Call):
            return f"{self._get_name_from_python_expr(expr.func)}(...)"
        elif isinstance(expr, ast.Subscript):
            return f"{self._get_name_from_python_expr(expr.value)}[...]"
        else:
            return expr.__class__.__name__

    def _create_node_id(self) -> str:
        """Generate a unique node ID"""
        return str(uuid.uuid4())

    def _collect_subtree(self, node_id: str, result: Dict[str, CodeNode]) -> None:
        """
        Recursively collect a node and all its descendants.

        Args:
            node_id: The ID of the node to start from
            result: Dictionary to populate with nodes
        """
        if node_id not in self.nodes:
            return

        node = self.nodes[node_id]
        result[node_id] = node

        for child_id in node.children:
            self._collect_subtree(child_id, result)

    def _calculate_metrics(self, code: str, language: CodeLanguage, root_node_id: str) -> CodeMetrics:
        """
        Calculate code metrics for the analyzed code.

        Args:
            code: The original source code
            language: The programming language
            root_node_id: ID of the root node

        Returns:
            CodeMetrics with calculated metrics
        """
        # Initialize metrics
        metrics = CodeMetrics()

        # Calculate basic metrics
        lines = code.splitlines()
        metrics.loc = len(lines)

        # Calculate logical lines of code (non-blank, non-comment)
        if language == CodeLanguage.PYTHON:
            # Simple heuristic for Python
            logical_lines = 0
            comment_lines = 0

            for line in lines:
                stripped = line.strip()
                if not stripped:
                    continue  # Skip blank lines

                if stripped.startswith('#'):
                    comment_lines += 1
                else:
                    logical_lines += 1

            metrics.lloc = logical_lines
            metrics.comment_lines = comment_lines

        # Calculate other metrics based on the AST
        if root_node_id in self.nodes:
            # Count functions and classes
            functions = self.find_nodes({"node_type": CodeNodeType.FUNCTION})
            classes = self.find_nodes({"node_type": CodeNodeType.CLASS})

            metrics.function_count = len(functions)
            metrics.class_count = len(classes)

            # Count dependencies
            imports = self.find_nodes({"node_type": CodeNodeType.IMPORT})
            metrics.dependency_count = len(imports)

            # Calculate cyclomatic complexity
            # Simple approximation: count control flow nodes
            control_flow_nodes = self.find_nodes({"node_type": CodeNodeType.CONTROL_FLOW})
            metrics.cyclomatic_complexity = 1 + len(control_flow_nodes)  # Base 1 + number of branches

            # Calculate function parameter counts
            for func in functions:
                arg_nodes = []
                for child_id in func.children:
                    child = self.nodes.get(child_id)
                    if child and child.node_type == CodeNodeType.ARGUMENT:
                        arg_nodes.append(child)

                func_name = func.name or "anonymous"
                if arg_nodes:
                    # Count direct argument children if available
                    params = 0
                    for arg_node in arg_nodes:
                        if arg_node.name == "arguments":
                            # If it's an arguments container, count its children
                            params += len(arg_node.children)
                        else:
                            # Otherwise count the node itself
                            params += 1

                    metrics.parameter_count[func_name] = params

        # Calculate maintainability index
        # Simplified formula: 171 - 5.2 * ln(HV) - 0.23 * CC - 16.2 * ln(SLOC)
        # where HV is Halstead Volume, CC is cyclomatic complexity, SLOC is source lines of code
        try:
            # Placeholder for Halstead volume (would need proper calculation)
            halstead_volume = metrics.lloc * 2.5  # Very rough approximation
            metrics.halstead_metrics["volume"] = halstead_volume

            metrics.maintainability_index = 171 - 5.2 * math.log(halstead_volume) - 0.23 * metrics.cyclomatic_complexity - 16.2 * math.log(metrics.lloc)
            # Normalize to 0-100 range
            metrics.maintainability_index = max(0, min(100, metrics.maintainability_index))
        except:
            metrics.maintainability_index = 50.0  # Default if calculation fails

        return metrics

    def _run_analysis_plugins(self, language: CodeLanguage, root_node_id: str,
                             options: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Run analysis plugins on the parsed code.

        Args:
            language: The code language
            root_node_id: ID of the root node
            options: Analysis options

        Returns:
            List of warnings/issues found by the plugins
        """
        all_warnings = []

        # Determine which plugins to run
        plugins_to_run = options.get("plugins", list(self.analysis_plugins.keys()))

        for plugin_id in plugins_to_run:
            if plugin_id in self.analysis_plugins:
                try:
                    plugin_func = self.analysis_plugins[plugin_id]
                    warnings = plugin_func(language, root_node_id, options)

                    # Add plugin ID to each warning
                    for warning in warnings:
                        warning["plugin"] = plugin_id

                    all_warnings.extend(warnings)
                except Exception as e:
                    self.logger.error(f"Error running plugin {plugin_id}: {str(e)}")

        return all_warnings

    def _plugin_complexity_checker(self, language: CodeLanguage, root_node_id: str,
                                 options: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Plugin to check code complexity.

        Args:
            language: The code language
            root_node_id: ID of the root node
            options: Analysis options

        Returns:
            List of complexity warnings
        """
        warnings = []

        # Get thresholds from options or use defaults
        max_function_complexity = options.get("max_function_complexity", 10)
        max_function_params = options.get("max_function_params", 5)

        # Find function nodes
        functions = self.find_nodes({"node_type": CodeNodeType.FUNCTION})

        for func in functions:
            # Check function complexity
            complexity = 1  # Base complexity

            # Add complexity for control flow statements
            subtree = self.get_subtree(func.node_id)
            for node in subtree.values():
                if node.node_type == CodeNodeType.CONTROL_FLOW:
                    complexity += 1

            if complexity > max_function_complexity:
                warnings.append({
                    "type": "complexity",
                    "message": f"Function '{func.name}' has high cyclomatic complexity ({complexity})",
                    "node_id": func.node_id,
                    "severity": "warning",
                    "line": func.line_start
                })

            # Check parameter count
            arg_nodes = []
            for child_id in func.children:
                child = self.nodes.get(child_id)
                if child and child.node_type == CodeNodeType.ARGUMENT:
                    arg_nodes.append(child)

            param_count = 0
            for arg_node in arg_nodes:
                if arg_node.name == "arguments":
                    # If it's an arguments container, count its children
                    param_count += len(arg_node.children)
                else:
                    # Otherwise count the node itself
                    param_count += 1

            if param_count > max_function_params:
                warnings.append({
                    "type": "parameters",
                    "message": f"Function '{func.name}' has too many parameters ({param_count})",
                    "node_id": func.node_id,
                    "severity": "warning",
                    "line": func.line_start
                })

        return warnings

    def _plugin_variable_naming(self, language: CodeLanguage, root_node_id: str,
                              options: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Plugin to check variable naming conventions.

        Args:
            language: The code language
            root_node_id: ID of the root node
            options: Analysis options

        Returns:
            List of naming convention warnings
        """
        warnings = []

        if language != CodeLanguage.PYTHON:
            return warnings  # Only implemented for Python for now

        # Get naming patterns from options or use defaults
        naming_patterns = options.get("naming_patterns", {
            "function": r"^[a-z][a-z0-9_]*$",  # snake_case
            "class": r"^[A-Z][a-zA-Z0-9]*$",   # PascalCase
            "variable": r"^[a-z][a-z0-9_]*$",  # snake_case
            "constant": r"^[A-Z][A-Z0-9_]*$"   # UPPER_CASE
        })

        # Check function names
        functions = self.find_nodes({"node_type": CodeNodeType.FUNCTION})
        for func in functions:
            if func.name and not re.match(naming_patterns["function"], func.name):
                warnings.append({
                    "type": "naming",
                    "message": f"Function name '{func.name}' does not follow naming convention",
                    "node_id": func.node_id,
                    "severity": "info",
                    "line": func.line_start
                })

        # Check class names
        classes = self.find_nodes({"node_type": CodeNodeType.CLASS})
        for cls in classes:
            if cls.name and not re.match(naming_patterns["class"], cls.name):
                warnings.append({
                    "type": "naming",
                    "message": f"Class name '{cls.name}' does not follow naming convention",
                    "node_id": cls.node_id,
                    "severity": "info",
                    "line": cls.line_start
                })

        # Check variable names
        variables = self.find_nodes({"node_type": CodeNodeType.VARIABLE})
        for var in variables:
            if var.name:
                # Check if it might be a constant (all caps)
                if re.match(r"^[A-Z][A-Z0-9_]*$", var.name):
                    continue  # Skip constants

                if not re.match(naming_patterns["variable"], var.name):
                    warnings.append({
                        "type": "naming",
                        "message": f"Variable name '{var.name}' does not follow naming convention",
                        "node_id": var.node_id,
                        "severity": "info",
                        "line": var.line_start
                    })

        return warnings

    def _plugin_security_checker(self, language: CodeLanguage, root_node_id: str,
                               options: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Plugin to check for common security issues.

        Args:
            language: The code language
            root_node_id: ID of the root node
            options: Analysis options

        Returns:
            List of security warnings
        """
        warnings = []

        if language == CodeLanguage.PYTHON:
            # Check for potentially unsafe function calls
            unsafe_functions = {
                "eval": "Evaluates arbitrary code which may be unsafe",
                "exec": "Executes arbitrary code which may be unsafe",
                "pickle.loads": "Deserializing untrusted data with pickle can lead to code execution",
                "marshal.loads": "Deserializing untrusted data with marshal can lead to code execution",
                "subprocess.call": "Check for shell=True which may allow command injection",
                "subprocess.Popen": "Check for shell=True which may allow command injection",
                "yaml.load": "Use yaml.safe_load instead for untrusted data",
                "os.system": "May allow command injection if input is not sanitized",
                "__import__": "Dynamic imports can lead to code execution"
            }

            # Find all function calls
            calls = self.find_nodes({"node_type": CodeNodeType.CALL})

            for call in calls:
                func_name = call.name

                # Check for direct matches
                if func_name in unsafe_functions:
                    warnings.append({
                        "type": "security",
                        "message": f"Potential security issue: {func_name} - {unsafe_functions[func_name]}",
                        "node_id": call.node_id,
                        "severity": "warning",
                        "line": call.line_start
                    })

                # Check for module.function matches
                for unsafe_func, description in unsafe_functions.items():
                    if '.' in unsafe_func and func_name.endswith('.' + unsafe_func.split('.')[-1]):
                        # This is a heuristic check - would need deeper analysis for accuracy
                        warnings.append({
                            "type": "security",
                            "message": f"Potential security issue: {func_name} - {description}",
                            "node_id": call.node_id,
                            "severity": "warning",
                            "line": call.line_start
                        })

            # Check for hardcoded secrets or credentials
            literals = self.find_nodes({"node_type": CodeNodeType.LITERAL})

            # Patterns that might indicate secrets
            secret_patterns = [
                (r'password', 'Possible hardcoded password'),
                (r'passwd', 'Possible hardcoded password'),
                (r'secret', 'Possible hardcoded secret'),
                (r'key', 'Possible hardcoded key'),
                (r'token', 'Possible hardcoded token'),
                (r'api.+key', 'Possible hardcoded API key')
            ]

            # Check assignments for potential secrets
            assignments = self.find_nodes({"node_type": CodeNodeType.STATEMENT})
            for assign in assignments:
                if not assign.name:
                    continue

                # Check variable name against secret patterns
                for pattern, message in secret_patterns:
                    if re.search(pattern, assign.name, re.IGNORECASE):
                        warnings.append({
                            "type": "security",
                            "message": f"{message} in variable '{assign.name}'",
                            "node_id": assign.node_id,
                            "severity": "warning",
                            "line": assign.line_start
                        })

            # Check for SQL injection vulnerabilities
            # This is a very simple heuristic and would need deeper analysis for accuracy
            string_format_calls = []
            for call in calls:
                if call.name in ['format', 'f-string', '%']:
                    string_format_calls.append(call)

            # Find strings that look like SQL queries
            sql_patterns = [
                r'SELECT\s+.+\s+FROM',
                r'INSERT\s+INTO',
                r'UPDATE\s+.+\s+SET',
                r'DELETE\s+FROM'
            ]

            for literal in literals:
                if not isinstance(literal.value, str):
                    continue

                for pattern in sql_patterns:
                    if re.search(pattern, literal.value, re.IGNORECASE):
                        # Check if this SQL string is used with string formatting
                        # This is a simplified check and not very accurate
                        warnings.append({
                            "type": "security",
                            "message": "Possible SQL query string - check for proper parameterization to avoid SQL injection",
                            "node_id": literal.node_id,
                            "severity": "info",
                            "line": literal.line_start
                        })
                        break

        elif language == CodeLanguage.JAVASCRIPT:
            # Similar checks for JavaScript would go here
            pass

        return warnings

ModuleNotFoundError: No module named 'grandnexus'

# Pattern Recognizer

In [ ]:
"""
pattern_recognizer.py - Pattern Recognition Module for GrandNexus

This module provides capabilities for detecting and extracting patterns from various
data sources including sequences, text, symbolic structures, and graph data. It serves
as a key perceptual component that helps the cognitive system identify regularities
and recurring motifs in observed information.

Key features:
- Sequential pattern detection (temporal patterns, repeating sequences)
- Structural pattern recognition (recurring structures in data)
- Approximate pattern matching with flexible similarity metrics
- Integration with semantic memory for contextual pattern understanding
- Emergent pattern discovery through unsupervised analysis
- Adaptive pattern prioritization based on relevance and attention
"""

import logging
import time
import threading
import uuid
import re
import math
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple, Union, Callable
from enum import Enum, auto
from collections import defaultdict, deque, Counter
import random

# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.graph.graph_engine import GraphEngine
from grandnexus.graph.graph_engine import create_graph_engine, AbstractGraphEngine

# Try importing optional dependencies
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    from sklearn.cluster import DBSCAN
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

# Pattern types
class PatternType(Enum):
    """Types of patterns the recognizer can detect"""
    SEQUENTIAL = auto()  # Ordered sequences of items
    TEMPORAL = auto()    # Time-dependent patterns
    STRUCTURAL = auto()  # Structure-based patterns (trees, graphs)
    LINGUISTIC = auto()  # Language patterns
    SYMBOLIC = auto()    # Symbolic or mathematical patterns
    BEHAVIORAL = auto()  # Patterns in behavior or action sequences
    SPATIAL = auto()     # Spatial arrangements
    EMERGENT = auto()    # Emergent patterns not fitting other categories

@dataclass
class Pattern:
    """Represents a detected pattern"""
    pattern_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    pattern_type: PatternType = PatternType.SEQUENTIAL
    pattern_data: Any = None  # The actual pattern content
    confidence: float = 0.5
    frequency: int = 1
    first_seen: float = field(default_factory=time.time)
    last_seen: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)

    # For structural patterns
    structure: Optional[Dict[str, Any]] = None

    # For pattern variations
    variations: List[Dict[str, Any]] = field(default_factory=list)

    # For extended context
    context_before: Optional[Any] = None
    context_after: Optional[Any] = None

    def update_frequency(self) -> None:
        """Update the frequency and last_seen timestamp"""
        self.frequency += 1
        self.last_seen = time.time()

    def add_variation(self, variation_data: Any, similarity: float) -> None:
        """Add a variation of this pattern"""
        self.variations.append({
            "data": variation_data,
            "similarity": similarity,
            "timestamp": time.time()
        })

    def to_dict(self) -> Dict[str, Any]:
        """Convert pattern to a dictionary for serialization"""
        result = {
            "pattern_id": self.pattern_id,
            "pattern_type": self.pattern_type.name,
            "confidence": self.confidence,
            "frequency": self.frequency,
            "first_seen": self.first_seen,
            "last_seen": self.last_seen,
            "metadata": self.metadata,
        }

        # Handle pattern_data based on type for serialization
        if isinstance(self.pattern_data, (str, int, float, bool, type(None))):
            result["pattern_data"] = self.pattern_data
        else:
            # For complex structures, serialize as string or use a specific serializer
            try:
                result["pattern_data"] = str(self.pattern_data)
            except:
                result["pattern_data"] = f"<{type(self.pattern_data).__name__}>"

        if self.structure:
            result["structure"] = self.structure

        if self.variations:
            result["variation_count"] = len(self.variations)

        return result

@dataclass
class PatternMatch:
    """Result of a pattern matching operation"""
    matched: bool
    pattern: Optional[Pattern] = None
    confidence: float = 0.0
    match_start: Optional[int] = None  # For sequential data
    match_end: Optional[int] = None    # For sequential data
    match_elements: List[Any] = field(default_factory=list)
    similarity: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class PatternSearchConfig:
    """Configuration for pattern search operations"""
    pattern_types: List[PatternType] = field(default_factory=lambda: list(PatternType))
    min_pattern_length: int = 2
    max_pattern_length: int = 10
    min_frequency: int = 2
    min_confidence: float = 0.5
    max_patterns: int = 20
    similarity_threshold: float = 0.7
    consider_variations: bool = True
    use_semantic_context: bool = False
    detect_hierarchical: bool = False
    max_search_time: float = 5.0  # seconds

class PatternRecognizer:
    """
    Core pattern recognition engine for GrandNexus, capable of identifying
    various pattern types across different data sources and modalities.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 graph_engine: Optional[GraphEngine] = None):
        """
        Initialize the pattern recognizer.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            semantic_memory: Reference to semantic memory for context
            working_memory: Reference to working memory for active patterns
            episodic_memory: Reference to episodic memory for historical patterns
            graph_engine: Reference to graph engine for structural patterns
        """
        self.logger = logging.getLogger("GrandNexus.Perception.PatternRecognizer")
        self.nexus_core = nexus_core
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory
        self.episodic_memory = episodic_memory
        self.graph_engine = graph_engine

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Pattern storage
        self.patterns = {}  # pattern_id -> Pattern
        self.pattern_index = defaultdict(list)  # Various indices for faster lookup

        # Pattern detectors for different types
        self.detectors = {}

        # History of recently seen data
        self.recent_sequences = deque(maxlen=100)
        self.recent_items = deque(maxlen=1000)

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"PatternRecognizer initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the pattern recognizer and register pattern detectors"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register built-in pattern detectors
                self._register_default_detectors()

                # Load any existing patterns from memory
                self._load_patterns_from_memory()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("PatternRecognizer initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"PatternRecognizer initialization failed: {str(e)}")
                return False

    def process(self, data: Any, data_type: str = "sequential",
               context: Optional[Dict[str, Any]] = None,
               detect_new_patterns: bool = True) -> List[PatternMatch]:
        """
        Process new data, matching against known patterns and optionally
        detecting new patterns.

        Args:
            data: The data to process (sequence, text, structure, etc.)
            data_type: Type of data (sequential, text, structural, etc.)
            context: Additional contextual information
            detect_new_patterns: Whether to look for new patterns

        Returns:
            List of pattern matches found in the data
        """
        if not self.initialized:
            self.initialize()

        # Prepare context if not provided
        if context is None:
            context = {}

        # Track the data for future pattern detection
        self._track_data(data, data_type, context)

        # Prepare result list
        matches = []

        # Match against known patterns
        pattern_matches = self._match_known_patterns(data, data_type, context)
        matches.extend(pattern_matches)

        # Detect new patterns if requested
        if detect_new_patterns:
            start_time = time.time()
            new_patterns = self._detect_new_patterns(data, data_type, context)
            detection_time = time.time() - start_time

            self.metrics["pattern_detection_time"].append(detection_time)

            # Log new pattern detection
            if new_patterns:
                self.logger.info(f"Detected {len(new_patterns)} new patterns in {data_type} data")

                # Register the new patterns
                for pattern in new_patterns:
                    self.register_pattern(pattern)

                # Match the new patterns against the current data
                new_matches = self._match_patterns(new_patterns, data, data_type, context)
                matches.extend(new_matches)

        # Update metrics
        self.metrics["matches_per_process"].append(len(matches))

        return matches

    def register_pattern(self, pattern: Pattern) -> bool:
        """
        Register a new pattern or update an existing similar pattern.

        Args:
            pattern: The pattern to register

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            try:
                # Check if this pattern is similar to an existing one
                similar_pattern_id = self._find_similar_pattern(pattern)

                if similar_pattern_id:
                    # Update the existing pattern
                    existing_pattern = self.patterns[similar_pattern_id]
                    existing_pattern.update_frequency()
                    existing_pattern.last_seen = time.time()

                    # Add this as a variation if it's not identical
                    if existing_pattern.pattern_data != pattern.pattern_data:
                        similarity = self._calculate_pattern_similarity(
                            existing_pattern.pattern_data,
                            pattern.pattern_data,
                            pattern.pattern_type
                        )
                        existing_pattern.add_variation(pattern.pattern_data, similarity)

                    # Update confidence if needed
                    if pattern.confidence > existing_pattern.confidence:
                        existing_pattern.confidence = pattern.confidence

                    # Update metadata with any new information
                    for key, value in pattern.metadata.items():
                        if key not in existing_pattern.metadata:
                            existing_pattern.metadata[key] = value

                    return True
                else:
                    # Register as a new pattern
                    self.patterns[pattern.pattern_id] = pattern

                    # Update indices for faster lookup
                    self._update_pattern_indices(pattern)

                    # Add to working memory if available
                    if self.working_memory:
                        pattern_dict = pattern.to_dict()
                        self.working_memory.store_item(
                            content=pattern_dict,
                            source="pattern_recognizer",
                            metadata={
                                "pattern_type": pattern.pattern_type.name,
                                "confidence": pattern.confidence
                            }
                        )

                    self.logger.info(f"Registered new pattern: {pattern.pattern_id} ({pattern.pattern_type.name})")
                    return True
            except Exception as e:
                self.logger.error(f"Error registering pattern: {str(e)}")
                return False

    def find_patterns(self, config: PatternSearchConfig,
                     data: Optional[Any] = None) -> List[Pattern]:
        """
        Find patterns matching the given configuration.

        Args:
            config: Configuration for pattern search
            data: Optional specific data to search in instead of history

        Returns:
            List of patterns matching the criteria
        """
        if not self.initialized:
            self.initialize()

        matches = []

        with self.lock:
            # Filter patterns by type
            candidates = []
            for pattern_id, pattern in self.patterns.items():
                if pattern.pattern_type in config.pattern_types and \
                   pattern.confidence >= config.min_confidence and \
                   pattern.frequency >= config.min_frequency:
                    candidates.append(pattern)

            # Sort by relevance (combination of confidence and frequency)
            candidates.sort(key=lambda p: p.confidence * p.frequency, reverse=True)

            # Limit to max_patterns
            candidates = candidates[:config.max_patterns]

            if data is not None:
                # If specific data is provided, match patterns against it
                for pattern in candidates:
                    detector = self._get_detector_for_type(pattern.pattern_type)
                    if detector:
                        matches = detector.match(pattern, data, {})
                        if matches:
                            matches.append(pattern)
            else:
                # Otherwise return the filtered patterns
                matches = candidates

        return matches

    def get_pattern(self, pattern_id: str) -> Optional[Pattern]:
        """Get a specific pattern by ID"""
        with self.lock:
            return self.patterns.get(pattern_id)

    def remove_pattern(self, pattern_id: str) -> bool:
        """Remove a pattern from the system"""
        with self.lock:
            if pattern_id in self.patterns:
                pattern = self.patterns[pattern_id]

                # Remove from indices
                self._remove_from_indices(pattern)

                # Remove from patterns dictionary
                del self.patterns[pattern_id]

                self.logger.info(f"Removed pattern: {pattern_id}")
                return True
            return False

    def get_pattern_metrics(self) -> Dict[str, Any]:
        """Get metrics about pattern recognition performance"""
        with self.lock:
            metrics_snapshot = {}

            # Overall pattern metrics
            metrics_snapshot["total_patterns"] = len(self.patterns)

            # Pattern types distribution
            type_counts = defaultdict(int)
            for pattern in self.patterns.values():
                type_counts[pattern.pattern_type.name] += 1
            metrics_snapshot["pattern_type_distribution"] = dict(type_counts)

            # Performance metrics
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }

            # Length metrics
            if self.patterns:
                pattern_lengths = []
                for pattern in self.patterns.values():
                    # Estimate length based on pattern_type
                    if pattern.pattern_type == PatternType.SEQUENTIAL and hasattr(pattern.pattern_data, "__len__"):
                        pattern_lengths.append(len(pattern.pattern_data))

                if pattern_lengths:
                    metrics_snapshot["avg_pattern_length"] = sum(pattern_lengths) / len(pattern_lengths)
                    metrics_snapshot["min_pattern_length"] = min(pattern_lengths)
                    metrics_snapshot["max_pattern_length"] = max(pattern_lengths)

            return metrics_snapshot

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_default_detectors(self) -> None:
        """Register built-in pattern detectors for different pattern types"""
        # Sequential pattern detector
        self.detectors[PatternType.SEQUENTIAL] = SequentialPatternDetector()

        # Temporal pattern detector
        self.detectors[PatternType.TEMPORAL] = TemporalPatternDetector()

        # Structural pattern detector (if graph_engine is available)
        if self.graph_engine:
            self.detectors[PatternType.STRUCTURAL] = StructuralPatternDetector()

        # Linguistic pattern detector
        self.detectors[PatternType.LINGUISTIC] = LinguisticPatternDetector()

        # Symbolic pattern detector
        self.detectors[PatternType.SYMBOLIC] = SymbolicPatternDetector()

    def _load_patterns_from_memory(self) -> None:
        """Load existing patterns from memory if available"""
        # This is a placeholder - in a real implementation,
        # patterns would be loaded from semantic_memory or episodic_memory
        pass

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.graph_engine:
                dependencies.append("graph_engine")

            self.nexus_core.register_module(
                name="pattern_recognizer",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _track_data(self, data: Any, data_type: str, context: Dict[str, Any]) -> None:
        """
        Track the data for future pattern detection by adding to history.

        Args:
            data: The data to track
            data_type: Type of data (sequential, text, structural, etc.)
            context: Additional contextual information
        """
        # Add to recent items
        self.recent_items.append((data, data_type, time.time(), context))

        # For sequential data, add to sequences
        if data_type == "sequential" or data_type == "temporal":
            self.recent_sequences.append((data, data_type, time.time(), context))

    def _match_known_patterns(self, data: Any, data_type: str,
                             context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match the given data against known patterns.

        Args:
            data: The data to match against patterns
            data_type: Type of data
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        matches = []

        # Get relevant patterns based on data_type
        relevant_patterns = []

        # Map data_type to PatternType
        pattern_type_map = {
            "sequential": PatternType.SEQUENTIAL,
            "temporal": PatternType.TEMPORAL,
            "structural": PatternType.STRUCTURAL,
            "text": PatternType.LINGUISTIC,
            "symbolic": PatternType.SYMBOLIC,
            "behavior": PatternType.BEHAVIORAL,
            "spatial": PatternType.SPATIAL
        }

        # Get pattern type from map, default to SEQUENTIAL
        pattern_type = pattern_type_map.get(data_type, PatternType.SEQUENTIAL)

        # Get patterns of matching type
        with self.lock:
            for pattern_id, pattern in self.patterns.items():
                if pattern.pattern_type == pattern_type:
                    relevant_patterns.append(pattern)

        # Match patterns using appropriate detector
        detector = self._get_detector_for_type(pattern_type)
        if detector:
            for pattern in relevant_patterns:
                pattern_matches = detector.match(pattern, data, context)
                matches.extend(pattern_matches)

        return matches

    def _detect_new_patterns(self, data: Any, data_type: str,
                            context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect new patterns in the data and history.

        Args:
            data: The current data
            data_type: Type of data
            context: Additional contextual information

        Returns:
            List of newly detected patterns
        """
        new_patterns = []

        # Map data_type to PatternType
        pattern_type_map = {
            "sequential": PatternType.SEQUENTIAL,
            "temporal": PatternType.TEMPORAL,
            "structural": PatternType.STRUCTURAL,
            "text": PatternType.LINGUISTIC,
            "symbolic": PatternType.SYMBOLIC,
            "behavior": PatternType.BEHAVIORAL,
            "spatial": PatternType.SPATIAL
        }

        # Get pattern type from map, default to SEQUENTIAL
        pattern_type = pattern_type_map.get(data_type, PatternType.SEQUENTIAL)

        # Use appropriate detector
        detector = self._get_detector_for_type(pattern_type)
        if detector:
            history = self._get_relevant_history(data_type)
            detected_patterns = detector.detect(data, history, context)
            new_patterns.extend(detected_patterns)

        return new_patterns

    def _match_patterns(self, patterns: List[Pattern], data: Any,
                       data_type: str, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match specific patterns against the given data.

        Args:
            patterns: List of patterns to match
            data: The data to match against
            data_type: Type of data
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        matches = []

        # Map data_type to PatternType
        pattern_type_map = {
            "sequential": PatternType.SEQUENTIAL,
            "temporal": PatternType.TEMPORAL,
            "structural": PatternType.STRUCTURAL,
            "text": PatternType.LINGUISTIC,
            "symbolic": PatternType.SYMBOLIC,
            "behavior": PatternType.BEHAVIORAL,
            "spatial": PatternType.SPATIAL
        }

        # Get pattern type from map, default to SEQUENTIAL
        pattern_type = pattern_type_map.get(data_type, PatternType.SEQUENTIAL)

        # Match patterns using appropriate detector
        for pattern in patterns:
            detector = self._get_detector_for_type(pattern.pattern_type)
            if detector:
                pattern_matches = detector.match(pattern, data, context)
                matches.extend(pattern_matches)

        return matches

    def _get_detector_for_type(self, pattern_type: PatternType) -> Optional[Any]:
        """
        Get the appropriate detector for a pattern type.

        Args:
            pattern_type: The type of pattern

        Returns:
            Pattern detector instance or None if not available
        """
        return self.detectors.get(pattern_type)

    def _get_relevant_history(self, data_type: str) -> List[Tuple[Any, str, float, Dict[str, Any]]]:
        """
        Get relevant historical data for pattern detection.

        Args:
            data_type: Type of data

        Returns:
            List of (data, data_type, timestamp, context) tuples
        """
        if data_type in ["sequential", "temporal"]:
            return list(self.recent_sequences)
        else:
            # Filter by data_type
            return [(d, dt, ts, ctx) for d, dt, ts, ctx in self.recent_items if dt == data_type]

    def _find_similar_pattern(self, pattern: Pattern) -> Optional[str]:
        """
        Find an existing pattern similar to the given one.

        Args:
            pattern: The pattern to find a similar one for

        Returns:
            ID of similar pattern or None if not found
        """
        # First check for exact matches
        for pattern_id, existing in self.patterns.items():
            if existing.pattern_type == pattern.pattern_type and existing.pattern_data == pattern.pattern_data:
                return pattern_id

        # Then check for similar patterns
        for pattern_id, existing in self.patterns.items():
            if existing.pattern_type == pattern.pattern_type:
                similarity = self._calculate_pattern_similarity(
                    existing.pattern_data,
                    pattern.pattern_data,
                    pattern.pattern_type
                )

                # Consider similar if above threshold
                if similarity >= 0.8:  # High threshold for similarity
                    return pattern_id

        return None

    def _calculate_pattern_similarity(self, pattern1: Any, pattern2: Any,
                                    pattern_type: PatternType) -> float:
        """
        Calculate similarity between two patterns.

        Args:
            pattern1: First pattern data
            pattern2: Second pattern data
            pattern_type: Type of patterns

        Returns:
            Similarity score between 0.0 and 1.0
        """
        if pattern1 == pattern2:
            return 1.0

        # Handle different pattern types
        if pattern_type == PatternType.SEQUENTIAL:
            return self._sequence_similarity(pattern1, pattern2)
        elif pattern_type == PatternType.LINGUISTIC:
            return self._text_similarity(pattern1, pattern2)
        elif pattern_type == PatternType.STRUCTURAL:
            return self._structural_similarity(pattern1, pattern2)
        elif pattern_type == PatternType.SYMBOLIC:
            return self._symbolic_similarity(pattern1, pattern2)
        else:
            # Default to basic comparison
            try:
                # For basic types that support equality
                return 1.0 if pattern1 == pattern2 else 0.0
            except:
                # For complex types, use string representation
                return self._text_similarity(str(pattern1), str(pattern2))

    def _sequence_similarity(self, seq1: Any, seq2: Any) -> float:
        """
        Calculate similarity between two sequences.

        Args:
            seq1: First sequence
            seq2: Second sequence

        Returns:
            Similarity score between 0.0 and 1.0
        """
        # Handle non-sequence types
        if not hasattr(seq1, "__len__") or not hasattr(seq2, "__len__"):
            return 0.0

        # Empty sequences
        if len(seq1) == 0 and len(seq2) == 0:
            return 1.0
        elif len(seq1) == 0 or len(seq2) == 0:
            return 0.0

        # Simple longest common subsequence approach
        # Real implementation would use a more sophisticated algorithm
        lcs_length = self._longest_common_subsequence_length(seq1, seq2)

        return lcs_length / max(len(seq1), len(seq2))

    def _text_similarity(self, text1: str, text2: str) -> float:
        """
        Calculate similarity between two text strings.

        Args:
            text1: First text
            text2: Second text

        Returns:
            Similarity score between 0.0 and 1.0
        """
        # Check if both are strings
        if not isinstance(text1, str) or not isinstance(text2, str):
            return 0.0

        # Simple Jaccard similarity of words
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())

        if not words1 and not words2:
            return 1.0
        elif not words1 or not words2:
            return 0.0

        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))

        return intersection / union

    def _structural_similarity(self, struct1: Any, struct2: Any) -> float:
        """
        Calculate similarity between two structures.

        Args:
            struct1: First structure
            struct2: Second structure

        Returns:
            Similarity score between 0.0 and 1.0
        """
        # This is a placeholder - a real implementation would depend
        # on the specific structure types
        return 0.5

    def _symbolic_similarity(self, symbol1: Any, symbol2: Any) -> float:
        """
        Calculate similarity between two symbolic expressions.

        Args:
            symbol1: First symbolic expression
            symbol2: Second symbolic expression

        Returns:
            Similarity score between 0.0 and 1.0
        """
        # This is a placeholder - a real implementation would use
        # tree edit distance, canonical forms, etc.
        return 0.5

    def _longest_common_subsequence_length(self, seq1: Any, seq2: Any) -> int:
        """
        Calculate the length of the longest common subsequence.

        Args:
            seq1: First sequence
            seq2: Second sequence

        Returns:
            Length of longest common subsequence
        """
        # Simple dynamic programming implementation
        m, n = len(seq1), len(seq2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]

        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if seq1[i-1] == seq2[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])

        return dp[m][n]

    def _update_pattern_indices(self, pattern: Pattern) -> None:
        """
        Update indices for efficient pattern lookup.

        Args:
            pattern: The pattern to index
        """
        # Index by pattern type
        self.pattern_index[f"type:{pattern.pattern_type.name}"].append(pattern.pattern_id)

        # Index by confidence range
        confidence_range = int(pattern.confidence * 10) / 10
        self.pattern_index[f"confidence:{confidence_range}"].append(pattern.pattern_id)

        # Index by frequency bucket (logarithmic scale)
        if pattern.frequency <= 1:
            freq_bucket = "1"
        elif pattern.frequency <= 5:
            freq_bucket = "2-5"
        elif pattern.frequency <= 10:
            freq_bucket = "6-10"
        elif pattern.frequency <= 50:
            freq_bucket = "11-50"
        else:
            freq_bucket = "50+"

        self.pattern_index[f"frequency:{freq_bucket}"].append(pattern.pattern_id)

        # Add metadata indices if available
        if pattern.metadata:
            for key, value in pattern.metadata.items():
                if isinstance(value, (str, int, float, bool)):
                    self.pattern_index[f"metadata:{key}:{value}"].append(pattern.pattern_id)

    def _remove_from_indices(self, pattern: Pattern) -> None:
        """
        Remove a pattern from all indices.

        Args:
            pattern: The pattern to remove from indices
        """
        # Remove from type index
        type_key = f"type:{pattern.pattern_type.name}"
        if type_key in self.pattern_index and pattern.pattern_id in self.pattern_index[type_key]:
            self.pattern_index[type_key].remove(pattern.pattern_id)

        # Remove from confidence index
        confidence_range = int(pattern.confidence * 10) / 10
        conf_key = f"confidence:{confidence_range}"
        if conf_key in self.pattern_index and pattern.pattern_id in self.pattern_index[conf_key]:
            self.pattern_index[conf_key].remove(pattern.pattern_id)

        # Remove from frequency index
        # Determine frequency bucket as in _update_pattern_indices
        if pattern.frequency <= 1:
            freq_bucket = "1"
        elif pattern.frequency <= 5:
            freq_bucket = "2-5"
        elif pattern.frequency <= 10:
            freq_bucket = "6-10"
        elif pattern.frequency <= 50:
            freq_bucket = "11-50"
        else:
            freq_bucket = "50+"

        freq_key = f"frequency:{freq_bucket}"
        if freq_key in self.pattern_index and pattern.pattern_id in self.pattern_index[freq_key]:
            self.pattern_index[freq_key].remove(pattern.pattern_id)

        # Remove from metadata indices
        if pattern.metadata:
            for key, value in pattern.metadata.items():
                if isinstance(value, (str, int, float, bool)):
                    meta_key = f"metadata:{key}:{value}"
                    if meta_key in self.pattern_index and pattern.pattern_id in self.pattern_index[meta_key]:
                        self.pattern_index[meta_key].remove(pattern.pattern_id)


# -----------------------------------------------------------------------------
# Pattern Detector implementations
# -----------------------------------------------------------------------------

class PatternDetectorBase:
    """Base class for pattern detectors"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect patterns in data and history.

        Args:
            data: Current data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected patterns
        """
        raise NotImplementedError("Subclasses must implement detect method")

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a pattern against data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        raise NotImplementedError("Subclasses must implement match method")


class SequentialPatternDetector(PatternDetectorBase):
    """Detector for sequential patterns"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect sequential patterns in data and history.

        Args:
            data: Current sequential data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected sequential patterns
        """
        patterns = []

        # Collect all sequential data from history
        all_sequences = []
        for hist_data, data_type, timestamp, hist_context in history:
            if data_type == "sequential":
                all_sequences.append(hist_data)

        # Add current data
        all_sequences.append(data)

        # Only perform detection if we have enough sequences
        if len(all_sequences) < 2:
            return patterns

        # Simple frequent subsequence detection
        # This is a placeholder for a more sophisticated algorithm
        # (e.g., PrefixSpan, SPADE, etc.)

        # First, find common subsequences across sequences
        min_length = context.get("min_pattern_length", 2)
        max_length = context.get("max_pattern_length", 10)

        # Extract potential subsequences from the current data
        if not hasattr(data, "__len__") or len(data) < min_length:
            return patterns

        # Extract all subsequences of length between min_length and max_length
        subsequences = []
        for length in range(min_length, min(max_length + 1, len(data) + 1)):
            for i in range(len(data) - length + 1):
                subsequence = data[i:i+length]
                subsequences.append(subsequence)

        # Count occurrences of each subsequence in all data
        subsequence_counts = Counter()
        for seq in all_sequences:
            if not hasattr(seq, "__len__"):
                continue

            for subseq in subsequences:
                if self._contains_subsequence(seq, subseq):
                    subsequence_counts[str(subseq)] += 1

        # Create patterns for frequent subsequences
        min_frequency = context.get("min_frequency", 2)
        for subseq_str, count in subsequence_counts.items():
            if count >= min_frequency:
                # Try to parse the string representation back to the original type
                # This is a simplification - real implementation would need to handle various types
                subseq = self._parse_sequence(subseq_str)

                # Create a new pattern
                pattern = Pattern(
                    pattern_type=PatternType.SEQUENTIAL,
                    pattern_data=subseq,
                    confidence=min(0.5 + 0.1 * count, 0.95),  # Higher confidence with more occurrences
                    frequency=count,
                    metadata={"source": "sequential_detector"}
                )

                patterns.append(pattern)

        return patterns

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a sequential pattern against data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        matches = []

        if not hasattr(data, "__len__") or not hasattr(pattern.pattern_data, "__len__"):
            return matches

        # Find all occurrences of the pattern in the data
        pattern_data = pattern.pattern_data
        pattern_len = len(pattern_data)

        if pattern_len > len(data):
            return matches

        # Exact matching
        for i in range(len(data) - pattern_len + 1):
            if data[i:i+pattern_len] == pattern_data:
                match = PatternMatch(
                    matched=True,
                    pattern=pattern,
                    confidence=pattern.confidence,
                    match_start=i,
                    match_end=i + pattern_len,
                    match_elements=data[i:i+pattern_len],
                    similarity=1.0
                )
                matches.append(match)

        # If no exact matches and we allow approximate matching
        if not matches and context.get("allow_approximate", True):
            # Approximate matching with similarity threshold
            similarity_threshold = context.get("similarity_threshold", 0.7)

            for i in range(len(data) - pattern_len + 1):
                # Calculate similarity between pattern and current window
                current_window = data[i:i+pattern_len]

                # Convert sequences to strings for comparison if needed
                if isinstance(pattern_data, str) and not isinstance(current_window, str):
                    current_window_str = str(current_window)
                    similarity = self._calculate_similarity(pattern_data, current_window_str)
                elif not isinstance(pattern_data, str) and isinstance(current_window, str):
                    pattern_data_str = str(pattern_data)
                    similarity = self._calculate_similarity(pattern_data_str, current_window)
                else:
                    similarity = self._calculate_similarity(pattern_data, current_window)

                if similarity >= similarity_threshold:
                    match = PatternMatch(
                        matched=True,
                        pattern=pattern,
                        confidence=pattern.confidence * similarity,
                        match_start=i,
                        match_end=i + pattern_len,
                        match_elements=current_window,
                        similarity=similarity
                    )
                    matches.append(match)

        return matches

    def _contains_subsequence(self, sequence: Any, subsequence: Any) -> bool:
        """
        Check if a sequence contains a subsequence.

        Args:
            sequence: The sequence to check in
            subsequence: The subsequence to look for

        Returns:
            True if subsequence is in sequence, False otherwise
        """
        if not hasattr(sequence, "__len__") or not hasattr(subsequence, "__len__"):
            return False

        if len(subsequence) > len(sequence):
            return False

        for i in range(len(sequence) - len(subsequence) + 1):
            if sequence[i:i+len(subsequence)] == subsequence:
                return True

        return False

    def _parse_sequence(self, sequence_str: str) -> Any:
        """
        Parse a string representation back to a sequence.

        Args:
            sequence_str: String representation of sequence

        Returns:
            The parsed sequence
        """
        # This is a simplified parsing logic
        # Real implementation would need to handle various sequence types
        try:
            # Try to eval if it looks like a list/tuple
            if (sequence_str.startswith('[') and sequence_str.endswith(']')) or \
               (sequence_str.startswith('(') and sequence_str.endswith(')')):
                return eval(sequence_str)
            else:
                # Return as string otherwise
                return sequence_str
        except:
            # If parsing fails, return the string as is
            return sequence_str

    def _calculate_similarity(self, seq1: Any, seq2: Any) -> float:
        """
        Calculate similarity between two sequences.

        Args:
            seq1: First sequence
            seq2: Second sequence

        Returns:
            Similarity score between 0.0 and 1.0
        """
        # String-based comparison for heterogeneous types
        if isinstance(seq1, str) and isinstance(seq2, str):
            # For strings, use Levenshtein distance or similar
            # This is a simplified version using Jaccard similarity
            words1 = set(seq1.split())
            words2 = set(seq2.split())

            if not words1 and not words2:
                return 1.0

            intersection = len(words1.intersection(words2))
            union = len(words1.union(words2))

            return intersection / union

        # Sequence-based comparison for homogeneous types
        if hasattr(seq1, "__len__") and hasattr(seq2, "__len__"):
            # If lengths are too different, limit similarity
            len_ratio = min(len(seq1), len(seq2)) / max(len(seq1), len(seq2)) if max(len(seq1), len(seq2)) > 0 else 1.0

            # Count matching elements
            matches = 0
            for i in range(min(len(seq1), len(seq2))):
                if seq1[i] == seq2[i]:
                    matches += 1

            element_similarity = matches / min(len(seq1), len(seq2)) if min(len(seq1), len(seq2)) > 0 else 0.0

            # Combine length ratio and element similarity
            return (len_ratio * 0.4 + element_similarity * 0.6)

        # Fallback for incomparable types
        return 0.0


class TemporalPatternDetector(PatternDetectorBase):
    """Detector for time-dependent patterns"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect temporal patterns in data and history.

        Args:
            data: Current data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected temporal patterns
        """
        patterns = []

        # Need enough history for temporal patterns
        if len(history) < 3:
            return patterns

        # Sort by timestamp
        sorted_history = sorted(history, key=lambda x: x[2])

        # Extract temporal sequence of values
        values = [hist_data for hist_data, _, _, _ in sorted_history]
        timestamps = [ts for _, _, ts, _ in sorted_history]

        # Add current data and timestamp
        current_time = time.time()
        values.append(data)
        timestamps.append(current_time)

        # Check for periodic patterns
        periodic_patterns = self._detect_periodicity(values, timestamps, context)
        patterns.extend(periodic_patterns)

        # Check for trend patterns
        trend_patterns = self._detect_trends(values, timestamps, context)
        patterns.extend(trend_patterns)

        return patterns

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a temporal pattern against data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        matches = []

        # Temporal patterns often require historical context to match
        # This is a simplified implementation

        if "temporal_type" not in pattern.metadata:
            return matches

        temporal_type = pattern.metadata["temporal_type"]

        if temporal_type == "periodic":
            # For periodic patterns, we check if this value fits the pattern
            expected_value = pattern.metadata.get("expected_value")
            if expected_value is not None:
                # Calculate similarity between expected and actual
                if isinstance(expected_value, (int, float)) and isinstance(data, (int, float)):
                    similarity = 1.0 - min(abs(expected_value - data) / max(abs(expected_value), 1.0), 1.0)
                else:
                    # For non-numeric types, use string comparison
                    similarity = 1.0 if str(expected_value) == str(data) else 0.0

                # If similar enough, we have a match
                similarity_threshold = context.get("similarity_threshold", 0.7)
                if similarity >= similarity_threshold:
                    match = PatternMatch(
                        matched=True,
                        pattern=pattern,
                        confidence=pattern.confidence * similarity,
                        similarity=similarity,
                        metadata={"expected_value": expected_value}
                    )
                    matches.append(match)

        elif temporal_type == "trend":
            # For trend patterns, we check if this value continues the trend
            trend_direction = pattern.metadata.get("trend_direction")
            last_value = pattern.metadata.get("last_value")

            if trend_direction is not None and last_value is not None:
                # Convert to numeric if needed
                if not isinstance(data, (int, float)):
                    try:
                        numeric_data = float(data)
                    except (ValueError, TypeError):
                        return matches
                else:
                    numeric_data = data

                if not isinstance(last_value, (int, float)):
                    try:
                        last_value = float(last_value)
                    except (ValueError, TypeError):
                        return matches

                # Check if value continues the trend
                continues_trend = False
                if trend_direction == "increasing" and numeric_data > last_value:
                    continues_trend = True
                elif trend_direction == "decreasing" and numeric_data < last_value:
                    continues_trend = True
                elif trend_direction == "stable" and abs(numeric_data - last_value) / max(abs(last_value), 1.0) < 0.1:
                    continues_trend = True

                if continues_trend:
                    match = PatternMatch(
                        matched=True,
                        pattern=pattern,
                        confidence=pattern.confidence,
                        similarity=1.0,
                        metadata={
                            "trend_direction": trend_direction,
                            "last_value": last_value,
                            "new_value": numeric_data
                        }
                    )
                    matches.append(match)

        return matches

    def _detect_periodicity(self, values: List[Any], timestamps: List[float],
                           context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect periodic patterns in temporal data.

        Args:
            values: Sequence of values
            timestamps: Corresponding timestamps
            context: Additional contextual information

        Returns:
            List of periodic patterns
        """
        patterns = []

        # Need at least 3 points for periodicity
        if len(values) < 3:
            return patterns

        # This is a placeholder - a real implementation would use
        # Fourier analysis, autocorrelation, or other techniques

        # For demonstration, we'll do a simple check for alternating values
        # if the last 4+ values alternate between two values
        if len(values) >= 4:
            recent_values = values[-4:]

            # Check if odd and even indices have the same values
            if all(recent_values[i] == recent_values[0] for i in range(0, len(recent_values), 2)) and \
               all(recent_values[i] == recent_values[1] for i in range(1, len(recent_values), 2)) and \
               recent_values[0] != recent_values[1]:

                # We have an alternating pattern
                pattern = Pattern(
                    pattern_type=PatternType.TEMPORAL,
                    pattern_data=[recent_values[0], recent_values[1]],
                    confidence=0.7,
                    frequency=2,  # Number of complete cycles
                    metadata={
                        "temporal_type": "periodic",
                        "period": 2,
                        "expected_value": recent_values[0] if len(recent_values) % 2 == 0 else recent_values[1]
                    }
                )

                patterns.append(pattern)

        return patterns

    def _detect_trends(self, values: List[Any], timestamps: List[float],
                      context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect trends in temporal data.

        Args:
            values: Sequence of values
            timestamps: Corresponding timestamps
            context: Additional contextual information

        Returns:
            List of trend patterns
        """
        patterns = []

        # Need at least 3 points for a trend
        if len(values) < 3:
            return patterns

        # Convert to numeric if possible
        numeric_values = []
        for val in values:
            if isinstance(val, (int, float)):
                numeric_values.append(val)
            else:
                try:
                    numeric_values.append(float(val))
                except (ValueError, TypeError):
                    # If we can't convert all values to numeric, we can't detect trends
                    return patterns

        # Simple trend detection - check if values are consistently increasing or decreasing
        increases = 0
        decreases = 0

        for i in range(1, len(numeric_values)):
            if numeric_values[i] > numeric_values[i-1]:
                increases += 1
            elif numeric_values[i] < numeric_values[i-1]:
                decreases += 1

        # Determine if there's a consistent trend
        if increases >= len(numeric_values) - 2 and increases > 0:
            # Increasing trend
            pattern = Pattern(
                pattern_type=PatternType.TEMPORAL,
                pattern_data=numeric_values,
                confidence=min(0.5 + 0.1 * increases, 0.9),
                frequency=1,
                metadata={
                    "temporal_type": "trend",
                    "trend_direction": "increasing",
                    "trend_strength": increases / (len(numeric_values) - 1),
                    "last_value": numeric_values[-1]
                }
            )
            patterns.append(pattern)

        elif decreases >= len(numeric_values) - 2 and decreases > 0:
            # Decreasing trend
            pattern = Pattern(
                pattern_type=PatternType.TEMPORAL,
                pattern_data=numeric_values,
                confidence=min(0.5 + 0.1 * decreases, 0.9),
                frequency=1,
                metadata={
                    "temporal_type": "trend",
                    "trend_direction": "decreasing",
                    "trend_strength": decreases / (len(numeric_values) - 1),
                    "last_value": numeric_values[-1]
                }
            )
            patterns.append(pattern)

        elif (increases + decreases) < len(numeric_values) / 3:
            # Stable trend (not much change)
            pattern = Pattern(
                pattern_type=PatternType.TEMPORAL,
                pattern_data=numeric_values,
                confidence=0.7,
                frequency=1,
                metadata={
                    "temporal_type": "trend",
                    "trend_direction": "stable",
                    "trend_strength": 1.0 - (increases + decreases) / (len(numeric_values) - 1),
                    "last_value": numeric_values[-1]
                }
            )
            patterns.append(pattern)

        return patterns


class StructuralPatternDetector(PatternDetectorBase):
    """Detector for structural patterns in graph-like data"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect structural patterns in data and history.

        Args:
            data: Current structured data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected structural patterns
        """
        # This is a placeholder - real implementation would depend on
        # specific graph data structure and available graph algorithms

        # Return empty list for now
        return []

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a structural pattern against data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        # This is a placeholder - real implementation would depend on
        # specific graph data structure and subgraph isomorphism algorithms

        # Return empty list for now
        return []


class LinguisticPatternDetector(PatternDetectorBase):
    """Detector for linguistic patterns in text data"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect linguistic patterns in data and history.

        Args:
            data: Current text data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected linguistic patterns
        """
        patterns = []

        # Ensure we have text data
        if not isinstance(data, str):
            return patterns

        # Collect text from history
        all_texts = []
        for hist_data, data_type, timestamp, hist_context in history:
            if data_type == "text" and isinstance(hist_data, str):
                all_texts.append(hist_data)

        # Add current data
        all_texts.append(data)

        # Only perform detection if we have enough text
        if len(all_texts) < 2:
            return patterns

        # Extract potential patterns
        text_patterns = []

        # Common phrases (simple n-gram extraction)
        text_patterns.extend(self._extract_common_phrases(all_texts))

        # Regular expressions (simple matching patterns)
        text_patterns.extend(self._extract_regex_patterns(all_texts))

        # Create Pattern objects for the detected patterns
        for pattern_text, pattern_info in text_patterns:
            pattern = Pattern(
                pattern_type=PatternType.LINGUISTIC,
                pattern_data=pattern_text,
                confidence=pattern_info.get("confidence", 0.5),
                frequency=pattern_info.get("frequency", 1),
                metadata={
                    "linguistic_type": pattern_info.get("type", "phrase"),
                    "source": "linguistic_detector"
                }
            )
            patterns.append(pattern)

        return patterns

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a linguistic pattern against text data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        matches = []

        # Ensure we have text data
        if not isinstance(data, str) or not isinstance(pattern.pattern_data, str):
            return matches

        # Get linguistic type from metadata
        ling_type = pattern.metadata.get("linguistic_type", "phrase")

        if ling_type == "phrase":
            # Simple substring matching
            pattern_text = pattern.pattern_data

            # Find all occurrences
            start_index = 0
            while True:
                start_index = data.find(pattern_text, start_index)
                if start_index == -1:
                    break

                end_index = start_index + len(pattern_text)

                match = PatternMatch(
                    matched=True,
                    pattern=pattern,
                    confidence=pattern.confidence,
                    match_start=start_index,
                    match_end=end_index,
                    match_elements=[pattern_text],
                    similarity=1.0
                )
                matches.append(match)

                # Move past this match
                start_index = end_index

        elif ling_type == "regex":
            # Regular expression matching
            try:
                pattern_regex = pattern.pattern_data
                for match in re.finditer(pattern_regex, data):
                    pattern_match = PatternMatch(
                        matched=True,
                        pattern=pattern,
                        confidence=pattern.confidence,
                        match_start=match.start(),
                        match_end=match.end(),
                        match_elements=[match.group(0)],
                        similarity=1.0,
                        metadata={"groups": match.groups()}
                    )
                    matches.append(pattern_match)
            except re.error:
                # Invalid regex
                pass

        return matches

    def _extract_common_phrases(self, texts: List[str]) -> List[Tuple[str, Dict[str, Any]]]:
        """
        Extract common phrases from a list of texts.

        Args:
            texts: List of text strings

        Returns:
            List of (phrase, info_dict) tuples
        """
        common_phrases = []

        # Simplified n-gram extraction - real implementation would be more sophisticated
        n_values = [2, 3, 4]  # bigrams, trigrams, 4-grams

        for n in n_values:
            # Extract n-grams from all texts
            all_ngrams = []

            for text in texts:
                words = text.split()
                if len(words) < n:
                    continue

                text_ngrams = [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]
                all_ngrams.extend(text_ngrams)

            # Count occurrences
            ngram_counts = Counter(all_ngrams)

            # Filter for common n-grams (appearing at least twice)
            for ngram, count in ngram_counts.items():
                if count >= 2:
                    common_phrases.append((ngram, {
                        "type": "phrase",
                        "frequency": count,
                        "confidence": min(0.5 + 0.1 * count, 0.9),
                        "n_value": n
                    }))

        return common_phrases

    def _extract_regex_patterns(self, texts: List[str]) -> List[Tuple[str, Dict[str, Any]]]:
        """
        Extract regex patterns from texts.

        Args:
            texts: List of text strings

        Returns:
            List of (regex_pattern, info_dict) tuples
        """
        regex_patterns = []

        # Predefined regex patterns to check for
        common_regexes = [
            (r'\b\d{3}-\d{3}-\d{4}\b', "phone_number"),
            (r'\b\w+@\w+\.\w+\b', "email"),
            (r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', "ip_address"),
            (r'\b\d{5}(-\d{4})?\b', "zip_code"),
            (r'\b(https?://)\S+\b', "url")
        ]

        for pattern, pattern_name in common_regexes:
            # Count how many texts contain this pattern
            match_count = 0
            for text in texts:
                if re.search(pattern, text):
                    match_count += 1

            # If pattern appears in multiple texts, add it
            if match_count >= 2:
                regex_patterns.append((pattern, {
                    "type": "regex",
                    "frequency": match_count,
                    "confidence": min(0.6 + 0.1 * match_count, 0.9),
                    "pattern_name": pattern_name
                }))

        return regex_patterns


class SymbolicPatternDetector(PatternDetectorBase):
    """Detector for symbolic patterns in mathematical or logical expressions"""

    def detect(self, data: Any, history: List[Tuple[Any, str, float, Dict[str, Any]]],
              context: Dict[str, Any]) -> List[Pattern]:
        """
        Detect symbolic patterns in data and history.

        Args:
            data: Current symbolic data
            history: List of (data, data_type, timestamp, context) tuples
            context: Additional contextual information

        Returns:
            List of detected symbolic patterns
        """
        # This is a placeholder - real implementation would depend on
        # symbolic expression parsing and pattern matching algorithms

        # Return empty list for now
        return []

    def match(self, pattern: Pattern, data: Any, context: Dict[str, Any]) -> List[PatternMatch]:
        """
        Match a symbolic pattern against data.

        Args:
            pattern: Pattern to match
            data: Data to match against
            context: Additional contextual information

        Returns:
            List of pattern matches
        """
        # This is a placeholder - real implementation would depend on
        # symbolic expression matching algorithms

        # Return empty list for now
        return []

# Attention Filter

In [ ]:
"""
attention_filter.py - Attentional Filter Module for GrandNexus

This module provides a framework for filtering and highlighting information based on
attentional mechanisms, allowing GrandNexus to focus on relevant data while ignoring
distractions. It implements various attentional strategies inspired by cognitive sciences,
including:

- Bottom-up attention: Driven by inherent properties of stimuli (salience, novelty)
- Top-down attention: Guided by current goals, expectations, and context
- Divided attention: Managing multiple sources of information simultaneously
- Sustained attention: Maintaining focus over time despite distractions

The module is designed to work closely with perception and reasoning components, enhancing
the system's ability to extract meaningful patterns from complex data streams.
"""

import logging
import threading
import time
import uuid
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Dict, List, Any, Optional, Callable, Union, Set, Tuple
import numpy as np
from collections import defaultdict, deque
import re
import random

# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.cognition.perception.perception_core import PerceptionCore


class AttentionMode(Enum):
    """Different modes of attentional filtering"""
    BOTTOM_UP = auto()    # Salience-driven attention
    TOP_DOWN = auto()     # Goal-directed attention
    DIVIDED = auto()      # Multi-focus attention
    SUSTAINED = auto()    # Long-term focus maintenance
    HYBRID = auto()       # Combination of strategies
    EXPLORATORY = auto()  # Novelty-seeking attention
    URGENT = auto()       # Urgent signal prioritization


@dataclass
class AttentionalFocus:
    """Representation of the current attentional focus"""
    focus_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    mode: AttentionMode = AttentionMode.HYBRID
    target_concepts: List[str] = field(default_factory=list)
    target_patterns: List[str] = field(default_factory=list)
    importance_threshold: float = 0.3
    activation_level: float = 1.0
    start_time: float = field(default_factory=time.time)
    duration: Optional[float] = None  # None means indefinite
    decay_rate: float = 0.05
    context: Dict[str, Any] = field(default_factory=dict)

    def is_active(self, current_time: Optional[float] = None) -> bool:
        """Check if the attentional focus is still active"""
        if current_time is None:
            current_time = time.time()

        if self.duration is not None:
            return (current_time - self.start_time) < self.duration
        return True

    def get_current_activation(self, current_time: Optional[float] = None) -> float:
        """Get the current activation level, accounting for decay"""
        if current_time is None:
            current_time = time.time()

        time_elapsed = current_time - self.start_time

        # Apply exponential decay
        decay_factor = np.exp(-self.decay_rate * time_elapsed)
        return self.activation_level * decay_factor

    def matches_content(self, content: Union[str, Dict[str, Any]]) -> Tuple[bool, float]:
        """
        Check if content matches this attentional focus.

        Args:
            content: String or dictionary content to check

        Returns:
            Tuple of (matches, score) where score indicates strength of match
        """
        if isinstance(content, str):
            text = content
            metadata = {}
        elif isinstance(content, dict):
            text = content.get("text", "")
            metadata = content
        else:
            # Unsupported content type
            return False, 0.0

        # Check for concept matches
        concept_matches = 0
        for concept in self.target_concepts:
            if concept.lower() in text.lower():
                concept_matches += 1

        # Check for pattern matches
        pattern_matches = 0
        for pattern in self.target_patterns:
            try:
                if re.search(pattern, text, re.IGNORECASE):
                    pattern_matches += 1
            except re.error:
                # If pattern is not a valid regex, treat it as a literal string
                if pattern.lower() in text.lower():
                    pattern_matches += 1

        # Calculate match score
        total_targets = len(self.target_concepts) + len(self.target_patterns)
        if total_targets == 0:
            return True, 1.0  # No targets means everything matches

        match_ratio = (concept_matches + pattern_matches) / total_targets

        # Check metadata match if available
        metadata_match = 0.0
        if metadata and self.context:
            matching_keys = set(metadata.keys()) & set(self.context.keys())
            if matching_keys:
                for key in matching_keys:
                    if metadata[key] == self.context[key]:
                        metadata_match += 1.0 / len(matching_keys)

        # Combine scores
        combined_score = 0.7 * match_ratio + 0.3 * metadata_match

        # Apply activation as a weight
        current_activation = self.get_current_activation()
        weighted_score = combined_score * current_activation

        return weighted_score >= self.importance_threshold, weighted_score


@dataclass
class FilteredPerception:
    """Result of applying attentional filters to perceptual input"""
    original_input: Any
    filtered_output: Any
    attention_score: float = 0.0
    matching_focus: Optional[str] = None  # ID of matching attentional focus
    filtered_by: List[str] = field(default_factory=list)  # IDs of applied filters
    metadata: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)


@dataclass
class SaliencyMap:
    """
    Represents a map of content importance based on various factors.
    This is used for bottom-up attention to highlight inherently salient information.
    """
    content_id: str
    base_map: Dict[str, float] = field(default_factory=dict)  # Element -> salience
    temporal_factors: Dict[str, float] = field(default_factory=dict)  # Time-based adjustments
    contextual_factors: Dict[str, float] = field(default_factory=dict)  # Context-based adjustments

    def get_combined_salience(self, element: str) -> float:
        """Get the combined salience value for an element"""
        base = self.base_map.get(element, 0.0)
        temporal = self.temporal_factors.get(element, 1.0)
        contextual = self.contextual_factors.get(element, 1.0)

        return base * temporal * contextual

    def sort_by_salience(self) -> List[Tuple[str, float]]:
        """Sort elements by their combined salience"""
        elements = list(set(self.base_map.keys()) |
                       set(self.temporal_factors.keys()) |
                       set(self.contextual_factors.keys()))

        salience_scores = [(elem, self.get_combined_salience(elem)) for elem in elements]
        return sorted(salience_scores, key=lambda x: x[1], reverse=True)


class AttentionFilter:
    """
    Main class for attentional filtering in GrandNexus.

    This module filters perceptual input based on current attentional focus,
    goals, and salience of the information itself. It helps GrandNexus focus
    on relevant information while preventing distraction from less important data.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 perception_core: Optional[PerceptionCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None):
        """
        Initialize the attention filter module.

        Args:
            nexus_core: Reference to the NexusCore for system-wide coordination
            perception_core: Reference to the perception module for input
            working_memory: Reference to working memory
            semantic_memory: Reference to semantic memory for concept lookups
        """
        self.logger = logging.getLogger("GrandNexus.Perception.AttentionFilter")
        self.nexus_core = nexus_core
        self.perception_core = perception_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Attentional focus management
        self.attentional_foci = {}  # ID -> AttentionalFocus
        self.default_focus = None  # Default focus when no specific focus matches

        # Salience tracking
        self.saliency_maps = {}  # Content ID -> SaliencyMap

        # Filter history and metrics
        self.filter_history = deque(maxlen=100)
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        # Filter registry
        self.filters = {}

        self.logger.info(f"AttentionFilter initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the attention filter and register default filters"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register built-in filters
                self._register_default_filters()

                # Create default attentional focus
                self._create_default_focus()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("AttentionFilter initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"AttentionFilter initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the attention filter"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                self.running = True
                self.logger.info("AttentionFilter started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start AttentionFilter: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the attention filter"""
        with self.lock:
            if not self.running:
                return True

            try:
                self.running = False
                self.logger.info("AttentionFilter stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping AttentionFilter: {str(e)}")
                return False

    def filter(self, input_data: Any, context: Optional[Dict[str, Any]] = None) -> FilteredPerception:
        """
        Apply attentional filtering to input data.

        Args:
            input_data: The input data to filter
            context: Optional context information for filtering

        Returns:
            FilteredPerception with filtered output and metadata
        """
        if not self.initialized:
            self.initialize()

        if context is None:
            context = {}

        # Create a working copy of the input data
        filtered_data = self._clone_input(input_data)

        # Apply each registered filter in sequence
        applied_filters = []
        attention_score = 0.0
        matching_focus_id = None

        # Determine active attentional foci
        active_foci = self._get_active_foci()

        # If input_data is a dictionary with a 'content_id', create/update saliency map
        if isinstance(input_data, dict) and 'content_id' in input_data:
            content_id = input_data['content_id']
            self._update_saliency_map(content_id, input_data, context)

        # First check if the input matches any attentional focus
        best_match_score = 0.0
        best_match_id = None

        for focus_id, focus in active_foci.items():
            matches, score = focus.matches_content(input_data)
            if matches and score > best_match_score:
                best_match_score = score
                best_match_id = focus_id

        # If no specific focus matches, use default focus
        if best_match_id is None and self.default_focus is not None:
            best_match_id = self.default_focus.focus_id
            best_match_score = 0.5  # Moderate score for default focus

        # Apply filters based on matching focus and context
        if best_match_id is not None:
            matching_focus = active_foci[best_match_id]
            matching_focus_id = best_match_id
            attention_score = best_match_score

            # Determine which filters to apply based on focus mode
            filter_keys = self._get_filters_for_mode(matching_focus.mode)

            for filter_key in filter_keys:
                if filter_key in self.filters:
                    filter_func = self.filters[filter_key]
                    try:
                        filter_result = filter_func(filtered_data, matching_focus, context)
                        if isinstance(filter_result, tuple) and len(filter_result) == 2:
                            # Filter returns (filtered_data, changed)
                            filtered_data, changed = filter_result
                            if changed:
                                applied_filters.append(filter_key)
                        else:
                            # Filter just returns filtered_data
                            filtered_data = filter_result
                            applied_filters.append(filter_key)
                    except Exception as e:
                        self.logger.error(f"Error applying filter {filter_key}: {str(e)}")

        # Update metrics
        self.metrics["attention_score"].append(attention_score)
        self.metrics["filters_applied"].append(len(applied_filters))

        # Create result
        result = FilteredPerception(
            original_input=input_data,
            filtered_output=filtered_data,
            attention_score=attention_score,
            matching_focus=matching_focus_id,
            filtered_by=applied_filters,
            metadata={"context": context}
        )

        # Update history
        self.filter_history.append(result)

        return result

    def create_focus(self, mode: AttentionMode,
                     target_concepts: Optional[List[str]] = None,
                     target_patterns: Optional[List[str]] = None,
                     importance_threshold: float = 0.3,
                     duration: Optional[float] = None,
                     context: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new attentional focus.

        Args:
            mode: Mode of attention to use
            target_concepts: Concepts to focus on
            target_patterns: Patterns to look for
            importance_threshold: Minimum importance for focus activation
            duration: Optional duration in seconds (None=indefinite)
            context: Additional contextual information

        Returns:
            ID of the created attentional focus
        """
        if target_concepts is None:
            target_concepts = []
        if target_patterns is None:
            target_patterns = []
        if context is None:
            context = {}

        focus = AttentionalFocus(
            mode=mode,
            target_concepts=target_concepts,
            target_patterns=target_patterns,
            importance_threshold=importance_threshold,
            duration=duration,
            context=context
        )

        with self.lock:
            self.attentional_foci[focus.focus_id] = focus
            self.logger.info(f"Created attentional focus {focus.focus_id} with mode {mode}")

        return focus.focus_id

    def update_focus(self, focus_id: str,
                    target_concepts: Optional[List[str]] = None,
                    target_patterns: Optional[List[str]] = None,
                    importance_threshold: Optional[float] = None,
                    duration: Optional[float] = None,
                    context: Optional[Dict[str, Any]] = None) -> bool:
        """
        Update an existing attentional focus.

        Args:
            focus_id: ID of the focus to update
            target_concepts: New concepts to focus on (None to keep existing)
            target_patterns: New patterns to look for (None to keep existing)
            importance_threshold: New threshold (None to keep existing)
            duration: New duration (None to keep existing)
            context: Additional contextual information (None to keep existing)

        Returns:
            True if update succeeded, False otherwise
        """
        with self.lock:
            if focus_id not in self.attentional_foci:
                self.logger.warning(f"Attentional focus {focus_id} not found")
                return False

            focus = self.attentional_foci[focus_id]

            if target_concepts is not None:
                focus.target_concepts = target_concepts
            if target_patterns is not None:
                focus.target_patterns = target_patterns
            if importance_threshold is not None:
                focus.importance_threshold = importance_threshold
            if duration is not None:
                focus.duration = duration
            if context is not None:
                focus.context = context

            self.logger.info(f"Updated attentional focus {focus_id}")
            return True

    def remove_focus(self, focus_id: str) -> bool:
        """
        Remove an attentional focus.

        Args:
            focus_id: ID of the focus to remove

        Returns:
            True if removal succeeded, False otherwise
        """
        with self.lock:
            if focus_id not in self.attentional_foci:
                self.logger.warning(f"Attentional focus {focus_id} not found")
                return False

            # Don't remove the default focus
            if self.default_focus is not None and focus_id == self.default_focus.focus_id:
                self.logger.warning("Cannot remove default attentional focus")
                return False

            del self.attentional_foci[focus_id]
            self.logger.info(f"Removed attentional focus {focus_id}")
            return True

    def get_focus(self, focus_id: str) -> Optional[AttentionalFocus]:
        """
        Get information about an attentional focus.

        Args:
            focus_id: ID of the focus to get

        Returns:
            AttentionalFocus if found, None otherwise
        """
        with self.lock:
            return self.attentional_foci.get(focus_id)

    def set_default_focus(self, focus_id: str) -> bool:
        """
        Set the default attentional focus.

        Args:
            focus_id: ID of the focus to set as default

        Returns:
            True if setting succeeded, False otherwise
        """
        with self.lock:
            if focus_id not in self.attentional_foci:
                self.logger.warning(f"Attentional focus {focus_id} not found")
                return False

            self.default_focus = self.attentional_foci[focus_id]
            self.logger.info(f"Set default attentional focus to {focus_id}")
            return True

    def register_filter(self, filter_key: str,
                        filter_func: Callable[[Any, AttentionalFocus, Dict[str, Any]], Any]) -> None:
        """
        Register a custom filter function.

        Args:
            filter_key: Unique key for the filter
            filter_func: Function that takes (data, focus, context) and returns filtered data
                         or (filtered_data, changed) tuple
        """
        with self.lock:
            self.filters[filter_key] = filter_func
            self.logger.info(f"Registered filter: {filter_key}")

    def get_filter_metrics(self) -> Dict[str, Any]:
        """Get current filtering performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def get_active_foci(self) -> Dict[str, Dict[str, Any]]:
        """
        Get information about all active attentional foci.

        Returns:
            Dictionary mapping focus IDs to information dictionaries
        """
        active_foci = self._get_active_foci()

        # Convert to serializable format
        result = {}
        for focus_id, focus in active_foci.items():
            result[focus_id] = {
                "mode": focus.mode.name,
                "target_concepts": focus.target_concepts,
                "target_patterns": focus.target_patterns,
                "activation": focus.get_current_activation(),
                "start_time": focus.start_time,
                "duration": focus.duration
            }

        return result

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_default_filters(self) -> None:
        """Register built-in filter functions"""
        # Bottom-up salience filter
        self.register_filter("salience", self._filter_by_salience)

        # Top-down selective attention
        self.register_filter("selective", self._filter_by_selective_attention)

        # Conceptual relevance filter
        self.register_filter("concept_relevance", self._filter_by_concept_relevance)

        # Pattern matching filter
        self.register_filter("pattern_matching", self._filter_by_pattern_matching)

        # Urgency filter
        self.register_filter("urgency", self._filter_by_urgency)

        # Novelty filter
        self.register_filter("novelty", self._filter_by_novelty)

    def _create_default_focus(self) -> None:
        """Create a default attentional focus"""
        default_focus_id = self.create_focus(
            mode=AttentionMode.HYBRID,
            target_concepts=[],
            target_patterns=[],
            importance_threshold=0.1  # Low threshold to match most things
        )

        self.default_focus = self.attentional_foci[default_focus_id]

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.perception_core:
                dependencies.append("perception_core")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")

            self.nexus_core.register_module(
                name="attention_filter",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _clone_input(self, input_data: Any) -> Any:
        """Create a working copy of input data"""
        if isinstance(input_data, dict):
            return input_data.copy()
        elif isinstance(input_data, list):
            return input_data.copy()
        elif isinstance(input_data, str):
            return input_data
        else:
            # For other types, we might need more sophisticated cloning
            # This is a simplified approach
            return input_data

    def _get_active_foci(self) -> Dict[str, AttentionalFocus]:
        """Get all currently active attentional foci"""
        current_time = time.time()

        with self.lock:
            active_foci = {}
            for focus_id, focus in self.attentional_foci.items():
                if focus.is_active(current_time):
                    active_foci[focus_id] = focus

            return active_foci

    def _get_filters_for_mode(self, mode: AttentionMode) -> List[str]:
        """Get appropriate filters for a given attention mode"""
        # Define filter sets for each mode
        filter_sets = {
            AttentionMode.BOTTOM_UP: ["salience", "novelty"],
            AttentionMode.TOP_DOWN: ["selective", "concept_relevance"],
            AttentionMode.DIVIDED: ["selective", "salience"],
            AttentionMode.SUSTAINED: ["selective", "concept_relevance", "pattern_matching"],
            AttentionMode.HYBRID: ["salience", "selective", "concept_relevance", "pattern_matching"],
            AttentionMode.EXPLORATORY: ["novelty", "salience"],
            AttentionMode.URGENT: ["urgency", "selective"]
        }

        return filter_sets.get(mode, ["selective"])  # Default to selective as fallback

    def _update_saliency_map(self, content_id: str, content: Dict[str, Any],
                            context: Dict[str, Any]) -> None:
        """Create or update a saliency map for content"""
        # This is a simplified implementation
        # In a real system, this would analyze content more deeply

        # Create base map if it doesn't exist
        if content_id not in self.saliency_maps:
            self.saliency_maps[content_id] = SaliencyMap(content_id=content_id)

        saliency_map = self.saliency_maps[content_id]

        # Update base map
        if "text" in content:
            text = content["text"]
            words = text.split()

            # Simple heuristics for salience:
            # - Words in ALL CAPS are more salient
            # - Words with exclamation marks are more salient
            # - Longer words may be more salient (e.g., technical terms)

            for word in words:
                clean_word = word.strip(".,!?;:()")
                if not clean_word:
                    continue

                salience = 0.5  # Default salience

                # ALL CAPS words
                if clean_word.isupper() and len(clean_word) > 1:
                    salience += 0.3

                # Words with emphasis
                if "!" in word:
                    salience += 0.2

                # Longer words
                if len(clean_word) > 8:
                    salience += 0.1

                saliency_map.base_map[clean_word] = salience

        # Update temporal factors (recency bias)
        current_time = time.time()
        for word in saliency_map.base_map:
            # Newer mentions get higher temporal factors
            saliency_map.temporal_factors[word] = 1.0

        # Update contextual factors based on current context
        if "goal" in context:
            goal = context["goal"]
            goal_words = goal.lower().split()

            for word in saliency_map.base_map:
                if word.lower() in goal_words:
                    saliency_map.contextual_factors[word] = 1.5  # Boost if related to goal
                else:
                    saliency_map.contextual_factors[word] = 1.0

    # -----------------------------------------------------------------------
    # Filter implementations
    # -----------------------------------------------------------------------

    def _filter_by_salience(self, data: Any, focus: AttentionalFocus,
                           context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on bottom-up salience.
        Highlights or enhances salient elements while suppressing less salient ones.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # This is a simplified implementation
        # In a real system, this would analyze content more deeply

        # For text data, we might highlight salient words
        if isinstance(data, str):
            # For now, we'll just return the original data
            # In a real implementation, we might add markup or structure
            return data, False

        elif isinstance(data, dict) and "text" in data:
            # If we have a content_id, we might have a saliency map
            content_id = data.get("content_id", "")
            if content_id in self.saliency_maps:
                saliency_map = self.saliency_maps[content_id]

                # Get top salient elements
                top_elements = saliency_map.sort_by_salience()[:5]  # Top 5 most salient

                # Add salience information to the data
                filtered_data = data.copy()
                filtered_data["salient_elements"] = top_elements

                return filtered_data, True

        # For other data types or no changes
        return data, False

    def _filter_by_selective_attention(self, data: Any, focus: AttentionalFocus,
                                      context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on top-down selective attention.
        Focuses on elements that match current goals and intentions.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # For text data, prioritize content related to target concepts
        if isinstance(data, str):
            text = data
            highlighted_text = text

            # Highlight target concepts
            for concept in focus.target_concepts:
                if concept.lower() in text.lower():
                    # In a real implementation, we might add markup or structure
                    # Here we just leave it unchanged
                    pass

            if highlighted_text != text:
                return highlighted_text, True
            return text, False

        elif isinstance(data, dict) and "text" in data:
            text = data["text"]
            filtered_data = data.copy()

            # Extract parts most relevant to focus
            relevant_parts = []

            # Split into sentences (simplistic)
            sentences = text.split(". ")

            for sentence in sentences:
                relevance = 0
                for concept in focus.target_concepts:
                    if concept.lower() in sentence.lower():
                        relevance += 1

                if relevance > 0:
                    relevant_parts.append(sentence)

            if relevant_parts:
                filtered_data["relevant_parts"] = relevant_parts
                return filtered_data, True

        # For other data types or no changes
        return data, False

   def _filter_by_selective_attention(self, data: Any, focus: AttentionalFocus,
                                      context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on top-down selective attention.
        Focuses on elements that match current goals and intentions.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # For text data, prioritize content related to target concepts
        if isinstance(data, str):
            text = data
            highlighted_text = text

            # Highlight target concepts
            for concept in focus.target_concepts:
                if concept.lower() in text.lower():
                    # In a real implementation, we might add markup or structure
                    # Here we just leave it unchanged
                    pass

            if highlighted_text != text:
                return highlighted_text, True
            return text, False

        elif isinstance(data, dict) and "text" in data:
            text = data["text"]
            filtered_data = data.copy()

            # Extract parts most relevant to focus
            relevant_parts = []

            # Split into sentences (simplistic)
            sentences = text.split(". ")

            for sentence in sentences:
                relevance = 0
                for concept in focus.target_concepts:
                    if concept.lower() in sentence.lower():
                        relevance += 1

                if relevance > 0:
                    relevant_parts.append(sentence)

            if relevant_parts:
                filtered_data["relevant_parts"] = relevant_parts
                return filtered_data, True

        # For other data types or no changes
        return data, False

    def _filter_by_concept_relevance(self, data: Any, focus: AttentionalFocus,
                                    context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on conceptual relevance.
        Enhances content related to targeted concepts in semantic memory.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # This filter requires semantic memory to work effectively
        if not self.semantic_memory:
            return data, False

        # For text data
        if isinstance(data, str):
            text = data
            # In a real implementation, we could look up related concepts
            # in semantic memory
            return text, False

        elif isinstance(data, dict) and "text" in data:
            text = data["text"]
            filtered_data = data.copy()

            # Extract concepts from text
            # This is a simplistic approach; a real implementation would use NLP
            words = set(text.lower().split())

            # Try to find related concepts in semantic memory
            related_concepts = []
            for concept in focus.target_concepts:
                # Placeholder for semantic memory query
                # In a real implementation, this would query the semantic memory
                # for concepts related to the target concept

                # For now, we'll just do simple string matching
                for word in words:
                    if concept.lower() in word or word in concept.lower():
                        related_concepts.append((word, concept))

            if related_concepts:
                filtered_data["related_concepts"] = related_concepts
                return filtered_data, True

        # For other data types or no changes
        return data, False

    def _filter_by_pattern_matching(self, data: Any, focus: AttentionalFocus,
                                  context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on pattern matching.
        Identifies content that matches specific patterns.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # For text data
        if isinstance(data, str):
            text = data
            matches = []

            for pattern in focus.target_patterns:
                try:
                    found = re.findall(pattern, text, re.IGNORECASE)
                    matches.extend(found)
                except re.error:
                    # If pattern is not a valid regex, treat it as a literal string
                    if pattern.lower() in text.lower():
                        matches.append(pattern)

            if matches:
                # In a real implementation, we might add markup or structure
                # Here we just return the original text
                return text, False
            return text, False

        elif isinstance(data, dict) and "text" in data:
            text = data["text"]
            filtered_data = data.copy()

            pattern_matches = []

            for pattern in focus.target_patterns:
                try:
                    found = re.findall(pattern, text, re.IGNORECASE)
                    if found:
                        pattern_matches.append((pattern, found))
                except re.error:
                    # If pattern is not a valid regex, treat it as a literal string
                    if pattern.lower() in text.lower():
                        pattern_matches.append((pattern, [pattern]))

            if pattern_matches:
                filtered_data["pattern_matches"] = pattern_matches
                return filtered_data, True

        # For other data types or no changes
        return data, False

    def _filter_by_urgency(self, data: Any, focus: AttentionalFocus,
                         context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on urgency.
        Prioritizes content that requires immediate attention.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # Check for urgency indicators in context
        is_urgent = (context.get("urgency", 0) > 0.7 or
                     context.get("emergency", False))

        # For text data
        if isinstance(data, str):
            text = data

            # Check for urgency markers in the text
            urgency_markers = ["urgent", "immediately", "emergency", "critical", "asap"]
            text_urgent = any(marker in text.lower() for marker in urgency_markers)

            if is_urgent or text_urgent:
                # In a real implementation, we might add markup or priority flags
                # Here we just return the original text
                return text, False
            return text, False

        elif isinstance(data, dict):
            filtered_data = data.copy()

            # Check for urgency in the data
            text_urgent = False
            if "text" in data:
                text = data["text"]
                urgency_markers = ["urgent", "immediately", "emergency", "critical", "asap"]
                text_urgent = any(marker in text.lower() for marker in urgency_markers)

            if is_urgent or text_urgent:
                filtered_data["is_urgent"] = True
                # Increase priority if that field exists
                if "priority" in filtered_data:
                    filtered_data["priority"] = max(filtered_data["priority"], 0.9)
                else:
                    filtered_data["priority"] = 0.9

                return filtered_data, True

        # For other data types or no changes
        return data, False

    def _filter_by_novelty(self, data: Any, focus: AttentionalFocus,
                         context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Filter based on novelty.
        Highlights content that contains new or unexpected information.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # This filter ideally requires working memory to track what's been seen
        if not self.working_memory:
            return data, False

        # For text data
        if isinstance(data, str):
            text = data
            # In a real implementation, we would check against working memory
            # to determine what's novel
            return text, False

        elif isinstance(data, dict) and "text" in data:
            text = data["text"]
            filtered_data = data.copy()

            # Simplified novelty detection
            # In a real implementation, this would query working memory
            # to check what's been seen recently

            # For demonstration, just randomly mark some content as novel
            if random.random() < 0.3:  # 30% chance
                filtered_data["is_novel"] = True
                return filtered_data, True

        # For other data types or no changes
        return data, False


# Create a submodule for specific attention filters

class SelectiveAttentionFilter:
    """
    Specialized filter for selective attention based on specific criteria.
    This can be used directly or registered with the main AttentionFilter.
    """

    def __init__(self, criteria: Dict[str, Any],
                working_memory: Optional[WorkingMemory] = None):
        """
        Initialize a selective attention filter.

        Args:
            criteria: Dictionary of criteria to filter by
            working_memory: Optional reference to working memory
        """
        self.criteria = criteria
        self.working_memory = working_memory
        self.logger = logging.getLogger("GrandNexus.Perception.SelectiveFilter")

    def filter(self, data: Any, focus: AttentionalFocus,
              context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Apply selective filtering based on criteria.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # Check if data matches criteria
        if isinstance(data, dict):
            filtered_data = data.copy()
            matches = []

            for key, value in self.criteria.items():
                if key in data and data[key] == value:
                    matches.append(key)

            if matches:
                filtered_data["matching_criteria"] = matches
                return filtered_data, True

        # No matches or unsupported data type
        return data, False


class TemporalAttentionFilter:
    """
    Specialized filter for attention based on temporal patterns.
    This can be used to detect sequences, trends, or changes over time.
    """

    def __init__(self, window_size: int = 5,
                working_memory: Optional[WorkingMemory] = None):
        """
        Initialize a temporal attention filter.

        Args:
            window_size: Number of recent items to consider
            working_memory: Optional reference to working memory
        """
        self.window_size = window_size
        self.working_memory = working_memory
        self.history = deque(maxlen=window_size)
        self.logger = logging.getLogger("GrandNexus.Perception.TemporalFilter")

    def filter(self, data: Any, focus: AttentionalFocus,
              context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Apply temporal filtering.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        # Add current data to history
        self.history.append(data)

        # Not enough history yet
        if len(self.history) < 2:
            return data, False

        # Check for temporal patterns
        if isinstance(data, dict):
            filtered_data = data.copy()

            # Check for changes
            previous = self.history[-2]
            if isinstance(previous, dict):
                changes = []

                for key in set(data.keys()) | set(previous.keys()):
                    if key in data and key in previous:
                        if data[key] != previous[key]:
                            changes.append((key, previous[key], data[key]))
                    elif key in data:
                        changes.append((key, None, data[key]))
                    elif key in previous:
                        changes.append((key, previous[key], None))

                if changes:
                    filtered_data["temporal_changes"] = changes
                    return filtered_data, True

        # No changes or unsupported data type
        return data, False


# Function to create a specialized attention filter for detecting anomalies

    def create_anomaly_filter(baseline: Dict[str, Any],
                         threshold: float = 0.2,
                         working_memory: Optional[WorkingMemory] = None) -> Callable:
    """
    Create an anomaly detection filter.

    Args:
        baseline: Baseline normal values
        threshold: Threshold for anomaly detection
        working_memory: Optional reference to working memory

    Returns:
        Filter function that can be registered with AttentionFilter
    """
    logger = logging.getLogger("GrandNexus.Perception.AnomalyFilter")

    def anomaly_filter(data: Any, focus: AttentionalFocus,
                      context: Dict[str, Any]) -> Tuple[Any, bool]:
        """
        Detect anomalies compared to baseline.

        Args:
            data: Data to filter
            focus: Current attentional focus
            context: Additional context

        Returns:
            Tuple of (filtered_data, changed)
        """
        if not isinstance(data, dict):
            return data, False

        filtered_data = data.copy()
        anomalies = []

        for key, base_value in baseline.items():
            if key in data:
                current_value = data[key]

                # Check for numerical anomalies
                if isinstance(base_value, (int, float)) and isinstance(current_value, (int, float)):
                    if abs(current_value - base_value) / max(1, abs(base_value)) > threshold:
                        anomalies.append((key, base_value, current_value))
                # Check for categorical anomalies
                elif current_value != base_value:
                    anomalies.append((key, base_value, current_value))

        if anomalies:
            filtered_data["anomalies"] = anomalies
            logger.info(f"Detected {len(anomalies)} anomalies")
            return filtered_data, True

        return data, False

    return anomaly_filter

class ContextualFilter:
    """
    Filter factory for creating context-specific attention filters.
    This class is designed to be used by AttentionFilter to dynamically
    generate filters based on specific contexts and needs.
    """

    @staticmethod
    def create_keyword_filter(keywords: List[str], importance_factor: float = 1.0) -> Callable:
        """
        Create a filter that highlights content containing specific keywords.

        Args:
            keywords: List of keywords to focus on
            importance_factor: Multiplier for matched content importance

        Returns:
            Filter function
        """
        def keyword_filter(data: Any, focus: AttentionalFocus, context: Dict[str, Any]) -> Tuple[Any, bool]:
            if isinstance(data, dict) and "text" in data:
                text = data["text"]
                filtered_data = data.copy()

                keyword_matches = []
                for keyword in keywords:
                    if keyword.lower() in text.lower():
                        keyword_matches.append(keyword)

                if keyword_matches:
                    filtered_data["keyword_matches"] = keyword_matches
                    filtered_data["keyword_importance"] = len(keyword_matches) * importance_factor
                    return filtered_data, True

            return data, False

        return keyword_filter

    @staticmethod
    def create_emotional_filter(emotions: List[str]) -> Callable:
        """
        Create a filter that detects emotional content.

        Args:
            emotions: List of emotions to detect (e.g., "joy", "anger")

        Returns:
            Filter function
        """
        # Simplified emotion keyword mapping
        emotion_keywords = {
            "joy": ["happy", "joy", "excited", "delighted", "pleased"],
            "anger": ["angry", "mad", "furious", "outraged", "irritated"],
            "sadness": ["sad", "unhappy", "depressed", "miserable", "gloomy"],
            "fear": ["afraid", "scared", "fearful", "terrified", "anxious"],
            "surprise": ["surprised", "shocked", "amazed", "astonished", "stunned"]
        }

        # Expand emotions list to keywords
        keywords = []
        for emotion in emotions:
            if emotion in emotion_keywords:
                keywords.extend(emotion_keywords[emotion])

        def emotion_filter(data: Any, focus: AttentionalFocus, context: Dict[str, Any]) -> Tuple[Any, bool]:
            if isinstance(data, dict) and "text" in data:
                text = data["text"]
                filtered_data = data.copy()

                emotion_matches = {}
                for emotion in emotions:
                    if emotion not in emotion_keywords:
                        continue

                    matches = 0
                    for keyword in emotion_keywords[emotion]:
                        matches += text.lower().count(keyword.lower())

                    if matches > 0:
                        emotion_matches[emotion] = matches

                if emotion_matches:
                    filtered_data["emotion_analysis"] = emotion_matches
                    return filtered_data, True

            return data, False

        return emotion_filter


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 837)

# Reasoning Core

In [ ]:
"""
reasoning_core.py - Core Reasoning Module for GrandNexus

This module provides the foundation for reasoning capabilities in GrandNexus.
It facilitates different forms of reasoning including deductive, inductive,
abductive, analogical, and causal reasoning, while allowing for emergent
behavior through flexible integration of reasoning strategies.

The reasoning module serves as a substrate for higher-level cognitive processes,
providing mechanisms for drawing inferences, solving problems, making decisions,
and generating hypotheses.

Key principles:
- Support diverse reasoning strategies without forcing a particular approach
- Allow for dynamic composition of reasoning methods based on context
- Facilitate emergent reasoning patterns through interaction between strategies
- Provide mechanisms for both certain and uncertain reasoning
- Support integration with memory and perception modules
"""

import logging
import threading
import time
import uuid
from typing import Dict, List, Any, Optional, Set, Union, Callable, Tuple
from dataclasses import dataclass, field
from enum import Enum, auto
import random
import math

# Will be imported from other modules
from grandnexus.cognition.cognition_manager import (
    CognitionManager, CognitiveModuleBase, CognitiveProcessType,
    CognitiveMode, CognitiveContext, CognitiveRequest, CognitiveResult
)
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.graph.graph_engine import GraphEngine
from grandnexus.graph.graph_engine import create_graph_engine, AbstractGraphEngine
from grandnexus.llm_interface.llm_engine import LLMEngine


class ReasoningType(Enum):
    """Types of reasoning processes"""
    DEDUCTIVE = auto()    # From general principles to specific conclusions
    INDUCTIVE = auto()    # From specific observations to general principles
    ABDUCTIVE = auto()    # Inference to the best explanation
    ANALOGICAL = auto()   # Mapping between similar situations
    CAUSAL = auto()       # Reasoning about cause and effect
    COUNTERFACTUAL = auto() # Reasoning about hypothetical scenarios
    STATISTICAL = auto()  # Probabilistic reasoning
    SPATIAL = auto()      # Reasoning about spatial relationships
    TEMPORAL = auto()     # Reasoning about time and sequence
    CREATIVE = auto()     # Generating novel combinations and ideas
    EMOTIONAL = auto()    # Reasoning about feelings and social context
    HYBRID = auto()       # Combination of multiple reasoning types


class ConfidenceLevel(Enum):
    """Levels of confidence in reasoning results"""
    CERTAIN = 1.0         # Logically certain (like mathematical proof)
    VERY_LIKELY = 0.9     # Strong evidence, highly probable
    LIKELY = 0.7          # Good evidence, more likely than not
    PLAUSIBLE = 0.5       # Possible explanation, balanced evidence
    UNLIKELY = 0.3        # Weak evidence, improbable but not impossible
    VERY_UNLIKELY = 0.1   # Very weak evidence, highly improbable
    UNKNOWN = 0.0         # Cannot determine confidence


@dataclass
class ReasoningStrategy:
    """
    A specific approach to reasoning that can be employed by the reasoning engine.
    Strategies can be composed and combined to create more complex reasoning patterns.
    """
    strategy_id: str
    reasoning_type: ReasoningType
    description: str
    handler: Callable  # Function that implements the strategy
    required_inputs: List[str] = field(default_factory=list)
    provided_outputs: List[str] = field(default_factory=list)
    compatibility: Set[ReasoningType] = field(default_factory=set)  # Compatible with these reasoning types
    synergy: Dict[str, float] = field(default_factory=dict)  # Strategy IDs and their synergy scores
    confidence_modifier: float = 0.0  # Adjustment to confidence when using this strategy
    complexity: int = 1  # Computational complexity (1-10)


@dataclass
class InferenceStep:
    """A single step in a reasoning process"""
    step_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    strategy_id: str = ""
    description: str = ""
    inputs: Dict[str, Any] = field(default_factory=dict)
    outputs: Dict[str, Any] = field(default_factory=dict)
    confidence: float = 1.0
    supporting_evidence: List[str] = field(default_factory=list)
    contradicting_evidence: List[str] = field(default_factory=list)
    sub_steps: List['InferenceStep'] = field(default_factory=list)
    timestamp: float = field(default_factory=time.time)


@dataclass
class ReasoningQuery:
    """A query for the reasoning engine"""
    query_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    question: Any = None  # The query/question to answer
    context: Dict[str, Any] = field(default_factory=dict)  # Additional context information
    preferred_strategies: List[str] = field(default_factory=list)
    excluded_strategies: List[str] = field(default_factory=list)
    required_reasoning_types: List[ReasoningType] = field(default_factory=list)
    min_confidence: float = 0.0
    max_steps: Optional[int] = None
    max_time: Optional[float] = None
    exploration_factor: float = 0.2  # How much to explore different reasoning paths


@dataclass
class ReasoningResult:
    """Result of a reasoning process"""
    result_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    query_id: str = ""
    conclusions: List[Dict[str, Any]] = field(default_factory=list)
    confidence: float = 0.0
    inference_steps: List[InferenceStep] = field(default_factory=list)
    reasoning_types_used: Set[ReasoningType] = field(default_factory=set)
    strategies_used: List[str] = field(default_factory=list)
    supporting_evidence: Dict[str, List[str]] = field(default_factory=dict)
    contradicting_evidence: Dict[str, List[str]] = field(default_factory=dict)
    processing_time: float = 0.0
    complete: bool = False
    annotations: Dict[str, Any] = field(default_factory=dict)


class ReasoningCore(CognitiveModuleBase):
    """
    Core class for the reasoning module in GrandNexus.

    Provides a flexible framework for different reasoning approaches,
    allowing them to be composed dynamically based on context. Acts as
    a substrate for emergent reasoning patterns rather than enforcing
    fixed reasoning pathways.
    """

    def __init__(self,
                 cognition_manager: Optional[CognitionManager] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 graph_engine: Optional[GraphEngine] = None,
                 llm_engine: Optional[LLMEngine] = None):
        """
        Initialize the Reasoning Core.

        Args:
            cognition_manager: Reference to the CognitionManager
            working_memory: Reference to the WorkingMemory module
            semantic_memory: Reference to the SemanticMemory module
            graph_engine: Reference to the GraphEngine module
            llm_engine: Reference to the LLM interface
        """
        self.cognition_manager = cognition_manager
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.graph_engine = graph_engine
        self.llm_engine = llm_engine

        # Logger setup
        self.logger = logging.getLogger("ReasoningCore")

        # Threading and synchronization
        self.lock = threading.RLock()

        # Reasoning strategies registry
        self.strategies: Dict[str, ReasoningStrategy] = {}

        # Reasoning workspace (shared state for reasoning processes)
        self.workspace: Dict[str, Dict[str, Any]] = {}

        # Active reasoning processes
        self.active_processes: Dict[str, Dict[str, Any]] = {}

        # Initialize basic reasoning strategies
        self._register_basic_strategies()

        self.logger.info("ReasoningCore initialized")

    def _register_basic_strategies(self) -> None:
        """Register built-in basic reasoning strategies"""
        # Simple deductive strategy
        self.register_strategy(ReasoningStrategy(
            strategy_id="simple_deduction",
            reasoning_type=ReasoningType.DEDUCTIVE,
            description="Simple deductive reasoning using modus ponens",
            handler=self._simple_deductive_reasoning,
            required_inputs=["premises", "rules"],
            provided_outputs=["conclusions"],
            compatibility={ReasoningType.DEDUCTIVE, ReasoningType.HYBRID},
            complexity=2
        ))

        # Simple inductive strategy
        self.register_strategy(ReasoningStrategy(
            strategy_id="simple_induction",
            reasoning_type=ReasoningType.INDUCTIVE,
            description="Simple inductive reasoning from examples to general rules",
            handler=self._simple_inductive_reasoning,
            required_inputs=["examples"],
            provided_outputs=["patterns", "generalizations"],
            compatibility={ReasoningType.INDUCTIVE, ReasoningType.HYBRID},
            complexity=3
        ))

        # Abductive strategy
        self.register_strategy(ReasoningStrategy(
            strategy_id="simple_abduction",
            reasoning_type=ReasoningType.ABDUCTIVE,
            description="Simple abductive reasoning to find explanations",
            handler=self._simple_abductive_reasoning,
            required_inputs=["observations"],
            provided_outputs=["explanations"],
            compatibility={ReasoningType.ABDUCTIVE, ReasoningType.HYBRID},
            confidence_modifier=-0.1,  # Abduction is inherently less certain
            complexity=4
        ))

        # Analogical strategy
        self.register_strategy(ReasoningStrategy(
            strategy_id="simple_analogy",
            reasoning_type=ReasoningType.ANALOGICAL,
            description="Simple analogical reasoning between source and target domains",
            handler=self._simple_analogical_reasoning,
            required_inputs=["source_domain", "target_domain"],
            provided_outputs=["mappings", "inferences"],
            compatibility={ReasoningType.ANALOGICAL, ReasoningType.HYBRID, ReasoningType.CREATIVE},
            complexity=5
        ))

        # LLM-assisted reasoning (if LLM engine is available)
        if self.llm_engine:
            self.register_strategy(ReasoningStrategy(
                strategy_id="llm_assisted",
                reasoning_type=ReasoningType.HYBRID,
                description="Use LLM to assist with general reasoning tasks",
                handler=self._llm_assisted_reasoning,
                required_inputs=["question", "context"],
                provided_outputs=["thoughts", "conclusions"],
                compatibility=set(ReasoningType),  # Compatible with all reasoning types
                confidence_modifier=-0.2,  # LLM responses have inherent uncertainty
                complexity=3
            ))

    def register_strategy(self, strategy: ReasoningStrategy) -> bool:
        """
        Register a reasoning strategy.

        Args:
            strategy: The strategy to register

        Returns:
            bool: True if successful, False otherwise
        """
        with self.lock:
            if strategy.strategy_id in self.strategies:
                self.logger.warning(f"Strategy with ID {strategy.strategy_id} already registered")
                return False

            self.strategies[strategy.strategy_id] = strategy
            self.logger.info(f"Registered reasoning strategy {strategy.strategy_id}")
            return True

    def unregister_strategy(self, strategy_id: str) -> bool:
        """
        Unregister a reasoning strategy.

        Args:
            strategy_id: ID of the strategy to unregister

        Returns:
            bool: True if successful, False otherwise
        """
        with self.lock:
            if strategy_id not in self.strategies:
                self.logger.warning(f"Strategy with ID {strategy_id} not found")
                return False

            del self.strategies[strategy_id]
            self.logger.info(f"Unregistered reasoning strategy {strategy_id}")
            return True

    def get_strategy(self, strategy_id: str) -> Optional[ReasoningStrategy]:
        """
        Get a registered reasoning strategy by ID.

        Args:
            strategy_id: ID of the strategy to retrieve

        Returns:
            Optional[ReasoningStrategy]: The strategy if found, None otherwise
        """
        return self.strategies.get(strategy_id)

    def get_strategies_by_type(self, reasoning_type: ReasoningType) -> List[ReasoningStrategy]:
        """
        Get all strategies of a specific reasoning type.

        Args:
            reasoning_type: The type of reasoning

        Returns:
            List[ReasoningStrategy]: List of matching strategies
        """
        return [s for s in self.strategies.values() if s.reasoning_type == reasoning_type]

    def create_reasoning_workspace(self, workspace_id: Optional[str] = None) -> str:
        """
        Create a new reasoning workspace.

        Args:
            workspace_id: Optional ID for the workspace (generated if None)

        Returns:
            str: ID of the created workspace
        """
        with self.lock:
            workspace_id = workspace_id or str(uuid.uuid4())
            if workspace_id in self.workspace:
                self.logger.warning(f"Workspace {workspace_id} already exists, will be overwritten")

            self.workspace[workspace_id] = {
                "created_at": time.time(),
                "data": {},
                "history": []
            }

            return workspace_id

    def get_workspace(self, workspace_id: str) -> Dict[str, Any]:
        """
        Get a reasoning workspace.

        Args:
            workspace_id: ID of the workspace

        Returns:
            Dict[str, Any]: The workspace data
        """
        with self.lock:
            if workspace_id not in self.workspace:
                self.logger.warning(f"Workspace {workspace_id} not found, creating new one")
                workspace_id = self.create_reasoning_workspace(workspace_id)

            return self.workspace[workspace_id]["data"]

    def update_workspace(self, workspace_id: str, updates: Dict[str, Any]) -> None:
        """
        Update a reasoning workspace.

        Args:
            workspace_id: ID of the workspace
            updates: Dictionary of updates to apply
        """
        with self.lock:
            if workspace_id not in self.workspace:
                self.logger.warning(f"Workspace {workspace_id} not found, creating new one")
                workspace_id = self.create_reasoning_workspace(workspace_id)

            # Record history of changes
            history_entry = {
                "timestamp": time.time(),
                "previous_state": self.workspace[workspace_id]["data"].copy(),
                "updates": updates.copy()
            }
            self.workspace[workspace_id]["history"].append(history_entry)

            # Apply updates
            self.workspace[workspace_id]["data"].update(updates)

    @CognitionManager.cognitive_capability(
        capability_type=CognitiveProcessType.REASONING,
        description="General reasoning capability",
        priority=50,
        compatible_modes={CognitiveMode.DELIBERATIVE, CognitiveMode.CREATIVE}
    )
    def reason(self,
              request: CognitiveRequest,
              context: CognitiveContext,
              query: Any,
              **kwargs) -> Dict[str, Any]:
        """
        Main reasoning capability. Processes the query using appropriate
        reasoning strategies based on context.

        Args:
            request: The cognitive request
            context: The cognitive context
            query: The query/question to reason about
            **kwargs: Additional arguments

        Returns:
            Dict[str, Any]: Reasoning results
        """
        start_time = time.time()

        try:
            self.logger.info(f"Processing reasoning request: {query}")

            # Create reasoning query
            reasoning_query = self._create_reasoning_query(query, context)

            # Create workspace for this reasoning process
            workspace_id = self.create_reasoning_workspace()

            # Initialize workspace with query and context
            initial_data = {
                "query": reasoning_query,
                "context_data": context.parameters,
                "working_memory_snapshot": {},  # Would be populated from working memory
                "intermediate_results": []
            }
            self.update_workspace(workspace_id, initial_data)

            # Execute reasoning process
            reasoning_result = self._execute_reasoning_process(reasoning_query, workspace_id)

            # Record processing time
            end_time = time.time()
            reasoning_result.processing_time = end_time - start_time

            # Convert reasoning result to dict for return
            return self._reasoning_result_to_dict(reasoning_result)

        except Exception as e:
            self.logger.error(f"Error in reasoning process: {e}", exc_info=True)

            # Return error information
            return {
                "error": str(e),
                "processing_time": time.time() - start_time,
                "query": str(query)
            }

    def _create_reasoning_query(self, query: Any, context: CognitiveContext) -> ReasoningQuery:
        """
        Create a reasoning query from the input query and context.

        Args:
            query: The input query
            context: The cognitive context

        Returns:
            ReasoningQuery: The created reasoning query
        """
        # Extract reasoning preferences from context
        preferred_strategies = context.parameters.get("preferred_strategies", [])
        excluded_strategies = context.parameters.get("excluded_strategies", [])

        # Parse required reasoning types
        required_types = []
        if "reasoning_types" in context.parameters:
            for type_name in context.parameters["reasoning_types"]:
                try:
                    if isinstance(type_name, str):
                        required_types.append(ReasoningType[type_name.upper()])
                    else:
                        required_types.append(type_name)
                except (KeyError, ValueError):
                    self.logger.warning(f"Unknown reasoning type: {type_name}")

        # Extract other parameters
        min_confidence = context.parameters.get("min_confidence", 0.0)
        max_steps = context.parameters.get("max_steps", None)
        max_time = context.parameters.get("max_time", None)
        exploration_factor = context.parameters.get("exploration_factor", 0.2)

        # Prepare context for the reasoning query
        query_context = {}

        # Add working memory state if available
        if self.working_memory:
            # In a real implementation, this would extract relevant information
            # from working memory based on the query
            query_context["working_memory"] = {"available": True}

        # Add semantic memory state if available
        if self.semantic_memory:
            query_context["semantic_memory"] = {"available": True}

        # Create the reasoning query
        return ReasoningQuery(
            question=query,
            context=query_context,
            preferred_strategies=preferred_strategies,
            excluded_strategies=excluded_strategies,
            required_reasoning_types=required_types,
            min_confidence=min_confidence,
            max_steps=max_steps,
            max_time=max_time,
            exploration_factor=exploration_factor
        )

    def _execute_reasoning_process(self, query: ReasoningQuery, workspace_id: str) -> ReasoningResult:
        """
        Execute a reasoning process for the given query.

        This is the core reasoning loop that selects and applies appropriate
        strategies based on the query and available information.

        Args:
            query: The reasoning query
            workspace_id: ID of the workspace to use

        Returns:
            ReasoningResult: Results of the reasoning process
        """
        # Initialize result
        result = ReasoningResult(
            query_id=query.query_id
        )

        # Fetch workspace
        workspace = self.get_workspace(workspace_id)

        # Track start time for time limit
        start_time = time.time()

        # Initialize reasoning process state
        current_step = 0

        # Flag to indicate if reasoning is complete
        reasoning_complete = False

        # Main reasoning loop
        while not reasoning_complete:
            # Check time limit
            if query.max_time and (time.time() - start_time) > query.max_time:
                self.logger.info(f"Reasoning time limit reached for query {query.query_id}")
                break

            # Check step limit
            if query.max_steps and current_step >= query.max_steps:
                self.logger.info(f"Reasoning step limit reached for query {query.query_id}")
                break

            # Select next reasoning strategy
            strategy_id = self._select_next_strategy(query, result, workspace)

            if not strategy_id:
                self.logger.info(f"No suitable strategy found for query {query.query_id}")
                break

            strategy = self.strategies[strategy_id]

            # Apply the strategy
            self.logger.debug(f"Applying strategy {strategy_id}")

            try:
                inference_step = InferenceStep(
                    strategy_id=strategy_id,
                    description=f"Applying {strategy.description}"
                )

                # Prepare inputs for the strategy
                strategy_inputs = self._prepare_strategy_inputs(strategy, workspace, query)
                inference_step.inputs = strategy_inputs

                # Execute the strategy
                strategy_result = strategy.handler(
                    query=query,
                    workspace=workspace,
                    inputs=strategy_inputs,
                    current_result=result
                )

                # Update inference step with results
                inference_step.outputs = strategy_result.get("outputs", {})
                inference_step.confidence = strategy_result.get("confidence", 0.5)

                if "sub_steps" in strategy_result:
                    inference_step.sub_steps = strategy_result["sub_steps"]

                if "supporting_evidence" in strategy_result:
                    inference_step.supporting_evidence = strategy_result["supporting_evidence"]

                if "contradicting_evidence" in strategy_result:
                    inference_step.contradicting_evidence = strategy_result["contradicting_evidence"]

                # Add step to result
                result.inference_steps.append(inference_step)

                # Update reasoning types used
                result.reasoning_types_used.add(strategy.reasoning_type)

                # Add strategy to used list
                result.strategies_used.append(strategy_id)

                # Update workspace with new information
                if "workspace_updates" in strategy_result:
                    self.update_workspace(workspace_id, strategy_result["workspace_updates"])

                # Check if new conclusions were generated
                if "conclusions" in strategy_result:
                    for conclusion in strategy_result["conclusions"]:
                        result.conclusions.append(conclusion)

                # Check if reasoning is complete
                if strategy_result.get("reasoning_complete", False):
                    reasoning_complete = True

                # Update evidence
                if "supporting_evidence" in strategy_result:
                    for conclusion_id, evidence in strategy_result.get("supporting_evidence", {}).items():
                        if conclusion_id not in result.supporting_evidence:
                            result.supporting_evidence[conclusion_id] = []
                        result.supporting_evidence[conclusion_id].extend(evidence)

                if "contradicting_evidence" in strategy_result:
                    for conclusion_id, evidence in strategy_result.get("contradicting_evidence", {}).items():
                        if conclusion_id not in result.contradicting_evidence:
                            result.contradicting_evidence[conclusion_id] = []
                        result.contradicting_evidence[conclusion_id].extend(evidence)

            except Exception as e:
                self.logger.error(f"Error applying strategy {strategy_id}: {e}", exc_info=True)

                # Create an error inference step
                error_step = InferenceStep(
                    strategy_id=strategy_id,
                    description=f"Error applying {strategy.description}: {str(e)}",
                    confidence=0.0
                )
                result.inference_steps.append(error_step)

            # Increment step counter
            current_step += 1

        # Calculate overall confidence
        if result.conclusions:
            # Average confidence of all conclusions, weighted by their individual confidence
            total_confidence = sum(c.get("confidence", 0.5) for c in result.conclusions)
            result.confidence = total_confidence / len(result.conclusions)
        else:
            result.confidence = 0.0

        # Mark as complete
        result.complete = reasoning_complete

        return result

    def _select_next_strategy(self,
                           query: ReasoningQuery,
                           current_result: ReasoningResult,
                           workspace: Dict[str, Any]) -> Optional[str]:
        """
        Select the next reasoning strategy to apply.

        This is a key function that determines the reasoning flow.
        It considers:
        - Query preferences
        - Current state of reasoning
        - Availability of required inputs for strategies
        - Compatibility between strategies
        - Exploration vs. exploitation tradeoff

        Args:
            query: The reasoning query
            current_result: Current reasoning result
            workspace: The reasoning workspace

        Returns:
            Optional[str]: ID of the selected strategy, or None if no suitable strategy
        """
        candidate_strategies = []

        # Consider preferred strategies first
        if query.preferred_strategies:
            for strategy_id in query.preferred_strategies:
                if strategy_id in self.strategies and strategy_id not in query.excluded_strategies:
                    candidate_strategies.append(strategy_id)

        # If no preferred strategies available, consider all strategies
        if not candidate_strategies:
            # Filter by required reasoning types
            if query.required_reasoning_types:
                for strategy in self.strategies.values():
                    if (strategy.reasoning_type in query.required_reasoning_types
                        and strategy.strategy_id not in query.excluded_strategies
                        and strategy.strategy_id not in current_result.strategies_used):
                        candidate_strategies.append(strategy.strategy_id)
            else:
                # Consider all strategies not excluded and not already used
                for strategy in self.strategies.values():
                    if (strategy.strategy_id not in query.excluded_strategies
                        and strategy.strategy_id not in current_result.strategies_used):
                        candidate_strategies.append(strategy.strategy_id)

        # If still no candidates, consider reusing strategies (except last one)
        if not candidate_strategies and current_result.strategies_used:
            for strategy_id in current_result.strategies_used[:-1]:
                if strategy_id not in query.excluded_strategies:
                    candidate_strategies.append(strategy_id)

        # If still no candidates, return None
        if not candidate_strategies:
            return None

        # Filter candidates by input availability
        valid_candidates = []
        for strategy_id in candidate_strategies:
            strategy = self.strategies[strategy_id]

            # Check if required inputs are available in workspace
            missing_inputs = []
            for input_name in strategy.required_inputs:
                if input_name not in workspace:
                    missing_inputs.append(input_name)

            # Special case: if missing inputs are standard (e.g., "query", "context"),
            # and corresponding data exists, consider the strategy valid
            if all(input_name in ["query", "context", "examples", "premises", "observations"]
                  for input_name in missing_inputs):
                valid_candidates.append(strategy_id)
            elif not missing_inputs:
                valid_candidates.append(strategy_id)

        # If no valid candidates, try to infer missing inputs
        if not valid_candidates:
            self.logger.debug(f"No strategies with all required inputs available, trying to infer missing inputs")

            # Try each candidate and see if we can infer its missing inputs
            for strategy_id in candidate_strategies:
                strategy = self.strategies[strategy_id]

                # Check which inputs are missing
                missing_inputs = []
                for input_name in strategy.required_inputs:
                    if input_name not in workspace:
                        missing_inputs.append(input_name)

                # Try to infer missing inputs
                if self._can_infer_inputs(strategy, missing_inputs, workspace, query):
                    valid_candidates.append(strategy_id)

        # If still no valid candidates, return None
        if not valid_candidates:
            return None

        # Apply exploration vs. exploitation
        if random.random() < query.exploration_factor:
            # Exploration: Choose a random strategy from valid candidates
            return random.choice(valid_candidates)
        else:
            # Exploitation: Choose based on confidence and synergy
            strategy_scores = {}

            for strategy_id in valid_candidates:
                strategy = self.strategies[strategy_id]

                # Base score is 1.0
                score = 1.0

                # Adjust based on compatibility with reasoning types already used
                for used_type in current_result.reasoning_types_used:
                    if used_type in strategy.compatibility:
                        score += 0.2

                # Adjust based on synergy with strategies already used
                for used_strategy in current_result.strategies_used:
                    if used_strategy in strategy.synergy:
                        score += strategy.synergy[used_strategy]

                # Store score
                strategy_scores[strategy_id] = score

            # Select strategy with highest score
            if strategy_scores:
                return max(strategy_scores.items(), key=lambda x: x[1])[0]
            else:
                # Fallback to random selection
                return random.choice(valid_candidates)

    def _can_infer_inputs(self,
                        strategy: ReasoningStrategy,
                        missing_inputs: List[str],
                        workspace: Dict[str, Any],
                        query: ReasoningQuery) -> bool:
        """
        Determine if missing inputs for a strategy can be inferred.

        Args:
            strategy: The strategy being considered
            missing_inputs: List of missing input names
            workspace: The reasoning workspace
            query: The reasoning query

        Returns:
            bool: True if inputs can be inferred, False otherwise
        """
        # This is a simplified implementation
        # In a real system, this would use more sophisticated inference methods

        if "query" in missing_inputs and "question" in workspace:
            # Can infer query from question
            return True

        if "premises" in missing_inputs and "context" in workspace:
            # Might be able to extract premises from context
            return True

        if "examples" in missing_inputs and ("observations" in workspace or "context" in workspace):
            # Might be able to extract examples from observations or context
            return True

        if "observations" in missing_inputs and ("examples" in workspace or "context" in workspace):
            # Might be able to interpret examples or context as observations
            return True

        # Default: cannot infer
        return False

    def _prepare_strategy_inputs(self,
                               strategy: ReasoningStrategy,
                               workspace: Dict[str, Any],
                               query: ReasoningQuery) -> Dict[str, Any]:
        """
        Prepare inputs for a reasoning strategy.

        This function:
        1. Extracts inputs already present in the workspace
        2. Attempts to infer missing inputs
        3. Creates default values for required inputs that cannot be found

        Args:
            strategy: The strategy to prepare inputs for
            workspace: The reasoning workspace
            query: The reasoning query

        Returns:
            Dict[str, Any]: Prepared inputs for the strategy
        """
        inputs = {}

        # First, extract inputs directly available in workspace
        for input_name in strategy.required_inputs:
            if input_name in workspace:
                inputs[input_name] = workspace[input_name]

        # Handle special cases for common inputs
        if "query" in strategy.required_inputs and "query" not in inputs:
            inputs["query"] = query.question

        if "context" in strategy.required_inputs and "context" not in inputs:
            inputs["context"] = query.context

        # Attempt to infer missing inputs
        missing_inputs = [name for name in strategy.required_inputs if name not in inputs]

        for input_name in missing_inputs:
            # Try to infer the input based on what's available
            inferred_value = self._infer_input(input_name, strategy, workspace, query)

            if inferred_value is not None:
                inputs[input_name] = inferred_value
            else:
                # Create an empty default value
                if input_name in ["premises", "rules", "examples", "observations"]:
                    inputs[input_name] = []
                elif input_name in ["source_domain", "target_domain"]:
                    inputs[input_name] = {}
                else:
                    inputs[input_name] = None

        return inputs

    def _infer_input(self,
                    input_name: str,
                    strategy: ReasoningStrategy,
                    workspace: Dict[str, Any],
                    query: ReasoningQuery) -> Optional[Any]:
        """
        Attempt to infer a missing input from available information.

        Args:
            input_name: Name of the input to infer
            strategy: The strategy requesting the input
            workspace: The reasoning workspace
            query: The reasoning query

        Returns:
            Optional[Any]: The inferred input value, or None if cannot be inferred
        """
        # This is a simplified implementation
        # In a real system, this would use more sophisticated inference methods

        # Inference rules for different input types
        if input_name == "premises" and "context" in workspace:
            # Extract statements from context that could be premises
            context = workspace["context"]
            if isinstance(context, dict) and "statements" in context:
                return context["statements"]
            return []

        if input_name == "rules" and "context" in workspace:
            # Extract rules from context
            context = workspace["context"]
            if isinstance(context, dict) and "rules" in context:
                return context["rules"]
            return []

        if input_name == "examples" and "observations" in workspace:
            # Interpret observations as examples
            return workspace["observations"]

        if input_name == "observations" and "examples" in workspace:
            # Interpret examples as observations
            return workspace["examples"]

        if input_name == "source_domain" and "context" in workspace:
            # Try to extract a source domain from context
            context = workspace["context"]
            if isinstance(context, dict) and "source_domain" in context:
                return context["source_domain"]
            return {}

        if input_name == "target_domain" and "context" in workspace:
            # Try to extract a target domain from context
            context = workspace["context"]
            if isinstance(context, dict) and "target_domain" in context:
                return context["target_domain"]
            return {}

        # Use LLM to infer input if available
        if self.llm_engine and input_name in ["premises", "rules", "examples", "observations"]:
            try:
                prompt = f"""
                Extract {input_name} from the following information:

                Question: {query.question}

                Extract only the {input_name} without additional commentary.
                Format your response as a list of items.
                """

                llm_response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 200,
                    "temperature": 0.3
                })

                # Parse the response (simplified)
                if hasattr(llm_response, 'text'):
                    response_text = llm_response.text
                else:
                    response_text = str(llm_response)

                # Basic extraction of list items
                items = []
                for line in response_text.split('\n'):
                    line = line.strip()
                    if line.startswith('- ') or line.startswith('* '):
                        items.append(line[2:].strip())
                    elif line.startswith('\d+\. '):
                        items.append(line[line.find(' ')+1:].strip())

                if items:
                    return items

            except Exception as e:
                self.logger.warning(f"Error using LLM to infer {input_name}: {e}")

        # Cannot infer
        return None

    def _reasoning_result_to_dict(self, result: ReasoningResult) -> Dict[str, Any]:
        """
        Convert a ReasoningResult to a dictionary.

        Args:
            result: The reasoning result to convert

        Returns:
            Dict[str, Any]: Dictionary representation of the result
        """
        return {
            "result_id": result.result_id,
            "query_id": result.query_id,
            "conclusions": result.conclusions,
            "confidence": result.confidence,
            "reasoning_types_used": [rt.name for rt in result.reasoning_types_used],
            "strategies_used": result.strategies_used,
            "inference_steps": [self._inference_step_to_dict(step) for step in result.inference_steps],
            "supporting_evidence": result.supporting_evidence,
            "contradicting_evidence": result.contradicting_evidence,
            "processing_time": result.processing_time,
            "complete": result.complete,
            "annotations": result.annotations
        }

    def _inference_step_to_dict(self, step: InferenceStep) -> Dict[str, Any]:
        """
        Convert an InferenceStep to a dictionary.

        Args:
            step: The inference step to convert

        Returns:
            Dict[str, Any]: Dictionary representation of the step
        """
        return {
            "step_id": step.step_id,
            "strategy_id": step.strategy_id,
            "description": step.description,
            "inputs": step.inputs,
            "outputs": step.outputs,
            "confidence": step.confidence,
            "supporting_evidence": step.supporting_evidence,
            "contradicting_evidence": step.contradicting_evidence,
            "sub_steps": [self._inference_step_to_dict(sub_step) for sub_step in step.sub_steps],
            "timestamp": step.timestamp
        }

    #------------------------------------------------------------------
    # Basic reasoning strategies
    #------------------------------------------------------------------

    def _simple_deductive_reasoning(self,
                                 query: ReasoningQuery,
                                 workspace: Dict[str, Any],
                                 inputs: Dict[str, Any],
                                 current_result: ReasoningResult) -> Dict[str, Any]:
        """
        Simple deductive reasoning strategy.

        Applies basic modus ponens: If A implies B, and A is true, then B is true.

        Args:
            query: The reasoning query
            workspace: The reasoning workspace
            inputs: Strategy inputs
            current_result: Current reasoning result

        Returns:
            Dict[str, Any]: Strategy results
        """
        premises = inputs.get("premises", [])
        rules = inputs.get("rules", [])

        # Check if we have the necessary inputs
        if not premises or not rules:
            return {
                "outputs": {
                    "message": "Insufficient inputs for deductive reasoning",
                    "conclusions": []
                },
                "confidence": 0.0,
                "workspace_updates": {},
                "conclusions": []
            }

        # Simple deductive reasoning (modus ponens)
        conclusions = []
        supporting_evidence = {}
        sub_steps = []

        # For each rule of the form "if A then B"
        for rule in rules:
            # Extract antecedent (if part) and consequent (then part)
            if "if" in rule and "then" in rule:
                antecedent = rule["if"]
                consequent = rule["then"]

                # Check if antecedent is in premises
                if antecedent in premises:
                    # Apply modus ponens
                    conclusion_id = str(uuid.uuid4())
                    conclusion = {
                        "id": conclusion_id,
                        "statement": consequent,
                        "confidence": rule.get("confidence", 0.95),
                        "derived_from": {
                            "rule": rule,
                            "premise": antecedent
                        }
                    }
                    conclusions.append(conclusion)

                    # Record supporting evidence
                    supporting_evidence[conclusion_id] = [
                        f"Premise: {antecedent}",
                        f"Rule: If {antecedent} then {consequent}"
                    ]

                    # Create sub-step
                    sub_step = InferenceStep(
                        description=f"Applied modus ponens with rule: If {antecedent} then {consequent}",
                        inputs={"premise": antecedent, "rule": rule},
                        outputs={"conclusion": consequent},
                        confidence=rule.get("confidence", 0.95)
                    )
                    sub_steps.append(sub_step)

        # Transitive reasoning: If A implies B and B implies C, then A implies C
        # This is a simple example of chaining multiple deductions
        new_conclusions = []
        for conclusion in conclusions:
            statement = conclusion["statement"]

            # Check if this statement is an antecedent in any rule
            for rule in rules:
                if "if" in rule and "then" in rule and rule["if"] == statement:
                    # Apply transitive reasoning
                    trans_conclusion_id = str(uuid.uuid4())
                    trans_conclusion = {
                        "id": trans_conclusion_id,
                        "statement": rule["then"],
                        "confidence": min(conclusion["confidence"], rule.get("confidence", 0.95)),
                        "derived_from": {
                            "rule": rule,
                            "prior_conclusion": conclusion
                        }
                    }
                    new_conclusions.append(trans_conclusion)

                    # Record supporting evidence
                    supporting_evidence[trans_conclusion_id] = [
                        f"Prior conclusion: {statement}",
                        f"Rule: If {rule['if']} then {rule['then']}"
                    ]

                    # Create sub-step
                    sub_step = InferenceStep(
                        description=f"Applied transitive reasoning with rule: If {rule['if']} then {rule['then']}",
                        inputs={"prior_conclusion": statement, "rule": rule},
                        outputs={"conclusion": rule["then"]},
                        confidence=min(conclusion["confidence"], rule.get("confidence", 0.95))
                    )
                    sub_steps.append(sub_step)

        # Add transitive conclusions
        conclusions.extend(new_conclusions)

        # Calculate overall confidence
        confidence = 0.95 if conclusions else 0.0

        # Check if reasoning is complete (if we derived any conclusions)
        reasoning_complete = len(conclusions) > 0

        return {
            "outputs": {
                "conclusions": conclusions
            },
            "workspace_updates": {
                "deductive_conclusions": conclusions
            },
            "confidence": confidence,
            "conclusions": conclusions,
            "reasoning_complete": reasoning_complete,
            "supporting_evidence": supporting_evidence,
            "sub_steps": sub_steps
        }

    def _simple_inductive_reasoning(self,
                                  query: ReasoningQuery,
                                  workspace: Dict[str, Any],
                                  inputs: Dict[str, Any],
                                  current_result: ReasoningResult) -> Dict[str, Any]:
        """
        Simple inductive reasoning strategy.

        Looks for patterns in examples and generates generalizations.

        Args:
            query: The reasoning query
            workspace: The reasoning workspace
            inputs: Strategy inputs
            current_result: Current reasoning result

        Returns:
            Dict[str, Any]: Strategy results
        """
        examples = inputs.get("examples", [])

        # Check if we have the necessary inputs
        if not examples:
            return {
                "outputs": {
                    "message": "Insufficient examples for inductive reasoning",
                    "patterns": [],
                    "generalizations": []
                },
                "confidence": 0.0,
                "workspace_updates": {},
                "conclusions": []
            }

        # Simple pattern detection in examples
        patterns = []
        generalizations = []
        supporting_evidence = {}
        sub_steps = []

        # Look for common attributes across examples
        if all(isinstance(example, dict) for example in examples):
            # For dictionaries, look for common keys and values
            common_keys = set(examples[0].keys())
            for example in examples[1:]:
                common_keys &= set(example.keys())

            # For each common key, check if values are consistent
            for key in common_keys:
                values = [example[key] for example in examples]

                # If all values are the same, this is a pattern
                if all(value == values[0] for value in values):
                    pattern_id = str(uuid.uuid4())
                    pattern = {
                        "id": pattern_id,
                        "type": "common_attribute",
                        "attribute": key,
                        "value": values[0],
                        "confidence": 0.7  # Lower confidence for inductive reasoning
                    }
                    patterns.append(pattern)

                    # Create generalization based on this pattern
                    generalization_id = str(uuid.uuid4())
                    generalization = {
                        "id": generalization_id,
                        "statement": f"All examples have {key} = {values[0]}",
                        "confidence": 0.7,
                        "derived_from": {
                            "pattern": pattern
                        }
                    }
                    generalizations.append(generalization)

                    # Record supporting evidence
                    supporting_evidence[generalization_id] = [
                        f"Example {i}: {example[key]}" for i, example in enumerate(examples)
                    ]

                    # Create sub-step
                    sub_step = InferenceStep(
                        description=f"Identified common attribute pattern: {key} = {values[0]}",
                        inputs={"examples": examples},
                        outputs={"pattern": pattern, "generalization": generalization},
                        confidence=0.7
                    )
                    sub_steps.append(sub_step)

                # If values follow a pattern (e.g., always increasing)
                elif all(isinstance(value, (int, float)) for value in values):
                    # Check for monotonic increase
                    if all(values[i] < values[i+1] for i in range(len(values)-1)):
                        pattern_id = str(uuid.uuid4())
                        pattern = {
                            "id": pattern_id,
                            "type": "increasing_attribute",
                            "attribute": key,
                            "values": values,
                            "confidence": 0.6  # Even lower confidence for trend detection
                        }
                        patterns.append(pattern)

                        # Create generalization based on this pattern
                        generalization_id = str(uuid.uuid4())
                        generalization = {
                            "id": generalization_id,
                            "statement": f"The value of {key} increases across examples",
                            "confidence": 0.6,
                            "derived_from": {
                                "pattern": pattern
                            }
                        }
                        generalizations.append(generalization)

                        # Record supporting evidence
                        supporting_evidence[generalization_id] = [
                            f"Example {i}: {key} = {example[key]}" for i, example in enumerate(examples)
                        ]

                        # Create sub-step
                        sub_step = InferenceStep(
                            description=f"Identified increasing trend in attribute: {key}",
                            inputs={"examples": examples},
                            outputs={"pattern": pattern, "generalization": generalization},
                            confidence=0.6
                        )
                        sub_steps.append(sub_step)

        # Use LLM for more sophisticated pattern detection if available
        elif self.llm_engine:
            try:
                # Prepare examples in a format suitable for LLM
                examples_text = "\n".join([f"Example {i}: {example}" for i, example in enumerate(examples)])

                prompt = f"""
                Analyze the following examples and identify patterns or generalizations:

                {examples_text}

                Provide your analysis in this format:

                Patterns:
                1. [pattern description]
                2. [pattern description]

                Generalizations:
                1. [generalization statement]
                2. [generalization statement]

                Please be specific and concrete in your analysis.
                """

                llm_response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 300,
                    "temperature": 0.3
                })

                # Parse the response (simplified)
                if hasattr(llm_response, 'text'):
                    response_text = llm_response.text
                else:
                    response_text = str(llm_response)

                # Extract patterns and generalizations
                llm_patterns_section = re.search(r'Patterns:(.*?)(?:Generalizations:|$)', response_text, re.DOTALL)
                llm_generalizations_section = re.search(r'Generalizations:(.*?)$', response_text, re.DOTALL)

                if llm_patterns_section:
                    llm_patterns_text = llm_patterns_section.group(1).strip()
                    pattern_lines = re.findall(r'\d+\.\s+(.*?)(?:\n|$)', llm_patterns_text)

                    for i, pattern_text in enumerate(pattern_lines):
                        pattern_id = str(uuid.uuid4())
                        pattern = {
                            "id": pattern_id,
                            "type": "llm_detected",
                            "description": pattern_text,
                            "confidence": 0.65  # Lower confidence for LLM-detected patterns
                        }
                        patterns.append(pattern)

                        # Create sub-step
                        sub_step = InferenceStep(
                            description=f"LLM detected pattern: {pattern_text}",
                            inputs={"examples": examples},
                            outputs={"pattern": pattern},
                            confidence=0.65
                        )
                        sub_steps.append(sub_step)

                if llm_generalizations_section:
                    llm_generalizations_text = llm_generalizations_section.group(1).strip()
                    generalization_lines = re.findall(r'\d+\.\s+(.*?)(?:\n|$)', llm_generalizations_text)

                    for i, generalization_text in enumerate(generalization_lines):
                        generalization_id = str(uuid.uuid4())
                        generalization = {
                            "id": generalization_id,
                            "statement": generalization_text,
                            "confidence": 0.6,
                            "derived_from": {
                                "llm_analysis": True
                            }
                        }
                        generalizations.append(generalization)

                        # Record supporting evidence
                        supporting_evidence[generalization_id] = [
                            f"Example {i}: {example}" for i, example in enumerate(examples)
                        ]

                        # Create sub-step
                        sub_step = InferenceStep(
                            description=f"LLM proposed generalization: {generalization_text}",
                            inputs={"examples": examples},
                            outputs={"generalization": generalization},
                            confidence=0.6
                        )
                        sub_steps.append(sub_step)

            except Exception as e:
                self.logger.warning(f"Error using LLM for pattern detection: {e}")

        # Calculate overall confidence
        confidence = 0.7 if generalizations else 0.0

        # Check if reasoning is complete (if we derived any generalizations)
        reasoning_complete = len(generalizations) > 0

        # Convert generalizations to conclusions
        conclusions = []
        for generalization in generalizations:
            conclusion_id = str(uuid.uuid4())
            conclusion = {
                "id": conclusion_id,
                "statement": generalization["statement"],
                "confidence": generalization["confidence"],
                "derived_from": generalization.get("derived_from", {})
            }
            conclusions.append(conclusion)

            # Transfer supporting evidence
            if generalization["id"] in supporting_evidence:
                supporting_evidence[conclusion_id] = supporting_evidence[generalization["id"]]

        return {
            "outputs": {
                "patterns": patterns,
                "generalizations": generalizations
            },
            "workspace_updates": {
                "inductive_patterns": patterns,
                "inductive_generalizations": generalizations
            },
            "confidence": confidence,
            "conclusions": conclusions,
            "reasoning_complete": reasoning_complete,
            "supporting_evidence": supporting_evidence,
            "sub_steps": sub_steps
        }

    def _simple_abductive_reasoning(self,
                                  query: ReasoningQuery,
                                  workspace: Dict[str, Any],
                                  inputs: Dict[str, Any],
                                  current_result: ReasoningResult) -> Dict[str, Any]:
        """
        Simple abductive reasoning strategy.

        Generates potential explanations for observations.

        Args:
            query: The reasoning query
            workspace: The reasoning workspace
            inputs: Strategy inputs
            current_result: Current reasoning result

        Returns:
            Dict[str, Any]: Strategy results
        """
        observations = inputs.get("observations", [])

        # Check if we have the necessary inputs
        if not observations:
            return {
                "outputs": {
                    "message": "Insufficient observations for abductive reasoning",
                    "explanations": []
                },
                "confidence": 0.0,
                "workspace_updates": {},
                "conclusions": []
            }

        # Simple abductive reasoning (inference to the best explanation)
        explanations = []
        supporting_evidence = {}
        contradicting_evidence = {}
        sub_steps = []

        # If LLM is available, use it for generating explanations
        if self.llm_engine:
            try:
                # Prepare observations in a format suitable for LLM
                observations_text = "\n".join([f"Observation {i}: {obs}" for i, obs in enumerate(observations)])

                prompt = f"""
                Given the following observations:

                {observations_text}

                Generate 3-5 possible explanations that could account for these observations.
                For each explanation:
                1. Provide a clear statement of the explanation
                2. Explain how it accounts for the observations
                3. Rate your confidence in this explanation (0-100%)
                4. Identify any observations that support or contradict this explanation

                Format your response as:

                Explanation 1: [explanation statement]
                - Accounts for observations by: [explanation]
                - Confidence: [percentage]
                - Supporting evidence: [list of supporting observations]
                - Contradicting evidence: [list of contradicting observations]

                Explanation 2: ...
                """

                llm_response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 500,
                    "temperature": 0.7  # Higher temperature for creative explanations
                })

                # Parse the response (simplified)
                if hasattr(llm_response, 'text'):
                    response_text = llm_response.text
                else:
                    response_text = str(llm_response)

                # Extract explanations
                explanation_sections = re.findall(r'Explanation \d+: (.*?)(?=Explanation \d+:|$)', response_text, re.DOTALL)

                for section in explanation_sections:
                    # Extract explanation statement
                    statement = section.split('\n')[0].strip()

                    # Extract confidence
                    confidence_match = re.search(r'Confidence:\s*(\d+)', section)
                    confidence = 0.5  # Default
                    if confidence_match:
                        confidence = int(confidence_match.group(1)) / 100  # Convert percentage to 0-1

                    # Extract supporting evidence
                    supporting_match = re.search(r'Supporting evidence:(.*?)(?=Contradicting evidence:|$)', section, re.DOTALL)
                    supporting = []
                    if supporting_match:
                        supporting_text = supporting_match.group(1)
                        supporting = [item.strip() for item in re.findall(r'- (.*?)(?=\n|$)', supporting_text)]

                    # Extract contradicting evidence
                    contradicting_match = re.search(r'Contradicting evidence:(.*?)$', section, re.DOTALL)
                    contradicting = []
                    if contradicting_match:
                        contradicting_text = contradicting_match.group(1)
                        contradicting = [item.strip() for item in re.findall(r'- (.*?)(?=\n|$)', contradicting_text)]

                    # Create explanation
                    explanation_id = str(uuid.uuid4())
                    explanation = {
                        "id": explanation_id,
                        "statement": statement,
                        "confidence": confidence,
                        "derived_from": {
                            "observations": observations
                        }
                    }
                    explanations.append(explanation)

                    # Record evidence
                    if supporting:
                        supporting_evidence[explanation_id] = supporting

                    if contradicting:
                        contradicting_evidence[explanation_id] = contradicting

                    # Create sub-step
                    sub_step = InferenceStep(
                        description=f"Generated explanation: {statement}",
                        inputs={"observations": observations},
                        outputs={"explanation": explanation},
                        confidence=confidence,
                        supporting_evidence=supporting,
                        contradicting_evidence=contradicting
                    )
                    sub_steps.append(sub_step)

            except Exception as e:
                self.logger.warning(f"Error using LLM for abductive reasoning: {e}")

        # Fallback: Generate simple explanations based on observations
        if not explanations:
            for obs in observations:
                explanation_id = str(uuid.uuid4())
                explanation = {
                    "id": explanation_id,
                    "statement": f"A possible explanation for '{obs}' is that it is a common occurrence",
                    "confidence": 0.3,
                    "derived_from": {
                        "observation": obs
                    }
                }
                explanations.append(explanation)

                # Record supporting evidence
                supporting_evidence[explanation_id] = [obs]

                # Create sub-step
                sub_step = InferenceStep(
                    description=f"Generated simple explanation for observation: {obs}",
                    inputs={"observation": obs},
                    outputs={"explanation": explanation},
                    confidence=0.3
                )
                sub_steps.append(sub_step)

        # Rank explanations by confidence
        explanations.sort(key=lambda x: x["confidence"], reverse=True)

        # Calculate overall confidence (based on best explanation)
        confidence = explanations[0]["confidence"] if explanations else 0.0

        # Check if reasoning is complete (if we derived any explanations)
        reasoning_complete = len(explanations) > 0

        # Convert explanations to conclusions
        conclusions = []
        for explanation in explanations:
            conclusion_id = str(uuid.uuid4())
            conclusion = {
                "id": conclusion_id,
                "statement": explanation["statement"],
                "confidence": explanation["confidence"],
                "derived_from": explanation.get("derived_from", {})
            }
            conclusions.append(conclusion)

            # Transfer supporting evidence
            if explanation["id"] in supporting_evidence:
                supporting_evidence[conclusion_id] = supporting_evidence[explanation["id"]]

            # Transfer contradicting evidence
            if explanation["id"] in contradicting_evidence:
                contradicting_evidence[conclusion_id] = contradicting_evidence[explanation["id"]]

        return {
            "outputs": {
                "explanations": explanations
            },
            "workspace_updates": {
                "abductive_explanations": explanations
            },
            "confidence": confidence,
            "conclusions": conclusions,
            "reasoning_complete": reasoning_complete,
            "supporting_evidence": supporting_evidence,
            "contradicting_evidence": contradicting_evidence,
            "sub_steps": sub_steps
        }

    def _simple_analogical_reasoning(self,
                                   query: ReasoningQuery,
                                   workspace: Dict[str, Any],
                                   inputs: Dict[str, Any],
                                   current_result: ReasoningResult) -> Dict[str, Any]:
        """
        Simple analogical reasoning strategy.

        Maps relationships from a source domain to a target domain.

        Args:
            query: The reasoning query
            workspace: The reasoning workspace
            inputs: Strategy inputs
            current_result: Current reasoning result

        Returns:
            Dict[str, Any]: Strategy results
        """
        source_domain = inputs.get("source_domain", {})
        target_domain = inputs.get("target_domain", {})

        # Check if we have the necessary inputs
        if not source_domain or not target_domain:
            return {
                "outputs": {
                    "message": "Insufficient domain information for analogical reasoning",
                    "mappings": [],
                    "inferences": []
                },
                "confidence": 0.0,
                "workspace_updates": {},
                "conclusions": []
            }

        # Simple analogical reasoning
        mappings = []
        inferences = []
        supporting_evidence = {}
        sub_steps = []

        # Extract entities from both domains
        source_entities = source_domain.get("entities", [])
        target_entities = target_domain.get("entities", [])

        # Extract relationships from source domain
        source_relationships = source_domain.get("relationships", [])

        # Map entities based on similarity
        entity_mappings = {}

        # If we have attributes for entities, use them for mapping
            if all("attributes" in entity for entity in source_entities) and all("attributes" in entity for entity in target_entities):
                # For each source entity
                for source_entity in source_entities:
                    # Find most similar target entity based on attributes
                    best_match = None
                    best_score = 0.0

                    for target_entity in target_entities:
                        # Calculate attribute similarity
                        score = self._calculate_attribute_similarity(
                            source_entity["attributes"],
                            target_entity["attributes"]
                        )

                        if score > best_score:
                            best_score = score
                            best_match = target_entity

                    # If we found a reasonable match
                    if best_match and best_score > 0.3:
                        mapping_id = str(uuid.uuid4())
                        mapping = {
                            "id": mapping_id,
                            "source_entity": source_entity["name"],
                            "target_entity": best_match["name"],
                            "confidence": best_score,
                            "mapped_attributes": self._get_shared_attributes(source_entity["attributes"], best_match["attributes"])
                        }
                        mappings.append(mapping)

                        # Record entity mapping
                        entity_mappings[source_entity["name"]] = best_match["name"]

                        # Create sub-step
                        sub_step = InferenceStep(
                            description=f"Mapped {source_entity['name']} to {best_match['name']} (similarity: {best_score:.2f})",
                            inputs={"source_entity": source_entity, "target_entity": best_match},
                            outputs={"mapping": mapping},
                            confidence=best_score
                        )
                        sub_steps.append(sub_step)

            # If no attribute-based mapping was possible, use simple name-based mapping
            if not mappings and len(source_entities) == len(target_entities):
                for i, source_entity in enumerate(source_entities):
                    target_entity = target_entities[i]

                    mapping_id = str(uuid.uuid4())
                    mapping = {
                        "id": mapping_id,
                        "source_entity": source_entity["name"],
                        "target_entity": target_entity["name"],
                        "confidence": 0.3,  # Lower confidence for simple position-based mapping
                        "mapped_attributes": []
                    }
                    mappings.append(mapping)

                    # Record entity mapping
                    entity_mappings[source_entity["name"]] = target_entity["name"]

                    # Create sub-step
                    sub_step = InferenceStep(
                        description=f"Mapped {source_entity['name']} to {target_entity['name']} (positional mapping)",
                        inputs={"source_entity": source_entity, "target_entity": target_entity},
                        outputs={"mapping": mapping},
                        confidence=0.3
                    )
                    sub_steps.append(sub_step)

            # Transfer relationships from source to target domain
            for relationship in source_relationships:
                # Get source entities involved in this relationship
                source_entity1 = relationship.get("entity1")
                source_entity2 = relationship.get("entity2", None)  # May be None for unary relationships
                relation_type = relationship.get("type")

                # Check if we have mappings for these entities
                if source_entity1 in entity_mappings:
                    target_entity1 = entity_mappings[source_entity1]
                    target_entity2 = None

                    if source_entity2 and source_entity2 in entity_mappings:
                        target_entity2 = entity_mappings[source_entity2]

                    # Create inference
                    inference_id = str(uuid.uuid4())

                    if target_entity2:
                        # Binary relationship inference
                        inference = {
                            "id": inference_id,
                            "statement": f"{relation_type}({target_entity1}, {target_entity2})",
                            "confidence": 0.5,  # Lower confidence for analogical inference
                            "derived_from": {
                                "source_relationship": relationship,
                                "mapping1": next((m for m in mappings if m["source_entity"] == source_entity1), None),
                                "mapping2": next((m for m in mappings if m["source_entity"] == source_entity2), None)
                            }
                        }
                    else:
                        # Unary relationship inference
                        inference = {
                            "id": inference_id,
                            "statement": f"{relation_type}({target_entity1})",
                            "confidence": 0.5,
                            "derived_from": {
                                "source_relationship": relationship,
                                "mapping": next((m for m in mappings if m["source_entity"] == source_entity1), None)
                            }
                        }

                    inferences.append(inference)

                    # Record supporting evidence
                    if source_entity2:
                        supporting_evidence[inference_id] = [
                            f"Source relationship: {relation_type}({source_entity1}, {source_entity2})",
                            f"Mapping: {source_entity1} -> {target_entity1}",
                            f"Mapping: {source_entity2} -> {target_entity2}"
                        ]
                    else:
                        supporting_evidence[inference_id] = [
                            f"Source relationship: {relation_type}({source_entity1})",
                            f"Mapping: {source_entity1} -> {target_entity1}"
                        ]

                    # Create sub-step
                    if target_entity2:
                        sub_step = InferenceStep(
                            description=f"Transferred relationship {relation_type} from ({source_entity1}, {source_entity2}) to ({target_entity1}, {target_entity2})",
                            inputs={"relationship": relationship, "mappings": entity_mappings},
                            outputs={"inference": inference},
                            confidence=0.5
                        )
                    else:
                        sub_step = InferenceStep(
                            description=f"Transferred relationship {relation_type} from {source_entity1} to {target_entity1}",
                            inputs={"relationship": relationship, "mappings": entity_mappings},
                            outputs={"inference": inference},
                            confidence=0.5
                        )

                    sub_steps.append(sub_step)

        # Use LLM for more sophisticated analogical reasoning if available
        if self.llm_engine and not inferences:
            try:
                # Prepare domain descriptions for LLM
                source_text = f"Source domain: {source_domain.get('name', 'Source')}\n"
                source_text += "Entities:\n"
                for entity in source_entities:
                    source_text += f"- {entity['name']}"
                    if "attributes" in entity:
                        attrs = ", ".join(f"{k}: {v}" for k, v in entity["attributes"].items())
                        source_text += f" ({attrs})"
                    source_text += "\n"

                source_text += "Relationships:\n"
                for rel in source_relationships:
                    if "entity2" in rel:
                        source_text += f"- {rel['type']}({rel['entity1']}, {rel['entity2']})\n"
                    else:
                        source_text += f"- {rel['type']}({rel['entity1']})\n"

                target_text = f"Target domain: {target_domain.get('name', 'Target')}\n"
                target_text += "Entities:\n"
                for entity in target_entities:
                    target_text += f"- {entity['name']}"
                    if "attributes" in entity:
                        attrs = ", ".join(f"{k}: {v}" for k, v in entity["attributes"].items())
                        target_text += f" ({attrs})"
                    target_text += "\n"

                prompt = f"""
                I'll provide information about a source domain and a target domain.
                Use analogical reasoning to map entities between domains and infer relationships in the target domain.

                {source_text}

                {target_text}

                Please provide:
                1. Entity mappings from source to target domain
                2. Inferred relationships in the target domain

                Format your response as:

                Entity mappings:
                - [source_entity] maps to [target_entity] (confidence: [0-100%])
                - ...

                Inferred relationships:
                - [relationship_type]([target_entity1], [target_entity2]) (confidence: [0-100%])
                - ...

                Explain your reasoning for each mapping and inference.
                """

                llm_response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 500,
                    "temperature": 0.4
                })

                # Parse the response (simplified)
                if hasattr(llm_response, 'text'):
                    response_text = llm_response.text
                else:
                    response_text = str(llm_response)

                # Extract mappings
                mapping_section = re.search(r'Entity mappings:(.*?)(?=Inferred relationships:|$)', response_text, re.DOTALL)
                if mapping_section:
                    mapping_text = mapping_section.group(1).strip()
                    mapping_lines = re.findall(r'- (.*?)(?=\n|$)', mapping_text)

                    entity_mappings = {}
                    for line in mapping_lines:
                        # Extract source, target, and confidence
                        match = re.match(r'(.*?)\s+maps to\s+(.*?)\s+\(confidence:\s+(\d+)%\)', line)
                        if match:
                            source_entity = match.group(1).strip()
                            target_entity = match.group(2).strip()
                            confidence = int(match.group(3)) / 100

                            mapping_id = str(uuid.uuid4())
                            mapping = {
                                "id": mapping_id,
                                "source_entity": source_entity,
                                "target_entity": target_entity,
                                "confidence": confidence,
                                "llm_generated": True
                            }
                            mappings.append(mapping)

                            # Record entity mapping
                            entity_mappings[source_entity] = target_entity

                            # Create sub-step
                            sub_step = InferenceStep(
                                description=f"LLM mapped {source_entity} to {target_entity} (confidence: {confidence:.2f})",
                                inputs={"source_entity": source_entity, "target_entity": target_entity},
                                outputs={"mapping": mapping},
                                confidence=confidence
                            )
                            sub_steps.append(sub_step)

                # Extract inferences
                inference_section = re.search(r'Inferred relationships:(.*?)$', response_text, re.DOTALL)
                if inference_section:
                    inference_text = inference_section.group(1).strip()
                    inference_lines = re.findall(r'- (.*?)(?=\n|$)', inference_text)

                    for line in inference_lines:
                        # Extract relationship and confidence
                        match = re.match(r'(.*?)\s+\(confidence:\s+(\d+)%\)', line)
                        if match:
                            relationship = match.group(1).strip()
                            confidence = int(match.group(2)) / 100

                            # Parse relationship text
                            rel_match = re.match(r'(\w+)\((.*?)\)', relationship)
                            if rel_match:
                                rel_type = rel_match.group(1)
                                entities = [e.strip() for e in rel_match.group(2).split(',')]

                                inference_id = str(uuid.uuid4())
                                inference = {
                                    "id": inference_id,
                                    "statement": relationship,
                                    "confidence": confidence,
                                    "llm_generated": True
                                }
                                inferences.append(inference)

                                # Record supporting evidence based on entity mappings
                                supporting_evidence[inference_id] = [
                                    f"LLM-inferred relationship with confidence {confidence:.2f}"
                                ]

                                # Add any mapping evidence if we can find it
                                for entity in entities:
                                    for source, target in entity_mappings.items():
                                        if target == entity:
                                            supporting_evidence[inference_id].append(
                                                f"Mapping: {source} -> {target}"
                                            )

                                # Create sub-step
                                sub_step = InferenceStep(
                                    description=f"LLM inferred relationship: {relationship} (confidence: {confidence:.2f})",
                                    inputs={"entity_mappings": entity_mappings},
                                    outputs={"inference": inference},
                                    confidence=confidence
                                )
                                sub_steps.append(sub_step)

            except Exception as e:
                self.logger.warning(f"Error using LLM for analogical reasoning: {e}")

        # Calculate overall confidence
        confidence = 0.5 if mappings and inferences else 0.0

        # Check if reasoning is complete (if we derived any inferences)
        reasoning_complete = len(inferences) > 0

        # Convert inferences to conclusions
        conclusions = []
        for inference in inferences:
            conclusion_id = str(uuid.uuid4())
            conclusion = {
                "id": conclusion_id,
                "statement": inference["statement"],
                "confidence": inference["confidence"],
                "derived_from": inference.get("derived_from", {})
            }
            conclusions.append(conclusion)

            # Transfer supporting evidence
            if inference["id"] in supporting_evidence:
                supporting_evidence[conclusion_id] = supporting_evidence[inference["id"]]

        return {
            "outputs": {
                "mappings": mappings,
                "inferences": inferences
            },
            "workspace_updates": {
                "analogical_mappings": mappings,
                "analogical_inferences": inferences
            },
            "confidence": confidence,
            "conclusions": conclusions,
            "reasoning_complete": reasoning_complete,
            "supporting_evidence": supporting_evidence,
            "sub_steps": sub_steps
        }

    def _calculate_attribute_similarity(self, attrs1: Dict[str, Any], attrs2: Dict[str, Any]) -> float:
        """
        Calculate similarity between two sets of attributes.

        Args:
            attrs1: First set of attributes
            attrs2: Second set of attributes

        Returns:
            float: Similarity score (0-1)
        """
        if not attrs1 or not attrs2:
            return 0.0

        shared_keys = set(attrs1.keys()) & set(attrs2.keys())
        if not shared_keys:
            return 0.0

        # Calculate similarity for each shared key
        key_similarities = []

        for key in shared_keys:
            value1 = attrs1[key]
            value2 = attrs2[key]

            # Calculate value similarity based on type
            if type(value1) != type(value2):
                # Different types
                key_similarities.append(0.0)
            elif isinstance(value1, (int, float)) and isinstance(value2, (int, float)):
                # Numerical values
                max_val = max(abs(value1), abs(value2))
                if max_val == 0:
                    key_similarities.append(1.0)  # Both zero
                else:
                    diff = abs(value1 - value2) / max_val
                    key_similarities.append(max(0.0, 1.0 - diff))
            elif isinstance(value1, str) and isinstance(value2, str):
                # String values - simple exact match
                if value1.lower() == value2.lower():
                    key_similarities.append(1.0)
                else:
                    # Simple partial match
                    key_similarities.append(0.3)
            else:
                # Other types - simple equality check
                key_similarities.append(1.0 if value1 == value2 else 0.0)

        # Compute overall similarity
        return sum(key_similarities) / len(key_similarities)

    def _get_shared_attributes(self, attrs1: Dict[str, Any], attrs2: Dict[str, Any]) -> List[str]:
        """
        Get list of attributes that have similar values in both attribute sets.

        Args:
            attrs1: First set of attributes
            attrs2: Second set of attributes

        Returns:
            List[str]: List of shared attribute names
        """
        shared = []
        shared_keys = set(attrs1.keys()) & set(attrs2.keys())

        for key in shared_keys:
            value1 = attrs1[key]
            value2 = attrs2[key]

            if type(value1) == type(value2):
                if isinstance(value1, (int, float)):
                    # Consider numerical values similar if within 20%
                    max_val = max(abs(value1), abs(value2))
                    if max_val == 0 or abs(value1 - value2) / max_val < 0.2:
                        shared.append(key)
                elif value1 == value2:
                    # Exact match for other types
                    shared.append(key)

        return shared

    def _llm_assisted_reasoning(self,
                             query: ReasoningQuery,
                             workspace: Dict[str, Any],
                             inputs: Dict[str, Any],
                             current_result: ReasoningResult) -> Dict[str, Any]:
        """
        Use LLM to assist with general reasoning tasks.

        Args:
            query: The reasoning query
            workspace: The reasoning workspace
            inputs: Strategy inputs
            current_result: Current reasoning result

        Returns:
            Dict[str, Any]: Strategy results
        """
        question = inputs.get("question", query.question)
        context = inputs.get("context", {})

        # Check if LLM engine is available
        if not self.llm_engine:
            return {
                "outputs": {
                    "message": "LLM engine not available for reasoning",
                    "thoughts": [],
                    "conclusions": []
                },
                "confidence": 0.0,
                "workspace_updates": {},
                "conclusions": []
            }

        try:
            # Prepare context information
            context_text = ""

            # Include workspace data in context
            if workspace:
                for key, value in workspace.items():
                    if key in ["deductive_conclusions", "inductive_patterns",
                             "inductive_generalizations", "abductive_explanations"]:
                        context_text += f"\n{key.replace('_', ' ').title()}:\n"
                        for item in value:
                            if "statement" in item:
                                context_text += f"- {item['statement']}\n"
                            elif "description" in item:
                                context_text += f"- {item['description']}\n"

            # Include memory information if available
            if "working_memory_snapshot" in workspace:
                context_text += "\nRelevant Memory:\n"
                for item in workspace["working_memory_snapshot"]:
                    context_text += f"- {item}\n"

            # Create a reasoning prompt with chain-of-thought
            prompt = f"""
            I need your help with a reasoning task. Please think step by step.

            Question: {question}

            {context_text if context_text else ""}

            Please analyze this question carefully using the following structure:

            1. Break down the key aspects of the question
            2. Consider the available information and relevant knowledge
            3. Explore possible approaches to addressing the question
            4. Work through each step of your reasoning explicitly
            5. Evaluate the strength of your conclusions
            6. Provide your final answer with a confidence level (0-100%)

            Start your response with "Reasoning steps:" and end with "Final conclusion:"
            """

            # Call LLM for reasoning
            llm_response = self.llm_engine.complete({
                "prompt": prompt,
                "max_tokens": 800,
                "temperature": 0.3
            })

            # Parse the response
            if hasattr(llm_response, 'text'):
                response_text = llm_response.text
            else:
                response_text = str(llm_response)

            # Extract reasoning steps
            steps_match = re.search(r'Reasoning steps:(.*?)(?=Final conclusion:|$)', response_text, re.DOTALL)
            steps_text = steps_match.group(1).strip() if steps_match else ""

            # Extract steps as list
            steps = []
            step_items = re.split(r'\d+\.\s+', steps_text)
            for item in step_items:
                if item.strip():
                    steps.append(item.strip())

            # Extract final conclusion
            conclusion_match = re.search(r'Final conclusion:(.*?)$', response_text, re.DOTALL)
            conclusion_text = conclusion_match.group(1).strip() if conclusion_match else ""

            # Extract confidence if present
            confidence = 0.5  # Default
            confidence_match = re.search(r'confidence[:\s]+(\d+)%', response_text, re.IGNORECASE)
            if confidence_match:
                confidence = int(confidence_match.group(1)) / 100

            # Create conclusions
            conclusion_id = str(uuid.uuid4())
            conclusion = {
                "id": conclusion_id,
                "statement": conclusion_text,
                "confidence": confidence,
                "derived_from": {
                    "llm_reasoning": True,
                    "steps": steps
                }
            }

            # Create supporting evidence
            supporting_evidence = {
                conclusion_id: [f"Reasoning step: {step}" for step in steps]
            }

            # Create inference steps
            sub_steps = []
            for i, step in enumerate(steps):
                sub_step = InferenceStep(
                    description=f"Step {i+1}: {step[:100]}{'...' if len(step) > 100 else ''}",
                    outputs={"thought": step},
                    confidence=confidence
                )
                sub_steps.append(sub_step)

            # Overall confidence (same as conclusion confidence)
            result_confidence = confidence

            # Reasoning is complete
            reasoning_complete = True

            return {
                "outputs": {
                    "thoughts": steps,
                    "conclusion": conclusion_text
                },
                "workspace_updates": {
                    "llm_reasoning_steps": steps,
                    "llm_conclusion": conclusion_text
                },
                "confidence": result_confidence,
                "conclusions": [conclusion],
                "reasoning_complete": reasoning_complete,
                "supporting_evidence": supporting_evidence,
                "sub_steps": sub_steps
            }

        except Exception as e:
            self.logger.error(f"Error in LLM-assisted reasoning: {e}", exc_info=True)

            return {
                "outputs": {
                    "error": str(e)
                },
                "confidence": 0.0,
                "conclusions": [],
                "reasoning_complete": False
            }

    #------------------------------------------------------------------
    # Utility methods
    #------------------------------------------------------------------

    @CognitionManager.cognitive_capability(
        capability_type=CognitiveProcessType.REASONING,
        description="Solve a problem using multi-strategy approach",
        priority=40,
        compatible_modes={CognitiveMode.DELIBERATIVE}
    )
    def solve_problem(self,
                     request: CognitiveRequest,
                     context: CognitiveContext,
                     query: Any,
                     **kwargs) -> Dict[str, Any]:
        """
        Higher-level problem-solving capability that combines multiple reasoning strategies.

        Args:
            request: The cognitive request
            context: The cognitive context
            query: The problem to solve
            **kwargs: Additional arguments

        Returns:
            Dict[str, Any]: Problem-solving results
        """
        try:
            self.logger.info(f"Solving problem: {query}")

            # Create a reasoning workspace for this problem
            workspace_id = self.create_reasoning_workspace()

            # Initialize problem-solving state
            problem_state = {
                "problem": query,
                "context": context.parameters,
                "approach": "multi-strategy",
                "steps_taken": [],
                "current_state": "initial",
                "goal_state": "solved"
            }

            self.update_workspace(workspace_id, {"problem_state": problem_state})

            # Problem-solving involves multiple phases:
            # 1. Problem analysis
            # 2. Strategy selection
            # 3. Solution generation
            # 4. Solution evaluation

            # Phase 1: Problem analysis
            analysis_result = self._analyze_problem(query, workspace_id)

            self.update_workspace(workspace_id, {
                "problem_analysis": analysis_result,
                "problem_state": {**problem_state, "current_state": "analyzed"}
            })

            # Phase 2: Strategy selection
            strategies = self._select_problem_solving_strategies(analysis_result, workspace_id)

            self.update_workspace(workspace_id, {
                "selected_strategies": strategies,
                "problem_state": {**problem_state, "current_state": "strategies_selected"}
            })

            # Phase 3: Solution generation using selected strategies
            solutions = []
            for strategy in strategies:
                strategy_result = self._apply_problem_solving_strategy(strategy, workspace_id)
                if strategy_result.get("solution"):
                    solutions.append({
                        "strategy": strategy,
                        "solution": strategy_result["solution"],
                        "confidence": strategy_result.get("confidence", 0.5)
                    })

            self.update_workspace(workspace_id, {
                "generated_solutions": solutions,
                "problem_state": {**problem_state, "current_state": "solutions_generated"}
            })

            # Phase 4: Solution evaluation and selection
            final_solution = self._evaluate_solutions(solutions, workspace_id)

            self.update_workspace(workspace_id, {
                "final_solution": final_solution,
                "problem_state": {**problem_state, "current_state": "solved"}
            })

            # Prepare result
            result = {
                "problem": query,
                "analysis": analysis_result,
                "strategies_used": [s["name"] for s in strategies],
                "solution": final_solution.get("solution"),
                "confidence": final_solution.get("confidence", 0.0),
                "reasoning_trace": final_solution.get("reasoning_trace", [])
            }

            return result

        except Exception as e:
            self.logger.error(f"Error in problem solving: {e}", exc_info=True)

            return {
                "error": str(e),
                "problem": query
            }

    def _analyze_problem(self, problem: Any, workspace_id: str) -> Dict[str, Any]:
        """
        Analyze a problem to determine its characteristics.

        Args:
            problem: The problem to analyze
            workspace_id: ID of the workspace

        Returns:
            Dict[str, Any]: Problem analysis results
        """
        # This is a simplified implementation
        analysis = {
            "problem_type": "unknown",
            "key_elements": [],
            "constraints": [],
            "potential_approaches": []
        }

        # Use LLM for problem analysis if available
        if self.llm_engine:
            try:
                prompt = f"""
                Analyze the following problem:

                Problem: {problem}

                Please provide a structured analysis with the following:

                1. Problem type (e.g., logical, mathematical, decision-making, etc.)
                2. Key elements of the problem
                3. Constraints or requirements
                4. Potential approaches for solving this problem

                Format your response with clear section headers.
                """

                llm_response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 500,
                    "temperature": 0.3
                })

                # Parse the response
                if hasattr(llm_response, 'text'):
                    response_text = llm_response.text
                else:
                    response_text = str(llm_response)

                # Extract problem type
                type_match = re.search(r'Problem type[:\s]+(.*?)(?=\n|$)', response_text)
                if type_match:
                    analysis["problem_type"] = type_match.group(1).strip()

                # Extract key elements
                elements_section = re.search(r'Key elements[:\s]+(.*?)(?=\n\d+\.\s|Constraints|Potential approaches|$)', response_text, re.DOTALL)
                if elements_section:
                    elements_text = elements_section.group(1).strip()
                    analysis["key_elements"] = [
                        item.strip() for item in re.findall(r'(?:^|\n)- (.*?)(?=\n|$)', elements_text)
                    ]

                # Extract constraints
                constraints_section = re.search(r'Constraints[:\s]+(.*?)(?=\n\d+\.\s|Potential approaches|$)', response_text, re.DOTALL)
                if constraints_section:
                    constraints_text = constraints_section.group(1).strip()
                    analysis["constraints"] = [
                        item.strip() for item in re.findall(r'(?:^|\n)- (.*?)(?=\n|$)', constraints_text)
                    ]

                # Extract potential approaches
                approaches_section = re.search(r'Potential approaches[:\s]+(.*?)$', response_text, re.DOTALL)
                if approaches_section:
                    approaches_text = approaches_section.group(1).strip()
                    analysis["potential_approaches"] = [
                        item.strip() for item in re.findall(r'(?:^|\n)- (.*?)(?=\n|$)', approaches_text)
                    ]

            except Exception as e:
                self.logger.warning(f"Error using LLM for problem analysis: {e}")

        # Fallback: Simple keyword-based analysis
        if analysis["problem_type"] == "unknown":
            problem_text = str(problem).lower()

            if any(word in problem_text for word in ["prove", "theorem", "logical", "deduce"]):
                analysis["problem_type"] = "logical"
            elif any(word in problem_text for word in ["calculate", "compute", "math", "equation"]):
                analysis["problem_type"] = "mathematical"
            elif any(word in problem_text for word in ["decide", "choose", "best", "optimal"]):
                analysis["problem_type"] = "decision-making"
            elif any(word in problem_text for word in ["why", "explain", "reason", "cause"]):
                analysis["problem_type"] = "explanatory"
            elif any(word in problem_text for word in ["compare", "contrast", "similar", "different"]):
                analysis["problem_type"] = "comparative"
            else:
                analysis["problem_type"] = "general"

            # Basic key elements extraction (just tokenize and take nouns/phrases)
            words = problem_text.split()
            if len(words) > 2:
                analysis["key_elements"] = [" ".join(words[:3]), " ".join(words[-3:])]

        return analysis

    def _select_problem_solving_strategies(self, analysis: Dict[str, Any], workspace_id: str) -> List[Dict[str, Any]]:
        """
        Select appropriate problem-solving strategies based on problem analysis.

        Args:
            analysis: Problem analysis results
            workspace_id: ID of the workspace

        Returns:
            List[Dict[str, Any]]: Selected strategies
        """
        strategies = []
        problem_type = analysis.get("problem_type", "unknown")

        # Select strategies based on problem type
        if problem_type == "logical":
            strategies.append({
                "name": "deductive_reasoning",
                "reasoning_type": ReasoningType.DEDUCTIVE,
                "description": "Use deductive reasoning to draw logical conclusions",
                "confidence": 0.8
            })
        elif problem_type == "mathematical":
            strategies.append({
                "name": "mathematical_reasoning",
                "reasoning_type": ReasoningType.DEDUCTIVE,
                "description": "Apply mathematical principles and calculations",
                "confidence": 0.8
            })
        elif problem_type == "decision-making":
            strategies.append({
                "name": "decision_analysis",
                "reasoning_type": ReasoningType.CAUSAL,
                "description": "Analyze decision factors and their implications",
                "confidence": 0.7
            })
            strategies.append({
                "name": "counterfactual_reasoning",
                "reasoning_type": ReasoningType.COUNTERFACTUAL,
                "description": "Consider alternative scenarios and outcomes",
                "confidence": 0.6
            })
        elif problem_type == "explanatory":
            strategies.append({
                "name": "abductive_reasoning",
                "reasoning_type": ReasoningType.ABDUCTIVE,
                "description": "Infer the most likely explanation",
                "confidence": 0.7
            })
            strategies.append({
                "name": "causal_analysis",
                "reasoning_type": ReasoningType.CAUSAL,
                "description": "Analyze cause-and-effect relationships",
                "confidence": 0.7
            })
        elif problem_type == "comparative":
            strategies.append({
                "name": "analogical_reasoning",
                "reasoning_type": ReasoningType.ANALOGICAL,
                "description": "Compare and map between domains",
                "confidence": 0.7
            })

        # Fallback: Add general strategies if no specific ones were selected
        if not strategies:
            strategies.append({
                "name": "general_problem_solving",
                "reasoning_type": ReasoningType.HYBRID,
                "description": "Apply general problem-solving methods",
                "confidence": 0.5
            })

        # Always consider adding LLM-assisted reasoning as a complementary strategy
        if self.llm_engine:
            strategies.append({
                "name": "llm_assisted",
                "reasoning_type": ReasoningType.HYBRID,
                "description": "Use LLM to assist with reasoning",
                "confidence": 0.6
            })

        return strategies

    def _apply_problem_solving_strategy(self, strategy: Dict[str, Any], workspace_id: str) -> Dict[str, Any]:
        """
        Apply a problem-solving strategy.

        Args:
            strategy: The strategy to apply
            workspace_id: ID of the workspace

        Returns:
            Dict[str, Any]: Strategy application results
        """
        # Get workspace data
        workspace = self.get_workspace(workspace_id)
        problem = workspace.get("problem_state", {}).get("problem", "")
        problem_analysis = workspace.get("problem_analysis", {})

        # Initialize result
        result = {
            "strategy": strategy["name"],
            "reasoning_trace": [],
            "solution": None,
            "confidence": 0.0
        }

        # Apply strategy based on its type
        if strategy["name"] == "deductive_reasoning":
            # Apply deductive reasoning strategy
            # This is a simplified implementation

            # Extract premises and rules from problem and analysis
            premises = []
            rules = []

            # Simple keyword-based extraction of premises and rules
            if isinstance(problem, str):
                sentences = problem.split(".")
                for sentence in sentences:
                    sentence = sentence.strip().lower()

                    # Look for premises (simple heuristic)
                    if sentence.startswith(("given", "assume", "consider", "suppose")):
                        premises.append(sentence)

                    # Look for rules (simple heuristic)
                    if "if" in sentence and "then" in sentence:
                        parts = sentence.split("if")[1].split("then")
                        if len(parts) == 2:
                            rule = {
                                "if": parts[0].strip(),
                                "then": parts[1].strip(),
                                "confidence": 0.9
                            }
                            rules.append(rule)

            # Add key elements from analysis as potential premises
            for element in problem_analysis.get("key_elements", []):
                if element not in premises:
                    premises.append(element)

            # If we have premises and rules, apply deductive reasoning
            if premises and rules:
                # Call simple deductive reasoning
                deduction_input = {
                    "premises": premises,
                    "rules": rules
                }

                deduction_result = self._simple_deductive_reasoning(
                    query=None,
                    workspace=workspace,
                    inputs=deduction_input,
                    current_result=None
                )

                # Extract conclusions
                conclusions = deduction_result.get("conclusions", [])

                if conclusions:
                    # Use the highest confidence conclusion as the solution
                    best_conclusion = max(conclusions, key=lambda x: x.get("confidence", 0))

                    result["solution"] = best_conclusion["statement"]
                    result["confidence"] = best_conclusion["confidence"]

                    # Add reasoning trace
                    for step in deduction_result.get("sub_steps", []):
                        result["reasoning_trace"].append(step.description)
                else:
                    result["reasoning_trace"].append("No conclusions derived from deductive reasoning")
            else:
                result["reasoning_trace"].append("Insufficient premises or rules for deductive reasoning")

        elif strategy["name"] == "abductive_reasoning":
            # Apply abductive reasoning strategy
            # This is a simplified implementation

            # Extract observations from problem and analysis
            observations = []

            # Extract observations from problem
            if isinstance(problem, str):
                sentences = problem.split(".")
                for sentence in sentences:
                    sentence = sentence.strip()
                    if sentence and not sentence.lower().startswith(("why", "how", "what")):
                        observations.append(sentence)

            # Add key elements as observations
            for element in problem_analysis.get("key_elements", []):
                if element not in observations:
                    observations.append(element)

            # If we have observations, apply abductive reasoning
            if observations:
                # Call simple abductive reasoning
                abduction_input = {
                    "observations": observations
                }

                abduction_result = self._simple_abductive_reasoning(
                    query=None,
                    workspace=workspace,
                    inputs=abduction_input,
                    current_result=None
                )

                # Extract explanations
                explanations = abduction_result.get("outputs", {}).get("explanations", [])

                if explanations:
                    # Use the highest confidence explanation as the solution
                    best_explanation = max(explanations, key=lambda x: x.get("confidence", 0))

                    result["solution"] = best_explanation["statement"]
                    result["confidence"] = best_explanation["confidence"]

                    # Add reasoning trace
                    for step in abduction_result.get("sub_steps", []):
                        result["reasoning_trace"].append(step.description)
                else:
                    result["reasoning_trace"].append("No explanations derived from abductive reasoning")
            else:
                result["reasoning_trace"].append("Insufficient observations for abductive reasoning")

        elif strategy["name"] == "llm_assisted":
            # Apply LLM-assisted reasoning strategy
            if self.llm_engine:
                llm_input = {
                    "question": problem,
                    "context": problem_analysis
                }

                llm_result = self._llm_assisted_reasoning(
                    query=None,
                    workspace=workspace,
                    inputs=llm_input,
                    current_result=None
                )

                # Extract conclusion
                conclusion = llm_result.get("outputs", {}).get("conclusion")

                if conclusion:
                    result["solution"] = conclusion
                    result["confidence"] = llm_result.get("confidence", 0.6)

                    # Add reasoning trace
                    thoughts = llm_result.get("outputs", {}).get("thoughts", [])
                    for thought in thoughts:
                        result["reasoning_trace"].append(thought)
                else:
                    result["reasoning_trace"].append("No conclusion from LLM-assisted reasoning")
            else:
                result["reasoning_trace"].append("LLM engine not available")

        elif strategy["name"] == "general_problem_solving":
            # Apply general problem-solving strategy
            # This is a placeholder implementation

            # Simply combine all available information
            description = f"General problem-solving for: {problem}"
            approaches = problem_analysis.get("potential_approaches", [])

            if approaches:
                # Choose the first suggested approach
                approach = approaches[0]
                result["solution"] = f"Using the approach: {approach}"
                result["confidence"] = 0.4
                result["reasoning_trace"].append(f"Selected approach: {approach}")
            else:
                result["reasoning_trace"].append("No specific approaches identified")

        # Return the result
        return result

    def _evaluate_solutions(self, solutions: List[Dict[str, Any]], workspace_id: str) -> Dict[str, Any]:
        """
        Evaluate and select the best solution from multiple candidates.

        Args:
            solutions: List of candidate solutions
            workspace_id: ID of the workspace

        Returns:
            Dict[str, Any]: Selected best solution
        """
        if not solutions:
            return {
                "solution": None,
                "confidence": 0.0,
                "reasoning_trace": ["No solutions to evaluate"]
            }

        # Get workspace data
        workspace = self.get_workspace(workspace_id)
        problem = workspace.get("problem_state", {}).get("problem", "")

        # Initialize evaluation result
        evaluation = {
            "evaluated_solutions": [],
            "reasoning_trace": [],
            "selected_solution": None,
            "confidence": 0.0
        }

        # Evaluate each solution
        for solution in solutions:
            solution_eval = {
                "strategy": solution["strategy"],
                "solution": solution["solution"],
                "base_confidence": solution["confidence"],
                "adjusted_confidence": solution["confidence"],  # Will be adjusted
                "strengths": [],
                "weaknesses": []
            }

            # Simple evaluation heuristics
            if solution["confidence"] > 0.7:
                solution_eval["strengths"].append("High confidence from strategy")
            elif solution["confidence"] < 0.5:
                solution_eval["weaknesses"].append("Low confidence from strategy")

            if len(solution.get("reasoning_trace", [])) > 3:
                solution_eval["strengths"].append("Detailed reasoning trace")
            else:
                solution_eval["weaknesses"].append("Limited reasoning steps")

            # Use LLM to evaluate solution if available
            if self.llm_engine:
                try:
                    prompt = f"""
                    Evaluate this solution to the given problem:

                    Problem: {problem}

                    Proposed solution: {solution["solution"]}

                    Strategy used: {solution["strategy"]}

                    Please evaluate this solution on:
                    1. Correctness (is it accurate and valid?)
                    2. Completeness (does it fully address the problem?)
                    3. Clarity (is it clearly explained?)

                    For each criterion, provide a rating (0-10) and brief explanation.
                    End with an overall assessment and confidence score (0-100%).
                    """

                    llm_response = self.llm_engine.complete({
                        "prompt": prompt,
                        "max_tokens": 300,
                        "temperature": 0.3
                    })

                    # Parse the response
                    if hasattr(llm_response, 'text'):
                        response_text = llm_response.text
                    else:
                        response_text = str(llm_response)

                    # Extract ratings
                    correctness_match = re.search(r'Correctness.*?(\d+)/10', response_text)
                    completeness_match = re.search(r'Completeness.*?(\d+)/10', response_text)
                    clarity_match = re.search(r'Clarity.*?(\d+)/10', response_text)

                    correctness = int(correctness_match.group(1)) / 10 if correctness_match else 0.5
                    completeness = int(completeness_match.group(1)) / 10 if completeness_match else 0.5
                    clarity = int(clarity_match.group(1)) / 10 if clarity_match else 0.5

                    # Extract overall confidence
                    confidence_match = re.search(r'confidence.*?(\d+)%', response_text, re.IGNORECASE)
                    overall_confidence = int(confidence_match.group(1)) / 100 if confidence_match else 0.5

                    # Adjust solution confidence based on evaluation
                    solution_eval["adjusted_confidence"] = (solution["confidence"] + overall_confidence) / 2

                    # Extract strengths and weaknesses
                    if correctness > 0.7:
                        solution_eval["strengths"].append(f"High correctness ({correctness:.1f}/1.0)")
                    elif correctness < 0.5:
                        solution_eval["weaknesses"].append(f"Low correctness ({correctness:.1f}/1.0)")

                    if completeness > 0.7:
                        solution_eval["strengths"].append(f"High completeness ({completeness:.1f}/1.0)")
                    elif completeness < 0.5:
                        solution_eval["weaknesses"].append(f"Low completeness ({completeness:.1f}/1.0)")

                    if clarity > 0.7:
                        solution_eval["strengths"].append(f"High clarity ({clarity:.1f}/1.0)")
                    elif clarity < 0.5:
                        solution_eval["weaknesses"].append(f"Low clarity ({clarity:.1f}/1.0)")

                except Exception as e:
                    self.logger.warning(f"Error using LLM for solution evaluation: {e}")
                    # Fall back to the strategy's confidence
                    solution_eval["adjusted_confidence"] = solution["confidence"]

            evaluation["evaluated_solutions"].append(solution_eval)

        # Select the best solution based on adjusted confidence
        if evaluation["evaluated_solutions"]:
            best_solution = max(evaluation["evaluated_solutions"], key=lambda x: x["adjusted_confidence"])

            evaluation["selected_solution"] = best_solution["solution"]
            evaluation["confidence"] = best_solution["adjusted_confidence"]

            # Add reasoning trace
            evaluation["reasoning_trace"].append(f"Selected solution from strategy: {best_solution['strategy']}")
            evaluation["reasoning_trace"].append(f"Confidence: {best_solution['adjusted_confidence']:.2f}")
            evaluation["reasoning_trace"].append("Strengths: " + ", ".join(best_solution["strengths"]))
            if best_solution["weaknesses"]:
                evaluation["reasoning_trace"].append("Weaknesses: " + ", ".join(best_solution["weaknesses"]))

            # Add the original reasoning trace from the selected solution
            for solution in solutions:
                if solution["strategy"] == best_solution["strategy"]:
                    evaluation["reasoning_trace"].extend(solution.get("reasoning_trace", []))
                    break

        return evaluation


# Standalone execution for testing
if __name__ == "__main__":
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Create a ReasoningCore instance
    reasoning = ReasoningCore()

    # Create a context and request for testing
    context = CognitiveContext(
        parameters={}
    )

    # Test with a simple query
    test_query = "If all birds can fly and a penguin is a bird, can a penguin fly?"

    request = CognitiveRequest(query=test_query, context=context)
    result = reasoning.reason(request, context, test_query)

    print(f"\nReasoning result for: {test_query}")
    if "error" in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Conclusions:")
        for conclusion in result.get("conclusions", []):
            print(f"- {conclusion['statement']} (confidence: {conclusion['confidence']:.2f})")

        print("\nInference steps:")
        for step in result.get("inference_steps", []):
            print(f"- {step['description']}")

    print("-" * 40)

IndentationError: unexpected indent (<ipython-input-15-6997f806d90a>, line 1685)

# Neural Reasoner

In [ ]:
"""
neural_reasoner.py - Neural reasoning module for GrandNexus

This module implements neural network-based reasoning capabilities for GrandNexus,
allowing for vector-based reasoning, pattern matching, and semantic similarity
computation within the cognitive architecture. It serves as a complement to
symbolic reasoning approaches, excelling at fuzzy pattern recognition and handling
uncertainty in knowledge.

Key features:
- Integration with embedding spaces for semantic reasoning
- Neural retrieval and similarity-based inference
- Hybrid neuro-symbolic operations supporting higher reasoning modules
- Adaptive weighting of neural vs symbolic evidence
"""

import logging
import threading
import time
import uuid
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from dataclasses import dataclass, field
from enum import Enum, auto
import numpy as np
from collections import defaultdict

# Import from GrandNexus modules
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.graph.graph_engine import create_graph_engine, AbstractGraphEngine

# -----------------------------------------------------------------------------
# Data Structures and Types
# -----------------------------------------------------------------------------

class EvidenceSourceType(Enum):
    """Types of sources that can provide reasoning evidence"""
    EMBEDDING = auto()       # Embeddings space similarity
    RETRIEVAL = auto()       # Retrieved information from memory
    GRAPH = auto()           # Graph-based inference
    PATTERN = auto()         # Pattern matching
    ANALOGY = auto()         # Analogical reasoning
    EXTERNAL = auto()        # External source (e.g., LLM)

class ReasoningTask(Enum):
    """Types of neural reasoning tasks"""
    SIMILARITY = auto()      # Compute similarity between concepts
    CLASSIFICATION = auto()  # Classify a concept
    CLUSTERING = auto()      # Group similar concepts
    COMPLETION = auto()      # Complete a partial pattern
    RETRIEVAL = auto()       # Retrieve similar concepts
    INFERENCE = auto()       # Infer new knowledge
    ANALOGY = auto()         # Find or complete analogies

@dataclass
class NeuralEvidence:
    """Evidence retrieved or computed by neural reasoning processes"""
    evidence_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    source_type: EvidenceSourceType = EvidenceSourceType.EMBEDDING
    content: Any = None
    confidence: float = 0.5
    relevance: float = 0.8
    embedding: Optional[np.ndarray] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class NeuralReasoningRequest:
    """Request for neural reasoning"""
    request_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    task_type: ReasoningTask = ReasoningTask.SIMILARITY
    query: Any = None
    query_embedding: Optional[np.ndarray] = None
    context: Optional[Dict[str, Any]] = None
    parameters: Dict[str, Any] = field(default_factory=dict)
    evidence_sources: List[EvidenceSourceType] = field(default_factory=list)
    timestamp: float = field(default_factory=time.time)

@dataclass
class NeuralReasoningResult:
    """Result of neural reasoning"""
    request_id: str
    conclusion: Any = None
    confidence: float = 0.0
    evidence: List[NeuralEvidence] = field(default_factory=list)
    embedding: Optional[np.ndarray] = None
    explanation: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    processing_time: float = 0.0

# -----------------------------------------------------------------------------
# Neural Reasoner Class
# -----------------------------------------------------------------------------

class NeuralReasoner:
    """
    Neural reasoning component for GrandNexus that leverages vector embeddings
    and neural networks to perform reasoning operations that complement
    traditional symbolic approaches.

    This module excels at similarity-based reasoning, analogy detection,
    and handling fuzzy or uncertain knowledge.
    """

    def __init__(self,
                 nexus_core: Optional[NexusCore] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 graph_engine: Optional[AbstractGraphEngine] = None,
                 embedding_dim: int = 768,
                 similarity_threshold: float = 0.7,
                 use_llm_embeddings: bool = True):
        """
        Initialize the neural reasoner.

        Args:
            nexus_core: Reference to the central nexus core
            semantic_memory: Reference to semantic memory for concept retrieval
            episodic_memory: Reference to episodic memory for experience retrieval
            working_memory: Reference to working memory for context
            graph_engine: Reference to graph engine for graph operations
            embedding_dim: Dimension of embedding vectors
            similarity_threshold: Default threshold for similarity matches
            use_llm_embeddings: Whether to use LLM-based embeddings
        """
        self.logger = logging.getLogger("GrandNexus.Reasoning.Neural")

        # Store module references
        self.nexus_core = nexus_core
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory
        self.working_memory = working_memory
        self.graph_engine = graph_engine

        # Configuration
        self.embedding_dim = embedding_dim
        self.similarity_threshold = similarity_threshold
        self.use_llm_embeddings = use_llm_embeddings

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Cache frequently accessed embeddings
        self.embedding_cache = {}
        self.max_cache_size = 1000

        # Performance metrics and activation tracking
        self.metrics = defaultdict(list)
        self.task_history = []
        self.last_activation_time = time.time()

        # Adaptive weights for different evidence sources
        self.evidence_weights = {
            EvidenceSourceType.EMBEDDING: 1.0,
            EvidenceSourceType.RETRIEVAL: 0.8,
            EvidenceSourceType.GRAPH: 0.7,
            EvidenceSourceType.PATTERN: 0.6,
            EvidenceSourceType.ANALOGY: 0.5,
            EvidenceSourceType.EXTERNAL: 0.9
        }

        self.logger.info(f"NeuralReasoner initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the neural reasoner and establish connections"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Check for essential dependencies
                if self.semantic_memory is None:
                    self.logger.warning("Neural reasoner initialized without semantic memory")

                # Initialize any pre-trained models or weights if needed
                # [In a real implementation, this could load pre-trained models or embeddings]

                # Connect with NexusCore for messaging/coordination if available
                if self.nexus_core:
                    self._register_with_nexus()

                self.initialized = True
                self.logger.info("Neural reasoner initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"Failed to initialize neural reasoner: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the neural reasoner"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start any background processes if needed
                # [In a real implementation, this might start background workers]

                self.running = True
                self.logger.info("Neural reasoner started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start neural reasoner: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the neural reasoner"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Stop any background processes
                # [In a real implementation, this would clean up resources]

                self.running = False
                self.logger.info("Neural reasoner stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping neural reasoner: {str(e)}")
                return False

    def process(self, request: NeuralReasoningRequest) -> NeuralReasoningResult:
        """
        Process a neural reasoning request.

        Args:
            request: The neural reasoning request to process

        Returns:
            The result of neural reasoning
        """
        if not self.running:
            self.logger.warning("Neural reasoner not running, attempting to start")
            if not self.start():
                raise RuntimeError("Neural reasoner could not be started")

        self.logger.info(f"Processing request {request.request_id}, task: {request.task_type}")
        self.last_activation_time = time.time()

        start_time = time.time()

        # Prepare the result object
        result = NeuralReasoningResult(request_id=request.request_id)

        try:
            # Process different types of reasoning tasks
            if request.task_type == ReasoningTask.SIMILARITY:
                self._process_similarity_task(request, result)
            elif request.task_type == ReasoningTask.CLASSIFICATION:
                self._process_classification_task(request, result)
            elif request.task_type == ReasoningTask.CLUSTERING:
                self._process_clustering_task(request, result)
            elif request.task_type == ReasoningTask.COMPLETION:
                self._process_completion_task(request, result)
            elif request.task_type == ReasoningTask.RETRIEVAL:
                self._process_retrieval_task(request, result)
            elif request.task_type == ReasoningTask.INFERENCE:
                self._process_inference_task(request, result)
            elif request.task_type == ReasoningTask.ANALOGY:
                self._process_analogy_task(request, result)
            else:
                raise ValueError(f"Unknown reasoning task type: {request.task_type}")

            # Record processing time
            result.processing_time = time.time() - start_time

            # Update metrics
            self.metrics["processing_time"].append(result.processing_time)
            self.metrics["confidence"].append(result.confidence)

            # Store in history
            self.task_history.append({
                "request_id": request.request_id,
                "task_type": request.task_type,
                "timestamp": time.time(),
                "confidence": result.confidence
            })

            self.logger.info(f"Completed request {request.request_id} in {result.processing_time:.3f}s")
            return result

        except Exception as e:
            # Log error and return a result indicating failure
            self.logger.error(f"Error processing request {request.request_id}: {str(e)}")

            result.confidence = 0.0
            result.explanation = f"Error: {str(e)}"
            result.processing_time = time.time() - start_time

            return result

    def compute_similarity(self, entity1: Any, entity2: Any,
                          method: str = "cosine") -> float:
        """
        Compute semantic similarity between two entities.

        Args:
            entity1: First entity (can be text, concept ID, or embedding)
            entity2: Second entity (can be text, concept ID, or embedding)
            method: Similarity method ('cosine', 'dot', 'euclidean')

        Returns:
            Similarity score between 0 and 1
        """
        # Get embeddings for both entities
        emb1 = self._get_embedding(entity1)
        emb2 = self._get_embedding(entity2)

        if emb1 is None or emb2 is None:
            self.logger.warning(f"Could not get embeddings for similarity computation")
            return 0.0

        # Compute similarity based on specified method
        if method == "cosine":
            return self._cosine_similarity(emb1, emb2)
        elif method == "dot":
            return self._dot_similarity(emb1, emb2)
        elif method == "euclidean":
            return self._euclidean_similarity(emb1, emb2)
        else:
            self.logger.warning(f"Unknown similarity method: {method}, using cosine")
            return self._cosine_similarity(emb1, emb2)

    def find_nearest(self, query: Any, candidates: List[Any],
                    top_k: int = 5, method: str = "cosine") -> List[Tuple[Any, float]]:
        """
        Find the nearest entities to a query from a list of candidates.

        Args:
            query: Query entity (text, concept ID, or embedding)
            candidates: List of candidate entities
            top_k: Number of top results to return
            method: Similarity method to use

        Returns:
            List of (entity, similarity) tuples for the top matches
        """
        # Get query embedding
        query_embedding = self._get_embedding(query)
        if query_embedding is None:
            self.logger.warning(f"Could not get embedding for query")
            return []

        # Compute similarity with all candidates
        similarities = []
        for candidate in candidates:
            candidate_embedding = self._get_embedding(candidate)
            if candidate_embedding is not None:
                if method == "cosine":
                    sim = self._cosine_similarity(query_embedding, candidate_embedding)
                elif method == "dot":
                    sim = self._dot_similarity(query_embedding, candidate_embedding)
                elif method == "euclidean":
                    sim = self._euclidean_similarity(query_embedding, candidate_embedding)
                else:
                    sim = self._cosine_similarity(query_embedding, candidate_embedding)

                similarities.append((candidate, sim))

        # Sort by similarity (descending) and return top_k
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:top_k]

    def complete_pattern(self, pattern: List[Any], options: List[Any]) -> Tuple[Any, float]:
        """
        Complete a pattern by finding the best continuation.

        Args:
            pattern: List of entities forming a pattern
            options: List of candidate entities to complete the pattern

        Returns:
            Tuple of (best_completion, confidence)
        """
        if not pattern or len(pattern) < 2:
            self.logger.warning("Pattern too short for completion")
            return (None, 0.0)

        if not options:
            self.logger.warning("No options provided for pattern completion")
            return (None, 0.0)

        # Convert pattern to embeddings
        pattern_embeddings = [self._get_embedding(item) for item in pattern]
        pattern_embeddings = [emb for emb in pattern_embeddings if emb is not None]

        if len(pattern_embeddings) < 2:
            self.logger.warning("Insufficient valid embeddings in pattern")
            return (None, 0.0)

        # Simple pattern continuation: predict by vector arithmetic
        # Example: A is to B as C is to ?
        # We compute D = C + (B - A) in the embedding space
        if len(pattern_embeddings) >= 3:
            # Take the last 3 embeddings to form A, B, C for A:B::C:?
            a, b, c = pattern_embeddings[-3:]
            target = c + (b - a)

            # Normalize target
            target = target / np.linalg.norm(target)

            # Find closest option to target
            best_option = None
            best_similarity = -1.0

            for option in options:
                option_emb = self._get_embedding(option)
                if option_emb is not None:
                    sim = self._cosine_similarity(target, option_emb)
                    if sim > best_similarity:
                        best_similarity = sim
                        best_option = option

            return (best_option, best_similarity)

        # Fallback: average embedding similarity to pattern
        similarities = []
        for option in options:
            option_emb = self._get_embedding(option)
            if option_emb is not None:
                # Compute average similarity to pattern items
                avg_sim = sum(self._cosine_similarity(option_emb, pattern_emb)
                              for pattern_emb in pattern_embeddings) / len(pattern_embeddings)
                similarities.append((option, avg_sim))

        if not similarities:
            return (None, 0.0)

        # Return the option with highest similarity
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[0]

    def find_cluster_centers(self, entities: List[Any], n_clusters: int = 3) -> List[Any]:
        """
        Find cluster centers among entities in embedding space.

        Args:
            entities: List of entities to cluster
            n_clusters: Number of clusters to identify

        Returns:
            List of entities representing cluster centers
        """
        # Convert entities to embeddings
        embeddings = []
        valid_entities = []

        for entity in entities:
            emb = self._get_embedding(entity)
            if emb is not None:
                embeddings.append(emb)
                valid_entities.append(entity)

        if len(embeddings) < n_clusters:
            self.logger.warning(f"Not enough valid entities for {n_clusters} clusters")
            return valid_entities  # Return all valid entities if too few

        # Convert to numpy array
        embeddings_array = np.array(embeddings)

        # Simple k-means clustering
        # In a real implementation, we would use a full sklearn KMeans
        # Here we'll do a simplified version

        # Initialize random centers
        np.random.seed(42)  # For reproducibility
        center_indices = np.random.choice(len(embeddings), n_clusters, replace=False)
        centers = embeddings_array[center_indices]

        # Run a few iterations of k-means
        for _ in range(5):
            # Assign to clusters
            distances = np.array([[np.linalg.norm(emb - center) for center in centers]
                                 for emb in embeddings_array])
            cluster_assignments = np.argmin(distances, axis=1)

            # Update centers
            for i in range(n_clusters):
                cluster_points = embeddings_array[cluster_assignments == i]
                if len(cluster_points) > 0:
                    centers[i] = np.mean(cluster_points, axis=0)

        # Find entities closest to final centers
        center_entities = []
        for center in centers:
            distances = [np.linalg.norm(emb - center) for emb in embeddings_array]
            closest_idx = np.argmin(distances)
            center_entities.append(valid_entities[closest_idx])

        return center_entities

    def adapt_weights(self, feedback: Dict[EvidenceSourceType, float]) -> None:
        """
        Adapt weights of different evidence sources based on feedback.

        Args:
            feedback: Dictionary mapping evidence sources to feedback scores (-1 to 1)
        """
        with self.lock:
            for source_type, score in feedback.items():
                if source_type in self.evidence_weights:
                    # Apply bounded update with diminishing returns for extreme values
                    current = self.evidence_weights[source_type]
                    delta = score * 0.1 * (1.0 - current if score > 0 else current)
                    self.evidence_weights[source_type] = max(0.1, min(1.0, current + delta))

            self.logger.info(f"Updated evidence weights: {self.evidence_weights}")

    def get_activation(self) -> float:
        """
        Get the current activation level of the neural reasoner.

        Returns:
            Activation level between 0 and 1, indicating recent usage
        """
        time_since_active = time.time() - self.last_activation_time
        # Exponential decay of activation
        return np.exp(-time_since_active / 3600)  # Half-life of about 1 hour

    def clear_cache(self) -> None:
        """Clear the embedding cache"""
        with self.lock:
            self.embedding_cache.clear()
            self.logger.info("Embedding cache cleared")

    # ------------------------------------------------------------------------
    # Internal methods
    # ------------------------------------------------------------------------

    def _register_with_nexus(self) -> None:
        """Register with NexusCore for coordination"""
        try:
            # Register as a module
            dependencies = []
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.graph_engine:
                dependencies.append("graph_engine")

            self.nexus_core.register_module(
                name="neural_reasoner",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _get_embedding(self, entity: Any) -> Optional[np.ndarray]:
        """
        Get embedding for an entity. The entity can be:
        - A text string
        - A concept ID from semantic memory
        - A pre-computed embedding

        If caching is enabled, will check cache first.

        Args:
            entity: The entity to get embedding for

        Returns:
            Embedding vector or None if unavailable
        """
        # Check if input is already an embedding
        if isinstance(entity, np.ndarray) and entity.ndim == 1:
            return entity / np.linalg.norm(entity)  # Normalize

        # Generate cache key
        cache_key = str(entity)

        # Check cache first
        if cache_key in self.embedding_cache:
            return self.embedding_cache[cache_key]

        # Try to get embedding based on entity type
        embedding = None

        if isinstance(entity, str):
            # Check if it's a concept ID in semantic memory
            if self.semantic_memory:
                try:
                    concept = self.semantic_memory.get_concept(entity)
                    if concept and concept.embedding is not None:
                        embedding = concept.embedding
                except Exception as e:
                    self.logger.debug(f"Error retrieving concept {entity}: {str(e)}")

            # If not found, treat as text and get embedding
            if embedding is None and self.use_llm_embeddings:
                embedding = self._get_text_embedding(entity)

        # Store in cache if valid
        if embedding is not None:
            # Normalize embedding
            embedding = embedding / np.linalg.norm(embedding)

            # Add to cache
            self.embedding_cache[cache_key] = embedding

            # Manage cache size
            if len(self.embedding_cache) > self.max_cache_size:
                # Simple strategy: remove a random key
                keys = list(self.embedding_cache.keys())
                key_to_remove = keys[np.random.randint(0, len(keys))]
                del self.embedding_cache[key_to_remove]

        return embedding

    def _get_text_embedding(self, text: str) -> Optional[np.ndarray]:
        """
        Get embedding for a text string using LLM or other embedding sources.

        Args:
            text: Text to get embedding for

        Returns:
            Embedding vector or None if unavailable
        """
        # In a real implementation, this would call an embedding model or service
        # For example, using OpenAI or a local model via HuggingFace

        # Simplified mock implementation for demonstration purposes
        try:
            # Check if we have access to an LLM interface through NexusCore
            if self.nexus_core:
                llm_interface = self.nexus_core.get_module("llm_interface")
                if llm_interface and hasattr(llm_interface, "create_embedding"):
                    self.logger.debug(f"Getting embedding for text via llm_interface")
                    return llm_interface.create_embedding(text)

            # Fallback to a mock embedding (random but deterministic based on input)
            self.logger.debug(f"Generating mock embedding for text")
            # Create a deterministic but unique hash from the text
            np.random.seed(hash(text) % 2**32)
            embedding = np.random.randn(self.embedding_dim)
            return embedding / np.linalg.norm(embedding)  # Normalize

        except Exception as e:
            self.logger.warning(f"Error generating embedding for text: {str(e)}")
            return None

    # Similarity metrics

    def _cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Compute cosine similarity between two vectors.
        Assumes vectors are already normalized.

        Args:
            vec1, vec2: Vectors to compare

        Returns:
            Cosine similarity score (0-1)
        """
        # For normalized vectors, cosine similarity = dot product
        similarity = np.dot(vec1, vec2)

        # Clamp to -1..1 range (in case of numerical issues)
        similarity = max(-1.0, min(1.0, similarity))

        # Map from -1..1 to 0..1
        return (similarity + 1) / 2

    def _dot_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Compute dot product similarity, scaled to 0-1 range.

        Args:
            vec1, vec2: Vectors to compare

        Returns:
            Normalized similarity score (0-1)
        """
        # For normalized vectors, we can map from -1..1 to 0..1
        raw_dot = np.dot(vec1, vec2)
        return (raw_dot + 1) / 2

    def _euclidean_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Compute similarity based on Euclidean distance.

        Args:
            vec1, vec2: Vectors to compare

        Returns:
            Similarity score (0-1), where 1 = identical
        """
        distance = np.linalg.norm(vec1 - vec2)
        # Convert distance to similarity: 0 distance = 1 similarity, large distance = 0 similarity
        return 1 / (1 + distance)

    # Task processing methods

    def _process_similarity_task(self, request: NeuralReasoningRequest,
                                result: NeuralReasoningResult) -> None:
        """
        Process similarity computation between entities.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        if not isinstance(request.query, list) or len(request.query) != 2:
            raise ValueError("Similarity task requires query to be a list of two items")

        entity1, entity2 = request.query

        # Get embeddings if not provided
        emb1 = request.query_embedding if request.query_embedding is not None else self._get_embedding(entity1)
        emb2 = self._get_embedding(entity2)

        if emb1 is None or emb2 is None:
            raise ValueError("Could not get embeddings for similarity computation")

        # Get similarity method from parameters or use default
        method = request.parameters.get("similarity_method", "cosine")

        # Compute similarity
        if method == "cosine":
            similarity = self._cosine_similarity(emb1, emb2)
        elif method == "dot":
            similarity = self._dot_similarity(emb1, emb2)
        elif method == "euclidean":
            similarity = self._euclidean_similarity(emb1, emb2)
        else:
            self.logger.warning(f"Unknown similarity method: {method}, using cosine")
            similarity = self._cosine_similarity(emb1, emb2)

        # Create evidence
        evidence = NeuralEvidence(
            source_type=EvidenceSourceType.EMBEDDING,
            content=f"Similarity between '{entity1}' and '{entity2}': {similarity:.3f}",
            confidence=0.9,  # High confidence for direct computation
            relevance=1.0,
            embedding=None,
            metadata={
                "entity1": entity1,
                "entity2": entity2,
                "similarity": similarity,
                "method": method
            }
        )

        # Populate result
        result.conclusion = similarity
        result.confidence = 0.9  # High confidence for direct computation
        result.evidence = [evidence]
        result.explanation = f"Computed {method} similarity between '{entity1}' and '{entity2}'"

    def _process_classification_task(self, request: NeuralReasoningRequest,
                                    result: NeuralReasoningResult) -> None:
        """
        Process classification of entities into categories.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract query and ensure it's the right structure
        if not isinstance(request.query, dict) or "entity" not in request.query or "categories" not in request.query:
            raise ValueError("Classification task requires query with 'entity' and 'categories'")

        entity = request.query["entity"]
        categories = request.query["categories"]

        if not isinstance(categories, list) or len(categories) < 2:
            raise ValueError("Categories must be a list with at least 2 items")

        # Get embedding for entity
        entity_emb = request.query_embedding if request.query_embedding is not None else self._get_embedding(entity)

        if entity_emb is None:
            raise ValueError(f"Could not get embedding for entity: {entity}")

        # Compute similarity with each category
        similarities = []
        category_embeddings = {}

        for category in categories:
            category_emb = self._get_embedding(category)
            if category_emb is not None:
                sim = self._cosine_similarity(entity_emb, category_emb)
                similarities.append((category, sim))
                category_embeddings[category] = category_emb

        if not similarities:
            raise ValueError("Could not compute similarities with any category")

        # Sort by similarity
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Apply any threshold from parameters
        threshold = request.parameters.get("classification_threshold", 0.5)

        # Filter by threshold
        valid_categories = [(cat, sim) for cat, sim in similarities if sim >= threshold]

        # For each valid category, add an evidence
        evidence_list = []
        for category, similarity in valid_categories:
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.EMBEDDING,
                content=f"Entity '{entity}' matches category '{category}' with score {similarity:.3f}",
                confidence=similarity,
                relevance=1.0,
                metadata={
                    "entity": entity,
                    "category": category,
                    "similarity": similarity,
                    "threshold": threshold
                }
            )
            evidence_list.append(evidence)

        # Compute overall confidence based on contrast between top categories
        if len(similarities) >= 2:
            top_sim = similarities[0][1]
            second_sim = similarities[1][1]
            contrast = (top_sim - second_sim) / top_sim if top_sim > 0 else 0
            confidence = top_sim * (0.5 + 0.5 * contrast)  # Boost confidence if clear contrast
        else:
            confidence = similarities[0][1] if similarities else 0.0

        # Populate result
        best_category = similarities[0][0] if similarities else None
        result.conclusion = best_category
        result.confidence = confidence
        result.evidence = evidence_list
        result.embedding = entity_emb

        # Build explanation
        if valid_categories:
            if len(valid_categories) == 1:
                result.explanation = f"Entity '{entity}' classified as '{best_category}' with confidence {confidence:.3f}"
            else:
                categories_str = ", ".join([f"'{cat}' ({sim:.3f})" for cat, sim in valid_categories[:3]])
                result.explanation = f"Entity '{entity}' matches multiple categories: {categories_str}"
        else:
            result.explanation = f"Entity '{entity}' did not match any category above threshold {threshold}"

    def _process_clustering_task(self, request: NeuralReasoningRequest,
                                result: NeuralReasoningResult) -> None:
        """
        Process clustering of entities.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract query and ensure it's the right structure
        if not isinstance(request.query, list):
            raise ValueError("Clustering task requires query to be a list of entities")

        entities = request.query

        if len(entities) < 2:
            raise ValueError("Need at least 2 entities for clustering")

        # Get number of clusters from parameters or derive from data size
        n_clusters = request.parameters.get("n_clusters")
        if n_clusters is None:
            # Heuristic: sqrt of number of entities, bounded
            n_clusters = min(max(2, int(np.sqrt(len(entities)))), 10)

        # Get embeddings for all entities
        entity_embeddings = []
        valid_entities = []

        for entity in entities:
            emb = self._get_embedding(entity)
            if emb is not None:
                entity_embeddings.append(emb)
                valid_entities.append(entity)

        if len(valid_entities) < n_clusters:
            n_clusters = len(valid_entities)
            self.logger.warning(f"Reduced number of clusters to {n_clusters} due to insufficient valid entities")

        if len(valid_entities) < 2:
            raise ValueError("Need at least 2 valid entities with embeddings for clustering")

        # Convert to numpy array for clustering
        embeddings_array = np.array(entity_embeddings)

        # Simple k-means clustering
        # In a real implementation, we would use a full sklearn KMeans
        # Here we'll do a simplified version

        # Initialize random centers
        np.random.seed(42)  # For reproducibility
        center_indices = np.random.choice(len(embeddings_array), n_clusters, replace=False)
        centers = embeddings_array[center_indices].copy()

        # Run a few iterations of k-means
        for _ in range(5):
            # Assign to clusters
            distances = np.array([[np.linalg.norm(emb - center) for center in centers]
                                for emb in embeddings_array])
            cluster_assignments = np.argmin(distances, axis=1)

            # Update centers
            for i in range(n_clusters):
                cluster_points = embeddings_array[cluster_assignments == i]
                if len(cluster_points) > 0:
                    centers[i] = np.mean(cluster_points, axis=0)

        # Collect results
        clusters = [[] for _ in range(n_clusters)]
        for idx, cluster_idx in enumerate(cluster_assignments):
            clusters[cluster_idx].append(valid_entities[idx])

        # Find entities closest to final centers (cluster representatives)
        representatives = []
        for i, center in enumerate(centers):
            if len(clusters[i]) > 0:  # Ensure cluster is not empty
                distances = [np.linalg.norm(embeddings_array[j] - center)
                             for j, entity in enumerate(valid_entities)
                             if cluster_assignments[j] == i]
                closest_idx = np.argmin(distances)
                # Find the entity in this cluster with this index
                cluster_entities = [valid_entities[j] for j, ca in enumerate(cluster_assignments) if ca == i]
                representatives.append(cluster_entities[closest_idx])
            else:
                representatives.append(None)

        # Create evidence for each cluster
        evidence_list = []
        for i, (cluster, representative) in enumerate(zip(clusters, representatives)):
            if representative is not None:
                evidence = NeuralEvidence(
                    source_type=EvidenceSourceType.PATTERN,
                    content=f"Cluster {i+1} with {len(cluster)} entities, representative: '{representative}'",
                    confidence=0.8,  # Fixed confidence for clustering
                    relevance=1.0,
                    metadata={
                        "cluster_index": i,
                        "cluster_size": len(cluster),
                        "cluster_entities": cluster,
                        "representative": representative
                    }
                )
                evidence_list.append(evidence)

        # Compute overall quality metric (simplified)
        # In a real implementation, we would use proper clustering metrics like silhouette score
        # Here we'll use a simple heuristic: average distance to cluster center
        intra_cluster_distances = []
        for i in range(n_clusters):
            cluster_points = embeddings_array[cluster_assignments == i]
            if len(cluster_points) > 0:
                distances = [np.linalg.norm(point - centers[i]) for point in cluster_points]
                intra_cluster_distances.extend(distances)

        avg_distance = np.mean(intra_cluster_distances) if intra_cluster_distances else 1.0
        quality_score = 1.0 / (1.0 + avg_distance)  # Convert to 0-1 score, higher is better

        # Populate result
        result.conclusion = {
            "clusters": clusters,
            "representatives": representatives,
            "num_clusters": n_clusters,
            "quality_score": quality_score
        }
        result.confidence = quality_score
        result.evidence = evidence_list

        # Build explanation
        result.explanation = f"Clustered {len(valid_entities)} entities into {n_clusters} groups with quality score {quality_score:.3f}"

    def _process_completion_task(self, request: NeuralReasoningRequest,
                                result: NeuralReasoningResult) -> None:
        """
        Process pattern completion task.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract query and ensure it's the right structure
        if not isinstance(request.query, dict) or "pattern" not in request.query or "options" not in request.query:
            raise ValueError("Completion task requires query with 'pattern' and 'options'")

        pattern = request.query["pattern"]
        options = request.query["options"]

        if not isinstance(pattern, list) or len(pattern) < 2:
            raise ValueError("Pattern must be a list with at least 2 items")

        if not isinstance(options, list) or len(options) < 1:
            raise ValueError("Options must be a list with at least 1 item")

        # Convert pattern to embeddings
        pattern_embeddings = []
        valid_pattern_items = []

        for item in pattern:
            emb = self._get_embedding(item)
            if emb is not None:
                pattern_embeddings.append(emb)
                valid_pattern_items.append(item)

        if len(pattern_embeddings) < 2:
            raise ValueError("Need at least 2 valid items with embeddings in the pattern")

        # Determine completion type from parameters
        completion_type = request.parameters.get("completion_type", "analogy")

        if completion_type == "analogy" and len(pattern_embeddings) >= 3:
            # A:B::C:? style analogy
            a, b, c = pattern_embeddings[-3:]
            target = c + (b - a)
            # Normalize
            target = target / np.linalg.norm(target)

            # Find closest option
            best_option = None
            best_similarity = -1.0

            for option in options:
                option_emb = self._get_embedding(option)
                if option_emb is not None:
                    sim = self._cosine_similarity(target, option_emb)
                    if sim > best_similarity:
                        best_similarity = sim
                        best_option = option

            if best_option is None:
                raise ValueError("Could not find a valid completion option")

            # Create evidence
            a_item, b_item, c_item = valid_pattern_items[-3:]
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.ANALOGY,
                content=f"Analogy completion: '{a_item}' is to '{b_item}' as '{c_item}' is to '{best_option}'",
                confidence=best_similarity,
                relevance=1.0,
                metadata={
                    "a": a_item,
                    "b": b_item,
                    "c": c_item,
                    "d": best_option,
                    "similarity": best_similarity,
                    "completion_type": "analogy"
                }
            )

            # Populate result
            result.conclusion = best_option
            result.confidence = best_similarity
            result.evidence = [evidence]
            result.explanation = f"Completed analogy pattern using vector arithmetic: '{a_item}' is to '{b_item}' as '{c_item}' is to '{best_option}' (confidence: {best_similarity:.3f})"

        else:
            # Simpler pattern continuation using average similarity
            similarities = []
            for option in options:
                option_emb = self._get_embedding(option)
                if option_emb is not None:
                    # Compute average similarity to pattern items
                    avg_sim = sum(self._cosine_similarity(option_emb, pattern_emb)
                                for pattern_emb in pattern_embeddings) / len(pattern_embeddings)
                    similarities.append((option, avg_sim))

            if not similarities:
                raise ValueError("Could not find a valid completion option")

            # Get best match
            similarities.sort(key=lambda x: x[1], reverse=True)
            best_option, best_similarity = similarities[0]

            # Create evidence
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.PATTERN,
                content=f"Pattern completion: '{best_option}' continues pattern with similarity {best_similarity:.3f}",
                confidence=best_similarity,
                relevance=1.0,
                metadata={
                    "pattern_items": valid_pattern_items,
                    "completion": best_option,
                    "similarity": best_similarity,
                    "completion_type": "continuation"
                }
            )

            # Populate result
            result.conclusion = best_option
            result.confidence = best_similarity
            result.evidence = [evidence]
            result.explanation = f"Completed pattern using average similarity, adding '{best_option}' (confidence: {best_similarity:.3f})"

    def _process_retrieval_task(self, request: NeuralReasoningRequest,
                               result: NeuralReasoningResult) -> None:
        """
        Process retrieval of similar entities.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Get query embedding
        if request.query_embedding is not None:
            query_emb = request.query_embedding
        elif request.query is not None:
            query_emb = self._get_embedding(request.query)
        else:
            raise ValueError("Retrieval task requires query or query_embedding")

        if query_emb is None:
            raise ValueError("Could not get embedding for query")

        # Get parameters
        top_k = request.parameters.get("top_k", 5)
        similarity_threshold = request.parameters.get("similarity_threshold", self.similarity_threshold)

        # Use different evidence sources based on request
        evidence_sources = request.evidence_sources or [EvidenceSourceType.RETRIEVAL]

        all_results = []

        # Use semantic memory if available and requested
        if self.semantic_memory is not None and EvidenceSourceType.RETRIEVAL in evidence_sources:
            try:
                # Try to query semantic memory with embedding
                query = {
                    "semantic_query": request.query if isinstance(request.query, str) else None,
                    "embedding": query_emb,
                    "max_results": top_k
                }

                semantic_results = self.semantic_memory.query_concepts(query)

                for concept in semantic_results:
                    # Calculate similarity if not provided
                    if hasattr(concept, 'embedding') and concept.embedding is not None:
                        similarity = self._cosine_similarity(query_emb, concept.embedding)
                    else:
                        similarity = 0.5  # Default if no embedding

                    if similarity >= similarity_threshold:
                        # Create evidence
                        evidence = NeuralEvidence(
                            source_type=EvidenceSourceType.RETRIEVAL,
                            content=concept,
                            confidence=similarity,
                            relevance=similarity,
                            embedding=concept.embedding if hasattr(concept, 'embedding') else None,
                            metadata={
                                "concept_id": concept.concept_id if hasattr(concept, 'concept_id') else None,
                                "name": concept.name if hasattr(concept, 'name') else str(concept),
                                "similarity": similarity,
                                "source": "semantic_memory"
                            }
                        )
                        all_results.append((evidence, similarity))
            except Exception as e:
                self.logger.warning(f"Error retrieving from semantic memory: {str(e)}")

        # Use graph engine if available and requested
        if self.graph_engine is not None and EvidenceSourceType.GRAPH in evidence_sources:
            try:
                # Try to find similar nodes in the graph
                node_criteria = {}
                if isinstance(request.query, str):
                    node_criteria = {"label": request.query}  # Simple text match

                # For a real implementation, this would use embedding-based graph search
                # Here we use a simplified approach
                node_ids = self.graph_engine.query_nodes(node_criteria)

                for node_id in node_ids[:top_k]:
                    try:
                        node = self.graph_engine.get_node(node_id)

                        # Calculate similarity if node has embedding
                        if hasattr(node, 'embedding') and node.embedding is not None:
                            similarity = self._cosine_similarity(query_emb, node.embedding)
                        else:
                            similarity = 0.4  # Lower default for graph match without embedding

                        if similarity >= similarity_threshold:
                            # Create evidence
                            evidence = NeuralEvidence(
                                source_type=EvidenceSourceType.GRAPH,
                                content=node,
                                confidence=similarity,
                                relevance=similarity,
                                embedding=node.embedding if hasattr(node, 'embedding') else None,
                                metadata={
                                    "node_id": node_id,
                                    "label": node.label if hasattr(node, 'label') else str(node),
                                    "similarity": similarity,
                                    "source": "graph_engine"
                                }
                            )
                            all_results.append((evidence, similarity))
                    except Exception as e:
                        self.logger.debug(f"Error retrieving node {node_id}: {str(e)}")
            except Exception as e:
                self.logger.warning(f"Error retrieving from graph engine: {str(e)}")

        # Use episodic memory if available and requested
        if self.episodic_memory is not None and EvidenceSourceType.RETRIEVAL in evidence_sources:
            try:
                # Here we would query episodic memory with the embedding
                # This is a simplified placeholder - real implementation would depend on the episodic memory interface

                # Add episodic memory results to all_results
                pass
            except Exception as e:
                self.logger.warning(f"Error retrieving from episodic memory: {str(e)}")

        # Sort all results by weighted similarity
        all_results.sort(key=lambda x: x[1], reverse=True)

        # Take top results
        top_results = all_results[:top_k]

        if not top_results:
            # No results found
            result.conclusion = []
            result.confidence = 0.0
            result.evidence = []
            result.explanation = "No items retrieved matching the query"
            return

        # Extract evidence and items
        evidence_list = [item[0] for item in top_results]
        retrieved_items = [e.content for e in evidence_list]

        # Weighted average confidence based on top matches
        confidence = sum(item[1] for item in top_results) / len(top_results)

        # Populate result
        result.conclusion = retrieved_items
        result.confidence = confidence
        result.evidence = evidence_list
        result.embedding = query_emb

        # Build explanation
        if isinstance(request.query, str):
            query_str = f"'{request.query}'"
        else:
            query_str = "the provided query"

        top_matches_str = ", ".join([
            f"'{e.metadata.get('name', str(e.content))}' ({e.confidence:.2f})"
            for e in evidence_list[:3]
        ])

        result.explanation = f"Retrieved {len(retrieved_items)} items similar to {query_str}. Top matches: {top_matches_str}"

    def _process_inference_task(self, request: NeuralReasoningRequest,
                               result: NeuralReasoningResult) -> None:
        """
        Process inference from neural evidence.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract inference type from parameters
        inference_type = request.parameters.get("inference_type", "property")

        # Handle different types of inferences
        if inference_type == "property":
            self._process_property_inference(request, result)
        elif inference_type == "relation":
            self._process_relation_inference(request, result)
        elif inference_type == "category":
            self._process_category_inference(request, result)
        else:
            raise ValueError(f"Unknown inference type: {inference_type}")

    def _process_property_inference(self, request: NeuralReasoningRequest,
                                   result: NeuralReasoningResult) -> None:
        """
        Infer properties of an entity based on similar entities.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract query entity
        if not isinstance(request.query, dict) or "entity" not in request.query:
            raise ValueError("Property inference requires query with 'entity'")

        entity = request.query["entity"]
        properties_to_infer = request.query.get("properties", [])

        # Get entity embedding
        entity_emb = request.query_embedding if request.query_embedding is not None else self._get_embedding(entity)

        if entity_emb is None:
            raise ValueError(f"Could not get embedding for entity: {entity}")

        # Find similar entities in semantic memory
        similar_entities = []
        properties_counts = defaultdict(int)
        properties_confidence = defaultdict(float)
        evidence_list = []

        # Use semantic memory if available
        if self.semantic_memory is not None:
            try:
                query = {
                    "semantic_query": entity if isinstance(entity, str) else None,
                    "embedding": entity_emb,
                    "max_results": 10
                }

                similar_concepts = self.semantic_memory.query_concepts(query)

                for concept in similar_concepts:
                    # Skip if it's the same entity
                    if hasattr(concept, 'concept_id') and str(concept.concept_id) == str(entity):
                        continue

                    # Calculate similarity
                    if hasattr(concept, 'embedding') and concept.embedding is not None:
                        similarity = self._cosine_similarity(entity_emb, concept.embedding)
                    else:
                        similarity = 0.5  # Default

                    # Only consider sufficiently similar entities
                    if similarity >= self.similarity_threshold:
                        similar_entities.append((concept, similarity))

                        # Extract properties from concept
                        if hasattr(concept, 'properties'):
                            for prop_name, prop_value in concept.properties.items():
                                if not properties_to_infer or prop_name in properties_to_infer:
                                    properties_counts[prop_name] += 1
                                    properties_confidence[prop_name] += similarity

                        # Create evidence
                        evidence = NeuralEvidence(
                            source_type=EvidenceSourceType.RETRIEVAL,
                            content=f"Similar entity: {concept.name if hasattr(concept, 'name') else str(concept)}",
                            confidence=similarity,
                            relevance=similarity,
                            metadata={
                                "entity": concept.concept_id if hasattr(concept, 'concept_id') else str(concept),
                                "similarity": similarity,
                                "properties": concept.properties if hasattr(concept, 'properties') else {}
                            }
                        )
                        evidence_list.append(evidence)
            except Exception as e:
                self.logger.warning(f"Error retrieving similar entities: {str(e)}")

        # Now infer properties based on what we found
        inferred_properties = {}

        for prop_name, count in properties_counts.items():
            if count >= 2:  # Require at least 2 similar entities to have this property
                confidence = properties_confidence[prop_name] / count
                inferred_properties[prop_name] = {
                    'confidence': confidence,
                    'support': count
                }

        if not inferred_properties:
            # No properties inferred
            result.conclusion = {}
            result.confidence = 0.0
            result.evidence = evidence_list
            result.explanation = f"Could not infer any properties for '{entity}' from similar entities"
            return

        # Add inferred properties evidence
        for prop_name, info in inferred_properties.items():
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.INFERENCE,
                content=f"Inferred property: {prop_name} for '{entity}'",
                confidence=info['confidence'],
                relevance=1.0,
                metadata={
                    "property": prop_name,
                    "confidence": info['confidence'],
                    "support": info['support'],
                    "inference_type": "property"
                }
            )
            evidence_list.append(evidence)

        # Compute overall confidence (average of property confidences)
        confidence = sum(info['confidence'] for info in inferred_properties.values()) / len(inferred_properties)

        # Populate result
        result.conclusion = inferred_properties
        result.confidence = confidence
        result.evidence = evidence_list
        result.embedding = entity_emb

        # Build explanation
        props_str = ", ".join([f"'{prop}' ({info['confidence']:.2f})"
                              for prop, info in list(inferred_properties.items())[:3]])

        result.explanation = f"Inferred {len(inferred_properties)} properties for '{entity}'. Top properties: {props_str}"

    def _process_relation_inference(self, request: NeuralReasoningRequest,
                                   result: NeuralReasoningResult) -> None:
        """
        Infer relations between entities.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract source and target entities
        if not isinstance(request.query, dict) or "source" not in request.query or "target" not in request.query:
            raise ValueError("Relation inference requires query with 'source' and 'target'")

        source = request.query["source"]
        target = request.query["target"]

        # Get embeddings
        source_emb = self._get_embedding(source)
        target_emb = self._get_embedding(target)

        if source_emb is None or target_emb is None:
            raise ValueError(f"Could not get embeddings for entities")

        # Check for direct relation in graph if available
        direct_relations = []
        if self.graph_engine is not None:
            try:
                # Convert to node IDs if needed
                source_id = source
                target_id = target

                # Check for edges between source and target
                edges = self.graph_engine.get_node_neighbors(source_id)
                for edge_id, edge_type, neighbor_id in edges:
                    if neighbor_id == target_id:
                        direct_relations.append((edge_type, edge_id))
            except Exception as e:
                self.logger.debug(f"Error checking direct relations: {str(e)}")

        # Find similar entity pairs in the graph to infer likely relations
        similar_relations = defaultdict(float)
        relation_evidence = defaultdict(list)

        # Helper to find similar pairs
        def find_similar_pairs():
            # Find entities similar to source
            similar_sources = self._find_similar_entities(source, source_emb, 5)

            # Find entities similar to target
            similar_targets = self._find_similar_entities(target, target_emb, 5)

            # Check relations between similar entity pairs
            for sim_source, source_sim in similar_sources:
                for sim_target, target_sim in similar_targets:
                    # Skip if it's the original pair
                    if str(sim_source) == str(source) and str(sim_target) == str(target):
                        continue

                    # Check relations between this pair
                    if self.graph_engine is not None:
                        try:
                            # Get source ID
                            source_id = sim_source

                            # Check edges
                            edges = self.graph_engine.get_node_neighbors(source_id)
                            for edge_id, edge_type, neighbor_id in edges:
                                if neighbor_id == str(sim_target):
                                    # Weight by product of similarities
                                    weight = source_sim * target_sim
                                    similar_relations[edge_type] += weight

                                    # Add as evidence
                                    relation_evidence[edge_type].append({
                                        "source": sim_source,
                                        "target": sim_target,
                                        "source_sim": source_sim,
                                        "target_sim": target_sim,
                                        "weight": weight,
                                        "edge_id": edge_id
                                    })
                        except Exception as e:
                            self.logger.debug(f"Error checking relations between similar entities: {str(e)}")

        # Find similar pairs to infer relations
        find_similar_pairs()

        # Prepare evidence list
        evidence_list = []

        # Add direct relation evidence if any
        for edge_type, edge_id in direct_relations:
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.GRAPH,
                content=f"Direct relation '{edge_type}' exists between '{source}' and '{target}'",
                confidence=1.0,  # High confidence for direct relation
                relevance=1.0,
                metadata={
                    "source": source,
                    "target": target,
                    "relation": edge_type,
                    "edge_id": edge_id,
                    "direct": True
                }
            )
            evidence_list.append(evidence)

        # Add similar relation evidence
        for relation_type, weight in similar_relations.items():
            # Normalize weight to 0-1 based on number of supporting pairs
            support_count = len(relation_evidence[relation_type])
            confidence = min(0.9, weight / support_count) if support_count > 0 else 0.0

            # Only include if confidence above threshold
            if confidence >= 0.3:  # Minimum threshold for relations
                # Get supporting examples (up to 3)
                examples = relation_evidence[relation_type][:3]
                examples_str = ", ".join([
                    f"'{ex['source']}' -> '{ex['target']}'" for ex in examples
                ])

                evidence = NeuralEvidence(
                    source_type=EvidenceSourceType.INFERENCE,
                    content=f"Inferred relation '{relation_type}' between '{source}' and '{target}' based on similar pairs",
                    confidence=confidence,
                    relevance=1.0,
                    metadata={
                        "source": source,
                        "target": target,
                        "relation": relation_type,
                        "support_count": support_count,
                        "weight": weight,
                        "examples": examples
                    }
                )
                evidence_list.append(evidence)

        # Combine direct and inferred relations
        inferred_relations = {}

        # First add direct relations with highest confidence
        for edge_type, _ in direct_relations:
            inferred_relations[edge_type] = {
                'confidence': 1.0,
                'direct': True
            }

        # Then add inferred relations if not already present
        for relation_type, weight in similar_relations.items():
            if relation_type not in inferred_relations:
                support_count = len(relation_evidence[relation_type])
                confidence = min(0.9, weight / support_count) if support_count > 0 else 0.0

                if confidence >= 0.3:  # Minimum threshold
                    inferred_relations[relation_type] = {
                        'confidence': confidence,
                        'direct': False,
                        'support_count': support_count
                    }

        if not inferred_relations:
            # No relations inferred
            result.conclusion = {}
            result.confidence = 0.0
            result.evidence = evidence_list
            result.explanation = f"Could not infer any relations between '{source}' and '{target}'"
            return

        # Compute overall confidence
        confidence = max(info['confidence'] for info in inferred_relations.values())

        # Populate result
        result.conclusion = inferred_relations
        result.confidence = confidence
        result.evidence = evidence_list

        # Build explanation
        if direct_relations:
            rel_types = [edge_type for edge_type, _ in direct_relations]
            rel_str = ", ".join([f"'{rel}'" for rel in rel_types])
            result.explanation = f"Found direct relation(s) {rel_str} between '{source}' and '{target}'"
        else:
            inferred_rels = [(rel, info['confidence']) for rel, info in inferred_relations.items()
                            if not info.get('direct', False)]
            inferred_rels.sort(key=lambda x: x[1], reverse=True)

            if inferred_rels:
                rel_str = ", ".join([f"'{rel}' ({conf:.2f})" for rel, conf in inferred_rels[:3]])
                result.explanation = f"Inferred relation(s) {rel_str} between '{source}' and '{target}'"
            else:
                result.explanation = f"No relations inferred between '{source}' and '{target}'"

    def _process_category_inference(self, request: NeuralReasoningRequest,
                                   result: NeuralReasoningResult) -> None:
        """
        Infer categories for an entity.

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract entity
        if not isinstance(request.query, dict) or "entity" not in request.query:
            raise ValueError("Category inference requires query with 'entity'")

        entity = request.query["entity"]

        # Get entity embedding
        entity_emb = request.query_embedding if request.query_embedding is not None else self._get_embedding(entity)

        if entity_emb is None:
            raise ValueError(f"Could not get embedding for entity: {entity}")

        # Find similar entities in semantic memory
        similar_entities = []
        categories_counts = defaultdict(int)
        categories_confidence = defaultdict(float)
        evidence_list = []

        # Use semantic memory if available
        if self.semantic_memory is not None:
            try:
                query = {
                    "semantic_query": entity if isinstance(entity, str) else None,
                    "embedding": entity_emb,
                    "max_results": 10
                }

                similar_concepts = self.semantic_memory.query_concepts(query)

                for concept in similar_concepts:
                    # Skip if it's the same entity
                    if hasattr(concept, 'concept_id') and str(concept.concept_id) == str(entity):
                        continue

                    # Calculate similarity
                    if hasattr(concept, 'embedding') and concept.embedding is not None:
                        similarity = self._cosine_similarity(entity_emb, concept.embedding)
                    else:
                        similarity = 0.5  # Default

                    # Only consider sufficiently similar entities
                    if similarity >= self.similarity_threshold:
                        similar_entities.append((concept, similarity))

                        # Extract categories from concept
                        if hasattr(concept, 'categories'):
                            for category in concept.categories:
                                categories_counts[category] += 1
                                categories_confidence[category] += similarity

                        # Also check for "is_a" relations in graph if available
                        if self.graph_engine is not None and hasattr(concept, 'concept_id'):
                            try:
                                edges = self.graph_engine.get_node_neighbors(concept.concept_id)
                                for _, edge_type, neighbor_id in edges:
                                    if edge_type == "is_a" or edge_type == "instance_of" or edge_type == "type":
                                        # Get the target node (category)
                                        category_node = self.graph_engine.get_node(neighbor_id)
                                        if category_node:
                                            category = category_node.label if hasattr(category_node, 'label') else str(neighbor_id)
                                            categories_counts[category] += 1
                                            categories_confidence[category] += similarity
                            except Exception as e:
                                self.logger.debug(f"Error checking graph relations: {str(e)}")

                        # Create evidence
                        evidence = NeuralEvidence(
                            source_type=EvidenceSourceType.RETRIEVAL,
                            content=f"Similar entity: {concept.name if hasattr(concept, 'name') else str(concept)}",
                            confidence=similarity,
                            relevance=similarity,
                            metadata={
                                "entity": concept.concept_id if hasattr(concept, 'concept_id') else str(concept),
                                "similarity": similarity,
                                "categories": concept.categories if hasattr(concept, 'categories') else []
                            }
                        )
                        evidence_list.append(evidence)
            except Exception as e:
                self.logger.warning(f"Error retrieving similar entities: {str(e)}")

        # Now infer categories based on what we found
        inferred_categories = {}

        for category, count in categories_counts.items():
            if count >= 2:  # Require at least 2 similar entities to have this category
                confidence = categories_confidence[category] / count
                inferred_categories[category] = {
                    'confidence': confidence,
                    'support': count
                }

        if not inferred_categories:
            # No categories inferred
            result.conclusion = {}
            result.confidence = 0.0
            result.evidence = evidence_list
            result.explanation = f"Could not infer any categories for '{entity}' from similar entities"
            return

        # Add inferred categories evidence
        for category, info in inferred_categories.items():
            evidence = NeuralEvidence(
                source_type=EvidenceSourceType.INFERENCE,
                content=f"Inferred category: {category} for '{entity}'",
                confidence=info['confidence'],
                relevance=1.0,
                metadata={
                    "category": category,
                    "confidence": info['confidence'],
                    "support": info['support'],
                    "inference_type": "category"
                }
            )
            evidence_list.append(evidence)

        # Compute overall confidence (average of category confidences)
        confidence = sum(info['confidence'] for info in inferred_categories.values()) / len(inferred_categories)

        # Populate result
        result.conclusion = inferred_categories
        result.confidence = confidence
        result.evidence = evidence_list
        result.embedding = entity_emb

        # Build explanation
        categories_str = ", ".join([f"'{cat}' ({info['confidence']:.2f})"
                                   for cat, info in list(inferred_categories.items())[:3]])

        result.explanation = f"Inferred {len(inferred_categories)} categories for '{entity}'. Top categories: {categories_str}"

    def _process_analogy_task(self, request: NeuralReasoningRequest,
                             result: NeuralReasoningResult) -> None:
        """
        Process analogy-based reasoning (A is to B as C is to ?).

        Args:
            request: The reasoning request
            result: Result object to populate
        """
        # Extract query and ensure it's the right structure
        if not isinstance(request.query, dict) or "a" not in request.query or "b" not in request.query or "c" not in request.query:
            raise ValueError("Analogy task requires query with 'a', 'b', and 'c'")

        # Extract analogy components
        a = request.query["a"]
        b = request.query["b"]
        c = request.query["c"]
        options = request.query.get("options", [])

        # Get embeddings
        a_emb = self._get_embedding(a)
        b_emb = self._get_embedding(b)
        c_emb = self._get_embedding(c)

        if a_emb is None or b_emb is None or c_emb is None:
            raise ValueError("Could not get embeddings for all analogy components")

        # Calculate target embedding using vector arithmetic
        # D = C + (B - A)
        target_emb = c_emb + (b_emb - a_emb)

        # Normalize
        target_emb = target_emb / np.linalg.norm(target_emb)

        # If options are provided, find the best match
        if options:
            similarities = []
            for option in options:
                option_emb = self._get_embedding(option)
                if option_emb is not None:
                    sim = self._cosine_similarity(target_emb, option_emb)
                    similarities.append((option, sim))

            if not similarities:
                raise ValueError("Could not compute embeddings for any options")

            # Sort by similarity
            similarities.sort(key=lambda x: x[1], reverse=True)
            best_match, similarity = similarities[0]

            # Add evidence for top matches
            evidence_list = []
            for option, sim in similarities[:3]:  # Top 3 matches
                evidence = NeuralEvidence(
                    source_type=EvidenceSourceType.ANALOGY,
                    content=f"Analogy match: '{option}' with similarity {sim:.3f}",
                    confidence=sim,
                    relevance=1.0,
                    metadata={
                        "a": a,
                        "b": b,
                        "c": c,
                        "d": option,
                        "similarity": sim
                    }
                )
                evidence_list.append(evidence)

            # Populate result
            result.conclusion = best_match
            result.confidence = similarity
            result.evidence = evidence_list
            result.embedding = target_emb

            # Build explanation
            result.explanation = f"Analogy '{a}' is to '{b}' as '{c}' is to '{best_match}' with confidence {similarity:.3f}"

        else:
            # If no options provided, find similar entities to target embedding
            similar_entities = self._find_similar_entities_to_embedding(target_emb, 5)

            if not similar_entities:
                result.conclusion = None
                result.confidence = 0.0
                result.evidence = []
                result.embedding = target_emb
                result.explanation = f"Could not find any entities similar to the target of analogy '{a}' is to '{b}' as '{c}' is to ?"
                return

            # Add evidence for similar entities
            evidence_list = []
            for entity, sim in similar_entities:
                evidence = NeuralEvidence(
                    source_type=EvidenceSourceType.ANALOGY,
                    content=f"Analogy completion: '{entity}' with similarity {sim:.3f}",
                    confidence=sim,
                    relevance=1.0,
                    metadata={
                        "a": a,
                        "b": b,
                        "c": c,
                        "d": entity,
                        "similarity": sim
                    }
                )
                evidence_list.append(evidence)

            # Best match is the highest similarity
            best_match, similarity = similar_entities[0]

            # Populate result
            result.conclusion = best_match
            result.confidence = similarity
            result.evidence = evidence_list
            result.embedding = target_emb

            # Build explanation
            result.explanation = f"Analogy '{a}' is to '{b}' as '{c}' is to '{best_match}' with confidence {similarity:.3f}"

    def _find_similar_entities(self, entity: Any, entity_emb: np.ndarray,
                              top_k: int = 5) -> List[Tuple[Any, float]]:
        """
        Find entities similar to the given entity.

        Args:
            entity: Entity to find similar entities for
            entity_emb: Embedding of the entity
            top_k: Number of top similar entities to return

        Returns:
            List of (entity, similarity) tuples
        """
        similar_entities = []

        # Use semantic memory if available
        if self.semantic_memory is not None:
            try:
                query = {
                    "semantic_query": entity if isinstance(entity, str) else None,
                    "embedding": entity_emb,
                    "max_results": top_k * 2  # Get more and then filter
                }

                similar_concepts = self.semantic_memory.query_concepts(query)

                for concept in similar_concepts:
                    # Skip if it's the same entity
                    if hasattr(concept, 'concept_id') and str(concept.concept_id) == str(entity):
                        continue

                    # Calculate similarity
                    if hasattr(concept, 'embedding') and concept.embedding is not None:
                        similarity = self._cosine_similarity(entity_emb, concept.embedding)
                    else:
                        similarity = 0.5  # Default

                    # Only consider sufficiently similar entities
                    if similarity >= self.similarity_threshold:
                        entity_id = concept.concept_id if hasattr(concept, 'concept_id') else str(concept)
                        similar_entities.append((entity_id, similarity))
            except Exception as e:
                self.logger.debug(f"Error finding similar entities in semantic memory: {str(e)}")

        # Use graph engine if available and needed
        if self.graph_engine is not None and len(similar_entities) < top_k:
            try:
                # Find similar nodes in the graph
                # This is a simplified approach - in a real implementation,
                # would use proper embedding-based similarity search

                # Assume entity is an ID or can be converted to string
                entity_str = str(entity)

                # Get properties of the entity if it's a node
                try:
                    entity_node = self.graph_engine.get_node(entity_str)
                    node_criteria = {}

                    if hasattr(entity_node, 'label'):
                        node_criteria["label"] = entity_node.label

                    # Find nodes with matching criteria
                    node_ids = self.graph_engine.query_nodes(node_criteria)

                    for node_id in node_ids:
                        # Skip if it's the same entity
                        if node_id == entity_str:
                            continue

                        try:
                            node = self.graph_engine.get_node(node_id)

                            # Calculate similarity if node has embedding
                            if hasattr(node, 'embedding') and node.embedding is not None:
                                similarity = self._cosine_similarity(entity_emb, node.embedding)

                                # Only consider sufficiently similar entities
                                if similarity >= self.similarity_threshold:
                                    similar_entities.append((node_id, similarity))
                        except Exception as e:
                            self.logger.debug(f"Error processing node {node_id}: {str(e)}")

                except Exception as e:
                    self.logger.debug(f"Entity not found in graph or error: {str(e)}")
            except Exception as e:
                self.logger.debug(f"Error finding similar entities in graph: {str(e)}")

        # Sort by similarity and return top_k
        similar_entities.sort(key=lambda x: x[1], reverse=True)
        return similar_entities[:top_k]

    def _find_similar_entities_to_embedding(self, embedding: np.ndarray,
                                          top_k: int = 5) -> List[Tuple[Any, float]]:
        """
        Find entities similar to the given embedding.

        Args:
            embedding: Embedding vector to find similar entities for
            top_k: Number of top similar entities to return

        Returns:
            List of (entity, similarity) tuples
        """
        similar_entities = []

        # Use semantic memory if available
        if self.semantic_memory is not None:
            try:
                query = {
                    "embedding": embedding,
                    "max_results": top_k
                }

                similar_concepts = self.semantic_memory.query_concepts(query)

                for concept in similar_concepts:
                    # Calculate similarity
                    if hasattr(concept, 'embedding') and concept.embedding is not None:
                        similarity = self._cosine_similarity(embedding, concept.embedding)
                    else:
                        similarity = 0.5  # Default

                    # Only consider sufficiently similar entities
                    if similarity >= self.similarity_threshold:
                        entity_id = concept.name if hasattr(concept, 'name') else str(concept)
                        similar_entities.append((entity_id, similarity))
            except Exception as e:
                self.logger.debug(f"Error finding similar entities in semantic memory: {str(e)}")

        # Sort by similarity and return top_k
        similar_entities.sort(key=lambda x: x[1], reverse=True)
        return similar_entities[:top_k]

# End of NeuralReasoner class

# Hybrid Inference

In [ ]:
"""
hybrid_inference.py - Raisonnement Neuro-Symbolique pour GrandNexus

Ce module implémente un moteur d'inférence hybride qui combine:
- Raisonnement symbolique (basé sur logique)
- Raisonnement neuronal (basé sur embeddings/réseaux)
- Frameworks neuro-symboliques (LNN, DeepProbLog ou équivalents)

Caractéristiques principales:
- Intégration bidirectionnelle entre logique formelle et représentations neuronales
- Capacité à manipuler l'incertitude et la confiance dans les inférences
- Flexibilité pour différents types de tâches cognitives
- Coopération avec les modules de mémoire et d'apprentissage
- Plusieurs stratégies d'inférence hybride pour s'adapter au contexte
"""

import logging
import threading
import time
import uuid
import math
from typing import Dict, List, Any, Optional, Callable, Union, Set, Tuple
from enum import Enum, auto
from dataclasses import dataclass, field
import numpy as np
from collections import defaultdict, deque
import random

# Imports from GrandNexus core and related modules
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory, Concept, Relation
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.graph.graph_engine import AbstractGraphEngine, GraphNode, GraphEdge
from grandnexus.graph.graph_engine import create_graph_engine, AbstractGraphEngine
from grandnexus.reasoning.reasoning_core import ReasoningCore
from grandnexus.reasoning.symbolic_engine import SymbolicEngine
from grandnexus.reasoning.neural_reasoner import NeuralReasoner
from grandnexus.llm_interface.llm_engine import LLMEngine

# Types d'inférence hybride
class InferenceType(Enum):
    """Types d'inférence hybride supportés par le moteur."""
    DEDUCTION = auto()         # Raisonnement déductif (A → B, A ⊢ B)
    INDUCTION = auto()         # Raisonnement inductif (généralisation)
    ABDUCTION = auto()         # Raisonnement abductif (hypothèse explicative)
    ANALOGY = auto()           # Raisonnement par analogie
    PROBABILISTIC = auto()     # Raisonnement probabiliste
    FUZZY = auto()             # Logique floue
    SPATIAL = auto()           # Raisonnement spatial
    TEMPORAL = auto()          # Raisonnement temporel
    CAUSAL = auto()            # Raisonnement causal
    LLM_GUIDED = auto()        # Inférence guidée par un LLM

# Mode d'opération du moteur hybride
class HybridMode(Enum):
    """Modes de fonctionnement du moteur d'inférence hybride."""
    SYMBOLIC_FIRST = auto()    # Commence par le raisonnement symbolique
    NEURAL_FIRST = auto()      # Commence par le raisonnement neuronal
    PARALLEL = auto()          # Les deux approches en parallèle, puis fusion
    ITERATIVE = auto()         # Alterne entre approches en plusieurs passes
    ADAPTIVE = auto()          # Choisit dynamiquement la meilleure approche
    LLM_ORCHESTRATED = auto()  # LLM dirige le flux de raisonnement

@dataclass
class SymbolicConstraint:
    """Représente une contrainte logique formelle."""
    constraint_id: str
    expression: str            # Représentation textuelle (ex: "A → B")
    variables: Set[str] = field(default_factory=set)
    weight: float = 1.0        # Poids/importance (0-1)
    confidence: float = 1.0    # Confiance (0-1)
    source: str = "user"       # Source de la contrainte

@dataclass
class InferenceQuery:
    """Requête d'inférence à traiter par le moteur hybride."""
    query_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    question: str = ""
    target_variables: Set[str] = field(default_factory=set)
    constraints: List[SymbolicConstraint] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)
    known_facts: Dict[str, Any] = field(default_factory=dict)
    inference_types: List[InferenceType] = field(default_factory=list)
    hybrid_mode: HybridMode = HybridMode.ADAPTIVE
    time_limit: Optional[float] = None
    max_inference_depth: int = 5
    use_llm: bool = True       # Utiliser ou non l'assistance d'un LLM

@dataclass
class InferenceStep:
    """Une étape unique dans un processus d'inférence."""
    step_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    type: str = ""             # Type d'étape (symbolique, neurale, etc.)
    description: str = ""
    input_facts: Dict[str, Any] = field(default_factory=dict)
    output_facts: Dict[str, Any] = field(default_factory=dict)
    confidence: float = 1.0
    reasoning: str = ""        # Explication textuelle du raisonnement
    timestamp: float = field(default_factory=time.time)

@dataclass
class InferenceResult:
    """Résultat complet d'un processus d'inférence."""
    query_id: str
    success: bool = False
    answers: Dict[str, Any] = field(default_factory=dict)
    confidence: float = 0.0
    steps: List[InferenceStep] = field(default_factory=list)
    explanation: str = ""
    execution_time: float = 0.0
    symbolic_contribution: float = 0.0  # 0-1 proportion de contribution
    neural_contribution: float = 0.0    # 0-1 proportion de contribution
    llm_contribution: float = 0.0       # 0-1 proportion de contribution

class HybridInferenceEngine:
    """
    Moteur d'inférence neuro-symbolique pour GrandNexus.

    Ce moteur intègre des approches symboliques et neuronales pour le raisonnement,
    permettant une inférence plus robuste, explicable et flexible que les approches
    purement symboliques ou purement neuronales.
    """

    def __init__(self,
                 reasoning_core: Optional[ReasoningCore] = None,
                 symbolic_engine: Optional[SymbolicEngine] = None,
                 neural_reasoner: Optional[NeuralReasoner] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 graph_engine: Optional[AbstractGraphEngine] = None,
                 llm_engine: Optional[LLMEngine] = None,
                 nexus_core: Optional[NexusCore] = None):
        """
        Initialise le moteur d'inférence hybride.

        Args:
            reasoning_core: Référence au module de raisonnement principal
            symbolic_engine: Moteur de raisonnement symbolique
            neural_reasoner: Moteur de raisonnement neuronal
            semantic_memory: Mémoire sémantique du système
            working_memory: Mémoire de travail du système
            episodic_memory: Mémoire épisodique du système
            graph_engine: Moteur de graphe pour navigation de connaissances
            llm_engine: Interface avec un modèle de langage
            nexus_core: Référence au noyau central de GrandNexus
        """
        self.logger = logging.getLogger("GrandNexus.Reasoning.HybridInference")

        # Références aux autres modules
        self.reasoning_core = reasoning_core
        self.symbolic_engine = symbolic_engine
        self.neural_reasoner = neural_reasoner
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory
        self.episodic_memory = episodic_memory
        self.graph_engine = graph_engine
        self.llm_engine = llm_engine
        self.nexus_core = nexus_core

        # État interne
        self.instance_id = str(uuid.uuid4())[:8]
        self.lock = threading.RLock()
        self.running = False
        self.initialized = False

        # Registre des stratégies d'inférence
        self.inference_strategies = {}

        # Cache de résultats récents
        self.results_cache = {}
        self.cache_expiry = 3600  # 1 heure

        # Métriques de performance
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        # Règles symboliques et représentations neuronales
        self.symbolic_rules = []
        self.neural_models = {}

        # Compteurs d'utilisation pour ajustement adaptatif
        self.strategy_usage_counts = defaultdict(int)
        self.strategy_success_rates = defaultdict(lambda: (0, 0))  # (success, total)

        # Paramètres adaptatifs
        self.params = {
            "symbolic_weight": 0.5,      # Poids accordé à l'inférence symbolique
            "neural_weight": 0.5,        # Poids accordé à l'inférence neurale
            "confidence_threshold": 0.7, # Seuil minimal de confiance
            "max_branching_factor": 5,   # Facteur de branchement maximum en recherche
            "use_approximate": False,    # Permettre des inférences approximatives
            "combine_method": "weighted" # Comment combiner inférences multiples
        }

        self.logger.info(f"HybridInferenceEngine initialisé avec ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialise les composants du moteur d'inférence hybride."""
        if self.initialized:
            return True

        try:
            with self.lock:
                # Enregistrer les stratégies d'inférence par défaut
                self._register_default_strategies()

                # Configurer les interfaces avec les autres modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Initialiser les modèles neuro-symboliques
                self._initialize_neuro_symbolic_models()

                self.initialized = True
                self.logger.info("HybridInferenceEngine initialization complete")
                return True
        except Exception as e:
            self.logger.error(f"Échec de l'initialisation: {str(e)}")
            return False

    def start(self) -> bool:
        """Démarre le moteur d'inférence hybride."""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            self.running = True
            self.logger.info("HybridInferenceEngine démarré")
            return True

    def stop(self) -> bool:
        """Arrête le moteur d'inférence hybride."""
        with self.lock:
            if not self.running:
                return True

            self.running = False
            self.logger.info("HybridInferenceEngine arrêté")
            return True

    def process_query(self, query: InferenceQuery) -> InferenceResult:
        """
        Traite une requête d'inférence en utilisant le mode hybride spécifié.

        Args:
            query: La requête d'inférence à traiter

        Returns:
            InferenceResult: Le résultat de l'inférence avec explications
        """
        if not self.running:
            self.logger.warning("Tentative d'utilisation du moteur non démarré")
            if not self.start():
                raise RuntimeError("Impossible de démarrer le moteur d'inférence")

        self.logger.info(f"Traitement de la requête d'inférence: {query.question}")

        # Vérifier le cache pour des requêtes identiques récentes
        cache_key = self._compute_cache_key(query)
        if cache_key in self.results_cache:
            result, timestamp = self.results_cache[cache_key]
            if time.time() - timestamp < self.cache_expiry:
                self.logger.info(f"Résultat trouvé en cache pour {query.query_id}")
                # Créer une copie pour éviter la modification du cache
                return self._copy_result(result)

        # Chronométrer l'exécution
        start_time = time.time()

        # Sélectionner la stratégie d'inférence basée sur le mode hybride
        if query.hybrid_mode == HybridMode.ADAPTIVE:
            hybrid_mode = self._select_adaptive_mode(query)
        else:
            hybrid_mode = query.hybrid_mode

        # Exécuter l'inférence selon le mode hybride choisi
        result = self._execute_hybrid_inference(query, hybrid_mode)

        # Compléter le résultat
        result.execution_time = time.time() - start_time

        # Mettre en cache le résultat
        self.results_cache[cache_key] = (result, time.time())

        # Enregistrer des métriques pour l'apprentissage adaptatif
        self._update_metrics(query, result)

        return result

    def add_symbolic_rule(self, rule: SymbolicConstraint) -> bool:
        """
        Ajoute une règle symbolique au moteur d'inférence.

        Args:
            rule: La règle/contrainte à ajouter

        Returns:
            bool: True si l'ajout a réussi, False sinon
        """
        with self.lock:
            # Vérifier que la règle est valide
            if not rule.expression:
                self.logger.warning("Tentative d'ajout d'une règle vide")
                return False

            # Vérifier les doublons
            for existing_rule in self.symbolic_rules:
                if existing_rule.constraint_id == rule.constraint_id:
                    self.logger.warning(f"Règle {rule.constraint_id} déjà existante, mise à jour")
                    # Remplacer l'ancienne règle
                    self.symbolic_rules.remove(existing_rule)
                    break

            # Ajouter la règle
            self.symbolic_rules.append(rule)
            self.logger.info(f"Règle symbolique ajoutée: {rule.constraint_id} - {rule.expression}")

            # Si le moteur symbolique est disponible, lui passer aussi la règle
            if self.symbolic_engine:
                try:
                    # Interface supposée - peut nécessiter une adaptation
                    self.symbolic_engine.add_rule(rule)
                except Exception as e:
                    self.logger.warning(f"Échec de l'ajout au moteur symbolique: {str(e)}")

            return True

    def adjust_parameters(self, new_params: Dict[str, Any]) -> bool:
        """
        Ajuste les paramètres du moteur d'inférence hybride.

        Args:
            new_params: Dictionnaire de paramètres à modifier

        Returns:
            bool: True si les paramètres ont été modifiés, False sinon
        """
        with self.lock:
            for key, value in new_params.items():
                if key in self.params:
                    old_value = self.params[key]
                    self.params[key] = value
                    self.logger.info(f"Paramètre {key} modifié: {old_value} -> {value}")
                else:
                    self.logger.warning(f"Paramètre inconnu: {key}")

            return True

    def get_metrics(self) -> Dict[str, Any]:
        """
        Récupère les métriques de performance du moteur d'inférence.

        Returns:
            Dict: Métriques de performance
        """
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def explain_inference(self, result: InferenceResult) -> str:
        """
        Génère une explication en langage naturel du processus d'inférence.

        Args:
            result: Résultat d'inférence à expliquer

        Returns:
            str: Explication détaillée du raisonnement
        """
        if not result.steps:
            return "Aucune étape d'inférence disponible pour explication."

        # Si le LLM est disponible, l'utiliser pour générer une explication cohérente
        if self.llm_engine and result.llm_contribution > 0.1:
            try:
                # Préparer le prompt avec les étapes
                steps_text = "\n".join([
                    f"Étape {i+1}: {step.type} - {step.description}\n"
                    f"Raisonnement: {step.reasoning}\n"
                    f"Confiance: {step.confidence:.2f}"
                    for i, step in enumerate(result.steps)
                ])

                prompt = (
                    f"Explication du raisonnement:\n\n"
                    f"Question: {result.query_id}\n"
                    f"Étapes d'inférence:\n{steps_text}\n\n"
                    f"Réponses: {result.answers}\n\n"
                    f"Veuillez expliquer ce raisonnement de manière claire et cohérente:"
                )

                # Appeler le LLM pour obtenir une explication
                response = self.llm_engine.complete({
                    "prompt": prompt,
                    "max_tokens": 300,
                    "temperature": 0.7
                })

                if response and hasattr(response, 'text') and response.text:
                    return response.text
            except Exception as e:
                self.logger.warning(f"Échec de l'explication via LLM: {str(e)}")

        # Méthode alternative: construire une explication structurée
        explanation = [f"Explication du processus d'inférence pour: {result.query_id}\n"]

        for i, step in enumerate(result.steps):
            explanation.append(f"Étape {i+1}: {step.type}")
            explanation.append(f"  {step.description}")
            if step.reasoning:
                explanation.append(f"  Raisonnement: {step.reasoning}")
            if step.confidence < 1.0:
                explanation.append(f"  Confiance: {step.confidence:.2f}")
            explanation.append("")

        explanation.append(f"Conclusion:")
        for var, value in result.answers.items():
            explanation.append(f"  {var} = {value}")
        explanation.append(f"Confiance globale: {result.confidence:.2f}")

        contribution_info = []
        if result.symbolic_contribution > 0.01:
            contribution_info.append(f"{result.symbolic_contribution:.0%} symbolique")
        if result.neural_contribution > 0.01:
            contribution_info.append(f"{result.neural_contribution:.0%} neuronal")
        if result.llm_contribution > 0.01:
            contribution_info.append(f"{result.llm_contribution:.0%} LLM")

        if contribution_info:
            explanation.append(f"Contributions: {', '.join(contribution_info)}")

        return "\n".join(explanation)

    def register_inference_strategy(self, name: str,
                                   strategy_fn: Callable[[InferenceQuery], InferenceResult],
                                   applicable_modes: List[HybridMode],
                                   applicable_types: List[InferenceType]) -> bool:
        """
        Enregistre une stratégie d'inférence personnalisée.

        Args:
            name: Nom unique de la stratégie
            strategy_fn: Fonction implémentant la stratégie
            applicable_modes: Modes hybrides compatibles
            applicable_types: Types d'inférence compatibles

        Returns:
            bool: True si l'enregistrement a réussi, False sinon
        """
        with self.lock:
            if name in self.inference_strategies:
                self.logger.warning(f"Stratégie {name} déjà enregistrée, remplacement")

            self.inference_strategies[name] = {
                'function': strategy_fn,
                'modes': applicable_modes,
                'types': applicable_types,
                'last_used': 0,
                'usage_count': 0,
                'success_rate': 0.0
            }

            self.logger.info(f"Stratégie d'inférence enregistrée: {name}")
            return True

    def unregister_inference_strategy(self, name: str) -> bool:
        """
        Supprime une stratégie d'inférence du registre.

        Args:
            name: Nom de la stratégie à supprimer

        Returns:
            bool: True si la suppression a réussi, False sinon
        """
        with self.lock:
            if name not in self.inference_strategies:
                self.logger.warning(f"Stratégie {name} non trouvée")
                return False

            del self.inference_strategies[name]
            self.logger.info(f"Stratégie d'inférence supprimée: {name}")
            return True

    # -----------------------------------------------------------------------
    # Méthodes internes
    # -----------------------------------------------------------------------

    def _register_default_strategies(self) -> None:
        """Enregistre les stratégies d'inférence par défaut."""
        # Stratégie symbolique-première
        self.register_inference_strategy(
            name="symbolic_first",
            strategy_fn=self._symbolic_first_strategy,
            applicable_modes=[HybridMode.SYMBOLIC_FIRST, HybridMode.ADAPTIVE],
            applicable_types=[InferenceType.DEDUCTION, InferenceType.ABDUCTION,
                             InferenceType.PROBABILISTIC, InferenceType.FUZZY]
        )

        # Stratégie neurale-première
        self.register_inference_strategy(
            name="neural_first",
            strategy_fn=self._neural_first_strategy,
            applicable_modes=[HybridMode.NEURAL_FIRST, HybridMode.ADAPTIVE],
            applicable_types=[InferenceType.INDUCTION, InferenceType.ANALOGY,
                             InferenceType.SPATIAL, InferenceType.CAUSAL]
        )

        # Stratégie parallèle
        self.register_inference_strategy(
            name="parallel",
            strategy_fn=self._parallel_strategy,
            applicable_modes=[HybridMode.PARALLEL, HybridMode.ADAPTIVE],
            applicable_types=[t for t in InferenceType]  # Tous les types
        )

        # Stratégie itérative
        self.register_inference_strategy(
            name="iterative",
            strategy_fn=self._iterative_strategy,
            applicable_modes=[HybridMode.ITERATIVE, HybridMode.ADAPTIVE],
            applicable_types=[InferenceType.DEDUCTION, InferenceType.INDUCTION,
                             InferenceType.ABDUCTION, InferenceType.PROBABILISTIC]
        )

        # Stratégie LLM-orchestrée (si LLM disponible)
        if self.llm_engine:
            self.register_inference_strategy(
                name="llm_orchestrated",
                strategy_fn=self._llm_orchestrated_strategy,
                applicable_modes=[HybridMode.LLM_ORCHESTRATED, HybridMode.ADAPTIVE],
                applicable_types=[InferenceType.LLM_GUIDED, InferenceType.ANALOGY,
                                 InferenceType.CAUSAL, InferenceType.TEMPORAL]
            )

    def _register_with_nexus_core(self) -> None:
        """Enregistre ce module auprès du NexusCore."""
        if not self.nexus_core:
            self.logger.warning("Pas de NexusCore disponible pour l'enregistrement")
            return

        try:
            # Préparer les dépendances
            dependencies = []
            if self.reasoning_core:
                dependencies.append("reasoning_core")
            if self.symbolic_engine:
                dependencies.append("symbolic_engine")
            if self.neural_reasoner:
                dependencies.append("neural_reasoner")
            if self.semantic_memory:
                dependencies.append("semantic_memory")

            # Enregistrement auprès du NexusCore
            self.nexus_core.register_module(
                name="hybrid_inference",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Enregistrement réussi auprès du NexusCore")
        except Exception as e:
            self.logger.error(f"Échec de l'enregistrement auprès du NexusCore: {str(e)}")

    def _initialize_neuro_symbolic_models(self) -> None:
        """Initialise les modèles neuro-symboliques spécifiques."""
        try:
            # Charger les modèles LNN (Logical Neural Networks) si disponible
            # Note: Ceci est un placeholder - dans une implémentation réelle,
            # nous initialiserions les frameworks spécifiques
            self.neural_models["lnn"] = {
                "type": "lnn",
                "initialized": True,
                "description": "Logical Neural Network pour inférence symbolique avec activation neurale"
            }

            # DeepProbLog
            self.neural_models["deepproblog"] = {
                "type": "deepproblog",
                "initialized": True,
                "description": "DeepProbLog pour programmation logique probabiliste neuronale"
            }

            self.logger.info("Modèles neuro-symboliques initialisés")
        except Exception as e:
            self.logger.warning(f"Échec de l'initialisation des modèles neuro-symboliques: {str(e)}")

    def _execute_hybrid_inference(self, query: InferenceQuery,
                                 hybrid_mode: HybridMode) -> InferenceResult:
        """
        Exécute l'inférence selon le mode hybride spécifié.

        Args:
            query: La requête d'inférence
            hybrid_mode: Le mode hybride à utiliser

        Returns:
            InferenceResult: Résultat de l'inférence
        """
        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Sélectionner la stratégie d'inférence appropriée
        strategy_name = self._select_strategy(query, hybrid_mode)

        if not strategy_name or strategy_name not in self.inference_strategies:
            self.logger.warning(f"Aucune stratégie trouvée pour le mode {hybrid_mode}")
            return result

        # Exécuter la stratégie
        strategy = self.inference_strategies[strategy_name]
        try:
            # Chronométrer l'exécution
            start_time = time.time()

            # Appeler la fonction de stratégie
            result = strategy['function'](query)

            # Mettre à jour les statistiques d'utilisation
            strategy['last_used'] = time.time()
            strategy['usage_count'] += 1

            # Mettre à jour le taux de succès
            success = bool(result.success)
            old_success, old_total = self.strategy_success_rates[strategy_name]
            self.strategy_success_rates[strategy_name] = (old_success + (1 if success else 0), old_total + 1)

            self.logger.info(f"Inférence exécutée via stratégie '{strategy_name}', succès={success}")

            # Compléter le résultat si nécessaire
            if not result.explanation:
                result.explanation = self.explain_inference(result)

            return result

        except Exception as e:
            self.logger.error(f"Erreur lors de l'exécution de la stratégie {strategy_name}: {str(e)}")
            result.explanation = f"Erreur d'inférence: {str(e)}"
            return result

    def _select_strategy(self, query: InferenceQuery,
                        hybrid_mode: HybridMode) -> Optional[str]:
        """
        Sélectionne la stratégie d'inférence la plus appropriée.

        Args:
            query: La requête d'inférence
            hybrid_mode: Le mode d'inférence hybride

        Returns:
            str: Nom de la stratégie sélectionnée, ou None si aucune trouvée
        """
        # Trouver les stratégies compatibles avec le mode et les types d'inférence
        compatible_strategies = []

        for name, strategy in self.inference_strategies.items():
            # Vérifier la compatibilité du mode
            if hybrid_mode not in strategy['modes']:
                continue

            # Vérifier la compatibilité des types d'inférence
            types_compatible = False
            for inference_type in query.inference_types:
                if inference_type in strategy['types']:
                    types_compatible = True
                    break

            if not types_compatible and query.inference_types:
                continue

            compatible_strategies.append(name)

        if not compatible_strategies:
            return None

        # Si une seule stratégie est compatible, la retourner
        if len(compatible_strategies) == 1:
            return compatible_strategies[0]

        # Si mode adaptatif, utiliser les statistiques d'utilisation et de succès
        if hybrid_mode == HybridMode.ADAPTIVE:
            # Calculer un score pour chaque stratégie basé sur:
            # - Taux de succès historique
            # - Pertinence pour les types d'inférence demandés
            # - Avec un peu d'exploration aléatoire

            scores = {}
            for name in compatible_strategies:
                # Calculer le taux de succès
                success, total = self.strategy_success_rates[name]
                success_rate = success / total if total > 0 else 0.5

                # Score de base = taux de succès
                scores[name] = success_rate

                # Bonus pour l'expérience (mais éviter de trop favoriser les stratégies déjà utilisées)
                usage_bonus = min(0.2, self.inference_strategies[name]['usage_count'] / 100)
                scores[name] += usage_bonus

                # Petit facteur aléatoire pour l'exploration (max 0.1)
                scores[name] += random.random() * 0.1

            # Sélectionner la stratégie avec le meilleur score
            return max(scores, key=scores.get) if scores else None

        # Sinon, utiliser une stratégie compatible de manière déterministe
        else:
            # Par défaut, prendre la première stratégie compatible
            return compatible_strategies[0]

    def _select_adaptive_mode(self, query: InferenceQuery) -> HybridMode:
        """
        Sélectionne dynamiquement le meilleur mode hybride pour la requête.

        Args:
            query: La requête d'inférence

        Returns:
            HybridMode: Le mode hybride sélectionné
        """
        # Pour la décision adaptative, nous considérons:
        # 1. La nature de la requête (types d'inférence)
        # 2. Les données disponibles (faits connus, contraintes)
        # 3. L'historique des succès/échecs précédents

        # Règles heuristiques pour la sélection du mode:

        # Si nous avons beaucoup de contraintes symboliques, privilégier symbolique
        if len(query.constraints) > 5:
            return HybridMode.SYMBOLIC_FIRST

        # Si nous avons peu de contraintes mais beaucoup de contexte, privilégier neuronal
        if len(query.constraints) < 2 and len(query.context) > 5:
            return HybridMode.NEURAL_FIRST

        # Si la requête est une LLM_GUIDED, utiliser ce mode
        if InferenceType.LLM_GUIDED in query.inference_types and self.llm_engine:
            return HybridMode.LLM_ORCHESTRATED

        # Si la requête inclut des raisonnements spatiaux ou analogiques, privilégier neuronal
        if (InferenceType.SPATIAL in query.inference_types or
            InferenceType.ANALOGY in query.inference_types):
            return HybridMode.NEURAL_FIRST

        # Si la requête est déductive ou abductive, privilégier symbolique
        if (InferenceType.DEDUCTION in query.inference_types or
            InferenceType.ABDUCTION in query.inference_types):
            return HybridMode.SYMBOLIC_FIRST

        # Si timing est critique (petit time_limit), utiliser parallèle pour optimiser
        if query.time_limit and query.time_limit < 1.0:
            return HybridMode.PARALLEL

        # Pour les questions complexes et multi-étapes, préférer itératif
        if query.max_inference_depth > 3:
            return HybridMode.ITERATIVE

        # Par défaut, utiliser parallèle qui est généralement équilibré
        return HybridMode.PARALLEL

    def _compute_cache_key(self, query: InferenceQuery) -> str:
        """
        Génère une clé de cache pour une requête d'inférence.

        Args:
            query: La requête d'inférence

        Returns:
            str: Clé de cache unique
        """
        # Éléments clés à inclure dans la signature de cache
        elements = [
            query.question,
            sorted(query.target_variables),
            tuple((c.constraint_id, c.expression) for c in query.constraints),
            sorted((k, str(v)) for k, v in query.known_facts.items())
        ]

        # Créer une signature en chaîne
        signature = str(elements)

        # Hacher pour avoir une clé compacte
        import hashlib
        return hashlib.md5(signature.encode('utf-8')).hexdigest()

    def _copy_result(self, result: InferenceResult) -> InferenceResult:
        """
        Crée une copie d'un résultat d'inférence.

        Args:
            result: Le résultat à copier

        Returns:
            InferenceResult: Copie du résultat
        """
        # Cette implémentation est simplifiée, une vraie copie profonde serait plus robuste
        import copy
        return copy.deepcopy(result)

    def _update_metrics(self, query: InferenceQuery, result: InferenceResult) -> None:
        """
        Met à jour les métriques de performance.

        Args:
            query: La requête d'inférence
            result: Le résultat d'inférence
        """
        self.metrics["success_rate"].append(1.0 if result.success else 0.0)
        self.metrics["confidence"].append(result.confidence)
        self.metrics["execution_time"].append(result.execution_time)
        self.metrics["steps_count"].append(len(result.steps))

        # Suivre les contribuions
        self.metrics["symbolic_contribution"].append(result.symbolic_contribution)
        self.metrics["neural_contribution"].append(result.neural_contribution)
        self.metrics["llm_contribution"].append(result.llm_contribution)

    # -----------------------------------------------------------------------
    # Implémentations des stratégies d'inférence
    # -----------------------------------------------------------------------

    def _symbolic_first_strategy(self, query: InferenceQuery) -> InferenceResult:
        """
        Stratégie symbolique-première: commence par un raisonnement
        symbolique puis affine avec neuronal si nécessaire.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat d'inférence
        """
        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Étape 1: Appliquer l'inférence symbolique
        symbolic_result = self._apply_symbolic_reasoning(query)
        result.steps.extend(symbolic_result.steps)

        # Si le résultat symbolique est suffisant (haute confiance), le retourner
        if symbolic_result.success and symbolic_result.confidence >= self.params["confidence_threshold"]:
            result.success = True
            result.answers = symbolic_result.answers
            result.confidence = symbolic_result.confidence
            result.symbolic_contribution = 1.0
            result.neural_contribution = 0.0
            return result

        # Étape 2: Affiner avec inférence neurale
        neural_input = query
        # Injecter les résultats symboliques comme faits connus pour l'inférence neurale
        for var, value in symbolic_result.answers.items():
            neural_input.known_facts[var] = value

        neural_result = self._apply_neural_reasoning(neural_input)
        result.steps.extend(neural_result.steps)

        # Combiner les résultats
        combined_answers = {}
        combined_answers.update(symbolic_result.answers)

        # Pour les variables qui n'ont pas été résolues symboliquement ou avec faible confiance
        for var, value in neural_result.answers.items():
            if var not in combined_answers or symbolic_result.confidence < 0.7:
                combined_answers[var] = value

        result.answers = combined_answers

        # Calculer la confiance combinée - plus élevée pour les réponses symboliques
        symbolic_weight = self.params["symbolic_weight"]
        neural_weight = self.params["neural_weight"]

        # Calcul de la contribution proportionnelle
        if symbolic_result.success and neural_result.success:
            # Si les deux ont réussi, pondérer leurs contributions
            result.symbolic_contribution = 0.7  # Dominant car stratégie symbolique-première
            result.neural_contribution = 0.3
            result.confidence = (symbolic_result.confidence * symbolic_weight +
                               neural_result.confidence * neural_weight) / (symbolic_weight + neural_weight)
        elif symbolic_result.success:
            result.symbolic_contribution = 1.0
            result.confidence = symbolic_result.confidence
        elif neural_result.success:
            result.neural_contribution = 1.0
            result.confidence = neural_result.confidence
        else:
            # Aucun n'a réussi
            result.confidence = 0.0

        result.success = bool(result.answers)

        return result

    def _neural_first_strategy(self, query: InferenceQuery) -> InferenceResult:
        """
        Stratégie neurale-première: commence par un raisonnement
        neuronal puis vérifie/corrige avec symbolique.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat d'inférence
        """
        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Étape 1: Appliquer l'inférence neurale
        neural_result = self._apply_neural_reasoning(query)
        result.steps.extend(neural_result.steps)

        # Si l'inférence neurale a échoué complètement, pas la peine de continuer
        if not neural_result.success:
            # Essayer quand même l'inférence symbolique comme fallback
            symbolic_result = self._apply_symbolic_reasoning(query)
            result.steps.extend(symbolic_result.steps)
            result.success = symbolic_result.success
            result.answers = symbolic_result.answers
            result.confidence = symbolic_result.confidence
            result.symbolic_contribution = 1.0
            return result

        # Étape 2: Vérifier et contraindre avec inférence symbolique
        symbolic_input = query
        # Injecter les résultats neuraux comme hypothèses pour l'inférence symbolique
        for var, value in neural_result.answers.items():
            if var not in symbolic_input.known_facts:
                symbolic_input.known_facts[var] = value

        symbolic_result = self._apply_symbolic_reasoning(symbolic_input)
        result.steps.extend(symbolic_result.steps)

        # Combiner les résultats - privilégier symbolique pour les contraintes dures
        combined_answers = {}

        # Commencer par les réponses neurales
        combined_answers.update(neural_result.answers)

        # Remplacer par les réponses symboliques qui sont plus fiables
        for var, value in symbolic_result.answers.items():
            # Si confiance élevée, toujours préférer la réponse symbolique
            if var in symbolic_result.answers and symbolic_result.confidence > 0.8:
                combined_answers[var] = value

        result.answers = combined_answers

        # Calculer la confiance combinée et les contributions
        if symbolic_result.success and neural_result.success:
            # Si les deux ont réussi, pondérer leurs contributions
            result.neural_contribution = 0.7  # Dominant car stratégie neurale-première
            result.symbolic_contribution = 0.3

            symbolic_weight = self.params["symbolic_weight"] * 0.5  # Réduit car stratégie neurale-première
            neural_weight = self.params["neural_weight"] * 1.5  # Augmenté car stratégie neurale-première

            result.confidence = (symbolic_result.confidence * symbolic_weight +
                               neural_result.confidence * neural_weight) / (symbolic_weight + neural_weight)
        elif neural_result.success:
            result.neural_contribution = 1.0
            result.confidence = neural_result.confidence
        elif symbolic_result.success:
            result.symbolic_contribution = 1.0
            result.confidence = symbolic_result.confidence
        else:
            # Aucun n'a réussi
            result.confidence = 0.0

        result.success = bool(result.answers)

        return result

    def _parallel_strategy(self, query: InferenceQuery) -> InferenceResult:
        """
        Stratégie parallèle: exécute les inférences symbolique et neurale
        en parallèle, puis fusionne les résultats.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat d'inférence
        """
        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Exécuter les deux types d'inférence en parallèle
        # Note: Dans une implémentation réelle, ceci utiliserait des threads ou asyncio
        # Pour simplifier, nous les exécutons séquentiellement

        symbolic_result = self._apply_symbolic_reasoning(query)
        neural_result = self._apply_neural_reasoning(query)

        # Ajouter les étapes des deux raisonnements
        result.steps.extend(symbolic_result.steps)
        result.steps.extend(neural_result.steps)

        # Fusionner les réponses
        combined_answers = {}

        # Méthode de fusion basée sur la confiance
        for var in set(list(symbolic_result.answers.keys()) + list(neural_result.answers.keys())):
            symbolic_value = symbolic_result.answers.get(var)
            neural_value = neural_result.answers.get(var)

            symbolic_confidence = symbolic_result.confidence if var in symbolic_result.answers else 0
            neural_confidence = neural_result.confidence if var in neural_result.answers else 0

            # Si une seule source a une réponse, l'utiliser
            if symbolic_value is not None and neural_value is None:
                combined_answers[var] = symbolic_value
            elif neural_value is not None and symbolic_value is None:
                combined_answers[var] = neural_value
            # Si les deux ont une réponse, prendre celle avec la plus haute confiance
            elif symbolic_value is not None and neural_value is not None:
                if symbolic_confidence >= neural_confidence:
                    combined_answers[var] = symbolic_value
                else:
                    combined_answers[var] = neural_value

        result.answers = combined_answers

        # Calculer la confiance et les contributions
        if symbolic_result.success and neural_result.success:
            # Si les deux ont réussi, leur contribution est à peu près égale
            result.symbolic_contribution = 0.5
            result.neural_contribution = 0.5

            # Confiance pondérée basée sur les poids configurés
            symbolic_weight = self.params["symbolic_weight"]
            neural_weight = self.params["neural_weight"]

            result.confidence = (symbolic_result.confidence * symbolic_weight +
                               neural_result.confidence * neural_weight) / (symbolic_weight + neural_weight)
        elif symbolic_result.success:
            result.symbolic_contribution = 1.0
            result.confidence = symbolic_result.confidence
        elif neural_result.success:
            result.neural_contribution = 1.0
            result.confidence = neural_result.confidence
        else:
            result.confidence = 0.0

        result.success = bool(result.answers)

        return result

    def _iterative_strategy(self, query: InferenceQuery) -> InferenceResult:
        """
        Stratégie itérative: alterne entre raisonnements symbolique et
        neuronal en plusieurs passes, chacun utilisant les résultats précédents.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat d'inférence
        """
        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Copier la requête pour la modifier à chaque itération
        current_query = query  # Dans une implémentation réelle, faire une copie profonde

        # Variables pour suivre les contributions
        symbolic_contrib = 0.0
        neural_contrib = 0.0

        # Exécuter plusieurs passes, alternant symbolique et neuronal
        max_iterations = query.max_inference_depth or 3

        for i in range(max_iterations):
            # Iteration paire: symbolique, impaire: neuronale
            if i % 2 == 0:
                iter_result = self._apply_symbolic_reasoning(current_query)
                symbolic_weight = 1.0 / (i + 1)  # Poids diminue à chaque itération
                symbolic_contrib += symbolic_weight
            else:
                iter_result = self._apply_neural_reasoning(current_query)
                neural_weight = 1.0 / (i + 1)  # Poids diminue à chaque itération
                neural_contrib += neural_weight

            # Ajouter les étapes
            result.steps.extend(iter_result.steps)

            # Mettre à jour les faits connus pour la prochaine itération
            for var, value in iter_result.answers.items():
                if var not in current_query.known_facts:
                    current_query.known_facts[var] = value

            # Si nous avons toutes les variables cibles, ou si la confiance est très haute, arrêter
            all_targets_found = all(var in current_query.known_facts for var in query.target_variables)
            if all_targets_found and iter_result.confidence > 0.9:
                break

            # Si aucun progrès dans cette itération, arrêter
            if not iter_result.answers:
                break

        # Construire le résultat final
        result.answers = current_query.known_facts.copy()

        # Filtrer pour ne garder que les variables cibles si spécifiées
        if query.target_variables:
            result.answers = {var: val for var, val in result.answers.items()
                            if var in query.target_variables}

        # Calculer les contributions normalisées
        total_contrib = symbolic_contrib + neural_contrib
        if total_contrib > 0:
            result.symbolic_contribution = symbolic_contrib / total_contrib
            result.neural_contribution = neural_contrib / total_contrib

        # Estimer la confiance globale - plus complexe dans une stratégie itérative
        # Ici nous utilisons une heuristique simple: moyenne des confiances des étapes
        confidences = [step.confidence for step in result.steps if hasattr(step, 'confidence')]
        result.confidence = sum(confidences) / len(confidences) if confidences else 0.0

        result.success = bool(result.answers)

        return result

    def _llm_orchestrated_strategy(self, query: InferenceQuery) -> InferenceResult:
        """
        Stratégie LLM-orchestrée: utilise un LLM pour guider et intégrer
        les inférences symboliques et neuronales.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat d'inférence
        """
        # Cette stratégie n'est disponible que si un LLM est configuré
        if not self.llm_engine:
            self.logger.warning("Stratégie LLM-orchestrée invoquée sans LLM disponible")
            return InferenceResult(query_id=query.query_id, success=False)

        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Construire un prompt pour le LLM qui explique la requête et demande un plan
        constraints_text = "\n".join([f"- {c.expression}" for c in query.constraints])
        facts_text = "\n".join([f"- {var}: {value}" for var, value in query.known_facts.items()])

        plan_prompt = (
            f"Question/Requête: {query.question}\n\n"
            f"Variables cibles: {', '.join(query.target_variables)}\n\n"
            f"Contraintes connues:\n{constraints_text}\n\n"
            f"Faits connus:\n{facts_text}\n\n"
            f"Veuillez élaborer un plan d'inférence structuré pour résoudre cette question. "
            f"Précisez quand il faut utiliser le raisonnement symbolique (logique, contraintes) "
            f"et quand il faut utiliser le raisonnement neuronal (similarité, analogie)."
        )

        # Demander un plan au LLM
        try:
            plan_response = self.llm_engine.complete({
                "prompt": plan_prompt,
                "max_tokens": 500,
                "temperature": 0.7
            })

            plan_text = plan_response.text if hasattr(plan_response, 'text') else str(plan_response)

            # Ajouter l'étape de planification
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="llm_planning",
                description="Planification du processus d'inférence par LLM",
                reasoning=plan_text,
                confidence=0.9
            ))

            # Analyser le plan pour extraire les étapes d'inférence
            # Cette implémentation est simplifiée - une version réelle utiliserait
            # une analyse plus sophistiquée du plan généré par le LLM

            # Supposons que le plan liste clairement "Étape 1", "Étape 2", etc.
            plan_steps = []
            current_step = None

            for line in plan_text.split('\n'):
                line = line.strip()
                if not line:
                    continue

                # Détecter les en-têtes d'étapes
                if line.lower().startswith(("étape", "step", "phase")):
                    if current_step:
                        plan_steps.append(current_step)
                    current_step = {"description": line, "type": "unknown", "content": []}

                    # Essayer de déterminer le type d'étape
                    lower_line = line.lower()
                    if any(kw in lower_line for kw in ["symbolique", "logique", "déduction", "contrainte"]):
                        current_step["type"] = "symbolic"
                    elif any(kw in lower_line for kw in ["neural", "neuronal", "analogie", "similarité"]):
                        current_step["type"] = "neural"
                elif current_step:
                    current_step["content"].append(line)

            # Ajouter la dernière étape
            if current_step:
                plan_steps.append(current_step)

            # Exécuter les étapes du plan
            current_facts = query.known_facts.copy()

            for i, plan_step in enumerate(plan_steps):
                step_type = plan_step["type"]
                step_desc = plan_step["description"]

                # Créer une sous-requête pour cette étape
                sub_query = query  # Dans un vrai code, faire une copie profonde
                sub_query.known_facts = current_facts

                # Exécuter l'étape selon son type
                if step_type == "symbolic":
                    step_result = self._apply_symbolic_reasoning(sub_query)
                elif step_type == "neural":
                    step_result = self._apply_neural_reasoning(sub_query)
                else:
                    # Type inconnu, par défaut essayer les deux et prendre le meilleur
                    symbolic_result = self._apply_symbolic_reasoning(sub_query)
                    neural_result = self._apply_neural_reasoning(sub_query)

                    # Choisir le résultat avec le plus de réponses ou la plus haute confiance
                    if len(symbolic_result.answers) > len(neural_result.answers):
                        step_result = symbolic_result
                    elif len(neural_result.answers) > len(symbolic_result.answers):
                        step_result = neural_result
                    else:
                        step_result = symbolic_result if symbolic_result.confidence >= neural_result.confidence else neural_result

                # Mettre à jour les faits connus
                current_facts.update(step_result.answers)

                # Ajouter les étapes de cette sous-inférence
                result.steps.extend(step_result.steps)

            # Intégration finale des résultats par le LLM
            integration_prompt = (
                f"Question initiale: {query.question}\n\n"
                f"Variables cibles: {', '.join(query.target_variables)}\n\n"
                f"Résultats obtenus:\n"
            )

            for var, value in current_facts.items():
                integration_prompt += f"- {var}: {value}\n"

            integration_prompt += (
                f"\nVeuillez fournir une réponse finale intégrée à la question initiale, "
                f"en indiquant votre niveau de confiance."
            )

            integration_response = self.llm_engine.complete({
                "prompt": integration_prompt,
                "max_tokens": 300,
                "temperature": 0.7
            })

            integration_text = integration_response.text if hasattr(integration_response, 'text') else str(integration_response)

            # Ajouter l'étape d'intégration
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="llm_integration",
                description="Intégration finale des résultats par LLM",
                reasoning=integration_text,
                confidence=0.85
            ))

            # Définir les résultats finaux
            result.answers = current_facts
            if query.target_variables:
                result.answers = {var: val for var, val in result.answers.items()
                               if var in query.target_variables}

            # Évaluer les contributions
            step_types = [step.type for step in result.steps if hasattr(step, 'type')]
            symbolic_count = sum(1 for t in step_types if 'symbolic' in t)
            neural_count = sum(1 for t in step_types if 'neural' in t)
            llm_count = sum(1 for t in step_types if 'llm' in t)

            total_steps = symbolic_count + neural_count + llm_count
            if total_steps > 0:
                result.symbolic_contribution = symbolic_count / total_steps
                result.neural_contribution = neural_count / total_steps
                result.llm_contribution = llm_count / total_steps
            else:
                result.llm_contribution = 1.0

            # Estimer la confiance globale
            result.confidence = 0.8  # Valeur par défaut pour la stratégie LLM-orchestrée

            # Essayer d'extraire une confiance du texte d'intégration
            confidence_keywords = {
                "très confiant": 0.9,
                "confiant": 0.8,
                "assez confiant": 0.7,
                "moyennement confiant": 0.6,
                "peu confiant": 0.5,
                "incertain": 0.4,
                "très incertain": 0.3
            }

            for keyword, conf_value in confidence_keywords.items():
                if keyword in integration_text.lower():
                    result.confidence = conf_value
                    break

            result.success = bool(result.answers)
            result.explanation = integration_text

            return result

        except Exception as e:
            self.logger.error(f"Erreur dans la stratégie LLM-orchestrée: {str(e)}")
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="error",
                description=f"Erreur lors de l'exécution de la stratégie LLM-orchestrée: {str(e)}",
                confidence=0.0
            ))
            result.success = False
            return result

    def _apply_symbolic_reasoning(self, query: InferenceQuery) -> InferenceResult:
        """
        Applique le raisonnement symbolique à une requête.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat de l'inférence symbolique
        """
        # Cette méthode est un placeholder - dans une implémentation réelle,
        # elle appellerait un moteur symbolique sophistiqué (par ex. via symbolic_engine)

        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Si un moteur symbolique est disponible, l'utiliser
        if self.symbolic_engine:
            try:
                # Interface supposée - adapter selon l'API réelle du moteur
                engine_result = self.symbolic_engine.solve(
                    constraints=[c.expression for c in query.constraints],
                    known_facts=query.known_facts,
                    target_variables=query.target_variables
                )

                # Transformer le résultat du moteur symbolique
                result.answers = engine_result.get("solutions", {})
                result.confidence = engine_result.get("confidence", 0.8)
                result.success = engine_result.get("success", False)

                # Ajouter les étapes de raisonnement
                for i, step in enumerate(engine_result.get("steps", [])):
                    result.steps.append(InferenceStep(
                        step_id=str(uuid.uuid4()),
                        type="symbolic",
                        description=f"Étape symbolique {i+1}",
                        reasoning=step.get("explanation", ""),
                        input_facts=step.get("input", {}),
                        output_facts=step.get("output", {}),
                        confidence=step.get("confidence", 0.8)
                    ))

                return result

            except Exception as e:
                self.logger.error(f"Erreur du moteur symbolique: {str(e)}")
                # Continuer avec l'implémentation de secours

        # Implémentation simplifiée de secours (si pas de moteur ou erreur)
        # Ceci n'est qu'une simulation très basique de raisonnement symbolique

        # Simuler une étape d'analyse des contraintes
        result.steps.append(InferenceStep(
            step_id=str(uuid.uuid4()),
            type="symbolic_analysis",
            description="Analyse des contraintes symboliques",
            input_facts=query.known_facts,
            confidence=0.7,
            reasoning="Analyse des contraintes disponibles pour déterminer les relations logiques."
        ))

        # Simuler une résolution de contraintes extrêmement simple
        # Dans une vraie implémentation, ce serait un solveur de contraintes
        answers = query.known_facts.copy()

        # Raisonnement symbolique simple sur les contraintes
        for constraint in query.constraints:
            # Exemple: traitement très basique pour des règles d'implication simples A → B
            if "→" in constraint.expression or "->" in constraint.expression:
                parts = constraint.expression.replace("→", "->").split("->")
                if len(parts) == 2:
                    antecedent = parts[0].strip()
                    consequent = parts[1].strip()

                    # Si l'antécédent est connu et vrai, inférer le conséquent
                    if antecedent in answers and answers[antecedent]:
                        answers[consequent] = True

                        result.steps.append(InferenceStep(
                            step_id=str(uuid.uuid4()),
                            type="symbolic_deduction",
                            description=f"Application de la règle d'implication: {constraint.expression}",
                            input_facts={antecedent: answers[antecedent]},
                            output_facts={consequent: True},
                            confidence=constraint.confidence,
                            reasoning=f"Puisque {antecedent} est vrai, on peut déduire que {consequent} est vrai."
                        ))

        # Déterminer le succès et filtrer les réponses pour les variables cibles
        target_answers = {}
        for var in query.target_variables:
            if var in answers:
                target_answers[var] = answers[var]

        result.answers = target_answers if query.target_variables else answers
        result.success = bool(result.answers)
        result.confidence = 0.7 if result.success else 0.0

        return result

    def _apply_neural_reasoning(self, query: InferenceQuery) -> InferenceResult:
        """
        Applique le raisonnement neuronal à une requête.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat de l'inférence neuronale
        """
        # Cette méthode est un placeholder - dans une implémentation réelle,
        # elle appellerait un modèle neuronal (par ex. via neural_reasoner)

        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Si un raisonneur neuronal est disponible, l'utiliser
        if self.neural_reasoner:
            try:
                # Interface supposée - adapter selon l'API réelle du raisonneur
                reasoner_result = self.neural_reasoner.infer(
                    question=query.question,
                    context=query.context,
                    known_facts=query.known_facts
                )

                # Transformer le résultat du raisonneur neuronal
                result.answers = reasoner_result.get("predictions", {})
                result.confidence = reasoner_result.get("confidence", 0.7)
                result.success = reasoner_result.get("success", False)

                # Ajouter les étapes de raisonnement
                for i, step in enumerate(reasoner_result.get("steps", [])):
                    result.steps.append(InferenceStep(
                        step_id=str(uuid.uuid4()),
                        type="neural",
                        description=f"Étape neuronale {i+1}",
                        reasoning=step.get("explanation", ""),
                        input_facts=step.get("input", {}),
                        output_facts=step.get("output", {}),
                        confidence=step.get("confidence", 0.7)
                    ))

                return result

            except Exception as e:
                self.logger.error(f"Erreur du raisonneur neuronal: {str(e)}")
                # Continuer avec l'implémentation de secours

        # Implémentation simplifiée de secours (si pas de raisonneur ou erreur)
        # Ceci n'est qu'une simulation très basique de raisonnement neuronal

        # Simuler une étape d'encodage
        result.steps.append(InferenceStep(
            step_id=str(uuid.uuid4()),
            type="neural_encoding",
            description="Encodage neuronal de la requête et du contexte",
            confidence=0.8,
            reasoning="Transformation de la requête et du contexte en représentations vectorielles."
        ))

        # Simuler une recherche de similitude
        has_semantic_memory = self.semantic_memory is not None

        if has_semantic_memory and query.question:
            # Simuler une recherche dans la mémoire sémantique
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="neural_retrieval",
                description="Recherche de concepts similaires en mémoire sémantique",
                confidence=0.6,
                reasoning="Recherche de concepts et relations pertinents à la requête."
            ))

            # Simuler des réponses basées sur les variables cibles
            answers = {}
            for var in query.target_variables:
                # Simuler une prédiction avec une valeur aléatoire pour la démonstration
                # Dans une vraie implémentation, cela serait une inférence basée sur embeddings
                answers[var] = f"valeur_prédite_pour_{var}"

                result.steps.append(InferenceStep(
                    step_id=str(uuid.uuid4()),
                    type="neural_prediction",
                    description=f"Prédiction neuronale pour la variable {var}",
                    output_facts={var: answers[var]},
                    confidence=0.65,
                    reasoning=f"Inférence de la valeur la plus probable pour {var} basée sur similitudes."
                ))

            result.answers = answers
            result.success = bool(result.answers)
            result.confidence = 0.65 if result.success else 0.0
        else:
            # Pas de mémoire sémantique ou pas de question
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="neural_fallback",
                description="Raisonnement neuronal non applicable",
                confidence=0.0,
                reasoning="Impossible d'appliquer un raisonnement neuronal dans ce contexte."
            ))

            result.success = False
            result.confidence = 0.0

        return result

    def _apply_neuro_symbolic_reasoning(self, query: InferenceQuery) -> InferenceResult:
        """
        Applique un raisonnement neuro-symbolique unifié à une requête.
        Utilise des frameworks comme LNN ou DeepProbLog pour combiner
        les approches symboliques et neuronales en un seul modèle.

        Args:
            query: La requête d'inférence

        Returns:
            InferenceResult: Résultat de l'inférence neuro-symbolique
        """
        # Cette méthode est un placeholder - dans une implémentation réelle,
        # elle utiliserait un framework neuro-symbolique complet

        # Initialiser le résultat
        result = InferenceResult(query_id=query.query_id)
        result.steps = []

        # Vérifier si nous avons un modèle LNN initialisé
        if "lnn" in self.neural_models and self.neural_models["lnn"]["initialized"]:
            # Simuler une étape LNN
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="neuro_symbolic_lnn",
                description="Raisonnement avec Logical Neural Network",
                confidence=0.75,
                reasoning="Application de LNN pour intégrer règles logiques et activation neuronale."
            ))

            # Simuler la propagation de l'activation dans LNN
            result.steps.append(InferenceStep(
                step_id=str(uuid.uuid4()),
                type="neuro_symbolic_activation",
                description="Propagation de l'activation dans le réseau logique",
                confidence=0.8,
                reasoning="Initialisation des neurones logiques et propagation des valeurs jusqu'à convergence."
            ))

            # Simuler extraction de résultats depuis LNN
            answers = {}
            for var in query.target_variables:
                # Simuler une prédiction
                answers[var] = f"ns_valeur_pour_{var}"

                result.steps.append(InferenceStep(
                    step_id=str(uuid.uuid4()),
                    type="neuro_symbolic_extraction",
                    description=f"Extraction de valeur pour {var} depuis LNN",
                    output_facts={var: answers[var]},
                    confidence=0.85,
                    reasoning=f"La valeur pour {var} a été extraite avec une confiance élevée via LNN."
                ))

            result.answers = answers
            result.success = bool(result.answers)
            result.confidence = 0.85 if result.success else 0.0

            # Dans un vrai système, les contributions seraient plus nuancées
            result.symbolic_contribution = 0.5
            result.neural_contribution = 0.5

            return result

        # Si pas de framework neuro-symbolique disponible, retomber sur l'approche hybride
        self.logger.info("Aucun framework neuro-symbolique disponible, recours à l'approche hybride")

        # Utiliser la stratégie parallèle comme fallback
        return self._parallel_strategy(query)

ModuleNotFoundError: No module named 'grandnexus'

# Planning

In [ ]:
"""
planning.py - Planning and Goal Decomposition Module for GrandNexus

This module provides planning capabilities for GrandNexus, enabling the system to:
- Decompose high-level goals into actionable sub-goals and tasks
- Create and manage execution plans with dependencies
- Monitor plan execution and handle contingencies
- Adapt plans based on feedback and changing contexts

The planning approach combines:
- Hierarchical Task Networks (HTN) for goal decomposition
- STRIPS-like operators for action representation
- Dynamic re-planning capabilities for adaptation
- Belief-Desire-Intention (BDI) inspired goal management

Design principles:
- Flexible: Plans can be modified during execution
- Contextual: Planning takes into account the current state and context
- Emergent: Complex plans can emerge from simpler building blocks
- Adaptive: Plans adjust to feedback and changing circumstances
"""

import logging
import threading
import uuid
import time
import heapq
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Set, Tuple, Callable, Union
from collections import deque, defaultdict

# Import from GrandNexus core modules
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.reasoning.reasoning_core import ReasoningCore

# Define goal and plan states
class GoalState(Enum):
    """States a goal can be in"""
    PENDING = auto()    # Not yet activated
    ACTIVE = auto()     # Currently being pursued
    ACHIEVED = auto()   # Successfully completed
    FAILED = auto()     # Failed to achieve
    SUSPENDED = auto()  # Temporarily suspended
    ABANDONED = auto()  # Permanently abandoned

class TaskState(Enum):
    """States a task/action can be in"""
    PENDING = auto()    # Not yet started
    READY = auto()      # Ready to execute (all preconditions met)
    EXECUTING = auto()  # Currently executing
    COMPLETED = auto()  # Successfully completed
    FAILED = auto()     # Failed to complete
    BLOCKED = auto()    # Waiting for preconditions

class PlanningApproach(Enum):
    """Different planning approaches that can be selected"""
    HIERARCHICAL = auto()  # Hierarchical Task Network planning
    FORWARD = auto()       # Forward state-space planning
    BACKWARD = auto()      # Backward state-space planning
    PARTIAL_ORDER = auto() # Partial-order planning
    REACTIVE = auto()      # Reactive planning (minimal lookahead)
    ADAPTIVE = auto()      # Adaptive planning (context-dependent strategy)

@dataclass
class Condition:
    """Represents a condition that must be true for a task to execute or a goal to be achieved"""
    condition_type: str  # e.g., "state_check", "resource_check", "capability_check"
    parameters: Dict[str, Any] = field(default_factory=dict)
    negated: bool = False

    def check(self, state: Dict[str, Any]) -> bool:
        """
        Check if this condition is satisfied in the given state.

        Args:
            state: Current state of the world

        Returns:
            True if condition is satisfied, False otherwise
        """
        # This is a simplified implementation
        # A real implementation would have logic for different condition types

        if self.condition_type == "state_check":
            # Check if a state variable has the expected value
            var_name = self.parameters.get("variable", "")
            expected_value = self.parameters.get("value")

            if var_name in state:
                result = state[var_name] == expected_value
                return not result if self.negated else result
            return False

        elif self.condition_type == "capability_check":
            # Check if a capability is available
            capability = self.parameters.get("capability", "")
            result = capability in state.get("capabilities", [])
            return not result if self.negated else result

        elif self.condition_type == "resource_check":
            # Check if a resource level meets the requirement
            resource = self.parameters.get("resource", "")
            min_level = self.parameters.get("min_level", 0)

            if resource in state.get("resources", {}):
                result = state["resources"][resource] >= min_level
                return not result if self.negated else result
            return False

        # Add more condition types as needed

        return False

@dataclass
class Effect:
    """Represents an effect of executing a task or achieving a goal"""
    effect_type: str  # e.g., "state_change", "resource_change"
    parameters: Dict[str, Any] = field(default_factory=dict)

    def apply(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Apply this effect to the given state.

        Args:
            state: Current state of the world

        Returns:
            Updated state with effect applied
        """
        # This is a simplified implementation
        # A real implementation would have logic for different effect types

        # Make a copy of the state to avoid modifying the original
        new_state = state.copy()

        if self.effect_type == "state_change":
            # Change a state variable to a new value
            var_name = self.parameters.get("variable", "")
            new_value = self.parameters.get("value")

            if var_name:
                new_state[var_name] = new_value

        elif self.effect_type == "resource_change":
            # Change a resource level
            resource = self.parameters.get("resource", "")
            change = self.parameters.get("change", 0)

            if resource:
                if "resources" not in new_state:
                    new_state["resources"] = {}
                if resource not in new_state["resources"]:
                    new_state["resources"][resource] = 0

                new_state["resources"][resource] += change

        # Add more effect types as needed

        return new_state

@dataclass
class Task:
    """
    Represents an executable task in a plan.

    Tasks are the basic building blocks of plans. They have preconditions
    that must be met before execution, effects that occur after execution,
    and a handler function that performs the actual execution.
    """
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    name: str = ""
    description: str = ""
    preconditions: List[Condition] = field(default_factory=list)
    effects: List[Effect] = field(default_factory=list)
    handler: Optional[Callable[[Dict[str, Any]], Tuple[bool, Dict[str, Any]]]] = None
    estimated_duration: float = 0.0
    priority: float = 0.5
    state: TaskState = TaskState.PENDING
    retry_count: int = 0
    max_retries: int = 3
    failure_handlers: List[Callable[[Dict[str, Any]], None]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def can_execute(self, state: Dict[str, Any]) -> bool:
        """
        Check if all preconditions are satisfied and the task can be executed.

        Args:
            state: Current state of the world

        Returns:
            True if the task can be executed, False otherwise
        """
        for condition in self.preconditions:
            if not condition.check(state):
                return False
        return True

    def predict_effects(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Predict the state after executing this task.

        Args:
            state: Current state of the world

        Returns:
            Predicted state after task execution
        """
        new_state = state.copy()

        for effect in self.effects:
            new_state = effect.apply(new_state)

        return new_state

    def execute(self, state: Dict[str, Any]) -> Tuple[bool, Dict[str, Any]]:
        """
        Execute the task with the given state.

        Args:
            state: Current state of the world

        Returns:
            Tuple of (success, new_state)
        """
        if not self.can_execute(state):
            return False, state

        self.state = TaskState.EXECUTING

        try:
            if self.handler:
                # Call the handler function
                success, new_state = self.handler(state)
            else:
                # No handler, just apply the effects
                success = True
                new_state = self.predict_effects(state)

            if success:
                self.state = TaskState.COMPLETED
            else:
                self.state = TaskState.FAILED
                self.retry_count += 1

                # Try to handle the failure
                for handler in self.failure_handlers:
                    handler(state)

                # Check if we can retry
                if self.retry_count <= self.max_retries:
                    self.state = TaskState.PENDING

            return success, new_state

        except Exception as e:
            self.state = TaskState.FAILED
            self.retry_count += 1

            # Try to handle the failure
            for handler in self.failure_handlers:
                try:
                    handler(state)
                except:
                    pass

            return False, state

@dataclass
class Goal:
    """
    Represents a goal to be achieved.

    Goals are high-level objectives that can be decomposed into sub-goals
    and eventually into tasks. They have conditions that define when the
    goal is achieved.
    """
    goal_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    name: str = ""
    description: str = ""
    achievement_conditions: List[Condition] = field(default_factory=list)
    parent_goal_id: Optional[str] = None
    sub_goals: List[str] = field(default_factory=list)
    tasks: List[str] = field(default_factory=list)
    state: GoalState = GoalState.PENDING
    priority: float = 0.5
    deadline: Optional[float] = None
    creation_time: float = field(default_factory=time.time)
    last_updated: float = field(default_factory=time.time)
    achievement_time: Optional[float] = None
    context: Dict[str, Any] = field(default_factory=dict)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def is_achieved(self, state: Dict[str, Any]) -> bool:
        """
        Check if the goal is achieved in the given state.

        Args:
            state: Current state of the world

        Returns:
            True if the goal is achieved, False otherwise
        """
        for condition in self.achievement_conditions:
            if not condition.check(state):
                return False
        return True

    def has_expired(self, current_time: float) -> bool:
        """
        Check if the goal has expired (past its deadline).

        Args:
            current_time: Current time

        Returns:
            True if the goal has expired, False otherwise
        """
        return self.deadline is not None and current_time > self.deadline

    def update_state(self, new_state: GoalState, current_time: float) -> None:
        """
        Update the state of the goal.

        Args:
            new_state: New goal state
            current_time: Current time
        """
        self.state = new_state
        self.last_updated = current_time

        if new_state == GoalState.ACHIEVED:
            self.achievement_time = current_time

@dataclass
class Plan:
    """
    Represents a plan for achieving a goal.

    A plan consists of a collection of tasks (and potentially sub-plans)
    with dependencies between them. It specifies the order in which tasks
    should be executed to achieve a goal.
    """
    plan_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    goal_id: str = ""
    tasks: Dict[str, Task] = field(default_factory=dict)
    task_dependencies: Dict[str, List[str]] = field(default_factory=lambda: defaultdict(list))
    sub_plans: Dict[str, "Plan"] = field(default_factory=dict)
    state: Dict[str, Any] = field(default_factory=dict)
    current_task_id: Optional[str] = None
    completed_tasks: List[str] = field(default_factory=list)
    failed_tasks: List[str] = field(default_factory=list)
    creation_time: float = field(default_factory=time.time)
    last_updated: float = field(default_factory=time.time)
    completion_time: Optional[float] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def get_ready_tasks(self) -> List[str]:
        """
        Get tasks that are ready to be executed (all dependencies completed).

        Returns:
            List of task IDs
        """
        ready_tasks = []

        for task_id, task in self.tasks.items():
            if task.state != TaskState.PENDING:
                continue

            dependencies = self.task_dependencies.get(task_id, [])
            all_deps_completed = True

            for dep_id in dependencies:
                dep_task = self.tasks.get(dep_id)
                if dep_task is None or dep_task.state != TaskState.COMPLETED:
                    all_deps_completed = False
                    break

            if all_deps_completed and task.can_execute(self.state):
                ready_tasks.append(task_id)

        return ready_tasks

    def is_completed(self) -> bool:
        """
        Check if the plan is completed (all tasks completed).

        Returns:
            True if the plan is completed, False otherwise
        """
        return all(task.state == TaskState.COMPLETED for task in self.tasks.values())

    def is_failed(self) -> bool:
        """
        Check if the plan has failed (any task failed with no retries left).

        Returns:
            True if the plan has failed, False otherwise
        """
        return any(task.state == TaskState.FAILED and task.retry_count > task.max_retries
                   for task in self.tasks.values())

    def get_progress(self) -> float:
        """
        Get the progress of the plan (percentage of completed tasks).

        Returns:
            Progress as a float between 0 and 1
        """
        if not self.tasks:
            return 0.0

        completed = sum(1 for task in self.tasks.values() if task.state == TaskState.COMPLETED)
        return completed / len(self.tasks)

    def update_state(self, new_state: Dict[str, Any]) -> None:
        """
        Update the state of the plan.

        Args:
            new_state: New state
        """
        self.state.update(new_state)
        self.last_updated = time.time()

class PlanningModule:
    """
    Planning module for GrandNexus.

    This module is responsible for:
    - Goal management and decomposition
    - Plan generation and execution
    - Task scheduling and monitoring
    - Adaptive re-planning

    It integrates with other cognitive modules to create, execute,
    and adapt plans based on the system's goals and context.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 reasoning_core: Optional[ReasoningCore] = None):
        """
        Initialize the planning module.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            working_memory: Reference to working memory for context
            semantic_memory: Reference to semantic memory for knowledge
            episodic_memory: Reference to episodic memory for past experiences
            reasoning_core: Reference to reasoning core for inference
        """
        self.logger = logging.getLogger("GrandNexus.Reasoning.Planning")
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory
        self.reasoning_core = reasoning_core

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.lock = threading.RLock()

        # Goals and plans
        self.goals: Dict[str, Goal] = {}
        self.plans: Dict[str, Plan] = {}

        # Task library (reusable task templates)
        self.task_library: Dict[str, Task] = {}

        # Goal decomposers (functions that decompose goals into sub-goals and tasks)
        self.goal_decomposers: Dict[str, Callable[[Goal, Dict[str, Any]], Tuple[List[Goal], List[Task]]]] = {}

        # Planning algorithms
        self.planning_algorithms: Dict[PlanningApproach, Callable[[Goal, Dict[str, Any]], Plan]] = {}

        # Active execution
        self.executing_plans: Set[str] = set()
        self.execution_thread: Optional[threading.Thread] = None
        self.stop_execution = threading.Event()

        # Performance metrics
        self.metrics: Dict[str, List[float]] = defaultdict(list)

        self.logger.info(f"PlanningModule initialized with ID={self.instance_id}")

    def start(self) -> bool:
        """Start the planning module"""
        with self.lock:
            if self.running:
                return True

            # Initialize planning algorithms
            self._register_planning_algorithms()

            # Initialize task library
            self._initialize_task_library()

            # Initialize goal decomposers
            self._register_goal_decomposers()

            # Start execution thread
            self.stop_execution.clear()
            self.execution_thread = threading.Thread(target=self._execution_loop, daemon=True)
            self.execution_thread.start()

            self.running = True
            self.logger.info("PlanningModule started")
            return True

    def stop(self) -> bool:
        """Stop the planning module"""
        with self.lock:
            if not self.running:
                return True

            # Stop execution thread
            self.stop_execution.set()
            if self.execution_thread:
                self.execution_thread.join(timeout=5.0)
                self.execution_thread = None

            self.running = False
            self.logger.info("PlanningModule stopped")
            return True

    def add_goal(self, goal: Goal) -> str:
        """
        Add a new goal to be planned for.

        Args:
            goal: The goal to add

        Returns:
            Goal ID
        """
        with self.lock:
            self.goals[goal.goal_id] = goal
            self.logger.info(f"Added goal {goal.goal_id}: {goal.name}")
            return goal.goal_id

    def get_goal(self, goal_id: str) -> Optional[Goal]:
        """
        Get a goal by ID.

        Args:
            goal_id: Goal ID

        Returns:
            Goal object, or None if not found
        """
        with self.lock:
            return self.goals.get(goal_id)

    def update_goal(self, goal_id: str, updates: Dict[str, Any]) -> bool:
        """
        Update a goal's properties.

        Args:
            goal_id: Goal ID
            updates: Dictionary of properties to update

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            goal = self.goals.get(goal_id)
            if not goal:
                return False

            # Update goal properties
            for key, value in updates.items():
                if hasattr(goal, key):
                    setattr(goal, key, value)

            goal.last_updated = time.time()
            return True

    def remove_goal(self, goal_id: str) -> bool:
        """
        Remove a goal.

        Args:
            goal_id: Goal ID

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            if goal_id not in self.goals:
                return False

            # Remove associated plans
            for plan_id, plan in list(self.plans.items()):
                if plan.goal_id == goal_id:
                    if plan_id in self.executing_plans:
                        self.executing_plans.remove(plan_id)
                    del self.plans[plan_id]

            del self.goals[goal_id]
            return True

    def create_plan_for_goal(self, goal_id: str, approach: PlanningApproach = PlanningApproach.HIERARCHICAL,
                            context: Optional[Dict[str, Any]] = None) -> Optional[str]:
        """
        Create a plan for achieving a goal.

        Args:
            goal_id: Goal ID
            approach: Planning approach to use
            context: Additional context for planning

        Returns:
            Plan ID if successful, None otherwise
        """
        with self.lock:
            goal = self.goals.get(goal_id)
            if not goal:
                self.logger.warning(f"Goal {goal_id} not found")
                return None

            # Check if there's already an active plan for this goal
            for plan in self.plans.values():
                if plan.goal_id == goal_id and plan.goal_id in self.executing_plans:
                    self.logger.warning(f"Already have an executing plan for goal {goal_id}")
                    return plan.plan_id

            # Prepare planning context
            if context is None:
                context = {}

            # Get the current state
            current_state = self._get_current_state()

            # Choose planning algorithm
            planning_algo = self.planning_algorithms.get(approach)
            if not planning_algo:
                self.logger.error(f"No planning algorithm available for approach {approach}")
                return None

            try:
                # Generate plan
                plan = planning_algo(goal, {**current_state, **context})

                # Store plan
                self.plans[plan.plan_id] = plan

                self.logger.info(f"Created plan {plan.plan_id} for goal {goal_id}")
                return plan.plan_id
            except Exception as e:
                self.logger.error(f"Error creating plan for goal {goal_id}: {str(e)}")
                return None

    def get_plan(self, plan_id: str) -> Optional[Plan]:
        """
        Get a plan by ID.

        Args:
            plan_id: Plan ID

        Returns:
            Plan object, or None if not found
        """
        with self.lock:
            return self.plans.get(plan_id)

    def execute_plan(self, plan_id: str) -> bool:
        """
        Start executing a plan.

        Args:
            plan_id: Plan ID

        Returns:
            True if execution started, False otherwise
        """
        with self.lock:
            plan = self.plans.get(plan_id)
            if not plan:
                self.logger.warning(f"Plan {plan_id} not found")
                return False

            # Add to executing plans
            self.executing_plans.add(plan_id)

            # Update goal state
            goal = self.goals.get(plan.goal_id)
            if goal:
                goal.update_state(GoalState.ACTIVE, time.time())

            self.logger.info(f"Started execution of plan {plan_id}")
            return True

    def pause_plan(self, plan_id: str) -> bool:
        """
        Pause execution of a plan.

        Args:
            plan_id: Plan ID

        Returns:
            True if paused, False otherwise
        """
        with self.lock:
            if plan_id not in self.executing_plans:
                return False

            self.executing_plans.remove(plan_id)

            plan = self.plans.get(plan_id)
            if plan and plan.current_task_id:
                # Mark current task as pending
                task = plan.tasks.get(plan.current_task_id)
                if task and task.state == TaskState.EXECUTING:
                    task.state = TaskState.PENDING

            # Update goal state
            if plan:
                goal = self.goals.get(plan.goal_id)
                if goal:
                    goal.update_state(GoalState.SUSPENDED, time.time())

            self.logger.info(f"Paused execution of plan {plan_id}")
            return True

    def cancel_plan(self, plan_id: str) -> bool:
        """
        Cancel execution of a plan.

        Args:
            plan_id: Plan ID

        Returns:
            True if cancelled, False otherwise
        """
        with self.lock:
            if plan_id in self.executing_plans:
                self.executing_plans.remove(plan_id)

            plan = self.plans.get(plan_id)
            if not plan:
                return False

            # Update goal state
            goal = self.goals.get(plan.goal_id)
            if goal:
                goal.update_state(GoalState.ABANDONED, time.time())

            # Remove plan
            del self.plans[plan_id]

            self.logger.info(f"Cancelled plan {plan_id}")
            return True

    def get_active_goals(self) -> List[Goal]:
        """
        Get all active goals.

        Returns:
            List of active goals
        """
        with self.lock:
            return [goal for goal in self.goals.values() if goal.state == GoalState.ACTIVE]

    def get_executing_plans(self) -> List[Plan]:
        """
        Get all currently executing plans.

        Returns:
            List of executing plans
        """
        with self.lock:
            return [self.plans[plan_id] for plan_id in self.executing_plans if plan_id in self.plans]

    def add_task_to_library(self, task_template: Task) -> str:
        """
        Add a task template to the task library.

        Args:
            task_template: Task template to add

        Returns:
            Task ID
        """
        with self.lock:
            self.task_library[task_template.task_id] = task_template
            return task_template.task_id

    def get_task_from_library(self, task_id: str) -> Optional[Task]:
        """
        Get a task template from the library.

        Args:
            task_id: Task ID

        Returns:
            Task template, or None if not found
        """
        with self.lock:
            return self.task_library.get(task_id)

    def register_goal_decomposer(self, goal_type: str,
                                decomposer: Callable[[Goal, Dict[str, Any]], Tuple[List[Goal], List[Task]]]) -> None:
        """
        Register a goal decomposer function.

        Args:
            goal_type: Type of goal this decomposer handles
            decomposer: Function that decomposes goals into sub-goals and tasks
        """
        with self.lock:
            self.goal_decomposers[goal_type] = decomposer
            self.logger.info(f"Registered goal decomposer for {goal_type}")

    def get_planning_metrics(self) -> Dict[str, Any]:
        """
        Get planning performance metrics.

        Returns:
            Dictionary of metrics
        """
        with self.lock:
            metrics = {}
            for key, values in self.metrics.items():
                if values:
                    metrics[key] = {
                        "current": values[-1],
                        "average": sum(values) / len(values),
                        "min": min(values),
                        "max": max(values)
                    }
            return metrics

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_planning_algorithms(self) -> None:
        """Register available planning algorithms"""
        self.planning_algorithms[PlanningApproach.HIERARCHICAL] = self._hierarchical_planning
        self.planning_algorithms[PlanningApproach.FORWARD] = self._forward_planning
        self.planning_algorithms[PlanningApproach.BACKWARD] = self._backward_planning
        self.planning_algorithms[PlanningApproach.PARTIAL_ORDER] = self._partial_order_planning
        self.planning_algorithms[PlanningApproach.REACTIVE] = self._reactive_planning
        self.planning_algorithms[PlanningApproach.ADAPTIVE] = self._adaptive_planning

    def _initialize_task_library(self) -> None:
        """Initialize the task library with basic task templates"""
        # This would be populated with domain-specific tasks
        # Here we just provide a simple example

        # Example: Wait task
        wait_task = Task(
            task_id="task_wait",
            name="Wait",
            description="Wait for a specified duration",
            handler=lambda state: (True, state),  # Just returns success
            estimated_duration=1.0
        )
        self.task_library[wait_task.task_id] = wait_task

        # Additional tasks would be defined here

    def _register_goal_decomposers(self) -> None:
        """Register goal decomposers for different goal types"""
        # This would be populated with domain-specific decomposers
        # Here we just provide a simple example

        # Example: Generic goal decomposer
        self.register_goal_decomposer("generic", self._generic_goal_decomposer)

        # Additional decomposers would be registered here

    def _generic_goal_decomposer(self, goal: Goal, context: Dict[str, Any]) -> Tuple[List[Goal], List[Task]]:
        """
        Generic goal decomposer that handles simple goals.

        Args:
            goal: Goal to decompose
            context: Planning context

        Returns:
            Tuple of (sub-goals, tasks)
        """
        # This is a placeholder implementation
        # In a real system, this would have more sophisticated decomposition logic

        # Example: Create a simple task to achieve the goal
        task = Task(
            task_id=f"task_{uuid.uuid4().hex[:8]}",
            name=f"Achieve {goal.name}",
            description=f"Generic task to achieve goal: {goal.description}",
            effects=[
                # Assume the task will achieve all goal conditions
                Effect(
                    effect_type="state_change",
                    parameters={
                        "variable": f"goal_{goal.goal_id}_achieved",
                        "value": True
                    }
                )
            ]
        )

        return [], [task]

    def _get_current_state(self) -> Dict[str, Any]:
        """
        Get the current state of the world for planning.

        Returns:
            Dictionary representing the current state
        """
        state = {}

        # Get basic system state
        state["current_time"] = time.time()
        state["capabilities"] = ["reasoning", "perception", "learning"]  # Basic capabilities

        # Get state from working memory if available
        if self.working_memory:
            try:
                items = self.working_memory.get_all_items()

                # Extract relevant state information from working memory
                for item in items:
                    if item.content and isinstance(item.content, dict) and "state" in item.content:
                        state.update(item.content["state"])
            except Exception as e:
                self.logger.warning(f"Error getting state from working memory: {str(e)}")

        # Include information about active goals
        active_goals = [goal.goal_id for goal in self.goals.values() if goal.state == GoalState.ACTIVE]
        state["active_goals"] = active_goals

        return state

    def _hierarchical_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Hierarchical Task Network (HTN) planning algorithm.

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        self.logger.info(f"Creating hierarchical plan for goal {goal.goal_id}: {goal.name}")

        # Create a new plan
        plan = Plan(
            plan_id=f"plan_{uuid.uuid4().hex[:8]}",
            goal_id=goal.goal_id,
            state=context.copy()
        )

        # Determine goal type
        goal_type = goal.metadata.get("type", "generic")

        # Get appropriate decomposer
        decomposer = self.goal_decomposers.get(goal_type, self.goal_decomposers.get("generic"))
        if not decomposer:
            raise ValueError(f"No decomposer available for goal type: {goal_type}")

        # Decompose goal into sub-goals and tasks
        sub_goals, tasks = decomposer(goal, context)

        # Add tasks to the plan
        for task in tasks:
            plan.tasks[task.task_id] = task

        # Process sub-goals recursively
        for sub_goal in sub_goals:
            # Store the sub-goal
            self.goals[sub_goal.goal_id] = sub_goal

            # Link to parent goal
            goal.sub_goals.append(sub_goal.goal_id)

            # Create a sub-plan for the sub-goal
            sub_plan = self._hierarchical_planning(sub_goal, context)

            # Add sub-plan to the main plan
            plan.sub_plans[sub_plan.plan_id] = sub_plan

            # Create a task to execute the sub-plan
            sub_plan_task = Task(
                task_id=f"task_exec_plan_{sub_plan.plan_id}",
                name=f"Execute plan for {sub_goal.name}",
                description=f"Execute sub-plan to achieve {sub_goal.name}",
                handler=lambda state, plan_id=sub_plan.plan_id: self._execute_sub_plan(plan_id, state)
            )

            plan.tasks[sub_plan_task.task_id] = sub_plan_task

        # Set up task dependencies based on effects and preconditions
        self._setup_task_dependencies(plan)

        return plan

    def _forward_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Forward state-space planning algorithm.

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        # This is a simplified implementation of forward planning
        self.logger.info(f"Creating forward plan for goal {goal.goal_id}: {goal.name}")

        # Create a new plan
        plan = Plan(
            plan_id=f"plan_{uuid.uuid4().hex[:8]}",
            goal_id=goal.goal_id,
            state=context.copy()
        )

        # Get all applicable task templates from the library
        applicable_tasks = []
        for task_id, task_template in self.task_library.items():
            # Create a copy of the template
            task = Task(
                task_id=f"task_{uuid.uuid4().hex[:8]}",
                name=task_template.name,
                description=task_template.description,
                preconditions=task_template.preconditions.copy(),
                effects=task_template.effects.copy(),
                handler=task_template.handler,
                estimated_duration=task_template.estimated_duration,
                priority=task_template.priority
            )
            applicable_tasks.append(task)

        # Start from the initial state
        current_state = context.copy()

        # Keep adding tasks until the goal is achieved or no more progress can be made
        while not goal.is_achieved(current_state):
            # Find a task that can be executed in the current state
            next_task = None
            next_state = None

            for task in applicable_tasks:
                if task.can_execute(current_state):
                    # Predict the effects of executing this task
                    predicted_state = task.predict_effects(current_state)

                    # Check if this gets us closer to the goal
                    # (Simple heuristic: count how many goal conditions are satisfied)
                    current_satisfied = sum(1 for cond in goal.achievement_conditions if cond.check(current_state))
                    predicted_satisfied = sum(1 for cond in goal.achievement_conditions if cond.check(predicted_state))

                    if predicted_satisfied > current_satisfied:
                        next_task = task
                        next_state = predicted_state
                        break

            if next_task is None:
                # No task can get us closer to the goal
                self.logger.warning(f"Could not find a path to achieve goal {goal.goal_id}")
                break

            # Add the task to the plan
            plan.tasks[next_task.task_id] = next_task

            # Update the current state
            current_state = next_state

            # Remove the task from the applicable list to avoid adding it again
            applicable_tasks.remove(next_task)

        # Set up task dependencies (sequential for forward planning)
        task_ids = list(plan.tasks.keys())
        for i in range(1, len(task_ids)):
            plan.task_dependencies[task_ids[i]].append(task_ids[i-1])

        return plan

    def _backward_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Backward state-space planning algorithm.

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        # This is a simplified implementation of backward planning
        self.logger.info(f"Creating backward plan for goal {goal.goal_id}: {goal.name}")

        # Create a new plan
        plan = Plan(
            plan_id=f"plan_{uuid.uuid4().hex[:8]}",
            goal_id=goal.goal_id,
            state=context.copy()
        )

        # Start with the goal conditions as the current sub-goals
        subgoals = goal.achievement_conditions.copy()

        # Keep track of the planned tasks in reverse order
        planned_tasks = []

        # Keep adding tasks until all sub-goals are achieved in the initial state
        while subgoals:
            subgoal = subgoals.pop(0)

            # Check if the sub-goal is already satisfied in the initial state
            if subgoal.check(context):
                continue

            # Find a task that can achieve this sub-goal
            achieving_task = None

            for task_id, task_template in self.task_library.items():
                # Check if any of the task's effects would achieve the sub-goal
                for effect in task_template.effects:
                    # This is a simplified check - in a real system,
                    # we would need a more sophisticated check
                    if (effect.effect_type == "state_change" and
                        effect.parameters.get("variable") == subgoal.parameters.get("variable") and
                        effect.parameters.get("value") == subgoal.parameters.get("value")):

                        # Create a copy of the template
                        achieving_task = Task(
                            task_id=f"task_{uuid.uuid4().hex[:8]}",
                            name=task_template.name,
                            description=task_template.description,
                            preconditions=task_template.preconditions.copy(),
                            effects=task_template.effects.copy(),
                            handler=task_template.handler,
                            estimated_duration=task_template.estimated_duration,
                            priority=task_template.priority
                        )
                        break

                if achieving_task:
                    break

            if achieving_task is None:
                # No task can achieve this sub-goal
                self.logger.warning(f"Could not find a task to achieve sub-goal: {subgoal}")
                raise ValueError(f"No task can achieve sub-goal: {subgoal}")

            # Add the task's preconditions as new sub-goals
            subgoals.extend(achieving_task.preconditions)

            # Add the task to the planned tasks
            planned_tasks.append(achieving_task)

        # Reverse the list of tasks to get the correct order
        planned_tasks.reverse()

        # Add tasks to the plan
        for task in planned_tasks:
            plan.tasks[task.task_id] = task

        # Set up task dependencies (sequential for backward planning)
        task_ids = list(plan.tasks.keys())
        for i in range(1, len(task_ids)):
            plan.task_dependencies[task_ids[i]].append(task_ids[i-1])

        return plan

    def _partial_order_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Partial-Order Planning (POP) algorithm.

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        # This would be a more complex implementation of POP
        # For simplicity, we'll delegate to hierarchical planning for now
        self.logger.info(f"Delegating partial-order planning to hierarchical planning for goal {goal.goal_id}")
        return self._hierarchical_planning(goal, context)

    def _reactive_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Reactive planning algorithm (minimal lookahead).

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        # This is a simplified implementation of reactive planning
        self.logger.info(f"Creating reactive plan for goal {goal.goal_id}: {goal.name}")

        # Create a new plan
        plan = Plan(
            plan_id=f"plan_{uuid.uuid4().hex[:8]}",
            goal_id=goal.goal_id,
            state=context.copy()
        )

        # For reactive planning, we just create a single "monitor and react" task
        react_task = Task(
            task_id=f"task_react_{uuid.uuid4().hex[:8]}",
            name=f"Monitor and react for {goal.name}",
            description=f"Continuously monitor the environment and react to achieve {goal.name}",
            handler=lambda state: self._reactive_handler(goal, state)
        )

        plan.tasks[react_task.task_id] = react_task

        return plan

    def _adaptive_planning(self, goal: Goal, context: Dict[str, Any]) -> Plan:
        """
        Adaptive planning algorithm (selects best approach based on context).

        Args:
            goal: Goal to plan for
            context: Planning context

        Returns:
            A Plan to achieve the goal
        """
        # Analyze the goal and context to select the best planning approach
        self.logger.info(f"Selecting best planning approach for goal {goal.goal_id}: {goal.name}")

        # Check if the goal has a specific planning approach specified
        if "preferred_approach" in goal.metadata:
            approach_name = goal.metadata["preferred_approach"]
            if hasattr(PlanningApproach, approach_name.upper()):
                approach = getattr(PlanningApproach, approach_name.upper())
                self.logger.info(f"Using preferred approach: {approach}")
                return self.planning_algorithms[approach](goal, context)

        # Criteria for selecting an approach

        # If the goal is complex (has many achievement conditions)
        if len(goal.achievement_conditions) > 3:
            self.logger.info("Complex goal detected, using hierarchical planning")
            return self._hierarchical_planning(goal, context)

        # If the goal is time-critical (has a tight deadline)
        if goal.deadline and goal.deadline - time.time() < 60:  # less than 60 seconds
            self.logger.info("Time-critical goal detected, using reactive planning")
            return self._reactive_planning(goal, context)

        # If the goal requires looking ahead
        if "requires_lookahead" in goal.metadata and goal.metadata["requires_lookahead"]:
            self.logger.info("Goal requires lookahead, using forward planning")
            return self._forward_planning(goal, context)

        # Default to hierarchical planning
        self.logger.info("Using default hierarchical planning")
        return self._hierarchical_planning(goal, context)

    def _setup_task_dependencies(self, plan: Plan) -> None:
        """
        Set up dependencies between tasks based on preconditions and effects.

        Args:
            plan: The plan to set up dependencies for
        """
        # Clear existing dependencies
        plan.task_dependencies = defaultdict(list)

        # For each task
        for task_id, task in plan.tasks.items():
            # Check which other tasks need to come before this one
            for other_id, other_task in plan.tasks.items():
                if task_id == other_id:
                    continue

                # Check if any of other_task's effects satisfy task's preconditions
                for precond in task.preconditions:
                    for effect in other_task.effects:
                        if (effect.effect_type == "state_change" and
                            effect.parameters.get("variable") == precond.parameters.get("variable") and
                            effect.parameters.get("value") == precond.parameters.get("value")):

                            # other_task needs to come before task
                            plan.task_dependencies[task_id].append(other_id)
                            break

    def _execution_loop(self) -> None:
        """Background thread for executing plans"""
        self.logger.info("Plan execution loop started")

        while not self.stop_execution.is_set():
            try:
                # Process each executing plan
                with self.lock:
                    executing_plans = list(self.executing_plans)

                for plan_id in executing_plans:
                    try:
                        self._process_plan(plan_id)
                    except Exception as e:
                        self.logger.error(f"Error processing plan {plan_id}: {str(e)}")

                # Sleep briefly to avoid hogging the CPU
                time.sleep(0.1)

            except Exception as e:
                self.logger.error(f"Error in execution loop: {str(e)}")

        self.logger.info("Plan execution loop stopped")

    def _process_plan(self, plan_id: str) -> None:
        """
        Process a single plan, executing ready tasks.

        Args:
            plan_id: ID of the plan to process
        """
        with self.lock:
            plan = self.plans.get(plan_id)
            if not plan:
                if plan_id in self.executing_plans:
                    self.executing_plans.remove(plan_id)
                return

            # Check if the plan is completed or failed
            if plan.is_completed():
                self.logger.info(f"Plan {plan_id} completed successfully")
                self.executing_plans.remove(plan_id)
                plan.completion_time = time.time()

                # Update goal state
                goal = self.goals.get(plan.goal_id)
                if goal:
                    goal.update_state(GoalState.ACHIEVED, time.time())

                # Record metrics
                self.metrics["plan_success_rate"].append(1.0)
                return

            if plan.is_failed():
                self.logger.warning(f"Plan {plan_id} failed")
                self.executing_plans.remove(plan_id)

                # Update goal state
                goal = self.goals.get(plan.goal_id)
                if goal:
                    goal.update_state(GoalState.FAILED, time.time())

                # Record metrics
                self.metrics["plan_success_rate"].append(0.0)
                return

            # Get ready tasks
            ready_tasks = plan.get_ready_tasks()

            if not ready_tasks and not plan.current_task_id:
                self.logger.warning(f"Plan {plan_id} has no ready tasks")
                return

            # If there's a current task, check its status
            if plan.current_task_id:
                current_task = plan.tasks.get(plan.current_task_id)

                if current_task and current_task.state == TaskState.EXECUTING:
                    # Task is still executing, nothing to do
                    return

                elif current_task and current_task.state == TaskState.COMPLETED:
                    # Task completed, clear current task
                    plan.completed_tasks.append(plan.current_task_id)
                    plan.current_task_id = None

                elif current_task and current_task.state == TaskState.FAILED:
                    # Task failed
                    plan.failed_tasks.append(plan.current_task_id)
                    plan.current_task_id = None

                    # Check if we can retry
                    if current_task.retry_count <= current_task.max_retries:
                        # Reset task state
                        current_task.state = TaskState.PENDING
                    else:
                        # Can't retry, mark as failed
                        self.logger.warning(f"Task {current_task.task_id} failed with no retries left")
                else:
                    # Invalid current task
                    plan.current_task_id = None

            # If no current task and there are ready tasks, start one
            if not plan.current_task_id and ready_tasks:
                # Sort ready tasks by priority (highest first)
                ready_tasks.sort(key=lambda task_id: plan.tasks[task_id].priority, reverse=True)

                # Get the highest priority task
                next_task_id = ready_tasks[0]
                next_task = plan.tasks[next_task_id]

                self.logger.info(f"Executing task {next_task_id}: {next_task.name}")

                # Set as current task
                plan.current_task_id = next_task_id

                # Execute the task
                success, new_state = next_task.execute(plan.state)

                # Update plan state
                plan.update_state(new_state)

                # Record metrics
                self.metrics["task_success_rate"].append(1.0 if success else 0.0)

    def _execute_sub_plan(self, plan_id: str, state: Dict[str, Any]) -> Tuple[bool, Dict[str, Any]]:
        """
        Execute a sub-plan as part of a task handler.

        Args:
            plan_id: ID of the sub-plan to execute
            state: Current state

        Returns:
            Tuple of (success, new_state)
        """
        with self.lock:
            sub_plan = self.plans.get(plan_id)
            if not sub_plan:
                return False, state

            # Initialize sub-plan with current state
            sub_plan.state = state.copy()

            # Execute tasks in the sub-plan
            completed_tasks = []

            # Keep executing tasks until the plan is complete or failed
            while not sub_plan.is_completed() and not sub_plan.is_failed():
                # Get ready tasks
                ready_tasks = sub_plan.get_ready_tasks()

                if not ready_tasks:
                    break

                # Sort ready tasks by priority (highest first)
                ready_tasks.sort(key=lambda task_id: sub_plan.tasks[task_id].priority, reverse=True)

                # Execute each ready task
                for task_id in ready_tasks:
                    task = sub_plan.tasks[task_id]

                    self.logger.info(f"Executing sub-plan task {task_id}: {task.name}")

                    # Execute the task
                    success, new_state = task.execute(sub_plan.state)

                    # Update sub-plan state
                    sub_plan.update_state(new_state)

                    if success:
                        task.state = TaskState.COMPLETED
                        completed_tasks.append(task_id)
                    else:
                        task.state = TaskState.FAILED
                        return False, state

            return sub_plan.is_completed(), sub_plan.state

    def _reactive_handler(self, goal: Goal, state: Dict[str, Any]) -> Tuple[bool, Dict[str, Any]]:
        """
        Handler for reactive planning tasks.

        Args:
            goal: Goal to achieve
            state: Current state

        Returns:
            Tuple of (success, new_state)
        """
        # This is a simplified implementation
        # In a real system, this would continuously monitor and react

        # Check if the goal is already achieved
        if goal.is_achieved(state):
            return True, state

        # Try to find a task that can get us closer to the goal
        for task_id, task_template in self.task_library.items():
            # Create a copy of the template
            task = Task(
                task_id=f"task_{uuid.uuid4().hex[:8]}",
                name=task_template.name,
                description=task_template.description,
                preconditions=task_template.preconditions.copy(),
                effects=task_template.effects.copy(),
                handler=task_template.handler,
                estimated_duration=task_template.estimated_duration,
                priority=task_template.priority
            )

            # Check if the task can be executed
            if task.can_execute(state):
                # Predict effects
                new_state = task.predict_effects(state)

                # Check if this gets us closer to the goal
                current_satisfied = sum(1 for cond in goal.achievement_conditions if cond.check(state))
                predicted_satisfied = sum(1 for cond in goal.achievement_conditions if cond.check(new_state))

                if predicted_satisfied > current_satisfied:
                    # Execute the task
                    self.logger.info(f"Reactively executing task: {task.name}")
                    success, actual_state = task.execute(state)

                    if success:
                        return False, actual_state  # Return False to indicate we need to continue reacting

        # No suitable task found
        self.logger.warning(f"No suitable task found for reactive planning for goal {goal.goal_id}")
        return False, state

ModuleNotFoundError: No module named 'grandnexus'

# Uncertainty

In [ ]:
"""
uncertainty.py - Uncertainty Management and Probabilistic Reasoning Module for GrandNexus

This module provides capabilities for managing uncertainty, representing probabilistic beliefs,
and performing probabilistic reasoning. It enables GrandNexus to reason about uncertain information,
model confidence levels, and make decisions in the face of incomplete or contradictory evidence.

Key features:
- Multiple uncertainty representation methods (probabilities, intervals, fuzzy logic)
- Bayesian belief networks for causal reasoning under uncertainty
- Monte Carlo sampling for complex probability distributions
- Belief revision and evidence integration
- Conflict resolution for contradictory information
- Decision-making under uncertainty with various policies
- Integration with memory and reasoning modules for consistent belief management
"""

import logging
import threading
import time
import uuid
import random
import math
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union, Set, Callable
from enum import Enum, auto
from collections import defaultdict, deque

# Try importing optional dependencies with fallbacks
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    from scipy import stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.reasoning import reasoning_core

# Uncertainty representation types
class UncertaintyType(Enum):
    """Types of uncertainty representations supported by the module"""
    PROBABILITY = auto()       # Single probability value
    INTERVAL = auto()          # Probability interval
    DISTRIBUTION = auto()      # Probability distribution
    FUZZY = auto()             # Fuzzy membership
    DEMPSTER_SHAFER = auto()   # Belief and plausibility
    POSSIBILITY = auto()       # Possibility theory
    VERBAL = auto()            # Verbal/linguistic uncertainty

@dataclass
class UncertainBelief:
    """
    Represents a belief with associated uncertainty information.
    Acts as a container for various uncertainty representations.
    """
    belief_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    content: Any = None  # The actual belief content (statement, proposition, etc.)
    uncertainty_type: UncertaintyType = UncertaintyType.PROBABILITY
    # Different uncertainty representations based on type
    probability: Optional[float] = None  # Single probability value
    interval: Optional[Tuple[float, float]] = None  # Min and max probability
    distribution_params: Optional[Dict[str, Any]] = None  # Parameters for a distribution
    fuzzy_membership: Optional[float] = None  # Fuzzy logic membership
    belief_mass: Optional[Dict[str, float]] = None  # Dempster-Shafer belief masses
    possibility: Optional[float] = None  # Possibility value
    verbal_confidence: Optional[str] = None  # Verbal expression of uncertainty

    # Metadata and tracking
    source: Optional[str] = None  # Where this belief came from
    timestamp: float = field(default_factory=time.time)
    last_updated: float = field(default_factory=time.time)
    update_count: int = 0
    evidence: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        """Validate the belief after initialization"""
        self._validate()

    def _validate(self):
        """Validate that appropriate uncertainty values are provided for the type"""
        if self.uncertainty_type == UncertaintyType.PROBABILITY and self.probability is None:
            raise ValueError("Probability value must be provided for PROBABILITY type")
        elif self.uncertainty_type == UncertaintyType.INTERVAL and self.interval is None:
            raise ValueError("Interval must be provided for INTERVAL type")
        elif self.uncertainty_type == UncertaintyType.DISTRIBUTION and self.distribution_params is None:
            raise ValueError("Distribution parameters must be provided for DISTRIBUTION type")
        elif self.uncertainty_type == UncertaintyType.FUZZY and self.fuzzy_membership is None:
            raise ValueError("Fuzzy membership must be provided for FUZZY type")
        elif self.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER and self.belief_mass is None:
            raise ValueError("Belief mass must be provided for DEMPSTER_SHAFER type")
        elif self.uncertainty_type == UncertaintyType.POSSIBILITY and self.possibility is None:
            raise ValueError("Possibility value must be provided for POSSIBILITY type")
        elif self.uncertainty_type == UncertaintyType.VERBAL and self.verbal_confidence is None:
            raise ValueError("Verbal confidence must be provided for VERBAL type")

    def update(self,
               new_uncertainty_value: Any = None,
               evidence: Optional[Dict[str, Any]] = None) -> None:
        """
        Update the belief with new uncertainty value and optional evidence.

        Args:
            new_uncertainty_value: New value for the uncertain belief
            evidence: Optional evidence for the update
        """
        self.last_updated = time.time()
        self.update_count += 1

        if evidence:
            evidence['timestamp'] = time.time()
            self.evidence.append(evidence)

        if new_uncertainty_value is not None:
            if self.uncertainty_type == UncertaintyType.PROBABILITY:
                self.probability = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.INTERVAL:
                self.interval = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.DISTRIBUTION:
                self.distribution_params = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.FUZZY:
                self.fuzzy_membership = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
                self.belief_mass = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.POSSIBILITY:
                self.possibility = new_uncertainty_value
            elif self.uncertainty_type == UncertaintyType.VERBAL:
                self.verbal_confidence = new_uncertainty_value

    def to_probability(self) -> float:
        """Convert the belief to a single probability value for comparison"""
        if self.uncertainty_type == UncertaintyType.PROBABILITY:
            return self.probability
        elif self.uncertainty_type == UncertaintyType.INTERVAL:
            # Use interval midpoint
            return (self.interval[0] + self.interval[1]) / 2
        elif self.uncertainty_type == UncertaintyType.DISTRIBUTION:
            # For distributions, return mean or expected value
            dist_type = self.distribution_params.get('type', 'normal')
            if dist_type == 'normal':
                return self.distribution_params.get('mean', 0.5)
            elif dist_type == 'beta':
                alpha = self.distribution_params.get('alpha', 1)
                beta = self.distribution_params.get('beta', 1)
                return alpha / (alpha + beta)
            else:
                return 0.5  # Default
        elif self.uncertainty_type == UncertaintyType.FUZZY:
            return self.fuzzy_membership
        elif self.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
            # Sum belief mass for the positive case
            return sum(self.belief_mass.get(k, 0) for k in self.belief_mass if 'true' in k.lower())
        elif self.uncertainty_type == UncertaintyType.POSSIBILITY:
            # Convert possibility to probability (simplified)
            return min(1.0, self.possibility * 0.5)
        elif self.uncertainty_type == UncertaintyType.VERBAL:
            # Map verbal expressions to probabilities
            verbal_map = {
                "impossible": 0.0,
                "highly unlikely": 0.1,
                "unlikely": 0.2,
                "somewhat unlikely": 0.3,
                "uncertain": 0.5,
                "somewhat likely": 0.7,
                "likely": 0.8,
                "highly likely": 0.9,
                "certain": 1.0
            }
            return verbal_map.get(self.verbal_confidence.lower(), 0.5)
        else:
            return 0.5  # Default

    def to_dict(self) -> Dict[str, Any]:
        """Convert the belief to a dictionary representation"""
        result = {
            "belief_id": self.belief_id,
            "uncertainty_type": self.uncertainty_type.name,
            "timestamp": self.timestamp,
            "last_updated": self.last_updated,
            "update_count": self.update_count,
            "source": self.source,
            "metadata": self.metadata
        }

        # Include the content if it's a simple type
        if isinstance(self.content, (str, int, float, bool, type(None))):
            result["content"] = self.content
        else:
            # Try to get a string representation
            result["content"] = str(self.content)

        # Include the appropriate uncertainty value based on type
        if self.uncertainty_type == UncertaintyType.PROBABILITY:
            result["probability"] = self.probability
        elif self.uncertainty_type == UncertaintyType.INTERVAL:
            result["interval"] = self.interval
        elif self.uncertainty_type == UncertaintyType.DISTRIBUTION:
            result["distribution_params"] = self.distribution_params
        elif self.uncertainty_type == UncertaintyType.FUZZY:
            result["fuzzy_membership"] = self.fuzzy_membership
        elif self.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
            result["belief_mass"] = self.belief_mass
        elif self.uncertainty_type == UncertaintyType.POSSIBILITY:
            result["possibility"] = self.possibility
        elif self.uncertainty_type == UncertaintyType.VERBAL:
            result["verbal_confidence"] = self.verbal_confidence

        # Include recent evidence
        if self.evidence:
            result["recent_evidence"] = self.evidence[-5:]  # Last 5 pieces

        return result

@dataclass
class BayesianNode:
    """Node in a Bayesian network for probabilistic reasoning"""
    node_id: str
    name: str
    states: List[str]  # Possible states for this variable
    cpt: Dict[str, Dict[str, float]]  # Conditional probability table
    parents: List[str] = field(default_factory=list)  # Parent node IDs
    children: List[str] = field(default_factory=list)  # Child node IDs
    description: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def get_probability(self, state: str, parent_states: Dict[str, str]) -> float:
        """
        Get probability for this node being in given state given parent states.

        Args:
            state: State to get probability for
            parent_states: Dictionary mapping parent node IDs to their states

        Returns:
            Probability value
        """
        if not self.parents:
            # If no parents, use unconditional probability
            return self.cpt.get("unconditional", {}).get(state, 0.0)

        # Create CPT lookup key from parent states
        lookup_key = "_".join(parent_states.get(parent, "") for parent in self.parents)

        # Get probability from CPT
        return self.cpt.get(lookup_key, {}).get(state, 0.0)

@dataclass
class BayesianNetwork:
    """Bayesian network for probabilistic reasoning with causal relationships"""
    network_id: str
    name: str
    nodes: Dict[str, BayesianNode] = field(default_factory=dict)
    description: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def add_node(self, node: BayesianNode) -> None:
        """Add a node to the network"""
        self.nodes[node.node_id] = node

    def add_edge(self, parent_id: str, child_id: str) -> None:
        """Add a directed edge from parent to child"""
        if parent_id not in self.nodes or child_id not in self.nodes:
            raise ValueError(f"Both nodes must exist in the network")

        # Update parent and child lists
        if parent_id not in self.nodes[child_id].parents:
            self.nodes[child_id].parents.append(parent_id)

        if child_id not in self.nodes[parent_id].children:
            self.nodes[parent_id].children.append(child_id)

    def get_node(self, node_id: str) -> Optional[BayesianNode]:
        """Get a node by ID"""
        return self.nodes.get(node_id)

    def get_all_nodes(self) -> List[BayesianNode]:
        """Get all nodes in the network"""
        return list(self.nodes.values())

class UncertaintyManager:
    """
    Core uncertainty management module for GrandNexus.

    This module provides capabilities for representing and reasoning with
    uncertain information, enabling GrandNexus to manage beliefs with
    various forms of uncertainty and perform probabilistic reasoning.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 reasoning_core: Optional[Any] = None):
        """
        Initialize the uncertainty manager.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            working_memory: Reference to working memory
            semantic_memory: Reference to semantic memory
            reasoning_core: Reference to reasoning core
        """
        self.logger = logging.getLogger("GrandNexus.Reasoning.Uncertainty")
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.reasoning_core = reasoning_core

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Belief storage
        self.beliefs = {}  # belief_id -> UncertainBelief
        self.belief_index = defaultdict(list)  # Various indices for faster lookup

        # Bayesian networks
        self.networks = {}  # network_id -> BayesianNetwork

        # Reasoning history
        self.reasoning_history = deque(maxlen=100)

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"UncertaintyManager initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the uncertainty manager"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("UncertaintyManager initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"UncertaintyManager initialization failed: {str(e)}")
                return False

    def create_belief(self, content: Any,
                     uncertainty_type: UncertaintyType = UncertaintyType.PROBABILITY,
                     uncertainty_value: Any = None,
                     source: Optional[str] = None,
                     metadata: Optional[Dict[str, Any]] = None,
                     evidence: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new uncertain belief.

        Args:
            content: The content of the belief
            uncertainty_type: Type of uncertainty representation
            uncertainty_value: Value for the uncertainty representation
            source: Source of the belief
            metadata: Additional metadata
            evidence: Evidence supporting the belief

        Returns:
            ID of the created belief
        """
        if not self.initialized:
            self.initialize()

        if metadata is None:
            metadata = {}

        # Prepare the appropriate uncertainty representation
        kwargs = {
            "content": content,
            "uncertainty_type": uncertainty_type,
            "source": source,
            "metadata": metadata
        }

        if uncertainty_type == UncertaintyType.PROBABILITY:
            kwargs["probability"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.INTERVAL:
            kwargs["interval"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.DISTRIBUTION:
            kwargs["distribution_params"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.FUZZY:
            kwargs["fuzzy_membership"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
            kwargs["belief_mass"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.POSSIBILITY:
            kwargs["possibility"] = uncertainty_value
        elif uncertainty_type == UncertaintyType.VERBAL:
            kwargs["verbal_confidence"] = uncertainty_value

        # Add evidence if provided
        if evidence:
            evidence['timestamp'] = time.time()
            kwargs["evidence"] = [evidence]

        # Create the belief
        belief = UncertainBelief(**kwargs)

        with self.lock:
            # Store the belief
            self.beliefs[belief.belief_id] = belief

            # Update indices for faster lookup
            self._update_belief_indices(belief)

            # Store in working memory if available
            if self.working_memory:
                self.working_memory.store_item(
                    content=belief.to_dict(),
                    source="uncertainty_manager",
                    metadata={
                        "type": "uncertain_belief",
                        "uncertainty_type": uncertainty_type.name
                    }
                )

        self.logger.info(f"Created uncertain belief: {belief.belief_id}")
        return belief.belief_id

    def get_belief(self, belief_id: str) -> Optional[UncertainBelief]:
        """Get a belief by ID"""
        with self.lock:
            return self.beliefs.get(belief_id)

    def query_beliefs(self, content: Optional[Any] = None,
                     uncertainty_type: Optional[UncertaintyType] = None,
                     min_probability: Optional[float] = None,
                     max_probability: Optional[float] = None,
                     source: Optional[str] = None,
                     metadata_filter: Optional[Dict[str, Any]] = None) -> List[UncertainBelief]:
        """
        Query beliefs matching given criteria.

        Args:
            content: Content to match (exact match)
            uncertainty_type: Type of uncertainty to filter for
            min_probability: Minimum probability (converted if needed)
            max_probability: Maximum probability (converted if needed)
            source: Source to filter for
            metadata_filter: Metadata key-value pairs to match

        Returns:
            List of matching beliefs
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            # Start with all beliefs
            belief_ids = set(self.beliefs.keys())

            # Apply filters
            if content is not None:
                # Filter by content (exact match for now)
                content_filter = set()
                for belief_id, belief in self.beliefs.items():
                    if belief.content == content:
                        content_filter.add(belief_id)
                belief_ids &= content_filter

            if uncertainty_type is not None:
                # Filter by uncertainty type
                type_key = f"type:{uncertainty_type.name}"
                if type_key in self.belief_index:
                    belief_ids &= set(self.belief_index[type_key])

            if min_probability is not None or max_probability is not None:
                # Filter by probability (converting if needed)
                probability_filter = set()
                for belief_id in belief_ids:
                    belief = self.beliefs[belief_id]
                    prob = belief.to_probability()

                    if min_probability is not None and prob < min_probability:
                        continue
                    if max_probability is not None and prob > max_probability:
                        continue

                    probability_filter.add(belief_id)

                belief_ids = probability_filter

            if source is not None:
                # Filter by source
                source_key = f"source:{source}"
                if source_key in self.belief_index:
                    belief_ids &= set(self.belief_index[source_key])

            if metadata_filter is not None:
                # Filter by metadata
                for key, value in metadata_filter.items():
                    meta_key = f"metadata:{key}:{value}"
                    if meta_key in self.belief_index:
                        belief_ids &= set(self.belief_index[meta_key])

            # Return matching beliefs
            return [self.beliefs[belief_id] for belief_id in belief_ids]

    def update_belief(self, belief_id: str,
                     new_uncertainty_value: Any,
                     evidence: Optional[Dict[str, Any]] = None,
                     update_method: str = "replace") -> bool:
        """
        Update an existing belief with new uncertainty information.

        Args:
            belief_id: ID of the belief to update
            new_uncertainty_value: New value for the uncertainty
            evidence: Optional evidence for the update
            update_method: How to combine with existing value ('replace', 'bayesian', 'weighted')

        Returns:
            True if update was successful, False otherwise
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            belief = self.beliefs.get(belief_id)
            if not belief:
                self.logger.warning(f"Belief {belief_id} not found for update")
                return False

            try:
                old_value = None

                # Get the old value based on uncertainty type
                if belief.uncertainty_type == UncertaintyType.PROBABILITY:
                    old_value = belief.probability
                elif belief.uncertainty_type == UncertaintyType.INTERVAL:
                    old_value = belief.interval
                elif belief.uncertainty_type == UncertaintyType.DISTRIBUTION:
                    old_value = belief.distribution_params
                elif belief.uncertainty_type == UncertaintyType.FUZZY:
                    old_value = belief.fuzzy_membership
                elif belief.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
                    old_value = belief.belief_mass
                elif belief.uncertainty_type == UncertaintyType.POSSIBILITY:
                    old_value = belief.possibility
                elif belief.uncertainty_type == UncertaintyType.VERBAL:
                    old_value = belief.verbal_confidence

                # Apply the update based on the method
                updated_value = self._combine_uncertainty_values(
                    old_value, new_uncertainty_value,
                    belief.uncertainty_type, update_method
                )

                # Update the belief
                belief.update(updated_value, evidence)

                # Log update
                self.logger.info(f"Updated belief: {belief_id}")

                # Update in working memory if available
                if self.working_memory:
                    self.working_memory.store_item(
                        content=belief.to_dict(),
                        source="uncertainty_manager",
                        metadata={
                            "type": "updated_belief",
                            "uncertainty_type": belief.uncertainty_type.name
                        }
                    )

                return True
            except Exception as e:
                self.logger.error(f"Error updating belief {belief_id}: {str(e)}")
                return False

    def create_bayesian_network(self, name: str,
                              description: Optional[str] = None,
                              metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new Bayesian network for causal probabilistic reasoning.

        Args:
            name: Name of the network
            description: Optional description
            metadata: Additional metadata

        Returns:
            ID of the created network
        """
        if not self.initialized:
            self.initialize()

        network_id = str(uuid.uuid4())

        if metadata is None:
            metadata = {}

        # Create the network
        network = BayesianNetwork(
            network_id=network_id,
            name=name,
            description=description,
            metadata=metadata
        )

        with self.lock:
            # Store the network
            self.networks[network_id] = network

        self.logger.info(f"Created Bayesian network: {network_id} ({name})")
        return network_id

    def add_node_to_network(self, network_id: str, node: BayesianNode) -> bool:
        """
        Add a node to a Bayesian network.

        Args:
            network_id: ID of the network
            node: Node to add

        Returns:
            True if addition was successful, False otherwise
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            network = self.networks.get(network_id)
            if not network:
                self.logger.warning(f"Network {network_id} not found")
                return False

            try:
                network.add_node(node)
                self.logger.info(f"Added node {node.node_id} to network {network_id}")
                return True
            except Exception as e:
                self.logger.error(f"Error adding node to network: {str(e)}")
                return False

    def add_edge_to_network(self, network_id: str,
                          parent_id: str, child_id: str) -> bool:
        """
        Add a directed edge between nodes in a Bayesian network.

        Args:
            network_id: ID of the network
            parent_id: ID of the parent node
            child_id: ID of the child node

        Returns:
            True if addition was successful, False otherwise
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            network = self.networks.get(network_id)
            if not network:
                self.logger.warning(f"Network {network_id} not found")
                return False

            try:
                network.add_edge(parent_id, child_id)
                self.logger.info(f"Added edge {parent_id} -> {child_id} to network {network_id}")
                return True
            except Exception as e:
                self.logger.error(f"Error adding edge to network: {str(e)}")
                return False

    def query_bayesian_network(self, network_id: str,
                             query_nodes: List[str],
                             evidence: Dict[str, str],
                             method: str = "exact") -> Dict[str, Dict[str, float]]:
        """
        Query a Bayesian network to compute probabilities given evidence.

        Args:
            network_id: ID of the network
            query_nodes: List of node IDs to query
            evidence: Dictionary mapping node IDs to their observed states
            method: Inference method ('exact', 'approximate', 'sampling')

        Returns:
            Dictionary mapping node IDs to probability distributions over states
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            network = self.networks.get(network_id)
            if not network:
                self.logger.warning(f"Network {network_id} not found")
                return {}

            try:
                if method == "exact":
                    return self._exact_inference(network, query_nodes, evidence)
                elif method == "approximate":
                    return self._approximate_inference(network, query_nodes, evidence)
                elif method == "sampling":
                    return self._sampling_inference(network, query_nodes, evidence)
                else:
                    self.logger.warning(f"Unknown inference method: {method}")
                    return {}
            except Exception as e:
                self.logger.error(f"Error querying network: {str(e)}")
                return {}

    def monte_carlo_simulation(self,
                              variables: Dict[str, Dict[str, Any]],
                              function: Callable,
                              num_samples: int = 1000) -> Dict[str, Any]:
        """
        Perform Monte Carlo simulation to propagate uncertainty through a function.

        Args:
            variables: Dictionary mapping variable names to distribution parameters
            function: Function that takes variable values and returns a result
            num_samples: Number of Monte Carlo samples

        Returns:
            Dictionary with simulation results (mean, std dev, quantiles, etc.)
        """
        if not self.initialized:
            self.initialize()

        if not HAS_NUMPY:
            self.logger.warning("NumPy is required for Monte Carlo simulation")
            return {"error": "NumPy is required for Monte Carlo simulation"}

        try:
            # Generate samples for each variable
            samples = {}
            for var_name, var_params in variables.items():
                dist_type = var_params.get("type", "normal")

                if dist_type == "normal":
                    mean = var_params.get("mean", 0.0)
                    std = var_params.get("std", 1.0)
                    samples[var_name] = np.random.normal(mean, std, num_samples)
                elif dist_type == "uniform":
                    low = var_params.get("low", 0.0)
                    high = var_params.get("high", 1.0)
                    samples[var_name] = np.random.uniform(low, high, num_samples)
                elif dist_type == "beta":
                    alpha = var_params.get("alpha", 1.0)
                    beta = var_params.get("beta", 1.0)
                    samples[var_name] = np.random.beta(alpha, beta, num_samples)
                else:
                    self.logger.warning(f"Unknown distribution type: {dist_type}")
                    return {"error": f"Unknown distribution type: {dist_type}"}

            # Run the function on each sample
            results = []
            for i in range(num_samples):
                sample_values = {var: samples[var][i] for var in variables}
                result = function(**sample_values)
                results.append(result)

            # Compute statistics
            results_array = np.array(results)

            return {
                "mean": float(np.mean(results_array)),
                "std": float(np.std(results_array)),
                "min": float(np.min(results_array)),
                "max": float(np.max(results_array)),
                "quantiles": {
                    "0.05": float(np.quantile(results_array, 0.05)),
                    "0.25": float(np.quantile(results_array, 0.25)),
                    "0.5": float(np.quantile(results_array, 0.5)),
                    "0.75": float(np.quantile(results_array, 0.75)),
                    "0.95": float(np.quantile(results_array, 0.95))
                },
                "num_samples": num_samples
            }
        except Exception as e:
            self.logger.error(f"Error in Monte Carlo simulation: {str(e)}")
            return {"error": str(e)}

    def reconcile_conflicting_beliefs(self,
                                     belief_ids: List[str],
                                     reconciliation_strategy: str = "weighted_average") -> Optional[str]:
        """
        Reconcile multiple potentially conflicting beliefs into a single belief.

        Args:
            belief_ids: List of belief IDs to reconcile
            reconciliation_strategy: Strategy for reconciliation

        Returns:
            ID of the reconciled belief, or None if reconciliation failed
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            # Get all beliefs
            beliefs = []
            for belief_id in belief_ids:
                belief = self.beliefs.get(belief_id)
                if belief:
                    beliefs.append(belief)

            if not beliefs:
                self.logger.warning("No beliefs found for reconciliation")
                return None

            # Check if all beliefs have the same content
            content = beliefs[0].content
            if not all(belief.content == content for belief in beliefs):
                self.logger.warning("Cannot reconcile beliefs with different content")
                return None

            try:
                # Apply reconciliation strategy
                if reconciliation_strategy == "weighted_average":
                    # Convert all to probabilities
                    probs = [belief.to_probability() for belief in beliefs]

                    # Use update count as weight (more updates = more evidence)
                    weights = [belief.update_count + 1 for belief in beliefs]  # +1 to avoid zero weight

                    # Calculate weighted average
                    weighted_prob = sum(p * w for p, w in zip(probs, weights)) / sum(weights)

                    # Create new belief with reconciled probability
                    reconciled_id = self.create_belief(
                        content=content,
                        uncertainty_type=UncertaintyType.PROBABILITY,
                        uncertainty_value=weighted_prob,
                        source="reconciliation",
                        metadata={
                            "reconciliation_strategy": reconciliation_strategy,
                            "source_beliefs": belief_ids
                        },
                        evidence={
                            "source": "reconciliation",
                            "description": f"Reconciled from {len(beliefs)} beliefs using {reconciliation_strategy}"
                        }
                    )

                    return reconciled_id

                elif reconciliation_strategy == "dempster_shafer":
                    # Implement Dempster-Shafer combination rule
                    # This is a simplified version
                    belief_masses = {}

                    for belief in beliefs:
                        if belief.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
                            # Use existing belief masses
                            for state, mass in belief.belief_mass.items():
                                if state not in belief_masses:
                                    belief_masses[state] = mass
                                else:
                                    # Basic combination (simplified)
                                    belief_masses[state] = belief_masses[state] + mass - belief_masses[state] * mass
                        else:
                            # Convert to simple belief mass
                            prob = belief.to_probability()
                            if "true" not in belief_masses:
                                belief_masses["true"] = prob
                            else:
                                belief_masses["true"] = belief_masses["true"] + prob - belief_masses["true"] * prob

                            if "false" not in belief_masses:
                                belief_masses["false"] = 1 - prob
                            else:
                                belief_masses["false"] = belief_masses["false"] + (1 - prob) - belief_masses["false"] * (1 - prob)

                    # Create new belief with combined belief masses
                    reconciled_id = self.create_belief(
                        content=content,
                        uncertainty_type=UncertaintyType.DEMPSTER_SHAFER,
                        uncertainty_value=belief_masses,
                        source="reconciliation",
                        metadata={
                            "reconciliation_strategy": reconciliation_strategy,
                            "source_beliefs": belief_ids
                        },
                        evidence={
                            "source": "reconciliation",
                            "description": f"Reconciled from {len(beliefs)} beliefs using {reconciliation_strategy}"
                        }
                    )

                    return reconciled_id

                elif reconciliation_strategy == "highest_confidence":
                    # Select the belief with highest confidence
                    highest_conf_belief = max(beliefs, key=lambda b: b.to_probability())

                    # Create new belief based on the highest confidence one
                    reconciled_id = self.create_belief(
                        content=content,
                        uncertainty_type=highest_conf_belief.uncertainty_type,
                        uncertainty_value=self._get_uncertainty_value(highest_conf_belief),
                        source="reconciliation",
                        metadata={
                            "reconciliation_strategy": reconciliation_strategy,
                            "source_beliefs": belief_ids
                        },
                        evidence={
                            "source": "reconciliation",
                            "description": f"Selected from {len(beliefs)} beliefs using {reconciliation_strategy}"
                        }
                    )

                    return reconciled_id

                else:
                    self.logger.warning(f"Unknown reconciliation strategy: {reconciliation_strategy}")
                    return None

            except Exception as e:
                self.logger.error(f"Error reconciling beliefs: {str(e)}")
                return None

    def calculate_certainty_factor(self, evidence_factors: List[float],
                                 combination_rule: str = "cf_model") -> float:
        """
        Calculate a certainty factor from multiple pieces of evidence.

        Args:
            evidence_factors: List of certainty factors from evidence
            combination_rule: Rule for combining certainty factors

        Returns:
            Combined certainty factor
        """
        if not evidence_factors:
            return 0.0

        try:
            if combination_rule == "cf_model":
                # MYCIN certainty factor model
                cf = evidence_factors[0]

                for i in range(1, len(evidence_factors)):
                    cf2 = evidence_factors[i]

                    if cf >= 0 and cf2 >= 0:
                        # Both positive
                        cf = cf + cf2 - cf * cf2
                    elif cf < 0 and cf2 < 0:
                        # Both negative
                        cf = cf + cf2 + cf * cf2
                    else:
                        # One positive, one negative
                        cf = (cf + cf2) / (1 - min(abs(cf), abs(cf2)))

                return cf

            elif combination_rule == "min_max":
                # Conservative combination using min/max
                pos_factors = [f for f in evidence_factors if f > 0]
                neg_factors = [f for f in evidence_factors if f < 0]

                pos_cf = max(pos_factors) if pos_factors else 0
                neg_cf = min(neg_factors) if neg_factors else 0

                return pos_cf + neg_cf  # Sum of positive and negative CFs

            elif combination_rule == "average":
                # Simple average
                return sum(evidence_factors) / len(evidence_factors)

            else:
                self.logger.warning(f"Unknown combination rule: {combination_rule}")
                return sum(evidence_factors) / len(evidence_factors)

        except Exception as e:
            self.logger.error(f"Error calculating certainty factor: {str(e)}")
            return 0.0

    def fuzzy_inference(self, fuzzy_rules: List[Dict[str, Any]],
                       inputs: Dict[str, float]) -> Dict[str, float]:
        """
        Perform fuzzy logic inference based on fuzzy rules.

        Args:
            fuzzy_rules: List of fuzzy rules
            inputs: Input values for fuzzy variables

        Returns:
            Dictionary of output fuzzy variable values
        """
        if not self.initialized:
            self.initialize()

        try:
            # Simple Mamdani-style fuzzy inference
            rule_outputs = []

            for rule in fuzzy_rules:
                # Evaluate antecedent (if-part)
                if 'antecedent' not in rule:
                    continue

                antecedent_value = self._evaluate_fuzzy_expression(rule['antecedent'], inputs)

                # Apply consequent (then-part) if antecedent has nonzero value
                if antecedent_value > 0 and 'consequent' in rule:
                    consequent = rule['consequent']
                    rule_outputs.append({
                        'variable': consequent['variable'],
                        'value': consequent['value'],
                        'strength': antecedent_value
                    })

            # Combine rule outputs for each output variable
            output_values = {}

            for output in rule_outputs:
                var_name = output['variable']

                if var_name not in output_values:
                    output_values[var_name] = 0.0

                # Take maximum for each output variable (standard Mamdani approach)
                output_values[var_name] = max(output_values[var_name], output['strength'])

            return output_values

        except Exception as e:
            self.logger.error(f"Error in fuzzy inference: {str(e)}")
            return {}

    def get_metrics(self) -> Dict[str, Any]:
        """Get metrics about uncertainty management performance"""
        with self.lock:
            metrics_snapshot = {}

            # Overall belief metrics
            metrics_snapshot["total_beliefs"] = len(self.beliefs)
            metrics_snapshot["total_networks"] = len(self.networks)

            # Uncertainty type distribution
            type_counts = defaultdict(int)
            for belief in self.beliefs.values():
                type_counts[belief.uncertainty_type.name] += 1
            metrics_snapshot["uncertainty_type_distribution"] = dict(type_counts)

            # Performance metrics
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }

            return metrics_snapshot

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.reasoning_core:
                dependencies.append("reasoning_core")

            self.nexus_core.register_module(
                name="uncertainty",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _update_belief_indices(self, belief: UncertainBelief) -> None:
        """
        Update indices for faster belief lookup.

        Args:
            belief: The belief to index
        """
        # Index by uncertainty type
        self.belief_index[f"type:{belief.uncertainty_type.name}"].append(belief.belief_id)

        # Index by source if available
        if belief.source:
            self.belief_index[f"source:{belief.source}"].append(belief.belief_id)

        # Index by probability range (converted if needed)
        prob = belief.to_probability()
        prob_range = int(prob * 10) / 10  # Round to nearest 0.1
        self.belief_index[f"probability:{prob_range}"].append(belief.belief_id)

        # Add metadata indices if available
        if belief.metadata:
            for key, value in belief.metadata.items():
                if isinstance(value, (str, int, float, bool)):
                    self.belief_index[f"metadata:{key}:{value}"].append(belief.belief_id)

    def _get_uncertainty_value(self, belief: UncertainBelief) -> Any:
        """Get the appropriate uncertainty value from a belief based on its type"""
        if belief.uncertainty_type == UncertaintyType.PROBABILITY:
            return belief.probability
        elif belief.uncertainty_type == UncertaintyType.INTERVAL:
            return belief.interval
        elif belief.uncertainty_type == UncertaintyType.DISTRIBUTION:
            return belief.distribution_params
        elif belief.uncertainty_type == UncertaintyType.FUZZY:
            return belief.fuzzy_membership
        elif belief.uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
            return belief.belief_mass
        elif belief.uncertainty_type == UncertaintyType.POSSIBILITY:
            return belief.possibility
        elif belief.uncertainty_type == UncertaintyType.VERBAL:
            return belief.verbal_confidence
        else:
            return None

    def _combine_uncertainty_values(self, old_value: Any, new_value: Any,
                                  uncertainty_type: UncertaintyType,
                                  method: str) -> Any:
        """
        Combine old and new uncertainty values using specified method.

        Args:
            old_value: Existing uncertainty value
            new_value: New uncertainty value
            uncertainty_type: Type of uncertainty
            method: Combination method ('replace', 'bayesian', 'weighted')

        Returns:
            Combined uncertainty value
        """
        if method == "replace":
            # Simple replacement
            return new_value

        elif method == "bayesian":
            # Bayesian update (only for probability)
            if uncertainty_type == UncertaintyType.PROBABILITY:
                # Simplified Bayesian update (for binary hypothesis)
                # P(h|e) = P(e|h) * P(h) / [P(e|h) * P(h) + P(e|~h) * P(~h)]
                # Assuming P(e|h) = new_value and P(e|~h) = 1 - new_value
                prior = old_value
                likelihood = new_value

                numerator = likelihood * prior
                denominator = likelihood * prior + (1 - likelihood) * (1 - prior)

                if denominator == 0:
                    return prior  # Avoid division by zero

                return numerator / denominator

            elif uncertainty_type == UncertaintyType.INTERVAL:
                # For interval, use Bayesian update on endpoints
                prior_min, prior_max = old_value
                likelihood_min, likelihood_max = new_value

                # Update min
                num_min = likelihood_min * prior_min
                denom_min = likelihood_min * prior_min + (1 - likelihood_min) * (1 - prior_min)
                if denom_min == 0:
                    posterior_min = prior_min
                else:
                    posterior_min = num_min / denom_min

                # Update max
                num_max = likelihood_max * prior_max
                denom_max = likelihood_max * prior_max + (1 - likelihood_max) * (1 - prior_max)
                if denom_max == 0:
                    posterior_max = prior_max
                else:
                    posterior_max = num_max / denom_max

                return (posterior_min, posterior_max)

            else:
                # For other types, fall back to replacement
                self.logger.warning(f"Bayesian update not supported for {uncertainty_type.name}")
                return new_value

        elif method == "weighted":
            # Weighted combination
            if uncertainty_type == UncertaintyType.PROBABILITY:
                # Weighted average of probabilities
                return (old_value + new_value) / 2

            elif uncertainty_type == UncertaintyType.INTERVAL:
                # Weighted average of interval endpoints
                old_min, old_max = old_value
                new_min, new_max = new_value

                return ((old_min + new_min) / 2, (old_max + new_max) / 2)

            elif uncertainty_type == UncertaintyType.DISTRIBUTION:
                # For distribution, try to average parameters
                result = {}

                # Merge keys
                all_keys = set(old_value.keys()) | set(new_value.keys())

                for key in all_keys:
                    if key in old_value and key in new_value:
                        # Average numeric values
                        if isinstance(old_value[key], (int, float)) and isinstance(new_value[key], (int, float)):
                            result[key] = (old_value[key] + new_value[key]) / 2
                        else:
                            # For non-numeric, keep new value
                            result[key] = new_value[key]
                    elif key in new_value:
                        result[key] = new_value[key]
                    else:
                        result[key] = old_value[key]

                return result

            elif uncertainty_type == UncertaintyType.FUZZY:
                # Average of fuzzy memberships
                return (old_value + new_value) / 2

            elif uncertainty_type == UncertaintyType.DEMPSTER_SHAFER:
                # Simplified combination of belief masses
                result = {}

                # Merge keys
                all_keys = set(old_value.keys()) | set(new_value.keys())

                for key in all_keys:
                    if key in old_value and key in new_value:
                        # Simple average for masses
                        result[key] = (old_value[key] + new_value[key]) / 2
                    elif key in new_value:
                        result[key] = new_value[key] / 2  # Reduce weight for new-only keys
                    else:
                        result[key] = old_value[key] / 2  # Reduce weight for old-only keys

                # Normalize masses
                total_mass = sum(result.values())
                if total_mass > 0:
                    for key in result:
                        result[key] /= total_mass

                return result

            elif uncertainty_type == UncertaintyType.POSSIBILITY:
                # Average of possibility values
                return (old_value + new_value) / 2

            elif uncertainty_type == UncertaintyType.VERBAL:
                # For verbal, prefer the new value
                return new_value

            else:
                return new_value

        else:
            # Unknown method
            self.logger.warning(f"Unknown update method: {method}")
            return new_value

    def _exact_inference(self, network: BayesianNetwork,
                       query_nodes: List[str],
                       evidence: Dict[str, str]) -> Dict[str, Dict[str, float]]:
        """
        Perform exact inference on a Bayesian network.

        Args:
            network: The Bayesian network
            query_nodes: Nodes to query
            evidence: Observed evidence

        Returns:
            Dictionary mapping node IDs to probability distributions
        """
        # This is a simplified implementation for small networks
        # For real applications, use libraries like pgmpy or pyAgrum

        # Get all nodes
        all_nodes = list(network.nodes.keys())

        # Find nodes that are not in evidence or query
        hidden_nodes = [n for n in all_nodes if n not in evidence and n not in query_nodes]

        # For each query node, compute marginal distribution
        result = {}

        for query_node in query_nodes:
            node = network.get_node(query_node)
            if not node:
                continue

            # Get possible states for this node
            states = node.states

            # Initialize distribution
            distribution = {state: 0.0 for state in states}

            # For a small network, we can enumerate all possible hidden node states
            # This is inefficient for larger networks
            hidden_state_combinations = self._generate_state_combinations(network, hidden_nodes)

            for hidden_states in hidden_state_combinations:
                # Combine with evidence
                full_assignment = {**evidence, **hidden_states}

                # Compute probability of this assignment
                assignment_prob = 1.0

                for node_id, node_obj in network.nodes.items():
                    state = full_assignment.get(node_id)
                    if not state:
                        continue

                    # Get parent states
                    parent_states = {p: full_assignment.get(p) for p in node_obj.parents}

                    # Skip if parent states are incomplete
                    if any(v is None for v in parent_states.values()):
                        continue

                    # Get conditional probability
                    state_prob = node_obj.get_probability(state, parent_states)

                    # Update assignment probability
                    assignment_prob *= state_prob

                # For each possible state of the query node
                for state in states:
                    # Create a new assignment with this state
                    state_assignment = {**full_assignment, query_node: state}

                    # Compute probability with this state
                    state_prob = 1.0

                    for node_id, node_obj in network.nodes.items():
                        node_state = state_assignment.get(node_id)
                        if not node_state:
                            continue

                        # Get parent states
                        parent_states = {p: state_assignment.get(p) for p in node_obj.parents}

                        # Skip if parent states are incomplete
                        if any(v is None for v in parent_states.values()):
                            continue

                        # Get conditional probability
                        node_prob = node_obj.get_probability(node_state, parent_states)

                        # Update state probability
                        state_prob *= node_prob

                    # Add to state distribution
                    distribution[state] += state_prob

            # Normalize distribution
            total_prob = sum(distribution.values())
            if total_prob > 0:
                for state in distribution:
                    distribution[state] /= total_prob

            # Store result
            result[query_node] = distribution

        return result

    def _approximate_inference(self, network: BayesianNetwork,
                             query_nodes: List[str],
                             evidence: Dict[str, str]) -> Dict[str, Dict[str, float]]:
        """
        Perform approximate inference on a Bayesian network.

        Args:
            network: The Bayesian network
            query_nodes: Nodes to query
            evidence: Observed evidence

        Returns:
            Dictionary mapping node IDs to probability distributions
        """
        # For simple implementation, use sampling inference
        return self._sampling_inference(network, query_nodes, evidence, num_samples=10000)

    def _sampling_inference(self, network: BayesianNetwork,
                          query_nodes: List[str],
                          evidence: Dict[str, str],
                          num_samples: int = 1000) -> Dict[str, Dict[str, float]]:
        """
        Perform inference via sampling.

        Args:
            network: The Bayesian network
            query_nodes: Nodes to query
            evidence: Observed evidence
            num_samples: Number of samples to draw

        Returns:
            Dictionary mapping node IDs to probability distributions
        """
        # Initialize result dictionary
        result = {}
        for query_node in query_nodes:
            node = network.get_node(query_node)
            if node:
                result[query_node] = {state: 0.0 for state in node.states}

        # Rejection sampling
        valid_samples = 0

        for _ in range(num_samples):
            # Generate a random sample
            sample = self._generate_random_sample(network)

            # Check if sample is consistent with evidence
            if all(sample.get(node) == state for node, state in evidence.items()):
                valid_samples += 1

                # Update counts for query nodes
                for query_node in query_nodes:
                    state = sample.get(query_node)
                    if state is not None and query_node in result:
                        result[query_node][state] += 1

        # Normalize counts to probabilities
        if valid_samples > 0:
            for query_node in query_nodes:
                if query_node in result:
                    for state in result[query_node]:
                        result[query_node][state] /= valid_samples

        return result

    def _generate_random_sample(self, network: BayesianNetwork) -> Dict[str, str]:
        """
        Generate a random sample from a Bayesian network.

        Args:
            network: The Bayesian network

        Returns:
            Dictionary mapping node IDs to sampled states
        """
        # Topologically sort nodes
        sorted_nodes = self._topological_sort(network)

        # Generate sample
        sample = {}

        for node_id in sorted_nodes:
            node = network.get_node(node_id)
            if not node:
                continue

            # Get parent states
            parent_states = {p: sample.get(p) for p in node.parents}

            # Skip if parent states are incomplete
            if any(v is None for v in parent_states.values()) and node.parents:
                continue

            # Get probability distribution for each state
            state_probs = {}
            total_prob = 0.0

            for state in node.states:
                prob = node.get_probability(state, parent_states)
                state_probs[state] = prob
                total_prob += prob

            # Normalize if needed
            if total_prob > 0:
                for state in state_probs:
                    state_probs[state] /= total_prob

            # Sample a state
            r = random.random()
            cumulative_prob = 0.0

            for state, prob in state_probs.items():
                cumulative_prob += prob
                if r <= cumulative_prob:
                    sample[node_id] = state
                    break

            # Fallback if no state was selected
            if node_id not in sample and node.states:
                sample[node_id] = random.choice(node.states)

        return sample

    def _topological_sort(self, network: BayesianNetwork) -> List[str]:
        """
        Topologically sort nodes in a Bayesian network.

        Args:
            network: The Bayesian network

        Returns:
            List of node IDs in topological order
        """
        # Get all nodes
        nodes = list(network.nodes.keys())

        # Initialize
        visited = set()
        temp_mark = set()
        result = []

        def visit(node_id):
            if node_id in temp_mark:
                # Cycle detected, just continue
                return
            if node_id in visited:
                return

            temp_mark.add(node_id)

            # Visit all parents
            node = network.get_node(node_id)
            if node:
                for parent in node.parents:
                    visit(parent)

            temp_mark.remove(node_id)
            visited.add(node_id)
            result.append(node_id)

        # Visit all nodes
        for node_id in nodes:
            if node_id not in visited:
                visit(node_id)

        # Reverse to get correct order
        return list(reversed(result))

    def _generate_state_combinations(self, network: BayesianNetwork,
                                   nodes: List[str]) -> List[Dict[str, str]]:
        """
        Generate all possible state combinations for a list of nodes.

        Args:
            network: The Bayesian network
            nodes: List of node IDs

        Returns:
            List of dictionaries mapping node IDs to states
        """
        if not nodes:
            return [{}]

        # Start with first node
        node_id = nodes[0]
        node = network.get_node(node_id)

        if not node:
            return self._generate_state_combinations(network, nodes[1:])

        # Get states for this node
        states = node.states

        # Generate combinations for remaining nodes
        remaining_combinations = self._generate_state_combinations(network, nodes[1:])

        # Combine with states for this node
        result = []
        for state in states:
            for combination in remaining_combinations:
                result.append({**{node_id: state}, **combination})

        return result

    def _evaluate_fuzzy_expression(self, expression: Dict[str, Any],
                                 inputs: Dict[str, float]) -> float:
        """
        Evaluate a fuzzy logic expression.

        Args:
            expression: The fuzzy expression
            inputs: Input values

        Returns:
            Fuzzy value
        """
        if 'type' not in expression:
            return 0.0

        expr_type = expression['type']

        if expr_type == 'input':
            # Direct input reference
            var_name = expression.get('variable')
            if var_name in inputs:
                return inputs[var_name]
            return 0.0

        elif expr_type == 'and':
            # Fuzzy AND (minimum)
            operands = expression.get('operands', [])
            if not operands:
                return 0.0

            values = [self._evaluate_fuzzy_expression(op, inputs) for op in operands]
            return min(values)

        elif expr_type == 'or':
            # Fuzzy OR (maximum)
            operands = expression.get('operands', [])
            if not operands:
                return 0.0

            values = [self._evaluate_fuzzy_expression(op, inputs) for op in operands]
            return max(values)

        elif expr_type == 'not':
            # Fuzzy NOT (complement)
            operand = expression.get('operand')
            if not operand:
                return 0.0

            value = self._evaluate_fuzzy_expression(operand, inputs)
            return 1.0 - value

        elif expr_type == 'membership':
            # Fuzzy membership function
            var_name = expression.get('variable')
            if var_name not in inputs:
                return 0.0

            value = inputs[var_name]
            membership_type = expression.get('membership_type', 'triangle')

            if membership_type == 'triangle':
                a = expression.get('a', 0.0)
                b = expression.get('b', 0.5)
                c = expression.get('c', 1.0)

                if value <= a or value >= c:
                    return 0.0
                elif a < value <= b:
                    return (value - a) / (b - a)
                else:  # b < value < c
                    return (c - value) / (c - b)

            elif membership_type == 'trapezoid':
                a = expression.get('a', 0.0)
                b = expression.get('b', 0.25)
                c = expression.get('c', 0.75)
                d = expression.get('d', 1.0)

                if value <= a or value >= d:
                    return 0.0
                elif a < value <= b:
                    return (value - a) / (b - a)
                elif b < value <= c:
                    return 1.0
                else:  # c < value < d
                    return (d - value) / (d - c)

            elif membership_type == 'gaussian':
                mean = expression.get('mean', 0.5)
                std = expression.get('std', 0.2)

                return math.exp(-((value - mean) ** 2) / (2 * std ** 2))

            else:
                return 0.0

        else:
            return 0.0

ModuleNotFoundError: No module named 'grandnexus'

# Learning Core

In [ ]:
"""
learning_core.py - Core Learning Module for GrandNexus

This module serves as the central orchestrator for all learning-related
capabilities in the GrandNexus cognitive architecture. It manages various
learning paradigms through a flexible, emergence-oriented approach rather
than strict prescriptive rules.

The design philosophy emphasizes:
- Adaptivity: Learning strategies adapt based on context and performance
- Emergence: Complex learning behaviors emerge from simple interaction rules
- Integration: Seamless connection with other cognitive modules
- Flexibility: Support for multiple learning paradigms without rigidity
"""

import logging
import threading
import time
import uuid
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from enum import Enum, auto
from dataclasses import dataclass, field
import numpy as np
from collections import defaultdict, deque

# Import from GrandNexus core modules (assuming these exist)
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory

# Learning paradigm types
class LearningParadigm(Enum):
    SUPERVISED = auto()
    UNSUPERVISED = auto()
    REINFORCEMENT = auto()
    META = auto()
    CONTINUOUS = auto()
    CURRICULUM = auto()
    INTRINSIC = auto()
    TRANSFER = auto()
    EMERGENT = auto()  # Special type for emergent learning patterns

# Learning event types
class LearningEventType(Enum):
    OBSERVATION = auto()    # New observation received
    FEEDBACK = auto()       # Feedback on a prediction/action
    REFLECTION = auto()     # Internal learning assessment
    ADAPTATION = auto()     # Learning parameter adjustment
    DISCOVERY = auto()      # New pattern or knowledge discovered
    INTEGRATION = auto()    # Knowledge integration across domains
    FORGETTING = auto()     # Strategic forgetting to avoid overfitting
    CURIOSITY = auto()      # Intrinsic interest in unexplored areas

@dataclass
class LearningEvent:
    """Represents a discrete learning event in the system"""
    event_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    event_type: LearningEventType = LearningEventType.OBSERVATION
    timestamp: float = field(default_factory=time.time)
    paradigm: LearningParadigm = LearningParadigm.SUPERVISED
    content: Any = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    source_module: Optional[str] = None
    confidence: float = 1.0
    importance: float = 0.5

@dataclass
class LearningTask:
    """Represents a learning task to be processed"""
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    description: str = ""
    paradigm: LearningParadigm = LearningParadigm.SUPERVISED
    data: Any = None
    parameters: Dict[str, Any] = field(default_factory=dict)
    callback: Optional[Callable] = None
    priority: float = 0.5
    deadline: Optional[float] = None
    status: str = "pending"
    result: Any = None

@dataclass
class LearningStrategy:
    """Defines a learning strategy with its parameters and activation conditions"""
    strategy_id: str
    name: str
    paradigm: LearningParadigm
    parameters: Dict[str, Any] = field(default_factory=dict)
    activation_threshold: float = 0.5
    current_activation: float = 0.0
    last_used: float = 0.0
    success_rate: float = 0.5
    affinity_map: Dict[str, float] = field(default_factory=dict)  # Affinity to contexts/domains

    def is_applicable(self, task: LearningTask, context: Dict[str, Any]) -> float:
        """
        Determine applicability score of this strategy to the given task and context.
        Higher scores indicate better fit.

        This is a simplistic implementation - in a real system, this would be more sophisticated,
        potentially using meta-learning or contextual bandits to determine strategy selection.
        """
        # Base score is the activation level
        score = self.current_activation

        # Paradigm matching bonus
        if task.paradigm == self.paradigm:
            score += 0.3

        # Context affinity - match task elements with the strategy's affinities
        context_keys = set(context.keys())
        for key, affinity in self.affinity_map.items():
            if key in context_keys:
                score += affinity * context[key]

        # Success rate bonus
        score += self.success_rate * 0.2

        # Recency penalty (gradual decay of relevance if not used recently)
        time_since_used = time.time() - self.last_used if self.last_used > 0 else 0
        recency_penalty = min(0.5, time_since_used / 3600.0)  # Max 0.5 penalty after 1 hour
        score -= recency_penalty

        return max(0.0, min(1.0, score))  # Bound between 0 and 1

class AdaptiveLearningParameter:
    """
    Represents a learning parameter that can self-adjust based on feedback
    and contextual cues. This enables emergent adaptation without requiring
    manual parameter tuning.
    """
    def __init__(self, name: str, initial_value: float,
                 min_value: float, max_value: float,
                 adaptation_rate: float = 0.1):
        self.name = name
        self.value = initial_value
        self.min_value = min_value
        self.max_value = max_value
        self.adaptation_rate = adaptation_rate
        self.history = deque(maxlen=100)  # Track recent values for analysis
        self.context_influence = {}  # How different contexts influence this parameter

    def adapt(self, feedback: float, context: Dict[str, float]) -> None:
        """
        Adapt parameter based on feedback and context.
        feedback: -1.0 to 1.0 (negative means decrease, positive means increase)
        context: dict mapping context factors to their strength (0.0 to 1.0)
        """
        # Base adaptation from direct feedback
        adaptation = feedback * self.adaptation_rate

        # Context-specific adaptations
        for ctx_name, ctx_strength in context.items():
            if ctx_name in self.context_influence:
                ctx_factor = self.context_influence[ctx_name]
                adaptation += ctx_factor * ctx_strength * self.adaptation_rate * 0.5

        # Apply adaptation with bounds
        new_value = self.value + adaptation
        self.value = max(self.min_value, min(self.max_value, new_value))
        self.history.append((time.time(), self.value, feedback, context))

    def learn_context_influence(self) -> None:
        """
        Analyze history to learn how different contexts influence this parameter.
        This is a simple implementation - more sophisticated approaches could be used.
        """
        if len(self.history) < 30:  # Need sufficient data
            return

        # Extract all unique context factors
        context_factors = set()
        for _, _, _, context in self.history:
            context_factors.update(context.keys())

        # Analyze correlation between each context factor and parameter changes
        for factor in context_factors:
            correlations = []
            for i in range(1, len(self.history)):
                prev_time, prev_value, _, prev_ctx = self.history[i-1]
                curr_time, curr_value, _, curr_ctx = self.history[i]

                if factor in curr_ctx:
                    factor_presence = curr_ctx[factor]
                    value_change = curr_value - prev_value
                    correlations.append((factor_presence, value_change))

            if correlations:
                # Calculate simple correlation coefficient
                factor_values = [c[0] for c in correlations]
                changes = [c[1] for c in correlations]

                if len(set(factor_values)) > 1 and len(set(changes)) > 1:
                    covariance = np.cov(factor_values, changes)[0, 1]
                    std_factor = np.std(factor_values)
                    std_changes = np.std(changes)

                    if std_factor > 0 and std_changes > 0:
                        correlation = covariance / (std_factor * std_changes)
                        self.context_influence[factor] = correlation

class LearningCore:
    """
    Central orchestrator for all learning functions in GrandNexus.

    This module coordinates between different learning paradigms and strategies,
    manages learning tasks, and facilitates the emergence of novel learning
    behaviors through interaction patterns.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None):
        """
        Initialize the learning core with references to other system modules.

        Args:
            nexus_core: Reference to the NexusCore for system-wide coordination
            working_memory: Working memory module for temporary storage
            semantic_memory: Semantic memory for structured knowledge
            episodic_memory: Episodic memory for experience-based learning
        """
        self.logger = logging.getLogger("GrandNexus.Learning.Core")
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Learning strategies registry
        self.strategies: Dict[str, LearningStrategy] = {}

        # Event tracking
        self.event_history = deque(maxlen=1000)
        self.event_patterns = {}  # Discovered patterns in learning events

        # Task management
        self.tasks = deque()
        self.active_tasks = set()
        self.completed_tasks = deque(maxlen=100)

        # Adaptive parameters
        self.parameters = {}

        # Module connections and signal pathways
        self.signal_pathways = defaultdict(float)  # (source, target) -> strength

        # Emergence tracking
        self.emergent_patterns = []
        self.activation_landscape = defaultdict(float)  # Concept activation levels

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"LearningCore initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Set up initial learning strategies and parameters"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register basic learning strategies
                self._register_default_strategies()

                # Set up adaptive parameters
                self._initialize_adaptive_parameters()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("LearningCore initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"LearningCore initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the learning core operations"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start background processes
                self._start_background_processes()

                self.running = True
                self.logger.info("LearningCore started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start LearningCore: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the learning core operations"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Stop background processes
                self._stop_background_processes()

                self.running = False
                self.logger.info("LearningCore stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping LearningCore: {str(e)}")
                return False

    def register_strategy(self, strategy: LearningStrategy) -> bool:
        """
        Register a new learning strategy.

        Args:
            strategy: The LearningStrategy to register

        Returns:
            bool: True if registration succeeded, False otherwise
        """
        with self.lock:
            if strategy.strategy_id in self.strategies:
                self.logger.warning(f"Strategy {strategy.strategy_id} already exists, updating")

            self.strategies[strategy.strategy_id] = strategy
            self.logger.info(f"Registered learning strategy: {strategy.name} ({strategy.strategy_id})")
            return True

    def unregister_strategy(self, strategy_id: str) -> bool:
        """Remove a learning strategy from the registry"""
        with self.lock:
            if strategy_id not in self.strategies:
                self.logger.warning(f"Strategy {strategy_id} not found")
                return False

            del self.strategies[strategy_id]
            self.logger.info(f"Unregistered learning strategy: {strategy_id}")
            return True

    def submit_learning_task(self, task: LearningTask) -> str:
        """
        Submit a learning task for processing.

        Args:
            task: The LearningTask to process

        Returns:
            str: The task ID for tracking
        """
        with self.lock:
            self.tasks.append(task)
            self.logger.info(f"Learning task submitted: {task.task_id} - {task.description}")
            return task.task_id

    def process_learning_event(self, event: LearningEvent) -> None:
        """
        Process a learning event, updating internal state and triggering
        appropriate responses based on the event type.

        Args:
            event: The LearningEvent to process
        """
        with self.lock:
            # Record the event
            self.event_history.append(event)

            # Update activation landscape based on event
            self._update_activation_landscape(event)

            # Process different event types
            if event.event_type == LearningEventType.OBSERVATION:
                self._process_observation(event)
            elif event.event_type == LearningEventType.FEEDBACK:
                self._process_feedback(event)
            elif event.event_type == LearningEventType.REFLECTION:
                self._process_reflection(event)
            elif event.event_type == LearningEventType.DISCOVERY:
                self._process_discovery(event)
            elif event.event_type == LearningEventType.CURIOSITY:
                self._process_curiosity(event)

            # Check for emergent patterns in recent events
            self._detect_event_patterns()

    def get_task_status(self, task_id: str) -> Optional[LearningTask]:
        """Get the current status of a learning task"""
        with self.lock:
            # Check active tasks
            for task in self.active_tasks:
                if task.task_id == task_id:
                    return task

            # Check pending tasks
            for task in self.tasks:
                if task.task_id == task_id:
                    return task

            # Check completed tasks
            for task in self.completed_tasks:
                if task.task_id == task_id:
                    return task

            return None

    def get_learning_metrics(self) -> Dict[str, Any]:
        """Get current learning performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def adapt_to_context(self, context: Dict[str, Any]) -> None:
        """
        Adapt learning strategies and parameters based on the current context.

        Args:
            context: Dictionary of contextual factors
        """
        with self.lock:
            # Adjust strategy activations based on context
            self._adapt_strategies_to_context(context)

            # Adjust parameters based on context
            self._adapt_parameters_to_context(context)

            # Log adaptation
            self.logger.info(f"Adapted learning system to context: {context}")

    def get_strategy_activations(self) -> Dict[str, float]:
        """Get current activation levels of all learning strategies"""
        with self.lock:
            return {s_id: strategy.current_activation
                    for s_id, strategy in self.strategies.items()}

    def get_activation_landscape(self) -> Dict[str, float]:
        """Get the current concept activation landscape"""
        with self.lock:
            return dict(self.activation_landscape)

    def get_emergent_patterns(self) -> List[Dict[str, Any]]:
        """Get discovered emergent patterns in learning behavior"""
        with self.lock:
            return self.emergent_patterns.copy()

    def reinforce_pathway(self, source: str, target: str,
                           strength: float = 0.1) -> None:
        """
        Strengthen a signal pathway between source and target concepts.
        This is a core mechanism for emergent learning - strengthening
        pathways that are frequently activated together.

        Args:
            source: Source concept/node identifier
            target: Target concept/node identifier
            strength: Amount to strengthen the pathway (0.0 to 1.0)
        """
        with self.lock:
            key = (source, target)
            current = self.signal_pathways[key]
            # Sigmoid-like update to keep within bounds
            self.signal_pathways[key] = current + strength * (1.0 - current)

            # If connection becomes strong, log it as potentially emergent
            if current < 0.7 and self.signal_pathways[key] >= 0.7:
                self.logger.info(f"Emergent strong connection: {source} -> {target}")

                # Check if this connects previously separate subsystems
                self._check_for_systemic_emergence(source, target)

    def propagate_activation(self, initial_activations: Dict[str, float],
                             steps: int = 3) -> Dict[str, float]:
        """
        Propagate activation through the network of concepts and signal pathways.
        This is a key mechanism for associative learning and emergence.

        Args:
            initial_activations: Dictionary mapping concepts to initial activation
            steps: Number of propagation steps to run

        Returns:
            Dictionary of resulting concept activations
        """
        with self.lock:
            # Start with initial activations
            current = defaultdict(float)
            for concept, activation in initial_activations.items():
                current[concept] = activation

            # Propagate for specified number of steps
            for _ in range(steps):
                next_activations = defaultdict(float)

                # First, copy decay of current activations
                decay_factor = 0.8  # Signals decay by 20% per step
                for concept, activation in current.items():
                    next_activations[concept] += activation * decay_factor

                # Then propagate signals along pathways
                for (source, target), strength in self.signal_pathways.items():
                    if source in current and current[source] > 0.1:  # Threshold
                        propagated = current[source] * strength
                        next_activations[target] += propagated

                # Normalize activations (prevent runaway feedback loops)
                for concept in next_activations:
                    next_activations[concept] = min(1.0, next_activations[concept])

                current = next_activations

            return dict(current)

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_default_strategies(self) -> None:
        """Register the default set of learning strategies"""
        # Supervised learning strategy
        self.register_strategy(LearningStrategy(
            strategy_id="supervised_basic",
            name="Basic Supervised Learning",
            paradigm=LearningParadigm.SUPERVISED,
            parameters={
                "learning_rate": 0.01,
                "batch_size": 32,
                "epochs": 10
            },
            activation_threshold=0.3,
            current_activation=0.5
        ))

        # Reinforcement learning strategy
        self.register_strategy(LearningStrategy(
            strategy_id="reinforcement_basic",
            name="Basic Reinforcement Learning",
            paradigm=LearningParadigm.REINFORCEMENT,
            parameters={
                "discount_factor": 0.99,
                "exploration_rate": 0.1,
                "learning_rate": 0.001
            },
            activation_threshold=0.3,
            current_activation=0.4
        ))

        # Continuous learning strategy
        self.register_strategy(LearningStrategy(
            strategy_id="continuous_ewc",
            name="Elastic Weight Consolidation",
            paradigm=LearningParadigm.CONTINUOUS,
            parameters={
                "ewc_lambda": 5000,
                "fisher_sample_size": 200
            },
            activation_threshold=0.3,
            current_activation=0.2,
            affinity_map={"memory_constraint": 0.8}
        ))

        # Meta-learning strategy
        self.register_strategy(LearningStrategy(
            strategy_id="meta_maml",
            name="Model-Agnostic Meta-Learning",
            paradigm=LearningParadigm.META,
            parameters={
                "inner_lr": 0.01,
                "outer_lr": 0.001,
                "inner_steps": 5
            },
            activation_threshold=0.4,
            current_activation=0.1,
            affinity_map={"new_task": 0.9, "few_shot": 0.8}
        ))

        # Intrinsic motivation strategy
        self.register_strategy(LearningStrategy(
            strategy_id="intrinsic_rnd",
            name="Random Network Distillation",
            paradigm=LearningParadigm.INTRINSIC,
            parameters={
                "reward_scale": 0.1,
                "predictor_lr": 0.001
            },
            activation_threshold=0.2,
            current_activation=0.3,
            affinity_map={"exploration": 0.9, "sparse_reward": 0.7}
        ))

    def _initialize_adaptive_parameters(self) -> None:
        """Initialize adaptive parameters for learning system"""
        # Learning rate parameter
        self.parameters["learning_rate"] = AdaptiveLearningParameter(
            name="learning_rate",
            initial_value=0.01,
            min_value=0.0001,
            max_value=0.1,
            adaptation_rate=0.05
        )

        # Exploration rate parameter
        self.parameters["exploration_rate"] = AdaptiveLearningParameter(
            name="exploration_rate",
            initial_value=0.2,
            min_value=0.01,
            max_value=0.5,
            adaptation_rate=0.1
        )

        # Memory retention parameter
        self.parameters["memory_retention"] = AdaptiveLearningParameter(
            name="memory_retention",
            initial_value=0.8,
            min_value=0.5,
            max_value=0.99,
            adaptation_rate=0.02
        )

        # Curiosity factor parameter
        self.parameters["curiosity_factor"] = AdaptiveLearningParameter(
            name="curiosity_factor",
            initial_value=0.3,
            min_value=0.1,
            max_value=0.9,
            adaptation_rate=0.1
        )

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="learning_core",
                module=self,
                dependencies=dependencies
            )

            # Set up message handlers
            # (implementation depends on NexusCore's messaging system)

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _start_background_processes(self) -> None:
        """Start background threads for ongoing learning processes"""
        # Implementation would depend on specific background processes needed
        pass

    def _stop_background_processes(self) -> None:
        """Stop background learning processes"""
        # Implementation would depend on specific background processes
        pass

    def _process_observation(self, event: LearningEvent) -> None:
        """Process an observation learning event"""
        # Implementation would depend on how observations are handled
        pass

    def _process_feedback(self, event: LearningEvent) -> None:
        """Process feedback on a learning outcome"""
        # Extract feedback information
        task_id = event.metadata.get("task_id")
        if not task_id:
            self.logger.warning("Feedback event missing task_id")
            return

        success = event.metadata.get("success", False)
        score = event.metadata.get("score", 0.0)

        # Update metrics
        self.metrics["feedback_scores"].append(score)

        # Update strategy success rates
        if "strategy_id" in event.metadata:
            strategy_id = event.metadata["strategy_id"]
            if strategy_id in self.strategies:
                strategy = self.strategies[strategy_id]
                # Weighted update of success rate
                current = strategy.success_rate
                strategy.success_rate = 0.9 * current + 0.1 * (1.0 if success else 0.0)
                strategy.last_used = time.time()

                # If strategy performed very well or very poorly, adjust activation
                if score > 0.8:
                    strategy.current_activation = min(1.0, strategy.current_activation + 0.05)
                elif score < 0.2:
                    strategy.current_activation = max(0.1, strategy.current_activation - 0.05)

    def _process_reflection(self, event: LearningEvent) -> None:
        """Process a reflection on internal learning processes"""
        # Implementation would depend on how reflections are used
        pass

    def _process_discovery(self, event: LearningEvent) -> None:
        """Process a knowledge discovery event"""
        if "concept" in event.metadata:
            concept = event.metadata["concept"]
            # Boost activation for the discovered concept
            self.activation_landscape[concept] += 0.5

            # If related concepts are provided, create or strengthen pathways
            if "related_concepts" in event.metadata:
                for related in event.metadata["related_concepts"]:
                    self.reinforce_pathway(concept, related, 0.3)
                    self.reinforce_pathway(related, concept, 0.1)  # Asymmetric reinforcement

    def _process_curiosity(self, event: LearningEvent) -> None:
        """Process a curiosity-driven learning event"""
        # Implementation would depend on curiosity mechanisms
        pass

    def _update_activation_landscape(self, event: LearningEvent) -> None:
        """Update the activation landscape based on learning events"""
        # Extract concepts from event
        concepts = []

        # Check event content for concepts
        if isinstance(event.content, str):
            # Simple example - in real implementation, would use NLP or concept extraction
            words = event.content.split()
            concepts.extend(words)

        # Check metadata for explicit concepts
        if "concepts" in event.metadata:
            concepts.extend(event.metadata["concepts"])

        # Update activation for each concept
        for concept in concepts:
            # Increase activation (with saturation)
            current = self.activation_landscape[concept]
            # Sigmoid-like update to prevent runaway activation
            self.activation_landscape[concept] = current + event.importance * (1.0 - current)

        # Decay activations for all concepts
        for concept in list(self.activation_landscape.keys()):
            self.activation_landscape[concept] *= 0.95  # 5% decay per event

            # Remove concepts with negligible activation
            if self.activation_landscape[concept] < 0.01:
                del self.activation_landscape[concept]

    def _detect_event_patterns(self) -> None:
        """
        Detect patterns in recent learning events that might indicate
        emergent learning behaviors or opportunities.
        """
        # We need enough events to detect patterns
        if len(self.event_history) < 10:
            return

        # Look for repeated sequences of event types
        recent_types = [e.event_type for e in self.event_history][-10:]

        # Check for alternating patterns
        alternating = True
        for i in range(2, len(recent_types), 2):
            if recent_types[i] != recent_types[i-2]:
                alternating = False
                break

        if alternating and len(recent_types) >= 6:
            pattern_key = f"{recent_types[0]}-{recent_types[1]}"
            if pattern_key not in self.event_patterns:
                self.event_patterns[pattern_key] = {
                    'count': 1,
                    'first_seen': time.time()
                }
                self.logger.info(f"Detected new event pattern: Alternating {pattern_key}")
            else:
                self.event_patterns[pattern_key]['count'] += 1

        # Check for frequency of specific event types
        type_counts = {}
        for e_type in recent_types:
            type_counts[e_type] = type_counts.get(e_type, 0) + 1

        # Look for dominance of specific event types
        for e_type, count in type_counts.items():
            if count >= 7:  # 70% or more
                pattern_key = f"dominant-{e_type}"
                if pattern_key not in self.event_patterns:
                    self.event_patterns[pattern_key] = {
                        'count': 1,
                        'first_seen': time.time()
                    }
                    self.logger.info(f"Detected new event pattern: Dominant {e_type}")
                else:
                    self.event_patterns[pattern_key]['count'] += 1

    def _adapt_strategies_to_context(self, context: Dict[str, Any]) -> None:
        """
        Adapt strategy activation levels based on the current context.
        This allows different learning approaches to emerge as dominant
        depending on the situation.

        Args:
            context: Dictionary of contextual factors
        """
        # Convert context to normalized form for strategy evaluation
        normalized_context = {}
        for key, value in context.items():
            if isinstance(value, bool):
                normalized_context[key] = 1.0 if value else 0.0
            elif isinstance(value, (int, float)):
                # Scale to 0-1 range (assuming most values are in 0-100 range)
                normalized_context[key] = min(1.0, max(0.0, value / 100.0))
            else:
                # For strings or other types, just indicate presence
                normalized_context[key] = 1.0

        # Calculate applicability scores for each strategy
        scores = {}
        for strategy_id, strategy in self.strategies.items():
            scores[strategy_id] = strategy.is_applicable(
                LearningTask(paradigm=strategy.paradigm),  # Dummy task matching the strategy
                normalized_context
            )

        # Adjust activations based on scores
        # Using a softmax-like normalization to create competition
        if scores:
            max_score = max(scores.values())
            exp_scores = {k: np.exp(v - max_score) for k, v in scores.items()}
            sum_exp = sum(exp_scores.values())

            for strategy_id, exp_score in exp_scores.items():
                # Calculate target activation (softmax-normalized)
                target = exp_score / sum_exp

                # Gradually move current activation toward target
                current = self.strategies[strategy_id].current_activation
                # Move 10% of the way toward target
                new_activation = current + 0.1 * (target - current)

                # Update strategy activation
                self.strategies[strategy_id].current_activation = new_activation

                # Log significant changes
                if abs(new_activation - current) > 0.1:
                    self.logger.info(f"Strategy {strategy_id} activation changed: {current:.2f} -> {new_activation:.2f}")

    def _adapt_parameters_to_context(self, context: Dict[str, Any]) -> None:
        """
        Adapt learning parameters based on the current context.

        Args:
            context: Dictionary of contextual factors
        """
        # Normalize context for parameter adaptation
        normalized_context = {}
        for key, value in context.items():
            if isinstance(value, bool):
                normalized_context[key] = 1.0 if value else 0.0
            elif isinstance(value, (int, float)):
                normalized_context[key] = min(1.0, max(0.0, value / 100.0))
            else:
                normalized_context[key] = 1.0

        # Process specific contextual factors that should influence parameters

        # If high time pressure, decrease exploration
        if "time_pressure" in normalized_context and normalized_context["time_pressure"] > 0.7:
            self.parameters["exploration_rate"].adapt(-0.5, normalized_context)

        # If new environment, increase exploration
        if "new_environment" in normalized_context and normalized_context["new_environment"] > 0.5:
            self.parameters["exploration_rate"].adapt(0.7, normalized_context)
            self.parameters["curiosity_factor"].adapt(0.5, normalized_context)

        # If performance is recently poor, adjust learning rate
        if "recent_performance" in normalized_context:
            perf = normalized_context["recent_performance"]
            if perf < 0.3:  # Poor performance
                # Try a bigger learning rate change
                self.parameters["learning_rate"].adapt(0.4, normalized_context)
            elif perf > 0.8:  # Great performance
                # Slightly reduce learning rate to stabilize
                self.parameters["learning_rate"].adapt(-0.1, normalized_context)

        # If this is critical task, increase memory retention
        if "task_importance" in normalized_context and normalized_context["task_importance"] > 0.8:
            self.parameters["memory_retention"].adapt(0.3, normalized_context)

        # Periodically analyze parameter history to learn context influences
        if random.random() < 0.05:  # 5% chance per adaptation
            for param in self.parameters.values():
                param.learn_context_influence()

    def _check_for_systemic_emergence(self, source: str, target: str) -> None:
        """
        Check if a new strong connection between source and target
        represents an emergent bridge between previously separate subsystems.

        Args:
            source: Source concept/node
            target: Target concept/node
        """
        # Build a graph of strong connections
        strong_connections = defaultdict(set)
        for (s, t), strength in self.signal_pathways.items():
            if strength >= 0.5:  # Consider connections with at least 0.5 strength
                strong_connections[s].add(t)

        # Check if source and target were previously in separate connected components
        # by performing a flood fill from source before this new connection

        # Temporarily remove the direct connection
        if target in strong_connections[source]:
            strong_connections[source].remove(target)

        # Perform a breadth-first search from source
        visited = set([source])
        queue = deque([source])
        while queue:
            current = queue.popleft()
            for neighbor in strong_connections[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        # If target wasn't reachable before, this is a bridge between subsystems
        if target not in visited:
            # This is an emergent bridge!
            self.logger.info(f"Emergent bridge detected between subsystems via {source} -> {target}")

            # Record this as an emergent pattern
            self.emergent_patterns.append({
                'type': 'bridge',
                'source': source,
                'target': target,
                'time': time.time(),
                'source_subsystem': list(visited),
                'target_subsystem': [n for n in strong_connections if n not in visited]
            })

    def _select_learning_strategy(self, task: LearningTask, context: Dict[str, Any]) -> Optional[str]:
        """
        Select the most appropriate learning strategy for a given task and context.

        Args:
            task: The learning task to process
            context: Current contextual factors

        Returns:
            strategy_id of selected strategy, or None if no suitable strategy found
        """
        candidates = []

        # Evaluate each strategy's applicability to the task
        for strategy_id, strategy in self.strategies.items():
            if strategy.current_activation >= strategy.activation_threshold:
                score = strategy.is_applicable(task, context)
                if score > 0:
                    candidates.append((strategy_id, score))

        if not candidates:
            return None

        # Sort by score (highest first)
        candidates.sort(key=lambda x: x[1], reverse=True)

        # Introduce some exploration (don't always pick the highest)
        if random.random() < self.parameters["exploration_rate"].value:
            # Pick randomly from top half of candidates
            top_half = candidates[:max(1, len(candidates)//2)]
            return random.choice(top_half)[0]
        else:
            # Pick the highest scoring strategy
            return candidates[0][0]

    def _execute_learning_task(self, task: LearningTask) -> None:
        """
        Execute a learning task using the appropriate strategy.

        Args:
            task: The learning task to execute
        """
        # Prepare execution context
        context = {}

        # Extract task-relevant contextual factors
        if self.working_memory:
            # Get current working memory state
            try:
                working_mem_items = self.working_memory.get_all_items()
                context["working_memory_size"] = len(working_mem_items)

                # Extract relevant items for the task
                relevant_items = []
                if task.description:
                    # Find items related to the task description
                    for item in working_mem_items:
                        # Simplified relevance check - would be more sophisticated in real system
                        if any(word in item.content for word in task.description.split()):
                            relevant_items.append(item)

                context["relevant_items"] = len(relevant_items)
            except Exception as e:
                self.logger.warning(f"Error accessing working memory: {str(e)}")

        # Determine task complexity
        if isinstance(task.data, dict) and "complexity" in task.data:
            context["task_complexity"] = task.data["complexity"]
        else:
            # Estimate complexity based on data size
            try:
                data_size = len(task.data) if hasattr(task.data, "__len__") else 1
                context["task_complexity"] = min(1.0, data_size / 1000)  # Normalize to 0-1
            except (TypeError, AttributeError):
                context["task_complexity"] = 0.5  # Default medium complexity

        # Select strategy
        strategy_id = self._select_learning_strategy(task, context)

        if not strategy_id:
            self.logger.warning(f"No suitable strategy found for task {task.task_id}")
            task.status = "failed"
            if task.callback:
                task.callback(task)
            return

        strategy = self.strategies[strategy_id]
        self.logger.info(f"Selected strategy '{strategy.name}' for task {task.task_id}")

        # Mark task as in progress
        task.status = "running"

        # Execute the appropriate learning algorithm based on strategy
        try:
            if strategy.paradigm == LearningParadigm.SUPERVISED:
                result = self._execute_supervised_learning(task, strategy, context)
            elif strategy.paradigm == LearningParadigm.REINFORCEMENT:
                result = self._execute_reinforcement_learning(task, strategy, context)
            elif strategy.paradigm == LearningParadigm.CONTINUOUS:
                result = self._execute_continuous_learning(task, strategy, context)
            elif strategy.paradigm == LearningParadigm.META:
                result = self._execute_meta_learning(task, strategy, context)
            elif strategy.paradigm == LearningParadigm.INTRINSIC:
                result = self._execute_intrinsic_learning(task, strategy, context)
            else:
                raise ValueError(f"Unsupported learning paradigm: {strategy.paradigm}")

            # Update task with result
            task.result = result
            task.status = "completed"

            # Record success metrics
            self.metrics["task_success_rate"].append(1.0)

            # Update strategy usage statistics
            strategy.last_used = time.time()

            # Log completion
            self.logger.info(f"Task {task.task_id} completed successfully")

        except Exception as e:
            self.logger.error(f"Error executing task {task.task_id}: {str(e)}")
            task.status = "failed"
            task.result = {"error": str(e)}

            # Record failure metrics
            self.metrics["task_success_rate"].append(0.0)

        # Move task to completed
        if task in self.active_tasks:
            self.active_tasks.remove(task)
        self.completed_tasks.append(task)

        # Execute callback if provided
        if task.callback:
            try:
                task.callback(task)
            except Exception as e:
                self.logger.error(f"Error in task callback: {str(e)}")

    def _execute_supervised_learning(self, task: LearningTask,
                                     strategy: LearningStrategy,
                                     context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute a supervised learning task.

        Args:
            task: The learning task with data
            strategy: The selected learning strategy
            context: Contextual factors

        Returns:
            Dictionary with learning results
        """
        # This is a placeholder implementation
        # In a real system, this would be much more sophisticated

        self.logger.info(f"Executing supervised learning for task {task.task_id}")

        # Extract parameters
        learning_rate = task.parameters.get("learning_rate",
                                           strategy.parameters.get("learning_rate",
                                                                 self.parameters["learning_rate"].value))

        batch_size = task.parameters.get("batch_size",
                                        strategy.parameters.get("batch_size", 32))

        epochs = task.parameters.get("epochs",
                                    strategy.parameters.get("epochs", 10))

        # Extract data
        if not task.data or not isinstance(task.data, dict):
            raise ValueError("Task data must be a dictionary for supervised learning")

        X_train = task.data.get("X_train")
        y_train = task.data.get("y_train")
        X_val = task.data.get("X_val")
        y_val = task.data.get("y_val")

        if X_train is None or y_train is None:
            raise ValueError("Missing training data (X_train, y_train)")

        # In a real implementation, this would create and train a model
        # Here we'll just simulate the training process

        # Simulated training
        model = {"type": "supervised_model"}
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Simulate training epoch
            train_loss = 1.0 / (epoch + 1)  # Simulated loss that decreases over time
            train_losses.append(train_loss)

            # Simulate validation if validation data provided
            if X_val is not None and y_val is not None:
                val_loss = train_loss * 1.2  # Slightly higher than training loss
                val_losses.append(val_loss)

        # Prepare results
        result = {
            "model": model,
            "train_losses": train_losses,
            "epochs_completed": epochs
        }

        if val_losses:
            result["val_losses"] = val_losses
            result["best_val_loss"] = min(val_losses)

        return result

    def _execute_reinforcement_learning(self, task: LearningTask,
                                       strategy: LearningStrategy,
                                       context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute a reinforcement learning task.

        Args:
            task: The learning task with environment data
            strategy: The selected learning strategy
            context: Contextual factors

        Returns:
            Dictionary with learning results
        """
        # Placeholder implementation
        self.logger.info(f"Executing reinforcement learning for task {task.task_id}")

        # Extract parameters
        learning_rate = task.parameters.get("learning_rate",
                                          strategy.parameters.get("learning_rate",
                                                                self.parameters["learning_rate"].value))

        discount_factor = task.parameters.get("discount_factor",
                                            strategy.parameters.get("discount_factor", 0.99))

        exploration_rate = task.parameters.get("exploration_rate",
                                             strategy.parameters.get("exploration_rate",
                                                                   self.parameters["exploration_rate"].value))

        # Extract environment data
        if not task.data or not isinstance(task.data, dict):
            raise ValueError("Task data must be a dictionary for reinforcement learning")

        env = task.data.get("environment")
        episodes = task.data.get("episodes", 100)

        if env is None:
            raise ValueError("Missing environment specification")

        # In a real implementation, this would interact with the environment
        # Here we'll just simulate the learning process

        # Simulated training
        policy = {"type": "reinforcement_policy"}
        rewards = []

        for episode in range(episodes):
            # Simulate an episode
            episode_reward = (1 - np.exp(-episode/20)) * 100  # Reward improves over time
            rewards.append(episode_reward)

        # Prepare results
        result = {
            "policy": policy,
            "episode_rewards": rewards,
            "episodes_completed": episodes,
            "final_exploration_rate": exploration_rate * (0.1 ** (episodes / 100))
        }

        return result

    def _execute_continuous_learning(self, task: LearningTask,
                                    strategy: LearningStrategy,
                                    context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute a continuous learning task.

        Args:
            task: The learning task with stream data
            strategy: The selected learning strategy
            context: Contextual factors

        Returns:
            Dictionary with learning results
        """
        # Placeholder implementation
        self.logger.info(f"Executing continuous learning for task {task.task_id}")

        # Extract parameters based on strategy type
        if strategy.strategy_id == "continuous_ewc":
            ewc_lambda = task.parameters.get("ewc_lambda",
                                           strategy.parameters.get("ewc_lambda", 5000))

            fisher_sample_size = task.parameters.get("fisher_sample_size",
                                                    strategy.parameters.get("fisher_sample_size", 200))

        # Extract data
        if not task.data or not isinstance(task.data, dict):
            raise ValueError("Task data must be a dictionary for continuous learning")

        previous_model = task.data.get("previous_model")
        new_data = task.data.get("new_data")
        task_id = task.data.get("task_id")

        if previous_model is None:
            raise ValueError("Missing previous model")
        if new_data is None:
            raise ValueError("Missing new data")

        # In a real implementation, this would update the model with EWC or other CL method
        # Here we'll just simulate the process

        # Simulate model update
        updated_model = {"type": "updated_model", "base": previous_model, "task_id": task_id}
        performance = {}

        # Simulate performance on current and previous tasks
        performance["current_task"] = 0.85  # Good performance on current task
        performance["previous_tasks"] = 0.75  # Slight forgetting on previous tasks

        # Prepare results
        result = {
            "model": updated_model,
            "performance": performance,
            "forgetting_measure": 1 - performance["previous_tasks"]
        }

        return result

    def _execute_meta_learning(self, task: LearningTask,
                              strategy: LearningStrategy,
                              context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute a meta-learning task.

        Args:
            task: The learning task with meta-tasks
            strategy: The selected learning strategy
            context: Contextual factors

        Returns:
            Dictionary with learning results
        """
        # Placeholder implementation
        self.logger.info(f"Executing meta-learning for task {task.task_id}")

        # Extract parameters based on strategy
        if strategy.strategy_id == "meta_maml":
            inner_lr = task.parameters.get("inner_lr",
                                         strategy.parameters.get("inner_lr", 0.01))

            outer_lr = task.parameters.get("outer_lr",
                                         strategy.parameters.get("outer_lr", 0.001))

            inner_steps = task.parameters.get("inner_steps",
                                            strategy.parameters.get("inner_steps", 5))

        # Extract data
        if not task.data or not isinstance(task.data, dict):
            raise ValueError("Task data must be a dictionary for meta-learning")

        meta_train_tasks = task.data.get("meta_train_tasks")
        meta_val_tasks = task.data.get("meta_val_tasks")
        meta_test_tasks = task.data.get("meta_test_tasks")

        if meta_train_tasks is None:
            raise ValueError("Missing meta-training tasks")

        # In a real implementation, this would perform MAML training
        # Here we'll just simulate the process

        # Simulate meta-training
        meta_model = {"type": "meta_model"}
        meta_train_losses = []
        meta_val_losses = []

        for epoch in range(10):  # Simulate 10 meta-epochs
            # Simulate meta-training epoch
            train_loss = 1.0 / (epoch + 1)  # Simulated loss that decreases over time
            meta_train_losses.append(train_loss)

            # Simulate meta-validation if available
            if meta_val_tasks:
                val_loss = train_loss * 1.1  # Slightly higher than training loss
                meta_val_losses.append(val_loss)

        # Simulate meta-testing if available
        meta_test_results = None
        if meta_test_tasks:
            meta_test_results = []
            for i, test_task in enumerate(meta_test_tasks):
                # Simulate adaptation to test task
                task_result = {
                    "task_id": f"test_task_{i}",
                    "pre_adaptation_loss": 0.8,
                    "post_adaptation_loss": 0.3,
                    "adaptation_steps": inner_steps
                }
                meta_test_results.append(task_result)

        # Prepare results
        result = {
            "meta_model": meta_model,
            "meta_train_losses": meta_train_losses
        }

        if meta_val_losses:
            result["meta_val_losses"] = meta_val_losses

        if meta_test_results:
            result["meta_test_results"] = meta_test_results
            result["average_adaptation_improvement"] = 0.5  # Simulated improvement after adaptation

        return result

    def _execute_intrinsic_learning(self, task: LearningTask,
                                   strategy: LearningStrategy,
                                   context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute an intrinsic motivation-based learning task.

        Args:
            task: The learning task with environment data
            strategy: The selected learning strategy
            context: Contextual factors

        Returns:
            Dictionary with learning results
        """
        # Placeholder implementation
        self.logger.info(f"Executing intrinsic motivation learning for task {task.task_id}")

        # Extract parameters based on strategy
        if strategy.strategy_id == "intrinsic_rnd":
            reward_scale = task.parameters.get("reward_scale",
                                             strategy.parameters.get("reward_scale", 0.1))

            predictor_lr = task.parameters.get("predictor_lr",
                                             strategy.parameters.get("predictor_lr", 0.001))

        # Extract data
        if not task.data or not isinstance(task.data, dict):
            raise ValueError("Task data must be a dictionary for intrinsic learning")

        env = task.data.get("environment")
        episodes = task.data.get("episodes", 100)

        if env is None:
            raise ValueError("Missing environment specification")

        # In a real implementation, this would set up RND networks and train
        # Here we'll just simulate the process

        # Simulate training
        policy = {"type": "intrinsic_policy"}
        extrinsic_rewards = []
        intrinsic_rewards = []
        prediction_errors = []

        for episode in range(episodes):
            # Simulate an episode
            extrinsic_reward = (1 - np.exp(-episode/30)) * 100  # External reward improves over time

            # Intrinsic reward starts high and decreases as environment becomes familiar
            intrinsic_reward = 100 * np.exp(-episode/20)

            # Prediction error follows the same pattern as intrinsic reward
            prediction_error = 1.0 * np.exp(-episode/20)

            extrinsic_rewards.append(extrinsic_reward)
            intrinsic_rewards.append(intrinsic_reward)
            prediction_errors.append(prediction_error)

        # Prepare results
        result = {
            "policy": policy,
            "extrinsic_rewards": extrinsic_rewards,
            "intrinsic_rewards": intrinsic_rewards,
            "prediction_errors": prediction_errors,
            "episodes_completed": episodes,
            "exploration_coverage": 0.85  # Simulated coverage of state space
        }

        return result

# Continuous Learner

In [ ]:
"""
continuous_learner.py - Continuous Learning Module for GrandNexus

This module provides continuous learning capabilities for GrandNexus, allowing the system
to learn from a stream of data over time while mitigating catastrophic forgetting.
It leverages the Avalanche framework for continual learning strategies.

The module supports:
- Sequential task learning with various continual learning strategies
- Dynamic model adaptation to new data without catastrophic forgetting
- Experience replay from episodic memory
- Knowledge consolidation into semantic memory
- Performance monitoring across tasks

Key design principles:
- Adaptivity: Models maintain performance on previous tasks while learning new ones
- Efficiency: Strategies balance computational resources with forgetting prevention
- Integration: Deep connection with memory systems for experience storage and retrieval
- Emergence: Complex learning behaviors emerge from simple learning rules
"""

import logging
import threading
import time
import uuid
from typing import Dict, List, Any, Optional, Tuple, Set, Union, Callable
from dataclasses import dataclass, field
from enum import Enum, auto
from collections import defaultdict, deque
import copy
import random

# Import from GrandNexus core modules
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.learning.learning_core import LearningCore, LearningParadigm, LearningTask, LearningResult

# Try to import Avalanche
try:
    import torch
    import numpy as np
    from torch.utils.data import Dataset, DataLoader

    # Avalanche imports
    import avalanche
    from avalanche.benchmarks import NCScenario, ni_benchmark
    from avalanche.training.strategies import Naive, CWRStar, GEM, AGEM, EWC, LwF, SynapticIntelligence
    from avalanche.models import SimpleMLP, MTSimpleMLP, SimpleCNN
    from avalanche.evaluation.metrics import forgetting_metrics, accuracy_metrics, loss_metrics

    HAS_AVALANCHE = True
except ImportError:
    HAS_AVALANCHE = False

# Define continual learning strategies
class CLStrategy(Enum):
    """Continual learning strategies available in the system"""
    NAIVE = auto()          # Naive fine-tuning (baseline, prone to forgetting)
    EWC = auto()            # Elastic Weight Consolidation
    LWF = auto()            # Learning without Forgetting
    SI = auto()             # Synaptic Intelligence
    GEM = auto()            # Gradient Episodic Memory
    AGEM = auto()           # Averaged GEM
    REPLAY = auto()         # Experience Replay
    CWR_STAR = auto()       # CWR*
    DER = auto()            # Dark Experience Replay
    CUSTOM = auto()         # Custom strategy

# Define task types
class TaskType(Enum):
    """Types of continual learning tasks"""
    CLASSIFICATION = auto()  # Standard classification task
    REGRESSION = auto()      # Regression task
    EMBEDDING = auto()       # Learning embeddings
    GENERATION = auto()      # Generative task
    CUSTOM = auto()          # Custom task type

@dataclass
class CLExperience:
    """
    Represents a learning experience (task) in continual learning.

    An experience is a set of data for a particular task or time period
    that the model learns from before moving to the next experience.
    """
    experience_id: str
    task_id: int  # Numeric task identifier used by Avalanche
    name: str
    data: Any  # The actual data for this experience
    metadata: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)

    # Metrics for this experience
    metrics: Dict[str, float] = field(default_factory=dict)

    # For tracking train/eval history
    train_history: List[Dict[str, float]] = field(default_factory=list)
    eval_history: List[Dict[str, float]] = field(default_factory=list)

    def add_train_metric(self, metrics: Dict[str, float]) -> None:
        """Add training metrics for this experience"""
        self.train_history.append({
            'timestamp': time.time(),
            **metrics
        })

        # Update latest metrics
        self.metrics.update(metrics)

    def add_eval_metric(self, metrics: Dict[str, float]) -> None:
        """Add evaluation metrics for this experience"""
        self.eval_history.append({
            'timestamp': time.time(),
            **metrics
        })

        # Update latest metrics
        self.metrics.update(metrics)

@dataclass
class CLScenario:
    """
    Represents a continual learning scenario.

    A scenario consists of a sequence of experiences (tasks) that the
    model encounters, along with configuration for how learning should
    proceed through these experiences.
    """
    scenario_id: str
    name: str
    task_type: TaskType
    experiences: List[CLExperience] = field(default_factory=list)
    avalanche_scenario: Optional[Any] = None  # Avalanche's NCScenario
    strategy: CLStrategy = CLStrategy.EWC
    strategy_params: Dict[str, Any] = field(default_factory=dict)
    metadata: Dict[str, Any] = field(default_factory=dict)

    # State tracking
    current_experience_idx: int = -1
    model: Any = None  # PyTorch model
    cl_strategy: Any = None  # Avalanche strategy instance

    creation_time: float = field(default_factory=time.time)
    last_updated: float = field(default_factory=time.time)

    @property
    def current_experience(self) -> Optional[CLExperience]:
        """Get the current experience being learned"""
        if 0 <= self.current_experience_idx < len(self.experiences):
            return self.experiences[self.current_experience_idx]
        return None

    def advance_to_next_experience(self) -> bool:
        """Move to the next experience in the scenario"""
        if self.current_experience_idx + 1 < len(self.experiences):
            self.current_experience_idx += 1
            self.last_updated = time.time()
            return True
        return False

    def has_more_experiences(self) -> bool:
        """Check if there are more experiences to learn"""
        return self.current_experience_idx + 1 < len(self.experiences)

class GrandNexusDataset(Dataset):
    """
    Custom PyTorch Dataset for GrandNexus data.

    This dataset adapts data from GrandNexus memory systems
    to a format usable by PyTorch and Avalanche.
    """
    def __init__(self, data: List[Tuple[Any, Any]], transform=None, target_transform=None):
        """
        Initialize the dataset.

        Args:
            data: List of (input, target) tuples
            transform: Optional transform for inputs
            target_transform: Optional transform for targets
        """
        self.data = data
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self) -> int:
        """Get the number of samples in the dataset"""
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[Any, Any]:
        """
        Get a sample from the dataset.

        Args:
            idx: Index of the sample

        Returns:
            Tuple of (input, target)
        """
        x, y = self.data[idx]

        if self.transform:
            x = self.transform(x)

        if self.target_transform:
            y = self.target_transform(y)

        return x, y

class ContinuousLearner:
    """
    ContinuousLearner module for GrandNexus.

    This module implements continual learning using the Avalanche framework,
    allowing the system to learn from a stream of data while mitigating
    catastrophic forgetting. It integrates with GrandNexus memory systems
    for experience storage and retrieval.
    """

    def __init__(self, learning_core: Optional[LearningCore] = None,
                 nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None):
        """
        Initialize the ContinuousLearner module.

        Args:
            learning_core: Reference to the main LearningCore module
            nexus_core: Reference to the NexusCore for system integration
            working_memory: Reference to working memory for context
            semantic_memory: Reference to semantic memory for knowledge storage
            episodic_memory: Reference to episodic memory for experience storage
        """
        self.logger = logging.getLogger("GrandNexus.Learning.ContinuousLearner")

        if not HAS_AVALANCHE:
            self.logger.warning("Avalanche framework not available. Limited functionality.")

        # Store references to other modules
        self.learning_core = learning_core
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Scenarios and models
        self.scenarios: Dict[str, CLScenario] = {}
        self.active_scenarios: Set[str] = set()

        # Strategy factories
        self.strategy_factories = {}

        # Background learning thread
        self.learning_thread = None
        self.stop_learning = threading.Event()

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"ContinuousLearner initialized with ID={self.instance_id}")

        # Device for PyTorch (CPU/GPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if HAS_AVALANCHE else None

    def initialize(self) -> bool:
        """Initialize the continuous learner module"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register strategy factories
                if HAS_AVALANCHE:
                    self._register_strategy_factories()

                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("ContinuousLearner initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"ContinuousLearner initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the continuous learner module"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            # Start background learning thread
            self.stop_learning.clear()
            self.learning_thread = threading.Thread(target=self._learning_loop, daemon=True)
            self.learning_thread.start()

            self.running = True
            self.logger.info("ContinuousLearner started")
            return True

    def stop(self) -> bool:
        """Stop the continuous learner module"""
        with self.lock:
            if not self.running:
                return True

            # Stop background learning thread
            self.stop_learning.set()
            if self.learning_thread:
                self.learning_thread.join(timeout=5.0)
                self.learning_thread = None

            self.running = False
            self.logger.info("ContinuousLearner stopped")
            return True

    def create_scenario(self, name: str, task_type: TaskType,
                      strategy: CLStrategy = CLStrategy.EWC,
                      strategy_params: Optional[Dict[str, Any]] = None,
                      metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new continual learning scenario.

        Args:
            name: Name of the scenario
            task_type: Type of task (classification, regression, etc.)
            strategy: Continual learning strategy to use
            strategy_params: Parameters for the strategy
            metadata: Additional information about the scenario

        Returns:
            Scenario ID
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot create scenario: Avalanche framework not available")
            return ""

        with self.lock:
            scenario_id = f"scenario_{uuid.uuid4().hex[:8]}"

            if strategy_params is None:
                strategy_params = {}

            if metadata is None:
                metadata = {}

            scenario = CLScenario(
                scenario_id=scenario_id,
                name=name,
                task_type=task_type,
                strategy=strategy,
                strategy_params=strategy_params,
                metadata=metadata
            )

            self.scenarios[scenario_id] = scenario
            self.logger.info(f"Created scenario {scenario_id}: {name}")

            return scenario_id

    def add_experience(self, scenario_id: str, name: str, data: Any,
                     task_id: Optional[int] = None,
                     metadata: Optional[Dict[str, Any]] = None) -> Optional[str]:
        """
        Add a learning experience to a scenario.

        Args:
            scenario_id: ID of the scenario to add experience to
            name: Name of the experience
            data: Data for the experience (X_train, y_train, X_val, y_val)
            task_id: Task ID for the experience (auto-assigned if None)
            metadata: Additional information about the experience

        Returns:
            Experience ID if successful, None otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot add experience: Avalanche framework not available")
            return None

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return None

            # Assign task ID if not provided
            if task_id is None:
                task_id = len(scenario.experiences)

            # Create experience
            experience_id = f"exp_{scenario_id}_{uuid.uuid4().hex[:6]}"

            if metadata is None:
                metadata = {}

            experience = CLExperience(
                experience_id=experience_id,
                task_id=task_id,
                name=name,
                data=data,
                metadata=metadata
            )

            # Add to scenario
            scenario.experiences.append(experience)
            scenario.last_updated = time.time()

            # If this is the first experience, set current_experience_idx to 0
            if scenario.current_experience_idx == -1 and len(scenario.experiences) == 1:
                scenario.current_experience_idx = 0

            # Reset Avalanche scenario to force regeneration
            scenario.avalanche_scenario = None

            self.logger.info(f"Added experience {experience_id} to scenario {scenario_id}")
            return experience_id

    def initialize_model(self, scenario_id: str, model_config: Optional[Dict[str, Any]] = None) -> bool:
        """
        Initialize a model for a scenario.

        Args:
            scenario_id: ID of the scenario to initialize model for
            model_config: Configuration for the model

        Returns:
            True if successful, False otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot initialize model: Avalanche framework not available")
            return False

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return False

            if not scenario.experiences:
                self.logger.warning(f"Scenario {scenario_id} has no experiences")
                return False

            # Default model config
            if model_config is None:
                model_config = {}

            # Get example data from first experience to determine input/output shapes
            first_exp = scenario.experiences[0]
            example_data = first_exp.data

            try:
                # Extract input shape and output size from example data
                if isinstance(example_data, dict):
                    X = example_data.get("X_train") or example_data.get("inputs")
                    y = example_data.get("y_train") or example_data.get("targets")
                elif isinstance(example_data, tuple) and len(example_data) >= 2:
                    X, y = example_data[:2]
                else:
                    raise ValueError("Invalid data format in experience")

                # Determine input shape
                if isinstance(X, torch.Tensor):
                    input_shape = tuple(X.shape[1:])  # Remove batch dimension
                elif isinstance(X, np.ndarray):
                    input_shape = tuple(X.shape[1:])
                else:
                    # Try to convert to numpy
                    X = np.array(X)
                    input_shape = tuple(X.shape[1:])

                # Determine output size
                if isinstance(y, torch.Tensor):
                    if len(y.shape) > 1 and y.shape[1] > 1:
                        # One-hot encoded
                        output_size = y.shape[1]
                    else:
                        # Class indices
                        output_size = len(torch.unique(y))
                elif isinstance(y, np.ndarray):
                    if len(y.shape) > 1 and y.shape[1] > 1:
                        output_size = y.shape[1]
                    else:
                        output_size = len(np.unique(y))
                else:
                    # Try to determine output size
                    unique_classes = set(y)
                    output_size = len(unique_classes)

                # Create appropriate model based on task type and data
                if scenario.task_type == TaskType.CLASSIFICATION:
                    # Determine if we need multi-head
                    multi_task = model_config.get("multi_task", False)

                    # For images (assuming channel-first)
                    if len(input_shape) == 3:
                        # Use CNN for image data
                        channels = input_shape[0]
                        model = SimpleCNN(channels, output_size)
                    else:
                        # Use MLP for other data
                        input_size = int(np.prod(input_shape))
                        hidden_size = model_config.get("hidden_size", 256)
                        hidden_layers = model_config.get("hidden_layers", 2)

                        if multi_task:
                            scenario.model = MTSimpleMLP(
                                input_size=input_size,
                                hidden_size=hidden_size,
                                num_layers=hidden_layers,
                                output_size=output_size
                            )
                        else:
                            scenario.model = SimpleMLP(
                                input_size=input_size,
                                hidden_size=hidden_size,
                                num_layers=hidden_layers,
                                output_size=output_size
                            )

                elif scenario.task_type == TaskType.REGRESSION:
                    # MLP for regression
                    input_size = int(np.prod(input_shape))
                    hidden_size = model_config.get("hidden_size", 256)
                    hidden_layers = model_config.get("hidden_layers", 2)

                    scenario.model = SimpleMLP(
                        input_size=input_size,
                        hidden_size=hidden_size,
                        num_layers=hidden_layers,
                        output_size=output_size
                    )

                # Move model to device
                scenario.model = scenario.model.to(self.device)

                self.logger.info(f"Initialized model for scenario {scenario_id}")
                return True

            except Exception as e:
                self.logger.error(f"Error initializing model for scenario {scenario_id}: {str(e)}")
                return False

    def setup_avalanche_scenario(self, scenario_id: str) -> bool:
        """
        Set up an Avalanche scenario from a GrandNexus scenario.

        Args:
            scenario_id: ID of the scenario to set up

        Returns:
            True if successful, False otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot set up Avalanche scenario: Avalanche framework not available")
            return False

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return False

            if not scenario.experiences:
                self.logger.warning(f"Scenario {scenario_id} has no experiences")
                return False

            if scenario.avalanche_scenario is not None:
                # Already set up
                return True

            try:
                # Create train/eval datasets for each experience
                train_datasets = []
                eval_datasets = []
                task_labels = []

                for exp in scenario.experiences:
                    data = exp.data

                    # Extract train/eval data
                    if isinstance(data, dict):
                        X_train = data.get("X_train") or data.get("inputs")
                        y_train = data.get("y_train") or data.get("targets")
                        X_val = data.get("X_val") or data.get("validation_inputs")
                        y_val = data.get("y_val") or data.get("validation_targets")
                    elif isinstance(data, tuple) and len(data) >= 2:
                        if len(data) >= 4:
                            X_train, y_train, X_val, y_val = data[:4]
                        else:
                            X_train, y_train = data[:2]
                            X_val, y_val = None, None
                    else:
                        raise ValueError(f"Invalid data format in experience {exp.experience_id}")

                    # Convert to tensors if needed
                    if not isinstance(X_train, torch.Tensor):
                        X_train = torch.tensor(X_train, dtype=torch.float32)
                    if not isinstance(y_train, torch.Tensor):
                        if scenario.task_type == TaskType.CLASSIFICATION:
                            y_train = torch.tensor(y_train, dtype=torch.long)
                        else:
                            y_train = torch.tensor(y_train, dtype=torch.float32)

                    # Create train dataset
                    train_data = [(x, y) for x, y in zip(X_train, y_train)]
                    train_dataset = GrandNexusDataset(train_data)
                    train_datasets.append(train_dataset)

                    # Create eval dataset if available
                    if X_val is not None and y_val is not None:
                        if not isinstance(X_val, torch.Tensor):
                            X_val = torch.tensor(X_val, dtype=torch.float32)
                        if not isinstance(y_val, torch.Tensor):
                            if scenario.task_type == TaskType.CLASSIFICATION:
                                y_val = torch.tensor(y_val, dtype=torch.long)
                            else:
                                y_val = torch.tensor(y_val, dtype=torch.float32)

                        eval_data = [(x, y) for x, y in zip(X_val, y_val)]
                        eval_dataset = GrandNexusDataset(eval_data)
                        eval_datasets.append(eval_dataset)
                    else:
                        # Use train data for eval if no separate eval data
                        eval_datasets.append(train_dataset)

                    # Task labels
                    task_labels.append(exp.task_id)

                # Create Avalanche scenario
                scenario.avalanche_scenario = ni_benchmark(
                    train_datasets=train_datasets,
                    test_datasets=eval_datasets,
                    task_labels=task_labels,
                    complete_test_set_only=False
                )

                self.logger.info(f"Set up Avalanche scenario for {scenario_id} with {len(scenario.experiences)} experiences")
                return True

            except Exception as e:
                self.logger.error(f"Error setting up Avalanche scenario for {scenario_id}: {str(e)}")
                return False

    def setup_cl_strategy(self, scenario_id: str) -> bool:
        """
        Set up a continual learning strategy for a scenario.

        Args:
            scenario_id: ID of the scenario to set up strategy for

        Returns:
            True if successful, False otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot set up CL strategy: Avalanche framework not available")
            return False

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return False

            if scenario.model is None:
                self.logger.warning(f"Scenario {scenario_id} has no model initialized")
                return False

            if scenario.avalanche_scenario is None:
                if not self.setup_avalanche_scenario(scenario_id):
                    return False

            try:
                # Get strategy factory
                strategy_factory = self.strategy_factories.get(scenario.strategy)
                if strategy_factory is None:
                    self.logger.error(f"No factory for strategy {scenario.strategy}")
                    return False

                # Create strategy
                strategy_params = scenario.strategy_params.copy()

                # Add common parameters
                strategy_params["model"] = scenario.model
                strategy_params["optimizer"] = torch.optim.Adam(scenario.model.parameters(), lr=0.001)

                if scenario.task_type == TaskType.CLASSIFICATION:
                    strategy_params["criterion"] = torch.nn.CrossEntropyLoss()
                else:  # Regression
                    strategy_params["criterion"] = torch.nn.MSELoss()

                # Create strategy
                scenario.cl_strategy = strategy_factory(**strategy_params)

                self.logger.info(f"Set up {scenario.strategy} strategy for scenario {scenario_id}")
                return True

            except Exception as e:
                self.logger.error(f"Error setting up CL strategy for {scenario_id}: {str(e)}")
                return False

    def train_current_experience(self, scenario_id: str,
                               train_epochs: int = 1) -> Dict[str, float]:
        """
        Train the model on the current experience in a scenario.

        Args:
            scenario_id: ID of the scenario to train
            train_epochs: Number of epochs to train for

        Returns:
            Training metrics
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot train: Avalanche framework not available")
            return {}

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return {}

            if scenario.current_experience_idx < 0 or scenario.current_experience_idx >= len(scenario.experiences):
                self.logger.warning(f"No current experience for scenario {scenario_id}")
                return {}

            if scenario.model is None:
                self.logger.warning(f"No model initialized for scenario {scenario_id}")
                return {}

            if scenario.cl_strategy is None:
                if not self.setup_cl_strategy(scenario_id):
                    return {}

            try:
                # Get current experience
                current_exp = scenario.current_experience
                avalanche_exp = scenario.avalanche_scenario.train_stream[scenario.current_experience_idx]

                self.logger.info(f"Training on experience {current_exp.name} ({scenario_id})")

                # Train
                metrics = {}
                scenario.cl_strategy.train(
                    experiences=avalanche_exp,
                    eval_streams=[],
                    num_workers=0,
                    eval_every=-1
                )

                # Extract metrics
                for name, value in scenario.cl_strategy.evaluator.get_last_metrics().items():
                    metrics[name] = float(value)

                # Add training metrics to experience
                current_exp.add_train_metric(metrics)

                # Update global metrics
                for name, value in metrics.items():
                    self.metrics[f"train_{name}"].append(value)

                self.logger.info(f"Completed training on experience {current_exp.name} ({scenario_id})")
                return metrics

            except Exception as e:
                self.logger.error(f"Error training experience for {scenario_id}: {str(e)}")
                return {}

    def evaluate_on_experiences(self, scenario_id: str,
                              experience_indices: Optional[List[int]] = None) -> Dict[str, Dict[str, float]]:
        """
        Evaluate the model on specific experiences in a scenario.

        Args:
            scenario_id: ID of the scenario to evaluate
            experience_indices: Indices of experiences to evaluate on (None for all)

        Returns:
            Evaluation metrics for each experience
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot evaluate: Avalanche framework not available")
            return {}

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return {}

            if scenario.model is None:
                self.logger.warning(f"No model initialized for scenario {scenario_id}")
                return {}

            if scenario.cl_strategy is None:
                self.logger.warning(f"No strategy initialized for scenario {scenario_id}")
                return {}

            try:
                # Determine which experiences to evaluate on
                if experience_indices is None:
                    # Evaluate on all experiences up to the current one
                    experience_indices = list(range(scenario.current_experience_idx + 1))

                # Filter valid indices
                experience_indices = [idx for idx in experience_indices if 0 <= idx < len(scenario.experiences)]

                if not experience_indices:
                    self.logger.warning(f"No valid experience indices for scenario {scenario_id}")
                    return {}

                results = {}

                # Evaluate on each experience
                for idx in experience_indices:
                    exp = scenario.experiences[idx]
                    avalanche_exp = scenario.avalanche_scenario.test_stream[idx]

                    self.logger.info(f"Evaluating on experience {exp.name} ({scenario_id})")

                    # Evaluate
                    metrics = scenario.cl_strategy.eval(avalanche_exp).metrics

                    # Extract metrics
                    exp_metrics = {}
                    for name, value in metrics.items():
                        if isinstance(value, torch.Tensor):
                            exp_metrics[name] = float(value.item())
                        else:
                            exp_metrics[name] = float(value)

                    # Add evaluation metrics to experience
                    exp.add_eval_metric(exp_metrics)

                    # Add to results
                    results[exp.experience_id] = exp_metrics

                    # Update global metrics
                    for name, value in exp_metrics.items():
                        self.metrics[f"eval_{name}"].append(value)

                self.logger.info(f"Completed evaluation for scenario {scenario_id}")
                return results

            except Exception as e:
                self.logger.error(f"Error evaluating experiences for {scenario_id}: {str(e)}")
                return {}

    def train_full_scenario(self, scenario_id: str, train_epochs: int = 1,
                          evaluate_after_each: bool = True) -> Dict[str, Any]:
        """
        Train through all experiences in a scenario sequentially.

        Args:
            scenario_id: ID of the scenario to train
            train_epochs: Number of epochs to train for each experience
            evaluate_after_each: Whether to evaluate after each experience

        Returns:
            Training summary
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot train: Avalanche framework not available")
            return {}

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return {}

            if not scenario.experiences:
                self.logger.warning(f"Scenario {scenario_id} has no experiences")
                return {}

            if scenario.model is None:
                # Initialize model
                if not self.initialize_model(scenario_id):
                    return {}

            # Reset current experience
            scenario.current_experience_idx = 0

            # Set up strategy
            if scenario.cl_strategy is None:
                if not self.setup_cl_strategy(scenario_id):
                    return {}

            self.logger.info(f"Starting training for full scenario {scenario_id}")

            # Add to active scenarios
            self.active_scenarios.add(scenario_id)

            summary = {
                "scenario_id": scenario_id,
                "experiences": [],
                "forgetting": {},
                "final_accuracy": {}
            }

            try:
                # Train each experience
                while scenario.current_experience_idx < len(scenario.experiences):
                    exp = scenario.current_experience

                    # Train on current experience
                    train_metrics = self.train_current_experience(scenario_id, train_epochs)

                    # Evaluate if requested
                    eval_metrics = {}
                    if evaluate_after_each:
                        eval_results = self.evaluate_on_experiences(scenario_id)

                        # Extract metrics for current experience
                        if exp.experience_id in eval_results:
                            eval_metrics = eval_results[exp.experience_id]

                            # Update summary with accuracy
                            if "Top1_Acc_Stream/eval_phase/test_stream" in eval_metrics:
                                summary["final_accuracy"][exp.experience_id] = eval_metrics["Top1_Acc_Stream/eval_phase/test_stream"]

                    # Record experience results
                    summary["experiences"].append({
                        "experience_id": exp.experience_id,
                        "name": exp.name,
                        "train_metrics": train_metrics,
                        "eval_metrics": eval_metrics
                    })

                    # Move to next experience
                    if not scenario.advance_to_next_experience():
                        break

                # Evaluate forgetting
                if evaluate_after_each and len(scenario.experiences) > 1:
                    # Evaluate on all experiences to measure forgetting
                    all_eval_results = self.evaluate_on_experiences(scenario_id)

                    # Calculate forgetting for each previously learned experience
                    for i in range(len(scenario.experiences) - 1):
                        exp = scenario.experiences[i]

                        if exp.experience_id in all_eval_results:
                            current_acc = all_eval_results[exp.experience_id].get("Top1_Acc_Stream/eval_phase/test_stream", 0.0)
                            max_acc = max(
                                metrics.get("Top1_Acc_Stream/eval_phase/test_stream", 0.0)
                                for metrics in exp.eval_history
                            )
                            forgetting = max(0.0, max_acc - current_acc)

                            summary["forgetting"][exp.experience_id] = forgetting

                self.logger.info(f"Completed training for full scenario {scenario_id}")

                # Remove from active scenarios
                self.active_scenarios.remove(scenario_id)

                return summary

            except Exception as e:
                self.logger.error(f"Error training full scenario {scenario_id}: {str(e)}")

                # Remove from active scenarios
                if scenario_id in self.active_scenarios:
                    self.active_scenarios.remove(scenario_id)

                return {"error": str(e)}

    def get_scenario(self, scenario_id: str) -> Optional[CLScenario]:
        """
        Get a scenario by ID.

        Args:
            scenario_id: Scenario ID

        Returns:
            Scenario object or None if not found
        """
        with self.lock:
            return self.scenarios.get(scenario_id)

    def get_experience(self, scenario_id: str, experience_id: str) -> Optional[CLExperience]:
        """
        Get an experience by ID within a scenario.

        Args:
            scenario_id: Scenario ID
            experience_id: Experience ID

        Returns:
            Experience object or None if not found
        """
        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                return None

            for exp in scenario.experiences:
                if exp.experience_id == experience_id:
                    return exp

            return None

    def get_forgetting_metrics(self, scenario_id: str) -> Dict[str, float]:
        """
        Calculate forgetting metrics for a scenario.

        Args:
            scenario_id: Scenario ID

        Returns:
            Dictionary of forgetting metrics
        """
        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                return {}

            forgetting = {}

            # Only calculate forgetting if we have multiple experiences with evaluation history
            if len(scenario.experiences) <= 1:
                return {}

            for i, exp in enumerate(scenario.experiences[:-1]):  # Exclude last experience
                if not exp.eval_history:
                    continue

                # Get maximum accuracy achieved for this experience
                max_acc = max(
                    metrics.get("Top1_Acc_Stream/eval_phase/test_stream", 0.0)
                    for metrics in exp.eval_history
                )

                # Get most recent accuracy
                current_acc = exp.metrics.get("Top1_Acc_Stream/eval_phase/test_stream", 0.0)

                # Calculate forgetting
                forgetting[exp.experience_id] = max(0.0, max_acc - current_acc)

            return forgetting

    def get_learning_metrics(self) -> Dict[str, Any]:
        """
        Get performance metrics for continuous learning.

        Returns:
            Dictionary of learning metrics
        """
        with self.lock:
            metrics_summary = {}

            # Process each metric
            for key, values in self.metrics.items():
                if values:
                    metrics_summary[key] = {
                        "current": values[-1],
                        "average": sum(values) / len(values),
                        "min": min(values),
                        "max": max(values)
                    }

            return metrics_summary

    def save_model(self, scenario_id: str, path: str) -> bool:
        """
        Save a trained model to disk.

        Args:
            scenario_id: Scenario ID
            path: Path to save the model

        Returns:
            True if successful, False otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot save model: PyTorch not available")
            return False

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario or scenario.model is None:
                self.logger.warning(f"No model available for scenario {scenario_id}")
                return False

            try:
                # Save model state
                torch.save(scenario.model.state_dict(), path)
                self.logger.info(f"Saved model for scenario {scenario_id} to {path}")
                return True
            except Exception as e:
                self.logger.error(f"Error saving model for scenario {scenario_id}: {str(e)}")
                return False

    def load_model(self, scenario_id: str, path: str) -> bool:
        """
        Load a saved model from disk.

        Args:
            scenario_id: Scenario ID
            path: Path to load the model from

        Returns:
            True if successful, False otherwise
        """
        if not HAS_AVALANCHE:
            self.logger.error("Cannot load model: PyTorch not available")
            return False

        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return False

            if scenario.model is None:
                # Initialize model first
                if not self.initialize_model(scenario_id):
                    return False

            try:
                # Load model state
                scenario.model.load_state_dict(torch.load(path))
                self.logger.info(f"Loaded model for scenario {scenario_id} from {path}")
                return True
            except Exception as e:
                self.logger.error(f"Error loading model for scenario {scenario_id}: {str(e)}")
                return False

    def store_experience_in_memory(self, scenario_id: str, experience_id: str) -> bool:
        """
        Store an experience in episodic memory for future replay.

        Args:
            scenario_id: Scenario ID
            experience_id: Experience ID

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            # Check if we have episodic memory
            if not self.episodic_memory:
                self.logger.warning("No episodic memory available")
                return False

            # Get experience
            experience = self.get_experience(scenario_id, experience_id)
            if not experience:
                self.logger.warning(f"Experience {experience_id} not found in scenario {scenario_id}")
                return False

            try:
                # Store in episodic memory
                memory_id = self.episodic_memory.store(
                    content={
                        "type": "learning_experience",
                        "scenario_id": scenario_id,
                        "experience_id": experience_id,
                        "name": experience.name,
                        "task_id": experience.task_id,
                        "data_summary": {
                            "size": self._get_data_size(experience.data),
                            "timestamp": experience.timestamp
                        },
                        "metrics": experience.metrics
                    },
                    metadata={
                        "scenario_id": scenario_id,
                        "experience_id": experience_id,
                        "type": "learning_experience"
                    },
                    relevance=0.7  # Medium-high relevance
                )

                self.logger.info(f"Stored experience {experience_id} in episodic memory as {memory_id}")
                return True
            except Exception as e:
                self.logger.error(f"Error storing experience in memory: {str(e)}")
                return False

    def create_replay_experience(self, scenario_id: str,
                               num_experiences: int = 3,
                               experience_filter: Optional[Callable[[CLExperience], bool]] = None) -> Optional[str]:
        """
        Create a replay experience from previous experiences in episodic memory.

        Args:
            scenario_id: Scenario ID
            num_experiences: Number of past experiences to combine
            experience_filter: Function to filter experiences

        Returns:
            Experience ID if successful, None otherwise
        """
        with self.lock:
            # Check if we have episodic memory
            if not self.episodic_memory:
                self.logger.warning("No episodic memory available")
                return None

            # Get scenario
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                self.logger.warning(f"Scenario {scenario_id} not found")
                return None

            try:
                # Query episodic memory for learning experiences
                if hasattr(self.episodic_memory, "query"):
                    memories = self.episodic_memory.query(
                        metadata_filter={
                            "type": "learning_experience",
                            "scenario_id": scenario_id
                        },
                        max_results=50  # Get a good sample
                    )
                else:
                    self.logger.warning("Episodic memory does not support querying")
                    return None

                if not memories:
                    self.logger.warning(f"No experiences found in memory for scenario {scenario_id}")
                    return None

                # Retrieve full memory items
                memory_items = []
                for memory_id in memories:
                    if isinstance(memory_id, tuple):
                        memory_id = memory_id[0]  # Extract ID if tuple
                    memory = self.episodic_memory.retrieve(memory_id)
                    if memory:
                        memory_items.append(memory)

                if not memory_items:
                    self.logger.warning("Could not retrieve memory items")
                    return None

                # Extract experiences from memory
                experiences = []
                for memory in memory_items:
                    content = memory.content
                    if content and isinstance(content, dict):
                        exp_id = content.get("experience_id")
                        if exp_id:
                            exp = self.get_experience(scenario_id, exp_id)
                            if exp:
                                experiences.append(exp)

                # Apply filter if provided
                if experience_filter:
                    experiences = [exp for exp in experiences if experience_filter(exp)]

                if not experiences:
                    self.logger.warning("No valid experiences found after filtering")
                    return None

                # Sort by task_id (to get a range of tasks)
                experiences.sort(key=lambda exp: exp.task_id)

                # Select experiences (evenly distributed if possible)
                selected = []
                if len(experiences) <= num_experiences:
                    selected = experiences
                else:
                    # Try to get a diverse set
                    step = max(1, len(experiences) // num_experiences)
                    indices = [i * step for i in range(num_experiences)]
                    indices = [min(i, len(experiences) - 1) for i in indices]  # Ensure valid indices
                    selected = [experiences[i] for i in indices]

                # Combine data from selected experiences
                combined_data = self._combine_experience_data(selected)

                # Create new replay experience
                return self.add_experience(
                    scenario_id=scenario_id,
                    name=f"Replay({', '.join(exp.name for exp in selected)})",
                    data=combined_data,
                    metadata={
                        "type": "replay",
                        "source_experiences": [exp.experience_id for exp in selected]
                    }
                )

            except Exception as e:
                self.logger.error(f"Error creating replay experience: {str(e)}")
                return None

    def process_learning_task(self, task: LearningTask) -> LearningResult:
        """
        Process a learning task from the learning core.

        Args:
            task: Learning task to process

        Returns:
            Learning result
        """
        if task.paradigm != LearningParadigm.CONTINUOUS:
            return LearningResult(
                success=False,
                logs=["Task paradigm is not CONTINUOUS"],
                metrics={}
            )

        try:
            # Extract task parameters
            task_type_str = task.parameters.get("task_type", "CLASSIFICATION")
            task_type = getattr(TaskType, task_type_str, TaskType.CLASSIFICATION)

            strategy_str = task.parameters.get("strategy", "EWC")
            strategy = getattr(CLStrategy, strategy_str, CLStrategy.EWC)

            scenario_name = task.parameters.get("scenario_name", f"Scenario-{uuid.uuid4().hex[:6]}")

            # Create scenario
            scenario_id = self.create_scenario(
                name=scenario_name,
                task_type=task_type,
                strategy=strategy,
                strategy_params=task.parameters.get("strategy_params", {})
            )

            if not scenario_id:
                return LearningResult(
                    success=False,
                    logs=["Failed to create scenario"],
                    metrics={}
                )

            # Add experiences
            experiences = task.data.get("experiences", [])
            for exp_data in experiences:
                exp_name = exp_data.get("name", f"Experience-{uuid.uuid4().hex[:6]}")
                exp_data_obj = exp_data.get("data")

                if exp_data_obj:
                    self.add_experience(
                        scenario_id=scenario_id,
                        name=exp_name,
                        data=exp_data_obj,
                        metadata=exp_data.get("metadata", {})
                    )

            # Train if requested
            results = {}
            if task.parameters.get("train", False):
                # Initialize model
                if not self.initialize_model(scenario_id):
                    return LearningResult(
                        success=False,
                        logs=["Failed to initialize model"],
                        metrics={}
                    )

                # Train full scenario
                train_epochs = task.parameters.get("train_epochs", 1)
                eval_after = task.parameters.get("evaluate_after_each", True)

                results = self.train_full_scenario(
                    scenario_id=scenario_id,
                    train_epochs=train_epochs,
                    evaluate_after_each=eval_after
                )

            return LearningResult(
                success=True,
                logs=[f"Created scenario {scenario_id} with {len(experiences)} experiences"],
                metrics=results,
                learned_model=self.get_scenario(scenario_id)
            )

        except Exception as e:
            return LearningResult(
                success=False,
                logs=[f"Error processing learning task: {str(e)}"],
                metrics={}
            )

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_strategy_factories(self) -> None:
        """Register factories for creating continual learning strategies"""
        if not HAS_AVALANCHE:
            return

        # Naive (baseline, fine-tuning)
        self.strategy_factories[CLStrategy.NAIVE] = lambda **kwargs: Naive(**kwargs)

        # EWC (Elastic Weight Consolidation)
        self.strategy_factories[CLStrategy.EWC] = lambda **kwargs: EWC(
            ewc_lambda=kwargs.pop("ewc_lambda", 100.0),
            **kwargs
        )

        # LwF (Learning without Forgetting)
        self.strategy_factories[CLStrategy.LWF] = lambda **kwargs: LwF(
            alpha=kwargs.pop("alpha", 1.0),
            temperature=kwargs.pop("temperature", 2.0),
            **kwargs
        )

        # SI (Synaptic Intelligence)
        self.strategy_factories[CLStrategy.SI] = lambda **kwargs: SynapticIntelligence(
            si_lambda=kwargs.pop("si_lambda", 1.0),
            **kwargs
        )

        # GEM (Gradient Episodic Memory)
        self.strategy_factories[CLStrategy.GEM] = lambda **kwargs: GEM(
            patterns_per_experience=kwargs.pop("patterns_per_experience", 100),
            memory_strength=kwargs.pop("memory_strength", 0.5),
            **kwargs
        )

        # AGEM (Averaged GEM)
        self.strategy_factories[CLStrategy.AGEM] = lambda **kwargs: AGEM(
            patterns_per_experience=kwargs.pop("patterns_per_experience", 100),
            sample_size=kwargs.pop("sample_size", 64),
            **kwargs
        )

        # CWR* (CWR Star)
        self.strategy_factories[CLStrategy.CWR_STAR] = lambda **kwargs: CWRStar(**kwargs)

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module
            dependencies = []
            if self.learning_core:
                dependencies.append("learning_core")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="continuous_learner",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _learning_loop(self) -> None:
        """Background thread for continuous learning"""
        self.logger.info("Starting continuous learning background thread")

        while not self.stop_learning.is_set():
            try:
                # Process active scenarios
                with self.lock:
                    active_scenarios = list(self.active_scenarios)

                # Process each active scenario
                for scenario_id in active_scenarios:
                    try:
                        self._process_scenario(scenario_id)
                    except Exception as e:
                        self.logger.error(f"Error processing scenario {scenario_id}: {str(e)}")

                # Sleep to avoid hogging CPU
                time.sleep(1.0)

            except Exception as e:
                self.logger.error(f"Error in learning loop: {str(e)}")

        self.logger.info("Continuous learning background thread stopped")

    def _process_scenario(self, scenario_id: str) -> None:
        """
        Process a single active scenario.

        Args:
            scenario_id: Scenario ID
        """
        with self.lock:
            scenario = self.scenarios.get(scenario_id)
            if not scenario:
                if scenario_id in self.active_scenarios:
                    self.active_scenarios.remove(scenario_id)
                return

            # Skip if the scenario is not properly initialized
            if scenario.model is None or scenario.cl_strategy is None:
                return

            # Check if there are more experiences to process
            if not scenario.has_more_experiences():
                # Remove from active scenarios
                self.active_scenarios.remove(scenario_id)
                self.logger.info(f"Scenario {scenario_id} completed all experiences")
                return

    def _get_data_size(self, data: Any) -> Dict[str, int]:
        """
        Get the size of data in an experience.

        Args:
            data: Experience data

        Returns:
            Dictionary with data size information
        """
        size_info = {"total": 0}

        try:
            if isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, (torch.Tensor, np.ndarray)):
                        size_info[key] = value.shape[0]
                        size_info["total"] += value.shape[0]
            elif isinstance(data, tuple) and len(data) >= 2:
                X, y = data[:2]
                if isinstance(X, (torch.Tensor, np.ndarray)):
                    size_info["X"] = X.shape[0]
                    size_info["total"] += X.shape[0]
                if isinstance(y, (torch.Tensor, np.ndarray)):
                    size_info["y"] = y.shape[0]
        except Exception:
            pass

        return size_info

    def _combine_experience_data(self, experiences: List[CLExperience]) -> Any:
        """
        Combine data from multiple experiences.

        Args:
            experiences: List of experiences to combine

        Returns:
            Combined data
        """
        if not experiences:
            return None

        try:
            # Determine data format (dict or tuple)
            first_data = experiences[0].data
            is_dict_format = isinstance(first_data, dict)

            if is_dict_format:
                # Dictionary format
                combined = {}

                # Identify keys to combine
                keys_to_combine = set()
                for exp in experiences:
                    if isinstance(exp.data, dict):
                        keys_to_combine.update(exp.data.keys())

                # Combine data for each key
                for key in keys_to_combine:
                    data_list = []
                    for exp in experiences:
                        if isinstance(exp.data, dict) and key in exp.data:
                            data = exp.data[key]
                            if data is not None:
                                data_list.append(data)

                    if data_list:
                        # Combine data (assuming tensors or arrays)
                        if isinstance(data_list[0], torch.Tensor):
                            combined[key] = torch.cat(data_list, dim=0)
                        elif isinstance(data_list[0], np.ndarray):
                            combined[key] = np.concatenate(data_list, axis=0)
                        else:
                            # Try to convert to numpy first
                            try:
                                np_list = [np.array(d) for d in data_list]
                                combined[key] = np.concatenate(np_list, axis=0)
                            except:
                                # Just store the list
                                combined[key] = data_list

                return combined

            else:
                # Tuple format (X_train, y_train, X_val, y_val)
                X_train_list = []
                y_train_list = []
                X_val_list = []
                y_val_list = []

                for exp in experiences:
                    data = exp.data
                    if isinstance(data, tuple):
                        if len(data) >= 1 and data[0] is not None:
                            X_train_list.append(data[0])
                        if len(data) >= 2 and data[1] is not None:
                            y_train_list.append(data[1])
                        if len(data) >= 3 and data[2] is not None:
                            X_val_list.append(data[2])
                        if len(data) >= 4 and data[3] is not None:
                            y_val_list.append(data[3])

                # Combine data
                combined_data = []

                # X_train
                if X_train_list:
                    if isinstance(X_train_list[0], torch.Tensor):
                        combined_data.append(torch.cat(X_train_list, dim=0))
                    elif isinstance(X_train_list[0], np.ndarray):
                        combined_data.append(np.concatenate(X_train_list, axis=0))
                    else:
                        combined_data.append(X_train_list)
                else:
                    combined_data.append(None)

                # y_train
                if y_train_list:
                    if isinstance(y_train_list[0], torch.Tensor):
                        combined_data.append(torch.cat(y_train_list, dim=0))
                    elif isinstance(y_train_list[0], np.ndarray):
                        combined_data.append(np.concatenate(y_train_list, axis=0))
                    else:
                        combined_data.append(y_train_list)
                else:
                    combined_data.append(None)

                # X_val
                if X_val_list:
                    if isinstance(X_val_list[0], torch.Tensor):
                        combined_data.append(torch.cat(X_val_list, dim=0))
                    elif isinstance(X_val_list[0], np.ndarray):
                        combined_data.append(np.concatenate(X_val_list, axis=0))
                    else:
                        combined_data.append(X_val_list)
                else:
                    combined_data.append(None)

                # y_val
                if y_val_list:
                    if isinstance(y_val_list[0], torch.Tensor):
                        combined_data.append(torch.cat(y_val_list, dim=0))
                    elif isinstance(y_val_list[0], np.ndarray):
                        combined_data.append(np.concatenate(y_val_list, axis=0))
                    else:
                        combined_data.append(y_val_list)
                else:
                    combined_data.append(None)

                return tuple(combined_data)

        except Exception as e:
            self.logger.error(f"Error combining experience data: {str(e)}")
            return None

ModuleNotFoundError: No module named 'grandnexus'

# Meta Learner

In [ ]:
"""
meta_learner.py - Meta-Learning Module for GrandNexus

This module implements meta-learning capabilities, allowing the system to learn
how to learn efficiently across tasks. It focuses on few-shot learning techniques
that enable rapid adaptation to new tasks with minimal examples.

Key features:
- Model-Agnostic Meta-Learning (MAML) implementation
- Prototypical Networks for few-shot classification
- Task generation and sampling for meta-training
- Integration with standard learning pipelines
- Adaptation metrics and performance tracking

The design emphasizes:
- Flexibility across domains and task types
- Efficient parameter adaptation
- Integration with both symbolic and neural components
- Emergent learning strategies through meta-optimization
"""

import logging
import time
import threading
import uuid
import math
import random
from typing import Dict, List, Any, Optional, Tuple, Callable, Union, Set
from dataclasses import dataclass, field
from enum import Enum, auto
from collections import defaultdict, deque
import copy

import numpy as np

# Import learn2learn if available, otherwise provide stub functionality
try:
    import learn2learn as l2l
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_LEARN2LEARN = True
except ImportError:
    HAS_LEARN2LEARN = False

# Assume these exist in the GrandNexus architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.learning.learning_core import LearningCore

class MetaLearningParadigm(Enum):
    """Types of meta-learning approaches supported by the module"""
    MAML = auto()               # Model-Agnostic Meta-Learning
    PROTO_NET = auto()          # Prototypical Networks
    RELATION_NET = auto()       # Relation Networks
    MATCHING_NET = auto()       # Matching Networks
    REPTILE = auto()            # Reptile algorithm
    ANIL = auto()               # Almost No Inner Loop
    META_SGD = auto()           # Meta-SGD
    CUSTOM = auto()             # Custom meta-learning approach

@dataclass
class TaskDescription:
    """Description of a learning task for meta-learning"""
    task_id: str
    name: str
    task_type: str              # "classification", "regression", "reinforcement", etc.
    n_way: int = 0              # Number of classes (for classification)
    k_shot: int = 0             # Number of examples per class
    query_size: int = 0         # Number of query examples
    data_source: str = ""       # Source of task data
    domain: str = ""            # Domain of the task
    difficulty: float = 0.5     # Estimated difficulty (0-1)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class MetaLearningResult:
    """Results of a meta-learning operation"""
    success: bool
    paradigm: MetaLearningParadigm
    model_state: Optional[Any] = None
    metrics: Dict[str, float] = field(default_factory=dict)
    adaptation_history: List[Dict[str, Any]] = field(default_factory=list)
    error_message: Optional[str] = None
    timing: Dict[str, float] = field(default_factory=dict)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        """Initialize additional fields after creation"""
        self.id = str(uuid.uuid4())
        self.timestamp = time.time()

@dataclass
class AdaptationRequest:
    """Request to adapt a meta-learned model to a new task"""
    task_id: str
    support_data: Any           # Training examples (support set)
    query_data: Optional[Any] = None  # Test examples (query set)
    adaptation_steps: int = 5   # Number of adaptation steps to perform
    learning_rate: float = 0.01 # Learning rate for adaptation
    metrics: List[str] = field(default_factory=list)  # Metrics to track
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class MetaLearnerConfig:
    """Configuration for MetaLearner"""
    paradigm: MetaLearningParadigm = MetaLearningParadigm.MAML
    model_type: str = "neural"  # "neural", "bayesian", "symbolic", etc.
    meta_batch_size: int = 4    # Number of tasks per meta-batch
    meta_lr: float = 0.001      # Meta learning rate
    inner_lr: float = 0.01      # Inner learning rate (for MAML)
    inner_steps: int = 5        # Number of inner steps (for MAML)
    n_way: int = 5              # Default n-way classification
    k_shot: int = 1             # Default k-shot learning
    task_batch_size: int = 16   # Batch size for task training
    evaluation_tasks: int = 10  # Number of tasks for evaluation
    load_model_path: str = ""   # Path to load pre-trained model
    save_model_path: str = ""   # Path to save model
    device: str = "cpu"         # Device to use ("cpu", "cuda", etc.)
    requires_grad: bool = True  # Whether to require gradients
    first_order: bool = False   # Whether to use first-order approximation (MAML)
    train_tasks: List[TaskDescription] = field(default_factory=list)
    val_tasks: List[TaskDescription] = field(default_factory=list)
    test_tasks: List[TaskDescription] = field(default_factory=list)

class MetaLearner:
    """
    MetaLearner module for GrandNexus, implementing few-shot learning capabilities
    to enable rapid adaptation to new tasks with minimal examples.
    """

    def __init__(self, config: Optional[MetaLearnerConfig] = None,
                 nexus_core: Optional[NexusCore] = None,
                 learning_core: Optional[LearningCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None):
        """
        Initialize the meta-learner with configuration and module references.

        Args:
            config: Configuration for the meta-learner
            nexus_core: Reference to the NexusCore for system-wide coordination
            learning_core: Reference to the main LearningCore module
            working_memory: Reference to working memory for temporary storage
            semantic_memory: Reference to semantic memory for knowledge
            episodic_memory: Reference to episodic memory for experiences
        """
        # Set up logging
        self.logger = logging.getLogger("GrandNexus.Learning.MetaLearner")

        # Store module references
        self.nexus_core = nexus_core
        self.learning_core = learning_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory

        # Initialize configuration
        self.config = config or MetaLearnerConfig()

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.running = False
        self.lock = threading.RLock()

        # Meta-learning models and state
        self.meta_model = None
        self.meta_optimizer = None
        self.model_registry = {}  # Storage for different meta-models

        # Task management
        self.task_registry = {}  # All registered tasks
        self.task_families = defaultdict(list)  # Group tasks by family/domain

        # Performance tracking
        self.metrics = defaultdict(lambda: deque(maxlen=100))
        self.adaptation_history = defaultdict(list)

        # Check for required dependencies
        self.has_dependencies = self._check_dependencies()

        self.logger.info(f"MetaLearner initialized with ID={self.instance_id}, "
                        f"Dependencies={self.has_dependencies}")

    def initialize(self) -> bool:
        """Initialize the meta-learner and prepare it for use"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Check dependencies
                if not self.has_dependencies:
                    self.logger.warning("Meta-learning dependencies not available. "
                                      "Some functionality will be limited.")

                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Set up the meta-learning model
                if HAS_LEARN2LEARN and self.config.paradigm != MetaLearningParadigm.CUSTOM:
                    self._setup_meta_model()

                # Load tasks if provided in config
                self._register_config_tasks()

                self.initialized = True
                self.logger.info("MetaLearner initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"MetaLearner initialization failed: {str(e)}")
                return False

    def register_task(self, task: TaskDescription) -> str:
        """
        Register a new task for meta-learning.

        Args:
            task: The task description to register

        Returns:
            The task ID
        """
        with self.lock:
            if task.task_id in self.task_registry:
                self.logger.warning(f"Task {task.task_id} already exists, updating")

            self.task_registry[task.task_id] = task

            # Add to task family
            if task.domain:
                self.task_families[task.domain].append(task.task_id)

            self.logger.info(f"Registered task: {task.name} ({task.task_id})")
            return task.task_id

    def meta_train(self, epochs: int = 10, tasks: Optional[List[str]] = None,
                  save_model: bool = False) -> MetaLearningResult:
        """
        Perform meta-training on a set of tasks.

        Args:
            epochs: Number of meta-training epochs
            tasks: Optional list of task IDs to train on, or None to use config tasks
            save_model: Whether to save the model after training

        Returns:
            MetaLearningResult with training results
        """
        if not self.initialized:
            self.initialize()

        if not self.has_dependencies:
            return MetaLearningResult(
                success=False,
                paradigm=self.config.paradigm,
                error_message="Meta-learning dependencies not available"
            )

        # Start timing
        start_time = time.time()
        timing = {}

        try:
            # Determine tasks to use
            task_ids = tasks if tasks is not None else [t.task_id for t in self.config.train_tasks]
            if not task_ids:
                raise ValueError("No tasks specified for meta-training")

            # Setup meta-learning (this varies by paradigm)
            setup_start = time.time()
            if self.config.paradigm == MetaLearningParadigm.MAML:
                result = self._train_maml(task_ids, epochs)
            elif self.config.paradigm == MetaLearningParadigm.PROTO_NET:
                result = self._train_proto_net(task_ids, epochs)
            elif self.config.paradigm == MetaLearningParadigm.REPTILE:
                result = self._train_reptile(task_ids, epochs)
            elif self.config.paradigm == MetaLearningParadigm.ANIL:
                result = self._train_anil(task_ids, epochs)
            elif self.config.paradigm == MetaLearningParadigm.CUSTOM:
                result = self._train_custom(task_ids, epochs)
            else:
                raise ValueError(f"Unsupported meta-learning paradigm: {self.config.paradigm}")

            timing["setup"] = time.time() - setup_start

            # Save model if requested
            if save_model and result.success and self.config.save_model_path:
                save_start = time.time()
                self._save_meta_model(self.config.save_model_path)
                timing["save_model"] = time.time() - save_start

            # Add timing information
            timing["total"] = time.time() - start_time
            result.timing = timing

            return result

        except Exception as e:
            self.logger.error(f"Error during meta-training: {str(e)}")
            return MetaLearningResult(
                success=False,
                paradigm=self.config.paradigm,
                error_message=f"Training error: {str(e)}",
                timing=timing
            )

    def adapt(self, request: AdaptationRequest) -> MetaLearningResult:
        """
        Adapt the meta-learned model to a new task.

        Args:
            request: The adaptation request with task data

        Returns:
            MetaLearningResult with adaptation results
        """
        if not self.initialized:
            self.initialize()

        if not self.has_dependencies or self.meta_model is None:
            return MetaLearningResult(
                success=False,
                paradigm=self.config.paradigm,
                error_message="Meta-learning model not available"
            )

        # Start timing
        start_time = time.time()
        timing = {}

        try:
            # Get task description
            task = self.task_registry.get(request.task_id)
            if not task:
                raise ValueError(f"Unknown task ID: {request.task_id}")

            # Adaptation strategy depends on the paradigm
            adapt_start = time.time()
            if self.config.paradigm == MetaLearningParadigm.MAML:
                result = self._adapt_maml(request, task)
            elif self.config.paradigm == MetaLearningParadigm.PROTO_NET:
                result = self._adapt_proto_net(request, task)
            elif self.config.paradigm == MetaLearningParadigm.REPTILE:
                result = self._adapt_reptile(request, task)
            elif self.config.paradigm == MetaLearningParadigm.ANIL:
                result = self._adapt_anil(request, task)
            elif self.config.paradigm == MetaLearningParadigm.CUSTOM:
                result = self._adapt_custom(request, task)
            else:
                raise ValueError(f"Unsupported meta-learning paradigm: {self.config.paradigm}")

            timing["adaptation"] = time.time() - adapt_start

            # Evaluate on query set if provided
            if request.query_data is not None:
                eval_start = time.time()
                evaluation = self._evaluate_adaptation(request, result)
                result.metrics.update(evaluation)
                timing["evaluation"] = time.time() - eval_start

            # Add timing information
            timing["total"] = time.time() - start_time
            result.timing = timing

            # Track adaptation history
            self.adaptation_history[request.task_id].append({
                "timestamp": time.time(),
                "metrics": result.metrics,
                "steps": request.adaptation_steps
            })

            return result

        except Exception as e:
            self.logger.error(f"Error during adaptation: {str(e)}")
            return MetaLearningResult(
                success=False,
                paradigm=self.config.paradigm,
                error_message=f"Adaptation error: {str(e)}",
                timing=timing
            )

    def evaluate(self, task_ids: Optional[List[str]] = None,
               n_tasks: int = 5) -> Dict[str, Any]:
        """
        Evaluate the meta-learner on a set of test tasks.

        Args:
            task_ids: Optional list of task IDs to evaluate on, or None to use config
            n_tasks: Number of tasks to evaluate if randomly sampling

        Returns:
            Dictionary with evaluation results
        """
        if not self.initialized:
            self.initialize()

        if not self.has_dependencies or self.meta_model is None:
            return {
                "success": False,
                "error": "Meta-learning model not available"
            }

        # Determine tasks to use
        if task_ids is None:
            if self.config.test_tasks:
                task_ids = [t.task_id for t in self.config.test_tasks]
            else:
                # Randomly sample tasks
                all_tasks = list(self.task_registry.keys())
                if not all_tasks:
                    return {
                        "success": False,
                        "error": "No tasks available for evaluation"
                    }
                task_ids = random.sample(all_tasks, min(n_tasks, len(all_tasks)))

        # Evaluate on each task
        results = []
        overall_metrics = defaultdict(list)

        for task_id in task_ids:
            task = self.task_registry.get(task_id)
            if not task:
                self.logger.warning(f"Skipping unknown task ID: {task_id}")
                continue

            # Create adaptation request
            try:
                # Normally we would get the actual data here
                # For simplicity, we're creating a placeholder AdaptationRequest
                request = AdaptationRequest(
                    task_id=task_id,
                    support_data=None,  # Would get real data in a full implementation
                    query_data=None,    # Would get real data in a full implementation
                    adaptation_steps=self.config.inner_steps
                )

                # Adapt to the task
                result = self.adapt(request)
                results.append({
                    "task_id": task_id,
                    "success": result.success,
                    "metrics": result.metrics
                })

                # Collect metrics across tasks
                if result.success:
                    for key, value in result.metrics.items():
                        overall_metrics[key].append(value)

            except Exception as e:
                self.logger.error(f"Error evaluating task {task_id}: {str(e)}")
                results.append({
                    "task_id": task_id,
                    "success": False,
                    "error": str(e)
                })

        # Calculate aggregated metrics
        aggregated_metrics = {}
        for key, values in overall_metrics.items():
            if values:
                aggregated_metrics[key] = {
                    "mean": sum(values) / len(values),
                    "std": np.std(values) if len(values) > 1 else 0.0,
                    "min": min(values),
                    "max": max(values)
                }

        return {
            "success": True,
            "task_results": results,
            "aggregated_metrics": aggregated_metrics,
            "tasks_evaluated": len(results),
            "timestamp": time.time()
        }

    def load_model(self, model_path: str) -> bool:
        """
        Load a pre-trained meta-learning model.

        Args:
            model_path: Path to the model file

        Returns:
            True if the model was loaded successfully, False otherwise
        """
        if not self.initialized:
            self.initialize()

        if not self.has_dependencies:
            self.logger.error("Cannot load model: dependencies not available")
            return False

        try:
            # This would load the model from disk
            # For now, we'll just log the action
            self.logger.info(f"Would load model from {model_path}")
            return True
        except Exception as e:
            self.logger.error(f"Error loading model: {str(e)}")
            return False

    def get_metrics(self) -> Dict[str, Any]:
        """Get current meta-learning performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def get_task_metrics(self, task_id: str) -> List[Dict[str, Any]]:
        """
        Get adaptation history and metrics for a specific task.

        Args:
            task_id: The task ID to get metrics for

        Returns:
            List of adaptation history entries
        """
        with self.lock:
            return self.adaptation_history.get(task_id, [])

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _check_dependencies(self) -> bool:
        """Check if required dependencies are available"""
        return HAS_LEARN2LEARN

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.learning_core:
                dependencies.append("learning_core")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="meta_learner",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _setup_meta_model(self) -> None:
        """Set up the meta-learning model based on configuration"""
        if not HAS_LEARN2LEARN:
            self.logger.warning("learn2learn not available, skipping model setup")
            return

        # This is a simplified setup for illustration
        # In a real implementation, model architecture would depend on the task

        # Create base model (simple MLP for demonstration)
        if self.config.model_type == "neural":
            # Simple model with embedding layer, hidden layer, and output layer
            model = torch.nn.Sequential(
                torch.nn.Linear(20, 40),
                torch.nn.ReLU(),
                torch.nn.Linear(40, 40),
                torch.nn.ReLU(),
                torch.nn.Linear(40, 5)  # Assuming 5-way classification
            )

            # Wrap the model based on meta-learning paradigm
            if self.config.paradigm == MetaLearningParadigm.MAML:
                self.meta_model = l2l.algorithms.MAML(
                    model,
                    lr=self.config.inner_lr,
                    first_order=self.config.first_order
                )
            elif self.config.paradigm == MetaLearningParadigm.REPTILE:
                self.meta_model = l2l.algorithms.Reptile(
                    model,
                    lr=self.config.inner_lr
                )
            elif self.config.paradigm == MetaLearningParadigm.ANIL:
                # ANIL is like MAML but only adapts the last layer
                features = torch.nn.Sequential(
                    torch.nn.Linear(20, 40),
                    torch.nn.ReLU(),
                    torch.nn.Linear(40, 40),
                    torch.nn.ReLU(),
                )
                head = torch.nn.Linear(40, 5)
                model = torch.nn.Sequential(features, head)

                self.meta_model = l2l.algorithms.MAML(
                    model,
                    lr=self.config.inner_lr,
                    first_order=self.config.first_order,
                    allow_nograd=True  # Allow freezing parts of the network
                )

                # Freeze feature extractor for ANIL
                for param in features.parameters():
                    param.requires_grad = False

            # Create optimizer
            self.meta_optimizer = torch.optim.Adam(
                self.meta_model.parameters(),
                lr=self.config.meta_lr
            )

            self.logger.info(f"Meta-learning model set up with paradigm {self.config.paradigm}")

    def _register_config_tasks(self) -> None:
        """Register tasks provided in the configuration"""
        for task in self.config.train_tasks + self.config.val_tasks + self.config.test_tasks:
            self.register_task(task)

    def _save_meta_model(self, path: str) -> None:
        """Save the meta-model to disk"""
        if not HAS_LEARN2LEARN or self.meta_model is None:
            self.logger.warning("No meta-model to save")
            return

        try:
            # Save model state
            # For simplicity, we're just logging this action
            self.logger.info(f"Would save model to {path}")
        except Exception as e:
            self.logger.error(f"Error saving model: {str(e)}")

    def _train_maml(self, task_ids: List[str], epochs: int) -> MetaLearningResult:
        """
        Train using Model-Agnostic Meta-Learning (MAML).

        Args:
            task_ids: List of task IDs to train on
            epochs: Number of meta-training epochs

        Returns:
            MetaLearningResult with training results
        """
        if not HAS_LEARN2LEARN:
            return MetaLearningResult(
                success=False,
                paradigm=MetaLearningParadigm.MAML,
                error_message="learn2learn not available"
            )

        # This is a simplified implementation
        # In a real system, this would use actual task data

        # Create loss function
        loss_fn = torch.nn.CrossEntropyLoss()

        # Track metrics
        meta_train_losses = []
        adaptation_history = []

        # Meta-training loop
        for epoch in range(epochs):
            meta_batch_loss = 0.0

            # Sample tasks for this meta-batch
            batch_task_ids = random.sample(task_ids, min(self.config.meta_batch_size, len(task_ids)))

            for task_id in batch_task_ids:
                # Clone the model for this task
                learner = self.meta_model.clone()

                # Simulate task data (support and query sets)
                # In a real implementation, this would load the actual task data
                task = self.task_registry[task_id]
                support_data = torch.randn(task.k_shot * task.n_way, 20)  # Random features
                support_labels = torch.cat([torch.full((task.k_shot,), i) for i in range(task.n_way)])
                query_data = torch.randn(task.query_size, 20)  # Random features
                query_labels = torch.randint(0, task.n_way, (task.query_size,))

                # Adapt the model on the support set (inner loop)
                for _ in range(self.config.inner_steps):
                    support_preds = learner(support_data)
                    support_loss = loss_fn(support_preds, support_labels.long())
                    learner.adapt(support_loss)

                # Evaluate on query set (meta-objective)
                query_preds = learner(query_data)
                query_loss = loss_fn(query_preds, query_labels.long())
                meta_batch_loss += query_loss

            # Meta-optimization step (outer loop)
            self.meta_optimizer.zero_grad()
            meta_batch_loss.backward()
            self.meta_optimizer.step()

            # Log progress
            avg_loss = meta_batch_loss.item() / len(batch_task_ids)
            meta_train_losses.append(avg_loss)

            # Record adaptation history
            adaptation_history.append({
                "epoch": epoch,
                "meta_loss": avg_loss
            })

            self.logger.info(f"Epoch {epoch+1}/{epochs}: Meta-loss = {avg_loss:.4f}")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.MAML,
            model_state=self.meta_model.state_dict(),
            metrics={"final_meta_loss": meta_train_losses[-1]} if meta_train_losses else {},
            adaptation_history=adaptation_history
        )

    def _train_proto_net(self, task_ids: List[str], epochs: int) -> MetaLearningResult:
        """Train using Prototypical Networks"""
        # Simplified implementation for illustration
        self.logger.info(f"Would train Prototypical Network on {len(task_ids)} tasks for {epochs} epochs")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.PROTO_NET,
            metrics={"final_meta_loss": 0.5},  # Placeholder
            adaptation_history=[]
        )

    def _train_reptile(self, task_ids: List[str], epochs: int) -> MetaLearningResult:
        """Train using Reptile algorithm"""
        # Simplified implementation for illustration
        self.logger.info(f"Would train Reptile on {len(task_ids)} tasks for {epochs} epochs")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.REPTILE,
            metrics={"final_meta_loss": 0.5},  # Placeholder
            adaptation_history=[]
        )

    def _train_anil(self, task_ids: List[str], epochs: int) -> MetaLearningResult:
        """Train using Almost No Inner Loop (ANIL)"""
        # ANIL is similar to MAML but only adapts the last layer
        # This is a simplified implementation similar to _train_maml

        return self._train_maml(task_ids, epochs)  # Reuse MAML implementation

    def _train_custom(self, task_ids: List[str], epochs: int) -> MetaLearningResult:
        """Train using custom meta-learning approach"""
        # Placeholder for custom implementations
        self.logger.info(f"Would train custom meta-learner on {len(task_ids)} tasks for {epochs} epochs")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.CUSTOM,
            metrics={"final_meta_loss": 0.5},  # Placeholder
            adaptation_history=[]
        )

    def _adapt_maml(self, request: AdaptationRequest,
                   task: TaskDescription) -> MetaLearningResult:
        """
        Adapt MAML model to a new task.

        Args:
            request: The adaptation request
            task: The task description

        Returns:
            MetaLearningResult with adaptation results
        """
        if not HAS_LEARN2LEARN or self.meta_model is None:
            return MetaLearningResult(
                success=False,
                paradigm=MetaLearningParadigm.MAML,
                error_message="MAML model not available"
            )

        # Create a copy of the meta-model for adaptation
        learner = self.meta_model.clone()

        # Create loss function
        loss_fn = torch.nn.CrossEntropyLoss()

        # In a real implementation, this would use the actual data from request
        # Here we generate synthetic data for illustration
        support_data = torch.randn(task.k_shot * task.n_way, 20)  # Random features
        support_labels = torch.cat([torch.full((task.k_shot,), i) for i in range(task.n_way)])

        # Adaptation history
        adaptation_history = []
        adaptation_losses = []

        # Adapt the model on the support set
        for step in range(request.adaptation_steps):
            # Forward pass
            support_preds = learner(support_data)
            support_loss = loss_fn(support_preds, support_labels.long())

            # Adaptation step
            learner.adapt(support_loss)

            # Track loss
            adaptation_losses.append(support_loss.item())

            # Record adaptation history
            adaptation_history.append({
                "step": step,
                "loss": support_loss.item()
            })

        # Calculate accuracy on support set
        with torch.no_grad():
            support_preds = learner(support_data)
            _, support_preds_class = torch.max(support_preds, dim=1)
            support_accuracy = (support_preds_class == support_labels).float().mean().item()

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.MAML,
            model_state=learner.state_dict(),
            metrics={
                "final_adaptation_loss": adaptation_losses[-1] if adaptation_losses else 0.0,
                "support_accuracy": support_accuracy
            },
            adaptation_history=adaptation_history
        )

    def _adapt_proto_net(self, request: AdaptationRequest,
                        task: TaskDescription) -> MetaLearningResult:
        """
        Adapt Prototypical Network to a new task.

        Args:
            request: The adaptation request
            task: The task description

        Returns:
            MetaLearningResult with adaptation results
        """
        # Simplified implementation for illustration
        self.logger.info(f"Would adapt Prototypical Network to task {task.name}")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.PROTO_NET,
            metrics={"support_accuracy": 0.85},  # Placeholder
            adaptation_history=[]
        )

    def _adapt_reptile(self, request: AdaptationRequest,
                      task: TaskDescription) -> MetaLearningResult:
        """
        Adapt Reptile model to a new task.

        Args:
            request: The adaptation request
            task: The task description

        Returns:
            MetaLearningResult with adaptation results
        """
        # Simplified implementation for illustration
        self.logger.info(f"Would adapt Reptile to task {task.name}")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.REPTILE,
            metrics={"support_accuracy": 0.82},  # Placeholder
            adaptation_history=[]
        )

    def _adapt_anil(self, request: AdaptationRequest,
                   task: TaskDescription) -> MetaLearningResult:
        """
        Adapt ANIL model (Almost No Inner Loop) to a new task.

        Args:
            request: The adaptation request
            task: The task description

        Returns:
            MetaLearningResult with adaptation results
        """
        # ANIL is similar to MAML but only adapts the last layer
        # This would be similar to _adapt_maml but with specific layers frozen

        return self._adapt_maml(request, task)  # Reuse MAML implementation for now

    def _adapt_custom(self, request: AdaptationRequest,
                     task: TaskDescription) -> MetaLearningResult:
        """
        Adapt custom meta-learning model to a new task.

        Args:
            request: The adaptation request
            task: The task description

        Returns:
            MetaLearningResult with adaptation results
        """
        # Placeholder for custom implementations
        self.logger.info(f"Would adapt custom meta-learner to task {task.name}")

        return MetaLearningResult(
            success=True,
            paradigm=MetaLearningParadigm.CUSTOM,
            metrics={"support_accuracy": 0.80},  # Placeholder
            adaptation_history=[]
        )

    def _evaluate_adaptation(self, request: AdaptationRequest,
                            result: MetaLearningResult) -> Dict[str, float]:
        """
        Evaluate adapted model on query data.

        Args:
            request: The adaptation request
            result: The adaptation result

        Returns:
            Dictionary of evaluation metrics
        """
        if not HAS_LEARN2LEARN or result.model_state is None:
            return {"query_accuracy": 0.0}

        # Create evaluation metrics
        metrics = {}

        try:
            # In a real implementation, this would use the actual query data
            # Here we generate synthetic data for illustration
            task = self.task_registry[request.task_id]
            query_data = torch.randn(task.query_size, 20)  # Random features
            query_labels = torch.randint(0, task.n_way, (task.query_size,))

            # Load the adapted model
            adapted_model = copy.deepcopy(self.meta_model)
            adapted_model.load_state_dict(result.model_state)

            # Evaluate on query set
            with torch.no_grad():
                query_preds = adapted_model(query_data)
                _, query_preds_class = torch.max(query_preds, dim=1)
                query_accuracy = (query_preds_class == query_labels).float().mean().item()

            metrics["query_accuracy"] = query_accuracy

            # Calculate additional requested metrics
            for metric_name in request.metrics:
                if metric_name == "loss":
                    loss_fn = torch.nn.CrossEntropyLoss()
                    query_loss = loss_fn(query_preds, query_labels.long()).item()
                    metrics["query_loss"] = query_loss
                # Add more metrics as needed

        except Exception as e:
            self.logger.error(f"Error during evaluation: {str(e)}")
            metrics["error"] = str(e)

        return metrics

ModuleNotFoundError: No module named 'grandnexus'

# Reinforcement

In [ ]:
"""
reinforcement.py - Reinforcement Learning Module for GrandNexus

This module implements reinforcement learning capabilities for the
GrandNexus cognitive architecture. It provides a flexible foundation
for learning from experience through reward signals, supporting various
policy types and learning algorithms.

Key features:
- Support for multiple RL algorithms (Q-Learning, DQN, PPO, etc.)
- Integration with both symbolic and neural components
- Experience replay for efficient learning
- Policy abstraction for flexible agent behavior
- Exploration strategies with adaptive parameters
- Integration with episodic and semantic memory modules
"""

import logging
import threading
import time
import uuid
import random
import math
from enum import Enum, auto
from typing import Dict, List, Any, Optional, Callable, Tuple, Union, Set
from dataclasses import dataclass, field
from collections import deque, defaultdict
import copy
import numpy as np

# Assuming these exist in the architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory

# Try importing PyTorch for deep RL methods, but make it optional
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# Algorithm types
class RLAlgorithm(Enum):
    """Types of reinforcement learning algorithms supported"""
    Q_LEARNING = auto()
    SARSA = auto()
    DQN = auto()
    A2C = auto()
    PPO = auto()
    SAC = auto()
    CUSTOM = auto()

# Exploration strategies
class ExplorationStrategy(Enum):
    """Types of exploration strategies"""
    EPSILON_GREEDY = auto()
    BOLTZMANN = auto()
    UCB = auto()
    THOMPSON = auto()
    NOISY_NETS = auto()
    PARAMETER_NOISE = auto()
    CUSTOM = auto()

@dataclass
class State:
    """
    Representation of an environment state.

    This is deliberately flexible to support both discrete and continuous
    state spaces, as well as both numeric and symbolic representations.
    """
    features: Any  # Can be vector, tensor, dictionary, symbolic state, etc.
    metadata: Dict[str, Any] = field(default_factory=dict)

    # Hash implementation for dictionary key usage
    def __hash__(self):
        if isinstance(self.features, (tuple, list, np.ndarray)):
            return hash(tuple(self.features))
        return hash(str(self.features))

    # Equality for dictionary comparison
    def __eq__(self, other):
        if not isinstance(other, State):
            return False
        return hash(self) == hash(other)

@dataclass
class Action:
    """
    Representation of an agent action.

    Supports both discrete and continuous action spaces,
    as well as both numeric and symbolic representations.
    """
    value: Any  # Can be int, float, vector, symbolic action, etc.
    metadata: Dict[str, Any] = field(default_factory=dict)

    # Hash implementation for dictionary key usage
    def __hash__(self):
        if isinstance(self.value, (tuple, list, np.ndarray)):
            return hash(tuple(self.value))
        return hash(str(self.value))

    # Equality for dictionary comparison
    def __eq__(self, other):
        if not isinstance(other, Action):
            return False
        return hash(self) == hash(other)

@dataclass
class Experience:
    """
    Representation of a single reinforcement learning experience
    (state, action, reward, next_state).
    """
    state: State
    action: Action
    reward: float
    next_state: State
    done: bool = False
    info: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)
    importance: float = 0.5  # Priority for experience replay

@dataclass
class RLEnvironment:
    """Abstract representation of an environment for RL"""
    name: str
    state_space: Any  # Description of possible states
    action_space: Any  # Description of possible actions
    max_steps: int = 1000
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class RLPolicy:
    """
    Policy for determining actions based on states.

    A policy can be represented in many ways:
    - Q-table for discrete state/action spaces
    - Neural network for continuous spaces
    - Rule-based for symbolic approaches
    """
    policy_id: str
    policy_type: str  # "q_table", "neural_network", "rule_based", etc.
    parameters: Dict[str, Any] = field(default_factory=dict)
    model: Any = None  # Specific implementation (Q-table, neural net, etc.)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class RLConfig:
    """Configuration for the reinforcement learning module"""
    algorithm: RLAlgorithm = RLAlgorithm.Q_LEARNING
    learning_rate: float = 0.01
    discount_factor: float = 0.99
    exploration_strategy: ExplorationStrategy = ExplorationStrategy.EPSILON_GREEDY
    initial_exploration_rate: float = 1.0
    final_exploration_rate: float = 0.01
    exploration_decay: float = 0.995
    batch_size: int = 32
    update_frequency: int = 4
    target_update_frequency: int = 1000  # For DQN-like algorithms
    replay_buffer_size: int = 10000
    prioritized_replay: bool = False
    double_q: bool = False  # Whether to use double Q-learning
    use_episodic_memory: bool = True
    use_semantic_memory: bool = True
    neural_network_config: Dict[str, Any] = field(default_factory=lambda: {
        "hidden_layers": [128, 128],
        "activation": "relu"
    })
    custom_parameters: Dict[str, Any] = field(default_factory=dict)

class ExperienceBuffer:
    """
    Buffer for storing and sampling experiences for experience replay.

    Supports both uniform and prioritized sampling.
    """

    def __init__(self, capacity: int = 10000, use_priorities: bool = False,
                alpha: float = 0.6, beta: float = 0.4, beta_increment: float = 0.001):
        """
        Initialize experience buffer.

        Args:
            capacity: Maximum number of experiences to store
            use_priorities: Whether to use prioritized experience replay
            alpha: Priority exponent (how much prioritization to use)
            beta: Importance sampling exponent
            beta_increment: How much to increase beta over time
        """
        self.capacity = capacity
        self.use_priorities = use_priorities
        self.alpha = alpha
        self.beta = beta
        self.beta_increment = beta_increment

        # Buffer storage
        self.buffer = deque(maxlen=capacity)
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.position = 0
        self.size = 0

        # Stats
        self.add_count = 0
        self.sample_count = 0

    def add(self, experience: Experience) -> None:
        """
        Add an experience to the buffer.

        Args:
            experience: The experience to add
        """
        if self.size < self.capacity:
            self.buffer.append(experience)
            self.size += 1
        else:
            self.buffer[self.position] = experience

        # Set maximum priority for new experiences
        if self.use_priorities:
            max_priority = self.priorities.max() if self.size > 0 else 1.0
            self.priorities[self.position] = max_priority

        # Update position
        self.position = (self.position + 1) % self.capacity
        self.add_count += 1

    def sample(self, batch_size: int) -> Tuple[List[Experience], List[int], np.ndarray]:
        """
        Sample a batch of experiences from the buffer.

        Args:
            batch_size: Number of experiences to sample

        Returns:
            Tuple of (experiences, indices, weights)
        """
        # Ensure we have enough experiences
        batch_size = min(batch_size, self.size)

        # Sample according to priorities if enabled
        if self.use_priorities:
            # Calculate sampling probabilities
            priorities = self.priorities[:self.size]
            probabilities = priorities ** self.alpha
            probabilities /= probabilities.sum()

            # Sample indices
            indices = np.random.choice(self.size, batch_size, replace=False, p=probabilities)

            # Calculate importance sampling weights
            weights = (self.size * probabilities[indices]) ** -self.beta
            weights /= weights.max()

            # Incrementally increase beta
            self.beta = min(1.0, self.beta + self.beta_increment)
        else:
            # Uniform sampling
            indices = np.random.choice(self.size, batch_size, replace=False)
            weights = np.ones(batch_size, dtype=np.float32)

        # Get experiences from buffer
        experiences = [self.buffer[i] for i in indices]

        self.sample_count += 1
        return experiences, indices, weights

    def update_priorities(self, indices: List[int], priorities: List[float]) -> None:
        """
        Update priorities for experiences.

        Args:
            indices: Indices of experiences to update
            priorities: New priorities for these experiences
        """
        if not self.use_priorities:
            return

        for idx, priority in zip(indices, priorities):
            # Ensure positive priority
            self.priorities[idx] = max(1e-6, priority)

    def clear(self) -> None:
        """Clear the buffer"""
        self.buffer.clear()
        self.priorities = np.zeros(self.capacity, dtype=np.float32)
        self.position = 0
        self.size = 0

    def get_stats(self) -> Dict[str, Any]:
        """Get buffer statistics"""
        return {
            "size": self.size,
            "capacity": self.capacity,
            "utilization": self.size / max(1, self.capacity),
            "add_count": self.add_count,
            "sample_count": self.sample_count,
            "beta": self.beta if self.use_priorities else None,
            "use_priorities": self.use_priorities
        }

class ReinforcementLearning:
    """
    Core reinforcement learning module for GrandNexus.

    This module provides the functionality for agents to learn from
    experience through interaction with environments, using various
    reinforcement learning algorithms.
    """

    def __init__(self, config: Optional[RLConfig] = None,
                nexus_core: Optional[NexusCore] = None,
                episodic_memory: Optional[EpisodicMemory] = None,
                working_memory: Optional[WorkingMemory] = None,
                semantic_memory: Optional[SemanticMemory] = None):
        """
        Initialize the reinforcement learning module.

        Args:
            config: Configuration for the RL module
            nexus_core: Reference to the NexusCore
            episodic_memory: Reference to episodic memory (for storing experiences)
            working_memory: Reference to working memory (for active learning processes)
            semantic_memory: Reference to semantic memory (for storing learned policies)
        """
        self.logger = logging.getLogger("GrandNexus.Learning.Reinforcement")

        # Store configuration
        self.config = config or RLConfig()

        # Store module references
        self.nexus_core = nexus_core
        self.episodic_memory = episodic_memory
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.lock = threading.RLock()
        self.initialized = False
        self.running = False

        # Storage for environments, policies, and experiences
        self.environments = {}
        self.policies = {}
        self.experience_buffers = {}
        self.active_learning = {}  # Tracks active learning processes

        # Exploration parameters
        self.exploration_rate = self.config.initial_exploration_rate

        # Learning metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        # Create default experience buffer
        self.default_buffer = ExperienceBuffer(
            capacity=self.config.replay_buffer_size,
            use_priorities=self.config.prioritized_replay
        )

        self.logger.info(f"ReinforcementLearning initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """
        Initialize the reinforcement learning module.

        Returns:
            True if initialization succeeded, False otherwise
        """
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Initialize algorithm-specific components
                self._initialize_algorithm()

                self.initialized = True
                self.logger.info("ReinforcementLearning initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"ReinforcementLearning initialization failed: {str(e)}")
                return False

    def register_environment(self, environment: RLEnvironment) -> None:
        """
        Register an environment for reinforcement learning.

        Args:
            environment: The environment to register
        """
        with self.lock:
            self.environments[environment.name] = environment
            self.logger.info(f"Registered environment: {environment.name}")

    def create_policy(self, name: str, policy_type: str, environment_name: str,
                     parameters: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new policy for a registered environment.

        Args:
            name: Name for the policy
            policy_type: Type of policy ("q_table", "neural_network", etc.)
            environment_name: Name of the environment this policy is for
            parameters: Additional parameters for the policy

        Returns:
            Policy ID string
        """
        with self.lock:
            if environment_name not in self.environments:
                raise ValueError(f"Environment {environment_name} not registered")

            # Generate policy ID
            policy_id = f"{name}_{str(uuid.uuid4())[:8]}"

            # Create policy based on type
            if policy_type == "q_table":
                model = self._create_q_table(self.environments[environment_name])
            elif policy_type == "neural_network":
                if not HAS_TORCH:
                    raise ImportError("PyTorch is required for neural network policies")
                model = self._create_neural_network(self.environments[environment_name])
            elif policy_type == "rule_based":
                model = {}  # Placeholder for rule-based policy
            else:
                raise ValueError(f"Unsupported policy type: {policy_type}")

            # Create policy object
            policy = RLPolicy(
                policy_id=policy_id,
                policy_type=policy_type,
                parameters=parameters or {},
                model=model,
                metadata={"environment_name": environment_name}
            )

            # Store policy
            self.policies[policy_id] = policy

            # Create experience buffer for this policy if one doesn't exist
            if policy_id not in self.experience_buffers:
                self.experience_buffers[policy_id] = ExperienceBuffer(
                    capacity=self.config.replay_buffer_size,
                    use_priorities=self.config.prioritized_replay
                )

            self.logger.info(f"Created policy {name} ({policy_id}) for environment {environment_name}")
            return policy_id

    def select_action(self, policy_id: str, state: State) -> Action:
        """
        Select an action according to the given policy and state.

        Args:
            policy_id: ID of the policy to use
            state: Current state of the environment

        Returns:
            Selected action
        """
        if policy_id not in self.policies:
            raise ValueError(f"Policy {policy_id} not found")

        policy = self.policies[policy_id]

        # Check if exploration should be used
        if random.random() < self.exploration_rate:
            # Explore: select random action
            environment_name = policy.metadata["environment_name"]
            environment = self.environments[environment_name]

            # Handle different action space types
            if isinstance(environment.action_space, list):
                action_value = random.choice(environment.action_space)
            elif isinstance(environment.action_space, range):
                action_value = random.choice(list(environment.action_space))
            elif hasattr(environment.action_space, 'sample'):
                # OpenAI Gym-like interface
                action_value = environment.action_space.sample()
            else:
                # Fallback: assume it's a number of discrete actions
                try:
                    num_actions = int(environment.action_space)
                    action_value = random.randint(0, num_actions - 1)
                except (ValueError, TypeError):
                    raise ValueError(f"Unsupported action space: {environment.action_space}")

            action = Action(value=action_value, metadata={"exploration": True})
        else:
            # Exploit: select best action according to policy
            if policy.policy_type == "q_table":
                action = self._select_action_q_table(policy, state)
            elif policy.policy_type == "neural_network":
                action = self._select_action_neural_network(policy, state)
            elif policy.policy_type == "rule_based":
                action = self._select_action_rule_based(policy, state)
            else:
                raise ValueError(f"Unsupported policy type: {policy.policy_type}")

        return action

    def store_experience(self, policy_id: str, experience: Experience) -> None:
        """
        Store an experience for later learning.

        Args:
            policy_id: ID of the policy that generated the experience
            experience: The experience to store
        """
        with self.lock:
            # Store in policy-specific buffer
            if policy_id in self.experience_buffers:
                self.experience_buffers[policy_id].add(experience)

            # Also store in default buffer
            self.default_buffer.add(experience)

            # Store in episodic memory if available
            if self.episodic_memory is not None and self.config.use_episodic_memory:
                self._store_in_episodic_memory(experience)

    def update_policy(self, policy_id: str, experiences: Optional[List[Experience]] = None) -> Dict[str, Any]:
        """
        Update a policy based on stored experiences.

        Args:
            policy_id: ID of the policy to update
            experiences: Optional specific experiences to learn from
                         (if None, samples from buffer)

        Returns:
            Dictionary with update metrics
        """
        with self.lock:
            if policy_id not in self.policies:
                raise ValueError(f"Policy {policy_id} not found")

            policy = self.policies[policy_id]

            # Get experiences
            if experiences is None:
                # Sample from buffer
                buffer = self.experience_buffers.get(policy_id, self.default_buffer)
                if buffer.size < self.config.batch_size:
                    # Not enough experiences yet
                    return {"updated": False, "reason": "insufficient_data"}

                experiences, indices, weights = buffer.sample(self.config.batch_size)
            else:
                # Use provided experiences
                indices = None
                weights = np.ones(len(experiences))

            # Update policy based on type
            if policy.policy_type == "q_table":
                metrics = self._update_q_table(policy, experiences, weights)
            elif policy.policy_type == "neural_network":
                metrics = self._update_neural_network(policy, experiences, weights)
            elif policy.policy_type == "rule_based":
                metrics = self._update_rule_based(policy, experiences, weights)
            else:
                raise ValueError(f"Unsupported policy type: {policy.policy_type}")

            # Update priorities if using prioritized replay
            if self.config.prioritized_replay and indices is not None:
                buffer = self.experience_buffers.get(policy_id, self.default_buffer)
                if "td_errors" in metrics:
                    priorities = np.abs(metrics["td_errors"]) + 1e-6  # Avoid zero priority
                    buffer.update_priorities(indices, priorities)

            # Update metrics
            self._update_metrics(policy_id, metrics)

            # Update exploration rate
            self._update_exploration_rate()

            # Store updated policy in semantic memory if available
            if self.semantic_memory is not None and self.config.use_semantic_memory:
                self._store_policy_in_semantic_memory(policy)

            return {"updated": True, **metrics}

    def train(self, policy_id: str, environment_name: str, episodes: int = 100,
             max_steps_per_episode: Optional[int] = None,
             step_callback: Optional[Callable[[int, int, Dict[str, Any]], None]] = None,
             episode_callback: Optional[Callable[[int, Dict[str, Any]], None]] = None) -> Dict[str, Any]:
        """
        Train a policy on an environment for a specified number of episodes.

        Args:
            policy_id: ID of the policy to train
            environment_name: Name of the environment to train on
            episodes: Number of episodes to train for
            max_steps_per_episode: Maximum steps per episode (overrides environment default)
            step_callback: Optional callback after each step
            episode_callback: Optional callback after each episode

        Returns:
            Dictionary with training metrics
        """
        if not self.initialized:
            self.initialize()

        if policy_id not in self.policies:
            raise ValueError(f"Policy {policy_id} not found")

        if environment_name not in self.environments:
            raise ValueError(f"Environment {environment_name} not registered")

        environment = self.environments[environment_name]

        # Check if this environment has a step function
        if not hasattr(environment, 'step') or not callable(environment.step):
            raise ValueError(f"Environment {environment_name} does not have a step method")

        # Initialize learning process
        process_id = f"{policy_id}_{environment_name}_{str(uuid.uuid4())[:8]}"
        self.active_learning[process_id] = {
            "policy_id": policy_id,
            "environment_name": environment_name,
            "episodes": episodes,
            "current_episode": 0,
            "start_time": time.time(),
            "status": "running",
            "metrics": {}
        }

        # Set maximum steps
        max_steps = max_steps_per_episode or environment.max_steps

        # Training metrics
        total_rewards = []
        episode_lengths = []

        try:
            # Training loop
            for episode in range(episodes):
                # Reset environment
                state = environment.reset()

                # Convert to State object if necessary
                if not isinstance(state, State):
                    state = State(features=state)

                # Episode variables
                episode_reward = 0
                episode_steps = 0
                done = False

                # Episode loop
                while not done and episode_steps < max_steps:
                    # Select action
                    action = self.select_action(policy_id, state)

                    # Take action in environment
                    next_state, reward, done, info = environment.step(action.value)

                    # Convert to State object if necessary
                    if not isinstance(next_state, State):
                        next_state = State(features=next_state)

                    # Create experience
                    experience = Experience(
                        state=state,
                        action=action,
                        reward=reward,
                        next_state=next_state,
                        done=done,
                        info=info
                    )

                    # Store experience
                    self.store_experience(policy_id, experience)

                    # Update policy
                    if episode_steps % self.config.update_frequency == 0:
                        self.update_policy(policy_id)

                    # Update state
                    state = next_state

                    # Update episode metrics
                    episode_reward += reward
                    episode_steps += 1

                    # Call step callback if provided
                    if step_callback:
                        step_callback(episode, episode_steps, {
                            "reward": reward,
                            "done": done,
                            "policy_id": policy_id,
                            "environment_name": environment_name
                        })

                # Update episode metrics
                total_rewards.append(episode_reward)
                episode_lengths.append(episode_steps)

                # Update training process status
                self.active_learning[process_id]["current_episode"] = episode + 1
                self.active_learning[process_id]["metrics"] = {
                    "mean_reward": np.mean(total_rewards[-100:]),
                    "mean_episode_length": np.mean(episode_lengths[-100:]),
                    "last_reward": episode_reward,
                    "last_episode_length": episode_steps,
                    "exploration_rate": self.exploration_rate
                }

                # Call episode callback if provided
                if episode_callback:
                    episode_callback(episode, {
                        "reward": episode_reward,
                        "steps": episode_steps,
                        "policy_id": policy_id,
                        "environment_name": environment_name
                    })

            # Mark training process as completed
            self.active_learning[process_id]["status"] = "completed"
            self.active_learning[process_id]["end_time"] = time.time()

            # Return training metrics
            return {
                "process_id": process_id,
                "status": "completed",
                "episodes_completed": episodes,
                "total_steps": sum(episode_lengths),
                "mean_reward": np.mean(total_rewards),
                "mean_episode_length": np.mean(episode_lengths),
                "rewards": total_rewards,
                "episode_lengths": episode_lengths
            }

        except Exception as e:
            # Mark training process as failed
            self.active_learning[process_id]["status"] = "failed"
            self.active_learning[process_id]["error"] = str(e)
            self.active_learning[process_id]["end_time"] = time.time()

            self.logger.error(f"Training failed: {str(e)}")

            # Re-raise exception
            raise

    def evaluate(self, policy_id: str, environment_name: str, episodes: int = 10,
                max_steps_per_episode: Optional[int] = None) -> Dict[str, Any]:
        """
        Evaluate a policy on an environment.

        Args:
            policy_id: ID of the policy to evaluate
            environment_name: Name of the environment to evaluate on
            episodes: Number of episodes to evaluate for
            max_steps_per_episode: Maximum steps per episode (overrides environment default)

        Returns:
            Dictionary with evaluation metrics
        """
        if not self.initialized:
            self.initialize()

        if policy_id not in self.policies:
            raise ValueError(f"Policy {policy_id} not found")

        if environment_name not in self.environments:
            raise ValueError(f"Environment {environment_name} not registered")

        environment = self.environments[environment_name]

        # Check if this environment has a step function
        if not hasattr(environment, 'step') or not callable(environment.step):
            raise ValueError(f"Environment {environment_name} does not have a step method")

        # Set maximum steps
        max_steps = max_steps_per_episode or environment.max_steps

        # Temporarily disable exploration
        original_exploration_rate = self.exploration_rate
        self.exploration_rate = 0.0

        # Evaluation metrics
        total_rewards = []
        episode_lengths = []

        try:
            # Evaluation loop
            for episode in range(episodes):
                # Reset environment
                state = environment.reset()

                # Convert to State object if necessary
                if not isinstance(state, State):
                    state = State(features=state)

                # Episode variables
                episode_reward = 0
                episode_steps = 0
                done = False

                # Episode loop
                while not done and episode_steps < max_steps:
                    # Select action (greedy policy)
                    action = self.select_action(policy_id, state)

                    # Take action in environment
                    next_state, reward, done, info = environment.step(action.value)

                    # Convert to State object if necessary
                    if not isinstance(next_state, State):
                        next_state = State(features=next_state)

                    # Update state
                    state = next_state

                    # Update episode metrics
                    episode_reward += reward
                    episode_steps += 1

                # Update evaluation metrics
                total_rewards.append(episode_reward)
                episode_lengths.append(episode_steps)

            # Restore exploration rate
            self.exploration_rate = original_exploration_rate

            # Return evaluation metrics
            return {
                "episodes": episodes,
                "total_steps": sum(episode_lengths),
                "mean_reward": np.mean(total_rewards),
                "mean_episode_length": np.mean(episode_lengths),
                "rewards": total_rewards,
                "episode_lengths": episode_lengths,
                "min_reward": np.min(total_rewards),
                "max_reward": np.max(total_rewards),
                "std_reward": np.std(total_rewards)
            }

        except Exception as e:
            # Restore exploration rate
            self.exploration_rate = original_exploration_rate

            self.logger.error(f"Evaluation failed: {str(e)}")

            # Re-raise exception
            raise

    def save_policy(self, policy_id: str, filepath: str) -> bool:
        """
        Save a policy to disk.

        Args:
            policy_id: ID of the policy to save
            filepath: Path to save the policy to

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            if policy_id not in self.policies:
                raise ValueError(f"Policy {policy_id} not found")

            try:
                policy = self.policies[policy_id]

                # Handle different policy types
                if policy.policy_type == "q_table":
                    # For Q-table, we can just save the table
                    np.save(filepath, policy.model)
                elif policy.policy_type == "neural_network" and HAS_TORCH:
                    # For neural network, save the state dict
                    torch.save(policy.model.state_dict(), filepath)
                elif policy.policy_type == "rule_based":
                    # For rule-based, save rules as JSON
                    import json
                    with open(filepath, 'w') as f:
                        json.dump(policy.model, f)
                else:
                    raise ValueError(f"Saving not implemented for policy type: {policy.policy_type}")

                self.logger.info(f"Saved policy {policy_id} to {filepath}")
                return True
            except Exception as e:
                self.logger.error(f"Failed to save policy {policy_id}: {str(e)}")
                return False

    def load_policy(self, policy_id: str, filepath: str) -> bool:
        """
        Load a policy from disk.

        Args:
            policy_id: ID of the policy to load into
            filepath: Path to load the policy from

        Returns:
            True if successful, False otherwise
        """
        with self.lock:
            if policy_id not in self.policies:
                raise ValueError(f"Policy {policy_id} not found")

            try:
                policy = self.policies[policy_id]

                # Handle different policy types
                if policy.policy_type == "q_table":
                    # For Q-table, load the table
                    policy.model = np.load(filepath)
                elif policy.policy_type == "neural_network" and HAS_TORCH:
                    # For neural network, load the state dict
                    policy.model.load_state_dict(torch.load(filepath))
                    policy.model.eval()  # Set to evaluation mode
                elif policy.policy_type == "rule_based":
                    # For rule-based, load rules from JSON
                    import json
                    with open(filepath, 'r') as f:
                        policy.model = json.load(f)
                else:
                    raise ValueError(f"Loading not implemented for policy type: {policy.policy_type}")

                self.logger.info(f"Loaded policy {policy_id} from {filepath}")
                return True
            except Exception as e:
                self.logger.error(f"Failed to load policy {policy_id}: {str(e)}")
                return False

    def get_policy_stats(self, policy_id: str) -> Dict[str, Any]:
        """
        Get statistics for a policy.

        Args:
            policy_id: ID of the policy to get statistics for

        Returns:
            Dictionary with policy statistics
        """
        with self.lock:
            if policy_id not in self.policies:
                raise ValueError(f"Policy {policy_id} not found")

            policy = self.policies[policy_id]

            # Basic stats for all policy types
            stats = {
                "policy_id": policy_id,
                "policy_type": policy.policy_type,
                "environment": policy.metadata.get("environment_name", "unknown")
            }

            # Policy-specific stats
            if policy.policy_type == "q_table":
                # Stats for Q-table
                q_table = policy.model
                stats.update({
                    "states": q_table.shape[0],
                    "actions": q_table.shape[1],
                    "min_q": float(np.min(q_table)),
                    "max_q": float(np.max(q_table)),
                    "mean_q": float(np.mean(q_table)),
                    "nonzero_states": int(np.sum(np.any(q_table != 0, axis=1)))
                })
            elif policy.policy_type == "neural_network" and HAS_TORCH:
                # Stats for neural network
                stats.update({
                    "model_structure": str(policy.model),
                    "parameter_count": sum(p.numel() for p in policy.model.parameters())
                })
            elif policy.policy_type == "rule_based":
                # Stats for rule-based
                stats.update({
                    "rule_count": len(policy.model) if isinstance(policy.model, dict) else 0
                })

            # Add buffer stats if available
            if policy_id in self.experience_buffers:
                stats["buffer"] = self.experience_buffers[policy_id].get_stats()

            return stats

    def get_active_learning_status(self, process_id: Optional[str] = None) -> Dict[str, Any]:
        """
        Get status of active learning processes.

        Args:
            process_id: Optional specific process ID to get status for

        Returns:
            Dictionary with learning process status
        """
        with self.lock:
            if process_id is not None:
                if process_id not in self.active_learning:
                    raise ValueError(f"Learning process {process_id} not found")
                return self.active_learning[process_id]
            else:
                return {pid: status for pid, status in self.active_learning.items()}

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")

            self.nexus_core.register_module(
                name="reinforcement",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _initialize_algorithm(self) -> None:
        """Initialize components specific to the selected algorithm"""
        algorithm = self.config.algorithm

        # Algorithm-specific initialization
        if algorithm == RLAlgorithm.DQN and HAS_TORCH:
            # Initialize target network for DQN
            self.target_network = None
            self.logger.info("Initialized DQN components")
        elif algorithm == RLAlgorithm.PPO and HAS_TORCH:
            # Initialize PPO components
            self.logger.info("Initialized PPO components")

        self.logger.info(f"Initialized algorithm: {algorithm.name}")

    def _create_q_table(self, environment: RLEnvironment) -> np.ndarray:
        """
        Create a Q-table for an environment.

        Args:
            environment: The environment to create a Q-table for

        Returns:
            Numpy array representing the Q-table
        """
        # Determine state space size
        if isinstance(environment.state_space, int):
            # Discrete state space
            state_size = environment.state_space
        elif isinstance(environment.state_space, list) and len(environment.state_space) > 0:
            # List of discrete states
            state_size = len(environment.state_space)
        elif isinstance(environment.state_space, tuple) and len(environment.state_space) == 2:
            # Continuous range (min, max) - discretize
            state_size = 10  # Default discretization
        else:
            # Fallback
            state_size = 10
            self.logger.warning(f"Using default state size of {state_size} for Q-table")

        # Determine action space size
        if isinstance(environment.action_space, int):
            # Discrete action space
            action_size = environment.action_space
        elif isinstance(environment.action_space, list):
            # List of discrete actions
            action_size = len(environment.action_space)
        else:
            # Fallback
            action_size = 4  # Common default
            self.logger.warning(f"Using default action size of {action_size} for Q-table")

        # Create Q-table
        q_table = np.zeros((state_size, action_size))

        return q_table

    def _create_neural_network(self, environment: RLEnvironment) -> nn.Module:
        """
        Create a neural network policy for an environment.

        Args:
            environment: The environment to create a neural network for

        Returns:
            PyTorch neural network
        """
        if not HAS_TORCH:
            raise ImportError("PyTorch is required for neural network policies")

        # Determine input size (state dimensions)
        if hasattr(environment.state_space, 'shape'):
            # Gym-like interface
            input_size = np.prod(environment.state_space.shape)
        elif isinstance(environment.state_space, int):
            # Discrete state space
            input_size = environment.state_space
        elif isinstance(environment.state_space, tuple) and len(environment.state_space) > 0:
            # Tuple of dimensions
            input_size = np.prod(environment.state_space)
        else:
            # Fallback
            input_size = 4  # Common default
            self.logger.warning(f"Using default input size of {input_size} for neural network")

        # Determine output size (action dimensions)
        if hasattr(environment.action_space, 'n'):
            # Gym-like discrete action space
            output_size = environment.action_space.n
        elif isinstance(environment.action_space, int):
            # Discrete action space
            output_size = environment.action_space
        elif isinstance(environment.action_space, list):
            # List of discrete actions
            output_size = len(environment.action_space)
        else:
            # Fallback
            output_size = 2  # Common default
            self.logger.warning(f"Using default output size of {output_size} for neural network")

        # Get neural network configuration
        nn_config = self.config.neural_network_config
        hidden_layers = nn_config.get("hidden_layers", [128, 128])
        activation_name = nn_config.get("activation", "relu")

        # Map activation name to function
        activation_map = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "sigmoid": nn.Sigmoid(),
            "leaky_relu": nn.LeakyReLU()
        }
        activation = activation_map.get(activation_name, nn.ReLU())

        # Create model
        layers = []

        # Input layer
        prev_size = int(input_size)

        # Hidden layers
        for size in hidden_layers:
            layers.append(nn.Linear(prev_size, size))
            layers.append(activation)
            prev_size = size

        # Output layer
        layers.append(nn.Linear(prev_size, output_size))

        # Create model
        model = nn.Sequential(*layers)

        return model

    def _select_action_q_table(self, policy: RLPolicy, state: State) -> Action:
        """
        Select action using Q-table policy.

        Args:
            policy: The policy to use
            state: Current state

        Returns:
            Selected action
        """
        q_table = policy.model

        # Map state to Q-table index
        if isinstance(state.features, (int, np.integer)):
            # Direct index
            state_idx = state.features
        elif isinstance(state.features, (list, tuple, np.ndarray)):
            # For simplicity, just take the first element
            # In a real implementation, you'd need a proper state discretization method
            state_idx = int(state.features[0]) if len(state.features) > 0 else 0
        else:
            # Fallback
            state_idx = 0

        # Ensure index is within bounds
        state_idx = max(0, min(state_idx, q_table.shape[0] - 1))

        # Get action with highest Q-value
        action_idx = np.argmax(q_table[state_idx])

        # Get corresponding action value
        environment_name = policy.metadata["environment_name"]
        environment = self.environments[environment_name]

        if isinstance(environment.action_space, list):
            action_value = environment.action_space[action_idx]
        else:
            action_value = action_idx

        return Action(value=action_value, metadata={"q_value": float(q_table[state_idx][action_idx])})

    def _select_action_neural_network(self, policy: RLPolicy, state: State) -> Action:
        """
        Select action using neural network policy.

        Args:
            policy: The policy to use
            state: Current state

        Returns:
            Selected action
        """
        if not HAS_TORCH:
            raise ImportError("PyTorch is required for neural network policies")

        # Convert state to tensor
        if isinstance(state.features, (list, tuple, np.ndarray)):
            # Convert to tensor
            state_tensor = torch.FloatTensor(state.features)
        elif isinstance(state.features, (int, float)):
            # Single value
            state_tensor = torch.FloatTensor([state.features])
        else:
            # Fallback
            state_tensor = torch.zeros(1)
            self.logger.warning(f"Unable to convert state to tensor: {state.features}")

        # Get action values from model
        model = policy.model
        model.eval()  # Set to evaluation mode
        with torch.no_grad():
            action_values = model(state_tensor)

        # Get action with highest value
        action_idx = torch.argmax(action_values).item()

        # Get corresponding action value
        environment_name = policy.metadata["environment_name"]
        environment = self.environments[environment_name]

        if isinstance(environment.action_space, list):
            action_value = environment.action_space[action_idx]
        else:
            action_value = action_idx

        return Action(value=action_value, metadata={"q_value": float(action_values[action_idx])})

    def _select_action_rule_based(self, policy: RLPolicy, state: State) -> Action:
        """
        Select action using rule-based policy.

        Args:
            policy: The policy to use
            state: Current state

        Returns:
            Selected action
        """
        rules = policy.model

        # This is a simplified implementation
        # In a real system, you'd have a more sophisticated rule engine

        environment_name = policy.metadata["environment_name"]
        environment = self.environments[environment_name]

        # Default action (fallback)
        if isinstance(environment.action_space, list):
            default_action = environment.action_space[0]
        else:
            default_action = 0

        # Check if there are rules for this state
        state_key = str(state.features)
        if state_key in rules:
            action_value = rules[state_key]
        else:
            action_value = default_action

        return Action(value=action_value, metadata={"rule_based": True})

    def _update_q_table(self, policy: RLPolicy, experiences: List[Experience],
                      weights: np.ndarray) -> Dict[str, Any]:
        """
        Update Q-table policy based on experiences.

        Args:
            policy: The policy to update
            experiences: List of experiences to learn from
            weights: Importance weights for experiences

        Returns:
            Dictionary with update metrics
        """
        q_table = policy.model
        learning_rate = self.config.learning_rate
        discount_factor = self.config.discount_factor

        # Metrics
        td_errors = []

        # Process each experience
        for i, experience in enumerate(experiences):
            # Extract components
            state = experience.state
            action = experience.action
            reward = experience.reward
            next_state = experience.next_state
            done = experience.done
            weight = weights[i]

            # Map states to Q-table indices
            if isinstance(state.features, (int, np.integer)):
                state_idx = state.features
            elif isinstance(state.features, (list, tuple, np.ndarray)):
                state_idx = int(state.features[0]) if len(state.features) > 0 else 0
            else:
                state_idx = 0

            if isinstance(next_state.features, (int, np.integer)):
                next_state_idx = next_state.features
            elif isinstance(next_state.features, (list, tuple, np.ndarray)):
                next_state_idx = int(next_state.features[0]) if len(next_state.features) > 0 else 0
            else:
                next_state_idx = 0

            # Ensure indices are within bounds
            state_idx = max(0, min(state_idx, q_table.shape[0] - 1))
            next_state_idx = max(0, min(next_state_idx, q_table.shape[0] - 1))

            # Map action to Q-table index
            if isinstance(action.value, (int, np.integer)):
                action_idx = action.value
            elif isinstance(action.value, (list, tuple, np.ndarray)):
                action_idx = int(action.value[0]) if len(action.value) > 0 else 0
            else:
                # Try to map to index in environment action space
                environment_name = policy.metadata["environment_name"]
                environment = self.environments[environment_name]

                if isinstance(environment.action_space, list):
                    try:
                        action_idx = environment.action_space.index(action.value)
                    except ValueError:
                        action_idx = 0
                else:
                    action_idx = 0

            # Ensure action index is within bounds
            action_idx = max(0, min(action_idx, q_table.shape[1] - 1))

            # Get current Q-value
            current_q = q_table[state_idx, action_idx]

            # Compute target Q-value
            if done:
                target_q = reward
            else:
                # Standard Q-learning target
                target_q = reward + discount_factor * np.max(q_table[next_state_idx])

            # Compute TD error
            td_error = target_q - current_q
            td_errors.append(td_error)

            # Update Q-value with weighted learning
            q_table[state_idx, action_idx] += learning_rate * weight * td_error

        # Return metrics
        return {
            "td_errors": np.array(td_errors),
            "mean_td_error": float(np.mean(np.abs(td_errors))),
            "max_td_error": float(np.max(np.abs(td_errors))),
            "q_table_mean": float(np.mean(q_table)),
            "q_table_max": float(np.max(q_table)),
            "q_table_min": float(np.min(q_table))
        }

    def _update_neural_network(self, policy: RLPolicy, experiences: List[Experience],
                             weights: np.ndarray) -> Dict[str, Any]:
        """
        Update neural network policy based on experiences.

        Args:
            policy: The policy to update
            experiences: List of experiences to learn from
            weights: Importance weights for experiences

        Returns:
            Dictionary with update metrics
        """
        if not HAS_TORCH:
            raise ImportError("PyTorch is required for neural network policies")

        model = policy.model
        learning_rate = self.config.learning_rate
        discount_factor = self.config.discount_factor

        # Create optimizer if not already present
        if not hasattr(policy, 'optimizer'):
            policy.optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        # Create target network for DQN if needed and not already present
        if self.config.algorithm == RLAlgorithm.DQN and not hasattr(policy, 'target_network'):
            policy.target_network = copy.deepcopy(model)
            policy.target_update_counter = 0

        # Prepare batches
        batch_states = []
        batch_actions = []
        batch_rewards = []
        batch_next_states = []
        batch_dones = []

        # Process each experience
        for experience in experiences:
            # Extract components
            state = experience.state
            action = experience.action
            reward = experience.reward
            next_state = experience.next_state
            done = experience.done

            # Convert states to tensors
            if isinstance(state.features, (list, tuple, np.ndarray)):
                state_tensor = state.features
            else:
                state_tensor = [state.features]

            if isinstance(next_state.features, (list, tuple, np.ndarray)):
                next_state_tensor = next_state.features
            else:
                next_state_tensor = [next_state.features]

            # Convert action to index
            if isinstance(action.value, (int, np.integer)):
                action_idx = action.value
            else:
                # Try to map to index in environment action space
                environment_name = policy.metadata["environment_name"]
                environment = self.environments[environment_name]

                if isinstance(environment.action_space, list):
                    try:
                        action_idx = environment.action_space.index(action.value)
                    except ValueError:
                        action_idx = 0
                else:
                    action_idx = 0

            # Add to batches
            batch_states.append(state_tensor)
            batch_actions.append(action_idx)
            batch_rewards.append(reward)
            batch_next_states.append(next_state_tensor)
            batch_dones.append(float(done))

        # Convert batches to tensors
        states = torch.FloatTensor(batch_states)
        actions = torch.LongTensor(batch_actions)
        rewards = torch.FloatTensor(batch_rewards)
        next_states = torch.FloatTensor(batch_next_states)
        dones = torch.FloatTensor(batch_dones)
        weights = torch.FloatTensor(weights)

        # Get current Q values
        model.train()  # Set to training mode
        current_q = model(states)
        current_q = current_q.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Compute target Q values
        with torch.no_grad():
            if self.config.algorithm == RLAlgorithm.DQN:
                # DQN target
                if self.config.double_q:
                    # Double Q-learning
                    next_actions = model(next_states).max(1)[1].unsqueeze(1)
                    next_q = policy.target_network(next_states).gather(1, next_actions).squeeze(1)
                else:
                    # Standard DQN
                    next_q = policy.target_network(next_states).max(1)[0]
            else:
                # Standard Q-learning
                next_q = model(next_states).max(1)[0]

            target_q = rewards + (1 - dones) * discount_factor * next_q

        # Compute loss (weighted MSE)
        td_errors = target_q - current_q
        loss = (weights * td_errors.pow(2)).mean()

        # Update model
        policy.optimizer.zero_grad()
        loss.backward()
        policy.optimizer.step()

        # Update target network if needed
        if self.config.algorithm == RLAlgorithm.DQN:
            policy.target_update_counter += 1
            if policy.target_update_counter >= self.config.target_update_frequency:
                policy.target_network.load_state_dict(model.state_dict())
                policy.target_update_counter = 0

        # Return metrics
        return {
            "loss": float(loss.item()),
            "td_errors": td_errors.detach().numpy(),
            "mean_td_error": float(td_errors.abs().mean().item()),
            "max_td_error": float(td_errors.abs().max().item())
        }

    def _update_rule_based(self, policy: RLPolicy, experiences: List[Experience],
                         weights: np.ndarray) -> Dict[str, Any]:
        """
        Update rule-based policy based on experiences.

        Args:
            policy: The policy to update
            experiences: List of experiences to learn from
            weights: Importance weights for experiences

        Returns:
            Dictionary with update metrics
        """
        # This is a simplified implementation
        # In a real system, you'd have a more sophisticated rule learning mechanism

        rules = policy.model

        # Process each experience
        updated_rules = 0
        for experience in experiences:
            # Extract components
            state = experience.state
            action = experience.action
            reward = experience.reward

            # Only update rules for positive rewards
            if reward > 0:
                # Create rule for this state
                state_key = str(state.features)
                rules[state_key] = action.value
                updated_rules += 1

        # Return metrics
        return {
            "updated_rules": updated_rules,
            "total_rules": len(rules)
        }

    def _store_in_episodic_memory(self, experience: Experience) -> None:
        """
        Store an experience in episodic memory.

        Args:
            experience: The experience to store
        """
        if self.episodic_memory is None:
            return

        try:
            # Create an episodic memory event
            self.episodic_memory.add_event(
                episode_id="reinforcement_learning",  # This should be a real episode ID
                content={
                    "type": "experience",
                    "state": experience.state.features,
                    "action": experience.action.value,
                    "reward": experience.reward,
                    "next_state": experience.next_state.features,
                    "done": experience.done
                },
                metadata={
                    "source": "reinforcement_learning",
                    "timestamp": experience.timestamp,
                    "importance": experience.importance
                }
            )
        except Exception as e:
            self.logger.error(f"Failed to store experience in episodic memory: {str(e)}")

    def _store_policy_in_semantic_memory(self, policy: RLPolicy) -> None:
        """
        Store a policy in semantic memory.

        Args:
            policy: The policy to store
        """
        if self.semantic_memory is None:
            return

        try:
            # Store policy metadata
            self.semantic_memory.add_concept({
                "name": f"policy_{policy.policy_id}",
                "description": f"Reinforcement learning policy of type {policy.policy_type}",
                "properties": {
                    "policy_id": policy.policy_id,
                    "policy_type": policy.policy_type,
                    "environment": policy.metadata.get("environment_name", "unknown"),
                    "last_updated": time.time()
                }
            })
        except Exception as e:
            self.logger.error(f"Failed to store policy in semantic memory: {str(e)}")

    def _update_metrics(self, policy_id: str, update_metrics: Dict[str, Any]) -> None:
        """
        Update learning metrics.

        Args:
            policy_id: ID of the policy being updated
            update_metrics: Metrics from the update
        """
        # Update global metrics
        for key, value in update_metrics.items():
            if isinstance(value, (int, float)):
                metric_key = f"{policy_id}_{key}"
                self.metrics[metric_key].append(value)

        # Also track exploration rate
        self.metrics["exploration_rate"].append(self.exploration_rate)

    def _update_exploration_rate(self) -> None:
        """Update the exploration rate according to the decay schedule"""
        # Simple exponential decay
        self.exploration_rate = max(
            self.config.final_exploration_rate,
            self.exploration_rate * self.config.exploration_decay
        )

ModuleNotFoundError: No module named 'grandnexus'

# Intrinsic

In [ ]:
"""
intrinsic_motivation.py - Intrinsic Motivation and Curiosity Module for GrandNexus

This module implements computational curiosity and intrinsic motivation mechanisms,
enabling GrandNexus to explore and learn in the absence of external rewards.
It draws inspiration from both neuroscience and computational theories of curiosity,
providing a flexible foundation for self-motivated learning.

Key features:
- Multiple intrinsic reward generators (novelty, surprise, uncertainty, competence)
- Integration with both memory systems and learning modules
- Adaptive curiosity mechanisms that evolve with experience
- Attention direction based on information-theoretic principles
- Support for both symbolic and neural approaches to curiosity
"""

import logging
import threading
import time
import uuid
import random
import math
from enum import Enum, auto
from typing import Dict, List, Any, Optional, Callable, Tuple, Union, Set
from dataclasses import dataclass, field
from collections import deque, defaultdict
import copy
import numpy as np

# Assuming these exist in the architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.learning.learning_core import LearningCore

# Optional imports for advanced functionality
try:
    from sklearn.neighbors import NearestNeighbors
    from sklearn.cluster import KMeans
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# Intrinsic motivation types
class MotivationType(Enum):
    """Types of intrinsic motivation mechanisms"""
    NOVELTY = auto()       # Driven by encountering new states/experiences
    SURPRISE = auto()      # Driven by prediction errors
    UNCERTAINTY = auto()   # Driven by epistemic uncertainty
    COMPETENCE = auto()    # Driven by mastery (learning progress)
    EMPOWERMENT = auto()   # Driven by control over environment (options/choices)
    CURIOSITY = auto()     # General/mixed curiosity drive
    CUSTOM = auto()        # Custom motivation type

class AttentionFocusType(Enum):
    """Types of attention focus mechanisms"""
    ENTROPY = auto()       # Focus on high entropy areas
    UNCERTAINTY = auto()   # Focus on uncertain areas
    PREDICTION_ERROR = auto() # Focus on areas with high prediction error
    LEARNING_PROGRESS = auto() # Focus on areas with high learning progress
    RANDOM = auto()        # Random focus (baseline)
    CUSTOM = auto()        # Custom focus type

@dataclass
class CuriosityState:
    """
    Represents the current state of curiosity and focus of attention.
    This serves as a high-level aggregation of different intrinsic drives.
    """
    interest_level: float = 0.5         # Overall interest/curiosity level (0-1)
    attention_focus: Optional[Any] = None  # Current focus of attention
    focus_strength: float = 0.5         # Strength of current focus (0-1)
    active_drives: List[MotivationType] = field(default_factory=list)
    novelty_seeking: float = 0.5        # Current novelty seeking level (0-1)
    exploration_rate: float = 0.5       # Current exploration bias (0-1)
    timestamp: float = field(default_factory=time.time)

@dataclass
class ObservationData:
    """
    Container for an observation that can be evaluated for intrinsic motivation.
    Flexible to support both symbolic and neural representations.
    """
    content: Any                        # The actual observation
    context: Dict[str, Any] = field(default_factory=dict)  # Contextual information
    source: str = "unknown"             # Source of the observation
    timestamp: float = field(default_factory=time.time)
    embedding: Optional[np.ndarray] = None  # Vector representation if available
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class MotivationSignal:
    """
    Represents an intrinsic motivation signal generated from an observation.
    """
    observation: ObservationData
    motivation_type: MotivationType
    strength: float                     # Strength of the signal (0-1)
    explanation: str = ""               # Human-readable explanation
    target: Optional[Any] = None        # What aspect should be explored/attended to
    timestamp: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class IntrinsicMotivationConfig:
    """Configuration for the intrinsic motivation module"""
    # General settings
    enabled_types: List[MotivationType] = field(default_factory=lambda: [
        MotivationType.NOVELTY, MotivationType.SURPRISE, MotivationType.CURIOSITY
    ])
    default_weights: Dict[MotivationType, float] = field(default_factory=lambda: {
        MotivationType.NOVELTY: 1.0,
        MotivationType.SURPRISE: 1.0,
        MotivationType.UNCERTAINTY: 0.8,
        MotivationType.COMPETENCE: 0.6,
        MotivationType.EMPOWERMENT: 0.5,
        MotivationType.CURIOSITY: 1.0,
    })

    # Memory and integration
    embedding_dimension: int = 128      # Dimension for embeddings
    memory_capacity: int = 1000         # How many observation embeddings to store
    memory_decay: float = 0.995         # Rate of memory decay (forgetting)

    # Novelty detection
    novelty_threshold: float = 0.8      # Minimum distance to be considered novel
    novelty_k_neighbors: int = 5        # k for nearest neighbor novelty detection

    # Surprise model parameters
    surprise_window_size: int = 50      # Window size for surprise detection
    surprise_min_count: int = 10        # Minimum observations before surprise detection

    # Uncertainty estimation
    uncertainty_ensemble_size: int = 5  # Number of models in uncertainty ensemble
    uncertainty_threshold: float = 0.3  # Threshold for high uncertainty

    # Competence progress tracking
    competence_window_size: int = 100   # Window size for competence tracking
    competence_smoothing: float = 0.1   # Smoothing factor for competence progress

    # Empowerment estimation
    empowerment_horizon: int = 3        # Horizon for empowerment calculation
    empowerment_samples: int = 10       # Number of samples for empowerment estimation

    # Attention mechanism
    attention_decay: float = 0.9        # Decay of attention strength
    attention_focus_type: AttentionFocusType = AttentionFocusType.UNCERTAINTY
    attention_switch_threshold: float = 0.2  # Threshold for switching attention

    # Adaptation parameters
    adaptive_weights: bool = True       # Whether weights adapt based on success
    learning_rate: float = 0.01         # Rate for adapting weights

    # Integration with other modules
    use_episodic_memory: bool = True
    use_semantic_memory: bool = True
    use_working_memory: bool = True

    # Custom parameters
    custom_parameters: Dict[str, Any] = field(default_factory=dict)


class NoveltyDetector:
    """
    Detects novelty in observations based on distance to previous observations.
    Uses a nearest-neighbor approach for efficient novelty calculation.
    """

    def __init__(self, config: IntrinsicMotivationConfig):
        """
        Initialize novelty detector.

        Args:
            config: Configuration parameters
        """
        self.config = config
        self.embedding_memory = []
        self.timestamps = []
        self.k = config.novelty_k_neighbors
        self.threshold = config.novelty_threshold
        self.nn_model = None

        # Logging for debugging
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation.Novelty")

    def compute_novelty(self, observation: ObservationData) -> float:
        """
        Compute novelty score for an observation.

        Args:
            observation: The observation to evaluate

        Returns:
            Novelty score in [0, 1] range
        """
        # Ensure we have an embedding
        if observation.embedding is None:
            self.logger.warning("Cannot compute novelty for observation without embedding")
            return 0.5  # Default mid-range novelty

        # If memory is empty, this is maximally novel
        if not self.embedding_memory:
            self._add_to_memory(observation)
            return 1.0

        # Compute distance to existing memories
        if HAS_SKLEARN and len(self.embedding_memory) >= self.k:
            # Use efficient nearest neighbors for large memories
            novelty = self._compute_novelty_nn(observation.embedding)
        else:
            # Manual calculation for small memories
            novelty = self._compute_novelty_manual(observation.embedding)

        # Add to memory if novel or randomly (to ensure diversity)
        if novelty > self.threshold or random.random() < 0.1:
            self._add_to_memory(observation)

        return novelty

    def _compute_novelty_nn(self, embedding: np.ndarray) -> float:
        """
        Compute novelty using nearest neighbors algorithm.

        Args:
            embedding: Vector representation of the observation

        Returns:
            Novelty score in [0, 1] range
        """
        # Ensure embedding is 2D
        if embedding.ndim == 1:
            embedding = embedding.reshape(1, -1)

        # Initialize or update model if needed
        if self.nn_model is None or len(self.embedding_memory) > len(self.timestamps):
            self._update_nn_model()

        # Get distances to k nearest neighbors
        distances, _ = self.nn_model.kneighbors(embedding)

        # Compute mean distance to k nearest neighbors, normalized to [0, 1]
        mean_distance = np.mean(distances)

        # Apply sigmoid-like normalization
        novelty = 2.0 / (1.0 + np.exp(-5.0 * mean_distance)) - 1.0

        return float(min(1.0, max(0.0, novelty)))

    def _compute_novelty_manual(self, embedding: np.ndarray) -> float:
        """
        Compute novelty manually for small memory sizes.

        Args:
            embedding: Vector representation of the observation

        Returns:
            Novelty score in [0, 1] range
        """
        # Compute distances to all memories
        distances = []
        for mem_embedding in self.embedding_memory:
            # Euclidean distance
            distance = np.sqrt(np.sum((embedding - mem_embedding) ** 2))
            distances.append(distance)

        # Sort distances and take mean of k smallest
        distances.sort()
        k = min(self.k, len(distances))
        if k == 0:
            return 1.0

        mean_distance = np.mean(distances[:k])

        # Apply sigmoid-like normalization
        novelty = 2.0 / (1.0 + np.exp(-5.0 * mean_distance)) - 1.0

        return float(min(1.0, max(0.0, novelty)))

    def _add_to_memory(self, observation: ObservationData) -> None:
        """
        Add observation to memory.

        Args:
            observation: Observation to add
        """
        # Add embedding to memory
        self.embedding_memory.append(observation.embedding)
        self.timestamps.append(observation.timestamp)

        # Limit memory size
        if len(self.embedding_memory) > self.config.memory_capacity:
            # Remove oldest or randomly (with preference for low-novelty items)
            if random.random() < 0.7:  # 70% chance to remove oldest
                self.embedding_memory.pop(0)
                self.timestamps.pop(0)
            else:
                # Remove a random item, weighted by age (older more likely)
                now = time.time()
                ages = [now - ts for ts in self.timestamps]
                weights = [age for age in ages]
                if sum(weights) > 0:
                    weights = [w / sum(weights) for w in weights]
                    idx = random.choices(range(len(weights)), weights=weights)[0]
                else:
                    idx = random.randint(0, len(self.embedding_memory) - 1)

                self.embedding_memory.pop(idx)
                self.timestamps.pop(idx)

            # Clear model to force retraining
            self.nn_model = None

    def _update_nn_model(self) -> None:
        """Update the nearest neighbors model with current memory"""
        if not HAS_SKLEARN:
            return

        # Convert memory to numpy array
        memory_array = np.array(self.embedding_memory)

        # Create and fit model
        self.nn_model = NearestNeighbors(n_neighbors=min(self.k, len(memory_array)))
        self.nn_model.fit(memory_array)

    def reset(self) -> None:
        """Clear memory and reset the detector"""
        self.embedding_memory = []
        self.timestamps = []
        self.nn_model = None


class SurpriseDetector:
    """
    Detects surprise (prediction errors) in observations.
    Maintains prediction models and tracks their errors over time.
    """

    def __init__(self, config: IntrinsicMotivationConfig):
        """
        Initialize surprise detector.

        Args:
            config: Configuration parameters
        """
        self.config = config
        self.window_size = config.surprise_window_size
        self.min_count = config.surprise_min_count

        # History of observations and predictions
        self.observation_history = deque(maxlen=self.window_size)
        self.prediction_errors = deque(maxlen=self.window_size)

        # Simple prediction models
        self.prediction_models = {}

        # Logging
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation.Surprise")

    def compute_surprise(self, observation: ObservationData) -> float:
        """
        Compute surprise score for an observation.

        Args:
            observation: The observation to evaluate

        Returns:
            Surprise score in [0, 1] range
        """
        # Make prediction
        prediction = self._predict(observation)

        # Compute error between prediction and actual
        error = self._compute_prediction_error(observation, prediction)

        # Update history
        self.observation_history.append(observation)
        self.prediction_errors.append(error)

        # Update model
        self._update_model(observation)

        # Compute surprise based on how unusual this error is
        if len(self.prediction_errors) < self.min_count:
            # Not enough data yet, return moderate surprise
            surprise = 0.5
        else:
            # Compute how many standard deviations away from mean
            mean_error = np.mean(self.prediction_errors)
            std_error = np.std(self.prediction_errors) + 1e-5  # Avoid division by zero
            z_score = (error - mean_error) / std_error

            # Convert to [0, 1] range using sigmoid-like function
            surprise = 1.0 / (1.0 + np.exp(-z_score))

        return float(min(1.0, max(0.0, surprise)))

    def _predict(self, observation: ObservationData) -> Any:
        """
        Make a prediction for an observation.

        Args:
            observation: Observation to predict

        Returns:
            Prediction of observation content
        """
        # If we have no history, return the observation itself (no surprise)
        if not self.observation_history:
            return observation.content

        # Get context keys to use for prediction
        context_keys = set(observation.context.keys())

        # Find a suitable model based on context
        model_key = None
        for key in self.prediction_models:
            if set(key.split('|')) <= context_keys:
                model_key = key
                break

        if model_key is not None and self.prediction_models[model_key].get('count', 0) >= self.min_count:
            # Use existing model
            model = self.prediction_models[model_key]

            # Very simple prediction: most frequent value seen in this context
            counts = model.get('counts', {})
            if counts:
                prediction = max(counts.items(), key=lambda x: x[1])[0]
                return prediction

        # Fallback: use most recent observation
        return self.observation_history[-1].content

    def _compute_prediction_error(self, observation: ObservationData, prediction: Any) -> float:
        """
        Compute error between prediction and actual observation.

        Args:
            observation: Actual observation
            prediction: Predicted observation

        Returns:
            Error value
        """
        # Handle different types of content
        if observation.embedding is not None and hasattr(prediction, 'embedding'):
            # Use embedding distance if available
            error = np.sqrt(np.sum((observation.embedding - prediction.embedding) ** 2))
        elif isinstance(observation.content, (int, float)) and isinstance(prediction, (int, float)):
            # Numerical value
            error = abs(observation.content - prediction)
        elif isinstance(observation.content, str) and isinstance(prediction, str):
            # String
            error = 1.0 if observation.content != prediction else 0.0
        else:
            # Default: binary match/no-match
            error = 1.0 if observation.content != prediction else 0.0

        return error

    def _update_model(self, observation: ObservationData) -> None:
        """
        Update prediction model with new observation.

        Args:
            observation: New observation
        """
        # Create/update model for this context
        # Use relevant context keys
        context_keys = sorted(observation.context.keys())
        model_key = '|'.join(context_keys)

        if model_key not in self.prediction_models:
            self.prediction_models[model_key] = {
                'counts': {},
                'count': 0
            }

        model = self.prediction_models[model_key]

        # Update counts of observed values
        content_key = str(observation.content)
        if content_key not in model['counts']:
            model['counts'][content_key] = 0
        model['counts'][content_key] += 1
        model['count'] += 1

        # Limit number of models
        if len(self.prediction_models) > 100:  # Arbitrary limit
            # Remove model with fewest observations
            min_model = min(self.prediction_models.items(), key=lambda x: x[1]['count'])
            del self.prediction_models[min_model[0]]

    def reset(self) -> None:
        """Clear history and reset the detector"""
        self.observation_history.clear()
        self.prediction_errors.clear()
        self.prediction_models = {}


class UncertaintyEstimator:
    """
    Estimates epistemic uncertainty for observations.
    Uses ensemble or Bayesian methods for uncertainty quantification.
    """

    def __init__(self, config: IntrinsicMotivationConfig):
        """
        Initialize uncertainty estimator.

        Args:
            config: Configuration parameters
        """
        self.config = config
        self.ensemble_size = config.uncertainty_ensemble_size
        self.threshold = config.uncertainty_threshold

        # Knowledge coverage map (for symbolic approach)
        self.knowledge_map = {}

        # Ensemble models (for neural approach)
        self.ensemble_models = []

        # Logging
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation.Uncertainty")

    def compute_uncertainty(self, observation: ObservationData) -> float:
        """
        Compute uncertainty score for an observation.

        Args:
            observation: The observation to evaluate

        Returns:
            Uncertainty score in [0, 1] range
        """
        # If neural approach is available and observation has embedding
        if HAS_TORCH and observation.embedding is not None and self.ensemble_models:
            uncertainty = self._compute_neural_uncertainty(observation)
        else:
            # Fallback to symbolic approach
            uncertainty = self._compute_symbolic_uncertainty(observation)

        return uncertainty

    def _compute_neural_uncertainty(self, observation: ObservationData) -> float:
        """
        Compute uncertainty using neural ensemble.

        Args:
            observation: Observation to evaluate

        Returns:
            Uncertainty score in [0, 1] range
        """
        if not self.ensemble_models:
            # No models yet, maximum uncertainty
            return 1.0

        # Get predictions from all models
        predictions = []
        for model in self.ensemble_models:
            if HAS_TORCH:
                with torch.no_grad():
                    embedding = torch.FloatTensor(observation.embedding).unsqueeze(0)
                    pred = model(embedding).squeeze(0).numpy()
                    predictions.append(pred)

        if not predictions:
            return 1.0

        # Compute variance across models
        predictions = np.array(predictions)
        variance = np.mean(np.var(predictions, axis=0))

        # Normalize to [0, 1]
        uncertainty = 2.0 / (1.0 + np.exp(-5.0 * variance)) - 1.0

        return float(min(1.0, max(0.0, uncertainty)))

    def _compute_symbolic_uncertainty(self, observation: ObservationData) -> float:
        """
        Compute uncertainty using symbolic knowledge map.

        Args:
            observation: Observation to evaluate

        Returns:
            Uncertainty score in [0, 1] range
        """
        # Generate a key for this observation
        key = self._get_observation_key(observation)

        # Check if we have seen similar observations before
        if key in self.knowledge_map:
            count = self.knowledge_map[key]['count']

            # Uncertainty decreases with count, but never reaches zero
            uncertainty = 1.0 / (1.0 + count)

            # Update count
            self.knowledge_map[key]['count'] += 1
        else:
            # Never seen before, maximum uncertainty
            uncertainty = 1.0

            # Add to knowledge map
            self.knowledge_map[key] = {
                'count': 1,
                'first_seen': time.time()
            }

        return uncertainty

    def _get_observation_key(self, observation: ObservationData) -> str:
        """
        Generate a key for the observation to use in the knowledge map.

        Args:
            observation: Observation to generate key for

        Returns:
            String key
        """
        # Use context and content to generate key
        context_key = '|'.join(f"{k}:{v}" for k, v in sorted(observation.context.items()))

        # Handle different content types
        if isinstance(observation.content, (int, float)):
            # Discretize numerical value
            bin_size = 0.1
            binned = int(observation.content / bin_size) * bin_size
            content_key = f"num:{binned}"
        elif isinstance(observation.content, str):
            content_key = f"str:{observation.content}"
        else:
            content_key = f"obj:{type(observation.content).__name__}"

        return f"{context_key}|{content_key}"

    def update_models(self, observations: List[ObservationData]) -> None:
        """
        Update uncertainty models with new observations.

        Args:
            observations: List of observations to learn from
        """
        # Only update neural models if available
        if not HAS_TORCH or not observations:
            return

        # Ensure all observations have embeddings
        embeddings = [obs.embedding for obs in observations if obs.embedding is not None]
        if not embeddings:
            return

        # Create ensemble if not already created
        if not self.ensemble_models:
            self._create_ensemble(embeddings[0].shape[0])

        # Convert to tensors
        X = torch.FloatTensor(embeddings)

        # Simple "targets" - just predict the embedding itself
        y = X.clone()

        # Train each model slightly differently
        for i, model in enumerate(self.ensemble_models):
            # Add random noise to targets for diversity
            noisy_y = y + torch.randn_like(y) * 0.1 * (i + 1)

            # Simple training (in practice, would use proper training loop)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
            for _ in range(5):  # Just a few iterations for demonstration
                optimizer.zero_grad()
                pred = model(X)
                loss = F.mse_loss(pred, noisy_y)
                loss.backward()
                optimizer.step()

    def _create_ensemble(self, input_dim: int) -> None:
        """
        Create ensemble of models for uncertainty estimation.

        Args:
            input_dim: Dimension of input embeddings
        """
        if not HAS_TORCH:
            return

        # Clear any existing models
        self.ensemble_models = []

        # Create ensemble
        for _ in range(self.ensemble_size):
            # Simple MLP model
            model = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 64),
                nn.ReLU(),
                nn.Linear(64, input_dim)
            )

            # Initialize weights differently for each model
            for param in model.parameters():
                nn.init.normal_(param, mean=0.0, std=0.02)

            self.ensemble_models.append(model)

    def reset(self) -> None:
        """Clear knowledge and reset the estimator"""
        self.knowledge_map = {}
        self.ensemble_models = []


class CompetenceProgressTracker:
    """
    Tracks learning progress (competence) in different areas.
    Intrinsic motivation arises from improving performance.
    """

    def __init__(self, config: IntrinsicMotivationConfig):
        """
        Initialize competence progress tracker.

        Args:
            config: Configuration parameters
        """
        self.config = config
        self.window_size = config.competence_window_size
        self.smoothing = config.competence_smoothing

        # Track performance history for different skills/areas
        self.performance_history = {}

        # Logging
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation.Competence")

    def compute_competence_progress(self, observation: ObservationData,
                                   performance: float, skill_area: str) -> float:
        """
        Compute competence progress for a skill area.

        Args:
            observation: The context observation
            performance: Performance measure in [0, 1]
            skill_area: Identifier for the skill/area

        Returns:
            Competence progress score in [0, 1] range
        """
        # Initialize history for this skill if needed
        if skill_area not in self.performance_history:
            self.performance_history[skill_area] = {
                'values': deque(maxlen=self.window_size),
                'timestamps': deque(maxlen=self.window_size),
                'smoothed': 0.0,
                'previous_smoothed': 0.0
            }

        history = self.performance_history[skill_area]

        # Update history
        history['values'].append(performance)
        history['timestamps'].append(observation.timestamp)

        # Update smoothed average
        if not history['values']:
            history['smoothed'] = performance
        else:
            # Exponential moving average
            history['previous_smoothed'] = history['smoothed']
            history['smoothed'] = (1 - self.smoothing) * history['smoothed'] + self.smoothing * performance

        # Compute progress (derivative of performance)
        if len(history['values']) < 2:
            # Not enough data yet
            progress = 0.5  # Neutral
        else:
            # Compare current smoothed to previous
            progress_raw = history['smoothed'] - history['previous_smoothed']

            # Normalize to [-1, 1], then to [0, 1]
            progress = (np.tanh(5.0 * progress_raw) + 1.0) / 2.0

        return float(progress)

    def get_high_progress_areas(self, threshold: float = 0.7) -> List[Tuple[str, float]]:
        """
        Get skill areas with high learning progress.

        Args:
            threshold: Minimum progress to consider high

        Returns:
            List of (skill_area, progress) tuples
        """
        high_progress = []

        for skill_area, history in self.performance_history.items():
            if len(history['values']) < 2:
                continue

            # Compute current progress
            progress_raw = history['smoothed'] - history['previous_smoothed']
            progress = (np.tanh(5.0 * progress_raw) + 1.0) / 2.0

            if progress >= threshold:
                high_progress.append((skill_area, float(progress)))

        # Sort by progress (highest first)
        high_progress.sort(key=lambda x: x[1], reverse=True)

        return high_progress

    def get_low_progress_areas(self, threshold: float = 0.3) -> List[Tuple[str, float]]:
        """
        Get skill areas with low learning progress (plateaued).

        Args:
            threshold: Maximum progress to consider low

        Returns:
            List of (skill_area, progress) tuples
        """
        low_progress = []

        for skill_area, history in self.performance_history.items():
            if len(history['values']) < self.window_size // 2:
                # Not enough data yet
                continue

            # Compute current progress
            progress_raw = history['smoothed'] - history['previous_smoothed']
            progress = (np.tanh(5.0 * progress_raw) + 1.0) / 2.0

            if progress <= threshold:
                low_progress.append((skill_area, float(progress)))

        # Sort by progress (lowest first)
        low_progress.sort(key=lambda x: x[1])

        return low_progress

    def reset(self) -> None:
        """Clear history and reset the tracker"""
        self.performance_history = {}


class EmpowermentEstimator:
    """
    Estimates empowerment (information-theoretic measure of control/options).
    This is a simplified implementation of empowerment principles.
    """

    def __init__(self, config: IntrinsicMotivationConfig):
        """
        Initialize empowerment estimator.

        Args:
            config: Configuration parameters
        """
        self.config = config
        self.horizon = config.empowerment_horizon
        self.samples = config.empowerment_samples

        # Model of state transitions (for empowerment simulation)
        self.transition_model = {}

        # Logging
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation.Empowerment")

    def compute_empowerment(self, observation: ObservationData,
                           available_actions: List[Any]) -> float:
        """
        Compute empowerment score for an observation.

        Args:
            observation: The observation to evaluate
            available_actions: List of actions available in this state

        Returns:
            Empowerment score in [0, 1] range
        """
        # If no actions, no empowerment
        if not available_actions:
            return 0.0

        # First-order empowerment: simply number of available actions
        action_count = len(available_actions)
        first_order = min(1.0, action_count / 10.0)  # Normalize to [0, 1]

        # If we have a transition model, estimate higher-order empowerment
        state_key = self._get_state_key(observation)
        if state_key in self.transition_model and self.horizon > 1:
            # Sample action sequences and count unique outcomes
            unique_outcomes = set()

            for _ in range(self.samples):
                # Sample a random action sequence
                current_state = state_key
                for _ in range(self.horizon):
                    if current_state not in self.transition_model:
                        break

                    transitions = self.transition_model[current_state]
                    if not transitions:
                        break

                    # Sample a random action
                    action = random.choice(list(transitions.keys()))

                    # Sample next state from possible outcomes
                    next_states = transitions[action]
                    if not next_states:
                        break

                    # Weight by probability
                    total = sum(prob for _, prob in next_states)
                    if total <= 0:
                        break

                    rand_val = random.random() * total
                    cumulative = 0
                    next_state = None

                    for state, prob in next_states:
                        cumulative += prob
                        if cumulative >= rand_val:
                            next_state = state
                            break

                    if next_state is None:
                        break

                    current_state = next_state

                # Add final state to set of unique outcomes
                unique_outcomes.add(current_state)

            # Empowerment is log of number of unique reachable states
            if unique_outcomes:
                # Normalize to [0, 1] assuming maximum of 2^horizon states
                higher_order = min(1.0, len(unique_outcomes) / (2.0 ** self.horizon))

                # Combine first and higher order
                empowerment = 0.3 * first_order + 0.7 * higher_order
            else:
                empowerment = first_order
        else:
            empowerment = first_order

        return empowerment

    def update_model(self, current_state: ObservationData,
                    action: Any, next_state: ObservationData) -> None:
        """
        Update transition model with observed transition.

        Args:
            current_state: Starting state
            action: Action taken
            next_state: Resulting state
        """
        # Get state keys
        current_key = self._get_state_key(current_state)
        next_key = self._get_state_key(next_state)

        # Initialize model for this state if needed
        if current_key not in self.transition_model:
            self.transition_model[current_key] = {}

        # Initialize actions if needed
        action_key = str(action)
        if action_key not in self.transition_model[current_key]:
            self.transition_model[current_key][action_key] = []

        # Check if we've seen this transition before
        for i, (state, count) in enumerate(self.transition_model[current_key][action_key]):
            if state == next_key:
                # Increment count
                self.transition_model[current_key][action_key][i] = (state, count + 1)
                break
        else:
            # New transition
            self.transition_model[current_key][action_key].append((next_key, 1))

        # Limit model size
        if len(self.transition_model) > 1000:  # Arbitrary limit
            # Remove least recently used states
            # In a real implementation, would track usage timestamps
            keys = list(self.transition_model.keys())
            to_remove = keys[:len(keys)//5]  # Remove 20%
            for key in to_remove:
                del self.transition_model[key]

    def _get_state_key(self, observation: ObservationData) -> str:
        """
        Generate a key for the state.

        Args:
            observation: Observation to generate key for

        Returns:
            String key
        """
        # If embedding available, use a discretized version
        if observation.embedding is not None:
            # Simple discretization: round each dimension to 1 decimal place
            discretized = np.round(observation.embedding * 10) / 10
            key = str(discretized.tolist())
        else:
            # Use content and context
            context_str = '|'.join(f"{k}:{v}" for k, v in sorted(observation.context.items()))
            key = f"{observation.content}|{context_str}"

        return key

    def reset(self) -> None:
        """Clear model and reset the estimator"""
        self.transition_model = {}


class IntrinsicMotivation:
    """
    Main module for intrinsic motivation in GrandNexus.

    Coordinates different intrinsic motivation mechanisms to generate
    curiosity signals that drive exploration and learning.
    """

    def __init__(self, config: Optional[IntrinsicMotivationConfig] = None,
                nexus_core: Optional[NexusCore] = None,
                episodic_memory: Optional[EpisodicMemory] = None,
                working_memory: Optional[WorkingMemory] = None,
                semantic_memory: Optional[SemanticMemory] = None,
                learning_core: Optional[LearningCore] = None):
        """
        Initialize the intrinsic motivation module.

        Args:
            config: Configuration for the module
            nexus_core: Reference to the NexusCore
            episodic_memory: Reference to episodic memory
            working_memory: Reference to working memory
            semantic_memory: Reference to semantic memory
            learning_core: Reference to learning core
        """
        self.logger = logging.getLogger("GrandNexus.Learning.IntrinsicMotivation")

        # Store configuration
        self.config = config or IntrinsicMotivationConfig()

        # Store module references
        self.nexus_core = nexus_core
        self.episodic_memory = episodic_memory
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.learning_core = learning_core

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.lock = threading.RLock()
        self.initialized = False
        self.running = False

        # Current curiosity state
        self.curiosity_state = CuriosityState()

        # Curiosity mechanisms
        self.novelty_detector = None
        self.surprise_detector = None
        self.uncertainty_estimator = None
        self.competence_tracker = None
        self.empowerment_estimator = None

        # Motivation history
        self.recent_motivations = deque(maxlen=100)

        # Weights for different motivation types
        self.motivation_weights = dict(self.config.default_weights)

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"IntrinsicMotivation initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """
        Initialize the intrinsic motivation module.

        Returns:
            True if initialization succeeded, False otherwise
        """
        if self.initialized:
            return True

        with self.lock:
            try:
                # Initialize motivation mechanisms
                self._initialize_motivation_mechanisms()

                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("IntrinsicMotivation initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"IntrinsicMotivation initialization failed: {str(e)}")
                return False

    def process_observation(self, observation: ObservationData) -> List[MotivationSignal]:
        """
        Process an observation and generate intrinsic motivation signals.

        Args:
            observation: The observation to evaluate

        Returns:
            List of motivation signals
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            # Generate embeddings if needed
            if observation.embedding is None:
                observation = self._generate_embedding(observation)

            # Generate motivation signals
            signals = []

            # Novelty
            if MotivationType.NOVELTY in self.config.enabled_types:
                novelty = self.novelty_detector.compute_novelty(observation)
                if novelty > 0.3:  # Threshold to generate signal
                    signals.append(MotivationSignal(
                        observation=observation,
                        motivation_type=MotivationType.NOVELTY,
                        strength=novelty,
                        explanation=f"Novel observation (novelty={novelty:.2f})"
                    ))
                self.metrics["novelty"].append(novelty)

            # Surprise
            if MotivationType.SURPRISE in self.config.enabled_types:
                surprise = self.surprise_detector.compute_surprise(observation)
                if surprise > 0.3:  # Threshold to generate signal
                    signals.append(MotivationSignal(
                        observation=observation,
                        motivation_type=MotivationType.SURPRISE,
                        strength=surprise,
                        explanation=f"Surprising observation (surprise={surprise:.2f})"
                    ))
                self.metrics["surprise"].append(surprise)

            # Uncertainty
            if MotivationType.UNCERTAINTY in self.config.enabled_types:
                uncertainty = self.uncertainty_estimator.compute_uncertainty(observation)
                if uncertainty > 0.3:  # Threshold to generate signal
                    signals.append(MotivationSignal(
                        observation=observation,
                        motivation_type=MotivationType.UNCERTAINTY,
                        strength=uncertainty,
                        explanation=f"Uncertain knowledge (uncertainty={uncertainty:.2f})"
                    ))
                self.metrics["uncertainty"].append(uncertainty)

            # Store in memory (both intrinsic and working memory)
            self._store_in_memory(observation, signals)

            # Update curiosity state based on signals
            self._update_curiosity_state(signals)

            # Add to history
            for signal in signals:
                self.recent_motivations.append(signal)

            # Update metrics
            self.metrics["signal_count"].append(len(signals))

            # Log strong signals
            for signal in signals:
                if signal.strength > 0.7:
                    self.logger.info(f"Strong {signal.motivation_type.name} signal: {signal.explanation}")

            return signals

    def update_competence(self, skill_area: str, performance: float,
                         observation: ObservationData) -> Optional[MotivationSignal]:
        """
        Update competence tracking for a skill area and generate motivation signal.

        Args:
            skill_area: Identifier for the skill/area
            performance: Performance measure in [0, 1]
            observation: Contextual observation

        Returns:
            Motivation signal if significant progress detected, None otherwise
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            if MotivationType.COMPETENCE not in self.config.enabled_types:
                return None

            # Compute competence progress
            progress = self.competence_tracker.compute_competence_progress(
                observation, performance, skill_area)

            self.metrics["competence_progress"].append(progress)

            # Generate signal if significant progress
            if progress > 0.6:  # Threshold for significant progress
                signal = MotivationSignal(
                    observation=observation,
                    motivation_type=MotivationType.COMPETENCE,
                    strength=progress,
                    explanation=f"Learning progress in {skill_area} (progress={progress:.2f})",
                    target=skill_area
                )

                self.recent_motivations.append(signal)
                return signal

            return None

    def update_empowerment(self, observation: ObservationData,
                          actions: List[Any], next_state: Optional[ObservationData] = None,
                          action_taken: Optional[Any] = None) -> Optional[MotivationSignal]:
        """
        Update empowerment model and generate motivation signal.

        Args:
            observation: Current state observation
            actions: Available actions in this state
            next_state: Optional next state (if action was taken)
            action_taken: Optional action that was taken

        Returns:
            Motivation signal if high empowerment detected, None otherwise
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            if MotivationType.EMPOWERMENT not in self.config.enabled_types:
                return None

            # Update transition model if action was taken
            if next_state is not None and action_taken is not None:
                self.empowerment_estimator.update_model(
                    observation, action_taken, next_state)

            # Compute empowerment
            empowerment = self.empowerment_estimator.compute_empowerment(
                observation, actions)

            self.metrics["empowerment"].append(empowerment)

            # Generate signal if high empowerment
            if empowerment > 0.7:  # Threshold for high empowerment
                signal = MotivationSignal(
                    observation=observation,
                    motivation_type=MotivationType.EMPOWERMENT,
                    strength=empowerment,
                    explanation=f"High control/options (empowerment={empowerment:.2f})"
                )

                self.recent_motivations.append(signal)
                return signal

            return None

    def get_curiosity_state(self) -> CuriosityState:
        """
        Get the current curiosity state.

        Returns:
            Current curiosity state
        """
        with self.lock:
            return copy.deepcopy(self.curiosity_state)

    def get_attention_focus(self) -> Tuple[Any, float]:
        """
        Get the current attention focus and its strength.

        Returns:
            Tuple of (focus_target, strength)
        """
        with self.lock:
            return (self.curiosity_state.attention_focus,
                    self.curiosity_state.focus_strength)

    def get_motivation_weights(self) -> Dict[MotivationType, float]:
        """
        Get the current weights for different motivation types.

        Returns:
            Dictionary mapping motivation types to weights
        """
        with self.lock:
            return dict(self.motivation_weights)

    def set_motivation_weights(self, weights: Dict[MotivationType, float]) -> None:
        """
        Set weights for different motivation types.

        Args:
            weights: Dictionary mapping motivation types to weights
        """
        with self.lock:
            for motivation_type, weight in weights.items():
                if motivation_type in self.motivation_weights:
                    self.motivation_weights[motivation_type] = max(0.0, min(1.0, weight))

    def reset(self) -> None:
        """
        Reset all intrinsic motivation mechanisms.

        This clears all history and model state, but keeps configuration.
        """
        with self.lock:
            # Reset mechanisms
            if self.novelty_detector:
                self.novelty_detector.reset()

            if self.surprise_detector:
                self.surprise_detector.reset()

            if self.uncertainty_estimator:
                self.uncertainty_estimator.reset()

            if self.competence_tracker:
                self.competence_tracker.reset()

            if self.empowerment_estimator:
                self.empowerment_estimator.reset()

            # Reset state
            self.curiosity_state = CuriosityState()
            self.recent_motivations.clear()

            # Reset metrics
            for key in list(self.metrics.keys()):
                self.metrics[key].clear()

            self.logger.info("IntrinsicMotivation reset complete")

    def get_recent_motivations(self) -> List[MotivationSignal]:
        """
        Get recent motivation signals.

        Returns:
            List of recent motivation signals
        """
        with self.lock:
            return list(self.recent_motivations)

    def get_motivation_metrics(self) -> Dict[str, Any]:
        """
        Get metrics for intrinsic motivation performance.

        Returns:
            Dictionary with metrics
        """
        with self.lock:
            metrics = {}

            # Process metric queues
            for key, values in self.metrics.items():
                if values:
                    metrics[key] = {
                        "current": values[-1],
                        "mean": float(np.mean(values)),
                        "max": float(np.max(values)),
                        "min": float(np.min(values))
                    }

            # Add current state info
            metrics["curiosity_state"] = {
                "interest_level": self.curiosity_state.interest_level,
                "focus_strength": self.curiosity_state.focus_strength,
                "novelty_seeking": self.curiosity_state.novelty_seeking,
                "exploration_rate": self.curiosity_state.exploration_rate,
                "active_drives": [drive.name for drive in self.curiosity_state.active_drives]
            }

            # Add weights
            metrics["motivation_weights"] = {k.name: v for k, v in self.motivation_weights.items()}

            return metrics

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _initialize_motivation_mechanisms(self) -> None:
        """Initialize the various intrinsic motivation mechanisms"""
        # Novelty detection
        self.novelty_detector = NoveltyDetector(self.config)

        # Surprise detection
        self.surprise_detector = SurpriseDetector(self.config)

        # Uncertainty estimation
        self.uncertainty_estimator = UncertaintyEstimator(self.config)

        # Competence tracking
        self.competence_tracker = CompetenceProgressTracker(self.config)

        # Empowerment estimation
        self.empowerment_estimator = EmpowermentEstimator(self.config)

        self.logger.info("Intrinsic motivation mechanisms initialized")

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.learning_core:
                dependencies.append("learning_core")

            self.nexus_core.register_module(
                name="intrinsic_motivation",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _generate_embedding(self, observation: ObservationData) -> ObservationData:
        """
        Generate embedding for an observation if not already present.

        Args:
            observation: Observation to generate embedding for

        Returns:
            Observation with embedding added (if possible)
        """
        # If already has embedding, return as is
        if observation.embedding is not None:
            return observation

        # Try to generate embedding
        # This would typically use a neural network or other embedding model
        # For simplicity, we'll just create a random embedding if we can't do better

        # Check if semantic memory can provide an embedding
        if self.semantic_memory is not None and self.config.use_semantic_memory:
            # Try to get embedding from semantic memory
            # This is a placeholder - actual implementation would depend on SemanticMemory API
            try:
                if isinstance(observation.content, str):
                    concept = self.semantic_memory.get_concept(observation.content)
                    if concept and hasattr(concept, 'embedding') and concept.embedding is not None:
                        new_observation = copy.copy(observation)
                        new_observation.embedding = concept.embedding
                        return new_observation
            except Exception as e:
                self.logger.warning(f"Error getting embedding from semantic memory: {str(e)}")

        # Fallback: create simple random embedding (for demonstration only)
        # In a real system, would use a proper embedding model
        dim = self.config.embedding_dimension
        embedding = np.random.randn(dim)
        embedding = embedding / np.linalg.norm(embedding)  # Normalize

        # Create new observation with embedding
        new_observation = copy.copy(observation)
        new_observation.embedding = embedding

        return new_observation

    def _store_in_memory(self, observation: ObservationData,
                        signals: List[MotivationSignal]) -> None:
        """
        Store observation and signals in memory systems.

        Args:
            observation: Observation to store
            signals: Motivation signals generated
        """
        # Store in working memory if available
        if self.working_memory is not None and self.config.use_working_memory:
            try:
                # Store the observation
                item_id = self.working_memory.store_item(
                    content=observation.content,
                    source="intrinsic_motivation",
                    metadata={
                        "context": observation.context,
                        "signals": [
                            {
                                "type": signal.motivation_type.name,
                                "strength": signal.strength,
                                "explanation": signal.explanation
                            }
                            for signal in signals
                        ]
                    },
                    importance=max([0.5] + [signal.strength for signal in signals])
                )
            except Exception as e:
                self.logger.warning(f"Error storing in working memory: {str(e)}")

        # Store in episodic memory if available and signals are significant
        if (self.episodic_memory is not None and self.config.use_episodic_memory and
            any(signal.strength > 0.6 for signal in signals)):
            try:
                # Create an episodic memory event
                event_id = self.episodic_memory.add_event(
                    episode_id="intrinsic_motivation",  # This should be a real episode ID
                    content={
                        "observation": observation.content,
                        "signals": [
                            {
                                "type": signal.motivation_type.name,
                                "strength": signal.strength,
                                "explanation": signal.explanation
                            }
                            for signal in signals
                        ]
                    },
                    metadata={
                        "source": "intrinsic_motivation",
                        "context": observation.context,
                        "importance": max([0.5] + [signal.strength for signal in signals])
                    }
                )
            except Exception as e:
                self.logger.warning(f"Error storing in episodic memory: {str(e)}")

    def _update_curiosity_state(self, signals: List[MotivationSignal]) -> None:
        """
        Update the curiosity state based on new motivation signals.

        Args:
            signals: New motivation signals
        """
        # Default decay of current state
        self.curiosity_state.interest_level *= self.config.attention_decay
        self.curiosity_state.focus_strength *= self.config.attention_decay

        # If no signals, gradually return to baseline
        if not signals:
            baseline = 0.3
            self.curiosity_state.interest_level = (
                0.9 * self.curiosity_state.interest_level + 0.1 * baseline
            )
            return

        # Combine signals according to current weights
        total_strength = 0.0
        weighted_strength = 0.0
        active_drives = set()

        for signal in signals:
            weight = self.motivation_weights.get(signal.motivation_type, 1.0)
            weighted_strength += signal.strength * weight
            total_strength += signal.strength
            active_drives.add(signal.motivation_type)

        # Update interest level (with smoothing)
        if total_strength > 0:
            avg_weighted_strength = weighted_strength / total_strength
            self.curiosity_state.interest_level = (
                0.7 * self.curiosity_state.interest_level +
                0.3 * avg_weighted_strength
            )

        # Update active drives
        self.curiosity_state.active_drives = list(active_drives)

        # Update novelty seeking based on novelty signals
        novelty_signals = [s for s in signals if s.motivation_type == MotivationType.NOVELTY]
        if novelty_signals:
            avg_novelty = sum(s.strength for s in novelty_signals) / len(novelty_signals)
            self.curiosity_state.novelty_seeking = (
                0.8 * self.curiosity_state.novelty_seeking + 0.2 * avg_novelty
            )

        # Update exploration rate based on multiple factors
        # Higher for novelty and uncertainty, lower for competence
        exploration_factors = []

        for signal in signals:
            if signal.motivation_type in [MotivationType.NOVELTY, MotivationType.UNCERTAINTY]:
                exploration_factors.append(signal.strength)
            elif signal.motivation_type == MotivationType.COMPETENCE:
                exploration_factors.append(0.3)  # Lower when making progress (exploiting)

        if exploration_factors:
            avg_factor = sum(exploration_factors) / len(exploration_factors)
            self.curiosity_state.exploration_rate = (
                0.8 * self.curiosity_state.exploration_rate + 0.2 * avg_factor
            )

        # Update attention focus
        self._update_attention_focus(signals)

        # Update timestamp
        self.curiosity_state.timestamp = time.time()

    def _update_attention_focus(self, signals: List[MotivationSignal]) -> None:
        """
        Update the attention focus based on signals.

        Args:
            signals: New motivation signals
        """
        # If no signals, maintain current focus with decay
        if not signals:
            return

        # Find the strongest signal
        strongest = max(signals, key=lambda s: s.strength)

        # If much stronger than current focus, switch
        if (strongest.strength > self.curiosity_state.focus_strength +
            self.config.attention_switch_threshold):
            # Switch focus
            self.curiosity_state.attention_focus = strongest.target or strongest.observation.content
            self.curiosity_state.focus_strength = strongest.strength

            self.logger.info(f"Attention focus switched to {self.curiosity_state.attention_focus} "
                            f"({strongest.motivation_type.name}, "
                            f"strength={strongest.strength:.2f})")
        else:
            # Slightly update focus strength
            self.curiosity_state.focus_strength = (
                0.8 * self.curiosity_state.focus_strength + 0.2 * strongest.strength
            )

    def _adapt_weights(self, feedback: float,
                      motivation_type: MotivationType) -> None:
        """
        Adapt weights based on feedback.

        Args:
            feedback: Feedback value (-1 to 1)
            motivation_type: Type of motivation that led to the action
        """
        if not self.config.adaptive_weights:
            return

        # Update weight for this motivation type
        if motivation_type in self.motivation_weights:
            current = self.motivation_weights[motivation_type]
            # Adjust weight (bounded)
            new_weight = current + self.config.learning_rate * feedback
            self.motivation_weights[motivation_type] = max(0.1, min(1.0, new_weight))

            # Log significant changes
            if abs(new_weight - current) > 0.1:
                self.logger.info(f"Adapted weight for {motivation_type.name}: "
                                f"{current:.2f} -> {self.motivation_weights[motivation_type]:.2f}")



ModuleNotFoundError: No module named 'grandnexus'

# curriculum

In [ ]:
"""
curriculum.py - Curriculum Learning Module for GrandNexus

This module manages structured learning sequences by organizing tasks based on
their complexity, prerequisites, and learning objectives. It enables GrandNexus
to learn in a progressive manner, mastering foundational skills before tackling
more complex ones, similar to how educational curricula are designed.

Key features:
- Task difficulty estimation and progression management
- Prerequisite relationship tracking between skills/concepts
- Adaptive task selection based on current performance
- Integration with memory systems for skill mastery tracking
- Learning path generation for specific objectives
- Dynamic curriculum adjustment based on learning outcomes
"""

import logging
import threading
import time
import uuid
import random
import math
from enum import Enum, auto
from typing import Dict, List, Any, Optional, Callable, Tuple, Union, Set
from dataclasses import dataclass, field
from collections import deque, defaultdict, Counter
import copy
import heapq
import numpy as np

# Assuming these exist in the architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.learning.learning_core import LearningCore
from grandnexus.learning.intrinsic_motivation import IntrinsicMotivation

# Task status enum
class TaskStatus(Enum):
    """Status of a learning task in the curriculum"""
    NOT_STARTED = auto()    # Task has not been started yet
    IN_PROGRESS = auto()    # Task is currently being learned
    COMPLETED = auto()      # Task has been successfully completed
    MASTERED = auto()       # Task has been mastered (high proficiency)
    FAILED = auto()         # Task was attempted but not completed successfully
    SKIPPED = auto()        # Task was skipped (e.g., too difficult, not relevant)
    EXPIRED = auto()        # Task is no longer relevant

# Difficulty level enum
class DifficultyLevel(Enum):
    """Difficulty levels for tasks"""
    TRIVIAL = 0
    EASY = 1
    MEDIUM = 2
    HARD = 3
    VERY_HARD = 4
    EXPERT = 5

@dataclass
class LearningTask:
    """
    Represents a specific learning task within a curriculum.
    Tasks are the building blocks of curricula.
    """
    task_id: str
    name: str
    description: str = ""
    difficulty: DifficultyLevel = DifficultyLevel.MEDIUM
    prerequisites: List[str] = field(default_factory=list)  # List of prerequisite task IDs
    estimated_time: float = 0.0  # Estimated time to complete in minutes
    status: TaskStatus = TaskStatus.NOT_STARTED
    proficiency: float = 0.0  # Current proficiency level (0.0 to 1.0)
    attempts: int = 0  # Number of attempts made
    last_attempted: Optional[float] = None  # Timestamp of last attempt
    metadata: Dict[str, Any] = field(default_factory=dict)
    # Function to execute this task (if executable)
    task_function: Optional[Callable] = None

@dataclass
class LearningUnit:
    """
    A group of related learning tasks that form a cohesive unit.
    Similar to a course module or chapter.
    """
    unit_id: str
    name: str
    description: str = ""
    tasks: List[str] = field(default_factory=list)  # List of task IDs
    dependencies: List[str] = field(default_factory=list)  # List of unit IDs this depends on
    completion_threshold: float = 0.8  # Proportion of tasks needed to consider unit complete
    progress: float = 0.0  # Current progress (0.0 to 1.0)
    status: TaskStatus = TaskStatus.NOT_STARTED
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Curriculum:
    """
    A structured learning program composed of learning units and tasks.
    Defines a complete learning path for a subject or skill area.
    """
    curriculum_id: str
    name: str
    description: str = ""
    units: List[str] = field(default_factory=list)  # List of unit IDs
    learning_objectives: List[str] = field(default_factory=list)
    target_audience: str = ""
    estimated_duration: float = 0.0  # Total estimated time in hours
    progress: float = 0.0  # Overall curriculum progress (0.0 to 1.0)
    status: TaskStatus = TaskStatus.NOT_STARTED
    creation_date: float = field(default_factory=time.time)
    last_modified: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class PerformanceRecord:
    """
    Records performance on a specific task attempt.
    Used to track learning progress over time.
    """
    record_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    task_id: str = ""
    timestamp: float = field(default_factory=time.time)
    score: float = 0.0  # Performance score (0.0 to 1.0)
    time_spent: float = 0.0  # Time spent in minutes
    mistakes: int = 0  # Number of mistakes or errors
    completed: bool = False  # Whether the task was completed
    feedback: str = ""  # Feedback on performance
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class CurriculumConfig:
    """Configuration settings for the curriculum module"""
    # Mastery thresholds
    mastery_threshold: float = 0.9  # Score needed to consider a task mastered
    proficiency_decay_rate: float = 0.005  # Daily decay rate for unused skills

    # Task selection parameters
    difficulty_weight: float = 0.3  # How much difficulty influences task selection
    novelty_weight: float = 0.2  # Preference for tasks not attempted recently
    relevance_weight: float = 0.4  # Importance of task to current goals

    # Adaptive curriculum parameters
    adaptation_enabled: bool = True  # Whether to adapt curriculum based on performance
    max_consecutive_failures: int = 3  # Maximum failures before suggesting easier tasks
    review_interval_days: int = 14  # Days before suggesting review of previously mastered tasks

    # Integration parameters
    use_intrinsic_motivation: bool = True  # Use intrinsic motivation for task selection
    use_episodic_memory: bool = True  # Record performance in episodic memory
    use_semantic_memory: bool = True  # Store curriculum structure in semantic memory

    # Custom parameters
    custom_parameters: Dict[str, Any] = field(default_factory=dict)

class CurriculumManager:
    """
    Core class for managing learning curricula within GrandNexus.

    This module organizes learning into structured sequences, tracks progress,
    and selects appropriate tasks based on the system's current capabilities
    and learning goals.
    """

    def __init__(self, config: Optional[CurriculumConfig] = None,
                nexus_core: Optional[NexusCore] = None,
                episodic_memory: Optional[EpisodicMemory] = None,
                working_memory: Optional[WorkingMemory] = None,
                semantic_memory: Optional[SemanticMemory] = None,
                learning_core: Optional[LearningCore] = None,
                intrinsic_motivation: Optional[IntrinsicMotivation] = None):
        """
        Initialize the curriculum manager.

        Args:
            config: Configuration settings
            nexus_core: Reference to NexusCore
            episodic_memory: Reference to episodic memory
            working_memory: Reference to working memory
            semantic_memory: Reference to semantic memory
            learning_core: Reference to learning core
            intrinsic_motivation: Reference to intrinsic motivation module
        """
        self.logger = logging.getLogger("GrandNexus.Learning.Curriculum")

        # Store configuration
        self.config = config or CurriculumConfig()

        # Store module references
        self.nexus_core = nexus_core
        self.episodic_memory = episodic_memory
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.learning_core = learning_core
        self.intrinsic_motivation = intrinsic_motivation

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.lock = threading.RLock()
        self.initialized = False
        self.running = False

        # Storage for curricula, units, tasks, and performance records
        self.curricula = {}
        self.learning_units = {}
        self.learning_tasks = {}
        self.performance_records = {}

        # Current learning state
        self.active_curriculum = None
        self.active_unit = None
        self.active_task = None
        self.current_focus = []  # List of tasks currently in focus

        # Skill mastery tracking
        self.skill_mastery = {}  # Mapping of skill IDs to mastery levels

        # Learning path generation
        self.learning_paths = {}  # Stored learning paths for different objectives

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"CurriculumManager initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """
        Initialize the curriculum manager.

        Returns:
            True if initialization was successful, False otherwise
        """
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Load any persistent curricula
                self._load_from_storage()

                self.initialized = True
                self.logger.info("CurriculumManager initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"CurriculumManager initialization failed: {str(e)}")
                return False

    def create_curriculum(self, name: str, description: str,
                         learning_objectives: List[str],
                         metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new curriculum.

        Args:
            name: Name of the curriculum
            description: Description of what this curriculum teaches
            learning_objectives: List of learning objectives
            metadata: Additional information about the curriculum

        Returns:
            The ID of the created curriculum
        """
        with self.lock:
            # Generate curriculum ID
            curriculum_id = f"curriculum_{str(uuid.uuid4())[:8]}"

            # Create curriculum
            curriculum = Curriculum(
                curriculum_id=curriculum_id,
                name=name,
                description=description,
                learning_objectives=learning_objectives,
                metadata=metadata or {}
            )

            # Store curriculum
            self.curricula[curriculum_id] = curriculum

            self.logger.info(f"Created curriculum: {name} ({curriculum_id})")
            return curriculum_id

    def create_learning_unit(self, curriculum_id: str, name: str,
                           description: str, dependencies: Optional[List[str]] = None,
                           metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new learning unit within a curriculum.

        Args:
            curriculum_id: ID of the curriculum this unit belongs to
            name: Name of the learning unit
            description: Description of the unit's content
            dependencies: IDs of other units this one depends on
            metadata: Additional information about the unit

        Returns:
            The ID of the created learning unit
        """
        with self.lock:
            if curriculum_id not in self.curricula:
                raise ValueError(f"Curriculum {curriculum_id} not found")

            # Generate unit ID
            unit_id = f"unit_{str(uuid.uuid4())[:8]}"

            # Create learning unit
            unit = LearningUnit(
                unit_id=unit_id,
                name=name,
                description=description,
                dependencies=dependencies or [],
                metadata=metadata or {}
            )

            # Store unit
            self.learning_units[unit_id] = unit

            # Add to curriculum
            self.curricula[curriculum_id].units.append(unit_id)
            self.curricula[curriculum_id].last_modified = time.time()

            self.logger.info(f"Created learning unit: {name} ({unit_id}) in curriculum {curriculum_id}")
            return unit_id

    def create_learning_task(self, unit_id: str, name: str,
                           description: str, difficulty: DifficultyLevel,
                           prerequisites: Optional[List[str]] = None,
                           estimated_time: float = 30.0,
                           task_function: Optional[Callable] = None,
                           metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Create a new learning task within a unit.

        Args:
            unit_id: ID of the unit this task belongs to
            name: Name of the task
            description: Description of what the task involves
            difficulty: Difficulty level of the task
            prerequisites: IDs of tasks that should be completed before this one
            estimated_time: Estimated time to complete in minutes
            task_function: Optional function to execute this task
            metadata: Additional information about the task

        Returns:
            The ID of the created task
        """
        with self.lock:
            if unit_id not in self.learning_units:
                raise ValueError(f"Learning unit {unit_id} not found")

            # Generate task ID
            task_id = f"task_{str(uuid.uuid4())[:8]}"

            # Create learning task
            task = LearningTask(
                task_id=task_id,
                name=name,
                description=description,
                difficulty=difficulty,
                prerequisites=prerequisites or [],
                estimated_time=estimated_time,
                task_function=task_function,
                metadata=metadata or {}
            )

            # Store task
            self.learning_tasks[task_id] = task

            # Add to unit
            self.learning_units[unit_id].tasks.append(task_id)

            self.logger.info(f"Created learning task: {name} ({task_id}) in unit {unit_id}")
            return task_id

    def get_curriculum(self, curriculum_id: str) -> Optional[Curriculum]:
        """
        Get a curriculum by ID.

        Args:
            curriculum_id: ID of the curriculum to retrieve

        Returns:
            The curriculum if found, None otherwise
        """
        with self.lock:
            return self.curricula.get(curriculum_id)

    def get_learning_unit(self, unit_id: str) -> Optional[LearningUnit]:
        """
        Get a learning unit by ID.

        Args:
            unit_id: ID of the learning unit to retrieve

        Returns:
            The learning unit if found, None otherwise
        """
        with self.lock:
            return self.learning_units.get(unit_id)

    def get_learning_task(self, task_id: str) -> Optional[LearningTask]:
        """
        Get a learning task by ID.

        Args:
            task_id: ID of the learning task to retrieve

        Returns:
            The learning task if found, None otherwise
        """
        with self.lock:
            return self.learning_tasks.get(task_id)

    def set_active_curriculum(self, curriculum_id: str) -> bool:
        """
        Set the active curriculum being worked on.

        Args:
            curriculum_id: ID of the curriculum to activate

        Returns:
            True if successful, False if curriculum not found
        """
        with self.lock:
            if curriculum_id not in self.curricula:
                self.logger.warning(f"Curriculum {curriculum_id} not found")
                return False

            self.active_curriculum = curriculum_id

            # Mark as in progress if not already
            if self.curricula[curriculum_id].status == TaskStatus.NOT_STARTED:
                self.curricula[curriculum_id].status = TaskStatus.IN_PROGRESS

            self.logger.info(f"Set active curriculum to: {self.curricula[curriculum_id].name}")
            return True

    def set_active_unit(self, unit_id: str) -> bool:
        """
        Set the active learning unit being worked on.

        Args:
            unit_id: ID of the learning unit to activate

        Returns:
            True if successful, False if unit not found
        """
        with self.lock:
            if unit_id not in self.learning_units:
                self.logger.warning(f"Learning unit {unit_id} not found")
                return False

            self.active_unit = unit_id

            # Mark as in progress if not already
            if self.learning_units[unit_id].status == TaskStatus.NOT_STARTED:
                self.learning_units[unit_id].status = TaskStatus.IN_PROGRESS

            self.logger.info(f"Set active unit to: {self.learning_units[unit_id].name}")
            return True

    def get_next_task(self) -> Optional[LearningTask]:
        """
        Get the next recommended learning task from the active curriculum.

        Returns:
            The next recommended task, or None if no suitable task is found
        """
        with self.lock:
            if not self.active_curriculum:
                self.logger.warning("No active curriculum set")
                return None

            if not self.active_unit:
                # Recommend next unit if none active
                next_unit_id = self._get_next_unit_id()
                if next_unit_id:
                    self.set_active_unit(next_unit_id)
                else:
                    self.logger.warning("No suitable unit found")
                    return None

            # Get recommended task from active unit
            next_task_id = self._get_next_task_id()
            if not next_task_id:
                # Try finding a task from another unit
                next_unit_id = self._get_next_unit_id()
                if next_unit_id:
                    self.set_active_unit(next_unit_id)
                    next_task_id = self._get_next_task_id()

                if not next_task_id:
                    self.logger.warning("No suitable task found")
                    return None

            # Set as active task
            self.active_task = next_task_id

            # Mark as in progress
            task = self.learning_tasks[next_task_id]
            if task.status == TaskStatus.NOT_STARTED:
                task.status = TaskStatus.IN_PROGRESS

            # Add to current focus
            if next_task_id not in self.current_focus:
                self.current_focus.append(next_task_id)
                # Limit focus size
                if len(self.current_focus) > 5:
                    self.current_focus.pop(0)

            return task

    def record_performance(self, task_id: str, score: float, time_spent: float,
                          completed: bool, mistakes: int = 0,
                          feedback: str = "", metadata: Optional[Dict[str, Any]] = None) -> str:
        """
        Record performance on a learning task.

        Args:
            task_id: ID of the task
            score: Performance score (0.0 to 1.0)
            time_spent: Time spent on the task in minutes
            completed: Whether the task was completed
            mistakes: Number of mistakes made
            feedback: Feedback on the performance
            metadata: Additional data about the performance

        Returns:
            ID of the created performance record
        """
        with self.lock:
            if task_id not in self.learning_tasks:
                raise ValueError(f"Learning task {task_id} not found")

            # Create performance record
            record_id = f"perf_{str(uuid.uuid4())[:8]}"
            record = PerformanceRecord(
                record_id=record_id,
                task_id=task_id,
                score=score,
                time_spent=time_spent,
                completed=completed,
                mistakes=mistakes,
                feedback=feedback,
                metadata=metadata or {}
            )

            # Store record
            self.performance_records[record_id] = record

            # Update task status and proficiency
            task = self.learning_tasks[task_id]
            task.attempts += 1
            task.last_attempted = time.time()

            # Update proficiency (weighted average with previous proficiency)
            if task.attempts == 1:
                task.proficiency = score
            else:
                # Higher weight for most recent attempt
                task.proficiency = 0.3 * task.proficiency + 0.7 * score

            # Update status based on performance
            if completed:
                if score >= self.config.mastery_threshold:
                    task.status = TaskStatus.MASTERED
                else:
                    task.status = TaskStatus.COMPLETED
            else:
                if task.attempts >= self.config.max_consecutive_failures:
                    task.status = TaskStatus.FAILED
                else:
                    task.status = TaskStatus.IN_PROGRESS

            # Update unit progress
            if task_id in self.learning_tasks and self.active_unit:
                unit = self.learning_units[self.active_unit]
                completed_tasks = sum(1 for tid in unit.tasks
                                     if tid in self.learning_tasks and
                                     self.learning_tasks[tid].status in
                                     [TaskStatus.COMPLETED, TaskStatus.MASTERED])
                unit.progress = completed_tasks / max(1, len(unit.tasks))

                # Update unit status if all tasks completed
                if unit.progress >= unit.completion_threshold:
                    unit.status = TaskStatus.COMPLETED

            # Update curriculum progress
            if self.active_curriculum:
                curriculum = self.curricula[self.active_curriculum]
                completed_units = sum(1 for uid in curriculum.units
                                     if uid in self.learning_units and
                                     self.learning_units[uid].status in
                                     [TaskStatus.COMPLETED, TaskStatus.MASTERED])
                curriculum.progress = completed_units / max(1, len(curriculum.units))

                # Update curriculum status if all units completed
                if curriculum.progress >= 1.0:
                    curriculum.status = TaskStatus.COMPLETED

            # Update skill mastery
            self._update_skill_mastery(task_id, score, completed)

            # Store in memory systems
            self._store_performance_in_memory(record)

            # Generate learning feedback
            if self.config.adaptation_enabled:
                self._adapt_curriculum(task_id, score, completed)

            self.logger.info(f"Recorded performance for task {task.name}: "
                           f"score={score:.2f}, completed={completed}")

            # Notify intrinsic motivation if available
            if (self.intrinsic_motivation and self.config.use_intrinsic_motivation and
                not getattr(record, 'from_intrinsic_motivation', False)):
                try:
                    self.intrinsic_motivation.update_competence(
                        skill_area=task.metadata.get('skill_area', task.name),
                        performance=score,
                        observation=self._create_observation_data(task, record)
                    )
                except Exception as e:
                    self.logger.warning(f"Error updating intrinsic motivation: {str(e)}")

            # Update metrics
            self.metrics["task_scores"].append(score)
            self.metrics["task_completion"].append(1.0 if completed else 0.0)

            return record_id

    def get_learning_path(self, objective: str) -> List[str]:
        """
        Get a recommended learning path for a specific objective.

        Args:
            objective: Learning objective to achieve

        Returns:
            List of task IDs forming a learning path
        """
        with self.lock:
            # Check if we already have a computed path
            if objective in self.learning_paths:
                return self.learning_paths[objective]

            # Find tasks related to this objective
            relevant_tasks = []
            for task_id, task in self.learning_tasks.items():
                if (objective.lower() in task.name.lower() or
                    objective.lower() in task.description.lower() or
                    objective.lower() in str(task.metadata).lower()):
                    relevant_tasks.append(task_id)

            if not relevant_tasks:
                self.logger.warning(f"No tasks found for objective: {objective}")
                return []

            # Build dependency graph
            graph = self._build_dependency_graph()

            # Find path to each relevant task
            all_paths = []
            for task_id in relevant_tasks:
                path = self._find_path_to_task(task_id, graph)
                if path:
                    all_paths.append(path)

            # Combine paths and remove duplicates
            combined_path = []
            for path in all_paths:
                for task_id in path:
                    if task_id not in combined_path:
                        combined_path.append(task_id)

            # Sort by difficulty and prerequisites
            sorted_path = self._topological_sort(combined_path, graph)

            # Store for future reference
            self.learning_paths[objective] = sorted_path

            return sorted_path

    def get_skill_mastery(self, skill_id: str) -> float:
        """
        Get the current mastery level for a skill.

        Args:
            skill_id: ID of the skill

        Returns:
            Mastery level from 0.0 to 1.0
        """
        with self.lock:
            return self.skill_mastery.get(skill_id, 0.0)

    def get_performance_history(self, task_id: str) -> List[PerformanceRecord]:
        """
        Get performance history for a specific task.

        Args:
            task_id: ID of the task

        Returns:
            List of performance records for the task
        """
        with self.lock:
            return [record for record in self.performance_records.values()
                   if record.task_id == task_id]

    def get_curriculum_progress(self, curriculum_id: str) -> Dict[str, Any]:
        """
        Get detailed progress information for a curriculum.

        Args:
            curriculum_id: ID of the curriculum

        Returns:
            Dictionary with progress information
        """
        with self.lock:
            if curriculum_id not in self.curricula:
                raise ValueError(f"Curriculum {curriculum_id} not found")

            curriculum = self.curricula[curriculum_id]

            # Collect unit progress
            units_progress = {}
            for unit_id in curriculum.units:
                if unit_id in self.learning_units:
                    unit = self.learning_units[unit_id]

                    # Collect task progress
                    tasks_progress = {}
                    for task_id in unit.tasks:
                        if task_id in self.learning_tasks:
                            task = self.learning_tasks[task_id]
                            tasks_progress[task_id] = {
                                "name": task.name,
                                "status": task.status.name,
                                "proficiency": task.proficiency,
                                "attempts": task.attempts
                            }

                    units_progress[unit_id] = {
                        "name": unit.name,
                        "progress": unit.progress,
                        "status": unit.status.name,
                        "tasks": tasks_progress
                    }

            return {
                "name": curriculum.name,
                "progress": curriculum.progress,
                "status": curriculum.status.name,
                "units": units_progress
            }

    def get_metrics(self) -> Dict[str, Any]:
        """
        Get performance metrics for the curriculum manager.

        Returns:
            Dictionary with metrics
        """
        with self.lock:
            metrics = {}

            # Process metric queues
            for key, values in self.metrics.items():
                if values:
                    metrics[key] = {
                        "current": values[-1],
                        "mean": float(np.mean(values)),
                        "max": float(np.max(values)),
                        "min": float(np.min(values))
                    }

            # Add curriculum stats
            metrics["curricula_count"] = len(self.curricula)
            metrics["units_count"] = len(self.learning_units)
            metrics["tasks_count"] = len(self.learning_tasks)
            metrics["performance_records_count"] = len(self.performance_records)

            if self.active_curriculum:
                metrics["active_curriculum"] = {
                    "id": self.active_curriculum,
                    "name": self.curricula[self.active_curriculum].name,
                    "progress": self.curricula[self.active_curriculum].progress
                }

            return metrics

    def reset_progress(self, curriculum_id: Optional[str] = None) -> None:
        """
        Reset progress for a curriculum or all curricula.

        Args:
            curriculum_id: ID of curriculum to reset, or None for all
        """
        with self.lock:
            if curriculum_id:
                if curriculum_id not in self.curricula:
                    raise ValueError(f"Curriculum {curriculum_id} not found")

                # Reset curriculum progress
                self.curricula[curriculum_id].progress = 0.0
                self.curricula[curriculum_id].status = TaskStatus.NOT_STARTED

                # Reset units in this curriculum
                for unit_id in self.curricula[curriculum_id].units:
                    if unit_id in self.learning_units:
                        self.learning_units[unit_id].progress = 0.0
                        self.learning_units[unit_id].status = TaskStatus.NOT_STARTED

                        # Reset tasks in this unit
                        for task_id in self.learning_units[unit_id].tasks:
                            if task_id in self.learning_tasks:
                                self.learning_tasks[task_id].status = TaskStatus.NOT_STARTED
                                self.learning_tasks[task_id].proficiency = 0.0
                                self.learning_tasks[task_id].attempts = 0
                                self.learning_tasks[task_id].last_attempted = None

                self.logger.info(f"Reset progress for curriculum: {self.curricula[curriculum_id].name}")
            else:
                # Reset all curricula
                for curr_id, curriculum in self.curricula.items():
                    curriculum.progress = 0.0
                    curriculum.status = TaskStatus.NOT_STARTED

                # Reset all units
                for unit_id, unit in self.learning_units.items():
                    unit.progress = 0.0
                    unit.status = TaskStatus.NOT_STARTED

                # Reset all tasks
                for task_id, task in self.learning_tasks.items():
                    task.status = TaskStatus.NOT_STARTED
                    task.proficiency = 0.0
                    task.attempts = 0
                    task.last_attempted = None

                self.logger.info(f"Reset progress for all curricula")

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.learning_core:
                dependencies.append("learning_core")
            if self.intrinsic_motivation:
                dependencies.append("intrinsic_motivation")

            self.nexus_core.register_module(
                name="curriculum",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _load_from_storage(self) -> None:
        """Load curricula from persistent storage"""
        # In a real implementation, this would load from disk or database
        # For simplicity, we'll just log a message
        self.logger.info("No persistent curricula to load")

    def _get_next_unit_id(self) -> Optional[str]:
        """
        Get the ID of the next recommended unit in the active curriculum.

        Returns:
            The ID of the next unit, or None if no suitable unit is found
        """
        if not self.active_curriculum:
            return None

        curriculum = self.curricula[self.active_curriculum]

        # Find incomplete units
        candidate_units = []
        for unit_id in curriculum.units:
            if unit_id in self.learning_units:
                unit = self.learning_units[unit_id]
                if unit.status in [TaskStatus.NOT_STARTED, TaskStatus.IN_PROGRESS]:
                    # Check if dependencies are satisfied
                    dependencies_met = True
                    for dep_id in unit.dependencies:
                        if dep_id in self.learning_units:
                            dep_unit = self.learning_units[dep_id]
                            if dep_unit.status != TaskStatus.COMPLETED:
                                dependencies_met = False
                                break

                    if dependencies_met:
                        candidate_units.append(unit_id)

        if not candidate_units:
            return None

        # Sort by progress (prefer units already in progress)
        candidate_units.sort(key=lambda uid:
                             1 if self.learning_units[uid].status == TaskStatus.IN_PROGRESS else 0,
                             reverse=True)

        return candidate_units[0]

    def _get_next_task_id(self) -> Optional[str]:
        """
        Get the ID of the next recommended task in the active unit.

        Returns:
            The ID of the next task, or None if no suitable task is found
        """
        if not self.active_unit:
            return None

        unit = self.learning_units[self.active_unit]

        # Find incomplete tasks
        candidate_tasks = []
        for task_id in unit.tasks:
            if task_id in self.learning_tasks:
                task = self.learning_tasks[task_id]
                if task.status in [TaskStatus.NOT_STARTED, TaskStatus.IN_PROGRESS]:
                    # Check if prerequisites are satisfied
                    prerequisites_met = True
                    for prereq_id in task.prerequisites:
                        if prereq_id in self.learning_tasks:
                            prereq_task = self.learning_tasks[prereq_id]
                            if prereq_task.status not in [TaskStatus.COMPLETED, TaskStatus.MASTERED]:
                                prerequisites_met = False
                                break

                    if prerequisites_met:
                        candidate_tasks.append(task_id)

        if not candidate_tasks:
            return None

        # If intrinsic motivation is available, use it to influence task selection
        if self.intrinsic_motivation and self.config.use_intrinsic_motivation:
            try:
                # Get curiosity state
                curiosity_state = self.intrinsic_motivation.get_curiosity_state()

                # Calculate task scores based on novelty, difficulty, and interest
                task_scores = []
                for task_id in candidate_tasks:
                    task = self.learning_tasks[task_id]

                    # Base score
                    score = 0.5

                    # Adjust for novelty preference
                    novelty_factor = 0.0
                    if task.last_attempted is None:
                        # Never attempted
                        novelty_factor = 1.0
                    else:
                        # Inversely proportional to recency
                        days_since = (time.time() - task.last_attempted) / (24 * 3600)
                        novelty_factor = min(1.0, days_since / 7.0)  # Max after a week

                    score += curiosity_state.novelty_seeking * novelty_factor * self.config.novelty_weight

                    # Adjust for difficulty preference
                    difficulty_factor = task.difficulty.value / DifficultyLevel.EXPERT.value
                    # If exploration is high, prefer harder tasks
                    if curiosity_state.exploration_rate > 0.7:
                        score += difficulty_factor * self.config.difficulty_weight
                    else:
                        # Otherwise prefer tasks of moderate difficulty
                        moderate_difficulty = 1.0 - abs(difficulty_factor - 0.5) * 2
                        score += moderate_difficulty * self.config.difficulty_weight

                    # Adjust for relevance to current focus
                    if curiosity_state.attention_focus:
                        relevance = 0.0
                        focus_str = str(curiosity_state.attention_focus).lower()

                        if focus_str in task.name.lower() or focus_str in task.description.lower():
                            relevance = 1.0
                        elif any(focus_str in str(meta_val).lower()
                               for meta_val in task.metadata.values()):
                            relevance = 0.8

                        score += relevance * self.config.relevance_weight * curiosity_state.focus_strength

                    task_scores.append((task_id, score))

                # Sort by score (highest first)
                task_scores.sort(key=lambda x: x[1], reverse=True)

                if task_scores:
                    self.logger.debug(f"Selected task {task_scores[0][0]} with score {task_scores[0][1]:.2f}")
                    return task_scores[0][0]

            except Exception as e:
                self.logger.warning(f"Error using intrinsic motivation for task selection: {str(e)}")

        # Fallback: sort by difficulty (easiest first)
        candidate_tasks.sort(key=lambda tid: self.learning_tasks[tid].difficulty.value)

        return candidate_tasks[0]

    def _update_skill_mastery(self, task_id: str, score: float, completed: bool) -> None:
        """
        Update skill mastery based on task performance.

        Args:
            task_id: ID of the task
            score: Performance score
            completed: Whether the task was completed
        """
        if task_id not in self.learning_tasks:
            return

        task = self.learning_tasks[task_id]

        # Get skill areas from task metadata
        skill_areas = []
        if 'skill_area' in task.metadata:
            if isinstance(task.metadata['skill_area'], list):
                skill_areas.extend(task.metadata['skill_area'])
            else:
                skill_areas.append(task.metadata['skill_area'])

        if 'skills' in task.metadata:
            if isinstance(task.metadata['skills'], list):
                skill_areas.extend(task.metadata['skills'])
            elif isinstance(task.metadata['skills'], dict):
                skill_areas.extend(task.metadata['skills'].keys())
            else:
                skill_areas.append(task.metadata['skills'])

        # If no skills specified, use task name
        if not skill_areas:
            skill_areas.append(task.name)

        # Update mastery for each skill
        for skill in skill_areas:
            skill_id = str(skill)

            # Get current mastery
            current_mastery = self.skill_mastery.get(skill_id, 0.0)

            # If task was completed, update mastery
            if completed:
                # Weight based on task difficulty
                difficulty_weight = (task.difficulty.value + 1) / (DifficultyLevel.EXPERT.value + 1)

                # Higher difficulty gives more mastery gain
                mastery_gain = score * difficulty_weight * 0.2

                # New mastery is weighted average of current and gain
                new_mastery = current_mastery * 0.8 + mastery_gain

                # Cap at 1.0
                new_mastery = min(1.0, new_mastery)

                # Update mastery
                self.skill_mastery[skill_id] = new_mastery

                if new_mastery > current_mastery + 0.1:
                    self.logger.info(f"Skill mastery increased for {skill_id}: "
                                    f"{current_mastery:.2f} -> {new_mastery:.2f}")

    def _adapt_curriculum(self, task_id: str, score: float, completed: bool) -> None:
        """
        Adapt curriculum based on learning performance.

        Args:
            task_id: ID of the task
            score: Performance score
            completed: Whether the task was completed
        """
        if not self.config.adaptation_enabled:
            return

        task = self.learning_tasks[task_id]

        # Handle failed tasks
        if not completed and score < 0.3:
            consecutive_failures = 0

            # Check if there are multiple recent failures
            for record in reversed(list(self.performance_records.values())):
                if record.task_id == task_id and record.score < 0.3:
                    consecutive_failures += 1
                else:
                    break

            if consecutive_failures >= self.config.max_consecutive_failures:
                # Suggest easier alternative tasks
                self._suggest_easier_alternatives(task_id)

        # Handle completed tasks
        elif completed and score > 0.7:
            # Check if mastery review is needed for any related skills
            self._check_for_review_needed(task)

    def _suggest_easier_alternatives(self, task_id: str) -> List[str]:
        """
        Suggest easier alternative tasks for a failed task.

        Args:
            task_id: ID of the failed task

        Returns:
            List of suggested alternative task IDs
        """
        if task_id not in self.learning_tasks:
            return []

        task = self.learning_tasks[task_id]

        # Find easier tasks with similar skills
        alternatives = []

        # Get task skills
        task_skills = set()
        if 'skill_area' in task.metadata:
            if isinstance(task.metadata['skill_area'], list):
                task_skills.update(task.metadata['skill_area'])
            else:
                task_skills.add(task.metadata['skill_area'])

        if 'skills' in task.metadata:
            if isinstance(task.metadata['skills'], list):
                task_skills.update(task.metadata['skills'])
            elif isinstance(task.metadata['skills'], dict):
                task_skills.update(task.metadata['skills'].keys())
            else:
                task_skills.add(task.metadata['skills'])

        # If no skills, use task name
        if not task_skills:
            task_skills.add(task.name)

        # Find all tasks with similar skills but lower difficulty
        for tid, t in self.learning_tasks.items():
            if tid == task_id:
                continue

            # Check if easier
            if t.difficulty.value >= task.difficulty.value:
                continue

            # Check for skill overlap
            t_skills = set()
            if 'skill_area' in t.metadata:
                if isinstance(t.metadata['skill_area'], list):
                    t_skills.update(t.metadata['skill_area'])
                else:
                    t_skills.add(t.metadata['skill_area'])

            if 'skills' in t.metadata:
                if isinstance(t.metadata['skills'], list):
                    t_skills.update(t.metadata['skills'])
                elif isinstance(t.metadata['skills'], dict):
                    t_skills.update(t.metadata['skills'].keys())
                else:
                    t_skills.add(t.metadata['skills'])

            # If no skills, use task name
            if not t_skills:
                t_skills.add(t.name)

            # Check overlap
            if task_skills.intersection(t_skills):
                alternatives.append(tid)

        # Log suggestion
        if alternatives:
            alt_names = [self.learning_tasks[tid].name for tid in alternatives[:3]]
            self.logger.info(f"Suggesting easier alternatives for {task.name}: {', '.join(alt_names)}")

        return alternatives

    def _check_for_review_needed(self, task: LearningTask) -> List[str]:
        """
        Check if any related skills need review.

        Args:
            task: The current task

        Returns:
            List of task IDs recommended for review
        """
        # Get current task skills
        task_skills = set()
        if 'skill_area' in task.metadata:
            if isinstance(task.metadata['skill_area'], list):
                task_skills.update(task.metadata['skill_area'])
            else:
                task_skills.add(task.metadata['skill_area'])

        if 'skills' in task.metadata:
            if isinstance(task.metadata['skills'], list):
                task_skills.update(task.metadata['skills'])
            elif isinstance(task.metadata['skills'], dict):
                task_skills.update(task.metadata['skills'].keys())
            else:
                task_skills.add(task.metadata['skills'])

        # Find tasks that haven't been practiced recently
        now = time.time()
        review_candidates = []

        for tid, t in self.learning_tasks.items():
            if tid == task.task_id:
                continue

            # Check if previously mastered
            if t.status != TaskStatus.MASTERED:
                continue

            # Check if not practiced recently
            if t.last_attempted:
                days_since = (now - t.last_attempted) / (24 * 3600)
                if days_since < self.config.review_interval_days:
                    continue

            # Check for skill overlap
            t_skills = set()
            if 'skill_area' in t.metadata:
                if isinstance(t.metadata['skill_area'], list):
                    t_skills.update(t.metadata['skill_area'])
                else:
                    t_skills.add(t.metadata['skill_area'])

            if 'skills' in t.metadata:
                if isinstance(t.metadata['skills'], list):
                    t_skills.update(t.metadata['skills'])
                elif isinstance(t.metadata['skills'], dict):
                    t_skills.update(t.metadata['skills'].keys())
                else:
                    t_skills.add(t.metadata['skills'])

            # If skills overlap, add to review candidates
            if task_skills.intersection(t_skills):
                review_candidates.append(tid)

        # Log recommendation
        if review_candidates:
            review_names = [self.learning_tasks[tid].name for tid in review_candidates[:3]]
            self.logger.info(f"Recommending review of: {', '.join(review_names)}")

        return review_candidates

    def _store_performance_in_memory(self, record: PerformanceRecord) -> None:
        """
        Store performance record in memory systems.

        Args:
            record: The performance record to store
        """
        # Store in episodic memory if available
        if self.episodic_memory and self.config.use_episodic_memory:
            try:
                task_id = record.task_id
                task = self.learning_tasks.get(task_id)

                if task:
                    # Create episodic memory event
                    self.episodic_memory.add_event(
                        episode_id="curriculum_learning",  # This should be a real episode ID
                        content={
                            "type": "task_performance",
                            "task_name": task.name,
                            "score": record.score,
                            "completed": record.completed,
                            "feedback": record.feedback
                        },
                        metadata={
                            "source": "curriculum",
                            "task_id": task_id,
                            "importance": min(1.0, record.score + 0.3) if record.completed else 0.3
                        }
                    )
            except Exception as e:
                self.logger.warning(f"Error storing in episodic memory: {str(e)}")

        # Store in semantic memory if available
        if self.semantic_memory and self.config.use_semantic_memory:
            try:
                task_id = record.task_id
                task = self.learning_tasks.get(task_id)

                if task:
                    # Get skill areas from task
                    skill_areas = []
                    if 'skill_area' in task.metadata:
                        if isinstance(task.metadata['skill_area'], list):
                            skill_areas.extend(task.metadata['skill_area'])
                        else:
                            skill_areas.append(task.metadata['skill_area'])

                    # Create skill concepts or update them
                    for skill in skill_areas:
                        # This is simplified - actual usage would depend on semantic memory API
                        skill_concept = {
                            "name": str(skill),
                            "description": f"Skill area related to task {task.name}",
                            "properties": {
                                "mastery": self.skill_mastery.get(str(skill), 0.0),
                                "last_practiced": time.time()
                            }
                        }

                        self.semantic_memory.add_concept(skill_concept)
            except Exception as e:
                self.logger.warning(f"Error storing in semantic memory: {str(e)}")

    def _create_observation_data(self, task: LearningTask,
                               record: PerformanceRecord) -> Any:
        """
        Create observation data for intrinsic motivation.

        Args:
            task: The task
            record: The performance record

        Returns:
            Observation data object
        """
        if not self.intrinsic_motivation:
            return None

        # Format depends on intrinsic motivation module's expectations
        # This is a simplified approach
        try:
            from grandnexus.learning.intrinsic_motivation import ObservationData

            observation = ObservationData(
                content=f"Task performance: {task.name}, score={record.score:.2f}",
                context={
                    "task_id": task.task_id,
                    "task_name": task.name,
                    "difficulty": task.difficulty.value,
                    "completed": record.completed,
                    "skill_area": task.metadata.get("skill_area", task.name)
                },
                source="curriculum",
                metadata={
                    "from_curriculum": True,
                    "performance_record_id": record.record_id
                }
            )

            return observation
        except (ImportError, AttributeError):
            # Fall back to a simple dictionary
            return {
                "content": f"Task performance: {task.name}, score={record.score:.2f}",
                "context": {
                    "task_id": task.task_id,
                    "task_name": task.name,
                    "difficulty": task.difficulty.value,
                    "completed": record.completed
                },
                "source": "curriculum",
                "timestamp": time.time()
            }

    def _build_dependency_graph(self) -> Dict[str, List[str]]:
        """
        Build a dependency graph for tasks and units.

        Returns:
            Dictionary mapping task IDs to their dependencies
        """
        graph = {}

        # Add task prerequisites
        for task_id, task in self.learning_tasks.items():
            graph[task_id] = task.prerequisites.copy()

        # Add unit dependencies (all tasks in dependent units are prerequisites)
        for unit_id, unit in self.learning_units.items():
            for task_id in unit.tasks:
                if task_id not in graph:
                    graph[task_id] = []

                # Add prerequisites from dependent units
                for dep_unit_id in unit.dependencies:
                    if dep_unit_id in self.learning_units:
                        dep_unit = self.learning_units[dep_unit_id]
                        for dep_task_id in dep_unit.tasks:
                            if dep_task_id != task_id and dep_task_id not in graph[task_id]:
                                graph[task_id].append(dep_task_id)

        return graph

    def _find_path_to_task(self, task_id: str,
                         graph: Dict[str, List[str]]) -> List[str]:
        """
        Find a path to a task including all prerequisites.

        Args:
            task_id: Target task ID
            graph: Dependency graph

        Returns:
            List of task IDs forming a path to the target
        """
        if task_id not in graph:
            return []

        # Use DFS to find all dependencies
        visited = set()
        path = []

        def dfs(node):
            if node in visited:
                return

            visited.add(node)

            # Visit dependencies first
            for dep in graph.get(node, []):
                dfs(dep)

            # Add node to path
            path.append(node)

        dfs(task_id)

        return path

    def _topological_sort(self, tasks: List[str],
                        graph: Dict[str, List[str]]) -> List[str]:
        """
        Sort tasks topologically (prerequisites before dependents).

        Args:
            tasks: List of task IDs to sort
            graph: Dependency graph

        Returns:
            Topologically sorted list of task IDs
        """
        # Filter tasks to include only those in the graph
        valid_tasks = [t for t in tasks if t in graph]

        # Create subgraph
        subgraph = {t: [d for d in graph[t] if d in valid_tasks] for t in valid_tasks}

        # Count incoming edges
        in_degree = {t: 0 for t in valid_tasks}
        for t in valid_tasks:
            for dep in subgraph[t]:
                in_degree[dep] = in_degree.get(dep, 0) + 1

        # Start with nodes with no dependencies
        queue = [t for t in valid_tasks if in_degree.get(t, 0) == 0]
        result = []

        while queue:
            node = queue.pop(0)
            result.append(node)

            # Remove edges
            for t in valid_tasks:
                if node in subgraph[t]:
                    in_degree[t] -= 1
                    if in_degree[t] == 0:
                        queue.append(t)

        # Check for cycles
        if len(result) != len(valid_tasks):
            # Fall back to difficulty-based sorting
            return sorted(valid_tasks,
                         key=lambda t: self.learning_tasks[t].difficulty.value if t in self.learning_tasks else 0)

        return result

ModuleNotFoundError: No module named 'grandnexus'

# Monitor

In [ ]:
"""
monitor.py - Cognitive Process Monitoring Module for GrandNexus

This module provides real-time monitoring and analytics of GrandNexus's cognitive processes.
It acts as an awareness layer, observing the behavior and performance of all cognitive
components, collecting metrics, detecting anomalies, and providing insights
that can inform metacognitive decisions.

Key features:
- Real-time monitoring of all cognitive module activities
- Collection and aggregation of performance metrics
- Detection of processing bottlenecks and anomalies
- Resource utilization tracking
- Cognitive load estimation
- Integration with adaptive mechanisms for self-regulation

The design emphasizes:
- Minimal interference with monitored processes
- Extensible metrics collection framework
- Configurable thresholds and monitoring policies
- Integration with other metacognitive components
"""

import logging
import threading
import time
import uuid
import psutil
import os
from typing import Dict, List, Any, Optional, Set, Tuple, Union, Callable
from enum import Enum, auto
from dataclasses import dataclass, field
from collections import defaultdict, deque
import statistics
import numpy as np
import datetime

# Assuming these exist in GrandNexus architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.metacognition.confidence import ConfidenceEvaluator

# Metric types for categorization
class MetricType(Enum):
    """Types of metrics that can be monitored"""
    PERFORMANCE = auto()      # Processing speed, throughput, latency
    RESOURCE = auto()         # Memory usage, CPU usage, etc.
    ACTIVITY = auto()         # Module operation counts, call frequency
    QUALITY = auto()          # Confidence scores, precision, error rates
    STABILITY = auto()        # Error counts, exception rates, etc.
    COGNITIVE_LOAD = auto()   # Estimated load on various cognitive resources
    EMERGING = auto()         # Metrics that emerge from system interactions

class MetricPriority(Enum):
    """Priority levels for metrics, influencing sampling rate and alerting"""
    CRITICAL = 0   # Essential metrics requiring immediate attention if thresholds crossed
    HIGH = 1       # Important metrics requiring prompt attention
    MEDIUM = 2     # Standard metrics for regular monitoring
    LOW = 3        # Background metrics collected less frequently
    DEBUG = 4      # Metrics collected only during debug sessions

@dataclass
class MetricDefinition:
    """Definition of a metric to be monitored"""
    name: str
    description: str
    type: MetricType
    unit: str = ""
    priority: MetricPriority = MetricPriority.MEDIUM
    sampling_interval: float = 1.0  # seconds
    smoothing_factor: float = 0.2   # For exponential smoothing (0-1)
    alert_threshold_low: Optional[float] = None
    alert_threshold_high: Optional[float] = None
    derived: bool = False  # True if calculated from other metrics

@dataclass
class MetricValue:
    """A single metric measurement with metadata"""
    metric_id: str
    timestamp: float
    value: float
    source_module: str
    context: Dict[str, Any] = field(default_factory=dict)

@dataclass
class MonitoredSystem:
    """Information about the system being monitored"""
    cpu_count: int
    total_memory: int  # bytes
    start_time: float
    platform: str
    modules: Set[str] = field(default_factory=set)

@dataclass
class MonitorAlert:
    """Alert raised when a metric crosses a threshold"""
    alert_id: str
    metric_id: str
    timestamp: float
    severity: str  # "warning", "error", "critical"
    message: str
    value: float
    threshold: float
    is_active: bool = True

@dataclass
class PerformanceReport:
    """Comprehensive report on system performance"""
    timestamp: float
    time_period: str  # e.g., "last_minute", "last_hour", "last_day"
    metrics_summary: Dict[str, Dict[str, float]]  # metric_id -> {avg, min, max, std}
    bottlenecks: List[Dict[str, Any]]
    resource_usage: Dict[str, float]
    cognitive_load: Dict[str, float]
    alerts: List[MonitorAlert]
    recommendations: List[Dict[str, Any]] = field(default_factory=list)

class CognitiveMonitor:
    """
    Central monitoring system for tracking the performance and behavior
    of GrandNexus's cognitive modules.

    This module acts as an observational layer, collecting metrics,
    detecting anomalies, and providing insights without directly
    intervening in cognitive processes.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 confidence_evaluator: Optional[ConfidenceEvaluator] = None,
                 config: Optional[Dict[str, Any]] = None):
        """
        Initialize the cognitive monitor.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            confidence_evaluator: Reference to confidence evaluation module
            config: Optional configuration dictionary
        """
        self.logger = logging.getLogger("GrandNexus.Metacognition.Monitor")
        self.nexus_core = nexus_core
        self.confidence_evaluator = confidence_evaluator

        # Initialize configuration
        self.config = config or {}
        self.monitoring_interval = self.config.get("monitoring_interval", 1.0)  # seconds
        self.history_length = self.config.get("history_length", 3600)  # number of samples to keep
        self.alert_check_interval = self.config.get("alert_check_interval", 5.0)  # seconds
        self.report_interval = self.config.get("report_interval", 300.0)  # seconds

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.start_time = time.time()
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # System information
        self.system_info = MonitoredSystem(
            cpu_count=os.cpu_count() or 1,
            total_memory=psutil.virtual_memory().total,
            start_time=self.start_time,
            platform=f"Python {sys.version.split()[0]} on {sys.platform}"
        )

        # Metrics registry
        self.metrics: Dict[str, MetricDefinition] = {}

        # Metrics storage
        self.metric_histories: Dict[str, deque[MetricValue]] = defaultdict(
            lambda: deque(maxlen=self.history_length)
        )

        # Current values (latest for each metric)
        self.current_values: Dict[str, MetricValue] = {}

        # Smoothed values (exponential moving average)
        self.smoothed_values: Dict[str, float] = {}

        # Alerts
        self.active_alerts: Dict[str, MonitorAlert] = {}
        self.alert_history: deque[MonitorAlert] = deque(maxlen=1000)

        # Bottleneck detection
        self.bottleneck_thresholds = self.config.get("bottleneck_thresholds", {
            "cpu_usage": 90.0,  # percentage
            "memory_usage": 85.0,  # percentage
            "queue_growth_rate": 0.2,  # items per second
            "response_time": 2.0  # seconds
        })

        # Performance reports
        self.reports: deque[PerformanceReport] = deque(maxlen=100)

        # Monitoring threads
        self.monitor_thread = None
        self.alert_thread = None
        self.report_thread = None
        self.stop_event = threading.Event()

        # Custom metric collectors (module_name -> list of collector functions)
        self.custom_collectors: Dict[str, List[Callable[[], Dict[str, float]]]] = defaultdict(list)

        self.logger.info(f"CognitiveMonitor initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the monitor with standard metrics"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register standard metrics
                self._register_standard_metrics()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Register for callbacks and events
                self._subscribe_to_events()

                self.initialized = True
                self.logger.info("CognitiveMonitor initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"CognitiveMonitor initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the monitoring processes"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start the monitoring thread
                self.stop_event.clear()
                self.monitor_thread = threading.Thread(
                    target=self._monitoring_loop,
                    name="Monitor_Thread",
                    daemon=True
                )
                self.monitor_thread.start()

                # Start the alert checking thread
                self.alert_thread = threading.Thread(
                    target=self._alert_check_loop,
                    name="Alert_Thread",
                    daemon=True
                )
                self.alert_thread.start()

                # Start the report generation thread
                self.report_thread = threading.Thread(
                    target=self._report_generation_loop,
                    name="Report_Thread",
                    daemon=True
                )
                self.report_thread.start()

                self.running = True
                self.logger.info("CognitiveMonitor started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start CognitiveMonitor: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the monitoring processes"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Signal threads to stop
                self.stop_event.set()

                # Wait for threads to terminate
                if self.monitor_thread and self.monitor_thread.is_alive():
                    self.monitor_thread.join(timeout=5.0)

                if self.alert_thread and self.alert_thread.is_alive():
                    self.alert_thread.join(timeout=5.0)

                if self.report_thread and self.report_thread.is_alive():
                    self.report_thread.join(timeout=5.0)

                self.running = False
                self.logger.info("CognitiveMonitor stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping CognitiveMonitor: {str(e)}")
                return False

    def register_metric(self, metric: MetricDefinition) -> bool:
        """
        Register a new metric to be monitored.

        Args:
            metric: The metric definition

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            if metric.name in self.metrics:
                self.logger.warning(f"Metric {metric.name} already exists, updating")

            self.metrics[metric.name] = metric
            self.logger.info(f"Registered metric: {metric.name} ({metric.type.name})")
            return True

    def record_metric(self, metric_name: str, value: float,
                     source_module: str, context: Optional[Dict[str, Any]] = None) -> bool:
        """
        Record a metric value.

        Args:
            metric_name: Name of the metric
            value: Metric value
            source_module: Name of the module reporting the metric
            context: Optional contextual information

        Returns:
            True if recording succeeded, False otherwise
        """
        if metric_name not in self.metrics:
            self.logger.warning(f"Unknown metric: {metric_name}")
            return False

        try:
            # Create metric value object
            metric_value = MetricValue(
                metric_id=metric_name,
                timestamp=time.time(),
                value=value,
                source_module=source_module,
                context=context or {}
            )

            # Store the value
            with self.lock:
                # Add to history
                self.metric_histories[metric_name].append(metric_value)

                # Update current value
                self.current_values[metric_name] = metric_value

                # Update smoothed value
                smoothing_factor = self.metrics[metric_name].smoothing_factor
                if metric_name in self.smoothed_values:
                    self.smoothed_values[metric_name] = (
                        smoothing_factor * value +
                        (1 - smoothing_factor) * self.smoothed_values[metric_name]
                    )
                else:
                    self.smoothed_values[metric_name] = value

            return True
        except Exception as e:
            self.logger.error(f"Error recording metric {metric_name}: {str(e)}")
            return False

    def get_current_value(self, metric_name: str) -> Optional[float]:
        """
        Get the current value of a metric.

        Args:
            metric_name: Name of the metric

        Returns:
            Current value, or None if the metric doesn't exist
        """
        with self.lock:
            metric_value = self.current_values.get(metric_name)
            return metric_value.value if metric_value else None

    def get_smoothed_value(self, metric_name: str) -> Optional[float]:
        """
        Get the smoothed value of a metric.

        Args:
            metric_name: Name of the metric

        Returns:
            Smoothed value, or None if the metric doesn't exist
        """
        with self.lock:
            return self.smoothed_values.get(metric_name)

    def get_metric_history(self, metric_name: str,
                         time_window: Optional[float] = None) -> List[MetricValue]:
        """
        Get the history of a metric over a time window.

        Args:
            metric_name: Name of the metric
            time_window: Optional time window in seconds (if None, return all history)

        Returns:
            List of metric values within the time window
        """
        with self.lock:
            if metric_name not in self.metric_histories:
                return []

            history = list(self.metric_histories[metric_name])

            if time_window is not None:
                cutoff_time = time.time() - time_window
                history = [v for v in history if v.timestamp >= cutoff_time]

            return history

    def get_active_alerts(self) -> List[MonitorAlert]:
        """Get all currently active alerts"""
        with self.lock:
            return list(self.active_alerts.values())

    def get_alert_history(self, time_window: Optional[float] = None) -> List[MonitorAlert]:
        """
        Get alert history over a time window.

        Args:
            time_window: Optional time window in seconds (if None, return all history)

        Returns:
            List of alerts within the time window
        """
        with self.lock:
            alerts = list(self.alert_history)

            if time_window is not None:
                cutoff_time = time.time() - time_window
                alerts = [a for a in alerts if a.timestamp >= cutoff_time]

            return alerts

    def get_latest_report(self) -> Optional[PerformanceReport]:
        """Get the most recent performance report"""
        with self.lock:
            return self.reports[-1] if self.reports else None

    def get_reports(self, time_window: Optional[float] = None) -> List[PerformanceReport]:
        """
        Get performance reports over a time window.

        Args:
            time_window: Optional time window in seconds (if None, return all reports)

        Returns:
            List of performance reports within the time window
        """
        with self.lock:
            reports = list(self.reports)

            if time_window is not None:
                cutoff_time = time.time() - time_window
                reports = [r for r in reports if r.timestamp >= cutoff_time]

            return reports

    def register_custom_collector(self, module_name: str,
                                collector_func: Callable[[], Dict[str, float]]) -> bool:
        """
        Register a custom metric collector function for a module.

        Args:
            module_name: Name of the module
            collector_func: Function that returns a dict mapping metric names to values

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            self.custom_collectors[module_name].append(collector_func)
            self.logger.info(f"Registered custom collector for module: {module_name}")
            return True

    def trigger_custom_report(self, metrics_filter: Optional[Set[str]] = None,
                            time_window: float = 300.0) -> PerformanceReport:
        """
        Generate a custom performance report on demand.

        Args:
            metrics_filter: Optional set of metric names to include
            time_window: Time window in seconds for metrics analysis

        Returns:
            Generated performance report
        """
        # Collect metrics within time window
        metrics_data = self._collect_metrics_data(time_window, metrics_filter)

        # Detect bottlenecks
        bottlenecks = self._detect_bottlenecks(metrics_data)

        # Get resource usage
        resource_usage = self._get_current_resource_usage()

        # Estimate cognitive load
        cognitive_load = self._estimate_cognitive_load()

        # Get active alerts
        alerts = list(self.active_alerts.values())

        # Generate recommendations
        recommendations = self._generate_recommendations(
            metrics_data, bottlenecks, resource_usage, alerts
        )

        # Create report
        report = PerformanceReport(
            timestamp=time.time(),
            time_period=f"last_{int(time_window)}s",
            metrics_summary=metrics_data,
            bottlenecks=bottlenecks,
            resource_usage=resource_usage,
            cognitive_load=cognitive_load,
            alerts=alerts,
            recommendations=recommendations
        )

        # Store the report
        with self.lock:
            self.reports.append(report)

        return report

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.confidence_evaluator:
                dependencies.append("confidence_evaluator")

            self.nexus_core.register_module(
                name="cognitive_monitor",
                module=self,
                dependencies=dependencies
            )

            # Get all registered modules
            modules = self.nexus_core.list_modules()
            self.system_info.modules.update(modules)

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _subscribe_to_events(self) -> None:
        """Subscribe to events from NexusCore and other modules"""
        if not self.nexus_core:
            return

        # In a real implementation, this would set up event subscriptions
        # For now, it's just a placeholder
        self.logger.info("Event subscriptions set up")

    def _register_standard_metrics(self) -> None:
        """Register standard system and cognitive metrics"""
        # System resource metrics
        self.register_metric(MetricDefinition(
            name="system.cpu_usage",
            description="Overall CPU usage percentage",
            type=MetricType.RESOURCE,
            unit="%",
            priority=MetricPriority.HIGH,
            alert_threshold_high=90.0
        ))

        self.register_metric(MetricDefinition(
            name="system.memory_usage",
            description="Overall memory usage percentage",
            type=MetricType.RESOURCE,
            unit="%",
            priority=MetricPriority.HIGH,
            alert_threshold_high=85.0
        ))

        self.register_metric(MetricDefinition(
            name="system.memory_available",
            description="Available system memory",
            type=MetricType.RESOURCE,
            unit="MB",
            priority=MetricPriority.MEDIUM,
            alert_threshold_low=500.0
        ))

        # Core metrics
        self.register_metric(MetricDefinition(
            name="core.message_queue_size",
            description="Size of the NexusCore message queue",
            type=MetricType.ACTIVITY,
            unit="messages",
            priority=MetricPriority.HIGH,
            alert_threshold_high=100
        ))

        self.register_metric(MetricDefinition(
            name="core.messages_per_second",
            description="Rate of message processing",
            type=MetricType.PERFORMANCE,
            unit="msg/s",
            priority=MetricPriority.MEDIUM
        ))

        self.register_metric(MetricDefinition(
            name="core.active_modules",
            description="Number of active modules",
            type=MetricType.ACTIVITY,
            unit="modules",
            priority=MetricPriority.MEDIUM
        ))

        # Performance metrics
        self.register_metric(MetricDefinition(
            name="performance.response_time",
            description="Average response time for requests",
            type=MetricType.PERFORMANCE,
            unit="ms",
            priority=MetricPriority.HIGH,
            alert_threshold_high=5000.0
        ))

        self.register_metric(MetricDefinition(
            name="performance.throughput",
            description="Overall system throughput",
            type=MetricType.PERFORMANCE,
            unit="ops/s",
            priority=MetricPriority.MEDIUM
        ))

        # Cognitive metrics
        self.register_metric(MetricDefinition(
            name="cognitive.working_memory_load",
            description="Load on working memory",
            type=MetricType.COGNITIVE_LOAD,
            unit="%",
            priority=MetricPriority.HIGH,
            derived=True,
            alert_threshold_high=90.0
        ))

        self.register_metric(MetricDefinition(
            name="cognitive.semantic_memory_access_rate",
            description="Rate of semantic memory accesses",
            type=MetricType.ACTIVITY,
            unit="accesses/s",
            priority=MetricPriority.MEDIUM
        ))

        self.register_metric(MetricDefinition(
            name="cognitive.reasoning_confidence",
            description="Average confidence in reasoning results",
            type=MetricType.QUALITY,
            unit="%",
            priority=MetricPriority.HIGH,
            alert_threshold_low=70.0
        ))

        self.register_metric(MetricDefinition(
            name="cognitive.learning_rate",
            description="Rate of new knowledge acquisition",
            type=MetricType.PERFORMANCE,
            unit="items/min",
            priority=MetricPriority.MEDIUM
        ))

        # Stability metrics
        self.register_metric(MetricDefinition(
            name="stability.error_rate",
            description="Rate of errors across all modules",
            type=MetricType.STABILITY,
            unit="errors/min",
            priority=MetricPriority.CRITICAL,
            alert_threshold_high=5.0
        ))

        self.register_metric(MetricDefinition(
            name="stability.module_restarts",
            description="Number of module restarts",
            type=MetricType.STABILITY,
            unit="restarts",
            priority=MetricPriority.HIGH,
            alert_threshold_high=3.0
        ))

    def _monitoring_loop(self) -> None:
        """Main monitoring loop that collects metrics at regular intervals"""
        self.logger.info("Monitoring loop started")

        last_collection_times = defaultdict(float)

        while not self.stop_event.is_set():
            try:
                current_time = time.time()

                # Collect system metrics
                self._collect_system_metrics()

                # Collect metrics from each module based on sampling interval
                for metric_name, metric_def in self.metrics.items():
                    if current_time - last_collection_times[metric_name] >= metric_def.sampling_interval:
                        self._collect_metric(metric_name, metric_def)
                        last_collection_times[metric_name] = current_time

                # Collect metrics from custom collectors
                self._run_custom_collectors()

                # Collect derived metrics
                self._calculate_derived_metrics()

                # Sleep until next collection cycle
                time.sleep(self.monitoring_interval)

            except Exception as e:
                self.logger.error(f"Error in monitoring loop: {str(e)}")
                time.sleep(5.0)  # Sleep longer on error

    def _alert_check_loop(self) -> None:
        """Loop that checks for metrics crossing alert thresholds"""
        self.logger.info("Alert check loop started")

        while not self.stop_event.is_set():
            try:
                # Check all metrics against their thresholds
                self._check_alert_thresholds()

                # Sleep until next check
                time.sleep(self.alert_check_interval)

            except Exception as e:
                self.logger.error(f"Error in alert check loop: {str(e)}")
                time.sleep(5.0)  # Sleep longer on error

    def _report_generation_loop(self) -> None:
        """Loop that generates periodic performance reports"""
        self.logger.info("Report generation loop started")

        next_report_time = time.time() + self.report_interval

        while not self.stop_event.is_set():
            try:
                current_time = time.time()

                if current_time >= next_report_time:
                    # Generate a standard report
                    self._generate_periodic_report()
                    next_report_time = current_time + self.report_interval

                # Sleep for a bit
                time.sleep(min(30.0, next_report_time - current_time))

            except Exception as e:
                self.logger.error(f"Error in report generation loop: {str(e)}")
                time.sleep(5.0)  # Sleep longer on error
                next_report_time = time.time() + self.report_interval

    def _collect_system_metrics(self) -> None:
        """Collect basic system metrics"""
        try:
            # CPU usage
            cpu_percent = psutil.cpu_percent(interval=None)
            self.record_metric("system.cpu_usage", cpu_percent, "system")

            # Memory usage
            memory = psutil.virtual_memory()
            memory_percent = memory.percent
            memory_available_mb = memory.available / (1024 * 1024)

            self.record_metric("system.memory_usage", memory_percent, "system")
            self.record_metric("system.memory_available", memory_available_mb, "system")

            # Collect NexusCore metrics if available
            if self.nexus_core:
                try:
                    core_status = self.nexus_core.get_status()

                    # Message queue size
                    if "message_queue_size" in core_status:
                        self.record_metric(
                            "core.message_queue_size",
                            core_status["message_queue_size"],
                            "nexus_core"
                        )

                    # Active modules
                    if "active_modules" in core_status:
                        self.record_metric(
                            "core.active_modules",
                            len(core_status["active_modules"]),
                            "nexus_core"
                        )

                except Exception as e:
                    self.logger.warning(f"Error collecting NexusCore metrics: {str(e)}")

        except Exception as e:
            self.logger.error(f"Error collecting system metrics: {str(e)}")

    def _collect_metric(self, metric_name: str, metric_def: MetricDefinition) -> None:
        """
        Collect a specific metric based on its definition.
        This would typically dispatch to specialized collection methods.
        """
        # Most metrics are collected directly from modules
        # This method handles special cases or polling metrics
        pass

    def _run_custom_collectors(self) -> None:
        """Run all registered custom metric collectors"""
        for module_name, collectors in self.custom_collectors.items():
            for collector in collectors:
                try:
                    metrics = collector()
                    for metric_name, value in metrics.items():
                        if metric_name in self.metrics:
                            self.record_metric(metric_name, value, module_name)
                        else:
                            self.logger.warning(f"Unknown metric from collector: {metric_name}")

                except Exception as e:
                    self.logger.error(f"Error in custom collector for {module_name}: {str(e)}")

    def _calculate_derived_metrics(self) -> None:
        """Calculate metrics that are derived from other metrics"""
        # Example: Calculate messages per second from queue size changes
        try:
            # Working memory load
            # (This would typically use actual working memory metrics)
            # Here we'll just use a placeholder calculation
            if "system.memory_usage" in self.current_values:
                memory_usage = self.current_values["system.memory_usage"].value
                # Simplified model relating system memory to cognitive load
                working_memory_load = min(100.0, memory_usage * 1.2)
                self.record_metric("cognitive.working_memory_load", working_memory_load, "monitor")

            # Message processing rate
            message_queue_history = self.get_metric_history("core.message_queue_size", 10.0)
            if len(message_queue_history) >= 2:
                oldest = message_queue_history[0]
                newest = message_queue_history[-1]
                time_diff = newest.timestamp - oldest.timestamp
                if time_diff > 0:
                    queue_diff = oldest.value - newest.value  # Processed = decreased queue
                    if queue_diff > 0:  # Only if queue decreased
                        rate = queue_diff / time_diff
                        self.record_metric("core.messages_per_second", rate, "monitor")

        except Exception as e:
            self.logger.error(f"Error calculating derived metrics: {str(e)}")

    def _check_alert_thresholds(self) -> None:
        """Check all metrics against their alert thresholds"""
        try:
            # Check each metric with defined thresholds
            for metric_name, metric_def in self.metrics.items():
                current_value = self.get_current_value(metric_name)
                if current_value is None:
                    continue

                # Check high threshold
                if (metric_def.alert_threshold_high is not None and
                    current_value > metric_def.alert_threshold_high):
                    self._trigger_alert(
                        metric_name,
                        current_value,
                        metric_def.alert_threshold_high,
                        "high",
                        f"{metric_name} exceeded high threshold: {current_value} > {metric_def.alert_threshold_high} {metric_def.unit}"
                    )
                else:
                    # Check if there's an active high alert to resolve
                    self._resolve_alert(metric_name, "high")

                # Check low threshold
                if (metric_def.alert_threshold_low is not None and
                    current_value < metric_def.alert_threshold_low):
                    self._trigger_alert(
                        metric_name,
                        current_value,
                        metric_def.alert_threshold_low,
                        "low",
                        f"{metric_name} below low threshold: {current_value} < {metric_def.alert_threshold_low} {metric_def.unit}"
                    )
                else:
                    # Check if there's an active low alert to resolve
                    self._resolve_alert(metric_name, "low")

        except Exception as e:
            self.logger.error(f"Error checking alert thresholds: {str(e)}")

    def _trigger_alert(self, metric_name: str, current_value: float,
                      threshold: float, threshold_type: str, message: str) -> None:
        """
        Trigger an alert for a metric crossing a threshold.

        Args:
            metric_name: Name of the metric
            current_value: Current value of the metric
            threshold: Threshold that was crossed
            threshold_type: Type of threshold ('high' or 'low')
            message: Alert message
        """
        # Determine severity based on metric priority
        metric_def = self.metrics.get(metric_name)
        if not metric_def:
            return

        severity_map = {
            MetricPriority.CRITICAL: "critical",
            MetricPriority.HIGH: "error",
            MetricPriority.MEDIUM: "warning",
            MetricPriority.LOW: "info",
            MetricPriority.DEBUG: "debug"
        }
        severity = severity_map.get(metric_def.priority, "warning")

        # Create alert ID
        alert_id = f"{metric_name}_{threshold_type}_{int(time.time())}"

        # Check if this alert is already active
        existing_alert_id = f"{metric_name}_{threshold_type}"
        if existing_alert_id in self.active_alerts:
            # Update existing alert
            alert = self.active_alerts[existing_alert_id]
            alert.timestamp = time.time()
            alert.value = current_value
            alert.message = message

            # Log the continuation of the alert
            self.logger.warning(f"Alert continuing: {message}")
            return

        # Create new alert
        alert = MonitorAlert(
            alert_id=alert_id,
            metric_id=metric_name,
            timestamp=time.time(),
            severity=severity,
            message=message,
            value=current_value,
            threshold=threshold,
            is_active=True
        )

        # Store the alert
        with self.lock:
            self.active_alerts[existing_alert_id] = alert
            self.alert_history.append(alert)

        # Log the alert
        log_method = getattr(self.logger, severity, self.logger.warning)
        log_method(f"Alert triggered: {message}")

        # Notify subscribers
        self._notify_alert(alert)

    def _resolve_alert(self, metric_name: str, threshold_type: str) -> None:
        """
        Resolve an active alert when the metric returns to normal range.

        Args:
            metric_name: Name of the metric
            threshold_type: Type of threshold ('high' or 'low')
        """
        # Check if this alert is active
        existing_alert_id = f"{metric_name}_{threshold_type}"
        if existing_alert_id not in self.active_alerts:
            return

        # Get the alert
        alert = self.active_alerts[existing_alert_id]

        # Mark as resolved
        alert.is_active = False

        # Remove from active alerts
        with self.lock:
            del self.active_alerts[existing_alert_id]

        # Log the resolution
        self.logger.info(f"Alert resolved: {alert.message}")

        # Notify subscribers
        self._notify_alert_resolution(alert)

    def _notify_alert(self, alert: MonitorAlert) -> None:
        """
        Notify subscribers about a new alert.

        Args:
            alert: The alert that was triggered
        """
        # In a real implementation, this would notify subscribers
        # Here it's just a placeholder
        if self.nexus_core:
            try:
                # Example: Send a message through NexusCore
                self.nexus_core.send_message(
                    source="cognitive_monitor",
                    target="*",  # Broadcast to all modules
                    message_type="alert",
                    content={
                        "alert_id": alert.alert_id,
                        "metric_id": alert.metric_id,
                        "timestamp": alert.timestamp,
                        "severity": alert.severity,
                        "message": alert.message,
                        "value": alert.value,
                        "threshold": alert.threshold
                    }
                )
            except Exception as e:
                self.logger.error(f"Error notifying alert: {str(e)}")

    def _notify_alert_resolution(self, alert: MonitorAlert) -> None:
        """
        Notify subscribers about an alert resolution.

        Args:
            alert: The alert that was resolved
        """
        # In a real implementation, this would notify subscribers
        # Here it's just a placeholder
        if self.nexus_core:
            try:
                # Example: Send a message through NexusCore
                self.nexus_core.send_message(
                    source="cognitive_monitor",
                    target="*",  # Broadcast to all modules
                    message_type="alert_resolution",
                    content={
                        "alert_id": alert.alert_id,
                        "metric_id": alert.metric_id,
                        "timestamp": time.time(),
                        "message": f"Resolved: {alert.message}"
                    }
                )
            except Exception as e:
                self.logger.error(f"Error notifying alert resolution: {str(e)}")

    def _generate_periodic_report(self) -> None:
        """Generate a standard periodic performance report"""
        try:
            # Generate report for the last reporting interval
            report = self.trigger_custom_report(time_window=self.report_interval)

            # Log report generation
            self.logger.info(f"Generated periodic performance report at {datetime.datetime.now()}")

            # Notify subscribers
            self._notify_report(report)

        except Exception as e:
            self.logger.error(f"Error generating periodic report: {str(e)}")

    def _notify_report(self, report: PerformanceReport) -> None:
        """
        Notify subscribers about a new performance report.

        Args:
            report: The performance report
        """
        # In a real implementation, this would notify subscribers
        # Here it's just a placeholder
        if self.nexus_core:
            try:
                # Example: Send a message through NexusCore
                self.nexus_core.send_message(
                    source="cognitive_monitor",
                    target="metacognition",  # Send to metacognition modules
                    message_type="performance_report",
                    content={
                        "timestamp": report.timestamp,
                        "time_period": report.time_period,
                        "bottlenecks": report.bottlenecks,
                        "resource_usage": report.resource_usage,
                        "cognitive_load": report.cognitive_load,
                        "active_alerts": len(report.alerts),
                        "recommendations": report.recommendations
                    }
                )
            except Exception as e:
                self.logger.error(f"Error notifying report: {str(e)}")

    def _collect_metrics_data(self, time_window: float,
                             metrics_filter: Optional[Set[str]] = None) -> Dict[str, Dict[str, float]]:
        """
        Collect metrics data for a report.

        Args:
            time_window: Time window in seconds
            metrics_filter: Optional set of metric names to include

        Returns:
            Dictionary mapping metric names to summary statistics
        """
        result = {}

        # Determine which metrics to include
        metrics_to_include = set(self.metrics.keys())
        if metrics_filter:
            metrics_to_include = metrics_to_include.intersection(metrics_filter)

        # For each metric, collect summary statistics
        for metric_name in metrics_to_include:
            history = self.get_metric_history(metric_name, time_window)

            if not history:
                continue

            values = [entry.value for entry in history]

            # Calculate summary statistics
            summary = {
                "avg": statistics.mean(values) if values else 0.0,
                "min": min(values) if values else 0.0,
                "max": max(values) if values else 0.0,
                "current": values[-1] if values else 0.0
            }

            # Calculate standard deviation if there are enough values
            if len(values) > 1:
                summary["std"] = statistics.stdev(values)
            else:
                summary["std"] = 0.0

            # Add to result
            result[metric_name] = summary

        return result

    def _detect_bottlenecks(self, metrics_data: Dict[str, Dict[str, float]]) -> List[Dict[str, Any]]:
        """
        Detect performance bottlenecks based on metrics.

        Args:
            metrics_data: Dictionary of metrics summary data

        Returns:
            List of detected bottlenecks with details
        """
        bottlenecks = []

        # Check CPU usage
        if "system.cpu_usage" in metrics_data:
            cpu_usage = metrics_data["system.cpu_usage"]["current"]
            if cpu_usage > self.bottleneck_thresholds.get("cpu_usage", 90.0):
                bottlenecks.append({
                    "type": "resource",
                    "resource": "cpu",
                    "severity": "high" if cpu_usage > 95.0 else "medium",
                    "value": cpu_usage,
                    "threshold": self.bottleneck_thresholds.get("cpu_usage", 90.0),
                    "description": f"CPU usage is {cpu_usage:.1f}%, indicating potential CPU bottleneck"
                })

        # Check memory usage
        if "system.memory_usage" in metrics_data:
            memory_usage = metrics_data["system.memory_usage"]["current"]
            if memory_usage > self.bottleneck_thresholds.get("memory_usage", 85.0):
                bottlenecks.append({
                    "type": "resource",
                    "resource": "memory",
                    "severity": "high" if memory_usage > 95.0 else "medium",
                    "value": memory_usage,
                    "threshold": self.bottleneck_thresholds.get("memory_usage", 85.0),
                    "description": f"Memory usage is {memory_usage:.1f}%, indicating potential memory bottleneck"
                })

        # Check message queue growth
        if "core.message_queue_size" in metrics_data:
            queue_size = metrics_data["core.message_queue_size"]["current"]
            queue_avg = metrics_data["core.message_queue_size"]["avg"]

            if queue_size > 50 and queue_size > queue_avg * 1.5:
                bottlenecks.append({
                    "type": "processing",
                    "resource": "message_queue",
                    "severity": "medium",
                    "value": queue_size,
                    "threshold": queue_avg * 1.5,
                    "description": f"Message queue size ({queue_size}) is growing, indicating processing bottleneck"
                })

        # Check response time
        if "performance.response_time" in metrics_data:
            response_time = metrics_data["performance.response_time"]["current"]
            if response_time > self.bottleneck_thresholds.get("response_time", 2.0):
                bottlenecks.append({
                    "type": "performance",
                    "resource": "response_time",
                    "severity": "high" if response_time > 5.0 else "medium",
                    "value": response_time,
                    "threshold": self.bottleneck_thresholds.get("response_time", 2.0),
                    "description": f"Response time is {response_time:.2f}s, indicating performance bottleneck"
                })

        # Check working memory load
        if "cognitive.working_memory_load" in metrics_data:
            memory_load = metrics_data["cognitive.working_memory_load"]["current"]
            if memory_load > 85.0:
                bottlenecks.append({
                    "type": "cognitive",
                    "resource": "working_memory",
                    "severity": "high" if memory_load > 95.0 else "medium",
                    "value": memory_load,
                    "threshold": 85.0,
                    "description": f"Working memory load is {memory_load:.1f}%, indicating cognitive bottleneck"
                })

        return bottlenecks

    def _get_current_resource_usage(self) -> Dict[str, float]:
        """
        Get current resource usage for various system resources.

        Returns:
            Dictionary mapping resource names to usage values
        """
        try:
            # System resources
            cpu_percent = psutil.cpu_percent(interval=None)
            memory = psutil.virtual_memory()

            # Disk usage
            disk = psutil.disk_usage('/')

            result = {
                "cpu": cpu_percent,
                "memory": memory.percent,
                "memory_available_mb": memory.available / (1024 * 1024),
                "disk": disk.percent,
                "disk_free_gb": disk.free / (1024 * 1024 * 1024)
            }

            # Core resources if available
            if self.nexus_core:
                try:
                    core_status = self.nexus_core.get_status()

                    # Add relevant core metrics
                    if "message_queue_size" in core_status:
                        result["message_queue"] = core_status["message_queue_size"]

                    if "active_modules" in core_status:
                        result["active_modules"] = len(core_status["active_modules"])

                except Exception as e:
                    self.logger.warning(f"Error getting NexusCore status: {str(e)}")

            return result

        except Exception as e:
            self.logger.error(f"Error getting resource usage: {str(e)}")
            return {"error": 1.0}

    def _estimate_cognitive_load(self) -> Dict[str, float]:
        """
        Estimate cognitive load across different cognitive resources.

        Returns:
            Dictionary mapping cognitive resource names to load values
        """
        result = {}

        # Working memory load
        working_memory_load = self.get_current_value("cognitive.working_memory_load")
        if working_memory_load is not None:
            result["working_memory"] = working_memory_load

        # Processing load (based on CPU)
        cpu_usage = self.get_current_value("system.cpu_usage")
        if cpu_usage is not None:
            result["processing"] = min(100.0, cpu_usage * 1.1)  # Slightly higher than CPU

        # Attention load (based on number of active modules and tasks)
        if self.nexus_core:
            try:
                core_status = self.nexus_core.get_status()
                active_modules = len(core_status.get("active_modules", []))

                # Estimate attention load based on active modules
                # This is a simplified model
                attention_load = min(100.0, active_modules * 10.0)  # 10% per active module
                result["attention"] = attention_load

            except Exception:
                pass

        # Learning load (based on learning metrics if available)
        learning_rate = self.get_current_value("cognitive.learning_rate")
        if learning_rate is not None:
            # Simplified model: higher learning rate = higher load
            result["learning"] = min(100.0, learning_rate * 5.0)  # 5% per item/min

        # Overall cognitive load (weighted average)
        if result:
            weights = {
                "working_memory": 0.4,
                "processing": 0.3,
                "attention": 0.2,
                "learning": 0.1
            }

            weighted_sum = 0.0
            weight_sum = 0.0

            for resource, load in result.items():
                if resource in weights:
                    weighted_sum += load * weights[resource]
                    weight_sum += weights[resource]

            if weight_sum > 0:
                result["overall"] = weighted_sum / weight_sum

        return result

    def _generate_recommendations(self, metrics_data: Dict[str, Dict[str, float]],
                                bottlenecks: List[Dict[str, Any]],
                                resource_usage: Dict[str, float],
                                alerts: List[MonitorAlert]) -> List[Dict[str, Any]]:
        """
        Generate system recommendations based on current state.

        Args:
            metrics_data: Dictionary of metrics data
            bottlenecks: List of detected bottlenecks
            resource_usage: Dictionary of resource usage
            alerts: List of active alerts

        Returns:
            List of recommendations
        """
        recommendations = []

        # CPU recommendations
        if "cpu" in resource_usage and resource_usage["cpu"] > 90.0:
            recommendations.append({
                "type": "resource_management",
                "target": "system",
                "priority": "high",
                "action": "reduce_load",
                "description": "Reduce computational load by deferring non-critical tasks"
            })

        # Memory recommendations
        if "memory" in resource_usage and resource_usage["memory"] > 85.0:
            recommendations.append({
                "type": "resource_management",
                "target": "system",
                "priority": "high",
                "action": "free_memory",
                "description": "Free memory by clearing caches or offloading data"
            })

        # Message queue recommendations
        if "message_queue" in resource_usage and resource_usage["message_queue"] > 100:
            recommendations.append({
                "type": "processing",
                "target": "nexus_core",
                "priority": "medium",
                "action": "increase_processing_threads",
                "description": "Increase message processing capacity to reduce queue backlog"
            })

        # Working memory recommendations
        working_memory_load = self.get_current_value("cognitive.working_memory_load")
        if working_memory_load is not None and working_memory_load > 90.0:
            recommendations.append({
                "type": "cognitive",
                "target": "working_memory",
                "priority": "high",
                "action": "consolidate_memory",
                "description": "Consolidate working memory items to reduce cognitive load"
            })

        # Error rate recommendations
        error_rate = self.get_current_value("stability.error_rate")
        if error_rate is not None and error_rate > 5.0:
            recommendations.append({
                "type": "stability",
                "target": "system",
                "priority": "critical",
                "action": "investigate_errors",
                "description": "Investigate and address high error rate across modules"
            })

        # Bottleneck-specific recommendations
        for bottleneck in bottlenecks:
            if bottleneck["type"] == "resource" and bottleneck["resource"] == "cpu":
                # Check if we already have a CPU recommendation
                if not any(r["target"] == "system" and r["action"] == "reduce_load" for r in recommendations):
                    recommendations.append({
                        "type": "resource_management",
                        "target": "system",
                        "priority": "high" if bottleneck["severity"] == "high" else "medium",
                        "action": "reduce_load",
                        "description": "Reduce CPU load by limiting parallel tasks"
                    })

            elif bottleneck["type"] == "cognitive" and bottleneck["resource"] == "working_memory":
                # Check if we already have a working memory recommendation
                if not any(r["target"] == "working_memory" for r in recommendations):
                    recommendations.append({
                        "type": "cognitive",
                        "target": "working_memory",
                        "priority": "high" if bottleneck["severity"] == "high" else "medium",
                        "action": "prune_memory",
                        "description": "Prune less relevant items from working memory"
                    })

        # Add module-specific recommendations based on active alerts
        for alert in alerts:
            if "core.message_queue_size" in alert.metric_id and alert.severity == "error":
                if not any(r["action"] == "increase_processing_threads" for r in recommendations):
                    recommendations.append({
                        "type": "processing",
                        "target": "nexus_core",
                        "priority": "high",
                        "action": "increase_processing_threads",
                        "description": "Increase message processing capacity to address queue overflow"
                    })

            elif "stability.error_rate" in alert.metric_id and alert.severity == "critical":
                if not any(r["action"] == "investigate_errors" for r in recommendations):
                    recommendations.append({
                        "type": "stability",
                        "target": "system",
                        "priority": "critical",
                        "action": "investigate_errors",
                        "description": "Critical error rate detected, investigate immediately"
                    })

        return recommendations

ModuleNotFoundError: No module named 'grandnexus'

# Confidence

In [ ]:
"""
confidence.py - Confidence Evaluation and Uncertainty Management for GrandNexus

This module provides a comprehensive framework for assessing the confidence and
uncertainty in GrandNexus's cognitive processes. It enables the system to be aware
of its own limitations, the reliability of its knowledge, and the certainty of its
inferences, providing a foundation for metacognitive awareness.

Key features:
- Multi-level confidence assessment for different cognitive processes
- Uncertainty propagation through inference chains
- Evidence-based confidence scoring
- Calibration of confidence metrics against observed outcomes
- Integration with memory systems to track confidence over time
- Decision-making support under uncertainty

The design emphasizes:
- Calibrated confidence scores that align with actual performance
- Integration with all cognitive modules (reasoning, learning, perception)
- Principled uncertainty management based on probability theory
- Transparency about confidence levels in system outputs
"""

import logging
import threading
import time
import uuid
import math
from typing import Dict, List, Any, Optional, Set, Tuple, Union, Callable
from enum import Enum, auto
from dataclasses import dataclass, field
from collections import defaultdict, deque
import statistics
import numpy as np

# Assuming these exist in GrandNexus architecture
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory

# Confidence source types
class ConfidenceSource(Enum):
    """Sources of confidence information"""
    PERCEPTION = auto()      # Confidence from perceptual processes
    REASONING = auto()       # Confidence from reasoning processes
    MEMORY = auto()          # Confidence from memory retrieval
    LEARNING = auto()        # Confidence from learning processes
    PRIOR = auto()           # Prior/background confidence
    COMBINED = auto()        # Combined from multiple sources
    EXTERNAL = auto()        # Provided externally (e.g., user feedback)

@dataclass
class ConfidenceScore:
    """Represents a confidence assessment for a specific item/process"""
    score_id: str
    source: ConfidenceSource
    value: float  # 0.0 to 1.0
    timestamp: float
    item_id: Optional[str] = None  # ID of the item being assessed
    module_name: str = ""
    context: Dict[str, Any] = field(default_factory=dict)
    evidence: List[Dict[str, Any]] = field(default_factory=list)
    uncertainty_type: Optional[str] = None  # e.g., "aleatoric", "epistemic"

@dataclass
class ConfidenceThresholds:
    """Thresholds for different confidence levels"""
    very_high: float = 0.95
    high: float = 0.8
    medium: float = 0.6
    low: float = 0.4
    very_low: float = 0.2

@dataclass
class ConfidenceStats:
    """Statistics about a set of confidence assessments"""
    count: int = 0
    mean: float = 0.0
    median: float = 0.0
    std_dev: float = 0.0
    min: float = 0.0
    max: float = 0.0
    calibration_error: float = 0.0  # Difference between confidence and actual accuracy

class ConfidenceEvaluator:
    """
    Central component for evaluating confidence across GrandNexus's cognitive processes.

    This module provides mechanisms to assess, track, and manage uncertainty in the
    system's knowledge and reasoning, enabling metacognitive awareness and appropriate
    action selection based on confidence levels.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 config: Optional[Dict[str, Any]] = None):
        """
        Initialize the confidence evaluator.

        Args:
            nexus_core: Reference to the NexusCore for system integration
            working_memory: Reference to working memory module
            semantic_memory: Reference to semantic memory module
            episodic_memory: Reference to episodic memory module
            config: Optional configuration dictionary
        """
        self.logger = logging.getLogger("GrandNexus.Metacognition.Confidence")
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory

        # Initialize configuration
        self.config = config or {}
        self.confidence_thresholds = ConfidenceThresholds(
            very_high=self.config.get("threshold_very_high", 0.95),
            high=self.config.get("threshold_high", 0.8),
            medium=self.config.get("threshold_medium", 0.6),
            low=self.config.get("threshold_low", 0.4),
            very_low=self.config.get("threshold_very_low", 0.2)
        )

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.initialized = False
        self.lock = threading.RLock()

        # Confidence history
        self.confidence_history: Dict[str, deque[ConfidenceScore]] = defaultdict(
            lambda: deque(maxlen=self.config.get("history_length", 1000))
        )

        # Aggregated confidence by module/source
        self.aggregated_confidence: Dict[str, Dict[str, Any]] = {}

        # Calibration data (predicted confidence vs. actual correctness)
        self.calibration_data: Dict[str, List[Tuple[float, bool]]] = defaultdict(list)

        # Confidence models for different modules/sources
        self.confidence_models: Dict[str, Any] = {}

        # Custom confidence evaluators (module_name -> list of evaluator functions)
        self.custom_evaluators: Dict[str, List[Callable[..., float]]] = defaultdict(list)

        self.logger.info(f"ConfidenceEvaluator initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the confidence evaluator"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Initialize baseline confidence models
                self._initialize_confidence_models()

                self.initialized = True
                self.logger.info("ConfidenceEvaluator initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"ConfidenceEvaluator initialization failed: {str(e)}")
                return False

    def record_confidence(self, source: ConfidenceSource, value: float,
                         module_name: str, item_id: Optional[str] = None,
                         context: Optional[Dict[str, Any]] = None,
                         evidence: Optional[List[Dict[str, Any]]] = None,
                         uncertainty_type: Optional[str] = None) -> str:
        """
        Record a confidence assessment.

        Args:
            source: Source of the confidence assessment
            value: Confidence value (0.0 to 1.0)
            module_name: Name of the module making the assessment
            item_id: Optional ID of the item being assessed
            context: Optional contextual information
            evidence: Optional evidence supporting the assessment
            uncertainty_type: Optional type of uncertainty

        Returns:
            ID of the recorded confidence score
        """
        if not self.initialized:
            self.initialize()

        # Validate confidence value
        value = max(0.0, min(1.0, value))  # Ensure between 0 and 1

        # Create confidence score object
        score_id = str(uuid.uuid4())
        score = ConfidenceScore(
            score_id=score_id,
            source=source,
            value=value,
            timestamp=time.time(),
            item_id=item_id,
            module_name=module_name,
            context=context or {},
            evidence=evidence or [],
            uncertainty_type=uncertainty_type
        )

        # Store the score
        with self.lock:
            # Add to history
            self.confidence_history[module_name].append(score)

            # Update aggregated confidence
            self._update_aggregated_confidence(score)

        return score_id

    def get_confidence(self, item_id: str) -> Optional[float]:
        """
        Get the most recent confidence score for an item.

        Args:
            item_id: ID of the item

        Returns:
            Confidence value (0.0 to 1.0), or None if not found
        """
        if not self.initialized:
            self.initialize()

        with self.lock:
            # Search for the most recent confidence score for this item
            most_recent = None
            most_recent_time = 0.0

            for module_history in self.confidence_history.values():
                for score in module_history:
                    if score.item_id == item_id and score.timestamp > most_recent_time:
                        most_recent = score
                        most_recent_time = score.timestamp

            return most_recent.value if most_recent else None

    def combine_confidence_scores(self, scores: List[ConfidenceScore],
                                 method: str = "weighted_average") -> float:
        """
        Combine multiple confidence scores into a single assessment.

        Args:
            scores: List of confidence scores to combine
            method: Method for combining scores ("weighted_average", "min", "max", etc.)

        Returns:
            Combined confidence value
        """
        if not scores:
            return 0.0

        if method == "weighted_average":
            # Calculate weighted average based on recency and source reliability
            weights = []
            values = []

            for score in scores:
                # Base weight
                weight = 1.0

                # Adjust weight based on recency (newer scores have higher weight)
                age = time.time() - score.timestamp
                recency_factor = max(0.5, math.exp(-age / 3600.0))  # Decay over hour
                weight *= recency_factor

                # Adjust weight based on source reliability
                source_reliability = self._get_source_reliability(score.source, score.module_name)
                weight *= source_reliability

                weights.append(weight)
                values.append(score.value)

            # Compute weighted average
            if sum(weights) > 0:
                return sum(w * v for w, v in zip(weights, values)) / sum(weights)
            else:
                return statistics.mean(values)

        elif method == "min":
            # Return the minimum confidence (most conservative)
            return min(score.value for score in scores)

        elif method == "max":
            # Return the maximum confidence (most optimistic)
            return max(score.value for score in scores)

        elif method == "geometric_mean":
            # Geometric mean (good for probabilities)
            return math.pow(math.prod(score.value for score in scores), 1.0 / len(scores))

        else:
            # Default to simple average
            return statistics.mean(score.value for score in scores)

    def evaluate_perception_confidence(self, perception_result: Any,
                                     context: Optional[Dict[str, Any]] = None) -> float:
        """
        Evaluate confidence in a perception result.

        Args:
            perception_result: Result from a perception process
            context: Optional contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        if not self.initialized:
            self.initialize()

        # Initialize context if needed
        if context is None:
            context = {}

        # Check if perception_result has a confidence attribute
        if hasattr(perception_result, "confidence"):
            return float(perception_result.confidence)

        # If it's a dictionary with confidence key
        if isinstance(perception_result, dict) and "confidence" in perception_result:
            return float(perception_result["confidence"])

        # Apply custom evaluators for perception
        for evaluator in self.custom_evaluators.get("perception", []):
            try:
                confidence = evaluator(perception_result, context)
                if confidence is not None:
                    return confidence
            except Exception as e:
                self.logger.warning(f"Error in custom perception evaluator: {str(e)}")

        # Default confidence model
        return self._default_perception_confidence(perception_result, context)

    def evaluate_reasoning_confidence(self, reasoning_result: Any,
                                    context: Optional[Dict[str, Any]] = None) -> float:
        """
        Evaluate confidence in a reasoning result.

        Args:
            reasoning_result: Result from a reasoning process
            context: Optional contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        if not self.initialized:
            self.initialize()

        # Initialize context if needed
        if context is None:
            context = {}

        # Check if reasoning_result has a confidence attribute
        if hasattr(reasoning_result, "confidence"):
            return float(reasoning_result.confidence)

        # If it's a dictionary with confidence key
        if isinstance(reasoning_result, dict) and "confidence" in reasoning_result:
            return float(reasoning_result["confidence"])

        # Apply custom evaluators for reasoning
        for evaluator in self.custom_evaluators.get("reasoning", []):
            try:
                confidence = evaluator(reasoning_result, context)
                if confidence is not None:
                    return confidence
            except Exception as e:
                self.logger.warning(f"Error in custom reasoning evaluator: {str(e)}")

        # Default confidence model
        return self._default_reasoning_confidence(reasoning_result, context)

    def evaluate_memory_confidence(self, memory_item: Any,
                                 context: Optional[Dict[str, Any]] = None) -> float:
        """
        Evaluate confidence in a memory retrieval.

        Args:
            memory_item: Retrieved memory item
            context: Optional contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        if not self.initialized:
            self.initialize()

        # Initialize context if needed
        if context is None:
            context = {}

        # Check if memory_item has a confidence or importance attribute
        if hasattr(memory_item, "confidence"):
            return float(memory_item.confidence)

        if hasattr(memory_item, "importance"):
            # Scale importance to confidence-like value
            return min(1.0, float(memory_item.importance) * 1.25)

        # If it's a dictionary with confidence key
        if isinstance(memory_item, dict):
            if "confidence" in memory_item:
                return float(memory_item["confidence"])
            if "importance" in memory_item:
                return min(1.0, float(memory_item["importance"]) * 1.25)

        # Apply custom evaluators for memory
        for evaluator in self.custom_evaluators.get("memory", []):
            try:
                confidence = evaluator(memory_item, context)
                if confidence is not None:
                    return confidence
            except Exception as e:
                self.logger.warning(f"Error in custom memory evaluator: {str(e)}")

        # Default confidence model
        return self._default_memory_confidence(memory_item, context)

    def propagate_confidence(self, input_confidences: List[float],
                           operation: str = "chain_rule") -> float:
        """
        Propagate confidence through a multi-step process.

        Args:
            input_confidences: List of confidence values for inputs
            operation: Type of confidence propagation operation

        Returns:
            Propagated confidence value
        """
        if not input_confidences:
            return 0.0

        if operation == "chain_rule":
            # Probabilistic chain rule (product of probabilities)
            return math.prod(input_confidences)

        elif operation == "noisy_or":
            # Noisy-OR model (1 - Product(1 - p))
            return 1.0 - math.prod(1.0 - p for p in input_confidences)

        elif operation == "min":
            # Weakest link model
            return min(input_confidences)

        elif operation == "max":
            # Strongest evidence model
            return max(input_confidences)

        elif operation == "average":
            # Simple average
            return statistics.mean(input_confidences)

        else:
            # Default to chain rule
            return math.prod(input_confidences)

    def register_calibration_data(self, module_name: str, predicted_confidence: float,
                                was_correct: bool) -> None:
        """
        Register actual outcomes to calibrate confidence assessments.

        Args:
            module_name: Name of the module that made the prediction
            predicted_confidence: The confidence that was assigned
            was_correct: Whether the prediction was actually correct
        """
        with self.lock:
            self.calibration_data[module_name].append((predicted_confidence, was_correct))

            # Update calibration statistics if we have enough data
            if len(self.calibration_data[module_name]) >= 10:
                self._update_calibration_stats(module_name)

    def get_calibration_stats(self, module_name: Optional[str] = None) -> Dict[str, Any]:
        """
        Get calibration statistics for confidence assessments.

        Args:
            module_name: Optional module name to get stats for (if None, get overall stats)

        Returns:
            Dictionary with calibration statistics
        """
        with self.lock:
            if module_name:
                # Return stats for specific module
                if module_name in self.aggregated_confidence:
                    return self.aggregated_confidence[module_name].get("calibration", {})
                return {}

            # Return overall stats
            overall_stats = {}

            # Combine calibration data from all modules
            all_data = []
            for module_data in self.calibration_data.values():
                all_data.extend(module_data)

            if all_data:
                # Calculate calibration metrics
                overall_stats = self._calculate_calibration_metrics(all_data)

            return overall_stats

    def register_custom_evaluator(self, module_name: str,
                                evaluator_func: Callable[..., float]) -> bool:
        """
        Register a custom confidence evaluator function.

        Args:
            module_name: Name of the module (e.g., "perception", "reasoning")
            evaluator_func: Function that takes (result, context) and returns confidence

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            self.custom_evaluators[module_name].append(evaluator_func)
            self.logger.info(f"Registered custom evaluator for module: {module_name}")
            return True

    def get_confidence_level(self, confidence_value: float) -> str:
        """
        Map a confidence value to a descriptive level.

        Args:
            confidence_value: Confidence value (0.0 to 1.0)

        Returns:
            String describing the confidence level
        """
        if confidence_value >= self.confidence_thresholds.very_high:
            return "very_high"
        elif confidence_value >= self.confidence_thresholds.high:
            return "high"
        elif confidence_value >= self.confidence_thresholds.medium:
            return "medium"
        elif confidence_value >= self.confidence_thresholds.low:
            return "low"
        else:
            return "very_low"

    def get_uncertainty_interval(self, confidence_value: float,
                               distribution: str = "gaussian") -> Tuple[float, float]:
        """
        Calculate an uncertainty interval around an estimate.

        Args:
            confidence_value: Confidence value (0.0 to 1.0)
            distribution: Probability distribution model

        Returns:
            Tuple of (lower_bound, upper_bound) representing uncertainty interval
        """
        # Convert confidence to standard deviation
        if distribution == "gaussian":
            # Convert confidence to standard deviation
            # Confidence of 1.0 -> std_dev of 0.01
            # Confidence of 0.5 -> std_dev of 0.25
            # Confidence of 0.0 -> std_dev of 0.5
            std_dev = 0.5 - (confidence_value * 0.49)

            # 95% confidence interval is approximately ±2 standard deviations
            interval = std_dev * 2.0

            # Limit the interval to [0, 1]
            lower_bound = max(0.0, 0.5 - interval)
            upper_bound = min(1.0, 0.5 + interval)

            return (lower_bound, upper_bound)

        elif distribution == "beta":
            # Use a Beta distribution shaped by confidence
            # High confidence -> concentrated distribution
            # Low confidence -> spread out distribution
            if confidence_value > 0.5:
                alpha = 2.0 + (confidence_value - 0.5) * 18.0  # 2 to 20
                beta = 2.0
            else:
                alpha = 2.0
                beta = 2.0 + (0.5 - confidence_value) * 18.0  # 2 to 20

            # Calculate 95% confidence interval
            # For a true implementation, use scipy.stats.beta.ppf
            # This is a rough approximation
            width = 2.0 / (alpha + beta)
            lower_bound = max(0.0, 0.5 - width)
            upper_bound = min(1.0, 0.5 + width)

            return (lower_bound, upper_bound)

        else:
            # Simple heuristic for unknown distributions
            interval = 1.0 - confidence_value
            lower_bound = max(0.0, 0.5 - interval / 2.0)
            upper_bound = min(1.0, 0.5 + interval / 2.0)

            return (lower_bound, upper_bound)

    def should_defer_decision(self, confidence_value: float,
                            importance: float = 0.5,
                            threshold: Optional[float] = None) -> bool:
        """
        Determine if a decision should be deferred due to low confidence.

        Args:
            confidence_value: Confidence value (0.0 to 1.0)
            importance: Importance of the decision (0.0 to 1.0)
            threshold: Optional explicit threshold (if None, calculated based on importance)

        Returns:
            True if the decision should be deferred, False otherwise
        """
        # If threshold is provided, use it directly
        if threshold is not None:
            return confidence_value < threshold

        # Calculate threshold based on importance
        # Higher importance -> higher confidence required
        adaptive_threshold = 0.5 + (importance * 0.3)

        return confidence_value < adaptive_threshold

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="confidence_evaluator",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _initialize_confidence_models(self) -> None:
        """Initialize baseline confidence models for different sources"""
        # This would typically load or configure confidence estimation models
        # For now, we use simple heuristic models
        self.confidence_models["perception"] = {"type": "heuristic"}
        self.confidence_models["reasoning"] = {"type": "heuristic"}
        self.confidence_models["memory"] = {"type": "heuristic"}
        self.confidence_models["combined"] = {"type": "weighted_average"}

    def _update_aggregated_confidence(self, score: ConfidenceScore) -> None:
        """
        Update aggregated confidence statistics for a module/source.

        Args:
            score: The confidence score to incorporate
        """
        module_name = score.module_name
        source_name = score.source.name

        # Initialize if not exists
        if module_name not in self.aggregated_confidence:
            self.aggregated_confidence[module_name] = {
                "sources": {},
                "overall": {
                    "count": 0,
                    "sum": 0.0,
                    "sum_sq": 0.0,
                    "min": 1.0,
                    "max": 0.0,
                    "last_update": 0.0
                },
                "calibration": {}
            }

        # Initialize source if not exists
        if source_name not in self.aggregated_confidence[module_name]["sources"]:
            self.aggregated_confidence[module_name]["sources"][source_name] = {
                "count": 0,
                "sum": 0.0,
                "sum_sq": 0.0,
                "min": 1.0,
                "max": 0.0,
                "last_update": 0.0
            }

        # Update source statistics
        source_stats = self.aggregated_confidence[module_name]["sources"][source_name]
        source_stats["count"] += 1
        source_stats["sum"] += score.value
        source_stats["sum_sq"] += score.value ** 2
        source_stats["min"] = min(source_stats["min"], score.value)
        source_stats["max"] = max(source_stats["max"], score.value)
        source_stats["last_update"] = score.timestamp

        # Update overall statistics
        overall_stats = self.aggregated_confidence[module_name]["overall"]
        overall_stats["count"] += 1
        overall_stats["sum"] += score.value
        overall_stats["sum_sq"] += score.value ** 2
        overall_stats["min"] = min(overall_stats["min"], score.value)
        overall_stats["max"] = max(overall_stats["max"], score.value)
        overall_stats["last_update"] = score.timestamp

        # Calculate derived statistics
        self._calculate_derived_statistics(source_stats)
        self._calculate_derived_statistics(overall_stats)

    def _calculate_derived_statistics(self, stats: Dict[str, Any]) -> None:
        """
        Calculate derived statistics from raw counts/sums.

        Args:
            stats: Dictionary of statistics to update
        """
        if stats["count"] > 0:
            stats["mean"] = stats["sum"] / stats["count"]

            # Variance
            if stats["count"] > 1:
                variance = (stats["sum_sq"] / stats["count"]) - (stats["mean"] ** 2)
                stats["std_dev"] = math.sqrt(max(0.0, variance))
            else:
                stats["std_dev"] = 0.0

    def _update_calibration_stats(self, module_name: str) -> None:
        """
        Update calibration statistics for a module.

        Args:
            module_name: Name of the module to update
        """
        if module_name not in self.calibration_data:
            return

        # Get calibration data
        data = self.calibration_data[module_name]
        if not data:
            return

        # Calculate calibration metrics
        calibration_metrics = self._calculate_calibration_metrics(data)

        # Update module's calibration stats
        if module_name in self.aggregated_confidence:
            self.aggregated_confidence[module_name]["calibration"] = calibration_metrics

    def _calculate_calibration_metrics(self, data: List[Tuple[float, bool]]) -> Dict[str, Any]:
        """
        Calculate calibration metrics from prediction-outcome pairs.

        Args:
            data: List of (predicted_confidence, was_correct) pairs

        Returns:
            Dictionary with calibration metrics
        """
        if not data:
            return {}

        # Basic metrics
        total = len(data)
        correct = sum(1 for _, correct in data if correct)
        accuracy = correct / total if total > 0 else 0.0

        # Group by confidence bins
        bins = {}
        bin_size = 0.1  # 10 bins (0.0-0.1, 0.1-0.2, etc.)

        for confidence, correct in data:
            bin_index = min(int(confidence / bin_size), 9)  # Ensure bin_index in [0, 9]
            if bin_index not in bins:
                bins[bin_index] = {"count": 0, "correct": 0}
            bins[bin_index]["count"] += 1
            if correct:
                bins[bin_index]["correct"] += 1

        # Calculate calibration error
        calibration_error = 0.0
        bin_stats = []

        for bin_index, bin_data in bins.items():
            if bin_data["count"] > 0:
                bin_confidence = (bin_index + 0.5) * bin_size  # Midpoint of bin
                bin_accuracy = bin_data["correct"] / bin_data["count"]
                bin_error = abs(bin_confidence - bin_accuracy)
                calibration_weight = bin_data["count"] / total
                calibration_error += bin_error * calibration_weight

                bin_stats.append({
                    "bin": bin_index,
                    "confidence_range": f"{bin_index * bin_size:.1f}-{(bin_index + 1) * bin_size:.1f}",
                    "count": bin_data["count"],
                    "accuracy": bin_accuracy,
                    "error": bin_error
                })

        # Calculate Brier score (mean squared error)
        brier_score = sum((confidence - (1.0 if correct else 0.0)) ** 2
                         for confidence, correct in data) / total

        return {
            "total_predictions": total,
            "accuracy": accuracy,
            "calibration_error": calibration_error,
            "brier_score": brier_score,
            "bins": bin_stats
        }

    def _get_source_reliability(self, source: ConfidenceSource, module_name: str) -> float:
        """
        Get the reliability factor for a confidence source.

        Args:
            source: The confidence source
            module_name: Name of the module

        Returns:
            Reliability factor (0.0 to 1.0)
        """
        # Check if we have calibration data for this source/module
        source_name = source.name

        if (module_name in self.aggregated_confidence and
            "sources" in self.aggregated_confidence[module_name] and
            source_name in self.aggregated_confidence[module_name]["sources"]):

            # If we have calibration data, use it to determine reliability
            if "calibration" in self.aggregated_confidence[module_name]:
                calibration = self.aggregated_confidence[module_name]["calibration"]
                if "calibration_error" in calibration:
                    # Convert calibration error to reliability
                    # Lower error = higher reliability
                    error = calibration["calibration_error"]
                    reliability = max(0.5, 1.0 - error)  # Minimum reliability of 0.5
                    return reliability

        # Default reliability factors based on source type
        default_reliability = {
            ConfidenceSource.PERCEPTION: 0.8,
            ConfidenceSource.REASONING: 0.85,
            ConfidenceSource.MEMORY: 0.9,
            ConfidenceSource.LEARNING: 0.75,
            ConfidenceSource.PRIOR: 0.7,
            ConfidenceSource.COMBINED: 0.85,
            ConfidenceSource.EXTERNAL: 0.95  # High trust in external sources
        }

        return default_reliability.get(source, 0.8)

    def _default_perception_confidence(self, perception_result: Any,
                                     context: Dict[str, Any]) -> float:
        """
        Default heuristic for estimating perception confidence.

        Args:
            perception_result: Result from a perception process
            context: Contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        # Default moderate confidence
        base_confidence = 0.7

        # Simple heuristics based on perception_result type
        if hasattr(perception_result, "score"):
            # If it has a score attribute, use that
            return min(1.0, float(perception_result.score))

        # Check for similarity score in context
        if "similarity_score" in context:
            sim_score = context["similarity_score"]
            if isinstance(sim_score, (int, float)):
                return min(1.0, float(sim_score))

        # Check for ambiguity in context
        if "ambiguity" in context:
            ambiguity = context["ambiguity"]
            if isinstance(ambiguity, (int, float)):
                return max(0.0, 1.0 - float(ambiguity))

        # Check for noise level in context
        if "noise_level" in context:
            noise = context["noise_level"]
            if isinstance(noise, (int, float)):
                return max(0.0, 1.0 - float(noise))

        # Return base confidence
        return base_confidence

    def _default_reasoning_confidence(self, reasoning_result: Any,
                                    context: Dict[str, Any]) -> float:
        """
        Default heuristic for estimating reasoning confidence.

        Args:
            reasoning_result: Result from a reasoning process
            context: Contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        # Default moderate confidence
        base_confidence = 0.75

        # Check for explicit confidence indicators
        if hasattr(reasoning_result, "certainty"):
            return min(1.0, float(reasoning_result.certainty))

        if hasattr(reasoning_result, "probability"):
            return min(1.0, float(reasoning_result.probability))

        # Check for inference steps in reasoning_result
        if hasattr(reasoning_result, "inference_steps"):
            steps = reasoning_result.inference_steps
            if steps:
                # More steps can mean more uncertainty
                step_count = len(steps)
                step_factor = max(0.6, 1.0 / (1.0 + 0.1 * step_count))
                base_confidence *= step_factor

        # Check for contradictions in context
        if "contradictions" in context:
            contradictions = context["contradictions"]
            if contradictions:
                contradiction_factor = max(0.3, 1.0 - (0.2 * len(contradictions)))
                base_confidence *= contradiction_factor

        # Check for assumptions in context
        if "assumptions" in context:
            assumptions = context["assumptions"]
            if assumptions:
                assumption_factor = max(0.5, 1.0 - (0.1 * len(assumptions)))
                base_confidence *= assumption_factor

        # Return adjusted confidence
        return base_confidence

    def _default_memory_confidence(self, memory_item: Any,
                                 context: Dict[str, Any]) -> float:
        """
        Default heuristic for estimating memory retrieval confidence.

        Args:
            memory_item: Retrieved memory item
            context: Contextual information

        Returns:
            Confidence value (0.0 to 1.0)
        """
        # Default moderate confidence
        base_confidence = 0.8

        # Check for age/recency of memory
        if hasattr(memory_item, "timestamp"):
            age = time.time() - memory_item.timestamp
            age_hours = age / 3600.0

            # Memories decay over time
            recency_factor = 1.0 / (1.0 + (age_hours / 24.0))  # Half confidence after 24 hours
            base_confidence *= recency_factor

        # Check for reinforcement count
        if hasattr(memory_item, "access_count"):
            count = memory_item.access_count

            # More access = stronger memory
            access_factor = min(1.5, 1.0 + (0.1 * count))  # Up to 50% boost
            base_confidence *= access_factor

            # But cap at 1.0
            base_confidence = min(1.0, base_confidence)

        # Check for retrieval conditions
        if "retrieval_score" in context:
            retrieval_score = context["retrieval_score"]
            if isinstance(retrieval_score, (int, float)):
                retrieval_factor = min(1.0, float(retrieval_score))
                base_confidence *= retrieval_factor

        # Return adjusted confidence
        return base_confidence

ModuleNotFoundError: No module named 'grandnexus'

# Adaptation

In [ ]:
"""
adaptation.py - Dynamic Parameter Adaptation Module for GrandNexus

This module implements meta-cognitive capabilities to dynamically adjust
system parameters based on performance monitoring, environmental changes,
and operational needs. It serves as a key component of the system's
self-optimization capability.

Key features:
- Parameter tracking and change history management
- Adaptive strategies based on performance feedback
- Multi-dimensional parameter space navigation
- Interference detection between parameter adjustments
- Reinforcement learning based parameter optimization
- Hierarchical parameter dependency modeling
"""

import logging
import time
import threading
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from enum import Enum, auto
import math
import json
import numpy as np
from collections import deque, defaultdict

# Assuming these modules exist in the GrandNexus framework
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.metacognition.monitor import MetaMonitor
from grandnexus.metacognition.confidence import ConfidenceEstimator

# Parameter adjustment strategies
class AdaptationStrategy(Enum):
    """Strategies for parameter adaptation"""
    GRADIENT_ASCENT = auto()     # Adjust parameters to maximize performance metric
    GRADIENT_DESCENT = auto()    # Adjust parameters to minimize some metric (like error)
    BINARY_SEARCH = auto()       # Try midpoint between known good/bad values
    RANDOM_SEARCH = auto()       # Try random values within bounds
    BAYESIAN_OPT = auto()        # Bayesian optimization approach
    DISCRETE_SWEEP = auto()      # Sweep across discrete values
    EVOLUTIONARY = auto()        # Genetic algorithm-like approach
    REINFORCEMENT = auto()       # RL-based parameter tuning
    HEURISTIC = auto()           # Manual rule-based adjustments
    MULTI_ARMED_BANDIT = auto()  # Exploration vs exploitation approach

# Parameter change types
class ChangeType(Enum):
    """Types of parameter changes"""
    AUTOMATIC = auto()     # System-initiated change
    FEEDBACK = auto()      # Change based on explicit feedback
    SCHEDULED = auto()     # Pre-scheduled change
    EMERGENCY = auto()     # Change due to system emergency
    EXPERIMENTAL = auto()  # Change to experiment with new settings
    RESET = auto()         # Reset to default/previous value
    CALIBRATION = auto()   # Calibration-related adjustment

@dataclass
class ParameterChange:
    """Record of a parameter change"""
    change_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    parameter_name: str = ""
    previous_value: Any = None
    new_value: Any = None
    timestamp: float = field(default_factory=time.time)
    strategy: AdaptationStrategy = AdaptationStrategy.HEURISTIC
    change_type: ChangeType = ChangeType.AUTOMATIC
    reason: str = ""
    performance_before: Optional[Dict[str, float]] = None
    performance_after: Optional[Dict[str, float]] = None
    confidence: float = 0.5  # Confidence in this change (0-1)
    status: str = "pending"  # pending, applied, reverted, superseded
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class AdaptableParameter:
    """
    A parameter that can be dynamically adjusted.
    Includes constraints, history tracking, and metadata.
    """
    name: str
    value: Any
    default_value: Any
    min_value: Optional[Any] = None
    max_value: Optional[Any] = None
    step_size: Optional[Any] = None
    param_type: str = "float"  # float, int, bool, str, enum, etc.
    description: str = ""
    tags: List[str] = field(default_factory=list)
    affects: List[str] = field(default_factory=list)  # Other parameters this influences
    depends_on: List[str] = field(default_factory=list)  # Parameters this depends on
    is_critical: bool = False
    acceptable_range: Optional[Tuple[Any, Any]] = None  # Narrower than min/max for safety
    change_history: List[str] = field(default_factory=list)  # List of change IDs
    performance_correlation: Dict[str, float] = field(default_factory=dict)  # Metric -> correlation
    change_sensitivity: float = 1.0  # How aggressively to change this parameter (0-1)
    exposure_level: str = "internal"  # internal, exposed, controlled, locked

    def validate_value(self, new_value: Any) -> bool:
        """Check if a proposed value is valid for this parameter"""
        # Check type compatibility
        if self.param_type == "float":
            if not isinstance(new_value, (int, float)):
                return False
        elif self.param_type == "int":
            if not isinstance(new_value, int):
                return False
        elif self.param_type == "bool":
            if not isinstance(new_value, bool):
                return False

        # Check bounds
        if self.min_value is not None and new_value < self.min_value:
            return False
        if self.max_value is not None and new_value > self.max_value:
            return False

        return True

    def update_value(self, new_value: Any, change_record: ParameterChange) -> bool:
        """Update the parameter value and record the change"""
        if not self.validate_value(new_value):
            return False

        self.value = new_value
        self.change_history.append(change_record.change_id)
        return True

    def reset_to_default(self) -> Any:
        """Reset the parameter to its default value"""
        old_value = self.value
        self.value = self.default_value
        return old_value

    def suggest_adjustment(self,
                          performance_metrics: Dict[str, float],
                          strategy: AdaptationStrategy,
                          step_multiplier: float = 1.0) -> Optional[Tuple[Any, float]]:
        """
        Suggest an adjustment to this parameter based on performance metrics.

        Args:
            performance_metrics: Current performance metrics
            strategy: Adaptation strategy to use
            step_multiplier: Multiply the step size by this factor

        Returns:
            Tuple of (suggested_value, confidence) or None if no adjustment recommended
        """
        # This is a simplified implementation - in a real system this would be more sophisticated

        if strategy == AdaptationStrategy.GRADIENT_ASCENT:
            # Try to increase parameter if it's positively correlated with performance
            target_metric = max(self.performance_correlation.items(), key=lambda x: x[1])[0]
            if target_metric in performance_metrics and self.performance_correlation[target_metric] > 0.2:
                if self.param_type == "float" or self.param_type == "int":
                    if self.max_value is not None and self.value >= self.max_value:
                        return None  # Already at max

                    # Calculate step size
                    if self.step_size is not None:
                        step = self.step_size * step_multiplier * self.change_sensitivity
                    else:
                        # Default step is 5% of range or 0.1 of current value
                        if self.min_value is not None and self.max_value is not None:
                            step = (self.max_value - self.min_value) * 0.05 * step_multiplier * self.change_sensitivity
                        else:
                            step = max(0.1 * abs(self.value), 0.01) * step_multiplier * self.change_sensitivity

                    if self.param_type == "int":
                        step = max(1, int(step))

                    new_value = self.value + step
                    if self.max_value is not None:
                        new_value = min(new_value, self.max_value)

                    # Confidence based on correlation strength
                    confidence = min(0.9, abs(self.performance_correlation[target_metric]))
                    return (new_value, confidence)

        elif strategy == AdaptationStrategy.GRADIENT_DESCENT:
            # Try to decrease parameter if it's negatively correlated with performance
            target_metric = min(self.performance_correlation.items(), key=lambda x: x[1])[0]
            if target_metric in performance_metrics and self.performance_correlation[target_metric] < -0.2:
                if self.param_type == "float" or self.param_type == "int":
                    if self.min_value is not None and self.value <= self.min_value:
                        return None  # Already at min

                    # Calculate step size
                    if self.step_size is not None:
                        step = self.step_size * step_multiplier * self.change_sensitivity
                    else:
                        # Default step is 5% of range or 0.1 of current value
                        if self.min_value is not None and self.max_value is not None:
                            step = (self.max_value - self.min_value) * 0.05 * step_multiplier * self.change_sensitivity
                        else:
                            step = max(0.1 * abs(self.value), 0.01) * step_multiplier * self.change_sensitivity

                    if self.param_type == "int":
                        step = max(1, int(step))

                    new_value = self.value - step
                    if self.min_value is not None:
                        new_value = max(new_value, self.min_value)

                    # Confidence based on correlation strength
                    confidence = min(0.9, abs(self.performance_correlation[target_metric]))
                    return (new_value, confidence)

        elif strategy == AdaptationStrategy.BINARY_SEARCH:
            # This would require knowledge of previous good/bad values
            # Simplified implementation:
            if hasattr(self, '_binary_search_upper') and hasattr(self, '_binary_search_lower'):
                midpoint = (self._binary_search_upper + self._binary_search_lower) / 2
                if self.param_type == "int":
                    midpoint = int(midpoint)
                return (midpoint, 0.7)
            else:
                # Initialize binary search bounds
                if self.min_value is not None and self.max_value is not None:
                    self._binary_search_lower = self.min_value
                    self._binary_search_upper = self.max_value
                    midpoint = (self._binary_search_upper + self._binary_search_lower) / 2
                    if self.param_type == "int":
                        midpoint = int(midpoint)
                    return (midpoint, 0.5)

        elif strategy == AdaptationStrategy.RANDOM_SEARCH:
            import random

            if self.param_type == "float":
                if self.min_value is not None and self.max_value is not None:
                    # Random value within bounds
                    new_value = self.min_value + random.random() * (self.max_value - self.min_value)
                    return (new_value, 0.3)  # Low confidence due to randomness

            elif self.param_type == "int":
                if self.min_value is not None and self.max_value is not None:
                    # Random integer within bounds
                    new_value = random.randint(self.min_value, self.max_value)
                    return (new_value, 0.3)

            elif self.param_type == "bool":
                # Toggle the value
                return (not self.value, 0.5)

        # Default: no adjustment
        return None

@dataclass
class ParameterGroup:
    """Group of related parameters that should be considered together"""
    group_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    name: str = ""
    description: str = ""
    parameters: List[str] = field(default_factory=list)  # List of parameter names
    joint_constraints: List[str] = field(default_factory=list)  # E.g., "param1 + param2 <= 1.0"
    tags: List[str] = field(default_factory=list)
    priority: int = 0  # Higher means more important to optimize

class AdaptationEngine:
    """
    Core engine for dynamically adapting system parameters based on
    performance metrics, environmental conditions, and operational feedback.

    This module lies at the heart of GrandNexus's self-optimization and
    metacognitive capabilities, allowing the system to intelligently
    adjust its own operations.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 meta_monitor: Optional[MetaMonitor] = None,
                 confidence_estimator: Optional[ConfidenceEstimator] = None,
                 working_memory: Optional[WorkingMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None):
        """
        Initialize the adaptation engine.

        Args:
            nexus_core: Reference to the NexusCore
            meta_monitor: Reference to the metacognitive monitoring module
            confidence_estimator: Reference to the confidence estimator
            working_memory: Reference to working memory
            episodic_memory: Reference to episodic memory
        """
        self.logger = logging.getLogger("GrandNexus.Metacognition.Adaptation")
        self.nexus_core = nexus_core
        self.meta_monitor = meta_monitor
        self.confidence_estimator = confidence_estimator
        self.working_memory = working_memory
        self.episodic_memory = episodic_memory

        # Core state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Parameters registry
        self.parameters: Dict[str, AdaptableParameter] = {}

        # Parameter groups
        self.parameter_groups: Dict[str, ParameterGroup] = {}

        # Parameter change history
        self.change_history: Dict[str, ParameterChange] = {}
        self.recent_changes = deque(maxlen=100)  # IDs of recent changes

        # Performance metrics history
        self.performance_history: Dict[str, deque] = defaultdict(lambda: deque(maxlen=100))

        # Adaptation strategies
        self.default_strategy = AdaptationStrategy.GRADIENT_ASCENT
        self.strategy_history: Dict[AdaptationStrategy, float] = {
            strategy: 0.0 for strategy in AdaptationStrategy
        }  # Strategy -> success rate

        # Adaptation schedule and state
        self.adaptation_interval = 60  # seconds
        self.last_adaptation_time = 0
        self.adaptation_enabled = True
        self.current_experiment = None

        self.logger.info(f"AdaptationEngine initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the adaptation engine"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Initialize parameter registry
                self._initialize_default_parameters()

                self.initialized = True
                self.logger.info("AdaptationEngine initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"AdaptationEngine initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the adaptation engine"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start the adaptation background thread
                self.running = True
                self.adaptation_thread = threading.Thread(
                    target=self._adaptation_loop,
                    daemon=True
                )
                self.adaptation_thread.start()

                self.logger.info("AdaptationEngine started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start AdaptationEngine: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the adaptation engine"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Signal the adaptation thread to stop
                self.running = False

                # Wait for the thread to terminate (with timeout)
                if hasattr(self, 'adaptation_thread'):
                    self.adaptation_thread.join(timeout=5.0)

                self.logger.info("AdaptationEngine stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping AdaptationEngine: {str(e)}")
                return False

    def register_parameter(self, parameter: AdaptableParameter) -> bool:
        """
        Register a parameter for adaptation.

        Args:
            parameter: The AdaptableParameter to register

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            if parameter.name in self.parameters:
                self.logger.warning(f"Parameter {parameter.name} already exists, will be overwritten")

            self.parameters[parameter.name] = parameter

            # If this parameter has tags that match any group, add it to those groups
            for group_id, group in self.parameter_groups.items():
                if any(tag in parameter.tags for tag in group.tags):
                    if parameter.name not in group.parameters:
                        group.parameters.append(parameter.name)
                        self.logger.info(f"Added parameter {parameter.name} to group {group.name} based on tags")

            self.logger.info(f"Registered parameter: {parameter.name}")
            return True

    def register_parameter_group(self, group: ParameterGroup) -> bool:
        """
        Register a group of related parameters.

        Args:
            group: The ParameterGroup to register

        Returns:
            True if registration succeeded, False otherwise
        """
        with self.lock:
            if group.group_id in self.parameter_groups:
                self.logger.warning(f"Parameter group {group.group_id} already exists, will be overwritten")

            self.parameter_groups[group.group_id] = group

            # Validate that all parameters exist
            missing_params = [p for p in group.parameters if p not in self.parameters]
            if missing_params:
                self.logger.warning(f"Group {group.name} references non-existent parameters: {missing_params}")

            self.logger.info(f"Registered parameter group: {group.name} with {len(group.parameters)} parameters")
            return True

    def set_parameter(self, name: str, value: Any,
                     reason: str = "",
                     change_type: ChangeType = ChangeType.AUTOMATIC) -> Optional[str]:
        """
        Set a parameter to a new value.

        Args:
            name: Parameter name
            value: New value
            reason: Reason for the change
            change_type: Type of change

        Returns:
            Change ID if successful, None otherwise
        """
        with self.lock:
            if name not in self.parameters:
                self.logger.error(f"Parameter {name} not found")
                return None

            parameter = self.parameters[name]

            # Validate the new value
            if not parameter.validate_value(value):
                self.logger.error(f"Invalid value {value} for parameter {name}")
                return None

            # Record performance before change
            performance_before = {}
            if self.meta_monitor:
                try:
                    # Get current performance metrics
                    metrics = self.meta_monitor.get_current_metrics()
                    if metrics:
                        performance_before = metrics
                except Exception as e:
                    self.logger.warning(f"Failed to get performance metrics: {str(e)}")

            # Create change record
            change = ParameterChange(
                parameter_name=name,
                previous_value=parameter.value,
                new_value=value,
                timestamp=time.time(),
                strategy=AdaptationStrategy.HEURISTIC,  # Manual/explicit change
                change_type=change_type,
                reason=reason,
                performance_before=performance_before,
                status="applied",
                confidence=1.0  # High confidence for explicit changes
            )

            # Store the change record
            self.change_history[change.change_id] = change
            self.recent_changes.append(change.change_id)

            # Update the parameter
            success = parameter.update_value(value, change)
            if not success:
                self.logger.error(f"Failed to update parameter {name}")
                change.status = "failed"
                return None

            self.logger.info(f"Parameter {name} changed from {change.previous_value} to {value}")

            # Schedule performance after check
            threading.Timer(5.0, self._check_performance_after, args=[change.change_id]).start()

            return change.change_id

    def get_parameter(self, name: str) -> Optional[Any]:
        """
        Get the current value of a parameter.

        Args:
            name: Parameter name

        Returns:
            Current parameter value or None if not found
        """
        with self.lock:
            if name not in self.parameters:
                self.logger.error(f"Parameter {name} not found")
                return None

            return self.parameters[name].value

    def get_parameter_info(self, name: str) -> Optional[Dict[str, Any]]:
        """
        Get detailed information about a parameter.

        Args:
            name: Parameter name

        Returns:
            Dictionary with parameter information or None if not found
        """
        with self.lock:
            if name not in self.parameters:
                self.logger.error(f"Parameter {name} not found")
                return None

            param = self.parameters[name]

            # Prepare change history information
            changes = []
            for change_id in param.change_history[-10:]:  # Get last 10 changes
                if change_id in self.change_history:
                    change = self.change_history[change_id]
                    changes.append({
                        "change_id": change_id,
                        "timestamp": change.timestamp,
                        "previous_value": change.previous_value,
                        "new_value": change.new_value,
                        "reason": change.reason,
                        "status": change.status
                    })

            # Prepare parameter information
            info = {
                "name": param.name,
                "value": param.value,
                "default_value": param.default_value,
                "type": param.param_type,
                "description": param.description,
                "is_critical": param.is_critical,
                "min_value": param.min_value,
                "max_value": param.max_value,
                "step_size": param.step_size,
                "tags": param.tags,
                "affects": param.affects,
                "depends_on": param.depends_on,
                "change_sensitivity": param.change_sensitivity,
                "exposure_level": param.exposure_level,
                "performance_correlation": param.performance_correlation,
                "recent_changes": changes
            }

            return info

    def list_parameters(self, tags: Optional[List[str]] = None) -> List[Dict[str, Any]]:
        """
        List all registered parameters, optionally filtered by tags.

        Args:
            tags: Optional list of tags to filter by

        Returns:
            List of parameter information dictionaries
        """
        with self.lock:
            result = []

            for name, param in self.parameters.items():
                # Skip if we're filtering by tags and this parameter doesn't match
                if tags and not any(tag in param.tags for tag in tags):
                    continue

                result.append({
                    "name": param.name,
                    "value": param.value,
                    "type": param.param_type,
                    "description": param.description,
                    "tags": param.tags
                })

            return result

    def list_parameter_groups(self) -> List[Dict[str, Any]]:
        """
        List all registered parameter groups.

        Returns:
            List of group information dictionaries
        """
        with self.lock:
            result = []

            for group_id, group in self.parameter_groups.items():
                result.append({
                    "group_id": group_id,
                    "name": group.name,
                    "description": group.description,
                    "parameters": group.parameters,
                    "tags": group.tags,
                    "priority": group.priority
                })

            return result

    def update_performance_metrics(self, metrics: Dict[str, float]) -> None:
        """
        Update the system with current performance metrics.

        Args:
            metrics: Dictionary of metric name -> value
        """
        with self.lock:
            # Store metrics in history
            for name, value in metrics.items():
                self.performance_history[name].append((time.time(), value))

            # Update parameter correlations
            self._update_parameter_correlations(metrics)

    def suggest_parameter_adjustments(self) -> List[Dict[str, Any]]:
        """
        Suggest parameter adjustments based on current performance.

        Returns:
            List of suggested adjustments with confidence levels
        """
        with self.lock:
            if not self.adaptation_enabled:
                return []

            # Get current performance metrics
            current_metrics = {}
            if self.meta_monitor:
                try:
                    metrics = self.meta_monitor.get_current_metrics()
                    if metrics:
                        current_metrics = metrics
                except Exception as e:
                    self.logger.warning(f"Failed to get performance metrics: {str(e)}")

            # If no metrics, use recent history
            if not current_metrics:
                for metric, history in self.performance_history.items():
                    if history:
                        current_metrics[metric] = history[-1][1]

            # If still no metrics, we can't make good suggestions
            if not current_metrics:
                self.logger.warning("No performance metrics available for parameter suggestions")
                return []

            suggestions = []

            # Select adaptation strategy - prefer strategies that worked well in the past
            # This is a simple weighted random selection
            strategy_weights = {}
            for strategy, success_rate in self.strategy_history.items():
                # Base weight on success rate, but ensure some chance for all strategies
                strategy_weights[strategy] = 0.1 + success_rate

            # Normalize weights
            total_weight = sum(strategy_weights.values())
            if total_weight > 0:
                strategy_weights = {s: w/total_weight for s, w in strategy_weights.items()}

            import random
            r = random.random()
            cum_weight = 0
            selected_strategy = self.default_strategy
            for strategy, weight in strategy_weights.items():
                cum_weight += weight
                if r <= cum_weight:
                    selected_strategy = strategy
                    break

            # Let each parameter suggest an adjustment
            for name, param in self.parameters.items():
                # Skip parameters with no correlation data
                if not param.performance_correlation:
                    continue

                suggestion = param.suggest_adjustment(
                    current_metrics,
                    selected_strategy
                )

                if suggestion:
                    new_value, confidence = suggestion
                    suggestions.append({
                        "parameter": name,
                        "current_value": param.value,
                        "suggested_value": new_value,
                        "confidence": confidence,
                        "strategy": selected_strategy.name
                    })

            # Sort by confidence (highest first)
            suggestions.sort(key=lambda x: x["confidence"], reverse=True)

            return suggestions

    def apply_suggestions(self, max_changes: int = 1,
                         min_confidence: float = 0.5) -> List[str]:
        """
        Automatically apply parameter adjustment suggestions.

        Args:
            max_changes: Maximum number of changes to apply
            min_confidence: Minimum confidence threshold for applying changes

        Returns:
            List of change IDs for applied changes
        """
        with self.lock:
            if not self.adaptation_enabled:
                return []

            suggestions = self.suggest_parameter_adjustments()

            # Filter by confidence
            suggestions = [s for s in suggestions if s["confidence"] >= min_confidence]

            applied_changes = []

            # Apply up to max_changes suggestions
            for suggestion in suggestions[:max_changes]:
                change_id = self.set_parameter(
                    suggestion["parameter"],
                    suggestion["suggested_value"],
                    reason=f"Automatic adjustment using {suggestion['strategy']} strategy",
                    change_type=ChangeType.AUTOMATIC
                )

                if change_id:
                    # Update the change record with strategy information
                    change = self.change_history[change_id]
                    change.strategy = getattr(AdaptationStrategy, suggestion["strategy"])
                    change.confidence = suggestion["confidence"]

                    applied_changes.append(change_id)

            return applied_changes

    def revert_change(self, change_id: str,
                     reason: str = "Manual revert") -> Optional[str]:
        """
        Revert a parameter change.

        Args:
            change_id: ID of the change to revert
            reason: Reason for the revert

        Returns:
            New change ID if successful, None otherwise
        """
        with self.lock:
            if change_id not in self.change_history:
                self.logger.error(f"Change {change_id} not found")
                return None

            change = self.change_history[change_id]

            # Skip if already reverted or failed
            if change.status in ["reverted", "failed"]:
                self.logger.warning(f"Change {change_id} is already {change.status}")
                return None

            # Get the affected parameter
            param_name = change.parameter_name
            if param_name not in self.parameters:
                self.logger.error(f"Parameter {param_name} no longer exists")
                return None

            # Create a new change that reverts to the previous value
            revert_id = self.set_parameter(
                param_name,
                change.previous_value,
                reason=reason,
                change_type=ChangeType.RESET
            )

            if revert_id:
                # Update status of the original change
                change.status = "reverted"

                # Link the revert change to the original
                revert_change = self.change_history[revert_id]
                revert_change.metadata["reverts"] = change_id

                self.logger.info(f"Change {change_id} reverted by change {revert_id}")

            return revert_id

    def reset_parameter(self, name: str) -> Optional[str]:
        """
        Reset a parameter to its default value.

        Args:
            name: Parameter name

        Returns:
            Change ID if successful, None otherwise
        """
        with self.lock:
            if name not in self.parameters:
                self.logger.error(f"Parameter {name} not found")
                return None

            parameter = self.parameters[name]

            # Create a change record for the reset
            change_id = self.set_parameter(
                name,
                parameter.default_value,
                reason="Reset to default value",
                change_type=ChangeType.RESET
            )

            return change_id

    def disable_adaptation(self) -> None:
        """Temporarily disable automatic parameter adaptation."""
        with self.lock:
            self.adaptation_enabled = False
            self.logger.info("Parameter adaptation disabled")

    def enable_adaptation(self) -> None:
        """Re-enable automatic parameter adaptation."""
        with self.lock:
            self.adaptation_enabled = True
            self.logger.info("Parameter adaptation enabled")

    def run_experiment(self, experiment_name: str,
                     parameters: Dict[str, Any],
                     duration: float = 300.0,
                     reason: str = "") -> str:
        """
        Run an experiment with temporary parameter values.

        Args:
            experiment_name: Name of the experiment
            parameters: Dictionary of parameter name -> value
            duration: How long to run the experiment in seconds
            reason: Reason for the experiment

        Returns:
            Experiment ID (actually a UUID)
        """
        with self.lock:
            # Create an experiment record
            experiment_id = str(uuid.uuid4())

            # Store original values
            original_values = {}
            change_ids = []

            # Set the experimental values
            for name, value in parameters.items():
                if name in self.parameters:
                    original_values[name] = self.parameters[name].value

                    change_id = self.set_parameter(
                        name,
                        value,
                        reason=f"Experiment: {experiment_name} - {reason}",
                        change_type=ChangeType.EXPERIMENTAL
                    )

                    if change_id:
                        change_ids.append(change_id)
                else:
                    self.logger.warning(f"Experiment parameter {name} not found")

            # Store experiment data
            self.current_experiment = {
                "id": experiment_id,
                "name": experiment_name,
                "start_time": time.time(),
                "end_time": time.time() + duration,
                "original_values": original_values,
                "change_ids": change_ids,
                "reason": reason
            }

            # Schedule the end of the experiment
            threading.Timer(duration, self._end_experiment, args=[experiment_id]).start()

            self.logger.info(f"Started experiment {experiment_name} with ID {experiment_id}")

            return experiment_id

    def export_parameter_config(self) -> Dict[str, Any]:
        """
        Export the current parameter configuration.

        Returns:
            Dictionary of parameter configurations
        """
        with self.lock:
            config = {}

            for name, param in self.parameters.items():
                config[name] = {
                    "value": param.value,
                    "type": param.param_type,
                    "description": param.description,
                    "is_critical": param.is_critical,
                    "min_value": param.min_value,
                    "max_value": param.max_value
                }

            return config

    def import_parameter_config(self, config: Dict[str, Dict[str, Any]],
                               reason: str = "Imported configuration") -> List[str]:
        """
        Import parameter configuration from a dictionary.

        Args:
            config: Dictionary of parameter name -> configuration
            reason: Reason for the import

        Returns:
            List of change IDs for applied changes
        """
        with self.lock:
            change_ids = []

            for name, param_config in config.items():
                if name in self.parameters and "value" in param_config:
                    change_id = self.set_parameter(
                        name,
                        param_config["value"],
                        reason=reason,
                        change_type=ChangeType.AUTOMATIC
                    )

                    if change_id:
                        change_ids.append(change_id)

            return change_ids

    def get_change_history(self, limit: int = 100) -> List[Dict[str, Any]]:
        """
        Get the history of parameter changes.

        Args:
            limit: Maximum number of changes to return

        Returns:
            List of change records (most recent first)
        """
        with self.lock:
            changes = []

            # Get the most recent change IDs
            recent_ids = list(self.recent_changes)[-limit:]

            for change_id in reversed(recent_ids):
                if change_id in self.change_history:
                    change = self.change_history[change_id]
                    changes.append({
                        "change_id": change_id,
                        "parameter_name": change.parameter_name,
                        "previous_value": change.previous_value,
                        "new_value": change.new_value,
                        "timestamp": change.timestamp,
                        "reason": change.reason,
                        "strategy": change.strategy.name,
                        "change_type": change.change_type.name,
                        "status": change.status,
                        "confidence": change.confidence
                    })

            return changes

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []

            if self.meta_monitor:
                dependencies.append("meta_monitor")
            if self.confidence_estimator:
                dependencies.append("confidence_estimator")
            if self.working_memory:
                dependencies.append("working_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")

            self.nexus_core.register_module(
                name="adaptation_engine",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _initialize_default_parameters(self) -> None:
        """Initialize default adaptable parameters"""
        # System-wide parameters
        self.register_parameter(AdaptableParameter(
            name="system.logging_level",
            value="INFO",
            default_value="INFO",
            param_type="enum",
            description="System-wide logging level",
            tags=["system", "logging"],
            acceptable_range=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"],
            is_critical=True,
            exposure_level="exposed"
        ))

        self.register_parameter(AdaptableParameter(
            name="system.thread_pool_size",
            value=8,
            default_value=8,
            param_type="int",
            min_value=1,
            max_value=32,
            step_size=1,
            description="Number of worker threads in the thread pool",
            tags=["system", "performance", "resources"],
            is_critical=True,
            exposure_level="internal"
        ))

        # Metacognition parameters
        self.register_parameter(AdaptableParameter(
            name="metacog.confidence_threshold",
            value=0.7,
            default_value=0.7,
            param_type="float",
            min_value=0.0,
            max_value=1.0,
            step_size=0.05,
            description="Threshold for high-confidence decisions",
            tags=["metacognition", "confidence"],
            exposure_level="exposed"
        ))

        self.register_parameter(AdaptableParameter(
            name="metacog.monitoring_frequency",
            value=1.0,
            default_value=1.0,
            param_type="float",
            min_value=0.1,
            max_value=10.0,
            step_size=0.1,
            description="Frequency of metacognitive monitoring in Hz",
            tags=["metacognition", "monitoring"],
            exposure_level="internal"
        ))

        # Adaptation engine's own parameters
        self.register_parameter(AdaptableParameter(
            name="adaptation.interval",
            value=self.adaptation_interval,
            default_value=60.0,
            param_type="float",
            min_value=5.0,
            max_value=3600.0,
            step_size=5.0,
            description="Interval between adaptation cycles in seconds",
            tags=["adaptation", "timing"],
            is_critical=False,
            exposure_level="internal"
        ))

        self.register_parameter(AdaptableParameter(
            name="adaptation.max_changes_per_cycle",
            value=1,
            default_value=1,
            param_type="int",
            min_value=0,
            max_value=10,
            step_size=1,
            description="Maximum parameter changes per adaptation cycle",
            tags=["adaptation", "limits"],
            is_critical=True,
            exposure_level="exposed"
        ))

        # Create a parameter group
        self.register_parameter_group(ParameterGroup(
            name="System Resources",
            description="Parameters affecting system resource utilization",
            parameters=["system.thread_pool_size"],
            tags=["system", "resources", "performance"],
            priority=5
        ))

        self.register_parameter_group(ParameterGroup(
            name="Metacognition",
            description="Parameters for metacognitive processes",
            parameters=["metacog.confidence_threshold", "metacog.monitoring_frequency"],
            tags=["metacognition"],
            priority=3
        ))

        # Add more default parameters and groups as needed

    def _adaptation_loop(self) -> None:
        """Background thread for periodic parameter adaptation"""
        self.logger.info("Adaptation loop started")

        while self.running:
            try:
                # Check if adaptation is enabled
                if self.adaptation_enabled:
                    # Check if it's time for adaptation
                    current_time = time.time()

                    # Get the current interval setting (may have been changed)
                    interval = self.get_parameter("adaptation.interval")
                    if interval is None:
                        interval = self.adaptation_interval

                    if current_time - self.last_adaptation_time >= interval:
                        # Get max changes setting
                        max_changes = self.get_parameter("adaptation.max_changes_per_cycle")
                        if max_changes is None:
                            max_changes = 1

                        # Apply parameter suggestions
                        changes = self.apply_suggestions(max_changes=max_changes)

                        if changes:
                            self.logger.info(f"Applied {len(changes)} parameter changes")

                        self.last_adaptation_time = current_time

                # Check for expired experiment
                if self.current_experiment and current_time >= self.current_experiment["end_time"]:
                    self._end_experiment(self.current_experiment["id"])

                # Sleep a bit to avoid busy-waiting
                # We use small sleeps and check self.running frequently to ensure quick shutdown
                time.sleep(1.0)

            except Exception as e:
                self.logger.error(f"Error in adaptation loop: {str(e)}")
                time.sleep(5.0)  # Sleep a bit longer on error

    def _update_parameter_correlations(self, metrics: Dict[str, float]) -> None:
        """
        Update the correlation between parameters and performance metrics.

        Args:
            metrics: Current performance metrics
        """
        # This is a simplistic implementation
        # In a real system, this would use more sophisticated statistical methods

        # For each parameter, calculate correlation with each metric
        for param_name, param in self.parameters.items():
            # Skip if no change history
            if not param.change_history:
                continue

            # Get the most recent changes
            recent_changes = []
            for change_id in param.change_history[-10:]:  # Last 10 changes
                if change_id in self.change_history:
                    change = self.change_history[change_id]
                    if change.performance_before and change.performance_after:
                        recent_changes.append(change)

            # Skip if not enough data
            if len(recent_changes) < 3:
                continue

            # Calculate direction of correlation
            for metric_name, current_value in metrics.items():
                correlations = []

                for change in recent_changes:
                    if metric_name in change.performance_before and metric_name in change.performance_after:
                        before = change.performance_before[metric_name]
                        after = change.performance_after[metric_name]

                        # Calculate the change in parameter and metric
                        param_change = change.new_value - change.previous_value
                        metric_change = after - before

                        # Skip if no change in parameter
                        if param_change == 0:
                            continue

                        # Calculate the sign of correlation: +1 if same direction, -1 if opposite
                        correlation = 1 if (param_change > 0) == (metric_change > 0) else -1

                        correlations.append(correlation)

                # Calculate average correlation
                if correlations:
                    avg_correlation = sum(correlations) / len(correlations)
                    param.performance_correlation[metric_name] = avg_correlation

    def _check_performance_after(self, change_id: str) -> None:
        """
        Check performance metrics after a parameter change.

        Args:
            change_id: ID of the change to check
        """
        with self.lock:
            if change_id not in self.change_history:
                return

            change = self.change_history[change_id]

            # Skip if already has performance_after
            if change.performance_after:
                return

            # Get current performance metrics
            performance_after = {}
            if self.meta_monitor:
                try:
                    metrics = self.meta_monitor.get_current_metrics()
                    if metrics:
                        performance_after = metrics
                        change.performance_after = performance_after

                        # Calculate impact on performance
                        impact = {}
                        if change.performance_before:
                            for metric, after_value in performance_after.items():
                                if metric in change.performance_before:
                                    before_value = change.performance_before[metric]
                                    impact[metric] = after_value - before_value

                            # Update strategy success history
                            if impact:
                                # Count this as a success if any important metric improved
                                # This is a simplistic approach
                                success = any(v > 0 for k, v in impact.items()
                                             if k in ["throughput", "accuracy", "reward"])

                                # Update strategy success rate
                                strategy = change.strategy
                                current_rate = self.strategy_history[strategy]
                                # Weighted update of success rate (30% new, 70% history)
                                self.strategy_history[strategy] = 0.7 * current_rate + 0.3 * (1.0 if success else 0.0)
                except Exception as e:
                    self.logger.warning(f"Failed to get performance metrics after change: {str(e)}")

    def _end_experiment(self, experiment_id: str) -> None:
        """
        End an experiment by restoring original parameter values.

        Args:
            experiment_id: ID of the experiment to end
        """
        with self.lock:
            if not self.current_experiment or self.current_experiment["id"] != experiment_id:
                return

            experiment = self.current_experiment

            # Restore original values
            for name, value in experiment["original_values"].items():
                self.set_parameter(
                    name,
                    value,
                    reason=f"End of experiment: {experiment['name']}",
                    change_type=ChangeType.RESET
                )

            self.logger.info(f"Ended experiment {experiment['name']} with ID {experiment_id}")

            # Clear current experiment
            self.current_experiment = None

# Reflection

In [ ]:
"""
reflection.py - Self-Analysis and Cognitive Process Reflection Module for GrandNexus

This module implements meta-cognitive self-reflection capabilities, allowing
the system to analyze its own reasoning processes, decisions, and cognitive
operations. It supports both automatic periodic reflection and on-demand
reflection triggered by specific events or confidence thresholds.

Key capabilities:
- Process trace analysis of reasoning and decision chains
- Detection of cognitive biases and reasoning errors
- Explanatory generation for system decisions and actions
- Identification of knowledge gaps and uncertainty sources
- Abstraction of lessons learned from episodic experiences
- Strategic adjustment of cognitive processes based on reflection outcomes

The module works closely with monitor.py (for process monitoring),
confidence.py (for uncertainty assessment), and adaption.py (for parameter
adjustment) to form the complete metacognitive subsystem of GrandNexus.
"""

import logging
import time
import threading
import uuid
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from collections import deque, defaultdict
import re
import json

# Assuming these modules exist in the GrandNexus framework
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.working import WorkingMemory
from grandnexus.metacognition.monitor import MetaMonitor
from grandnexus.metacognition.confidence import ConfidenceEstimator
from grandnexus.metacognition.adaptation import AdaptationEngine

# Reflection types
class ReflectionType(Enum):
    """Types of reflective analysis"""
    PROCESS_TRACE = auto()      # Analysis of cognitive process traces
    DECISION_ANALYSIS = auto()   # Analysis of decision-making
    ERROR_ANALYSIS = auto()      # Analysis of errors or failures
    KNOWLEDGE_GAP = auto()       # Identification of knowledge gaps
    BIAS_DETECTION = auto()      # Detection of cognitive biases
    STRATEGY_EVALUATION = auto() # Evaluation of cognitive strategies
    PERFORMANCE_ANALYSIS = auto() # Analysis of system performance
    MODEL_EVALUATION = auto()    # Evaluation of internal models
    EXPLANATION_GENERATION = auto() # Generation of explanations
    LESSON_EXTRACTION = auto()   # Extraction of lessons learned

# Targets of reflection (what is being reflected upon)
class ReflectionTarget(Enum):
    """Targets of reflection"""
    REASONING_TRACE = auto()     # A specific reasoning process
    DECISION = auto()            # A specific decision
    ERROR = auto()               # A specific error
    KNOWLEDGE = auto()           # Knowledge state
    BIAS = auto()                # Potential cognitive bias
    STRATEGY = auto()            # Cognitive strategy
    PERFORMANCE = auto()         # System performance
    MODEL = auto()               # Internal model
    EXPLANATION = auto()         # Explanation generation
    LESSON = auto()              # Lesson learned

@dataclass
class ReflectionTrigger:
    """Represents an event or condition that triggers reflection"""
    trigger_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    name: str = ""
    description: str = ""
    target_type: ReflectionTarget = ReflectionTarget.REASONING_TRACE
    reflection_type: ReflectionType = ReflectionType.PROCESS_TRACE
    condition: Optional[Callable] = None  # Function that evaluates if reflection is needed
    priority: float = 0.5  # 0.0 to 1.0
    enabled: bool = True
    cooldown: float = 60.0  # Minimum seconds between triggers of this type
    last_triggered: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ReflectionTask:
    """A reflection task to be processed"""
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    target_type: ReflectionTarget = ReflectionTarget.REASONING_TRACE
    reflection_type: ReflectionType = ReflectionType.PROCESS_TRACE
    target_data: Any = None  # The data being reflected upon
    context: Dict[str, Any] = field(default_factory=dict)
    priority: float = 0.5
    created_at: float = field(default_factory=time.time)
    deadline: Optional[float] = None
    trigger_id: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ReflectionResult:
    """The result of a reflection task"""
    task_id: str
    success: bool = False
    insights: List[Dict[str, Any]] = field(default_factory=list)
    summary: str = ""
    recommendations: List[Dict[str, Any]] = field(default_factory=list)
    confidence: float = 0.0
    completion_time: float = field(default_factory=time.time)
    processing_time: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

class ReflectionStrategy:
    """Abstract base class for reflection strategies"""
    def __init__(self, name: str, reflection_type: ReflectionType):
        self.name = name
        self.reflection_type = reflection_type
        self.logger = logging.getLogger(f"GrandNexus.Metacognition.Reflection.Strategy.{name}")

    def can_handle(self, task: ReflectionTask) -> bool:
        """Check if this strategy can handle the given task"""
        return task.reflection_type == self.reflection_type

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Perform reflection on the given task.
        Must be implemented by subclasses.
        """
        raise NotImplementedError("Subclasses must implement this method")

class ReflectionModule:
    """
    Core module for system self-reflection and meta-cognitive analysis.

    This module enables GrandNexus to analyze its own cognitive processes,
    detect errors or inefficiencies, generate explanations, and learn from
    its experiences to improve its operations over time.
    """

    def __init__(self, nexus_core: Optional[NexusCore] = None,
                 meta_monitor: Optional[MetaMonitor] = None,
                 confidence_estimator: Optional[ConfidenceEstimator] = None,
                 adaptation_engine: Optional[AdaptationEngine] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 working_memory: Optional[WorkingMemory] = None):
        """
        Initialize the reflection module.

        Args:
            nexus_core: Reference to the NexusCore for system-wide coordination
            meta_monitor: Reference to the metacognitive monitoring module
            confidence_estimator: Reference to the confidence estimation module
            adaptation_engine: Reference to the parameter adaptation engine
            episodic_memory: Reference to episodic memory for reflection
            semantic_memory: Reference to semantic memory for reflection
            working_memory: Reference to working memory for reflection context
        """
        self.logger = logging.getLogger("GrandNexus.Metacognition.Reflection")
        self.nexus_core = nexus_core
        self.meta_monitor = meta_monitor
        self.confidence_estimator = confidence_estimator
        self.adaptation_engine = adaptation_engine
        self.episodic_memory = episodic_memory
        self.semantic_memory = semantic_memory
        self.working_memory = working_memory

        # Core state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Task management
        self.current_tasks = set()
        self.pending_tasks = []  # Priority queue (heapq)
        self.completed_tasks = deque(maxlen=100)

        # Reflection strategies
        self.strategies: Dict[ReflectionType, List[ReflectionStrategy]] = {
            rtype: [] for rtype in ReflectionType
        }

        # Reflection triggers
        self.triggers: Dict[str, ReflectionTrigger] = {}

        # Reflection history
        self.reflection_history = deque(maxlen=1000)

        # Thread pool for parallel reflection
        self.thread_pool = []
        self.max_threads = 4

        # LLM integration - if the system has LLM capabilities
        self.llm_available = False
        self.llm_module = None

        # Learning from reflection
        self.lessons_learned = []

        # Performance metrics
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        self.logger.info(f"ReflectionModule initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """Initialize the reflection module"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Register with NexusCore if available
                if self.nexus_core:
                    self._register_with_nexus_core()

                # Register default reflection strategies
                self._register_default_strategies()

                # Register default reflection triggers
                self._register_default_triggers()

                # Check for LLM availability
                self._check_llm_availability()

                self.initialized = True
                self.logger.info("ReflectionModule initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"ReflectionModule initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the reflection module"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start the reflection background thread
                self.running = True
                self.reflection_thread = threading.Thread(
                    target=self._reflection_loop,
                    daemon=True
                )
                self.reflection_thread.start()

                self.logger.info("ReflectionModule started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start ReflectionModule: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the reflection module"""
        with self.lock:
            if not self.running:
                return True

            try:
                # Signal the reflection thread to stop
                self.running = False

                # Wait for the thread to terminate (with timeout)
                if hasattr(self, 'reflection_thread'):
                    self.reflection_thread.join(timeout=5.0)

                self.logger.info("ReflectionModule stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping ReflectionModule: {str(e)}")
                return False

    def register_strategy(self, strategy: ReflectionStrategy) -> bool:
        """
        Register a reflection strategy.

        Args:
            strategy: The strategy to register

        Returns:
            bool: True if registration succeeded, False otherwise
        """
        with self.lock:
            rtype = strategy.reflection_type
            self.strategies[rtype].append(strategy)
            self.logger.info(f"Registered reflection strategy: {strategy.name} for {rtype.name}")
            return True

    def register_trigger(self, trigger: ReflectionTrigger) -> bool:
        """
        Register a reflection trigger.

        Args:
            trigger: The trigger to register

        Returns:
            bool: True if registration succeeded, False otherwise
        """
        with self.lock:
            if trigger.trigger_id in self.triggers:
                self.logger.warning(f"Trigger {trigger.trigger_id} already exists, updating")

            self.triggers[trigger.trigger_id] = trigger
            self.logger.info(f"Registered reflection trigger: {trigger.name}")
            return True

    def submit_reflection_task(self, task: ReflectionTask) -> str:
        """
        Submit a reflection task for processing.

        Args:
            task: The ReflectionTask to process

        Returns:
            str: The task ID for tracking
        """
        with self.lock:
            # Use heapq to maintain a priority queue
            import heapq

            # Create a tuple of (priority, creation time, task) for sorting
            # We negate priority because heapq is a min-heap (lower values come first)
            # but we want higher priority tasks to come first
            entry = (-task.priority, task.created_at, task)
            heapq.heappush(self.pending_tasks, entry)

            self.logger.info(f"Reflection task submitted: {task.task_id} - {task.reflection_type.name}")
            return task.task_id

    def get_task_status(self, task_id: str) -> Optional[Dict[str, Any]]:
        """Get the current status of a reflection task"""
        with self.lock:
            # Check current tasks
            for task in self.current_tasks:
                if task.task_id == task_id:
                    return {
                        "task_id": task_id,
                        "status": "in_progress",
                        "type": task.reflection_type.name,
                        "created_at": task.created_at
                    }

            # Check pending tasks
            for _, _, task in self.pending_tasks:
                if task.task_id == task_id:
                    return {
                        "task_id": task_id,
                        "status": "pending",
                        "type": task.reflection_type.name,
                        "created_at": task.created_at
                    }

            # Check completed tasks
            for result in self.completed_tasks:
                if result.task_id == task_id:
                    return {
                        "task_id": task_id,
                        "status": "completed",
                        "success": result.success,
                        "completion_time": result.completion_time,
                        "processing_time": result.processing_time
                    }

            return None

    def get_reflection_result(self, task_id: str) -> Optional[ReflectionResult]:
        """Get the result of a completed reflection task"""
        with self.lock:
            for result in self.completed_tasks:
                if result.task_id == task_id:
                    return result

            return None

    def get_reflection_insights(self, limit: int = 10,
                               reflection_type: Optional[ReflectionType] = None) -> List[Dict[str, Any]]:
        """
        Get recent insights from reflection activities.

        Args:
            limit: Maximum number of insights to return
            reflection_type: Optional filter by reflection type

        Returns:
            List of insight dictionaries
        """
        with self.lock:
            # Collect all insights from recent reflection results
            all_insights = []

            for result in self.completed_tasks:
                if not result.success or not result.insights:
                    continue

                # Skip if filtering by type and doesn't match
                if reflection_type is not None:
                    task_found = False
                    for task in self.reflection_history:
                        if task.task_id == result.task_id and task.reflection_type == reflection_type:
                            task_found = True
                            break

                    if not task_found:
                        continue

                # Add insights with metadata
                for insight in result.insights:
                    all_insights.append({
                        "task_id": result.task_id,
                        "timestamp": result.completion_time,
                        "confidence": result.confidence,
                        **insight
                    })

            # Sort by timestamp (newest first) and limit
            all_insights.sort(key=lambda x: x["timestamp"], reverse=True)
            return all_insights[:limit]

    def process_event(self, event_type: str, event_data: Dict[str, Any]) -> None:
        """
        Process an event that might trigger reflection.

        Args:
            event_type: The type of event
            event_data: Event data dictionary
        """
        with self.lock:
            # Check all triggers to see if any apply
            current_time = time.time()

            for trigger_id, trigger in self.triggers.items():
                # Skip if trigger is disabled
                if not trigger.enabled:
                    continue

                # Skip if in cooldown
                if current_time - trigger.last_triggered < trigger.cooldown:
                    continue

                # Check if condition applies
                should_trigger = False

                if trigger.condition is not None:
                    try:
                        # Call the condition function with event data
                        should_trigger = trigger.condition(event_type, event_data)
                    except Exception as e:
                        self.logger.error(f"Error evaluating trigger condition: {str(e)}")
                        continue
                else:
                    # Simple matching based on event type
                    if event_type == trigger.name:
                        should_trigger = True

                if should_trigger:
                    # Create a reflection task from the trigger
                    task = ReflectionTask(
                        target_type=trigger.target_type,
                        reflection_type=trigger.reflection_type,
                        target_data=event_data,
                        context={"event_type": event_type},
                        priority=trigger.priority,
                        trigger_id=trigger_id,
                        metadata={"triggered_by_event": True}
                    )

                    # Submit the task
                    self.submit_reflection_task(task)

                    # Update trigger timestamp
                    trigger.last_triggered = current_time

                    self.logger.info(f"Event {event_type} triggered reflection: {trigger.name}")

    def reflect_on_process(self, process_data: Dict[str, Any],
                          priority: float = 0.5) -> str:
        """
        Initiate reflection on a cognitive process.

        Args:
            process_data: Data describing the cognitive process
            priority: Priority level for the reflection task

        Returns:
            Task ID for the new reflection task
        """
        task = ReflectionTask(
            target_type=ReflectionTarget.REASONING_TRACE,
            reflection_type=ReflectionType.PROCESS_TRACE,
            target_data=process_data,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    def reflect_on_decision(self, decision_data: Dict[str, Any],
                           priority: float = 0.5) -> str:
        """
        Initiate reflection on a decision.

        Args:
            decision_data: Data describing the decision
            priority: Priority level for the reflection task

        Returns:
            Task ID for the new reflection task
        """
        task = ReflectionTask(
            target_type=ReflectionTarget.DECISION,
            reflection_type=ReflectionType.DECISION_ANALYSIS,
            target_data=decision_data,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    def reflect_on_error(self, error_data: Dict[str, Any],
                        priority: float = 0.7) -> str:
        """
        Initiate reflection on an error.

        Args:
            error_data: Data describing the error
            priority: Priority level for the reflection task

        Returns:
            Task ID for the new reflection task
        """
        task = ReflectionTask(
            target_type=ReflectionTarget.ERROR,
            reflection_type=ReflectionType.ERROR_ANALYSIS,
            target_data=error_data,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    def generate_explanation(self, explanandum: Any,
                            context: Optional[Dict[str, Any]] = None,
                            priority: float = 0.6) -> str:
        """
        Generate an explanation for a system output or process.

        Args:
            explanandum: The thing to be explained
            context: Additional context for explanation
            priority: Priority level for the explanation task

        Returns:
            Task ID for the new reflection task
        """
        if context is None:
            context = {}

        task = ReflectionTask(
            target_type=ReflectionTarget.EXPLANATION,
            reflection_type=ReflectionType.EXPLANATION_GENERATION,
            target_data=explanandum,
            context=context,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    def extract_lessons(self, experience_data: Dict[str, Any],
                       priority: float = 0.4) -> str:
        """
        Extract lessons learned from an experience.

        Args:
            experience_data: Data describing the experience
            priority: Priority level for the extraction task

        Returns:
            Task ID for the new reflection task
        """
        task = ReflectionTask(
            target_type=ReflectionTarget.LESSON,
            reflection_type=ReflectionType.LESSON_EXTRACTION,
            target_data=experience_data,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    def get_lessons_learned(self, category: Optional[str] = None,
                           limit: int = 10) -> List[Dict[str, Any]]:
        """
        Get lessons learned from past reflections.

        Args:
            category: Optional category filter
            limit: Maximum number of lessons to return

        Returns:
            List of lesson dictionaries
        """
        with self.lock:
            if category:
                filtered_lessons = [l for l in self.lessons_learned if l.get("category") == category]
            else:
                filtered_lessons = self.lessons_learned

            # Sort by importance or recency
            sorted_lessons = sorted(filtered_lessons,
                                    key=lambda x: x.get("importance", 0.5),
                                    reverse=True)

            return sorted_lessons[:limit]

    def evaluate_strategy(self, strategy_data: Dict[str, Any],
                         priority: float = 0.5) -> str:
        """
        Evaluate the effectiveness of a cognitive strategy.

        Args:
            strategy_data: Data describing the strategy
            priority: Priority level for the evaluation task

        Returns:
            Task ID for the new reflection task
        """
        task = ReflectionTask(
            target_type=ReflectionTarget.STRATEGY,
            reflection_type=ReflectionType.STRATEGY_EVALUATION,
            target_data=strategy_data,
            priority=priority,
            metadata={"source": "explicit_request"}
        )

        return self.submit_reflection_task(task)

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []

            if self.meta_monitor:
                dependencies.append("meta_monitor")
            if self.confidence_estimator:
                dependencies.append("confidence_estimator")
            if self.adaptation_engine:
                dependencies.append("adaptation_engine")
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.working_memory:
                dependencies.append("working_memory")

            self.nexus_core.register_module(
                name="reflection_module",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _register_default_strategies(self) -> None:
        """Register default reflection strategies"""
        # Process trace analysis
        self.register_strategy(ProcessTraceReflectionStrategy())

        # Decision analysis
        self.register_strategy(DecisionAnalysisStrategy())

        # Error analysis
        self.register_strategy(ErrorAnalysisStrategy())

        # Knowledge gap detection
        self.register_strategy(KnowledgeGapStrategy())

        # Explanation generation
        self.register_strategy(ExplanationGenerationStrategy())

        # Lesson extraction
        self.register_strategy(LessonExtractionStrategy())

        # Add more default strategies as needed

    def _register_default_triggers(self) -> None:
        """Register default reflection triggers"""
        # Low confidence decision trigger
        self.register_trigger(ReflectionTrigger(
            name="low_confidence_decision",
            description="Trigger reflection when a decision has low confidence",
            target_type=ReflectionTarget.DECISION,
            reflection_type=ReflectionType.DECISION_ANALYSIS,
            condition=lambda event_type, data: (
                event_type == "decision_made" and
                "confidence" in data and
                data["confidence"] < 0.5
            ),
            priority=0.7,
            cooldown=30.0
        ))

        # Error trigger
        self.register_trigger(ReflectionTrigger(
            name="error_occurred",
            description="Trigger reflection when an error occurs",
            target_type=ReflectionTarget.ERROR,
            reflection_type=ReflectionType.ERROR_ANALYSIS,
            condition=lambda event_type, data: event_type == "error_occurred",
            priority=0.8,
            cooldown=60.0
        ))

        # Performance anomaly trigger
        self.register_trigger(ReflectionTrigger(
            name="performance_anomaly",
            description="Trigger reflection when performance deviates significantly",
            target_type=ReflectionTarget.PERFORMANCE,
            reflection_type=ReflectionType.PERFORMANCE_ANALYSIS,
            condition=lambda event_type, data: (
                event_type == "performance_metrics" and
                "anomaly_detected" in data and
                data["anomaly_detected"]
            ),
            priority=0.6,
            cooldown=300.0
        ))

        # Add more default triggers as needed

    def _check_llm_availability(self) -> None:
        """Check for the availability of LLM module for reflection"""
        try:
            # Try to get the LLM module from NexusCore
            if self.nexus_core:
                llm_module = self.nexus_core.get_module("llm_interface")
                if llm_module:
                    self.llm_module = llm_module
                    self.llm_available = True
                    self.logger.info("LLM interface available for reflection")
        except Exception as e:
            self.logger.warning(f"Failed to check LLM availability: {str(e)}")

    def _reflection_loop(self) -> None:
        """Background thread for processing reflection tasks"""
        self.logger.info("Reflection loop started")

        while self.running:
            try:
                # Process pending tasks if any
                self._process_next_task()

                # Sleep a bit to avoid busy-waiting
                # We use small sleeps and check self.running frequently to ensure quick shutdown
                time.sleep(0.1)

            except Exception as e:
                self.logger.error(f"Error in reflection loop: {str(e)}")
                time.sleep(1.0)  # Sleep a bit longer on error

    def _process_next_task(self) -> None:
        """Process the next task from the queue"""
        with self.lock:
            # Check if there are any pending tasks
            if not self.pending_tasks:
                return

            # Check if we have capacity to process more tasks
            if len(self.current_tasks) >= self.max_threads:
                return

            import heapq

            # Get the highest priority task
            priority, created_at, task = heapq.heappop(self.pending_tasks)

            # Add to current tasks
            self.current_tasks.add(task)

            # Add to reflection history
            self.reflection_history.append(task)

        # Start a new thread to process the task
        thread = threading.Thread(
            target=self._execute_reflection_task,
            args=(task,),
            daemon=True
        )
        thread.start()

        # Add to thread pool
        with self.lock:
            self.thread_pool.append(thread)

    def _execute_reflection_task(self, task: ReflectionTask) -> None:
        """
        Execute a reflection task in a separate thread.

        Args:
            task: The task to execute
        """
        start_time = time.time()
        result = None

        try:
            # Find strategies that can handle this task
            strategies = [s for s in self.strategies[task.reflection_type] if s.can_handle(task)]

            if not strategies:
                self.logger.warning(f"No strategy available for task: {task.task_id}")
                result = ReflectionResult(
                    task_id=task.task_id,
                    success=False,
                    summary="No strategy available for this reflection type",
                    confidence=0.0,
                    processing_time=time.time() - start_time
                )
            else:
                # Use the first available strategy (in a more advanced implementation,
                # we might select the best strategy or combine multiple)
                strategy = strategies[0]

                # Execute the strategy
                result = strategy.reflect(task)

                # Ensure task_id is set correctly
                result.task_id = task.task_id

                # Set processing time
                result.processing_time = time.time() - start_time

                # Process recommendations if successful
                if result.success and result.recommendations:
                    self._process_recommendations(result.recommendations)

                # Extract lessons if successful
                if result.success and task.reflection_type == ReflectionType.LESSON_EXTRACTION:
                    self._store_lessons(result.insights)

        except Exception as e:
            self.logger.error(f"Error executing reflection task {task.task_id}: {str(e)}")
            result = ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary=f"Error during reflection: {str(e)}",
                confidence=0.0,
                processing_time=time.time() - start_time
            )

        finally:
            # Add to completed tasks and remove from current tasks
            with self.lock:
                if result:
                    self.completed_tasks.append(result)

                if task in self.current_tasks:
                    self.current_tasks.remove(task)

    def _process_recommendations(self, recommendations: List[Dict[str, Any]]) -> None:
        """
        Process recommendations from reflection results.

        Args:
            recommendations: List of recommendation dictionaries
        """
        for recommendation in recommendations:
            # Process different types of recommendations
            rec_type = recommendation.get("type", "")

            if rec_type == "parameter_adjustment":
                # Recommendation to adjust a parameter
                param_name = recommendation.get("parameter")
                new_value = recommendation.get("value")
                reason = recommendation.get("reason", "Based on reflection")
                confidence = recommendation.get("confidence", 0.5)

                if param_name and new_value is not None and self.adaptation_engine:
                    try:
                        # Only apply high-confidence recommendations automatically
                        if confidence >= 0.8:
                            self.adaptation_engine.set_parameter(
                                param_name,
                                new_value,
                                reason=reason
                            )
                            self.logger.info(f"Applied parameter adjustment: {param_name} = {new_value}")
                        else:
                            # Log lower confidence recommendations without applying
                            self.logger.info(f"Suggested parameter adjustment: {param_name} = {new_value} (confidence: {confidence})")
                    except Exception as e:
                        self.logger.error(f"Error applying parameter adjustment: {str(e)}")

            elif rec_type == "knowledge_update":
                # Recommendation to update semantic knowledge
                concept = recommendation.get("concept")
                update = recommendation.get("update")
                confidence = recommendation.get("confidence", 0.5)

                if concept and update and self.semantic_memory and confidence >= 0.7:
                    try:
                        # Apply knowledge update
                        # Implementation depends on semantic memory interface
                        self.logger.info(f"Applied knowledge update for concept: {concept}")
                    except Exception as e:
                        self.logger.error(f"Error applying knowledge update: {str(e)}")

            elif rec_type == "strategy_change":
                # Recommendation to change cognitive strategy
                module = recommendation.get("module")
                strategy = recommendation.get("strategy")
                reason = recommendation.get("reason", "Based on reflection")

                if module and strategy and self.nexus_core:
                    try:
                        # Notify the target module about the strategy change
                        target_module = self.nexus_core.get_module(module)
                        if target_module and hasattr(target_module, "set_strategy"):
                            target_module.set_strategy(strategy, reason=reason)
                            self.logger.info(f"Changed strategy for {module} to {strategy}")
                    except Exception as e:
                        self.logger.error(f"Error changing strategy: {str(e)}")

            elif rec_type == "resource_allocation":
                # Recommendation to adjust resource allocation
                resource = recommendation.get("resource")
                allocation = recommendation.get("allocation")
                reason = recommendation.get("reason", "Based on reflection")

                # Implementation would depend on resource management interface
                self.logger.info(f"Resource allocation recommendation: {resource} = {allocation}")

            elif rec_type == "further_reflection":
                # Recommendation for additional reflection
                reflection_type = recommendation.get("reflection_type")
                target_data = recommendation.get("target_data")
                priority = recommendation.get("priority", 0.5)

                if reflection_type and target_data:
                    try:
                        # Create a new reflection task
                        task = ReflectionTask(
                            target_type=ReflectionTarget[reflection_type.upper()],
                            reflection_type=ReflectionType[reflection_type.upper()],
                            target_data=target_data,
                            priority=priority,
                            metadata={"source": "recommendation"}
                        )

                        self.submit_reflection_task(task)
                        self.logger.info(f"Scheduled further reflection: {reflection_type}")
                    except Exception as e:
                        self.logger.error(f"Error scheduling further reflection: {str(e)}")

            else:
                # Unknown recommendation type
                self.logger.warning(f"Unknown recommendation type: {rec_type}")

    def _store_lessons(self, insights: List[Dict[str, Any]]) -> None:
        """
        Store lessons learned from reflection.

        Args:
            insights: List of insight dictionaries containing lessons
        """
        for insight in insights:
            if "lesson" in insight:
                lesson = {
                    "lesson": insight["lesson"],
                    "category": insight.get("category", "general"),
                    "context": insight.get("context", {}),
                    "importance": insight.get("importance", 0.5),
                    "confidence": insight.get("confidence", 0.5),
                    "timestamp": time.time(),
                    "source_task_id": insight.get("source_task_id")
                }

                # Add to lessons learned
                with self.lock:
                    self.lessons_learned.append(lesson)

                # Store in semantic memory if available
                if self.semantic_memory:
                    try:
                        # Implementation depends on semantic memory interface
                        self.logger.info(f"Stored lesson in semantic memory: {lesson['lesson']}")
                    except Exception as e:
                        self.logger.error(f"Error storing lesson in semantic memory: {str(e)}")

    def _generate_llm_reflection(self,
                                task: ReflectionTask,
                                prompt_template: str) -> Optional[Dict[str, Any]]:
        """
        Use LLM to generate reflection on a task.

        Args:
            task: The reflection task
            prompt_template: Template for LLM prompt

        Returns:
            Dictionary with LLM-generated reflection or None if failed
        """
        if not self.llm_available or not self.llm_module:
            return None

        try:
            # Format the prompt
            prompt = self._format_reflection_prompt(prompt_template, task)

            # Call LLM module
            response = self.llm_module.generate(
                prompt=prompt,
                max_tokens=1000,
                temperature=0.7
            )

            # Parse the response
            result = self._parse_llm_response(response)

            return result
        except Exception as e:
            self.logger.error(f"Error generating LLM reflection: {str(e)}")
            return None

    def _format_reflection_prompt(self, template: str, task: ReflectionTask) -> str:
        """
        Format a prompt template for LLM reflection.

        Args:
            template: The prompt template
            task: The reflection task

        Returns:
            Formatted prompt string
        """
        # Extract task data as string representation
        data_str = self._format_data_for_prompt(task.target_data)

        # Format the template with task data
        prompt = template.format(
            task_type=task.reflection_type.name,
            target_type=task.target_type.name,
            data=data_str,
            context=str(task.context)
        )

        return prompt

    def _format_data_for_prompt(self, data: Any) -> str:
        """
        Format task data for inclusion in a prompt.

        Args:
            data: The data to format

        Returns:
            Formatted string representation
        """
        if isinstance(data, dict):
            # Format dictionary as JSON string
            return json.dumps(data, indent=2)
        elif isinstance(data, list):
            # Format list items
            return "\n".join([f"- {self._format_data_for_prompt(item)}" for item in data])
        else:
            # Convert to string
            return str(data)

    def _parse_llm_response(self, response: str) -> Dict[str, Any]:
        """
        Parse LLM response into structured reflection result.

        Args:
            response: The raw LLM response

        Returns:
            Dictionary with parsed reflection
        """
        # This is a simple implementation that assumes a particular format
        # A more robust implementation would use a structured response format (e.g., JSON)

        result = {
            "summary": "",
            "insights": [],
            "recommendations": []
        }

        # Extract summary (everything up to first heading)
        match = re.search(r'^(.*?)(?:#|$)', response, re.DOTALL)
        if match:
            result["summary"] = match.group(1).strip()

        # Extract insights
        insights_match = re.search(r'(?:#+ *Insights? *\n)(.*?)(?:#|$)', response, re.DOTALL)
        if insights_match:
            insights_text = insights_match.group(1).strip()
            insight_items = re.findall(r'- (.*?)(?:\n-|\n#|$)', insights_text + "\n#", re.DOTALL)

            for item in insight_items:
                result["insights"].append({"description": item.strip()})

        # Extract recommendations
        recommendations_match = re.search(r'(?:#+ *Recommendations? *\n)(.*?)(?:#|$)', response, re.DOTALL)
        if recommendations_match:
            recommendations_text = recommendations_match.group(1).strip()
            recommendation_items = re.findall(r'- (.*?)(?:\n-|\n#|$)', recommendations_text + "\n#", re.DOTALL)

            for item in recommendation_items:
                result["recommendations"].append({"description": item.strip()})

        return result


# Concrete Reflection Strategy Implementations

class ProcessTraceReflectionStrategy(ReflectionStrategy):
    """Strategy for analyzing cognitive process traces"""

    def __init__(self):
        super().__init__("ProcessTraceAnalysis", ReflectionType.PROCESS_TRACE)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Analyze a reasoning process trace.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing insights and recommendations
        """
        process_data = task.target_data

        # Validate input
        if not isinstance(process_data, dict) or "steps" not in process_data:
            return ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary="Invalid process data format. Expected dictionary with 'steps' key.",
                confidence=0.0
            )

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.8
        )

        # Extract process steps
        steps = process_data.get("steps", [])

        # Analyze step coherence
        insights = []
        recommendations = []

        # Check for logical gaps between steps
        has_gaps = False
        gap_indices = []

        for i in range(1, len(steps)):
            prev_step = steps[i-1]
            curr_step = steps[i]

            # Simple heuristic: check if current step references previous step outputs
            if "outputs" in prev_step and "inputs" in curr_step:
                # Check for overlap between previous outputs and current inputs
                prev_outputs = set(str(o) for o in prev_step.get("outputs", []))
                curr_inputs = set(str(i) for i in curr_step.get("inputs", []))

                if not prev_outputs or not curr_inputs or not any(o in s for o in prev_outputs for s in curr_inputs):
                    has_gaps = True
                    gap_indices.append(i)

        if has_gaps:
            insights.append({
                "type": "coherence_issue",
                "description": f"Logical gaps detected between steps: {gap_indices}",
                "confidence": 0.7
            })

            recommendations.append({
                "type": "process_improvement",
                "description": "Add intermediate steps to bridge logical gaps",
                "steps": gap_indices
            })

        # Check for redundant steps
        redundant_indices = []
        for i in range(len(steps)):
            for j in range(i+1, len(steps)):
                # Simple heuristic: check for similar operation types and inputs
                if steps[i].get("operation") == steps[j].get("operation"):
                    if self._are_similar(steps[i].get("inputs", []), steps[j].get("inputs", [])):
                        redundant_indices.append((i, j))

        if redundant_indices:
            insights.append({
                "type": "efficiency_issue",
                "description": f"Potentially redundant steps: {redundant_indices}",
                "confidence": 0.6
            })

            recommendations.append({
                "type": "process_improvement",
                "description": "Consolidate redundant steps to improve efficiency",
                "steps": redundant_indices
            })

        # Check for error patterns
        error_steps = []
        for i, step in enumerate(steps):
            if step.get("error") or step.get("status") == "error":
                error_steps.append(i)

        if error_steps:
            insights.append({
                "type": "error_pattern",
                "description": f"Errors detected in steps: {error_steps}",
                "confidence": 0.9
            })

            recommendations.append({
                "type": "error_handling",
                "description": "Implement error recovery strategies for error-prone steps",
                "steps": error_steps
            })

        # Generate summary
        if insights:
            summary = f"Analysis of {len(steps)} reasoning steps revealed {len(insights)} insights."
            if has_gaps:
                summary += f" Logical gaps found between {len(gap_indices)} step pairs."
            if redundant_indices:
                summary += f" {len(redundant_indices)} potentially redundant steps identified."
            if error_steps:
                summary += f" Errors detected in {len(error_steps)} steps."
        else:
            summary = f"Analysis of {len(steps)} reasoning steps found no significant issues."

        # Update result
        result.summary = summary
        result.insights = insights
        result.recommendations = recommendations

        return result

    def _are_similar(self, list1, list2, threshold=0.7):
        """Check if two lists contain similar elements"""
        if not list1 or not list2:
            return False

        # Convert elements to strings for comparison
        set1 = set(str(x) for x in list1)
        set2 = set(str(x) for x in list2)

        # Calculate Jaccard similarity
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))

        if union == 0:
            return False

        similarity = intersection / union
        return similarity >= threshold


class DecisionAnalysisStrategy(ReflectionStrategy):
    """Strategy for analyzing decision-making processes"""

    def __init__(self):
        super().__init__("DecisionAnalysis", ReflectionType.DECISION_ANALYSIS)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Analyze a decision-making process.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing insights and recommendations
        """
        decision_data = task.target_data

        # Validate input
        if not isinstance(decision_data, dict):
            return ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary="Invalid decision data format. Expected dictionary.",
                confidence=0.0
            )

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.8
        )

        # Extract decision components
        options = decision_data.get("options", [])
        criteria = decision_data.get("criteria", [])
        weights = decision_data.get("weights", {})
        scores = decision_data.get("scores", {})
        chosen_option = decision_data.get("chosen", None)
        confidence_score = decision_data.get("confidence", 0.5)

        # Analyze decision components
        insights = []
        recommendations = []

        # Check if all criteria were used
        unused_criteria = []
        for criterion in criteria:
            if criterion not in weights:
                unused_criteria.append(criterion)

        if unused_criteria:
            insights.append({
                "type": "unused_criteria",
                "description": f"Decision criteria not used: {unused_criteria}",
                "confidence": 0.9
            })

            recommendations.append({
                "type": "decision_improvement",
                "description": "Include all defined criteria in the decision process",
                "criteria": unused_criteria
            })

        # Check for balanced criteria weighting
        if weights:
            max_weight = max(weights.values())
            min_weight = min(weights.values())
            weight_range = max_weight - min_weight

            if weight_range > 0.7:  # Arbitrary threshold
                insights.append({
                    "type": "imbalanced_weights",
                    "description": "Large disparity in criteria weights may oversimplify decision",
                    "confidence": 0.6,
                    "max_weight": max_weight,
                    "min_weight": min_weight
                })

        # Check if chosen option was actually highest scored
        if chosen_option and scores:
            option_scores = {}
            for option in options:
                if option in scores:
                    option_scores[option] = sum(scores[option].get(c, 0) * weights.get(c, 1) for c in criteria)

            if option_scores:
                highest_scored = max(option_scores.items(), key=lambda x: x[1])[0]

                if highest_scored != chosen_option:
                    insights.append({
                        "type": "inconsistent_choice",
                        "description": f"Chosen option ({chosen_option}) was not the highest scored ({highest_scored})",
                        "confidence": 0.9,
                        "scores": option_scores
                    })

                    recommendations.append({
                        "type": "decision_improvement",
                        "description": "Align chosen option with highest scored option or document override reasoning",
                        "highest_scored": highest_scored,
                        "chosen": chosen_option
                    })

        # Check for low confidence
        if confidence_score < 0.5:
            insights.append({
                "type": "low_confidence",
                "description": f"Low confidence in decision ({confidence_score})",
                "confidence": 0.9
            })

            recommendations.append({
                "type": "confidence_improvement",
                "description": "Gather more information before finalizing decision",
                "confidence_score": confidence_score
            })

        # Generate summary
        summary = f"Analysis of decision with {len(options)} options and {len(criteria)} criteria."
        if unused_criteria:
            summary += f" {len(unused_criteria)} criteria were unused."
        if chosen_option and scores and highest_scored != chosen_option:
            summary += f" Chosen option ({chosen_option}) differs from highest scored ({highest_scored})."
        if confidence_score < 0.5:
            summary += f" Decision confidence is low ({confidence_score})."

        # Update result
        result.summary = summary
        result.insights = insights
        result.recommendations = recommendations

        return result


class ErrorAnalysisStrategy(ReflectionStrategy):
    """Strategy for analyzing errors"""

    def __init__(self):
        super().__init__("ErrorAnalysis", ReflectionType.ERROR_ANALYSIS)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Analyze an error.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing insights and recommendations
        """
        error_data = task.target_data

        # Validate input
        if not isinstance(error_data, dict):
            return ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary="Invalid error data format. Expected dictionary.",
                confidence=0.0
            )

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.8
        )

        # Extract error components
        error_type = error_data.get("type", "unknown")
        error_message = error_data.get("message", "")
        module = error_data.get("module", "")
        context = error_data.get("context", {})
        stack_trace = error_data.get("stack_trace", "")

        # Analyze error
        insights = []
        recommendations = []

        # Check for known error patterns
        known_pattern = self._match_known_error_pattern(error_type, error_message)
        if known_pattern:
            insights.append({
                "type": "known_error_pattern",
                "description": f"Error matches known pattern: {known_pattern['pattern']}",
                "confidence": 0.9,
                "pattern": known_pattern
            })

            recommendations.append({
                "type": "error_handling",
                "description": known_pattern.get("recommendation", "Apply standard fix for this error"),
                "error_type": error_type
            })

        # Check for resource-related errors
        if any(term in error_message.lower() for term in ["memory", "timeout", "resource", "capacity"]):
            insights.append({
                "type": "resource_issue",
                "description": "Error may be related to resource constraints",
                "confidence": 0.7
            })

            recommendations.append({
                "type": "resource_allocation",
                "description": "Consider adjusting resource allocation",
                "resource": "memory" if "memory" in error_message.lower() else "processing",
                "allocation": 1.2  # Arbitrary increase factor
            })

        # Check for input validation errors
        if any(term in error_message.lower() for term in ["invalid", "expected", "type error", "value error"]):
            insights.append({
                "type": "input_validation",
                "description": "Error suggests input validation issues",
                "confidence": 0.8
            })

            recommendations.append({
                "type": "parameter_adjustment",
                "description": "Strengthen input validation",
                "parameter": "system.validation_strictness",
                "value": 0.8,
                "confidence": 0.7
            })

        # Generate summary
        summary = f"Analysis of {error_type} error in {module}: {error_message}"
        if known_pattern:
            summary += f" This is a known error pattern: {known_pattern['pattern']}."

        # Update result
        result.summary = summary
        result.insights = insights
        result.recommendations = recommendations

        return result

    def _match_known_error_pattern(self, error_type, error_message):
        """Match error against known patterns"""
        # This would be a database or dictionary of known error patterns
        # For demonstration, we'll use a simple list

        known_patterns = [
            {
                "pattern": "memory_allocation_failure",
                "type_match": ["MemoryError", "OutOfMemoryError"],
                "message_regex": r"(out of memory|allocation failed|memory exceeded)",
                "recommendation": "Increase memory allocation or implement memory optimization"
            },
            {
                "pattern": "timeout_error",
                "type_match": ["TimeoutError", "DeadlineExceededError"],
                "message_regex": r"(timeout|deadline exceeded|took too long)",
                "recommendation": "Increase timeout threshold or optimize processing speed"
            },
            {
                "pattern": "dependency_failure",
                "type_match": ["ImportError", "ModuleNotFoundError"],
                "message_regex": r"(not found|cannot import|no module named)",
                "recommendation": "Verify module dependencies and availability"
            }
        ]

        # Check each pattern for matches
        for pattern in known_patterns:
            # Check type match
            if error_type in pattern["type_match"]:
                # Check message regex
                if re.search(pattern["message_regex"], error_message.lower()):
                    return pattern

        return None


class KnowledgeGapStrategy(ReflectionStrategy):
    """Strategy for identifying knowledge gaps"""

    def __init__(self):
        super().__init__("KnowledgeGapDetection", ReflectionType.KNOWLEDGE_GAP)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Identify knowledge gaps.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing insights and recommendations
        """
        knowledge_data = task.target_data

        # Validate input
        if not isinstance(knowledge_data, dict):
            return ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary="Invalid knowledge data format. Expected dictionary.",
                confidence=0.0
            )

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.7
        )

        # Extract knowledge components
        query = knowledge_data.get("query", "")
        available_knowledge = knowledge_data.get("available_knowledge", [])
        missing_knowledge = knowledge_data.get("missing_knowledge", [])
        confidence_scores = knowledge_data.get("confidence_scores", {})

        # Analyze knowledge gaps
        insights = []
        recommendations = []

        # Check for explicit missing knowledge
        if missing_knowledge:
            insights.append({
                "type": "explicit_knowledge_gap",
                "description": f"Explicitly identified missing knowledge: {missing_knowledge}",
                "confidence": 0.9,
                "missing": missing_knowledge
            })

            recommendations.append({
                "type": "knowledge_acquisition",
                "description": "Prioritize acquisition of missing knowledge",
                "targets": missing_knowledge
            })

        # Check for low confidence areas
        low_confidence_areas = []
        for area, score in confidence_scores.items():
            if score < 0.4:  # Arbitrary threshold
                low_confidence_areas.append((area, score))

        if low_confidence_areas:
            insights.append({
                "type": "low_confidence_knowledge",
                "description": f"Knowledge areas with low confidence: {[a[0] for a in low_confidence_areas]}",
                "confidence": 0.8,
                "areas": low_confidence_areas
            })

            recommendations.append({
                "type": "knowledge_verification",
                "description": "Verify knowledge in low confidence areas",
                "areas": [a[0] for a in low_confidence_areas]
            })

        # Check for knowledge relevance
        if query and available_knowledge:
            # In a real implementation, this would use more sophisticated
            # relevance assessment, possibly using semantic similarity

            # For demonstration, we'll use a simple keyword matching
            query_keywords = set(query.lower().split())
            relevant_count = 0

            for knowledge_item in available_knowledge:
                item_text = str(knowledge_item).lower()
                if any(keyword in item_text for keyword in query_keywords):
                    relevant_count += 1

            relevance_ratio = relevant_count / len(available_knowledge) if available_knowledge else 0

            if relevance_ratio < 0.3:  # Arbitrary threshold
                insights.append({
                    "type": "low_knowledge_relevance",
                    "description": "Available knowledge has low relevance to the query",
                    "confidence": 0.6,
                    "relevance_ratio": relevance_ratio
                })

                recommendations.append({
                    "type": "knowledge_organization",
                    "description": "Improve knowledge retrieval relevance",
                    "query": query
                })

        # Generate summary
        summary = f"Knowledge gap analysis for query: '{query}'."
        if missing_knowledge:
            summary += f" {len(missing_knowledge)} missing knowledge items identified."
        if low_confidence_areas:
            summary += f" {len(low_confidence_areas)} knowledge areas with low confidence."

        # Update result
        result.summary = summary
        result.insights = insights
        result.recommendations = recommendations

        return result


class ExplanationGenerationStrategy(ReflectionStrategy):
    """Strategy for generating explanations"""

    def __init__(self):
        super().__init__("ExplanationGeneration", ReflectionType.EXPLANATION_GENERATION)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Generate an explanation.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing the explanation
        """
        explanandum = task.target_data
        context = task.context

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.8
        )

        # Generate explanation
        explanation = self._generate_explanation(explanandum, context)

        # Generate summary
        summary = f"Explanation generated for: {str(explanandum)[:100]}"

        # Create insights with the explanation
        insights = [{
            "type": "explanation",
            "description": explanation,
            "confidence": 0.8
        }]

        # Update result
        result.summary = summary
        result.insights = insights

        return result

    def _generate_explanation(self, explanandum, context):
        """
        Generate an explanation for the given item.

        This is a placeholder implementation. In a real system, this would use
        more sophisticated techniques, possibly involving LLMs, causal inference,
        or domain-specific explanation methods.
        """
        # For demonstration, we'll create a simple template-based explanation
        explanation = f"The system produced this result based on the following factors:\n\n"

        if isinstance(explanandum, dict):
            for key, value in explanandum.items():
                explanation += f"- {key}: {value}\n"
        else:
            explanation += f"- Input: {explanandum}\n"

        if context:
            explanation += "\nContextual factors that influenced this result:\n\n"

            for key, value in context.items():
                explanation += f"- {key}: {value}\n"

        explanation += "\nThe reasoning process involved evaluating the available information "
        explanation += "against established patterns and applying relevant rules or models."

        return explanation


class LessonExtractionStrategy(ReflectionStrategy):
    """Strategy for extracting lessons from experiences"""

    def __init__(self):
        super().__init__("LessonExtraction", ReflectionType.LESSON_EXTRACTION)

    def reflect(self, task: ReflectionTask) -> ReflectionResult:
        """
        Extract lessons from an experience.

        Args:
            task: The reflection task

        Returns:
            ReflectionResult containing extracted lessons
        """
        experience_data = task.target_data

        # Validate input
        if not isinstance(experience_data, dict):
            return ReflectionResult(
                task_id=task.task_id,
                success=False,
                summary="Invalid experience data format. Expected dictionary.",
                confidence=0.0
            )

        # Initialize result
        result = ReflectionResult(
            task_id=task.task_id,
            success=True,
            confidence=0.7
        )

        # Extract experience components
        experience_type = experience_data.get("type", "general")
        outcome = experience_data.get("outcome", "")
        success = experience_data.get("success", False)
        performance = experience_data.get("performance", {})
        steps = experience_data.get("steps", [])
        failures = experience_data.get("failures", [])
        context = experience_data.get("context", {})

        # Extract lessons
        insights = []
        recommendations = []

        # Lesson from outcome
        if outcome:
            insights.append({
                "type": "outcome_lesson",
                "description": f"Experience resulted in: {outcome}",
                "confidence": 0.9,
                "lesson": f"In {experience_type} scenarios, outcomes tend to be: {outcome}",
                "category": experience_type,
                "importance": 0.7 if success else 0.8  # Failures are slightly more important to learn from
            })

        # Lessons from performance
        if performance:
            # Identify top and bottom performance areas
            if len(performance) > 0:
                top_areas = sorted(performance.items(), key=lambda x: x[1], reverse=True)[:2]
                bottom_areas = sorted(performance.items(), key=lambda x: x[1])[:2]

                if top_areas:
                    insights.append({
                        "type": "strength_lesson",
                        "description": f"Strong performance in: {[a[0] for a in top_areas]}",
                        "confidence": 0.8,
                        "lesson": f"Strategies used in {[a[0] for a in top_areas]} are effective and should be maintained",
                        "category": "strengths",
                        "importance": 0.6
                    })

                if bottom_areas:
                    insights.append({
                        "type": "weakness_lesson",
                        "description": f"Weak performance in: {[a[0] for a in bottom_areas]}",
                        "confidence": 0.8,
                        "lesson": f"Improvement needed in {[a[0] for a in bottom_areas]} areas with adjusted strategies",
                        "category": "weaknesses",
                        "importance": 0.8
                    })

                    recommendations.append({
                        "type": "performance_improvement",
                        "description": f"Focus improvement efforts on {[a[0] for a in bottom_areas]}",
                        "areas": [a[0] for a in bottom_areas]
                    })

        # Lessons from failures
        if failures:
            failure_categories = {}

            # Group failures by type
            for failure in failures:
                failure_type = failure.get("type", "unknown")
                if failure_type not in failure_categories:
                    failure_categories[failure_type] = []

                failure_categories[failure_type].append(failure)

            # Extract lessons for each failure category
            for f_type, f_list in failure_categories.items():
                insights.append({
                    "type": "failure_lesson",
                    "description": f"{len(f_list)} failures of type {f_type}",
                    "confidence": 0.9,
                    "lesson": f"Prevent {f_type} failures by ensuring proper validation and handling",
                    "category": "failures",
                    "importance": 0.9  # Failures are high importance for learning
                })

                # If multiple failures of same type, recommend specific handling
                if len(f_list) >= 2:
                    recommendations.append({
                        "type": "error_handling",
                        "description": f"Implement specific handling for {f_type} failures",
                        "failure_type": f_type
                    })

        # Lessons from steps (if process-oriented experience)
        if steps and len(steps) > 0:
            # Identify bottlenecks or high-latency steps
            latency_steps = []
            for step in steps:
                if "latency" in step and step["latency"] > 0.5:  # Arbitrary threshold
                    latency_steps.append(step)

            if latency_steps:
                insights.append({
                    "type": "process_lesson",
                    "description": f"High latency identified in {len(latency_steps)} steps",
                    "confidence": 0.7,
                    "lesson": f"Process efficiency can be improved by optimizing high-latency steps",
                    "category": "process",
                    "importance": 0.7
                })

                recommendations.append({
                    "type": "process_improvement",
                    "description": "Optimize high-latency steps",
                    "steps": [s.get("name", i) for i, s in enumerate(latency_steps)]
                })

            # Identify optimal step sequencing
            if success and len(steps) > 3:
                insights.append({
                    "type": "sequence_lesson",
                    "description": "Successful process sequence identified",
                    "confidence": 0.6,
                    "lesson": "This step sequence can be applied to similar problems",
                    "category": "process",
                    "importance": 0.7
                })

        # Contextual lessons
        if context:
            # Identify key contextual factors
            insights.append({
                "type": "context_lesson",
                "description": f"Key contextual factors: {list(context.keys())}",
                "confidence": 0.7,
                "lesson": f"In {experience_type} scenarios, pay attention to these contextual factors: {list(context.keys())}",
                "category": "context",
                "importance": 0.6
            })

        # Generate summary
        if success:
            summary = f"Lessons extracted from successful {experience_type} experience."
        else:
            summary = f"Lessons extracted from unsuccessful {experience_type} experience."

        summary += f" {len(insights)} lessons identified across {len(set(i.get('category', 'general') for i in insights))} categories."

        # Update result
        result.summary = summary
        result.insights = insights
        result.recommendations = recommendations

        return result

ModuleNotFoundError: No module named 'grandnexus'

# Graph Engine

In [ ]:
"""
graph_engine.py - Advanced Graph Engine Module for GrandNexus

This module provides a comprehensive graph-based representation system for GrandNexus,
integrating seamlessly with memory and cognitive modules.

Features:
- Abstract base class with consistent interface
- Multiple graph engine implementations (InMemory, Neo4j)
- Cognitive graph operations (activation spreading, pattern mining)
- Advanced query capabilities
- Integration with semantic and episodic memory
- Support for both symbolic and subsymbolic representations
"""

import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json

# Optional dependencies for vector operations and mathematical functions
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

# Optional dependency for in-memory graph analytics
try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

# Optional dependency for Neo4j
try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False


# -----------------------------------------------------------------------------
# Core Data Structures
# -----------------------------------------------------------------------------

@dataclass
class GraphNode:
    """
    Represents a node in the graph with cognitive properties.

    Attributes:
        node_id: Unique identifier for the node
        label: Primary category or type of the node
        properties: Dictionary of node attributes
        embedding: Optional vector representation for similarity operations
        activation: Current activation level (0.0 to 1.0, higher = more activated)
        last_accessed: Timestamp of last access (for decay calculations)
        importance: Long-term importance score (0.0 to 1.0)
        creation_time: When the node was created
    """
    node_id: str
    label: str
    properties: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[Any] = None
    activation: float = 0.0
    last_accessed: float = field(default_factory=time.time)
    importance: float = 0.5
    creation_time: float = field(default_factory=time.time)

    def access(self) -> None:
        """Update activation and last_accessed time when node is accessed."""
        self.activation = 1.0
        self.last_accessed = time.time()

    def decay_activation(self, decay_rate: float, current_time: Optional[float] = None) -> None:
        """
        Apply decay to node activation based on time since last access.

        Args:
            decay_rate: Rate of activation decay (higher = faster decay)
            current_time: Current time or None to use current system time
        """
        if current_time is None:
            current_time = time.time()

        time_since_access = max(0.001, current_time - self.last_accessed)
        decay_factor = math.exp(-decay_rate * time_since_access)
        self.activation *= decay_factor

        # Ensure activation stays in valid range
        self.activation = max(0.0, min(1.0, self.activation))


@dataclass
class GraphEdge:
    """
    Represents a directed edge in the graph with cognitive properties.

    Attributes:
        edge_id: Unique identifier for the edge
        source_id: ID of the source node
        target_id: ID of the target node
        type: Relationship type (e.g., "IS_A", "PART_OF")
        properties: Dictionary of edge attributes
        weight: Strength of the connection (0.0 to 1.0, higher = stronger)
        bidirectional: Whether the relationship applies in both directions
        creation_time: When the edge was created
    """
    edge_id: str
    source_id: str
    target_id: str
    type: str
    properties: Dict[str, Any] = field(default_factory=dict)
    weight: float = 1.0
    bidirectional: bool = False
    creation_time: float = field(default_factory=time.time)


@dataclass
class GraphPattern:
    """
    Represents a discovered pattern in the graph.

    Attributes:
        pattern_id: Unique identifier for the pattern
        nodes: List of node IDs in this pattern
        edges: List of edge IDs in this pattern
        frequency: Number of occurrences in the graph
        significance: Measure of pattern importance
        embedding: Vector representation of the pattern
    """
    pattern_id: str
    nodes: List[str]
    edges: List[str]
    frequency: int = 1
    significance: float = 0.5
    embedding: Optional[Any] = None


@dataclass
class GraphQuery:
    """
    A flexible query specification for graph operations.

    Attributes:
        node_criteria: Criteria for matching nodes
        edge_criteria: Criteria for matching edges
        start_nodes: Optional set of starting node IDs
        max_depth: Maximum traversal depth
        max_results: Maximum number of results to return
        include_inactive: Whether to include nodes below activation threshold
        min_activation: Minimum activation level to consider
        sort_by: How to sort results ("activation", "importance", "recency")
    """
    node_criteria: Dict[str, Any] = field(default_factory=dict)
    edge_criteria: Dict[str, Any] = field(default_factory=dict)
    start_nodes: Optional[List[str]] = None
    max_depth: int = 3
    max_results: int = 100
    include_inactive: bool = False
    min_activation: float = 0.0
    sort_by: str = "activation"


# -----------------------------------------------------------------------------
# Exception Classes
# -----------------------------------------------------------------------------

class GraphEngineError(Exception):
    """Base class for graph-related errors."""
    pass

class NodeNotFoundError(GraphEngineError):
    """Raised when a node is not found."""
    pass

class EdgeNotFoundError(GraphEngineError):
    """Raised when an edge is not found."""
    pass

class InvalidQueryError(GraphEngineError):
    """Raised when a query is malformed."""
    pass


# -----------------------------------------------------------------------------
# Abstract Base Class
# -----------------------------------------------------------------------------

class AbstractGraphEngine(ABC):
    """
    Abstract base class defining the interface for graph engines in GrandNexus.

    This class defines the core operations that any graph engine implementation
    must support, ensuring a consistent interface regardless of the underlying
    storage or processing mechanism.
    """

    @abstractmethod
    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """
        Add a node to the graph.

        Args:
            node_id: Optional node ID (generated if None)
            label: Primary type or category of the node
            properties: Node attributes

        Returns:
            The ID of the created node
        """
        pass

    @abstractmethod
    def get_node(self, node_id: str) -> GraphNode:
        """
        Retrieve a node by ID.

        Args:
            node_id: ID of the node to retrieve

        Returns:
            The GraphNode object

        Raises:
            NodeNotFoundError: If node with given ID doesn't exist
        """
        pass

    @abstractmethod
    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """
        Update a node's properties.

        Args:
            node_id: ID of the node to update
            properties: Properties to update (merged with existing)

        Raises:
            NodeNotFoundError: If node with given ID doesn't exist
        """
        pass

    @abstractmethod
    def delete_node(self, node_id: str) -> None:
        """
        Delete a node and its connected edges.

        Args:
            node_id: ID of the node to delete

        Raises:
            NodeNotFoundError: If node with given ID doesn't exist
        """
        pass

    @abstractmethod
    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """
        Add an edge between two nodes.

        Args:
            source_id: ID of the source node
            target_id: ID of the target node
            edge_type: Type of relationship
            edge_id: Optional edge ID (generated if None)
            properties: Edge attributes
            weight: Strength of connection (0.0 to 1.0)
            bidirectional: Whether the relationship is bi-directional

        Returns:
            The ID of the created edge

        Raises:
            NodeNotFoundError: If source or target node doesn't exist
        """
        pass

    @abstractmethod
    def get_edge(self, edge_id: str) -> GraphEdge:
        """
        Retrieve an edge by ID.

        Args:
            edge_id: ID of the edge to retrieve

        Returns:
            The GraphEdge object

        Raises:
            EdgeNotFoundError: If edge with given ID doesn't exist
        """
        pass

    @abstractmethod
    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """
        Update an edge's properties.

        Args:
            edge_id: ID of the edge to update
            properties: Properties to update (merged with existing)

        Raises:
            EdgeNotFoundError: If edge with given ID doesn't exist
        """
        pass

    @abstractmethod
    def delete_edge(self, edge_id: str) -> None:
        """
        Delete an edge.

        Args:
            edge_id: ID of the edge to delete

        Raises:
            EdgeNotFoundError: If edge with given ID doesn't exist
        """
        pass

    @abstractmethod
    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """
        Get neighboring nodes connected to a node.

        Args:
            node_id: ID of the node
            edge_types: Optional filter for specific edge types
            direction: "outgoing", "incoming", or "both"

        Returns:
            List of (neighbor_id, edge_id, edge_type) tuples

        Raises:
            NodeNotFoundError: If node with given ID doesn't exist
        """
        pass

    @abstractmethod
    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """
        Find nodes matching the specified criteria.

        Args:
            criteria: Dictionary of property constraints

        Returns:
            List of matching node IDs
        """
        pass

    @abstractmethod
    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """
        Find edges matching the specified criteria.

        Args:
            criteria: Dictionary of property constraints

        Returns:
            List of matching edge IDs
        """
        pass

    @abstractmethod
    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """
        Find paths between two nodes.

        Args:
            start_id: ID of the starting node
            end_id: ID of the ending node
            max_depth: Maximum path length
            edge_types: Optional filter for specific edge types

        Returns:
            List of paths, where each path is a list of node IDs
        """
        pass

    @abstractmethod
    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """
        Execute a complex graph query.

        Args:
            query: GraphQuery object specifying the query parameters

        Returns:
            Dictionary with query results
        """
        pass

    @abstractmethod
    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """
        Spread activation from source nodes through the graph.

        Args:
            source_nodes: Starting nodes for activation spread
            decay_factor: How quickly activation diminishes (0-1)
            max_depth: Maximum steps to spread

        Returns:
            Dictionary mapping node IDs to activation values
        """
        pass

    @abstractmethod
    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """
        Get the currently activated subgraph.

        Args:
            threshold: Minimum activation level

        Returns:
            Tuple of (node_ids, edge_ids) for the activated subgraph
        """
        pass

    @abstractmethod
    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """
        Detect frequent patterns in the graph.

        Args:
            min_frequency: Minimum number of occurrences
            max_size: Maximum pattern size (nodes)

        Returns:
            List of GraphPattern objects
        """
        pass

    @abstractmethod
    def clear(self) -> None:
        """
        Clear all nodes and edges from the graph.
        """
        pass

    @abstractmethod
    def serialize(self) -> Dict[str, Any]:
        """
        Serialize the graph to a dictionary.

        Returns:
            Dictionary representation of the graph
        """
        pass

    @abstractmethod
    def deserialize(self, data: Dict[str, Any]) -> None:
        """
        Load the graph from a serialized representation.

        Args:
            data: Dictionary containing serialized graph
        """
        pass

    @abstractmethod
    def close(self) -> None:
        """
        Close the graph engine and release any resources.
        """
        pass


# -----------------------------------------------------------------------------
# In-Memory Graph Engine Implementation
# -----------------------------------------------------------------------------

class InMemoryGraphEngine(AbstractGraphEngine):
    """
    In-memory implementation of the graph engine with cognitive capabilities.

    This implementation keeps all nodes and edges in memory, optimized for
    cognitive operations like activation spreading and pattern detection.
    Optionally uses NetworkX for advanced graph algorithms.
    """

    def __init__(self, use_networkx: bool = True, decay_rate: float = 0.1):
        """
        Initialize the in-memory graph engine.

        Args:
            use_networkx: Whether to use NetworkX for graph algorithms (if available)
            decay_rate: Default activation decay rate
        """
        self.logger = logging.getLogger("InMemoryGraphEngine")
        self.use_networkx = use_networkx and HAS_NETWORKX
        self.decay_rate = decay_rate

        # Primary storage
        self.nodes: Dict[str, GraphNode] = {}
        self.edges: Dict[str, GraphEdge] = {}

        # Indices for efficient access
        self.outgoing_edges: Dict[str, List[str]] = defaultdict(list)  # node_id -> [edge_ids]
        self.incoming_edges: Dict[str, List[str]] = defaultdict(list)  # node_id -> [edge_ids]
        self.edges_by_type: Dict[str, List[str]] = defaultdict(list)   # type -> [edge_ids]
        self.nodes_by_label: Dict[str, List[str]] = defaultdict(list)  # label -> [node_ids]

        # NetworkX graph (if enabled)
        self._nx_graph = nx.MultiDiGraph() if self.use_networkx else None

        # Concurrency control
        self.lock = threading.RLock()

        self.logger.info(f"InMemoryGraphEngine initialized (use_networkx={self.use_networkx})")

    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """Add a node to the graph."""
        with self.lock:
            # Generate ID if not provided
            node_id = node_id or str(uuid.uuid4())
            properties = properties or {}

            # Create node
            node = GraphNode(
                node_id=node_id,
                label=label,
                properties=properties.copy()
            )

            # Store node
            self.nodes[node_id] = node
            self.nodes_by_label[label].append(node_id)

            # Add to NetworkX graph if enabled
            if self.use_networkx and self._nx_graph is not None:
                attrs = dict(properties)
                attrs.update({
                    'label': label,
                    'activation': node.activation,
                    'importance': node.importance
                })
                self._nx_graph.add_node(node_id, **attrs)

            self.logger.debug(f"Added node {node_id} with label '{label}'")
            return node_id

    def get_node(self, node_id: str) -> GraphNode:
        """Retrieve a node by ID."""
        with self.lock:
            if node_id not in self.nodes:
                raise NodeNotFoundError(f"Node {node_id} not found")

            node = self.nodes[node_id]
            node.access()  # Update activation and last_accessed
            return node

    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """Update a node's properties."""
        with self.lock:
            if node_id not in self.nodes:
                raise NodeNotFoundError(f"Node {node_id} not found")

            node = self.nodes[node_id]

            # Update properties
            node.properties.update(properties)
            node.last_accessed = time.time()

            # Update NetworkX if enabled
            if self.use_networkx and self._nx_graph is not None:
                for key, value in properties.items():
                    self._nx_graph.nodes[node_id][key] = value

                # Update common attributes
                self._nx_graph.nodes[node_id]['activation'] = node.activation
                self._nx_graph.nodes[node_id]['importance'] = node.importance

            self.logger.debug(f"Updated node {node_id}")

    def delete_node(self, node_id: str) -> None:
        """Delete a node and its connected edges."""
        with self.lock:
            if node_id not in self.nodes:
                raise NodeNotFoundError(f"Node {node_id} not found")

            # Delete connected edges first
            edges_to_delete = []
            edges_to_delete.extend(self.outgoing_edges.get(node_id, []))
            edges_to_delete.extend(self.incoming_edges.get(node_id, []))

            for edge_id in edges_to_delete:
                if edge_id in self.edges:  # Check to avoid duplicate deletions
                    self._delete_edge_internal(edge_id)

            # Remove from label index
            label = self.nodes[node_id].label
            if node_id in self.nodes_by_label.get(label, []):
                self.nodes_by_label[label].remove(node_id)

            # Remove from main storage
            node = self.nodes.pop(node_id)

            # Remove from NetworkX if enabled
            if self.use_networkx and self._nx_graph is not None:
                if self._nx_graph.has_node(node_id):
                    self._nx_graph.remove_node(node_id)

            # Clean up edge indices
            if node_id in self.outgoing_edges:
                del self.outgoing_edges[node_id]
            if node_id in self.incoming_edges:
                del self.incoming_edges[node_id]

            self.logger.debug(f"Deleted node {node_id} and its connected edges")

    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """Add an edge between two nodes."""
        with self.lock:
            # Verify nodes exist
            if source_id not in self.nodes:
                raise NodeNotFoundError(f"Source node {source_id} not found")
            if target_id not in self.nodes:
                raise NodeNotFoundError(f"Target node {target_id} not found")

            # Generate ID if not provided
            edge_id = edge_id or str(uuid.uuid4())
            properties = properties or {}

            # Create edge
            edge = GraphEdge(
                edge_id=edge_id,
                source_id=source_id,
                target_id=target_id,
                type=edge_type,
                properties=properties.copy(),
                weight=weight,
                bidirectional=bidirectional
            )

            # Store edge
            self.edges[edge_id] = edge
            self.outgoing_edges[source_id].append(edge_id)
            self.incoming_edges[target_id].append(edge_id)
            self.edges_by_type[edge_type].append(edge_id)

            # Add to NetworkX graph if enabled
            if self.use_networkx and self._nx_graph is not None:
                attrs = dict(properties)
                attrs.update({
                    'edge_id': edge_id,
                    'type': edge_type,
                    'weight': weight,
                    'bidirectional': bidirectional
                })
                self._nx_graph.add_edge(source_id, target_id, **attrs)

            # If bidirectional, add inverse edge automatically
            if bidirectional and source_id != target_id:
                inverse_id = f"{edge_id}_inverse"
                inverse_props = properties.copy()
                inverse_props['inverse_of'] = edge_id

                inverse_edge = GraphEdge(
                    edge_id=inverse_id,
                    source_id=target_id,
                    target_id=source_id,
                    type=edge_type,
                    properties=inverse_props,
                    weight=weight,
                    bidirectional=True
                )

                self.edges[inverse_id] = inverse_edge
                self.outgoing_edges[target_id].append(inverse_id)
                self.incoming_edges[source_id].append(inverse_id)
                self.edges_by_type[edge_type].append(inverse_id)

                if self.use_networkx and self._nx_graph is not None:
                    inverse_attrs = dict(inverse_props)
                    inverse_attrs.update({
                        'edge_id': inverse_id,
                        'type': edge_type,
                        'weight': weight,
                        'bidirectional': True
                    })
                    self._nx_graph.add_edge(target_id, source_id, **inverse_attrs)

            self.logger.debug(f"Added edge {edge_id} from {source_id} to {target_id} with type '{edge_type}'")
            return edge_id

    def get_edge(self, edge_id: str) -> GraphEdge:
        """Retrieve an edge by ID."""
        with self.lock:
            if edge_id not in self.edges:
                raise EdgeNotFoundError(f"Edge {edge_id} not found")

            return self.edges[edge_id]

    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """Update an edge's properties."""
        with self.lock:
            if edge_id not in self.edges:
                raise EdgeNotFoundError(f"Edge {edge_id} not found")

            edge = self.edges[edge_id]

            # Update properties
            edge.properties.update(properties)

            # Update weight if provided
            if 'weight' in properties:
                edge.weight = float(properties['weight'])

            # Update NetworkX if enabled
            if self.use_networkx and self._nx_graph is not None:
                # Get the edge in NetworkX
                source_id, target_id = edge.source_id, edge.target_id
                if self._nx_graph.has_edge(source_id, target_id):
                    # Find the edge with matching edge_id
                    for _, _, edge_data in self._nx_graph.edges(data=True):
                        if edge_data.get('edge_id') == edge_id:
                            # Update properties
                            for key, value in properties.items():
                                edge_data[key] = value
                            break

            # If this is part of a bidirectional pair, update the inverse too
            if edge.bidirectional:
                inverse_id = f"{edge_id}_inverse"
                if inverse_id in self.edges:
                    inverse_edge = self.edges[inverse_id]
                    for key, value in properties.items():
                        if key != 'inverse_of':  # Skip this to prevent recursion
                            inverse_edge.properties[key] = value

                    if 'weight' in properties:
                        inverse_edge.weight = float(properties['weight'])

            self.logger.debug(f"Updated edge {edge_id}")

    def delete_edge(self, edge_id: str) -> None:
        """Delete an edge."""
        with self.lock:
            if edge_id not in self.edges:
                raise EdgeNotFoundError(f"Edge {edge_id} not found")

            self._delete_edge_internal(edge_id)

            # If this is part of a bidirectional pair, delete the inverse too
            inverse_id = f"{edge_id}_inverse"
            if inverse_id in self.edges:
                self._delete_edge_internal(inverse_id)

            self.logger.debug(f"Deleted edge {edge_id}")

    def _delete_edge_internal(self, edge_id: str) -> None:
        """Internal helper to delete an edge without checking existence."""
        edge = self.edges[edge_id]

        # Remove from indices
        if edge_id in self.outgoing_edges.get(edge.source_id, []):
            self.outgoing_edges[edge.source_id].remove(edge_id)
        if edge_id in self.incoming_edges.get(edge.target_id, []):
            self.incoming_edges[edge.target_id].remove(edge_id)
        if edge_id in self.edges_by_type.get(edge.type, []):
            self.edges_by_type[edge.type].remove(edge_id)

        # Remove from NetworkX if enabled
        if self.use_networkx and self._nx_graph is not None:
            source_id, target_id = edge.source_id, edge.target_id
            # Remove the specific edge with this edge_id
            edges_to_remove = []
            for u, v, k, data in self._nx_graph.edges(source_id, target_id, keys=True, data=True):
                if data.get('edge_id') == edge_id:
                    edges_to_remove.append((u, v, k))

            for u, v, k in edges_to_remove:
                self._nx_graph.remove_edge(u, v, k)

        # Remove from main storage
        del self.edges[edge_id]

    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """Get neighboring nodes connected to a node."""
        with self.lock:
            if node_id not in self.nodes:
                raise NodeNotFoundError(f"Node {node_id} not found")

            result = []

            if direction in ["outgoing", "both"]:
                for edge_id in self.outgoing_edges.get(node_id, []):
                    if edge_id in self.edges:  # Ensure edge still exists
                        edge = self.edges[edge_id]
                        if edge_types is None or edge.type in edge_types:
                            result.append((edge.target_id, edge_id, edge.type))

            if direction in ["incoming", "both"]:
                for edge_id in self.incoming_edges.get(node_id, []):
                    if edge_id in self.edges:  # Ensure edge still exists
                        edge = self.edges[edge_id]
                        if edge_types is None or edge.type in edge_types:
                            result.append((edge.source_id, edge_id, edge.type))

            # Update activation for the node
            self.nodes[node_id].access()

            return result

    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """Find nodes matching the specified criteria."""
        with self.lock:
            result = []

            # Optimize for label queries
            if 'label' in criteria and len(criteria) == 1:
                label = criteria['label']
                return list(self.nodes_by_label.get(label, []))

            # General matching
            for node_id, node in self.nodes.items():
                if self._match_criteria(node.properties, criteria):
                    result.append(node_id)

            return result

    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """Find edges matching the specified criteria."""
        with self.lock:
            result = []

            # Optimize for type queries
            if 'type' in criteria and len(criteria) == 1:
                edge_type = criteria['type']
                return list(self.edges_by_type.get(edge_type, []))

            # General matching
            for edge_id, edge in self.edges.items():
                if self._match_criteria(edge.properties, criteria):
                    result.append(edge_id)

            return result

    def _match_criteria(self, properties: Dict[str, Any], criteria: Dict[str, Any]) -> bool:
        """Check if properties match the given criteria."""
        for key, value in criteria.items():
            if key not in properties:
                return False

            prop_value = properties[key]

            # Handle comparison operators
            if isinstance(value, dict):
                for op, op_value in value.items():
                    if op == '$gt':
                        if not (prop_value > op_value):
                            return False
                    elif op == '$gte':
                        if not (prop_value >= op_value):
                            return False
                    elif op == '$lt':
                        if not (prop_value < op_value):
                            return False
                    elif op == '$lte':
                        if not (prop_value <= op_value):
                            return False
                    elif op == '$ne':
                        if not (prop_value != op_value):
                            return False
                    elif op == '$in':
                        if not (prop_value in op_value):
                            return False
                    else:
                        # Unknown operator
                        return False
            else:
                # Simple equality
                if prop_value != value:
                    return False

        return True

    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """Find paths between two nodes."""
        with self.lock:
            # Verify nodes exist
            if start_id not in self.nodes:
                raise NodeNotFoundError(f"Start node {start_id} not found")
            if end_id not in self.nodes:
                raise NodeNotFoundError(f"End node {end_id} not found")

            # Use NetworkX if available for path finding
            if self.use_networkx and self._nx_graph is not None:
                return self._find_paths_networkx(start_id, end_id, max_depth, edge_types)
            else:
                return self._find_paths_bfs(start_id, end_id, max_depth, edge_types)

    def _find_paths_networkx(self, start_id: str, end_id: str, max_depth: int,
                           edge_types: Optional[List[str]]) -> List[List[str]]:
        """Find paths using NetworkX algorithms."""
        try:
            # If edge_types is specified, create a view of the graph with only those types
            if edge_types:
                view = nx.subgraph_view(
                    self._nx_graph,
                    filter_edge=lambda u, v, k: self._nx_graph[u][v][k].get('type') in edge_types
                )
            else:
                view = self._nx_graph

            # Find simple paths with cutoff
            all_paths = list(nx.all_simple_paths(view, start_id, end_id, cutoff=max_depth))
            return all_paths

        except (nx.NetworkXError, nx.NetworkXNoPath):
            return []

    def _find_paths_bfs(self, start_id: str, end_id: str, max_depth: int,
                      edge_types: Optional[List[str]]) -> List[List[str]]:
        """Find paths using breadth-first search (fallback if NetworkX is unavailable)."""
        if start_id == end_id:
            return [[start_id]]

        paths = []
        queue = deque([(start_id, [start_id], 0)])
        visited = set([start_id])

        while queue:
            node_id, path, depth = queue.popleft()

            if depth >= max_depth:
                continue

            neighbors = self.get_node_neighbors(node_id, edge_types, "outgoing")

            for neighbor_id, _, _ in neighbors:
                if neighbor_id == end_id:
                    paths.append(path + [neighbor_id])
                elif neighbor_id not in visited:
                    visited.add(neighbor_id)
                    queue.append((neighbor_id, path + [neighbor_id], depth + 1))

        return paths

    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """Execute a complex graph query."""
        with self.lock:
            result = {
                'nodes': [],
                'edges': [],
                'metadata': {
                    'query_time': time.time(),
                    'node_count': 0,
                    'edge_count': 0
                }
            }

            # Find starting nodes
            start_nodes = []
            if query.start_nodes:
                # Use specified start nodes
                start_nodes = [node_id for node_id in query.start_nodes if node_id in self.nodes]
            else:
                # Find nodes matching criteria
                start_nodes = self.query_nodes(query.node_criteria)

            if not start_nodes:
                return result

            # Apply activation threshold
            if not query.include_inactive and query.min_activation > 0:
                start_nodes = [node_id for node_id in start_nodes
                              if self.nodes[node_id].activation >= query.min_activation]

            # Build result nodes and edges
            processed_nodes = set()
            processed_edges = set()

            # Process start nodes
            for node_id in start_nodes:
                if node_id not in processed_nodes:
                    processed_nodes.add(node_id)
                    result['nodes'].append(self._node_to_dict(self.nodes[node_id]))

            # Expand graph if needed
            if query.max_depth > 0:
                for start_id in start_nodes:
                    self._expand_query_from_node(
                        start_id, query, processed_nodes, processed_edges, result, 0
                    )

            # Sort results if requested
            if query.sort_by == "activation":
                result['nodes'].sort(key=lambda n: n.get('activation', 0), reverse=True)
            elif query.sort_by == "importance":
                result['nodes'].sort(key=lambda n: n.get('importance', 0), reverse=True)
            elif query.sort_by == "recency":
                result['nodes'].sort(key=lambda n: n.get('last_accessed', 0), reverse=True)

            # Limit results if needed
            if query.max_results > 0:
                result['nodes'] = result['nodes'][:query.max_results]

                # Rebuild edges to only include those connected to returned nodes
                node_ids = {n['id'] for n in result['nodes']}
                result['edges'] = [e for e in result['edges']
                                  if e['source'] in node_ids and e['target'] in node_ids]

            # Update metadata
            result['metadata']['node_count'] = len(result['nodes'])
            result['metadata']['edge_count'] = len(result['edges'])

            return result

    def _expand_query_from_node(self, node_id: str, query: GraphQuery,
                              processed_nodes: Set[str], processed_edges: Set[str],
                              result: Dict[str, Any], depth: int) -> None:
        """Helper method to expand a query from a node recursively."""
        if depth >= query.max_depth:
            return

        # Get neighbors
        neighbors = self.get_node_neighbors(node_id, None, "both")

        for neighbor_id, edge_id, edge_type in neighbors:
            # Check edge criteria
            edge = self.edges[edge_id]
            if query.edge_criteria and not self._match_criteria(edge.properties, query.edge_criteria):
                continue

            # Process edge if not already processed
            if edge_id not in processed_edges:
                processed_edges.add(edge_id)
                result['edges'].append(self._edge_to_dict(edge))

            # Process neighbor node if not already processed
            if neighbor_id not in processed_nodes:
                neighbor = self.nodes[neighbor_id]

                # Check node criteria and activation
                if (query.node_criteria and not self._match_criteria(neighbor.properties, query.node_criteria)):
                    continue

                if not query.include_inactive and neighbor.activation < query.min_activation:
                    continue

                processed_nodes.add(neighbor_id)
                result['nodes'].append(self._node_to_dict(neighbor))

                # Recurse
                self._expand_query_from_node(
                    neighbor_id, query, processed_nodes, processed_edges, result, depth + 1
                )

    def _node_to_dict(self, node: GraphNode) -> Dict[str, Any]:
        """Convert a node to a dictionary representation."""
        return {
            'id': node.node_id,
            'label': node.label,
            'properties': node.properties.copy(),
            'activation': node.activation,
            'importance': node.importance,
            'last_accessed': node.last_accessed,
            'creation_time': node.creation_time
        }

    def _edge_to_dict(self, edge: GraphEdge) -> Dict[str, Any]:
        """Convert an edge to a dictionary representation."""
        return {
            'id': edge.edge_id,
            'source': edge.source_id,
            'target': edge.target_id,
            'type': edge.type,
            'properties': edge.properties.copy(),
            'weight': edge.weight,
            'bidirectional': edge.bidirectional,
            'creation_time': edge.creation_time
        }

    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """Spread activation from source nodes through the graph."""
        with self.lock:
            # Verify nodes exist
            for node_id in source_nodes:
                if node_id not in self.nodes:
                    raise NodeNotFoundError(f"Source node {node_id} not found")

            # Initialize activation
            activation = {}
            for node_id in self.nodes:
                if node_id in source_nodes:
                    activation[node_id] = 1.0  # Full activation for source nodes
                    self.nodes[node_id].activation = 1.0
                else:
                    activation[node_id] = self.nodes[node_id].activation

            # Spread activation with BFS
            queue = deque([(node_id, 0) for node_id in source_nodes])
            visited = set(source_nodes)

            while queue:
                node_id, depth = queue.popleft()

                if depth >= max_depth:
                    continue

                current_activation = activation[node_id]

                # Get outgoing neighbors
                neighbors = self.get_node_neighbors(node_id, None, "outgoing")

                # Calculate activation to spread
                spread_activation = current_activation * decay_factor

                for neighbor_id, edge_id, _ in neighbors:
                    edge = self.edges[edge_id]

                    # Weight by edge weight
                    neighbor_activation = spread_activation * edge.weight

                    # Add to existing activation
                    activation[neighbor_id] = min(1.0, activation[neighbor_id] + neighbor_activation)

                    # Update node activation
                    self.nodes[neighbor_id].activation = activation[neighbor_id]

                    # Continue spreading if not visited
                    if neighbor_id not in visited:
                        visited.add(neighbor_id)
                        queue.append((neighbor_id, depth + 1))

            # Return only non-zero activations
            return {node_id: act for node_id, act in activation.items() if act > 0}

    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """Get the currently activated subgraph."""
        with self.lock:
            # Find activated nodes
            activated_nodes = [node_id for node_id, node in self.nodes.items()
                              if node.activation >= threshold]

            # Find edges between activated nodes
            activated_edges = []
            for node_id in activated_nodes:
                for edge_id in self.outgoing_edges.get(node_id, []):
                    edge = self.edges.get(edge_id)
                    if edge and edge.target_id in activated_nodes:
                        activated_edges.append(edge_id)

            return activated_nodes, activated_edges

    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """Detect frequent patterns in the graph."""
        with self.lock:
            if self.use_networkx and self._nx_graph is not None and HAS_NUMPY:
                return self._detect_patterns_networkx(min_frequency, max_size)
            else:
                return self._detect_patterns_basic(min_frequency, max_size)

    def _detect_patterns_networkx(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Detect patterns using NetworkX algorithms."""
        try:
            # This is a simplified approach using motifs/subgraphs
            # A production implementation would use a more sophisticated algorithm
            patterns = []

            # Sample subgraphs of different sizes
            for size in range(2, max_size + 1):
                # Find connected components
                for c in nx.connected_components(self._nx_graph.to_undirected()):
                    if len(c) == size:
                        subgraph = self._nx_graph.subgraph(c)
                        pattern_id = f"pattern_{len(patterns)}"

                        # Create pattern
                        pattern = GraphPattern(
                            pattern_id=pattern_id,
                            nodes=list(subgraph.nodes()),
                            edges=[edge[2]['edge_id'] for edge in subgraph.edges(data=True)
                                   if 'edge_id' in edge[2]],
                            frequency=self._count_pattern_occurrences(subgraph),
                            significance=0.5,  # Placeholder
                            embedding=None  # Placeholder
                        )

                        if pattern.frequency >= min_frequency:
                            patterns.append(pattern)

            return patterns

        except Exception as e:
            self.logger.error(f"Error detecting patterns: {e}")
            return []

    def _count_pattern_occurrences(self, pattern_graph: 'nx.Graph') -> int:
        """Count occurrences of a pattern in the graph."""
        # This is a simplified implementation
        # In a production system, this would use subgraph isomorphism algorithms
        return 2  # Placeholder

    def _detect_patterns_basic(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Basic pattern detection fallback."""
        # This is a very simplified approach that just looks for star patterns
        # A production implementation would use more sophisticated algorithms
        patterns = []

        # Look for hub nodes (nodes with many connections)
        for node_id, node in self.nodes.items():
            outgoing = self.outgoing_edges.get(node_id, [])
            incoming = self.incoming_edges.get(node_id, [])

            if len(outgoing) + len(incoming) >= 3:  # Hub threshold
                # Create a star pattern
                pattern_id = f"pattern_{len(patterns)}"

                # Collect connected nodes and edges
                connected_nodes = [node_id]
                pattern_edges = []

                for edge_id in outgoing:
                    edge = self.edges[edge_id]
                    connected_nodes.append(edge.target_id)
                    pattern_edges.append(edge_id)

                # Truncate to max_size
                if len(connected_nodes) > max_size:
                    connected_nodes = connected_nodes[:max_size]
                    pattern_edges = pattern_edges[:max_size-1]

                pattern = GraphPattern(
                    pattern_id=pattern_id,
                    nodes=connected_nodes,
                    edges=pattern_edges,
                    frequency=min_frequency,  # Placeholder
                    significance=0.5,  # Placeholder
                    embedding=None  # Placeholder
                )

                patterns.append(pattern)

        return patterns

    def clear(self) -> None:
        """Clear all nodes and edges from the graph."""
        with self.lock:
            self.nodes.clear()
            self.edges.clear()
            self.outgoing_edges.clear()
            self.incoming_edges.clear()
            self.edges_by_type.clear()
            self.nodes_by_label.clear()

            if self.use_networkx and self._nx_graph is not None:
                self._nx_graph.clear()

            self.logger.info("Graph cleared")

    def serialize(self) -> Dict[str, Any]:
        """Serialize the graph to a dictionary."""
        with self.lock:
            data = {
                'nodes': [],
                'edges': [],
                'metadata': {
                    'timestamp': time.time(),
                    'node_count': len(self.nodes),
                    'edge_count': len(self.edges)
                }
            }

            # Serialize nodes
            for node_id, node in self.nodes.items():
                node_data = {
                    'id': node.node_id,
                    'label': node.label,
                    'properties': node.properties,
                    'activation': node.activation,
                    'importance': node.importance,
                    'creation_time': node.creation_time,
                    'last_accessed': node.last_accessed
                }

                # Handle embedding if present
                if node.embedding is not None and HAS_NUMPY:
                    if isinstance(node.embedding, np.ndarray):
                        node_data['embedding'] = node.embedding.tolist()
                    else:
                        node_data['embedding'] = node.embedding

                data['nodes'].append(node_data)

            # Serialize edges
            for edge_id, edge in self.edges.items():
                edge_data = {
                    'id': edge.edge_id,
                    'source': edge.source_id,
                    'target': edge.target_id,
                    'type': edge.type,
                    'properties': edge.properties,
                    'weight': edge.weight,
                    'bidirectional': edge.bidirectional,
                    'creation_time': edge.creation_time
                }

                data['edges'].append(edge_data)

            return data

    def deserialize(self, data: Dict[str, Any]) -> None:
        """Load the graph from a serialized representation."""
        with self.lock:
            # Clear existing data
            self.clear()

            # Load nodes first
            for node_data in data.get('nodes', []):
                node_id = node_data['id']

                # Create embedding if present
                embedding = None
                if 'embedding' in node_data and HAS_NUMPY:
                    embedding_data = node_data['embedding']
                    if embedding_data:
                        embedding = np.array(embedding_data)

                # Create node
                node = GraphNode(
                    node_id=node_id,
                    label=node_data['label'],
                    properties=node_data.get('properties', {}),
                    embedding=embedding,
                    activation=node_data.get('activation', 0.0),
                    last_accessed=node_data.get('last_accessed', time.time()),
                    importance=node_data.get('importance', 0.5),
                    creation_time=node_data.get('creation_time', time.time())
                )

                # Add to storage
                self.nodes[node_id] = node
                self.nodes_by_label[node.label].append(node_id)

                # Add to NetworkX if enabled
                if self.use_networkx and self._nx_graph is not None:
                    attrs = dict(node.properties)
                    attrs.update({
                        'label': node.label,
                        'activation': node.activation,
                        'importance': node.importance
                    })
                    self._nx_graph.add_node(node_id, **attrs)

            # Load edges
            for edge_data in data.get('edges', []):
                edge_id = edge_data['id']
                source_id = edge_data['source']
                target_id = edge_data['target']

                # Skip if source or target doesn't exist
                if source_id not in self.nodes or target_id not in self.nodes:
                    self.logger.warning(f"Skipping edge {edge_id}: source or target not found")
                    continue

                # Create edge
                edge = GraphEdge(
                    edge_id=edge_id,
                    source_id=source_id,
                    target_id=target_id,
                    type=edge_data['type'],
                    properties=edge_data.get('properties', {}),
                    weight=edge_data.get('weight', 1.0),
                    bidirectional=edge_data.get('bidirectional', False),
                    creation_time=edge_data.get('creation_time', time.time())
                )

                # Add to storage (avoid using add_edge to prevent duplicate bidirectional edges)
                self.edges[edge_id] = edge
                self.outgoing_edges[source_id].append(edge_id)
                self.incoming_edges[target_id].append(edge_id)
                self.edges_by_type[edge.type].append(edge_id)

                # Add to NetworkX if enabled
                if self.use_networkx and self._nx_graph is not None:
                    attrs = dict(edge.properties)
                    attrs.update({
                        'edge_id': edge_id,
                        'type': edge.type,
                        'weight': edge.weight,
                        'bidirectional': edge.bidirectional
                    })
                    self._nx_graph.add_edge(source_id, target_id, **attrs)

            self.logger.info(f"Deserialized graph with {len(self.nodes)} nodes and {len(self.edges)} edges")

    def close(self) -> None:
        """Close the graph engine and release any resources."""
        # For in-memory graph, there's not much to close
        # This method is primarily for interface consistency with other implementations
        self.logger.info("InMemoryGraphEngine closed")


# -----------------------------------------------------------------------------
# Neo4j Graph Engine Implementation
# -----------------------------------------------------------------------------

class Neo4jGraphEngine(AbstractGraphEngine):
    """
    Neo4j-based implementation of the graph engine.

    This implementation uses Neo4j as a backend store, providing scalability
    for large graphs while maintaining the cognitive capabilities.
    """

    def __init__(self, uri: str, user: str, password: str, database: str = "neo4j",
                decay_rate: float = 0.1, use_apoc: bool = False):
        """
        Initialize the Neo4j graph engine.

        Args:
            uri: Neo4j connection URI (e.g., "bolt://localhost:7687")
            user: Neo4j username
            password: Neo4j password
            database: Neo4j database name
            decay_rate: Default activation decay rate
            use_apoc: Whether to use APOC procedures (must be installed in Neo4j)
        """
        if not HAS_NEO4J:
            raise ImportError("Neo4j driver not available. Please install it with 'pip install neo4j'.")

        self.logger = logging.getLogger("Neo4jGraphEngine")
        self.uri = uri
        self.user = user
        self.password = password
        self.database = database
        self.decay_rate = decay_rate
        self.use_apoc = use_apoc

        # Initialize driver
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

        # Test connection
        self._test_connection()

        # Ensure constraints and indices
        self._setup_schema()

        # Local caches to avoid repeated database lookups
        self._node_cache: Dict[str, GraphNode] = {}
        self._edge_cache: Dict[str, GraphEdge] = {}

        # Concurrency control
        self.lock = threading.RLock()

        self.logger.info(f"Neo4jGraphEngine initialized (connected to {uri}, database={database})")

    def _test_connection(self) -> None:
        """Test the Neo4j connection."""
        try:
            with self.driver.session(database=self.database) as session:
                result = session.run("RETURN 1 AS test")
                test_value = result.single()["test"]
                if test_value != 1:
                    raise ConnectionError("Neo4j connection test failed")
        except Exception as e:
            self.logger.error(f"Failed to connect to Neo4j: {e}")
            raise ConnectionError(f"Could not connect to Neo4j: {e}")

    def _setup_schema(self) -> None:
        """Set up constraints and indices for optimized performance."""
        try:
            with self.driver.session(database=self.database) as session:
                # Create constraints
                session.run("CREATE CONSTRAINT node_id IF NOT EXISTS FOR (n:Node) REQUIRE n.node_id IS UNIQUE")
                session.run("CREATE CONSTRAINT edge_id IF NOT EXISTS FOR ()-[r:RELATES]-() REQUIRE r.edge_id IS UNIQUE")

                # Create indices
                session.run("CREATE INDEX node_label IF NOT EXISTS FOR (n:Node) ON (n.label)")
                session.run("CREATE INDEX node_activation IF NOT EXISTS FOR (n:Node) ON (n.activation)")
                session.run("CREATE INDEX edge_type IF NOT EXISTS FOR ()-[r:RELATES]-() ON (r.type)")
        except Exception as e:
            self.logger.warning(f"Failed to set up schema: {e}. Continuing without constraints.")

    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """Add a node to the graph."""
        with self.lock:
            # Generate ID if not provided
            node_id = node_id or str(uuid.uuid4())
            properties = properties or {}

            # Create node record
            cypher = """
            CREATE (n:Node {node_id: $node_id, label: $label,
                        activation: $activation,
                        importance: $importance,
                        creation_time: $creation_time,
                        last_accessed: $last_accessed})
            SET n += $properties
            RETURN n.node_id AS node_id
            """

            params = {
                "node_id": node_id,
                "label": label,
                "activation": 0.0,
                "importance": 0.5,
                "creation_time": time.time(),
                "last_accessed": time.time(),
                "properties": properties
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    created_id = result.single()["node_id"]

                    # Create and cache the node
                    node = GraphNode(
                        node_id=created_id,
                        label=label,
                        properties=properties.copy(),
                        activation=0.0,
                        importance=0.5
                    )
                    self._node_cache[created_id] = node

                    self.logger.debug(f"Added node {created_id} with label '{label}'")
                    return created_id
            except Exception as e:
                self.logger.error(f"Error adding node: {e}")
                raise GraphEngineError(f"Failed to add node: {e}")

    def get_node(self, node_id: str) -> GraphNode:
        """Retrieve a node by ID."""
        with self.lock:
            # Check cache first
            if node_id in self._node_cache:
                node = self._node_cache[node_id]
                node.access()  # Update activation and last_accessed
                self._update_node_activation(node_id, node.activation, node.last_accessed)
                return node

            # Query the database
            cypher = """
            MATCH (n:Node {node_id: $node_id})
            SET n.activation = 1.0, n.last_accessed = $current_time
            RETURN n.node_id AS id, n.label AS label,
                   n.activation AS activation, n.importance AS importance,
                   n.creation_time AS creation_time, n.last_accessed AS last_accessed,
                   properties(n) AS properties
            """

            params = {
                "node_id": node_id,
                "current_time": time.time()
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    record = result.single()

                    if not record:
                        raise NodeNotFoundError(f"Node {node_id} not found")

                    # Extract properties, excluding known attributes
                    props = dict(record["properties"])
                    for key in ["node_id", "label", "activation", "importance", "creation_time", "last_accessed"]:
                        if key in props:
                            del props[key]

                    # Create and cache node
                    node = GraphNode(
                        node_id=record["id"],
                        label=record["label"],
                        properties=props,
                        activation=record["activation"],
                        last_accessed=record["last_accessed"],
                        importance=record["importance"],
                        creation_time=record["creation_time"]
                    )

                    self._node_cache[node_id] = node
                    return node
            except Exception as e:
                if isinstance(e, NodeNotFoundError):
                    raise
                self.logger.error(f"Error retrieving node: {e}")
                raise GraphEngineError(f"Failed to retrieve node: {e}")

    def _update_node_activation(self, node_id: str, activation: float, last_accessed: float) -> None:
        """Update a node's activation and last_accessed time in the database."""
        cypher = """
        MATCH (n:Node {node_id: $node_id})
        SET n.activation = $activation, n.last_accessed = $last_accessed
        """

        params = {
            "node_id": node_id,
            "activation": activation,
            "last_accessed": last_accessed
        }

        try:
            with self.driver.session(database=self.database) as session:
                session.run(cypher, params)
        except Exception as e:
            self.logger.warning(f"Failed to update node activation: {e}")

    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """Update a node's properties."""
        with self.lock:
            # Update cache if present
            if node_id in self._node_cache:
                node = self._node_cache[node_id]
                node.properties.update(properties)
                node.last_accessed = time.time()

            # Update in database
            cypher = """
            MATCH (n:Node {node_id: $node_id})
            SET n += $properties, n.last_accessed = $current_time
            """

            params = {
                "node_id": node_id,
                "properties": properties,
                "current_time": time.time()
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    if result.consume().counters.nodes_created == 0 and \
                       result.consume().counters.properties_set == 0:
                        raise NodeNotFoundError(f"Node {node_id} not found")

                    self.logger.debug(f"Updated node {node_id}")
            except Exception as e:
                if isinstance(e, NodeNotFoundError):
                    raise
                self.logger.error(f"Error updating node: {e}")
                raise GraphEngineError(f"Failed to update node: {e}")

    def delete_node(self, node_id: str) -> None:
        """Delete a node and its connected edges."""
        with self.lock:
            # Remove from cache
            if node_id in self._node_cache:
                del self._node_cache[node_id]

            # Delete from database
            cypher = """
            MATCH (n:Node {node_id: $node_id})
            OPTIONAL MATCH (n)-[r]-()
            DELETE r, n
            """

            params = {"node_id": node_id}

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    if result.consume().counters.nodes_deleted == 0:
                        raise NodeNotFoundError(f"Node {node_id} not found")

                    # Clean up edge cache - remove edges connected to this node
                    edge_ids_to_remove = []
                    for edge_id, edge in self._edge_cache.items():
                        if edge.source_id == node_id or edge.target_id == node_id:
                            edge_ids_to_remove.append(edge_id)

                    for edge_id in edge_ids_to_remove:
                        if edge_id in self._edge_cache:
                            del self._edge_cache[edge_id]

                    self.logger.debug(f"Deleted node {node_id} and its connected edges")
            except Exception as e:
                if isinstance(e, NodeNotFoundError):
                    raise
                self.logger.error(f"Error deleting node: {e}")
                raise GraphEngineError(f"Failed to delete node: {e}")

    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """Add an edge between two nodes."""
        with self.lock:
            # Generate ID if not provided
            edge_id = edge_id or str(uuid.uuid4())
            properties = properties or {}

            # Create edge in database
            cypher = """
            MATCH (s:Node {node_id: $source_id})
            MATCH (t:Node {node_id: $target_id})
            CREATE (s)-[r:RELATES]->(t)
            SET r.edge_id = $edge_id, r.type = $edge_type, r.weight = $weight,
                r.bidirectional = $bidirectional, r.creation_time = $creation_time
            SET r += $properties
            RETURN r.edge_id AS edge_id
            """

            params = {
                "source_id": source_id,
                "target_id": target_id,
                "edge_id": edge_id,
                "edge_type": edge_type,
                "weight": weight,
                "bidirectional": bidirectional,
                "creation_time": time.time(),
                "properties": properties
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    created_id = result.single()["edge_id"]

                    # Create and cache the edge
                    edge = GraphEdge(
                        edge_id=created_id,
                        source_id=source_id,
                        target_id=target_id,
                        type=edge_type,
                        properties=properties.copy(),
                        weight=weight,
                        bidirectional=bidirectional
                    )
                    self._edge_cache[created_id] = edge

                    # If bidirectional, add inverse edge
                    if bidirectional and source_id != target_id:
                        inverse_id = f"{edge_id}_inverse"
                        inverse_cypher = """
                        MATCH (s:Node {node_id: $target_id})
                        MATCH (t:Node {node_id: $source_id})
                        CREATE (s)-[r:RELATES]->(t)
                        SET r.edge_id = $edge_id, r.type = $edge_type, r.weight = $weight,
                            r.bidirectional = $bidirectional, r.creation_time = $creation_time,
                            r.inverse_of = $original_edge_id
                        SET r += $properties
                        RETURN r.edge_id AS edge_id
                        """

                        inverse_params = {
                            "source_id": source_id,
                            "target_id": target_id,
                            "edge_id": inverse_id,
                            "edge_type": edge_type,
                            "weight": weight,
                            "bidirectional": bidirectional,
                            "creation_time": time.time(),
                            "properties": properties,
                            "original_edge_id": edge_id
                        }

                        result = session.run(inverse_cypher, inverse_params)
                        inverse_edge = GraphEdge(
                            edge_id=inverse_id,
                            source_id=target_id,
                            target_id=source_id,
                            type=edge_type,
                            properties={**properties, "inverse_of": edge_id},
                            weight=weight,
                            bidirectional=True
                        )
                        self._edge_cache[inverse_id] = inverse_edge

                    self.logger.debug(f"Added edge {created_id} from {source_id} to {target_id} with type '{edge_type}'")
                    return created_id
            except Exception as e:
                self.logger.error(f"Error adding edge: {e}")
                raise GraphEngineError(f"Failed to add edge: {e}")

    def get_edge(self, edge_id: str) -> GraphEdge:
        """Retrieve an edge by ID."""
        with self.lock:
            # Check cache first
            if edge_id in self._edge_cache:
                return self._edge_cache[edge_id]

            # Query the database
            cypher = """
            MATCH (s)-[r:RELATES]->(t)
            WHERE r.edge_id = $edge_id
            RETURN r.edge_id AS id, s.node_id AS source_id, t.node_id AS target_id,
                   r.type AS type, r.weight AS weight, r.bidirectional AS bidirectional,
                   r.creation_time AS creation_time, properties(r) AS properties
            """

            params = {"edge_id": edge_id}

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    record = result.single()

                    if not record:
                        raise EdgeNotFoundError(f"Edge {edge_id} not found")

                    # Extract properties, excluding known attributes
                    props = dict(record["properties"])
                    for key in ["edge_id", "type", "weight", "bidirectional", "creation_time"]:
                        if key in props:
                            del props[key]

                    # Create and cache edge
                    edge = GraphEdge(
                        edge_id=record["id"],
                        source_id=record["source_id"],
                        target_id=record["target_id"],
                        type=record["type"],
                        properties=props,
                        weight=record["weight"],
                        bidirectional=record["bidirectional"],
                        creation_time=record["creation_time"]
                    )

                    self._edge_cache[edge_id] = edge
                    return edge
            except Exception as e:
                if isinstance(e, EdgeNotFoundError):
                    raise
                self.logger.error(f"Error retrieving edge: {e}")
                raise GraphEngineError(f"Failed to retrieve edge: {e}")

    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """Update an edge's properties."""
        with self.lock:
            # Update cache if present
            if edge_id in self._edge_cache:
                edge = self._edge_cache[edge_id]
                edge.properties.update(properties)

            # Update in database
            cypher = """
            MATCH ()-[r:RELATES]->()
            WHERE r.edge_id = $edge_id
            SET r += $properties
            """

            params = {
                "edge_id": edge_id,
                "properties": properties
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    if result.consume().counters.properties_set == 0:
                        raise EdgeNotFoundError(f"Edge {edge_id} not found")

                    # Update weight in cache if it changed
                    if "weight" in properties and edge_id in self._edge_cache:
                        self._edge_cache[edge_id].weight = properties["weight"]

                    self.logger.debug(f"Updated edge {edge_id}")
            except Exception as e:
                if isinstance(e, EdgeNotFoundError):
                    raise
                self.logger.error(f"Error updating edge: {e}")
                raise GraphEngineError(f"Failed to update edge: {e}")

    def delete_edge(self, edge_id: str) -> None:
        """Delete an edge."""
        with self.lock:
            # Remove from cache
            if edge_id in self._edge_cache:
                edge = self._edge_cache[edge_id]
                bidirectional = edge.bidirectional
                del self._edge_cache[edge_id]

                # Handle bidirectional edge
                if bidirectional:
                    inverse_id = f"{edge_id}_inverse"
                    if inverse_id in self._edge_cache:
                        del self._edge_cache[inverse_id]

            # Delete from database
            cypher = """
            MATCH ()-[r:RELATES]->()
            WHERE r.edge_id = $edge_id
            DELETE r
            """

            params = {"edge_id": edge_id}

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    if result.consume().counters.relationships_deleted == 0:
                        raise EdgeNotFoundError(f"Edge {edge_id} not found")

                    # Delete inverse edge if it exists
                    inverse_id = f"{edge_id}_inverse"
                    session.run(cypher, {"edge_id": inverse_id})

                    self.logger.debug(f"Deleted edge {edge_id}")
            except Exception as e:
                if isinstance(e, EdgeNotFoundError):
                    raise
                self.logger.error(f"Error deleting edge: {e}")
                raise GraphEngineError(f"Failed to delete edge: {e}")

    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """Get neighboring nodes connected to a node."""
        with self.lock:
            # Prepare Cypher query based on direction
            if direction == "outgoing":
                cypher = """
                MATCH (n:Node {node_id: $node_id})-[r:RELATES]->(neighbor:Node)
                WHERE $edge_types IS NULL OR r.type IN $edge_types
                RETURN neighbor.node_id AS neighbor_id, r.edge_id AS edge_id, r.type AS edge_type
                """
            elif direction == "incoming":
                cypher = """
                MATCH (n:Node {node_id: $node_id})<-[r:RELATES]-(neighbor:Node)
                WHERE $edge_types IS NULL OR r.type IN $edge_types
                RETURN neighbor.node_id AS neighbor_id, r.edge_id AS edge_id, r.type AS edge_type
                """
            elif direction == "both":
                cypher = """
                MATCH (n:Node {node_id: $node_id})-[r:RELATES]-(neighbor:Node)
                WHERE $edge_types IS NULL OR r.type IN $edge_types
                RETURN neighbor.node_id AS neighbor_id, r.edge_id AS edge_id, r.type AS edge_type
                """
            else:
                raise ValueError(f"Invalid direction: {direction}. Must be 'outgoing', 'incoming', or 'both'.")

            params = {
                "node_id": node_id,
                "edge_types": edge_types
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    neighbors = []

                    for record in result:
                        neighbors.append((
                            record["neighbor_id"],
                            record["edge_id"],
                            record["edge_type"]
                        ))

                    # Activate the node
                    self._update_node_activation(node_id, 1.0, time.time())

                    return neighbors
            except Exception as e:
                self.logger.error(f"Error getting node neighbors: {e}")
                raise GraphEngineError(f"Failed to get node neighbors: {e}")

    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """Find nodes matching the specified criteria."""
        with self.lock:
            # Prepare WHERE clause from criteria
            where_clauses = []
            params = {}

            for i, (key, value) in enumerate(criteria.items()):
                param_name = f"param{i}"

                if isinstance(value, dict):
                    # Handle comparison operators
                    for op, op_value in value.items():
                        op_param = f"{param_name}_{op}"

                        if op == '$gt':
                            where_clauses.append(f"n.{key} > ${op_param}")
                        elif op == '$gte':
                            where_clauses.append(f"n.{key} >= ${op_param}")
                        elif op == '$lt':
                            where_clauses.append(f"n.{key} < ${op_param}")
                        elif op == '$lte':
                            where_clauses.append(f"n.{key} <= ${op_param}")
                        elif op == '$ne':
                            where_clauses.append(f"n.{key} <> ${op_param}")
                        elif op == '$in':
                            where_clauses.append(f"n.{key} IN ${op_param}")
                        else:
                            # Unknown operator, skip
                            continue

                        params[op_param] = op_value
                else:
                    # Simple equality
                    where_clauses.append(f"n.{key} = ${param_name}")
                    params[param_name] = value

            # Build query
            where_clause = " AND ".join(where_clauses) if where_clauses else "TRUE"

            cypher = f"""
            MATCH (n:Node)
            WHERE {where_clause}
            RETURN n.node_id AS node_id
            """

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    node_ids = [record["node_id"] for record in result]
                    return node_ids
            except Exception as e:
                self.logger.error(f"Error querying nodes: {e}")
                raise GraphEngineError(f"Failed to query nodes: {e}")

    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """Find edges matching the specified criteria."""
        with self.lock:
            # Prepare WHERE clause from criteria
            where_clauses = []
            params = {}

            for i, (key, value) in enumerate(criteria.items()):
                param_name = f"param{i}"

                if isinstance(value, dict):
                    # Handle comparison operators
                    for op, op_value in value.items():
                        op_param = f"{param_name}_{op}"

                        if op == '$gt':
                            where_clauses.append(f"r.{key} > ${op_param}")
                        elif op == '$gte':
                            where_clauses.append(f"r.{key} >= ${op_param}")
                        elif op == '$lt':
                            where_clauses.append(f"r.{key} < ${op_param}")
                        elif op == '$lte':
                            where_clauses.append(f"r.{key} <= ${op_param}")
                        elif op == '$ne':
                            where_clauses.append(f"r.{key} <> ${op_param}")
                        elif op == '$in':
                            where_clauses.append(f"r.{key} IN ${op_param}")
                        else:
                            # Unknown operator, skip
                            continue

                        params[op_param] = op_value
                else:
                    # Simple equality
                    where_clauses.append(f"r.{key} = ${param_name}")
                    params[param_name] = value

            # Build query
            where_clause = " AND ".join(where_clauses) if where_clauses else "TRUE"

            cypher = f"""
            MATCH ()-[r:RELATES]->()
            WHERE {where_clause}
            RETURN r.edge_id AS edge_id
            """

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    edge_ids = [record["edge_id"] for record in result]
                    return edge_ids
            except Exception as e:
                self.logger.error(f"Error querying edges: {e}")
                raise GraphEngineError(f"Failed to query edges: {e}")

    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """Find paths between two nodes."""
        with self.lock:
            # Verify nodes exist
            if not self._node_exists(start_id):
                raise NodeNotFoundError(f"Start node {start_id} not found")
            if not self._node_exists(end_id):
                raise NodeNotFoundError(f"End node {end_id} not found")

            # Prepare Cypher query
            type_filter = ""
            if edge_types:
                type_filter = "AND r.type IN $edge_types"

            # Use shortest path algorithm if available, otherwise variable length path
            if self.use_apoc:
                cypher = f"""
                MATCH (start:Node {{node_id: $start_id}}), (end:Node {{node_id: $end_id}})
                CALL apoc.algo.allSimplePaths(start, end, 'RELATES>', $max_depth) YIELD path
                WITH path, [node IN nodes(path) | node.node_id] AS node_ids
                WHERE ALL(r IN relationships(path) WHERE true {type_filter})
                RETURN node_ids
                """
            else:
                cypher = f"""
                MATCH p = (start:Node {{node_id: $start_id}})-[r:RELATES*..{max_depth}]->(end:Node {{node_id: $end_id}})
                WHERE ALL(rel IN relationships(p) WHERE true {type_filter})
                RETURN [node IN nodes(p) | node.node_id] AS node_ids
                """

            params = {
                "start_id": start_id,
                "end_id": end_id,
                "max_depth": max_depth,
                "edge_types": edge_types
            }

            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(cypher, params)
                    paths = [record["node_ids"] for record in result]
                    return paths
            except Exception as e:
                self.logger.error(f"Error finding paths: {e}")
                raise GraphEngineError(f"Failed to find paths: {e}")

    def _node_exists(self, node_id: str) -> bool:
        """Check if a node exists in the database."""
        # Check cache first
        if node_id in self._node_cache:
            return True

        # Query database
        cypher = "MATCH (n:Node {node_id: $node_id}) RETURN count(n) AS count"
        params = {"node_id": node_id}

        try:
            with self.driver.session(database=self.database) as session:
                result = session.run(cypher, params)
                return result.single()["count"] > 0
        except Exception as e:
            self.logger.error(f"Error checking if node exists: {e}")
            return False

    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """Execute a complex graph query."""
        with self.lock:
            result = {
                'nodes': [],
                'edges': [],
                'metadata': {
                    'query_time': time.time(),
                    'node_count': 0,
                    'edge_count': 0
                }
            }

            # Prepare node criteria
            node_where_clauses = []
            node_params = {}
            for i, (key, value) in enumerate(query.node_criteria.items()):
                param_name = f"n_param{i}"
                node_where_clauses.append(f"n.{key} = ${param_name}")
                node_params[param_name] = value

            # Prepare edge criteria
            edge_where_clauses = []
            edge_params = {}
            for i, (key, value) in enumerate(query.edge_criteria.items()):
                param_name = f"e_param{i}"
                edge_where_clauses.append(f"r.{key} = ${param_name}")
                edge_params[param_name] = value

            # Combine all parameters
            all_params = {**node_params, **edge_params}
            all_params.update({
                "start_nodes": query.start_nodes,
                "max_depth": query.max_depth,
                "min_activation": query.min_activation
            })

            # Build query
            node_where = " AND ".join(node_where_clauses) if node_where_clauses else "TRUE"
            edge_where = " AND ".join(edge_where_clauses) if edge_where_clauses else "TRUE"

            activation_filter = ""
            if not query.include_inactive and query.min_activation > 0:
                activation_filter = f"AND n.activation >= {query.min_activation}"

            # Execute query based on complexity
            try:
                with self.driver.session(database=self.database) as session:
                    # Strategy depends on whether start nodes are specified
                    if query.start_nodes and len(query.start_nodes) > 0:
                        # Exploration from start nodes
                        cypher = f"""
                        MATCH (start:Node)
                        WHERE start.node_id IN $start_nodes
                        OPTIONAL MATCH p = (start)-[r:RELATES*..{query.max_depth}]-(n:Node)
                        WHERE {node_where} AND ALL(rel IN relationships(p) WHERE {edge_where})
                        {activation_filter}
                        WITH DISTINCT n, r
                        RETURN collect(DISTINCT {{
                            id: n.node_id,
                            label: n.label,
                            properties: properties(n),
                            activation: n.activation,
                            importance: n.importance
                        }}) AS nodes,
                        collect(DISTINCT {{
                            id: r.edge_id,
                            source: startNode(r).node_id,
                            target: endNode(r).node_id,
                            type: r.type,
                            properties: properties(r)
                        }}) AS edges
                        """
                    else:
                        # Query based on criteria
                        cypher = f"""
                        MATCH (n:Node)
                        WHERE {node_where} {activation_filter}
                        WITH n
                        LIMIT {query.max_results}
                        OPTIONAL MATCH (n)-[r:RELATES]-(m:Node)
                        WHERE {edge_where}
                        RETURN collect(DISTINCT {{
                            id: n.node_id,
                            label: n.label,
                            properties: properties(n),
                            activation: n.activation,
                            importance: n.importance
                        }}) AS nodes,
                        collect(DISTINCT {{
                            id: r.edge_id,
                            source: startNode(r).node_id,
                            target: endNode(r).node_id,
                            type: r.type,
                            properties: properties(r)
                        }}) AS edges
                        """

                    result_record = session.run(cypher, all_params).single()

                    if result_record:
                        result['nodes'] = result_record["nodes"]
                        result['edges'] = result_record["edges"]

                    # Update metadata
                    result['metadata']['node_count'] = len(result['nodes'])
                    result['metadata']['edge_count'] = len(result['edges'])

                    return result
            except Exception as e:
                self.logger.error(f"Error executing query: {e}")
                raise GraphEngineError(f"Failed to execute query: {e}")

    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """Spread activation from source nodes through the graph."""
        with self.lock:
            # Verify nodes exist
            for node_id in source_nodes:
                if not self._node_exists(node_id):
                    raise NodeNotFoundError(f"Source node {node_id} not found")

            # Clear cache to ensure fresh activation values
            self._node_cache.clear()

            # First, set full activation for source nodes
            activate_cypher = """
            MATCH (n:Node)
            WHERE n.node_id IN $source_nodes
            SET n.activation = 1.0, n.last_accessed = $current_time
            """

            activation_results = {}

            try:
                with self.driver.session(database=self.database) as session:
                    # Set initial activation
                    session.run(activate_cypher, {
                        "source_nodes": source_nodes,
                        "current_time": time.time()
                    })

                    # Use different strategies depending on Neo4j version and extensions
                    if self.use_apoc:
                        # Use APOC for more efficient spreading
                        return self._spread_activation_apoc(source_nodes, decay_factor, max_depth)
                    else:
                        # Fallback to manual Cypher approach
                        return self._spread_activation_cypher(source_nodes, decay_factor, max_depth)
            except Exception as e:
                self.logger.error(f"Error spreading activation: {e}")
                raise GraphEngineError(f"Failed to spread activation: {e}")

    def _spread_activation_apoc(self, source_nodes: List[str], decay_factor: float,
                              max_depth: int) -> Dict[str, float]:
        """Spread activation using APOC procedures."""
        # This would use APOC's graph algorithms for more efficient propagation
        # Simplified version here
        cypher = """
        MATCH (start:Node)
        WHERE start.node_id IN $source_nodes
        CALL apoc.path.expandConfig(start, {
            relationshipFilter: 'RELATES>',
            minLevel: 1,
            maxLevel: $max_depth
        }) YIELD path
        WITH nodes(path) AS nodes, relationships(path) AS rels
        UNWIND nodes AS n
        WITH DISTINCT n,
             1.0 * (($decay_factor) ^ length(
                 shortestPath((start:Node)-[*]->(n))
             )) AS activation
        WHERE start.node_id IN $source_nodes
        SET n.activation = CASE
            WHEN n.activation < activation THEN activation
            ELSE n.activation
        END
        RETURN n.node_id AS node_id, n.activation AS activation
        """

        params = {
            "source_nodes": source_nodes,
            "max_depth": max_depth,
            "decay_factor": decay_factor
        }

        with self.driver.session(database=self.database) as session:
            result = session.run(cypher, params)
            activation_dict = {record["node_id"]: record["activation"] for record in result}
            return activation_dict

    def _spread_activation_cypher(self, source_nodes: List[str], decay_factor: float,
                               max_depth: int) -> Dict[str, float]:
        """Spread activation using pure Cypher."""
        # Iterative approach for Neo4j without APOC
        activation_dict = {node_id: 1.0 for node_id in source_nodes}

        with self.driver.session(database=self.database) as session:
            for depth in range(1, max_depth + 1):
                # Calculate decay for this level
                level_decay = decay_factor ** depth

                # Spread to neighbors at this depth
                cypher = """
                MATCH (start:Node)-[r:RELATES*EXACTLY {depth}]->(n:Node)
                WHERE start.node_id IN $source_nodes
                WITH DISTINCT n, $level_decay * r[-1].weight AS new_activation
                WHERE new_activation > n.activation
                SET n.activation = new_activation, n.last_accessed = $current_time
                RETURN n.node_id AS node_id, n.activation AS activation
                """

                params = {
                    "source_nodes": source_nodes,
                    "depth": depth,
                    "level_decay": level_decay,
                    "current_time": time.time()
                }

                result = session.run(cypher, params)
                for record in result:
                    activation_dict[record["node_id"]] = record["activation"]

            # Retrieve all activated nodes
            final_cypher = """
            MATCH (n:Node)
            WHERE n.activation > 0
            RETURN n.node_id AS node_id, n.activation AS activation
            """

            result = session.run(final_cypher)
            return {record["node_id"]: record["activation"] for record in result}

    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """Get the currently activated subgraph."""
        with self.lock:
            # Query for activated nodes
            nodes_cypher = """
            MATCH (n:Node)
            WHERE n.activation >= $threshold
            RETURN n.node_id AS node_id
            """

            # Query for edges between activated nodes
            edges_cypher = """
            MATCH (n:Node)-[r:RELATES]->(m:Node)
            WHERE n.activation >= $threshold AND m.activation >= $threshold
            RETURN r.edge_id AS edge_id
            """

            params = {"threshold": threshold}

            try:
                with self.driver.session(database=self.database) as session:
                    # Get activated nodes
                    nodes_result = session.run(nodes_cypher, params)
                    activated_nodes = [record["node_id"] for record in nodes_result]

                    # Get edges between activated nodes
                    edges_result = session.run(edges_cypher, params)
                    activated_edges = [record["edge_id"] for record in edges_result]

                    return activated_nodes, activated_edges
            except Exception as e:
                self.logger.error(f"Error getting activated subgraph: {e}")
                raise GraphEngineError(f"Failed to get activated subgraph: {e}")

    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """Detect frequent patterns in the graph."""
        with self.lock:
            # Pattern detection is complex - this is a simplified implementation
            # In a production system, more sophisticated algorithms would be used

            # Use APOC if available
            if self.use_apoc:
                return self._detect_patterns_apoc(min_frequency, max_size)
            else:
                return self._detect_patterns_basic(min_frequency, max_size)

    def _detect_patterns_apoc(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Detect patterns using APOC."""
        # This would use APOC's graph algorithms
        # Simplified placeholder implementation
        patterns = []

        cypher = """
        CALL apoc.periodic.iterate(
            'MATCH (n:Node) RETURN n',
            'MATCH (n)-[r:RELATES*..{max_size}]-(m)
             WITH nodes(r) AS nodes, relationships(r) AS rels
             WHERE size(nodes) <= {max_size}
             RETURN nodes, rels, count(*) AS freq
             ORDER BY freq DESC
             LIMIT 10',
            {batchSize:100, parallel:true}
        )
        YIELD batches, total
        RETURN batches, total
        """

        # Simplified response
        for i in range(3):  # Just return a few placeholder patterns
            pattern = GraphPattern(
                pattern_id=f"pattern_{i}",
                nodes=[f"node_{j}" for j in range(3)],
                edges=[f"edge_{j}" for j in range(2)],
                frequency=min_frequency + i,
                significance=0.5 + (i * 0.1)
            )
            patterns.append(pattern)

        return patterns

    def _detect_patterns_basic(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Basic pattern detection without APOC."""
        # Very simplified approach - in production, use a more sophisticated algorithm
        patterns = []

        cypher = """
        MATCH (n:Node)
        WITH n.label AS label, count(*) AS count
        WHERE count >= $min_frequency
        MATCH (n:Node {label: label})-[r:RELATES*..{max_size}]-(m:Node {label: label})
        WITH label, collect(n.node_id) AS nodes, collect(r.edge_id) AS edges
        RETURN label, nodes, edges, count
        ORDER BY count DESC
        LIMIT 5
        """

        params = {
            "min_frequency": min_frequency,
            "max_size": max_size
        }

        try:
            with self.driver.session(database=self.database) as session:
                result = session.run(cypher, params)

                for i, record in enumerate(result):
                    pattern = GraphPattern(
                        pattern_id=f"pattern_{i}_{record['label']}",
                        nodes=record["nodes"],
                        edges=record["edges"],
                        frequency=record["count"],
                        significance=0.5  # Placeholder
                    )
                    patterns.append(pattern)

                return patterns
        except Exception as e:
            self.logger.error(f"Error detecting patterns: {e}")
            return []  # Return empty list on error

    def clear(self) -> None:
        """Clear all nodes and edges from the graph."""
        with self.lock:
            # Reset caches
            self._node_cache.clear()
            self._edge_cache.clear()

            # Delete all data
            cypher = "MATCH (n) DETACH DELETE n"

            try:
                with self.driver.session(database=self.database) as session:
                    session.run(cypher)
                    self.logger.info("Graph cleared")
            except Exception as e:
                self.logger.error(f"Error clearing graph: {e}")
                raise GraphEngineError(f"Failed to clear graph: {e}")

    def serialize(self) -> Dict[str, Any]:
        """Serialize the graph to a dictionary."""
        with self.lock:
            data = {
                'nodes': [],
                'edges': [],
                'metadata': {
                    'timestamp': time.time(),
                    'uri': self.uri,
                    'database': self.database
                }
            }

            # Query all nodes
            nodes_cypher = """
            MATCH (n:Node)
            RETURN n.node_id AS id, n.label AS label, n.activation AS activation,
                   n.importance AS importance, n.creation_time AS creation_time,
                   n.last_accessed AS last_accessed, properties(n) AS properties
            """

            # Query all edges
            edges_cypher = """
            MATCH (s:Node)-[r:RELATES]->(t:Node)
            RETURN r.edge_id AS id, s.node_id AS source, t.node_id AS target,
                   r.type AS type, r.weight AS weight, r.bidirectional AS bidirectional,
                   r.creation_time AS creation_time, properties(r) AS properties
            """

            try:
                with self.driver.session(database=self.database) as session:
                    # Get nodes
                    nodes_result = session.run(nodes_cypher)
                    for record in nodes_result:
                        props = dict(record["properties"])
                        # Remove standard properties
                        for key in ["node_id", "label", "activation", "importance",
                                   "creation_time", "last_accessed"]:
                            if key in props:
                                del props[key]

                        node_data = {
                            'id': record["id"],
                            'label': record["label"],
                            'properties': props,
                            'activation': record["activation"],
                            'importance': record["importance"],
                            'creation_time': record["creation_time"],
                            'last_accessed': record["last_accessed"]
                        }
                        data['nodes'].append(node_data)

                    # Get edges
                    edges_result = session.run(edges_cypher)
                    for record in edges_result:
                        props = dict(record["properties"])
                        # Remove standard properties
                        for key in ["edge_id", "type", "weight", "bidirectional", "creation_time"]:
                            if key in props:
                                del props[key]

                        edge_data = {
                            'id': record["id"],
                            'source': record["source"],
                            'target': record["target"],
                            'type': record["type"],
                            'properties': props,
                            'weight': record["weight"],
                            'bidirectional': record["bidirectional"],
                            'creation_time': record["creation_time"]
                        }
                        data['edges'].append(edge_data)

                    # Update metadata
                    data['metadata']['node_count'] = len(data['nodes'])
                    data['metadata']['edge_count'] = len(data['edges'])

                    return data
            except Exception as e:
                self.logger.error(f"Error serializing graph: {e}")
                raise GraphEngineError(f"Failed to serialize graph: {e}")

    def deserialize(self, data: Dict[str, Any]) -> None:
        """Load the graph from a serialized representation."""
        with self.lock:
            # Clear existing data first
            self.clear()

            try:
                with self.driver.session(database=self.database) as session:
                    # Create nodes first
                    for node_data in data.get('nodes', []):
                        node_id = node_data['id']
                        label = node_data['label']

                        # Prepare properties
                        props = {
                            'node_id': node_id,
                            'label': label,
                            'activation': node_data.get('activation', 0.0),
                            'importance': node_data.get('importance', 0.5),
                            'creation_time': node_data.get('creation_time', time.time()),
                            'last_accessed': node_data.get('last_accessed', time.time())
                        }

                        # Add custom properties
                        props.update(node_data.get('properties', {}))

                        # Create node
                        create_node_cypher = """
                        CREATE (n:Node)
                        SET n = $props
                        """

                        session.run(create_node_cypher, {'props': props})

                    # Create edges
                    for edge_data in data.get('edges', []):
                        edge_id = edge_data['id']
                        source_id = edge_data['source']
                        target_id = edge_data['target']
                        edge_type = edge_data['type']

                        # Prepare properties
                        props = {
                            'edge_id': edge_id,
                            'type': edge_type,
                            'weight': edge_data.get('weight', 1.0),
                            'bidirectional': edge_data.get('bidirectional', False),
                            'creation_time': edge_data.get('creation_time', time.time())
                        }

                        # Add custom properties
                        props.update(edge_data.get('properties', {}))

                        # Create edge
                        create_edge_cypher = """
                        MATCH (s:Node {node_id: $source_id})
                        MATCH (t:Node {node_id: $target_id})
                        CREATE (s)-[r:RELATES]->(t)
                        SET r = $props
                        """

                        edge_params = {
                            'source_id': source_id,
                            'target_id': target_id,
                            'props': props
                        }

                        session.run(create_edge_cypher, edge_params)

                    self.logger.info(f"Deserialized graph with {len(data.get('nodes', []))} nodes and {len(data.get('edges', []))} edges")
            except Exception as e:
                self.logger.error(f"Error deserializing graph: {e}")
                raise GraphEngineError(f"Failed to deserialize graph: {e}")

    def close(self) -> None:
        """Close the graph engine and release any resources."""
        with self.lock:
            try:
                if self.driver:
                    self.driver.close()
                    self.logger.info("Neo4jGraphEngine closed")
            except Exception as e:
                self.logger.error(f"Error closing Neo4j driver: {e}")


# -----------------------------------------------------------------------------
# Factory function to create graph engines
# -----------------------------------------------------------------------------

def create_graph_engine(engine_type: str, **kwargs) -> AbstractGraphEngine:
    """
    Factory function to create the appropriate graph engine.

    Args:
        engine_type: Type of graph engine to create ("inmemory" or "neo4j")
        **kwargs: Additional arguments to pass to the graph engine constructor

    Returns:
        An instance of AbstractGraphEngine

    Raises:
        ValueError: If the engine type is not supported
    """
    if engine_type.lower() == "inmemory":
        return InMemoryGraphEngine(**kwargs)
    elif engine_type.lower() == "neo4j":
        return Neo4jGraphEngine(**kwargs)
    else:
        raise ValueError(f"Unsupported graph engine type: {engine_type}")

# Graph Engine 2

In [ ]:
"""
graph_engine.py - Advanced Graph Engine Module for GrandNexus

This module provides a comprehensive graph-based representation system for GrandNexus,
integrating seamlessly with memory and cognitive modules.

Features:
- Abstract base class with consistent interface
- Multiple graph engine implementations (InMemory, Neo4j)
- Cognitive graph operations (activation spreading, pattern mining)
- Advanced query capabilities
- Integration with semantic and episodic memory
- Support for both symbolic and subsymbolic representations
"""

import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json
import os
import random

# Optional dependencies
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False


# -----------------------------------------------------------------------------
# Core Data Structures
# -----------------------------------------------------------------------------

@dataclass
class GraphNode:
    """A node in the graph with cognitive properties"""
    node_id: str
    label: str
    properties: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[Any] = None
    activation: float = 0.0
    last_accessed: float = field(default_factory=time.time)
    importance: float = 0.5
    creation_time: float = field(default_factory=time.time)

    def access(self) -> None:
        """Update activation and last_accessed time when accessed"""
        current_time = time.time()
        self.last_accessed = current_time
        # Increase activation, with saturation
        self.activation = min(1.0, self.activation + 0.2)

    def decay_activation(self, decay_rate: float, current_time: Optional[float] = None) -> None:
        """Apply decay to node activation"""
        if current_time is None:
            current_time = time.time()

        time_since_access = current_time - self.last_accessed
        decay_factor = math.exp(-decay_rate * time_since_access)
        self.activation *= decay_factor


@dataclass
class GraphEdge:
    """A directed edge in the graph with cognitive properties"""
    edge_id: str
    source_id: str
    target_id: str
    type: str
    properties: Dict[str, Any] = field(default_factory=dict)
    weight: float = 1.0
    bidirectional: bool = False
    creation_time: float = field(default_factory=time.time)


@dataclass
class GraphPattern:
    """A discovered pattern in the graph"""
    pattern_id: str
    nodes: List[str]
    edges: List[str]
    frequency: int = 1
    significance: float = 0.5
    embedding: Optional[Any] = None


@dataclass
class GraphQuery:
    """A flexible query specification for graph operations"""
    node_criteria: Dict[str, Any] = field(default_factory=dict)
    edge_criteria: Dict[str, Any] = field(default_factory=dict)
    start_nodes: Optional[List[str]] = None
    max_depth: int = 3
    max_results: int = 100
    include_inactive: bool = False
    min_activation: float = 0.0
    sort_by: str = "activation"


# -----------------------------------------------------------------------------
# Exception Classes
# -----------------------------------------------------------------------------

class GraphEngineError(Exception):
    """Base class for graph-related errors"""
    pass

class NodeNotFoundError(GraphEngineError):
    """Raised when a node is not found"""
    pass

class EdgeNotFoundError(GraphEngineError):
    """Raised when an edge is not found"""
    pass

class InvalidQueryError(GraphEngineError):
    """Raised when a query is malformed"""
    pass


# -----------------------------------------------------------------------------
# Abstract Base Class
# -----------------------------------------------------------------------------

class AbstractGraphEngine(ABC):
    """Abstract base class defining the interface for graph engines"""

    @abstractmethod
    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """Add a node to the graph"""
        pass

    @abstractmethod
    def get_node(self, node_id: str) -> GraphNode:
        """Retrieve a node by ID"""
        pass

    @abstractmethod
    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """Update a node's properties"""
        pass

    @abstractmethod
    def delete_node(self, node_id: str) -> None:
        """Delete a node and its connected edges"""
        pass

    @abstractmethod
    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """Add an edge between two nodes"""
        pass

    @abstractmethod
    def get_edge(self, edge_id: str) -> GraphEdge:
        """Retrieve an edge by ID"""
        pass

    @abstractmethod
    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """Update an edge's properties"""
        pass

    @abstractmethod
    def delete_edge(self, edge_id: str) -> None:
        """Delete an edge"""
        pass

    @abstractmethod
    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """Get neighboring nodes connected to a node"""
        pass

    @abstractmethod
    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """Find nodes matching criteria"""
        pass

    @abstractmethod
    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """Find edges matching criteria"""
        pass

    @abstractmethod
    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """Find paths between two nodes"""
        pass

    @abstractmethod
    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """Execute a complex graph query"""
        pass

    @abstractmethod
    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """Spread activation from source nodes through the graph"""
        pass

    @abstractmethod
    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """Get the currently activated subgraph"""
        pass

    @abstractmethod
    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """Detect frequent patterns in the graph"""
        pass

    @abstractmethod
    def clear(self) -> None:
        """Clear all nodes and edges from the graph"""
        pass

    @abstractmethod
    def serialize(self) -> Dict[str, Any]:
        """Serialize the graph to a dictionary"""
        pass

    @abstractmethod
    def deserialize(self, data: Dict[str, Any]) -> None:
        """Load the graph from a serialized representation"""
        pass

    @abstractmethod
    def close(self) -> None:
        """Close the graph engine and release resources"""
        pass


# -----------------------------------------------------------------------------
# In-Memory Graph Engine Implementation
# -----------------------------------------------------------------------------

class InMemoryGraphEngine(AbstractGraphEngine):
    """In-memory implementation of the graph engine"""

    def __init__(self, use_networkx: bool = True, decay_rate: float = 0.1):
        """Initialize the in-memory graph engine"""
        self.logger = logging.getLogger("GrandNexus.Graph.InMemoryEngine")
        self.instance_id = str(uuid.uuid4())[:8]
        self.decay_rate = decay_rate

        # Data storage
        self.nodes = {}  # node_id -> GraphNode
        self.edges = {}  # edge_id -> GraphEdge
        self.outgoing_edges = defaultdict(list)  # source_id -> [edge_id]
        self.incoming_edges = defaultdict(list)  # target_id -> [edge_id]

        # Activation tracking
        self.activation_updated = {}  # node_id -> last_update_time

        # NetworkX graph (optional)
        self.use_networkx = use_networkx and HAS_NETWORKX
        self.nx_graph = None
        self.nx_graph_dirty = True  # Needs to be rebuilt

        if self.use_networkx:
            if not HAS_NETWORKX:
                self.logger.warning("NetworkX not available; falling back to basic implementation")
                self.use_networkx = False
            else:
                self.nx_graph = nx.DiGraph()

        self.logger.info(f"InMemoryGraphEngine initialized with ID={self.instance_id}")

    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """Add a node to the graph"""
        if node_id is None:
            node_id = str(uuid.uuid4())

        if node_id in self.nodes:
            self.logger.warning(f"Node {node_id} already exists, updating")
            self.update_node(node_id, properties or {})
            return node_id

        node = GraphNode(
            node_id=node_id,
            label=label,
            properties=properties or {},
            activation=0.1,  # Initial small activation
        )

        self.nodes[node_id] = node

        # Update NetworkX graph if used
        if self.use_networkx:
            self.nx_graph.add_node(
                node_id,
                label=label,
                properties=properties or {},
                activation=node.activation
            )
        else:
            self.nx_graph_dirty = True

        self.logger.debug(f"Added node {node_id} with label {label}")
        return node_id

    def get_node(self, node_id: str) -> GraphNode:
        """Retrieve a node by ID"""
        if node_id not in self.nodes:
            raise NodeNotFoundError(f"Node {node_id} not found")

        node = self.nodes[node_id]
        node.access()  # Update access time and activation
        return node

    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """Update a node's properties"""
        if node_id not in self.nodes:
            raise NodeNotFoundError(f"Node {node_id} not found")

        node = self.nodes[node_id]

        # Update existing properties
        node.properties.update(properties)

        # Update NetworkX graph if used
        if self.use_networkx:
            for key, value in properties.items():
                self.nx_graph.nodes[node_id][key] = value
        else:
            self.nx_graph_dirty = True

        self.logger.debug(f"Updated node {node_id}")

    def delete_node(self, node_id: str) -> None:
        """Delete a node and its connected edges"""
        if node_id not in self.nodes:
            raise NodeNotFoundError(f"Node {node_id} not found")

        # Remove all edges connected to this node
        incoming = self.incoming_edges[node_id].copy()
        outgoing = self.outgoing_edges[node_id].copy()

        for edge_id in incoming + outgoing:
            self._delete_edge_internal(edge_id)

        # Remove node
        del self.nodes[node_id]
        if node_id in self.incoming_edges:
            del self.incoming_edges[node_id]
        if node_id in self.outgoing_edges:
            del self.outgoing_edges[node_id]
        if node_id in self.activation_updated:
            del self.activation_updated[node_id]

        # Update NetworkX graph if used
        if self.use_networkx:
            if node_id in self.nx_graph:
                self.nx_graph.remove_node(node_id)
        else:
            self.nx_graph_dirty = True

        self.logger.debug(f"Deleted node {node_id}")

    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """Add an edge between two nodes"""
        # Verify nodes exist
        if source_id not in self.nodes:
            raise NodeNotFoundError(f"Source node {source_id} not found")
        if target_id not in self.nodes:
            raise NodeNotFoundError(f"Target node {target_id} not found")

        if edge_id is None:
            edge_id = str(uuid.uuid4())

        if edge_id in self.edges:
            self.logger.warning(f"Edge {edge_id} already exists, updating")
            self.update_edge(edge_id, properties or {})
            return edge_id

        edge = GraphEdge(
            edge_id=edge_id,
            source_id=source_id,
            target_id=target_id,
            type=edge_type,
            properties=properties or {},
            weight=weight,
            bidirectional=bidirectional
        )

        self.edges[edge_id] = edge
        self.outgoing_edges[source_id].append(edge_id)
        self.incoming_edges[target_id].append(edge_id)

        # Update NetworkX graph if used
        if self.use_networkx:
            self.nx_graph.add_edge(
                source_id,
                target_id,
                key=edge_id,
                type=edge_type,
                properties=properties or {},
                weight=weight,
                bidirectional=bidirectional
            )

            if bidirectional:
                self.nx_graph.add_edge(
                    target_id,
                    source_id,
                    key=f"{edge_id}_rev",
                    type=edge_type,
                    properties=properties or {},
                    weight=weight,
                    bidirectional=bidirectional
                )
        else:
            self.nx_graph_dirty = True

        self.logger.debug(f"Added edge {edge_id} from {source_id} to {target_id} of type {edge_type}")

        # If bidirectional, add reverse edge
        if bidirectional:
            # No need to add another edge in memory since the bidirectional flag
            # is already set, but make sure the edge is in both node's lists
            if edge_id not in self.outgoing_edges[target_id]:
                self.outgoing_edges[target_id].append(edge_id)
            if edge_id not in self.incoming_edges[source_id]:
                self.incoming_edges[source_id].append(edge_id)

        return edge_id

    def get_edge(self, edge_id: str) -> GraphEdge:
        """Retrieve an edge by ID"""
        if edge_id not in self.edges:
            raise EdgeNotFoundError(f"Edge {edge_id} not found")

        return self.edges[edge_id]

    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """Update an edge's properties"""
        if edge_id not in self.edges:
            raise EdgeNotFoundError(f"Edge {edge_id} not found")

        edge = self.edges[edge_id]

        # Update existing properties
        edge.properties.update(properties)

        # Update NetworkX graph if used
        if self.use_networkx:
            # Find the edge in the graph
            if edge.source_id in self.nx_graph and edge.target_id in self.nx_graph[edge.source_id]:
                for key, value in properties.items():
                    for k in self.nx_graph[edge.source_id][edge.target_id]:
                        if k == edge_id:
                            self.nx_graph[edge.source_id][edge.target_id][k][key] = value
        else:
            self.nx_graph_dirty = True

        self.logger.debug(f"Updated edge {edge_id}")

    def delete_edge(self, edge_id: str) -> None:
        """Delete an edge"""
        if edge_id not in self.edges:
            raise EdgeNotFoundError(f"Edge {edge_id} not found")

        self._delete_edge_internal(edge_id)
        self.logger.debug(f"Deleted edge {edge_id}")

    def _delete_edge_internal(self, edge_id: str) -> None:
        """Internal helper to delete an edge"""
        edge = self.edges[edge_id]

        # Remove from indices
        if edge_id in self.outgoing_edges[edge.source_id]:
            self.outgoing_edges[edge.source_id].remove(edge_id)
        if edge_id in self.incoming_edges[edge.target_id]:
            self.incoming_edges[edge.target_id].remove(edge_id)

        # Remove reverse connections if bidirectional
        if edge.bidirectional:
            if edge_id in self.outgoing_edges[edge.target_id]:
                self.outgoing_edges[edge.target_id].remove(edge_id)
            if edge_id in self.incoming_edges[edge.source_id]:
                self.incoming_edges[edge.source_id].remove(edge_id)

        # Remove from edges dict
        del self.edges[edge_id]

        # Update NetworkX graph if used
        if self.use_networkx:
            if edge.source_id in self.nx_graph and edge.target_id in self.nx_graph[edge.source_id]:
                # Find the edge with the matching key
                for k in list(self.nx_graph[edge.source_id][edge.target_id].keys()):
                    if k == edge_id:
                        self.nx_graph.remove_edge(edge.source_id, edge.target_id, key=k)

                # If bidirectional, remove reverse edge
                if edge.bidirectional:
                    for k in list(self.nx_graph[edge.target_id][edge.source_id].keys()):
                        if k == f"{edge_id}_rev":
                            self.nx_graph.remove_edge(edge.target_id, edge.source_id, key=k)
        else:
            self.nx_graph_dirty = True

    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """
        Get neighboring nodes connected to a node.

        Args:
            node_id: ID of the node
            edge_types: Optional filter for edge types
            direction: "outgoing", "incoming", or "both"

        Returns:
            List of (neighbor_id, edge_id, edge_type) tuples
        """
        if node_id not in self.nodes:
            raise NodeNotFoundError(f"Node {node_id} not found")

        neighbors = []

        # Get outgoing edges
        if direction in ["outgoing", "both"]:
            for edge_id in self.outgoing_edges[node_id]:
                edge = self.edges[edge_id]
                if edge_types is None or edge.type in edge_types:
                    neighbors.append((edge.target_id, edge_id, edge.type))

        # Get incoming edges
        if direction in ["incoming", "both"]:
            for edge_id in self.incoming_edges[node_id]:
                edge = self.edges[edge_id]
                if edge_types is None or edge.type in edge_types:
                    neighbors.append((edge.source_id, edge_id, edge.type))

        return neighbors

    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """Find nodes matching criteria"""
        matching_nodes = []

        for node_id, node in self.nodes.items():
            if self._match_criteria(node.properties, criteria):
                matching_nodes.append(node_id)

        return matching_nodes

    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """Find edges matching criteria"""
        matching_edges = []

        for edge_id, edge in self.edges.items():
            if self._match_criteria(edge.properties, criteria):
                matching_edges.append(edge_id)

        return matching_edges

    def _match_criteria(self, properties: Dict[str, Any], criteria: Dict[str, Any]) -> bool:
        """Check if properties match given criteria"""
        for key, value in criteria.items():
            # Handle special cases with operators
            if isinstance(value, dict) and len(value) == 1:
                op = list(value.keys())[0]
                op_value = value[op]

                if op == "$gt":
                    if key not in properties or properties[key] <= op_value:
                        return False
                elif op == "$lt":
                    if key not in properties or properties[key] >= op_value:
                        return False
                elif op == "$gte":
                    if key not in properties or properties[key] < op_value:
                        return False
                elif op == "$lte":
                    if key not in properties or properties[key] > op_value:
                        return False
                elif op == "$ne":
                    if key in properties and properties[key] == op_value:
                        return False
                elif op == "$in":
                    if key not in properties or properties[key] not in op_value:
                        return False
                elif op == "$contains":
                    if (key not in properties or
                        not isinstance(properties[key], (str, list, set, dict)) or
                        op_value not in properties[key]):
                        return False
                else:
                    # Unknown operator
                    return False
            else:
                # Direct equality match
                if key not in properties or properties[key] != value:
                    return False

        return True

    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """Find paths between two nodes"""
        if start_id not in self.nodes:
            raise NodeNotFoundError(f"Start node {start_id} not found")
        if end_id not in self.nodes:
            raise NodeNotFoundError(f"End node {end_id} not found")

        # Use NetworkX if available
        if self.use_networkx:
            return self._find_paths_networkx(start_id, end_id, max_depth, edge_types)

        # Fall back to BFS
        return self._find_paths_bfs(start_id, end_id, max_depth, edge_types)

    def _find_paths_networkx(self, start_id: str, end_id: str, max_depth: int,
                           edge_types: Optional[List[str]]) -> List[List[str]]:
        """Find paths using NetworkX algorithms"""
        # Make sure NetworkX graph is up to date
        if self.nx_graph_dirty:
            self._rebuild_nx_graph()

        # Create a subgraph with only the specified edge types if needed
        if edge_types:
            subgraph = nx.DiGraph()
            for u, v, data in self.nx_graph.edges(data=True):
                if data.get('type') in edge_types:
                    subgraph.add_edge(u, v)
        else:
            subgraph = self.nx_graph

        # Find all simple paths up to max_depth
        paths = []
        try:
            for path in nx.all_simple_paths(subgraph, start_id, end_id, cutoff=max_depth):
                paths.append(list(path))
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            # No path exists or node not found
            pass

        return paths

    def _find_paths_bfs(self, start_id: str, end_id: str, max_depth: int,
                      edge_types: Optional[List[str]]) -> List[List[str]]:
        """Find paths using BFS (fallback if NetworkX unavailable)"""
        paths = []
        queue = deque([(start_id, [start_id], 0)])  # (node, path_so_far, depth)
        visited = set()  # For cycle detection

        while queue:
            node_id, path, depth = queue.popleft()

            if depth > max_depth:
                continue

            if node_id == end_id:
                paths.append(path)
                continue

            # Get neighbors
            neighbors = self.get_node_neighbors(node_id, edge_types, "outgoing")

            for neighbor_id, _, _ in neighbors:
                if neighbor_id in path:  # Avoid cycles
                    continue

                new_path = path + [neighbor_id]
                new_depth = depth + 1

                # Store path state
                queue.append((neighbor_id, new_path, new_depth))

        return paths

    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """Execute a complex graph query"""
        results = {
            "nodes": {},
            "edges": {},
            "metadata": {
                "query_time": time.time(),
                "node_count": 0,
                "edge_count": 0,
                "start_nodes": query.start_nodes
            }
        }

        # Start with node criteria if no start nodes specified
        if not query.start_nodes:
            matching_nodes = self.query_nodes(query.node_criteria)
            start_nodes = matching_nodes
        else:
            start_nodes = query.start_nodes

        # Track processed nodes to avoid cycles
        processed_nodes = set()
        processed_edges = set()

        # Expand from each start node
        for node_id in start_nodes:
            if node_id not in self.nodes:
                continue

            self._expand_query_from_node(
                node_id, query, processed_nodes, processed_edges, results, 0
            )

        # Set metadata
        results["metadata"]["node_count"] = len(results["nodes"])
        results["metadata"]["edge_count"] = len(results["edges"])

        return results

    def _expand_query_from_node(self, node_id: str, query: GraphQuery,
                              processed_nodes: Set[str], processed_edges: Set[str],
                              result: Dict[str, Any], depth: int) -> None:
        """Helper to expand a query from a node recursively"""
        if depth > query.max_depth or node_id in processed_nodes:
            return

        # Check activation threshold
        node = self.nodes[node_id]
        if not query.include_inactive and node.activation < query.min_activation:
            return

        # Add node to result if it passes criteria
        if not query.node_criteria or self._match_criteria(node.properties, query.node_criteria):
            result["nodes"][node_id] = self._node_to_dict(node)
            processed_nodes.add(node_id)

            # Find neighbors according to edge criteria
            neighbors = self.get_node_neighbors(node_id)

            for neighbor_id, edge_id, edge_type in neighbors:
                edge = self.edges[edge_id]

                # Add edge to result if it passes criteria
                if edge_id not in processed_edges and (
                   not query.edge_criteria or
                   self._match_criteria(edge.properties, query.edge_criteria)):
                    result["edges"][edge_id] = self._edge_to_dict(edge)
                    processed_edges.add(edge_id)

                    # Recursively expand neighbor
                    self._expand_query_from_node(
                        neighbor_id, query, processed_nodes, processed_edges, result, depth + 1
                    )

    def _node_to_dict(self, node: GraphNode) -> Dict[str, Any]:
        """Convert node to dictionary representation"""
        return {
            "id": node.node_id,
            "label": node.label,
            "properties": node.properties,
            "activation": node.activation,
            "last_accessed": node.last_accessed,
            "importance": node.importance,
            "creation_time": node.creation_time
        }

    def _edge_to_dict(self, edge: GraphEdge) -> Dict[str, Any]:
        """Convert edge to dictionary representation"""
        return {
            "id": edge.edge_id,
            "source": edge.source_id,
            "target": edge.target_id,
            "type": edge.type,
            "properties": edge.properties,
            "weight": edge.weight,
            "bidirectional": edge.bidirectional,
            "creation_time": edge.creation_time
        }

    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """
        Spread activation from source nodes through the graph.

        Args:
            source_nodes: Starting nodes for activation
            decay_factor: How much activation decays per hop (0-1)
            max_depth: Maximum number of hops

        Returns:
            Dictionary mapping node IDs to activation values
        """
        # Initialize activation
        activation = defaultdict(float)

        # Set initial activation for source nodes
        for node_id in source_nodes:
            if node_id in self.nodes:
                self.nodes[node_id].access()  # Mark as accessed
                activation[node_id] = 1.0  # Full activation

        # Track visited nodes to avoid cycles
        visited = set(source_nodes)

        # For each depth level
        frontier = source_nodes
        for depth in range(1, max_depth + 1):
            new_frontier = []

            for node_id in frontier:
                if node_id not in self.nodes:
                    continue

                # Get outgoing neighbors
                neighbors = self.get_node_neighbors(node_id, direction="outgoing")

                # Calculate propagated activation (decayed by distance and edge weight)
                current_activation = activation[node_id]
                propagated_base = current_activation * (1 - decay_factor)

                for neighbor_id, edge_id, _ in neighbors:
                    if neighbor_id in visited:
                        continue

                    edge = self.edges[edge_id]

                    # Propagate activation, weighted by edge weight
                    propagated = propagated_base * edge.weight

                    # Add to neighbor's activation
                    activation[neighbor_id] += propagated

                    # Add to frontier for next iteration
                    new_frontier.append(neighbor_id)
                    visited.add(neighbor_id)

            frontier = new_frontier
            if not frontier:
                break  # No more nodes to process

        # Update node activation values in the graph
        current_time = time.time()
        for node_id, act_value in activation.items():
            if node_id in self.nodes:
                node = self.nodes[node_id]
                # Blend new activation with existing (favor higher values)
                node.activation = max(node.activation, act_value)
                # Update timestamp to reset decay
                if act_value > 0.1:  # Only if significant activation
                    node.last_accessed = current_time

        return dict(activation)

    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """
        Get the currently activated subgraph (nodes with activation >= threshold).

        Args:
            threshold: Minimum activation threshold

        Returns:
            Tuple of (node_ids, edge_ids) lists
        """
        # Decay all nodes' activation
        current_time = time.time()
        self._decay_all_activations(current_time)

        # Find activated nodes
        activated_nodes = []
        for node_id, node in self.nodes.items():
            if node.activation >= threshold:
                activated_nodes.append(node_id)

        # Find edges connecting activated nodes
        activated_edges = []
        for edge_id, edge in self.edges.items():
            if edge.source_id in activated_nodes and edge.target_id in activated_nodes:
                activated_edges.append(edge_id)

        return activated_nodes, activated_edges

    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """
        Detect frequent patterns in the graph.

        Args:
            min_frequency: Minimum number of occurrences for a pattern
            max_size: Maximum number of nodes in a pattern

        Returns:
            List of detected patterns
        """
        if self.use_networkx:
            return self._detect_patterns_networkx(min_frequency, max_size)
        else:
            return self._detect_patterns_basic(min_frequency, max_size)

    def _detect_patterns_networkx(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Detect patterns using NetworkX algorithms"""
        # Make sure NetworkX graph is up to date
        if self.nx_graph_dirty:
            self._rebuild_nx_graph()

        patterns = []

        # This is a simplified implementation
        # Real pattern mining would use more sophisticated algorithms

        # For small max_size, we can use a brute force approach to find common subgraphs
        for size in range(2, min(max_size + 1, 4)):  # Limit to size 3 for efficiency
            # Find all connected subgraphs of size 'size'
            for node in self.nx_graph.nodes():
                # Extract a subgraph around this node
                subgraph_nodes = [node]
                frontier = [node]
                visited = {node}

                # BFS to build a subgraph of up to 'size' nodes
                while len(subgraph_nodes) < size and frontier:
                    new_frontier = []
                    for n in frontier:
                        for neighbor in self.nx_graph.neighbors(n):
                            if neighbor not in visited and len(subgraph_nodes) < size:
                                visited.add(neighbor)
                                subgraph_nodes.append(neighbor)
                                new_frontier.append(neighbor)
                    frontier = new_frontier

                if len(subgraph_nodes) == size:
                    # Create the subgraph
                    subgraph = self.nx_graph.subgraph(subgraph_nodes)

                    # Check if this subgraph appears elsewhere
                    frequency = self._count_pattern_occurrences(subgraph)

                    if frequency >= min_frequency:
                        # Convert to EdgeGraph format
                        pattern_id = str(uuid.uuid4())

                        # Get edge IDs
                        edges = []
                        for u, v, data in subgraph.edges(data=True):
                            if 'key' in data:
                                edges.append(data['key'])

                        patterns.append(GraphPattern(
                            pattern_id=pattern_id,
                            nodes=list(subgraph_nodes),
                            edges=edges,
                            frequency=frequency,
                            significance=0.5 + (frequency - min_frequency) * 0.1  # Simple heuristic
                        ))

        return patterns

    def _count_pattern_occurrences(self, pattern_graph: 'nx.Graph') -> int:
        """Count occurrences of a pattern in the graph"""
        # This is a simplified approach - full subgraph isomorphism is NP-hard
        # We just check for subgraphs with the same node/edge counts and similar connections

        pattern_node_count = pattern_graph.number_of_nodes()
        pattern_edge_count = pattern_graph.number_of_edges()

        # Quick signature based on degree sequence (sorted)
        pattern_degree_seq = sorted([d for n, d in pattern_graph.degree()])

        # Count matches
        count = 0

        # Check each node as a potential starting point
        for start_node in self.nx_graph.nodes():
            # Get subgraph around this node
            subgraph_nodes = [start_node]
            frontier = [start_node]
            visited = {start_node}

            # BFS to build a subgraph of matching size
            while len(subgraph_nodes) < pattern_node_count and frontier:
                new_frontier = []
                for n in frontier:
                    for neighbor in self.nx_graph.neighbors(n):
                        if neighbor not in visited and len(subgraph_nodes) < pattern_node_count:
                            visited.add(neighbor)
                            subgraph_nodes.append(neighbor)
                            new_frontier.append(neighbor)
                frontier = new_frontier

            if len(subgraph_nodes) == pattern_node_count:
                subgraph = self.nx_graph.subgraph(subgraph_nodes)

                # Compare simple metrics
                if (subgraph.number_of_edges() == pattern_edge_count and
                    sorted([d for n, d in subgraph.degree()]) == pattern_degree_seq):
                    count += 1

        return count

    def _detect_patterns_basic(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Basic pattern detection fallback"""
        patterns = []

        # This is a very simplified approach
        # Real pattern mining would use more sophisticated algorithms

        # Look for common edge types connecting the same node types
        edge_type_counts = defaultdict(int)

        for edge in self.edges.values():
            source_node = self.nodes[edge.source_id]
            target_node = self.nodes[edge.target_id]

            # Create a pattern signature
            pattern_sig = f"{source_node.label}--{edge.type}-->{target_node.label}"
            edge_type_counts[pattern_sig] += 1

        # Check which patterns are frequent
        for pattern_sig, count in edge_type_counts.items():
            if count >= min_frequency:
                # Find instances of this pattern
                source_label, edge_type, target_label = pattern_sig.split("--")
                edge_type = edge_type.strip(">")

                # Get some example instances for this pattern
                instances = []
                for edge in self.edges.values():
                    source_node = self.nodes[edge.source_id]
                    target_node = self.nodes[edge.target_id]

                    if (source_node.label == source_label and
                        edge.type == edge_type and
                        target_node.label == target_label):
                        instances.append((edge.source_id, edge.target_id, edge.edge_id))

                        if len(instances) >= 5:  # Limit to 5 examples
                            break

                if instances:
                    pattern_id = str(uuid.uuid4())
                    pattern_nodes = []
                    pattern_edges = []

                    # Use the first instance as the pattern example
                    pattern_nodes.append(instances[0][0])
                    pattern_nodes.append(instances[0][1])
                    pattern_edges.append(instances[0][2])

                    patterns.append(GraphPattern(
                        pattern_id=pattern_id,
                        nodes=pattern_nodes,
                        edges=pattern_edges,
                        frequency=count,
                        significance=0.5 + (count - min_frequency) * 0.1  # Simple heuristic
                    ))

        return patterns

    def clear(self) -> None:
        """Clear all nodes and edges from the graph"""
        self.nodes.clear()
        self.edges.clear()
        self.outgoing_edges.clear()
        self.incoming_edges.clear()
        self.activation_updated.clear()

        if self.use_networkx:
            self.nx_graph.clear()
        else:
            self.nx_graph_dirty = True

        self.logger.info("Graph cleared")

    def serialize(self) -> Dict[str, Any]:
        """Serialize the graph to a dictionary"""
        # Decay activations before serialization
        current_time = time.time()
        self._decay_all_activations(current_time)

        nodes_data = {}
        for node_id, node in self.nodes.items():
            # Convert potentially non-serializable data
            properties = {}
            for k, v in node.properties.items():
                try:
                    json.dumps({k: v})
                    properties[k] = v
                except (TypeError, OverflowError):
                    # If not serializable, convert to string
                    properties[k] = str(v)

            nodes_data[node_id] = {
                "label": node.label,
                "properties": properties,
                "activation": node.activation,
                "last_accessed": node.last_accessed,
                "importance": node.importance,
                "creation_time": node.creation_time
            }

            # Handle embeddings specially (if present and numpy array)
            if node.embedding is not None:
                if HAS_NUMPY and isinstance(node.embedding, np.ndarray):
                    nodes_data[node_id]["embedding"] = node.embedding.tolist()
                else:
                    # Try to convert to list if possible
                    try:
                        nodes_data[node_id]["embedding"] = list(node.embedding)
                    except:
                        pass  # Skip if not convertible

        edges_data = {}
        for edge_id, edge in self.edges.items():
            # Convert potentially non-serializable data
            properties = {}
            for k, v in edge.properties.items():
                try:
                    json.dumps({k: v})
                    properties[k] = v
                except (TypeError, OverflowError):
                    # If not serializable, convert to string
                    properties[k] = str(v)

            edges_data[edge_id] = {
                "source_id": edge.source_id,
                "target_id": edge.target_id,
                "type": edge.type,
                "properties": properties,
                "weight": edge.weight,
                "bidirectional": edge.bidirectional,
                "creation_time": edge.creation_time
            }

        return {
            "nodes": nodes_data,
            "edges": edges_data,
            "metadata": {
                "timestamp": current_time,
                "instance_id": self.instance_id,
                "node_count": len(nodes_data),
                "edge_count": len(edges_data)
            }
        }

    def deserialize(self, data: Dict[str, Any]) -> None:
        """Load the graph from a serialized representation"""
        # Clear existing graph
        self.clear()

        # Load nodes
        for node_id, node_data in data.get("nodes", {}).items():
            # Handle embeddings if present
            embedding = node_data.get("embedding")
            if embedding is not None and HAS_NUMPY:
                embedding = np.array(embedding)

            node = GraphNode(
                node_id=node_id,
                label=node_data.get("label", "Node"),
                properties=node_data.get("properties", {}),
                embedding=embedding,
                activation=node_data.get("activation", 0.0),
                last_accessed=node_data.get("last_accessed", time.time()),
                importance=node_data.get("importance", 0.5),
                creation_time=node_data.get("creation_time", time.time())
            )

            self.nodes[node_id] = node

        # Load edges
        for edge_id, edge_data in data.get("edges", {}).items():
            edge = GraphEdge(
                edge_id=edge_id,
                source_id=edge_data.get("source_id"),
                target_id=edge_data.get("target_id"),
                type=edge_data.get("type", "RELATED_TO"),
                properties=edge_data.get("properties", {}),
                weight=edge_data.get("weight", 1.0),
                bidirectional=edge_data.get("bidirectional", False),
                creation_time=edge_data.get("creation_time", time.time())
            )

            self.edges[edge_id] = edge

            # Update indices
            self.outgoing_edges[edge.source_id].append(edge_id)
            self.incoming_edges[edge.target_id].append(edge_id)

            if edge.bidirectional:
                if edge_id not in self.outgoing_edges[edge.target_id]:
                    self.outgoing_edges[edge.target_id].append(edge_id)
                if edge_id not in self.incoming_edges[edge.source_id]:
                    self.incoming_edges[edge.source_id].append(edge_id)

        # Rebuild NetworkX graph if used
        if self.use_networkx:
            self._rebuild_nx_graph()
        else:
            self.nx_graph_dirty = True

        self.logger.info(f"Loaded graph with {len(self.nodes)} nodes and {len(self.edges)} edges")

    def close(self) -> None:
        """Close the graph engine and release resources"""
        # Nothing to do for in-memory engine
        pass

    def _decay_all_activations(self, current_time: float) -> None:
        """Decay all nodes' activation based on time since last access"""
        for node_id, node in self.nodes.items():
            node.decay_activation(self.decay_rate, current_time)

    def _rebuild_nx_graph(self) -> None:
        """Rebuild the NetworkX graph from scratch"""
        if not self.use_networkx:
            return

        self.nx_graph.clear()

        # Add nodes
        for node_id, node in self.nodes.items():
            self.nx_graph.add_node(
                node_id,
                label=node.label,
                properties=node.properties,
                activation=node.activation
            )

        # Add edges
        for edge_id, edge in self.edges.items():
            self.nx_graph.add_edge(
                edge.source_id,
                edge.target_id,
                key=edge_id,
                type=edge.type,
                properties=edge.properties,
                weight=edge.weight,
                bidirectional=edge.bidirectional
            )

            if edge.bidirectional:
                self.nx_graph.add_edge(
                    edge.target_id,
                    edge.source_id,
                    key=f"{edge_id}_rev",
                    type=edge.type,
                    properties=edge.properties,
                    weight=edge.weight,
                    bidirectional=edge.bidirectional
                )

        self.nx_graph_dirty = False


# -----------------------------------------------------------------------------
# Neo4j Graph Engine Implementation
# -----------------------------------------------------------------------------

class Neo4jGraphEngine(AbstractGraphEngine):
    """Neo4j-based implementation of the graph engine"""

    def __init__(self, uri: str, user: str, password: str, database: str = "neo4j",
                decay_rate: float = 0.1, use_apoc: bool = False):
        """Initialize the Neo4j graph engine"""
        self.logger = logging.getLogger("GrandNexus.Graph.Neo4jEngine")
        self.instance_id = str(uuid.uuid4())[:8]
        self.decay_rate = decay_rate

        # Neo4j connection
        self.uri = uri
        self.user = user
        self.password = password
        self.database = database
        self.use_apoc = use_apoc

        # Local cache for node/edge data
        self.node_cache = {}  # node_id -> GraphNode
        self.edge_cache = {}  # edge_id -> GraphEdge

        # Initialize driver
        if not HAS_NEO4J:
            raise ImportError("Neo4j Python driver is not installed")

        self.driver = GraphDatabase.driver(uri, auth=(user, password))

        # Test connection
        self._test_connection()

        # Set up schema
        self._setup_schema()

        self.logger.info(f"Neo4jGraphEngine initialized with ID={self.instance_id}")

    def _test_connection(self) -> None:
        """Test the Neo4j connection"""
        try:
            with self.driver.session(database=self.database) as session:
                result = session.run("RETURN 1 as test")
                record = result.single()
                if record["test"] != 1:
                    raise ConnectionError("Unexpected response from Neo4j")

            self.logger.info("Successfully connected to Neo4j")
        except Exception as e:
            self.logger.error(f"Failed to connect to Neo4j: {str(e)}")
            raise

    def _setup_schema(self) -> None:
        """Set up constraints and indices for optimized performance"""
        try:
            with self.driver.session(database=self.database) as session:
                # Create constraint on node IDs
                session.run("""
                    CREATE CONSTRAINT node_id_unique IF NOT EXISTS
                    FOR (n:Node)
                    REQUIRE n.id IS UNIQUE
                """)

                # Create constraint on edge IDs
                session.run("""
                    CREATE CONSTRAINT edge_id_unique IF NOT EXISTS
                    FOR ()-[r:RELATED_TO]->()
                    REQUIRE r.id IS UNIQUE
                """)

                # Create index on node labels
                session.run("""
                    CREATE INDEX node_label_index IF NOT EXISTS
                    FOR (n:Node)
                    ON (n.label)
                """)

                # Create index on edge types
                session.run("""
                    CREATE INDEX edge_type_index IF NOT EXISTS
                    FOR ()-[r:RELATED_TO]->()
                    ON (r.type)
                """)

                # Create index on activation
                session.run("""
                    CREATE INDEX node_activation_index IF NOT EXISTS
                    FOR (n:Node)
                    ON (n.activation)
                """)

            self.logger.info("Neo4j schema setup successful")
        except Exception as e:
            self.logger.warning(f"Failed to set up Neo4j schema: {str(e)}")

    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """Add a node to the graph"""
        if node_id is None:
            node_id = str(uuid.uuid4())

        if properties is None:
            properties = {}

        current_time = time.time()

        # Sanitize properties (ensure they're JSON-serializable)
        clean_props = {}
        for k, v in properties.items():
            try:
                json.dumps({k: v})
                clean_props[k] = v
            except (TypeError, OverflowError):
                # If not serializable, convert to string
                clean_props[k] = str(v)

        try:
            with self.driver.session(database=self.database) as session:
                # Check if node exists
                if self._node_exists(node_id):
                    # Update existing node
                    self.update_node(node_id, clean_props)
                    return node_id

                # Create node
                cypher = """
                CREATE (n:Node {
                    id: $id,
                    label: $label,
                    activation: $activation,
                    last_accessed: $last_accessed,
                    importance: $importance,
                    creation_time: $creation_time
                })
                SET n += $properties
                RETURN n
                """

                result = session.run(
                    cypher,
                    id=node_id,
                    label=label,
                    activation=0.1,  # Initial activation
                    last_accessed=current_time,
                    importance=0.5,  # Default importance
                    creation_time=current_time,
                    properties=clean_props
                )

                # Update cache
                self.node_cache[node_id] = GraphNode(
                    node_id=node_id,
                    label=label,
                    properties=clean_props,
                    activation=0.1,
                    last_accessed=current_time,
                    importance=0.5,
                    creation_time=current_time
                )

                self.logger.debug(f"Added node {node_id} with label {label}")
                return node_id

        except Exception as e:
            self.logger.error(f"Error adding node: {str(e)}")
            raise GraphEngineError(f"Failed to add node: {str(e)}")

    def get_node(self, node_id: str) -> GraphNode:
        """Retrieve a node by ID"""
        # Check cache first
        if node_id in self.node_cache:
            node = self.node_cache[node_id]
            node.access()  # Update access time and activation

            # Update activation in database (async)
            threading.Thread(
                target=self._update_node_activation,
                args=(node_id, node.activation, node.last_accessed)
            ).start()

            return node

        try:
            with self.driver.session(database=self.database) as session:
                cypher = """
                MATCH (n:Node {id: $id})
                SET n.activation = n.activation + 0.2,
                    n.last_accessed = $current_time
                RETURN n
                """

                current_time = time.time()

                result = session.run(cypher, id=node_id, current_time=current_time)
                record = result.single()

                if not record:
                    raise NodeNotFoundError(f"Node {node_id} not found")

                node_data = record["n"]

                # Extract node data
                node = GraphNode(
                    node_id=node_id,
                    label=node_data.get("label", "Node"),
                    properties={k: v for k, v in dict(node_data).items()
                               if k not in ["id", "label", "activation", "last_accessed",
                                           "importance", "creation_time"]},
                    activation=min(1.0, node_data.get("activation", 0.0) + 0.2),
                    last_accessed=current_time,
                    importance=node_data.get("importance", 0.5),
                    creation_time=node_data.get("creation_time", current_time)
                )

                # Update cache
                self.node_cache[node_id] = node

                return node

        except NodeNotFoundError:
            raise
        except Exception as e:
            self.logger.error(f"Error retrieving node: {str(e)}")
            raise GraphEngineError(f"Failed to retrieve node: {str(e)}")

    def _update_node_activation(self, node_id: str, activation: float, last_accessed: float) -> None:
        """Update a node's activation and last_accessed time"""
        try:
            with self.driver.session(database=self.database) as session:
                cypher = """
                MATCH (n:Node {id: $id})
                SET n.activation = $activation,
                    n.last_accessed = $last_accessed
                """

                session.run(
                    cypher,
                    id=node_id,
                    activation=min(1.0, activation),  # Cap at 1.0
                    last_accessed=last_accessed
                )

        except Exception as e:
            self.logger.warning(f"Failed to update node activation: {str(e)}")

    def update_node(self, node_id: str, properties: Dict[str, Any]) -> None:
        """Update a node's properties"""
        # Sanitize properties
        clean_props = {}
        for k, v in properties.items():
            try:
                json.dumps({k: v})
                clean_props[k] = v
            except (TypeError, OverflowError):
                clean_props[k] = str(v)

        try:
            with self.driver.session(database=self.database) as session:
                cypher = """
                MATCH (n:Node {id: $id})
                SET n += $properties
                RETURN n
                """

                result = session.run(cypher, id=node_id, properties=clean_props)
                record = result.single()

                if not record:
                    raise NodeNotFoundError(f"Node {node_id} not found")

                # Update cache if present
                if node_id in self.node_cache:
                    self.node_cache[node_id].properties.update(clean_props)

                self.logger.debug(f"Updated node {node_id}")

        except Exception as e:
            self.logger.error(f"Error updating node: {str(e)}")
            raise GraphEngineError(f"Failed to update node: {str(e)}")

    def delete_node(self, node_id: str) -> None:
        """Delete a node and its connected edges"""
        try:
            with self.driver.session(database=self.database) as session:
                # Delete node and its relationships
                cypher = """
                MATCH (n:Node {id: $id})
                OPTIONAL MATCH (n)-[r]-()
                DELETE r, n
                """

                result = session.run(cypher, id=node_id)

                # Update cache
                if node_id in self.node_cache:
                    del self.node_cache[node_id]

                # Also remove from edge cache any edges connected to this node
                edge_ids_to_remove = []
                for edge_id, edge in self.edge_cache.items():
                    if edge.source_id == node_id or edge.target_id == node_id:
                        edge_ids_to_remove.append(edge_id)

                for edge_id in edge_ids_to_remove:
                    if edge_id in self.edge_cache:
                        del self.edge_cache[edge_id]

                self.logger.debug(f"Deleted node {node_id}")

        except Exception as e:
            self.logger.error(f"Error deleting node: {str(e)}")
            raise GraphEngineError(f"Failed to delete node: {str(e)}")

    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """Add an edge between two nodes"""
        if edge_id is None:
            edge_id = str(uuid.uuid4())

        if properties is None:
            properties = {}

        # Sanitize properties
        clean_props = {}
        for k, v in properties.items():
            try:
                json.dumps({k: v})
                clean_props[k] = v
            except (TypeError, OverflowError):
                clean_props[k] = str(v)

        current_time = time.time()

        try:
            with self.driver.session(database=self.database) as session:
                # Check if nodes exist
                if not self._node_exists(source_id):
                    raise NodeNotFoundError(f"Source node {source_id} not found")
                if not self._node_exists(target_id):
                    raise NodeNotFoundError(f"Target node {target_id} not found")

                # Check if edge exists
                result = session.run("""
                    MATCH (s:Node {id: $source_id})-[r:RELATED_TO {id: $edge_id}]->(t:Node {id: $target_id})
                    RETURN r
                """, source_id=source_id, target_id=target_id, edge_id=edge_id)

                if result.single():
                    # Edge exists, update properties
                    self.update_edge(edge_id, clean_props)
                    return edge_id

                # Create edge
                direction = "<->" if bidirectional else "->"

                cypher = f"""
                MATCH (s:Node {{id: $source_id}}), (t:Node {{id: $target_id}})
                CREATE (s)-[r:RELATED_TO {{
                    id: $edge_id,
                    type: $edge_type,
                    weight: $weight,
                    bidirectional: $bidirectional,
                    creation_time: $creation_time
                }}]{direction}(t)
                SET r += $properties
                RETURN r
                """

                result = session.run(
                    cypher,
                    source_id=source_id,
                    target_id=target_id,
                    edge_id=edge_id,
                    edge_type=edge_type,
                    weight=weight,
                    bidirectional=bidirectional,
                    creation_time=current_time,
                    properties=clean_props
                )

                # Update cache
                edge = GraphEdge(
                    edge_id=edge_id,
                    source_id=source_id,
                    target_id=target_id,
                    type=edge_type,
                    properties=clean_props,
                    weight=weight,
                    bidirectional=bidirectional,
                    creation_time=current_time
                )

                self.edge_cache[edge_id] = edge

                self.logger.debug(f"Added edge {edge_id} from {source_id} to {target_id} of type {edge_type}")
                return edge_id

        except (NodeNotFoundError, EdgeNotFoundError):
            raise
        except Exception as e:
            self.logger.error(f"Error adding edge: {str(e)}")
            raise GraphEngineError(f"Failed to add edge: {str(e)}")

    def get_edge(self, edge_id: str) -> GraphEdge:
        """Retrieve an edge by ID"""
        # Check cache first
        if edge_id in self.edge_cache:
            return self.edge_cache[edge_id]

        try:
            with self.driver.session(database=self.database) as session:
                cypher = """
                MATCH ()-[r:RELATED_TO {id: $id}]->()
                RETURN r, startNode(r) as source, endNode(r) as target
                """

                result = session.run(cypher, id=edge_id)
                record = result.single()

                if not record:
                    raise EdgeNotFoundError(f"Edge {edge_id} not found")

                edge_data = record["r"]
                source_node = record["source"]
                target_node = record["target"]

                # Extract edge data
                edge = GraphEdge(
                    edge_id=edge_id,
                    source_id=source_node["id"],
                    target_id=target_node["id"],
                    type=edge_data.get("type", "RELATED_TO"),
                    properties={k: v for k, v in dict(edge_data).items()
                               if k not in ["id", "type", "weight", "bidirectional", "creation_time"]},
                    weight=edge_data.get("weight", 1.0),
                    bidirectional=edge_data.get("bidirectional", False),
                    creation_time=edge_data.get("creation_time", time.time())
                )

                # Update cache
                self.edge_cache[edge_id] = edge

                return edge

        except EdgeNotFoundError:
            raise
        except Exception as e:
            self.logger.error(f"Error retrieving edge: {str(e)}")
            raise GraphEngineError(f"Failed to retrieve edge: {str(e)}")

    def update_edge(self, edge_id: str, properties: Dict[str, Any]) -> None:
        """Update an edge's properties"""
        # Sanitize properties
        clean_props = {}
        for k, v in properties.items():
            try:
                json.dumps({k: v})
                clean_props[k] = v
            except (TypeError, OverflowError):
                clean_props[k] = str(v)

        try:
            with self.driver.session(database=self.database) as session:
                cypher = """
                MATCH ()-[r:RELATED_TO {id: $id}]->()
                SET r += $properties
                RETURN r
                """

                result = session.run(cypher, id=edge_id, properties=clean_props)
                record = result.single()

                if not record:
                    raise EdgeNotFoundError(f"Edge {edge_id} not found")

                # Update cache if present
                if edge_id in self.edge_cache:
                    self.edge_cache[edge_id].properties.update(clean_props)

                self.logger.debug(f"Updated edge {edge_id}")

        except Exception as e:
            self.logger.error(f"Error updating edge: {str(e)}")
            raise GraphEngineError(f"Failed to update edge: {str(e)}")

    def delete_edge(self, edge_id: str) -> None:
        """Delete an edge"""
        try:
            with self.driver.session(database=self.database) as session:
                # Delete edge
                cypher = """
                MATCH ()-[r:RELATED_TO {id: $id}]->()
                DELETE r
                """

                result = session.run(cypher, id=edge_id)

                # Update cache
                if edge_id in self.edge_cache:
                    del self.edge_cache[edge_id]

                self.logger.debug(f"Deleted edge {edge_id}")

        except Exception as e:
            self.logger.error(f"Error deleting edge: {str(e)}")
            raise GraphEngineError(f"Failed to delete edge: {str(e)}")

    def _node_exists(self, node_id: str) -> bool:
        """Check if a node exists in the database"""
        try:
            with self.driver.session(database=self.database) as session:
                result = session.run(
                    "MATCH (n:Node {id: $id}) RETURN count(n) as count",
                    id=node_id
                )
                record = result.single()
                return record and record["count"] > 0
        except Exception as e:
            self.logger.error(f"Error checking node existence: {str(e)}")
            return False

    def get_node_neighbors(self, node_id: str, edge_types: Optional[List[str]] = None,
                          direction: str = "outgoing") -> List[Tuple[str, str, str]]:
        """
        Get neighboring nodes connected to a node.

        Args:
            node_id: ID of the node
            edge_types: Optional filter for edge types
            direction: "outgoing", "incoming", or "both"

        Returns:
            List of (neighbor_id, edge_id, edge_type) tuples
        """
        if not self._node_exists(node_id):
            raise NodeNotFoundError(f"Node {node_id} not found")

        neighbors = []

        try:
            with self.driver.session(database=self.database) as session:
                # Construct Cypher based on direction and edge types
                edge_filter = ""
                if edge_types:
                    type_list = ", ".join([f"'{t}'" for t in edge_types])
                    edge_filter = f"WHERE r.type IN [{type_list}]"

                if direction == "outgoing":
                    cypher = f"""
                    MATCH (n:Node {{id: $id}})-[r:RELATED_TO]->(neighbor:Node)
                    {edge_filter}
                    RETURN neighbor.id as neighbor_id, r.id as edge_id, r.type as edge_type
                    """
                elif direction == "incoming":
                    cypher = f"""
                    MATCH (n:Node {{id: $id}})<-[r:RELATED_TO]-(neighbor:Node)
                    {edge_filter}
                    RETURN neighbor.id as neighbor_id, r.id as edge_id, r.type as edge_type
                    """
                else:  # both
                    cypher = f"""
                    MATCH (n:Node {{id: $id}})-[r:RELATED_TO]-(neighbor:Node)
                    {edge_filter}
                    RETURN neighbor.id as neighbor_id, r.id as edge_id, r.type as edge_type
                    """

                result = session.run(cypher, id=node_id)

                for record in result:
                    neighbors.append((
                        record["neighbor_id"],
                        record["edge_id"],
                        record["edge_type"]
                    ))

                return neighbors

        except NodeNotFoundError:
            raise
        except Exception as e:
            self.logger.error(f"Error retrieving neighbors: {str(e)}")
            raise GraphEngineError(f"Failed to retrieve neighbors: {str(e)}")

    def query_nodes(self, criteria: Dict[str, Any]) -> List[str]:
        """Find nodes matching criteria"""
        if not criteria:
            # If no criteria, return all nodes (up to a limit)
            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(
                        "MATCH (n:Node) RETURN n.id as id LIMIT 1000"
                    )
                    return [record["id"] for record in result]
            except Exception as e:
                self.logger.error(f"Error querying nodes: {str(e)}")
                return []

        # Convert criteria to Cypher WHERE clause
        where_clauses = []
        params = {}

        for key, value in criteria.items():
            # Handle special operators
            if isinstance(value, dict) and len(value) == 1:
                op = list(value.keys())[0]
                op_value = value[op]

                if op == "$gt":
                    where_clauses.append(f"n.{key} > ${key}")
                    params[key] = op_value
                elif op == "$lt":
                    where_clauses.append(f"n.{key} < ${key}")
                    params[key] = op_value
                elif op == "$gte":
                    where_clauses.append(f"n.{key} >= ${key}")
                    params[key] = op_value
                elif op == "$lte":
                    where_clauses.append(f"n.{key} <= ${key}")
                    params[key] = op_value
                elif op == "$ne":
                    where_clauses.append(f"n.{key} <> ${key}")
                    params[key] = op_value
                elif op == "$in":
                    where_clauses.append(f"n.{key} IN ${key}")
                    params[key] = op_value
                elif op == "$contains":
                    where_clauses.append(f"n.{key} CONTAINS ${key}")
                    params[key] = op_value
                else:
                    # Unknown operator, use equality
                    where_clauses.append(f"n.{key} = ${key}")
                    params[key] = value
            else:
                # Direct equality
                where_clauses.append(f"n.{key} = ${key}")
                params[key] = value

        where_clause = " AND ".join(where_clauses)

        try:
            with self.driver.session(database=self.database) as session:
                cypher = f"""
                MATCH (n:Node)
                WHERE {where_clause}
                RETURN n.id as id
                """

                result = session.run(cypher, **params)
                return [record["id"] for record in result]

        except Exception as e:
            self.logger.error(f"Error querying nodes: {str(e)}")
            return []

    def query_edges(self, criteria: Dict[str, Any]) -> List[str]:
        """Find edges matching criteria"""
        if not criteria:
            # If no criteria, return all edges (up to a limit)
            try:
                with self.driver.session(database=self.database) as session:
                    result = session.run(
                        "MATCH ()-[r:RELATED_TO]->() RETURN r.id as id LIMIT 1000"
                    )
                    return [record["id"] for record in result]
            except Exception as e:
                self.logger.error(f"Error querying edges: {str(e)}")
                return []

        # Convert criteria to Cypher WHERE clause
        where_clauses = []
        params = {}

        for key, value in criteria.items():
            # Handle special operators
            if isinstance(value, dict) and len(value) == 1:
                op = list(value.keys())[0]
                op_value = value[op]

                if op == "$gt":
                    where_clauses.append(f"r.{key} > ${key}")
                    params[key] = op_value
                elif op == "$lt":
                    where_clauses.append(f"r.{key} < ${key}")
                    params[key] = op_value
                elif op == "$gte":
                    where_clauses.append(f"r.{key} >= ${key}")
                    params[key] = op_value
                elif op == "$lte":
                    where_clauses.append(f"r.{key} <= ${key}")
                    params[key] = op_value
                elif op == "$ne":
                    where_clauses.append(f"r.{key} <> ${key}")
                    params[key] = op_value
                elif op == "$in":
                    where_clauses.append(f"r.{key} IN ${key}")
                    params[key] = op_value
                elif op == "$contains":
                    where_clauses.append(f"r.{key} CONTAINS ${key}")
                    params[key] = op_value
                else:
                    # Unknown operator, use equality
                    where_clauses.append(f"r.{key} = ${key}")
                    params[key] = value
            else:
                # Direct equality
                where_clauses.append(f"r.{key} = ${key}")
                params[key] = value

        where_clause = " AND ".join(where_clauses)

        try:
            with self.driver.session(database=self.database) as session:
                cypher = f"""
                MATCH ()-[r:RELATED_TO]->()
                WHERE {where_clause}
                RETURN r.id as id
                """

                result = session.run(cypher, **params)
                return [record["id"] for record in result]

        except Exception as e:
            self.logger.error(f"Error querying edges: {str(e)}")
            return []

    def find_paths(self, start_id: str, end_id: str, max_depth: int = 3,
                  edge_types: Optional[List[str]] = None) -> List[List[str]]:
        """Find paths between two nodes"""
        if not self._node_exists(start_id):
            raise NodeNotFoundError(f"Start node {start_id} not found")
        if not self._node_exists(end_id):
            raise NodeNotFoundError(f"End node {end_id} not found")

        # Construct edge type filter if provided
        edge_filter = ""
        if edge_types:
            type_list = ", ".join([f"'{t}'" for t in edge_types])
            edge_filter = f"WHERE ALL(r IN relationships(p) WHERE r.type IN [{type_list}])"

        try:
            with self.driver.session(database=self.database) as session:
                cypher = f"""
                MATCH (start:Node {{id: $start_id}}), (end:Node {{id: $end_id}}),
                      p = shortestPath((start)-[:RELATED_TO*..{max_depth}]->(end))
                {edge_filter}
                RETURN [node in nodes(p) | node.id] as path
                """

                result = session.run(cypher, start_id=start_id, end_id=end_id)
                paths = [record["path"] for record in result]

                # If no paths found with shortestPath, try to find all paths (up to a limit)
                if not paths:
                    cypher = f"""
                    MATCH (start:Node {{id: $start_id}}), (end:Node {{id: $end_id}}),
                          p = (start)-[:RELATED_TO*..{max_depth}]->(end)
                    {edge_filter}
                    RETURN [node in nodes(p) | node.id] as path
                    LIMIT 10
                    """

                    result = session.run(cypher, start_id=start_id, end_id=end_id)
                    paths = [record["path"] for record in result]

                return paths

        except (NodeNotFoundError, EdgeNotFoundError):
            raise
        except Exception as e:
            self.logger.error(f"Error finding paths: {str(e)}")
            raise GraphEngineError(f"Failed to find paths: {str(e)}")

    def execute_query(self, query: GraphQuery) -> Dict[str, Any]:
        """Execute a complex graph query"""
        results = {
            "nodes": {},
            "edges": {},
            "metadata": {
                "query_time": time.time(),
                "node_count": 0,
                "edge_count": 0,
                "start_nodes": query.start_nodes
            }
        }

        # Construct Cypher query based on GraphQuery object

        # Start with node criteria if no start nodes specified
        if not query.start_nodes:
            # Convert node criteria to Cypher WHERE clause
            node_where_clauses = []
            node_params = {}

            for key, value in query.node_criteria.items():
                # Handle special operators
                if isinstance(value, dict) and len(value) == 1:
                    op = list(value.keys())[0]
                    op_value = value[op]

                    if op == "$gt":
                        node_where_clauses.append(f"n.{key} > ${key}")
                        node_params[key] = op_value
                    elif op == "$lt":
                        node_where_clauses.append(f"n.{key} < ${key}")
                        node_params[key] = op_value
                    elif op == "$gte":
                        node_where_clauses.append(f"n.{key} >= ${key}")
                        node_params[key] = op_value
                    elif op == "$lte":
                        node_where_clauses.append(f"n.{key} <= ${key}")
                        node_params[key] = op_value
                    elif op == "$ne":
                        node_where_clauses.append(f"n.{key} <> ${key}")
                        node_params[key] = op_value
                    elif op == "$in":
                        node_where_clauses.append(f"n.{key} IN ${key}")
                        node_params[key] = op_value
                    elif op == "$contains":
                        node_where_clauses.append(f"n.{key} CONTAINS ${key}")
                        node_params[key] = op_value
                    else:
                        # Unknown operator, use equality
                        node_where_clauses.append(f"n.{key} = ${key}")
                        node_params[key] = value
                else:
                    # Direct equality
                    node_where_clauses.append(f"n.{key} = ${key}")
                    node_params[key] = value

            node_where_clause = " AND ".join(node_where_clauses) if node_where_clauses else "1=1"

            # Execute query to find matching start nodes
            try:
                with self.driver.session(database=self.database) as session:
                    cypher = f"""
                    MATCH (n:Node)
                    WHERE {node_where_clause}
                    RETURN n.id as id
                    """

                    result = session.run(cypher, **node_params)
                    start_nodes = [record["id"] for record in result]
            except Exception as e:
                self.logger.error(f"Error finding start nodes: {str(e)}")
                return results
        else:
            start_nodes = query.start_nodes

        # No start nodes found
        if not start_nodes:
            return results

        # Construct edge criteria WHERE clause
        edge_where_clauses = []
        edge_params = {}

        for key, value in query.edge_criteria.items():
            # Handle special operators (similar to node criteria)
            if isinstance(value, dict) and len(value) == 1:
                op = list(value.keys())[0]
                op_value = value[op]

                # Similar to node criteria processing
                param_key = f"edge_{key}"  # Avoid parameter name conflicts

                if op == "$gt":
                    edge_where_clauses.append(f"r.{key} > ${param_key}")
                    edge_params[param_key] = op_value
                elif op == "$lt":
                    edge_where_clauses.append(f"r.{key} < ${param_key}")
                    edge_params[param_key] = op_value
                # ... other operators similarly
                else:
                    # Default to equality
                    edge_where_clauses.append(f"r.{key} = ${param_key}")
                    edge_params[param_key] = value
            else:
                # Direct equality
                param_key = f"edge_{key}"
                edge_where_clauses.append(f"r.{key} = ${param_key}")
                edge_params[param_key] = value

        edge_where_clause = " AND ".join(edge_where_clauses) if edge_where_clauses else "1=1"

        # Add activation filter if needed
        activation_filter = ""
        if not query.include_inactive and query.min_activation > 0:
            activation_filter = f"AND n2.activation >= {query.min_activation}"

        # Execute the main query for each start node
        for start_node_id in start_nodes:
            try:
                with self.driver.session(database=self.database) as session:
                    # Construct a variable-length path query with filters
                    cypher = f"""
                    MATCH (n:Node {{id: $start_id}})-[r:RELATED_TO*1..{query.max_depth}]->(n2:Node)
                    WHERE {edge_where_clause} {activation_filter}
                    WITH n, r, n2
                    RETURN n.id as start_id, n2.id as node_id,
                           [rel in r | rel.id] as edge_ids,
                           n2 as node_data
                    LIMIT {query.max_results}
                    """

                    params = {"start_id": start_node_id, **edge_params}

                    result = session.run(cypher, **params)

                    for record in result:
                        node_id = record["node_id"]
                        node_data = record["node_data"]
                        edge_ids = record["edge_ids"]

                        # Add node to results if not already present
                        if node_id not in results["nodes"]:
                            # Convert node data to dictionary
                            node_dict = dict(node_data)

                            # Format for consistency with in-memory version
                            node_obj = {
                                "id": node_id,
                                "label": node_dict.get("label", "Node"),
                                "properties": {k: v for k, v in node_dict.items()
                                            if k not in ["id", "label", "activation",
                                                        "last_accessed", "importance",
                                                        "creation_time"]},
                                "activation": node_dict.get("activation", 0.0),
                                "last_accessed": node_dict.get("last_accessed", time.time()),
                                "importance": node_dict.get("importance", 0.5),
                                "creation_time": node_dict.get("creation_time", time.time())
                            }

                            results["nodes"][node_id] = node_obj

                        # Add edges to results
                        for edge_id in edge_ids:
                            if edge_id not in results["edges"]:
                                # Fetch edge details
                                edge = self.get_edge(edge_id)

                                # Format for consistency
                                edge_obj = {
                                    "id": edge.edge_id,
                                    "source": edge.source_id,
                                    "target": edge.target_id,
                                    "type": edge.type,
                                    "properties": edge.properties,
                                    "weight": edge.weight,
                                    "bidirectional": edge.bidirectional,
                                    "creation_time": edge.creation_time
                                }

                                results["edges"][edge_id] = edge_obj

            except Exception as e:
                self.logger.error(f"Error executing graph query for start node {start_node_id}: {str(e)}")
                # Continue with other start nodes

        # Update metadata
        results["metadata"]["node_count"] = len(results["nodes"])
        results["metadata"]["edge_count"] = len(results["edges"])

        return results

    def spread_activation(self, source_nodes: List[str], decay_factor: float = 0.5,
                         max_depth: int = 3) -> Dict[str, float]:
        """
        Spread activation from source nodes through the graph.

        Args:
            source_nodes: Starting nodes for activation
            decay_factor: How much activation decays per hop (0-1)
            max_depth: Maximum number of hops

        Returns:
            Dictionary mapping node IDs to activation values
        """
        # Choose appropriate implementation based on configuration
        if self.use_apoc:
            return self._spread_activation_apoc(source_nodes, decay_factor, max_depth)
        else:
            return self._spread_activation_cypher(source_nodes, decay_factor, max_depth)

    def _spread_activation_apoc(self, source_nodes: List[str], decay_factor: float,
                              max_depth: int) -> Dict[str, float]:
        """Spread activation using APOC procedures"""
        # Requires APOC installed on Neo4j

        # Initialize activation map
        activation = {}
        for node_id in source_nodes:
            activation[node_id] = 1.0  # Full activation

        try:
            with self.driver.session(database=self.database) as session:
                # Update activation for source nodes
                for node_id in source_nodes:
                    session.run("""
                        MATCH (n:Node {id: $id})
                        SET n.activation = 1.0,
                            n.last_accessed = $current_time
                    """, id=node_id, current_time=time.time())

                # Use APOC's path expander to spread activation
                cypher = """
                UNWIND $source_nodes AS source_id
                MATCH (source:Node {id: source_id})
                CALL apoc.path.expandConfig(source, {
                    relationshipFilter: 'RELATED_TO>',
                    minLevel: 1,
                    maxLevel: $max_depth
                }) YIELD path
                WITH path, source,
                     [n IN nodes(path) | n.id] AS node_ids,
                     [r IN relationships(path) | r.weight] AS edge_weights,
                     length(path) AS depth
                UNWIND range(0, size(node_ids) - 1) AS idx
                WITH node_ids[idx] AS node_id,
                     CASE WHEN idx = 0 THEN 1.0
                          ELSE REDUCE(weight = 1.0, w IN edge_weights[0..idx] |
                               weight * w * (1 - $decay_factor)) END AS act
                MERGE (target:Node {id: node_id})
                ON MATCH SET target.activation =
                    CASE WHEN target.activation < act
                    THEN act
                    ELSE target.activation END,
                    target.last_accessed = CASE WHEN act > 0.1
                                          THEN $current_time
                                          ELSE target.last_accessed END
                RETURN node_id, act as activation
                """

                result = session.run(
                    cypher,
                    source_nodes=source_nodes,
                    decay_factor=decay_factor,
                    max_depth=max_depth,
                    current_time=time.time()
                )

                # Collect results
                for record in result:
                    node_id = record["node_id"]
                    act = record["activation"]

                    # Update activation map (keep highest value)
                    activation[node_id] = max(activation.get(node_id, 0.0), act)

                return activation

        except Exception as e:
            self.logger.error(f"Error spreading activation (APOC): {str(e)}")
            return activation

    def _spread_activation_cypher(self, source_nodes: List[str], decay_factor: float,
                               max_depth: int) -> Dict[str, float]:
        """Spread activation using pure Cypher"""
        # Initialize activation map
        activation = {}
        for node_id in source_nodes:
            activation[node_id] = 1.0  # Full activation

        try:
            with self.driver.session(database=self.database) as session:
                # Update activation for source nodes
                current_time = time.time()
                for node_id in source_nodes:
                    session.run("""
                        MATCH (n:Node {id: $id})
                        SET n.activation = 1.0,
                            n.last_accessed = $current_time
                    """, id=node_id, current_time=current_time)

                # Spread activation level by level
                for depth in range(1, max_depth + 1):
                    # Propagation factor for this depth
                    prop_factor = (1 - decay_factor) ** depth

                    # Find nodes at exactly this depth from any source
                    cypher = """
                    UNWIND $source_nodes AS source_id
                    MATCH (source:Node {id: source_id}), (target:Node),
                          path = shortestPath((source)-[:RELATED_TO*..{depth}]->(target))
                    WHERE length(path) = $depth
                    WITH target.id AS node_id,
                         REDUCE(weight = 1.0, r IN relationships(path) |
                             weight * r.weight) AS path_weight
                    WITH node_id, MAX($prop_factor * path_weight) AS activation
                    MATCH (n:Node {id: node_id})
                    SET n.activation = CASE
                                        WHEN n.activation < activation
                                        THEN activation
                                        ELSE n.activation
                                      END,
                        n.last_accessed = CASE
                                            WHEN activation > 0.1
                                            THEN $current_time
                                            ELSE n.last_accessed
                                          END
                    RETURN node_id, activation
                    """

                    result = session.run(
                        cypher,
                        source_nodes=source_nodes,
                        depth=depth,
                        prop_factor=prop_factor,
                        current_time=current_time
                    )

                    # Update activation map
                    for record in result:
                        node_id = record["node_id"]
                        act = record["activation"]

                        # Keep highest activation
                        activation[node_id] = max(activation.get(node_id, 0.0), act)

                return activation

        except Exception as e:
            self.logger.error(f"Error spreading activation (Cypher): {str(e)}")
            return activation

    def get_activated_subgraph(self, threshold: float = 0.2) -> Tuple[List[str], List[str]]:
        """
        Get the currently activated subgraph (nodes with activation >= threshold).

        Args:
            threshold: Minimum activation threshold

        Returns:
            Tuple of (node_ids, edge_ids) lists
        """
        activated_nodes = []
        activated_edges = []

        try:
            with self.driver.session(database=self.database) as session:
                # Find activated nodes
                cypher_nodes = """
                MATCH (n:Node)
                WHERE n.activation >= $threshold
                RETURN n.id as id
                """

                result = session.run(cypher_nodes, threshold=threshold)
                activated_nodes = [record["id"] for record in result]

                # Find edges connecting activated nodes
                if activated_nodes:
                    cypher_edges = """
                    MATCH (s:Node)-[r:RELATED_TO]->(t:Node)
                    WHERE s.id IN $activated_nodes AND t.id IN $activated_nodes
                    RETURN r.id as id
                    """

                    result = session.run(cypher_edges, activated_nodes=activated_nodes)
                    activated_edges = [record["id"] for record in result]

                return activated_nodes, activated_edges

        except Exception as e:
            self.logger.error(f"Error getting activated subgraph: {str(e)}")
            return [], []

    def detect_patterns(self, min_frequency: int = 2, max_size: int = 5) -> List[GraphPattern]:
        """
        Detect frequent patterns in the graph.

        Args:
            min_frequency: Minimum number of occurrences for a pattern
            max_size: Maximum number of nodes in a pattern

        Returns:
            List of detected patterns
        """
        if self.use_apoc:
            return self._detect_patterns_apoc(min_frequency, max_size)
        else:
            return self._detect_patterns_basic(min_frequency, max_size)

    def _detect_patterns_apoc(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Detect patterns using APOC"""
        patterns = []

        try:
            with self.driver.session(database=self.database) as session:
                # Use APOC to find frequent subgraphs
                # This is a simplified version - real implementation would use
                # more sophisticated algorithms

                # Find frequent node label combinations
                cypher = """
                CALL apoc.periodic.iterate(
                    'MATCH (n:Node) RETURN n',
                    'WITH n
                     MATCH (n)-[r:RELATED_TO]->(m:Node)
                     WITH n.label as label1, r.type as rel_type, m.label as label2
                     RETURN label1, rel_type, label2, count(*) as freq
                     ORDER BY freq DESC',
                    {batchSize: 1000}
                )
                YIELD batches, total, timeTaken, committedOperations
                RETURN batches, total
                """

                # For now, use a simpler approach
                # Find frequent edge patterns
                cypher = """
                MATCH (n:Node)-[r:RELATED_TO]->(m:Node)
                WITH n.label as label1, r.type as rel_type, m.label as label2,
                     collect({source: n.id, target: m.id, edge: r.id}) as instances
                WHERE size(instances) >= $min_frequency
                RETURN label1, rel_type, label2, size(instances) as freq, instances[0..5] as sample
                ORDER BY freq DESC
                LIMIT 50
                """

                result = session.run(cypher, min_frequency=min_frequency)

                for record in result:
                    if record["sample"]:
                        pattern_id = str(uuid.uuid4())
                        sample = record["sample"][0]

                        patterns.append(GraphPattern(
                            pattern_id=pattern_id,
                            nodes=[sample["source"], sample["target"]],
                            edges=[sample["edge"]],
                            frequency=record["freq"],
                            significance=0.5 + (record["freq"] - min_frequency) * 0.1
                        ))

                return patterns

        except Exception as e:
            self.logger.error(f"Error detecting patterns (APOC): {str(e)}")
            return []

    def _detect_patterns_basic(self, min_frequency: int, max_size: int) -> List[GraphPattern]:
        """Basic pattern detection without APOC"""
        patterns = []

        try:
            with self.driver.session(database=self.database) as session:
                # Find frequent edge type patterns
                cypher = """
                MATCH (n:Node)-[r:RELATED_TO]->(m:Node)
                WITH n.label as source_label, r.type as edge_type, m.label as target_label,
                     count(*) as freq,
                     collect({source: n.id, target: m.id, edge: r.id})[0..5] as samples
                WHERE freq >= $min_frequency
                RETURN source_label, edge_type, target_label, freq, samples
                ORDER BY freq DESC
                LIMIT 50
                """

                result = session.run(cypher, min_frequency=min_frequency)

                for record in result:
                    if record["samples"]:
                        pattern_id = str(uuid.uuid4())
                        first_sample = record["samples"][0]

                        patterns.append(GraphPattern(
                            pattern_id=pattern_id,
                            nodes=[first_sample["source"], first_sample["target"]],
                            edges=[first_sample["edge"]],
                            frequency=record["freq"],
                            significance=0.5 + (record["freq"] - min_frequency) * 0.1
                        ))

                # Also look for star patterns (nodes with many connections)
                if max_size >= 3:
                    cypher = """
                    MATCH (center:Node)-[r:RELATED_TO]-(neighbor:Node)
                    WITH center, count(DISTINCT neighbor) as degree
                    WHERE degree >= $min_frequency
                    WITH center, degree
                    ORDER BY degree DESC
                    LIMIT 10
                    MATCH (center)-[r:RELATED_TO]-(neighbor:Node)
                    WITH center, degree, collect(neighbor.id)[0..5] as neighbors,
                         collect(r.id)[0..5] as edges
                    RETURN center.id as center_id, degree, neighbors, edges
                    """

                    result = session.run(cypher, min_frequency=min_frequency)

                    for record in result:
                        pattern_id = str(uuid.uuid4())
                        nodes = [record["center_id"]] + record["neighbors"]

                        patterns.append(GraphPattern(
                            pattern_id=pattern_id,
                            nodes=nodes[:max_size],
                            edges=record["edges"][:max_size-1],
                            frequency=record["degree"],
                            significance=0.5 + (record["degree"] - min_frequency) * 0.05
                        ))

                return patterns

        except Exception as e:
            self.logger.error(f"Error detecting patterns (basic): {str(e)}")
            return []

    def clear(self) -> None:
        """Clear all nodes and edges from the graph"""
        try:
            with self.driver.session(database=self.database) as session:
                # Delete all nodes and relationships
                session.run("MATCH (n:Node) DETACH DELETE n")

                # Clear caches
                self.node_cache.clear()
                self.edge_cache.clear()

                self.logger.info("Graph cleared")

        except Exception as e:
            self.logger.error(f"Error clearing graph: {str(e)}")
            raise GraphEngineError(f"Failed to clear graph: {str(e)}")

    def serialize(self) -> Dict[str, Any]:
        """Serialize the graph to a dictionary"""
        nodes_data = {}
        edges_data = {}

        try:
            with self.driver.session(database=self.database) as session:
                # Export nodes
                cypher_nodes = """
                MATCH (n:Node)
                RETURN n
                """

                result = session.run(cypher_nodes)

                for record in result:
                    node_data = dict(record["n"])
                    node_id = node_data.pop("id")

                    # Extract standard properties
                    nodes_data[node_id] = {
                        "label": node_data.pop("label", "Node"),
                        "properties": {k: v for k, v in node_data.items()
                                     if k not in ["activation", "last_accessed",
                                                 "importance", "creation_time"]},
                        "activation": node_data.get("activation", 0.0),
                        "last_accessed": node_data.get("last_accessed", time.time()),
                        "importance": node_data.get("importance", 0.5),
                        "creation_time": node_data.get("creation_time", time.time())
                    }

                # Export edges
                cypher_edges = """
                MATCH ()-[r:RELATED_TO]->()
                RETURN r, startNode(r).id as source_id, endNode(r).id as target_id
                """

                result = session.run(cypher_edges)

                for record in result:
                    edge_data = dict(record["r"])
                    edge_id = edge_data.pop("id")

                    # Extract standard properties
                    edges_data[edge_id] = {
                        "source_id": record["source_id"],
                        "target_id": record["target_id"],
                        "type": edge_data.pop("type", "RELATED_TO"),
                        "properties": {k: v for k, v in edge_data.items()
                                     if k not in ["weight", "bidirectional", "creation_time"]},
                        "weight": edge_data.get("weight", 1.0),
                        "bidirectional": edge_data.get("bidirectional", False),
                        "creation_time": edge_data.get("creation_time", time.time())
                    }

                return {
                    "nodes": nodes_data,
                    "edges": edges_data,
                    "metadata": {
                        "timestamp": time.time(),
                        "instance_id": self.instance_id,
                        "node_count": len(nodes_data),
                        "edge_count": len(edges_data)
                    }
                }

        except Exception as e:
            self.logger.error(f"Error serializing graph: {str(e)}")
            raise GraphEngineError(f"Failed to serialize graph: {str(e)}")

    def deserialize(self, data: Dict[str, Any]) -> None:
        """Load the graph from a serialized representation"""
        # Clear existing graph
        self.clear()

        try:
            with self.driver.session(database=self.database) as session:
                # Import nodes
                for node_id, node_data in data.get("nodes", {}).items():
                    # Prepare node properties
                    props = node_data.get("properties", {})
                    props.update({
                        "id": node_id,
                        "label": node_data.get("label", "Node"),
                        "activation": node_data.get("activation", 0.0),
                        "last_accessed": node_data.get("last_accessed", time.time()),
                        "importance": node_data.get("importance", 0.5),
                        "creation_time": node_data.get("creation_time", time.time())
                    })

                    # Create node
                    cypher = """
                    CREATE (n:Node $props)
                    """

                    session.run(cypher, props=props)

                # Import edges
                for edge_id, edge_data in data.get("edges", {}).items():
                    # Create edge
                    cypher = """
                    MATCH (s:Node {id: $source_id}), (t:Node {id: $target_id})
                    CREATE (s)-[r:RELATED_TO {
                        id: $edge_id,
                        type: $edge_type,
                        weight: $weight,
                        bidirectional: $bidirectional,
                        creation_time: $creation_time
                    }]->(t)
                    SET r += $properties
                    """

                    session.run(
                        cypher,
                        source_id=edge_data["source_id"],
                        target_id=edge_data["target_id"],
                        edge_id=edge_id,
                        edge_type=edge_data.get("type", "RELATED_TO"),
                        weight=edge_data.get("weight", 1.0),
                        bidirectional=edge_data.get("bidirectional", False),
                        creation_time=edge_data.get("creation_time", time.time()),
                        properties=edge_data.get("properties", {})
                    )

                self.logger.info(f"Loaded graph with {len(data.get('nodes', {}))} nodes and {len(data.get('edges', {}))} edges")

        except Exception as e:
            self.logger.error(f"Error deserializing graph: {str(e)}")
            raise GraphEngineError(f"Failed to deserialize graph: {str(e)}")

    def close(self) -> None:
        """Close the graph engine and release resources"""
        try:
            # Close the Neo4j driver
            self.driver.close()

            # Clear caches
            self.node_cache.clear()
            self.edge_cache.clear()

            self.logger.info("Neo4j connection closed")

        except Exception as e:
            self.logger.error(f"Error closing Neo4j connection: {str(e)}")


# -----------------------------------------------------------------------------
# Factory function to create graph engines
# -----------------------------------------------------------------------------

    def create_graph_engine(engine_type: str, **kwargs) -> AbstractGraphEngine:
        """
        Factory function to create the appropriate graph engine.

        Args:
            engine_type: Type of engine to create ("inmemory" or "neo4j")
            **kwargs: Arguments to pass to the engine constructor

        Returns:
            An instance of the appropriate graph engine

        Raises:
            ValueError: If engine_type is not recognized
            ImportError: If required dependencies are not available
        """
        engine_type = engine_type.lower()

        if engine_type == "inmemory":
            return InMemoryGraphEngine(**kwargs)

        elif engine_type == "neo4j":
            if not HAS_NEO4J:
                raise ImportError("Neo4j Python driver is not installed. Install with: pip install neo4j")

            # Validate required parameters
            required_params = ["uri", "user", "password"]
            missing_params = [p for p in required_params if p not in kwargs]

            if missing_params:
                raise ValueError(f"Missing required Neo4j parameters: {missing_params}")

            return Neo4jGraphEngine(**kwargs)

        else:
            raise ValueError(f"Unknown engine type: {engine_type}. Supported types: inmemory, neo4j")

# Pattern Mining

In [ ]:
"""
pattern_mining.py - Advanced Pattern Mining & Anomaly Detection for GrandNexus

This module provides advanced pattern analysis capabilities for the graph structure:
- Frequent subgraph mining
- Pattern matching and similarity search
- Anomaly detection
- Subgraph embedding and comparison

It works in tandem with the graph_engine.py module while maintaining separation of concerns:
- graph_engine.py handles the basic graph operations and structure
- pattern_mining.py provides higher-level cognitive pattern analysis
"""

import logging
import time
import threading
import math
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Callable
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import uuid

from grandnexus.graph.graph_engine import ( create_graph_engine,
    AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)
from grandnexus.graph.graph_pattern_interface import (
    PatternMiningInterface, PatternMatchResult, FrequentPattern
)

# Optional dependencies for vector operations
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

# Optional dependency for graph algorithms
try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False


class PatternMiningModule(PatternMiningInterface):
    """
    Provides advanced pattern-mining capabilities for GrandNexus.

    Works with GraphEngine to discover patterns, detect anomalies,
    compare subgraphs, and extract knowledge structures from the graph.
    """

    def __init__(self,
                 graph_engine: AbstractGraphEngine,
                 use_networkx: bool = True,
                 default_similarity_threshold: float = 0.7,
                 embedding_dim: int = 128,
                 pattern_cache_size: int = 1000):
        """
        Initialize the pattern mining module.

        Args:
            graph_engine: The graph engine instance to work with
            use_networkx: Whether to use NetworkX for advanced graph algorithms
            default_similarity_threshold: Default threshold for pattern matching
            embedding_dim: Dimension for pattern embeddings
            pattern_cache_size: Maximum number of patterns to cache
        """
        self.logger = logging.getLogger("PatternMiningModule")
        self.graph_engine = graph_engine
        self.use_networkx = use_networkx and HAS_NETWORKX
        self.default_similarity_threshold = default_similarity_threshold
        self.embedding_dim = embedding_dim
        self.pattern_cache_size = pattern_cache_size

        # Cache for discovered patterns
        self._pattern_cache = {}
        self._frequent_patterns = []
        self._nx_graph = None

        # Concurrency control
        self.lock = threading.RLock()

        # Initialize NetworkX graph if enabled
        if self.use_networkx:
            self._build_networkx_graph_if_needed()

        self.logger.info(f"PatternMiningModule initialized. NetworkX={self.use_networkx}, "
                        f"similarity_threshold={self.default_similarity_threshold:.2f}")

    def _build_networkx_graph_if_needed(self) -> Optional["nx.Graph"]:
        """Build or update the NetworkX representation of the graph."""
        if not self.use_networkx:
            return None

        with self.lock:
            try:
                self.logger.debug("Building/updating NetworkX graph representation")

                # Create a new MultiDiGraph to support multiple edge types between nodes
                g = nx.MultiDiGraph()

                # Get all nodes from the graph engine
                # We'll use a graph query to retrieve all nodes
                query = GraphQuery(
                    max_results=100000  # Set a high limit to get all nodes
                )

                result = self.graph_engine.execute_query(query)
                nodes_data = result.get('nodes', [])
                edges_data = result.get('edges', [])

                # Add nodes to NetworkX graph
                for node_data in nodes_data:
                    node_id = node_data['id']
                    properties = node_data.get('properties', {})
                    label = node_data.get('label', 'Node')

                    # Prepare node attributes
                    attrs = dict(properties)
                    attrs['label'] = label
                    attrs['activation'] = node_data.get('activation', 0.0)
                    attrs['importance'] = node_data.get('importance', 0.5)

                    g.add_node(node_id, **attrs)

                # Add edges to NetworkX graph
                for edge_data in edges_data:
                    source_id = edge_data['source']
                    target_id = edge_data['target']
                    edge_id = edge_data['id']
                    edge_type = edge_data.get('type', 'RELATES')
                    properties = edge_data.get('properties', {})

                    # Prepare edge attributes
                    attrs = dict(properties)
                    attrs['edge_id'] = edge_id
                    attrs['type'] = edge_type
                    attrs['weight'] = edge_data.get('weight', 1.0)

                    g.add_edge(source_id, target_id, key=edge_id, **attrs)

                self._nx_graph = g
                self.logger.debug(f"NetworkX graph built with {g.number_of_nodes()} nodes "
                                f"and {g.number_of_edges()} edges")
                return g

            except Exception as e:
                self.logger.error(f"Error building NetworkX graph: {e}")
                return None

    def find_patterns(self,
                      pattern_spec: Dict[str, Any],
                      similarity_threshold: Optional[float] = None,
                      max_results: int = 10,
                      use_cached: bool = True) -> List[PatternMatchResult]:
        """
        Searches the graph for subgraphs matching the given pattern specification.

        Args:
            pattern_spec: Dictionary describing the pattern to match
            similarity_threshold: Minimum similarity score (defaults to self.default_similarity_threshold)
            max_results: Maximum number of results to return
            use_cached: Whether to use/update the pattern cache

        Returns:
            List of PatternMatchResult objects
        """
        with self.lock:
            # Generate a cache key for this pattern
            pattern_hash = str(hash(str(pattern_spec)))
            if use_cached and pattern_hash in self._pattern_cache:
                self.logger.debug(f"Using cached pattern results for {pattern_hash}")
                return self._pattern_cache[pattern_hash][:max_results]

            similarity_threshold = similarity_threshold or self.default_similarity_threshold
            self.logger.info(f"Finding patterns with threshold={similarity_threshold:.2f}")

            # Choose implementation based on available tools
            if self.use_networkx and self._nx_graph:
                results = self._find_patterns_networkx(pattern_spec, similarity_threshold)
            else:
                results = self._find_patterns_direct(pattern_spec, similarity_threshold)

            # Sort by similarity score (descending)
            results.sort(key=lambda x: x.similarity_score, reverse=True)

            # Cache the results if requested
            if use_cached:
                self._pattern_cache[pattern_hash] = results

                # Prune cache if it exceeds the maximum size
                if len(self._pattern_cache) > self.pattern_cache_size:
                    # Remove the oldest entries
                    keys_to_remove = sorted(self._pattern_cache.keys())[:len(self._pattern_cache) - self.pattern_cache_size]
                    for key in keys_to_remove:
                        del self._pattern_cache[key]

            return results[:max_results]

    def _find_patterns_networkx(self, pattern_spec: Dict[str, Any],
                              similarity_threshold: float) -> List[PatternMatchResult]:
        """Find patterns using NetworkX graph algorithms."""
        results = []

        # Extract pattern constraints
        pattern_nodes = pattern_spec.get("nodes", [])
        pattern_edges = pattern_spec.get("edges", [])
        structure_info = pattern_spec.get("structure", {})
        max_depth = structure_info.get("max_depth", 2)

        if not pattern_nodes:
            return []

        # For each node in the graph, try it as a starting point
        for start_node in self._nx_graph.nodes():
            # Extract local subgraph around this node
            subgraph = nx.ego_graph(self._nx_graph, start_node, radius=max_depth)

            # Skip tiny subgraphs that are too small to match the pattern
            if len(subgraph) < len(pattern_nodes):
                continue

            # Check if the subgraph matches the pattern
            match_result = self._check_subgraph_match(subgraph, pattern_spec)

            if match_result and match_result.similarity_score >= similarity_threshold:
                results.append(match_result)

        return results

    def _find_patterns_direct(self, pattern_spec: Dict[str, Any],
                           similarity_threshold: float) -> List[PatternMatchResult]:
        """Find patterns using direct GraphEngine queries."""
        results = []

        # Extract pattern constraints
        pattern_nodes = pattern_spec.get("nodes", [])
        if not pattern_nodes:
            return []

        # Find candidate starting nodes
        first_node_spec = pattern_nodes[0]
        node_type = first_node_spec.get("type", "Node")
        constraints = first_node_spec.get("constraints", {})

        # Convert to GraphQuery
        query = GraphQuery(
            node_criteria={"type": node_type, **constraints},
            max_depth=1
        )

        try:
            # Execute the query to find potential starting nodes
            query_result = self.graph_engine.execute_query(query)
            candidate_nodes = [node['id'] for node in query_result.get('nodes', [])]

            # For each candidate node, expand and check if it matches the pattern
            for node_id in candidate_nodes:
                # Create a simple pattern match with this as the starting point
                result = self._check_direct_pattern_match(node_id, pattern_spec)

                if result and result.similarity_score >= similarity_threshold:
                    results.append(result)

                    # Limit results to avoid excessive computation
                    if len(results) >= 100:  # Internal limit
                        break

        except Exception as e:
            self.logger.error(f"Error finding patterns directly: {e}")

        return results

    def _check_subgraph_match(self, subgraph: "nx.Graph",
                            pattern_spec: Dict[str, Any]) -> Optional[PatternMatchResult]:
        """Check if a subgraph matches the pattern specification."""
        # Extract pattern constraints
        pattern_nodes = pattern_spec.get("nodes", [])
        pattern_edges = pattern_spec.get("edges", [])

        # Count nodes matching each required type
        node_type_matches = defaultdict(int)
        node_ids_by_type = defaultdict(list)

        for node_id in subgraph.nodes():
            node_data = subgraph.nodes[node_id]
            node_type = node_data.get("type", node_data.get("label", "unknown"))
            node_type_matches[node_type] += 1
            node_ids_by_type[node_type].append(node_id)

        # Check if we have enough nodes of each required type
        pattern_node_types = [n.get("type", "Node") for n in pattern_nodes]
        required_type_counts = Counter(pattern_node_types)

        # If we're missing any required node type, return None
        for req_type, count in required_type_counts.items():
            if node_type_matches[req_type] < count:
                return None

        # Check edges (simplified)
        edge_type_matches = defaultdict(int)
        edge_ids_by_type = defaultdict(list)

        for u, v, k, edge_data in subgraph.edges(data=True, keys=True):
            rel_type = edge_data.get("type", edge_data.get("relation_type", "unknown"))
            edge_type_matches[rel_type] += 1
            edge_ids_by_type[rel_type].append(edge_data.get("edge_id", k))

        # Check if we have the required edges
        pattern_edge_types = [e.get("type", "RELATES") for e in pattern_edges]
        required_edge_counts = Counter(pattern_edge_types)

        # If we're missing any required edge type, return None
        for req_type, count in required_edge_counts.items():
            if edge_type_matches[req_type] < count:
                return None

        # Calculate similarity score

        # Node type similarity
        node_type_sim = 0.0
        if required_type_counts:
            node_type_sim = sum(min(node_type_matches[t], required_type_counts[t])
                             for t in required_type_counts) / sum(required_type_counts.values())

        # Edge type similarity
        edge_type_sim = 1.0
        if required_edge_counts:
            edge_type_sim = sum(min(edge_type_matches[t], required_edge_counts[t])
                             for t in required_edge_counts) / sum(required_edge_counts.values())

        # Combined similarity
        similarity = (node_type_sim + edge_type_sim) / 2.0

        # Create the match result
        all_matched_nodes = []
        for node_type in required_type_counts:
            # Take only as many nodes as required by the pattern
            all_matched_nodes.extend(node_ids_by_type[node_type][:required_type_counts[node_type]])

        all_matched_edges = []
        for edge_type in required_edge_counts:
            # Take only as many edges as required by the pattern
            all_matched_edges.extend(edge_ids_by_type[edge_type][:required_edge_counts[edge_type]])

        return PatternMatchResult(
            subgraph_nodes=all_matched_nodes,
            subgraph_edges=all_matched_edges,
            similarity_score=similarity,
            pattern_type=pattern_spec.get("pattern_type", "custom"),
            extra_info={
                "node_type_similarity": node_type_sim,
                "edge_type_similarity": edge_type_sim,
                "matched_node_types": dict(node_type_matches),
                "matched_edge_types": dict(edge_type_matches)
            }
        )

    def _check_direct_pattern_match(self, start_node_id: str,
                                 pattern_spec: Dict[str, Any]) -> Optional[PatternMatchResult]:
        """Check if a pattern matches starting from a specific node using direct graph queries."""
        # Extract pattern constraints
        pattern_nodes = pattern_spec.get("nodes", [])
        pattern_edges = pattern_spec.get("edges", [])
        max_depth = pattern_spec.get("structure", {}).get("max_depth", 2)

        # Start with the first node matching the first pattern node
        matched_nodes = [start_node_id]
        matched_edges = []

        # Get neighboring nodes to a certain depth
        try:
            # For simplicity, just get the neighbors up to max_depth
            # In a real implementation, this would be more sophisticated
            current_nodes = [start_node_id]
            next_nodes = []
            visited = {start_node_id}

            for depth in range(max_depth):
                for node_id in current_nodes:
                    neighbors = self.graph_engine.get_node_neighbors(node_id, direction="both")

                    for neighbor_id, edge_id, edge_type in neighbors:
                        if neighbor_id not in visited:
                            visited.add(neighbor_id)
                            next_nodes.append(neighbor_id)

                            # Get the node to check its type
                            try:
                                neighbor = self.graph_engine.get_node(neighbor_id)

                                # Check if this matches any required node type
                                for pattern_node in pattern_nodes[1:]:  # Skip first, we already matched it
                                    required_type = pattern_node.get("type", "Node")
                                    if neighbor.label == required_type:
                                        matched_nodes.append(neighbor_id)
                                        matched_edges.append(edge_id)
                                        break
                            except NodeNotFoundError:
                                pass

                current_nodes = next_nodes
                next_nodes = []

            # Calculate a simple similarity score based on how many pattern nodes we matched
            nodes_matched = len(matched_nodes)
            nodes_required = len(pattern_nodes)

            similarity = nodes_matched / nodes_required if nodes_required > 0 else 0.0

            if similarity > 0:
                return PatternMatchResult(
                    subgraph_nodes=matched_nodes,
                    subgraph_edges=matched_edges,
                    similarity_score=similarity,
                    pattern_type=pattern_spec.get("pattern_type", "custom"),
                    extra_info={
                        "direct_match": True,
                        "starting_node": start_node_id
                    }
                )

        except Exception as e:
            self.logger.error(f"Error checking direct pattern match: {e}")

        return None

    def extract_frequent_subgraphs(self,
                                min_support: int = 2,
                                max_size: int = 5,
                                store_embeddings: bool = True) -> List[FrequentPattern]:
        """
        Discovers frequently occurring patterns in the graph.

        Args:
            min_support: Minimum number of occurrences to be considered frequent
            max_size: Maximum size (nodes) of patterns to consider
            store_embeddings: Whether to compute and store embeddings for patterns

        Returns:
            List of FrequentPattern objects
        """
        with self.lock:
            self.logger.info(f"Extracting frequent subgraphs (min_support={min_support}, max_size={max_size})")

            if self.use_networkx and self._nx_graph:
                patterns = self._extract_frequent_subgraphs_networkx(min_support, max_size)
            else:
                patterns = self._extract_frequent_subgraphs_direct(min_support, max_size)

            # Compute embeddings if requested
            if store_embeddings and patterns:
                for pattern in patterns:
                    if pattern.pattern_embedding is None:
                        # Create a temporary PatternMatchResult to compute the embedding
                        temp_result = PatternMatchResult(
                            subgraph_nodes=pattern.nodes,
                            subgraph_edges=pattern.edges
                        )
                        pattern.pattern_embedding = self._compute_pattern_embedding(temp_result)

            # Sort by support (descending)
            patterns.sort(key=lambda x: x.support, reverse=True)

            # Update the cache
            self._frequent_patterns = patterns

            self.logger.info(f"Discovered {len(patterns)} frequent patterns")
            return patterns

    def _extract_frequent_subgraphs_networkx(self, min_support: int, max_size: int) -> List[FrequentPattern]:
        """Extract frequent subgraphs using NetworkX."""
        if not HAS_NETWORKX or not self._nx_graph:
            return []

        # This is a simplified approach using subgraph sampling
        # A real implementation would use algorithms like gSpan, Subdue, etc.

        pattern_candidates = defaultdict(lambda: {"nodes": [], "edges": [], "occurrences": []})

        # Sample random starting points
        node_list = list(self._nx_graph.nodes())
        if len(node_list) < 3:
            self.logger.warning("Graph is too small for meaningful pattern mining")
            return []

        # For small graphs, use all nodes. For large graphs, sample a subset.
        sample_size = min(len(node_list), 100)
        try:
            starting_nodes = np.random.choice(node_list, size=sample_size, replace=False)
        except Exception:
            # Fallback if numpy is not available
            starting_nodes = node_list[:min(100, len(node_list))]

        for start_node in starting_nodes:
            # For each size from 2 to max_size
            for size in range(2, max_size + 1):
                # Get a local subgraph of approximately this size
                subgraph = self._sample_local_subgraph(start_node, size)

                if len(subgraph) < 2:  # Skip trivial subgraphs
                    continue

                # Compute a canonical signature for this subgraph
                signature = self._compute_subgraph_signature(subgraph)

                # Add to candidates
                node_ids = list(subgraph.nodes())
                edge_ids = []
                for u, v, k, data in subgraph.edges(data=True, keys=True):
                    edge_id = data.get("edge_id", k)
                    edge_ids.append(edge_id)

                pattern_candidates[signature]["nodes"].extend(node_ids)
                pattern_candidates[signature]["edges"].extend(edge_ids)
                pattern_candidates[signature]["occurrences"].append((node_ids, edge_ids))

        # Filter by support and convert to FrequentPattern objects
        frequent_patterns = []
        pattern_id_counter = 0

        for signature, info in pattern_candidates.items():
            occurrences = info["occurrences"]
            if len(occurrences) >= min_support:
                # Create a FrequentPattern
                pattern_id = f"pattern_{pattern_id_counter}"
                pattern_id_counter += 1

                # Use the first occurrence as the example
                example_nodes, example_edges = occurrences[0]

                pattern = FrequentPattern(
                    pattern_id=pattern_id,
                    nodes=example_nodes,
                    edges=example_edges,
                    support=len(occurrences),
                    signature=signature
                )

                frequent_patterns.append(pattern)

        return frequent_patterns

    def _extract_frequent_subgraphs_direct(self, min_support: int, max_size: int) -> List[FrequentPattern]:
        """Extract frequent subgraphs using direct GraphEngine queries."""
        # Simplified approach as a fallback when NetworkX is not available

        # Strategy:
        # 1. Group nodes by type/label
        # 2. For each label, find pairs of connected nodes
        # 3. Expand these pairs to form larger subgraphs
        # 4. Count occurrences of similar structures

        # Get all nodes with their labels
        nodes_by_label = defaultdict(list)

        # Query for all nodes
        query = GraphQuery(max_results=100000)  # High limit to get all nodes
        result = self.graph_engine.execute_query(query)

        for node in result.get('nodes', []):
            node_id = node['id']
            label = node.get('label', 'Node')
            nodes_by_label[label].append(node_id)

        # Find frequent patterns
        patterns = []
        pattern_id_counter = 0

        # Look for star patterns (nodes with multiple connections of the same type)
        # This is a simplified approach compared to full graph mining
        for label, node_ids in nodes_by_label.items():
            for node_id in node_ids:
                # Get all neighbors
                neighbors = self.graph_engine.get_node_neighbors(node_id, direction="both")

                # Group neighbors by edge type
                neighbors_by_type = defaultdict(list)
                edges_by_type = defaultdict(list)

                for neighbor_id, edge_id, edge_type in neighbors:
                    neighbors_by_type[edge_type].append(neighbor_id)
                    edges_by_type[edge_type].append(edge_id)

                # Look for edge types with multiple occurrences
                for edge_type, neighbor_ids in neighbors_by_type.items():
                    if len(neighbor_ids) >= min_support and len(neighbor_ids) + 1 <= max_size:
                        # We found a star pattern
                        pattern_id = f"pattern_{pattern_id_counter}"
                        pattern_id_counter += 1

                        # Create a pattern with the center node and its neighbors
                        pattern_nodes = [node_id] + neighbor_ids
                        pattern_edges = edges_by_type[edge_type]

                        # Create a signature
                        signature = f"star:{label}:{edge_type}:{len(neighbor_ids)}"

                        pattern = FrequentPattern(
                            pattern_id=pattern_id,
                            nodes=pattern_nodes,
                            edges=pattern_edges,
                            support=len(neighbor_ids),
                            signature=signature
                        )

                        patterns.append(pattern)

        return patterns

    def _sample_local_subgraph(self, start_node: str, target_size: int) -> "nx.Graph":
        """
        Samples a local subgraph of approximately target_size nodes starting from start_node.

        Args:
            start_node: Starting node ID
            target_size: Desired number of nodes

        Returns:
            A NetworkX graph (subgraph of original)
        """
        if not self._nx_graph or not self.use_networkx:
            return nx.Graph()  # Empty graph as fallback

        # Use BFS to expand from start_node
        visited = {start_node}
        frontier = [start_node]

        while len(visited) < target_size and frontier:
            current = frontier.pop(0)

            # Add neighbors in random order
            neighbors = list(self._nx_graph.neighbors(current))

            # Shuffle neighbors if numpy is available
            if HAS_NUMPY:
                try:
                    np.random.shuffle(neighbors)
                except Exception:
                    pass

            for neighbor in neighbors:
                if neighbor not in visited:
                    visited.add(neighbor)
                    frontier.append(neighbor)

                    if len(visited) >= target_size:
                        break

        # Extract the subgraph
        return self._nx_graph.subgraph(visited)

    def _compute_subgraph_signature(self, subgraph: "nx.Graph") -> str:
        """
        Computes a canonical signature for a subgraph.

        This is a simplified version - a real implementation would use more sophisticated
        isomorphism-invariant signatures.

        Args:
            subgraph: A NetworkX graph

        Returns:
            A string signature
        """
        # Count node types
        node_types = Counter()
        for node, data in subgraph.nodes(data=True):
            node_type = data.get("type", data.get("label", "unknown"))
            node_types[node_type] += 1

        # Count edge types
        edge_types = Counter()
        for u, v, data in subgraph.edges(data=True):
            if isinstance(data, dict):
                edge_type = data.get("type", data.get("relation_type", "unknown"))
                edge_types[edge_type] += 1

        # Create signature parts
        signature_parts = []

        # Add sorted node types
        node_sig = ";".join(f"{t}:{c}" for t, c in sorted(node_types.items()))
        signature_parts.append(f"N[{node_sig}]")

        # Add sorted edge types
        edge_sig = ";".join(f"{t}:{c}" for t, c in sorted(edge_types.items()))
        signature_parts.append(f"E[{edge_sig}]")

        # Add basic structural info
        signature_parts.append(f"S[{len(subgraph)}:{subgraph.number_of_edges()}]")

        return "|".join(signature_parts)

    def _compute_pattern_embedding(self, pattern: PatternMatchResult) -> Optional[Any]:
        """
        Computes a vector embedding for a pattern.

        Args:
            pattern: A PatternMatchResult object

        Returns:
            Numpy array or None if embeddings not supported
        """
        if not HAS_NUMPY:
            return None

        embedding_features = []
        node_ids = pattern.subgraph_nodes

        if not node_ids:
            return np.zeros(self.embedding_dim)

        # If we don't have NetworkX or it's not enabled, use a simpler approach
        if not self.use_networkx or not self._nx_graph:
            # Create a basic embedding using node and edge counts
            try:
                # Get node types
                node_types = Counter()
                for node_id in node_ids:
                    try:
                        node = self.graph_engine.get_node(node_id)
                        node_types[node.label] += 1
                    except NodeNotFoundError:
                        pass

                # Get edge types
                edge_types = Counter()
                for edge_id in pattern.subgraph_edges:
                    try:
                        edge = self.graph_engine.get_edge(edge_id)
                        edge_types[edge.type] += 1
                    except EdgeNotFoundError:
                        pass

                # Add node type counts as features
                for node_type, count in sorted(node_types.items()):
                    embedding_features.append(count)

                # Add edge type counts as features
                for edge_type, count in sorted(edge_types.items()):
                    embedding_features.append(count)

                # Add basic size information
                embedding_features.append(len(node_ids))
                embedding_features.append(len(pattern.subgraph_edges))

            except Exception as e:
                self.logger.warning(f"Error computing embedding: {e}")
                embedding_features = [0] * 5  # Fallback

        else:
            # Use NetworkX for more sophisticated embeddings
            try:
                # Extract the subgraph
                subgraph = self._nx_graph.subgraph(node_ids)

                # Get node type distribution
                node_types = Counter()
                for node_id in node_ids:
                    if node_id in self._nx_graph:
                        node_data = self._nx_graph.nodes[node_id]
                        node_type = node_data.get("type", node_data.get("label", "unknown"))
                        node_types[node_type] += 1

                # Get edge type distribution
                edge_types = Counter()
                for edge_id in pattern.subgraph_edges:
                    # Find the edge with this ID
                    for u, v, k, data in self._nx_graph.edges(data=True, keys=True):
                        if data.get("edge_id") == edge_id or k == edge_id:
                            edge_type = data.get("type", data.get("relation_type", "unknown"))
                            edge_types[edge_type] += 1
                            break

                # Add basic graph metrics as features
                if len(node_ids) > 1:
                    density = nx.density(subgraph)
                    embedding_features.append(density)

                    # Try to compute clustering
                    try:
                        clustering = nx.average_clustering(subgraph.to_undirected())
                        embedding_features.append(clustering)
                    except Exception:
                        embedding_features.append(0.0)

                    # Add degree distribution features
                    degrees = [d for _, d in subgraph.degree()]
                    if degrees:
                        avg_degree = sum(degrees) / len(degrees)
                        max_degree = max(degrees) if degrees else 0
                        embedding_features.extend([avg_degree, max_degree])
                else:
                    # Placeholders for single-node patterns
                    embedding_features.extend([0.0, 0.0, 0.0, 0.0])

                # Add node and edge type features
                for node_type, count in sorted(node_types.items()):
                    embedding_features.append(count)

                for edge_type, count in sorted(edge_types.items()):
                    embedding_features.append(count)

            except Exception as e:
                self.logger.warning(f"Error computing topological features: {e}")
                embedding_features = [0] * 5  # Fallback

        # Convert to numpy array and pad/truncate to fixed dimension
        embedding = np.array(embedding_features, dtype=np.float32)

        # Pad or truncate to the desired dimension
        if len(embedding) < self.embedding_dim:
            embedding = np.pad(embedding, (0, self.embedding_dim - len(embedding)))
        elif len(embedding) > self.embedding_dim:
            embedding = embedding[:self.embedding_dim]

        # Normalize
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm

        return embedding

    def find_similar_patterns(self,
                            pattern: Union[PatternMatchResult, Dict[str, Any]],
                            top_n: int = 5) -> List[Tuple[PatternMatchResult, float]]:
        """
        Finds patterns similar to the provided pattern using embedding similarity.

        Args:
            pattern: Either a PatternMatchResult or a pattern specification dict
            top_n: Maximum number of results to return

        Returns:
            List of (pattern, similarity_score) tuples
        """
        with self.lock:
            # Convert pattern to PatternMatchResult if needed
            if isinstance(pattern, dict):
                # It's a pattern spec - find a matching pattern first
                matches = self.find_patterns(pattern, max_results=1)
                if not matches:
                    return []
                query_pattern = matches[0]
            else:
                query_pattern = pattern

            # Compute embedding for the query pattern
            query_embedding = self._compute_pattern_embedding(query_pattern)
            if query_embedding is None:
                return []  # Can't compute similarities without embeddings

            # Get patterns to compare against
            all_patterns = []

            # First, check the frequent patterns we've discovered
            for freq_pattern in self._frequent_patterns:
                if freq_pattern.pattern_embedding is not None:
                    # Create a PatternMatchResult from the FrequentPattern
                    pattern_result = PatternMatchResult(
                        subgraph_nodes=freq_pattern.nodes,
                        subgraph_edges=freq_pattern.edges,
                        similarity_score=1.0,  # Will be replaced
                        pattern_type="frequent",
                        extra_info={"support": freq_pattern.support, "pattern_id": freq_pattern.pattern_id}
                    )
                    all_patterns.append((pattern_result, freq_pattern.pattern_embedding))

            # Also check the pattern cache
            for patterns in self._pattern_cache.values():
                for p in patterns:
                    # Compute embedding if needed
                    embedding = self._compute_pattern_embedding(p)
                    if embedding is not None:
                        all_patterns.append((p, embedding))

            # Calculate similarities
            similarities = []
            for p, emb in all_patterns:
                # Skip if it's the exact same pattern (same nodes)
                if set(p.subgraph_nodes) == set(query_pattern.subgraph_nodes):
                    continue

                # Calculate cosine similarity
                try:
                    similarity = np.dot(query_embedding, emb) / (
                        np.linalg.norm(query_embedding) * np.linalg.norm(emb))
                    similarities.append((p, float(similarity)))
                except Exception as e:
                    self.logger.warning(f"Error calculating similarity: {e}")

            # Sort by similarity (descending) and return top N
            similarities.sort(key=lambda x: x[1], reverse=True)
            return similarities[:top_n]

    def detect_anomalies(self,
                       property_outliers: bool = True,
                       structural_anomalies: bool = True,
                       threshold: float = 2.0) -> List[Dict[str, Any]]:
        """
        Detects nodes or relationships that appear anomalous by their properties or structure.

        Args:
            property_outliers: Whether to look for anomalies based on properties
            structural_anomalies: Whether to look for structural anomalies
            threshold: Z-score threshold for anomaly detection

        Returns:
            List of anomaly dictionaries with details
        """
        with self.lock:
            anomalies = []

            if not self.use_networkx or not self._nx_graph:
                # Fallback to direct graph engine when NetworkX is not available
                return self._detect_anomalies_direct(property_outliers, structural_anomalies, threshold)

            self.logger.info(f"Detecting anomalies (property={property_outliers}, "
                           f"structural={structural_anomalies}, threshold={threshold})")

            # Property-based anomaly detection
            if property_outliers:
                # Collect statistics by node type
                node_type_stats = defaultdict(lambda: defaultdict(list))

                # For each node, collect numeric properties by type
                for node_id, attrs in self._nx_graph.nodes(data=True):
                    node_type = attrs.get("type", attrs.get("label", "unknown"))

                    # Collect numeric properties
                    for key, val in attrs.items():
                        if isinstance(val, (int, float)) and key not in ('created_at', 'updated_at'):
                            node_type_stats[node_type][key].append((node_id, val))

                # For each node type and property, detect anomalies
                for node_type, prop_data in node_type_stats.items():
                    for prop_name, values in prop_data.items():
                        if len(values) <= 1:
                            continue  # Not enough data for statistics

                        # Extract values
                        node_ids, numeric_values = zip(*values)

                        # Calculate mean and standard deviation
                        if HAS_NUMPY:
                            mean_val = np.mean(numeric_values)
                            std_val = np.std(numeric_values)
                        else:
                            # Fallback to manual calculation
                            mean_val = sum(numeric_values) / len(numeric_values)
                            squared_diffs = [(x - mean_val) ** 2 for x in numeric_values]
                            std_val = math.sqrt(sum(squared_diffs) / len(numeric_values))

                        if std_val <= 0:
                            continue  # Avoid division by zero

                        # Detect anomalies
                        for node_id, val in values:
                            z_score = abs(val - mean_val) / std_val
                            if z_score > threshold:
                                anomalies.append({
                                    "node_id": node_id,
                                    "type": "property_outlier",
                                    "node_type": node_type,
                                    "property": prop_name,
                                    "value": val,
                                    "mean": mean_val,
                                    "std": std_val,
                                    "z_score": z_score,
                                    "score": z_score  # For uniform sorting
                                })

            # Structural anomaly detection
            if structural_anomalies:
                # Degree statistics by node type
                degree_stats = defaultdict(list)
                in_degree_stats = defaultdict(list)
                out_degree_stats = defaultdict(list)

                for node_id, attrs in self._nx_graph.nodes(data=True):
                    node_type = attrs.get("type", attrs.get("label", "unknown"))

                    # Calculate degrees
                    total_degree = self._nx_graph.degree(node_id)
                    in_degree = self._nx_graph.in_degree(node_id)
                    out_degree = self._nx_graph.out_degree(node_id)

                    degree_stats[node_type].append((node_id, total_degree))
                    in_degree_stats[node_type].append((node_id, in_degree))
                    out_degree_stats[node_type].append((node_id, out_degree))

                # Analyze degree statistics by type
                for degree_type, stats_dict in [
                    ("total", degree_stats),
                    ("in", in_degree_stats),
                    ("out", out_degree_stats)
                ]:
                    for node_type, values in stats_dict.items():
                        if len(values) <= 1:
                            continue

                        node_ids, degrees = zip(*values)

                        # Calculate statistics
                        if HAS_NUMPY:
                            mean_degree = np.mean(degrees)
                            std_degree = np.std(degrees)
                        else:
                            mean_degree = sum(degrees) / len(degrees)
                            squared_diffs = [(x - mean_degree) ** 2 for x in degrees]
                            std_degree = math.sqrt(sum(squared_diffs) / len(degrees))

                        if std_degree <= 0:
                            continue

                        for node_id, degree in values:
                            z_score = abs(degree - mean_degree) / std_degree
                            if z_score > threshold:
                                anomalies.append({
                                    "node_id": node_id,
                                    "type": "structural_anomaly",
                                    "anomaly_type": f"{degree_type}_degree_outlier",
                                    "node_type": node_type,
                                    "degree": degree,
                                    "mean": mean_degree,
                                    "std": std_degree,
                                    "z_score": z_score,
                                    "score": z_score
                                })

                # Detect clustering anomalies (optional)
                try:
                    # Calculate clustering coefficients
                    clustering = nx.clustering(self._nx_graph.to_undirected())

                    clustering_stats = defaultdict(list)
                    for node_id, cluster_coef in clustering.items():
                        if node_id in self._nx_graph:
                            node_type = self._nx_graph.nodes[node_id].get("type",
                                                                       self._nx_graph.nodes[node_id].get("label", "unknown"))
                            clustering_stats[node_type].append((node_id, cluster_coef))

                    # Analyze clustering statistics by type
                    for node_type, values in clustering_stats.items():
                        if len(values) <= 1:
                            continue

                        node_ids, coeffs = zip(*values)

                        # Calculate statistics
                        if HAS_NUMPY:
                            mean_coeff = np.mean(coeffs)
                            std_coeff = np.std(coeffs)
                        else:
                            mean_coeff = sum(coeffs) / len(coeffs)
                            squared_diffs = [(x - mean_coeff) ** 2 for x in coeffs]
                            std_coeff = math.sqrt(sum(squared_diffs) / len(coeffs))

                        if std_coeff <= 0:
                            continue

                        for node_id, coeff in values:
                            z_score = abs(coeff - mean_coeff) / std_coeff
                            if z_score > threshold:
                                anomalies.append({
                                    "node_id": node_id,
                                    "type": "structural_anomaly",
                                    "anomaly_type": "clustering_outlier",
                                    "node_type": node_type,
                                    "clustering": coeff,
                                    "mean": mean_coeff,
                                    "std": std_coeff,
                                    "z_score": z_score,
                                    "score": z_score
                                })
                except Exception as e:
                    self.logger.warning(f"Error calculating clustering: {e}")

            # Sort by anomaly score (descending)
            anomalies.sort(key=lambda x: x["score"], reverse=True)

            self.logger.info(f"Detected {len(anomalies)} anomalies")
            return anomalies

    def _detect_anomalies_direct(self, property_outliers: bool,
                              structural_anomalies: bool,
                              threshold: float) -> List[Dict[str, Any]]:
        """Detect anomalies using direct GraphEngine queries (fallback)."""
        anomalies = []

        # Query for all nodes
        query = GraphQuery(max_results=100000)  # High limit to get all nodes
        result = self.graph_engine.execute_query(query)
        nodes = result.get('nodes', [])

        if not nodes:
            return []

        # Property-based anomaly detection
        if property_outliers:
            # Group nodes by type
            nodes_by_type = defaultdict(list)

            for node in nodes:
                node_type = node.get('label', 'Node')
                nodes_by_type[node_type].append(node)

            # For each node type, look for property outliers
            for node_type, type_nodes in nodes_by_type.items():
                if len(type_nodes) <= 1:
                    continue

                # Collect property statistics
                property_values = defaultdict(list)

                for node in type_nodes:
                    for prop, value in node.get('properties', {}).items():
                        if isinstance(value, (int, float)):
                            property_values[prop].append((node['id'], value))

                # Analyze each property
                for prop_name, values in property_values.items():
                    if len(values) <= 1:
                        continue

                    # Extract values
                    node_ids, numeric_values = zip(*values)

                    # Calculate mean and standard deviation
                    if HAS_NUMPY:
                        mean_val = np.mean(numeric_values)
                        std_val = np.std(numeric_values)
                    else:
                        # Fallback to manual calculation
                        mean_val = sum(numeric_values) / len(numeric_values)
                        squared_diffs = [(x - mean_val) ** 2 for x in numeric_values]
                        std_val = math.sqrt(sum(squared_diffs) / len(numeric_values))

                    if std_val <= 0:
                        continue  # Avoid division by zero

                    # Detect anomalies
                    for node_id, val in values:
                        z_score = abs(val - mean_val) / std_val
                        if z_score > threshold:
                            anomalies.append({
                                "node_id": node_id,
                                "type": "property_outlier",
                                "node_type": node_type,
                                "property": prop_name,
                                "value": val,
                                "mean": mean_val,
                                "std": std_val,
                                "z_score": z_score,
                                "score": z_score
                            })

        # Structural anomaly detection
        if structural_anomalies:
            # Count degrees for all nodes
            degrees = {}

            for node in nodes:
                node_id = node['id']
                try:
                    neighbors = self.graph_engine.get_node_neighbors(node_id, direction="both")
                    degrees[node_id] = len(neighbors)
                except Exception:
                    degrees[node_id] = 0

            # Group by node type
            degree_by_type = defaultdict(list)

            for node in nodes:
                node_id = node['id']
                node_type = node.get('label', 'Node')
                if node_id in degrees:
                    degree_by_type[node_type].append((node_id, degrees[node_id]))

            # Analyze each node type
            for node_type, values in degree_by_type.items():
                if len(values) <= 1:
                    continue

                node_ids, degree_values = zip(*values)

                # Calculate statistics
                # Calculate statistics
                if HAS_NUMPY:
                    mean_degree = np.mean(degree_values)
                    std_degree = np.std(degree_values)
                else:
                    # Fallback to manual calculation
                    mean_degree = sum(degree_values) / len(degree_values)
                    squared_diffs = [(x - mean_degree) ** 2 for x in degree_values]
                    std_degree = math.sqrt(sum(squared_diffs) / len(degree_values))

                if std_degree <= 0:
                    continue

                # Find outliers
                for node_id, degree in values:
                    z_score = abs(degree - mean_degree) / std_degree
                    if z_score > threshold:
                        anomalies.append({
                            "node_id": node_id,
                            "type": "structural_anomaly",
                            "anomaly_type": "degree_outlier",
                            "node_type": node_type,
                            "degree": degree,
                            "mean": mean_degree,
                            "std": std_degree,
                            "z_score": z_score,
                            "score": z_score
                        })

        # Sort by anomaly score (descending)
        anomalies.sort(key=lambda x: x["score"], reverse=True)

        return anomalies

    def get_pattern_statistics(self) -> Dict[str, Any]:
        """
        Returns statistics about the discovered patterns and cached patterns.

        Returns:
            Dictionary of statistics
        """
        with self.lock:
            stats = {
                "cached_patterns": sum(len(patterns) for patterns in self._pattern_cache.values()),
                "frequent_patterns": len(self._frequent_patterns),
                "pattern_types": defaultdict(int),
                "patterns_with_embeddings": 0,
                "timestamp": time.time()
            }

            # Count pattern types in cache
            for patterns in self._pattern_cache.values():
                for p in patterns:
                    if p.pattern_type:
                        stats["pattern_types"][p.pattern_type] += 1

            # Count patterns with embeddings
            for p in self._frequent_patterns:
                if p.pattern_embedding is not None:
                    stats["patterns_with_embeddings"] += 1

            # If we have NetworkX, add some graph statistics
            if self.use_networkx and self._nx_graph:
                try:
                    stats["graph_statistics"] = {
                        "nodes": self._nx_graph.number_of_nodes(),
                        "edges": self._nx_graph.number_of_edges(),
                        "density": nx.density(self._nx_graph),
                        "connected_components": nx.number_weakly_connected_components(self._nx_graph),
                        "strongly_connected_components": nx.number_strongly_connected_components(self._nx_graph)
                    }
                except Exception as e:
                    self.logger.warning(f"Error calculating graph statistics: {e}")

            return dict(stats)

    def clear_caches(self) -> None:
        """
        Clears the pattern caches to free memory and force recomputation.
        """
        with self.lock:
            self._pattern_cache.clear()
            self._frequent_patterns = []
            self.logger.info("Pattern caches cleared")

    def rebuild_networkx_graph(self) -> None:
        """
        Forces a rebuild of the NetworkX graph representation.
        """
        with self.lock:
            self._nx_graph = None
            self._build_networkx_graph_if_needed()
            self.logger.info("NetworkX graph rebuilt")

    def export_patterns_to_graph(self) -> int:
        """
        Exports discovered frequent patterns back to the GraphEngine as nodes.
        This allows patterns to be part of the graph itself, enabling meta-reasoning.

        Returns:
            Number of patterns exported
        """
        with self.lock:
            count = 0

            for pattern in self._frequent_patterns:
                pattern_id = f"pattern_{pattern.pattern_id}"

                try:
                    # Add a node representing the pattern
                    properties = {
                        "type": "Pattern",
                        "support": pattern.support,
                        "signature": pattern.signature,
                        "node_count": len(pattern.nodes),
                        "edge_count": len(pattern.edges)
                    }

                    # Add the pattern node
                    try:
                        self.graph_engine.add_node(
                            node_id=pattern_id,
                            label="Pattern",
                            properties=properties
                        )
                    except Exception:
                        # Node might already exist, try updating
                        self.graph_engine.update_node(pattern_id, properties)

                    # Connect the pattern to its nodes
                    for node_id in pattern.nodes:
                        try:
                            self.graph_engine.add_edge(
                                source_id=pattern_id,
                                target_id=node_id,
                                edge_type="CONTAINS",
                                properties={"role": "pattern_member"}
                            )
                        except Exception as e:
                            self.logger.debug(f"Couldn't connect pattern to node {node_id}: {e}")

                    count += 1

                except Exception as e:
                    self.logger.warning(f"Error exporting pattern {pattern_id}: {e}")

            self.logger.info(f"Exported {count} patterns to graph")
            return count

    def compare_subgraphs(self, subgraph1: List[str], subgraph2: List[str]) -> float:
        """
        Compares two subgraphs (lists of node IDs) and returns a similarity score.

        Args:
            subgraph1: List of node IDs for the first subgraph
            subgraph2: List of node IDs for the second subgraph

        Returns:
            Similarity score between 0 and 1
        """
        if not subgraph1 or not subgraph2:
            return 0.0

        # Create PatternMatchResult objects for the subgraphs
        result1 = PatternMatchResult(subgraph_nodes=subgraph1)
        result2 = PatternMatchResult(subgraph_nodes=subgraph2)

        # Compute embeddings
        emb1 = self._compute_pattern_embedding(result1)
        emb2 = self._compute_pattern_embedding(result2)

        if emb1 is None or emb2 is None:
            # Fallback to Jaccard similarity if embeddings not available
            set1 = set(subgraph1)
            set2 = set(subgraph2)

            if not set1 or not set2:
                return 0.0

            jaccard = len(set1.intersection(set2)) / len(set1.union(set2))
            return jaccard

        # Calculate cosine similarity
        try:
            similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
            return float(similarity)
        except Exception as e:
            self.logger.warning(f"Error calculating similarity: {e}")
            return 0.0

    def find_pattern_instances(self, pattern: FrequentPattern, min_similarity: float = 0.8) -> List[List[str]]:
        """
        Finds all instances of a pattern in the graph.

        Args:
            pattern: The pattern to find instances of
            min_similarity: Minimum similarity score for a match

        Returns:
            List of node ID lists, each representing an instance of the pattern
        """
        # Convert pattern to a query specification
        pattern_spec = {
            "pattern_type": "template",
            "nodes": [],
            "edges": [],
            "structure": {"max_depth": len(pattern.nodes) + 1}
        }

        # Add node specifications
        node_types = {}
        try:
            for node_id in pattern.nodes:
                node = self.graph_engine.get_node(node_id)
                node_types[node_id] = node.label
                pattern_spec["nodes"].append({"type": node.label})
        except Exception as e:
            self.logger.warning(f"Error getting node types for pattern: {e}")
            return []

        # Add edge specifications
        try:
            edge_types = set()
            for edge_id in pattern.edges:
                edge = self.graph_engine.get_edge(edge_id)
                edge_types.add(edge.type)

                source_idx = pattern.nodes.index(edge.source_id)
                target_idx = pattern.nodes.index(edge.target_id)

                pattern_spec["edges"].append({
                    "source_idx": source_idx,
                    "target_idx": target_idx,
                    "type": edge.type
                })
        except Exception as e:
            self.logger.warning(f"Error getting edge types for pattern: {e}")
            # Continue with node types only

        # Find matches to the pattern
        matches = self.find_patterns(
            pattern_spec=pattern_spec,
            similarity_threshold=min_similarity,
            max_results=100
        )

        # Extract instances
        instances = []
        for match in matches:
            # Skip exact same instance
            if set(match.subgraph_nodes) == set(pattern.nodes):
                continue

            instances.append(match.subgraph_nodes)

        return instances

    def analyze_pattern_evolution(self, pattern_id: str,
                               time_window: Optional[Tuple[float, float]] = None) -> Dict[str, Any]:
        """
        Analyzes how a pattern has evolved over time.

        Args:
            pattern_id: ID of the pattern to analyze
            time_window: Optional tuple of (start_time, end_time)

        Returns:
            Dictionary with evolution metrics
        """
        # Find the pattern
        pattern = None
        for p in self._frequent_patterns:
            if p.pattern_id == pattern_id:
                pattern = p
                break

        if not pattern:
            return {"error": f"Pattern {pattern_id} not found"}

        # This is a placeholder - in a real implementation, you would:
        # 1. Query the graph for historical state or snapshots
        # 2. Trace the pattern over time
        # 3. Analyze changes in structure, support, etc.

        return {
            "pattern_id": pattern_id,
            "current_support": pattern.support,
            "nodes_count": len(pattern.nodes),
            "edges_count": len(pattern.edges),
            "note": "Full pattern evolution analysis requires temporal graph data"
        }

# Utility function to create a PatternMiningModule from a GraphEngine
    def create_pattern_mining(graph_engine: AbstractGraphEngine,
                            use_networkx: bool = True,
                            embedding_dim: int = 128) -> PatternMiningModule:
        """
        Factory function to create a PatternMiningModule for a given GraphEngine.

        Args:
            graph_engine: The GraphEngine to mine patterns from
            use_networkx: Whether to use NetworkX for advanced algorithms (if available)
            embedding_dim: Dimension for pattern embeddings

        Returns:
            A configured PatternMiningModule instance
        """
        return PatternMiningModule(
            graph_engine=graph_engine,
            use_networkx=use_networkx,
            default_similarity_threshold=0.7,
            embedding_dim=embedding_dim
        )

ModuleNotFoundError: No module named 'grandnexus'

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np

# Define the number of rows
num_rows = 1000000

# Define the possible values
states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking", "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

start_date = "2022-01-01"
end_date = "2022-12-31"
# Create a date range
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Adding issue weekday based on the "Issue Date"
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

df["issue_weekday"] = df["Issue Date"].dt.weekday.map(weekday_names)

# Grouping by issue_weekday and counting the Summons Number
df.groupby(["Issue Date"])["Ticket Number"
].count().sort_values()

# Graph Pattern

In [ ]:
"""
graph_pattern_interface.py - Interface between GraphEngine and PatternMining

This module defines the integration points between the graph engine and
pattern mining modules, allowing them to work as a cohesive unit while maintaining
separation of concerns. It serves as the bridge between low-level graph operations
and high-level cognitive pattern analysis.
"""

from abc import ABC, abstractmethod
from typing import Dict, Any, List, Optional, Union, Tuple, Set
from dataclasses import dataclass, field

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)

@dataclass
class PatternMatchResult:
    """
    Represents the result of a pattern matching operation.
    """
    subgraph_nodes: List[str] = field(default_factory=list)
    subgraph_edges: List[str] = field(default_factory=list)
    similarity_score: float = 0.0
    pattern_type: Optional[str] = None
    extra_info: Dict[str, Any] = field(default_factory=dict)

    def __repr__(self):
        pattern_info = f", type={self.pattern_type}" if self.pattern_type else ""
        return (f"<PatternMatchResult nodes={len(self.subgraph_nodes)}, "
                f"edges={len(self.subgraph_edges)}, sim={self.similarity_score:.2f}{pattern_info}>")


@dataclass
class FrequentPattern:
    """
    Represents a discovered frequent pattern in the graph.
    """
    pattern_id: str
    nodes: List[str]
    edges: List[str]
    support: int
    pattern_embedding: Optional[Any] = None
    signature: str = ""

    def __repr__(self):
        return f"<FrequentPattern id={self.pattern_id}, support={self.support}, nodes={len(self.nodes)}>"


class PatternMiningInterface(ABC):
    """
    Abstract interface that defines how PatternMining should interact with GraphEngine.
    This allows for multiple implementations while maintaining consistent interfaces.
    """

    @abstractmethod
    def find_patterns(self,
                     pattern_spec: Dict[str, Any],
                     similarity_threshold: Optional[float] = None,
                     max_results: int = 10) -> List[PatternMatchResult]:
        """
        Searches the graph for subgraphs matching the given pattern specification.
        """
        pass

    @abstractmethod
    def extract_frequent_subgraphs(self,
                                  min_support: int = 2,
                                  max_size: int = 5) -> List[FrequentPattern]:
        """
        Discovers frequently occurring patterns in the graph.
        """
        pass

    @abstractmethod
    def detect_anomalies(self,
                        property_outliers: bool = True,
                        structural_anomalies: bool = True,
                        threshold: float = 2.0) -> List[Dict[str, Any]]:
        """
        Detects nodes or relationships that appear anomalous.
        """
        pass

    @abstractmethod
    def find_similar_patterns(self,
                             pattern: Union[PatternMatchResult, Dict[str, Any]],
                             top_n: int = 5) -> List[Tuple[PatternMatchResult, float]]:
        """
        Finds patterns similar to the provided pattern using embedding similarity.
        """
        pass

    @abstractmethod
    def get_pattern_statistics(self) -> Dict[str, Any]:
        """
        Returns statistics about the discovered patterns.
        """
        pass


class GraphPatternBridge:
    """
    Bridge class that connects GraphEngine with PatternMining.
    This class maintains the state needed for both modules to collaborate effectively
    while respecting GrandNexus's fractal architecture.
    """

    def __init__(self, graph_engine: AbstractGraphEngine, pattern_mining: PatternMiningInterface):
        """
        Initialize the bridge with both required components.

        Args:
            graph_engine: Instance of a GraphEngine implementation
            pattern_mining: Instance of a PatternMining implementation
        """
        self.graph_engine = graph_engine
        self.pattern_mining = pattern_mining
        self.logger = logging.getLogger("GraphPatternBridge")

    def convert_pattern_to_graphpattern(self, pattern: FrequentPattern) -> GraphPattern:
        """
        Converts a FrequentPattern from PatternMining to a GraphPattern for GraphEngine.

        Args:
            pattern: A FrequentPattern object

        Returns:
            A GraphPattern object that can be used with GraphEngine
        """
        return GraphPattern(
            pattern_id=pattern.pattern_id,
            nodes=pattern.nodes,
            edges=pattern.edges,
            frequency=pattern.support,
            significance=0.5,  # Default value, could be calculated
            embedding=pattern.pattern_embedding
        )

    def convert_graphpattern_to_pattern(self, graph_pattern: GraphPattern) -> FrequentPattern:
        """
        Converts a GraphPattern from GraphEngine to a FrequentPattern for PatternMining.

        Args:
            graph_pattern: A GraphPattern object

        Returns:
            A FrequentPattern object that can be used with PatternMining
        """
        return FrequentPattern(
            pattern_id=graph_pattern.pattern_id,
            nodes=graph_pattern.nodes,
            edges=graph_pattern.edges,
            support=graph_pattern.frequency,
            pattern_embedding=graph_pattern.embedding,
            signature=""  # Would need to be computed
        )

    def store_discovered_patterns(self, patterns: List[FrequentPattern]) -> None:
        """
        Stores discovered patterns from PatternMining into the GraphEngine.
        This makes the patterns part of the graph itself, enabling meta-reasoning.

        Args:
            patterns: List of discovered FrequentPattern objects
        """
        for pattern in patterns:
            # Convert to GraphPattern
            graph_pattern = self.convert_pattern_to_graphpattern(pattern)

            # Create a special node for the pattern
            pattern_node_id = f"pattern_{pattern.pattern_id}"
            properties = {
                "type": "pattern",
                "support": pattern.support,
                "signature": pattern.signature,
            }

            try:
                # Add the pattern node
                self.graph_engine.add_node(node_id=pattern_node_id,
                                          label="Pattern",
                                          properties=properties)

                # Link the pattern node to all nodes in the pattern
                for node_id in pattern.nodes:
                    self.graph_engine.add_edge(
                        source_id=pattern_node_id,
                        target_id=node_id,
                        edge_type="PATTERN_CONTAINS",
                        properties={"pattern_id": pattern.pattern_id}
                    )
            except Exception as e:
                self.logger.error(f"Failed to store pattern {pattern.pattern_id}: {e}")

    def apply_pattern_activation(self, pattern: FrequentPattern, activation: float = 0.8) -> Dict[str, float]:
        """
        Activates all nodes in a pattern, allowing for pattern-based spreading activation.

        Args:
            pattern: The pattern whose nodes should be activated
            activation: Initial activation value for the pattern nodes

        Returns:
            Dictionary mapping node IDs to their activation values
        """
        # First, activate all nodes in the pattern
        for node_id in pattern.nodes:
            try:
                node = self.graph_engine.get_node(node_id)
                node.activation = activation
            except NodeNotFoundError:
                self.logger.warning(f"Node {node_id} from pattern {pattern.pattern_id} not found")

        # Then use spreading activation to propagate
        return self.graph_engine.spread_activation(
            source_nodes=pattern.nodes,
            decay_factor=0.5,
            max_depth=2
        )

    def get_pattern_subgraph(self, pattern_id: str) -> Tuple[List[GraphNode], List[GraphEdge]]:
        """
        Retrieves all nodes and edges that are part of a specific pattern.

        Args:
            pattern_id: ID of the pattern to retrieve

        Returns:
            Tuple of (nodes, edges) lists representing the pattern subgraph
        """
        pattern_node_id = f"pattern_{pattern_id}"

        try:
            # Find all nodes connected to the pattern node
            neighbors = self.graph_engine.get_node_neighbors(
                node_id=pattern_node_id,
                edge_types=["PATTERN_CONTAINS"]
            )

            node_ids = [neighbor_id for neighbor_id, _, _ in neighbors]
            nodes = [self.graph_engine.get_node(node_id) for node_id in node_ids]

            # Find all edges between these nodes
            edges = []
            for i, src_id in enumerate(node_ids):
                for dst_id in node_ids[i+1:]:
                    # Check for edges in both directions
                    out_neighbors = self.graph_engine.get_node_neighbors(
                        node_id=src_id,
                        direction="outgoing"
                    )
                    for neighbor_id, edge_id, _ in out_neighbors:
                        if neighbor_id == dst_id:
                            edges.append(self.graph_engine.get_edge(edge_id))

                    in_neighbors = self.graph_engine.get_node_neighbors(
                        node_id=src_id,
                        direction="incoming"
                    )
                    for neighbor_id, edge_id, _ in in_neighbors:
                        if neighbor_id == dst_id:
                            edges.append(self.graph_engine.get_edge(edge_id))

            return nodes, edges

        except NodeNotFoundError:
            self.logger.warning(f"Pattern node {pattern_node_id} not found")
            return [], []

    def register_with_nexus_core(self, nexus_core):
        """
        Registers this bridge with NexusCore to enable communication with other modules.

        Args:
            nexus_core: The NexusCore instance
        """
        # Register callbacks with NexusCore for handling pattern-related events
        nexus_core.register_handler("graph.pattern_discovered", self._handle_pattern_discovered)
        nexus_core.register_handler("memory.concept_stored", self._handle_concept_stored)
        nexus_core.register_handler("reasoning.relation_inferred", self._handle_relation_inferred)

    def _handle_pattern_discovered(self, event_data: Dict[str, Any]) -> None:
        """Handler for pattern discovery events"""
        # Implementation would dispatch to appropriate methods
        pass

    def _handle_concept_stored(self, event_data: Dict[str, Any]) -> None:
        """Handler for memory concept storage events"""
        # Check if new concept forms patterns with existing concepts
        pass

    def _handle_relation_inferred(self, event_data: Dict[str, Any]) -> None:
        """Handler for reasoning relation inference events"""
        # Check if new relation completes or modifies known patterns
        pass


# Import here to avoid circular imports
import logging

ModuleNotFoundError: No module named 'grandnexus'

# Graph Module

In [ ]:
"""
graph_module_manager.py - Integration point for Graph and PatternMining modules with NexusCore

This module serves as the central manager for all graph-related operations in GrandNexus.
It coordinates the GraphEngine and PatternMining modules, ensuring they work as a unified
system while maintaining clean separation of concerns.
"""

import logging
import traceback
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Type
import time

from grandnexus.core.nexus_core import NexusCore
from grandnexus.core.registry import ModuleRegistry

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError,
    create_graph_engine
)
from grandnexus.graph.pattern_mining import (
    PatternMiningModule, PatternMatchResult, FrequentPattern,
    create_pattern_mining
)
from grandnexus.graph.graph_pattern_interface import GraphPatternBridge


class GraphModuleManager:
    """
    Central manager for all graph-related functionality in GrandNexus.

    This class:
    1. Initializes and manages both GraphEngine and PatternMining
    2. Connects them to NexusCore for system-wide integration
    3. Provides a unified API for graph operations
    4. Handles messaging, events, and cross-module integration
    """

    def __init__(self, nexus_core: NexusCore, config: Dict[str, Any]):
        """
        Initialize the GraphModuleManager.

        Args:
            nexus_core: The NexusCore instance for system integration
            config: Configuration dictionary with graph settings
        """
        self.logger = logging.getLogger("GraphModuleManager")
        self.nexus_core = nexus_core
        self.config = config

        # Extract configuration
        graph_engine_type = config.get('graph_engine_type', 'inmemory')
        graph_engine_config = config.get('graph_engine_config', {})
        use_pattern_mining = config.get('use_pattern_mining', True)
        use_networkx = config.get('use_networkx', True)
        embedding_dim = config.get('embedding_dim', 128)

        # Initialize the GraphEngine
        self.logger.info(f"Initializing GraphEngine of type '{graph_engine_type}'")
        self.graph_engine = create_graph_engine(graph_engine_type, **graph_engine_config)

        # Initialize PatternMining if enabled
        self.pattern_mining = None
        if use_pattern_mining:
            self.logger.info("Initializing PatternMining module")
            self.pattern_mining = create_pattern_mining(
                graph_engine=self.graph_engine,
                use_networkx=use_networkx,
                embedding_dim=embedding_dim
            )

            # Create the bridge between GraphEngine and PatternMining
            self.bridge = GraphPatternBridge(
                graph_engine=self.graph_engine,
                pattern_mining=self.pattern_mining
            )
        else:
            self.logger.info("PatternMining module disabled")
            self.bridge = None

        # Register with NexusCore
        self._register_with_nexus_core()

        self.logger.info("GraphModuleManager initialized successfully")

    def _register_with_nexus_core(self):
        """Register handlers and callbacks with NexusCore."""
        # Register message handlers
        self.nexus_core.register_handler("graph.add_node", self._handle_add_node)
        self.nexus_core.register_handler("graph.get_node", self._handle_get_node)
        self.nexus_core.register_handler("graph.update_node", self._handle_update_node)
        self.nexus_core.register_handler("graph.delete_node", self._handle_delete_node)
        self.nexus_core.register_handler("graph.add_edge", self._handle_add_edge)
        self.nexus_core.register_handler("graph.get_edge", self._handle_get_edge)
        self.nexus_core.register_handler("graph.update_edge", self._handle_update_edge)
        self.nexus_core.register_handler("graph.delete_edge", self._handle_delete_edge)
        self.nexus_core.register_handler("graph.query", self._handle_query)
        self.nexus_core.register_handler("graph.activate", self._handle_activate)

        # Pattern mining handlers (if enabled)
        if self.pattern_mining:
            self.nexus_core.register_handler("graph.find_patterns", self._handle_find_patterns)
            self.nexus_core.register_handler("graph.extract_patterns", self._handle_extract_patterns)
            self.nexus_core.register_handler("graph.detect_anomalies", self._handle_detect_anomalies)
            self.nexus_core.register_handler("graph.similar_patterns", self._handle_similar_patterns)

        # Register for memory events
        self.nexus_core.register_handler("memory.concept_stored", self._handle_concept_stored)
        self.nexus_core.register_handler("memory.episodic_created", self._handle_episodic_created)

        # Register with ModuleRegistry
        ModuleRegistry.register("graph", self)

    # --------------------------------------------------------------------------
    # Core GraphEngine API handlers
    # --------------------------------------------------------------------------

    def _handle_add_node(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle add_node request."""
        try:
            node_id = self.graph_engine.add_node(
                node_id=message.get('node_id'),
                label=message.get('label', 'Node'),
                properties=message.get('properties', {})
            )

            # Publish event for other modules
            self.nexus_core.publish_event("graph.node_added", {
                "node_id": node_id,
                "label": message.get('label', 'Node'),
                "source_module": message.get('source_module')
            })

            return {"success": True, "node_id": node_id}
        except Exception as e:
            self.logger.error(f"Error adding node: {e}")
            traceback.print_exc()
            return {"success": False, "error": str(e)}

    def _handle_get_node(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle get_node request."""
        try:
            node_id = message.get('node_id')
            if not node_id:
                return {"success": False, "error": "No node_id provided"}

            node = self.graph_engine.get_node(node_id)

            # Convert to serializable format
            node_data = {
                "node_id": node.node_id,
                "label": node.label,
                "properties": node.properties,
                "activation": node.activation,
                "importance": node.importance,
                "last_accessed": node.last_accessed,
                "creation_time": node.creation_time
            }

            return {"success": True, "node": node_data}
        except NodeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error getting node: {e}")
            return {"success": False, "error": str(e)}

    def _handle_update_node(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle update_node request."""
        try:
            node_id = message.get('node_id')
            properties = message.get('properties', {})

            if not node_id:
                return {"success": False, "error": "No node_id provided"}

            self.graph_engine.update_node(node_id, properties)

            # Publish event for other modules
            self.nexus_core.publish_event("graph.node_updated", {
                "node_id": node_id,
                "properties": properties,
                "source_module": message.get('source_module')
            })

            return {"success": True}
        except NodeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error updating node: {e}")
            return {"success": False, "error": str(e)}

    def _handle_delete_node(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle delete_node request."""
        try:
            node_id = message.get('node_id')
            if not node_id:
                return {"success": False, "error": "No node_id provided"}

            self.graph_engine.delete_node(node_id)

            # Publish event for other modules
            self.nexus_core.publish_event("graph.node_deleted", {
                "node_id": node_id,
                "source_module": message.get('source_module')
            })

            return {"success": True}
        except NodeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error deleting node: {e}")
            return {"success": False, "error": str(e)}

    def _handle_add_edge(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle add_edge request."""
        try:
            source_id = message.get('source_id')
            target_id = message.get('target_id')
            edge_type = message.get('edge_type', 'RELATES')

            if not source_id or not target_id:
                return {"success": False, "error": "Source and target IDs required"}

            edge_id = self.graph_engine.add_edge(
                source_id=source_id,
                target_id=target_id,
                edge_type=edge_type,
                edge_id=message.get('edge_id'),
                properties=message.get('properties', {}),
                weight=message.get('weight', 1.0),
                bidirectional=message.get('bidirectional', False)
            )

            # Publish event for other modules
            self.nexus_core.publish_event("graph.edge_added", {
                "edge_id": edge_id,
                "source_id": source_id,
                "target_id": target_id,
                "edge_type": edge_type,
                "source_module": message.get('source_module')
            })

            return {"success": True, "edge_id": edge_id}
        except NodeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error adding edge: {e}")
            return {"success": False, "error": str(e)}

    def _handle_get_edge(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle get_edge request."""
        try:
            edge_id = message.get('edge_id')
            if not edge_id:
                return {"success": False, "error": "No edge_id provided"}

            edge = self.graph_engine.get_edge(edge_id)

            # Convert to serializable format
            edge_data = {
                "edge_id": edge.edge_id,
                "source_id": edge.source_id,
                "target_id": edge.target_id,
                "type": edge.type,
                "properties": edge.properties,
                "weight": edge.weight,
                "bidirectional": edge.bidirectional,
                "creation_time": edge.creation_time
            }

            return {"success": True, "edge": edge_data}
        except EdgeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error getting edge: {e}")
            return {"success": False, "error": str(e)}

    def _handle_update_edge(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle update_edge request."""
        try:
            edge_id = message.get('edge_id')
            properties = message.get('properties', {})

            if not edge_id:
                return {"success": False, "error": "No edge_id provided"}

            self.graph_engine.update_edge(edge_id, properties)

            # Publish event for other modules
            self.nexus_core.publish_event("graph.edge_updated", {
                "edge_id": edge_id,
                "properties": properties,
                "source_module": message.get('source_module')
            })

            return {"success": True}
        except EdgeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error updating edge: {e}")
            return {"success": False, "error": str(e)}

    def _handle_delete_edge(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle delete_edge request."""
        try:
            edge_id = message.get('edge_id')
            if not edge_id:
                return {"success": False, "error": "No edge_id provided"}

            self.graph_engine.delete_edge(edge_id)

            # Publish event for other modules
            self.nexus_core.publish_event("graph.edge_deleted", {
                "edge_id": edge_id,
                "source_module": message.get('source_module')
            })

            return {"success": True}
        except EdgeNotFoundError as e:
            return {"success": False, "error": str(e), "not_found": True}
        except Exception as e:
            self.logger.error(f"Error deleting edge: {e}")
            return {"success": False, "error": str(e)}

    def _handle_query(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle query request."""
        try:
            # Convert message data to GraphQuery object
            query_data = message.get('query', {})
            query = GraphQuery(
                node_criteria=query_data.get('node_criteria', {}),
                edge_criteria=query_data.get('edge_criteria', {}),
                start_nodes=query_data.get('start_nodes'),
                max_depth=query_data.get('max_depth', 3),
                max_results=query_data.get('max_results', 100),
                include_inactive=query_data.get('include_inactive', False),
                min_activation=query_data.get('min_activation', 0.0),
                sort_by=query_data.get('sort_by', "activation")
            )

            # Execute query
            result = self.graph_engine.execute_query(query)

            return {"success": True, "result": result}
        except Exception as e:
            self.logger.error(f"Error executing query: {e}")
            return {"success": False, "error": str(e)}

    def _handle_activate(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle activate (spread activation) request."""
        try:
            source_nodes = message.get('source_nodes', [])
            decay_factor = message.get('decay_factor', 0.5)
            max_depth = message.get('max_depth', 3)

            if not source_nodes:
                return {"success": False, "error": "No source_nodes provided"}

            # Spread activation
            result = self.graph_engine.spread_activation(
                source_nodes=source_nodes,
                decay_factor=decay_factor,
                max_depth=max_depth
            )

            # Publish event for other modules
            self.nexus_core.publish_event("graph.activation_spread", {
                "source_nodes": source_nodes,
                "activated_nodes": list(result.keys()),
                "source_module": message.get('source_module')
            })

            return {"success": True, "activations": result}
        except Exception as e:
            self.logger.error(f"Error spreading activation: {e}")
            return {"success": False, "error": str(e)}

    # --------------------------------------------------------------------------
    # Pattern Mining API handlers
    # --------------------------------------------------------------------------

    def _handle_find_patterns(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle find_patterns request."""
        if not self.pattern_mining:
            return {"success": False, "error": "Pattern mining not enabled"}

        try:
            pattern_spec = message.get('pattern_spec', {})
            similarity_threshold = message.get('similarity_threshold')
            max_results = message.get('max_results', 10)

            if not pattern_spec:
                return {"success": False, "error": "No pattern_spec provided"}

            # Find patterns
            patterns = self.pattern_mining.find_patterns(
                pattern_spec=pattern_spec,
                similarity_threshold=similarity_threshold,
                max_results=max_results
            )

            # Convert to serializable format
            pattern_results = []
            for p in patterns:
                pattern_results.append({
                    "nodes": p.subgraph_nodes,
                    "edges": p.subgraph_edges,
                    "similarity": p.similarity_score,
                    "pattern_type": p.pattern_type,
                    "extra_info": p.extra_info
                })

            # Publish event for other modules
            if pattern_results:
                self.nexus_core.publish_event("graph.patterns_found", {
                    "pattern_count": len(pattern_results),
                    "pattern_types": list(set(p.get('pattern_type') for p in pattern_results if p.get('pattern_type'))),
                    "source_module": message.get('source_module')
                })

            return {"success": True, "patterns": pattern_results}
        except Exception as e:
            self.logger.error(f"Error finding patterns: {e}")
            return {"success": False, "error": str(e)}

    def _handle_extract_patterns(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle extract_patterns (frequent pattern mining) request."""
        if not self.pattern_mining:
            return {"success": False, "error": "Pattern mining not enabled"}

        try:
            min_support = message.get('min_support', 2)
            max_size = message.get('max_size', 5)
            store_embeddings = message.get('store_embeddings', True)

            # Extract patterns
            patterns = self.pattern_mining.extract_frequent_subgraphs(
                min_support=min_support,
                max_size=max_size,
                store_embeddings=store_embeddings
            )

            # Convert to serializable format
            pattern_results = []
            for p in patterns:
                pattern_results.append({
                    "pattern_id": p.pattern_id,
                    "nodes": p.nodes,
                    "edges": p.edges,
                    "support": p.support,
                    "signature": p.signature
                })

            # Publish event for other modules
            if pattern_results:
                self.nexus_core.publish_event("graph.patterns_extracted", {
                    "pattern_count": len(pattern_results),
                    "source_module": message.get('source_module')
                })

                # If requested, store patterns in graph
                if message.get('store_in_graph', False) and self.bridge:
                    self.bridge.store_discovered_patterns(patterns)

            return {"success": True, "patterns": pattern_results}
        except Exception as e:
            self.logger.error(f"Error extracting patterns: {e}")
            return {"success": False, "error": str(e)}

    def _handle_detect_anomalies(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle detect_anomalies request."""
        if not self.pattern_mining:
            return {"success": False, "error": "Pattern mining not enabled"}

        try:
            property_outliers = message.get('property_outliers', True)
            structural_anomalies = message.get('structural_anomalies', True)
            threshold = message.get('threshold', 2.0)

            # Detect anomalies
            anomalies = self.pattern_mining.detect_anomalies(
                property_outliers=property_outliers,
                structural_anomalies=structural_anomalies,
                threshold=threshold
            )

            # Publish event for other modules
            if anomalies:
                self.nexus_core.publish_event("graph.anomalies_detected", {
                    "anomaly_count": len(anomalies),
                    "source_module": message.get('source_module')
                })

            return {"success": True, "anomalies": anomalies}
        except Exception as e:
            self.logger.error(f"Error detecting anomalies: {e}")
            return {"success": False, "error": str(e)}

    def _handle_similar_patterns(self, message: Dict[str, Any]) -> Dict[str, Any]:
        """Handle similar_patterns request."""
        if not self.pattern_mining:
            return {"success": False, "error": "Pattern mining not enabled"}

        try:
            pattern = message.get('pattern')
            top_n = message.get('top_n', 5)

            if not pattern:
                return {"success": False, "error": "No pattern provided"}

            # Find similar patterns
            similar = self.pattern_mining.find_similar_patterns(
                pattern=pattern,
                top_n=top_n
            )

            # Convert to serializable format
            similar_results = []
            for pattern, similarity in similar:
                similar_results.append({
                    "nodes": pattern.subgraph_nodes,
                    "edges": pattern.subgraph_edges,
                    "similarity": similarity,
                    "pattern_type": pattern.pattern_type
                })

            return {"success": True, "similar_patterns": similar_results}
        except Exception as e:
            self.logger.error(f"Error finding similar patterns: {e}")
            return {"success": False, "error": str(e)}

    # --------------------------------------------------------------------------
    # Event handlers for integration with other modules
    # --------------------------------------------------------------------------

    def _handle_concept_stored(self, event_data: Dict[str, Any]) -> None:
        """Handle memory.concept_stored events."""
        # When a concept is stored in memory, we can automatically add it to the graph
        concept_id = event_data.get('concept_id')
        concept_type = event_data.get('concept_type', 'Concept')
        properties = event_data.get('properties', {})

        if not concept_id:
            return

        try:
            # Check if node already exists
            try:
                self.graph_engine.get_node(concept_id)
                # Node exists, update it
                self.graph_engine.update_node(concept_id, properties)
            except NodeNotFoundError:
                # Node doesn't exist, create it
                self.graph_engine.add_node(
                    node_id=concept_id,
                    label=concept_type,
                    properties=properties
                )

            # Connect to related concepts if specified
            related_concepts = event_data.get('related_concepts', [])
            for relation in related_concepts:
                related_id = relation.get('concept_id')
                relation_type = relation.get('relation_type', 'RELATED_TO')

                if related_id:
                    try:
                        self.graph_engine.add_edge(
                            source_id=concept_id,
                            target_id=related_id,
                            edge_type=relation_type
                        )
                    except NodeNotFoundError:
                        # Related node doesn't exist yet, create it
                        self.graph_engine.add_node(
                            node_id=related_id,
                            label='Concept'
                        )
                        self.graph_engine.add_edge(
                            source_id=concept_id,
                            target_id=related_id,
                            edge_type=relation_type
                        )

            # If pattern mining is enabled, check for new patterns
            if self.pattern_mining and self.config.get('auto_mine_patterns', False):
                # Schedule a task to extract patterns
                self.nexus_core.schedule_task(
                    task_id=f"extract_patterns_{time.time()}",
                    handler="graph.extract_patterns",
                    data={
                        "min_support": 2,
                        "max_size": 5,
                        "store_in_graph": True
                    },
                    delay=5.0  # Wait 5 seconds to batch multiple additions
                )
        except Exception as e:
            self.logger.error(f"Error handling concept_stored event: {e}")

    def _handle_episodic_created(self, event_data: Dict[str, Any]) -> None:
        """Handle memory.episodic_created events."""
        # When an episodic memory is created, we can represent it in the graph
        episode_id = event_data.get('episode_id')
        episode_type = event_data.get('episode_type', 'Episode')
        entities = event_data.get('entities', [])

        if not episode_id:
            return

        try:
            # Create episode node
            try:
                self.graph_engine.add_node(
                    node_id=episode_id,
                    label=episode_type,
                    properties=event_data.get('properties', {})
                )
            except Exception:
                # Node might already exist
                pass

            # Connect to entities involved in the episode
            for entity in entities:
                entity_id = entity.get('entity_id')
                role = entity.get('role', 'INVOLVED_IN')

                if entity_id:
                    try:
                        # Create relationship: Entity -[INVOLVED_IN]-> Episode
                        self.graph_engine.add_edge(
                            source_id=entity_id,
                            target_id=episode_id,
                            edge_type=role
                        )
                    except NodeNotFoundError:
                        # Entity doesn't exist yet, create it
                        entity_type = entity.get('entity_type', 'Entity')
                        self.graph_engine.add_node(
                            node_id=entity_id,
                            label=entity_type
                        )
                        self.graph_engine.add_edge(
                            source_id=entity_id,
                            target_id=episode_id,
                            edge_type=role
                        )
        except Exception as e:
            self.logger.error(f"Error handling episodic_created event: {e}")

    # --------------------------------------------------------------------------
    # Public API for direct use by other Python modules
    # --------------------------------------------------------------------------

    def add_node(self, node_id: Optional[str] = None, label: str = "Node",
                properties: Optional[Dict[str, Any]] = None) -> str:
        """
        Add a node to the graph.

        Args:
            node_id: Optional node ID (generated if None)
            label: Node type/label
            properties: Node properties

        Returns:
            The ID of the node
        """
        return self.graph_engine.add_node(node_id, label, properties)

    def add_edge(self, source_id: str, target_id: str, edge_type: str,
                edge_id: Optional[str] = None, properties: Optional[Dict[str, Any]] = None,
                weight: float = 1.0, bidirectional: bool = False) -> str:
        """
        Add an edge between nodes.

        Args:
            source_id: Source node ID
            target_id: Target node ID
            edge_type: Type of relationship
            edge_id: Optional edge ID (generated if None)
            properties: Edge properties
            weight: Edge weight
            bidirectional: Whether the edge is bidirectional

        Returns:
            The ID of the edge
        """
        return self.graph_engine.add_edge(
            source_id, target_id, edge_type,
            edge_id, properties, weight, bidirectional
        )

    def query(self, **kwargs) -> Dict[str, Any]:
        """
        Execute a graph query.

        Args:
            **kwargs: Query parameters (node_criteria, edge_criteria, etc.)

        Returns:
            Query results
        """
        query = GraphQuery(**kwargs)
        return self.graph_engine.execute_query(query)

    def find_patterns(self, pattern_spec: Dict[str, Any], **kwargs) -> List[PatternMatchResult]:
        """
        Find patterns in the graph.

        Args:
            pattern_spec: Pattern specification
            **kwargs: Additional parameters

        Returns:
            List of pattern matches
        """
        if not self.pattern_mining:
            raise ValueError("Pattern mining is not enabled")

        return self.pattern_mining.find_patterns(pattern_spec, **kwargs)

    def spread_activation(self, source_nodes: List[str], **kwargs) -> Dict[str, float]:
        """
        Spread activation from source nodes.

        Args:
            source_nodes: Starting nodes for activation
            **kwargs: Additional parameters

        Returns:
            Dictionary of node activations
        """
        return self.graph_engine.spread_activation(source_nodes, **kwargs)

    def detect_anomalies(self, **kwargs) -> List[Dict[str, Any]]:
        """
        Detect anomalies in the graph.

        Args:
            **kwargs: Detection parameters

        Returns:
            List of anomalies
        """
        if not self.pattern_mining:
            raise ValueError("Pattern mining is not enabled")

        return self.pattern_mining.detect_anomalies(**kwargs)

    def create_subgraph_from_query(self, query: Dict[str, Any]) -> Tuple[List[str], List[str]]:
        """
        Create a subgraph based on a query.

        Args:
            query: Query specification

        Returns:
            Tuple of (node_ids, edge_ids)
        """
        result = self.query(**query)

        node_ids = [node['id'] for node in result.get('nodes', [])]
        edge_ids = [edge['id'] for edge in result.get('edges', [])]

        return node_ids, edge_ids

    # --------------------------------------------------------------------------
    # Maintenance and management methods
    # --------------------------------------------------------------------------

    def get_statistics(self) -> Dict[str, Any]:
        """
        Get statistics about the graph and pattern mining modules.

        Returns:
            Dictionary of statistics
        """
        stats = {
            "timestamp": time.time(),
            "graph": {}
        }

        # Get basic graph statistics
        try:
            node_query = GraphQuery(max_results=1)
            result = self.graph_engine.execute_query(node_query)
            stats["graph"]["node_count"] = len(result.get('nodes', []))
            stats["graph"]["edge_count"] = len(result.get('edges', []))

            # Get node types distribution
            node_types = {}
            for node in result.get('nodes', []):
                label = node.get('label', 'Unknown')
                node_types[label] = node_types.get(label, 0) + 1
            stats["graph"]["node_types"] = node_types

        except Exception as e:
            self.logger.error(f"Error getting graph statistics: {e}")
            stats["graph"]["error"] = str(e)

        # Get pattern mining statistics if enabled
        if self.pattern_mining:
            try:
                pattern_stats = self.pattern_mining.get_pattern_statistics()
                stats["pattern_mining"] = pattern_stats
            except Exception as e:
                self.logger.error(f"Error getting pattern mining statistics: {e}")
                stats["pattern_mining"] = {"error": str(e)}

        return stats

    def clear_caches(self) -> None:
        """Clear all caches to free memory."""
        try:
            if self.pattern_mining:
                self.pattern_mining.clear_caches()
                if hasattr(self.pattern_mining, "rebuild_networkx_graph"):
                    self.pattern_mining.rebuild_networkx_graph()

            self.logger.info("All caches cleared")
        except Exception as e:
            self.logger.error(f"Error clearing caches: {e}")
            raise

    def export_graph(self, format: str = "json") -> Dict[str, Any]:
        """
        Export the graph in the specified format.

        Args:
            format: Export format (currently only 'json' is supported)

        Returns:
            Exported graph data
        """
        if format.lower() != "json":
            raise ValueError(f"Unsupported export format: {format}")

        try:
            # Use the graph engine's serialization method
            return self.graph_engine.serialize()
        except Exception as e:
            self.logger.error(f"Error exporting graph: {e}")
            raise

    def import_graph(self, data: Dict[str, Any]) -> None:
        """
        Import graph data.

        Args:
            data: Graph data to import
        """
        try:
            # Use the graph engine's deserialization method
            self.graph_engine.deserialize(data)

            # Rebuild pattern mining if needed
            if self.pattern_mining and hasattr(self.pattern_mining, "rebuild_networkx_graph"):
                self.pattern_mining.rebuild_networkx_graph()

            self.logger.info("Graph imported successfully")
        except Exception as e:
            self.logger.error(f"Error importing graph: {e}")
            raise

    def shutdown(self) -> None:
        """Close and clean up resources."""
        try:
            # Close the graph engine
            if hasattr(self.graph_engine, "close"):
                self.graph_engine.close()

            self.logger.info("GraphModuleManager shut down")
        except Exception as e:
            self.logger.error(f"Error during shutdown: {e}")



ModuleNotFoundError: No module named 'grandnexus'

# Resource Manager 1

In [ ]:
"""
resource_manager.py - Advanced Resource & Scheduling System for GrandNexus

Key Features:
    1) Multi-node / HPC integration hooks:
       - Optional stubs for connecting to orchestration frameworks (e.g., Kubernetes, Ray, SLURM).
    2) Extended scheduling policies:
       - Priority-based, FIFO, Deadline-based, or ML-based (placeholders for integrating a
         learning model or offline RL approach).
    3) Deadlines, timeouts, and max waiting times:
       - Each TaskRequest can specify a deadline or max_waiting_time. The scheduler attempts
         to respect these constraints (Earliest-Deadline-First logic).
    4) Enhanced DRAS (Dynamic Resource Allocation Strategy):
       - Per-module resource partitions that can be updated dynamically.
       - Optionally load a learned model to adjust resource shares based on usage metrics.
    5) Anomaly detection stubs:
       - Hooks for detecting out-of-range CPU or memory usage that suggests a module is
         misbehaving or in a spinlock.
    6) Comprehensive logging, docstrings, type hints, error handling, and concurrency controls.
    7) Task interruption/cancellation:
       - More detailed placeholders for stopping tasks that are in progress, possibly across nodes.
    8) HPC/Cluster references:
       - If cluster_mode is True in the ResourceManagerConfig, one can integrate with real cluster
         managers (not implemented here, but the stubs are in place).
    9) ML-based scheduling placeholders:
       - A minimal “policy_model” interface is used to demonstrate how a learned model might
         advise on scheduling decisions.
    10) Preservation of advanced structures from previous versions (heapq, data classes, etc.)

"""

import logging
import psutil
import time
import queue
import threading
import heapq
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import (
    Any, Dict, List, Optional, Callable, Union, Deque, Tuple, Set
)


# ---------------------------------------------------------------------------------
# DATA CLASSES
# ---------------------------------------------------------------------------------

@dataclass(order=True)
class TaskRequest:
    """
    Represents a task request. The 'order=True' property enables comparison
    based on (priority, creation_time) to support heapq-based scheduling.

    Attributes:
        priority (int): Lower number = higher scheduling priority (if using priority-based).
        creation_time (float): Timestamp used for FIFO tie-break or deadline-based logic.
        task_id (str): Unique ID. By default, it's a UUID4 string.
        module_name (str): The module or origin that submitted this task.
        required_resources (Dict[str, float]): E.g. {"cpu":0.2, "memory":512} meaning 20% CPU, 512MB memory.
        callback (Callable): The function to execute for the task.
        callback_args (Tuple[Any, ...]): Positional arguments for callback.
        callback_kwargs (Dict[str, Any]): Keyword arguments for callback.
        max_runtime (Optional[float]): The maximum allowed runtime (seconds).
        retries (int): How many times the task can be retried if it fails.
        retry_delay (float): The delay (seconds) between retries.
        state (str): "pending", "running", "completed", "failed", "cancelled", etc.
        deadline (Optional[float]): A timestamp or None. If set, the scheduler might use EDF logic.
        max_waiting_time (Optional[float]): If the task waits in the queue too long, it can time out.
    """
    priority: int
    creation_time: float = field(default_factory=time.time, compare=True)
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()), compare=False)
    module_name: str = field(compare=False)
    required_resources: Dict[str, float] = field(default_factory=dict, compare=False)
    callback: Callable = field(compare=False)
    callback_args: Tuple[Any, ...] = field(default_factory=tuple, compare=False)
    callback_kwargs: Dict[str, Any] = field(default_factory=dict, compare=False)
    max_runtime: Optional[float] = field(default=None, compare=False)
    retries: int = field(default=0, compare=False)
    retry_delay: float = field(default=1.0, compare=False)
    state: str = field(default='pending', compare=False)
    deadline: Optional[float] = field(default=None, compare=False)
    max_waiting_time: Optional[float] = field(default=None, compare=False)

    def __repr__(self) -> str:
        return (f"<TaskRequest id={self.task_id[-8:]}, module={self.module_name}, "
                f"priority={self.priority}, state={self.state}>")

@dataclass
class ResourceUsageSnapshot:
    """
    Captures system or cluster resource usage at a specific time.

    Attributes:
        timestamp (float): The time the snapshot was taken.
        cpu_percent (float): Overall CPU usage.
        mem_percent (float): Overall memory usage.
        available_memory (float): Bytes of memory available.
        used_memory (float): Bytes of memory in use.
        process_usage (Dict[str, float]): {module: CPU usage} or {pid: usage}, up to the user approach.
        per_core_cpu_percent (List[float]): Detailed usage per CPU core.
        anomaly_score (float): A placeholder for anomaly detection logic (0=normal, >1=very anomalous).
    """
    timestamp: float = field(default_factory=time.time)
    cpu_percent: float = 0.0
    mem_percent: float = 0.0
    available_memory: float = 0.0
    used_memory: float = 0.0
    process_usage: Dict[str, float] = field(default_factory=dict)
    per_core_cpu_percent: List[float] = field(default_factory=list)
    anomaly_score: float = 0.0

    def __repr__(self) -> str:
        return (f"<ResourceUsageSnapshot time={self.timestamp:.1f}, CPU={self.cpu_percent:.1f}%, "
                f"MEM={self.mem_percent:.1f}%, anomaly={self.anomaly_score:.2f}>")

@dataclass
class ResourceManagerConfig:
    """
    Configuration object for the ResourceManager.

    Attributes:
        monitor_interval (float): Frequency of resource monitoring, in seconds.
        scheduler_interval (float): Frequency of scheduling loop, in seconds.
        cpu_soft_limit (float): Soft CPU usage limit (percentage).
        cpu_hard_limit (float): Hard CPU usage limit (percentage).
        mem_soft_limit (float): Soft memory usage limit (percentage).
        mem_hard_limit (float): Hard memory usage limit (percentage).
        max_task_queue_size (int): Max number of tasks in the queue before rejecting new ones.
        history_size (int): Max number of ResourceUsageSnapshots to keep.
        scheduling_policy (str): Could be "priority", "fifo", "deadline", or "ml".
        dynamic_resource_allocation (bool): If True, we track per-module resource partitions (DRAS).
        cluster_mode (bool): If True, we assume multi-node HPC context (stubs for integration with HPC managers).
        anomaly_detection (bool): If True, we attempt to compute an anomaly_score on each snapshot (stub).
        ml_policy_enabled (bool): If True, we attempt to consult an ML-based scheduling policy.
    """
    monitor_interval: float = 2.0
    scheduler_interval: float = 1.0
    cpu_soft_limit: float = 80.0
    cpu_hard_limit: float = 95.0
    mem_soft_limit: float = 80.0
    mem_hard_limit: float = 95.0
    max_task_queue_size: int = 200
    history_size: int = 100
    scheduling_policy: str = "priority"
    dynamic_resource_allocation: bool = True
    cluster_mode: bool = False
    anomaly_detection: bool = False
    ml_policy_enabled: bool = False


# ---------------------------------------------------------------------------------
# ML POLICY INTERFACE (Stub)
# ---------------------------------------------------------------------------------

class SchedulingPolicyModel:
    """
    A placeholder interface for an ML-based scheduling policy. A real model
    might implement .predict() to decide the next task to schedule or define
    a recommended 'priority' override.
    """

    def __init__(self, model_path: str = ""):
        """
        model_path: location of a trained ML or RL model. Implementation omitted.
        """
        self.model_path = model_path
        # load model from disk or initialize in memory, etc.

    def advise_task_start(self,
                          task: TaskRequest,
                          snapshot: ResourceUsageSnapshot) -> bool:
        """
        Decide if we can start 'task' given the system snapshot. Return True if yes,
        or False if no, or implement a more complex numeric score.
        """
        # Placeholder: Always return True. Real logic would incorporate e.g. neural net or RL.
        return True

    def advise_task_priority(self, task: TaskRequest) -> int:
        """
        Possibly override the task's priority. For demonstration, we do not override.
        """
        return task.priority


# ---------------------------------------------------------------------------------
# RESOURCE MANAGER CLASS
# ---------------------------------------------------------------------------------

class ResourceManager:
    """
    The advanced ResourceManager for GrandNexus, featuring HPC integration stubs,
    advanced scheduling logic, DRAS, anomaly detection hooks, and ML-based policy stubs.
    """

    def __init__(self, config: ResourceManagerConfig,
                 nexus_core: Optional[Any] = None,
                 ml_policy_model: Optional[SchedulingPolicyModel] = None):
        """
        Args:
            config: ResourceManagerConfig with intervals, limits, etc.
            nexus_core: Optional reference to the GrandNexus core or registry.
            ml_policy_model: If ml_policy_enabled, we can pass a loaded model here.
        """
        self.logger = logging.getLogger("ResourceManager")
        self.config = config
        self.nexus_core = nexus_core

        self.resource_snapshots: Deque[ResourceUsageSnapshot] = deque(maxlen=self.config.history_size)
        self.lock = threading.RLock()
        self.monitor_thread: Optional[threading.Thread] = None
        self.scheduler_thread: Optional[threading.Thread] = None
        self.running = False

        # Priority queue or store for tasks
        # We'll store (sort_key, creation_time, TaskRequest). Sort key depends on scheduling_policy.
        self.task_queue: List[Tuple[Any, float, TaskRequest]] = []
        self.running_tasks: Dict[str, TaskRequest] = {}
        self.completed_tasks: Dict[str, TaskRequest] = {}
        self.failed_tasks: Dict[str, TaskRequest] = {}

        # DRAS
        self._dras_enabled = self.config.dynamic_resource_allocation
        # The dictionary: module_name -> {"cpu": float, "memory": float, ...}
        # representing the allocated resources. Adjusted dynamically or by user calls.
        self._resource_allocation: Dict[str, Dict[str, float]] = defaultdict(
            lambda: {"cpu": 0.1, "memory": 128.0}  # default minimal allocation
        )

        self._ml_policy = ml_policy_model if ml_policy_model else SchedulingPolicyModel()

    # -----------------------------------------------------------------------------
    # PUBLIC METHODS
    # -----------------------------------------------------------------------------

    def start(self) -> None:
        """Begin the resource monitoring and scheduling loops."""
        with self.lock:
            if self.running:
                self.logger.warning("ResourceManager is already running.")
                return
            self.running = True

        self.monitor_thread = threading.Thread(
            target=self._monitor_loop,
            name="ResourceMonitor",
            daemon=True
        )
        self.scheduler_thread = threading.Thread(
            target=self._scheduler_loop,
            name="ResourceScheduler",
            daemon=True
        )
        self.monitor_thread.start()
        self.scheduler_thread.start()
        self.logger.info("ResourceManager started with advanced features enabled.")

    def stop(self) -> None:
        """Cleanly stop the manager, halting threads and preventing new tasks."""
        with self.lock:
            if not self.running:
                self.logger.warning("ResourceManager is not running.")
                return
            self.running = False

        if self.monitor_thread and self.monitor_thread.is_alive():
            self.monitor_thread.join(timeout=2.0)
        if self.scheduler_thread and self.scheduler_thread.is_alive():
            self.scheduler_thread.join(timeout=2.0)

        self.logger.info("ResourceManager stopped. No more scheduling or monitoring will occur.")

    def submit_task(self, task: TaskRequest) -> bool:
        """
        Submits a new task to the resource manager, returning True if accepted, False otherwise.
        We may incorporate scheduling_policy logic to adjust the 'sort_key' at insertion time.
        """
        with self.lock:
            if len(self.task_queue) + len(self.running_tasks) >= self.config.max_task_queue_size:
                self.logger.error(f"Task queue limit reached; cannot accept task {task.task_id}.")
                return False

            # Possibly override priority using ML policy
            if self.config.ml_policy_enabled:
                new_prio = self._ml_policy.advise_task_priority(task)
                if new_prio != task.priority:
                    self.logger.debug(f"ML policy overriding priority from {task.priority} to {new_prio} for {task.task_id}.")
                    task.priority = new_prio

            sort_key = self._compute_sort_key(task)
            heapq.heappush(self.task_queue, (sort_key, task.creation_time, task))
            self.logger.info(f"Task {task.task_id} submitted. (Queue size: {len(self.task_queue)})")

        return True

    def cancel_task(self, task_id: str) -> bool:
        """
        Cancels a task in the queue or running. If running, calls _interrupt_task stub.
        Returns True if found and cancelled, False if not found.
        """
        with self.lock:
            # 1) Check if running
            if task_id in self.running_tasks:
                t = self.running_tasks[task_id]
                t.state = "cancelled"
                self._interrupt_task(t)
                del self.running_tasks[task_id]
                self.logger.info(f"Cancelled running task {task_id}.")
                return True

            # 2) Check queue
            for idx, (sort_key, ctime, t) in enumerate(self.task_queue):
                if t.task_id == task_id:
                    t.state = "cancelled"
                    # remove from heap
                    del self.task_queue[idx]
                    heapq.heapify(self.task_queue)
                    self.logger.info(f"Cancelled queued task {task_id}.")
                    return True

            self.logger.warning(f"Task {task_id} not found for cancellation.")
            return False

    def clear_tasks(self, task_state: Optional[str] = None) -> int:
        """
        Clears tasks from queue/running/completed based on their state. If task_state is None, clears all.
        Returns the number of tasks cleared.
        """
        count = 0
        with self.lock:
            if task_state is None or task_state == "pending":
                count += len(self.task_queue)
                self.task_queue = []
            if task_state is None or task_state == "running":
                for rt_id in list(self.running_tasks.keys()):
                    t = self.running_tasks[rt_id]
                    if task_state is None or t.state == task_state:
                        t.state = "cancelled"
                        self._interrupt_task(t)
                        del self.running_tasks[rt_id]
                        count += 1
            if task_state is None or task_state in ["completed", "failed", "cancelled"]:
                for store_name in ["completed_tasks", "failed_tasks"]:
                    store_dict = getattr(self, store_name)
                    for key in list(store_dict.keys()):
                        if task_state is None or store_dict[key].state == task_state:
                            del store_dict[key]
                            count += 1

        self.logger.info(f"Cleared {count} tasks with state {task_state or 'all'}.")
        return count

    def update_dras_allocations(self, allocations: Dict[str, Dict[str, float]]) -> None:
        """
        Updates the dynamic resource allocations for modules. E.g.:
        {
          "ModuleA": {"cpu": 0.4, "memory": 1024},
          "ModuleB": {"cpu": 0.6, "memory": 2048}
        }
        """
        with self.lock:
            for mod, res in allocations.items():
                cpu_val = res.get("cpu", None)
                if cpu_val is not None:
                    if 0.0 <= cpu_val <= 1.0:
                        self._resource_allocation[mod]["cpu"] = cpu_val
                    else:
                        self.logger.warning(f"Invalid CPU for {mod}: {cpu_val}. Must be in [0,1].")
                mem_val = res.get("memory", None)
                if mem_val is not None and mem_val > 0:
                    self._resource_allocation[mod]["memory"] = mem_val
                self.logger.debug(f"DRAS allocation updated for {mod}: {self._resource_allocation[mod]}")

    def get_dras_allocations(self) -> Dict[str, Dict[str, float]]:
        """Returns a copy of the current resource allocation dictionary."""
        with self.lock:
            return dict((k, dict(v)) for k, v in self._resource_allocation.items())

    def adjust_task_priority(self, task_id: str, new_priority: int) -> bool:
        """
        Adjusts the priority of a queued task. Returns True if found, else False.
        """
        with self.lock:
            found = False
            for i, (skey, ctime, task) in enumerate(self.task_queue):
                if task.task_id == task_id:
                    found = True
                    task.priority = new_priority
                    # remove & re-insert
                    del self.task_queue[i]
                    heapq.heapify(self.task_queue)
                    skey = self._compute_sort_key(task)
                    heapq.heappush(self.task_queue, (skey, task.creation_time, task))
                    self.logger.info(f"Task {task_id} priority updated to {new_priority}.")
                    break

            if not found:
                self.logger.warning(f"Task {task_id} not found in queue for priority update.")
            return found

    def get_usage_history(self, limit: int = -1) -> List[ResourceUsageSnapshot]:
        """Returns up to 'limit' usage snapshots (or all if limit < 0)."""
        with self.lock:
            if limit < 0:
                return list(self.resource_snapshots)
            else:
                return list(self.resource_snapshots)[-limit:]

    def get_queued_tasks(self) -> List[TaskRequest]:
        """Returns all pending tasks in sorted order (like the queue sees them)."""
        with self.lock:
            sorted_entries = sorted(self.task_queue, key=lambda x: (x[0], x[1]))
            return [te[2] for te in sorted_entries]

    def get_running_tasks(self) -> List[TaskRequest]:
        """Returns a list of currently running tasks."""
        with self.lock:
            return list(self.running_tasks.values())

    def get_completed_tasks(self) -> List[TaskRequest]:
        """Returns a list of completed tasks."""
        with self.lock:
            return list(self.completed_tasks.values())

    def get_failed_tasks(self) -> List[TaskRequest]:
        """Returns a list of failed tasks."""
        with self.lock:
            return list(self.failed_tasks.values())

    # -----------------------------------------------------------------------------
    # INTERNAL / PROTECTED METHODS
    # -----------------------------------------------------------------------------

    def _monitor_loop(self) -> None:
        """
        Periodically collects resource usage stats. Optionally does anomaly detection,
        HPC calls, or distributed cluster checks.
        """
        self.logger.info("Monitor loop started.")
        while True:
            with self.lock:
                if not self.running:
                    break

            # Single-node usage
            try:
                cpu_percent = psutil.cpu_percent(interval=None)
                per_core = psutil.cpu_percent(interval=None, percpu=True)
                mem_stats = psutil.virtual_memory()

                # Optional: gather per-process usage if we want
                process_usage = {}
                # HPC/cluster logic stubs could go here if cluster_mode is True

                snap = ResourceUsageSnapshot(
                    cpu_percent=cpu_percent,
                    mem_percent=mem_stats.percent,
                    available_memory=mem_stats.available,
                    used_memory=mem_stats.used,
                    process_usage=process_usage,
                    per_core_cpu_percent=per_core
                )
                if self.config.anomaly_detection:
                    snap.anomaly_score = self._compute_anomaly_score(snap)

                with self.lock:
                    self.resource_snapshots.append(snap)
            except Exception as e:
                self.logger.exception(f"Exception in monitor loop: {e}", exc_info=True)

            time.sleep(self.config.monitor_interval)
        self.logger.info("Monitor loop ended.")

    def _scheduler_loop(self) -> None:
        """Periodically attempts to schedule tasks and react to resource limits."""
        self.logger.info("Scheduler loop started.")
        while True:
            with self.lock:
                if not self.running:
                    break
            self._perform_scheduling_cycle()
            time.sleep(self.config.scheduler_interval)
        self.logger.info("Scheduler loop ended.")

    def _perform_scheduling_cycle(self) -> None:
        """
        The main scheduling logic. Checks resource usage vs. soft/hard limits,
        attempts to start tasks from the queue if feasible. Also handles
        HPC expansions or job submissions if cluster_mode is True.
        """
        snapshot = None
        with self.lock:
            if self.resource_snapshots:
                snapshot = self.resource_snapshots[-1]

        if not snapshot:
            return

        # Hard-limit enforcement
        if snapshot.cpu_percent > self.config.cpu_hard_limit or snapshot.mem_percent > self.config.mem_hard_limit:
            self._handle_hard_limit_exceed(snapshot)
            # We might choose to skip scheduling new tasks this cycle
            return

        # If we are below soft-limits, attempt to schedule
        can_run_more = True
        if snapshot.cpu_percent >= self.config.cpu_soft_limit:
            can_run_more = False
        if snapshot.mem_percent >= self.config.mem_soft_limit:
            can_run_more = False

        if not can_run_more:
            self.logger.debug("Soft limit reached, deferring new tasks.")
            return

        # Actually pick tasks from the queue
        self._attempt_task_starts(snapshot)

        # HPC expansions or synergy: If cluster_mode is True, we might call external HPC APIs
        if self.config.cluster_mode:
            # e.g., self._submit_to_cluster(...)
            pass

    def _attempt_task_starts(self, snapshot: ResourceUsageSnapshot) -> None:
        """
        Pop tasks from the queue according to the scheduling policy,
        check feasibility, and start them if possible.
        """
        started_tasks = []
        # We'll try to start as many tasks as we can until we reach resource constraints
        # or run out of tasks.
        attempts = 0

        while True:
            with self.lock:
                if not self.task_queue:
                    break  # no more tasks
                # Peek the top
                top_entry = self.task_queue[0]
                sort_key, creation_time, task = top_entry

                # If the task is expired by max_waiting_time or deadline, skip it
                if self._is_task_expired(task):
                    # Remove it from queue
                    heapq.heappop(self.task_queue)
                    task.state = "failed"
                    task_str = "deadline" if (task.deadline and time.time() >= task.deadline) else "max_waiting"
                    self.logger.warning(f"Task {task.task_id} expired due to {task_str} constraint.")
                    self.failed_tasks[task.task_id] = task
                    continue

                # If we can't run the task (lack resources or ML policy says no), break out
                if not self._can_run_task(task, snapshot):
                    break

                # We can start the task. Remove from queue
                heapq.heappop(self.task_queue)
                task.state = "running"
                self.running_tasks[task.task_id] = task
                started_tasks.append(task)

            attempts += 1
            if attempts > 50:
                # Safety guard: don't start too many tasks in one cycle
                break

        # Outside lock: actually run the tasks
        for t in started_tasks:
            self._launch_task(t)

    def _launch_task(self, task: TaskRequest) -> None:
        """
        Actually run the task's callback. We do it in a new thread so scheduling loop can continue.
        For HPC or multi-node, we might do a cluster job submission instead.
        """
        def runner():
            self.logger.info(f"Starting task {task.task_id} from module {task.module_name} with priority {task.priority}.")
            start_time = time.time()
            try:
                # Possibly set a timer for max_runtime
                # or we rely on the callback to be well-behaved
                # or we run in a separate process we can kill if needed
                task.callback(*task.callback_args, **task.callback_kwargs)
                duration = time.time() - start_time
                with self.lock:
                    task.state = "completed"
                    self.completed_tasks[task.task_id] = task
                    if task.task_id in self.running_tasks:
                        del self.running_tasks[task.task_id]
                self.logger.info(f"Task {task.task_id} completed in {duration:.2f}s")
            except Exception as e:
                with self.lock:
                    self.logger.error(f"Task {task.task_id} failed: {e}", exc_info=True)
                    task.retries -= 1
                    if task.retries >= 0:
                        task.state = "pending"
                        # Re-queue it with a small delay
                        time.sleep(task.retry_delay)
                        skey = self._compute_sort_key(task)
                        heapq.heappush(self.task_queue, (skey, task.creation_time, task))
                    else:
                        task.state = "failed"
                        self.failed_tasks[task.task_id] = task
                    if task.task_id in self.running_tasks:
                        del self.running_tasks[task.task_id]

        threading.Thread(target=runner, daemon=True).start()

    def _can_run_task(self, task: TaskRequest, snapshot: ResourceUsageSnapshot) -> bool:
        """
        Check if we can run the given task right now. Takes into account:
            - Current system usage vs. soft limits
            - DRAS allocations if enabled
            - ML policy if config.ml_policy_enabled
        """
        if self.config.ml_policy_enabled:
            # If the policy returns false, we skip this task for now
            if not self._ml_policy.advise_task_start(task, snapshot):
                return False

        if self._dras_enabled:
            # Basic approach: we see if the module's CPU + required CPU would push us over some threshold
            module_alloc = self._resource_allocation[task.module_name]
            # e.g. if the module is allocated 0.3 CPU (30%), does task require 0.2? Then 0.5 total vs. 1.0
            # This is simplistic. Real DRAS might be more advanced
            hypothetical_cpu_usage = snapshot.cpu_percent + (task.required_resources.get("cpu", 0.0) * 100)
            if hypothetical_cpu_usage > self.config.cpu_soft_limit:
                return False
            # memory check
            # we interpret "memory" as MB
            needed_mem_bytes = task.required_resources.get("memory", 0) * 1024 * 1024
            if snapshot.available_memory < needed_mem_bytes:
                return False

            # If passed the checks, we also "account" for the new usage (a naive approach)
            # In a real system, you might store real-time usage per module
            # For demonstration, we'll just note the allocated CPU in the dictionary
            module_alloc["cpu"] += task.required_resources.get("cpu", 0)
            module_alloc["memory"] += task.required_resources.get("memory", 0)

        else:
            # If not using DRAS, we do a simpler check
            hypothetical_cpu_usage = snapshot.cpu_percent + (task.required_resources.get("cpu", 0.0) * 100)
            if hypothetical_cpu_usage > self.config.cpu_soft_limit:
                return False
            needed_mem_bytes = task.required_resources.get("memory", 0) * 1024 * 1024
            if snapshot.available_memory < needed_mem_bytes:
                return False

        return True

    def _compute_sort_key(self, task: TaskRequest) -> Any:
        """
        Decide how to sort tasks in the queue, based on self.config.scheduling_policy.
        Could produce a multi-component key. Lower is popped first from the min-heap.
        """
        if self.config.scheduling_policy == "fifo":
            return task.creation_time
        elif self.config.scheduling_policy == "deadline":
            # Earliest Deadline First: If no deadline, fallback to a large number
            dd = task.deadline if task.deadline else (time.time() + 9999999)
            return dd
        elif self.config.scheduling_policy == "ml":
            # We can do a more advanced approach, e.g. combining the priority with
            # a model-based score. For demonstration, let's just keep the 'priority' for now.
            return task.priority
        else:
            # "priority" or default
            return task.priority

    def _is_task_expired(self, task: TaskRequest) -> bool:
        """
        Check if a task has missed its deadline or exceeded max_waiting_time in the queue.
        """
        now = time.time()
        # Deadline
        if task.deadline and now >= task.deadline:
            return True
        # Max waiting time
        if task.max_waiting_time and (now - task.creation_time) >= task.max_waiting_time:
            return True
        return False

    def _interrupt_task(self, task: TaskRequest) -> None:
        """
        Actual method to forcibly interrupt a running task. The logic is heavily environment-dependent:
          - If tasks run in local threads, we might set a stop flag that tasks check.
          - If tasks run in separate processes, we might do an os.kill with a signal.
          - If tasks run in HPC or a container, we might issue a job cancel to the cluster manager.
        Here, we stub it out as a no-op besides logging.
        """
        self.logger.warning(f"Task {task.task_id} is being interrupted. Implementation needed.")

    def _compute_anomaly_score(self, snapshot: ResourceUsageSnapshot) -> float:
        """
        Placeholder for an anomaly detection approach. For example, we might track the
        standard deviation of CPU usage or memory usage vs. historical data, or an advanced
        ML outlier detection. Here we do something trivial:
        """
        # e.g. if CPU usage is 120% (which is impossible?), or memory usage is > 99%, that is suspicious
        # We'll do a naive approach: any usage above 90% => anomaly_score = usage - 90
        # so 95% => 5.0 anomaly, etc.
        anomaly = 0.0
        if snapshot.cpu_percent > 90:
            anomaly += (snapshot.cpu_percent - 90)
        if snapshot.mem_percent > 90:
            anomaly += (snapshot.mem_percent - 90)
        return anomaly

    def _handle_hard_limit_exceed(self, snapshot: ResourceUsageSnapshot) -> None:
        """
        If the system usage is above the hard limit, take immediate steps
        such as forcibly pausing or killing tasks, or request cluster expansion if cluster_mode = True.
        """
        self.logger.warning(f"Hard limit exceeded (CPU={snapshot.cpu_percent:.1f}% / MEM={snapshot.mem_percent:.1f}%).")
        # We can throttle tasks, forcibly reduce their priority, or kill the biggest resource hog
        # For demonstration, let's forcibly throttle everything
        for t_id, t in self.running_tasks.items():
            self.logger.warning(f"Forcibly throttling {t_id} due to hard limit exceed.")
            t.priority += 2  # degrade priority
        # HPC logic could expand cluster or request more memory from a container orchestrator, etc.

        # We might also skip scheduling new tasks until usage returns to normal

# ---------------------------------------------------------------------------------
# EXAMPLE USAGE (if run standalone)
# ---------------------------------------------------------------------------------

if __name__ == "__main__":
    logging.basicConfig(
        level=logging.DEBUG,
        format="%(asctime)s [%(name)s] %(levelname)s: %(message)s"
    )

    config = ResourceManagerConfig(
        monitor_interval=1.0,
        scheduler_interval=1.0,
        cpu_soft_limit=70.0,
        cpu_hard_limit=90.0,
        mem_soft_limit=75.0,
        mem_hard_limit=90.0,
        scheduling_policy="deadline",
        ml_policy_enabled=False,  # toggle for demonstration
        cluster_mode=False,
        anomaly_detection=True
    )

    # Optionally pass a trained ML policy here:
    # ml_policy = SchedulingPolicyModel("some_model_path")
    ml_policy = None

    manager = ResourceManager(config, nexus_core=None, ml_policy_model=ml_policy)
    manager.start()

    # Example tasks
    def example_task_callback(task: TaskRequest, msg: str, delay_s: float):
        print(f"[TASK {task.task_id}] {msg} - sleeping {delay_s}s ...")
        time.sleep(delay_s)
        print(f"[TASK {task.task_id}] Completed.")

    # Create a few tasks with deadlines
    t1 = TaskRequest(
        priority=1,
        module_name="ModuleA",
        required_resources={"cpu": 0.2, "memory": 128},
        callback=example_task_callback,
        callback_args=("Hello from T1", 3.0),
        deadline=time.time() + 15.0,  # 15s from now
    )
    t2 = TaskRequest(
        priority=0,
        module_name="ModuleB",
        required_resources={"cpu": 0.3, "memory": 256},
        callback=example_task_callback,
        callback_args=("Greetings from T2", 2.0),
        retries=1
    )
    t3 = TaskRequest(
        priority=2,
        module_name="ModuleC",
        required_resources={"cpu": 0.1, "memory": 64},
        callback=example_task_callback,
        callback_args=("Third task", 4.0),
        max_waiting_time=10.0,  # won't wait more than 10s in queue
    )

    manager.submit_task(t1)
    manager.submit_task(t2)
    manager.submit_task(t3)

    time.sleep(8)
    manager.cancel_task(t3.task_id)  # demonstration of cancelling

    time.sleep(10)
    print("\nUsage history so far:")
    for snap in manager.get_usage_history():
        print(snap)

    manager.stop()
    print("\n--- ResourceManager has stopped ---")
    print("Queued tasks:", manager.get_queued_tasks())
    print("Running tasks:", manager.get_running_tasks())
    print("Completed tasks:", manager.get_completed_tasks())
    print("Failed tasks:", manager.get_failed_tasks())

TypeError: non-default argument 'module_name' follows default argument

# Resource Manager 2

In [ ]:
"""
resource_manager.py - Advanced Resource & Scheduling System for GrandNexus

This module provides sophisticated resource management capabilities for GrandNexus,
including:

1) Smart scheduling of computational tasks based on priority, deadlines and resources
2) Dynamic resource allocation across different modules
3) Monitoring of system resources and performance
4) Adaptive load balancing and throttling
5) Error handling and recovery for resource-related issues

The ResourceManager serves as a central coordinator for computational resources,
ensuring that GrandNexus can efficiently manage its processing requirements
while maintaining responsiveness and stability.
"""

import logging
import threading
import time
import uuid
import queue
import heapq
import psutil
import os
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np

# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore


class TaskStatus(Enum):
    """Status of a task in the resource management system"""
    PENDING = auto()    # Waiting in queue
    RUNNING = auto()    # Currently executing
    COMPLETED = auto()  # Successfully completed
    FAILED = auto()     # Failed due to error
    CANCELLED = auto()  # Cancelled before completion
    TIMEOUT = auto()    # Timed out before completion
    DEFERRED = auto()   # Temporarily deferred due to resource constraints


class SchedulingPolicy(Enum):
    """Different policies for task scheduling"""
    FIFO = auto()             # First-in, first-out
    PRIORITY = auto()         # Based on task priority
    DEADLINE = auto()         # Earliest deadline first
    FAIR_SHARE = auto()       # Fair allocation between modules
    RESOURCE_AWARE = auto()   # Based on resource availability
    ADAPTIVE = auto()         # Adapts based on system state
    ML_BASED = auto()         # Uses ML model to decide scheduling


@dataclass(order=True)
class Task:
    """
    Represents a computational task to be managed by the ResourceManager.
    The order=True allows tasks to be compared based on priority and creation_time.
    """
    # Fields used for priority queue ordering
    priority: int
    creation_time: float = field(default_factory=time.time, compare=True)

    # Task identification
    task_id: str = field(default_factory=lambda: str(uuid.uuid4()), compare=False)
    source_module: str = field(default=None, compare=False)

    # Task details
    callback: Callable = field(compare=False)
    args: Tuple = field(default_factory=tuple, compare=False)
    kwargs: Dict[str, Any] = field(default_factory=dict, compare=False)
    resource_requirements: Dict[str, float] = field(default_factory=dict, compare=False)

    # Task control
    status: TaskStatus = field(default=TaskStatus.PENDING, compare=False)
    deadline: Optional[float] = field(default=None, compare=False)
    max_runtime: Optional[float] = field(default=None, compare=False)
    max_retries: int = field(default=0, compare=False)
    retry_count: int = field(default=0, compare=False)
    result: Any = field(default=None, compare=False)
    error: Optional[Exception] = field(default=None, compare=False)

    # Task tracking
    start_time: Optional[float] = field(default=None, compare=False)
    end_time: Optional[float] = field(default=None, compare=False)
    assigned_thread: Optional[threading.Thread] = field(default=None, compare=False)

    def __str__(self) -> str:
        """String representation for logging and debugging"""
        return (f"<Task id={self.task_id[:8]}, priority={self.priority}, "
                f"source={self.source_module}, status={self.status.name}>")


@dataclass
class ResourceSnapshot:
    """Snapshot of system resource usage at a given time"""
    timestamp: float = field(default_factory=time.time)
    cpu_usage: float = 0.0
    memory_usage: float = 0.0  # As percentage
    memory_available: int = 0  # In bytes
    disk_usage: float = 0.0    # As percentage
    io_counters: Dict[str, int] = field(default_factory=dict)
    per_module_cpu: Dict[str, float] = field(default_factory=dict)
    per_module_memory: Dict[str, int] = field(default_factory=dict)  # In bytes

    def __str__(self) -> str:
        """String representation for logging"""
        return (f"<ResourceSnapshot time={self.timestamp:.1f}, CPU={self.cpu_usage:.1f}%, "
                f"Memory={self.memory_usage:.1f}%, Avail={self.memory_available/(1024*1024):.1f}MB>")


@dataclass
class ResourceManagerConfig:
    """Configuration for ResourceManager behavior"""
    scheduling_policy: SchedulingPolicy = SchedulingPolicy.PRIORITY
    max_worker_threads: int = 4
    task_queue_size: int = 1000
    monitoring_interval: float = 1.0  # seconds
    cpu_threshold_soft: float = 75.0  # percentage
    cpu_threshold_hard: float = 90.0  # percentage
    memory_threshold_soft: float = 75.0  # percentage
    memory_threshold_hard: float = 90.0  # percentage
    thread_pool_resize_interval: float = 30.0  # seconds
    resource_snapshot_history: int = 60  # Keep last N snapshots
    enable_module_quotas: bool = True
    enable_adaptive_scheduling: bool = True
    enable_ml_scheduling: bool = False  # Requires ML model integration
    task_timeout_default: float = 300.0  # 5 minutes
    recovery_backoff_base: float = 2.0  # For exponential backoff


@dataclass
class ModuleQuota:
    """Resource quotas for a specific module"""
    module_name: str
    cpu_share: float = 0.25  # 0.0 to 1.0
    memory_share: float = 0.25  # 0.0 to 1.0
    max_concurrent_tasks: int = 2
    max_queued_tasks: int = 50
    task_priority_range: Tuple[int, int] = (1, 10)  # min, max
    current_task_count: int = 0

    def can_accept_task(self) -> bool:
        """Check if the module can accept another task"""
        return self.current_task_count < self.max_concurrent_tasks


class ResourceManager:
    """
    Central resource management system for GrandNexus.

    This class handles task scheduling, resource allocation, and performance
    monitoring to ensure efficient use of computational resources across
    the system.
    """

    def __init__(self, config: Optional[ResourceManagerConfig] = None,
                 nexus_core: Optional[NexusCore] = None):
        """
        Initialize the resource manager.

        Args:
            config: Configuration parameters (uses defaults if None)
            nexus_core: Reference to the NexusCore for system integration
        """
        self.logger = logging.getLogger("GrandNexus.ResourceManager")
        self.config = config or ResourceManagerConfig()
        self.nexus_core = nexus_core

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Task management
        self.task_queue = []  # heapq-based priority queue
        self.running_tasks = set()
        self.completed_tasks = deque(maxlen=100)
        self.failed_tasks = deque(maxlen=50)
        self.cancelled_tasks = deque(maxlen=50)

        # Worker thread pool
        self.workers = []
        self.worker_control = {}  # worker_id -> control dict

        # Resource monitoring
        self.resource_snapshots = deque(maxlen=self.config.resource_snapshot_history)
        self.module_stats = defaultdict(lambda: {
            'cpu_usage': deque(maxlen=30),
            'memory_usage': deque(maxlen=30),
            'tasks_completed': 0,
            'tasks_failed': 0,
            'total_runtime': 0.0
        })

        # Module quotas
        self.module_quotas = {}  # module_name -> ModuleQuota

        # Adaptive learning
        self.task_execution_stats = defaultdict(lambda: {
            'runtimes': deque(maxlen=20),
            'success_rate': 1.0,
            'resource_usage': deque(maxlen=20)
        })

        # ML-based scheduling (if enabled)
        self.scheduling_model = None
        if self.config.enable_ml_scheduling:
            self._init_scheduling_model()

        self.logger.info(f"ResourceManager initialized with ID={self.instance_id}")

    def initialize(self) -> bool:
        """
        Initialize the resource manager.

        Returns:
            True if initialization succeeded, False otherwise
        """
        if self.initialized:
            return True

        with self.lock:
            try:
                # Set up monitoring
                self._take_resource_snapshot()

                # Initialize worker threads
                self._init_worker_pool()

                # Set up default module quotas (if enabled)
                if self.config.enable_module_quotas:
                    self._init_default_quotas()

                # Connect to other modules
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("ResourceManager initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"ResourceManager initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """
        Start the resource manager operations.

        Returns:
            True if start succeeded, False otherwise
        """
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                # Start resource monitoring
                self._start_monitoring()

                # Start worker threads
                self._start_workers()

                self.running = True
                self.logger.info("ResourceManager started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start ResourceManager: {str(e)}")
                return False

    def stop(self) -> bool:
        """
        Stop the resource manager operations.

        Returns:
            True if stop succeeded, False otherwise
        """
        with self.lock:
            if not self.running:
                return True

            try:
                # Stop monitoring
                self._stop_monitoring()

                # Stop worker threads
                self._stop_workers()

                self.running = False
                self.logger.info("ResourceManager stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping ResourceManager: {str(e)}")
                return False

    def submit_task(self, source_module: str, callback: Callable,
                    args: Tuple = (), kwargs: Dict[str, Any] = None,
                    priority: int = 5, deadline: Optional[float] = None,
                    resource_requirements: Optional[Dict[str, float]] = None,
                    max_runtime: Optional[float] = None,
                    max_retries: int = 0) -> str:
        """
        Submit a task for execution.

        Args:
            source_module: Name of the module submitting the task
            callback: The function to call for task execution
            args: Positional arguments for the callback
            kwargs: Keyword arguments for the callback
            priority: Task priority (lower number = higher priority)
            deadline: Optional deadline timestamp for task completion
            resource_requirements: Dictionary of resource requirements
            max_runtime: Maximum runtime in seconds before timeout
            max_retries: Maximum number of retries on failure

        Returns:
            Task ID for tracking
        """
        if not self.running:
            if not self.start():
                raise RuntimeError("ResourceManager is not running and could not be started")

        if kwargs is None:
            kwargs = {}

        if resource_requirements is None:
            resource_requirements = {}

        with self.lock:
            # Check module quota if enabled
            if (self.config.enable_module_quotas and
                source_module in self.module_quotas):
                quota = self.module_quotas[source_module]

                # Enforce max queued tasks
                module_queued = sum(1 for task in self.task_queue
                                   if task.source_module == source_module)
                if module_queued >= quota.max_queued_tasks:
                    self.logger.warning(f"Module {source_module} exceeded queue quota")
                    return None

                # Enforce priority range
                min_prio, max_prio = quota.task_priority_range
                priority = max(min_prio, min(max_prio, priority))

            # Validate deadline if provided
            if deadline is not None and deadline < time.time():
                self.logger.warning(f"Task deadline already passed: {deadline}")
                return None

            # Set default max_runtime if not provided
            if max_runtime is None:
                max_runtime = self.config.task_timeout_default

            # Create task
            task = Task(
                priority=priority,
                source_module=source_module,
                callback=callback,
                args=args,
                kwargs=kwargs,
                resource_requirements=resource_requirements,
                deadline=deadline,
                max_runtime=max_runtime,
                max_retries=max_retries
            )

            # Add to queue
            heapq.heappush(self.task_queue, task)

            self.logger.info(f"Task {task.task_id} submitted from {source_module} with priority {priority}")

            return task.task_id

    def cancel_task(self, task_id: str) -> bool:
        """
        Cancel a pending or running task.

        Args:
            task_id: ID of the task to cancel

        Returns:
            True if task was found and cancelled, False otherwise
        """
        with self.lock:
            # Check pending tasks
            for i, task in enumerate(self.task_queue):
                if task.task_id == task_id:
                    # Mark as cancelled
                    task.status = TaskStatus.CANCELLED

                    # Remove from queue
                    self.task_queue[i] = self.task_queue[-1]
                    self.task_queue.pop()
                    heapq.heapify(self.task_queue)

                    # Add to cancelled tasks
                    self.cancelled_tasks.append(task)

                    self.logger.info(f"Cancelled pending task {task_id}")
                    return True

            # Check running tasks
            for task in self.running_tasks:
                if task.task_id == task_id:
                    # Mark as cancelled
                    task.status = TaskStatus.CANCELLED

                    # Interrupt if possible
                    if task.assigned_thread:
                        # Note: This is a placeholder. In a real system,
                        # we would need a proper way to interrupt tasks
                        self.logger.warning(f"Cannot reliably interrupt running task {task_id}")

                    self.logger.info(f"Marked running task {task_id} for cancellation")
                    return True

            self.logger.warning(f"Task {task_id} not found for cancellation")
            return False

    def get_task_status(self, task_id: str) -> Optional[Dict[str, Any]]:
        """
        Get the current status of a task.

        Args:
            task_id: ID of the task to check

        Returns:
            Dictionary with task status information, or None if not found
        """
        with self.lock:
            # Check all possible task locations
            for task_list in [self.task_queue, self.running_tasks,
                              self.completed_tasks, self.failed_tasks,
                              self.cancelled_tasks]:
                for task in task_list:
                    if getattr(task, 'task_id', None) == task_id:
                        # Convert to dict for API consumption
                        return {
                            'task_id': task.task_id,
                            'status': task.status.name,
                            'source_module': task.source_module,
                            'priority': task.priority,
                            'creation_time': task.creation_time,
                            'start_time': task.start_time,
                            'end_time': task.end_time,
                            'deadline': task.deadline,
                            'result': task.result if task.status == TaskStatus.COMPLETED else None,
                            'error': str(task.error) if task.error else None,
                            'retry_count': task.retry_count
                        }

            return None

    def get_module_stats(self, module_name: Optional[str] = None) -> Dict[str, Any]:
        """
        Get performance statistics for a specific module or all modules.

        Args:
            module_name: Name of the module, or None for all modules

        Returns:
            Dictionary with statistics
        """
        with self.lock:
            if module_name:
                if module_name in self.module_stats:
                    stats = self.module_stats[module_name]
                    return {
                        'module': module_name,
                        'avg_cpu': sum(stats['cpu_usage']) / len(stats['cpu_usage']) if stats['cpu_usage'] else 0,
                        'avg_memory': sum(stats['memory_usage']) / len(stats['memory_usage']) if stats['memory_usage'] else 0,
                        'tasks_completed': stats['tasks_completed'],
                        'tasks_failed': stats['tasks_failed'],
                        'total_runtime': stats['total_runtime'],
                        # Include quota if available
                        'quota': self._get_module_quota(module_name)
                    }
                return {'module': module_name, 'error': 'No stats available'}
            else:
                # Return stats for all modules
                result = {}
                for mod_name, stats in self.module_stats.items():
                    result[mod_name] = {
                        'avg_cpu': sum(stats['cpu_usage']) / len(stats['cpu_usage']) if stats['cpu_usage'] else 0,
                        'avg_memory': sum(stats['memory_usage']) / len(stats['memory_usage']) if stats['memory_usage'] else 0,
                        'tasks_completed': stats['tasks_completed'],
                        'tasks_failed': stats['tasks_failed'],
                        'total_runtime': stats['total_runtime'],
                        # Include quota if available
                        'quota': self._get_module_quota(mod_name)
                    }
                return result

    def set_module_quota(self, module_name: str,
                         cpu_share: Optional[float] = None,
                         memory_share: Optional[float] = None,
                         max_concurrent_tasks: Optional[int] = None,
                         max_queued_tasks: Optional[int] = None,
                         task_priority_range: Optional[Tuple[int, int]] = None) -> bool:
        """
        Set or update resource quotas for a module.

        Args:
            module_name: Name of the module
            cpu_share: Share of CPU resources (0.0 to 1.0)
            memory_share: Share of memory resources (0.0 to 1.0)
            max_concurrent_tasks: Maximum concurrent tasks
            max_queued_tasks: Maximum queued tasks
            task_priority_range: Tuple of (min_priority, max_priority)

        Returns:
            True if update succeeded, False otherwise
        """
        with self.lock:
            # Check if quota exists
            if module_name in self.module_quotas:
                quota = self.module_quotas[module_name]

                # Update fields if provided
                if cpu_share is not None:
                    quota.cpu_share = max(0.0, min(1.0, cpu_share))

                if memory_share is not None:
                    quota.memory_share = max(0.0, min(1.0, memory_share))

                if max_concurrent_tasks is not None:
                    quota.max_concurrent_tasks = max(1, max_concurrent_tasks)

                if max_queued_tasks is not None:
                    quota.max_queued_tasks = max(0, max_queued_tasks)

                if task_priority_range is not None:
                    min_prio, max_prio = task_priority_range
                    quota.task_priority_range = (min_prio, max_prio)

                self.logger.info(f"Updated quota for module {module_name}")
                return True
            else:
                # Create new quota
                quota = ModuleQuota(
                    module_name=module_name,
                    cpu_share=cpu_share if cpu_share is not None else 0.25,
                    memory_share=memory_share if memory_share is not None else 0.25,
                    max_concurrent_tasks=max_concurrent_tasks if max_concurrent_tasks is not None else 2,
                    max_queued_tasks=max_queued_tasks if max_queued_tasks is not None else 50,
                    task_priority_range=task_priority_range if task_priority_range is not None else (1, 10)
                )

                self.module_quotas[module_name] = quota
                self.logger.info(f"Created quota for module {module_name}")
                return True

    def get_system_status(self) -> Dict[str, Any]:
        """
        Get overall system status and resource usage.

        Returns:
            Dictionary with system status information
        """
        with self.lock:
            # Get latest resource snapshot
            snapshot = self.resource_snapshots[-1] if self.resource_snapshots else None

            # Count tasks by status
            pending_count = len(self.task_queue)
            running_count = len(self.running_tasks)
            completed_count = len(self.completed_tasks)
            failed_count = len(self.failed_tasks)
            cancelled_count = len(self.cancelled_tasks)

            # Calculate average task wait and execution times
            avg_wait_time = 0.0
            if completed_count > 0:
                wait_times = [task.start_time - task.creation_time
                             for task in self.completed_tasks
                             if task.start_time is not None]
                if wait_times:
                    avg_wait_time = sum(wait_times) / len(wait_times)

            avg_exec_time = 0.0
            if completed_count > 0:
                exec_times = [task.end_time - task.start_time
                             for task in self.completed_tasks
                             if task.start_time is not None and task.end_time is not None]
                if exec_times:
                    avg_exec_time = sum(exec_times) / len(exec_times)

            return {
                'timestamp': time.time(),
                'cpu_usage': snapshot.cpu_usage if snapshot else None,
                'memory_usage': snapshot.memory_usage if snapshot else None,
                'memory_available': snapshot.memory_available if snapshot else None,
                'tasks_pending': pending_count,
                'tasks_running': running_count,
                'tasks_completed': completed_count,
                'tasks_failed': failed_count,
                'tasks_cancelled': cancelled_count,
                'avg_wait_time': avg_wait_time,
                'avg_exec_time': avg_exec_time,
                'worker_count': len(self.workers),
                'scheduling_policy': self.config.scheduling_policy.name
            }

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _init_worker_pool(self) -> None:
        """Initialize the worker thread pool"""
        self.workers = []
        self.worker_control = {}

        for i in range(self.config.max_worker_threads):
            worker_id = f"worker-{i}"
            self.worker_control[worker_id] = {
                'running': False,
                'stop_event': threading.Event(),
                'tasks_processed': 0,
                'current_task': None
            }
            worker = threading.Thread(
                target=self._worker_loop,
                args=(worker_id,),
                name=worker_id,
                daemon=True
            )
            self.workers.append(worker)

        self.logger.info(f"Initialized worker pool with {len(self.workers)} workers")

    def _start_workers(self) -> None:
        """Start all worker threads"""
        for i, worker in enumerate(self.workers):
            worker_id = f"worker-{i}"
            self.worker_control[worker_id]['running'] = True
            self.worker_control[worker_id]['stop_event'].clear()
            worker.start()

        self.logger.info(f"Started {len(self.workers)} worker threads")

    def _stop_workers(self) -> None:
        """Stop all worker threads"""
        # Signal all workers to stop
        for worker_id in self.worker_control:
            self.worker_control[worker_id]['running'] = False
            self.worker_control[worker_id]['stop_event'].set()

        # Wait for workers to terminate
        for worker in self.workers:
            if worker.is_alive():
                worker.join(timeout=2.0)

        self.logger.info("Stopped all worker threads")

    def _worker_loop(self, worker_id: str) -> None:
        """
        Main worker thread loop.

        Args:
            worker_id: Identifier for this worker
        """
        control = self.worker_control[worker_id]
        self.logger.debug(f"Worker {worker_id} started")

        while control['running']:
            try:
                # Try to get a task
                task = self._get_next_task()

                if task:
                    # Process the task
                    control['current_task'] = task
                    task.assigned_thread = threading.current_thread()

                    self._execute_task(task)

                    control['tasks_processed'] += 1
                    control['current_task'] = None
                else:
                    # No tasks available, sleep briefly
                    control['stop_event'].wait(0.1)
            except Exception as e:
                self.logger.error(f"Error in worker {worker_id}: {str(e)}")
                # Sleep to avoid tight loop in case of persistent error
                time.sleep(0.5)

        self.logger.debug(f"Worker {worker_id} stopped")

    def _get_next_task(self) -> Optional[Task]:
        """
        Get the next task to execute based on scheduling policy.

        Returns:
            Next task to execute, or None if no tasks are ready
        """
        with self.lock:
            if not self.task_queue:
                return None

            # Apply scheduling policy
            if self.config.scheduling_policy == SchedulingPolicy.FIFO:
                # Find oldest task (ignoring priority)
                oldest_idx = 0
                oldest_time = float('inf')
                for i, task in enumerate(self.task_queue):
                    if task.creation_time < oldest_time:
                        oldest_time = task.creation_time
                        oldest_idx = i

                # Swap with last element and pop
                self.task_queue[oldest_idx], self.task_queue[-1] = self.task_queue[-1], self.task_queue[oldest_idx]
                task = self.task_queue.pop()

                # Fix heap
                if self.task_queue:
                    heapq.heapify(self.task_queue)

            elif self.config.scheduling_policy == SchedulingPolicy.DEADLINE:
                # Find task with earliest deadline (that's not passed)
                now = time.time()
                earliest_idx = None
                earliest_deadline = float('inf')

                for i, task in enumerate(self.task_queue):
                    if task.deadline is not None and task.deadline > now:
                        if task.deadline < earliest_deadline:
                            earliest_deadline = task.deadline
                            earliest_idx = i

                if earliest_idx is not None:
                    # Swap with last element and pop
                    self.task_queue[earliest_idx], self.task_queue[-1] = self.task_queue[-1], self.task_queue[earliest_idx]
                    task = self.task_queue.pop()

                    # Fix heap
                    if self.task_queue:
                        heapq.heapify(self.task_queue)
                else:
                    # No task with valid deadline, use priority
                    task = heapq.heappop(self.task_queue)

            elif self.config.scheduling_policy == SchedulingPolicy.FAIR_SHARE:
                # Group tasks by module
                tasks_by_module = defaultdict(list)
                for i, task in enumerate(self.task_queue):
                    tasks_by_module[task.source_module].append((i, task))

                # Find module with fewest running tasks
                module_counts = defaultdict(int)
                for task in self.running_tasks:
                    module_counts[task.source_module] += 1

                # Select from least-represented module
                selected_idx = None
                min_count = float('inf')

                for module, tasks in tasks_by_module.items():
                    count = module_counts[module]
                    if count < min_count and tasks:
                        min_count = count
                        # Select highest priority task from this module
                        tasks.sort(key=lambda x: x[1].priority)
                        selected_idx = tasks[0][0]

                if selected_idx is not None:
                    # Swap with last element and pop
                    self.task_queue[selected_idx], self.task_queue[-1] = self.task_queue[-1], self.task_queue[selected_idx]
                    task = self.task_queue.pop()

                    # Fix heap
                    if self.task_queue:
                        heapq.heapify(self.task_queue)
                else:
                    # Fallback to priority
                    task = heapq.heappop(self.task_queue)

            elif self.config.scheduling_policy == SchedulingPolicy.ML_BASED and self.scheduling_model:
                # Use ML model to score and select task
                # This is a placeholder for actual ML-based scheduling
                # In a real implementation, we would run the model on each task

                # Fallback to priority for now
                task = heapq.heappop(self.task_queue)

            else:
                # Default: Priority-based (heap is already sorted by priority)
                task = heapq.heappop(self.task_queue)

            # Check module quota if enabled
            if (self.config.enable_module_quotas and
                task.source_module in self.module_quotas):
                quota = self.module_quotas[task.source_module]

                # Count current tasks from this module
                module_running = sum(1 for t in self.running_tasks
                                    if t.source_module == task.source_module)

                if module_running >= quota.max_concurrent_tasks:
                    # Module quota exceeded, put task back and return None
                    heapq.heappush(self.task_queue, task)
                    return None

            # Update task status
            task.status = TaskStatus.RUNNING
            task.start_time = time.time()

            # Add to running tasks
            self.running_tasks.add(task)

            return task

    def _execute_task(self, task: Task) -> None:
        """
        Execute a task and handle results.

        Args:
            task: The task to execute
        """
        result = None
        error = None

        try:
            # Log task start
            self.logger.debug(f"Starting execution of task {task.task_id} from {task.source_module}")

            # Execute task
            start_time = time.time()
            result = task.callback(*task.args, **task.kwargs)
            end_time = time.time()

            # Calculate duration
            duration = end_time - start_time

            # Update task
            task.end_time = end_time
            task.result = result
            task.status = TaskStatus.COMPLETED

            # Log completion
            self.logger.debug(f"Task {task.task_id} completed in {duration:.3f}s")

            # Update module stats
            with self.lock:
                stats = self.module_stats[task.source_module]
                stats['tasks_completed'] += 1
                stats['total_runtime'] += duration

            # Update execution stats for this task type
            task_type = task.kwargs.get('task_type', 'default')
            with self.lock:
                self.task_execution_stats[task_type]['runtimes'].append(duration)
                # Update success rate (weighted average)
                current_rate = self.task_execution_stats[task_type]['success_rate']
                self.task_execution_stats[task_type]['success_rate'] = 0.9 * current_rate + 0.1

        except Exception as e:
            # Handle task failure
            error = e
            end_time = time.time()

            self.logger.error(f"Task {task.task_id} failed: {str(e)}")

            # Update task
            task.end_time = end_time
            task.error = error

            # Check for retry
            if task.retry_count < task.max_retries:
                # Schedule retry
                task.retry_count += 1
                task.status = TaskStatus.PENDING

                # Calculate backoff delay
                backoff = self.config.recovery_backoff_base ** task.retry_count

                # Reset task for retry
                task.start_time = None
                task.end_time = None
                task.error = None

                # Re-add to queue with original priority
                with self.lock:
                    heapq.heappush(self.task_queue, task)

                self.logger.info(f"Scheduled retry {task.retry_count}/{task.max_retries} "
                               f"for task {task.task_id} after {backoff:.1f}s")

            else:
                # Max retries exceeded, mark as failed
                task.status = TaskStatus.FAILED

                # Update module stats
                with self.lock:
                    stats = self.module_stats[task.source_module]
                    stats['tasks_failed'] += 1

                # Update execution stats for this task type
                task_type = task.kwargs.get('task_type', 'default')
                with self.lock:
                    # Update success rate (weighted average)
                    current_rate = self.task_execution_stats[task_type]['success_rate']
                    self.task_execution_stats[task_type]['success_rate'] = 0.9 * current_rate

                # Move to failed tasks
                with self.lock:
                    self.running_tasks.remove(task)
                    self.failed_tasks.append(task)

            return

        # Task completed successfully
        with self.lock:
            # Remove from running
            self.running_tasks.remove(task)

            # Add to completed
            self.completed_tasks.append(task)

    def _start_monitoring(self) -> None:
        """Start resource monitoring thread"""
        self.monitoring_thread = threading.Thread(
            target=self._monitoring_loop,
            daemon=True,
            name="resource-monitor"
        )
        self.monitoring_thread.start()
        self.logger.info("Started resource monitoring thread")

    def _stop_monitoring(self) -> None:
        """Stop resource monitoring thread"""
        if hasattr(self, 'monitoring_thread') and self.monitoring_thread.is_alive():
            # There's no clean way to stop a thread, so we just let it exit on its own
            # by checking self.running in the loop
            self.monitoring_thread.join(timeout=2.0)
            self.logger.info("Stopped resource monitoring thread")

    def _monitoring_loop(self) -> None:
        """Main loop for resource monitoring"""
        while self.running:
            try:
                # Take resource snapshot
                self._take_resource_snapshot()

                # Check for resource constraints
                self._check_resource_constraints()

                # Resize thread pool if needed (periodically)
                if (time.time() % self.config.thread_pool_resize_interval <
                    self.config.monitoring_interval):
                    self._resize_thread_pool()

                # Sleep until next monitoring interval
                time.sleep(self.config.monitoring_interval)

            except Exception as e:
                self.logger.error(f"Error in monitoring loop: {str(e)}")
                time.sleep(1.0)  # Sleep to avoid tight loop in case of error

    def _take_resource_snapshot(self) -> None:
        """Take a snapshot of current resource usage"""
        try:
            # Get system-wide CPU and memory usage
            cpu_percent = psutil.cpu_percent(interval=None)
            memory = psutil.virtual_memory()

            # Get disk usage for system drive
            disk = psutil.disk_usage('/')

            # Get IO counters
            io = psutil.disk_io_counters()
            io_dict = {
                'read_count': io.read_count if io else 0,
                'write_count': io.write_count if io else 0,
                'read_bytes': io.read_bytes if io else 0,
                'write_bytes': io.write_bytes if io else 0
            }

            # Get per-module resource usage (if possible)
            per_module_cpu = {}
            per_module_memory = {}

            # In a real implementation, we might track processes/threads per module
            # For simplicity, we'll just estimate based on running tasks

            for task in self.running_tasks:
                module = task.source_module
                # Assign some arbitrary CPU/memory values for demonstration
                per_module_cpu[module] = per_module_cpu.get(module, 0) + (cpu_percent / (len(self.running_tasks) or 1))
                per_module_memory[module] = per_module_memory.get(module, 0) + (memory.used / (len(self.running_tasks) or 1))

            # Create snapshot
            snapshot = ResourceSnapshot(
                cpu_usage=cpu_percent,
                memory_usage=memory.percent,
                memory_available=memory.available,
                disk_usage=disk.percent,
                io_counters=io_dict,
                per_module_cpu=per_module_cpu,
                per_module_memory=per_module_memory
            )

            # Add to history
            with self.lock:
                self.resource_snapshots.append(snapshot)

                # Update module stats
                for module, cpu in per_module_cpu.items():
                    self.module_stats[module]['cpu_usage'].append(cpu)

                for module, mem in per_module_memory.items():
                    self.module_stats[module]['memory_usage'].append(mem)

        except Exception as e:
            self.logger.error(f"Error taking resource snapshot: {str(e)}")

    def _check_resource_constraints(self) -> None:
        """Check if system is under resource constraints and take action"""
        if not self.resource_snapshots:
            return

        snapshot = self.resource_snapshots[-1]

        # Check CPU usage
        if snapshot.cpu_usage > self.config.cpu_threshold_hard:
            self.logger.warning(f"CPU usage critical: {snapshot.cpu_usage:.1f}% > {self.config.cpu_threshold_hard}%")
            self._handle_critical_cpu_usage()
        elif snapshot.cpu_usage > self.config.cpu_threshold_soft:
            self.logger.info(f"CPU usage high: {snapshot.cpu_usage:.1f}% > {self.config.cpu_threshold_soft}%")
            self._handle_high_cpu_usage()

        # Check memory usage
        if snapshot.memory_usage > self.config.memory_threshold_hard:
            self.logger.warning(f"Memory usage critical: {snapshot.memory_usage:.1f}% > {self.config.memory_threshold_hard}%")
            self._handle_critical_memory_usage()
        elif snapshot.memory_usage > self.config.memory_threshold_soft:
            self.logger.info(f"Memory usage high: {snapshot.memory_usage:.1f}% > {self.config.memory_threshold_soft}%")
            self._handle_high_memory_usage()

    def _handle_high_cpu_usage(self) -> None:
        """Handle high CPU usage (soft threshold)"""
        # Slow down task scheduling
        # We don't have to do anything specific - the monitoring thread
        # will continue to run, but worker threads will sleep more between tasks
        pass

    def _handle_critical_cpu_usage(self) -> None:
        """Handle critical CPU usage (hard threshold)"""
        # Consider suspending some tasks
        with self.lock:
            # Find lowest priority running tasks
            if self.running_tasks:
                tasks = sorted(self.running_tasks, key=lambda t: -t.priority)  # Reverse sort (highest priority last)
                for task in tasks[:max(1, len(tasks) // 4)]:  # Take ~25% of tasks
                    # Here we would ideally suspend the task
                    # But Python doesn't have a good way to suspend a thread
                    # So we'll just log it
                    self.logger.warning(f"Would suspend task {task.task_id} due to CPU constraints")

    def _handle_high_memory_usage(self) -> None:
        """Handle high memory usage (soft threshold)"""
        # Throttle new memory-intensive tasks
        pass

    def _handle_critical_memory_usage(self) -> None:
        """Handle critical memory usage (hard threshold)"""
        # Release memory if possible and delay memory-intensive tasks
        with self.lock:
            # Clear completed tasks history to free memory
            self.completed_tasks.clear()

            # GC and release memory
            import gc
            gc.collect()

    def _resize_thread_pool(self) -> None:
        """
        Dynamically resize the worker thread pool based on load.
        This is called periodically to adjust the number of worker threads.
        """
        # This is a simplified implementation
        # In a real system, this would be more sophisticated

        if not self.resource_snapshots:
            return

        snapshot = self.resource_snapshots[-1]

        # Calculate desired worker count based on load
        current_workers = len(self.workers)
        queue_size = len(self.task_queue)

        # Very simple heuristic:
        # - If queue is growing and CPU isn't maxed, add workers
        # - If queue is small and we have excess workers, remove some

        if queue_size > current_workers * 2 and snapshot.cpu_usage < 70:
            # Add a worker
            if current_workers < self.config.max_worker_threads:
                self._add_worker()
        elif queue_size < current_workers // 2 and current_workers > 2:
            # Remove a worker
            self._remove_worker()

    def _add_worker(self) -> None:
        """Add a new worker thread to the pool"""
        worker_id = f"worker-{len(self.workers)}"

        self.worker_control[worker_id] = {
            'running': True,
            'stop_event': threading.Event(),
            'tasks_processed': 0,
            'current_task': None
        }

        worker = threading.Thread(
            target=self._worker_loop,
            args=(worker_id,),
            name=worker_id,
            daemon=True
        )

        self.workers.append(worker)
        worker.start()

        self.logger.info(f"Added worker thread {worker_id}, now have {len(self.workers)} workers")

    def _remove_worker(self) -> None:
        """Remove a worker thread from the pool"""
        if not self.workers:
            return

        # Get the last worker
        worker_id = f"worker-{len(self.workers) - 1}"

        # Signal it to stop
        if worker_id in self.worker_control:
            self.worker_control[worker_id]['running'] = False
            self.worker_control[worker_id]['stop_event'].set()

            # Remove from list
            worker = self.workers.pop()

            # No need to join, it will exit on its own

            self.logger.info(f"Removed worker thread {worker_id}, now have {len(self.workers)} workers")

    def _init_scheduling_model(self) -> None:
        """Initialize the ML-based scheduling model"""
        # This is a placeholder for an actual ML model initialization
        # In a real implementation, this would load a trained model

        # For now, we'll just set a dummy model
        class DummySchedulingModel:
            def predict(self, task: Task, system_state: Dict[str, Any]) -> float:
                # Return a simple score based on priority and deadline
                score = 10 - task.priority  # Higher priority (lower number) = higher score

                # Factor in deadline if present
                if task.deadline:
                    time_to_deadline = max(0, task.deadline - time.time())
                    urgency = 10 / (1 + time_to_deadline / 60)  # More urgent as deadline approaches
                    score += urgency

                return score

        self.scheduling_model = DummySchedulingModel()
        self.logger.info("Initialized dummy scheduling model")

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            self.nexus_core.register_module(
                name="resource_manager",
                module=self,
                dependencies=[]  # No direct dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _init_default_quotas(self) -> None:
        """Initialize default resource quotas for known modules"""
        # Set up some reasonable defaults
        default_quotas = {
            "perception": {
                "cpu_share": 0.3,
                "memory_share": 0.2,
                "max_concurrent_tasks": 3,
                "max_queued_tasks": 50,
                "task_priority_range": (1, 8)
            },
            "reasoning": {
                "cpu_share": 0.4,
                "memory_share": 0.4,
                "max_concurrent_tasks": 2,
                "max_queued_tasks": 30,
                "task_priority_range": (1, 5)
            },
            "learning": {
                "cpu_share": 0.2,
                "memory_share": 0.3,
                "max_concurrent_tasks": 1,
                "max_queued_tasks": 20,
                "task_priority_range": (3, 10)
            },
            "memory_manager": {
                "cpu_share": 0.1,
                "memory_share": 0.4,
                "max_concurrent_tasks": 2,
                "max_queued_tasks": 40,
                "task_priority_range": (2, 7)
            }
        }

        for module_name, quota_params in default_quotas.items():
            self.set_module_quota(
                module_name=module_name,
                cpu_share=quota_params["cpu_share"],
                memory_share=quota_params["memory_share"],
                max_concurrent_tasks=quota_params["max_concurrent_tasks"],
                max_queued_tasks=quota_params["max_queued_tasks"],
                task_priority_range=quota_params["task_priority_range"]
            )

    def _get_module_quota(self, module_name: str) -> Optional[Dict[str, Any]]:
        """
        Get the quota settings for a module.

        Args:
            module_name: Name of the module

        Returns:
            Dictionary with quota settings or None if not found
        """
        if module_name in self.module_quotas:
            quota = self.module_quotas[module_name]
            return {
                "cpu_share": quota.cpu_share,
                "memory_share": quota.memory_share,
                "max_concurrent_tasks": quota.max_concurrent_tasks,
                "max_queued_tasks": quota.max_queued_tasks,
                "task_priority_range": quota.task_priority_range,
                "current_task_count": quota.current_task_count
            }
        return None

ModuleNotFoundError: No module named 'grandnexus'

# LLM Engine

In [ ]:
"""
llm_engine.py - Multi-provider LLM Interface for GrandNexus

This module provides a unified interface for interacting with various
Large Language Models (LLMs) and embedding models. It supports multiple
providers and offers caching, fallback mechanisms, and integration with
GrandNexus's memory and cognitive systems.

Key features:
- Support for multiple LLM providers (local models, OpenAI, Anthropic, etc.)
- Unified embedding generation interface
- Caching of results to reduce redundant API calls
- Integration with memory systems for context enhancement
- Fallback mechanisms for resilience
- Structured response parsing
"""

import logging
import threading
import time
import uuid
import json
import os
import re
import hashlib
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np



# Assuming these exist or will exist
from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.graph.graph_engine import GraphEngine

class LLMProvider(str, Enum):
    """Types of supported LLM providers"""
    LOCAL = "local"          # Local models (via transformers, etc.)
    OPENAI = "openai"        # OpenAI API
    ANTHROPIC = "anthropic"  # Anthropic/Claude API
    DEEPSEEK = "deepseek"    # DeepSeek API
    CUSTOM = "custom"        # Custom integration


@dataclass
class LLMConfig:
    """Configuration for the LLMEngine"""
    provider: LLMProvider = LLMProvider.LOCAL
    model_name: str = "text-embedding-3-small"  # Default model name
    api_key: Optional[str] = None
    temperature: float = 0.7
    max_tokens: int = 1024
    embedding_dimension: int = 1536
    context_window: int = 8192
    cache_enabled: bool = True
    cache_size: int = 1000
    request_timeout: float = 60.0
    fallback_providers: List[LLMProvider] = field(default_factory=list)
    synergy_with_memory: bool = True
    synergy_with_graph: bool = True


@dataclass
class LLMRequest:
    """Represents a request to an LLM"""
    prompt: str
    temperature: Optional[float] = None
    max_tokens: Optional[int] = None
    request_type: str = "completion"  # "completion", "embedding", "chat"
    context: Dict[str, Any] = field(default_factory=dict)
    streaming: bool = False
    stream_callback: Optional[Callable] = None
    request_id: str = field(default_factory=lambda: str(uuid.uuid4()))


@dataclass
class LLMResponse:
    """Result of an LLM request"""
    text: str
    embedding: Optional[np.ndarray] = None
    tokens_used: int = 0
    model_used: str = ""
    request_time: float = 0.0
    request_id: str = ""
    parsed_data: Optional[Dict[str, Any]] = None
    confidence: float = 1.0


class LLMEngine:
    """
    Core LLM Engine for GrandNexus, integrating multiple providers and models.

    This engine manages access to language models and embeddings, providing
    a unified interface regardless of the underlying provider. It also integrates
    with GrandNexus's memory systems for context enhancement and result storage.
    """

    def __init__(self, config: LLMConfig,
                 working_memory: Optional[WorkingMemory] = None,
                 semantic_memory: Optional[SemanticMemory] = None,
                 episodic_memory: Optional[EpisodicMemory] = None,
                 graph_engine: Optional[GraphEngine] = None,
                 nexus_core: Optional[NexusCore] = None):
        """
        Initialize the LLM engine.

        Args:
            config: Configuration for the LLM engine
            working_memory: Reference to working memory for context
            semantic_memory: Reference to semantic memory for knowledge retrieval
            episodic_memory: Reference to episodic memory for experience access
            graph_engine: Reference to graph engine for knowledge graph integration
            nexus_core: Reference to the NexusCore for system-wide coordination
        """
        self.logger = logging.getLogger("GrandNexus.LLMInterface.Engine")
        self.config = config
        self.nexus_core = nexus_core
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory
        self.graph_engine = graph_engine

        # Internal state
        self.instance_id = str(uuid.uuid4())[:8]
        self.running = False
        self.initialized = False
        self.lock = threading.RLock()

        # Provider clients
        self.clients = {}

        # Caching
        self.cache = {}
        self.cache_order = deque(maxlen=self.config.cache_size)

        # Request history and metrics
        self.request_history = deque(maxlen=100)
        self.metrics = defaultdict(lambda: deque(maxlen=100))

        # Parser registry
        self.parsers = {}

        self.logger.info(f"LLMEngine initialized with ID={self.instance_id}, provider={config.provider}")

    def initialize(self) -> bool:
        """Initialize the LLM engine and connect to providers"""
        if self.initialized:
            return True

        with self.lock:
            try:
                # Initialize provider clients
                self._init_provider_clients()

                # Initialize parsers
                self._init_response_parsers()

                # Connect to memory systems
                if self.config.synergy_with_memory:
                    self._init_memory_integration()

                # Connect to graph system
                if self.config.synergy_with_graph:
                    self._init_graph_integration()

                # Register with NexusCore
                if self.nexus_core:
                    self._register_with_nexus_core()

                self.initialized = True
                self.logger.info("LLMEngine initialization complete")
                return True
            except Exception as e:
                self.logger.error(f"LLMEngine initialization failed: {str(e)}")
                return False

    def start(self) -> bool:
        """Start the LLM engine"""
        if not self.initialized:
            if not self.initialize():
                return False

        with self.lock:
            if self.running:
                return True

            try:
                self.running = True
                self.logger.info("LLMEngine started successfully")
                return True
            except Exception as e:
                self.logger.error(f"Failed to start LLMEngine: {str(e)}")
                return False

    def stop(self) -> bool:
        """Stop the LLM engine"""
        with self.lock:
            if not self.running:
                return True

            try:
                self.running = False
                self.logger.info("LLMEngine stopped successfully")
                return True
            except Exception as e:
                self.logger.error(f"Error stopping LLMEngine: {str(e)}")
                return False

    def complete(self, request: LLMRequest) -> LLMResponse:
        """
        Generate a completion from a prompt.

        Args:
            request: The LLMRequest with prompt and parameters

        Returns:
            LLMResponse containing the generated text
        """
        if not self.initialized:
            self.initialize()

        # Check cache if enabled
        if self.config.cache_enabled:
            cache_key = self._create_cache_key(request)
            if cache_key in self.cache:
                self.logger.debug(f"Cache hit for request {request.request_id}")
                self.metrics["cache_hit_rate"].append(1.0)
                return self.cache[cache_key]
            else:
                self.metrics["cache_hit_rate"].append(0.0)

        # Prepare context
        context = request.context.copy()

        # Enrich prompt with memory if enabled
        enriched_prompt = request.prompt
        if self.config.synergy_with_memory:
            enriched_prompt = self._enrich_prompt_with_memory(request.prompt, context)

        # Set request parameters
        temperature = request.temperature if request.temperature is not None else self.config.temperature
        max_tokens = request.max_tokens if request.max_tokens is not None else self.config.max_tokens

        # Record start time
        start_time = time.time()

        try:
            # Call the provider
            response = self._call_provider({
                "prompt": enriched_prompt,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "request_type": request.request_type,
                "streaming": request.streaming,
                "stream_callback": request.stream_callback,
                "request_id": request.request_id
            })

            # Calculate request time
            request_time = time.time() - start_time
            response.request_time = request_time

            # Update metrics
            self.metrics["request_time"].append(request_time)
            self.metrics["tokens_used"].append(response.tokens_used)
            self.metrics["completion_success_rate"].append(1.0)

            # Store in cache if enabled
            if self.config.cache_enabled:
                self._add_to_cache(cache_key, response)

            # Record in history
            self.request_history.append({
                "request_id": request.request_id,
                "request_type": request.request_type,
                "timestamp": start_time,
                "success": True
            })

            return response

        except Exception as e:
            self.logger.error(f"Error in completion request: {str(e)}")

            # Update metrics
            self.metrics["completion_success_rate"].append(0.0)

            # Record in history
            self.request_history.append({
                "request_id": request.request_id,
                "request_type": request.request_type,
                "timestamp": start_time,
                "success": False,
                "error": str(e)
            })

            # Try fallback providers if available
            if self.config.fallback_providers:
                return self._call_fallback_provider(
                    request, e, self.config.fallback_providers[0]
                )

            # Return error response
            return LLMResponse(
                text=f"Error: {str(e)}",
                request_id=request.request_id,
                confidence=0.0
            )

    def create_embedding(self, text: str, model: Optional[str] = None) -> np.ndarray:
        """
        Create an embedding vector for a text.

        Args:
            text: The text to embed
            model: Optional specific embedding model to use

        Returns:
            Numpy array with the embedding vector
        """
        if not self.initialized:
            self.initialize()

        # Check cache if enabled
        if self.config.cache_enabled:
            cache_key = f"embed:{model or self.config.model_name}:{hashlib.md5(text.encode()).hexdigest()}"
            if cache_key in self.cache:
                self.logger.debug(f"Cache hit for embedding")
                self.metrics["cache_hit_rate"].append(1.0)
                return self.cache[cache_key].embedding
            else:
                self.metrics["cache_hit_rate"].append(0.0)

        # Record start time
        start_time = time.time()

        try:
            # Create embedding request
            request = LLMRequest(
                prompt=text,
                request_type="embedding",
                context={"model": model} if model else {}
            )

            # Call the provider
            response = self._call_provider({
                "prompt": text,
                "request_type": "embedding",
                "request_id": request.request_id,
                "model": model
            })

            # Calculate request time
            request_time = time.time() - start_time
            response.request_time = request_time

            # Update metrics
            self.metrics["request_time"].append(request_time)
            self.metrics["embedding_success_rate"].append(1.0)

            # Store in cache if enabled
            if self.config.cache_enabled:
                self._add_to_cache(cache_key, response)

            # Record in history
            self.request_history.append({
                "request_id": request.request_id,
                "request_type": "embedding",
                "timestamp": start_time,
                "success": True
            })

            return response.embedding

        except Exception as e:
            self.logger.error(f"Error in embedding request: {str(e)}")

            # Update metrics
            self.metrics["embedding_success_rate"].append(0.0)

            # Record in history
            self.request_history.append({
                "request_id": str(uuid.uuid4()),
                "request_type": "embedding",
                "timestamp": start_time,
                "success": False,
                "error": str(e)
            })

            # Try fallback providers if available
            if self.config.fallback_providers:
                fallback_response = self._call_fallback_provider(
                    LLMRequest(prompt=text, request_type="embedding"),
                    e, self.config.fallback_providers[0]
                )
                if fallback_response.embedding is not None:
                    return fallback_response.embedding

            # Return zero embedding as fallback
            return np.zeros(self.config.embedding_dimension)

    def parse_response(self, response: LLMResponse, parser_type: str) -> Dict[str, Any]:
        """
        Parse an LLM response using a specific parser.

        Args:
            response: The LLMResponse to parse
            parser_type: The type of parser to use

        Returns:
            Dictionary with parsed data
        """
        if not parser_type in self.parsers:
            self.logger.warning(f"Unknown parser type: {parser_type}")
            return {"error": f"Unknown parser type: {parser_type}"}

        try:
            parser_func = self.parsers[parser_type]
            parsed_data = parser_func(response.text)

            # Update the response with parsed data
            response.parsed_data = parsed_data

            return parsed_data
        except Exception as e:
            self.logger.error(f"Error parsing response with {parser_type}: {str(e)}")
            return {"error": f"Parsing error: {str(e)}"}

    def register_parser(self, parser_type: str, parser_func: Callable[[str], Dict[str, Any]]) -> None:
        """
        Register a custom response parser.

        Args:
            parser_type: Unique identifier for the parser
            parser_func: Function that takes response text and returns parsed data
        """
        with self.lock:
            self.parsers[parser_type] = parser_func
            self.logger.info(f"Registered parser: {parser_type}")

    def get_metrics(self) -> Dict[str, Any]:
        """Get current performance metrics"""
        with self.lock:
            metrics_snapshot = {}
            for key, values in self.metrics.items():
                if values:
                    metrics_snapshot[key] = {
                        'current': values[-1],
                        'average': sum(values) / len(values),
                        'min': min(values),
                        'max': max(values)
                    }
            return metrics_snapshot

    def clear_cache(self) -> None:
        """Clear the response cache"""
        with self.lock:
            self.cache.clear()
            self.cache_order.clear()
            self.logger.info("Cache cleared")

    # -----------------------------------------------------------------------
    # Internal methods
    # -----------------------------------------------------------------------

    def _init_provider_clients(self) -> None:
        """Initialize clients for the configured provider and fallbacks"""
        # Initialize main provider
        self._init_provider(self.config.provider)

        # Initialize fallback providers
        for provider in self.config.fallback_providers:
            self._init_provider(provider)

    def _init_provider(self, provider: LLMProvider) -> None:
        """
        Initialize a specific provider client.

        Args:
            provider: The provider to initialize
        """
        if provider == LLMProvider.LOCAL:
            self._init_local_model()
        elif provider == LLMProvider.OPENAI:
            self._init_openai_client()
        elif provider == LLMProvider.ANTHROPIC:
            self._init_anthropic_client()
        elif provider == LLMProvider.DEEPSEEK:
            self._init_deepseek_client()
        elif provider == LLMProvider.CUSTOM:
            self._init_custom_client()
        else:
            self.logger.warning(f"Unknown provider: {provider}")

    def _init_local_model(self) -> None:
        """Initialize a local model using Hugging Face Transformers"""
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

            # Check if we should use GPU
            device = "cuda" if torch.cuda.is_available() else "cpu"

            # Create a basic transformers pipeline
            model_name = self.config.model_name
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(model_name)

            # Move model to appropriate device
            model = model.to(device)

            # Create text generation pipeline
            text_generator = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                device=0 if device == "cuda" else -1
            )

            # Create embedding pipeline if needed
            try:
                from sentence_transformers import SentenceTransformer
                embedding_model = SentenceTransformer(model_name)
                embedding_model = embedding_model.to(device)
            except (ImportError, ValueError):
                self.logger.warning(f"No suitable embedding model found for {model_name}")
                embedding_model = None

            self.clients[LLMProvider.LOCAL] = {
                "text_generator": text_generator,
                "embedding_model": embedding_model,
                "tokenizer": tokenizer,
                "device": device
            }

            self.logger.info(f"Initialized local model: {model_name} on {device}")
        except ImportError:
            self.logger.error("Failed to import transformers. Please install with: pip install transformers torch sentence-transformers")
            raise
        except Exception as e:
            self.logger.error(f"Error initializing local model: {str(e)}")
            raise

    def _init_openai_client(self) -> None:
        """Initialize the OpenAI client"""
        try:
            import openai

            # Set up API key
            api_key = self.config.api_key
            if api_key is None:
                api_key = os.environ.get("OPENAI_API_KEY")

            if not api_key:
                raise ValueError("No OpenAI API key provided in config or environment")

            # Create client
            client = openai.OpenAI(api_key=api_key)

            self.clients[LLMProvider.OPENAI] = {
                "client": client
            }

            self.logger.info("Initialized OpenAI client")
        except ImportError:
            self.logger.error("Failed to import openai. Please install with: pip install openai")
            raise
        except Exception as e:
            self.logger.error(f"Error initializing OpenAI client: {str(e)}")
            raise

    def _init_anthropic_client(self) -> None:
        """Initialize the Anthropic/Claude client"""
        try:
            import anthropic

            # Set up API key
            api_key = self.config.api_key
            if api_key is None:
                api_key = os.environ.get("ANTHROPIC_API_KEY")

            if not api_key:
                raise ValueError("No Anthropic API key provided in config or environment")

            # Create client
            client = anthropic.Anthropic(api_key=api_key)

            self.clients[LLMProvider.ANTHROPIC] = {
                "client": client
            }

            self.logger.info("Initialized Anthropic client")
        except ImportError:
            self.logger.error("Failed to import anthropic. Please install with: pip install anthropic")
            raise
        except Exception as e:
            self.logger.error(f"Error initializing Anthropic client: {str(e)}")
            raise

    def _init_deepseek_client(self) -> None:
        """Initialize the DeepSeek client"""
        # This is a placeholder - actual implementation would depend on DeepSeek's API
        self.logger.warning("DeepSeek client initialization not implemented yet")

    def _init_custom_client(self) -> None:
        """Initialize a custom client"""
        # This is a placeholder for custom LLM integrations
        self.logger.warning("Custom client initialization not implemented yet")

    def _init_response_parsers(self) -> None:
        """Initialize standard response parsers"""
        # Register some basic parsers
        self.register_parser("json", self._parse_json_response)
        self.register_parser("list", self._parse_list_response)
        self.register_parser("key_value", self._parse_key_value_response)
        self.register_parser("structured_text", self._parse_structured_text)

    def _init_memory_integration(self) -> None:
        """Initialize integration with memory systems"""
        if not (self.working_memory or self.semantic_memory or self.episodic_memory):
            self.logger.warning("No memory modules available for integration")
            return

        self.logger.info("Initialized memory integration")

    def _init_graph_integration(self) -> None:
        """Initialize integration with graph engine"""
        if not self.graph_engine:
            self.logger.warning("No graph engine available for integration")
            return

        self.logger.info("Initialized graph integration")

    def _register_with_nexus_core(self) -> None:
        """Register with NexusCore for system-wide coordination"""
        if not self.nexus_core:
            self.logger.warning("No NexusCore available for registration")
            return

        try:
            # Register as a module with NexusCore
            dependencies = []
            if self.working_memory:
                dependencies.append("working_memory")
            if self.semantic_memory:
                dependencies.append("semantic_memory")
            if self.episodic_memory:
                dependencies.append("episodic_memory")
            if self.graph_engine:
                dependencies.append("graph_engine")

            self.nexus_core.register_module(
                name="llm_engine",
                module=self,
                dependencies=dependencies
            )

            self.logger.info("Successfully registered with NexusCore")
        except Exception as e:
            self.logger.error(f"Failed to register with NexusCore: {str(e)}")

    def _enrich_prompt_with_memory(self, prompt: str, context: Dict[str, Any]) -> str:
        """
        Enhance a prompt with relevant information from memory.

        Args:
            prompt: The original prompt
            context: Context information

        Returns:
            Enriched prompt with memory information
        """
        enriched_prompt = prompt

        try:
            # Extract potential memory cues from the prompt
            # This is a simplified approach - a real implementation would be more sophisticated
            memory_cues = self._extract_memory_cues(prompt)

            # Check working memory for relevant information
            if self.working_memory:
                working_mem_items = self._retrieve_from_working_memory(memory_cues, context)
                if working_mem_items:
                    working_context = "\nContext from working memory:\n"
                    for item in working_mem_items:
                        working_context += f"- {item}\n"
                    enriched_prompt = working_context + "\n" + enriched_prompt

            # Check semantic memory for relevant concepts
            if self.semantic_memory:
                semantic_concepts = self._retrieve_from_semantic_memory(memory_cues, context)
                if semantic_concepts:
                    semantic_context = "\nRelevant concepts:\n"
                    for concept, description in semantic_concepts:
                        semantic_context += f"- {concept}: {description}\n"
                    enriched_prompt = semantic_context + "\n" + enriched_prompt

            # Check episodic memory for relevant experiences
            if self.episodic_memory:
                episodic_items = self._retrieve_from_episodic_memory(memory_cues, context)
                if episodic_items:
                    episodic_context = "\nRelevant past interactions:\n"
                    for item in episodic_items:
                        episodic_context += f"- {item}\n"
                    enriched_prompt = episodic_context + "\n" + enriched_prompt

            return enriched_prompt
        except Exception as e:
            self.logger.error(f"Error enriching prompt with memory: {str(e)}")
            return prompt

    def _extract_memory_cues(self, prompt: str) -> List[str]:
        """
        Extract potential memory cues from a prompt.

        Args:
            prompt: The prompt to analyze

        Returns:
            List of extracted cues
        """
        # This is a simplified implementation
        # In a real system, this would use NLP techniques to extract key concepts

        words = prompt.split()
        # Filter out common stop words and punctuation
        stop_words = {"the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "with", "by"}
        cues = [word.strip(".,!?;:()[]{}\"'").lower() for word in words
                if len(word) > 3 and word.lower() not in stop_words]

        # Remove duplicates while preserving order
        unique_cues = []
        for cue in cues:
            if cue not in unique_cues:
                unique_cues.append(cue)

        return unique_cues[:10]  # Limit to top 10 cues

    def _retrieve_from_working_memory(self, cues: List[str],
                                     context: Dict[str, Any]) -> List[str]:
        """
        Retrieve relevant items from working memory.

        Args:
            cues: Memory cues to search for
            context: Additional context

        Returns:
            List of relevant items as strings
        """
        if not self.working_memory:
            return []

        # This is a simplified implementation
        # In a real system, this would use more sophisticated retrieval

        try:
            # Assume working_memory has a method to query items
            items = self.working_memory.get_all_items()

            relevant_items = []
            for item in items:
                # Check if any cue is in the item's content
                if hasattr(item, 'content'):
                    content = str(item.content)
                    if any(cue in content.lower() for cue in cues):
                        relevant_items.append(content)

            return relevant_items[:3]  # Limit to 3 items
        except Exception as e:
            self.logger.error(f"Error retrieving from working memory: {str(e)}")
            return []

    def _retrieve_from_semantic_memory(self, cues: List[str],
                                     context: Dict[str, Any]) -> List[Tuple[str, str]]:
        """
        Retrieve relevant concepts from semantic memory.

        Args:
            cues: Memory cues to search for
            context: Additional context

        Returns:
            List of (concept, description) tuples
        """
        if not self.semantic_memory:
            return []

        # This is a simplified implementation
        # In a real system, this would use more sophisticated retrieval

        try:
            relevant_concepts = []

            for cue in cues:
                # Assume semantic_memory has a method to query concepts
                query = {"text": cue}
                concepts = self.semantic_memory.query_concepts(query)

                for concept in concepts:
                    if hasattr(concept, 'name') and hasattr(concept, 'description'):
                        relevant_concepts.append((concept.name, concept.description))

            return relevant_concepts[:3]  # Limit to 3 concepts
        except Exception as e:
            self.logger.error(f"Error retrieving from semantic memory: {str(e)}")
            return []

    def _retrieve_from_episodic_memory(self, cues: List[str],
                                     context: Dict[str, Any]) -> List[str]:
        """
        Retrieve relevant episodes from episodic memory.

        Args:
            cues: Memory cues to search for
            context: Additional context

        Returns:
            List of relevant episodes as strings
        """
        if not self.episodic_memory:
            return []

        # This is a simplified implementation
        # In a real system, this would use more sophisticated retrieval

        try:
            relevant_episodes = []

            # Assume episodic_memory has a method to query episodes
            # This would depend on the actual implementation of EpisodicMemory

            return relevant_episodes[:2]  # Limit to 2 episodes
        except Exception as e:
            self.logger.error(f"Error retrieving from episodic memory: {str(e)}")
            return []

    def _enrich_response_with_graph(self, response: LLMResponse) -> LLMResponse:
        """
        Enhance an LLM response with information from the knowledge graph.

        Args:
            response: The LLM response to enhance

        Returns:
            Enhanced response
        """
        if not self.graph_engine or not self.config.synergy_with_graph:
            return response

        try:
            # Extract entities from the response
            entities = self._extract_entities(response.text)

            if not entities:
                return response

            # Query the graph for information about these entities
            graph_info = self._query_graph_for_entities(entities)

            if not graph_info:
                return response

            # Create enhanced response
            enhanced_response = LLMResponse(
                text=response.text,
                embedding=response.embedding,
                tokens_used=response.tokens_used,
                model_used=response.model_used,
                request_time=response.request_time,
                request_id=response.request_id,
                confidence=response.confidence,
                parsed_data=response.parsed_data
            )

            # Add graph information to parsed data
            if enhanced_response.parsed_data is None:
                enhanced_response.parsed_data = {}

            enhanced_response.parsed_data["graph_context"] = graph_info

            return enhanced_response
        except Exception as e:
            self.logger.error(f"Error enriching response with graph: {str(e)}")
            return response

    def _extract_entities(self, text: str) -> List[str]:
        """
        Extract entities from text for graph queries.

        Args:
            text: The text to analyze

        Returns:
            List of extracted entities
        """
        # This is a simplified implementation
        # In a real system, this would use NER or more sophisticated techniques

        # Split into sentences and words
        sentences = text.split(". ")
        entities = []

        for sentence in sentences:
            words = sentence.split()

            # Look for capitalized words that might be entities
            for i, word in enumerate(words):
                clean_word = word.strip(".,!?;:()[]{}\"'")

                # Simple heuristic: capitalized words not at the start of a sentence
                if (i > 0 and clean_word and clean_word[0].isupper() and
                    len(clean_word) > 1 and clean_word.lower() not in
                    {"i", "i'm", "i've", "i'll", "i'd"}):
                    entities.append(clean_word)

                # Also look for quoted phrases that might be entities
                if '"' in sentence:
                    quoted = re.findall(r'"([^"]*)"', sentence)
                    entities.extend(quoted)

        # Remove duplicates
        unique_entities = []
        for entity in entities:
            if entity not in unique_entities:
                unique_entities.append(entity)

        return unique_entities[:5]  # Limit to 5 entities

    def _query_graph_for_entities(self, entities: List[str]) -> Dict[str, Any]:
        """
        Query the knowledge graph for information about entities.

        Args:
            entities: List of entities to query

        Returns:
            Dictionary with graph information
        """
        if not self.graph_engine:
            return {}

        result = {}

        try:
            for entity in entities:
                # Create a query for the graph engine
                query = GraphQuery(
                    node_criteria={"label": entity},
                    max_depth=1,
                    max_results=5
                )

                # Execute the query
                query_result = self.graph_engine.execute_query(query)

                if query_result.get("nodes"):
                    result[entity] = query_result

            return result
        except Exception as e:
            self.logger.error(f"Error querying graph for entities: {str(e)}")
            return {}

    def _call_provider(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call the configured LLM provider.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from the provider
        """
        provider = self.config.provider

        if provider not in self.clients:
            raise ValueError(f"Provider {provider} not initialized")

        if provider == LLMProvider.LOCAL:
            return self._call_local_model(request_params)
        elif provider == LLMProvider.OPENAI:
            return self._call_openai(request_params)
        elif provider == LLMProvider.ANTHROPIC:
            return self._call_anthropic(request_params)
        elif provider == LLMProvider.DEEPSEEK:
            return self._call_deepseek(request_params)
        elif provider == LLMProvider.CUSTOM:
            return self._call_custom_provider(request_params)
        else:
            raise ValueError(f"Unsupported provider: {provider}")

    def _call_local_model(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call a local model using Transformers.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from the local model
        """
        client = self.clients[LLMProvider.LOCAL]

        if "request_type" not in request_params:
            raise ValueError("Missing request_type parameter")

        request_type = request_params["request_type"]

        if request_type == "completion" or request_type == "chat":
            # Use text generation pipeline
            if "text_generator" not in client:
                raise ValueError("Text generator not initialized")

            text_generator = client["text_generator"]
            prompt = request_params["prompt"]
            max_tokens = request_params.get("max_tokens", 100)
            temperature = request_params.get("temperature", 0.7)

            # Generate text
            result = text_generator(
                prompt,
                max_length=len(prompt.split()) + max_tokens,
                num_return_sequences=1,
                temperature=temperature
            )

            # Extract generated text (removing the prompt)
            generated_text = result[0]["generated_text"]
            if generated_text.startswith(prompt):
                generated_text = generated_text[len(prompt):].strip()

            # Count tokens
            tokenizer = client["tokenizer"]
            tokens = len(tokenizer.encode(generated_text))

            return LLMResponse(
                text=generated_text,
                tokens_used=tokens,
                model_used=self.config.model_name,
                request_id=request_params.get("request_id", "")
            )

        elif request_type == "embedding":
            # Use embedding model
            if "embedding_model" not in client or client["embedding_model"] is None:
                raise ValueError("Embedding model not initialized")

            embedding_model = client["embedding_model"]
            text = request_params["prompt"]

            # Generate embedding
            embedding = embedding_model.encode(text)

            # Ensure it's a numpy array
            if not isinstance(embedding, np.ndarray):
                embedding = np.array(embedding)

            return LLMResponse(
                text="",
                embedding=embedding,
                tokens_used=0,
                model_used=self.config.model_name,
                request_id=request_params.get("request_id", "")
            )

        else:
            raise ValueError(f"Unsupported request type: {request_type}")

    def _call_openai(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call the OpenAI API.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from OpenAI
        """
        client = self.clients[LLMProvider.OPENAI]["client"]

        if "request_type" not in request_params:
            raise ValueError("Missing request_type parameter")

        request_type = request_params["request_type"]
        model = request_params.get("model", "gpt-3.5-turbo")

        if request_type == "completion" or request_type == "chat":
            # Use chat completion API
            prompt = request_params["prompt"]
            max_tokens = request_params.get("max_tokens", 1000)
            temperature = request_params.get("temperature", 0.7)
            streaming = request_params.get("streaming", False)

            messages = [{"role": "user", "content": prompt}]

            if streaming:
                stream_callback = request_params.get("stream_callback")
                if not stream_callback:
                    raise ValueError("Stream callback required for streaming")

                response_text = ""

                # Stream the response
                stream = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    stream=True
                )

                for chunk in stream:
                    if chunk.choices and chunk.choices[0].delta.content:
                        content = chunk.choices[0].delta.content
                        response_text += content
                        stream_callback(content)

                # Return the full response
                return LLMResponse(
                    text=response_text,
                    tokens_used=len(response_text.split()),
                    model_used=model,
                    request_id=request_params.get("request_id", "")
                )
            else:
                # Non-streaming request
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    max_tokens=max_tokens,
                    temperature=temperature
                )

                response_text = response.choices[0].message.content

                return LLMResponse(
                    text=response_text,
                    tokens_used=response.usage.total_tokens,
                    model_used=model,
                    request_id=request_params.get("request_id", "")
                )

        elif request_type == "embedding":
            # Use embedding API
            text = request_params["prompt"]
            embedding_model = request_params.get("model", "text-embedding-3-small")

            response = client.embeddings.create(
                model=embedding_model,
                input=text
            )

            embedding = np.array(response.data[0].embedding)

            return LLMResponse(
                text="",
                embedding=embedding,
                tokens_used=response.usage.total_tokens,
                model_used=embedding_model,
                request_id=request_params.get("request_id", "")
            )

        else:
            raise ValueError(f"Unsupported request type: {request_type}")

    def _call_anthropic(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call the Anthropic/Claude API.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from Anthropic
        """
        client = self.clients[LLMProvider.ANTHROPIC]["client"]

        if "request_type" not in request_params:
            raise ValueError("Missing request_type parameter")

        request_type = request_params["request_type"]
        model = request_params.get("model", "claude-3-sonnet-20240229")

        if request_type == "completion" or request_type == "chat":
            # Use completion API
            prompt = request_params["prompt"]
            max_tokens = request_params.get("max_tokens", 1000)
            temperature = request_params.get("temperature", 0.7)
            streaming = request_params.get("streaming", False)

            if streaming:
                stream_callback = request_params.get("stream_callback")
                if not stream_callback:
                    raise ValueError("Stream callback required for streaming")

                response_text = ""

                # Stream the response
                with client.messages.stream(
                    model=model,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    messages=[
                        {"role": "user", "content": prompt}
                    ]
                ) as stream:
                    for text in stream.text_stream:
                        response_text += text
                        stream_callback(text)

                # Return the full response
                return LLMResponse(
                    text=response_text,
                    tokens_used=len(response_text.split()),
                    model_used=model,
                    request_id=request_params.get("request_id", "")
                )
            else:
                # Non-streaming request
                response = client.messages.create(
                    model=model,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    messages=[
                        {"role": "user", "content": prompt}
                    ]
                )

                response_text = response.content[0].text

                return LLMResponse(
                    text=response_text,
                    tokens_used=response.usage.output_tokens,
                    model_used=model,
                    request_id=request_params.get("request_id", "")
                )

        elif request_type == "embedding":
            # Anthropic doesn't provide embeddings, fallback to OpenAI if available
            if LLMProvider.OPENAI in self.clients:
                self.logger.warning("Anthropic doesn't provide embeddings, falling back to OpenAI")
                return self._call_openai(request_params)
            else:
                raise ValueError("Anthropic doesn't provide embeddings and no fallback available")

        else:
            raise ValueError(f"Unsupported request type: {request_type}")

    def _call_deepseek(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call the DeepSeek API.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from DeepSeek
        """
        # This is a placeholder - implementation would depend on DeepSeek's API
        raise NotImplementedError("DeepSeek API not implemented yet")

    def _call_custom_provider(self, request_params: Dict[str, Any]) -> LLMResponse:
        """
        Call a custom provider.

        Args:
            request_params: Parameters for the request

        Returns:
            LLMResponse from the custom provider
        """
        # This is a placeholder for custom LLM integrations
        raise NotImplementedError("Custom provider not implemented yet")

    def _call_fallback_provider(self, request: LLMRequest, original_error: Exception,
                              fallback_provider: LLMProvider) -> LLMResponse:
        """
        Call a fallback provider after a failure.

        Args:
            request: The original request
            original_error: The error from the primary provider
            fallback_provider: The fallback provider to use

        Returns:
            LLMResponse from the fallback provider
        """
        self.logger.info(f"Falling back to {fallback_provider} after error: {str(original_error)}")

        # Create a backup of the current provider
        original_provider = self.config.provider

        try:
            # Temporarily switch to fallback provider
            self.config.provider = fallback_provider

            # Retry the request
            return self.complete(request)
        except Exception as e:
            self.logger.error(f"Fallback to {fallback_provider} also failed: {str(e)}")

            # Try next fallback if available
            fallback_index = self.config.fallback_providers.index(fallback_provider)
            if fallback_index + 1 < len(self.config.fallback_providers):
                next_fallback = self.config.fallback_providers[fallback_index + 1]
                return self._call_fallback_provider(request, e, next_fallback)

            # All fallbacks failed, return error response
            return LLMResponse(
                text=f"Error: {str(original_error)}. Fallbacks also failed.",
                request_id=request.request_id,
                confidence=0.0
            )
        finally:
            # Restore original provider
            self.config.provider = original_provider

    def _create_cache_key(self, request: LLMRequest) -> str:
        """
        Create a cache key for a request.

        Args:
            request: The request to create a key for

        Returns:
            Cache key string
        """
        # Create a deterministic key based on request parameters
        components = [
            request.request_type,
            request.prompt,
            str(request.temperature),
            str(request.max_tokens)
        ]

        # Add sorted context items if present
        if request.context:
            context_str = json.dumps(request.context, sort_keys=True)
            components.append(context_str)

        # Join and hash
        key_string = "|".join(components)
        return hashlib.md5(key_string.encode()).hexdigest()

    def _add_to_cache(self, cache_key: str, response: LLMResponse) -> None:
        """
        Add a response to the cache.

        Args:
            cache_key: Cache key for the response
            response: Response to cache
        """
        # Check if key already exists
        if cache_key in self.cache:
            return

        # If cache is full, remove oldest item
        if len(self.cache) >= self.config.cache_size:
            if self.cache_order:
                oldest_key = self.cache_order.popleft()
                if oldest_key in self.cache:
                    del self.cache[oldest_key]

        # Add to cache and order tracker
        self.cache[cache_key] = response
        self.cache_order.append(cache_key)

    def _parse_json_response(self, text: str) -> Dict[str, Any]:
        """
        Parse a JSON response from LLM.

        Args:
            text: Response text to parse

        Returns:
            Parsed JSON data
        """
        # Try to extract JSON from the response
        try:
            # Look for JSON pattern
            json_pattern = r'```(?:json)?\s*([\s\S]*?)```'
            json_match = re.search(json_pattern, text)

            if json_match:
                json_str = json_match.group(1)
                return json.loads(json_str)

            # If no match with code blocks, try to parse the whole text
            return json.loads(text)
        except json.JSONDecodeError:
            # Try to be more lenient by fixing common issues
            fixed_text = text.strip()

            # Remove trailing commas in lists/objects
            fixed_text = re.sub(r',\s*}', '}', fixed_text)
            fixed_text = re.sub(r',\s*]', ']', fixed_text)

            # Add quotes around unquoted keys
            fixed_text = re.sub(r'([{,])\s*(\w+)\s*:', r'\1"\2":', fixed_text)

            try:
                return json.loads(fixed_text)
            except json.JSONDecodeError as e:
                raise ValueError(f"Failed to parse JSON response: {str(e)}")

    def _parse_list_response(self, text: str) -> Dict[str, Any]:
        """
        Parse a list response from LLM.

        Args:
            text: Response text to parse

        Returns:
            Dictionary with parsed list
        """
        items = []

        # Look for numbered or bulleted list items
        lines = text.split('\n')

        for line in lines:
            line = line.strip()

            # Check for numbered items (1. Item)
            numbered_match = re.match(r'^\d+\.\s+(.+)$', line)
            if numbered_match:
                items.append(numbered_match.group(1))
                continue

            # Check for bulleted items (- Item or * Item)
            bulleted_match = re.match(r'^[-*•]\s+(.+)$', line)
            if bulleted_match:
                items.append(bulleted_match.group(1))

        return {"items": items}

    def _parse_key_value_response(self, text: str) -> Dict[str, Any]:
        """
        Parse a key-value response from LLM.

        Args:
            text: Response text to parse

        Returns:
            Dictionary with parsed key-value pairs
        """
        result = {}

        # Look for key-value pairs (Key: Value)
        lines = text.split('\n')

        for line in lines:
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Check for key-value pattern
            kv_match = re.match(r'^([^:]+):\s*(.+)$', line)
            if kv_match:
                key = kv_match.group(1).strip()
                value = kv_match.group(2).strip()
                result[key] = value

        return result

    def _parse_structured_text(self, text: str) -> Dict[str, Any]:
        """
        Parse structured text response from LLM.

        Args:
            text: Response text to parse

        Returns:
            Dictionary with parsed structure
        """
        result = {"sections": []}
        current_section = None

        # Look for headers and content
        lines = text.split('\n')

        for line in lines:
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Check for headers (# Header)
            header_match = re.match(r'^#+\s+(.+)$', line)
            if header_match:
                # Save previous section if exists
                if current_section:
                    result["sections"].append(current_section)

                # Start new section
                current_section = {
                    "title": header_match.group(1),
                    "content": ""
                }
            elif current_section:
                # Add line to current section content
                if current_section["content"]:
                    current_section["content"] += "\n"
                current_section["content"] += line

        # Save last section if exists
        if current_section:
            result["sections"].append(current_section)

        return result

ModuleNotFoundError: No module named 'grandnexus'

# LLM Interface

In [ ]:
# llm_interface/llm_engine.py

import logging
import threading
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Callable, Union, Tuple
import numpy as np
from enum import Enum

class LLMProvider(str, Enum):
    """Types de providers LLM supportés"""
    LOCAL = "local"          # Modèle local (ex: via transformers)
    OPENAI = "openai"        # API OpenAI
    ANTHROPIC = "anthropic"  # API Anthropic/Claude
    DEEPSEEK = "deepseek"    # API DeepSeek
    CUSTOM = "custom"        # Intégration personnalisée

@dataclass
class LLMConfig:
    """Configuration pour le LLMEngine"""
    provider: LLMProvider = LLMProvider.LOCAL
    model_name: str = "text-embedding-3-small"  # Nom du modèle par défaut
    api_key: Optional[str] = None
    temperature: float = 0.7
    max_tokens: int = 1024
    embedding_dimension: int = 1536
    context_window: int = 8192
    cache_enabled: bool = True
    cache_size: int = 1000
    request_timeout: float = 60.0
    fallback_providers: List[LLMProvider] = field(default_factory=list)
    synergy_with_memory: bool = True
    synergy_with_graph: bool = True

@dataclass
class LLMRequest:
    """Représente une requête au LLM"""
    prompt: str
    temperature: Optional[float] = None
    max_tokens: Optional[int] = None
    request_type: str = "completion"  # "completion", "embedding", "chat"
    context: Dict[str, Any] = field(default_factory=dict)
    streaming: bool = False
    stream_callback: Optional[Callable] = None
    request_id: str = field(default_factory=lambda: str(uuid.uuid4()))

@dataclass
class LLMResponse:
    """Résultat d'une requête LLM"""
    text: str
    embedding: Optional[np.ndarray] = None
    tokens_used: int = 0
    model_used: str = ""
    request_time: float = 0.0
    request_id: str = ""
    parsed_data: Optional[Dict[str, Any]] = None
    confidence: float = 1.0

class LLMEngine:
    """
    Moteur LLM pour GrandNexus intégrant plusieurs providers et modèles.
    Gère l'accès aux LLM, les embeddings, et le caching.
    S'intègre avec les autres modules de GrandNexus.
    """

    def __init__(self,
                 config: LLMConfig,
                 working_memory: Optional[Any] = None,
                 semantic_memory: Optional[Any] = None,
                 episodic_memory: Optional[Any] = None,
                 graph_engine: Optional[Any] = None):
        """
        Initialise le moteur LLM avec la configuration et les références aux autres modules.
        """
        self.logger = logging.getLogger("LLMEngine")
        self.config = config
        self.working_memory = working_memory
        self.semantic_memory = semantic_memory
        self.episodic_memory = episodic_memory
        self.graph_engine = graph_engine

        self.lock = threading.RLock()
        self.response_cache = {}
        self.embedding_cache = {}
        self.active_clients = {}

        # Initialize provider clients based on config
        self._init_provider_clients()

        self.logger.info(f"LLMEngine initialized with provider: {config.provider}, model: {config.model_name}")

    def _init_provider_clients(self) -> None:
        """Initialise les clients d'API selon le provider configuré"""
        try:
            if self.config.provider == LLMProvider.LOCAL:
                self._init_local_model()
            elif self.config.provider == LLMProvider.OPENAI:
                self._init_openai_client()
            elif self.config.provider == LLMProvider.ANTHROPIC:
                self._init_anthropic_client()
            elif self.config.provider == LLMProvider.DEEPSEEK:
                self._init_deepseek_client()
            elif self.config.provider == LLMProvider.CUSTOM:
                self._init_custom_client()

            # Initialize fallback providers if specified
            for provider in self.config.fallback_providers:
                self._init_fallback_provider(provider)

        except Exception as e:
            self.logger.error(f"Error initializing LLM provider: {e}", exc_info=True)
            raise

    def _init_local_model(self):
        """Initialise un modèle local via HuggingFace Transformers"""
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer

            self.logger.info(f"Loading local model: {self.config.model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.config.model_name,
                torch_dtype="auto",
                device_map="auto"
            )
            self.active_clients["local"] = True
        except ImportError:
            self.logger.error("Transformers library not installed. Cannot use LOCAL provider.")
            self.active_clients["local"] = False
        except Exception as e:
            self.logger.error(f"Error loading local model: {e}", exc_info=True)
            self.active_clients["local"] = False

    def _init_openai_client(self):
        """Initialise le client OpenAI"""
        try:
            import openai
            openai.api_key = self.config.api_key
            self.active_clients["openai"] = openai
        except ImportError:
            self.logger.error("OpenAI library not installed. Cannot use OPENAI provider.")
            self.active_clients["openai"] = False

    def _init_anthropic_client(self):
        """Initialise le client Anthropic/Claude"""
        try:
            import anthropic
            self.anthropic_client = anthropic.Anthropic(api_key=self.config.api_key)
            self.active_clients["anthropic"] = self.anthropic_client
        except ImportError:
            self.logger.error("Anthropic library not installed. Cannot use ANTHROPIC provider.")
            self.active_clients["anthropic"] = False

    def _init_deepseek_client(self):
        """Initialise le client DeepSeek"""
        # Implémentation spécifique à DeepSeek basée sur leur API/SDK
        try:
            # Exemple fictif - à adapter selon l'API réelle de DeepSeek
            import deepseek
            self.deepseek_client = deepseek.Client(api_key=self.config.api_key)
            self.active_clients["deepseek"] = self.deepseek_client
        except ImportError:
            self.logger.error("DeepSeek library not installed. Cannot use DEEPSEEK provider.")
            self.active_clients["deepseek"] = False

    def _init_custom_client(self):
        """Initialise un client personnalisé"""
        # Implémentation pour intégration personnalisée
        self.active_clients["custom"] = True

    def _init_fallback_provider(self, provider: LLMProvider):
        """Initialise un provider de fallback"""
        if provider == LLMProvider.LOCAL:
            if "local" not in self.active_clients:
                self._init_local_model()
        elif provider == LLMProvider.OPENAI:
            if "openai" not in self.active_clients:
                self._init_openai_client()
        # etc.

    def complete(self, request: LLMRequest) -> LLMResponse:
        """
        Génère une complétion de texte à partir d'un prompt.
        Intègre caching et gestion des fallbacks.
        """
        with self.lock:
            # Check cache first if enabled
            if self.config.cache_enabled:
                cache_key = self._create_cache_key(request)
                cached_response = self.response_cache.get(cache_key)
                if cached_response:
                    self.logger.debug(f"Cache hit for request: {request.request_id}")
                    return cached_response

            start_time = time.time()

            # Enrichir le prompt avec le context si synergy activée
            if self.config.synergy_with_memory:
                enriched_prompt = self._enrich_prompt_with_memory(request.prompt, request.context)
                request.prompt = enriched_prompt

            # Tenter l'appel au provider principal
            try:
                response = self._call_provider(request)

                # Cache result if enabled
                if self.config.cache_enabled:
                    self._add_to_cache(cache_key, response)

                return response

            except Exception as e:
                self.logger.error(f"Error completing request with primary provider: {e}", exc_info=True)

                # Try fallback providers
                for provider in self.config.fallback_providers:
                    try:
                        self.logger.info(f"Trying fallback provider: {provider}")
                        fallback_response = self._call_fallback_provider(provider, request)
                        return fallback_response
                    except Exception as fallback_e:
                        self.logger.error(f"Fallback provider {provider} also failed: {fallback_e}")

                # All providers failed
                raise Exception(f"All LLM providers failed to complete request: {request.request_id}")

    def _call_provider(self, request: LLMRequest) -> LLMResponse:
        """Appelle le provider configuré"""
        if self.config.provider == LLMProvider.LOCAL:
            return self._call_local_model(request)
        elif self.config.provider == LLMProvider.OPENAI:
            return self._call_openai(request)
        elif self.config.provider == LLMProvider.ANTHROPIC:
            return self._call_anthropic(request)
        elif self.config.provider == LLMProvider.DEEPSEEK:
            return self._call_deepseek(request)
        elif self.config.provider == LLMProvider.CUSTOM:
            return self._call_custom_provider(request)
        else:
            raise ValueError(f"Unsupported provider: {self.config.provider}")

    def _call_local_model(self, request: LLMRequest) -> LLMResponse:
        """Appelle un modèle local via Transformers"""
        try:
            import torch

            start_time = time.time()

            inputs = self.tokenizer(request.prompt, return_tensors="pt").to(self.model.device)

            temp = request.temperature if request.temperature is not None else self.config.temperature
            max_new_tokens = request.max_tokens if request.max_tokens is not None else self.config.max_tokens

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temp,
                    do_sample=temp > 0,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            response_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Remove prompt from response if needed
            if response_text.startswith(request.prompt):
                response_text = response_text[len(request.prompt):].strip()

            tokens_used = outputs.shape[1]  # Total tokens including prompt

            end_time = time.time()

            return LLMResponse(
                text=response_text,
                tokens_used=tokens_used,
                model_used=self.config.model_name,
                request_time=end_time - start_time,
                request_id=request.request_id
            )
        except Exception as e:
            self.logger.error(f"Error in local model call: {e}", exc_info=True)
            raise

    def _call_openai(self, request: LLMRequest) -> LLMResponse:
        """Appelle l'API OpenAI"""
        client = self.active_clients.get("openai")
        if not client:
            raise Exception("OpenAI client not initialized")

        start_time = time.time()

        temp = request.temperature if request.temperature is not None else self.config.temperature
        max_tokens = request.max_tokens if request.max_tokens is not None else self.config.max_tokens

        response = client.chat.completions.create(
            model=self.config.model_name,
            messages=[{"role": "user", "content": request.prompt}],
            temperature=temp,
            max_tokens=max_tokens,
            timeout=self.config.request_timeout
        )

        end_time = time.time()

        return LLMResponse(
            text=response.choices[0].message.content,
            tokens_used=response.usage.total_tokens,
            model_used=self.config.model_name,
            request_time=end_time - start_time,
            request_id=request.request_id
        )

    def _call_anthropic(self, request: LLMRequest) -> LLMResponse:
        """Appelle l'API Anthropic/Claude"""
        client = self.active_clients.get("anthropic")
        if not client:
            raise Exception("Anthropic client not initialized")

        start_time = time.time()

        temp = request.temperature if request.temperature is not None else self.config.temperature
        max_tokens = request.max_tokens if request.max_tokens is not None else self.config.max_tokens

        response = client.messages.create(
            model=self.config.model_name,
            messages=[{"role": "user", "content": request.prompt}],
            temperature=temp,
            max_tokens=max_tokens
        )

        end_time = time.time()

        return LLMResponse(
            text=response.content[0].text,
            tokens_used=response.usage.input_tokens + response.usage.output_tokens,
            model_used=self.config.model_name,
            request_time=end_time - start_time,
            request_id=request.request_id
        )

    def _call_deepseek(self, request: LLMRequest) -> LLMResponse:
        """Appelle l'API DeepSeek"""
        client = self.active_clients.get("deepseek")
        if not client:
            raise Exception("DeepSeek client not initialized")

        # Implémentation spécifique à l'API DeepSeek
        # À adapter selon la documentation de leur API
        start_time = time.time()

        response = client.generate(
            prompt=request.prompt,
            temperature=request.temperature or self.config.temperature,
            max_tokens=request.max_tokens or self.config.max_tokens
        )

        end_time = time.time()

        return LLMResponse(
            text=response.get("text", ""),
            tokens_used=response.get("usage", {}).get("total_tokens", 0),
            model_used=self.config.model_name,
            request_time=end_time - start_time,
            request_id=request.request_id
        )

    def _call_custom_provider(self, request: LLMRequest) -> LLMResponse:
        """Appelle un provider personnalisé"""
        # Implémentation spécifique pour intégration personnalisée
        start_time = time.time()

        # Logique personnalisée d'appel LLM ici
        response_text = f"Custom provider response for: {request.prompt[:30]}..."

        end_time = time.time()

        return LLMResponse(
            text=response_text,
            tokens_used=len(request.prompt.split()),  # Estimation basique
            model_used="custom_model",
            request_time=end_time - start_time,
            request_id=request.request_id
        )

    def _call_fallback_provider(self, provider: LLMProvider, request: LLMRequest) -> LLMResponse:
        """Gère les appels aux providers de fallback"""
        # Temporairement changer de provider et appeler
        original_provider = self.config.provider
        self.config.provider = provider

        try:
            response = self._call_provider(request)
            return response
        finally:
            # Restaurer le provider original
            self.config.provider = original_provider

    def create_embedding(self, text: str) -> np.ndarray:
        """
        Crée un embedding vectoriel pour le texte d'entrée.
        Utilise le caching pour éviter des calculs redondants.
        """
        with self.lock:
            # Check cache
            if self.config.cache_enabled and text in self.embedding_cache:
                return self.embedding_cache[text]

            start_time = time.time()

            try:
                if self.config.provider == LLMProvider.OPENAI:
                    embedding = self._create_openai_embedding(text)
                elif self.config.provider == LLMProvider.LOCAL:
                    embedding = self._create_local_embedding(text)
                # Autres providers...
                else:
                    raise ValueError(f"Embedding not supported for provider: {self.config.provider}")

                # Cache result
                if self.config.cache_enabled:
                    self.embedding_cache[text] = embedding

                self.logger.debug(f"Created embedding in {time.time() - start_time:.3f}s")
                return embedding

            except Exception as e:
                self.logger.error(f"Error creating embedding: {e}", exc_info=True)
                raise

    def _create_openai_embedding(self, text: str) -> np.ndarray:
        """Crée un embedding via l'API OpenAI"""
        client = self.active_clients.get("openai")
        if not client:
            raise Exception("OpenAI client not initialized")

        response = client.embeddings.create(
            model="text-embedding-3-small",  # ou autre modèle d'embedding
            input=text
        )

        return np.array(response.data[0].embedding)

    def _create_local_embedding(self, text: str) -> np.ndarray:
        """Crée un embedding avec un modèle local"""
        try:
            from sentence_transformers import SentenceTransformer

            # Lazy initialize embedding model if needed
            if not hasattr(self, "embedding_model"):
                model_name = "all-MiniLM-L6-v2"  # Modèle léger par défaut
                self.embedding_model = SentenceTransformer(model_name)

            embedding = self.embedding_model.encode(text)
            return embedding

        except ImportError:
            self.logger.error("SentenceTransformers not installed. Cannot create local embeddings.")
            raise

    def _enrich_prompt_with_memory(self, prompt: str, context: Dict[str, Any]) -> str:
        """
        Enrichit le prompt avec des informations de la mémoire.
        Intègre des éléments pertinents des mémoires working, semantic et episodic.
        """
        if not self.config.synergy_with_memory:
            return prompt

        enriched_parts = []

        # Working Memory
        if self.working_memory:
            # On pourrait obtenir les éléments récents ou correspondant à des critères
            memory_items = []  # Placeholder pour la vraie implémentation
            if memory_items:
                enriched_parts.append("\nRelevant working memory:\n" + "\n".join(str(item) for item in memory_items))

        # Semantic Memory - concepts liés au contexte
        if self.semantic_memory and context.get("query_concepts"):
            # On pourrait chercher des concepts reliés au prompt
            concepts = []  # Placeholder pour la vraie implémentation
            if concepts:
                enriched_parts.append("\nRelevant concepts:\n" + "\n".join(str(concept) for concept in concepts))

        # Episodic Memory - épisodes pertinents
        if self.episodic_memory and context.get("query_episodes"):
            # On pourrait chercher des épisodes similaires ou pertinents
            episodes = []  # Placeholder pour la vraie implémentation
            if episodes:
                enriched_parts.append("\nRelevant episodes:\n" + "\n".join(str(episode) for episode in episodes))

        # S'il y a des enrichissements, les ajouter au prompt
        if enriched_parts:
            enriched_prompt = prompt + "\n\nContext Information:" + "".join(enriched_parts)
            return enriched_prompt

        return prompt

    def _enrich_response_with_graph(self, response: LLMResponse) -> LLMResponse:
        """
        Enrichit la réponse avec des informations du graphe de connaissances
        si la synergy avec le graphe est activée.
        """
        if not self.config.synergy_with_graph or not self.graph_engine:
            return response

        # Exemple d'enrichissement (à adapter selon vos besoins réels)
        try:
            # Extraire des entités clés de la réponse
            entities = self._extract_entities(response.text)

            if entities:
                # Rechercher ces entités dans le graphe
                graph_data = self._query_graph_for_entities(entities)

                if graph_data:
                    # Stocker les informations du graphe dans parsed_data
                    if response.parsed_data is None:
                        response.parsed_data = {}
                    response.parsed_data["graph_enrichment"] = graph_data

        except Exception as e:
            self.logger.error(f"Error enriching response with graph data: {e}", exc_info=True)

        return response

    def _extract_entities(self, text: str) -> List[str]:
        """
        Extrait des entités du texte.
        Placeholder pour une implémentation NER plus sophistiquée.
        """
        # Implémentation simpliste pour l'exemple
        import re
        # Cherche les mots commençant par une majuscule qui ne sont pas au début d'une phrase
        entities = re.findall(r'(?<![\.\?\!]\s)(?<!\A)([A-Z][a-z]+)', text)
        return list(set(entities))  # Éliminer les doublons

    def _query_graph_for_entities(self, entities: List[str]) -> Dict[str, Any]:
        """
        Interroge le graphe pour obtenir des informations sur les entités.
        """
        if not self.graph_engine:
            return {}

        result = {}

        for entity in entities:
            # Exemple simplifié - à adapter selon votre API graph_engine
            try:
                # Rechercher l'entité dans le graphe
                nodes = self.graph_engine.find_nodes({"name": entity})

                if nodes:
                    node_id = nodes[0]
                    # Obtenir les relations pour ce nœud
                    relations = self.graph_engine.get_relations(node_id)

                    # Stocker dans le résultat
                    result[entity] = {
                        "node_id": node_id,
                        "relations": relations
                    }
            except Exception as e:
                self.logger.error(f"Error querying graph for entity '{entity}': {e}")

        return result

    def _create_cache_key(self, request: LLMRequest) -> str:
        """Crée une clé de cache unique pour une requête"""
        # Générer une clé basée sur les paramètres essentiels
        key_parts = [
            request.prompt,
            str(request.temperature or self.config.temperature),
            str(request.max_tokens or self.config.max_tokens),
            request.request_type,
            self.config.model_name
        ]
        return "|".join(key_parts)

    def _add_to_cache(self, cache_key: str, response: LLMResponse) -> None:
        """Ajoute une réponse au cache"""
        # Limiter la taille du cache si nécessaire
        if self.config.cache_size > 0 and len(self.response_cache) >= self.config.cache_size:
            # Stratégie simple: supprimer une entrée aléatoire
            # Une stratégie LRU serait plus efficace en pratique
            if self.response_cache:
                self.response_cache.pop(next(iter(self.response_cache)))

        self.response_cache[cache_key] = response

    def parse_response(self, response: LLMResponse, parser_type: str) -> Dict[str, Any]:
        """
        Parse la réponse du LLM selon un type de parser spécifique.
        Utilise les fonctions de llm_interface.response_parser
        """
        if not response or not response.text:
            return {}

        try:
            # Importer dynamiquement le parser approprié
            parser_module = __import__("llm_interface.response_parser", fromlist=["parse_" + parser_type])
            parser_func = getattr(parser_module, "parse_" + parser_type, None)

            if parser_func:
                parsed_data = parser_func(response.text)
                return parsed_data
            else:
                self.logger.warning(f"Parser function 'parse_{parser_type}' not found")
                return {}

        except (ImportError, AttributeError) as e:
            self.logger.error(f"Error importing parser: {e}")
            return {}
        except Exception as e:
            self.logger.error(f"Error parsing response: {e}", exc_info=True)
            return {}

# LLM Interface Prompt

In [ ]:
# llm_interface/prompt_templates.py

from typing import Dict, Any, List, Optional

def concept_definition_prompt(concept_name: str) -> str:
    """
    Génère un prompt pour obtenir la définition d'un concept.
    """
    return f"""Please provide a detailed definition of the concept: "{concept_name}".
Include the following:
- Primary definition
- Key attributes or properties
- Related concepts or categories
- Examples if applicable
"""

def relation_validation_prompt(source_concept: str, relation_type: str, target_concept: str) -> str:
    """
    Génère un prompt pour valider une relation entre deux concepts.
    """
    return f"""Is the following relationship valid?
Source: {source_concept}
Relation Type: {relation_type}
Target: {target_concept}

Please analyze if this relationship is factually accurate and provide:
1. Whether this relationship is valid (yes/no)
2. Confidence level (0.0 to 1.0)
3. Brief explanation
"""

def chain_of_thought_prompt(question: str, context: Optional[Dict[str, Any]] = None) -> str:
    """
    Génère un prompt de raisonnement étape par étape (chain of thought).
    Inclut optionnellement du contexte.
    """
    context_str = ""
    if context:
        context_str = "\nContext Information:\n"
        for key, value in context.items():
            context_str += f"- {key}: {value}\n"

    return f"""Question: {question}
{context_str}
Let's think step by step to solve this problem:

1. """

def graph_enrichment_prompt(query: str, graph_data: Dict[str, Any]) -> str:
    """
    Génère un prompt qui incorpore des données du graphe de connaissances.
    """
    graph_context = "Graph Knowledge:\n"
    for entity, data in graph_data.items():
        graph_context += f"- {entity}:\n"
        if "properties" in data:
            graph_context += "  Properties:\n"
            for prop, value in data["properties"].items():
                graph_context += f"    - {prop}: {value}\n"
        if "relations" in data:
            graph_context += "  Relations:\n"
            for rel_type, targets in data["relations"].items():
                for target in targets:
                    graph_context += f"    - {rel_type} -> {target}\n"

    return f"""Query: {query}

{graph_context}

Based on the above graph knowledge, please provide a comprehensive answer:
"""

def learning_feedback_prompt(training_data: Dict[str, Any], model_performance: Dict[str, float]) -> str:
    """
    Génère un prompt pour obtenir du feedback sur les performances d'apprentissage.
    """
    metrics_str = "\n".join([f"- {metric}: {value}" for metric, value in model_performance.items()])

    return f"""Analyze the following learning performance:

Training Data Summary:
- Size: {training_data.get('size', 'Unknown')}
- Distribution: {training_data.get('distribution', 'Unknown')}
- Features: {', '.join(training_data.get('features', []))}

Performance Metrics:
{metrics_str}

Please provide:
1. Analysis of these results
2. Potential improvements
3. Whether this performance is satisfactory for production use
"""

def event_analysis_prompt(event_data: Dict[str, Any]) -> str:
    """
    Génère un prompt pour analyser un événement détecté par la perception.
    """
    event_str = "\n".join([f"- {key}: {value}" for key, value in event_data.items()])

    return f"""Analyze the following event:

Event Data:
{event_str}

Please provide:
1. What is happening in this event?
2. What are the key entities involved?
3. What are potential implications or next steps?
4. Confidence level in this analysis (0.0 to 1.0)
"""

# LLM Interface Response

In [ ]:
# llm_interface/response_parser.py

import json
import re
from typing import Dict, Any, List, Optional, Union

def parse_concept_definition(response: str) -> Dict[str, Any]:
   """
   Parse la réponse d'une définition de concept.
   Extrait la définition, les attributs, et les concepts liés.
   """
   result = {
       "definition": "",
       "attributes": [],
       "related_concepts": [],
       "examples": [],
       "confidence": 0.8  # Valeur par défaut
   }

   # Recherche de la définition principale
   definition_match = re.search(r"(definition|meaning):\s*([^\n]+)", response, re.IGNORECASE)
   if definition_match:
       result["definition"] = definition_match.group(2).strip()
   else:
       # Essayer de prendre la première phrase significative
       sentences = re.split(r'[.!?]', response)
       if sentences:
           result["definition"] = sentences[0].strip()

   # Recherche d'attributs/propriétés
   attr_match = re.search(r"(attributes|properties|characteristics):(.*?)(?:\n\n|\n[A-Z]|$)",
                          response, re.IGNORECASE | re.DOTALL)
   if attr_match:
       attr_text = attr_match.group(2).strip()
       # Extraire les éléments de liste
       attributes = re.findall(r"[-•*]\s*([^\n]+)", attr_text)
       if attributes:
           result["attributes"] = [attr.strip() for attr in attributes]
       else:
           # Séparer par points-virgules ou virgules si pas de formatage en liste
           attributes = re.split(r'[;,]', attr_text)
           result["attributes"] = [attr.strip() for attr in attributes if attr.strip()]

   # Recherche de concepts liés
   related_match = re.search(r"(related concepts|related categories|similar to):(.*?)(?:\n\n|\n[A-Z]|$)",
                             response, re.IGNORECASE | re.DOTALL)
   if related_match:
       related_text = related_match.group(2).strip()
       related = re.findall(r"[-•*]\s*([^\n]+)", related_text)
       if related:
           result["related_concepts"] = [rel.strip() for rel in related]
       else:
           related = re.split(r'[;,]', related_text)
           result["related_concepts"] = [rel.strip() for rel in related if rel.strip()]

   # Recherche d'exemples
   examples_match = re.search(r"(examples|instances|such as):(.*?)(?:\n\n|\n[A-Z]|$)",
                              response, re.IGNORECASE | re.DOTALL)
   if examples_match:
       examples_text = examples_match.group(2).strip()
       examples = re.findall(r"[-•*]\s*([^\n]+)", examples_text)
       if examples:
           result["examples"] = [ex.strip() for ex in examples]
       else:
           examples = re.split(r'[;,]', examples_text)
           result["examples"] = [ex.strip() for ex in examples if ex.strip()]

   return result

def parse_relation_validation(response: str) -> Dict[str, Any]:
   """
   Parse la réponse d'une validation de relation.
   Extrait si la relation est valide, la confiance, et l'explication.
   """
   result = {
       "is_valid": False,
       "confidence": 0.0,
       "explanation": ""
   }

   # Recherche si la relation est valide (oui/non)
   valid_match = re.search(r"(valid|relationship is valid|is valid):\s*(yes|no|true|false)",
                           response, re.IGNORECASE)
   if valid_match:
       valid_text = valid_match.group(2).lower()
       result["is_valid"] = valid_text in ["yes", "true"]

   # Recherche du niveau de confiance
   conf_match = re.search(r"(confidence|confidence level):\s*([0-9.]+)", response, re.IGNORECASE)
   if conf_match:
       try:
           conf_value = float(conf_match.group(2))
           if 0 <= conf_value <= 1:
               result["confidence"] = conf_value
       except ValueError:
           pass

   # Recherche de l'explication
   expl_match = re.search(r"(explanation|reasoning|rationale):\s*([^\n]+)", response, re.IGNORECASE)
   if expl_match:
       result["explanation"] = expl_match.group(2).strip()
   else:
       # Chercher des blocs qui pourraient contenir l'explication
       expl_section = re.search(r"(?:^|\n)([^:]+(?:\n[^:]+)+)", response)
       if expl_section:
           result["explanation"] = expl_section.group(1).strip()

   return result

def parse_chain_of_thought(response: str) -> Dict[str, Any]:
   """
   Parse une réponse de type "chaîne de raisonnement".
   Extrait les étapes, la conclusion et la confiance.
   """
   result = {
       "steps": [],
       "conclusion": "",
       "confidence": 0.0
   }

   # Extraire les étapes numérotées
   steps = re.findall(r"(?:\n|\A)(\d+)\.\s+([^\n]+)(?:\n|$)", response)
   if steps:
       result["steps"] = [step[1].strip() for step in steps]

   # Chercher la conclusion
   conclusion_match = re.search(r"(?:conclusion|therefore|thus|in summary|to summarize):\s*([^\n]+)",
                               response, re.IGNORECASE)
   if conclusion_match:
       result["conclusion"] = conclusion_match.group(1).strip()
   else:
       # Prendre la dernière phrase ou le dernier paragraphe comme conclusion
       paragraphs = response.split("\n\n")
       if paragraphs:
           last_para = paragraphs[-1].strip()
           if last_para and not any(last_para.startswith(f"{i}.") for i in range(1, 10)):
               result["conclusion"] = last_para

   # Chercher le niveau de confiance
   conf_match = re.search(r"(confidence|certainty):\s*([0-9.]+)", response, re.IGNORECASE)
   if conf_match:
       try:
           conf_value = float(conf_match.group(2))
           if 0 <= conf_value <= 1:
               result["confidence"] = conf_value
           else:
               # Normaliser si la confiance est exprimée en pourcentage
               result["confidence"] = min(conf_value / 100, 1.0)
       except ValueError:
           pass
   else:
       # Estimer la confiance en fonction des mots-clés
       confidence_words = {
           "certain": 1.0, "definitely": 1.0, "absolutely": 1.0,
           "likely": 0.8, "probably": 0.7, "most likely": 0.8,
           "possibly": 0.5, "perhaps": 0.4, "maybe": 0.4,
           "uncertain": 0.3, "unclear": 0.2, "doubtful": 0.1
       }

       for word, value in confidence_words.items():
           if re.search(r'\b' + re.escape(word) + r'\b', response, re.IGNORECASE):
               result["confidence"] = max(result["confidence"], value)

   return result

def parse_structured_json(response: str) -> Dict[str, Any]:
   """
   Tente d'extraire un objet JSON structuré de la réponse.
   Robuste aux divers formats ou textes additionnels.
   """
   # Rechercher un bloc de code JSON encadré par ```json...```
   json_block_match = re.search(r'```(?:json)?\s*({[\s\S]*?})\s*```', response)
   if json_block_match:
       json_str = json_block_match.group(1)
       try:
           return json.loads(json_str)
       except json.JSONDecodeError:
           pass

   # Chercher un bloc qui ressemble à du JSON même sans backticks
   json_obj_match = re.search(r'({[\s\S]*})', response)
   if json_obj_match:
       json_str = json_obj_match.group(1)
       try:
           return json.loads(json_str)
       except json.JSONDecodeError:
           pass

   # Dernier recours: essayer de traiter toute la réponse comme du JSON
   try:
       return json.loads(response)
   except json.JSONDecodeError:
       # Échec de l'extraction - renvoyer un dictionnaire vide
       return {}

def parse_entity_extraction(response: str) -> List[Dict[str, Any]]:
   """
   Extrait des entités nommées d'une réponse, avec leurs types et attributs.
   """
   entities = []

   # Essayer d'abord de parser un JSON structuré
   try:
       json_data = parse_structured_json(response)
       if isinstance(json_data, dict) and "entities" in json_data:
           return json_data["entities"]
       elif isinstance(json_data, list):
           return json_data
   except Exception:
       pass

   # Rechercher des sections d'entités
   entity_sections = re.findall(r"(?:^|\n)(?:Entity|Name):\s*([^\n]+)(?:\n(?:[^:\n]+:)?[^\n]*)*", response)

   for section in entity_sections:
       entity = {"name": section.strip(), "type": "UNKNOWN", "attributes": {}}

       # Chercher le type
       type_match = re.search(r"Type:\s*([^\n]+)", section)
       if type_match:
           entity["type"] = type_match.group(1).strip()

       # Chercher les attributs
       attr_matches = re.findall(r"([A-Za-z0-9_]+):\s*([^\n]+)", section)
       for attr_name, attr_value in attr_matches:
           if attr_name.lower() not in ["entity", "name", "type"]:
               entity["attributes"][attr_name] = attr_value.strip()

       entities.append(entity)

   # Si aucune entité n'a été trouvée, essayer une approche plus simple
   if not entities:
       simple_entities = re.findall(r"[-•*]\s*([^\n:]+)(?:\s*\(([^)]+)\))?", response)
       for entity_name, entity_type in simple_entities:
           entity = {
               "name": entity_name.strip(),
               "type": entity_type.strip() if entity_type else "UNKNOWN",
               "attributes": {}
           }
           entities.append(entity)

   return entities

def parse_classification(response: str) -> Dict[str, Any]:
   """
   Parse une réponse de classification.
   Extrait la classe, la probabilité et la justification.
   """
   result = {
       "class": "",
       "probability": 0.0,
       "explanation": ""
   }

   # Recherche de la classe
   class_match = re.search(r"(?:class|category|classification|label):\s*([^\n.]+)", response, re.IGNORECASE)
   if class_match:
       result["class"] = class_match.group(1).strip()

   # Recherche de la probabilité
   prob_match = re.search(r"(?:probability|confidence|certainty):\s*([0-9.]+)(?:%|)", response, re.IGNORECASE)
   if prob_match:
       try:
           prob_value = float(prob_match.group(1))
           # Normaliser si c'est un pourcentage
           if prob_value > 1 and prob_value <= 100:
               result["probability"] = prob_value / 100
           else:
               result["probability"] = prob_value
       except ValueError:
           pass

   # Recherche de l'explication
   expl_match = re.search(r"(?:explanation|reasoning|justification):\s*(.+?)(?:\n\n|$)",
                          response, re.IGNORECASE | re.DOTALL)
   if expl_match:
       result["explanation"] = expl_match.group(1).strip()

   return result

def parse_math_expression_interpretation(response: str) -> Dict[str, Any]:
   """
   Parse l'interprétation d'une expression mathématique.
   """
   result = {
       "interpretation": "",
       "variables": {},
       "simplified_form": "",
       "confidence": 0.0
   }

   # Extraire l'interprétation principale
   interp_match = re.search(r"(?:interpretation|meaning|represents):\s*(.+?)(?:\n\n|\n[A-Z]|$)",
                           response, re.IGNORECASE | re.DOTALL)
   if interp_match:
       result["interpretation"] = interp_match.group(1).strip()

   # Extraire les variables
   vars_section = re.search(r"(?:variables|where|symbols):\s*((?:[^\n]+\n?)+)", response, re.IGNORECASE)
   if vars_section:
       var_defs = re.findall(r"([a-zA-Z0-9_]+)\s*(?:is|represents|=|:)\s*([^\n,;]+)[,;]?",
                            vars_section.group(1))
       for var_name, var_meaning in var_defs:
           result["variables"][var_name.strip()] = var_meaning.strip()

   # Extraire la forme simplifiée
   simple_match = re.search(r"(?:simplified|simplifies to|reduced form):\s*(.+?)(?:\n\n|\n[A-Z]|$)",
                           response, re.IGNORECASE | re.DOTALL)
   if simple_match:
       result["simplified_form"] = simple_match.group(1).strip()

   # Extraire la confiance
   conf_match = re.search(r"(?:confidence|certainty):\s*([0-9.]+)", response, re.IGNORECASE)
   if conf_match:
       try:
           result["confidence"] = float(conf_match.group(1))
       except ValueError:
           pass

   return result

def parse_code_interpretation(response: str) -> Dict[str, Any]:
   """
   Parse l'interprétation d'un fragment de code.
   """
   result = {
       "purpose": "",
       "key_components": [],
       "potential_issues": [],
       "complexity": "",
       "confidence": 0.5
   }

   # Rechercher l'objectif principal
   purpose_match = re.search(r"(?:purpose|function|goal):\s*(.+?)(?:\n\n|\n[A-Z]|$)",
                            response, re.IGNORECASE | re.DOTALL)
   if purpose_match:
       result["purpose"] = purpose_match.group(1).strip()

   # Rechercher les composants clés
   comp_match = re.search(r"(?:components|key elements|main parts):(.*?)(?:\n\n|\n[A-Z]|$)",
                         response, re.IGNORECASE | re.DOTALL)
   if comp_match:
       comp_text = comp_match.group(1).strip()
       components = re.findall(r"[-•*]\s*([^\n]+)", comp_text)
       if components:
           result["key_components"] = [comp.strip() for comp in components]
       else:
           # Séparer par points si pas de formatage en liste
           components = re.split(r'\.', comp_text)
           result["key_components"] = [comp.strip() for comp in components if comp.strip()]

   # Rechercher les problèmes potentiels
   issues_match = re.search(r"(?:issues|problems|bugs|concerns):(.*?)(?:\n\n|\n[A-Z]|$)",
                           response, re.IGNORECASE | re.DOTALL)
   if issues_match:
       issues_text = issues_match.group(1).strip()
       issues = re.findall(r"[-•*]\s*([^\n]+)", issues_text)
       if issues:
           result["potential_issues"] = [issue.strip() for issue in issues]
       else:
           issues = re.split(r'\.', issues_text)
           result["potential_issues"] = [issue.strip() for issue in issues if issue.strip()]

   # Rechercher la complexité
   complex_match = re.search(r"(?:complexity|efficiency):\s*(.+?)(?:\n\n|\n[A-Z]|$)",
                            response, re.IGNORECASE | re.DOTALL)
   if complex_match:
       result["complexity"] = complex_match.group(1).strip()

   # Rechercher la confiance
   conf_match = re.search(r"(?:confidence|certainty):\s*([0-9.]+)", response, re.IGNORECASE)
   if conf_match:
       try:
           result["confidence"] = float(conf_match.group(1))
       except ValueError:
           pass

   return result

# LLM Interface Init

In [ ]:
# llm_interface/__init__.py

from .llm_engine import LLMEngine, LLMConfig, LLMRequest, LLMResponse, LLMProvider
from . import prompt_templates
from . import response_parser

__all__ = [
    'LLMEngine',
    'LLMConfig',
    'LLMRequest',
    'LLMResponse',
    'LLMProvider',
    'prompt_templates',
    'response_parser'
]

ImportError: attempted relative import with no known parent package

# Task
Test the core executable components of GrandNexus V6.4 by assembling the provided code modules into a coherent package, installing dependencies, and simulating basic interactions to demonstrate functionality.

## Identify and resolve dependencies

### Subtask:
Analyze the provided code cells to identify external libraries and internal dependencies between the modules.


**Reasoning**:
Analyze the import statements in the provided code cells to identify external and internal dependencies and list the unprovided modules.



In [ ]:
import re

code_cells = [
    """
# graph_engine.py
import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False
""",
    """
# graph_engine.py (Second version)
import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json
import os
import random

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False
""",
    """
# pattern_mining.py
import logging
import time
import threading
import math
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Callable
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import uuid

from grandnexus.graph.graph_engine import ( create_graph_engine,
    AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)
from grandnexus.graph.graph_pattern_interface import (
    PatternMiningInterface, PatternMatchResult, FrequentPattern
)

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False
""",
    """
# pdvSygkY4JRG
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np
""",
    """
# graph_pattern_interface.py
from abc import ABC, abstractmethod
from typing import Dict, Any, List, Optional, Union, Tuple, Set
from dataclasses import dataclass, field

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)

import logging
""",
    """
# graph_module_manager.py
import logging
import traceback
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Type
import time

from grandnexus.core.nexus_core import NexusCore
from grandnexus.core.registry import ModuleRegistry

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError,
    create_graph_engine
)
from grandnexus.graph.pattern_mining import (
    PatternMiningModule, PatternMatchResult, FrequentPattern,
    create_pattern_mining
)
from grandnexus.graph.graph_pattern_interface import GraphPatternBridge
""",
    """
# resource_manager.py (First version)
import logging
import psutil
import time
import queue
import threading
import heapq
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import (
    Any, Dict, List, Optional, Callable, Union, Deque, Tuple, Set
)
""",
    """
# resource_manager.py (Second version)
import logging
import threading
import time
import uuid
import queue
import heapq
import psutil
import os
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np

from grandnexus.core.nexus_core import NexusCore
""",
    """
# llm_engine.py
import logging
import threading
import time
import uuid
import json
import os
import re
import hashlib
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np

from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.graph.graph_engine import GraphEngine
""",
    """
# llm_interface/prompt_templates.py
from typing import Dict, Any, List, Optional
""",
    """
# llm_interface/response_parser.py
import json
import re
from typing import Dict, Any, List, Optional, Union
""",
    """
# llm_interface/__init__.py
from .llm_engine import LLMEngine, LLMConfig, LLMRequest, LLMResponse, LLMProvider
from . import prompt_templates
from . import response_parser
"""
]

external_dependencies = set()
internal_dependencies = defaultdict(set)
unprovided_modules = set()

module_names = {
    'graph_engine.py': 'graph.graph_engine',
    'pattern_mining.py': 'graph.pattern_mining',
    'graph_pattern_interface.py': 'graph.graph_pattern_interface',
    'graph_module_manager.py': 'graph.graph_module_manager',
    'resource_manager.py': 'resource_manager',
    'llm_engine.py': 'llm_engine',
    'llm_interface/prompt_templates.py': 'llm_interface.prompt_templates',
    'llm_interface/response_parser.py': 'llm_interface.response_parser',
    'llm_interface/__init__.py': 'llm_interface'
}

provided_grandnexus_modules = {
    'grandnexus.graph.graph_engine',
    'grandnexus.graph.pattern_mining',
    'grandnexus.graph.graph_pattern_interface',
    'grandnexus.graph.graph_module_manager',
    'grandnexus.resource_manager',
    'grandnexus.llm_engine', # Note: This is referenced, but the file is llm_interface/llm_engine.py
    'grandnexus.llm_interface.prompt_templates',
    'grandnexus.llm_interface.response_parser',
    'grandnexus.llm_interface'
}


for cell_content in code_cells:
    # Extract potential module name from comment or assume based on content
    cell_name_match = re.search(r'^#\s*(.*)\.py', cell_content, re.MULTILINE)
    if cell_name_match:
        cell_file_name = cell_name_match.group(1).strip() + '.py'
    elif 'graph_pattern_interface.py' in cell_content:
         cell_file_name = 'graph_pattern_interface.py'
    elif 'graph_module_manager.py' in cell_content:
         cell_file_name = 'graph_module_manager.py'
    elif 'llm_interface/prompt_templates.py' in cell_content:
         cell_file_name = 'llm_interface/prompt_templates.py'
    elif 'llm_interface/response_parser.py' in cell_content:
         cell_file_name = 'llm_interface/response_parser.py'
    elif 'llm_interface/__init__.py' in cell_content:
         cell_file_name = 'llm_interface/__init__.py'
    elif 'psutil' in cell_content and 'threading' in cell_content and 'heapq' in cell_content:
        # Heuristic to differentiate resource_manager versions
        if 'NexusCore' in cell_content:
             cell_file_name = 'resource_manager_v2.py'
        else:
             cell_file_name = 'resource_manager_v1.py'
    elif 'graph_engine' in cell_content and 'Neo4jGraphEngine' in cell_content:
        # Heuristic to differentiate graph_engine versions
        if 'os' in cell_content and 'random' in cell_content:
             cell_file_name = 'graph_engine_v2.py'
        else:
             cell_file_name = 'graph_engine_v1.py'
    elif 'pandas' in cell_content and 'numpy' in cell_content and 'Ticket Number' in cell_content:
         cell_file_name = 'ticket_analysis.py' # Give it a descriptive name
    else:
         cell_file_name = 'unknown_cell.py'

    current_module_base = module_names.get(cell_file_name)
    if current_module_base is None:
        # Try to map based on heuristics if not in the direct map
        if 'graph_engine_v' in cell_file_name:
            current_module_base = 'graph.graph_engine'
        elif 'resource_manager_v' in cell_file_name:
             current_module_base = 'resource_manager'
        elif 'llm_engine.py' in cell_file_name:
             current_module_base = 'llm_engine' # This naming is inconsistent with file path
        elif 'ticket_analysis.py' in cell_file_name:
             current_module_base = 'analysis.ticket_analysis' # Placeholder name
        else:
             current_module_base = cell_file_name.replace('/', '.').replace('.py', '')


    # Find all import statements
    imports = re.findall(r'^(?:import\s+([\w.]+))|(?:from\s+([\w.]+)\s+import)', cell_content, re.MULTILINE)

    for imp in imports:
        external_lib = imp[0] if imp[0] else imp[1] # Get the imported name

        if external_lib.startswith('grandnexus.'):
            # Internal GrandNexus dependency
            dep_module = external_lib

            if current_module_base:
                 internal_dependencies[f'grandnexus.{current_module_base}'].add(dep_module)

            # Check if the dependency is provided
            # Normalize dependency name (remove specific imported objects)
            base_dep_module = dep_module.split('.')[0:3] # grandnexus.module.submodule
            base_dep_module_name = '.'.join(base_dep_module)

            is_provided = False
            for provided_module_base in provided_grandnexus_modules:
                if dep_module.startswith(provided_module_base):
                    is_provided = True
                    break

            if not is_provided:
                unprovided_modules.add(dep_module)

        else:
            # External library
            # Clean up potential 'as' aliases and imported submodules for external libs
            base_lib = external_lib.split('.')[0]
            if base_lib not in ['logging', 'time', 'uuid', 'threading', 'abc', 'dataclasses', 'typing', 'collections', 'math', 'json', 'os', 'random', 're', 'hashlib', 'enum', 'auto', 'queue', 'heapq', 'psutil']: # Standard libraries
                external_dependencies.add(base_lib)


print("External Dependencies:")
for dep in sorted(list(external_dependencies)):
    print(f"- {dep}")

print("\nInternal Dependencies:")
for module, deps in internal_dependencies.items():
    print(f"- {module} depends on:")
    for dep in sorted(list(deps)):
        print(f"  - {dep}")

print("\nUnprovided GrandNexus Modules/Paths (based on imports):")
for mod in sorted(list(unprovided_modules)):
    print(f"- {mod}")


NameError: name 'defaultdict' is not defined

**Reasoning**:
The previous code failed because `defaultdict` was not imported. The corrected code block will import `defaultdict` and then proceed to analyze the import statements as planned.



In [ ]:
from collections import defaultdict
import re

code_cells = [
    """
# graph_engine.py
import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False
""",
    """
# graph_engine.py (Second version)
import logging
import time
import uuid
import threading
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union, Tuple, Set, Callable
from collections import defaultdict, deque
import math
import json
import os
import random

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

try:
    from neo4j import GraphDatabase, Driver
    HAS_NEO4J = True
except ImportError:
    HAS_NEO4J = False
""",
    """
# pattern_mining.py
import logging
import time
import threading
import math
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Callable
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import uuid

from grandnexus.graph.graph_engine import ( create_graph_engine,
    AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)
from grandnexus.graph.graph_pattern_interface import (
    PatternMiningInterface, PatternMatchResult, FrequentPattern
)

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False
""",
    """
# pdvSygkY4JRG
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np
""",
    """
# graph_pattern_interface.py
from abc import ABC, abstractmethod
from typing import Dict, Any, List, Optional, Union, Tuple, Set
from dataclasses import dataclass, field

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError
)

import logging
""",
    """
# graph_module_manager.py
import logging
import traceback
from typing import Dict, Any, List, Optional, Union, Tuple, Set, Type
import time

from grandnexus.core.nexus_core import NexusCore
from grandnexus.core.registry import ModuleRegistry

from grandnexus.graph.graph_engine import (
    create_graph_engine, AbstractGraphEngine, GraphNode, GraphEdge, GraphPattern,
    GraphQuery, NodeNotFoundError, EdgeNotFoundError,
    create_graph_engine
)
from grandnexus.graph.pattern_mining import (
    PatternMiningModule, PatternMatchResult, FrequentPattern,
    create_pattern_mining
)
from grandnexus.graph.graph_pattern_interface import GraphPatternBridge
""",
    """
# resource_manager.py (First version)
import logging
import psutil
import time
import queue
import threading
import heapq
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import (
    Any, Dict, List, Optional, Callable, Union, Deque, Tuple, Set
)
""",
    """
# resource_manager.py (Second version)
import logging
import threading
import time
import uuid
import queue
import heapq
import psutil
import os
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np

from grandnexus.core.nexus_core import NexusCore
""",
    """
# llm_engine.py
import logging
import threading
import time
import uuid
import json
import os
import re
import hashlib
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Callable, Union, Tuple, Set
from collections import defaultdict, deque
import numpy as np

from grandnexus.core.nexus_core import NexusCore
from grandnexus.memory.working import WorkingMemory
from grandnexus.memory.semantic import SemanticMemory
from grandnexus.memory.episodic import EpisodicMemory
from grandnexus.graph.graph_engine import GraphEngine
""",
    """
# llm_interface/prompt_templates.py
from typing import Dict, Any, List, Optional
""",
    """
# llm_interface/response_parser.py
import json
import re
from typing import Dict, Any, List, Optional, Union
""",
    """
# llm_interface/__init__.py
from .llm_engine import LLMEngine, LLMConfig, LLMRequest, LLMResponse, LLMProvider
from . import prompt_templates
from . import response_parser
"""
]

external_dependencies = set()
internal_dependencies = defaultdict(set)
unprovided_modules = set()

module_names = {
    'graph_engine.py': 'graph.graph_engine',
    'pattern_mining.py': 'graph.pattern_mining',
    'graph_pattern_interface.py': 'graph.graph_pattern_interface',
    'graph_module_manager.py': 'graph.graph_module_manager',
    'resource_manager.py': 'resource_manager',
    'llm_engine.py': 'llm_engine',
    'llm_interface/prompt_templates.py': 'llm_interface.prompt_templates',
    'llm_interface/response_parser.py': 'llm_interface.response_parser',
    'llm_interface/__init__.py': 'llm_interface'
}

provided_grandnexus_modules = {
    'grandnexus.graph.graph_engine',
    'grandnexus.graph.pattern_mining',
    'grandnexus.graph.graph_pattern_interface',
    'grandnexus.graph.graph_module_manager',
    'grandnexus.resource_manager',
    'grandnexus.llm_engine', # Note: This is referenced, but the file is llm_interface/llm_engine.py
    'grandnexus.llm_interface.prompt_templates',
    'grandnexus.llm_interface.response_parser',
    'grandnexus.llm_interface'
}


for cell_content in code_cells:
    # Extract potential module name from comment or assume based on content
    cell_name_match = re.search(r'^#\s*(.*)\.py', cell_content, re.MULTILINE)
    if cell_name_match:
        cell_file_name = cell_name_match.group(1).strip() + '.py'
    elif 'graph_pattern_interface.py' in cell_content:
         cell_file_name = 'graph_pattern_interface.py'
    elif 'graph_module_manager.py' in cell_content:
         cell_file_name = 'graph_module_manager.py'
    elif 'llm_interface/prompt_templates.py' in cell_content:
         cell_file_name = 'llm_interface/prompt_templates.py'
    elif 'llm_interface/response_parser.py' in cell_content:
         cell_file_name = 'llm_interface/response_parser.py'
    elif 'llm_interface/__init__.py' in cell_content:
         cell_file_name = 'llm_interface/__init__.py'
    elif 'psutil' in cell_content and 'threading' in cell_content and 'heapq' in cell_content:
        # Heuristic to differentiate resource_manager versions
        if 'NexusCore' in cell_content:
             cell_file_name = 'resource_manager_v2.py'
        else:
             cell_file_name = 'resource_manager_v1.py'
    elif 'graph_engine' in cell_content and 'Neo4jGraphEngine' in cell_content:
        # Heuristic to differentiate graph_engine versions
        if 'os' in cell_content and 'random' in cell_content:
             cell_file_name = 'graph_engine_v2.py'
        else:
             cell_file_name = 'graph_engine_v1.py'
    elif 'pandas' in cell_content and 'numpy' in cell_content and 'Ticket Number' in cell_content:
         cell_file_name = 'ticket_analysis.py' # Give it a descriptive name
    else:
         cell_file_name = 'unknown_cell.py'

    current_module_base = module_names.get(cell_file_name)
    if current_module_base is None:
        # Try to map based on heuristics if not in the direct map
        if 'graph_engine_v' in cell_file_name:
            current_module_base = 'graph.graph_engine'
        elif 'resource_manager_v' in cell_file_name:
             current_module_base = 'resource_manager'
        elif 'llm_engine.py' in cell_file_name:
             current_module_base = 'llm_engine' # This naming is inconsistent with file path
        elif 'ticket_analysis.py' in cell_file_name:
             current_module_base = 'analysis.ticket_analysis' # Placeholder name
        else:
             current_module_base = cell_file_name.replace('/', '.').replace('.py', '')


    # Find all import statements
    imports = re.findall(r'^(?:import\s+([\w.]+))|(?:from\s+([\w.]+)\s+import)', cell_content, re.MULTILINE)

    for imp in imports:
        external_lib = imp[0] if imp[0] else imp[1] # Get the imported name

        if external_lib.startswith('grandnexus.'):
            # Internal GrandNexus dependency
            dep_module = external_lib

            if current_module_base:
                 internal_dependencies[f'grandnexus.{current_module_base}'].add(dep_module)

            # Check if the dependency is provided
            # Normalize dependency name (remove specific imported objects)
            base_dep_module = dep_module.split('.')[0:3] # grandnexus.module.submodule
            base_dep_module_name = '.'.join(base_dep_module)

            is_provided = False
            for provided_module_base in provided_grandnexus_modules:
                if dep_module.startswith(provided_module_base):
                    is_provided = True
                    break

            if not is_provided:
                unprovided_modules.add(dep_module)

        else:
            # External library
            # Clean up potential 'as' aliases and imported submodules for external libs
            base_lib = external_lib.split('.')[0]
            if base_lib not in ['logging', 'time', 'uuid', 'threading', 'abc', 'dataclasses', 'typing', 'collections', 'math', 'json', 'os', 'random', 're', 'hashlib', 'enum', 'auto', 'queue', 'heapq', 'psutil']: # Standard libraries
                external_dependencies.add(base_lib)


print("External Dependencies:")
for dep in sorted(list(external_dependencies)):
    print(f"- {dep}")

print("\nInternal Dependencies:")
for module, deps in internal_dependencies.items():
    print(f"- {module} depends on:")
    for dep in sorted(list(deps)):
        print(f"  - {dep}")

print("\nUnprovided GrandNexus Modules/Paths (based on imports):")
for mod in sorted(list(unprovided_modules)):
    print(f"- {mod}")

External Dependencies:
- 
- neo4j
- numpy
- pandas
- traceback

Internal Dependencies:
- grandnexus.graph.pattern_mining depends on:
  - grandnexus.graph.graph_engine
  - grandnexus.graph.graph_pattern_interface
- grandnexus.graph.graph_pattern_interface depends on:
  - grandnexus.graph.graph_engine
- grandnexus.graph.graph_module_manager depends on:
  - grandnexus.core.nexus_core
  - grandnexus.core.registry
  - grandnexus.graph.graph_engine
  - grandnexus.graph.graph_pattern_interface
  - grandnexus.graph.pattern_mining
- grandnexus.resource_manager depends on:
  - grandnexus.core.nexus_core
- grandnexus.llm_engine depends on:
  - grandnexus.core.nexus_core
  - grandnexus.graph.graph_engine
  - grandnexus.memory.episodic
  - grandnexus.memory.semantic
  - grandnexus.memory.working

Unprovided GrandNexus Modules/Paths (based on imports):
- grandnexus.core.nexus_core
- grandnexus.core.registry
- grandnexus.memory.episodic
- grandnexus.memory.semantic
- grandnexus.memory.working


## Create a grandnexus package structure

### Subtask:
Organize the code into a proper Python package structure to resolve relative import issues.


**Reasoning**:
The first step is to create the necessary directory structure for the grandnexus package and its submodules based on the identified dependencies and the requested structure. This involves creating the main grandnexus directory and the core, graph, memory, and llm_interface subdirectories.



In [ ]:
import os

# Define the base directory for the package
base_dir = 'grandnexus'

# Define the subdirectories
subdirs = ['core', 'graph', 'memory', 'llm_interface']

# Create the base directory
os.makedirs(base_dir, exist_ok=True)
print(f"Created directory: {base_dir}")

# Create the subdirectories
for subdir in subdirs:
    path = os.path.join(base_dir, subdir)
    os.makedirs(path, exist_ok=True)
    print(f"Created directory: {path}")

Created directory: grandnexus
Created directory: grandnexus/core
Created directory: grandnexus/graph
Created directory: grandnexus/memory
Created directory: grandnexus/llm_interface


In [ ]:
%pip install numpy networkx neo4j pandas psutil regex cudf-pandas

# Note: Installation of specific LLM provider libraries (like openai, anthropic, deepseek,
# transformers, sentence-transformers) will be handled when we initialize the LLM engine
# based on the configured provider, as they might not all be necessary.
# We will also need to address the missing 'grandnexus.core' and 'grandnexus.memory'
# modules later in the plan by creating placeholders or simplifying the code.

ERROR: Could not find a version that satisfies the requirement cudf-pandas (from versions: none)
ERROR: No matching distribution found for cudf-pandas


In [ ]:
import os

# Create __init__.py in the base directory
base_init_path = os.path.join('grandnexus', '__init__.py')
with open(base_init_path, 'w') as f:
    f.write("# This file makes the 'grandnexus' directory a Python package\n")
print(f"Created: {base_init_path}")

# Create __init__.py in subdirectories
subdirs = ['core', 'graph', 'memory', 'llm_interface']
for subdir in subdirs:
    init_path = os.path.join('grandnexus', subdir, '__init__.py')
    with open(init_path, 'w') as f:
        f.write(f"# This file makes the '{subdir}' directory a Python package\n")
    print(f"Created: {init_path}")

# Create __init__.py in nested llm_interface directory
llm_interface_init_path = os.path.join('grandnexus', 'llm_interface', '__init__.py')
with open(llm_interface_init_path, 'w') as f:
     f.write("# This file makes the 'llm_interface' directory a Python package\n")
print(f"Created: {llm_interface_init_path}")